# Gemma / Olympic mathematics: Ngoc-matched PI configuration

Configuration reference: DATN-FPTU main, commit `786291e`. Training, PI, NAFC and evaluation formulas follow Ngoc.
The model is Gemma 3-1B-IT. Data remains the previously selected Olympic experiment: NuminaMath training and
Omni-MATH evaluation. Ngoc's MATH questions are not used.

| Setting | Matched configuration |
|---|---|
| Arms | PI monitor, PI + NAFC, PI + fixed replay, PI + KL, PI + FlipGuard |
| Seeds | 1, 2, 3: 15 training runs, all reported |
| Train | The frozen 2,000 NuminaMath-1.5 numerical-answer questions from olympiads_ref / olympiads |
| Test | The frozen 2,000 numerical-answer Omni-MATH questions |
| Response cap | 1,536 tokens throughout; no longer-cap test |
| PI | Budget 358; eta 0.05; kp 4; signed lambda in [-3, 3] |
| Training | TRL 1.13 + vLLM 0.19.1; BF16; 300 steps; 8 prompts x 8 generations |
| LoRA / optimizer | r32 all linear layers; lr 1e-5; 10 warmup steps; micro-batch 4 |
| Anchors | Initial pool 800; extend to the pre-frozen 2,048 if insufficient; choose 128 at >=6/8 |
| Split / audit | 64 repair + 64 held, split seed 0; 4 draws every 25 steps |
| Controls | KL beta 0.04; FlipGuard gamma 0.1; fixed replay gamma 0.5, max fraction 0.5 |
| Tests | 4 draws at steps 50, 100, 150, 200, 300, evaluation seed 1 |
| Base reference | 8 draws, then independent 4-draw noise check, at 1,536 tokens only |

The frozen train/test partitions are preserved from the Olympic experiment, not replaced with Ngoc's MATH split.
The initial anchor pool is the first 800 candidates in the already frozen Numina candidate order. If it cannot
provide 128 eligible anchors, selection extends to the remaining candidates in the pre-frozen 2,048-question bank.
This resource adjustment follows the observed shortage of 66/128 in the initial pool; it is not an identical
800-candidate replication. Threshold, training data, test data and decoding remain fixed. Selection batches are
cached, eligible initial results are reused, and the final selected 128 anchors receive a fresh 4-draw noise check.
A data adapter supplies those rows; prepare_buckets adds only the Olympic dataset CLI choice.
The other seven upstream scripts have no source-text edits. No Qwen scores, anchors, noise allowance or adapters
are reused. Omni-MATH is a competition-mathematics numeric subset, not an exclusively IMO benchmark or a full
official Omni-MATH score. Exact deduplication against the full Omni-MATH corpus was performed when freezing data.

The old output contains 64 validation questions, 128 anchors and 208 reference-test questions, all at 2,048 tokens.
It is preserved separately and is **not** resumed or pooled into this experiment. No optimizer training had started.
This new run must prepare Gemma-specific references and anchors with the upstream settings.

Select **A100**, fill the Hugging Face token below, then **Run all**. The notebook needs no repository or ZIP input
for a fresh run. Preparation includes upstream train-bucket sampling (2,000 x 8 responses) as well as test-reference
sampling (2,000 x [8 + 4]); these are shared by all 15 runs. No measured CU or single-session completion guarantee.
At the end, click **Download all output (.zip)**. Until downloaded, output exists only in the runtime.


## 1. Access and optional resume

In [ ]:
import os
HF_TOKEN = "" # @param {type:"string"}
RESTORE_MATCHED_OUTPUT = False # @param {type:"boolean"}
if not HF_TOKEN.strip():
    raise ValueError("Fill HF_TOKEN before Run all; accept the Gemma license on that Hugging Face account.")
os.environ["HF_TOKEN"] = HF_TOKEN.strip()
os.environ["USE_TF"] = "0"
os.environ["HF_HOME"] = "/content/hf_cache"


## 2. Install Ngoc's runtime

Training runs in fresh subprocesses, so they import the installed versions even if the notebook kernel previously
loaded other libraries. Math-Verify is installed without a version pin, as in Ngoc's notebook; its resolved version
is recorded and checked on resume. Do not bypass an installation or compatibility error.


In [ ]:
import sys, subprocess
from pathlib import Path
ROOT = Path("/content/gemma_ngoc")
ROOT.mkdir(exist_ok=True)
ARCHIVES = Path("/content/nafc_archives")
ARCHIVES.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
    "torch==2.10.0+cu128", "vllm==0.19.1", "trl==1.13.0", "transformers==5.18.0", "peft==0.21.1",
    "math-verify"], check=True)
subprocess.run([sys.executable, "-c", "import torch,vllm,trl,transformers,peft; print(torch.__version__,vllm.__version__,trl.__version__,transformers.__version__,peft.__version__); assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0), 'Select A100'"], check=True)


## 3. Embedded upstream code and matched plan

### `build_anchors.py`

In [ ]:
%%writefile /content/gemma_ngoc/build_anchors.py
"""
build_anchors.py — Chọn câu anchor cho ràng buộc đúng → sai và đo mức nhiễu của hai cách đếm (chạy 1 lần).

Anchor = câu model gốc giải đúng ≥ 6/8 mẫu (p̂ ≥ 0,75), lấy từ phần train KHÔNG dùng để train: bỏ đúng
`--train_limit` câu mà prepare_buckets.py đã lấy (shuffle seed 0), bỏ câu trùng đề với tập test.

Mức nhiễu ghi vào meta (model gốc chấm lại trên chính các anchor, 4 mẫu, seed khác):
  noise_robust = tỷ lệ anchor có ≤ 1/4 mẫu đúng            → mức nhiễu của cách đếm "thật"  (NAFC, robust_fixed)
  noise_raw    = tỷ lệ anchor mẫu đầu lần chọn đúng mà mẫu đầu lần chấm lại sai → mức nhiễu của cách đếm "thô"
                                                                                      (FlipGuard, raw_adaptive)
Ví dụ:
  python build_anchors.py --model Qwen/Qwen2.5-1.5B-Instruct --dataset math --train_limit 2000 --out anchors.json
"""

from __future__ import annotations

import argparse
import json
import random
import re
import statistics

from common import chat_messages, is_correct, load_split


def norm_q(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())


def select_pool(full: list[dict], used: list[dict], test: list[dict], pool: int, seed: int) -> tuple[list[dict], dict]:
    """Phần train chưa dùng, không trùng đề test, lấy ngẫu nhiên `pool` câu."""
    used_q = {norm_q(r["question"]) for r in used}
    test_q = {norm_q(r["question"]) for r in test}
    rest = [r for r in full if norm_q(r["question"]) not in used_q]
    clean = [r for r in rest if norm_q(r["question"]) not in test_q]
    rng = random.Random(seed + 1)
    chosen = rng.sample(clean, min(pool, len(clean)))
    return chosen, {"train_total": len(full), "train_used": len(used), "train_rest": len(rest),
                    "dup_with_test_removed": len(rest) - len(clean), "pool": len(chosen)}


def pick_anchors(scored: list[dict], n_anchor: int, hi: float, seed: int) -> list[dict]:
    cand = [x for x in scored if x["p_select"] >= hi and x["base_response"] is not None]
    rng = random.Random(seed)
    rng.shuffle(cand)
    return cand[:n_anchor]


def noise_floors(anchors: list[dict], lo: float) -> dict:
    n = max(len(anchors), 1)
    return {"noise_robust": sum(x["p_check"] <= lo for x in anchors) / n,
            "noise_raw": sum(x["base_first_correct"] and not x["check_first_correct"] for x in anchors) / n}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--dataset", default="math")
    ap.add_argument("--train_limit", type=int, default=2000, help="= LIMIT_TRAIN lúc tạo buckets train")
    ap.add_argument("--pool", type=int, default=800, help="số câu train chưa dùng đem chấm")
    ap.add_argument("--n_anchor", type=int, default=128)
    ap.add_argument("--k_select", type=int, default=8)
    ap.add_argument("--k_check", type=int, default=4)
    ap.add_argument("--hi", type=float, default=0.75)
    ap.add_argument("--lo", type=float, default=0.25)
    ap.add_argument("--max_len", type=int, default=1536)
    ap.add_argument("--temperature", type=float, default=1.0, help="= nhiệt độ lúc train và lúc đánh giá")
    ap.add_argument("--gpu_mem", type=float, default=0.85)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()

    full = load_split(args.dataset, "train", None)
    used = load_split(args.dataset, "train", args.train_limit)
    test = load_split(args.dataset, "test", None)
    pool, stats = select_pool(full, used, test, args.pool, args.seed)
    print("Chọn pool:", stats, flush=True)

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    tok = AutoTokenizer.from_pretrained(args.model)
    llm = LLM(args.model, gpu_memory_utilization=args.gpu_mem, max_model_len=args.max_len + 1024)

    def prompts(rows):
        return [tok.apply_chat_template(chat_messages(r["question"]), tokenize=False, add_generation_prompt=True)
                for r in rows]

    outs = llm.generate(prompts(pool), SamplingParams(n=args.k_select, temperature=args.temperature, top_p=1.0,
                                                      max_tokens=args.max_len, seed=args.seed))
    scored = []
    for r, o in zip(pool, outs):
        ok = [is_correct(c.text, r["answer"]) for c in o.outputs]
        good = [c for c, g in zip(o.outputs, ok) if g]
        scored.append({"question": r["question"], "answer": r["answer"], "p_select": sum(ok) / len(ok),
                       "base_first_correct": bool(ok[0]),
                       "base_response": good[0].text if good else None,
                       "base_response_ids": list(good[0].token_ids) if good else None,
                       "base_response_len": len(good[0].token_ids) if good else None})
    anchors = pick_anchors(scored, args.n_anchor, args.hi, args.seed)
    n_cand = sum(x["p_select"] >= args.hi for x in scored)
    if len(anchors) < args.n_anchor:
        print(f"CẢNH BÁO: chỉ có {len(anchors)} câu đạt p̂ ≥ {args.hi} trong pool {len(pool)}; tăng --pool nếu cần đủ "
              f"{args.n_anchor}.", flush=True)

    outs2 = llm.generate(prompts(anchors), SamplingParams(n=args.k_check, temperature=args.temperature, top_p=1.0,
                                                          max_tokens=args.max_len, seed=args.seed + 1))
    for x, o in zip(anchors, outs2):
        ok = [is_correct(c.text, x["answer"]) for c in o.outputs]
        x["p_check"] = sum(ok) / len(ok)
        x["check_first_correct"] = bool(ok[0])

    meta = {"model": args.model, "dataset": args.dataset, "train_limit": args.train_limit, **stats,
            "candidates": n_cand, "n_anchor": len(anchors), "k_select": args.k_select, "k_check": args.k_check,
            "hi": args.hi, "lo": args.lo, "max_len": args.max_len, "temperature": args.temperature,
            "pool_base_acc": statistics.mean(x["p_select"] for x in scored),
            "anchor_resp_len_mean": statistics.mean(x["base_response_len"] for x in anchors) if anchors else None,
            **noise_floors(anchors, args.lo)}
    json.dump({"meta": meta, "items": anchors}, open(args.out, "w"), ensure_ascii=False)
    print(json.dumps(meta, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    import os as _os, sys as _sys
    main()
    _sys.stdout.flush(); _sys.stderr.flush()
    _os._exit(0)   # không chờ tiến trình nền của vLLM

### `common.py`

In [ ]:
%%writefile /content/gemma_ngoc/common.py
"""
common.py — prompt, trích đáp án, chấm đúng/sai, nạp dữ liệu.

Dùng chung cho prepare_buckets.py, train.py, evaluate.py.
"""

from __future__ import annotations

import re

SYSTEM_PROMPT = "Please reason step by step, and put your final answer within \\boxed{}."

# Tên dataset và cột. Kiểm tra lại trên Hugging Face trước khi chạy nếu đổi dataset.
DATASETS = {
    "gsm8k": dict(path="openai/gsm8k", name="main", question="question", answer="answer"),
    "math500": dict(path="HuggingFaceH4/MATH-500", name=None, question="problem", answer="answer"),
    "math": dict(path="DigitalLearningGmbH/MATH-lighteval", name=None, question="problem", answer="solution"),
}


def gold_answer(dataset: str, raw: str) -> str:
    if dataset == "gsm8k":
        return raw.split("####")[-1].strip().replace(",", "")
    if dataset == "math":
        return (last_boxed(raw) or "").strip()
    return raw.strip()


def load_split(dataset: str, split: str, limit: int | None = None):
    """Trả về list dict {question, answer} đã chuẩn hoá."""
    from datasets import load_dataset

    cfg = DATASETS[dataset]
    ds = load_dataset(cfg["path"], cfg["name"], split=split) if cfg["name"] else load_dataset(cfg["path"], split=split)
    if limit and limit < len(ds):
        ds = ds.shuffle(seed=0).select(range(limit))   # lấy ngẫu nhiên, tránh dồn vào một chủ đề
    rows = [{"question": r[cfg["question"]], "answer": gold_answer(dataset, r[cfg["answer"]])} for r in ds]
    return [r for r in rows if r["answer"]]


def chat_messages(question: str) -> list[dict]:
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]


# ---------------------------------------------------------------------- #
def last_boxed(text: str) -> str | None:
    """Nội dung của \\boxed{...} cuối cùng, xử lý ngoặc lồng nhau."""
    idx = text.rfind("\\boxed")
    if idx < 0:
        return None
    i = text.find("{", idx)
    if i < 0:
        return None
    depth, j = 0, i
    while j < len(text):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[i + 1 : j]
        j += 1
    return None


def _normalize(s: str) -> str:
    s = s.strip().strip("$").replace(" ", "").replace("\\!", "").replace(",", "")
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac").replace("\\left", "").replace("\\right", "")
    s = s.rstrip(".")
    return s


def _as_number(s: str) -> float | None:
    s = _normalize(s)
    m = re.fullmatch(r"-?\d+(\.\d+)?", s)
    if m:
        return float(s)
    m = re.fullmatch(r"(-?)\\frac\{(-?\d+)\}\{(\d+)\}", s)
    if m and int(m.group(3)) != 0:
        sign = -1 if m.group(1) else 1
        return sign * int(m.group(2)) / int(m.group(3))
    return None


def is_correct(completion: str, gold: str) -> bool:
    """Chấm đúng/sai. Ưu tiên math_verify nếu đã cài, ngược lại so sánh chuẩn hoá."""
    pred = last_boxed(completion)
    if pred is None:
        return False
    try:
        from math_verify import parse, verify

        g, p = parse(f"${gold}$"), parse(f"${pred}$")
        if g and p:
            return bool(verify(g, p))
    except Exception:
        pass
    a, b = _as_number(pred), _as_number(gold)
    if a is not None and b is not None:
        return abs(a - b) < 1e-6 * max(1.0, abs(b))
    return _normalize(pred) == _normalize(gold)


def completion_text(c) -> str:
    """TRL truyền completion dạng chuỗi (standard) hoặc list message (conversational)."""
    if isinstance(c, list):
        return c[-1]["content"] if c else ""
    return c

### `constraints.py`

In [ ]:
%%writefile /content/gemma_ngoc/constraints.py
"""
constraints.py — Bộ điều khiển nhân tử Lagrange cho GRPO có ràng buộc độ dài.

File này KHÔNG import torch/trl, nên có thể test độc lập (xem test_constraints.py).

Các phương pháp:
  - "grpo"       : GRPO thường, chỉ reward đúng/sai (không ràng buộc)
  - "fixed"      : penalty cố định, lambda không đổi (cost clipped như LACONIC)
  - "leash"      : Leash (Li et al., 2025) — reward {-1,+1}, penalty một phía, clip [-1,1],
                   dual update theo độ dài trung bình toàn cục
  - "laconic"    : LACONIC (Liu et al., 2026) — reward {0,1}, clipped cost,
                   dual update theo độ dài trung bình toàn cục, trần Lambda = B/(Lmax-B)
  - "dalg"       : ĐỀ XUẤT — K nhân tử độc lập, mỗi nhóm độ khó một ngân sách B_k
  - "dalg_param" : ĐỀ XUẤT — lambda_k = clip(a + b*d_k, 0, Lambda_k), chỉ 2 tham số đối ngẫu

Tuỳ chọn ổn định hoá (RQ3):
  - optimistic=True : dùng gradient dự đoán 2*g_t - g_{t-1} thay cho g_t
  - init_lambdas    : khởi tạo lambda từ tập hiệu chỉnh (xem calibrate_lambda)
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field


def clipped_cost(length: int, budget: float) -> float:
    """Chi phí bị cắt của LACONIC: bằng 0 nếu trong ngân sách."""
    return max((length - budget) / budget, 0.0)


@dataclass
class LagrangianController:
    method: str
    budgets: list[float]            # B_k; với phương pháp toàn cục chỉ dùng budgets[0] = B
    max_len: int                    # L_max = max_completion_length
    difficulty: list[float] | None = None   # d_k trong [0,1], 0 = dễ, 1 = khó (dalg_param)
    eta: float = 0.002              # bước cập nhật đối ngẫu
    eta_param: float = 0.002        # bước cập nhật cho (a, b) của dalg_param
    fixed_lambda: float = 0.01
    optimistic: bool = False
    init_lambdas: list[float] | None = None
    leash_lambda_max: float = 1.0
    two_sided: bool = False         # λ được âm: ràng buộc coi là chủ động (E[L] = B), chống ép quá tay
    kp: float = 0.0                 # hệ số tỉ lệ (PI): λ = I + kp * vi_phạm, phản ứng ngay khi độ dài vượt/thiếu

    lambdas: list[float] = field(init=False)
    _prev_grad: list[float] = field(init=False)
    _a: float = field(init=False, default=0.0)
    _b: float = field(init=False, default=0.0)
    _prev_ab_grad: tuple[float, float] = field(init=False, default=(0.0, 0.0))
    _integral: list[float] = field(init=False)

    def __post_init__(self):
        valid = {"grpo", "fixed", "leash", "laconic", "dalg", "dalg_param"}
        if self.method not in valid:
            raise ValueError(f"method phải thuộc {valid}")
        k = self.num_groups
        if self.init_lambdas is not None:
            self.lambdas = [min(max(l, 0.0), self.ceiling(i)) for i, l in enumerate(self.init_lambdas[:k])]
        elif self.method == "fixed":
            self.lambdas = [self.fixed_lambda] * k
        else:
            self.lambdas = [0.0] * k
        self._prev_grad = [0.0] * k
        self._integral = list(self.lambdas)
        if self.method == "dalg_param":
            if self.difficulty is None or len(self.difficulty) != len(self.budgets):
                raise ValueError("dalg_param cần difficulty cùng độ dài với budgets")
            # Khởi tạo (a, b): đường thẳng qua lambda của nhóm dễ nhất và khó nhất (nếu có init), ngược lại (0, 0)
            if self.init_lambdas is not None:
                d0, d1 = self.difficulty[0], self.difficulty[-1]
                y0, y1 = self.lambdas[0], self.lambdas[-1]
                self._b = (y1 - y0) / (d1 - d0) if d1 != d0 else 0.0
                self._a = y0 - self._b * d0
            else:
                self._a, self._b = 0.0, 0.0
            self._sync_param_lambdas()

    # ------------------------------------------------------------------ #
    def state_dict(self) -> dict:
        """Lưu trạng thái đối ngẫu để tiếp tục train khi Colab bị ngắt."""
        return {"lambdas": self.lambdas, "prev_grad": self._prev_grad,
                "a": self._a, "b": self._b, "prev_ab_grad": list(self._prev_ab_grad),
                "integral": list(self._integral)}

    def load_state_dict(self, d: dict) -> None:
        self.lambdas = list(d["lambdas"])
        self._prev_grad = list(d["prev_grad"])
        self._a, self._b = d["a"], d["b"]
        self._prev_ab_grad = tuple(d["prev_ab_grad"])
        self._integral = list(d.get("integral", self.lambdas))

    @property
    def num_groups(self) -> int:
        return len(self.budgets) if self.method in ("dalg", "dalg_param") else 1

    def group_of(self, bucket: int) -> int:
        return bucket if self.method in ("dalg", "dalg_param") else 0

    def ceiling(self, k: int) -> float:
        """Trần của lambda. LACONIC: Lambda = B/(Lmax - B) để lời giải đúng luôn có reward > 0."""
        if self.method == "leash":
            return self.leash_lambda_max
        b = self.budgets[k] if self.method in ("dalg", "dalg_param") else self.budgets[0]
        return b / max(self.max_len - b, 1.0)

    def floor(self, k: int) -> float:
        """Sàn của λ: 0 cho ràng buộc bất đẳng thức; -trần khi two_sided."""
        return -self.ceiling(k) if self.two_sided else 0.0

    def budget(self, k: int) -> float:
        return self.budgets[k] if self.method in ("dalg", "dalg_param") else self.budgets[0]

    def _sync_param_lambdas(self):
        self.lambdas = [
            min(max(self._a + self._b * d, 0.0), self.ceiling(i))
            for i, d in enumerate(self.difficulty)
        ]

    # ------------------------------------------------------------------ #
    def shaped_rewards(self, correct: list[bool], lengths: list[int], buckets: list[int]) -> list[float]:
        """Primal: reward Lagrange cho từng completion (dùng lambda hiện tại)."""
        out = []
        for ok, L, bk in zip(correct, lengths, buckets):
            k = self.group_of(bk)
            B = self.budget(k)
            lam = self.lambdas[k]
            if self.method == "grpo":
                out.append(1.0 if ok else 0.0)
            elif self.method == "leash":
                r = 1.0 if ok else -1.0
                out.append(min(max(r - lam * clipped_cost(L, B), -1.0), 1.0))
            else:  # fixed, laconic, dalg, dalg_param
                r = 1.0 if ok else 0.0
                out.append(r - lam * clipped_cost(L, B))
        return out

    def dual_update(self, lengths: list[int], buckets: list[int]) -> dict[str, float]:
        """Dual: cập nhật lambda từ độ dài trung bình (dùng chi phí tuyến tính L/B - 1, như LACONIC/Leash)."""
        stats: dict[str, float] = {}
        k_total = self.num_groups
        sums = [0.0] * k_total
        counts = [0] * k_total
        for L, bk in zip(lengths, buckets):
            k = self.group_of(bk)
            sums[k] += L
            counts[k] += 1

        grads = [0.0] * k_total
        for k in range(k_total):
            if counts[k] == 0:
                continue
            mean_len = sums[k] / counts[k]
            grads[k] = mean_len / self.budget(k) - 1.0
            stats[f"constraint/mean_len_g{k}"] = mean_len
            stats[f"constraint/violation_g{k}"] = grads[k]

        if self.method in ("grpo", "fixed"):
            pass
        elif self.method == "dalg_param":
            # Gradient ascent trên (a, b): dlambda_k/da = 1, dlambda_k/db = d_k (khi lambda_k không bị cắt).
            # Mỗi nhóm đóng góp theo tỷ lệ số mẫu -> (a, b) học từ TOÀN BỘ batch, ít nhiễu hơn K lambda độc lập.
            n = sum(counts) or 1
            ga = gb = 0.0
            for k in range(k_total):
                if counts[k] == 0:
                    continue
                w = counts[k] / n
                ga += w * grads[k]
                gb += w * grads[k] * self.difficulty[k]
            if self.optimistic:
                pa, pb = self._prev_ab_grad
                step_a, step_b = 2 * ga - pa, 2 * gb - pb
            else:
                step_a, step_b = ga, gb
            self._prev_ab_grad = (ga, gb)
            self._a += self.eta_param * step_a
            self._b += self.eta_param * step_b
            self._sync_param_lambdas()
            stats["constraint/param_a"] = self._a
            stats["constraint/param_b"] = self._b
        else:  # leash, laconic, dalg
            for k in range(k_total):
                if counts[k] == 0:
                    continue
                g = 2 * grads[k] - self._prev_grad[k] if self.optimistic else grads[k]
                self._prev_grad[k] = grads[k]
                lo, hi = self.floor(k), self.ceiling(k)
                if self.kp > 0:   # PI: phần tích phân (có chống tích luỹ quá trần) + phần tỉ lệ
                    self._integral[k] = min(max(self._integral[k] + self.eta * g, lo), hi)
                    self.lambdas[k] = min(max(self._integral[k] + self.kp * grads[k], lo), hi)
                else:
                    self.lambdas[k] = min(max(self.lambdas[k] + self.eta * g, lo), hi)

        for k, lam in enumerate(self.lambdas):
            stats[f"constraint/lambda_g{k}"] = lam
        return stats


# ---------------------------------------------------------------------- #
def calibrate_lambda(
    samples_by_prompt: list[list[tuple[bool, int]]],
    budget: float,
    lam_max: float,
    beta: float = 0.5,
    iters: int = 40,
) -> float:
    """
    Ước lượng lambda ban đầu cho MỘT nhóm từ mẫu của model gốc (chạy offline, trước khi train).

    Ý tưởng (theo hướng dual one-shot của CAN / LARA): policy tối ưu của bài toán
    KL-regularized là pi_lam(o|q) ∝ pi_0(o|q) * exp((r - lam*c)/beta).
    Dùng các mẫu o ~ pi_0 làm importance samples, ước lượng E_{pi_lam}[L] rồi
    tìm lambda nhỏ nhất sao cho E[L] <= budget bằng tìm kiếm nhị phân.

    samples_by_prompt: mỗi phần tử là danh sách (đúng/sai, độ dài) của các mẫu cho một prompt.
    Trả về lam_max nếu kể cả lam_max vẫn không thỏa ràng buộc.
    """

    def expected_len(lam: float) -> float:
        total, n = 0.0, 0
        for group in samples_by_prompt:
            if not group:
                continue
            logits = [((1.0 if ok else 0.0) - lam * clipped_cost(L, budget)) / beta for ok, L in group]
            m = max(logits)
            ws = [math.exp(x - m) for x in logits]
            z = sum(ws)
            total += sum(w * L for w, (_, L) in zip(ws, group)) / z
            n += 1
        return total / max(n, 1)

    if expected_len(0.0) <= budget:
        return 0.0
    if expected_len(lam_max) > budget:
        return lam_max
    lo, hi = 0.0, lam_max
    for _ in range(iters):
        mid = (lo + hi) / 2
        if expected_len(mid) > budget:
            lo = mid
        else:
            hi = mid
    return hi


# ---------------------------------------------------------------------- #
@dataclass
class AccuracyController:
    """
    Phương pháp đề xuất v4 ("acccon"): tối thiểu độ dài với ràng buộc accuracy theo nhóm độ khó

        min E[L]   s.t.   acc_k(t) >= tau_k(t)   cho mọi nhóm k,

    trong đó tau_k(t) = accuracy của GRPO (không ràng buộc) ở nhóm k tại CÙNG bước train t, trừ eps_k.
    Ngưỡng bám theo đường học của GRPO nên lambda không bị dồn lên trần ở giai đoạn đầu (khi mọi
    phương pháp đều còn kém), tránh kiểu phản ứng trễ đã gặp ở LACONIC.

    Lagrangian: max  -E[L] + sum_k lambda_k * acc_k  ->  với câu thuộc nhóm k, advantage (chuẩn hoá tách rời)
        A = (lambda_k * z(đúng) - z(L)) / (1 + lambda_k)
    Dual: lambda_k <- clip(lambda_k + eta * (tau_k(t) - acc_ema_k), lam_min, lam_max),
    acc_ema_k là trung bình trượt của accuracy nhóm k trong batch (mỗi bước chỉ có vài câu mỗi nhóm).
    Khác DALG: việc "nén lan" sang nhóm khác không tự thoả được ràng buộc accuracy — nếu nó làm nhóm
    vừa mất accuracy thì lambda của nhóm vừa TĂNG và kéo lại, chứ không về 0.
    """
    ref_curves: list[list[float]]     # tau_k(t) đã trừ eps, mỗi nhóm một danh sách theo bước
    eta: float = 0.5
    lam_init: float = 1.0
    lam_min: float = 0.5            # = mức ép mạnh nhất LACONIC từng dùng (λ≈2 tương đương 0,5 ở đây)
    lam_max: float = 4.0
    ema: float = 0.9

    lambdas: list[float] = field(init=False)
    acc_ema: list[float | None] = field(init=False)
    step: int = field(init=False, default=0)

    def __post_init__(self):
        k = len(self.ref_curves)
        self.lambdas = [min(max(self.lam_init, self.lam_min), self.lam_max)] * k
        self.acc_ema = [None] * k

    @property
    def num_groups(self) -> int:
        return len(self.ref_curves)

    def target(self, k: int) -> float:
        c = self.ref_curves[k]
        return c[min(self.step, len(c) - 1)]

    def state_dict(self) -> dict:
        return {"lambdas": self.lambdas, "acc_ema": self.acc_ema, "step": self.step}

    def load_state_dict(self, d: dict) -> None:
        self.lambdas, self.acc_ema, self.step = list(d["lambdas"]), list(d["acc_ema"]), int(d["step"])

    def dual_update(self, correct: list[bool], buckets: list[int]) -> dict[str, float]:
        stats: dict[str, float] = {}
        for k in range(self.num_groups):
            idx = [i for i, b in enumerate(buckets) if b == k]
            if not idx:
                continue
            a = sum(correct[i] for i in idx) / len(idx)
            self.acc_ema[k] = a if self.acc_ema[k] is None else self.ema * self.acc_ema[k] + (1 - self.ema) * a
            tau = self.target(k)
            self.lambdas[k] = min(max(self.lambdas[k] + self.eta * (tau - self.acc_ema[k]), self.lam_min), self.lam_max)
            stats[f"constraint/acc_ema_g{k}"] = self.acc_ema[k]
            stats[f"constraint/target_g{k}"] = tau
        for k, lam in enumerate(self.lambdas):
            stats[f"constraint/lambda_g{k}"] = lam
        self.step += 1
        return stats

### `evaluate.py`

In [ ]:
%%writefile /content/gemma_ngoc/evaluate.py
"""
evaluate.py — Đánh giá một model (gốc hoặc đã train) theo đúng các metric của Proposal, mục 6.

Báo cáo: accuracy (pass@1, trung bình trên n mẫu), số token trung bình, và cả hai
THEO TỪNG NHÓM ĐỘ KHÓ, cùng tỷ lệ câu trả lời vượt ngân sách của nhóm.

Nhóm độ khó của tập test lấy từ file của prepare_buckets.py chạy trên split test
(nhóm được xác định bởi model GỐC, cố định cho mọi phương pháp).

Bản 14/10/2026: thêm trường "boxed" (tỷ lệ mẫu có \\boxed{}) cho từng câu, và tuỳ chọn --save_texts
để lưu nguyên văn lời giải (jsonl.gz) — dùng để kiểm tra bài "hỏng" là sai thật hay chỉ thiếu \\boxed{}.
Hai thay đổi này không đổi cách sinh hay cách chấm: acc, len, over, trunc tính y như cũ.

Ví dụ:
  python evaluate.py --model runs/dalg_s0/merged --buckets buckets_gsm8k_test.json --out results/dalg_s0.json
"""

from __future__ import annotations

import argparse
import json
import statistics

from common import chat_messages, is_correct, last_boxed


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--buckets", required=True, help="file json của split test")
    ap.add_argument("--n", type=int, default=4)
    ap.add_argument("--temperature", type=float, default=1.0, help="= nhiệt độ khi train, để đo đúng phân phối đã ràng buộc")
    ap.add_argument("--top_p", type=float, default=1.0)
    ap.add_argument("--seed", type=int, default=1, help="KHÁC seed lúc chia nhóm (0): mẫu đánh giá độc lập với mẫu chia nhóm")
    ap.add_argument("--max_len", type=int, default=None)
    ap.add_argument("--gpu_mem", type=float, default=0.85)
    ap.add_argument("--out", required=True)
    ap.add_argument("--save_texts", default=None, help="đường dẫn .jsonl.gz: lưu lời giải từng câu (mặc định không lưu)")
    args = ap.parse_args()

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    data = json.load(open(args.buckets))
    meta, items = data["meta"], data["items"]
    max_len = args.max_len or meta["max_len"]
    budgets = meta["budgets"]
    K = len(budgets)

    tok = AutoTokenizer.from_pretrained(args.model)
    prompts = [tok.apply_chat_template(chat_messages(it["question"]), tokenize=False, add_generation_prompt=True) for it in items]
    llm = LLM(args.model, gpu_memory_utilization=args.gpu_mem, max_model_len=max_len + 512)
    outs = llm.generate(prompts, SamplingParams(n=args.n, temperature=args.temperature, top_p=args.top_p,
                                                max_tokens=max_len, seed=args.seed))

    per_q = []
    all_ok = []
    for it, o in zip(items, outs):
        ok = [is_correct(c.text, it["answer"]) for c in o.outputs]
        all_ok.append(ok)
        ln = [len(c.token_ids) for c in o.outputs]
        per_q.append({"bucket": it["bucket"], "acc": sum(ok) / len(ok), "len": statistics.mean(ln),
                      "over": sum(L > budgets[it["bucket"]] for L in ln) / len(ln),
                      "trunc": sum(L >= max_len for L in ln) / len(ln),
                      "boxed": sum(last_boxed(c.text) is not None for c in o.outputs) / len(o.outputs)})

    if args.save_texts:
        import gzip
        with gzip.open(args.save_texts, "wt", encoding="utf-8") as f:
            for i, (it, o, ok) in enumerate(zip(items, outs, all_ok)):
                f.write(json.dumps({"i": i, "answer": it["answer"],
                                    "texts": [c.text for c in o.outputs],
                                    "ok": ok,
                                    "len": [len(c.token_ids) for c in o.outputs]}, ensure_ascii=False) + "\n")

    def summarize(rows):
        if not rows:
            return None
        return {"n": len(rows),
                "acc": round(100 * statistics.mean(r["acc"] for r in rows), 2),
                "mean_len": round(statistics.mean(r["len"] for r in rows), 1),
                "over_budget_rate": round(100 * statistics.mean(r["over"] for r in rows), 2),
                "truncated_rate": round(100 * statistics.mean(r["trunc"] for r in rows), 2),
                "boxed_rate": round(100 * statistics.mean(r["boxed"] for r in rows), 2)}

    result = {
        "model": args.model,
        "eval_set": f'{meta["dataset"]}/{meta["split"]}',
        "budget": meta["budget"],
        "decoding": {"temperature": args.temperature, "top_p": args.top_p, "n": args.n, "seed": args.seed},
        "budgets_by_bucket": budgets,
        "overall": summarize(per_q),
        "by_bucket": {k: summarize([r for r in per_q if r["bucket"] == k]) for k in range(K)},
        "per_question": per_q,
    }
    with open(args.out, "w") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(json.dumps({k: v for k, v in result.items() if k != "per_question"}, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    import os as _os, sys as _sys
    main()
    _sys.stdout.flush(); _sys.stderr.flush()
    _os._exit(0)   # thoát ngay: không chờ tiến trình nền của vLLM (từng làm ô treo hàng giờ)


### `fixflip.py`

In [ ]:
%%writefile /content/gemma_ngoc/fixflip.py
"""
fixflip.py — Module đo chung của đồ án: học thêm (sai → đúng) và làm hỏng (đúng → sai), có mức nhiễu.

Chỉ cần numpy. Dùng chung cho cả ba phần:
  - A, nhánh RL (Qwen, MATH, lấy mẫu T = 1, k mẫu mỗi câu),
  - C, nhánh SFT (OLMoE, HotpotQA; giải mã greedy thì k = 1),
  - bất kỳ cặp "model gốc / model sau train" nào chấm trên CÙNG một danh sách câu, CÙNG thứ tự.

Đầu vào chuẩn là `Scores`: với mỗi câu, tỷ lệ đúng p̂ trên k mẫu (cùng k cho mọi câu), tuỳ chọn độ dài
trung bình và nhãn nhóm (ví dụ nhóm độ khó). Đọc được file của evaluate.py, prepare_buckets.py, hoặc JSONL chung.

Các đại lượng (θ₀ = model gốc, θ = model mới, p₀/p₁ = tỷ lệ đúng từng câu):
  - Kỳ vọng 1 mẫu:   fix  = E[(1 − p₀) p₁] / E[1 − p₀]
                      flip = E[p₀ (1 − p₁)] / E[p₀]
    tức xác suất một mẫu của θ sai trên câu mà một mẫu của θ₀ đúng (flip), và ngược lại (fix).
    Luôn có  ΔAcc = E[p₁] − E[p₀] = E[(1 − p₀) p₁] − E[p₀ (1 − p₁)].
  - Chắc chắn:       câu hỏng thật   nếu p₀ ≥ hi và p₁ ≤ lo (mặc định 0,75 / 0,25), chia cho số câu có p₀ ≥ hi;
                      câu học thêm thật nếu p₀ ≤ lo và p₁ ≥ hi, chia cho số câu có p₀ ≤ lo.
  - Mức nhiễu:       cùng phép đo giữa hai lần chấm độc lập của CÙNG một model (model gốc chấm lại khi lấy mẫu,
                      hoặc hai seed cùng cấu hình khi greedy). Câu hỏng "vượt nhiễu" = flip − flip của mức nhiễu.

Ví dụ (dữ liệu phần A):
    base  = Scores.from_buckets_json("buckets_test_B440.json")     # model gốc, 8 mẫu, seed 0
    base2 = Scores.from_eval_json("results/base.json")             # model gốc chấm lại, 4 mẫu, seed 1
    noise = fixflip(base, base2)          # flip ≈ 0,391 (kỳ vọng 1 mẫu), robust_flip ≈ 0,039
    run   = Scores.from_eval_json("results/nafc_s0.json")
    print(fixflip(base, run))
    print(compare(base, run, Scores.from_eval_json("results/grpo_s0.json")))   # chênh lệch + KTC 95%

Dòng lệnh:
    python fixflip.py --base buckets_test_B440.json --base2 results/base.json \
        --runs results/grpo_s0.json results/nafc_s0.json --ref results/grpo_s0.json
"""

from __future__ import annotations

import argparse
import json
import math
from dataclasses import dataclass, field
from typing import Callable, Sequence

import numpy as np

HI, LO = 0.75, 0.25


# --------------------------------------------------------------------------------------------------------------- #
# Dữ liệu
# --------------------------------------------------------------------------------------------------------------- #
@dataclass
class Scores:
    """Kết quả chấm của MỘT model trên một danh sách câu cố định.

    p:      tỷ lệ đúng từng câu, trong [0, 1] (trung bình trên k mẫu)
    k:      số mẫu mỗi câu (1 nếu greedy)
    length: độ dài trung bình từng câu (token), tuỳ chọn
    group:  nhãn nhóm từng câu (int), tuỳ chọn
    ids:    mã câu, tuỳ chọn; nếu cả hai Scores đều có ids thì phép so kiểm tra khớp thứ tự
    """

    p: np.ndarray
    k: int = 1
    length: np.ndarray | None = None
    group: np.ndarray | None = None
    ids: list | None = None
    name: str = ""
    extra: dict = field(default_factory=dict)

    def __post_init__(self):
        self.p = np.asarray(self.p, dtype=float)
        if self.p.ndim != 1:
            raise ValueError("p phải là mảng 1 chiều (một giá trị mỗi câu)")
        if np.any((self.p < 0) | (self.p > 1)):
            raise ValueError("p phải nằm trong [0, 1]")
        if self.length is not None:
            self.length = np.asarray(self.length, dtype=float)
            assert self.length.shape == self.p.shape, "length phải cùng số câu với p"
        if self.group is not None:
            self.group = np.asarray(self.group)
            assert self.group.shape == self.p.shape, "group phải cùng số câu với p"
        if self.ids is not None:
            assert len(self.ids) == len(self.p), "ids phải cùng số câu với p"

    # ---- thống kê đơn ----
    def __len__(self):
        return len(self.p)

    @property
    def acc(self) -> float:
        return float(self.p.mean())

    @property
    def mean_len(self) -> float:
        return float(self.length.mean()) if self.length is not None else float("nan")

    def take(self, idx) -> "Scores":
        """Chọn một tập con câu (mảng bool hoặc chỉ số); dùng cho bootstrap và chia nhóm."""
        idx = np.asarray(idx)
        if idx.dtype == bool:
            idx = np.flatnonzero(idx)
        return Scores(p=self.p[idx], k=self.k,
                      length=None if self.length is None else self.length[idx],
                      group=None if self.group is None else self.group[idx],
                      ids=None if self.ids is None else [self.ids[i] for i in idx],
                      name=self.name, extra=self.extra)

    # ---- đọc file ----
    @classmethod
    def from_correct(cls, correct, length=None, group=None, ids=None, name="") -> "Scores":
        """Từ ma trận đúng/sai (số câu × k mẫu)."""
        c = np.asarray(correct, dtype=float)
        if c.ndim == 1:
            c = c[:, None]
        L = None if length is None else np.asarray(length, dtype=float)
        if L is not None and L.ndim == 2:
            L = L.mean(1)
        return cls(p=c.mean(1), k=c.shape[1], length=L, group=group, ids=ids, name=name)

    @classmethod
    def from_eval_json(cls, path: str, name: str | None = None) -> "Scores":
        """File của evaluate.py: per_question[i] = {acc, len, bucket, ...}; decoding.n = k."""
        d = json.load(open(path))
        q = d["per_question"]
        return cls(p=[x["acc"] for x in q], k=int(d.get("decoding", {}).get("n", 1)),
                   length=[x["len"] for x in q] if "len" in q[0] else None,
                   group=[x["bucket"] for x in q] if "bucket" in q[0] else None,
                   ids=[x["id"] for x in q] if "id" in q[0] else None,
                   name=name or path)

    @classmethod
    def from_buckets_json(cls, path: str, name: str | None = None) -> "Scores":
        """File của prepare_buckets.py: items[i] = {p, bucket, samples: [[đúng, độ dài], ...]}."""
        d = json.load(open(path))
        it = d["items"]
        return cls(p=[x["p"] for x in it], k=int(d.get("n", d.get("meta", {}).get("n", 1))),
                   length=[float(np.mean([s[1] for s in x["samples"]])) for x in it] if "samples" in it[0] else None,
                   group=[x["bucket"] for x in it] if "bucket" in it[0] else None,
                   name=name or path)

    @classmethod
    def from_jsonl(cls, path: str, name: str | None = None) -> "Scores":
        """JSONL chung, mỗi dòng một câu: {"id": ..., "correct": [0/1, ...], "length": [...]?, "group": ...?}.
        Định dạng khuyên dùng cho nhánh SFT (ghi ra từ script đánh giá của bất kỳ framework nào)."""
        rows = [json.loads(line) for line in open(path) if line.strip()]
        ks = {len(r["correct"]) for r in rows}
        if len(ks) != 1:
            raise ValueError(f"mọi câu phải có cùng số mẫu, thấy {sorted(ks)}")
        return cls.from_correct([r["correct"] for r in rows],
                                length=[np.mean(r["length"]) for r in rows] if "length" in rows[0] else None,
                                group=[r["group"] for r in rows] if "group" in rows[0] else None,
                                ids=[r["id"] for r in rows] if "id" in rows[0] else None,
                                name=name or path)

    @classmethod
    def load(cls, path: str, name: str | None = None) -> "Scores":
        if path.endswith(".jsonl"):
            return cls.from_jsonl(path, name)
        d = json.load(open(path))
        if "per_question" in d:
            return cls.from_eval_json(path, name)
        if "items" in d:
            return cls.from_buckets_json(path, name)
        raise ValueError(f"không nhận ra định dạng file {path}")


def _check_pair(a: Scores, b: Scores):
    if len(a) != len(b):
        raise ValueError(f"hai model chấm trên số câu khác nhau: {len(a)} và {len(b)} ({a.name} / {b.name})")
    if a.ids is not None and b.ids is not None and list(a.ids) != list(b.ids):
        raise ValueError(f"thứ tự câu không khớp giữa {a.name} và {b.name}")


# --------------------------------------------------------------------------------------------------------------- #
# Đo học thêm / làm hỏng
# --------------------------------------------------------------------------------------------------------------- #
def per_question(base: Scores, new: Scores, hi: float = HI, lo: float = LO) -> dict[str, np.ndarray]:
    """Các đại lượng TỪNG CÂU; mọi tỷ lệ tổng hợp = tổng tử số / tổng mẫu số. Dùng lại cho bootstrap."""
    _check_pair(base, new)
    p0, p1 = base.p, new.p
    return {
        "acc0": p0, "acc1": p1,
        "fix_num": (1 - p0) * p1, "fix_den": 1 - p0,                       # kỳ vọng 1 mẫu
        "flip_num": p0 * (1 - p1), "flip_den": p0,
        "rfix_num": ((p0 <= lo) & (p1 >= hi)).astype(float), "rfix_den": (p0 <= lo).astype(float),   # chắc chắn
        "rflip_num": ((p0 >= hi) & (p1 <= lo)).astype(float), "rflip_den": (p0 >= hi).astype(float),
    }


def _ratio(num, den):
    s = float(np.sum(den))
    return float(np.sum(num)) / s if s > 0 else float("nan")


def summarize_pq(q: dict[str, np.ndarray]) -> dict[str, float]:
    return {
        "acc_base": float(np.mean(q["acc0"])), "acc_new": float(np.mean(q["acc1"])),
        "delta": float(np.mean(q["acc1"]) - np.mean(q["acc0"])),
        "fix": _ratio(q["fix_num"], q["fix_den"]), "flip": _ratio(q["flip_num"], q["flip_den"]),
        "robust_fix": _ratio(q["rfix_num"], q["rfix_den"]), "robust_flip": _ratio(q["rflip_num"], q["rflip_den"]),
        "n_robust_fix": int(np.sum(q["rfix_num"])), "n_base_wrong": int(np.sum(q["rfix_den"])),
        "n_robust_flip": int(np.sum(q["rflip_num"])), "n_base_right": int(np.sum(q["rflip_den"])),
    }


def fixflip(base: Scores, new: Scores, hi: float = HI, lo: float = LO) -> dict[str, float]:
    """Học thêm / làm hỏng của `new` so với `base`, cả bản kỳ vọng 1 mẫu và bản chắc chắn."""
    out = summarize_pq(per_question(base, new, hi, lo))
    out["mean_len"] = new.mean_len
    return out


def noise_floor(scoring_a: Scores, scoring_b: Scores, hi: float = HI, lo: float = LO) -> dict[str, float]:
    """Mức nhiễu = fixflip giữa hai lần chấm độc lập của CÙNG một model."""
    return fixflip(scoring_a, scoring_b, hi, lo)


def by_group(base: Scores, new: Scores, hi: float = HI, lo: float = LO, groups=None) -> dict:
    """fixflip trong từng nhóm (mặc định nhóm của base, ví dụ nhóm độ khó dễ/vừa/khó)."""
    g = np.asarray(groups if groups is not None else base.group)
    if g is None or g.ndim == 0:
        raise ValueError("cần nhãn nhóm (base.group hoặc tham số groups)")
    return {k.item() if hasattr(k, "item") else k: fixflip(base.take(g == k), new.take(g == k), hi, lo)
            for k in np.unique(g)}


# --------------------------------------------------------------------------------------------------------------- #
# Bootstrap ghép cặp theo câu
# --------------------------------------------------------------------------------------------------------------- #
METRICS = ("acc_new", "delta", "fix", "flip", "robust_fix", "robust_flip")
_NUM_DEN = {"fix": ("fix_num", "fix_den"), "flip": ("flip_num", "flip_den"),
            "robust_fix": ("rfix_num", "rfix_den"), "robust_flip": ("rflip_num", "rflip_den")}


def _boot_idx(n: int, B: int, seed: int, chunk: int = 500):
    rng = np.random.default_rng(seed)
    done = 0
    while done < B:
        m = min(chunk, B - done)
        yield rng.integers(0, n, size=(m, n))
        done += m


def _metric_from_pq(q, metric, idx=None):
    def S(key):
        v = q[key]
        return v.sum() if idx is None else v[idx].sum(axis=-1)
    if metric == "acc_new":
        n = len(q["acc1"]) if idx is None else idx.shape[-1]
        return S("acc1") / n
    if metric == "delta":
        n = len(q["acc1"]) if idx is None else idx.shape[-1]
        return (S("acc1") - S("acc0")) / n
    num, den = _NUM_DEN[metric]
    d = S(den)
    return np.where(d > 0, S(num) / np.where(d > 0, d, 1), np.nan)


def bootstrap_ci(base: Scores, new: Scores, metric: str = "robust_flip", B: int = 4000, seed: int = 0,
                 hi: float = HI, lo: float = LO, alpha: float = 0.05) -> tuple[float, float, float]:
    """(ước lượng, cận dưới, cận trên) của một đại lượng, KTC bootstrap theo câu."""
    q = per_question(base, new, hi, lo)
    est = float(_metric_from_pq(q, metric))
    vals = np.concatenate([_metric_from_pq(q, metric, idx) for idx in _boot_idx(len(base), B, seed)])
    lo_, hi_ = np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return est, float(lo_), float(hi_)


def compare(base: Scores, x: Scores, y: Scores, metrics: Sequence[str] = METRICS, B: int = 4000, seed: int = 0,
            hi: float = HI, lo: float = LO, alpha: float = 0.05) -> dict[str, tuple[float, float, float]]:
    """Chênh lệch x − y (cùng base, cùng câu) cho từng đại lượng, với KTC bootstrap GHÉP CẶP:
    mỗi lần lấy lại mẫu dùng CÙNG tập câu cho cả x và y."""
    qx, qy = per_question(base, x, hi, lo), per_question(base, y, hi, lo)
    out = {}
    for m in metrics:
        est = float(_metric_from_pq(qx, m) - _metric_from_pq(qy, m))
        vals = np.concatenate([_metric_from_pq(qx, m, idx) - _metric_from_pq(qy, m, idx)
                               for idx in _boot_idx(len(base), B, seed)])
        lo_, hi_ = np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        out[m] = (est, float(lo_), float(hi_))
    return out


# --------------------------------------------------------------------------------------------------------------- #
# So ở cùng chi phí: nội suy từng câu giữa hai điểm kẹp hai bên
# --------------------------------------------------------------------------------------------------------------- #
def iso_interp(frontier: Sequence[Scores], target: float,
               cost: Callable[[Scores], float] = lambda s: s.mean_len) -> tuple[Scores, dict]:
    """Model "ảo" trên đường nối các điểm `frontier`, tại chi phí `target`.

    Nội suy tuyến tính TỪNG CÂU giữa hai điểm kẹp hai bên (như phần A so ở cùng độ dài). Không ngoại suy:
    ngoài khoảng thì lấy điểm đầu mút và báo `clamped`. Độ chính xác và học thêm / làm hỏng KỲ VỌNG 1 mẫu
    tuyến tính theo p₁ nên nội suy được chính xác; bản chắc chắn thì nên nội suy chính các tỷ lệ (interp_scalar).

    cost: hàm chi phí của một điểm, mặc định độ dài trung bình; dùng `lambda s: s.acc` để so ở cùng độ chính xác.
    """
    pts = sorted(frontier, key=cost)
    cs = [cost(s) for s in pts]
    for a, b in zip(pts, pts[1:]):
        _check_pair(a, b)
    if target <= cs[0] or target >= cs[-1]:
        j = 0 if target <= cs[0] else len(pts) - 1
        s = pts[j]
        return Scores(p=s.p.copy(), k=s.k, length=None if s.length is None else s.length.copy(), group=s.group,
                      ids=s.ids, name=f"{s.name}@{target:.3g}"), {"lo": j, "hi": j, "w": 0.0, "clamped": True}
    j = int(np.searchsorted(cs, target))
    w = (target - cs[j - 1]) / (cs[j] - cs[j - 1])
    a, b = pts[j - 1], pts[j]
    length = None if a.length is None or b.length is None else (1 - w) * a.length + w * b.length
    s = Scores(p=(1 - w) * a.p + w * b.p, k=min(a.k, b.k), length=length, group=a.group, ids=a.ids,
               name=f"interp({a.name},{b.name})@{target:.3g}")
    return s, {"lo": j - 1, "hi": j, "w": float(w), "clamped": False}


def interp_scalar(costs: Sequence[float], values: Sequence[float], target: float) -> float:
    """Nội suy tuyến tính một đại lượng vô hướng theo chi phí (không ngoại suy)."""
    order = np.argsort(costs)
    c, v = np.asarray(costs, float)[order], np.asarray(values, float)[order]
    return float(np.interp(target, c, v))


# --------------------------------------------------------------------------------------------------------------- #
# In bảng
# --------------------------------------------------------------------------------------------------------------- #
def pct(x: float, nd: int = 1) -> str:
    return "—" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{100 * x:.{nd}f}"


def table(base: Scores, runs: Sequence[Scores], noise: dict | None = None, hi: float = HI, lo: float = LO) -> str:
    """Bảng gọn: accuracy, độ dài, học thêm / làm hỏng (kỳ vọng và chắc chắn), làm hỏng vượt nhiễu."""
    hdr = (f"{'run':28s} {'acc':>6s} {'len':>6s} {'Δacc':>6s} {'fix':>6s} {'flip':>6s} "
           f"{'rfix':>6s} {'rflip':>6s} {'vượt nhiễu':>10s}")
    lines = [hdr, "-" * len(hdr)]
    for r in runs:
        m = fixflip(base, r, hi, lo)
        ex = m["robust_flip"] - noise["robust_flip"] if noise else float("nan")
        lines.append(f"{(r.name or '?')[-28:]:28s} {pct(m['acc_new']):>6s} {r.mean_len:6.0f} {pct(m['delta']):>6s} "
                     f"{pct(m['fix']):>6s} {pct(m['flip']):>6s} {pct(m['robust_fix']):>6s} {pct(m['robust_flip']):>6s} "
                     f"{pct(ex):>10s}")
    if noise:
        lines.append(f"{'mức nhiễu (gốc chấm lại)':28s} {'':6s} {'':6s} {'':6s} {pct(noise['fix']):>6s} "
                     f"{pct(noise['flip']):>6s} {pct(noise['robust_fix']):>6s} {pct(noise['robust_flip']):>6s}")
    lines.append(f"(%; fix/flip = kỳ vọng 1 mẫu; rfix/rflip = chắc chắn, ngưỡng {hi}/{lo})")
    return "\n".join(lines)


def fmt_compare(res: dict[str, tuple[float, float, float]], label: str = "") -> str:
    parts = [f"{m} {100 * e:+.2f} [{100 * l:+.2f}; {100 * h:+.2f}]" for m, (e, l, h) in res.items()]
    return (label + ": " if label else "") + " | ".join(parts)


def main():
    ap = argparse.ArgumentParser(description="Bảng học thêm / làm hỏng có mức nhiễu")
    ap.add_argument("--base", required=True, help="model gốc (vd buckets_test_B440.json, 8 mẫu)")
    ap.add_argument("--base2", default=None, help="model gốc chấm lại độc lập (vd results/base.json) để đo mức nhiễu")
    ap.add_argument("--runs", nargs="+", required=True)
    ap.add_argument("--ref", default=None, help="so từng run với run này (vd GRPO cùng seed), KTC bootstrap ghép cặp")
    ap.add_argument("--hi", type=float, default=HI)
    ap.add_argument("--lo", type=float, default=LO)
    ap.add_argument("--B", type=int, default=2000)
    args = ap.parse_args()

    base = Scores.load(args.base, "base")
    noise = noise_floor(base, Scores.load(args.base2, "base2"), args.hi, args.lo) if args.base2 else None
    runs = [Scores.load(p) for p in args.runs]
    print(table(base, runs, noise, args.hi, args.lo))
    if args.ref:
        ref = Scores.load(args.ref)
        for r in runs:
            if r.name == ref.name:
                continue
            print(fmt_compare(compare(base, r, ref, B=args.B, hi=args.hi, lo=args.lo), f"{r.name} − {ref.name}"))


if __name__ == "__main__":
    main()

### `flipcon.py`

In [ ]:
%%writefile /content/gemma_ngoc/flipcon.py
"""
flipcon.py — Ràng buộc đúng → sai (chống làm hỏng câu model gốc đã giải được) cho GRPO trên TRL.

Một lớp, năm chế độ; hai "công tắc" của ablation 2×2 là cách đếm câu hỏng và cách đặt λ:

  mode           đếm câu hỏng                 λ                                   vai trò
  ------------   --------------------------   ---------------------------------   -------------------------------
  nafc           thật: p^_θ ≤ 0,25 trên k mẫu  tự điều chỉnh theo ngân sách ε      phương pháp đề xuất (NAFC)
  flipguard      thô: 1 mẫu gốc đúng, 1 mẫu   γ cố định                           FlipGuard (Zhu và cs., 2024),
                 mới sai                                                           bản cho bài có đáp án kiểm được
  robust_fixed   thật                          γ cố định                           ô ablation
  raw_adaptive   thô                           tự điều chỉnh theo ngân sách ε      ô ablation
  monitor        thật                          luôn 0 (chỉ đo, không kéo)          đường đối chứng cho GRPO / KL
  nafc_replay    thật                          tự điều chỉnh theo ngân sách ε      NAFC on-policy: sửa bằng cách đưa anchor
                                                                                   hỏng vào batch GRPO (không dùng NLL)
  replay_fixed   thật (chỉ đo)                 γ cố định                           đối chứng: luôn trộn anchor ngẫu nhiên
                                                                                   (rehearsal thường, không có bộ phát hiện)

Cách hoạt động (mỗi `every` bước tối ưu, ngay sau khi TRL sinh lời giải của bước đó):
  1. Sinh k mẫu cho mỗi câu anchor bằng model hiện tại (vLLM colocate của TRL; không có vLLM thì HF generate).
  2. Đếm câu hỏng: F^ = tỷ lệ anchor bị hỏng theo cách đếm của chế độ; S = tập anchor đó.
  3. Cập nhật λ: tự điều chỉnh  λ ← clip(λ + η(F^ − ε), 0, λ_max);  cố định  λ = γ;  monitor  λ = 0.
  4. Ở mọi micro-batch cho đến lần kiểm tra sau: loss = loss GRPO + λ · NLL(lời giải đúng của model gốc | câu ∈ S),
     chia cho số bước tích luỹ gradient như loss GRPO (không thêm bước tối ưu nào, mọi nhánh cùng số bước).
  4'. Chế độ replay (nafc_replay, replay_fixed) KHÔNG dùng NLL. Ở mỗi lần sinh cho đến lần kiểm tra sau, thay
     n = ceil(λ · replay_frac_max · số prompt) nhóm prompt của batch bằng anchor (nafc_replay: lấy từ S;
     replay_fixed: lấy lần lượt từ mọi anchor). Anchor được sinh 8 lời giải, chấm và nhận advantage GRPO như mọi
     prompt khác, nên chịu đúng phần thưởng của phương pháp chính (kể cả phạt độ dài của LACONIC).
     Tổng số lời giải mỗi bước giữ nguyên, nên mọi nhánh cùng ngân sách tính toán.
  Tách anchor (v10, --anchor_split): chia cố định anchor thành nửa SỬA (được kéo / trộn vào train) và nửa ĐO
     (không bao giờ được train). λ cập nhật theo F^ trên nửa ĐO; S chỉ lấy anchor hỏng trong nửa SỬA. Nhờ vậy
     tín hiệu điều khiển và số liệu theo dõi không bị thổi phồng vì model đã học lại chính các anchor đó.

Anchor (từ build_anchors.py) là những câu model gốc đúng ≥ 6/8 mẫu, lấy từ phần train KHÔNG dùng để train;
file có sẵn mức nhiễu của cả hai cách đếm (model gốc chấm lại), nên mặc định ε = mức nhiễu + 2 điểm %.

Phần thuần Python (FlipController, AnchorBank.load, flags) kiểm thử không cần GPU: test_flipcon.py.
"""

from __future__ import annotations

import json
import math
import os
import time
from dataclasses import dataclass, field

MODES = ("nafc", "flipguard", "robust_fixed", "raw_adaptive", "monitor", "nafc_replay", "replay_fixed")
COUNTING = {"nafc": "robust", "flipguard": "raw", "robust_fixed": "robust", "raw_adaptive": "raw", "monitor": "robust",
            "nafc_replay": "robust", "replay_fixed": "robust"}
ADAPTIVE = {"nafc": True, "flipguard": False, "robust_fixed": False, "raw_adaptive": True, "monitor": False,
            "nafc_replay": True, "replay_fixed": False}
PULL = {"nafc": "nll", "flipguard": "nll", "robust_fixed": "nll", "raw_adaptive": "nll", "monitor": "none",
        "nafc_replay": "replay", "replay_fixed": "replay"}


def split_anchors(n: int, frac: float, seed: int = 0) -> tuple[list, list]:
    """Chia cố định n anchor: round(frac·n) anchor SỬA (được train), phần còn lại anchor ĐO (chỉ đo)."""
    import random
    perm = list(range(n))
    random.Random(seed).shuffle(perm)
    k = int(round(frac * n))
    return sorted(perm[:k]), sorted(perm[k:])


def n_replay_groups(lam: float, frac_max: float, n_groups: int, pool_size: int) -> int:
    """Số nhóm prompt thay bằng anchor: ceil(λ · frac_max · G), không vượt số anchor trong pool
    và luôn chừa ít nhất 1 prompt train."""
    if lam <= 0 or pool_size <= 0 or n_groups <= 1:
        return 0
    n = math.ceil(lam * frac_max * n_groups - 1e-9)
    return max(0, min(n, pool_size, n_groups - 1))


def replace_groups(inputs: list, n_replace: int, pool: list, ptr: int, make_item, key=None):
    """Thay n_replace nhóm prompt ĐẦU TIÊN của batch (dataset đã xáo) bằng anchor lấy lần lượt từ pool.
    Mọi phần tử cùng prompt (cùng nhóm GRPO) được thay bằng cùng một anchor.
    make_item(x, anchor_idx) -> phần tử mới. Trả về (inputs_mới, ptr_mới, danh_sách_anchor_đã_dùng)."""
    key = key or (lambda x: json.dumps(x["prompt"], sort_keys=True, ensure_ascii=False))
    order = []
    for x in inputs:
        k = key(x)
        if k not in order:
            order.append(k)
    chosen, used = {}, []
    for k in order[:n_replace]:
        a = pool[ptr % len(pool)]
        ptr += 1
        chosen[k] = a
        used.append(a)
    out = [make_item(x, chosen[key(x)]) if key(x) in chosen else x for x in inputs]
    return out, ptr, used


# --------------------------------------------------------------------------------------------------------------- #
# Đếm câu hỏng
# --------------------------------------------------------------------------------------------------------------- #
def robust_flags(correct: list[list[bool]], lo: float = 0.25) -> list[bool]:
    """Câu hỏng thật: tỷ lệ đúng của model hiện tại trên k mẫu ≤ lo (anchor đã chọn sẵn p^_gốc ≥ 0,75)."""
    return [len(c) > 0 and sum(c) / len(c) <= lo for c in correct]


def raw_flags(correct: list[list[bool]], base_first_correct: list[bool]) -> list[bool]:
    """Câu hỏng thô (kiểu FlipGuard): MỘT mẫu gốc đúng và MỘT mẫu mới sai (mẫu đầu tiên của mỗi bên)."""
    return [bool(b) and len(c) > 0 and not c[0] for c, b in zip(correct, base_first_correct)]


# --------------------------------------------------------------------------------------------------------------- #
# Bộ điều khiển λ
# --------------------------------------------------------------------------------------------------------------- #
@dataclass
class FlipController:
    mode: str = "nafc"
    eps: float = 0.06            # ngân sách: tỷ lệ anchor bị hỏng cho phép (cùng cách đếm với chế độ)
    eta: float = 1.0             # bước đối ngẫu
    gamma: float = 0.1           # λ cố định cho flipguard / robust_fixed
    lam_max: float = 1.0
    lam: float = 0.0
    selected: list = field(default_factory=list)   # chỉ số anchor trong S (lần kiểm tra gần nhất)
    history: list = field(default_factory=list)
    repair_idx: list | None = None   # tách anchor: anchor SỬA (None = không tách, mọi anchor vừa sửa vừa đo)
    held_idx: list | None = None     # tách anchor: anchor ĐO, λ chạy theo F^ trên tập này

    @property
    def split(self) -> bool:
        return self.held_idx is not None

    def __post_init__(self):
        if self.mode not in MODES:
            raise ValueError(f"mode phải thuộc {MODES}, nhận '{self.mode}'")
        if self.mode in ("flipguard", "robust_fixed", "replay_fixed"):
            self.lam = self.gamma
        if self.mode == "monitor":
            self.lam = 0.0

    @property
    def counting(self) -> str:
        return COUNTING[self.mode]

    @property
    def adaptive(self) -> bool:
        return ADAPTIVE[self.mode]

    @property
    def pull_kind(self) -> str:
        return PULL[self.mode]

    def replay_pool(self, n_anchor: int) -> list:
        """nafc_replay: anchor hỏng ở lần kiểm tra gần nhất (trong nửa SỬA nếu tách); replay_fixed: mọi anchor SỬA."""
        if self.mode == "replay_fixed":
            return list(self.repair_idx) if self.split else list(range(n_anchor))
        return list(self.selected)

    def update(self, robust: list[bool], raw: list[bool], step: int = -1, extra: dict | None = None) -> dict:
        """Một lần kiểm tra anchor: chọn S theo cách đếm của chế độ, cập nhật λ, trả về số liệu để ghi log."""
        n = max(len(robust), 1)
        f_robust, f_raw = sum(robust) / n, sum(raw) / n
        flags = robust if self.counting == "robust" else raw
        f = f_robust if self.counting == "robust" else f_raw
        self.selected = [i for i, x in enumerate(flags) if x]
        split_rec = {}
        if self.split:
            frac = lambda fl, idx: sum(1 for i in idx if fl[i]) / max(len(idx), 1)
            split_rec = {"F_robust_held": frac(robust, self.held_idx), "F_robust_repair": frac(robust, self.repair_idx),
                         "F_raw_held": frac(raw, self.held_idx), "F_raw_repair": frac(raw, self.repair_idx),
                         "n_held": len(self.held_idx), "n_repair": len(self.repair_idx)}
            f = frac(flags, self.held_idx)                                        # λ theo nửa ĐO
            rep_set = set(self.repair_idx)
            self.selected = [i for i in self.selected if i in rep_set]            # chỉ sửa nửa SỬA
        lam_before = self.lam
        if self.mode == "monitor":
            self.lam = 0.0
        elif self.adaptive:
            self.lam = min(max(self.lam + self.eta * (f - self.eps), 0.0), self.lam_max)
        else:
            self.lam = self.gamma
        rec = {"step": step, "mode": self.mode, "F_robust": f_robust, "F_raw": f_raw, "F_used": f, "eps": self.eps,
               "lam_before": lam_before, "lam": self.lam, "n_S": len(self.selected), "n_anchor": len(robust)}
        rec.update(split_rec)
        if extra:
            rec.update(extra)
        self.history.append(rec)
        return rec

    @property
    def pulling(self) -> bool:
        if self.mode == "replay_fixed":
            return self.lam > 0
        return self.lam > 0 and bool(self.selected)

    def state_dict(self) -> dict:
        return {"mode": self.mode, "eps": self.eps, "eta": self.eta, "gamma": self.gamma, "lam_max": self.lam_max,
                "lam": self.lam, "selected": list(self.selected), "history": list(self.history),
                "repair_idx": self.repair_idx, "held_idx": self.held_idx}

    def load_state_dict(self, d: dict) -> None:
        if d.get("mode") != self.mode:
            raise ValueError(f"checkpoint dùng chế độ {d.get('mode')}, lần chạy này là {self.mode}")
        if d.get("held_idx") != self.held_idx:
            raise ValueError("checkpoint dùng cách tách anchor khác lần chạy này (--anchor_split / --split_seed)")
        self.lam, self.selected, self.history = float(d["lam"]), list(d["selected"]), list(d["history"])


def default_eps(meta: dict, mode: str, margin: float = 0.02) -> float:
    """ε = mức nhiễu của CHÍNH cách đếm mà chế độ dùng (model gốc chấm lại trên anchor) + biên."""
    key = "noise_robust" if COUNTING[mode] == "robust" else "noise_raw"
    if key not in meta:
        raise KeyError(f"file anchor thiếu '{key}', chạy lại build_anchors.py")
    return float(meta[key]) + margin


# --------------------------------------------------------------------------------------------------------------- #
# Anchor
# --------------------------------------------------------------------------------------------------------------- #
@dataclass
class AnchorBank:
    questions: list
    answers: list
    base_responses: list           # một lời giải đúng của model gốc cho mỗi câu (text)
    base_response_ids: list        # token id tương ứng (nếu có), ưu tiên dùng khi tính NLL
    base_first_correct: list       # mẫu đầu tiên của model gốc có đúng không (cho cách đếm thô)
    meta: dict

    @classmethod
    def load(cls, path: str, limit: int | None = None) -> "AnchorBank":
        d = json.load(open(path))
        items = d["items"][:limit] if limit else d["items"]
        if not items:
            raise ValueError(f"{path} không có anchor nào")
        return cls(questions=[x["question"] for x in items], answers=[x["answer"] for x in items],
                   base_responses=[x["base_response"] for x in items],
                   base_response_ids=[x.get("base_response_ids") for x in items],
                   base_first_correct=[bool(x["base_first_correct"]) for x in items], meta=d["meta"])

    def __len__(self):
        return len(self.questions)


# --------------------------------------------------------------------------------------------------------------- #
# Tích hợp vào GRPOTrainer của TRL
# --------------------------------------------------------------------------------------------------------------- #
def make_flip_trainer(base_cls):
    """Trả về lớp con của GRPOTrainer (TRL 1.13) có thêm ràng buộc đúng → sai. Import torch/TRL muộn."""
    import torch
    import torch.nn.functional as F

    from common import chat_messages, is_correct

    class FlipGRPOTrainer(base_cls):
        def __init__(self, *args, flip_ctrl: FlipController, anchors: AnchorBank, flip_every: int = 25,
                     flip_k: int = 4, flip_per_micro: int = 1, replay_frac_max: float = 0.5, scorer=None,
                     **kwargs):
            super().__init__(*args, **kwargs)
            self.flip_ctrl, self.anchors = flip_ctrl, anchors
            self.flip_every, self.flip_k, self.flip_per_micro = flip_every, flip_k, flip_per_micro
            self.flip_scorer = scorer or is_correct
            self._flip_last_check = None
            self._flip_ptr = 0
            self.replay_frac_max = replay_frac_max
            self._replay_ptr = 0
            self._anchor_msgs = [chat_messages(q) for q in anchors.questions]
            tok = self.processing_class
            self._anchor_prompts = [tok.apply_chat_template(chat_messages(q), tokenize=False, add_generation_prompt=True)
                                    for q in anchors.questions]
            self._anchor_prompt_ids = [tok(p, add_special_tokens=False)["input_ids"] for p in self._anchor_prompts]
            eos = tok.eos_token_id
            self._anchor_resp_ids = []
            for text, ids in zip(anchors.base_responses, anchors.base_response_ids):
                r = list(ids) if ids else tok(text, add_special_tokens=False)["input_ids"]
                if eos is not None and (not r or r[-1] != eos):
                    r = r + [eos]
                self._anchor_resp_ids.append(r[: self.max_completion_length])
            self.flip_log_path = os.path.join(self.args.output_dir, "flip_log.jsonl")

        # ---------------- sinh mẫu cho anchor ----------------
        def _anchor_samples(self) -> list[list[str]]:
            k = self.flip_k
            if getattr(self, "use_vllm", False) and getattr(self.vllm_generation, "mode", None) == "colocate":
                from vllm import SamplingParams
                if self.state.global_step != self._last_loaded_step:   # TRL thường đã đồng bộ trong bước sinh
                    self.vllm_generation.sync_weights()
                    self._last_loaded_step = self.state.global_step
                sp = SamplingParams(n=k, temperature=self.temperature, top_p=1.0, max_tokens=self.max_completion_length)
                outs = self.vllm_generation.llm.generate(self._anchor_prompts, sp, use_tqdm=False)
                return [[c.text for c in o.outputs] for o in outs]
            # Đường dự phòng (CPU / không vLLM): HF generate từng câu
            from trl.models import unwrap_model_for_generation
            tok = self.processing_class
            texts = []
            with unwrap_model_for_generation(self.model_wrapped, self.accelerator) as m, torch.no_grad():
                was_training = m.training
                m.eval()
                for ids in self._anchor_prompt_ids:
                    x = torch.tensor([ids], device=self.accelerator.device)
                    out = m.generate(input_ids=x, attention_mask=torch.ones_like(x), do_sample=True,
                                     temperature=self.temperature, top_p=1.0, num_return_sequences=k,
                                     max_new_tokens=self.max_completion_length, pad_token_id=tok.pad_token_id)
                    texts.append(tok.batch_decode(out[:, x.size(1):], skip_special_tokens=True))
                if was_training:
                    m.train()
            return texts

        def _maybe_check_anchors(self):
            step = self.state.global_step
            if step % self.flip_every != 0 or step == self._flip_last_check:
                return
            self._flip_last_check = step
            t0 = time.time()
            samples = self._anchor_samples()
            correct = [[bool(self.flip_scorer(t, a)) for t in ts] for ts, a in zip(samples, self.anchors.answers)]
            robust = robust_flags(correct)
            raw = raw_flags(correct, self.anchors.base_first_correct)
            p = [sum(c) / max(len(c), 1) for c in correct]
            acc = sum(p) / max(len(p), 1)
            extra = {"anchor_acc": acc, "check_seconds": round(time.time() - t0, 1), "anchor_p": [round(v, 3) for v in p]}
            ctrl = self.flip_ctrl
            if ctrl.split:
                mean_on = lambda idx: sum(p[i] for i in idx) / max(len(idx), 1)
                extra.update(anchor_acc_held=mean_on(ctrl.held_idx), anchor_acc_repair=mean_on(ctrl.repair_idx))
            rec = ctrl.update(robust, raw, step=step, extra=extra)
            keys = ["F_robust", "F_raw", "lam", "n_S", "anchor_acc"]
            if ctrl.split:
                keys += ["F_robust_held", "F_robust_repair", "anchor_acc_held", "anchor_acc_repair"]
            for key in keys:
                self._metrics["train"][f"flip/{key}"].append(float(rec[key]))
            if self.accelerator.is_main_process:
                with open(self.flip_log_path, "a") as f:
                    f.write(json.dumps(rec) + "\n")
                tail = (f" | ĐO: F={rec['F_robust_held']:.3f} acc={rec['anchor_acc_held']:.3f}"
                        f" | SỬA: F={rec['F_robust_repair']:.3f} acc={rec['anchor_acc_repair']:.3f}") if ctrl.split else ""
                print(f"[flip] bước {step}: F_robust={rec['F_robust']:.3f} F_raw={rec['F_raw']:.3f} "
                      f"λ={rec['lam']:.3f} |S|={rec['n_S']} acc_anchor={acc:.3f}{tail} ({rec['check_seconds']}s)", flush=True)

        # ---------------- replay on-policy: đưa anchor vào batch GRPO ----------------
        def _replay(self, inputs):
            ctrl = self.flip_ctrl
            pool = ctrl.replay_pool(len(self.anchors))
            keyf = lambda x: json.dumps(x["prompt"], sort_keys=True, ensure_ascii=False)
            n_groups = len({keyf(x) for x in inputs})
            n = n_replay_groups(ctrl.lam, self.replay_frac_max, n_groups, len(pool)) if ctrl.pulling else 0
            if n == 0:
                self._metrics["train"]["flip/n_replay"].append(0.0)
                return inputs

            def make_item(x, a):
                y = dict(x)
                y["prompt"] = self._anchor_msgs[a]
                y["answer"] = self.anchors.answers[a]
                if "bucket" in y:
                    y["bucket"] = 0          # anchor là bài model gốc đúng >= 6/8 lần: nhóm dễ
                return y

            out, self._replay_ptr, used = replace_groups(inputs, n, pool, self._replay_ptr, make_item, key=keyf)
            self._metrics["train"]["flip/n_replay"].append(float(len(used)))
            return out

        def _generate_and_score_completions(self, inputs):
            if self.model.training and self.flip_ctrl.pull_kind == "replay":
                inputs = self._replay(inputs)
            out = super()._generate_and_score_completions(inputs)
            if self.model.training:
                self._maybe_check_anchors()
            return out

        # ---------------- loss học lại lời giải gốc trên S ----------------
        def _anchor_nll(self, model, idx: int):
            p, r = self._anchor_prompt_ids[idx], self._anchor_resp_ids[idx]
            dev = self.accelerator.device
            x = torch.tensor([p + r], device=dev)
            kw = {"input_ids": x, "attention_mask": torch.ones_like(x), "use_cache": False}
            if "logits_to_keep" in getattr(self, "model_kwarg_keys", ()):
                kw["logits_to_keep"] = len(r) + 1
            logits = model(**kw).logits[:, -(len(r) + 1):-1, :]
            target = torch.tensor([r], device=dev)
            return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), target.reshape(-1))

        def _compute_loss(self, model, inputs):
            loss = super()._compute_loss(model, inputs)
            if not self.model.training or not self.flip_ctrl.pulling or self.flip_ctrl.pull_kind != "nll":
                return loss
            S = self.flip_ctrl.selected
            nll = 0.0
            for _ in range(self.flip_per_micro):
                nll = nll + self._anchor_nll(model, S[self._flip_ptr % len(S)])
                self._flip_ptr += 1
            nll = nll / self.flip_per_micro
            self._metrics["train"]["flip/anchor_nll"].append(float(nll.detach()))
            return loss + self.flip_ctrl.lam * nll / self.current_gradient_accumulation_steps

    return FlipGRPOTrainer


def save_state(ctrl: FlipController, ckpt_dir: str) -> None:
    os.makedirs(ckpt_dir, exist_ok=True)
    json.dump(ctrl.state_dict(), open(os.path.join(ckpt_dir, "flip_state.json"), "w"))


def load_state(ctrl: FlipController, ckpt_dir: str) -> bool:
    p = os.path.join(ckpt_dir, "flip_state.json")
    if not os.path.exists(p):
        return False
    ctrl.load_state_dict(json.load(open(p)))
    return True


def read_flip_log(path: str) -> list[dict]:
    """Đọc flip_log.jsonl; nếu train tiếp từ checkpoint thì một bước có thể bị ghi hai lần → giữ bản sau cùng."""
    if not os.path.exists(path):
        return []
    by_step = {}
    for line in open(path):
        if line.strip():
            r = json.loads(line)
            by_step[r["step"]] = r
    return [by_step[s] for s in sorted(by_step)]


def summarize_log(recs: list[dict]) -> dict:
    if not recs:
        return {}
    lam = [r["lam"] for r in recs]
    return {"checks": len(recs), "lam_max_seen": max(lam), "lam_mean": sum(lam) / len(lam),
            "F_robust_mean": sum(r["F_robust"] for r in recs) / len(recs),
            "F_raw_mean": sum(r["F_raw"] for r in recs) / len(recs),
            "frac_checks_pulling": sum(r["lam"] > 0 and r["n_S"] > 0 for r in recs) / len(recs),
            "check_seconds_mean": sum(r.get("check_seconds", math.nan) for r in recs) / len(recs)}


### `frozen_questions.json`

In [ ]:
%%writefile /content/gemma_ngoc/frozen_questions.json
{
  "train": [
    {
      "question": "Let $0 \\leq a, b, c, d \\leq 2005$ be integers. What is the probability that $a b+c d$ is an even number?",
      "answer": "0.625"
    },
    {
      "question": "4. In a certain company, there are 100 shareholders, and any 66 of them own no less than $50 \\%$ of the company's shares. What is the largest percentage of all shares that one shareholder can own?",
      "answer": "25"
    },
    {
      "question": "9.2. On the board in the laboratory, two numbers are written. Every day, the senior researcher Pyotr Ivanovich erases both numbers from the board and writes down their arithmetic mean and harmonic mean ${ }^{2}$. In the morning of the first day, the numbers 1 and 2 were written on the board.\n\nFind the product of the numbers written on the board in the evening of the 2016th day.",
      "answer": "2"
    },
    {
      "question": "Do the three empty fields in the image belong to such natural numbers so that the product of the three numbers on each side of the triangle is the same.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_cd0342d3a611cc15345bg-1.jpg?height=297&width=346&top_left_y=622&top_left_x=889)\n\nWhat is the smallest and what is the largest number that can be written in the gray-shaded field under this condition?",
      "answer": "1336"
    },
    {
      "question": "3+ [ Divisibility of numbers. General properties ]\n\nHumpty-Dumpty walks along a straight line, either taking 37 steps to the left or 47 steps to the right in a minute.\n\nWhat is the shortest time in which he can be one step to the right of the starting point?",
      "answer": "59"
    },
    {
      "question": "[ Combinations and Permutations ]\n[ $\\underline{\\text { Directed Graphs }}$ ]\n\nIn the discussion, 15 deputies participated. Each of them, in their speech, criticized exactly $k$ of the remaining 14 deputies.\n\nFor what smallest $k$ can we assert that there will be two deputies who criticized each other?",
      "answer": "8"
    },
    {
      "question": "71. A segment $AB$ of unit length, which is a chord of a sphere with radius 1, is positioned at an angle of $\\pi / 3$ to the diameter $CD$ of this sphere. The distance from the end $C$ of the diameter to the nearest end $A$ of the chord $AB$ is $\\sqrt{2}$. Determine the length of the segment $BD$.",
      "answer": "1"
    },
    {
      "question": "1.1. In a cross-country skiing race, Petya and Vasya started simultaneously. Vasya ran the entire race at a constant speed of 12 km/h. Petya ran the first half of the distance at a speed of 9 km/h and fell behind Vasya. What should Petya's speed be on the second half of the distance to catch up with Vasya and finish at the same time as his friend? Express your answer in km/h.",
      "answer": "18"
    },
    {
      "question": "Task 1 - 201211\nIn the following schema, a digit is to be entered in each empty field so that a correctly calculated division problem is formed. In particular, none of the multi-digit numbers in the completed schema may receive a leading digit of 0.\n\nProve that there is exactly one entry of digits in the empty fields that meets these requirements! Determine this entry!\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-2089.jpg?height=662&width=679&top_left_y=443&top_left_x=997)",
      "answer": "123976316"
    },
    {
      "question": "7. In a bag, there are two black and two white Go stones. Now, stones are taken from the bag, one at a time. If a white stone is drawn, it is kept. If a black stone is drawn, a white stone must be put back. If there are no white stones left to put back, the game ends. What is the probability of successfully taking all the stones? $\\qquad$",
      "answer": "\\frac{7}{18}"
    },
    {
      "question": "2. A three-digit number $X$ was written with three different digits $A B C$. Four schoolchildren made the following statements. Petya: “The largest digit in the number $X$ is $B$”. Vasya: “$C=8$”. Tolya: “The largest digit is $C$”. Dima: “$C$ is the arithmetic mean of $A$ and $B$”. Find the number $X$, given that exactly one of the children is wrong.",
      "answer": "798"
    },
    {
      "question": "10. Let $m$ be the product of all positive integer divisors of 360,000 . Suppose the prime factors of $m$ are $p_{1}, p_{2}, \\ldots, p_{k}$, for some positive integer $k$, and $m=p_{1}^{e_{1}} p_{2}^{e_{2}} \\cdot \\ldots p_{k}^{e_{k}}$, for some positive integers $e_{1}, e_{2}, \\ldots, e_{k}$. Find $e_{1}+e_{2}+\\ldots+e_{k}$.",
      "answer": "630"
    },
    {
      "question": "【Question 18】\nIf $a, b, c, d, e$ are consecutive positive integers, and $a<b<c<d<e, b+c+d$ is a perfect square, $a+b+c+d+e$ is a perfect cube, what is the minimum value of $c$?",
      "answer": "675"
    },
    {
      "question": "4. It is known that $x+y+z=2016$ and $\\frac{1}{x+y}+\\frac{1}{y+z}+\\frac{1}{z+x}=\\frac{1}{32}$. Find $\\frac{z}{x+y}+\\frac{x}{y+z}+\\frac{y}{z+x}$\n\nANSWER: 60.",
      "answer": "63"
    },
    {
      "question": "1. On the Island of Knights and Liars, there live knights who always tell the truth and liars who always lie. One day, five residents of this island were asked one by one how many knights were among them.\n\n- One, - answered the first.\n- Two, - answered the second.\n- Three, - answered the third.\n- Don't believe them, they are all liars, - said the fourth.\n- You are the liar! - said the fifth to the fourth.\n\nHow many knights were there in reality?\n\nAnswer: 2.",
      "answer": "2"
    },
    {
      "question": "Task 6.4. On a computer keyboard, the key with the digit 1 is not working. For example, if you try to type the number 1231234, only the number 23234 will be printed.\n\nSasha tried to type an 8-digit number, but only 202020 was printed. How many 8-digit numbers meet this condition?",
      "answer": "28"
    },
    {
      "question": "A and B are shooting at a shooting range, but they only have one six-chambered revolver with one bullet. Therefore, they agreed to take turns randomly spinning the cylinder and shooting. A starts. Find the probability that the shot will occur when the revolver is with A.",
      "answer": "\\frac{6}{11}"
    },
    {
      "question": "6,7\n\nThe population of China is one billion people. It would seem that on a map of China with a scale of $1: 1000000$ (1 cm $: 10$ km), one million times fewer people could fit, compared to the entire territory of the country. However, in reality, not only 1000, but even 100 people would not be able to fit on this map. Can you explain this contradiction?",
      "answer": "1000000^2"
    },
    {
      "question": "11. In the sequence $\\left\\{a_{n}\\right\\}$, $a_{1}, a_{2}$ are given non-zero integers, $a_{n+2}=\\left|a_{n+1}-a_{n}\\right|$.\n(1) If $a_{16}=4, a_{17}=1$, find $a_{2018}$;\n(2) Prove: From $\\left\\{a_{n}\\right\\}$, it is always possible to select infinitely many terms to form two different constant subsequences.",
      "answer": "1"
    },
    {
      "question": "18. Find the largest integer $n$ such that $n$ is a divisor of $a^{5}-a$ for all integers $a$.",
      "answer": "30"
    },
    {
      "question": "14.1. $[9.1$ (15 points), 10.1 (15 points)] Calculate\n\n$$\n\\sqrt{1+2 \\sqrt{1+3 \\sqrt{1+\\ldots+2017 \\sqrt{1+2018 \\cdot 2020}}}}\n$$\n\nAnswer: 3.",
      "answer": "3"
    },
    {
      "question": "6. Nathaniel and Obediah play a game in which they take turns rolling a fair six-sided die and keep a running tally of the sum of the results of all rolls made. A player wins if, after he rolls, the number on the running tally is a multiple of 7 . Play continues until either player wins, or else indefinitely. If Nathaniel goes first, determine the probability that he ends up winning.",
      "answer": "\\frac{5}{11}"
    },
    {
      "question": "13.328. A fishing brigade planned to catch 1800 tons of fish within a certain period. For $1 / 3$ of this period, there was a storm, as a result of which the planned daily target was underachieved by 20 tons. However, in the remaining days, the brigade managed to catch 20 tons more than the daily norm, and the planned target was completed one day ahead of schedule. How many tons of fish were planned to be caught daily?",
      "answer": "100"
    },
    {
      "question": "How long would it take for the Earth to complete its axial rotation so that the centrifugal force at the equator would cancel out the weight of objects? The question also needs to be solved by trial and error: at the current rotational speed, at what distance from the Earth would the centrifugal force be equal to the gravitational force?",
      "answer": "5062"
    },
    {
      "question": "What is the sum of the $x$-intercept of the line with equation $20 x+16 y-40=0$ and the $y$-intercept of the line with equation $20 x+16 y-64=0$ ?",
      "answer": "6"
    },
    {
      "question": "8. (5 points) Teacher Wang is distributing bananas to the children. If each child gets 3 bananas, there will be 4 bananas left; if each child gets 4 bananas, there will be 3 bananas short. How many children are there $\\qquad$?",
      "answer": "7"
    },
    {
      "question": "$$\n\\begin{array}{l}\n\\text { 9. In } \\triangle A B C, \\angle A<\\angle B<\\angle C, \\\\\n\\frac{\\sin A+\\sin B+\\sin C}{\\cos A+\\cos B+\\cos C}=\\sqrt{3} .\n\\end{array}\n$$\n\nThen $\\angle B=$",
      "answer": "\\frac{\\pi}{3}"
    },
    {
      "question": "Golenischeva-Kumuzova T.I.\n\nYura has a calculator that allows multiplying a number by 3, adding 3 to a number, or (if the number is divisible by 3) dividing the number by 3. How can Yura use this calculator to get from the number 1 to the number 11?",
      "answer": "11"
    },
    {
      "question": "6. On the interval $\\left[\\frac{1}{2}, 2\\right]$, the function $f(x)=x^{2}+p x+q$ and $g(x)=2 x+\\frac{1}{x^{2}}$ achieve the same minimum value at the same point. The maximum value of $f(x)$ on $\\left[\\frac{1}{2}, 2\\right]$ is $\\qquad$ .",
      "answer": "4"
    },
    {
      "question": "16. Consider the function $f$ defined by\n$$\nf(x)=1+\\frac{2}{x} .\n$$\n\nFind the roots of the equation\n$$\n(\\underbrace{f \\circ f \\circ \\cdots \\circ f}_{10 \\text { times }})(x)=x,\n$$\nwhere \"o\" denotes composition of functions.",
      "answer": "-12"
    },
    {
      "question": "I am thinking of a five-digit number composed of even digits. If I swap the digit in the third position $\\mathrm{s}$ with any other, the number decreases. Furthermore, I will reveal that the first digit is double the last, and the second digit is double the second to last.\n\nWhat number am I thinking of?\n\n(M. Mach)\n\nHint. Can you compare the third digit with the others?",
      "answer": "88644"
    },
    {
      "question": "18. For each natural $n$, which is not a perfect square, the number of values of the variable $x$ is calculated, for which both numbers $x+\\sqrt{n}$ and $x^{2}+param1 \\cdot \\sqrt{n}$ are natural numbers less than param2. Find the total number of such values of $x$.\n\n| param1 | param2 | answer |\n| :---: | :---: | :---: |\n| 18 | 200 |  |\n| 20 | 210 |  |\n| 22 | 250 |  |\n| 24 | 300 |  |\n| 26 | 330 |  |",
      "answer": "108"
    },
    {
      "question": "122. Get rid of the irrationality in the denominator of the fraction\n\n$$\n\\frac{2+\\sqrt{3}}{\\sqrt{6}-\\sqrt{3}+\\sqrt{2}-1} .\n$$",
      "answer": "\\frac{1}{2}(\\sqrt{3}+1)(\\sqrt{2}+1)"
    },
    {
      "question": "Suppose that\n\n$$\n\\begin{aligned}\nM & =1^{5}+2^{4} \\times 3^{3}-4^{2} \\div 5^{1} \\\\\nN & =1^{5}-2^{4} \\times 3^{3}+4^{2} \\div 5^{1}\n\\end{aligned}\n$$\n\nWhat is the value of $M+N$ ?",
      "answer": "2"
    },
    {
      "question": "11. (20 points) Find the largest real number $k$ such that for any triangle with sides $a$, $b$, and $c$, the following inequality holds:\n$$\n\\frac{b c}{b+c-a}+\\frac{a c}{a+c-b}+\\frac{a b}{a+b-c} \\geqslant k(a+b+c) .\n$$",
      "answer": "1"
    },
    {
      "question": "5. The line segment connecting two points on a sphere is called a chord of the sphere. For a sphere with a radius of 4, the lengths of two chords $AB$ and $CD$ are $2\\sqrt{7}$ and $4\\sqrt{3}$, respectively. $M$ and $N$ are the midpoints of $AB$ and $CD$, respectively. The endpoints of each pair of chords can move on the surface of the sphere. There are the following four propositions:\n(1) Chords $AB$ and $CD$ may intersect at point $M$\n(2) Chords $AB$ and $CD$ may intersect at point $N$\n(3) The maximum value of $MN$ is 5\n(4) The minimum value of $MN$ is 1\nAmong these, the true propositions are",
      "answer": "(1)(3)(4)"
    },
    {
      "question": "The average age of the Kebulov family, consisting of a father, a mother, and several children, is 18 years. When the father, who is 38 years old, is not included, the average age of the family is 14 years. How many children do the Kebulovs have?\n\n(L. Hozová)",
      "answer": "4"
    },
    {
      "question": "44. Find a two-digit number that is equal to three times the product of its digits.",
      "answer": "2415"
    },
    {
      "question": "Kosukhin O.n.\n\nSasha found that there were exactly $n$ working digit buttons left on the calculator. It turned out that any natural number from 1 to 99999999 can either be entered using only the working buttons, or obtained as the sum of two natural numbers, each of which can be entered using only the working buttons. What is the smallest $n$ for which this is possible?",
      "answer": "5"
    },
    {
      "question": "$30.11 \\times 12 \\times 13 \\times 14 \\times 15 \\times 16 \\times 17 \\times 18$, the unit digit of the product is",
      "answer": "0"
    },
    {
      "question": "Variant 9.2.3. On the sides $AB$ and $AD$ of rectangle $ABCD$, points $M$ and $N$ are marked, respectively. It is known that $AN=3$, $NC=39$, $AM=10$, $MB=5$.\n\n(a) (1 point) Find the area of rectangle $ABCD$.\n\n(b) (3 points) Find the area of triangle $MNC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f4981519479532effc1g-10.jpg?height=425&width=614&top_left_y=389&top_left_x=420)",
      "answer": "585"
    },
    {
      "question": "# 3. CONDITION\n\nGiven a right triangle $A B C$ with legs $A C=3$ and $B C=4$. Construct triangle $A_{1} B_{1} C_{1}$ by sequentially moving point $A$ a certain distance parallel to segment $B C$ (point $A_{1}$ ), then point $B-$ parallel to segment $A_{1} C$ (point $B_{1}$ ), and finally point $C$ - parallel to segment $A_{1} B_{1}$ (point $C_{1}$ ). What is the length of segment $B_{1} C_{1}$, if it turns out that angle $A_{1} B_{1} C_{1}$ is a right angle and $A_{1} B_{1}=1$?",
      "answer": "12"
    },
    {
      "question": "7. Given the sets\n$$\n\\begin{array}{l}\nA=\\left\\{(x, y) \\mid x=m, y=-3 m+2, m \\in \\mathbf{Z}_{+}\\right\\}, \\\\\nB=\\left\\{(x, y) \\mid x=n, y=a\\left(a^{2}-n+1\\right), n \\in \\mathbf{Z}_{+}\\right\\} .\n\\end{array}\n$$\n\nThen the number of integers $a$ such that $A \\cap B \\neq \\varnothing$ is $\\qquad$.",
      "answer": "10"
    },
    {
      "question": "9.5. From the digits $1,2,3,4,5,6,7,8,9$, nine (not necessarily distinct) nine-digit numbers are formed; each digit is used exactly once in each number. What is the maximum number of zeros that the sum of these nine numbers can end with?\n\n(N. Agakhanov)",
      "answer": "8"
    },
    {
      "question": "【Example 2】Using 4 different colors to color the 4 sides of a small square wooden block, with each side painted a different color, how many different coloring patterns can there be?",
      "answer": "3"
    },
    {
      "question": "171 If three integers $a, b, c(a \\neq 0)$ make the equation $a x^{2}+b x+c=0$ have two roots $a$ and $b$, then $a+b+c=$",
      "answer": "18"
    },
    {
      "question": "23. In the figure below, $\\mathrm{ADE}$ is a triangle with $\\angle \\mathrm{AED}=120^{\\circ}$, and $\\mathrm{B}$ and $\\mathrm{C}$ are points on side $\\mathrm{AD}$ such that $\\mathrm{BCE}$ is an equilateral triangle. If $\\mathrm{AB}=4 \\mathrm{~cm}, \\mathrm{CD}=16 \\mathrm{~cm}$ and $\\mathrm{BC}=x \\mathrm{~cm}$, find the value of $x$.",
      "answer": "8"
    },
    {
      "question": "The figure shows a rectangle $A B C D$ with side $\\overline{A D}=4$ and where point $M$ is the midpoint of segment $A B$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-19.jpg?height=508&width=708&top_left_y=500&top_left_x=777)\n\nKnowing that segments $A C$ and $D M$ are orthogonal, calculate the area of rectangle $A B C D$.\n\n#",
      "answer": "16\\sqrt{2}"
    },
    {
      "question": "3. Quadrilateral $ABCD$ is inscribed in circle $\\omega$, the center of which lies on side $AB$. Circle $\\omega_{1}$ is externally tangent to circle $\\omega$ at point $C$. Circle $\\omega_{2}$ is tangent to circles $\\omega$ and $\\omega_{1}$ at points $D$ and $E$ respectively. Line $BD$ intersects circle $\\omega_{2}$ again at point $P$, and line $AC$ intersects circle $\\omega_{1}$ again at point $Q$. Find the angle $PEQ$.",
      "answer": "180"
    },
    {
      "question": "4. (6 points) The sum of the largest natural number less than 100 and the smallest natural number greater than 300, is the $\\qquad$ times of the largest natural number not greater than 200.\n\n\n\n保留了原文的格式和换行，翻译结果如上。注意，原文中的 $\\qquad$ 用于表示空格或留白，翻译时也保持不变。",
      "answer": "2"
    },
    {
      "question": "Example 2 A deck of cards consists of 52 cards, with 13 cards of each of the four suits: \"Diamonds,\" \"Clubs,\" \"Hearts,\" and \"Spades,\" numbered $2,3, \\cdots, 10, \\mathrm{~J}, \\mathrm{Q}, \\mathrm{K}, \\mathrm{A}$, respectively. Two cards of the same suit and adjacent numbers are called \"consecutive suited cards,\" and $\\mathrm{A}$ can be considered as 1, making it consecutive with 2. Determine the number of ways to draw 13 cards from this deck such that each number appears and no two cards form a consecutive suited pair.",
      "answer": "3^{13}-3"
    },
    {
      "question": "8. (4 points) Given a triangle $ABC$ with sides $AB=14$, $BC=13$, and $AC=15$. On side $AB$, point $K$ is marked; on side $AC$, point $L$ is marked; and on side $BC$, point $N$ is marked. It is known that $AK=\\frac{15}{14}$, $BN=9$, and $AL=1$. A line parallel to $KL$ is drawn through point $N$, intersecting side $AC$ at point $M$. Find the area of quadrilateral $KLMN$.",
      "answer": "30\\frac{1013}{1183}"
    },
    {
      "question": "Example 6 Find the smallest positive period of the function $y=4 \\sin \\left(3 x+\\frac{\\pi}{4}\\right)+3 \\cos \\left(3 x+\\frac{\\pi}{4}\\right)$.",
      "answer": "\\frac{2\\pi}{3}"
    },
    {
      "question": "# Task 10.5\n\nThe field is a $41 \\times 41$ grid, in one of the cells of which a tank is hidden. The fighter aircraft shoots one cell per shot. If a hit occurs, the tank moves to an adjacent cell along a side; if not, it remains in place. After each shot, the pilot does not know whether a hit occurred. To destroy the tank, it must be hit twice. What is the minimum number of shots required to guarantee that the tank is destroyed?\n\n## Points 7\n\n#",
      "answer": "2521"
    },
    {
      "question": "60th Putnam 1999 Problem A5 Find a constant k such that for any polynomial f(x) of degree 1999, we have |f(0)| ≤ k ∫ -1 1 |f(x)| dx.",
      "answer": "(1/3000)^{2000}"
    },
    {
      "question": "Opgave 3. Gegeven is een onbekende rij $a_{1}, a_{2}, a_{3}, \\ldots$ van gehele getallen die voldoet aan de volgende eigenschap: voor elk priemgetal $p$ en elk positief geheel getal $k$ geldt\n\n$$\na_{p k+1}=p a_{k}-3 a_{p}+13\n$$\n\nBepaal alle mogelijke waarden van $a_{2013}$.",
      "answer": "2016"
    },
    {
      "question": "In how many ways can an $n \\times n$ table be filled with positive integers from 1 to $n^{2}$ such that the difference between numbers written in adjacent cells is not greater than $(n+1)$? (Two cells are adjacent if they share an edge or a vertex.)",
      "answer": "32"
    },
    {
      "question": "B1. The product of two consecutive natural numbers after the number $n$ is 600 greater than the product of two consecutive natural numbers before the number $n$. Determine the number $n$. Write down the solution.",
      "answer": "100"
    },
    {
      "question": "11. In $\\triangle A B C$, it is known that $A=\\frac{\\pi}{3}, \\frac{A B}{A C}=\\frac{8}{5}$, and the inradius $r=2 \\sqrt{3}$. Then the area of this triangle is",
      "answer": "40\\sqrt{3}"
    },
    {
      "question": "G1.2 Let $x=\\frac{1-\\sqrt{3}}{1+\\sqrt{3}}$ and $y=\\frac{1+\\sqrt{3}}{1-\\sqrt{3}}$. If $b=2 x^{2}-3 x y+2 y^{2}$, find the value of $b$.",
      "answer": "25"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n$\\lim _{n \\rightarrow \\infty} \\sqrt{n^{3}+8}\\left(\\sqrt{n^{3}+2}-\\sqrt{n^{3}-1}\\right)$",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "11. II 5 (BUL 1) ${ }^{\\mathrm{IMO} 4}$ Consider a partition of an $8 \\times 8$ chessboard into $p$ rectangles whose interiors are disjoint such that each of them has an equal number of white and black cells. Assume that $a_{1}<a_{2}<\\cdots<a_{p}$, where $a_{i}$ denotes the number of white cells in the $i$ th rectangle. Find the maximal $p$ for which such a partition is possible and for that $p$ determine all possible corresponding sequences $a_{1}, a_{2}, \\ldots, a_{p}$.",
      "answer": "7"
    },
    {
      "question": "7-7. The recruits stood in a row, facing the same direction. Among them were three brothers: Peter, Nikolai, and Denis. In front of Peter were 50 people, in front of Nikolai 100, and in front of Denis 170. On the command \"About face!\" everyone turned to face the opposite direction. As a result, it turned out that in front of one of the brothers there are now four times as many people as in front of another. How many recruits could there be, including the brothers? List all possible options.",
      "answer": "211"
    },
    {
      "question": "2. Let $x_{1}$ and $x_{2}$ be the largest roots of the polynomials\n\n$$\n\\begin{gathered}\nf(x)=1-x-4 x^{2}+x^{4} \\\\\n\\text { and } \\\\\ng(x)=16-8 x-16 x^{2}+x^{4}\n\\end{gathered}\n$$\n\nrespectively. Find $\\frac{x_{2}}{x_{1}}$.",
      "answer": "2"
    },
    {
      "question": "## 126. Math Puzzle 11/75\n\nA gardener sold half of all his apples and half an apple to the first buyer, half of the remaining apples and another half an apple to the second buyer, half of the remaining apples and half an apple to the third buyer, and so on.\n\nTo the seventh buyer, he sold half of the remaining apples and another half an apple. Then he had none left.\n\nHow many apples did the gardener have at the beginning?",
      "answer": "127"
    },
    {
      "question": "$[$ [extension of a tetrahedron to a parallelepiped]\n\nSegment $A B(A B=1)$, being a chord of a sphere with radius 1, is positioned at an angle of $60^{\\circ}$ to the diameter $C D$ of this sphere. The distance from the end $C$ of the diameter to the nearest end $A$ of the chord $A B$ is $\\sqrt{2}$. Find $B D$.\n\n#",
      "answer": "1"
    },
    {
      "question": "(Moscow 2000, exercise A1) Let $x$ and $y$ be two different real numbers such that $x^{2}-2000 x=y^{2}-2000 y$. What is the value of $x+y$?",
      "answer": "2000"
    },
    {
      "question": "2. For natural numbers $a$ and $b$, it is known that $5a-1$ is divisible by $b$, $a-10$ is divisible by $b$, but $3a+5$ is not divisible by $b$. What values can the number $b$ take?",
      "answer": "49"
    },
    {
      "question": "[ Law of Cosines ]\n\nFind the sum of the squares of the distances from a point $M$, taken on the diameter of a certain circle, to the ends of any chord parallel to this diameter, if the radius of the circle is $R$, and the distance from the point $M$ to the center of the circle is $a$.",
      "answer": "3"
    },
    {
      "question": "4-1. Katya attached a square with a perimeter of 40 cm to a square with a perimeter of 100 cm as shown in the figure. What is the perimeter of the resulting figure in centimeters?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_535cfcd84333341737d0g-01.jpg?height=281&width=374&top_left_y=676&top_left_x=844)",
      "answer": "120"
    },
    {
      "question": "31. Given $\\overline{A B C D E F} \\times B=\\overline{E F A B C D}$, the same letter represents the same digit, and different letters represent different digits. Then the possible cases for $\\overline{A B C D E F}$ are $\\qquad$ kinds.",
      "answer": "2"
    },
    {
      "question": "In the homework for calculating the value of the expression\n\n$$\n2-3+4-5+6-7+8-9+10=\n$$\n\nRadek forgot to write two parentheses, so he got a result that was 18 greater than what he would have obtained if he had written the parentheses. Add the parentheses in two ways and write down what number Radek got and what number he should have gotten.\n\n(M. Dillingerová)",
      "answer": "-12"
    },
    {
      "question": "7. Let $\\sin \\alpha+\\sin \\beta=\\frac{4}{5} \\sqrt{2}, \\cos \\alpha+\\cos \\beta=\\frac{4}{5} \\sqrt{3}$, then $\\tan \\alpha+\\tan \\beta=$",
      "answer": "\\sqrt{6}"
    },
    {
      "question": "The wall clock in the kitchen is 2 seconds slow every day, and the antique clock in the room is 15 seconds fast every day. On Sunday at noon, the wall clock shows 12:01, and the antique clock shows 11:59. When during the week will the sum of the squares of the differences between the times shown by the clocks and the actual time be the smallest?",
      "answer": "\\frac{1020}{229}"
    },
    {
      "question": "7. Given a tetrahedron \\(ABCD\\) with a volume of \\(40\\), \\(AB = 7\\), \\(BC = 8\\), \\(AC = 9\\). If the projection of vertex \\(D\\) onto the plane \\(ABC\\) is exactly the incenter \\(H\\) of \\(\\triangle ABC\\), then the surface area of the tetrahedron is\n\n---\n\nThe translation preserves the original text's formatting and line breaks.",
      "answer": "60+12\\sqrt{5}"
    },
    {
      "question": "180. Find the sides of a rectangle whose area is numerically equal to the sum of its perimeter and diagonal, and the base is three times the height.\n\n32",
      "answer": "8+\\sqrt{10}"
    },
    {
      "question": "7. The distances from the space diagonal of a rectangular prism to the three edges that do not intersect it are $2 \\sqrt{5}, \\frac{30}{\\sqrt{13}}, \\frac{15}{\\sqrt{10}}$, then the volume of the rectangular prism is $\\qquad$",
      "answer": "750"
    },
    {
      "question": "4. Given non-negative real numbers $x, y, z$ satisfy $\\frac{x}{2}+1=2-\\frac{y}{3}=\\frac{z}{4}-3$, then the minimum value of $2 x+3 y+4 z$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "73"
    },
    {
      "question": "$3.438 \\sin 84^{\\circ} \\cdot \\sin 24^{\\circ} \\cdot \\sin 48^{\\circ} \\cdot \\sin 12^{\\circ}=\\frac{1}{16}$.",
      "answer": "\\frac{1}{16}"
    },
    {
      "question": "## Task 1 - V10821\n\nIn an LPG, potatoes are laid out using a four-row laying machine. With this modern machine, 3 cooperative farmers can lay out 5 hectares of potatoes in 10 hours.\n\nIn the past, when farmers had to lay out the potatoes by hand, one farmer could manage 0.5 hectares in 8 hours. Additionally, he needed 2 hours for loosening the soil and covering the potatoes.\n\nHow many hours of work (labor hours) are required for 1 hectare\n\na) with the potato laying machine,\n\nb) with manual labor?\n\nc) Compare the figures by calculating percentages.",
      "answer": "30"
    },
    {
      "question": "Every term of a sequence, starting from the second, is equal to the sum of the previous term and the sum of its digits. The first elements of the sequence are\n\n$$\n1,2,4,8,16,23,28,38,49, \\ldots\n$$\n\nIs it possible that 793210041 belongs to this sequence?",
      "answer": "793210041"
    },
    {
      "question": "The sequence $a_{1}, a_{2}, a_{3}, \\ldots$ is an arithmetic sequence with common difference 3 and $a_{1}=1$. The sequence $b_{1}, b_{2}, b_{3}, \\ldots$ is an arithmetic sequence with common difference 10 and $b_{1}=2$. What is the smallest integer larger than 2023 that appears in both sequences?",
      "answer": "2032"
    },
    {
      "question": "5. The edges of the cuboid are in the ratio $4: 3$, the diagonals of the lateral faces are in the ratio $\\sqrt{20}: \\sqrt{13}$, and the area of the diagonal section is in the ratio 2:1 to the volume of the cuboid. Determine the area and volume of this cuboid.",
      "answer": "\\frac{125}{576}"
    },
    {
      "question": "Determine a point $N$ on the directrix $FM$ of a parabola or its extension, such that the distances from $N$ to the focus and the directrix differ by a given length, $l$.",
      "answer": "2"
    },
    {
      "question": "4. On the island, there live knights who always tell the truth and liars who always lie. In the island's football team, there are 11 people. Player number 1 said: \"In our team, there are as many knights as there are liars.\" Player number 2 said: \"In our team, the number of knights and the number of liars differ by one,\" and so on. Player number 11 said: \"In our team, the number of knights and the number of liars differ by ten.\" How many knights are in the team, and if there are any, what are their numbers? Find all possible answers to this question.",
      "answer": "0"
    },
    {
      "question": "35. What is the smallest positive integer $n$, where $n \\neq 11$, such that the highest common factor of $n-11$ and $3 n+20$ is greater than 1 ?",
      "answer": "64"
    },
    {
      "question": "Let's determine the remainder of the sum $1+22+333+4444+\\ldots+\\underbrace{s s s . \\ldots s}_{s \\text { times }}$ when divided by $(s-1)$ in the base $s+1$ number system.\n\n---\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\n---\n\nDetermine the remainder of the sum $1+22+333+4444+\\ldots+\\underbrace{s s s . \\ldots s}_{s \\text { times }}$ when divided by $(s-1)$ in the base $s+1$ number system.",
      "answer": "1"
    },
    {
      "question": "216. Blindness in Bats. One naturalist, trying to mystify Colonel Crackham, told him that he had studied the question of blindness in bats.\n\n- I found,- he said, - that the ingrained habit of bats to sleep during the day in dark corners and fly only at night has led to the spread of blindness among them, although some individuals saw well with both or one eye. Two of the bats I examined saw with the right eye, three with the left, four did not see with the left eye, and five did not see with the right eye.\n\nCould you calculate the smallest number of bats the naturalist had to examine to obtain such results?",
      "answer": "7"
    },
    {
      "question": "4. (5 points) Count the number of squares in the figure: $\\qquad$.\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "18"
    },
    {
      "question": "$\\underline{\\text { Folklore }}$\n\nGiven a polynomial $P(x)$ with integer coefficients. It is known that $P(1)=2013, P(2013)=1, P(k)=k$, where $k$ is some integer. Find $k$.\n\n#",
      "answer": "1007"
    },
    {
      "question": "3.13. $\\left\\{\\begin{array}{l}\\left(x_{3}+x_{4}+x_{5}\\right)^{5}=3 x_{1}, \\\\ \\left(x_{4}+x_{5}+x_{1}\\right)^{5}=3 x_{2}, \\\\ \\left(x_{5}+x_{1}+x_{2}\\right)^{5}=3 x_{3}, \\\\ \\left(x_{1}+x_{2}+x_{3}\\right)^{5}=3 x_{4}, \\\\ \\left(x_{2}+x_{3}+x_{4}\\right)^{5}=3 x_{5} .\\end{array}\\right.$\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_3d5ef740640a71a48194g-033.jpg?height=402&width=443&top_left_y=787&top_left_x=1155)\n\n## 3.3. Positive solutions",
      "answer": "0"
    },
    {
      "question": "$10 x$ is a positive integer, the last two digits of $3^{x}$ form a prime number, then the two values of $x$ closest to 2008 are $\\qquad$ and $\\qquad$ .",
      "answer": "20082009"
    },
    {
      "question": "11. As shown in the figure, in the Cartesian coordinate system, the ellipse $\\Gamma$ : $\\frac{x^{2}}{2}+y^{2}=1$ has left and right foci $F_{1}, F_{2}$, respectively. Let $P$ be a point on $\\Gamma$ in the first quadrant, and the extensions of $P F_{1}, P F_{2}$ intersect $\\Gamma$ at points $Q_{1}\\left(x_{1}, y_{1}\\right), Q_{2}\\left(x_{2}, y_{2}\\right)$, respectively. Find the maximum value of $y_{1}-y_{2}$.",
      "answer": "\\frac{2\\sqrt{2}}{3}"
    },
    {
      "question": "The squares of the natural numbers from 1 to 99 were written one after another, forming the number 14916253649... What is the digit that occupies the 100th position? (The positions are counted from left to right, so the $1^{\\underline{a}}$ position is the 1, the $2^{\\underline{a}}$ is the 4, and so on.)",
      "answer": "9"
    },
    {
      "question": "A lead bullet at $20^{\\circ}$ temperature, completely inelastic, falls freely from a height of $100 \\mathrm{~m}$ onto a solid plane. If all the apparently lost energy was converted into heat absorbed by the bullet, the questions are:\n\n1. What is the temperature of the bullet immediately after the fall?\n2. What speed would the bullet need to be dropped from above to raise its temperature to the melting point of lead?\n\nThe specific heat of lead $C=0.0315$, the mechanical equivalent of heat $E=425$, the melting point of lead $T=335^{\\circ}$, and the intensity of gravitational force $g=9.80 \\frac{\\mathrm{m}}{\\mathrm{s}^{2}}$.",
      "answer": "93.2"
    },
    {
      "question": "Problem 4. Determine the four-digit number $\\overline{x y z t}$ for which\n\n$$\n\\overline{x y z t}+4 \\cdot \\overline{y z t}+2 \\cdot \\overline{z t}=2018\n$$",
      "answer": "1174"
    },
    {
      "question": "42nd Swedish 2002 Problem 2 A, B, C can walk at 5km/hr. They have a car that can accomodate any two of them whch travels at 50km/hr. Can they reach a point 62km away in less than 3 hrs?",
      "answer": "2.95"
    },
    {
      "question": "In the zoo, there were 200 parrots. One day, they each made a statement in turn. Starting from the second,\n\nthe statements were \"Among the previously made statements, more than 70% are false.\" How many false statements did the parrots make\n\n#",
      "answer": "140"
    },
    {
      "question": "Problem 2. To number the pages of a textbook, 411 digits are needed. How many pages does the textbook have?",
      "answer": "173"
    },
    {
      "question": "2. the real numbers $a_{1}, a_{2}, \\ldots, a_{16}$ fulfill the two conditions\n\n$$\n\\sum_{i=1}^{16} a_{i}=100 \\quad \\text { and } \\quad \\sum_{i=1}^{16} a_{i}^{2}=1000\n$$\n\nWhat is the largest possible value that $a_{16}$ can take?\n\n## Solution",
      "answer": "25"
    },
    {
      "question": "5. The numbers from 100 to 999 are written without spaces. What is the remainder when the resulting 2700-digit number is divided by 7?",
      "answer": "2"
    },
    {
      "question": "\\begin{array}{l}\n\\text { [The ratio in which the bisector divides the side. } \\\\ \\text { [The ratio of the areas of triangles with a common base or common height] }]\\end{array}$\n\nIn triangle $ABC$, sides $AB$ and $BC$ are equal to each other, $AC=2$, and $\\angle ACB=30^{\\circ}$. From vertex $A$ to the side $BC$, the bisector $AE$ and the median $AD$ are drawn. Find the area of triangle $ADE$.",
      "answer": "\\frac{2\\sqrt{3}-3}{6}"
    },
    {
      "question": "6.5. On the board, there are 5 integers. By adding them in pairs, the following set of 10 numbers was obtained: $-1,5,8,9,11,12,14,18,20,24$. Determine which numbers are written on the board. In your answer, write their product.",
      "answer": "-2002"
    },
    {
      "question": "## Task A-2.5.\n\nDetermine all pairs $\\{a, b\\}$ of distinct real numbers such that the equations\n\n$$\nx^{2} + a x + b = 0 \\quad \\text{and} \\quad x^{2} + b x + a = 0\n$$\n\nhave at least one common solution in the set of real numbers.",
      "answer": "-1"
    },
    {
      "question": "11. Given $z_{1}, z_{2}, z_{3} \\in \\mathbb{C}$ and $\\left.\\mid z_{1}\\right\\}=\\left\\{z_{2} \\mid=\\sqrt{5}\\right\\} z_{3} \\mid$ and $z_{1}+z_{3}=z_{2}$, then $\\frac{z_{1} z_{2}}{z_{3}^{2}}=$",
      "answer": "-5"
    },
    {
      "question": "Draw two circles with the same center, one with a radius of $1 \\mathrm{~cm}$ and the other with a radius of $3 \\mathrm{~cm}$. In the region outside the circle with a radius of $1 \\mathrm{~cm}$ and inside the circle with a radius of $3 \\mathrm{~cm}$, draw circles that are simultaneously tangent to both circles, as shown in the given figure.\n(a) What should be the radius of these circles?\n\n(b) What is the maximum number of these circles that can be drawn without overlapping?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_280b19a87fa67143351eg-031.jpg?height=252&width=257&top_left_y=248&top_left_x=1590)",
      "answer": "6"
    },
    {
      "question": "2. On a sheet of lined paper, two rectangles are outlined. The first rectangle has a vertical side shorter than the horizontal side, while the second has the opposite. Find the maximum possible area of their common part, if the first rectangle contains 2015 cells, and the second - 2016.",
      "answer": "1302"
    },
    {
      "question": "5.082 If we expand all the brackets in the expression $(1+x)^{9}+(1+x)^{10}+\\ldots$ $\\ldots+(1+x)^{14}$ and combine like terms, we will get some polynomial. Determine the coefficient of $x^{9}$ in this polynomial without expanding the brackets.",
      "answer": "3003"
    },
    {
      "question": "20. [12] Let $f(x)=x^{3}-3 x$. Compute the number of positive divisors of\n$$\n\\left\\lfloor f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(\\frac{5}{2}\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right\\rfloor\n$$\nwhere $f$ is applied 8 times.",
      "answer": "6562"
    },
    {
      "question": "## Task A-1.4.\n\nIn the plane, one hundred circles with the same center and radii $1,2, \\ldots, 100$ are drawn. The smallest circle is colored red, and each of the 99 circular rings bounded by two circles is colored red or green such that adjacent areas are of different colors.\n\nDetermine the total area of the green-colored regions.",
      "answer": "5050\\pi"
    },
    {
      "question": "Shapovalov A.v.\n\nThere are 100 numbers. When each of them was increased by 1, the sum of their squares did not change. Each number was increased by 1 again.\n\nWill the sum of the squares change this time, and if so, by how much?",
      "answer": "200"
    },
    {
      "question": "Subject 2\n\nSolve in $\\mathbb{R}$ the equation: $\\frac{1}{\\{x\\}}=\\frac{1}{x}+\\frac{1}{[x]}$, where $\\{x\\}$ represents the fractional part of $x$ and $[x]$ represents the integer part of $x$.",
      "answer": "\\frac{1+\\sqrt{5}}{2}"
    },
    {
      "question": "3. Given the sequence $\\left\\{x_{n}\\right\\}$, and $x_{n+1}=\\frac{x_{n}+(2-\\sqrt{3})}{1-x_{n}(2-\\sqrt{3})}$, then $x_{1001}-x_{401}=$",
      "answer": "0"
    },
    {
      "question": "Three. (50 points) Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies: $a_{1}=1599$, when $k^{2}<n<(k+1)^{2}$, $a_{n}=\\left[\\frac{k+1}{2}\\right]+\\left[\\frac{k}{2}\\right]+1\\left(k \\in \\mathbf{N}^{*}\\right)$, does there exist a positive integer $n$ such that the equation $\\sum_{i=1}^{n} a_{i}=2006$ is satisfied? If it exists, find all such numbers $n$; if not, explain the reason.",
      "answer": "68"
    },
    {
      "question": "If in a right-angled triangle we draw the midlines $K_{1}$ and $K_{2}$ corresponding to the legs $a$ and $b$, then\n\n$$\n16 K_{2}^{2}-4 K_{1}^{2}=15 a^{2}\n$$",
      "answer": "15^{2}"
    },
    {
      "question": "$$\n\\begin{array}{l}\n\\text { 10. Given } f(x)=\\left\\{\\begin{array}{ll}\n-2 x, & x<0 ; \\\\\nx^{2}-1, & x \\geqslant 0,\n\\end{array}\\right. \\text { the equation } \\\\\nf(x)+2 \\sqrt{1-x^{2}}+\\left|f(x)-2 \\sqrt{1-x^{2}}\\right|-2 a x-4=0\n\\end{array}\n$$\n\nhas three real roots $x_{1}<x_{2}<x_{3}$. If $x_{3}-x_{2}=2\\left(x_{2}-x_{1}\\right)$, then the real number $a=$ $\\qquad$",
      "answer": "\\frac{\\sqrt{17}-3}{2}"
    },
    {
      "question": "Example 2.5. $I=\\int_{-1}^{1} x|x| d x$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\nHowever, since the text provided is already in a form that is commonly used in English for mathematical expressions, the translation is essentially the same:\n\nExample 2.5. $I=\\int_{-1}^{1} x|x| d x$.",
      "answer": "0"
    },
    {
      "question": "In a group of 200 people, only $1 \\%$ are women. Determine the number of men who must leave the group so that $98 \\%$ of the remaining people are male.\n\n#",
      "answer": "100"
    },
    {
      "question": "88. The edge of a regular tetrahedron is $a$. The plane $\\boldsymbol{P}$ passes through the vertex $B$ and the midpoints of the edges $A C$ and $A D$. A sphere touches the lines $A B, A C, A D$ and that part of the plane $P$ which is enclosed within the tetrahedron. Find the radius of the sphere.",
      "answer": "\\frac{\\sqrt{2}}{5\\\\sqrt{11}}"
    },
    {
      "question": "[ Divisibility of numbers. General properties ]  $[$ Classical combinatorics (other). $]$\n\nHow many natural numbers less than a thousand are not divisible by either 5 or 7?",
      "answer": "686"
    },
    {
      "question": "18. Collinear points $A, B$, and $C$ are given in the Cartesian plane such that $A=(a, 0)$ lies along the $x$-axis, $B$ lies along the line $y=x, C$ lies along the line $y=2 x$, and $A B / B C=2$. If $D=(a, a)$, the circumcircle of triangle $A D C$ intersects $y=x$ again at $E$, and ray $A E$ intersects $y=2 x$ at $F$, evaluate $A E / E F$.",
      "answer": "7"
    },
    {
      "question": "Example 14 Find the number of pairwise non-congruent triangles with vertices at the vertices of a regular 2000-gon.\n\n保留源文本的换行和格式，翻译结果如下：\n\nExample 14 Find the number of pairwise non-congruent triangles with vertices at the vertices of a regular 2000-gon.",
      "answer": "333333"
    },
    {
      "question": "In a fruit shop, Jaime noticed that an orange costs the same as half an apple plus half a real, and he also noticed that a third of an apple costs the same as a quarter of an orange plus half a real. With the value of 5 oranges plus 5 reals, how many apples can Jaime buy?\n\n#",
      "answer": "5"
    },
    {
      "question": "8. (1 mark) Given that $0.3010<\\log 2<0.3011$ and $0.4771<\\log 3<0.4772$. Find the leftmost digit of $12^{37}$.\n(1 分) 設 $0.3010<\\log 2<0.3011$ 及 $0.4771<\\log 3<0.4772$, 求 $12^{37}$ 最左的一位數字。",
      "answer": "8"
    },
    {
      "question": "4. If the orthocenter of $\\triangle O A B$ is exactly the focus of the parabola $y^{2}=4 x$, where $O$ is the origin, $A, B$ are on the parabola, then the area $S$ of $\\triangle O A B$ is $\\qquad$ .",
      "answer": "10\\sqrt{5}"
    },
    {
      "question": "I1.4 If $\\log _{2} a+\\log _{2} b \\geq \\gamma$, determine the smallest positive value $\\delta$ for $a+b$.",
      "answer": "16"
    },
    {
      "question": "Example 4 (2004 American Invitational Mathematics Examination) A circle of radius 1 is randomly placed inside a $15 \\times 36$ rectangle $ABCD$. Find the probability that this circle does not intersect with the diagonal $AC$.",
      "answer": "\\frac{375}{442}"
    },
    {
      "question": "4. Let $a \\triangle b$ and $a \\nabla b$ represent the minimum and maximum of the two numbers $a$ and $b$, respectively. For example, $3 \\triangle 4=3, 3 \\nabla 4=4$. For different natural numbers $x$, the number of possible values for $6 \\triangle[4 \\nabla(x \\triangle 5)]$ is $\\qquad$.",
      "answer": "2"
    },
    {
      "question": "While investigating a case, Detective Bashkovsky discovered that the key witness was the one from the Petrovs family who came home before the others on that fateful day. The investigation revealed the following facts.\n\n1. Neighbor Marya Kuzminichna wanted to borrow some salt from the Petrovs, rang their doorbell, but no one answered. When? Who knows? It was already dark...\n2. Galina Efimovna Petrova, coming home in the evening, found both children in the kitchen, and her husband on the sofa - he had a headache.\n3. Husband Anatoly Ivanovich claimed that as soon as he came home, he immediately lay down on the sofa and dozed off, saw no one, heard nothing, the neighbor definitely didn't come - the doorbell would have woken him up.\n4. Daughter Svetlana said that upon returning home, she immediately went to her room, knew nothing about her father, but as usual, stumbled over Dima's shoe in the hallway.\n5. Dmitry didn't remember when he came home, didn't see his father, but heard Svetka swearing about the shoe.\n\n\"Hmm,\" thought Bashkovsky. \"What is the probability that Dmitry returned home before his father?\"",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "11.4. (SFRY, 70). The diagonals of a convex pentagon $A B C D E$, intersecting, form a pentagon $A_{1} B_{1} C_{1} D_{1} E_{1}$ and a five-pointed star.\n\na) Find the sum of the angles of this star at the vertices $A$, $B, C, D, E$.\n\nb) Find the ratio of the area of the pentagon $A_{1} B_{1} C_{1} D_{1} E_{1}$ to the area of the pentagon $A B C D E$ under the condition of the regularity of the latter.",
      "answer": "\\frac{7-3\\sqrt{5}}{2}"
    },
    {
      "question": "3. In a square $A B C D$ with side length 2, $E$ is the midpoint of $A B$. Now, fold the figure along segments $E C, E D$ so that segments $E A, E B$ coincide. This results in a tetrahedron $C D E A$ (point $B$ has coincided with point $A$). The volume of this tetrahedron is",
      "answer": "\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "Example 7 (2006 Beijing College Entrance Examination Final Question) In the sequence $\\left\\{a_{n}\\right\\}$, if $a_{1}, a_{2}$ are positive integers, and $a_{n}=\\left|a_{n-1}-a_{n-2}\\right|$, $n=3,4,5, \\cdots$, then $\\left\\{a_{n}\\right\\}$ is called an \"absolute difference sequence\".\n(1) Give an example of an \"absolute difference sequence\" whose first five terms are not zero (only the first ten terms are required to be written);\n(2) If in the \"absolute difference sequence\" $\\left\\{a_{n}\\right\\}$, $a_{20}=3, a_{21}=$ 0, and the sequence $\\left\\{b_{n}\\right\\}$ satisfies $b_{n}=a_{n}+a_{n+1}+a_{n+2}, n=1,2$, $3, \\cdots$, determine whether the limits of $a_{n}$ and $b_{n}$ exist as $n \\rightarrow \\infty$. If they exist, find their limit values;\n(3) Prove: Any \"absolute difference sequence\" contains infinitely many zero terms.",
      "answer": "6"
    },
    {
      "question": "# Task No. 7.4\n\n## Condition:\n\nProfessor Severus Snape has prepared three potions, each with a volume of 480 ml. The first potion makes the person who drinks it smart, the second makes them beautiful, and the third makes them strong. To ensure the potion works, one needs to drink at least 30 ml of it. Severus Snape was about to drink his potions when he was called to the headmaster's office and left, leaving the labeled potions in large jugs on his desk. Taking advantage of his absence, Harry, Hermione, and Ron approached the table with the potions and began to sample them.\n\nHermione was the first to try the potions: she approached the first jug with the potion of wisdom and drank half of it, then poured the remainder into the second jug with the potion of beauty, thoroughly mixed the contents, and drank half of it. Next, it was Harry's turn: he drank half of the third jug with the potion of strength, then poured the remainder into the second jug, thoroughly mixed everything in this jug, and drank half of it. Now all the contents were in the second jug, which was left for Ron. What percentage of the contents of this jug does Ron need to drink to ensure that each of the three potions will definitely affect him?",
      "answer": "50"
    },
    {
      "question": "1. (2 points) For the polynomial $\\left(x^{2}-x+1\\right)^{100}$, find the sum of the coefficients of the even powers of $x$.",
      "answer": "\\frac{1+3^{100}}{2}"
    },
    {
      "question": "Problem 4. A row of 11 numbers is written such that the sum of any three consecutive numbers is 18. Additionally, the sum of all the numbers is 64. Find the central number.",
      "answer": "8"
    },
    {
      "question": "Example 6. A continuous random variable $X$ is given by the probability density function $p(x)=\\cos x$ in the interval $(0 ; \\pi / 2)$; outside this interval, $p(x)=0$. Find the expected value of the function $Y=X^{2}$.",
      "answer": "\\frac{\\pi^{2}}{4}-2"
    },
    {
      "question": "A fishing cooperative wanted to get an estimate of how many specimens of a certain fish species, which have already reached a certain size, live in one of their ponds. (Hereafter, these will be referred to as suitable fish.) For this purpose, they marked 50 suitable fish from their daily catch and released them back into the water. The next day, in their catch from 3 different locations in the pond, they counted a total of 67, 94, and 43 suitable fish, respectively, and among these, 6, 10, and 3 were marked. How could they estimate the number of suitable fish after the second day's catch?",
      "answer": "537"
    },
    {
      "question": "84 (1194). a) Find the sum of the cubes of the roots of the equation $2 x^{2}-$ $-5 x+1=0$.\n\nb) Find the value of the expression $\\frac{x_{1}}{x_{2}}+\\frac{x_{2}}{x_{1}}$, where $x_{1}$ and $x_{2}-$ are the roots of the equation $2 x^{2}-11 x+13=0$.",
      "answer": "2\\frac{17}{26}"
    },
    {
      "question": "\\section*{Problem 15}\n\nWhat is the minimal value of \\(\\mathrm{b} /(\\mathrm{c}+\\mathrm{d})+\\mathrm{c} /(\\mathrm{a}+\\mathrm{b})\\) for positive real numbers \\(\\mathrm{b}\\) and \\(\\mathrm{c}\\) and nonnegative real numbers a and \\(\\mathrm{d}\\) such that \\(\\mathrm{b}+\\mathrm{c} \\geq \\mathrm{a}+\\mathrm{d}\\) ?",
      "answer": "\\sqrt{2}-\\frac{1}{2}"
    },
    {
      "question": "2. Given $-\\frac{\\pi}{2}<\\alpha<\\frac{\\pi}{2}, 2 \\tan \\beta=\\tan 2 \\alpha, \\tan (\\beta-\\alpha)=-2 \\sqrt{2}$. Then $\\cos \\alpha=$",
      "answer": "\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "11. A circle is drawn inside a regular hexagon so that it touches all six sides of the hexagon. The area of the circle is $\\pi \\times 64 \\sqrt{3}$. What is the area of the hexagon?",
      "answer": "384"
    },
    {
      "question": "# 2. Inflation over two years will be\n\n$$\n\\left((1+0.025)^{\\wedge 2-1}\\right)^{*} 100 \\% = 5.0625 \\%\n$$\n\nThe real return on a bank deposit, taking into account the extension for the second year, will be $(1.06 * 1.06 / (1+0.050625) - 1) * 100 = 6.95 \\%$\n\n## Evaluation Criteria:\n\nA maximum of 25 points for a correct and justified solution. Of these:\n\n2 points for the correct calculation of the investment amount.\n\n5 points for the correct calculation of the nominal return on the bank deposit. If this value was calculated incorrectly, the participant could receive points for the following steps in the solution:\n\n- 3 points for calculating the amount the family can withdraw after closing the deposit, taking into account its extension for the second year.\n- 1 point for calculating the amount the family can withdraw after closing the deposit, without correctly accounting for its extension for the second year.\n\n7 points for the correct calculation of the expected return on investment in construction. If this value was calculated incorrectly, the participant could receive points for the following steps in the solution:\n\n- a maximum of 2 points for conceptually incorrect consideration of the probabilities of different outcomes.\n- 2 points for correctly accounting for the 13% income tax.\n\n1 point for the correct choice of investment method. If the correct choice was made based on incorrect calculations of the return in any of the investment options, the participant did not receive a point for this part of the solution.\n\n10 points for the correct calculation of the return. If this value was calculated incorrectly, the participant could receive points for the following steps in the solution:\n\n- 4 points for a logically correct method of calculating the return in %, provided that the real return was calculated incorrectly.\n- 8 points for calculating the return in % as the difference between the nominal return and inflation over 2 years, provided that the nominal return was calculated correctly.\n- 1 point for incorrect accounting of inflation over two years.\n\nIf the entire logic of the calculations is followed, but the participant makes only arithmetic errors, then -1 point for each such error.\n\n## Task 5\n\nThe banking sector has always been a tempting target for criminals, so commercial and government banks are implementing more and more protection technologies.\n\nYou head the information security department of a large commercial bank. Propose your protection strategy for the following objects of the banking infrastructure (at least one existing method for each object):\n\n- employee profiles;\n- bank cards;\n- ATMs;\n- online banking;\n- online store payments;\n- central automated banking system.",
      "answer": "6.95"
    },
    {
      "question": "4. If the functions $f(x)$ and $g(x)$ are defined on $\\mathbf{R}$, and $f(x-y)=f(x) g(y)-g(x) f(y)$, if $f(-2)=f(1) \\neq 0$, then $g(1)+g(-1)=$ (answer with a number).\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "-1"
    },
    {
      "question": "34. For an earth excavation project, Team A can complete it alone in 16 days, and Team B can complete it alone in 20 days. Now, both teams are working together, and their work efficiency increases by $20 \\%$. When $\\frac{1}{4}$ of the project is completed, they suddenly encounter groundwater, which affects the construction progress, reducing the daily excavation by 47.25 cubic meters of soil. As a result, the entire project took 10 days to complete. The total amount of soil to be excavated for the entire project is $\\qquad$ cubic meters.",
      "answer": "1100"
    },
    {
      "question": "16. On the island of Friends and Foes, every citizen is either a Friend (who always tells the truth) or a Foe (who always lies). Seven citizens are sitting in a circle. Each declares \"I am sitting between two Foes\". How many Friends are there in the circle?",
      "answer": "3"
    },
    {
      "question": "13. 3. 13 ${ }^{\\star \\star}$ The axial cross-section of a large wine glass is the graph of the function $y=x^{4}$. A small ball with radius $r$ is placed into the glass. When the radius $r$ is maximized, the ball can touch the lowest point of the bottom of the glass (i.e., what is the maximum value of $r$ such that a circle with radius $r$ located in the region $y \\geqslant x^{4}$ can pass through the origin)?",
      "answer": "\\frac{3}{2}\\sqrt[3]{2}"
    },
    {
      "question": "For $B_{n}\\left(n \\in \\mathbf{N}^{*}\\right)$. Let the point with coordinates $(1,1)$ be $M, P_{n}\\left(x_{n}, y_{n}\\right)$ is the circumcenter of $\\triangle A_{n} B_{n} M$, find the value of $x_{1}+x_{2}+\\cdots+x_{100}$.",
      "answer": "200\\frac{50}{101}"
    },
    {
      "question": "The sum of non-negative numbers $x_{1}, x_{2}, \\ldots, x_{10}$ is 1. Find the maximum possible value of the sum $x_{1} x_{2}+$ $x_{2} x_{3}+\\ldots+x_{9} x_{10}$",
      "answer": "0.25"
    },
    {
      "question": "34th CanMO 2002 Problem 1 What is the largest possible number of elements in a subset of {1, 2, 3, ... , 9} such that the sum of every pair (of distinct elements) in the subset is different?",
      "answer": "5"
    },
    {
      "question": "2. The sequence $\\left(a_{n}\\right)$ is defined by the following relations: $a_{1}=1, a_{2}=3, a_{n}=a_{n-1}-a_{n-2}+n$ (for $n \\geqslant 3$). Find $a_{1000}$.",
      "answer": "1002"
    },
    {
      "question": "\\section*{Problem 14}\n\nA spy-plane circles point A at a distance \\(10 \\mathrm{~km}\\) with speed \\(1000 \\mathrm{~km} / \\mathrm{h}\\). A missile is fired towards the plane from A at the same speed and moves so that it is always on the line between A and the plane. How long does it take to hit?",
      "answer": "18\\pi"
    },
    {
      "question": "1. How many positive numbers are there among the 2014 terms of the sequence: $\\sin 1^{\\circ}, \\sin 10^{\\circ}$, $\\sin 100^{\\circ}, \\sin 1000^{\\circ}, \\ldots ?$",
      "answer": "3"
    },
    {
      "question": "4. Determine the maximum value of the difference $x-y$ if\n\n$$\n2\\left(x^{2}+y^{2}\\right)=x+y\n$$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "13. Xiao Li and Xiao Zhang are running at a constant speed on a circular track. They start at the same time and place, with Xiao Li running clockwise and completing a lap every 72 seconds; Xiao Zhang running counterclockwise and completing a lap every 80 seconds. A quarter-circle arc interval is marked on the track, centered at the starting point. How many seconds do they spend in the marked interval simultaneously from the start until they both return to the starting point at the same time?",
      "answer": "46"
    },
    {
      "question": "O esqueleto de um cubo $6 \\times 6 \\times 6$, formado por cubinhos $1 \\times 1 \\times 1$ é mostrado na figura.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_0308bd91a070e7a80796g-3.jpg?height=500&width=437&top_left_y=387&top_left_x=638)\n\nFigura 64.1\n\n(a) Quantos cubinhos formam este esqueleto?\n\n(b) É dado um cubo $7 \\times 7 \\times 7$ formado por cubinhos $1 \\times 1 \\times 1$. Quantos cubinhos devemos retirar para obter um esqueleto do cubo $7 \\times$ $7 \\times 7$.\n\n##",
      "answer": "56"
    },
    {
      "question": "A train traveling between $A$ and $B$ arrives 20 minutes earlier in $B$ if its speed exceeds the scheduled speed by $5 \\mathrm{~km}$ per hour; however, it arrives 25 minutes late if its speed is $5 \\mathrm{~km}$ per hour less than the scheduled speed. What is the scheduled speed?",
      "answer": "45"
    },
    {
      "question": "$12 \\cdot 60$ In a right-angled triangle with side lengths all being positive integers, one of the legs is 1575 less than the hypotenuse, and the other leg is less than 1991. Find the length of the hypotenuse of this right-angled triangle.\n(China Beijing High School Grade 1 Mathematics Competition, 1991)",
      "answer": "1799"
    },
    {
      "question": "Problem 9.3. Given a convex quadrilateral $ABCD$, $X$ is the midpoint of diagonal $AC$. It turns out that $CD \\parallel BX$. Find $AD$, if it is known that $BX=3, BC=7, CD=6$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_790dd471148872cd0846g-02.jpg?height=224&width=495&top_left_y=1341&top_left_x=479)",
      "answer": "14"
    },
    {
      "question": "Task 2. Every student in the Netherlands receives a finite number of cards. Each card has a real number in the interval $[0,1]$ written on it. (The numbers on different cards do not have to be different.) Find the smallest real number $c>0$ for which the following holds, regardless of the numbers on the cards that everyone has received.\n\nEvery student whose sum of the numbers on the cards is at most 1000 can distribute the cards over 100 boxes so that the sum of the numbers in each box is at most $c$.",
      "answer": "11-\\frac{1}{91}"
    },
    {
      "question": "9. Given the sequence $\\left\\{a_{n}\\right\\}$ with the sum of the first $n$ terms being $S_{n}, a_{1}=\\frac{1}{5}$, and for any positive integers $m, n$, it holds that $a_{m+n}=a_{m} a_{n}$. If $S_{n}<a$ for any $n \\in \\mathbf{Z}_{+}$, then the minimum value of the real number $a$ is",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "50.What is the 1997th digit in the decimal expansion of the fraction $\\frac{1}{7}=0.142857 \\ldots ?$",
      "answer": "5"
    },
    {
      "question": "9. Given $f(x)=\\left\\{\\begin{array}{l}\\frac{3+x}{1+x^{2}}, 0 \\leq x \\leq 3, \\\\ \\frac{3}{5}, x>3 .\\end{array}\\right.$, the sequence $\\left\\{a_{n}\\right\\}$ satisfies: $0<a_{n} \\leq 3, n \\in N^{*}$, and $\\sum_{i=1}^{2009} a_{i}=\\frac{2009}{3}$, if the inequality $\\sum_{i=1}^{2009} f\\left(a_{i}\\right) \\leq x-\\ln (x-p)$ holds for all $x \\in(p,+\\infty)$, find the minimum value of the real number $p$.",
      "answer": "6026"
    },
    {
      "question": "3. Let the real numbers $x_{1}, x_{2}$, and $x_{3}$ be the roots of the equation $3 x^{3}-p x^{2}+28 x-p=0$, taking into account their possible multiplicities. Then $3 x^{3}-p x^{2}+28 x-p \\equiv 3\\left(x-x_{1}\\right)\\left(x-x_{2}\\right)\\left(x-x_{3}\\right)=$\n\n$$\n=3 x^{3}-3 x^{2}\\left(x_{1}+x_{2}+x_{3}\\right)+3 x\\left(x_{1} x_{2}+x_{2} x_{3}+x_{3} x_{1}\\right)-3 x_{1} x_{2} x_{3},\n$$\n\nfrom which $x_{1}+x_{2}+x_{3}=p / 3, \\quad x_{1} x_{2}+x_{2} x_{3}+x_{3} x_{1}=28 / 3$. Taking this into account, the target expression is transformed as follows:\n\n$$\n\\begin{aligned}\n&\\left(x_{1}-2\\right)^{2}+\\left(x_{2}-2\\right)^{2}+\\left(x_{3}-2\\right)^{2}=x_{1}^{2}+x_{2}^{2}+x_{3}^{2}-4\\left(x_{1}+x_{2}+x_{3}\\right)+12= \\\\\n&=\\left(x_{1}+x_{2}+x_{3}\\right)^{2}-2\\left(x_{1} x_{2}+x_{2} x_{3}+x_{3} x_{1}\\right)-4\\left(x_{1}+x_{2}+x_{3}\\right)+12= \\\\\n&= \\frac{p^{2}}{9}-\\frac{56}{3}-\\frac{4 p}{3}+12=\\frac{p^{2}-12 p-60}{9} .\n\\end{aligned}\n$$\n\nNow let's plot the graph of the equation $3 x^{3}-p x^{2}+28 x-p=0$ on the plane $(x, p)$. Since the parameter enters this equation linearly, it is more convenient to solve it for $p$:\n\n$$\np\\left(x^{2}+1\\right)=3 x^{3}+28 x=3 x\\left(x^{2}+1\\right)+25 x \\Rightarrow p(x)=3 x+\\frac{25 x}{x^{2}+1}\n$$\n\nWe will find the set of values of the parameter $p$ for which the equation has three real roots. For this, we will investigate $p(x)$:\n\n$$\np^{\\prime}(x)=3+\\frac{25\\left(1-x^{2}\\right)}{\\left(x^{2}+1\\right)^{2}}=\\frac{3 x^{4}-19 x^{2}+28}{\\left(x^{2}+1\\right)^{2}}=\\frac{\\left(x^{2}-4\\right)\\left(3 x^{2}-7\\right)}{\\left(x^{2}+1\\right)^{2}}\n$$\n\nThe odd function $p(x)$ has minima at points $\\bar{x}_{2}=-\\frac{\\sqrt{21}}{3}, \\bar{x}_{4}=2$ and maxima at points $\\bar{x}_{1}=-2, \\bar{x}_{3}=\\frac{\\sqrt{21}}{3}$,\n\nwhere\n\n$$\np_{1}=p\\left(\\bar{x}_{1}\\right)=-16, p_{2}=p\\left(\\bar{x}_{2}\\right)=-\\frac{7 \\sqrt{21}}{2}\n$$\n\n$$\np_{3}=p\\left(\\bar{x}_{3}\\right)=\\frac{7 \\sqrt{21}}{2}, p_{4}=p\\left(\\bar{x}_{4}\\right)=16\n$$\n\nFor all $c \\in\\left(p_{2} ; p_{1}\\right) \\cup\\left(p_{4} ; p_{3}\\right)$, the line $p=c$ intersects the graph at three points, and the equation $3 x^{3}-p x^{2}+28 x-p=0$ has three distinct roots. For parameter values that coincide with one of $p_{1}, p_{2}, p_{3}$, or $p_{4}$ (on the boundaries of the intervals), there are two points of intersection, which corresponds to the case of multiple roots.\n\nThe quadratic trinomial $\\frac{p^{2}-12 p-60}{9}$, decreasing for $p6$, takes the smallest possible value on the set $\\left[p_{2} ; p_{1}\\right] \\cup\\left[p_{4} ; p_{3}\\right]$ when $p=p_{4}=16$.\n\nThus, under the conditions of the problem,\n\n$$\n\\min \\left(\\left(x_{1}-2\\right)^{2}+\\left(x_{2}-2\\right)^{2}+\\left(x_{3}-2\\right)^{2}\\right)=\\frac{16^{2}-12 \\cdot 16-60}{9}=\\frac{4}{9}\n$$",
      "answer": "\\frac{4}{9}"
    },
    {
      "question": "A game is composed of the following rules:\n\ni) In each round, a common unbiased die is rolled.\n\nii) If the number 3 comes up, then player $A$ wins.\n\niii) If one of the numbers in the set $\\{4,5,6\\}$ comes up, then player $B$ wins.\n\niv) If one of the numbers in the set $\\{1,2\\}$ comes up, then the die is rolled again until it results in 3 or 4 or 5 or 6.\n\nWhat is the probability that player $B$ wins?",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "1.5 Remove the multiples of 3 and 4 from the sequence of natural numbers $1,2,3,4, \\cdots$, but retain all multiples of 5 (for example, 15 and 20 are not removed). After removing, write the remaining numbers in a sequence: $A_{1}=1, A_{2}=2, A_{3}=5, A_{4}=7, \\cdots$ Find the value of $A_{1988}$.\n(Advanced Math Test for Science Experimental Class Admission, 1988)",
      "answer": "3314"
    },
    {
      "question": "16. If an internal point of some $n$-sided prism is connected to all its vertices, then we obtain $n$ quadrilateral pyramids with a common vertex at this point, the bases of which are the lateral faces of the prism. Find the ratio of the sum of the volumes of these pyramids to the volume of the given prism.",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "13. If the six-digit number $a 2016 b$ is divisible by 12, then the number of such six-digit numbers is $\\qquad$.",
      "answer": "9"
    },
    {
      "question": "Problem 10.5. A circle $\\omega$ is inscribed in trapezoid $A B C D$, and $L$ is the point of tangency of $\\omega$ and side $C D$. It is known that $C L: L D=1: 4$. Find the area of trapezoid $A B C D$, if $B C=9$, $C D=30$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_c47c8a04f76ef67ab4e5g-35.jpg?height=305&width=407&top_left_y=231&top_left_x=517)",
      "answer": "972"
    },
    {
      "question": "Given a rectangle inscribed in a circle, we extend it to an octagon, four of whose vertices coincide with the vertices of the rectangle, and the other four are the intersection points of the perpendicular bisectors of the rectangle's sides and the circle. How should we choose the rectangle so that the ratio of the area of the octagon to the area of the rectangle is minimized?",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "5. Ari is watching several frogs jump straight towards him from the same place. Some frogs are larger, and some are smaller. When the larger frog makes 7 equal jumps, it is still $3 \\mathrm{~cm}$ away from Ari. When the smaller frog makes 10 equal jumps, it is still $1 \\mathrm{~cm}$ away from Ari. If the frogs are initially more than $2 \\mathrm{~m}$ away from Ari, what is the smallest possible length of the smaller frog's jump? The length of each jump in centimeters is a natural number.\n\nNo pocket calculators or any reference materials are allowed.\n\n## Ministry of Science and Education of the Republic of Croatia Agency for Education and Education Croatian Mathematical Society\n\n## County Competition in Mathematics February 26, 2024.  5th grade - elementary school\n\nEach task is worth 10 points. In addition to the final result, the procedure is also scored. To earn all points, it is necessary to find all solutions and confirm that there are no others, write down the procedure, and justify your conclusions.",
      "answer": "24"
    },
    {
      "question": "Connect the midpoints of the sides of a rectangle with the opposite vertices. What fraction of the area of the original rectangle is the area of the octagon determined by these eight lines?\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "(1) If $4^{a}-3 a^{b}=16, \\log _{2} a=\\frac{a+1}{b}$, then $a^{b}=$",
      "answer": "16"
    },
    {
      "question": "24. Let $a$ and $b$ be two real numbers. If the equation $a x+(b-3)=(5 a-1) x+3 b$ has more than one solution, what is the value of $100 a+4 b$ ?",
      "answer": "19"
    },
    {
      "question": "13.333. A ball falls from a height of 2 m 43 cm and, upon hitting the ground, bounces back up, each time reaching $2 / 3$ of the height from which it falls again. After how many bounces will the ball rise to a height of 32 cm?",
      "answer": "5"
    },
    {
      "question": "90. Use the digits 1 to 9 to form a two-digit number, a three-digit number, and a four-digit number, with each digit used only once. The maximum sum of these three numbers is $\\qquad$ .",
      "answer": "10656"
    },
    {
      "question": "53. A snail is crawling on a plane, turning $90^{\\circ}$ after each meter of its path. What is the maximum distance from the starting point it could be after crawling 300 m, having made a total of 99 left and 200 right turns?",
      "answer": "100\\sqrt{2}"
    },
    {
      "question": "1. In the diagram, $\\triangle A B C$ has angles that measure $60^{\\circ}$, $(5 x)^{\\circ}$ and $(3 x)^{\\circ}$. What is the value of $x$ ?",
      "answer": "15"
    },
    {
      "question": "7. For an arithmetic sequence with real number terms, the common difference is 4, and the square of the first term plus the sum of the remaining terms does not exceed 100. The maximum number of terms in such a sequence is $\\qquad$.",
      "answer": "8"
    },
    {
      "question": "NT 3. Find the largest integer $k(k \\geq 2)$, for which there exists an integer $n(n \\geq k)$ such that from any collection of $n$ consecutive positive integers one can always choose $k$ numbers, which verify the following conditions:\n\n1. each chosen number is not divisible by 6 , by 7 and by 8 ;\n2. the positive difference of any two different chosen numbers is not divisible by at least one of the numbers 6,7 or 8 .",
      "answer": "108"
    },
    {
      "question": "32. (USA 3) The vertex $A$ of the acute triangle $A B C$ is equidistant from the circumcenter $O$ and the orthocenter $H$. Determine all possible values for the measure of angle $A$.",
      "answer": "60"
    },
    {
      "question": "4. The real numbers $a, b, c, d$ are positive and satisfy $(a+c)(b+d)=a c+b d$. Find the minimum of\n\n$$\n\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{d}+\\frac{d}{a}\n$$\n\nAnswer: 8.",
      "answer": "8"
    },
    {
      "question": "Example 13 Let $a, b$ be non-zero complex numbers, and $\\frac{a}{b}$ is not a real number. Define:\n$$\n\\begin{array}{l}\nL_{a, b}=\\{r a+s b \\mid r, s \\in \\mathbf{Z}\\}, \\\\\nR_{a, b}=\\left\\{z \\mid z \\text { is a non-zero complex number, and } L_{a, b}=L_{z a, z}\\right\\} .\n\\end{array}\n$$\n\nTry to find the maximum number of elements in the set $R_{a, b}$ as $a, b$ vary.",
      "answer": "6"
    },
    {
      "question": "Problem 5. A square with an integer side length was cut into 2020 squares. It is known that the areas of 2019 squares are 1, and the area of the 2020th square is not equal to 1. Find all possible values that the area of the 2020th square can take. In your answer, provide the smallest of the obtained area values.",
      "answer": "112225"
    },
    {
      "question": "10-3. Point $O$ is the center of the circle. What is the value of angle $x$ in degrees?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d9f67fd4a24dc8d44db8g-25.jpg?height=488&width=870&top_left_y=2269&top_left_x=593)",
      "answer": "9"
    },
    {
      "question": "1. Let positive integers $a_{1}, a_{2}, \\cdots, a_{31}, b_{1}, b_{2}, \\cdots, b_{31}$ satisfy\n$$\n\\begin{array}{l}\n\\text { (1) } a_{1}<a_{2}<\\cdots<a_{31} \\leqslant 2015, \\\\\nb_{1}<b_{2}<\\cdots<b_{31} \\leqslant 2015 ; \\\\\n\\text { (2) } a_{1}+a_{2}+\\cdots+a_{31}=b_{1}+b_{2}+\\cdots+b_{31} \\text {. } \\\\\n\\text { Find } S=\\left|a_{1}-b_{1}\\right|+\\left|a_{2}-b_{2}\\right|+\\cdots+\\left|a_{31}-b_{31}\\right|\n\\end{array}\n$$\n\nthe maximum value.\n(Supplied by He Yijie)",
      "answer": "30720"
    },
    {
      "question": "## Task A-1.2.\n\nAll natural numbers from 1 to some number were written on a board. After one of the numbers was erased, the arithmetic mean of the remaining numbers on the board is $\\frac{673}{18}$. Which number was erased?",
      "answer": "9"
    },
    {
      "question": "[Pythagorean Theorem (direct and inverse).] [ Theorem on the sum of the squares of the diagonals ]\n\nThe height of a rhombus, drawn from the vertex of the obtuse angle, divides its side into segments of length $a$ and $b$. Find the diagonals of the rhombus.",
      "answer": "65"
    },
    {
      "question": "1. The numerical sequence $\\left\\{a_{n}\\right\\}_{n=1}^{\\infty}$ is defined such that $a_{1}=\\log _{2}\\left(\\log _{2} f(2)\\right), \\quad a_{2}=$ $\\log _{2}\\left(\\log _{2} f(f(2))\\right), \\ldots, a_{n}=\\log _{2}(\\log _{2} \\underbrace{f(f(\\ldots f}_{n}(2)))), \\ldots$, where $f(x)=x^{x}$. Determine the index $n$ for which $a_{n}=2059+2^{2059}$.\n\n(12 points)",
      "answer": "5"
    },
    {
      "question": "3 [ Perpendicularity of a line and a plane (other).]\n\nThe base of the pyramid is an equilateral triangle with a side length of 6. One of the lateral edges is perpendicular to the base plane and equals 4. Find the radius of the sphere circumscribed around the pyramid.\n\n#",
      "answer": "4"
    },
    {
      "question": "3. [5] Suppose\n$$\nh \\cdot a \\cdot r \\cdot v \\cdot a \\cdot r \\cdot d=m \\cdot i \\cdot t=h \\cdot m \\cdot m \\cdot t=100 \\text {. }\n$$\n\nFind $(r \\cdot a \\cdot d) \\cdot(t \\cdot r \\cdot i \\cdot v \\cdot i \\cdot a)$.",
      "answer": "10000"
    },
    {
      "question": "Problem 11.8. In a $28 \\times 35$ table, some $k$ cells are painted red, another $r-$ in pink, and the remaining $s-$ in blue. It is known that\n\n- $k \\geqslant r \\geqslant s$\n- each boundary cell has at least 2 neighbors of the same color;\n- each non-boundary cell has at least 3 neighbors of the same color.\n\nWhat is the smallest value that the quantity $k-s$ can take?\n\n(A cell is called boundary if it is adjacent to the edge of the table. Neighbors are cells that share a common side.)",
      "answer": "28"
    },
    {
      "question": "9.6. Petya and Vasya came up with ten quadratic trinomials. Then Vasya sequentially named consecutive natural numbers (starting from some number), and Petya substituted each named number into one of the trinomials of his choice and wrote down the obtained values on the board from left to right. It turned out that the numbers written on the board form an arithmetic progression (in this exact order). What is the maximum number of numbers Vasya could have named?\n\n(A. Golev)",
      "answer": "20"
    },
    {
      "question": "Example 6 Let $f(x)$ be an even function defined on $\\mathbf{R}$, whose graph is symmetric about the line $x=1$. For any $x_{1}, x_{2} \\in \\left[0, \\frac{1}{2}\\right]$, we have $f\\left(x_{1}+x_{2}\\right)=f\\left(x_{1}\\right) \\cdot f\\left(x_{2}\\right)$. Also, $f(1)=a>0$.\n(1) Find $f\\left(\\frac{1}{2}\\right)$ and $f\\left(\\frac{1}{4}\\right)$;\n(2) Prove that $f(x)$ is a periodic function;\n(3) Let $a_{n}=f\\left(2 n+\\frac{1}{2 n}\\right)$, find $\\lim _{n \\rightarrow \\infty}\\left(\\ln a_{n}\\right)$.",
      "answer": "0"
    },
    {
      "question": "8.2. Little kids were eating candies. Each one ate 7 candies less than all the others together, but still more than one candy. How many candies were eaten in total?",
      "answer": "21"
    },
    {
      "question": "4. As shown in Figure 9, the side length of square $A B C D$ is 10, and points $P$ and $Q$ are on the line segment connecting the midpoints of $A D$ and $B C$. Connecting $P A$, $P C$, $Q A$, and $Q C$, the square is divided into three parts of equal area. Find the length of line segment $P Q$.",
      "answer": "\\frac{20}{3}"
    },
    {
      "question": "5. The number 2015 was divided into 12 addends, after which all numbers that can be obtained by adding some of these addends (from one to nine) were written down. What is the minimum number of numbers that could have been written down?",
      "answer": "10"
    },
    {
      "question": "2. (15 points) Two medians of a triangle, equal to 18 and 24, are perpendicular to each other. Find the length of the third median of this triangle.",
      "answer": "30"
    },
    {
      "question": "13. Lucio buys a table in the shape of a regular hexagon and a rectangular tablecloth with an area of $1024 \\mathrm{~cm}^{2}$ that has the shorter side equal to the side of the hexagon and covers exactly the portion of the table between two opposite sides. What is the area of the table, expressed in $\\mathrm{cm}^{2}$?",
      "answer": "1536"
    },
    {
      "question": "(11 Let $\\sec x+\\tan x=\\frac{22}{7}, \\csc x+\\cot x=\\frac{m}{n}(m, n$ have no common divisor greater than 1), then $m+n=$ $\\qquad$",
      "answer": "44"
    },
    {
      "question": "40. Two different natural numbers $a, b$, their least common multiple is 64, then their greatest common divisor has\n$\\qquad$ possible situations.",
      "answer": "6"
    },
    {
      "question": "54. The task is very difficult-\n\nNot everyone can find:\n\nWhat does the star,\n\nBicycle and hedgehog equal?\n\n| BICYCLE | HEDGEHOG | 7 |\n| :---: | :---: | :---: |\n| + STAR | HEDGEHOG | 4 |\n| 6 | BICYCLE HEDGEHOG |  |\n| 1 BICYCLE | 0 | STAR |\n\nDecode the rebuses:",
      "answer": "1503"
    },
    {
      "question": "In the figure below, $A B C D$ and $E F G H$ are rectangles with parallel sides. Given that $A E=10$, $B F=20$ and $D H=30$, determine the length of the segment $C G$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-29.jpg?height=471&width=751&top_left_y=581&top_left_x=755)",
      "answer": "20\\sqrt{3}"
    },
    {
      "question": "91. A pasture is full of grass, which grows at a uniform rate every day. 17 cows can finish eating the grass in 30 days, 19 cows can finish in 20 days. Now, a certain number of cows eat for 5 days, then 3 cows are sold, and the remaining cows finish eating the grass in 2 more days. How many cows were there initially eating the grass (grass grows uniformly)?\n\n原有 $\\qquad$头牛吃草\n\nInitially, there were $\\qquad$ cows eating the grass.",
      "answer": "31"
    },
    {
      "question": "14. Let $a, b, c$ be positive real numbers such that\n$$\n\\left\\{\\begin{array}{l}\na^{2}+a b+b^{2}=25, \\\\\nb^{2}+b c+c^{2}=49, \\\\\nc^{2}+a+a^{2}=64 .\n\\end{array}\\right.\n$$\n\nFind $(a+b+c)^{2}$.",
      "answer": "129"
    },
    {
      "question": "96. A train takes 26 seconds to fully pass a bridge that is 335 meters long, and it takes 63 seconds to pass a bridge that is 1075 meters long at the same speed. The length of this train is _ meters.",
      "answer": "185"
    },
    {
      "question": "10,11 |\n\nLet $x, y, z$ be positive numbers and $x y z(x+y+z)=1$. Find the minimum value of the expression $(x+y)(x+z)$.",
      "answer": "2"
    },
    {
      "question": "3. Equilateral $\\triangle A B C$ and square $A B D E$ share a common side $A B$, the cosine of the dihedral angle $C-A B-D$ is $\\frac{\\sqrt{3}}{3}, M, N$ are the midpoints of $A C, B C$ respectively, then the cosine of the angle formed by $E M, A N$ is $\\qquad$ .",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "2. Point $B$ is internal to segment $\\overline{A C}$. In the same half-plane relative to line $A C$, equilateral triangles $\\triangle A B M$ and $\\triangle B C N$ are constructed. Lines $A N$ and $C M$ intersect at point $L$. Determine the angle $\\measuredangle C L N$.",
      "answer": "60"
    },
    {
      "question": "## 10.5 Find:\n\na) the real part of the complex number $z=\\frac{(2-i)^{3}}{3+4 i}$;\n\nb) the imaginary part of the complex number $z=\\left(\\frac{1+i}{1-i}\\right)^{11}$.\n\nPerform the operations (10.6-10.8):",
      "answer": "-1.52"
    },
    {
      "question": "## Task B-1.2.\n\nWith which digit does the number $2^{2022}+3^{2022}+7^{2022}$ end?",
      "answer": "2"
    },
    {
      "question": "Example 8 Determine the maximum value of $m^{2}+n^{2}$, where $m, n \\in\\{1,2, \\cdots, 2005\\}$ satisfy $\\left(n^{2}+2 m n-2 m^{2}\\right)^{2}=1$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "702036"
    },
    {
      "question": "## Task 1 - 290811\n\nOn a bottle of standard 40 percent acetic acid, the following instructions are given: \"The contents of this bottle ( $200 \\mathrm{ml}$ ), mixed with $800 \\mathrm{ml}$ of water, yield a ten percent vinegar.\"\n\nIs this correct?",
      "answer": "8"
    },
    {
      "question": "9. Given the function $f(x)=a x^{3}+b x^{2}+c x+d(a \\neq 0)$, when $0 \\leqslant x \\leqslant 1$, $\\left|f^{\\prime}(x)\\right| \\leqslant 1$, try to find the maximum value of $a$.",
      "answer": "\\frac{8}{3}"
    },
    {
      "question": "4. Find all positive real numbers $a$ such that there exists a positive integer $n$ and $n$ pairwise disjoint infinite sets $A_{1}, A_{2}, \\cdots, A_{n}$ satisfying $A_{1} \\cup A_{2} \\cup \\cdots \\cup A_{n}=\\mathbf{N}^{*}$, and for any two numbers $b>c$ in each $A_{i}$, we have $b-c \\geqslant a^{i}$.",
      "answer": "2"
    },
    {
      "question": "23. Container A contains 2 liters of sulfuric acid solution with a concentration of $30 \\%$, and container B contains 3 liters of sulfuric acid solution with a concentration of $10 \\%$. Now, sulfuric acid solutions with concentrations of $6 \\%$ and $22 \\%$ are being added to containers A and B, respectively, at rates of 0.1 liters per minute and 0.3 liters per minute. After $\\qquad$ minutes, the concentrations of the sulfuric acid solutions in containers A and B will be equal.",
      "answer": "20"
    },
    {
      "question": "$6 \\cdot 18$ Let $N$ be the set of positive integers, and define the function $f$ on $N$ as follows:\n(i) $f(1)=1, f(3)=3$;\n(ii) For $n \\in N$, we have\n$$\n\\begin{array}{l}\nf(2 n)=f(n), \\\\\nf(4 n+1)=2 f(2 n+1)-f(n), \\\\\nf(4 n+3)=3 f(2 n+1)-2 f(n) .\n\\end{array}\n$$\n\nFind all $n$ such that $n \\leqslant 1988$ and $f(n)=n$.",
      "answer": "92"
    },
    {
      "question": "3. Six children are sledding in a train formation down a hill. In how many different ways can they slide down if one of the children, who has peculiarities, believes that it is contraindicated for him to sit in even-numbered positions.",
      "answer": "360"
    },
    {
      "question": "[ Numerical inequalities. Comparing numbers.]\n\nWhat is greater: $1234567 \\cdot 1234569$ or $1234568^{2}$ ?",
      "answer": "1234568^{2}"
    },
    {
      "question": "8,9 trapezoids are constructed with circles on their diameters, intersecting at points $M$ and $N$. Chord $M N$ intersects the base $A D$ at point $E$. Find the ratio $A E: E D$.\n#",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "Problem 10. (4 points)\n\nTo buy new headphones costing 275 rubles, Katya decided to save money on sports activities. Until now, she has been buying a single-visit ticket to the swimming pool, including a visit to the sauna for 250 rubles, to warm up. However, summer has arrived, and the need to visit the sauna has disappeared. Visiting only the swimming pool costs 200 rubles more than visiting the sauna. How many times does Katya need to visit the swimming pool without the sauna to save enough to buy the headphones? (Provide the answer as a whole number, without spaces or units of measurement.)\n\nAnswer: 11.\n\n## Comment:",
      "answer": "11"
    },
    {
      "question": "12. Let the set $M=\\{1,2,3,4,5,6,7,8,9,10\\}, A=\\left\\{(x, y, z) \\mid x, y, z \\in M\\right.$, and $\\left.9 \\mid\\left(x^{3}+y^{3}+z^{3}\\right)\\right\\}$, then the number of elements in set $A$ is $\\qquad$.",
      "answer": "243"
    },
    {
      "question": "9. The decagon $A B C D P Q R S T U V W$ has two parallel bases: square $A B C D$ and octagon $P Q R S T U V W$, all angles of which are equal, and eight lateral faces: triangles $A P Q, B R S, C T U, D V W$ and rectangles $D A P W, A B R Q, B C T S$ and $C D V U$. It is known that the area of the section of the decagon by a plane passing through points $A, S$ and $U$ is $\\frac{1372}{585},|A B|=1 / 3,|P Q|=2 \\sqrt{2} / 3$. Find the distance between the bases of the decagon. Write the found value in the answer, rounding it if necessary.",
      "answer": "2.65"
    },
    {
      "question": "C4. Players $A$ and $B$ play a game with $N \\geq 2012$ coins and 2012 boxes arranged around a circle. Initially $A$ distributes the coins among the boxes so that there is at least 1 coin in each box. Then the two of them make moves in the order $B, A, B, A, \\ldots$ by the following rules:\n- On every move of his $B$ passes 1 coin from every box to an adjacent box.\n- On every move of hers $A$ chooses several coins that were not involved in $B$ 's previous move and are in different boxes. She passes every chosen coin to an adjacent box.\n\nPlayer A's goal is to ensure at least 1 coin in each box after every move of hers, regardless of how $B$ plays and how many moves are made. Find the least $N$ that enables her to succeed.",
      "answer": "4022"
    },
    {
      "question": "45th Putnam 1984 Problem A5 V is the pyramidal region x, y, z >= 0, x + y + z <= 1. Evaluate ∫ V x y 9 z 8 (1 - x - y - z) 4 dx dy dz. Solution",
      "answer": "\\frac{9!8!4!}{25!}"
    },
    {
      "question": "Moskvitin N.A.\n\nIn triangle $A B C$, $A B = B C$. A perpendicular $E D$ is dropped from point $E$ on side $A B$ to $B C$. It turns out that $A E = E D$. Find the angle $D A C$.",
      "answer": "45"
    },
    {
      "question": "6. How many $3 \\times 5$ rectangular pieces of cardboard can be cut from a $17 \\times 22$ rectangular piece of cardboard, when the amount of waste is minimised?",
      "answer": "24"
    },
    {
      "question": "Old cobbler Karl made boots and sent his son Hans to the market to sell them for 25 talers. At the market, two disabled men (one without a left leg, the other without a right leg) approached the boy and asked him to sell them each a boot. Hans agreed and sold each boot for 12.5 talers.\n\nWhen the boy came home and told his father everything, Karl decided that the disabled men should have been sold the boots cheaper - each for 10 talers. He gave Hans 5 talers and told him to return 2.5 talers to each disabled man.\n\nWhile the boy was looking for the disabled men at the market, he saw that sweets were being sold, couldn't resist, and spent 3 talers on candies. After that, he found the disabled men and gave them the remaining money - one taler each. Returning home, Hans realized how badly he had acted. He told everything to his father and asked for forgiveness. The cobbler got very angry and punished his son by putting him in a dark storeroom.\n\nSitting in the storeroom, Hans thought about it. It turned out that since he had returned one taler to each, the disabled men had paid 11.5 talers for each boot:\n\n$12.5-1=11.5$. So, the boots cost 23 talers: $2 \\cdot 11.5=23$. And Hans had spent 3 talers on candies, so in total it comes to 26 talers:\n\n$23+3=26$. But there were only 25 talers! Where did the extra taler come from?",
      "answer": "20"
    },
    {
      "question": "3. Find the number of natural numbers $k$, not exceeding 454500, such that $k^{2}-k$ is divisible by 505.",
      "answer": "3600"
    },
    {
      "question": "6. (7 points) Several sages lined up in a column. All of them wore either black or white caps. It turned out that among any 10 consecutive sages, there were an equal number of sages with white and black caps, while among any 12 consecutive sages - not an equal number. What is the maximum number of sages that could be\n\nAnswer: 15 sages.",
      "answer": "15"
    },
    {
      "question": "Let's share a thought: divide every natural number by the sum of the squares of its digits (for single-digit numbers, divide by the square of the number).\n\nIs there a smallest quotient among the quotients obtained this way, and if so, which one is it?",
      "answer": "\\frac{1}{9}"
    },
    {
      "question": "7. Anchurian Checkers ${ }^{1}$ (from 7th grade, 1 point). A tournament of Anchurian Checkers is held in several rounds. If an even number of players participate in a round, they are divided into random pairs. If the number of players is odd, random pairs are selected by lot, and one player remains without a pair and does not participate in the round. A draw cannot occur; the loser in each pair is eliminated from the tournament, while the winners and the player without a pair, if there is one, advance to the next round, which is conducted according to the same rules. This continues until only two players remain, who play the final round, that is, the last game, which determines the winner of the tournament.[^5]\n\nXIV Correspondence Internet Olympiad in Probability Theory and Statistics for Schoolchildren\n\n(c) Moscow Center for Continuous Mathematical Education, 2021\n\n26 participants arrived at the Anchurian Checkers tournament, and all of them play equally well, meaning that in any game played by any two of them, the chances of the opponents are equal. Among the players are Denis and Oleg. Find the probability that they will play against each other.",
      "answer": "\\frac{1}{13}"
    },
    {
      "question": "[The ratio in which the bisector divides the side.] [Pythagorean Theorem (direct and inverse).]\n\nIn triangle $ABC$, the height $BM$, the bisector $BN$, and the median $BL$ are drawn. It is known that $AM = MN = NL$. Find the tangent of angle $A$ of this triangle.",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "Problem 4.7. At the festival of namesakes, 45 Alexanders, 122 Borises, 27 Vasily, and several Gennady arrived. At the beginning of the festival, all of them stood in a row so that no two people with the same name stood next to each other. What is the minimum number of Gennady that could have arrived at the festival?",
      "answer": "49"
    },
    {
      "question": "2. A cube with edge $a=1 / \\sqrt{2}$ is illuminated by a cylindrical beam of light with radius $\\rho=\\sqrt{2-\\sqrt{3}}$, directed along the main diagonal of the cube (the axis of the beam contains the main diagonal). Find the area of the illuminated part of the cube's surface.",
      "answer": "(\\sqrt{3}-\\frac{3}{2})(\\pi+3)"
    },
    {
      "question": "20. Patrick asks SpongeBob: \"What is your lucky number?\" SpongeBob says: \"There is a sequence of numbers: $0,1,3,8$, $21,55,144,377,987, \\cdots \\cdots$ Starting from the third number, each number multiplied by 3 is exactly equal to the sum of the numbers immediately before and after it. My lucky number is the remainder when the 2020th number in this sequence is divided by 6.\" So, SpongeBob's lucky number is $\\qquad$ . (If it is divisible, then the remainder is 0)",
      "answer": "2"
    },
    {
      "question": "Problem 2. Pentagon $A B C D E$ is inscribed in circle $\\omega$. Diagonal $A C$ is the diameter of circle $\\omega$. Find $\\angle B E C$, if $\\angle A D B=20^{\\circ}$.",
      "answer": "70"
    },
    {
      "question": "17. In a certain country, an aviation network has already been established, where any city is connected to no more than three other cities, and from any city to another city, it takes no more than one transfer. How many cities can this country have at most?",
      "answer": "10"
    },
    {
      "question": "3 [Inscribed, circumscribed, and exscribed circles; their radii]\n\nTriangle $A B C$ is isosceles. The radius $O A$ of the circumscribed circle forms an angle $O A C$ with the base $A C$, which is $20^{\\circ}$. Find the angle $B A C$.",
      "answer": "35"
    },
    {
      "question": "Petr and Pavel were picking apples and pears in the orchard. On Monday, Petr ate 2 more pears than Pavel and 2 fewer apples than Pavel. On Tuesday, Petr ate 4 fewer pears than on Monday. Pavel ate 3 more pears than Petr and 3 fewer apples than Petr on Tuesday. Pavel ate 12 apples over the two days and the same number of apples as pears on Tuesday. In the evening on Tuesday, both boys found out that the number of apples they had eaten together over the two days was the same as the number of pears they had eaten together. How many apples did Petr eat on Monday and how many pears did Pavel eat on Tuesday?",
      "answer": "7"
    },
    {
      "question": "## Task 1 - 160721\n\nAfter the coming-of-age ceremony, all students in a class had their individual photos taken. Each student had enough prints made of their photo, and then each student in this class exchanged their photo with each of their classmates.\n\nHow many students in this class exchanged photos with each other if exactly 812 photographs changed hands?",
      "answer": "29"
    },
    {
      "question": "3.48 Let a $10 \\times 10$ grid table $A$ contain an integer in each cell. Denote the sum of the numbers in the $i$-th row as $s_{i}$, and the sum of the numbers in the $j$-th column as $t_{j}, i, j=1,2, \\cdots, 10$. Then fill a new $10 \\times 10$ grid table $B$ in the following way: in the cell at the $i$-th row and $j$-th column, write the smaller of $s_{i}$ and $t_{j}$. It is known that the cells in table $B$ can be numbered from 1 to 100 such that the number in the $k$-th cell is no greater than $k$. Find the maximum possible sum of all the numbers in table $A$.",
      "answer": "955"
    },
    {
      "question": "Bogdanov I.i.\n\nPetya wants to list all possible sequences of 100 natural numbers, in each of which a triple appears at least once, and any two adjacent members differ by no more than 1. How many sequences will he have to list?",
      "answer": "3^{100}-2^{100}"
    },
    {
      "question": "18 Real numbers $x, y, z$ not all positive satisfy $k\\left(x^{2}-x+1\\right)\\left(y^{2}-y+\\right.$ 1) $\\left(z^{2}-z+1\\right) \\geqslant(x y z)^{2}-x y z+1$, find the minimum value of the real number $k$.",
      "answer": "\\frac{16}{9}"
    },
    {
      "question": "Problem 6. Having defeated Koschei, Ivan demanded gold to ransom Vasilisa from the bandits. Koschei led him to a cave and said:\n\n\"In the chest lie gold ingots. But you cannot simply take them: they are enchanted. Put one or several into your bag. Then I will put one or several from the bag back into the chest, but it must be a different number. We will take turns moving them: you into the bag, I into the chest, each time a new number. When a new move becomes impossible, you can take your bag with the ingots.\"\n\nWhat is the maximum number of ingots Ivan can take, no matter how Koschei acts, if the chest initially contains a) 13; b) 14 gold ingots? How can he do it?\n\n[8 points for both parts, 5 points for one] (A. V. Shapovalov)",
      "answer": "13"
    },
    {
      "question": "7. If $x, y, k$ are positive reals such that\n$$\n3=k^{2}\\left(\\frac{x^{2}}{y^{2}}+\\frac{y^{2}}{x^{2}}\\right)+k\\left(\\frac{x}{y}+\\frac{y}{x}\\right) \\text {, }\n$$\nfind the maximum possible value of $k$.",
      "answer": "\\frac{-1+\\sqrt{7}}{2}"
    },
    {
      "question": "SI. 1 Given that $\\frac{3}{a}+\\frac{1}{u}=\\frac{7}{2}$ and $\\frac{2}{a}-\\frac{3}{u}=6$ are simultaneous equations in $a$ and $u$. Solve for $a$.",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "2. Add 3 digits after 325 to form a six-digit number, so that it can be divisible by $3$, $4$, and $5$, and make this value as small as possible. Then this new six-digit number is $\\qquad$ _.",
      "answer": "325020"
    },
    {
      "question": "$\\left[\\begin{array}{l}{[\\text { Transfer of sides, diagonals, etc. ] }} \\\\ {[\\text { Area of a trapezoid }}\\end{array}\\right]$\n\nFind the area of a trapezoid with bases 11 and 4 and diagonals 9 and 12.",
      "answer": "54"
    },
    {
      "question": "3. Initially, the numbers $1, 2, 3, \\cdots, 2013$ are written on the blackboard. In each step, you can erase two numbers $a, b$ from the blackboard and write down $a+b$. What is the minimum number of steps required to ensure that the sum of any remaining numbers on the blackboard is not equal to 2014?\n(Lin Chang provided the problem)",
      "answer": "503"
    },
    {
      "question": "1. (10 points) $\\frac{1-\\frac{1}{3}}{1 \\times \\frac{1}{2} \\times \\frac{1}{3}}+\\frac{\\frac{1}{3}-\\frac{1}{5}}{\\frac{1}{3} \\times \\frac{1}{4} \\times \\frac{1}{5}}+\\cdots+\\frac{\\frac{1}{2015}-\\frac{1}{2017}}{\\frac{1}{2015} \\times \\frac{1}{2016} \\times \\frac{1}{2017}}=$ $\\qquad$",
      "answer": "2034144"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0} \\frac{e^{2 x}-e^{3 x}}{\\operatorname{arctg} x-x^{2}}$",
      "answer": "-1"
    },
    {
      "question": "4. All natural numbers, the sum of the digits in the representation of which is divisible by 5, are listed in ascending order: $5,14,19,23,28,32, \\ldots$ What is the smallest positive difference between consecutive numbers in this sequence? Provide an example and explain why it cannot be smaller.\n\n---\n\nThe smallest positive difference between consecutive numbers in this sequence is 1. An example is the pair of numbers 14 and 15. The sum of the digits of 14 is $1 + 4 = 5$, which is divisible by 5. The sum of the digits of 15 is $1 + 5 = 6$, which is not divisible by 5. However, the next number in the sequence is 19, and the sum of its digits is $1 + 9 = 10$, which is divisible by 5. Therefore, the smallest difference is 1, as seen between 14 and 15.\n\nTo explain why it cannot be smaller: The smallest possible difference between two numbers is 1. If the difference were 0, the numbers would be the same, which is not possible for consecutive numbers in the sequence. If the difference were greater than 1, there would be a gap between the numbers, which means there would be a number in between that could potentially have a sum of digits divisible by 5, contradicting the requirement that the sequence lists all such numbers in ascending order. Therefore, the smallest possible difference is 1.",
      "answer": "1"
    },
    {
      "question": "5.3. For the celebration of the Name Day in the 5th grade parallel, several pizzas were ordered. 22 pizzas were ordered for all the boys, with each boy getting an equal share. Each girl also received an equal share, but half as much as each boy. How many pizzas were ordered if it is known that there are 13 girls in this parallel, and there are more boys than girls? Pizzas can be divided into parts.",
      "answer": "23"
    },
    {
      "question": "4. Suppose 40 objects are placed along a circle at equal distances. In how many ways can 3 objects be chosen from among them so that no two of the three chosen objects are adjacent nor diametrically opposite?",
      "answer": "7720"
    },
    {
      "question": "4.4.9 ** Let $x_{1}, x_{2}, \\cdots, x_{n}$ and $a_{1}, a_{2}, \\cdots, a_{n}(n>2)$ be two sets of arbitrary real numbers satisfying the conditions:\n(1) $x_{1}+x_{2}+\\cdots+x_{n}=0$;\n(2) $\\left|x_{1}\\right|+\\left|x_{2}\\right|+\\cdots+\\left|x_{n}\\right|=1$;\n(3) $a_{1} \\geqslant a_{2} \\geqslant \\cdots \\geqslant a_{n}$. To make the inequality $\\mid a_{1} x_{1}+a_{2} x_{2}+\\cdots+ a_{n} x_{n} \\mid \\leqslant A\\left(a_{1}-a_{n}\\right)$ hold, find the minimum value of the number $A$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "For example, if you throw a die, you have two chances to throw. You can choose to throw once or twice, but the number of the last throw will be the final result. Find the mathematical expectation of the points obtained by such a throw. If you have three chances, still taking the number of the last throw as the final result, find the mathematical expectation of the points obtained by such a throw.",
      "answer": "\\frac{14}{3}"
    },
    {
      "question": "10. (16 points) Figure 1 is a rhombus paper piece composed of 2 small equilateral triangles; Figure 2 is a fixed regular hexagonal board $A B C D E F$, which is made up of 24 equally sized small equilateral triangles. Now, 12 rhombus paper pieces are used to completely cover the hexagonal board, there are $\\qquad$ different covering methods.",
      "answer": "20"
    },
    {
      "question": "$2 \\cdot 56$ Find the smallest natural number, such that when its last digit is moved to the first position, the number is multiplied by 5.",
      "answer": "142857"
    },
    {
      "question": "15. (6 points) A motorcycle travels a 60-kilometer distance at an average speed of 20 kilometers per hour, and on the return trip, its average speed is 30 kilometers per hour. What is the average speed of the motorcycle for the entire round trip?",
      "answer": "24"
    },
    {
      "question": "# 2. Solution:\n\n$\\sin ^{4} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2018} 2019 x=1 \\Leftrightarrow\\left(1-\\cos ^{2} 2019 x\\right)^{2}+\\cos ^{2019} 2022 x \\cdot \\cos ^{2018} 2019 x=1 \\Leftrightarrow$ $\\cos ^{4} 2019 x-2 \\cos ^{2} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2018} 2019 x=0 \\Leftrightarrow$ $\\cos ^{2} 2019 x\\left(\\cos ^{2} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2016} 2019 x-2\\right)=0 \\Leftrightarrow$ 1) $\\cos 2019 x=0, x=\\frac{\\pi}{4038}+\\frac{\\pi n}{2019}, n \\in Z$ or 2) $\\cos ^{2} 2019 x+\\cos ^{2019} 2022 x \\cdot \\cos ^{2016} 2019 x=2 \\Leftrightarrow$ $\\left\\{\\begin{array}{c}\\cos ^{2} 2019 x=1, \\\\ \\cos ^{2019} 2022 x \\cdot \\cos ^{2016} 2019 x=1,\\end{array} \\Leftrightarrow\\left\\{\\begin{array}{l}\\sin 2019 x=0, \\\\ \\cos 2022 x=1,\\end{array} \\Leftrightarrow\\left\\{\\begin{array}{l}x=\\frac{\\pi n}{2019}, n \\in Z, \\\\ x=\\frac{\\pi m}{1011}, m \\in Z,\\end{array}\\right.\\right.\\right.$ $\\frac{n}{2019}=\\frac{m}{1011}, n=\\frac{2019 m}{1011}=\\frac{673 m}{337}, n \\in Z, m \\in Z \\Rightarrow m=337 k, \\quad x=\\frac{\\pi k}{3}, k \\in Z$",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "## Task 4 - 250934\n\nThe eight numbers 1, $2, \\ldots, 8$ are to be distributed to the vertices of a cube in such a way that the following conditions are met:\n\n(1) Each vertex of the cube is assigned exactly one of the eight numbers, and each of these numbers must appear in the distribution.\n\n(2) If one adds the four numbers at the vertices of each face of the cube, the sum on all six faces is the same.\n\nAs many distributions of the eight numbers to the vertices should be compiled as possible, such that each of these distributions meets conditions (1), (2), and no two of these distributions are congruent to each other, i.e., can be transformed into each other by rotation or reflection.\n\nDetermine the number of distributions that appear in such a compilation!",
      "answer": "2"
    },
    {
      "question": "Problem 2. Let $a, b, c, d$ be natural numbers such that $a+b+c+d=2018$. Find the minimum value of the expression\n\n$$\nE=(a-b)^{2}+2(a-c)^{2}+3(a-d)^{2}+4(b-c)^{2}+5(b-d)^{2}+6(c-d)^{2}\n$$",
      "answer": "14"
    },
    {
      "question": "$A B C$ is an equilateral triangle $(A B=a)$. Around vertex $A$, we draw two arcs, one with radius $r$ beyond the extensions of sides $A B$ and $A C$, and the other with radius $a+r$ between the rays of angle $B A C$. Similarly, we proceed around vertices $B$ and $C$, and the six arcs enclose a so-called Reuleaux $\\mathbf{R}$ domain of width $a+2 r$. We rotate this domain within a square frame of side $a+2 r$. What segments of the frame's sides does $\\mathbf{R}$ rub against during this process: 11[^0]\n\n\n[^0]:    ${ }^{1}$ Readers should cut out the shapes from cardboard, or draw one on transparent paper and move it over the other.",
      "answer": "(\\sqrt{3}-1)"
    },
    {
      "question": "Problem 9.2. Vanya thought of two natural numbers, the product of which equals 7200. What is the greatest value that the GCD of these numbers can take?",
      "answer": "60"
    },
    {
      "question": "# 9. Solution.\n\n1st method. An elementary outcome in the random experiment is a triplet of positions where the children in green caps stand. Consider the event $A$ \"all three green caps are together.\" This event is favorable in 9 elementary outcomes. The event $B$ \"two green caps are together, and the third is separate\" is favorable in 45 elementary outcomes (9 ways to choose two adjacent places, and the third place must be one of the 5 places not adjacent to the already chosen ones). The total number of ways to choose a triplet is $C_{9}^{3}=84$. Then the probability of the event \"some two green caps are together\" is $\\mathrm{P}(A)+\\mathrm{P}(B)=\\frac{54}{84}=\\frac{9}{14}$, and the desired probability is $1-\\mathrm{P}(A)-\\mathrm{P}(B)=\\frac{5}{14}$.\n\n2nd method. Number the children in green caps and form pairs \"green-red\" such that a red cap is placed to the right of each green cap. To the right of the first green cap, any of the six red caps can be placed, to the right of the second, one of the five remaining, and to the right of the third, one of the four remaining red caps. This results in three pairs \"green-red\" and three separate red caps. There are exactly $6 \\cdot 5 \\cdot 4=120$ such groupings. Arrange these pairs and separate caps (a total of six elements) in a circle. This can be done in 5! ways, and no two green caps will be adjacent, as a red cap is located to the right of each green cap.\n\nThe total number of ways to arrange 9 children in a circle is 8!.\n\nThus, the desired probability is $\\frac{120 \\cdot 5!}{8!}=\\frac{120}{8 \\cdot 7 \\cdot 6}=\\frac{5}{14}$.\n\n3rd method is similar to the 2nd method from the first variant. Other methods of solution are possible.",
      "answer": "\\frac{5}{14}"
    },
    {
      "question": "1. Calculate: $7.625-6 \\frac{1}{3}+5.75-1 \\frac{3}{8}=$",
      "answer": "5\\frac{2}{3}"
    },
    {
      "question": "2. Let the sequence $\\left\\{a_{n}\\right\\}$ satisfy, $a_{1}=1, a_{n+1}=5 a_{n}+1(n=1,2, \\cdots)$, then $\\sum_{n=1}^{2018} a_{n}=$",
      "answer": "\\frac{5^{2019}}{16}-\\frac{8077}{16}"
    },
    {
      "question": "11.5. In Flower City, there live $99^{2}$ dwarfs. Some of the dwarfs are knights (always tell the truth), while the rest are liars (always lie). The houses in the city are located in the cells of a $99 \\times 99$ square (a total of $99^{2}$ houses, arranged in 99 vertical and 99 horizontal streets). Each house is inhabited by exactly one dwarf. The house number is denoted by a pair of numbers $(x ; y)$, where $1 \\leqslant x \\leqslant 99$ is the number of the vertical street (numbers increase from left to right), and $1 \\leqslant y \\leqslant 99$ is the number of the horizontal street (numbers increase from bottom to top). The Flower City distance between two houses with numbers $\\left(x_{1} ; y_{1}\\right)$ and $\\left(x_{2} ; y_{2}\\right)$ is the number $\\rho=\\left|x_{1}-x_{2}\\right|+\\left|y_{1}-y_{2}\\right|$. It is known that on each street, whether vertical or horizontal, there live at least $k$ knights. In addition, all dwarfs know which house Knight Knowitall lives in. You want to find his house, but you do not know what Knowitall looks like. You can approach any house and ask the dwarf living there: “What is the Flower City distance from your house to the house of Knowitall?”. For what smallest $k$ can you guarantee to find the house of Knowitall? (V. Novikov)",
      "answer": "75"
    },
    {
      "question": "For what value of $a$ does the polynomial $P(x)=x^{1000}+a x^{2}+9$ divide by $x+1$?",
      "answer": "-10"
    },
    {
      "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the equations in polar coordinates.\n\n$$\n\\rho=8 \\cos \\varphi, 0 \\leq \\varphi \\leq \\frac{\\pi}{4}\n$$",
      "answer": "2\\pi"
    },
    {
      "question": "## Problem Statement\n\nFind the cosine of the angle between vectors $\\overrightarrow{A B}$ and $\\overrightarrow{A C}$.\n\n$A(2, -8, -1), B(4, -6, 0), C(-2, -5, -1)$",
      "answer": "-\\frac{2}{15}"
    },
    {
      "question": "5. (15 points) As shown in the figure, the side length of the square is 80 cm, $O$ is the center of the square, $A$ is the midpoint of $O B$, and within the square, a circle is drawn with $A$ as the center and $O A$ as the radius. Another circle is drawn with $B$ as the center and $O B$ as the radius, forming a special shape with one side of the square. Rotating this shape around point $O$ three times in a clockwise direction forms a windmill shape. What is the area of this windmill (shaded part) in square centimeters? (Take $\\pi$ as 3.14)",
      "answer": "912"
    },
    {
      "question": "Example 4.8 Find the number of positive integer solutions to the equation $x_{1}+2 x_{2}+4 x_{3}=21$.\n\n untranslated text remains the same as requested.",
      "answer": "20"
    },
    {
      "question": "257. Given a regular tetrahedron with edge $a$. Find the radius of a sphere centered at the center of the tetrahedron, such that the combined volume of the part of the tetrahedron outside the sphere and the part of the sphere outside the tetrahedron is minimized.",
      "answer": "\\frac{1}{3}\\sqrt{\\frac{2}{3}}"
    },
    {
      "question": "4. In tetrahedron $ABCD$, $\\angle ADB = \\angle BDC = \\angle CDA = 60^{\\circ}$, the areas of $\\triangle ADB$, $\\triangle BDC$, and $\\triangle CDA$ are $\\frac{\\sqrt{3}}{2}$, $2$, and $1$, respectively. Then the volume of the tetrahedron is . $\\qquad$",
      "answer": "\\frac{2\\sqrt{6}}{9}"
    },
    {
      "question": "## Problem Statement\n\nCalculate the areas of figures bounded by lines given in polar coordinates.\n\n$$\nr=\\cos \\phi, r=\\sqrt{2} \\sin \\left(\\phi-\\frac{\\pi}{4}\\right),\\left(-\\frac{\\pi}{4} \\leq \\phi \\leq \\frac{\\pi}{2}\\right)\n$$",
      "answer": "\\frac{\\pi}{4}-\\frac{1}{4}"
    },
    {
      "question": "Kluci found an old minefield plan, see the picture. The numbers are on the fields where there are no mines, and they indicate the number of adjacent mined fields. Determine how many mines there are in the field and where they are. (Fields are adjacent if they share a vertex or a side.)\n\n| 1 |  | 2 |  | 2 |\n| :--- | :--- | :--- | :--- | :--- |\n|  | 3 |  | 3 |  |\n| 3 |  |  |  | 3 |\n|  | 2 |  |  |  |\n|  |  |  | 2 |  |",
      "answer": "8"
    },
    {
      "question": "15. Let the base of the pyramid $M-ABCD$ be a square, and $MA=MD, MA \\perp AB$. If the area of $\\triangle AMD$ is 1, find the radius of the largest sphere that can fit inside this pyramid.",
      "answer": "\\sqrt{2}-1"
    },
    {
      "question": "Zlobin C.A.\n\nThe greatest common divisor of natural numbers $m$ and $n$ is 1. What is the greatest possible value of $\\gcd(m+2000 n, n+2000 m) ?$",
      "answer": "3999999"
    },
    {
      "question": "Problem 9.2. In four classes of a school, there are more than 70 children, all of whom came to the grade meeting (no other children were present at the meeting).\n\nEach girl who came was asked: \"How many people from your class, including you, came to the meeting?\"\n\nEach boy who came was asked: \"How many boys from your class, including you, came to the meeting?\"\n\nAmong the answers, the numbers $7,9,10,12,15,16,19$ and 21 were encountered (all children answered correctly).\n\n(a) (1 point) How many children are there in the largest class of the grade?\n\n(b) (3 points) How many girls came to the grade meeting?",
      "answer": "33"
    },
    {
      "question": "75. If $x+y=\\sqrt{3 \\sqrt{5}-\\sqrt{2}}, x-y=\\sqrt{3 \\sqrt{2}-\\sqrt{5}}$, then $xy=$",
      "answer": "\\sqrt{5}-\\sqrt{2}"
    },
    {
      "question": "11. Given $n^{2}=n \\times n$, find the last digit of $1^{2}+2^{2}+3^{2}+\\ldots+2016^{2}+2017^{2}$",
      "answer": "5"
    },
    {
      "question": "10.3. Three people are playing table tennis, with the player who loses a game giving way to the player who did not participate in it. In the end, it turned out that the first player played 10 games, the second - 21. How many games did the third player play?",
      "answer": "11"
    },
    {
      "question": "Problem 4. In triangle $A B C$ with the ratio of sides $A B: A C=7: 2$, the bisector of angle $B A C$ intersects side $B C$ at point $L$. Find the length of segment $A L$, if the length of the vector $2 \\cdot \\overrightarrow{A B}+7 \\cdot \\overrightarrow{A C}$ is 2016.",
      "answer": "224"
    },
    {
      "question": "Problem 11.5. The tangent lines to the circumcircle $k$ of an isosceles $\\triangle A B C$, $A C=B C$, at the points $B$ and $C$ meet at point $X$. If $A X$ meets $k$ at point $Y$, find the ratio $\\frac{A Y}{B Y}$.\n\nEmil Kolev",
      "answer": "2"
    },
    {
      "question": "(2) Let $X=\\{1,2, \\cdots, 2001\\}$. Find the smallest positive integer $m$, such that: for any $m$-element subset $W$ of $X$, there exist $u, v \\in W$ (where $u$ and $v$ can be the same), such that $u+v$ is a power of 2.",
      "answer": "999"
    },
    {
      "question": "3. (17 points) Find the smallest natural number that is simultaneously twice a perfect square and three times a perfect cube.\n\n#",
      "answer": "648"
    },
    {
      "question": "1. Vasya and Petya drew a five-pointed star each. All the angles at the vertices of Petya's star are acute, while Vasya's star has an obtuse angle. Each of them claims that the sum of the angles at the vertices of his star is greater. Who is right?\n![](https://cdn.mathpix.com/cropped/2024_05_06_ad0ddfff017bcb029f08g-1.jpg?height=328&width=758&top_left_y=1595&top_left_x=654)",
      "answer": "180"
    },
    {
      "question": "The midpoint of each side of the parallelogram is connected to the ends of the opposite side. Find the area of the octagon formed by the intersections of the drawn segments, if the area of the parallelogram is 1.\n\n#",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "39. Yura borrowed a book from Lena for three days. On the first day, he read half of the book, on the second day, he read a third of the remaining pages, and on the third day, he read a number of pages equal to half the number of pages he read in the first two days. Did Yura manage to finish the book in three days?",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "1. (8 points) Calculate: $2013+201+20-13+3=$",
      "answer": "2224"
    },
    {
      "question": "7. (10 points) The difference in area between two squares is 2016 square centimeters. If the side lengths of such pairs of squares are both integer centimeters, then how many pairs of squares satisfy the above condition.",
      "answer": "12"
    },
    {
      "question": "8. (6 points) The three-digit number $2 a 3$ plus 326 equals $5 b 9$. If $5 b 9$ is a multiple of 9, then $a+b=$\n\n untranslated portion:\n```\n(6 points) The three-digit number $2 a 3$ plus 326 equals $5 b 9$. If $5 b 9$ is a multiple of 9, then $a+b=$\n```",
      "answer": "6"
    },
    {
      "question": "Kanel-Belov A.Y.\n\nA cube with a side of 20 is divided into 8000 unit cubes, and a number is written in each cube. It is known that in each column of 20 cubes, parallel to the edge of the cube, the sum of the numbers is 1 (columns in all three directions are considered). In a certain cube, the number 10 is written. Through this cube, three layers $1 \\times 20 \\times 20$, parallel to the faces of the cube, pass. Find the sum of all numbers outside these layers.",
      "answer": "333"
    },
    {
      "question": "6. Given 4 spheres with a radius of 1, each pair of which are externally tangent, then the edge length of the circumscribed regular tetrahedron of the geometric body formed by these 4 spheres is",
      "answer": "2\\sqrt{6}+2"
    },
    {
      "question": "9. Solution. To get a complete collection, it is necessary that among the 9 coupons from Semy, there are 6 that Temy is missing. The probability of this is\n\n$$\n\\frac{C_{6}^{6} \\cdot C_{11}^{3}}{C_{17}^{9}}=\\frac{3}{442} \\approx 0.007\n$$",
      "answer": "0.007"
    },
    {
      "question": "33. How many six-digit numbers are there in which all digits are odd?",
      "answer": "15625"
    },
    {
      "question": "$10 \\cdot 23$ has a four-digit number with thousands, hundreds, tens, and units digits being $a, b, c, d$ respectively, the sum of the digits is $26$, the tens digit of the product of $b, d$ equals $(a+c)$, and $(bd - c^2)$ is a power of 2. Find this four-digit number (explain the reasoning).\n(Heilongjiang Province, China Mathematics Competition, 1979)",
      "answer": "1979"
    },
    {
      "question": "XXII OM - III - Task 3\n\nHow many locks at least need to be placed on the treasury so that with a certain distribution of keys among the 11-member committee authorized to open the treasury, any 6 members can open it, but no 5 can? Determine the distribution of keys among the committee members with the minimum number of locks.",
      "answer": "462"
    },
    {
      "question": "9. (12 points) Fill in the blanks with the numbers $2$, $0$, $1$, $5$, or leave them blank, so that each row and each column contains one $2$, $0$, $1$, $5$. The same number cannot be diagonally adjacent. Therefore, the first five positions of the fifth row are $\\qquad$ (use 9 to represent a blank).\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "15992"
    },
    {
      "question": "Florinda went for a walk. She left point $A$ and walked $1 \\mathrm{~m}$. At this point, she turned left at an angle of $90^{\\circ}$ and walked $2 \\mathrm{~m}$. At the last point, she turned left again and walked $3 \\mathrm{~m}$. She continued walking in this manner until, on the last segment, she walked $30 \\mathrm{~m}$ and reached her final point, which we will call point $B$. The following figure illustrates the first seven segments of Florinda's walk (note that point $B$ is not illustrated).\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_463eeeabc25d438954b1g-13.jpg?height=523&width=536&top_left_y=1470&top_left_x=837)\n\nCalculate the distance between points $A$ and $B$.\n\n#",
      "answer": "\\sqrt{481}"
    },
    {
      "question": "(15) Let $\\left\\{a_{n}\\right\\}$ be a geometric sequence, and each term is greater than 1, then $\\lg a_{1} \\lg a_{2012} \\sum_{i=1}^{20111} \\frac{1}{\\lg a_{i} \\lg a_{i+1}}$ has the value $\\qquad$.",
      "answer": "2011"
    },
    {
      "question": "A number was written on the board. At each step, we add to it the largest of its digits (for example, if we have written 142, the next number will be 146). What is the maximum number of odd numbers that can be written consecutively by proceeding in this way?\n\n## High school statements",
      "answer": "5"
    },
    {
      "question": "8. (10 points) Two small rulers form a set of rulers, and the small rulers can slide along the large ruler. Each unit on the large ruler is marked with a natural number. The first small ruler divides 11 units on the large ruler into 10, and the second small ruler divides 9 units on the large ruler into 10. The starting points of both small rulers are 0, and they are both marked from 1 to 10. Now, measure the distance between points A and B, where point A is at the 0 unit on the large ruler, and point B is between the 18th and 19th units on the large ruler. When the 0 unit of the first small ruler is placed at point B, its unit 3 coincides exactly with a unit on the large ruler. If the 0 unit of the second small ruler is placed at point B, then the \\_\\_\\_\\_\\_ unit of the second small ruler coincides exactly with a unit on the large ruler.",
      "answer": "7"
    },
    {
      "question": "The integers 390 and 9450 have three common positive divisors that are prime numbers. What is the sum of these prime numbers?",
      "answer": "10"
    },
    {
      "question": "11th USAMO 1982 Problem 1 A graph has 1982 points. Given any four points, there is at least one joined to the other three. What is the smallest number of points which are joined to 1981 points? Solution",
      "answer": "1979"
    },
    {
      "question": "## 226. Math Puzzle $3 / 84$\n\nSabine brings back 30 empty bottles, for which she receives $0.30 \\mathrm{M}$ refund per bottle.\n\nHow many bottles of soda costing $0.30 \\mathrm{M}$ each - for which she of course has to pay a deposit as well - could she buy again, and how much money, if any, would she have left over?",
      "answer": "29"
    },
    {
      "question": "7. Let $a, b, c$ be three distinct real numbers such that the equations $x^{2}+a x+1=0$ and $x^{2}+b x+c=0$ have a common real root, and the equations $x^{2}+x+a=0$ and $x^{2}+c x+b=0$ also have a common real root, then the value of $a+b+c$ is $\\qquad$.",
      "answer": "-3"
    },
    {
      "question": "Example 8. Find the Wronskian determinant for the functions: $y_{1}(x)=\\sin x$,\n\n$$\ny_{2}(x)=\\sin \\left(x+\\frac{\\pi}{8}\\right), y_{3}(x)=\\sin \\left(x-\\frac{\\pi}{8}\\right)\n$$",
      "answer": "0"
    },
    {
      "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a$ and $b$.\n\n$a=2 p+3 q$\n\n$b=p-2 q$\n\n$|p|=6$\n\n$|q|=7$\n\n$(\\widehat{p, q})=\\frac{\\pi}{3}$",
      "answer": "147\\sqrt{3}"
    },
    {
      "question": "A3. We call a positive integer alternating if the digits of the number alternate between even and odd. Thus, 2381 and 3218 are alternating, but 2318 is not. We call a number completely alternating if the number itself is alternating and double the number is also alternating. Thus, 505 is completely alternating, because both 505 and 1010 are alternating numbers.\n\nHow many completely alternating four-digit numbers are there?\n\nNote: a four-digit number may not start with a 0.",
      "answer": "70"
    },
    {
      "question": "Kondakov G.V.\n\nDivide the segment $[-1,1]$ into black and white segments so that the integrals of any a) linear function;\n\nb) quadratic trinomial over the white and black segments are equal.\n\n#",
      "answer": "6"
    },
    {
      "question": "8. Specify the greatest value of the parameter $p$ for which the equation has at least one solution\n\n$2 \\cos \\left(2 \\pi-\\frac{\\pi x^{2}}{6}\\right) \\cos \\left(\\frac{\\pi}{3} \\sqrt{9-x^{2}}\\right)-3=p-2 \\sin \\left(-\\frac{\\pi x^{2}}{6}\\right) \\cos \\left(\\frac{\\pi}{3} \\sqrt{9-x^{2}}\\right)$. (16 points)",
      "answer": "-2"
    },
    {
      "question": "G2.2 Let $b=89+899+8999+89999+899999$, find the value of $b$.",
      "answer": "999985"
    },
    {
      "question": "5. Write down the 0 times, 1 times, 2 times, 3 times, 4 times, 5 times, ... of the natural number 15 in order: $0, 15, 30, 45, 60, 75, \\ldots$\n\nThis sequence of numbers can go on indefinitely, and each subsequent number is always larger than the previous one. Any natural number is either one of these numbers or lies between two adjacent numbers. We call this sequence a strictly increasing infinite sequence. Each number from left to right is called the first term, the second term, the third term, ..., i.e., the first term is 0, the second term is 15, the third term is $30, \\ldots$, and so on. Therefore, the number that lies between the 135th term and the 136th term of this sequence, and whose difference with the smaller of these two terms is 6, is",
      "answer": "2016"
    },
    {
      "question": "6. Stopped Clock (from 6th grade. 1 point). On the clock, there are two hands: the hour hand and the minute hand. At a random moment in time, the clock stopped. Find the probability that the angle between the hands on the stopped clock is acute.",
      "answer": "0.5"
    },
    {
      "question": "## Task B-2.3.\n\nThe lengths of the sides of the hexagon $A B C D E F$ are in order $|A B|=b,|B C|=|C D|=a,|D E|=b$, $|E F|=|F A|=a, a \\neq b$. A circle with radius 8 and center at point $O$ is circumscribed around the hexagon. If $\\cos \\varangle E F O=\\frac{\\sqrt{2}}{4}$, calculate the perimeter and area of this hexagon.",
      "answer": "16\\sqrt{2}+24"
    },
    {
      "question": "5. Find the area of the cross-section of a regular hexagonal pyramid $S A B C D E F$ by a plane passing through the vertex $F$ of the base $A B C D E F$ and parallel to the median $C M$ of the lateral face $S C D$ and the apothem $S N$ of the lateral face $S A F$, if the side of the base of the pyramid is $4 \\sqrt{7}$, and the distance from the vertex $S$ to the cutting plane is $\\sqrt{7}$. (20 points)",
      "answer": "\\frac{202\\sqrt{3}}{5}"
    },
    {
      "question": "Problem 9. The function $f$ is defined on the set of triples of integers and takes real values. It is known that for any four integers $a, b, c$ and $n$ the equalities $f(n a, n b, n c)=n \\cdot f(a, b, c)$, $f(a+n, b+n, c+n)=f(a, b, c)+n$, $f(a, b, c)=f(c, b, a)$ hold. Find $f(24,25,26)$.",
      "answer": "25"
    },
    {
      "question": "G3.3 若方程 $\\sqrt[3]{5+\\sqrt{x}}+\\sqrt[3]{5-\\sqrt{x}}=1$, 求實數根 $x$ 。 Solve $\\sqrt[3]{5+\\sqrt{x}}+\\sqrt[3]{5-\\sqrt{x}}=1$ for real number $x$.",
      "answer": "52"
    },
    {
      "question": "9. Let $1<b<\\frac{5}{4}$, the number of distinct real roots of the equation $\\left|x^{2}-1\\right|=x+b$ is\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly. \n\n---\n\n9. Let $1<b<\\frac{5}{4}$, the number of distinct real roots of the equation $\\left|x^{2}-1\\right|=x+b$ is",
      "answer": "4"
    },
    {
      "question": "(4) If $4n+1, 6n+1$ are both perfect squares, then the smallest positive integer value of $n$ is\n$\\qquad$",
      "answer": "20"
    },
    {
      "question": "4. Let $I$ be the incenter of $\\triangle A B C$, and $3 \\overrightarrow{I A}+4 \\overrightarrow{I B}+5 \\overrightarrow{I C}=0$, then the size of angle $C$ is",
      "answer": "\\frac{\\pi}{2}"
    },
    {
      "question": "4. In a family, there are six children. Five of them are respectively 2, 6, 8, 12, and 14 years older than the youngest, and the age of each child is a prime number. How old is the youngest?",
      "answer": "5"
    },
    {
      "question": "Problem 3. (Option 2)\n\nFind the sum of the squares of the roots of the equation $\\left(x^{2}+6 x\\right)^{2}-1580\\left(x^{2}+6 x\\right)+1581=0$.",
      "answer": "3232"
    },
    {
      "question": "14. The function $f(x)$ defined on the interval $[1,2017]$ satisfies $f(1)=f(2017)$, and for any $x, y \\in [1,2017]$, it holds that $|f(x)-f(y)| \\leqslant 2|x-y|$. If the real number $m$ satisfies that for any $x, y \\in [1,2017]$, it holds that $|f(x)-f(y)| \\leqslant m$, find the minimum value of $m$.",
      "answer": "2016"
    },
    {
      "question": "# Task № 5.4\n\n## Condition:\n\nA Dog, a Cat, and a Mouse are running around a circular lake. They started simultaneously in the same direction from the same point and finished simultaneously, all running at constant speeds.\n\nThe Dog ran 12 laps, the Cat ran 5 laps, and the Mouse ran 2 laps. How many total overtakes were made from the start to the finish?\n\nIf two or more overtakes occur simultaneously, each overtake is counted separately. The start and finish moments are not counted as overtakes.",
      "answer": "17"
    },
    {
      "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{\\pi} 2^{4} \\cdot \\cos ^{8}\\left(\\frac{x}{2}\\right) d x\n$$",
      "answer": "\\frac{35\\pi}{8}"
    },
    {
      "question": "3. 11088 is $110\\%$ of the containers in December compared to November. This means that in November, $11088: 1.10=10080$ containers were manufactured, which is $105\\%$ compared to October. This means that in October, $10080: 1.05=9600$ containers were manufactured, which is $120\\%$ compared to September. Therefore, in September, $9600: 1.20=8000$ containers were manufactured.",
      "answer": "8000"
    },
    {
      "question": "A8. The sum of the numbers 1 to 123 is 7626 . One number is omitted so that the sum is now a multiple of 4 . How many different numbers could be omitted?",
      "answer": "31"
    },
    {
      "question": "Problem 11.2. In the store, there are 9 headphones, 13 computer mice, and 5 keyboards. In addition, there are 4 sets of \"keyboard and mouse\" and 5 sets of \"headphones and mouse\". In how many ways can you buy three items: headphones, a keyboard, and a mouse? Answer: 646.",
      "answer": "646"
    },
    {
      "question": "\\section*{Exercise 1 - 281021}\n\nThe smallest positive natural number is sought, whose digit representation (in the decimal system) consists only of the digits 0 and 1 and which is divisible by 450.",
      "answer": "11111111100"
    },
    {
      "question": "4 [ Excircle ]\n\nLet $A B C$ be an equilateral triangle. Three planes through the lines $A B, B C$, and $A C$ form an angle $\\phi$ with the plane $A B C$ and intersect at point $D 1$. In addition, three planes through the same lines form an angle $2 \\phi$ with the plane $A B C$ and intersect at point $D 2$. Find $\\phi$, given that points $D 1$ and $D 2$ are at equal distances from the plane $A B C$.",
      "answer": "30"
    },
    {
      "question": "7.98 Two people are playing a math game. Player A selects a set of 1-digit integers $x_{1}, x_{2}, \\cdots, x_{n}$ as the answer, which can be positive or negative. Player B can ask: what is the sum $a_{1} x_{1}+a_{2} x_{2}+\\cdots+a_{n} x_{n}$? Here, $a_{1}, a_{2}, \\cdots, a_{n}$ can be any set of numbers. Determine the minimum number of questions Player B needs to ask to guess the answer:",
      "answer": "1"
    },
    {
      "question": "Problem 10-5. Consider all reduced quadratic trinomials $x^{2}+p x+$ $q$ with integer coefficients $p$ and $q$. Let's call the range of such a trinomial the set of its values at all integer points $x=0, \\pm 1, \\pm 2, \\ldots$ What is the maximum number of such trinomials that can be chosen so that their ranges do not intersect pairwise?\n\nAnswer: 2.",
      "answer": "2"
    },
    {
      "question": "10. Determine the value of\n$$\n2002+\\frac{1}{2}\\left(2001+\\frac{1}{2}\\left(2000+\\cdots+\\frac{1}{2}\\left(3+\\frac{1}{2} \\cdot 2\\right)\\right) \\cdots\\right)\n$$",
      "answer": "4002"
    },
    {
      "question": "[ Numerical inequalities. Comparing numbers.]\n\nWhat is greater: $7^{92}$ or $8^{91}$ ?",
      "answer": "8^{91}"
    },
    {
      "question": "I3.4 If -2 and $216 c$ are the roots of the equation $p x^{2}+d x=1$, find the value of $d$.",
      "answer": "-\\frac{7}{18}"
    },
    {
      "question": "A3. What is the largest integer for which each pair of consecutive digits is a square?",
      "answer": "81649"
    },
    {
      "question": "6. On the table, there are three cones standing on their bases, touching each other. The radii of their bases are 1, 12, and 12, and the vertex angles are $-4 \\operatorname{arctg} \\frac{1}{3}, 4 \\operatorname{arctg} \\frac{2}{3}$, and $4 \\operatorname{arctg} \\frac{2}{3}$ respectively (the vertex angle of a cone is the angle between its generators in the axial section). A sphere is placed on the table, touching all the cones. Find the radius of the sphere.",
      "answer": "\\frac{40}{21}"
    },
    {
      "question": "B1. Mirko performs in the circus in the \"cannon launch\" act. His motion, which is shown in the figure, is described by the function $f$ with the rule\n\n$$\nf(x)=x-\\frac{1}{50} x^{2}\n$$\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_d6b556bd14f8001ca6fag-07.jpg?height=277&width=825&top_left_y=224&top_left_x=1092)\n\na) How far from the launch point should the center of the net be placed so that Mirko lands safely in it?\n\nb) What is the maximum height above the ground that Mirko reaches?",
      "answer": "50"
    },
    {
      "question": "Problem 2. In the class, two students sit at each desk. The number of desks where two boys sit is twice the number of desks where two girls sit. And the number of desks where two girls sit is twice the number of desks where a boy and a girl sit. How many boys are there in the class if there are 10 girls?\n\nAnswer: there are 18 boys in the class.",
      "answer": "18"
    },
    {
      "question": "8. Suppose there are 6 different elements of class A and 4 different elements of class B. An experiment requires an even number of elements from class A and at least 2 elements from class B. How many experimental schemes are there?\n\n保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "353"
    },
    {
      "question": "## SUBJECT 3\n\nLet the number $\\mathrm{n}=9+99+999+\\ldots+\\underbrace{99 \\ldots 9}_{\\text {2016 digits }}+2016$.\n\na) Show that the number n is divisible by 90.\n\nb) Find the quotient and remainder of the division of $\\mathrm{n}$ by 11111.",
      "answer": "10"
    },
    {
      "question": "G2.3 Determine the sum of all real roots of the following equation $|x+3|-|x-1|=x+1$.",
      "answer": "-3"
    },
    {
      "question": "13. [9] Suppose $\\triangle A B C$ has lengths $A B=5, B C=8$, and $C A=7$, and let $\\omega$ be the circumcircle of $\\triangle A B C$. Let $X$ be the second intersection of the external angle bisector of $\\angle B$ with $\\omega$, and let $Y$ be the foot of the perpendicular from $X$ to $B C$. Find the length of $Y C$.",
      "answer": "\\frac{13}{2}"
    },
    {
      "question": "11. (20 points) Let $x, y$ be complex numbers, find the maximum value of $\\frac{|3 x+4 y|}{\\sqrt{|x|^{2}+|y|^{2}+\\left|x^{2}+y^{2}\\right|}}$.",
      "answer": "\\frac{5\\sqrt{2}}{2}"
    },
    {
      "question": "## Task B-1.1.\n\nFor positive real numbers $a$ and $b$, define $a \\star b=\\frac{a-b}{a+b}$. What is\n\n$$\n\\frac{(a \\star b)+1}{(b \\star a)-1}+\\frac{(a \\star b)+2}{(b \\star a)-2}+\\ldots+\\frac{(a \\star b)+k}{(b \\star a)-k}+\\ldots+\\frac{(a \\star b)+2020}{(b \\star a)-2020}\n$$",
      "answer": "-2020"
    },
    {
      "question": "In the following figure, all the small squares on the board are equal. What percentage of the larger square does the painted region cover?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_e67c0440f1fa8bc62cdcg-21.jpg?height=219&width=217&top_left_y=1482&top_left_x=994)\n\n#",
      "answer": "18.75"
    },
    {
      "question": "3.184. Given the fraction $\\frac{5}{1+\\sqrt[3]{32 \\cos ^{4} 15^{\\circ}-10-8 \\sqrt{3}}}$. Simplify the expression under the cube root, and then reduce the fraction.",
      "answer": "1-\\sqrt[3]{4}+\\sqrt[3]{16}"
    },
    {
      "question": "## Task 5 - 290735\n\nWe consider the product of all natural numbers from 1 to 1000 inclusive.\n\nDetermine the number of zeros with which this product ends!",
      "answer": "249"
    },
    {
      "question": "## Task A-4.5.\n\nWhat is the probability that for two randomly chosen numbers $x, y$ from the set $[-2,2]$ the following holds\n\n$$\n|x|+|y| \\geqslant 1 \\quad \\text { and } \\quad \\| x|-| y|| \\leqslant 1 ?\n$$",
      "answer": "\\frac{5}{8}"
    },
    {
      "question": "Folklore\n\nA narrow ribbon is wound around a cylindrical column that is 20 meters high and 3 meters in diameter, rising from the base to the top in seven full turns. What is the length of the ribbon?",
      "answer": "\\sqrt{400+441\\pi^{2}}"
    },
    {
      "question": "The cities of Coco da Selva and Quixajuba are connected by a bus line. Buses leave Coco da Selva for Quixajuba every hour, with the first one departing at midnight on the dot. Buses leave Quixajuba for Coco da Selva every hour, with the first one departing at half past midnight. The bus journey takes exactly 5 hours.\n\nIf a bus leaves Coco da Selva at noon, how many buses coming from Quixajuba does it encounter during the journey?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d889f7c81233a28648eeg-1.jpg?height=226&width=317&top_left_y=1643&top_left_x=1506)\n\nFigure 21.2\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d889f7c81233a28648eeg-1.jpg?height=249&width=305&top_left_y=1943&top_left_x=1521)\n\nFigure 21.3\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d889f7c81233a28648eeg-1.jpg?height=220&width=214&top_left_y=2260&top_left_x=1566)\n\nFigure 21.4",
      "answer": "10"
    },
    {
      "question": "5. For any $n \\in \\mathbf{N}^{*}$, the parabola $y=\\left(n^{2}+n\\right) x^{2}-(2 n+1) x+1$ intersects the $x$-axis at points $A_{n}$ and $B_{n}$. Then $\\left|A_{1} B_{1}\\right|+\\left|A_{2} B_{2}\\right|+\\cdots+\\left|A_{2009} B_{2009}\\right|=$ $\\qquad$",
      "answer": "\\frac{2009}{2010}"
    },
    {
      "question": "9.5. On the side $AC$ of triangle $ABC$, a point $M$ is taken. It turns out that $AM = BM + MC$ and $\\angle BMA = \\angle MBC + \\angle BAC$. Find $\\angle BMA$.",
      "answer": "60"
    },
    {
      "question": "Let's find a point that is at distances $m, n, p$ from three planes. How many solutions does this problem have?",
      "answer": "8"
    },
    {
      "question": "3. Coaches Ivan and Marko are organizing a treasure hunt through two forest trails. Katja walked both trails, Ivan's and Marko's, and took a total of 2024 steps of the same length. If she had taken 204 fewer steps on Ivan's trail and 24 more steps on Marko's trail, she would have taken the same number of steps on both trails. How many steps did Katja take on Ivan's trail, and how many on Marko's trail?",
      "answer": "898"
    },
    {
      "question": "8. Arrange the numbers $1,2, \\cdots, 6$ in any order. If the number $k$ appears exactly in the $k$-th position, it is called a match. The expected number of matches is $\\qquad$",
      "answer": "1"
    },
    {
      "question": "Task B-4.3. Determine all numbers $z \\in \\mathbb{C}$ that are solutions to the equation $z^{3}+i^{2013}=0$. Calculate the area of the figure whose vertices in the Gaussian plane are determined by the solutions to this equation.",
      "answer": "\\frac{3\\sqrt{3}}{4}"
    },
    {
      "question": "7.2. There are nuts in three boxes. In the first box, there are six nuts fewer than in the other two boxes combined, and in the second box, there are ten fewer nuts than in the other two boxes combined. How many nuts are in the third box? Justify your answer.",
      "answer": "8"
    },
    {
      "question": "6. Let's call the distance between numbers the absolute value of their difference. It is known that the sum of the distances from sixteen consecutive natural numbers to some number $a$ is 636, and the sum of the distances from these same sixteen numbers to the number $a^{2}$ is 591. Find all possible values of $a$.",
      "answer": "-\\frac{5}{4}"
    },
    {
      "question": "12. Two squares $A B C D$ and $B E F G$ with side lengths of $8 \\mathrm{~cm}$ and $6 \\mathrm{~cm}$ respectively are placed side by side as shown in the figure. Connecting $D E$ intersects $B G$ at $P$. What is the area of the shaded region $A P E G$ in the figure?",
      "answer": "18"
    },
    {
      "question": "What are the last two digits of $9^{h}$, starting from the number $a=1$,\n\n$$\nb=2^{a}, \\quad c=3^{b}, \\quad d=4^{c}, \\quad e=5^{d}, \\quad f=6^{e}, \\quad g=7^{f}, \\quad h=8^{g} ?\n$$",
      "answer": "21"
    },
    {
      "question": "13. $[8]$ Find the smallest positive integer $n$ such that $\\frac{5^{n+1}+2^{n+1}}{5^{n}+2^{n}}>4.99$.",
      "answer": "7"
    },
    {
      "question": "10-2-1. Petya writes down a sequence of numbers: if the current number is equal to $x$, then the next one is $\\frac{1}{1-x}$. The first number in the sequence is 2. What is the five hundredth number?",
      "answer": "-1"
    },
    {
      "question": "Task 10.4. In the central cell of a $21 \\times 21$ board, there is a chip. In one move, the chip can be moved to an adjacent cell by side. Alina made 10 moves. How many cells can the chip end up in?",
      "answer": "121"
    },
    {
      "question": "3. Given that the radius of the circumcircle $\\odot O$ of square $ABCD$ is 10. Draw a semicircle outside the square with $AB$ as the diameter. Find the area of the region of the semicircle that lies outside $\\odot O$.",
      "answer": "50"
    },
    {
      "question": "3 $[\\quad$ Case Analysis $\\quad]$\n\nGiven five different positive numbers that can be divided into two groups such that the sums of the numbers in these groups are equal. In how many ways can this be done?\n\n#",
      "answer": "1"
    },
    {
      "question": "4- 120 If $\\operatorname{tg} x+\\operatorname{tg} y=25$, and $\\operatorname{ctg} x+\\operatorname{ctg} y=30$. Find $\\operatorname{tg}(x+y)$.",
      "answer": "150"
    },
    {
      "question": "Problem 3. Senya cannot write some letters and always makes mistakes in them. In the word TETRAHEDRON he would make five mistakes, in the word DODECAHEDRON - six, and in the word ICOSAHEDRON - seven. How many mistakes would he make in the word OCTAHEDRON?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_5f8bb630dfa91fa0065fg-2.jpg?height=41&width=471&top_left_y=799&top_left_x=802)",
      "answer": "5"
    },
    {
      "question": "3. Let $n$ be a natural number, for any real numbers $x, y, z$ there is always $\\left(x^{2}+y^{2}+z^{2}\\right) \\leqslant n\\left(x^{4}+y^{4}+z^{4}\\right)$, then the minimum value of $n$ is $\\qquad$",
      "answer": "3"
    },
    {
      "question": "4. $\\mathrm{ABCD}$ is a rhombus. Points $E$ and $F$ lie on sides $AB$ and $BC$ respectively, such that $\\frac{AE}{BE}=\\frac{BF}{CF}=5$. Triangle $DEF$ is equilateral. Find the angles of the rhombus. Solution.\n\nThe notation is shown in the figure. The situation where angle $D$ of the rhombus is acute is impossible (see below). Mark point $P$ on side $AB$ such that $AP=EB=FC$. Triangles $CDF$ and $ADP$ are congruent, as they have equal angles $\\angle A=\\angle C$ and corresponding equal sides $AD=DC, AP=FC$. Considering that triangle $FED$ is equilateral by the condition, we get that triangle $EDP$ is isosceles $(ED=FD=PD)$. (At this point, note that if angle $\\angle A$ of the rhombus is obtuse, then angle $\\angle APD$ is acute, and in the isosceles triangle $EDP$, the angle at the base is obtuse,\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_aa449fffebf3e1dc9820g-3.jpg?height=499&width=1037&top_left_y=116&top_left_x=521)\n\nwhich is impossible). Mark the equal angles $x=\\angle A=\\angle C$ and $y=\\angle ADP=\\angle CDF$. We find $\\angle EPD=x+y$ (the exterior angle of triangle $ADP$) and $\\angle PDE=180^{\\circ}-2(x+y)$. Summing all the angles at vertex $D$ of the rhombus ($\\left.\\angle D=180^{\\circ}-x\\right)$, we get the equation\n\n$$\ny+180^{\\circ}-2(x+y)+60^{\\circ}+y=180^{\\circ}-x\n$$\n\nFrom this, $x=60^{\\circ}$, the angles of the rhombus are $60^{\\circ}$ and $120^{\\circ}$.\n\nThe end of the solution can be different. After proving that triangle $EDP$ is isosceles, note that its exterior angles are equal: $\\angle APD=\\angle BED$. From this, triangles $ADP$ and $BED$ are congruent by the two indicated angles and sides $(AP=EB, PD=ED)$. Therefore, $BD=AD$, and triangle $ABD$ is equilateral. Thus, $\\angle BAD=60^{\\circ}$.",
      "answer": "60"
    },
    {
      "question": "$2+$ $[$ higher degree equations (miscellaneous). ]\n\nSolve the equation $\\left[x^{3}\\right]+\\left[x^{2}\\right]+[x]=\\{x\\}-1$.\n\n#",
      "answer": "-1"
    },
    {
      "question": "3. A store bought a batch of greeting cards at 21 cents each, and sold them for a total of 14.57 yuan. If each card was sold at the same price, and the selling price did not exceed twice the purchase price, then the store made a profit of $\\qquad$yuan.",
      "answer": "4.7"
    },
    {
      "question": "A triangle $A B C$ has an area equal to 944. Let $D$ be the midpoint of $[A B]$, $E$ the midpoint of $[B C]$, and $F$ the midpoint of $[A E]$. What is the area of $D E F$?",
      "answer": "118"
    },
    {
      "question": "3. From Nizhny Novgorod to Astrakhan, a steamboat takes 5 days, and on the return trip - 7 days. How long will it take for rafts to float from Nizhny Novgorod to Astrakhan?",
      "answer": "35"
    },
    {
      "question": "Let real numbers $a_{1}, a_{2}, \\cdots, a_{n} \\in[0,2]$, and define $a_{n+1}=a_{1}$, find the maximum value of $\\frac{\\sum_{i=1}^{n} a_{i}^{2} a_{i+1}+8 n}{\\sum_{i=1}^{n} a_{i}^{2}}$.",
      "answer": "4"
    },
    {
      "question": "Example 1-25 5 boys, 3 girls sit around a circular table. If there are no additional requirements, then $Q(8,8)=7!=5040$. If it is required that boy $B_{1}$ does not sit next to girl $G_{1}$, how many arrangements are there? If it is required that the 3 girls do not sit next to each other, how many arrangements are there?",
      "answer": "1440"
    },
    {
      "question": "An Olympic sequence is a sequence $s_{1}, s_{2}, \\ldots, s_{2023}$ where each of the 2023 terms is equal to 1 or -1. A slowly increasing sequence is a sequence of integers $t_{1}, t_{2}, \\ldots, t_{n}$ such that $1 \\leqslant t_{1}<t_{2}<\\cdots<t_{n} \\leqslant 2023$, and each of the $n-1$ differences $t_{i+1}-t_{i}$ is 1 or 2.\nFind the largest integer $C$ such that, for any Olympic sequence $s_{1}, s_{2}, \\ldots, s_{2023}$, there exists an integer $n \\geqslant 1$ and a slowly increasing sequence $t_{1}, t_{2}, \\ldots, t_{n}$ of length $n$ such that\n\n$$\n\\left|s_{t_{1}}+s_{t_{2}}+\\cdots+s_{t_{n}}\\right| \\geqslant C\n$$",
      "answer": "507"
    },
    {
      "question": "2. Let $\\triangle A B C$ be an obtuse isosceles triangle with base $\\overline{A B}$. On the base, there is a point $D$ such that $|\\angle A C D|=35^{\\circ}$. On the leg $\\overline{B C}$, there is a point $E$ such that $|C E|=|C D|$. Determine $|\\angle B D E|$.",
      "answer": "17.5"
    },
    {
      "question": "4. Find all natural numbers $n$ for which all numbers $n+1, n+3, n+7$, $n+9, n+13$ and $n+15$ are prime.",
      "answer": "4"
    },
    {
      "question": "4. We have\n\n$$\n\\begin{aligned}\n& a_{n}=\\prod_{k=2}^{n} \\frac{k^{2}}{k-1}=\\frac{2 \\cdot 2}{1} \\cdot \\frac{3 \\cdot 3}{2} \\cdot \\ldots \\ldots . \\cdot \\frac{n \\cdot n}{n-1}=\\frac{n!\\cdot n!}{(n-1)!}=n!\\cdot n=n!\\cdot[(n+1)-1]=(n+1)!-n! \\\\\n& b_{n}=\\sum_{k=2}^{n}[(k+1)!-k!]=(n+1)!-2, c_{n}=\\frac{(n+1)!-2}{(n+1)!}=1-\\frac{2}{(n+1)!} \\\\\n& d_{n}=\\left(1+\\frac{-2}{(n+1)!}\\right)^{(n+1)!}\n\\end{aligned}\n$$\n\n$$\n\\begin{gathered}\n\\text { We have } \\lim _{n \\rightarrow \\infty} c_{n}=\\lim _{n \\rightarrow \\infty}\\left(1+\\frac{-2}{(n+1)!}\\right)=1+0=1 \\text { and } \\lim _{n \\rightarrow \\infty} d_{n}=\\lim _{n \\rightarrow \\infty}\\left(1+\\frac{-2}{(n+1)!}\\right)^{(n+1)!}= \\\\\n=\\lim _{n \\rightarrow \\infty}\\left[\\left(1+\\frac{-2}{(n+1)!}\\right)^{\\frac{(n+1)!}{-2}}\\right]^{\\frac{-2}{(n+1)!}(n+1)!}=e^{-2}=\\frac{1}{e^{2}} \\quad \\text {.................................................................................... }\n\\end{gathered}\n$$\n\n## MATHEMATICS OLYMPIAD  Local stage, 16.02.2013  11th Grade\n\n## Problems:",
      "answer": "1"
    },
    {
      "question": "3. (8 points) Longteng Primary School Grade 5 has four classes. Class 5-1 has 42 students, Class 5-2 is $\\frac{6}{7}$ of Class 5-1, Class 5-3 is $\\frac{5}{6}$ of Class 5-2, and Class 5-4 is 1.2 times the number of students in Class 5-3. The total number of students in Grade 5 is.",
      "answer": "144"
    },
    {
      "question": "4. On each of the lines $y=0$ and $y=2$, there are 64 points with abscissas $1,2,3, \\ldots, 64$. In how many ways can three points be chosen from the marked 128 points so that they form the vertices of a right triangle?",
      "answer": "8420"
    },
    {
      "question": "## Task 3 - V00703\n\nThe tenth part of a number is increased by 3. The same value results when one hundredth of this number is decreased by 6!\n\nWhat is it?",
      "answer": "-100"
    },
    {
      "question": "Putnam 1992 Problem A2 Let the coefficient of x 1992 in the power series (1 + x) α = 1 + αx + ... be C(α). Find ∫ 0 1 C(-y-1) ∑ k=1 1992 1/(y+k) dy.",
      "answer": "1992"
    },
    {
      "question": "4. Determine all three-digit numbers with distinct digits that are divisible by 6 and whose sum of digits is divisible by 6. How many such numbers are there?",
      "answer": "54"
    },
    {
      "question": "## Task 15/73\n\nHow many different line segments are necessary to draw the pattern shown in the figure without tracing any segment more than once?\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_1d2be4837fb839c2b31dg-234.jpg?height=306&width=437&top_left_y=972&top_left_x=1252)",
      "answer": "5"
    },
    {
      "question": "10.7. For natural numbers $a>b>1$, define the sequence $x_{1}, x_{2}, \\ldots$ by the formula $x_{n}=\\frac{a^{n}-1}{b^{n}-1}$. Find the smallest $d$ such that this sequence does not contain $d$ consecutive terms that are prime numbers, for any $a$ and $b$.\n\n(V. Senderov)",
      "answer": "2"
    },
    {
      "question": "13. How many positive integers $x$, less than 10000, are there for which the difference $2^{x}-x^{2}$ is not divisible by 7?",
      "answer": "7142"
    },
    {
      "question": "Little Abel received as a gift a $2 \\times n$ board and $n$ tiles of size $2 \\times 1$. For example, the following figure shows the case where $n=10$, that is, when Abel has a $2 \\times 10$ board and 10 tiles of size $2 \\times 1$.\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-15.jpg?height=404&width=996&top_left_y=553&top_left_x=630)\n\nHe plays by filling the board using the $n$ tiles. For example, for $n=10$ Abel could fill it in the ways illustrated below:\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-15.jpg?height=580&width=1042&top_left_y=1144&top_left_x=608)\n\nHowever, observe that there are many other ways in which Abel can fill his board.\n\na) Calculate the total number of ways in which Abel can fill his board in the cases where $n=1, n=2$, and $n=3$, that is, in the cases where the boards have dimensions $2 \\times 1, 2 \\times 2$, and $2 \\times 3$.\n\nb) Let $a_{n}$ be the number of ways in which Abel can fill a $2 \\times n$ board using $n$ tiles of size $2 \\times 1$. Show that $a_{10}=a_{9}+a_{8}$.\n\nc) Calculate the total number of ways in which Abel can fill his board when $n=10$.",
      "answer": "89"
    },
    {
      "question": "3. (7 points) There are 6 blue, 7 red, and 9 white bulbs. In how many ways can they be arranged into a garland (using all the bulbs) so that no two white bulbs are adjacent?",
      "answer": "3435432"
    },
    {
      "question": "11.4. Solve the equation:\n\n$$\n\\begin{gathered}\n\\frac{10}{x+10}+\\frac{10 \\cdot 9}{(x+10)(x+9)}+\\frac{10 \\cdot 9 \\cdot 8}{(x+10)(x+9)(x+8)}+\\cdots+ \\\\\n+\\frac{10 \\cdot 9 \\ldots 2 \\cdot 1}{(x+10)(x+9) \\ldots(x+1)}=11\n\\end{gathered}\n$$",
      "answer": "-\\frac{1}{11}"
    },
    {
      "question": "40. Find a five-digit number that has the following property: when multiplied by 9, the result is a number composed of the same digits but in reverse order, i.e., the number is reversed.",
      "answer": "10989"
    },
    {
      "question": "1. The number n is the product of three (not necessarily distinct) prime numbers. If we increase each of them by 1, the product of the increased numbers will be 963 more than the original product. Determine the original number $n$.",
      "answer": "2013"
    },
    {
      "question": "Exercise 7. An integer $k>1$ is said to be superb if there exist $m, n$, and $a$, three strictly positive integers such that\n\n$$\n5^{\\mathrm{m}}+63 \\mathrm{n}+49=\\mathrm{a}^{\\mathrm{k}}\n$$\n\nDetermine the smallest superb integer.",
      "answer": "5"
    },
    {
      "question": "24. If $x, y$ and $z$ are real numbers such that $x+y+z=9$ and $x y+y z+z x=24$, find the largest possible value of $z$.",
      "answer": "5"
    },
    {
      "question": "13.402 The capacities of three vessels $A$, B, C, each of which has the shape of a cube, are in the ratio $1: 8: 27$, and the volumes of water poured into them are in the ratio 1:2:3. After transferring part of the water from vessel A to vessel B and from vessel B to vessel C, the water level in all three vessels became the same. Then, $128 \\frac{4}{7}$ liters of water were transferred from vessel C to vessel B, and after that, from vessel B to vessel A, so that the water depth in vessel A became twice that in vessel B. As a result, it turned out that there was now $10 \\theta$ liters less water in vessel A than there was initially. How much water was there initially in each vessel?",
      "answer": "500"
    },
    {
      "question": "12. As shown in the figure, with the midpoint $O$ of the line segment $A B$ of length $4 \\mathrm{~cm}$ as the center and a radius of $2 \\mathrm{~cm}$, draw a circle intersecting the perpendicular bisector of $A B$ at points $E$ and $F$. Then, with $B$ and $A$ as centers and a radius of $4 \\mathrm{~cm}$, draw circular arcs intersecting the ray $A E$ at point $C$ and the ray $B E$ at point $D$. Finally, with $E$ as the center and $D E$ as the radius, draw the circular arc $D C$. Find the area of the \"oval\" $A F B C D A$ formed by these 4 real curve arcs. (Use $\\pi$ for the value of pi, do not use an approximate value)",
      "answer": "(12-4\\sqrt{2})\\pi-4"
    },
    {
      "question": "2.110. $\\frac{\\sqrt{c-d}}{c^{2} \\sqrt{2 c}} \\cdot\\left(\\sqrt{\\frac{c-d}{c+d}}+\\sqrt{\\frac{c^{2}+c d}{c^{2}-c d}}\\right) ; \\quad c=2 ; d=1 / 4$.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Problem 3. In the school, there are 50 teachers, of whom 29 drink coffee, 28 drink tea, and 16 do not drink either coffee or tea. How many teachers drink only coffee, and how many drink only tea?",
      "answer": "6"
    },
    {
      "question": "A plane can intersect at most how many edges of a regular 10-sided prism?",
      "answer": "12"
    },
    {
      "question": "At the edge of a circular lake, there are stones numbered from 1 to 10, in a clockwise direction. Frog starts from stone 1 and jumps only on these 10 stones in a clockwise direction.\n\na) If Frog jumps 2 stones at a time, that is, from stone 1 to stone 3, from stone 3 to stone 5, and so on, on which stone will Frog be after 100 jumps?\n\nb) If on the first jump, Frog goes to stone 2, on the second jump to stone 4, on the third jump to stone 7, that is, in each jump he jumps one more stone than in the previous jump. On which stone will Frog be after 100 jumps?",
      "answer": "1"
    },
    {
      "question": "Example 3 (36th Canadian Mathematical Olympiad) Let $T$ be the set of all positive divisors of $20044^{100}$, and let $S$ be a subset of $T$ such that no element of $S$ is an integer multiple of any other element of $S$. Find the maximum possible number of elements in $S$.\n\nThe set $T$ consists of all positive divisors of $20044^{100}$. A subset $S$ of $T$ is such that no element of $S$ is an integer multiple of any other element of $S$. Determine the maximum possible number of elements in $S$.",
      "answer": "10201"
    },
    {
      "question": "Example 8 (40th IMO National Team Selection Test) For non-negative real numbers $x_{1}, x_{2}$, $\\cdots, x_{n}$ satisfying the condition $\\sum_{i=1}^{n} x_{i}=1$, find the maximum value of $\\sum_{i=1}^{n}\\left(x_{j}^{4}-x_{j}^{5}\\right)$.",
      "answer": "\\frac{1}{12}"
    },
    {
      "question": "Yatsenko I.V.\n\nVanya thought of a simple three-digit number, all digits of which are different.\n\nWhat digit can it end with if its last digit is equal to the sum of the first two?",
      "answer": "7"
    },
    {
      "question": "11.4. In a semicircle with a radius of 18 cm, a semicircle with a radius of 9 cm is constructed on one of the halves of the diameter, and a circle is inscribed, touching the larger semicircle from the inside, the smaller semicircle from the outside, and the second half of the diameter. Find the radius of this circle.\n\n#",
      "answer": "8"
    },
    {
      "question": "9.5. What is the maximum number of figures consisting of 4 1x1 squares, as shown in the diagram, that can be cut out from a $6 \\times 6$ table, if cutting can only be done along the grid lines?\n\n#",
      "answer": "8"
    },
    {
      "question": "We randomly select four points independently on the surface of a sphere. What is the probability that the tetrahedron defined by them contains the center of the sphere?",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "Seven dwarfs stood around the perimeter of their garden, one in each corner, and stretched a rope around the entire garden. Snow White came out from Smudla and walked along the rope. First, she walked four meters to the east, where she met Prof. From there, she continued two meters to the north until she reached Rejpal. From Rejpal, she walked west and after two meters, she met Stydlín. She then continued three meters to the north, until she reached Štístko. She set off to the west and after four meters, she met Kejchal, from where she had three meters to the south to Dřímal. Finally, she followed the rope to the shortest path back to Smudla, thus completing her walk around the garden.\n\nHow many square meters does the entire garden have?\n\n(M. Mach)\n\nHint. Draw the shape of the garden, preferably on graph paper.",
      "answer": "22"
    },
    {
      "question": "1. What is the minimum number of cells that need to be marked in a $20 \\times 20$ table so that each vertical or horizontal strip of $1 \\times 12$ contains at least one marked cell.",
      "answer": "32"
    },
    {
      "question": "5. Point $M$ divides the diagonal $A C$ of square $A B C D$ in the ratio $A M: M C=2: 1$. A line passing through point $M$ divides the square into two parts, the areas of which are in the ratio $9: 31$. In what ratio does this line divide the perimeter of the square?",
      "answer": "\\frac{27}{53}"
    },
    {
      "question": "5. (10 points) The product of two decimal numbers, when rounded to the nearest tenth, is 27.6. It is known that both decimals have one decimal place and their units digits are both 5. What is the exact product of these two decimals? $\\qquad$ .",
      "answer": "27.55"
    },
    {
      "question": "7.2. There are 11 kg of cereal. How can you measure out 1 kg of cereal using two weighings on a balance scale, if you have one 3 kg weight?",
      "answer": "1"
    },
    {
      "question": "The graph of the function $y=|x-1|+|x+1|$ and the line $y=c$ enclose a region with an area of 30. What is the value of the constant $c$?",
      "answer": "8"
    },
    {
      "question": "In the drawing to the side, triangle $ABC$ is equilateral and $BD = CE = AF = \\frac{AB}{3}$. The ratio $\\frac{EG}{GD}$ can be written in the form $\\frac{m}{n}, \\operatorname{gcd}(m, n)=1$. What is the value of $m+n$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_12f29789e10e6749ebb9g-37.jpg?height=460&width=516&top_left_y=1150&top_left_x=884)\n\n#",
      "answer": "5"
    },
    {
      "question": "12. Find the smallest natural number greater than 1 that is at least 600 times greater than each of its prime divisors.",
      "answer": "1944"
    },
    {
      "question": "10. If real numbers $a, b, c, d, e$ satisfy $a+b+c+d+e=8, a^{2}+b^{2}+c^{2}+d^{2}+e^{2}=16$, then the sum of the maximum and minimum values of $e$ is $\\qquad$",
      "answer": "\\frac{16}{5}"
    },
    {
      "question": "10th Australian 1989 Problem B3 Let N be the positive integers. The function f : N → N satisfies f(1) = 5, f( f(n) ) = 4n + 9 and f(2 n ) = 2 n+1 + 3 for all n. Find f(1789). Solution",
      "answer": "3581"
    },
    {
      "question": "Let $A B C$ be an isosceles triangle with $A C=B C$. The point $D$ lies on the side $A B$ such that the semicircle with diameter $[B D]$ and center $O$ is tangent to the side $A C$ in the point $P$ and intersects the side $B C$ at the point $Q$. The radius $O P$ intersects the chord $D Q$ at the point $E$ such that $5 \\cdot P E=3 \\cdot D E$. Find the ratio $\\frac{A B}{B C}$.",
      "answer": "\\frac{6}{5}"
    },
    {
      "question": "4. The four-digit number $\\overline{a b c d}$ is divisible by 3, and $a, b, c$ are permutations of three consecutive integers. Then the number of such four-digit numbers is\n$\\qquad$ .",
      "answer": "184"
    },
    {
      "question": "13. Sarah and Hagar play a game of darts. Let $O_{0}$ be a circle of radius 1. On the $n$th turn, the player whose turn it is throws a dart and hits a point $p_{n}$ randomly selected from the points of $O_{n-1}$. The player then draws the largest circle that is centered at $p_{n}$ and contained in $O_{n-1}$, and calls this circle $O_{n}$. The player then colors every point that is inside $O_{n-1}$ but not inside $O_{n}$ her color. Sarah goes first, and the two players alternate turns. Play continues indefinitely. If Sarah's color is red, and Hagar's color is blue, what is the expected value of the area of the set of points colored red?",
      "answer": "\\frac{6\\pi}{7}"
    },
    {
      "question": "5. In the number $2016 * * * * 02 *$, each of the 5 asterisks needs to be replaced with any of the digits $0,2,4,6,7,8$ (digits can be repeated) so that the resulting 11-digit number is divisible by 6. In how many ways can this be done?",
      "answer": "2160"
    },
    {
      "question": "1. Solve the equation: $(x-2016)(x-2017)(x-2018)=(x-2017)(x-2018)(x-2019)$.",
      "answer": "20172018"
    },
    {
      "question": "1B. The binomial coefficients of the second, third, and fourth terms in the expansion of the binomial $(1+x)^{n}$ form an arithmetic progression. Calculate $n$.",
      "answer": "7"
    },
    {
      "question": "96. Let $X$ be the number of points that fall when a die is rolled once. Find the variance of the random variable $X$.",
      "answer": "\\frac{35}{12}"
    },
    {
      "question": "$3 \\cdot 3$ Simplify the fraction $\\frac{x^{8}+x^{6}+x^{4}+x^{2}+1}{x^{4}+x^{3}+x^{2}+x+1}$, and calculate the value of the fraction when $x=-0.02$.",
      "answer": "\\frac{312499999}{306250000}"
    },
    {
      "question": "3. In triangle ABC, the median BM is drawn. It is known that $\\angle A B M=40^{\\circ}$, and $\\angle C B M=70^{\\circ}$. Find the ratio $A B: B M$.",
      "answer": "2"
    },
    {
      "question": "10. There are 4 identical squares. By joining them edge-to-edge without overlapping, they can form $\\qquad$ different plane figures. (Figures that can coincide after translation, rotation, or reflection are considered the same figure)",
      "answer": "5"
    },
    {
      "question": "G1.2 Let $a_{1}, a_{2}, a_{3}, \\ldots$ be an arithmetic sequence with common difference 1 and $a_{1}+a_{2}+a_{3}+\\ldots+a_{100}=2012$. If $P=a_{2}+a_{4}+a_{6}+\\ldots+a_{100}$, find the value of $P$.",
      "answer": "1031"
    },
    {
      "question": "Russian 1999 Problem 1 The digits of n strictly increase from left to right. Find the sum of the digits of 9n.",
      "answer": "9"
    },
    {
      "question": "564. $y=x^{2}+2$.\n\nTranslate the above text into English, keeping the original text's line breaks and format, and output the translation result directly.\n\n564. $y=x^{2}+2$.",
      "answer": "2"
    },
    {
      "question": "5. The complex numbers $z_{1}, z_{2}$ satisfy\n$$\n\\begin{array}{l}\n\\left|\\left(\\left|z_{1}\\right| z_{1}\\right)+\\left(\\left|z_{2}\\right| z_{2}\\right)\\right| \\\\\n=\\left|\\left(\\left|z_{1}\\right| z_{2}\\right)+\\left(z_{1}\\left|z_{2}\\right|\\right)\\right|,\n\\end{array}\n$$\n\nand $\\left|z_{1}-\\frac{1}{z_{2}}\\right|+\\left|z_{2}-\\frac{1}{z_{1}}\\right|$ has a maximum value and a minimum value whose sum is $4 \\sqrt{3}$. Then $\\left(\\left|z_{1} \\overline{z_{2}}\\right|+\\left|z_{2} \\overline{z_{1}}\\right|\\right)^{2016}=$",
      "answer": "6^{2016}"
    },
    {
      "question": "1. Ivan Ivanovich approached a source with two empty cans; one held 10 liters, and the other held 8 liters. Water from the source flowed in two streams - one stronger, the other weaker. Ivan Ivanovich simultaneously placed the cans under the streams and, when half of the smaller can was filled, he switched the cans. To Ivan Ivanovich's surprise, the cans filled up simultaneously. How many times more water does the stronger stream provide compared to the weaker one?",
      "answer": "2"
    },
    {
      "question": "3. Given a triangle $ABC$ with the largest side $BC$. The bisector of its angle $C$ intersects the altitudes $AA_{1}$ and $BB_{1}$ at points $P$ and $Q$ respectively, and the circumcircle of $ABC$ at point $L$. Find $\\angle ACB$, if it is known that $AP=LQ$.",
      "answer": "60"
    },
    {
      "question": "## Zadatak B-4.1.\n\nRiješite jednadžbu\n\n$$\n\\log _{2020} \\sqrt{x}+\\log _{2020} \\sqrt[4]{x}+\\log _{2020} \\sqrt[8]{x}+\\ldots+\\log _{2020} \\sqrt[2^{n}]{x}+\\ldots=\\frac{2}{\\log _{2} x}+\\log _{x} 505\n$$",
      "answer": "2020"
    },
    {
      "question": "66. In the winter basketball games, player Xiao Ming scored 23 points, 14 points, 11 points, and 20 points in the sixth, seventh, eighth, and ninth games, respectively. His average score in the first nine games is higher than his average score in the first five games. If his average score in the first ten games is higher than 18 points, then his score in the tenth game must be at least $\\qquad$ points.",
      "answer": "29"
    },
    {
      "question": "6. On the table, there are three cones standing on their bases, touching each other. The radii of their bases are 6, 24, and 24. A truncated cone is placed on the table with its smaller base down, and it shares a generatrix with each of the other cones. Find the radius of the smaller base of the truncated cone.",
      "answer": "2"
    },
    {
      "question": "10. The common difference of an arithmetic sequence with real terms is 4, and the square of the first term plus the sum of the remaining terms does not exceed 100. Such a sequence can have at most $\\qquad$ terms.",
      "answer": "8"
    },
    {
      "question": "10.419 A circle with its center on side $AB$ of triangle $ABC$ touches the other two sides of the triangle. Find the area of the circle if $a=13$ cm, $b=14$ cm, $c=15 \\mathbf{cm}$, where $a, b$ and $c$ are the lengths of the sides of the triangle.",
      "answer": "\\frac{3136}{81}\\pi"
    },
    {
      "question": "4. Let $f(t)=\\sqrt{1+t^{2}}-t$. Calculate the value of the expression\n\n$$\nf(x) f(y)+f(y) f(z)+f(z) f(x)\n$$\n\nif $x>0, y>0, z>0$ and $x y+y z+z x=1$.",
      "answer": "1"
    },
    {
      "question": "[ [ $\\left[\\begin{array}{l}\\text { Divisibility of numbers. General properties } \\\\ \\text { p }\\end{array}\\right]$\n\nFind such an $n$ that the number $10^{n}-1$ is divisible by a) 7 ; b) 13 ; c) 91 ; d) 819.",
      "answer": "6"
    },
    {
      "question": "## Task 1 - 090621\n\nKlaus participated in a track race as a member of the cycling section of a company sports club. After the race, his brother Reiner asked Klaus about the outcome of the race. Klaus said:\n\n\"When I crossed the finish line, no rider was next to me; exactly one third of the participating riders had already reached the finish line, and exactly half of all participants were still behind me.\"\n\nState which position Klaus finished in this race and how many riders in total participated in the race!",
      "answer": "3"
    },
    {
      "question": "Problem 4.5. Hooligan Dima made a construction in the shape of a $3 \\times 5$ rectangle using 38 wooden toothpicks. Then he simultaneously set fire to two adjacent corners of this rectangle, marked in the figure.\n\nIt is known that one toothpick burns for 10 seconds. How many seconds will it take for the entire construction to burn?\n\n(Fire spreads along the toothpicks at a constant speed. The fire continues to spread from each burned toothpick to all adjacent unburned toothpicks.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f1bf0225c3b69484645g-04.jpg?height=458&width=359&top_left_y=110&top_left_x=547)",
      "answer": "65"
    },
    {
      "question": "7. The equation $x^{2}-a|x|+a^{2}-3=0$ has a unique real solution for $x$, then $a=$ $\\qquad$ .",
      "answer": "-\\sqrt{3}"
    },
    {
      "question": "9. [6] Five guys each have a positive integer (the integers are not necessarily distinct). The greatest common divisor of any two guys' numbers is always more than 1 , but the greatest common divisor of all the numbers is 1 . What is the minimum possible value of the product of the numbers?",
      "answer": "32400"
    },
    {
      "question": "15 If $m, n \\in\\left\\{x \\mid x=a_{2} \\times 10^{2}+a_{1} \\times 10+a_{0}\\right\\}$, where $a_{i} \\in\\{1,2,3, 4,5,6,7\\}, i=0,1,2$, and $m+n=636$, then the number of distinct points $(m, n)$ representing points on the plane is $\\qquad$",
      "answer": "90"
    },
    {
      "question": "Find the root of the following equation, with a precision of 3 significant digits.\n\n$$\n(\\sqrt{5}-\\sqrt{2})(1+x)=(\\sqrt{6}-\\sqrt{3})(1-x)\n$$",
      "answer": "-0.0678"
    },
    {
      "question": "731. Calculate the approximate value:\n1) $1.08^{3.96}$;\n2) $\\frac{\\sin 1.49 \\cdot \\operatorname{arctg} 0.07}{2^{2.95}}$.",
      "answer": "0.01"
    },
    {
      "question": "It is known that the point symmetric to the center of the inscribed circle of triangle $ABC$ with respect to side $BC$ lies on the circumcircle of this triangle. Find angle $A$.\n\n#",
      "answer": "60"
    },
    {
      "question": "145. Three friends had a monkey. They bought a certain number of mango fruits and hid them. At night, one of the friends wanted to indulge, went to the storeroom, and wanted to take a third of the fruits, but it turned out that one fruit was left over, and he gave it to the monkey, and ate his share. After some time, the second one came and, not suspecting that the first one had already taken his share, divided the remainder into three equal parts, and again one fruit was left over, and he gave it to the monkey and ate his share. Finally, the third one came and also divided the remainder into three equal parts, and again one fruit was left over, which he gave to the monkey, and ate his share.\n\nThe next morning, all three came to the storeroom and divided the remaining fruits equally among themselves, and again one fruit was left over, which they gave to the monkey.\n\nDetermine the smallest number of fruits that satisfies the above conditions of distribution.\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_926f0bb262c8f9569516g-029.jpg?height=268&width=394&top_left_y=1491&top_left_x=837)\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_926f0bb262c8f9569516g-030.jpg?height=297&width=1307&top_left_y=248&top_left_x=383)\n\n## Problem from the tractate \"Bava Batra\".\n\nTalmud.",
      "answer": "79"
    },
    {
      "question": "390. A circle is divided into five equal arcs by five points. Find the measure of each of them.",
      "answer": "72"
    },
    {
      "question": "Given a sequence $\\left\\{a_{n}\\right\\}$ with all terms no less than 1, it satisfies:\n$$\na_{1}=1, \\quad a_{2}=1+\\frac{\\sqrt{2}}{2}, \\quad\\left(\\frac{a_{n}}{a_{n+1}-1}\\right)^{2}+\\left(\\frac{a_{n}-1}{a_{n-1}}\\right)^{2}=2 .\n$$\n\nTry to find:\n(1) The general term formula of the sequence $\\left\\{a_{n}\\right\\}$;\n(2) The value of $\\lim _{n \\rightarrow+\\infty} \\frac{a_{n}}{n}$.",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "25. In triangle $ABC$, the altitude $BH$ is drawn, point $O$ is the center of the circumscribed circle around it, and the length of its radius is $R$. Find the measure in radians of the largest of the angles $ACB$ and $BAC$, given that $R=(5 / 8) \\cdot B H=(5 / 6) \\cdot O H$. If necessary, round the found value to two decimal places.",
      "answer": "1.82"
    },
    {
      "question": "99. How many diagonals does a convex decagon have?",
      "answer": "35"
    },
    {
      "question": "And $c$ is a constant), the difference in eccentricity between $C_{1}$ and $C_{2}$ does not exceed 1, and $C_{2}$ has an asymptote with a slope no less than $\\frac{4}{3}$,\n$C_{1}$ and $C_{2}$ intersect the positive $x$-axis at points $A$ and $B$, respectively, and the intersection point of the two curves in the first quadrant is $D$. Try to find the maximum area of $\\triangle A B D$ and the equations of $C_{1}$ and $C_{2}$ at this time.",
      "answer": "\\frac{9\\sqrt{5}}{50}^{2}"
    },
    {
      "question": "18. If in a convex decagon no three diagonals intersect at the same point inside the decagon, how many segments can the diagonals be divided into by their intersection points inside the decagon?",
      "answer": "455"
    },
    {
      "question": "## Task 1 - 320731\n\na) Four red balls, two yellow balls, and one blue ball are to be distributed into two boxes $A$ and $B$ such that box $A$ contains three balls and box $B$ contains four balls.\n\nHow many such distributions are there in total?\n\nb) Now, balls of the same color are distinguished by an additional numbering. The distributions then differ not only in how many balls of each color are in boxes $A$ and $B$, but also in which numbers they carry.\n\nHow many such distributions are there in total?",
      "answer": "35"
    },
    {
      "question": "[ Arithmetic. Mental calculation, etc.]\n\nFor the book, 100 rubles were paid, and it remains to pay as much as it would remain to pay if it had been paid as much as it remains to pay. How much does the book cost?",
      "answer": "200"
    },
    {
      "question": "Ten points are marked around a circle, as illustrated in the figure.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_0308bd91a070e7a80796g-2.jpg?height=297&width=280&top_left_y=454&top_left_x=1068)\n\n(a) How many chords can be formed by connecting any two of these points? (A chord is a line segment connecting two points on a circle.)\n\n(b) How many triangles can be formed by connecting any three of these points?",
      "answer": "45"
    },
    {
      "question": "55 (978). There are 4 balls of different masses. Using a balance scale without weights, how many weighings are needed to arrange these balls in order of decreasing mass?",
      "answer": "5"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nIn the diagram, $\\triangle A B C$ is isosceles with $A B=B C$. If $\\angle B A C=t^{\\circ}$, what is the measure of $\\angle A B C$, in degrees?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_23a04f66c1167e793e90g-1.jpg?height=263&width=374&top_left_y=2053&top_left_x=903)",
      "answer": "76"
    },
    {
      "question": "Problem 8.3. In triangle $ABC$, the sides $AC=14$ and $AB=6$ are known. A circle with center $O$, constructed on side $AC$ as the diameter, intersects side $BC$ at point $K$. It turns out that $\\angle BAK = \\angle ACB$. Find the area of triangle $BOC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0973a8d23c1bf92cb27dg-24.jpg?height=378&width=422&top_left_y=1434&top_left_x=512)",
      "answer": "21"
    },
    {
      "question": "(1) The area of the inscribed square in the ellipse $\\frac{x^{2}}{5^{2}}+\\frac{y^{2}}{3^{2}}=1$ is",
      "answer": "\\frac{450}{17}"
    },
    {
      "question": "12. In the Tumba-Yumba tribe with a population of 30 people, a trader arrives. After studying the customs of the tribe, the trader proposes to play a game. For each natural exchange of goods conducted in the market by two tribespeople, the trader gives each participant one gold coin. If at the end of the day, two different tribespeople end up with the same number of coins, all goods and coins go to the trader. A tribesperson trades goods only with another known tribesperson, and the cunning trader knows this. By the end of the day, the trader finds a group of tribespeople with the same number of coins. The tribespeople are shocked, the trader is happy, and the offending tribespeople are expelled from the market. Then the Chief appears and offers the trader a rematch. Let the trader distribute 270 coins among the remaining tribespeople so that no one has the same number of coins (i.e., each tribesperson should have a unique number of coins). If the trader cannot do this, all the coins and goods remain in the tribe.\n\n- Is it true that in the original game, the tribespeople had no chance?\n- Is it true that the Chief can return all the goods and gold?\n- Is it true that the Chief knew how many tribespeople were expelled from the market?\n- What is the maximum number of tribespeople that needed to be expelled for the Chief to win?\n- Is it true that if the trader had lost the first game, the tribespeople could have lost the second game?\n\nExplain your answers.",
      "answer": "24"
    },
    {
      "question": "4. From the 12 edges and the 12 face diagonals of a cube, select $n$ lines such that any two of the selected line segments are skew lines. The maximum value of $n$ is $\\qquad$ .",
      "answer": "4"
    },
    {
      "question": "Each of the thirty sixth-graders has one pen, one pencil, and one ruler. After their participation in the Olympiad, it turned out that 26 students lost a pen, 23 - a ruler, and 21 - a pencil. Find the smallest possible number of sixth-graders who lost all three items.",
      "answer": "10"
    },
    {
      "question": "3.2 Two workers together produced 72 parts per shift. After the first worker increased their labor productivity by $15 \\%$, and the second by $25 \\%$, together they started producing 86 parts per shift. How many parts does each worker produce per shift after the increase in labor productivity?",
      "answer": "46"
    },
    {
      "question": "4. The function $f(x, y)$ satisfies for all non-negative integers $x, y$:\n(1) $f(0, y)=y+1$;\n(2) $f(x+1,0)=f(x, 1)$;\n(3) $f(x+1, y+1)=f[x, f(x+1, y)]$.\nDetermine $f(4,1981)$.\n(IMO -22 Problem)",
      "answer": "2^{(1984)}-3"
    },
    {
      "question": "【Question 3】As shown in the figure, points $C$, $E$, and $B$ are collinear, $C B \\perp A B$, $A E / / D C$, $A B=8$, $C E=5$, then the area of $\\triangle A E D$ is $\\qquad$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "20"
    },
    {
      "question": "## Task B-4.2.\n\nHow many complex numbers $z$ are there such that $|z|=1$ and the difference $z^{5!}-z^{4!}$ is a real number?",
      "answer": "240"
    },
    {
      "question": "[Area of a Triangle (using two sides and the angle between them)]\n\nA certain triangle can be cut out from a paper strip of unit width, but it cannot be cut out from any strip of lesser width. What area can this triangle have?\n\n#",
      "answer": "\\frac{1}{\\sqrt{3}}"
    },
    {
      "question": "2 [Rectangles and Squares. Properties and Characteristics]\n\nAuto: Dorricheno $C$.\n\nWhich are more numerous: rectangles with integer sides and a perimeter of 1996, or rectangles with integer sides and a perimeter of 1998?\n\n(Rectangles $a \\times b$ and $b \\times a$ are considered the same.)",
      "answer": "499"
    },
    {
      "question": "4. In a regular 2017-gon, all diagonals are drawn. Petya randomly selects some $\\mathrm{N}$ diagonals. What is the smallest $N$ such that among the selected diagonals, there are guaranteed to be two of the same length?",
      "answer": "1008"
    },
    {
      "question": "## 17. ILLUSTRATIVE EXERCISE\n\nA positive integer is called triangular if it can be written in the form $\\frac{n(n+1)}{2}$ for some positive integer $n$. How many pairs $(a, b)$ of triangular numbers are there such that $b-a=2007$? (It is recalled that 223 is a prime number).\n\nSOLUTION",
      "answer": "6"
    },
    {
      "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the parametric equations.\n\n$$\n\\begin{aligned}\n& \\left\\{\\begin{array}{l}\nx=2(t-\\sin t) \\\\\ny=2(1-\\cos t)\n\\end{array}\\right. \\\\\n& 0 \\leq t \\leq \\frac{\\pi}{2}\n\\end{aligned}\n$$",
      "answer": "4(2-\\sqrt{2})"
    },
    {
      "question": "1. Inside a cylinder with a base radius of 6, there are two spheres with radii also of 6, and the distance between their centers is 13. If a plane is made to be tangent to these two spheres and intersects the cylindrical surface to form an ellipse, then the sum of the lengths of the major and minor axes of this ellipse is $\\qquad$",
      "answer": "25"
    },
    {
      "question": "18 Let $S=1+\\frac{1}{\\sqrt{2}}+\\frac{1}{\\sqrt{3}}+\\cdots+\\frac{1}{\\sqrt{n}}$, to make $[S]=1994$, what should the value of $n$ be? ([S] represents the greatest integer not exceeding $S$).",
      "answer": "995006"
    },
    {
      "question": "Kite $A B C D$ is symmetric with respect to the diagonal $A C$. The length of $A C$ is $12 \\mathrm{~cm}$, the length of $B C$ is $6 \\mathrm{~cm}$, and the internal angle at vertex $B$ is a right angle. Points $E$, $F$ are given on sides $A B, A D$ such that triangle $E C F$ is equilateral.\n\nDetermine the length of segment $E F$.\n\n(K. Pazourek)",
      "answer": "4\\sqrt{3}"
    },
    {
      "question": "6-1. The physical education teacher lined up the class so that everyone was facing him. To the right of Kolya, there are 12 people, to the left of Sasha - 20 people, and to the right of him - 8 people. How many people are standing to the left of Kolya?",
      "answer": "16"
    },
    {
      "question": "I took a classic black and white chessboard, which was made up of $8 \\times 8$ square tiles with sides of $3 \\mathrm{~cm}$. I rearranged the tiles within the given frame so that one black rectangle, one black square, and one contiguous white shape were formed. The individual tiles still touched each other with their entire sides after rearrangement. The black shapes did not touch (not even at a corner), and each of them had at least one side in common with the edge of the chessboard.\n\nDetermine the maximum possible perimeter of the white shape and draw how it might look in such a case.\n\n(M. Mach)\n\nHint. What dimensions could the square have?",
      "answer": "168"
    },
    {
      "question": "One, Fill-in-the-blank Questions (8 points per question, total 64 points)\n1. Given the three sides of a triangle are consecutive natural numbers. If the largest angle is twice the smallest angle, then the perimeter of the triangle is $\\qquad$",
      "answer": "15"
    },
    {
      "question": "In an article about combinatorics, to determine the number of permutations, we essentially used an arrangement of notebooks that differs from the otherwise commonly used arrangement. In this arrangement, the rank of two notebooks is compared by sequentially checking how many lower-ranked elements are preceded by the second, third, fourth, etc., ranked elements. The notebook is of higher rank in which, proceeding in the above order, we first find an element that precedes more lower-ranked elements.\n\nFor example, comparing the groups 4231 and 3214: the 2 in both groups precedes one element, the 3 in the first group precedes one element, and in the second group, it precedes two elements, so in this special arrangement, the second group is of higher rank. (In the article, we saw that the situation is reversed in the usual arrangement.)\n\na) Write down all permutations of the elements 1, 2, 3, 4 in this special arrangement!\n\nb) What is the rank of the permutation 5341726 of the elements 1, 2, 3, 4, 5, 6, 7:\n\nI. in this special arrangement?\n\nII. in the usual arrangement?",
      "answer": "2271"
    },
    {
      "question": "5. Tangram is a familiar puzzle toy. Now, please use the cards provided to make a set of tangrams as shown in Figure 1, and take the four pieces numbered 1 to 4 to make a four-piece tangram.\n(1) Use the four pieces of the four-piece tangram to form the step shape shown in Figure 2. Draw the assembly method directly on Figure 2 with thick lines.\n(2) Figure 3 shows a four-level step of a cube, where the length, width, and height of each step are equal. It is known that the height $\\mathrm{AD}=h=3$ cm, the width $\\mathrm{DE}=b=1$ cm, and the length $\\mathrm{AC}=a=\\mathbf{8}$ cm. A clever mouse crawls from point A to point B along the surface of the steps (assuming it can crawl vertically on vertical surfaces), and it takes the shortest path. There is also a smart mousetrap that can be placed at any point on DE, FG, or HI. If it is placed on DE, then its movement path must be perpendicular to DE. Similarly, if it is placed on FG or HI, then its movement path must be perpendicular to FG or HI. It is known that the mouse and the smart mousetrap start at the same time, and the speed of the mouse $v=17$ cm/s. Prove that to catch the mouse exactly, the speed of the smart mousetrap is independent of its placement position, and find its speed.",
      "answer": "8"
    },
    {
      "question": "15. Given that $\\alpha$ is an acute angle, find the minimum value of $\\left(1+\\frac{1}{\\sin \\alpha}\\right)\\left(1+\\frac{1}{\\cos \\alpha}\\right)$.",
      "answer": "3+2\\sqrt{2}"
    },
    {
      "question": "## Task 1\n\nThree pioneers collected a total of $139 \\mathrm{~kg}$ of scrap. The first pioneer brought $36 \\mathrm{~kg}$ to the collection point, and the second brought twice as much.\n\nHow many kilograms of scrap did the third pioneer collect?",
      "answer": "31"
    },
    {
      "question": "6. Let the non-empty set $A \\subseteq\\{1,2,3,4,5,6,7\\}$, when $a \\in A$ it must also have $8-a \\in A$, there are $\\qquad$ such $A$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "15"
    },
    {
      "question": "16. (3 points) $A, B, C, D$ four boxes contain $8, 6, 3, 1$ balls respectively. The first child finds the box with the fewest balls, then takes one ball from each of the other boxes and puts it into this box; the second child also finds the box with the fewest balls, and then also takes one ball from each of the other boxes and puts it into this box, ..., after the 50th child has finished, the number of balls in box $A$ is $\\qquad$ .",
      "answer": "6"
    },
    {
      "question": "8. In how many different ways can integers $a, b, c \\in [1,100]$ be chosen so that the points with coordinates $A(-1, a), B(0, b)$, and $C(1, c)$ form a right triangle?",
      "answer": "974"
    },
    {
      "question": "6. (8 points) By expanding the expression $(1+\\sqrt{5})^{209}$ using the binomial theorem, we obtain terms of the form $C_{209}^{k}(\\sqrt{5})^{k}$. Find the value of $k$ for which such a term attains its maximum value.",
      "answer": "145"
    },
    {
      "question": "The rectangle $A B C D$ is divided into five equal rectangles. If the perimeter of $A B C D$ is $20 \\mathrm{~cm}$, determine its area.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_80f7f1185898b39e7876g-04.jpg?height=414&width=499&top_left_y=2032&top_left_x=687)\n\n#",
      "answer": "\\frac{3000}{121}^2"
    },
    {
      "question": "9. Half-planes $\\alpha_{1}, \\alpha_{2}, \\ldots, \\alpha_{n}$ have a common edge $l$. What is the sum of the dihedral angles $\\alpha_{1} \\widehat{l \\alpha_{2}}, \\alpha_{2} l \\alpha_{3}, \\ldots, \\alpha_{n-1} l \\alpha_{n}$, $\\widehat{\\alpha_{n} l} l \\alpha_{1}$, which together fill the entire space?",
      "answer": "2\\pi"
    },
    {
      "question": "I4.2 In Figure $1, \\triangle B A C$ is a right-angled triangle, $A B=A C=m \\mathrm{~cm}$. Suppose that the circle with diameter $A B$ intersects the line $B C$ at $D$, and the total area of the shaded region is $n \\mathrm{~cm}^{2}$. Find the value of $n$.",
      "answer": "\\frac{9}{4}"
    },
    {
      "question": "We call a phone number $d_{1} d_{2} d_{3}-d_{4} d_{5} d_{6} d_{7}$ cool if the number $d_{1} d_{2} d_{3}$ is equal to $d_{4} d_{5} d_{6}$ or to $d_{5} d_{6} d_{7}$. For example, $234-2347$ is a cool phone number. Assume that each $d_{i}$ can be any digit from 0 to 9. How many cool phone numbers exist? \n\n#",
      "answer": "19990"
    },
    {
      "question": "11. It is necessary to clearly follow the requirements for the volume of work if it is specified in the assignment.\n\n## Task 1 (Maximum 12 points)\n\nLet there be 5 banks in the country Alpha. The probability that a bank will close is the same for all banks and is equal to 0.05. Banks close independently of each other. At some point, a crisis occurred in the country Alpha, and the probability that a bank will close became 0.25 (it remains the same for all banks, and banks still close independently of each other). Find how much, in absolute terms, the probability that at least one bank will close has changed. Round your answer to the nearest hundredth.",
      "answer": "0.54"
    },
    {
      "question": "Problem 8.2. The numbers from 1 to 9 were placed in the cells of a $3 \\times 3$ table such that the sum of the numbers on one diagonal is 7, and on the other diagonal is 21. What is the sum of the numbers in the five shaded cells?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_adefdf2067937cb67fc7g-28.jpg?height=416&width=428&top_left_y=105&top_left_x=513)",
      "answer": "25"
    },
    {
      "question": "5. Let $a^{b}=\\frac{1}{8}$. What is the value of $a^{-3 b}$ ?",
      "answer": "512"
    },
    {
      "question": "B2 This month, I spent 26 days exercising for 20 minutes or more, 24 days exercising 40 minutes or more, and 4 days of exercising 2 hours exactly. I never exercise for less than 20 minutes or for more than 2 hours.\nWhat is the minimum number of hours I could have exercised this month?",
      "answer": "22"
    },
    {
      "question": "1. Given the complex number $z$ satisfies $\\frac{z-\\mathrm{i}}{z-1}$ is a pure imaginary number (where $\\mathrm{i}$ is the imaginary unit), then the maximum value of $|z|$ is",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "## Task B-4.3.\n\nA circle with center at point $S$ is inscribed in triangle $A B C$. The circle touches the sides of the triangle at points $D \\in \\overline{B C}, E \\in \\overline{A C}$ and $F \\in \\overline{A B}$. If $|A E|=3 \\mathrm{~cm},|C S|=2 \\sqrt{7} \\mathrm{~cm}$, and the measure of the angle at vertex $A$ is $60^{\\circ}$, calculate the area of triangle $A B C$.",
      "answer": "10\\sqrt{3}^2"
    },
    {
      "question": "81. Select several numbers from $1,2,3,4,5,6$ so that the sum of the selected numbers is a multiple of 3 but not a multiple of 5. There are $\\qquad$ different ways to do this.",
      "answer": "19"
    },
    {
      "question": "2. Determine the largest value of $x$ for which\n$$\n\\left|x^{2}-4 x-39601\\right| \\geq\\left|x^{2}+4 x-39601\\right|\n$$",
      "answer": "199"
    },
    {
      "question": "2. Laura has 2010 lamps and 2010 switches in front of her, with different switches controlling different lamps. She wants to find the correspondence between the switches and the lamps. For this, Charlie operates the switches. Each time Charlie presses some switches, and the number of lamps that light up is the same as the number of switches pressed (Note: The switches return to their original state after each operation). Questions:\n(1) What is the maximum number of different operations Charlie can perform so that Laura can accurately determine the correspondence between the switches and the lamps?\n(2) If Laura operates the switches herself, what is the minimum number of operations she needs to find the correspondence between the switches and the lamps?",
      "answer": "11"
    },
    {
      "question": "1. (5 points) Find the value of $n$ for which the following equality holds:\n\n$$\n\\frac{1}{1+\\sqrt{2}}+\\frac{1}{\\sqrt{2}+\\sqrt{3}}+\\frac{1}{\\sqrt{3}+\\sqrt{4}}+\\ldots+\\frac{1}{\\sqrt{n}+\\sqrt{n+1}}=2011\n$$",
      "answer": "4048143"
    },
    {
      "question": "7. (36th Austrian Mathematical Olympiad) Let $a$ be an integer, and $|a| \\leqslant 2005$. Find the number of $a$ such that the system of equations $\\left\\{\\begin{array}{l}x^{2}=y+a, \\\\ y^{2}=x+a\\end{array}\\right.$ has integer solutions.",
      "answer": "90"
    },
    {
      "question": "IS. $3 W$ and $F$ are two integers which are greater than 20. If the product of $W$ and $F$ is $b$ and the sum of $W$ and $F$ is $c$, find the value of $c$.",
      "answer": "57"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$$\n\\lim _{n \\rightarrow \\infty} \\frac{1-2+3-4+\\ldots+(2 n-1)-2 n}{\\sqrt[3]{n^{3}+2 n+2}}\n$$",
      "answer": "-1"
    },
    {
      "question": "Can $n!$ end with exactly five zeros?\n\n#",
      "answer": "24"
    },
    {
      "question": "1. The picture shows several circles connected by segments. Sasha chooses a natural number \\( n \\) and places different natural numbers in the circles so that for all these numbers the following property holds:\n\nif numbers \\( a \\) and \\( b \\) are not connected by a segment, then the sum \\( a + b \\) must be coprime with \\( n \\), while if they are connected, then the numbers \\( a + b \\) and \\( n \\) must have a common natural divisor greater than 1. For what smallest \\( n \\) does such an arrangement exist?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0b0e793e58d3be992fedg-01.jpg?height=274&width=277&top_left_y=800&top_left_x=907)",
      "answer": "35"
    },
    {
      "question": "8. On a birthday card printed with April 29, a child added two positive integers $x$ and $y$ between 4 and 29, resulting in a five-digit number $\\overline{4 x y 29}$. It turns out that this number is exactly the square of the positive integer $T$ corresponding to their birthday: $\\overline{4 x y 29}=T^{2}$. Then the number $T$ corresponding to the child's birthday is $T=$ $\\qquad$",
      "answer": "223"
    },
    {
      "question": "L OM - I - Task 5\n\nFind all pairs of positive integers $ x $, $ y $ satisfying the equation $ y^x = x^{50} $.",
      "answer": "8"
    },
    {
      "question": "13. There are two bags, $A$ and $B$. Bag $A$ contains 6 cards, one of which is labeled 0, two are labeled 1, and three are labeled 2; Bag $B$ contains 7 cards, four of which are labeled 0, one is labeled 1, and two are labeled 2. One card is drawn from bag $A$ and two cards are drawn from bag $B$, making a total of three cards. Find\n(1) the probability that all three cards drawn are 0;\n(2) the probability that the product of the numbers on the three cards is 4.",
      "answer": "\\frac{4}{63}"
    },
    {
      "question": "4. (8 points) There are four people, A, B, C, and D. B owes A 1 yuan, C owes B 2 yuan, D owes C 3 yuan, and A owes D 4 yuan. To settle all debts among them, A needs to pay $\\qquad$ yuan.\n\n保留源文本的换行和格式，翻译结果如下：\n\n4. (8 points) There are four people, A, B, C, and D. B owes A 1 yuan, C owes B 2 yuan, D owes C 3 yuan, and A owes D 4 yuan. To settle all debts among them, A needs to pay $\\qquad$ yuan.",
      "answer": "3"
    },
    {
      "question": "Problem 5. If Mare buys 20 eggs from the market, 30 denars will remain from the amount she brought, and to buy 30 eggs, 20 denars are missing. How many denars did Mare bring to the market?",
      "answer": "130"
    },
    {
      "question": "## Task 1 - 080811\n\nThe figure shows a five-pointed star, where the points $A, B, C, D, E$ are the vertices of a regular pentagon.\n\nDetermine the size of the angle $\\angle A C E$!\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0621.jpg?height=354&width=352&top_left_y=448&top_left_x=1366)",
      "answer": "36"
    },
    {
      "question": "10.348. A convex quadrilateral is divided by its diagonals into four triangles; the areas of three of them are 10, 20, and $30 \\mathrm{~cm}^{2}$, and each is less than the area of the fourth triangle. Find the area of the given quadrilateral.",
      "answer": "120^2"
    },
    {
      "question": "4. Let $x \\in \\mathbf{R}$, then the minimum value of the function $f(x)=\\sqrt{x^{2}+1}+\\sqrt{(x-12)^{2}+16}$ is",
      "answer": "13"
    },
    {
      "question": "7. Let the pair of positive integers $(x, y)$ satisfy $\\frac{1}{\\sqrt{x}}+\\frac{1}{\\sqrt{y}}=\\frac{1}{\\sqrt{20}}$. Then $xy$ has $\\qquad$ different possible values.",
      "answer": "2"
    },
    {
      "question": "11-8. In a caravan, there are 100 camels, both one-humped and two-humped, and there are at least one of each. If you take any 62 camels, they will have at least half of the total number of humps in the caravan. Let $N$ be the number of two-humped camels. How many values (in the range from 1 to 99) can $N$ take?",
      "answer": "72"
    },
    {
      "question": "## Task 24/84\n\nDetermine all four-digit natural numbers n (in the decimal system) that satisfy the following conditions:\n\n1. The cross sum is 23.\n2. The alternating cross sum is -5.\n3. The cross product is 360.\n4. The sum of the digits in the first and third positions is equal to the digit in the second position.",
      "answer": "19858915"
    },
    {
      "question": "20. (6 points) A group of monkeys needs to transport peaches from location $A$ to location $B$. Every 3 minutes, a monkey starts from $A$ and heads to $B$, and the entire journey takes 12 minutes. A rabbit starts running from $B$ to $A$ at the same time a monkey arrives at $B$. On its way, the rabbit encounters 5 monkeys coming towards it, and continues to $A$, where it meets another monkey just starting from $A$. If the rabbit's running speed is 3 kilometers/hour, then the distance between $A$ and $B$ is $\\qquad$",
      "answer": "300"
    },
    {
      "question": "In a table tennis tournament, $n$ Asian and $2 n$ European players participated. The number of matches won by the Europeans is 5/7 of the number of matches won by the Asians. How many players participated in the tournament?",
      "answer": "9"
    },
    {
      "question": "## Task A-1.2.\n\nLet $a$ and $b$ be positive real numbers such that\n\n$$\n\\frac{a}{b}+\\frac{b}{a}=3 \\quad \\text { and } \\quad \\frac{a^{2}}{b}+\\frac{b^{2}}{a}=10\n$$\n\nDetermine $\\frac{1}{a}+\\frac{1}{b}$.",
      "answer": "1"
    },
    {
      "question": "5. (3 points) Three circles with radii 1, 1, and \\(2 \\sqrt{\\frac{13-6 \\sqrt{3}}{13}}\\) are arranged such that the triangle formed by their centers is equilateral with a side length of \\(\\sqrt{3}\\). Find the radius of the circumcircle of the triangle, each vertex of which is the point of intersection of two of these circles, farthest from the center of the third circle.\n\nAnswer: \\(4 \\sqrt{3}-6\\)",
      "answer": "4\\sqrt{3}-6"
    },
    {
      "question": "6. Points $E$ and $F$ are the midpoints of sides $\\overline{B C}$ and $\\overline{C D}$ of rectangle $A B C D$. What is the area of triangle $A E F$ if the area of rectangle $44 \\mathrm{~cm}^{2}$?",
      "answer": "16.5^2"
    },
    {
      "question": "9. (16 points) Let\n$$\n\\begin{aligned}\nP(z)= & z^{4}-(6 \\mathrm{i}+6) z^{3}+24 \\mathrm{i}^{2}- \\\\\n& (18 \\mathrm{i}-18) z-13 .\n\\end{aligned}\n$$\n\nFind the area of the convex quadrilateral formed by the four points in the complex plane corresponding to the four roots of $P(z)=0$.",
      "answer": "2"
    },
    {
      "question": "7. The dog is chasing a rabbit that is 90 of its own jumps ahead of it. While the rabbit jumps 6 times, (20) the dog jumps only 5 times, and two dog jumps are as long as three rabbit jumps. How many jumps will the rabbit make until the dog catches it?",
      "answer": "360"
    },
    {
      "question": "## Task 2 - 090612\n\nDuring the holidays, Klaus was in the countryside. From his observations, the following joke problem emerged:\n\n$$\n1 \\frac{1}{2} \\text { chickens lay in } 1 \\frac{1}{2} \\text { days } 1 \\frac{1}{2} \\text { eggs }\n$$\n\nDetermine the total number of eggs that 7 chickens would lay in 6 days at the same laying rate!",
      "answer": "28"
    },
    {
      "question": "7、A book's page numbers are consecutive natural numbers, $1,2,3, \\cdots$, when these page numbers are added together, one page number is added twice, resulting in the incorrect sum of 1997, then the page number that was added twice is $\\qquad$",
      "answer": "44"
    },
    {
      "question": "Question 9: Given that $f(x)$ is an odd function defined on $R$, and when $x<0$, $f(x)$ is monotonically increasing, $f(-1)=0$. Let $\\varphi(x)=\\sin ^{2} x+m \\cdot \\cos x-2 m$, set $M=\\left\\{m \\mid\\right.$ for $\\left.\\forall x \\in\\left[0, \\frac{\\pi}{2}\\right], \\varphi(x)<0\\right\\}, N=\\{m \\mid$ for $\\left.\\forall \\mathrm{x} \\in\\left[0, \\frac{\\pi}{2}\\right], \\mathrm{f}(\\varphi(\\mathrm{x}))<0\\right\\}$, find $\\mathrm{M} \\cap \\mathrm{N}$.",
      "answer": "4-2\\sqrt{2}"
    },
    {
      "question": "3. In a cube with edge length 1, construct a large inscribed sphere $O_{1}$, and then construct a smaller sphere $O_{2}$ inside the cube, such that it is externally tangent to the larger sphere and also tangent to three faces of the cube. Then the surface area of sphere $O_{2}$ is $\\qquad$ .",
      "answer": "(7-4\\sqrt{3})\\pi"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nGray has $t$ dollars consisting of $\\$ 1$ and $\\$ 2$ coins. If she has the same number of $\\$ 1$ and $\\$ 2$ coins, how many $\\$ 1$ coins does she have?\n\n##",
      "answer": "4"
    },
    {
      "question": "B1. Every three-digit number can be written in reverse. 176 can be reversed to 671.\n\nDetermine all three-digit numbers that are equal to 3 times their reversed plus the sum of their digits.\n\nIn formula: $\\overline{d c}=3 \\times \\overline{d a}+(a+b+c)$",
      "answer": "441882"
    },
    {
      "question": "15th Putnam 1955 Problem A6 For what positive integers n does the polynomial p(x) ≡ x n + (2 + x) n + (2 - x) n have a rational root. Solution",
      "answer": "1"
    },
    {
      "question": "9. The base of the pyramid $T A B C D$ is an isosceles trapezoid $A B C D$, the length of the smaller base $B C$ of which is $\\sqrt{3}$. The ratio of the areas of the parts of the trapezoid $A B C D$, into which it is divided by the midline, is $5: 7$. All lateral faces of the pyramid $T A B C D$ are inclined to the base plane at an angle of $30^{\\circ}$. The plane $A K N$, where points $K$ and $N$ are the midpoints of edges $T B$ and $T C$ respectively, divides the pyramid into two parts. Find the volume of the larger part.\n\n$(16$ points)",
      "answer": "0.875"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 1} \\frac{x^{2}-1}{\\ln x}$",
      "answer": "2"
    },
    {
      "question": "Sure, here is the translated text:\n\n```\nI. Fill-in-the-Blanks (8 points per question, total 64 points)\n1. Given the sets\n$$\nA=\\{x \\mid 5 x-a \\leqslant 0\\}, B=\\{x \\mid 6 x-b>0\\}\n$$\n\nwhere $a, b \\in \\mathbf{N}_{+}$.\nIf $A \\cap B \\cap \\mathbf{N}=\\{2,3,4\\}$, then the number of integer pairs $(a, b)$ is . $\\qquad$\n```",
      "answer": "30"
    },
    {
      "question": "## Task 1 - 030511\n\nThe VEB Simson Suhl currently produces 235 microcars KR 50 daily. In 1958, production was only 42 microcars per day.\n\nHow many more microcars were produced in 1963 compared to 1958? Assume the number of working days in a year is 300.",
      "answer": "57900"
    },
    {
      "question": "11. (20 points) As shown in Figure 1, for the ellipse $\\Gamma: \\frac{x^{2}}{4}+y^{2}=1$, two chords $A B$ and $C D$ satisfy $\\overrightarrow{D C}=2 \\overrightarrow{A B}$. Let the intersection of lines $A D$ and $B C$ be point $P$.\n(1) Find the maximum value of $|A B|+|C D|$; 360227734\n(2) If point $P$ lies on the parabola $y=\\frac{1}{4} x^{2}-3$, find the maximum area of quadrilateral $A B C D$.",
      "answer": "\\frac{(3\\sqrt{13}-6)\\sqrt{2+2\\sqrt{13}}}{4}"
    },
    {
      "question": "## Task 3 - 341213\n\nThe circles in the figure can be filled with real numbers such that the given numbers appear in the outer circles and that in each of the seven inner circles, the arithmetic mean of the six adjacent circles is placed.\n\nDetermine which number $x$ stands in the central circle.\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-2349.jpg?height=409&width=397&top_left_y=2257&top_left_x=1315)",
      "answer": "\\frac{20}{3}"
    },
    {
      "question": "## 37. International Conference\n\nFour French, two representatives from the Republic of Ivory Coast, three English and four Swedes gathered at a conference to discuss forestry issues. The conference participants from the same country initially sat next to each other on a bench with 13 seats during the opening ceremony, and later, during the working sessions, at a round table with 13 seats.\n\nIn how many different ways can the conference participants sit next to each other in each of these two cases?",
      "answer": "165888"
    },
    {
      "question": "4. A table tennis tournament proceeds according to the following rule. In each round, if the number of participants is odd, one is randomly selected to automatically advance to the next round. The remaining participants $\\mathrm{z}$ are paired by random draw. The players of each pair compete against each other, and the winner of each pair advances to the next round. Let $f(n)$ denote the number of rounds in a tournament with $n$ participants. (For example, $f(5)=3$.) Determine $f(2013)$ and find the smallest natural number $n$ for which $f(n)=f(2013)$.\n\n## 57th Mathematical Competition for High School Students in Slovenia\n\nCelje, April 20, 2013\n\n## Solutions to Problems and Scoring\n\n(April 21, 2013, $14: 53$ )",
      "answer": "1025"
    },
    {
      "question": "3. A balloon that blows up in the shape of a perfect cube is being blown up at a rate such that at time $t$ fortnights, it has surface area $6 t$ square furlongs. At how many cubic furlongs per fortnight is the air being pumped in when the surface area is 144 square furlongs?",
      "answer": "3\\sqrt{6}"
    },
    {
      "question": "7. Determine all natural numbers of the form $\\overline{a b c d e f}$ such that in their representation they have the digits 1, 2, 3, 4, 5, and 6, none of which repeat, and the following conditions are met:\n\n$\\overline{a b}$ is divisible by 2,\n\n$\\overline{a b c}$ is divisible by 3,\n\n$\\overline{a b c d}$ is divisible by 4,\n\n$\\overline{a b c d e}$ is divisible by 5,\n\n$\\overline{a b c d e f}$ is divisible by 6.\n\n## Ministry of Science and Education of the Republic of Croatia  Agency for Education and Upbringing  Croatian Mathematical Society\n\n## SCHOOL/CITY COMPETITION IN MATHEMATICS  January 26, 2023.  8th grade - elementary school\n\nIn addition to the final result, the procedure is also graded. To receive full points, it is necessary to find all solutions and confirm that there are no others, write down the procedure, and justify your conclusions. Problems 1. - 5. are worth six points each, and 6. and 7. are worth ten points each.\n\n## Problems worth 6 points:",
      "answer": "123654321654"
    },
    {
      "question": "4. Let the edge length of the regular tetrahedron $ABCD$ be 2. Then the sum of the areas of all sections of the circumsphere of this tetrahedron by planes that are equidistant from the four vertices of the tetrahedron is $\\qquad$",
      "answer": "\\frac{59\\pi}{6}"
    },
    {
      "question": "8. Given the hyperbola $H: x^{2}-y^{2}=1$ and a point $M$ in the first quadrant on $H$, the line $l$ is tangent to $H$ at point $M$ and intersects the asymptotes of $H$ at points $P, Q$ (with $P$ in the first quadrant). If point $R$ is on the same asymptote as $Q$, then the minimum value of $\\overrightarrow{R P} \\cdot \\overrightarrow{R Q}$ is $\\qquad$",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "G5 Inside the square $A B C D$, the equilateral triangle $\\triangle A B E$ is constructed. Let $M$ be an interior point of the triangle $\\triangle A B E$ such that $M B=\\sqrt{2}, M C=\\sqrt{6}, M D=\\sqrt{5}$ and $M E=\\sqrt{3}$. Find the area of the square $A B C D$.",
      "answer": "3+\\sqrt{6}"
    },
    {
      "question": "Task B-4.4. Determine all values of the real parameter $p$ for which the equation $x^{4}-(3 p+2) x^{2}+p^{2}=0$ has four solutions that form four consecutive terms of an arithmetic sequence.",
      "answer": "6"
    },
    {
      "question": "What is the probability that two interns were born on the same day (not necessarily in the same year)? Note: There are 62 interns.",
      "answer": "99.59095749"
    },
    {
      "question": "11.213 Calculate the volume of a triangular pyramid, in which two opposite edges are 4 and 12 cm, and each of the other edges is 7 cm.",
      "answer": "24^3"
    },
    {
      "question": "5. If four dice are thrown at random, the probability that the smallest number shown by these four dice is exactly 3 is\n$\\qquad$ (Supplied by Zhao Bin)",
      "answer": "\\frac{175}{1296}"
    },
    {
      "question": "I4.4 In Figure 2, there are a square $A B C D$ with side length $(R-1) \\mathrm{cm}$ and an equilateral triangle $A E F$. ( $E$ and $F$ are points on $B C$ and $C D$ respectively). If the area of $\\triangle A E F$ is $(S-3) \\mathrm{cm}^{2}$, find the value of $S$.",
      "answer": "2\\sqrt{3}"
    },
    {
      "question": "19. Let $a, b, c, d$ be real numbers such that\n$$\n\\left\\{\\begin{array}{l}\na^{2}+b^{2}+2 a-4 b+4=0 \\\\\nc^{2}+d^{2}-4 c+4 d+4=0 .\n\\end{array}\\right.\n$$\n\nLet $m$ and $M$ be the minimum and the maximum values of $(a-c)^{2}+(b-d)^{2}$, respectively. What is $m \\times M$ ?",
      "answer": "16"
    },
    {
      "question": "3. Find the last three digits of the non-zero natural number $n$, knowing that when $29 n$ is divided by 250, the remainder is 67, and when $23 n$ is divided by 200, the remainder is 29.\n\n(Gazeta Matematica)",
      "answer": "123"
    },
    {
      "question": "If $a, b$, and $c$ are positive real numbers with $2 a+4 b+8 c=16$, what is the largest possible value of $a b c$ ?",
      "answer": "\\frac{64}{27}"
    },
    {
      "question": "6. A five-digit number abcde satisfies:\n$$\nac>d, dd, b>e \\text {. }\n$$\n\nFor example, 34201, 49412. If the digits of the number change with the position in a way similar to the monotonicity of a sine function over one period, then the five-digit number is said to conform to the \"sine rule.\" The number of five-digit numbers that conform to the sine rule is $\\qquad$.",
      "answer": "2892"
    },
    {
      "question": "5. Let the even function $f(x)$ satisfy: $f(1)=2$, and when $x y \\neq 0$, $f\\left(\\sqrt{x^{2}+y^{2}}\\right)=\\frac{f(x) f(y)}{f(x)+f(y)}$. Then $f(5)=$ $\\qquad$.",
      "answer": "\\frac{2}{25}"
    },
    {
      "question": "3. In $\\triangle A B C$, $\\angle C=90^{\\circ}, \\angle A$ and $\\angle B$ are bisected and intersect at point $P$, and $P E \\perp A B$ at point $E$. If $B C=2, A C=3$,\nthen $A E \\cdot E B=$ $\\qquad$",
      "answer": "3"
    },
    {
      "question": "\\section*{Task 2 - 011112}\n\nA steamboat travels from A to B in 3 hours and, with the same engine power, from B to A in \\(4 \\frac{1}{2}\\) hours. How long does a vessel propelled only by the current take for the journey from A to B?",
      "answer": "18"
    },
    {
      "question": "Evoikinov M.A.\n\nA pirate has five bags of coins, each containing 30 coins. He knows that one bag contains gold coins, another contains silver coins, a third contains bronze coins, and each of the two remaining bags contains an equal number of gold, silver, and bronze coins. You can simultaneously take any number of coins from any of the bags and see what kind of coins they are (coins are taken out only once). What is the smallest number of coins that need to be taken to definitely determine the contents of at least one bag?",
      "answer": "5"
    },
    {
      "question": "7. (15 points) The rules of a \"fun game\" are as follows: The player has four skills: Water, Fire, Wind, and Earth. Each skill requires a certain number of crystals to activate, and the specific effects of each skill are as follows:\nWater: Consumes 4 crystals, and simultaneously reduces the enemy's health by 4 points (if the enemy's health is less than 4 points, it is directly killed); Fire: Consumes 10 crystals, and simultaneously reduces the enemy's health by 11 points (if the enemy's health is less than 11 points, it is directly killed); Wind: Consumes 10 crystals, and simultaneously reduces the enemy's health by 5 points (if the enemy's health is less than 5 points, it is directly killed) and the next skill's crystal consumption is halved (for example: if the Water skill is used after the Wind skill, the Water skill only consumes 2 crystals);\nEarth: Consumes 18 crystals, and simultaneously halves the enemy's health (if the enemy's health is odd, add 1 first and then divide by 2): If the enemy's health is 120 points, then by reasonably selecting skills, at least $\\qquad$ crystals are needed to kill the enemy (the enemy is killed when its health is reduced to 0).",
      "answer": "63"
    },
    {
      "question": "14.1.11 * Try to append 3 digits after $523 \\cdots$ so that the resulting 6-digit number is divisible by $7$, $8$, and $9$.",
      "answer": "523656523152"
    },
    {
      "question": "52. Equal Distances. A few days ago, Professor Rackbrain stunned his students with the following puzzle:\n\n- When between three and four o'clock is the minute hand at the same distance from VIII as the hour hand is from XII?",
      "answer": "3"
    },
    {
      "question": "6. Solve the system of equations for positive real numbers:\n\n$$\n\\frac{1}{x y}=\\frac{x}{z}+1, \\quad \\frac{1}{y z}=\\frac{y}{x}+1, \\quad \\frac{1}{z x}=\\frac{z}{y}+1\n$$",
      "answer": "\\frac{1}{\\sqrt{2}}"
    },
    {
      "question": "4. On his birthday, Piglet baked a large cake weighing 10 kg and invited 100 guests. Among them was Winnie-the-Pooh, who is fond of sweets. The birthday boy announced the rule for dividing the cake: the first guest cuts a piece of the cake that is $1 \\%$, the second guest cuts a piece of the cake that is $2 \\%$ of the remaining part, the third guest cuts a piece of the cake that is $3 \\%$ of the remaining part, and so on. What place in line should Winnie-the-Pooh take to get the largest piece of cake?",
      "answer": "10"
    },
    {
      "question": "## Exercise 1. Calculate\n\n$$\n\\sqrt{32 \\sqrt{16 \\sqrt{8 \\sqrt{4}}}}\n$$\n\nOnly a numerical answer is expected here.",
      "answer": "16"
    },
    {
      "question": "8. In a $5 \\times 5$ matrix, each element is either 0 or 1, and it satisfies: the sum of the elements in each of the five rows is equal, but the sum of the elements in the five columns are all different. The number of such matrices is $\\qquad$",
      "answer": "26400"
    },
    {
      "question": "## Task 36/79\n\nSolve the equation\n\n$$\n\\binom{4 k}{k} \\cdot k^{2}=\\binom{4 k}{k+1} \\cdot(k+1)\n$$\n\nwith $k \\in \\mathbb{N}, k>0$, and calculate the value of both terms for the found solution(s).",
      "answer": "1980"
    },
    {
      "question": "A parttal párhuzamosan, attól 200 méterre halad egy vitorlás a Balatonon. Valaki folyamatosan egy irányban úszva szeretné elérni a közeledő hajót. A parthoz képest milyen szögben kell elindulnia, ha a vitorlás sebessége $15 \\mathrm{~km} / \\mathrm{h}$, az úszó sebessége $2 \\mathrm{~km} / \\mathrm{h}$, és induláskor a parton mérve $2 \\mathrm{~km}$ távolságban van a hajótól？\n\nA boat is sailing parallel to the shore, 200 meters away from it, on Lake Balaton. Someone wants to swim in a straight line to reach the approaching boat. At what angle relative to the shore should they start swimming if the boat's speed is $15 \\mathrm{~km} / \\mathrm{h}$, the swimmer's speed is $2 \\mathrm{~km} / \\mathrm{h}$, and at the start, they are $2 \\mathrm{~km}$ away from the boat, measured along the shore?",
      "answer": "42.54"
    },
    {
      "question": "B2. Let\n\n$$\nf(x)=\\ln x \\quad \\text { and } \\quad g(x)=\\frac{1}{3} x^{3}-x^{2}-6 x+4\n$$\n\nbe two functions. Let $p$ be the tangent to the graph of $f$ at the point $S(a, b)$, and $q$ be the tangent to the graph of $g$ at the point $T(a, c)$. The tangent $p$ is perpendicular to the tangent $q$. Calculate the value of $a$.",
      "answer": "3"
    },
    {
      "question": "1. Simplify:\n$$\n\\sum_{k=1}^{2016}(k \\sqrt{k+1}+(k+1) \\sqrt{k})^{-1}=\n$$",
      "answer": "1-\\frac{1}{\\sqrt{2017}}"
    },
    {
      "question": "Problem 7.1.1. Let $A, B, C, D, E, F, G, H$ - be different digits from 0 to 7 - satisfy the equation\n\n$$\n\\overline{A B C}+\\overline{D E}=\\overline{F G H}\n$$\n\nFind $\\overline{D E}$, if $\\overline{A B C}=146$.\n\n(The notation $\\overline{A B C}$ represents a three-digit number consisting of digits $A, B, C$, similarly constructed are the numbers $\\overline{F G H}$ and $\\overline{D E$).",
      "answer": "57"
    },
    {
      "question": "Point $O$ divides the segment $A B$ into segments $O A=6$ and $O B=4$. A circle is drawn with center at point $O$, and tangents are drawn from $A$ and $B$ to the circle, intersecting at point $M$, with the points of tangency lying on the same side of the line $A B$. Find the radius of the circle if $O M=12$.",
      "answer": "\\frac{6\\sqrt{21}}{7}"
    },
    {
      "question": "## Task 3 - 100923\n\nGiven are two real numbers $m \\neq 0$ and $n$. Furthermore, let $f$ be the function defined by $f(x)=m x+n$ for all real numbers.\n\na) Determine all numbers $x_{0}$ for which $2 \\cdot f\\left(x_{0}\\right)=f\\left(x_{0}+2\\right)$ holds (i.e., for which the function value at $x_{0}+2$ is twice as large as the one at $x_{0}$) when $m=1$ and $n=0$!\n\nb) Determine all numbers $x_{0}$ for which $2 \\cdot f\\left(x_{0}\\right)=f\\left(x_{0}+2\\right)$ holds for any given real numbers $m \\neq 0$ and $n$!",
      "answer": "2"
    },
    {
      "question": "## Task B-3.1.\n\nCalculate: $\\frac{1}{\\left(337^{\\frac{1}{2}}+6^{\\frac{1}{2}}\\right)^{-2}}-\\left(\\frac{\\sqrt{337}-\\sqrt{6}}{337^{\\frac{3}{2}}-6^{\\frac{3}{2}}}\\right)^{-1}$.",
      "answer": "\\sqrt{2022}"
    },
    {
      "question": "A circle with center $O$ is inscribed with a triangle $ABC (A>90^{\\circ})$. The extension of the angle bisector $AF$ of angle $A$ of this triangle intersects the circle at point $L$, and the radius $AO$ intersects the side $BC$ at point $E$. Let $AH$ be the altitude of triangle $ABC$. Find the ratio of the area of triangle $OAL$ to the area of quadrilateral $OEF L$, given that $AL=4\\sqrt{2}$, $AH=\\sqrt{2\\sqrt{3}}$, and $\\angle AEH=60^{\\circ}$.",
      "answer": "\\frac{4}{3}"
    },
    {
      "question": "13.361. A red pencil costs 27 kopecks, a blue one - 23 kopecks. No more than 9 rubles 40 kopecks can be spent on purchasing pencils. It is necessary to purchase the maximum possible total number of red and blue pencils. At the same time, the number of red pencils should be as few as possible, but the number of blue pencils should not differ from the number of red pencils by more than 10. How many red and how many blue pencils should be purchased under the given conditions?",
      "answer": "14"
    },
    {
      "question": "40. In an equilateral triangle $ABC$ with a side length of 6 (Fig. 42), find the scalar product of the vectors: a) $\\overrightarrow{A B}$ and $\\overrightarrow{A C} ;$ b) $\\overrightarrow{A B}$ and $\\overrightarrow{B C}$.",
      "answer": "18"
    },
    {
      "question": "7. The number $n+2015$ is divisible by 2016, and the number $n+2016$ is divisible by 2015. Find the smallest natural $n$ for which this is possible.\n\nANS: 4058209.",
      "answer": "4058209"
    },
    {
      "question": "4. (4 points) In a parallelogram with sides 4 and 7, the bisectors of the four interior angles are drawn. Find the ratio of the area of the quadrilateral formed by the intersection of the bisectors to the area of the parallelogram.",
      "answer": "\\frac{9}{56}"
    },
    {
      "question": "$3+$\n\n$p$ and $8 p^{2}+1$ are prime numbers. Find $p$.",
      "answer": "3"
    },
    {
      "question": "7. In $\\triangle A B C$, it is known that $a+c=2 b$, then $\\operatorname{tg} \\frac{A}{2} \\operatorname{tg} \\frac{C}{2}=$ $\\qquad$ .",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "1. A sequence of real numbers $a_{0}, a_{1}, a_{2}, \\ldots$ satisfies the relation\n\n$$\na_{m+n}+a_{m-n}-m+n-1=\\frac{a_{2 m}+a_{2 n}}{2}\n$$\n\nfor all $m, n \\in \\mathbb{N}, m \\geqslant n$. If $a_{1}=3$, find $a_{2004}$.\n\n(Kunap)",
      "answer": "2004^2+2004+1"
    },
    {
      "question": "Problem 6.3. Three merchants: Foma, Yerema, and Julius met in Novgorod. If Foma gives Yerema 70 gold coins, then Yerema and Julius will have the same amount of money. If Foma gives Yerema 40 gold coins, then Foma and Julius will have the same amount of money. How many gold coins should Foma give Yerema so that they both have the same amount of money?",
      "answer": "55"
    },
    {
      "question": "8. Let $\\{a, b, c, d\\}$ be a subset of $\\{1,2, \\cdots, 17\\}$. If 17 divides $(a-b+c-d)$, then $\\{a, b, c, d\\}$ is called a \"good subset\". Then, the number of good subsets is $\\qquad$",
      "answer": "476"
    },
    {
      "question": "Example 8 Find the maximum value of $n$ such that there exists an arithmetic sequence $a_{1}, a_{2}, \\cdots, a_{n}(n \\geqslant 3)$ satisfying\n$$\n\\sum_{i=1}^{n}\\left|a_{i}\\right|=\\sum_{i=1}^{n}\\left|a_{i}+1\\right|=\\sum_{i=1}^{n}\\left|a_{i}-2\\right|=507 .\n$$\n(2005, China Southeast Mathematical Olympiad)",
      "answer": "26"
    },
    {
      "question": "In triangle $ABC$, with the usual notations, $a > b$, and the segments $f_{1}$ and $f_{2}$, which are the distances from the internal and external angle bisectors of angle $C$ to the line $AB$, respectively, satisfy\n\n$$\n\\frac{f_{2}}{f_{1}}=\\frac{a+b}{a-b} \\sqrt{3}\n$$\n\nDetermine the angle at $C$.",
      "answer": "120"
    },
    {
      "question": "G8.2 If the area of the equilateral triangle $P Q R$ is $6+b \\sqrt{3}$, find the value of $b$.",
      "answer": "4"
    },
    {
      "question": "2.298. $\\frac{\\sqrt{21+8 \\sqrt{5}}}{4+\\sqrt{5}} \\cdot \\sqrt{9-4 \\sqrt{5}}=\\sqrt{5}-2$.",
      "answer": "\\sqrt{5}-2"
    },
    {
      "question": "An Eilitnip number is a three-digit positive integer with the properties that, for some integer $k$ with $0 \\leq k \\leq 7$ :\n\n- its digits are $k, k+1$ and $k+2$ in some order, and\n- it is divisible by $k+3$.\n\nDetermine the number of Eilitnip numbers.",
      "answer": "10"
    },
    {
      "question": "(15) Real numbers $x, y, z$ satisfy $x^{2}+y^{2}+z^{2}=1$, then the maximum value of $\\sqrt{2} x y+y z$ is\n$\\qquad$ .",
      "answer": "\\frac{\\sqrt{3}}{2}"
    },
    {
      "question": "Question 25, Given that the graph of the function $y=f(x)$ is centrally symmetric about the point $(1,1)$ and axially symmetric about the line $x+y=0$. If $x \\in(0,1)$, $f(x)=\\log _{2}(x+1)$, then the value of $f\\left(\\log _{2} 10\\right)$ is $\\qquad$ -",
      "answer": "\\frac{17}{5}"
    },
    {
      "question": "(5) Find the largest real number $\\lambda$ such that for a real-coefficient polynomial $f(x)=x^{3}+$ $a x^{2}+b x+c$ with all roots being non-negative real numbers, if $x \\geqslant 0$, then\n$$\nf(x) \\geqslant \\lambda(x-a)^{3}\n$$\n\nWhen does the equality hold in the above inequality?",
      "answer": "-\\frac{1}{27}"
    },
    {
      "question": "The circle constructed on the larger lateral side $AB$ of the right trapezoid $ABCD$ as its diameter intersects the base $AD$ at its midpoint. It is known that $AB=10, CD=6$. Find the midline of the trapezoid.",
      "answer": "12"
    },
    {
      "question": "120. Find $\\lim _{x \\rightarrow 3}\\left(x^{2}-7 x+4\\right)$.",
      "answer": "-8"
    },
    {
      "question": "2. Let ABCD be a parallelogram and E, F the midpoints of segments [BC], and respectively [CD]. If $\\mathrm{AE} \\cap \\mathrm{BF}=\\{\\mathrm{G}\\}$, and $\\mathrm{H} \\in(\\mathrm{AG}), \\mathrm{with}[\\mathrm{AH}] \\equiv[\\mathrm{HG}]$, find $\\frac{H G}{H E}$.\n\n(Sorin Furtună, Stelică Pană, Olympiads and Mathematics Competitions V-VIII 2014, Bîrchi Publishing)",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "Task 2. The houses on a street are numbered from 1 to 100. How many times does the digit $7$ appear in the house numbers?",
      "answer": "20"
    },
    {
      "question": "3. In the number $2 * 0 * 1 * 6 * 0 *$, each of the 5 asterisks needs to be replaced with any of the digits $0,1,2,3,4,5,6,7,8$ (digits can repeat) so that the resulting 10-digit number is divisible by 45. In how many ways can this be done?",
      "answer": "1458"
    },
    {
      "question": "Problem 4.3. Zhenya drew a square with a side of 3 cm, and then erased one of these sides. A figure in the shape of the letter \"P\" was obtained. The teacher asked Zhenya to place dots along this letter \"P\", starting from the edge, so that the next dot was 1 cm away from the previous one, as shown in the picture, and then count how many dots he got. He got 10 dots.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_5b323dccc142e96a9907g-03.jpg?height=331&width=329&top_left_y=196&top_left_x=562)\n\nThen the teacher decided to complicate the task and asked to count the number of dots, but for the letter \"P\" obtained in the same way from a square with a side of 10 cm. How many dots will Zhenya have this time?",
      "answer": "31"
    },
    {
      "question": "Example 4 Find the number of solutions to $\\sin x=\\lg x$.\nAnalysis This is a transcendental equation, which cannot be solved using elementary methods (thus making it impossible to determine the number of solutions directly).\nLet $y_{1}=\\sin x=f_{1}(x), y_{2}=\\lg x=f_{2}(x)$, then the number of solutions to the original equation is converted to the number of intersection points of the graphs of the two functions $f_{1}(x)$ and $f_{2}(x)$. The discussion of the existence and number of intersection points within a certain range relies on the algebraic analysis of the properties of the functions.",
      "answer": "31"
    },
    {
      "question": "A fair die is rolled six times. Find the mathematical expectation of the number of different faces that appear.\n\n#",
      "answer": "\\frac{6^{6}-5^{6}}{6^{5}}"
    },
    {
      "question": "3. Apples were divided into two unequal piles. When half of the apples from the first pile were moved to the second pile, and then half of the apples from the second pile were moved back to the first pile, there were 18 apples in the first pile and 8 in the second. How many apples were in each pile originally?",
      "answer": "20"
    },
    {
      "question": "Consider the points $P(0,0), Q(4,0)$ and $R(1,2)$. The line with equation $y=m x+b$ is parallel to $P R$ and passes through the midpoint of $Q R$. What is the value of $b$ ?",
      "answer": "-4"
    },
    {
      "question": "10. The last digit of a four-digit number is 2. If the last 2 is moved to the first position, the new number is 180 greater than the original number. The original number is $\\qquad$ -",
      "answer": "2022"
    },
    {
      "question": "8. If the internal radius $R=2+\\sqrt{6} \\mathrm{~cm}$ of a hollow sphere contains 4 solid spheres of radius $r$, then the maximum value that $r$ can take is\n\n保留了原文的换行和格式。",
      "answer": "2"
    },
    {
      "question": "Problem 9. The function $G$ is defined on the set of triples of integers and takes real values. It is known that for any four integers $a, b, c$ and $n$, the equalities $G(n a, n b, n c)=n \\cdot G(a, b, c)$, $G(a+n, b+n, c+n)=G(a, b, c)+n$, $G(a, b, c)=G(c, b, a)$ hold. Find $G(89,90,91)$.",
      "answer": "90"
    },
    {
      "question": "8. [6] $A B C D$ is a convex quadrilateral such that $A B<\\underline{A D}$. The diagonal $\\overline{A C}$ bisects $\\angle B A D$, and $m \\angle A B D=130^{\\circ}$. Let $E$ be a point on the interior of $\\overline{A D}$, and $m \\angle B A D=40^{\\circ}$. Given that $B C=$ $C D=D E$, determine $m \\angle A C E$ in degrees.",
      "answer": "55"
    },
    {
      "question": "[ Identical Transformations ]\n\nRepresent the numerical expression $2 \\cdot 2009^{2}+2 \\cdot 2010^{2}$ as the sum of squares of two natural numbers.\n\n#",
      "answer": "4019^{2}+1^{2}"
    },
    {
      "question": "3. Find the last digit of the integer part of the number $(\\sqrt{37}+\\sqrt{35})^{2016}$.",
      "answer": "1"
    },
    {
      "question": "4、A row of 20 seats, some of which are already occupied. If one more person sits down, there will be two adjacent seats occupied. The minimum number of people already seated is\n$\\qquad$ people",
      "answer": "7"
    },
    {
      "question": "11. Let $S(k)$ denote the sum of all the digits in the decimal representation of a positive integer $k$. Let $n$ be the smallest positive integer satisfying the condition $S(n)+S(n+1)=$ param1. As the answer to the problem, write down a five-digit number such that its first two digits coincide with the first two digits of $n$ and its last three digits coincide with the last three digits of $n$. For example, if $\\mathrm{n}=1234567890$, then the answer must be 12890.\n\n| param1 | Answer |\n| :---: | :---: |\n| 2016 | 59989 |\n| 664 | 49989 |\n| 1580 | 39989 |\n| 4000 | 79989 |\n\n## Condition\n\nLet $S(k)$ denote the sum of all the digits of the number $k$. Let $n$ be the smallest natural number such that $S(n)+S(n+1)=2016$. As the answer to the problem, write down a five-digit number such that its first two digits coincide with the first two digits of $n$, and its last three digits coincide with the last three digits of $n$. For example, if $\\mathrm{n}=1234567890$, then the answer should be 12890.\n\n## Answer.\n\n59989\n\n## Solution\n\nIf the number $n$ does not end in 9, then $S(n+1)=S(n)+1$, and the sum $S(n)+S(n+1)$ is odd, which means it cannot equal 2016. Therefore, the number $n$ must end in 9. If it ends in one 9, then $S(n+1)=S(n)-8$, and thus $S(n)=1012$. The smallest number with such a sum of digits is 599...989 (the total number of 9s is 111). If the number of 9s at the end of $n$ is not less than two, then by similar reasoning, the sum $S(n)$ will be larger, and the number $n$ will also be larger. Therefore, the answer is 59989.\n\n12. Let $A C$ be the largest side of triangle $A B C$. Points $K$ and $M$ are chosen on side $A C$ such that $A M=A B$ and $C K=C B$. It is given that the radius of the circumcircle of triangle $K B M$ is param1, and the radius of the incircle of triangle $A B C$ is param2, and this incircle touches side $B C$ at point $T$. Find the square of the length of segment $B T$.\n\nLet $A C$ be the largest side of triangle $A B C$. Points $K$ and $M$ are chosen on side $A C$ so that $A M=$ $A B$ and $C K=C B$. It is given that the radius of the circumcircle of the triangle $K B M$ equals param1, and the radius of the incircle of the triangle $A B C$ equals param2. Find the length of $B T$ squared if $T$ is the point where the incircle of $A B C$ touches its side $B C$.\n\n| param1 | param2 | Answer |\n| :---: | :---: | :---: |\n| 7 | 5 | 24 |\n| 8 | 5 | 39 |\n| 9 | 7 | 32 |\n| 11 | 8 | 57 |\n| 13 | 9 | 88 |\n\n## Condition\n\nLet $A C$ be the largest side of triangle $A B C$. Points $K$ and $M$ are chosen on side $A C$ such that $A M=A B$ and $C K=C B$. It is given that the radius of the circumcircle of triangle $K B M$ is 7, and the radius of the incircle of triangle $A B C$ is 5, and this incircle touches side $B C$ at point $T$. Find the square of the length of segment $B T$.\n\n## Answer.\n\n24",
      "answer": "24"
    },
    {
      "question": "8. Let $F_{1}$ and $F_{2}$ be the two foci of the ellipse $\\frac{x^{2}}{9}+\\frac{y^{2}}{4}=1$, and $P$ be a point on the ellipse such that $\\left|P F_{1}\\right|:\\left|P F_{2}\\right|$ $=2: 1$. Then the area of $\\triangle P F_{1} F_{2}$ is $\\qquad$.",
      "answer": "4"
    },
    {
      "question": "## Task 4 - 270614\n\nKerstin cuts a rectangular piece of paper so that the cut runs from the midpoint of one rectangular side to the midpoint of the opposite side. This results in two smaller rectangles; Kerstin places them exactly on top of each other so that only a smaller rectangle is visible, consisting of two layers.\n\nKerstin cuts this two-layer rectangle in the same way and again places the resulting rectangular pieces exactly on top of each other. This process continues: overlapping rectangles are cut, and the resulting pieces are placed on top of each other.\n\n(Of course, this cannot be done indefinitely; the stack of paper to be cut becomes thicker each time.)\n\n(1) State the number of paper pieces that have been created after the\n\na) second, b) third, c) fourth cutting!\n\n(2) Assuming that cutting could be done indefinitely, why would it still be impossible for exactly 160 paper pieces to ever result from a single cut?",
      "answer": "160"
    },
    {
      "question": "In the quadratic equation $x^{2}+z_{1} x+z_{2}+m=0$, $z_{1}, z_{2}, m$ are all complex numbers, and $z_{1}^{2}-4 z_{2}=16+20 \\mathrm{i}$. Suppose the two roots $\\alpha, \\beta$ of this equation satisfy $|\\alpha-\\beta|=$ $2 \\sqrt{7}$, find the maximum and minimum values of $|m|$.",
      "answer": "\\sqrt{41}+7"
    },
    {
      "question": "23. [12] If $x$ and $y$ are real numbers such that $\\frac{(x-4)^{2}}{4}+\\frac{y^{2}}{9}=1$, find the largest possible value of $\\frac{x^{2}}{4}+\\frac{y^{2}}{9}$.",
      "answer": "9"
    },
    {
      "question": "2. (i) Expand and simplify $(x+1)\\left(x^{2}-x+1\\right)$.\n(ii) Find all powers of 2 that are one more than a cube.\n(A power of 2 is a number that can be written in the form $2^{n}$, where $n$ is an integer greater than or equal to 0 .)",
      "answer": "12"
    },
    {
      "question": "Task A-3.4. (8 points)\n\nIf for the lengths $a, b, c$ of the sides of a triangle it holds that $(a+b+c)(a+b-c)=3 a b$, determine the angle opposite side $c$.",
      "answer": "60"
    },
    {
      "question": "1. As shown in the figure, in $\\triangle A B C$, $\\angle A=60^{\\circ}, A B>A C$, point $O$ is the circumcenter. The two altitudes $B E$ and $C F$ intersect at point $H$. Points $M$ and $N$ lie on segments $B H$ and $H F$ respectively, and satisfy $B M=C N$. Find the value of $\\frac{M H+N H}{O H}$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "4. (6) In each cell of a $100 \\times 100$ table, a number from the interval $[-1,1]$ is written. The sum of all the numbers is 0, the sum of all the numbers in each row in absolute value is at least $c$, and the sum of all the numbers in each column in absolute value is at least $c$. Find the largest possible value of $c$.",
      "answer": "50"
    },
    {
      "question": "60. In a parallelogram, the distances from a point inside to the four sides are $1,2,3,4$, then the minimum area of such a parallelogram is . $\\qquad$",
      "answer": "21"
    },
    {
      "question": "8. (10 points) In a plane, there are 5 points, among which no 3 points are collinear. Connecting line segments with these points as endpoints, apart from these 5 points, these line segments have at least $\\qquad$ intersection points.",
      "answer": "1"
    },
    {
      "question": "4. Given complex numbers $z, z_{1}, z_{2}\\left(z_{1} \\neq z_{2}\\right)$ satisfy:\n$$\nz_{1}^{2}=z_{2}^{2}=-2-2 \\sqrt{3} \\mathrm{i},\\left|z-z_{1}\\right|=\\left|z-z_{2}\\right|=4 \\text {. }\n$$\n\nThen $|z|=$ $\\qquad$",
      "answer": "2\\sqrt{3}"
    },
    {
      "question": "8. (3 points) Fill in the circles in the figure with $10, 15, 20, 30, 40$, and $60$ so that the product of the 3 numbers at the vertices of the three small triangles $A, B, C$ are all equal. What is the maximum equal product? $\\qquad$ .",
      "answer": "18000"
    },
    {
      "question": "33(1109). Find the members of the proportion $x_{1}: x_{2}=x_{3}: x_{4}$, where the first term is 6 more than the second, and the third is 5 more than the fourth. The sum of the squares of all terms is 793.",
      "answer": "2"
    },
    {
      "question": "7. (3 points) A rectangular wooden block with a length of 25 cm, a width of 10 cm, and a height of 4 cm is sawed into several equal-sized cubes, which are then assembled into a large cube. The surface area of this large cube is $\\qquad$ square centimeters.",
      "answer": "600"
    },
    {
      "question": "Example 1 Solve the equation: $\\lg (3-x)-\\lg (3+x)=\\lg (1-x)-\\lg (2 x+1)$.",
      "answer": "0"
    },
    {
      "question": "Two boxes together contain 65 balls of various sizes. Each ball is white, black, red, or yellow. Each time we take five balls of the same color, at least two are of the same size.\n\n(a) What is the maximum number of types of balls that exist in the boxes? Two balls are considered of different types when they have different colors or sizes.\n\n(b) Show that there are at least three balls, which are in the same box, and which are of the same type.\n\nHint: Observe that for a token to be removed it had to be flipped an odd number of times, and all circles have an even number of neighbors.\nHint: There are at most 4 distinct sizes of balls for each color.\n\nHint: Being a/600 and b/700 the two fractions, check which factors the numerator and the denominator of the sum can have in common.\n\nFacts that Help: A fraction is said to be irreducible if the numerator and the denominator do not have common prime factors.\nHint: Observe that the particular values of \\(x_{1}, x_{2}, \\ldots, x_{n}\\) are not important, but rather the number of these that are equal to 1 and -2.",
      "answer": "16"
    },
    {
      "question": "Gapochkin A.i.\n\nHow many integers from 1 to 1997 have a sum of digits that is divisible by 5?",
      "answer": "399"
    },
    {
      "question": "Let's determine the distance between the tangents parallel to the line $y=x$ for the parabolas given by the equations $y=x^{2}+1$ and $x=y^{2}+1$.\n\n---\n\nTranslate the above text into English, keeping the original text's line breaks and format, and output the translation result directly. \n\n---\n\nLet's determine the distance between the tangents parallel to the line $y=x$ for the parabolas given by the equations $y=x^{2}+1$ and $x=y^{2}+1$.",
      "answer": "\\frac{3\\sqrt{2}}{4}"
    },
    {
      "question": "Adámek was recalculating his collection of rainbow marbles. He found that he could divide them into equal piles in several ways. If he divided them into three piles, there would be $\\mathrm{v}$ more marbles in each pile than if he divided them into four piles.\n\nHow many rainbow marbles did Adámek have?\n\n(E. Semerádová)\n\nHint. Imagine actually regrouping the marbles from four piles into three.",
      "answer": "96"
    },
    {
      "question": "4. (5 points) Definition: $m \\otimes n=m \\times m-n \\times n$, then $2 \\otimes 4-4 \\otimes 6-6 \\otimes 8-\\cdots-98 \\otimes 100=$",
      "answer": "9972"
    },
    {
      "question": "12. (3 points) A rectangular photo frame is 40 cm long and 32 cm wide. A photo that is 32 cm long and 28 cm wide is placed inside. The area of the part of the frame not covered by the photo is $\\qquad$ square centimeters.",
      "answer": "384"
    },
    {
      "question": "Solve the following equation:\n\n$$\n\\sqrt{x+\\sqrt{x}}-\\sqrt{x-\\sqrt{x}}=\\frac{3}{2} \\sqrt{\\frac{x}{x+\\sqrt{x}}}\n$$",
      "answer": "\\frac{25}{16}"
    },
    {
      "question": "[\n\nTwo circles touch each other internally. A line passing through the center of the larger circle intersects it at points $A$ and $D$, and the smaller circle at points $B$ and $C$. Find the ratio of the radii of the circles if $A B: B C: C D=3: 7: 2$.\n\n#",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "We have a T-shaped figure composed of 22 squares with a side length of $1 \\mathrm{~cm}$. A line $p$ passes through point $V$, dividing the figure into two parts, white and gray. Determine in what ratio the line $p$ divides the segment $A B$, given that the areas of the white and gray regions are equal.\n\n(L. Šimünek)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_89c7e734c425fc1afd89g-1.jpg?height=591&width=593&top_left_y=1943&top_left_x=800)",
      "answer": "\\frac{3}{1}"
    },
    {
      "question": "Example 6 Determine an integer $k$ not less than 1, such that the expression $\\sin k x \\cdot \\sin ^{k} x+$ $\\cos k x \\cdot \\cos ^{k} x-\\cos ^{k} 2 x$ is independent of $x$.",
      "answer": "3"
    },
    {
      "question": "11.4. Given the function $f(x)=\\left(1-x^{3}\\right)^{-1 / 3}$. Find $f(f(f \\ldots f(2018) \\ldots))$ (the function $f$ is applied 2019 times)",
      "answer": "2018"
    },
    {
      "question": "Paris has two million inhabitants. A human being has, at most, 600000 hairs on their head. What is the largest number of Parisians that we can hope to find who have exactly the same number of hairs on their head?",
      "answer": "4"
    },
    {
      "question": "What is the greatest number of acute angles that can occur in a convex polygon?\n\n#",
      "answer": "3"
    },
    {
      "question": "6. Find the smallest real number $r$ such that a rod of length 1 can be broken into four parts, each of length at most $r$, so that no three of these parts can form a triangle.",
      "answer": "\\frac{3}{7}"
    },
    {
      "question": "3. Given the sequence $x_{n}$, satisfying $(n+1) x_{n+1}=x_{n}+n$, and $x_{1}=2$, then $x_{2009}=$ $\\qquad$",
      "answer": "\\frac{2009!+1}{2009!}"
    },
    {
      "question": "## 5. Music School\n\nIn a certain music school, each student plays exactly one instrument: piano, guitar, trumpet, or violin. Half of all students play the piano, and half of the half of all students play the guitar. The number of students who play the trumpet is equal to a quarter of the number of students who play the guitar, and 72 students play the violin. How many students are there in the music school?\n\n## Result: $\\quad 384$",
      "answer": "384"
    },
    {
      "question": "3. For the tetrahedron $ABCD$, the 6 edge lengths are 7, 13, 18, 27, 36, 41, and it is known that $AB=41$, then $CD=$ $\\qquad$",
      "answer": "13"
    },
    {
      "question": "Problem 3.6. Petya can draw only 4 things: the sun, a ball, a tomato, and a banana. But he does it extremely realistically! Today he drew several things, among which there were exactly 15 yellow, 18 round, and 13 edible. What is the maximum number of balls he could have drawn?\n\nPetya believes that all tomatoes are round and red, all balls are round and can be of any color, and all bananas are yellow and not round.",
      "answer": "18"
    },
    {
      "question": "12.11. (CSSR, 80). The set $M$ is obtained from the plane by removing three distinct points $A, B$, and $C$. Find the smallest number of convex sets whose union is the set $M$.",
      "answer": "4"
    },
    {
      "question": "Example 1. Find $\\lim _{x \\rightarrow 0} \\frac{\\sin x e^{x}-5 x}{4 x^{2}+7 x}$.",
      "answer": "-\\frac{4}{7}"
    },
    {
      "question": "[ Law of Cosines $\\quad]$ [ Properties and Criteria of Parallelograms ]\n\nGiven a parallelogram with sides 1 and 2 and an acute angle of $60^{\\circ}$. On two of its opposite sides, isosceles triangles with angles of $120^{\\circ}$ at the vertices are constructed outside the parallelogram. Find the distance between these vertices.",
      "answer": "\\sqrt{\\frac{13}{3}}"
    },
    {
      "question": "12. (17 points) A rectangular wooden board, if 4 decimeters are cut off from both the longer and shorter sides, the area is reduced by 168 square decimeters. Please ask: What is the original perimeter of the rectangle in decimeters?",
      "answer": "92"
    },
    {
      "question": "On a circle line, there are 2018 points. Each of these points is labeled with an integer. Each number is greater than the sum of the two numbers that immediately precede it in a clockwise direction.\n\nDetermine the maximum possible number of positive numbers that can be among the 2018 numbers.\n\n(Walther Janous)",
      "answer": "1008"
    },
    {
      "question": "Example 6. Find the variance of the discrete random variable $X$ representing the number of points that fall when tossing a die.",
      "answer": "\\frac{35}{12}"
    },
    {
      "question": "Bogosiov I.I.\n\n2011 numbers are written on the board. It turns out that the sum of any three of the written numbers is also a written number.\n\nWhat is the smallest number of zeros that can be among these numbers?\n\n#",
      "answer": "2.8"
    },
    {
      "question": "# Problem 4. (8 points)\n\nLena receives 50,000 rubles per month and spends 45,000 rubles per month. She gets her salary on the 6th of each month. Lena has: a deposit in the bank $\\mathrm{X}$ with a $1\\%$ monthly interest rate with monthly capitalization and the possibility of topping up, but no withdrawals allowed, with the calculation day being the 6th of each month; a debit card, on which 6% of the minimum balance for the month is credited on the 6th of each month at an annual rate of $8\\%$; a credit card with a payment due date of the 7th of each month and a limit of 100,000 rubles, with a $3\\%$ fee for cash withdrawals or transfers.\n\nLena can pay all her expenses using her bank card. Describe the optimal strategy for Lena and calculate the annual income she can earn from her cooperation with the bank (in rubles).",
      "answer": "4860"
    },
    {
      "question": "21. (2004 National College Entrance Examination, Tianjin Paper) From 4 boys and 2 girls, 3 people are randomly selected to participate in a speech contest. Let the random variable $\\xi$ represent the number of girls among the 3 selected people.\n(I) Find the distribution of $\\xi$;\n(II) Find the mathematical expectation of $\\xi$;\n(III) Find the probability that \"the number of girls $\\xi \\leqslant 1$ among the 3 selected people\".",
      "answer": "\\frac{4}{5}"
    },
    {
      "question": "Example 5 Given the set $M=\\left\\{x \\mid x^{2}-2 x+m=0, x \\in \\mathbf{R}\\right\\}$ is non-empty, then the sum of all elements in set $M$ is $\\qquad$ .",
      "answer": "2"
    },
    {
      "question": "11. (6 points) It snowed heavily at night, and in the morning, Xiaolong and his father measured the length of a circular path in the garden together, starting from the same point and walking in the same direction. Xiaolong's step length is 54 cm, and his father's step length is 72 cm. After each of them walked one complete circle and returned to the starting point, there were only 60 footprints left in the snow. What is the length of this path? $\\qquad$ meters.",
      "answer": "21.6"
    },
    {
      "question": "21. In a $9 \\times 9$ grid, each small square is colored either black or white, such that the number of black squares adjacent to each white square is greater than the number of white squares, and the number of white squares adjacent to each black square is greater than the number of black squares (squares sharing a common edge are considered adjacent), find the maximum difference in the number of black and white squares in such a coloring method.",
      "answer": "3"
    },
    {
      "question": "1. Given complex numbers $z_{1}, z_{2}$ corresponding to points $A, B$ on the complex plane, and $\\left|z_{1}\\right|=2, z_{1}^{2}-2 z_{1} z_{2}+4 z_{2}^{2}=0, O$ is the origin, then the perimeter of $\\triangle O A B$ is $\\qquad$ .",
      "answer": "3+\\sqrt{3}"
    },
    {
      "question": "2. Given $x, y \\in\\left[-\\frac{\\pi}{4}, \\frac{\\pi}{4}\\right], a \\in R$, and\n$$\n\\left\\{\\begin{array}{l}\nx^{3}+\\sin x-2 a=0, \\\\\n4 y^{3}+\\sin y \\cdot \\cos y+a=0 .\n\\end{array}\\right.\n$$\n\nThen $\\cos (x+2 y)=$ . $\\qquad$",
      "answer": "1"
    },
    {
      "question": "6. Try to advance as far as possible in the analytical solution of the problem given below. In case $B$ is necessary, a computer may be used at the final stage.\n\nPoint $A$ is located on a meadow, point $B$ - on a sandy wasteland. The distance between the points is 24 km. The boundary between the wasteland and the meadow is a straight line. The distance from point $A$ to the boundary is 8 km, and the distance from point $B$ to the boundary is 4 km. Find the minimum time it takes for a pedestrian to get from point $A$ to point $B$, if their maximum speed on the wasteland is 3 km/hour, and on the meadow 6 km/hour.",
      "answer": "4.89"
    },
    {
      "question": "11.071. Find the ratio of the volume of a regular hexagonal pyramid to the volume of a regular triangular pyramid, given that the sides of the bases of these pyramids are equal, and their apothems are twice the length of the sides of the base.",
      "answer": "\\frac{6\\sqrt{1833}}{47}"
    },
    {
      "question": "4. In the sequence $\\left\\{a_{n}\\right\\}$, $a_{k}+a_{k+1}=2 k+1\\left(k \\in \\mathbf{N}^{*}\\right)$, then $a_{1}+a_{100}$ equals",
      "answer": "101"
    },
    {
      "question": "9. The Yangtze Farm has a pasture, where grass grows uniformly every day. If 24 cows are grazed on the pasture, they will finish the grass in 6 days; if only 21 cows are grazed, it will take 8 days to finish the grass. How many days will it take to finish the grass if 36 cows are grazed?",
      "answer": "3"
    },
    {
      "question": "I1.3 If the equation $x^{2}-c x+b=0$ has two distinct real roots and their difference is 1, determine the greatest possible value of the sum of the roots, $c$.",
      "answer": "11"
    },
    {
      "question": "Suppose that $a, b, c, d$ and $e$ are consecutive positive integers with $a<b<c<d<e$. If $a^{2}+b^{2}+c^{2}=d^{2}+e^{2}$, what is the value of $a$ ?",
      "answer": "10"
    },
    {
      "question": "1. A trip to St. Petersburg is being organized for 30 schoolchildren along with their parents, some of whom will be driving cars. Each car can accommodate 5 people, including the driver. What is the minimum number of parents that need to be invited on the trip?\n\nANSWER: 10",
      "answer": "10"
    },
    {
      "question": "G8.3 If the area of the circle $E F G$ is $\\frac{(c+4 \\sqrt{3}) \\pi}{3}$, find the value of $c$",
      "answer": "7"
    },
    {
      "question": "14. Harry Potter used a wand to change a fraction, and after the change, he found that the numerator increased by $20 \\%$, and the denominator decreased by $19 \\%$, then the new fraction is increased by $\\qquad$ \\%. (rounded to 1\\%)",
      "answer": "48"
    },
    {
      "question": "$$\n\\text { 4.62 Solve the equation } \\frac{x}{2+\\frac{x}{2+\\frac{x}{2+\\ddots \\frac{x}{2+\\frac{x}{1+\\sqrt{1+x}}}}}}=1\n$$\n(In the above expression, there are 1985 twos.)",
      "answer": "3"
    },
    {
      "question": "6. A circle passes through the vertices $K$ and $P$ of triangle $K P M$ and intersects its sides $K M$ and $P M$ at points $F$ and $B$ respectively, such that $K F: F M=3: 1, P B: B M=6: 5$. Find $K P$, if $B F=\\sqrt{15}$.",
      "answer": "2\\sqrt{33}"
    },
    {
      "question": "Example 7 Given that the sum of several positive integers is 2019. Find the maximum value of their product.",
      "answer": "3^{673}"
    },
    {
      "question": "3. What are the sizes of the angles of a scalene triangle if the middle angle by size is the arithmetic mean of the remaining two angles and if the length of the longest side of the triangle is twice the length of the shortest side?",
      "answer": "30"
    },
    {
      "question": "For the polynomial\n$$\np(x)=985 x^{2021}+211 x^{2020}-211 \\text {, }\n$$\n\nlet its 2021 complex roots be $x_{1}, x_{2}, \\cdots, x_{2021}$. Then\n$$\n\\sum_{k=1}^{2021} \\frac{1}{x_{k}^{2}+1}=\n$$\n$\\qquad$",
      "answer": "2021"
    },
    {
      "question": "Three, a football invitational tournament involves sixteen cities, each sending two teams, Team A and Team B. According to the competition rules, each pair of teams plays at most one match. Additionally, the two teams from the same city do not play against each other. After a period of matches, it was found that, except for Team A from City A, all other teams have played a different number of matches. How many matches has Team B from City A played? Please prove your conclusion.",
      "answer": "15"
    },
    {
      "question": "5. $\\frac{\\sin 7^{\\circ}+\\sin 8^{\\circ} \\cos 15^{\\circ}}{\\cos 7^{\\circ}-\\sin 8^{\\circ} \\sin 15^{\\circ}}=$",
      "answer": "2-\\sqrt{3}"
    },
    {
      "question": "5. (10 points) Class 1 of Grade 4 used the class fund to purchase three types of stationery, A, B, and C, with unit prices of 3 yuan, 2 yuan, and 1 yuan, respectively. It is known that the number of B stationery purchased is 2 less than the number of A stationery purchased, and the cost of A stationery does not exceed half of the total cost. If the total cost for the three types of stationery is exactly 66 yuan, then the maximum number of A stationery that can be purchased is $\\qquad$ pieces.",
      "answer": "11"
    },
    {
      "question": "16 Given that $\\alpha+\\beta=17$ and $\\alpha \\beta=70$, find the value of $|\\alpha-\\beta|$.",
      "answer": "3"
    },
    {
      "question": "![](https://cdn.mathpix.com/cropped/2024_05_06_d3d405875929faed9b27g-21.jpg?height=184&width=2021&top_left_y=917&top_left_x=16)\n\nKoschei the Deathless has kidnapped the three daughters of the tsar. Ivan the Tsarevich set out to rescue them. He comes to Koschei, and Koschei says to him: \"Tomorrow morning, you will see five enchanted girls. Three of them are the tsar's daughters, and the other two are mine. They will be indistinguishable to you, but they will be able to recognize each other. I will approach one of them and ask her about each of the five: 'Is this a tsarevna?' She can answer truthfully or falsely, but she is allowed to call exactly two of them tsarevnas (including herself). Then I will question each of the other girls in the same way, and they must also name exactly two tsarevnas. If you can guess who the real tsarevnas are, I will let you go unharmed. And if you can also figure out which one is the eldest, which one is the middle, and which one is the youngest, you can take them with you.\" Ivan can pass a note to the tsarevnas to instruct them whom to call tsarevnas. Can he, regardless of the answers of Koschei's daughters,\n\na) return alive?\n\nb) take the tsarevnas with him?\n\n#",
      "answer": "99"
    },
    {
      "question": "5. Given distinct complex numbers $m, n$ satisfying $m n \\neq 0$, the set $\\left\\{m^{2}, n^{2}\\right\\}=\\{m, n\\}$, then $m+n=$\n\n untranslated portion:\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\nNote: The last sentence is a note about the translation request and is not part of the text to be translated. Here is the translation with the note removed:\n\n5. Given distinct complex numbers $m, n$ satisfying $m n \\neq 0$, the set $\\left\\{m^{2}, n^{2}\\right\\}=\\{m, n\\}$, then $m+n=$",
      "answer": "-1"
    },
    {
      "question": "[Examples and counterexamples. Constructions] Evaluation + example $\\quad]$\n\nOn an island, there are 100 knights and 100 liars. Each of them has at least one friend. One day, exactly 100 people said: \"All my friends are knights,\" and exactly 100 people said: \"All my friends are liars.\" What is the smallest possible number of pairs of friends, one of whom is a knight and the other a liar?",
      "answer": "50"
    },
    {
      "question": "5. Let a regular n-gon be denoted as $A_{1} A_{2} \\ldots A_{n}$. The point $A_{3}$ is reflected over the axis $A_{2} A_{4}$, resulting in the point $A_{3}^{\\prime}$. Then, the point $A_{3}^{\\prime}$ is reflected over the axis $A_{1} A_{3}$, resulting in the point $A_{3}^{\\prime \\prime}$. For which $n \\geqq 4$ is the point $A_{3}^{\\prime \\prime}$ identical to the intersection of the lines $A_{1} A_{2}$ and $A_{3} A_{4}$?\n\n(Jaroslav Zhouf)",
      "answer": "10"
    },
    {
      "question": "15. (6 points) The sum of the unit digit and the tens digit of a certain two-digit number is 12. After swapping the unit digit and the tens digit, the resulting two-digit number is 36 less than the original number. The original number is $\\qquad$",
      "answer": "84"
    },
    {
      "question": "3. A chess piece moves along the edges of a regular hexagon with a side length of $1 \\mathrm{~cm}$, starting from point $A$. On the first move, it travels $1 \\mathrm{~cm}$ to reach point $B$, on the second move, it travels $2 \\mathrm{~cm}$ to reach point $D$, $\\cdots \\cdots$, on the $n$-th move, it travels $n \\mathrm{~cm}$, until the 100th move is completed. At the point where the chess piece arrives the most times (the initial $A$ point also counts as one arrival), it has arrived $\\qquad$ times.",
      "answer": "34"
    },
    {
      "question": "3. A $3 \\times 3$ table needs to be filled with the numbers 2014, 2015, and 2016, such that the sum of the numbers in each row is the same. In how many different ways can this be done?",
      "answer": "831"
    },
    {
      "question": "28. (1) How many different ways are there to choose 4 letters from $a, b, b, c, c, c, d, d, d, d$? (2) How many different four-digit numbers can be formed by choosing 4 numbers from $1,2,2,3,3,3,4,4,4,4$?",
      "answer": "175"
    },
    {
      "question": "4. Given $A(0,0), B(1,0), C(0,2), D(3,3)$, point $P$ is any point in the plane, then the minimum value of $|P A|+|P B|+|P C|+|P D|$ is $\\qquad$.",
      "answer": "2\\sqrt{3}+\\sqrt{5}"
    },
    {
      "question": "## Task 1 - 150711\n\nTwo mathematicians are discussing their different phone numbers. It turned out the following:\n\n(1) Each of the two phone numbers is a three-digit prime number.\n\n(2) Each individual digit in the two phone numbers, when considered as a single-digit number, is also a prime number.\n\n(3) The digits in the tens place of the two phone numbers are the same. The digit in the hundreds place of one phone number is the digit in the units place of the other, and vice versa.\n\nDetermine the phone numbers, and justify the result without using a table of prime numbers as evidence!",
      "answer": "337733"
    },
    {
      "question": "## Task 1\n\nClass 3a transferred 120 marks to the solidarity account in preparation for the 30th birthday of our republic. Class 3b transferred 98 marks. How many marks more than Class 3b did Class 3a pay into the solidarity account?",
      "answer": "22"
    },
    {
      "question": "Four. (50 points) There are 2008 students participating in a large public welfare activity. If two students know each other, then these two students are considered as a cooperative group.\n(1) Find the minimum number of cooperative groups $m$, such that no matter how the students know each other, there exist three students who are pairwise in a cooperative group;\n(2) If the number of cooperative groups is $\\left[\\frac{m}{22}\\right]$, prove: there exist four students $A, B, C, D$ such that $A$ and $B$, $B$ and $C$, $C$ and $D$, $D$ and $A$ are each in a cooperative group.",
      "answer": "1008017"
    },
    {
      "question": "47. Over the past five years, 27 collective farmers have been awarded. Moreover, in each subsequent year, more were awarded than in the previous year. In the last year, three times as many were awarded as in the first year. How many collective farmers were awarded in the third year?",
      "answer": "5"
    },
    {
      "question": "6. Given the circle $x^{2}+y^{2}=4$ and the point $P(2,1)$, two perpendicular lines are drawn through point $P$ intersecting the circle at points $A, B$ and $C, D$ respectively, with point $A$ lying on the segment $P B$ and point $D$ lying on the segment $P C$. The maximum value of the area of quadrilateral $A B C D$ is $\\qquad$ .",
      "answer": "\\sqrt{15}"
    },
    {
      "question": "6. A farmer presented 6 types of sour cream in barrels of $9,13,17,19,20,38$ liters at the market. On the first day, he sold sour cream from three barrels completely, and on the second day, from two more barrels completely. The volume of sour cream sold on the first day was twice the volume of sour cream sold on the second day. Which barrels were emptied on the first day? In your answer, indicate the largest possible sum of the volumes of sour cream sold on the first day.\n\n#",
      "answer": "66"
    },
    {
      "question": "Given a circle with diameter $AB$ and an arbitrary point $P$ on it. The circle centered at $P$ with radius $PA$ intersects the line $AB$ at point $C$. When will the area of triangle $APC$ be the largest?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "8. (1995 China National Team Selection Test for IMO), Let $S=\\left\\{A=\\left(a_{1}, \\cdots, a_{8}\\right) \\mid a_{i}=0\\right.$ or $\\left.1, i=1,2, \\cdots, 8\\right\\}$. For two elements $A=\\left(a_{1}, \\cdots, a_{8}\\right)$ and $B=\\left(b_{1}, \\cdots, b_{8}\\right)$ in $S$, denote $d(A, B)=\\sum_{i=1}^{8}\\left|a_{i}-b_{i}\\right|$, and call it the distance between $A$ and $B$. How many elements can be selected from $S$ at most, such that the distance between any two of them is $\\geqslant 5$?",
      "answer": "4"
    },
    {
      "question": "1.1. (4 points) Mitya is 11 years older than Shura. When Mitya was as old as Shura is now, he was twice as old as she was. How old is Mitya?",
      "answer": "33"
    },
    {
      "question": "80. Three students' scores in a math exam are all different, and none of them scored 0 or 100. Each student knows their own score and ranking, but not the scores or rankings of the other two. They then provide information to each other:\n甲 says: “My score is a multiple of 10.”\n乙 says: “My score is a multiple of 12.”\n丙 says: “My score is a multiple of 14.”\nAfter thinking, 乙 says: “Now I know your scores.”\nSo, 乙's score is $\\qquad$ points.",
      "answer": "84"
    },
    {
      "question": "1. (Easy) Find the largest value of $x$ such that $\\sqrt[3]{x}+\\sqrt[3]{10-x}=1$.",
      "answer": "5+2\\sqrt{13}"
    },
    {
      "question": "2. [5] How many perfect squares divide $10^{10}$ ?",
      "answer": "36"
    },
    {
      "question": "## Task $7 / 68$\n\nWe are looking for all natural numbers $n$ for which $\\sum_{k=1}^{n} k$ is a three-digit number with identical digits.",
      "answer": "666"
    },
    {
      "question": "2.1. The vertices of a cube are labeled with numbers $\\pm 1$, and on its faces are numbers equal to the product of the numbers at the vertices of that face. Find all possible values that the sum of these 14 numbers can take. In your answer, specify their product.",
      "answer": "-20160"
    },
    {
      "question": "2. Three bear cubs were dividing three pieces of cheese weighing 10 g, 12 g, and 15 g. A fox came to help them. She can simultaneously bite and eat 1 g of cheese from any two pieces. Can the fox leave the bear cubs equal pieces of cheese?",
      "answer": "7"
    },
    {
      "question": "The equation of a circle is $x^{2}+y^{2}=100$; at the point $M_{1}(6, y>0)$ on the circumference of this circle, we draw a tangent. From the two endpoints of the diameter $EC$ lying on the x-axis, we draw perpendiculars to the tangent, intersecting it at points $A$ and $B$. What is the area of the triangle whose base is $AB$ and whose vertex is the origin $O$ of the coordinate system?",
      "answer": "80"
    },
    {
      "question": "2. Let $A=\\{a \\mid-1 \\leqslant a \\leqslant 2\\}$, then the area of the plane point set $B=\\{(x, y) \\mid x, y \\in A, x+y \\geqslant 0\\}$ is $\\qquad$ .",
      "answer": "7"
    },
    {
      "question": "90. In a sequence of integers (positive), each term, starting from the third, is equal to the absolute difference of the two preceding ones.\n\nWhat is the maximum number of terms such a sequence can have if each of its terms does not exceed 1967?",
      "answer": "2952"
    },
    {
      "question": "8. (6 points) Aunt Zhang and Aunt Li have the same monthly salary. Aunt Zhang deposits 30% of her salary into the bank each month, and the rest is used for daily expenses. Aunt Li's monthly daily expenses are 10% more than Aunt Zhang's, and the remaining money is also deposited into the bank. After one year, Aunt Li found that the total amount she deposited into the bank over 12 months was 5880 yuan less than Aunt Zhang's. What is Aunt Li's monthly salary? $\\qquad$",
      "answer": "7000"
    },
    {
      "question": "447. Find the greatest common divisor of the numbers 111111 and 111111111.",
      "answer": "111"
    },
    {
      "question": "Problem 8. In triangle $A B C$, side $B C=28$. The bisector $B L$ is divided by the point of intersection of the bisectors of the triangle in the ratio $4: 3$, counting from the vertex. Find the radius of the circumscribed circle around triangle $A B C$, if the radius of the inscribed circle in it is 12.",
      "answer": "50"
    },
    {
      "question": "In the following figure, the angles marked in gray have the same measure. Similarly, the angles marked in white also have the same measure. Determine the measure of angle $b$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_27f67f4520ebb8354fb2g-11.jpg?height=434&width=831&top_left_y=548&top_left_x=710)\n\n#",
      "answer": "30"
    },
    {
      "question": "10. The number of triangles with integer sides and the longest side being 11 is $\\qquad$.\n\nThe total number of such triangles is $\\qquad$.",
      "answer": "36"
    },
    {
      "question": "9. Given a cube $A B C D-A_{1} B_{1} C_{1} D_{1}$ with edge length 1, a sphere is constructed with vertex $A$ as the center and $\\frac{2 \\sqrt{3}}{3}$ as the radius. The length of the curve formed by the intersection of the sphere's surface with the cube's surface is $\\qquad$.",
      "answer": "\\frac{5\\sqrt{3}\\pi}{6}"
    },
    {
      "question": "Marayev A.\n\nCubes of size $1 \\times 1 \\times 1$ are glued together to form a cube of size $3 \\times 3 \\times 3$. What is the maximum number of cubes that can be removed from it so that the remaining figure has the following two properties:\n\n- from the side of each face of the original cube, the figure looks like a $3 \\times 3$ square (looking perpendicularly at this face, we do not see any gaps - 9 cubes of the figure are visible);\n- by moving from cube to cube through their common face, one can get from any cube to any other within the figure?",
      "answer": "13"
    },
    {
      "question": "A A trapezoid is divided into seven strips of equal width as shown. What fraction of the trapezoid's area is shaded? Explain why your answer is correct.",
      "answer": "\\frac{4}{7}"
    },
    {
      "question": "1. Calculate:\n$$\n\\frac{\\frac{7}{18} \\times 4 \\frac{1}{2}+\\frac{1}{6}}{13 \\frac{1}{3}-3 \\frac{3}{4} \\div \\frac{5}{16}} \\times 2 \\frac{7}{8}\n$$",
      "answer": "4\\frac{17}{128}"
    },
    {
      "question": "5. A supermarket is holding a promotional event, offering two packages in the discount area. Package $\\mathrm{A}$ includes 2 bottles of milk, and Package $\\mathrm{B}$ includes 3 bottles of cola. Each package comes with a free souvenir. After purchasing both types of packages, Xiao Long found that he bought a total of 50 bottles of milk and cola, and received 22 souvenirs. How many bottles of milk did Xiao Long buy?",
      "answer": "32"
    },
    {
      "question": "9. (16 points) In the acute triangle $\\triangle ABC$, $a, b, c$ are the sides opposite to $\\angle A, \\angle B, \\angle C$ respectively, and $a, b, c$ form an arithmetic sequence, $\\sin (A-C)=\\frac{\\sqrt{3}}{2}$. Find $\\sin (A+C)$.",
      "answer": "\\frac{\\sqrt{39}}{8}"
    },
    {
      "question": "Šárka enjoys celebrating, so in addition to birthdays, she came up with antibirthdays: the date of the antibirthday is created by swapping the day and month numbers in the birth date. She was born on 8.11., so her antibirthday is on 11.8. Her mother cannot celebrate her antibirthday: she was born on 23.7., her antibirthday would be on 7.23., which is not a valid date. Her brother can celebrate his antibirthday, but it falls on the same day as his birthday: he was born on 3.3.\n\nHow many days in a year are such that a person born on that day can celebrate their antibirthday, and it is on a different day than their birthday?\n\n(V. Hucíková)\n\nHint. What day of the month must a person be born on to be able to celebrate their antibirthday?\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\n---\n\nŠárka enjoys celebrating, so in addition to birthdays, she came up with antibirthdays: the date of the antibirthday is created by swapping the day and month numbers in the birth date. She was born on 8.11., so her antibirthday is on 11.8. Her mother cannot celebrate her antibirthday: she was born on 23.7., her antibirthday would be on 7.23., which is not a valid date. Her brother can celebrate his antibirthday, but it falls on the same day as his birthday: he was born on 3.3.\n\nHow many days in a year are such that a person born on that day can celebrate their antibirthday, and it is on a different day than their birthday?\n\n(V. Hucíková)\n\nHint. What day of the month must a person be born on to be able to celebrate their antibirthday?",
      "answer": "132"
    },
    {
      "question": "7. Let the edge length of a regular tetrahedron be $2 \\sqrt{6}$, and let a sphere be constructed with the center $O$ of the tetrahedron as its center. The total length of the curves formed by the intersection of the sphere's surface with the four faces of the tetrahedron is $4 \\pi$. Then the radius of the sphere $O$ is $\\qquad$",
      "answer": "\\frac{\\sqrt{5}}{2}"
    },
    {
      "question": "[ Inequalities. Case Analysis ]\n\nA magical moment is when the number of minutes on a digital clock matches the number of hours. To brew a magical potion, it must be put on the fire and taken off the fire at magical moments. To make it taste good, it must be cooked for 1.5 to 2 hours. How long does it take to cook a delicious magical potion?\n\n#",
      "answer": "1"
    },
    {
      "question": "5.1. Let's consider the number 616. The sum of its digits is 13, and the product of its digits is 36. What is the largest natural number whose sum of digits is 13 and the product of its digits is 36?",
      "answer": "3322111"
    },
    {
      "question": "$11 \\cdot 30 \\quad N$ is an integer, its $b$-ary representation is 777. Find the smallest positive integer $b$, such that $N$ is the fourth power of an integer in decimal.\n(9th Canadian Mathematics Competition, 1977)",
      "answer": "18"
    },
    {
      "question": "10.3 For the function $f(x)=2013+\\sin 2 \\pi x-8 x^{3}-12 x^{2}-18 x-a$, find the number of integer values of $a$ for each of which the equation\n\n$$\n\\underbrace{f(f(\\ldots f}_{2013 \\text { times }}(x) \\ldots))=2 x+1\n$$\n\nhas a unique solution on the interval $[49,50]$.",
      "answer": "60017"
    },
    {
      "question": "5. The sum of 10 positive integers is 24, the maximum value of the sum of their squares is $a$, and the minimum value is $b$, then $a +$ $b=$ . $\\qquad$",
      "answer": "294"
    },
    {
      "question": "10.5. Find all natural numbers $\\boldsymbol{n}$ such that $\\boldsymbol{n}$ is equal to the sum of three numbers, the first of which is the largest divisor of the number $\\boldsymbol{n} \\boldsymbol{- 1}$, different from $\\boldsymbol{n}-1$, the second is the largest divisor of the number $\\boldsymbol{n}-\\mathbf{2}$, different from $\\boldsymbol{n}-\\mathbf{2}$, and the third is the largest divisor of the number $\\boldsymbol{n}-\\mathbf{3}$, different from $\\boldsymbol{n}-\\mathbf{3}$.",
      "answer": "5866"
    },
    {
      "question": "10. Let $a, b, c \\in(0,1]$, $\\lambda$ be a real number, such that $\\frac{\\sqrt{3}}{\\sqrt{a+b+c}} \\geqslant 1+\\lambda(1-a)(1-b)(1-c)$ always holds, find the maximum value of $\\lambda$.\n\nSet $a, b, c \\in(0,1]$, $\\lambda$ as a real number, such that $\\frac{\\sqrt{3}}{\\sqrt{a+b+c}} \\geqslant 1+\\lambda(1-a)(1-b)(1-c)$ always holds, find the maximum value of $\\lambda$.",
      "answer": "\\frac{64}{27}"
    },
    {
      "question": "[ Skeletons of Polyhedral Figures ]\n\nA toy factory produces wireframe cubes, with small colorful beads located at their vertices. According to the GOST standard, each cube must use beads of all eight colors (white and the seven colors of the rainbow). How many different models of cubes can the factory produce?\n\n#",
      "answer": "1680"
    },
    {
      "question": "Problem 6. A square table of size $7 \\times 7$ with the four corner squares deleted is given.\n\nFigure 4.\n\na) What is the smallest number of squares which need to be colored black so that a 5 -square entirely uncolored Greek cross (Figure 4) cannot be found\n\n![](https://cdn.mathpix.com/cropped/2024_06_03_1ba58ed6da37450ce759g-4.jpg?height=211&width=212&top_left_y=1008&top_left_x=448)\non the table?\n\nb) Prove that it is possible to writte integers in each square in a way that the sum of the integers in each Greek cross is negative while the sum of all integers in the square table is positive.",
      "answer": "7"
    },
    {
      "question": "10.3. Find all natural $k$ such that the product of the first $k$ prime numbers, decreased by 1, is a perfect power of a natural number (greater than the first).\n\n(V. Senderov)",
      "answer": "1"
    },
    {
      "question": "Problem 11.1. The sequence $\\left\\{a_{n}\\right\\}_{n=1}^{\\infty}$ is defined by $a_{1}=0$ and $a_{n+1}=$ $a_{n}+4 n+3, n \\geq 1$.\n\na) Express $a_{n}$ as a function of $n$.\n\nb) Find the limit\n\n$$\n\\lim _{n \\rightarrow \\infty} \\frac{\\sqrt{a_{n}}+\\sqrt{a_{4 n}}+\\sqrt{a_{4^{2} n}}+\\cdots+\\sqrt{a_{4}{ }^{10} n}}{\\sqrt{a_{n}}+\\sqrt{a_{2 n}}+\\sqrt{a_{2^{2} n}}+\\cdots+\\sqrt{a_{2^{10} n}}}\n$$\n\nEmil Kolev",
      "answer": "683"
    },
    {
      "question": "1. In some cells of a $1 \\times 2100$ strip, one chip is placed. In each of the empty cells, a number is written that is equal to the absolute difference between the number of chips to the left and to the right of this cell. It is known that all the written numbers are distinct and non-zero. What is the minimum number of chips that can be placed in the cells?",
      "answer": "1400"
    },
    {
      "question": "1. At Andrei's birthday, Yana was the last to arrive, giving him a ball, and Eduard was the second to last, giving him a calculator. Testing the calculator, Andrei noticed that the product of the total number of his gifts and the number of gifts he had before Eduard arrived is exactly 16 more than the product of his age and the number of gifts he had before Yana arrived. How many gifts does Andrei have?",
      "answer": "18"
    },
    {
      "question": "[ Periodicity and Aperiodicity $]$ [ Classical Combinatorics (other)]\n\nIn an old manor, the house is surrounded by tall trees - pines, spruces, and birches. There are a total of 96 trees. These trees have a strange property: of the two trees growing two apart from any conifer, one is coniferous and the other is deciduous, and of the two trees growing four apart from any conifer, one is coniferous and the other is deciduous. How many birches are planted around the house?",
      "answer": "32"
    },
    {
      "question": "$1.53 \\frac{\\sqrt{7+4 \\sqrt{3}} \\cdot \\sqrt{19-8 \\sqrt{3}}}{4-\\sqrt{3}}-\\sqrt{3}=2$.",
      "answer": "2"
    },
    {
      "question": "4. The board for playing \"Battleship\" is $10 \\times 10$. What is the maximum number of ships of size $1 \\times 4$ that can be placed on it?",
      "answer": "24"
    },
    {
      "question": "Let's determine the value of the following number to four decimal places:\n\n$$\n2+\\sqrt{3+\\sqrt{5-\\sqrt{13+\\sqrt{48}}}}\n$$",
      "answer": "3.9319"
    },
    {
      "question": "Example 8. At a distribution base, there are electric bulbs manufactured by two factories. Among them, $60 \\%$ are made by the first factory and $40 \\%$ by the second. It is known that out of every 100 bulbs made by the first factory, 95 meet the standard, and out of 100 bulbs made by the second factory, 85 meet the standard. Determine the probability that a randomly selected bulb will meet the standard.",
      "answer": "0.91"
    },
    {
      "question": "[ Properties and characteristics of an isosceles triangle. ]\n\n[ Pythagorean Theorem (direct and inverse). ]\n\nThe lateral sides of a right trapezoid are 10 and 8. The diagonal of the trapezoid, drawn from the vertex of the acute angle, bisects this angle.\n\nFind the area of the trapezoid.",
      "answer": "104"
    },
    {
      "question": "4. The length of the escalator is 200 steps. When Petya walks down the escalator, he manages to count 50 steps. How many steps will he count if he runs twice as fast?",
      "answer": "80"
    },
    {
      "question": "Let $M=\\{1,2,3, \\cdots, 1995\\}, A$ be a subset of $M$ and satisfy the condition: when $x \\in A$, $15x \\notin A$. Then the maximum number of elements in $A$ is $\\qquad$ .",
      "answer": "1870"
    },
    {
      "question": "## Task 2 - 240732\n\na) Let $M$ be the set of all numbers $x$ that have the following properties (1), (2), (3):\n\n(1) $x$ is a six-digit natural number.\n\n(2) $x$ has the digit sum 29.\n\n(3) $x$ is divisible by 11.\n\nDetermine the largest element of the set $M$!\n\nb) Let $M^{\\prime}$ be the set of all numbers $x$ that, in addition to properties (1), (2), (3), also have the following property (4):\n\n(4) No two digits of $x$ are the same.\n\nDetermine the largest element of the set $M^{\\prime}$!",
      "answer": "999020"
    },
    {
      "question": "# 8.\n\nThe answer to the task should be some integer or a number written as a finite decimal. If the answer contains a fractional number, use a comma when writing it. Enter all characters (comma, digits) without spaces. There is no need to indicate units of measurement.\n\nThe base and the side of the triangle are 30 and 14, respectively. Find the area of this triangle if the median drawn to the base is 13.",
      "answer": "168"
    },
    {
      "question": "7. Let $\\triangle A B C$ be a right triangle with right angle $C$. Let $I$ be the incenter of $A B C$, and let $M$ lie on $A C$ and $N$ on $B C$, respectively, such that $M, I, N$ are collinear and $\\overline{M N}$ is parallel to $A B$. If $A B=36$ and the perimeter of $C M N$ is 48 , find the area of $A B C$.",
      "answer": "252"
    },
    {
      "question": "Berov s.L.\n\nSeryozha chose two different natural numbers $a$ and $b$. He wrote down four numbers in his notebook: $a, a+2, b$ and $b+2$. Then he wrote on the board all six pairwise products of the numbers from the notebook. What is the maximum number of perfect squares that can be among the numbers on the board?",
      "answer": "2"
    },
    {
      "question": "9. As shown in the figure, the area of trapezoid $A B C D$ is 117 square centimeters. $A D / / B C$, $E F=13$ centimeters, $M N=4$ centimeters, and it is also known that $E F \\perp M N$ at $O$. What is the total area of the shaded parts? $\\qquad$ square centimeters.",
      "answer": "65"
    },
    {
      "question": "In the diagram, the circle has centre $O$ and radius 6 . Point $A$ is outside the circle and points $B$ and $C$ are on the circle so that $A B$ is perpendicular to $B O, A C$ is perpendicular to $C O$, and $\\angle B A C=50^{\\circ}$. What is the area of the shaded region?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2efeb2001664c4e9869eg-2.jpg?height=396&width=572&top_left_y=325&top_left_x=836)",
      "answer": "13\\pi"
    },
    {
      "question": "Tokarev S.I.\n\nThe path from platform $A$ to platform $B$ was traveled by the electric train in $X$ minutes ($0<X<60$). Find $X$, given that the angle between the hour and minute hands was $X$ degrees both at the moment of departure from $A$ and at the moment of arrival at $B$.",
      "answer": "48"
    },
    {
      "question": "68. Nail two boards of the same length together, resulting in a board that is 39 centimeters long. The overlapping part in the middle is 11 centimeters long. Each of these two boards is $\\qquad$ centimeters long.",
      "answer": "25"
    },
    {
      "question": "37. Football teams were holding a round-robin tournament. One of the fans of this game, returning from vacation, found the following situation: the total number of points scored by all teams was 44; one team, which scored the fewest points, received one point; the two teams occupying the first and second places scored the same number of points. How many teams are participating in the tournament and how many games do they still have to play?",
      "answer": "9"
    },
    {
      "question": "16. Let $f(x)=\\frac{x^{2010}}{x^{2010}+(1-x)^{2010}}$. Find the value of\n$$\nf\\left(\\frac{1}{2011}\\right)+f\\left(\\frac{2}{2011}\\right)+f\\left(\\frac{3}{2011}\\right)+\\ldots+f\\left(\\frac{2010}{2011}\\right)\n$$",
      "answer": "1005"
    },
    {
      "question": "32. It is given that $p$ is a prime number such that\n$$\nx^{3}+y^{3}-3 x y=p-1\n$$\nfor some positive integers $x$ and $y$. Determine the largest possible value of $p$.",
      "answer": "5"
    },
    {
      "question": "3. Solve the equation\n\n$$\n2(x-6)=\\frac{x^{2}}{(1+\\sqrt{x+1})^{2}}\n$$",
      "answer": "8"
    },
    {
      "question": "14.5. a) Seven girls are forming a circle. In how many different ways can they stand in a circle?\n\nb) How many necklaces can be made from seven different beads?",
      "answer": "360"
    },
    {
      "question": "## Task A-3.2. (4 points)\n\nLegoplus is a body consisting of seven equal cubes joined in such a way that there is one cube that shares a common face with each of the remaining six cubes.\n\nEach face of the legoplus must be painted with one color. How many colors are minimally needed to do this so that no two adjacent faces are painted the same color?",
      "answer": "3"
    },
    {
      "question": "G2.3 Given that there are $c$ prime numbers less than 100 such that their unit digits are not square numbers, find the values of $c$.",
      "answer": "15"
    },
    {
      "question": "1. How many weeks can a year overlap? Assume that a year overlaps with a week if at least 6 and one day of this week falls within the given year.",
      "answer": "54"
    },
    {
      "question": "Task B-3.1. Solve the equation\n\n$$\n\\left(4-\\frac{x}{2013}\\right)^{10^{2013}}=\\left(\\frac{x}{671}\\right)^{10^{2013}}\n$$",
      "answer": "-40262013"
    },
    {
      "question": "G10.3The fraction $\\frac{p}{q}$ is in its simplest form. If $\\frac{7}{10}<\\frac{p}{q}<\\frac{11}{15}$ where $q$ is the smallest possible positive integer and $c=p q$. Find the value of $c$.",
      "answer": "35"
    },
    {
      "question": "## Task 1\n\nIn front of the circus tent, students from a first grade class are standing.\n\nThe children have lined up in pairs. The teacher gives out 2 admission tickets at a time. She distributes four times 2 tickets to the girls and five times 2 tickets to the boys.\n\nQuestions:\n\na) How many girls want to go to the circus?\n\nb) How many boys are standing in front of the circus?\n\nc) How many children receive an admission ticket from the teacher?",
      "answer": "8"
    },
    {
      "question": "For any positive integer $n$, let $f_{1}(n)$ denote the square of the sum of the digits of $n$ plus $r+1$, where $r$ is the integer such that $n=3q+r, 0 \\leqslant r<3$. For $k \\geqslant 2$, let $f_{k}(n)=f_{1}\\left(f_{k-1}(n)\\right)$. Therefore, $f_{1990}(2345)$ $=$ $\\qquad$.",
      "answer": "3"
    },
    {
      "question": "5. In the lottery box of Green Green Park, there are golden, silver, and colorful balls. The lottery participant can randomly draw 3 balls from the lottery box. Xiao Yangyang hopes to get 3 golden balls, Mei Yangyang hopes to get 2 golden balls and 1 silver ball, and Lan Yangyang hopes to get one of each kind of ball. Village Chief Man Yangyang says: “Only one of you three can be chosen to draw the lottery, no matter who is chosen, the probability of achieving your wish is the same.” There are at least $\\qquad$ balls in the lottery box.",
      "answer": "8"
    },
    {
      "question": "13.288. A master and his apprentice were tasked with manufacturing a batch of identical parts. After the master worked for 7 hours and the apprentice for 4 hours, it turned out that they had completed $5 / 9$ of the entire work. After working together for another 4 hours, they found that $1 / 18$ of the entire work remained to be done. How long would it take the apprentice to complete the entire work on his own?",
      "answer": "24"
    },
    {
      "question": "6. How many solutions does the rebus $\\frac{B+O+C+b+M+O+\\breve{U}}{K+J+A+C+C}=\\frac{22}{29}$ have, where different letters represent different digits, and the same letters represent the same digits? It is known that the digit 0 is not used.",
      "answer": "0"
    },
    {
      "question": "11. Andrew, Bob, and Chris are working together to finish a group project. If Andrew doesn't help, it would take 2 hours. If Bob doesn't help, it would take 3 hours. If Chris doesn't help, it would take 4 hours. Now they just learned that Dave can also help them out. If Dave works on the project alone, it would take him half a day. What is the least amount of time it would take for them to finish the group project?",
      "answer": "\\frac{8}{5}"
    },
    {
      "question": "3. Determine the smallest and largest value of the function\n\n$$\nf(x, y, z, t)=\\frac{a x^{2}+b y^{2}}{a x+b y}+\\frac{a z^{2}+b t^{2}}{a z+b t},(a>0, b>0)\n$$\n\nunder the conditions $x+z=y+t=1, x, y, z, t \\geq 0$.",
      "answer": "12"
    },
    {
      "question": "Let's determine the $x$ for which the\n\n$$\n|13-x|+|58-x|+|71-x|+|75-x|+|79-x|\n$$\n\nsum is minimal!",
      "answer": "71"
    },
    {
      "question": "[ Inscribed and described circles ]\n\nIn triangle $A B C$ with sides $A B=\\sqrt{3}, B C=4, A C=\\sqrt{7}$, the median $B D$ is drawn. The circles inscribed in triangles $A B D$ and $B D C$ touch $B D$ at points $M$ and $N$ respectively. Find $M N$.\n\n#",
      "answer": "2-\\frac{\\sqrt{3}}{2}"
    },
    {
      "question": "Problem 4.6.3 A rectangle with a perimeter of 96 cm was divided into 108 smaller identical rectangles by eight vertical and eleven horizontal cuts. What is the perimeter of each of them, if the total length of all the cuts is 438 cm?",
      "answer": "9"
    },
    {
      "question": "An equilateral triangle has sides of length $x+5, y+11$, and 14 . What is the value of $x+y$ ?\n\n##",
      "answer": "12"
    },
    {
      "question": "1. In a chess tournament, 10 players participated. Each player played one game with each of the other players.\n\na) How many games did each individual player play?\n\nb) How many games of chess were played in total at the tournament?\n\nExplain your answers.",
      "answer": "45"
    },
    {
      "question": "2. Let $S$ be a subset of $\\{1,2, \\ldots, 9\\}$ such that the sums of any two elements of $S$ are distinct. For example, the set $\\{1,2,3,5\\}$ has this property, but the set $\\{1,2,3,4,5\\}$ does not, because $\\{2,3\\}$ and $\\{1,4\\}$ both sum to 5. What is the maximum number of elements that $S$ can contain?",
      "answer": "5"
    },
    {
      "question": "23. In $\\triangle A B C, A B: A C=4: 3$ and $M$ is the midpoint of $B C$. $E$ is a point on $A B$ and $F$ is a point on $A C$ such that $A E: A F=2: 1$. It is also given that $E F$ and $A M$ intersect at $G$ with $G F=72 \\mathrm{~cm}$ and $G E=x \\mathrm{~cm}$. Find the value of $x$.",
      "answer": "108"
    },
    {
      "question": "# Task 3. (3 points)\n\nIn the country of Taxia, everyone pays as many percent in taxes as thousands of tugriks their salary amounts to. What salary is the most advantageous to have?\n\n(Salary is measured in a positive, not necessarily integer number of tugriks)",
      "answer": "50000"
    },
    {
      "question": "7. Find the value of\n$$\n\\sum_{a=1}^{\\infty} \\sum_{b=1}^{\\infty} \\sum_{c=1}^{\\infty} \\frac{a b(3 a+c)}{4^{a+b+c}(a+b)(b+c)(c+a)}\n$$",
      "answer": "\\frac{1}{54}"
    },
    {
      "question": "3. (20 points) In math class, each dwarf needs to find a three-digit number without zero digits, divisible by 3, such that when 297 is added to it, the result is a number with the same digits but in reverse order. What is the minimum number of dwarfs that should be in the class so that among the numbers they find, there are always at least two identical ones?",
      "answer": "19"
    },
    {
      "question": "[Example 3.2.6] Find a three-digit number such that the ratio of the number to the sum of its digits is minimized.",
      "answer": "199"
    },
    {
      "question": "【Question 2】Using 1 one, 2 twos, 2 threes to form some four-digit numbers, the total number of different four-digit numbers that can be formed is $\\qquad$.\n\nTranslating the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "30"
    },
    {
      "question": "8. When we play poker, if we get 2 cards of the same size (such as 2 fives), we say we have a \"pair of fives.\" If we get three cards of the same size (such as 3 Ks), we say we have a \"trio of Ks.\" If we get 4 cards of the same size, we say we have a \"bomb.\" In a deck of poker cards, at least ( ) cards must be drawn to ensure a \"bomb.\"",
      "answer": "42"
    },
    {
      "question": "## 167. Math Puzzle $4 / 79$\n\nThe workers of a company's press shop conducted an initiative shift. 50 tons of press material was the result, exceeding their plan by eight tons. What was the percentage of plan fulfillment?",
      "answer": "119"
    },
    {
      "question": "8. Let there be a geometric sequence $\\left(a+\\log _{2} 3\\right),\\left(a+\\log _{4} 3\\right),\\left(a+\\log _{8} 3\\right)$, with a common ratio of $\\qquad$ .",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "9-5-1. The numbers 1 and 7 are written on the board. In one move, the greatest common divisor of both numbers on the board is added to each of them. For example, if at some point the numbers 20 and 50 are on the board, they will be replaced by the numbers 30 and 60.\n\nWhat numbers will be on the board after 100 moves? Provide the answers in any order. Answer. 588 and 594.",
      "answer": "588594"
    },
    {
      "question": "Example 5 Calculate $\\sec \\frac{2 \\pi}{9}+\\sec \\frac{4 \\pi}{9}+\\sec \\frac{6 \\pi}{9}+\\sec \\frac{8 \\pi}{9}$.",
      "answer": "4"
    },
    {
      "question": "The alphabet consists of 24 letters, and to describe each permutation of these letters, $24 \\mathrm{~cm}^{2}$ of area is needed. How many times the surface area of the Earth is required to describe all permutations? (The radius of the Earth is $r=6370 \\mathrm{~km}$.)",
      "answer": "2921000"
    },
    {
      "question": "Example $\\mathbf{3}$ Given the quadratic equation in $x$, $a(1+\\mathrm{i}) x^{2}+\\left(1+a^{2} \\mathrm{i}\\right) x+a^{2}+\\mathrm{i}=0$, has real roots, find the value of the real number $a$.\n\nThe equation is $a(1+\\mathrm{i}) x^{2}+\\left(1+a^{2} \\mathrm{i}\\right) x+a^{2}+\\mathrm{i}=0$.",
      "answer": "-1"
    },
    {
      "question": "3. Find the sum of all four-digit numbers in which the digits $0,4,5,9$ do not appear.",
      "answer": "6479352"
    },
    {
      "question": "9. (6 points) A hollow cone and a hollow cylinder, with base inner radii and heights of $12 \\mathrm{~cm}, 20 \\mathrm{~cm}$ respectively, are combined to form a vertically placed container as shown in the figure. Some fine sand is poured into this container, filling the cone and part of the cylinder. It is measured that the sand in the cylinder is $5 \\mathrm{~cm}$ high. If the container is inverted, the height of the sand will be $\\qquad$ $\\mathrm{cm}$.",
      "answer": "11\\frac{2}{3}"
    },
    {
      "question": "Example 2 Given the function $f(x)=x^{2}+2 x \\cdot \\tan \\theta-1, x \\in[-1, \\sqrt{3}]$, where $\\theta \\in\\left(-\\frac{\\pi}{2}, \\frac{\\pi}{2}\\right)$\n(1) When $\\theta=-\\frac{\\pi}{6}$, find the maximum and minimum values of the function $f(x)$;\n(2) Find the range of $\\theta$ such that $y=f(x)$ is a monotonic function on $[-1, \\sqrt{3}]$.",
      "answer": "-\\frac{4}{3}"
    },
    {
      "question": "# 5. Option 1.\n\nThe chocolate bar has the shape of a square with a side of 100 cm, divided into pieces with a side of 1 cm. The sweet tooth ate all the pieces along each of the four sides, as shown in the figure. How many pieces did the sweet tooth eat in total?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7eebe7f078c503a48a3eg-4.jpg?height=279&width=287&top_left_y=419&top_left_x=911)",
      "answer": "784"
    },
    {
      "question": "12. If $p$ is the smallest positive prime such that for some integer $n, p$ divides $n^{2}+5 n+23$, then $p=$",
      "answer": "13"
    },
    {
      "question": "8. Given positive integers $a_{1}, a_{2}, \\cdots, a_{2018}$ satisfying\n$$\na_{1}<a_{2}<\\cdots<a_{2018} \\text {. }\n$$\n\nFor $i=1,2, \\cdots, 2018, b_{i}$ is the number of positive integers in $a_{1}, a_{2}, \\cdots, a_{2018}$ that do not exceed $i$. Then\n$$\n\\frac{\\sum_{k=1}^{2018} a_{k}+\\sum_{k=1}^{a_{2018}} b_{k}}{a_{2018}+1}=\n$$\n$\\qquad$",
      "answer": "2018"
    },
    {
      "question": "We draw the midlines on the base of a regular tetrahedron. We connect the midpoint of each midline with the vertices of the side face parallel to the midline. What fraction of the volume of the tetrahedron is the volume of the common part of the three tetrahedra thus obtained?",
      "answer": "\\frac{1}{10}"
    },
    {
      "question": "12. If $16^{\\sin ^{2} x}+16^{\\cos ^{2} x}=10$, then $\\cos 4 x=$\n$\\qquad$ .",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "Example 9. The probability of manufacturing a non-standard part is $p=0.004$. Find the probability that among 1000 parts there will be 5 non-standard ones.",
      "answer": "0.1428"
    },
    {
      "question": "3. (8 points) Xiaopang used all of this month's salary to buy a stock. On the first day, the stock price increased by $\\frac{1}{10}$, on the second day it decreased by $\\frac{1}{11}$, on the third day it increased by $\\frac{1}{12}$, and on the fourth day it decreased by $\\frac{1}{13}$. At this point, the value of his stock is exactly 5000 yuan. What was Xiaopang's salary this month? $\\qquad$ yuan.",
      "answer": "5000"
    },
    {
      "question": "2. Arrange the order of operations in the expression\n\n$$\n1891-(1600: a+8040: a) \\times c .\n$$\n\nand calculate its value when $a=40$ and $c=4$. Show how the expression can be modified without changing its numerical value.",
      "answer": "927"
    },
    {
      "question": "21.22 * Write an $n$-digit number using the digits 1 and 2, where no two adjacent digits are both 1, and let the number of such $n$-digit numbers be $f(n)$. Find $f(10)$.",
      "answer": "144"
    },
    {
      "question": "Task 1. (3 points) A piece of cheese is given. It is allowed to choose any positive real number $a$, different from 1, and to cut the piece in the ratio 1 : a by weight, then to cut one of the resulting pieces in the same ratio, and so on. Is it possible with a finite number of cuts to obtain pieces of cheese that can be distributed into two portions of the same weight?",
      "answer": "1+\\sqrt{2}"
    },
    {
      "question": "The sum of three numbers is 100, two are prime and one is the sum of the other two.\n\n(a) What is the largest of the three numbers?\n(b) Give an example of such three numbers.\n\n(c) How many solutions exist for this problem?",
      "answer": "4"
    },
    {
      "question": "B2. How many natural numbers less than 100000 have a sum of digits divisible by 9 or 5?",
      "answer": "31109"
    },
    {
      "question": "5. (3 points) Three-digit even numbers $A, B, C, D, E$ satisfy $A<B<C<D<E$. If $A+B+C+D+E=4306$, then $A$ is the smallest. $\\qquad$",
      "answer": "326"
    },
    {
      "question": "(12) The sum of all real roots of the equation $\\left|x^{2}-3 x+2\\right|+\\left|x^{2}+2 x-3\\right|=11$ is $\\qquad$ .",
      "answer": "\\frac{-19+5\\sqrt{97}}{20}"
    },
    {
      "question": "11. From the sequence of positive integers $1,2,3,4,5, \\cdots$, remove the multiples of 3 and 4, but retain all multiples of 5 (for example, 15 and 120 are retained). After removing, the remaining numbers form a new sequence: $1,2,5,7,10, \\cdots$. Find the 2010th term of the new sequence.",
      "answer": "3349"
    },
    {
      "question": "Example 1 Determine the periodicity of the function $f(x)=x-[x], x \\in \\mathbf{R}$ (where $[x]$ denotes the greatest integer not exceeding $x$) and sketch its graph.",
      "answer": "1"
    },
    {
      "question": "7. Find the smallest positive integer that is $20 \\%$ larger than one integer and $19 \\%$ smaller than another.",
      "answer": "162"
    },
    {
      "question": "Many years ago, a teacher who did not want to give a lecture ordered his students to calculate the sum of the numbers from 1 to 100. A very clever student named Gauss found a very simple way to accomplish the task by discovering the formula:\n\n$$\n1+2+3+\\ldots+n=\\frac{n(n+1)}{2}\n$$\n\nSince this story is from a long time ago, today's challenges given to students require more elaborate tasks.\n\n(a) Verify that every prime number greater than 3 leaves a remainder of 1 or 5 when divided by 6.\n\n(b) Verify that the sum of the prime numbers that are greater than 1 and less than 1000 is less than 166338.\n\n(c) In the above estimate, to avoid technical complications, we did not eliminate some numbers that are certainly not prime. Eliminate some of these numbers and verify that the sum of the primes that are greater than 1 and less than 1000 is less than 166000.",
      "answer": "166000"
    },
    {
      "question": "The point $K$ lies in the interior of the unit circle. Four lines are drawn through $K$ such that each two adjacent lines form an angle of $45^{\\circ}$, as shown in the picture. In such a way the circle is divided into 8 regions. Four of these regions are colored such that no two colored regions share more than one point. Find the maximal and minimal possible value for the total area of colored regions.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2f0af4b96a9a083248abg-2.jpg?height=414&width=417&top_left_y=381&top_left_x=842)\n\n#",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "[ Inequality problems. Case analysis]\n\nLёsha and Ira live in a house where each floor has 9 apartments (the house has one entrance). The floor number of Lёsha is equal to the apartment number of Ira, and the sum of their apartment numbers is 329. What is the apartment number of Lёsha?",
      "answer": "296"
    },
    {
      "question": "6. On the faces $BCD, ACD, ABD$, and $ABC$ of the tetrahedron $ABCD$, points $A_{1}, B_{1}, C_{1}$, and $D_{1}$ are marked, respectively. It is known that the lines $AA_{1}, BB_{1}, CC_{1}$, and $DD_{1}$ intersect at point $P$, and $\\frac{AP}{A_{1}P}=\\frac{BP}{B_{1}P}=\\frac{CP}{C_{1}P}=\\frac{DP}{D_{1}P}=r$. Find all possible values of $r$.",
      "answer": "3"
    },
    {
      "question": "My only son was born when I was 37 years old. This was exactly 32 years after my grandfather's death, and he died at the age of 64. My grandfather was 12 years older than my grandmother, and they got married in 1947, when my grandmother was 18 years old.\n\nIn which year was my son born?\n\nNote: Ignore any discrepancies related to specific birth dates; you can assume, for example, that all the people mentioned have their birthdays on the same day.\n\n(M. Smitková)\n\nHint: How old was the grandfather when he got married?",
      "answer": "2013"
    },
    {
      "question": "6. Given that a line with slope $k$ is drawn through a focus of the ellipse $x^{2}+2 y^{2}=3$, intersecting the ellipse at points $A$ and $B$. If $AB=2$, then $|k|=$ $\\qquad$ .",
      "answer": "\\sqrt{1+\\sqrt{3}}"
    },
    {
      "question": "[  [Theorem on the lengths of a tangent and a secant; the product of the entire secant and its external part. Pythagorean Theorem (direct and inverse). Area of a trapezoid\n\nA circle passes through vertices $A$ and $B$ of rectangle $A B C D$ and is tangent to side $C D$ at its midpoint. A line is drawn through vertex $D$ that is tangent to the same circle at point $E$, and then intersects the extension of side $A B$ at point $K$.\n\nFind the area of trapezoid $B C D K$, if $A B=10$ and $K E: K A=3: 2$.\n\n#",
      "answer": "210"
    },
    {
      "question": "9. In the Pokémon factory, there is a type of fruit can, the original packaging bottle's internal is a cylinder with a diameter of 18 cm and a height of 10 cm. Now the internal diameter of the packaging bottle is reduced by 6 cm, and the height is increased by 2 cm. Then, the capacity of the original 1000 bottles is equivalent to the capacity of $\\qquad$ bottles now.",
      "answer": "1875"
    },
    {
      "question": "5. A square and a regular hexagon are drawn with the same side length. If the area of the square is $\\sqrt{3}$, what is the area of the hexagon?",
      "answer": "\\frac{9}{2}"
    },
    {
      "question": "2. Circle $k$ is inscribed in trapezoid $A B C D, A B \\| C D$, touching side $A B$ at point $E$. If $A E=15, B E=10$ and $C D=8$, determine the radius of circle $k$.",
      "answer": "4\\sqrt{3}"
    },
    {
      "question": "## Task 5 - 030725\n\nIn a box, there are 70 balls, namely 20 red, 20 green, 20 yellow, and the rest are black or white. Brigitte is supposed to take out as many balls from this box in the dark so that among them, there are certainly at least 10 balls of the same color.\n\nHow many balls does she have to take out at least? Justify your answer!",
      "answer": "38"
    },
    {
      "question": "6. (10 points) Fill in the 6 boxes in the figure with the numbers $1 \\sim 6$ respectively, the smallest result that can be obtained is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "342"
    },
    {
      "question": "20. As shown in the figure, the shape is composed of ten line segments. Now, we are to color the line segments using three colors, with the requirement that adjacent segments must be colored differently (line segments sharing a common endpoint are considered adjacent). If colors can be reused, how many different coloring methods are there in total?\n\nThe shape is composed of ten line segments. Now, we are to color the line segments using three colors, with the requirement that adjacent segments must be colored differently (line segments sharing a common endpoint are considered adjacent). If colors can be reused, how many different coloring methods are there in total?",
      "answer": "30"
    },
    {
      "question": "3. In triangle $\\triangle A B C$, the base $A B$ of which lies on the x-axis, altitudes $A M, B N$ and $C K$ are drawn. Find the length of the base $A B$, if the coordinates of points $M(2,2)$ and $N(4,4)$ are known.",
      "answer": "4\\sqrt{5}"
    },
    {
      "question": "10. Let $p$, $q$, $r$ be prime numbers, and\n$$\np|(q r-1), q|(r p-1), r \\mid(p q-1) \\text {. }\n$$\n\nFind all possible values of $p q r$.",
      "answer": "30"
    },
    {
      "question": "4. Let $a, b \\in \\mathbf{N}^{*},(a, b)=1$. Find the number of natural numbers $c$ such that the indeterminate equation $a x+b y=c$ has no non-negative integer solutions.",
      "answer": "\\frac{1}{2}(1)(1)"
    },
    {
      "question": "495. Find all pairs of natural numbers if their sum is 60, and the least common multiple is 72.",
      "answer": "2436"
    },
    {
      "question": "7. Given $z \\in \\mathbf{C}$. If the equation $x^{2}-2 z x+\\frac{3}{4}+\\mathrm{i}=0$ (where $\\mathrm{i}$ is the imaginary unit) has real roots, then the minimum value of $|z|$ is . $\\qquad$",
      "answer": "1"
    },
    {
      "question": "Evdokimov M.A.\n\nIn each cell of a $5 \\times 5$ board, there is either a cross or a zero, and no three crosses stand in a row either horizontally, vertically, or diagonally. What is the maximum number of crosses that can be on the board?",
      "answer": "16"
    },
    {
      "question": "4. If $\\cos ^{4} \\theta+\\sin ^{4} \\theta+(\\cos \\theta \\cdot \\sin \\theta)^{4}+$ $\\frac{1}{\\cos ^{4} \\theta+\\sin ^{4} \\theta}=\\frac{41}{16}$, then $\\sin ^{2} \\theta=$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Let $x_{1}=1$ and $x_{n+1}=x_{n}+\\left\\lfloor\\frac{x_{n}}{n}\\right\\rfloor+2$ for $n=1,2,3, \\ldots$, where $\\lfloor x\\rfloor$ denotes the largest integer not greater than $x$. Determine $x_{1997}$.",
      "answer": "23913"
    },
    {
      "question": "4. In the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, $A B=$ $2 \\sqrt{2}, B C=A A_{1}=2, P$ is a moving point on the diagonal $A C_{1}$, $Q$ is a moving point on the base $A B C D$ (points $P$ and $Q$ can coincide). Then the minimum value of $B_{1} P+P Q$ is $\\qquad$",
      "answer": "4.3"
    },
    {
      "question": "A cube with an edge of $12 \\mathrm{~cm}$ was divided into smaller, identical cubes such that the sum of the surface areas of all the smaller cubes was eight times greater than the surface area of the original cube.\n\nDetermine how many small cubes there were and what the length of their edges was.\n\n(M. Volfová)",
      "answer": "512"
    },
    {
      "question": "13. On a square $A B C D$ a line segment $B E$ is drawn such that the point $E$ lies on the side $C D$. The perimeter of triangle $B C E$ is three-quarters of the perimeter of the square $A B C D$. The ratio of lengths $C E: C D$ is $\\lambda: 1$.\nWhat is the value of $960 \\times \\lambda$ ?",
      "answer": "720"
    },
    {
      "question": "In the diagram, $\\triangle A B C$ is right-angled at $C$. Point $D$ is on $A C$ so that $\\angle A B C=2 \\angle D B C$. If $D C=1$ and $B D=3$, determine the length of $A D$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_30_fa78072b1e75930324fdg-3.jpg?height=328&width=371&top_left_y=1170&top_left_x=931)",
      "answer": "\\frac{9}{7}"
    },
    {
      "question": "Mrs. Kucera was on a seven-day vacation, and Káta walked her dog and fed her rabbits for the entire duration. For this, she received a large cake and 700 CZK. After another vacation, this time a four-day one, Káta received the same cake and 340 CZK for walking and feeding according to the same rules.\n\nWhat was the value of the cake?",
      "answer": "140"
    },
    {
      "question": "3rd Swedish 1963 Problem 2 The squares of a chessboard have side 4. What is the circumference of the largest circle that can be drawn entirely on the black squares of the board?",
      "answer": "2\\sqrt{40}\\pi"
    },
    {
      "question": "3. Find the value of the expression $\\operatorname{tg} 20^{\\circ}+4 \\sin 20^{\\circ}$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "3. Let $O$ be the circumcenter of acute $\\triangle A B C$, with $A B=6, A C=10$. If $\\overrightarrow{A O}=x \\overrightarrow{A B}+y \\overrightarrow{A C}$, and $2 x+10 y=5$, then $\\cos \\angle B A C=$ $\\qquad$ .",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "## Problem Statement\n\nCalculate the volumes of the bodies bounded by the surfaces.\n\n$$\n\\frac{x^{2}}{16}+\\frac{y^{2}}{9}+\\frac{z^{2}}{36}=1, z=3, z=0\n$$",
      "answer": "11\\pi"
    },
    {
      "question": "2. (7 points) In rectangle $A B C D$, side $A B$ is equal to 6, and side $B C$ is equal to 11. Bisectors of the angles from vertices $B$ and $C$ intersect side $A D$ at points $X$ and $Y$ respectively. Find the length of segment $X Y$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2509cbb2376df96befe7g-1.jpg?height=317&width=417&top_left_y=1121&top_left_x=822)\n\n## Answer: 1.",
      "answer": "1"
    },
    {
      "question": "The base of the right prism $A B C A_{1} B_{1} C_{1}$ is an isosceles triangle $A B C$, where $A B=B C=5$, $\\angle A B C=2 \\arcsin 3 / 5$. A plane perpendicular to the line $A_{1} C$ intersects the edges $A C$ and $A_{1} C_{1}$ at points $D$ and $E$ respectively, such that $A D=1 / 3 A C, E C_{1}=1 / 3 A_{1} C_{1}$. Find the area of the section of the prism by this plane.",
      "answer": "\\frac{40}{3}"
    },
    {
      "question": "3.1. ** Let $\\left\\{a_{n}\\right\\}$ be an arithmetic sequence with common difference $d, d \\neq 0$, and $\\left\\{b_{n}\\right\\}$ be a geometric sequence with common ratio $q$. Given that $a_{1}=b_{1}, a_{3}=b_{3}, a_{7}-b_{5}$. Find the relationship between $n$ and $m$ when $a_{n}=b_{m}$.",
      "answer": "2^{(+1)/2}-1"
    },
    {
      "question": "6 Set $A=\\{0,1,2,3,4,5,9\\}, a, b \\in A$, and $a \\neq b$. The number of functions of the form $y=-a x^{2}+(4-b) x$ whose graph vertex is in the first quadrant is $\\qquad$ .",
      "answer": "21"
    },
    {
      "question": "IMO 1995 Problem A3 Determine all integers n > 3 for which there exist n points A 1 , ... , A n in the plane, no three collinear, and real numbers r 1 , ... , r n such that for any distinct i, j, k, the area of the triangle A i A j A k is r i + r j + r k .",
      "answer": "4"
    },
    {
      "question": "$\\left[\\begin{array}{l}\\text { Tangent Circles } \\\\ \\text { [Pythagorean Theorem (direct and inverse).] }\\end{array}\\right]$\n\nTwo circles with radii $\\sqrt{19}$ and $\\sqrt{76}$, touching each other externally, are inscribed in a semicircle (i.e., each circle touches this semicircle and its diameter). Find the radius of the semicircle.",
      "answer": "4\\sqrt{19}"
    },
    {
      "question": "4. In the plane, there is a set $S$ of $n$ points ( $n>2$ ) with the property: if $A, B \\in S$, then there exists $C \\in S$ such that triangle $A B C$ is equilateral. How many can the number $n$ be?",
      "answer": "3"
    },
    {
      "question": "6. (10 points) Expression: $\\frac{\\frac{1}{1}-\\frac{1}{3}}{\\frac{1}{1} \\times \\frac{1}{2} \\times \\frac{1}{3}}+\\frac{\\frac{1}{2}-\\frac{1}{4}}{\\frac{1}{2} \\times \\frac{1}{3} \\times \\frac{1}{4}}+\\frac{\\frac{1}{3}-\\frac{1}{5}}{\\frac{1}{3} \\times \\frac{1}{4} \\times \\frac{1}{5}}+\\cdots+\\frac{\\frac{1}{9}-\\frac{1}{11}}{\\frac{1}{9} \\times \\frac{1}{10} \\times \\frac{1}{11}}$ calculation result",
      "answer": "108"
    },
    {
      "question": "A car with a weight of $P=1800 \\mathrm{~kg}$ starts on a road where $\\rho=\\frac{1}{50}, e=\\frac{1}{40} ; 1 \\frac{1}{12}$ minutes after, while covering $900 \\mathrm{~m}$, it reaches its maximum speed: $v=12 \\frac{\\mathrm{m}}{\\mathrm{sec}}$. How many horsepower is its power? What is the traction force?[^0]\n\n\n[^0]:    ${ }^{1}$ In this problem, $\\rho$ is the coefficient of friction, and $e$ is the sine of the incline angle.",
      "answer": "24"
    },
    {
      "question": "(12) The complex number $z$ satisfies $|z|(3 z+2 \\mathrm{i})=2(\\mathrm{i} z-6)$, then $|z|=$",
      "answer": "2"
    },
    {
      "question": "4. The sequence $\\left\\{x_{n}\\right\\}$ is defined as follows: $x_{1}=\\frac{1}{2}, x_{k+1}=x_{k}^{2}+x_{k}, k=1,2, \\cdots$. Find the integer part of the following sum:\n$$\n\\frac{1}{1+x_{1}}+\\frac{1}{1+x_{2}}+\\cdots+\\frac{1}{1+x_{2005}}\n$$",
      "answer": "1"
    },
    {
      "question": "4. for $a<0.5$\n\n## Correct answer: for $a<1$\n\n## Question 3\n\nCorrect\n\nPoints: 1.00 out of a maximum of 1.00\n\nQuestion 4\n\nCorrect\n\nPoints: 1.00 out of a maximum of 1.00\n\n## Question 5\n\nCorrect\n\nPoints: 1.00 out of a maximum of 1.00\nWorker $A$ and worker $B$ can complete the work in 7.2 days, worker $A$ and worker $C$ in 9 days, worker $B$ and worker $C$ in 12 days. How many days will they take to complete the work if they all work together?",
      "answer": "6"
    },
    {
      "question": "Problem 6.7. Anya places pebbles on the sand. First, she placed one stone, then added pebbles to form a pentagon, then made a larger outer pentagon with pebbles, then another outer pentagon, and so on, as shown in the picture. The number of stones she had arranged on the first four pictures: 1, 5, 12, and 22. If she continues to create such pictures, how many stones will be on the 10th picture?\n\n- \n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-17.jpg?height=113&width=104&top_left_y=818&top_left_x=382)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-17.jpg?height=187&width=186&top_left_y=823&top_left_x=527)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-17.jpg?height=266&width=277&top_left_y=820&top_left_x=758)",
      "answer": "145"
    },
    {
      "question": "B4. For the second round of the Math Olympiad, 999 students are invited. Melanie prepares invitation letters in the order of participant numbers: $1,2,3, \\ldots$ For some values of $n \\geqslant 100$, she notices the following: the number of participant numbers from $1$ to $n$ that end in a 5 is exactly equal to the number formed by the last two digits of $n$.\n\nFor how many values of $n$ (with $100 \\leqslant n<1000$) does this hold?",
      "answer": "9"
    },
    {
      "question": "The family approached the bridge at night. Dad can cross it in 1 minute, Mom in 2, the child in 5, and Grandma in 10 minutes. They have one flashlight. The bridge can only hold two people at a time. How can they cross the bridge in 17 minutes? (If two people cross together, they move at the slower of their speeds. Moving across the bridge without a flashlight is not allowed. Shining the light from a distance is not allowed. Carrying each other is not allowed.)",
      "answer": "17"
    },
    {
      "question": "【16】There are four points $A, B, C, D$ arranged from left to right on a straight line. Using these four points as endpoints, 6 line segments can be formed. It is known that the lengths of these 6 line segments are $14, 21, 34, 35, 48, 69$ (unit: millimeters). Then the length of line segment $\\mathrm{BC}$ is ( ) millimeters.",
      "answer": "14"
    },
    {
      "question": "4B. Around are placed 10 children. Ivo needs to join them. They will all play the elimination game in the following way: they start counting from the number 1 in the direction of the clock's hands, and the one on whom the number 13 falls is eliminated from the game. The counting continues with the remaining children where it was interrupted, and the thirteenth one is again eliminated from the game, etc. At which position (between which two of the ten marked numbers, see image) should Ivo stand to be the last one in the game?\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_a6c2bf676fcc39861820g-03.jpg?height=377&width=360&top_left_y=840&top_left_x=666)",
      "answer": "7"
    },
    {
      "question": "$A M$ - bisector of triangle $A B C, B M=2, C M=3, D$ - the point of intersection of the extension of $A M$ with the circumcircle of the given triangle, $M D=2$. Find $A B$.",
      "answer": "\\sqrt{10}"
    },
    {
      "question": "## Task B-1.6.\n\nHow many ordered pairs of natural numbers $(m, n)$ are there for which the number $3^{m}+7^{n}$ is divisible by 10, given that $1 \\leqslant m \\leqslant 80, 81 \\leqslant n \\leqslant 185$?",
      "answer": "2100"
    },
    {
      "question": "## Task 36/80\n\nThe smallest natural number $n$ is sought for which the following holds\n\n$$\nn \\equiv 1(\\bmod 2) ; \\quad n \\equiv 1(\\bmod 3) ; \\quad n \\equiv 1(\\bmod 5) ; \\quad n \\equiv 0(\\bmod 7) ; \\quad n \\equiv 1(\\bmod 11)\n$$",
      "answer": "1981"
    },
    {
      "question": "I1.2 Given that $f(x)=-x^{2}+10 x+9$, and $2 \\leq x \\leq \\frac{a}{9}$. If $b$ is the difference of the maximum and minimum values of $f$, find the value of $b$.",
      "answer": "9"
    },
    {
      "question": "19. Let $f(x)$ be a function that satisfies the following conditions:\n(1) If $x>y$ and $f(x)+x \\geqslant w \\geqslant f(y)+y$, then there exists a real number $z \\in[y, x]$ such that $f(z)=w-z$;\n(2) The equation $f(x)=0$ has at least one solution, and among these solutions, there is one that is not greater than all the others;\n(3) $f(0)=1$;\n(4) $f(-2007) \\leqslant 2008$;\n(5) $f(x) f(y)=f[x f(y)+y f(x)+x y]$.\n\nFind: the value of $f(-2007)$.",
      "answer": "2008"
    },
    {
      "question": "4. Given that $f(x), g(x)$ are functions defined on $\\mathbf{R}$, the graph of $f(x)$ is symmetric about the line $x=1$, the graph of $g(x)$ is symmetric about the point $(1,-2)$, and $f(x)+g(x)=9^{x}+x^{3}+1$, then the value of $f(2) g(2)$ is $\\qquad$ .",
      "answer": "2016"
    },
    {
      "question": "14th Swedish 1974 Problem 3 Let a 1 = 1, a 2 = 2 a 1 , a 3 = 3 a 2 , a 4 = 4 a 3 , ... , a 9 = 9 a 8 . Find the last two digits of a 9 .",
      "answer": "21"
    },
    {
      "question": "9. As shown in the figure, in the cube $A B C D-A_{1} B_{1} C_{1} D_{1}$ with edge length 2, $E$ is the midpoint of $B C$, and point $P$ moves along the line segment $D_{1} E$. Then, the minimum distance from point $P$ to the line $C C_{1}$ is $\\qquad$.",
      "answer": "\\frac{2\\sqrt{5}}{5}"
    },
    {
      "question": "5. (10 points) If $\\frac{O}{11}-\\frac{7}{\\square}<\\frac{4}{5}$ holds, then the maximum sum of the non-zero natural numbers that can be filled in “ $O$ ” and “ $\\square$ ” is",
      "answer": "393"
    },
    {
      "question": "1. Let $n$ be a natural number, $a, b$ be positive real numbers, and satisfy $a+b=2$, then the minimum value of $\\frac{1}{1+a^{n}}+\\frac{1}{1+b^{n}}$ is $\\qquad$.",
      "answer": "1"
    },
    {
      "question": "8. (7 points) A bag contains 10 white balls, 5 black balls, and 4 blue balls. All balls are identical except for their color. If $n$ balls are taken out, what is the minimum value of $n$ to ensure that at least 2 balls of each color are obtained? $\\qquad$ .",
      "answer": "17"
    },
    {
      "question": "5. (10 points) Among all the factors of the four-digit number Shuangcheng Shuangcheng, 3 are prime numbers, and the other 39 are not prime. Then, \nthe four-digit number Shuangcheng Shuangcheng has $\\qquad$ factors.",
      "answer": "12"
    },
    {
      "question": "1. Given that the three interior angles $\\angle A, \\angle B, \\angle C$ of $\\triangle \\mathrm{ABC}$ satisfy $2 \\angle A - \\angle B, 3 \\angle B - 2 \\angle C$, and $90^{\\circ} - \\angle A$, the minimum of these is $\\alpha$, then the maximum value of $\\alpha$ is $\\qquad$",
      "answer": "\\frac{2\\pi}{9}"
    },
    {
      "question": "Let $f: \\mathbb{Z}_{>0} \\rightarrow \\mathbb{R}$ be a function for which it holds that for all $n>1$ there is a prime divisor $p$ of $n$ such that\n\n$$\nf(n)=f\\left(\\frac{n}{p}\\right)-f(p)\n$$\n\nMoreover, it is given that $f\\left(2^{2014}\\right)+f\\left(3^{2015}\\right)+f\\left(5^{2016}\\right)=2013$.\nCalculate $f\\left(2014^{2}\\right)+f\\left(2015^{3}\\right)+f\\left(2016^{5}\\right)$.",
      "answer": "\\frac{49}{3}"
    },
    {
      "question": "15. Let $a>0, x \\in[-1,1]$. When the function $y=-x^{2}-a x+b$ has a minimum value of -1 and a maximum value of 1, find the corresponding values of $x$ for which the function attains its minimum and maximum values.",
      "answer": "-2+2\\sqrt{2}"
    },
    {
      "question": "1. We understand a palindrome as a natural number that reads the same forwards and backwards, for example, 16 261. Find the largest four-digit palindrome whose square is also a palindrome.",
      "answer": "2002"
    },
    {
      "question": "## Task A-4.5.\n\nLet $z_{1}$, $z_{2}$, and $z_{3}$ be complex numbers such that $\\left|z_{1}\\right|=\\left|z_{2}\\right|=\\left|z_{3}\\right|=1$ and $4 z_{3}=$ $3\\left(z_{1}+z_{2}\\right)$. What is $\\left|z_{1}-z_{2}\\right|$?",
      "answer": "\\frac{2}{3}\\sqrt{5}"
    },
    {
      "question": "Garkovv S.B.\n\nThe sequence $\\left(a_{n}\\right)$ is such that $a_{n}=n^{2}$ for $1 \\leq n \\leq 5$ and for all natural $n$ the equality $a_{n+5} + a_{n+1} = a_{n+4} + a_{n}$ holds. Find $a_{2015}$.",
      "answer": "17"
    },
    {
      "question": "Shapovalov A.V.\n\nFor what largest $n$ can one choose $n$ points on the surface of a cube so that not all of them lie in the same face of the cube and are vertices of a regular (planar) $n$-gon.",
      "answer": "12"
    },
    {
      "question": "# 9. Problem 9.10\n\nIn a convex 10-gon \\(A_{1} A_{2} \\ldots A_{10}\\), all sides and all diagonals connecting vertices every other one (i.e., \\(A_{1} A_{3}, A_{2} A_{4}\\), etc.) are drawn, except for the side \\(A_{1} A_{10}\\) and the diagonals \\(A_{1} A_{9}, A_{2} A_{10}\\).\n\nWe call a path leading from \\(A_{1}\\) to \\(A_{10}\\) a non-self-intersecting broken line (i.e., one in which non-adjacent segments do not have common points) with endpoints \\(A_{1}\\) and \\(A_{10}\\), each segment of which coincides with one of the drawn sides or diagonals. Find the number of such paths.",
      "answer": "55"
    },
    {
      "question": "## Task Condition\n\nCalculate the area of the figure bounded by the graphs of the functions:\n\n$$\ny=(x+1)^{2}, y^{2}=x+1\n$$",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "## Task A-4.2.\n\nA rubber ball was thrown from a height of 200 meters. Each time it bounces off the surface, it reaches $4 / 5$ of the previous height: after the first bounce, it rises to 160 meters, after the second bounce to 128 meters, etc. What is the total distance the ball travels before it comes to a stop?",
      "answer": "1800"
    },
    {
      "question": "Problem 8.2. (15 points) Real numbers $x_{1}, x_{2}, x_{3}, x_{4}$ are such that\n\n$$\n\\left\\{\\begin{array}{l}\nx_{1}+x_{2} \\geqslant 12 \\\\\nx_{1}+x_{3} \\geqslant 13 \\\\\nx_{1}+x_{4} \\geqslant 14 \\\\\nx_{3}+x_{4} \\geqslant 22 \\\\\nx_{2}+x_{3} \\geqslant 23 \\\\\nx_{2}+x_{4} \\geq 24\n\\end{array}\\right.\n$$\n\nWhat is the smallest value that the sum $x_{1}+x_{2}+x_{3}+x_{4}$ can take?",
      "answer": "37"
    },
    {
      "question": "6.50 In an exam, there are 4 multiple-choice questions, each with 3 options. A group of students take the exam, and the result is that for any 3 people, there is 1 question where their answers are all different. How many students can participate in the exam at most?",
      "answer": "9"
    },
    {
      "question": "1. Determine the greatest value of the natural number $n$ such that the number $39^{n}$ is a divisor of the number 39! Explain your answer.",
      "answer": "3"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$\\lim _{n \\rightarrow \\infty} \\frac{n^{2}-\\sqrt{n^{3}+1}}{\\sqrt[3]{n^{6}+2}-n}$",
      "answer": "1"
    },
    {
      "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the equations in polar coordinates.\n\n$$\n\\rho=6(1+\\sin \\varphi),-\\frac{\\pi}{2} \\leq \\varphi \\leq 0\n$$",
      "answer": "12(2-\\sqrt{2})"
    },
    {
      "question": "Tokaeva I.\n\nLet $F_{1}, F_{2}, F_{3}, \\ldots$ be a sequence of convex quadrilaterals, where $F_{k+1}$ (for $k=1,2,3, \\ldots$) is obtained by cutting $F_{k}$ along a diagonal, flipping one of the parts, and gluing it back along the cut line to the other part. What is the maximum number of different quadrilaterals that this sequence can contain? (Different polygons are those that cannot be made to coincide by a movement.)",
      "answer": "6"
    },
    {
      "question": "1 The minimum value of the function $y=\\sin ^{4} x+\\cos ^{4} x+\\sec ^{4} x+\\csc ^{4} x$ is",
      "answer": "8.5"
    },
    {
      "question": "Pankrác, Servác, and Bonifác bought a boat. Pankrác paid $60\\%$ of the boat's price, Servác paid $40\\%$ of the remaining price, and Bonifác paid the missing amount, which was 30 gold coins.\n\nHow many gold coins did the boat, which the boys bought, cost?\n\n(L. Hozová)",
      "answer": "125"
    },
    {
      "question": "30.6. Let $A B C$ be an isosceles triangle with $A B=A C$. Let $D$ and $E$ be the feet of the perpendiculars from $B$ and $C$ to $\\overline{A C}$ and $\\overline{A B}$, respectively. Suppose that $\\overline{C E}$ and $\\overline{B D}$ intersect at point $H$. If $E H=1$ and $A D=4$, find $D E$.",
      "answer": "\\frac{8\\sqrt{17}}{17}"
    },
    {
      "question": "(9) Let the sequence $\\left\\{a_{n}\\right\\}$ satisfy $a_{n}=(2 n-1)(2 n+1)(2 n+3)$, then the greatest common divisor $d$ of $a_{1}, a_{2}, \\cdots$, $a_{2008}$ these 2008 terms is $\\qquad$.",
      "answer": "3"
    },
    {
      "question": "18. Numbers $a, b$ and $c$ are such that $\\frac{a}{b+c}=\\frac{b}{c+a}=\\frac{c}{a+b}=k$. How many possible values of $k$ are there?",
      "answer": "2"
    },
    {
      "question": "Exercise 3. On a rectangular box with dimensions $6 \\mathrm{~cm} \\times 4 \\mathrm{~cm}$, a ribbon has been placed diagonally as shown in the figure below (the ribbon is represented in gray in the figure).\n\n![](https://cdn.mathpix.com/cropped/2024_06_04_e23d725cd2c5ceb81650g-2.jpg?height=320&width=404&top_left_y=518&top_left_x=858)\n\nWe measured $A B=1 \\mathrm{~cm}$ and $B C=5 \\mathrm{~cm}$. Calculate the width of the ribbon.",
      "answer": "\\frac{4}{5}"
    },
    {
      "question": "Problem 5.4. The school principal, the caretaker, and the parent committee, failing to agree with each other, each bought a carpet for the school auditorium, which is $10 \\times 10$. After thinking about what to do, they decided to place all three carpets as shown in the picture: the first carpet $6 \\times 8$ - in one corner, the second carpet $6 \\times 6$ - in the opposite corner, and the third carpet $5 \\times 7$ - in one of the remaining corners (all dimensions are in meters).\n\nFind the area of the part of the hall covered by carpets in three layers (give the answer in square meters).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_86512448fd32765ac040g-10.jpg?height=337&width=1003&top_left_y=304&top_left_x=225)",
      "answer": "6"
    },
    {
      "question": "2.165. $\\frac{t^{2}-t-6-(t+3) \\sqrt{t^{2}-4}}{t^{2}+t-6-(t-3) \\sqrt{t^{2}-4}} ; t>2$.",
      "answer": "-\\sqrt{\\frac{+2}{-2}}"
    },
    {
      "question": "A4. A circle with radius 2 rolls around a regular pentagon with side 4. During this process, the circle remains pressed against the pentagon until it returns to its initial position.\n\nCalculate the area of the region that is covered by the circle during this revolution.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_6ccf422d4ecbc0dab674g-1.jpg?height=448&width=462&top_left_y=1666&top_left_x=1294)",
      "answer": "16(5+\\pi)"
    },
    {
      "question": "[Law of Cosines] [Law of Sines]\n\nFind the radius of the circumcircle of a triangle with sides 5 and 8 and an angle of $60^{\\circ}$ between them.",
      "answer": "\\frac{7\\sqrt{3}}{3}"
    },
    {
      "question": "7. From all four-digit numbers where each digit is unique and the sum of the digits is 10, if two numbers are randomly selected, what is the probability that 2017 is selected?",
      "answer": "\\frac{1}{48}"
    },
    {
      "question": "116. How many lines can divide a plane into 5 parts?",
      "answer": "4"
    },
    {
      "question": "6. A real estate agent is trying to sell the last apartment in a building for $482,100 \\mathrm{kn}$, which was the price of the penultimate apartment, and by doing so, the average price of the apartments he sold in that building would be $519,500 \\mathrm{kn}$. However, due to market saturation, he sells the apartment for only $451,500 \\mathrm{kn}$, making the average price of the apartments he sold in that building $517,700 \\mathrm{kn}$. How many apartments in that building has the real estate agent sold?",
      "answer": "17"
    },
    {
      "question": "2. (17 points) There are ten weights of different weights, each weighing an integer number of grams. It is known that the weight of the lightest and heaviest weight differs by 9 grams. One weight is lost. Find its weight if the total weight of the remaining weights is 2022 grams.",
      "answer": "223"
    },
    {
      "question": "4. Masha bought 2021 weights of pairwise distinct masses. Now Masha places one weight on each pan of a two-pan balance (weights placed on the balance previously are not removed). Each time the balance is in equilibrium, Masha rejoices. What is the maximum number of times she can find a reason to be happy",
      "answer": "673"
    },
    {
      "question": "On each wall of a regular octahedron, one of the numbers $1,2,3,4,5,6,7$ and 8 is written, with different numbers on different walls. For each wall, Jarda determined the sum of the number written on it and the numbers of the three adjacent walls. Thus, he obtained eight sums, which he also added together.\n\nWhat values can this final sum take?\n\n(J. Zhouf)\n\nHint. How many times is each number counted in the total sum?",
      "answer": "144"
    },
    {
      "question": "3. Determine how many times the number $((2014)^{2^{2014}}-1)$ is larger than the number written in the following form: $\\left.\\left.\\left((2014)^{2^{0}}+1\\right) \\cdot\\left((2014)^{2^{1}}+1\\right) \\cdot\\left((2014)^{2^{2}}+1\\right) \\cdot \\ldots \\quad \\cdot\\right)^{2^{2013}}+1\\right)$.",
      "answer": "2013"
    },
    {
      "question": "8. Find all values of the parameter $a$ for which the system\n\n$$\n\\left\\{\\begin{array}{l}\ny=\\frac{|x|-1}{|x+1|}-2 \\\\\n4|x-1.5-a|+|y-1-2a|=1\n\\end{array}\\right.\n$$\n\nhas a unique solution. In your answer, specify the smallest of all the obtained values of the parameter $a$.",
      "answer": "-1.5"
    },
    {
      "question": "12th Balkan 1995 Problem 1 Define a n by a 3 = (2 + 3)/(1 + 6), a n = (a n-1 + n)/(1 + n a n-1 ). Find a 1995 . Solution",
      "answer": "\\frac{1991009}{1991011}"
    },
    {
      "question": "We have a lighter and two pieces of string that each burn for one hour when lit at one end (but not necessarily at a constant speed). How can we measure three-quarters of an hour?",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "The sum of the lengths of all of the edges of rectangular prism $A B C D E F G H$ is 24 . If the total surface area of the prism is 11, determine the length of the diagonal $A H$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_30_22929074a9efc78b4168g-2.jpg?height=380&width=542&top_left_y=279&top_left_x=846)",
      "answer": "5"
    },
    {
      "question": "3. A group of toddlers in a kindergarten has 90 teeth in total. Any two toddlers together do not have more than 9 teeth. What is the minimum number of toddlers that can be in the group?",
      "answer": "23"
    },
    {
      "question": "What digit can the positive integer $n \\geq 3$ end with, if $n+n^{2}+\\ldots+n^{2 n-3}-4$ is a prime number?",
      "answer": "5"
    },
    {
      "question": "1. Given the sets $M=\\{x, x y, \\lg (x y)\\}$ and $N=\\{0,|x|, y\\}$, and $M=N$, then the value of $\\left(x+\\frac{1}{y}\\right)+$ $\\left(x^{2}+\\frac{1}{y^{2}}\\right)+\\left(x^{3}+\\frac{1}{y^{3}}\\right)+\\cdots+\\left(x^{2001}+\\frac{1}{y^{2(101)}}\\right)$ is $\\qquad$",
      "answer": "-2"
    },
    {
      "question": "7.1. In a row, the numbers $\\sqrt{7.301}, \\sqrt{7.302}, \\sqrt{7.303}, \\ldots, \\sqrt{16.002}, \\sqrt{16.003}$ are written (under the square root - consecutive terms of an arithmetic progression with a common difference of 0.001). Find the number of rational numbers among the listed ones.",
      "answer": "13"
    },
    {
      "question": "4. Let $a, b, c, d, e$ be positive integers. Their sum is 2018. Let $M=\\max (a+b, b+c, c+d, d+e)$. Find the smallest possible value of $M$.",
      "answer": "673"
    },
    {
      "question": "I2.2 If the height of a cylinder is doubled and the new radius is $h$ times the original, then the new volume is $k$ times the original. Find $k$.",
      "answer": "32"
    },
    {
      "question": "Starting from a number, you can either cube it or divide it by 8. Starting from the number 2 and using these two operations, can you reach 64? And $2^{2011}$?",
      "answer": "2^{2011}"
    },
    {
      "question": "(7) An ant starts from vertex $A$ of the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, and travels along the surface of the prism to reach vertex $C_{1}$. The shortest distance is 6. Then, the maximum volume of the rectangular prism is $\\qquad$ .",
      "answer": "12\\sqrt{3}"
    },
    {
      "question": "Given the hyperbola $\\frac{x^{2}}{a}-\\frac{y^{2}}{a}=1$ with its right focus at $F$, any line passing through $F$ intersects the right branch of the hyperbola at $M, N$. The perpendicular bisector of $M N$ intersects the $x$-axis at $P$. When $a$ takes any real number, the value of $\\frac{\\vdots F P}{|M N|}$ is $\\qquad$",
      "answer": "\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "25. There are 8 items with different masses, each of which is an integer number of grams and does not exceed 15 grams. Xiao Ping wants to find the item with the greatest mass using the fewest number of weighings on a balance. He used the following method:\n(1) Divide the 8 items into 2 groups, each with 4 items, and compare the masses of these 2 groups;\n(2) Take the 4 items from the heavier group and divide them into 2 groups, each with 2 items, and compare their masses;\n(3) Take the 2 items from the heavier group and divide them into 1 item each, and take the heavier one.\nXiao Ping did not get a balanced result on the balance in all 3 weighings, and finally obtained one item.\nHowever, the item he found is actually the 5th heaviest among the 8 items.\nQuestion: What is the mass of the item Xiao Ping found? And what is the mass of the second heaviest item?",
      "answer": "11"
    },
    {
      "question": "88. Divide the numbers $1, 2, 3, 4, 5, 6, 7, 8$ into three groups, and calculate the sum of the numbers in each group. If these three sums are all different, and the largest sum is twice the smallest sum, then the smallest sum is $\\qquad$.",
      "answer": "8"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0} \\frac{1+\\cos (x-\\pi)}{\\left(e^{3 x}-1\\right)^{2}}$",
      "answer": "\\frac{1}{18}"
    },
    {
      "question": "7.1. Come up with at least one three-digit PAU number (all digits are different) such that $(П+\\mathrm{A}+\\mathrm{У}) \\times \\Pi \\times \\mathrm{A} \\times \\mathrm{Y}=300$ (it is sufficient to provide one example)",
      "answer": "235"
    },
    {
      "question": "Shapovalov A.B.\n\n55 boxers participated in a tournament with a \"loser leaves\" system. The fights proceeded sequentially. It is known that in each match, the number of previous victories of the participants differed by no more than 1. What is the maximum number of fights the tournament winner could have conducted?",
      "answer": "8"
    },
    {
      "question": "(9) In the Cartesian coordinate system, the \"rectangular distance\" between points $P\\left(x_{1}, y_{1}\\right)$ and $Q\\left(x_{2}, y_{2}\\right)$ is defined as $d(P, Q)=\\left|x_{1}-x_{2}\\right|+\\left|y_{1}-y_{2}\\right|$. If the \"rectangular distance\" from $C(x, y)$ to points $A(1,3)$ and $B(6,9)$ is equal, where real numbers $x$ and $y$ satisfy $0 \\leqslant x \\leqslant 10,0 \\leqslant y \\leqslant 10$, then the sum of the lengths of all the loci of points $C$ that satisfy the condition is $\\qquad$.",
      "answer": "5(\\sqrt{2}+1)"
    },
    {
      "question": "2. Find the sum of the first twelve terms of an arithmetic progression if its fifth term $a_{5}=1$, and the seventeenth term $a_{17}=18$.",
      "answer": "37.5"
    },
    {
      "question": "2. Four cars $A, B, C$, and $D$ start simultaneously from the same point on a circular track. $A$ and $B$ drive clockwise, while $C$ and $D$ drive counterclockwise. All cars move at constant (but pairwise different) speeds. Exactly 7 minutes after the start of the race, $A$ meets $C$ for the first time, and at the same moment, $B$ meets $D$ for the first time. After another 46 minutes, $A$ and $B$ meet for the first time. After how much time from the start of the race will all the cars meet for the first time?",
      "answer": "371"
    },
    {
      "question": "3 、The product of two positive integers is 100, neither of these positive integers contains the digit 0, then the sum of these two positive integers is",
      "answer": "29"
    },
    {
      "question": "(4) (50 points) Let $A$ be a $3 \\times 9$ grid, with each small cell filled with a positive integer. A sub-grid of $A$ of size $m \\times n (1 \\leqslant m \\leqslant 3, 1 \\leqslant n \\leqslant 9)$ is called a \"good rectangle\" if the sum of all its numbers is a multiple of 10. A $1 \\times 1$ cell in $A$ is called a \"bad cell\" if it is not contained in any \"good rectangle\". Find the maximum number of \"bad cells\" in $A$.",
      "answer": "25"
    },
    {
      "question": "6.3. In the park, all bicycle paths run from north to south or from west to east. Petya and Kolya simultaneously started from point $A$ and rode their bicycles at constant speeds: Petya - along the route $A-B-C$, Kolya - along the route $A-D-E-F-C$ (see fig.), and both spent 12 minutes on the road. It is known that Kolya rides 1.2 times faster than Petya. How much time did he spend on the segment $D E$? The scale in the figure is not preserved.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_f265a9605ca3fc80eb59g-1.jpg?height=224&width=408&top_left_y=1437&top_left_x=885)",
      "answer": "1"
    },
    {
      "question": "15.1 If $144^{p}=10,1728^{q}=5$ and $a=12^{2 p-3 q}$, find the value of $a$.",
      "answer": "2"
    },
    {
      "question": "During a summer vacation, it rained 7 times in the morning or afternoon. If it rained in the morning, it did not rain in the afternoon. In total, there were 5 rainless mornings and 6 rainless afternoons. How long did the vacation last?",
      "answer": "9"
    },
    {
      "question": "Task 3. Determine the number of:\n\na) two-digit\n\nb) three-digit numbers\n\nthat are written only with even digits.",
      "answer": "20"
    },
    {
      "question": "7.4. Arseny sat down at the computer between 4 and 5 PM, when the hour and minute hands were pointing in opposite directions, and got up from it on the same day between 10 and 11 PM, when the hands coincided. How long did Arseny sit at the computer?",
      "answer": "6"
    },
    {
      "question": "Example 9. On five identical cards, letters are written: on two cards $l$, on the other three $i$. These cards are randomly laid out in a row. What is the probability that the word \"lilii\" will be formed?",
      "answer": "0.1"
    },
    {
      "question": "84. The probability of successfully performing an exercise for each of two athletes is 0.5. The athletes perform the exercise in turns, with each making two attempts. The one who performs the exercise first wins the prize. Find the probability of the athletes winning the prize.",
      "answer": "0.9375"
    },
    {
      "question": "4. How many orderings $\\left(a_{1}, \\ldots, a_{8}\\right)$ of $(1,2, \\ldots, 8)$ exist such that $a_{1}-a_{2}+a_{3}-a_{4}+a_{5}-a_{6}+a_{7}-a_{8}=0$ ?",
      "answer": "4608"
    },
    {
      "question": "Task B-3.4. A group of children found a wooden board in the shape of a quadrilateral and decided to use it for the game \"pick-a-point\". Since the target was of an unusual shape, they had to adapt the game rules. They asked for advice from Mark's older brother, a good mathematician. He measured, calculated, and wrote something... and came to an important conclusion. He established that there is a point $O$ equally distant from all sides of the quadrilateral. Then he divided the board into areas by drawing lines from point $O$ to all vertices $A, B, C, D$. He assigned points to the areas inversely proportional to their area. Which area yields the most points and why? Mark's brother added a condition that points are only awarded if the distance from the hit point $T$ to point $O$ is not greater than the distance from point $O$ to any side of the board. Determine the ratio of the area of the part of the board that does not yield points to the area of the part that does yield points.\n\nMark's brother measured these values $|A B|=5 \\mathrm{dm},|B C|=3 \\mathrm{dm},|A D|=4 \\mathrm{dm}$, $|\\angle B A D|=60^{\\circ}$.",
      "answer": "\\frac{7}{10\\pi}(5\\sqrt{3}-\\sqrt{5})-1"
    },
    {
      "question": "11.2. On the board, there are 4 numbers. Vasya multiplied the first of these numbers by $\\sin \\alpha$, the second - by $\\cos \\alpha$, the third - by $\\operatorname{tg} \\alpha$, and the fourth - by $\\operatorname{ctg} \\alpha$ (for some angle $\\alpha$) and obtained a set of the same 4 numbers (possibly in a different order). What is the maximum number of different numbers that could have been written on the board?",
      "answer": "3"
    },
    {
      "question": "[ Arithmetic. Mental calculation, etc.]\n\nA customer walks into a store, picks out an item costing 20 rubles, and gives the seller a hundred-ruble note. The seller looks - no change. He goes to the neighboring department, gets the hundred-ruble note changed. He gives the customer the item and the change. The customer leaves. Suddenly, the seller from the neighboring department comes over, bringing the hundred-ruble note. It's a counterfeit! Our seller gives him his own hundred-ruble note. How much did our poor seller lose in the end?",
      "answer": "100"
    },
    {
      "question": "3. Solve the system of equations:\n\n$$\n\\left\\{\\begin{array}{l}\n2 x^{2}+3 y+5=2 \\sqrt{2 z+5} \\\\\n2 y^{2}+3 z+5=2 \\sqrt{2 x+5} \\\\\n2 z^{2}+3 x+5=2 \\sqrt{2 y+5}\n\\end{array}\\right.\n$$",
      "answer": "-0.5"
    },
    {
      "question": "8. [6] Let $A B C$ be an equilateral triangle with side length 2 , and let $\\Gamma$ be a circle with radius $\\frac{1}{2}$ centered at the center of the equilateral triangle. Determine the length of the shortest path that starts somewhere on $\\Gamma$, visits all three sides of $A B C$, and ends somewhere on $\\Gamma$ (not necessarily at the starting point). Express your answer in the form of $\\sqrt{p}-q$, where $p$ and $q$ are rational numbers written as reduced fractions.",
      "answer": "\\sqrt{\\frac{28}{3}}-1"
    },
    {
      "question": "7. In the tetrahedron $S A B C$, $\\triangle A B C$ is an equilateral triangle. The projection of point $A$ onto the plane $S B C$ is $H$, which is the orthocenter of $\\triangle S B C$, and the dihedral angle $H-A B-C$ has a plane angle of $30^{\\circ}$. Given $S A=2 \\sqrt{3}$, the volume of the tetrahedron $S A B C$ is $\\qquad$.",
      "answer": "\\frac{9}{4}\\sqrt{3}"
    },
    {
      "question": "9. Let $B$ be a point on the circle centred at $O$ with diameter $A C$ and let $D$ and $E$ be the circumcentres of the triangles $O A B$ and $O B C$ respectively. Given that $\\sin \\angle B O C=\\frac{4}{5}$ and $A C=24$, find the area of the triangle $B D E$.",
      "answer": "45"
    },
    {
      "question": "11. The town of Coppersville uses some special coins. Edward has five 5 p coins, four $4 p$ coins, three $3 \\mathrm{p}$ coins, two $2 \\mathrm{p}$ coins and one $1 \\mathrm{p}$ coin. The smallest amount he cannot pay with his coins alone is $W$ p.\nWhat is the value of $W$ ?",
      "answer": "56"
    },
    {
      "question": "B3. Solve the irrational equation $\\sqrt{x^{2}+4}-\\sqrt{4 x}=0$.",
      "answer": "2"
    },
    {
      "question": "4  [ Proof by contradiction $]$\n\nInside an isosceles right triangle $ABC$ with hypotenuse $AB$, a point $M$ is taken such that angle $MAB$ is $15^{\\circ}$ greater than angle $MAC$, and angle $MCB$ is $15^{\\circ}$ greater than angle $MBC$. Find angle $BMC$.",
      "answer": "150"
    },
    {
      "question": "5. Given that $a, b$ are positive numbers, and $\\frac{a+b}{a^{2}+a b+b^{2}}=\\frac{4}{49}$, then the value of $a+b$ is",
      "answer": "16"
    },
    {
      "question": "7、Convert the fraction $\\frac{5}{7}$ into a decimal, then the sum of all digits from the 1st to the 100th position after the decimal point is $\\qquad$\n\nConvert the fraction $\\frac{5}{7}$ into a decimal, then the sum of all digits from the 1st to the 100th position after the decimal point is $\\qquad$",
      "answer": "446"
    },
    {
      "question": "8. In a unit square, if three of its four sides are colored with three different colors, then the unit square is called \"colorful\". As shown in the figure, a $1 \\times 3$ grid table contains 10 unit line segments in its grid lines. Now, we need to color these 10 line segments, with each line segment being colored red, yellow, or blue, such that all three unit squares are colorful. The number of such coloring ways is $\\qquad$ (answer in numerical form).",
      "answer": "5184"
    },
    {
      "question": "## Problem II - 5\n\nThe natural numbers 22, 23, and 24 have the following property: the exponents of the prime factors in their factorization are all odd:\n\n$$\n22=2^{1} \\cdot 11^{1} ; \\quad 23=23^{1} ; \\quad 24=2^{3} \\cdot 3^{1}\n$$\n\nWhat is the largest number of consecutive natural numbers that can have this property? Justify your answer.",
      "answer": "7"
    },
    {
      "question": "2. Two players take turns to write natural numbers on a board. The rules forbid writing numbers greater than $\\mathrm{p}$ and also divisors of previously written numbers. The player who has no move loses.\nDetermine which player has a winning strategy for $\\mathrm{p}=10$ and describe this strategy.",
      "answer": "6"
    },
    {
      "question": "1. Given vectors $\\overrightarrow{A P}=(1, \\sqrt{3}), \\overrightarrow{P B}=(-\\sqrt{3}, 1)$, then the angle between vector $\\overrightarrow{A P}$ and $\\overrightarrow{A B}$ is $\\qquad$",
      "answer": "\\frac{\\pi}{4}"
    },
    {
      "question": "## Task A-1.2. (4 points)\n\nLet $a$ be a real number. Determine the sum of all three solutions of the equation\n\n$$\nx^{3}-a^{2} x+a x-x+a^{2}-a=0\n$$",
      "answer": "0"
    },
    {
      "question": "7. [7] Suppose $a$ and $b$ be positive integers not exceeding 100 such that\n$$\na b=\\left(\\frac{\\operatorname{lcm}(a, b)}{\\operatorname{gcd}(a, b)}\\right)^{2} .\n$$\n\nCompute the largest possible value of $a+b$.",
      "answer": "78"
    },
    {
      "question": "The unit volume regular tetrahedron, $T$, is magnified by a factor of two from a point $P$ on one of its symmetry axes. The volume of the common part of the original and the magnified tetrahedron is $1 / 8$. How far is the center of $T$ from $P$? (H)",
      "answer": "\\frac{5}{6}\\sqrt[6]{243}"
    },
    {
      "question": "2. Half-year grade. By the end of the half-year, Vasya Petrov had the following grades in mathematics in his journal: $4,1,2,5,2$. Before assigning the half-year grade, the math teacher told Vasya:\n\n- Vasya, you can choose the method to determine your half-year grade. I offer two options. Method A: the arithmetic mean of the current grades, rounded to the nearest whole number. Method B: the median of the current grades.\n\nThe best method for Vasya is the one that gives him the highest grade for the half-year.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_853fe0fb72821c0a3a48g-10.jpg?height=328&width=534&top_left_y=1912&top_left_x=1366)\n\na) (from 6th grade. 1 point). Which method is the best for Vasya?\n\nb) (from 6th grade. 2 points). Then the teacher thought and added:\n\n- Keep in mind, Vasya, that if you manage to choose the best method for yourself, I will add two more fives to your journal, and then calculate the half-year grade.\n\nProve that under these conditions, Method A is not the best for Vasya.",
      "answer": "3"
    },
    {
      "question": "6. Let $g(x)$ be a strictly increasing function defined for all $x \\geq 0$. It is known that the range of $t$ satisfying\n$$\n\\mathrm{g}\\left(2 t^{2}+t+5\\right)<\\mathrm{g}\\left(t^{2}-3 t+2\\right)\n$$\nis $b<t<a$. Find $a-b$.",
      "answer": "2"
    },
    {
      "question": "7.1. In a hotel, rooms are arranged in a row in order from 1 to 10000. Masha and Alina checked into the hotel in two different rooms. The sum of the room numbers they are staying in is 2022, and the sum of the room numbers of all the rooms between them is 3033. In which room is Masha staying, if her room has a lower number than Alina's room?",
      "answer": "1009"
    },
    {
      "question": "13.076. The collection of corn from the fields of a livestock farm was 4340 centners. The following year, it is planned to obtain 5520 centners of corn by increasing the area by 14 hectares and increasing the yield by 5 centners per hectare. Determine the area occupied by corn and the yield in centners per hectare (the yield was less than 40 centners per hectare).",
      "answer": "124"
    },
    {
      "question": "A regular hexagon $S$ has an inscribed triangle $H$ such that one side of $H$ is parallel to one side of $S$. What is the maximum possible ratio of the area of $H$ to the area of $S$?",
      "answer": "\\frac{3}{8}"
    },
    {
      "question": "5.086 In how many ways can the players of two football teams be arranged in a single row so that no two players from the same team stand next to each other?",
      "answer": "2\\cdot(11!)^2"
    },
    {
      "question": "The director of the school decided to take a photo of the 2008 graduates. He arranged the students in parallel rows, all with the same number of students, but this arrangement was too wide for the camera's field of view. To solve this problem, the director noticed that it was enough to remove one student from each row and place them in a new row. This arrangement did not please the director because the new row had 4 students fewer than the others. He then decided to remove one more student from each row and place them in the new row he created, and found that this way all the rows had the same number of students, and finally took the photo. How many students appeared in the photo?",
      "answer": "24"
    },
    {
      "question": "13. On the Island of Knights and Rascals, Knights tell the truth except when they are wrong, and Rascals always lie. During a meeting, 40 islanders sit around a large round table, and each one says: \"I am next to a Knight and a Rascal.\" Knowing that 3 Knights present are wrong, how many Knights are at the meeting?",
      "answer": "27"
    },
    {
      "question": "Problem 10.1. An equilateral triangle with a side of 10 is divided into 100 small equilateral triangles with a side of 1. Find the number of rhombi consisting of 8 small triangles (such rhombi can be rotated).\n![](https://cdn.mathpix.com/cropped/2024_05_06_d4986512fdb0330805a0g-32.jpg?height=280&width=582&top_left_y=1198&top_left_x=434)",
      "answer": "84"
    },
    {
      "question": "6. What can you buy for a ruble? Nine boxes of matches cost 9 rubles and some kopecks, while ten such boxes cost 11 rubles and some kopecks. How much does one box cost?",
      "answer": "1"
    },
    {
      "question": "In a parallelogram, there are two circles. One of them, with a radius of 3, is inscribed in the parallelogram, and the second circle touches two sides of the parallelogram and the first circle. The distance between the points of tangency lying on the same side of the parallelogram is 3. Find the area of the parallelogram.",
      "answer": "\\frac{75}{2}"
    },
    {
      "question": "## 15. The Emir and His Oil\n\nMy emirate encompasses a vast desert, in the middle of which I have built my palace. It also includes territorial waters along the coastline, which I value greatly. The territorial waters are quite extensive (covering an area that reaches a third of the desert's area), and part of my oil field is located there. Knowing that the area of the desert with oil deposits is three times the area of the marine territory without oil and that one-seventh of my territory without oil is in the sea, could you determine exactly what fraction of my oil field is in the sea?",
      "answer": "\\frac{2}{5}"
    },
    {
      "question": "21st Putnam 1960 Problem A6 A player throws a fair die (prob 1/6 for each of 1, 2, 3, 4, 5, 6 and each throw independent) repeatedly until his total score ≥ n. Let p(n) be the probability that his final score is n. Find lim p(n). Solution",
      "answer": "\\frac{2}{7}"
    },
    {
      "question": "10.5. To a natural number $N$, the largest divisor of $N$ less than $N$ was added, and the result was a power of ten. Find all such $N$.\n\n(N. Agakhanov)",
      "answer": "75"
    },
    {
      "question": "72. Railway Network (I). There are five cities; no three of them lie on the same straight line. These cities need to be connected by a railway network consisting of four straight roads; at that, railway lines can be built one over another on viaducts.\n\nHow many such different railway networks exist?",
      "answer": "125"
    },
    {
      "question": "3. In city \"N\", there are 10 horizontal and 12 vertical streets, of which a pair of horizontal and a pair of vertical streets form the rectangular boundary of the city, while the rest divide it into blocks that are squares with a side length of 100 meters. Each block has an address consisting of two integers $(i ; j), i=1,2, . ., 9, j=1,2, \\ldots, 11-$ the numbers of the streets that bound it from below and from the left. Taxis transport passengers from one block to another, adhering to the following rules: 1) pick-up and drop-off can be made at any point on the boundary of the block at the passenger's request; 2) it is forbidden to enter inside the block; 3) transportation is carried out along the shortest path; 4) a fee of 1 coin is charged for every 100 meters traveled (rounding the distance to the nearest 100 meters in favor of the driver). How many blocks are there in the city? What is the maximum and minimum fare that the driver can charge the passenger for a ride from block $(7,1)$ to block $(2 ; 10)$ without violating the rules?",
      "answer": "99"
    },
    {
      "question": "8. From $1,2, \\cdots, 20$ choose 5 different numbers, the probability that at least two of them are consecutive is",
      "answer": "\\frac{232}{323}"
    },
    {
      "question": "1. In triangle ABC, the altitude to side BC, the perpendicular bisector of the arc $\\mathrm{ABC}$, and the perpendicular bisector of side $\\mathrm{AB}$ intersect at one point. Determine the measure of angle $\\mathrm{ABC}$.",
      "answer": "60"
    },
    {
      "question": "1. The numbers $62,63,64,65,66,67,68,69$ and 70 are divided by, in some order, the numbers $1,2,3,4,5,6,7,8$ and 9 , resulting in nine integers. The sum of these nine integers is $S$. What are the possible values of $S$ ?",
      "answer": "187"
    },
    {
      "question": "4. Xiao Ming's mother went to the store to buy soap, the unit prices of brand A soap and brand B soap are 6 yuan and 9 yuan respectively. If Xiao Ming's mother used all her money to buy brand A soap, she could buy 2 more bars than if she used all her money to buy brand B soap, and there would be no money left. So, how much money did Xiao Ming's mother bring?",
      "answer": "36"
    },
    {
      "question": "Let $(x, y, z, t) \\in\\left(\\mathbb{N}^{*}\\right)^{4}$ be a quadruplet satisfying $x+y=z+t$ and $2 x y=z t$, and such that $x \\geqslant y$. Find the largest value of $m$ such that $m \\leqslant x / y$ for sure.",
      "answer": "3+2\\sqrt{2}"
    },
    {
      "question": "5.1. Among all the simple fractions, where the numerator and denominator are two-digit numbers, find the smallest fraction greater than $\\frac{3}{4}$. In your answer, specify its numerator.",
      "answer": "73"
    },
    {
      "question": "[ Rectangles and squares. Properties and characteristics ] [ Congruent triangles. Criteria for congruence (other).]\n\nMitya is going to fold a square sheet of paper $ABCD$. Mitya calls a fold beautiful if side $AB$ intersects side $CD$ and the four resulting right triangles are congruent. Before this, Vanya chooses a random point $F$ on the sheet. Find the probability that Mitya can make a beautiful fold passing through point $F$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Calculate the remainder of the Euclidean division of $2022^{2023^{2024}}$ by 19.",
      "answer": "8"
    },
    {
      "question": "## Exercise 11.\n\n1) Alice wants to color the integers between 2 and 8 (inclusive) using $k$ colors. She wishes that if $m$ and $n$ are integers between 2 and 8 such that $m$ is a multiple of $n$ and $m \\neq n$, then $m$ and $n$ are of different colors. Determine the smallest integer $k$ for which Alice can color the integers $2,3, \\ldots, 8$ using $k$ colors.\n2) Alice wants to color the integers between 2 and 31 (inclusive) using $k$ colors. She wishes that if $m$ and $n$ are integers between 2 and 31 such that $m$ is a multiple of $n$ and $m \\neq n$, then $m$ and $n$ are of different colors. Determine the smallest integer $k$ for which Alice can color the integers $2,3, \\ldots, 31$ using $k$ colors.",
      "answer": "4"
    },
    {
      "question": "Determine the product of all positive integers $n$ for which $3(n!+1)$ is divisible by $2 n-5$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\nHowever, it seems there is a note or an additional request in Chinese at the end of your message. If you need the Chinese text translated as well, please let me know! For now, I'll assume you only need the provided Dutch text translated to English. If that's correct, the translation is as above.",
      "answer": "2016"
    },
    {
      "question": "1.5 When a piece of paper with digits is flipped upside down, the digits $0,1,8$ remain unchanged, the digits 6 and 9 swap, and other digits have no meaning when flipped. Find the number of nine-digit numbers that remain unchanged when the paper is flipped upside down.",
      "answer": "1500"
    },
    {
      "question": "5. In a football tournament where each team played against each other once, teams A, B, C, D, and E participated. For a win, a team received 3 points, for a draw 1 point, and for a loss 0 points. In the end, it turned out that teams A, B, C, D, and E each had 7 points. What is the maximum number of points that team $\\mathrm{E}$ could have?",
      "answer": "7"
    },
    {
      "question": "Task 1. Which of the numbers 723, 732, and 273 will decrease the most and by how much if in each of them the digits 7 and 3 swap places?",
      "answer": "396"
    },
    {
      "question": "## Properties and characteristics of a parallelogram [Trigonometric ratios in a right triangle]\n\nOn the side $C D$ of parallelogram $A B C D$ with an obtuse angle at vertex $D$, an equilateral triangle $C D E$ is constructed such that points $A$ and $E$ lie on opposite sides of line $C D$. It is known that the distances from points $D$ and $E$ to line $B C$ are 3 and 8, respectively, and the distance from point $E$ to line $A B$ is 13. Find the area of parallelogram $A B C D$.",
      "answer": "28\\sqrt{3}"
    },
    {
      "question": "The ratio between the number of men and women in the city of Campo Verde is $\\frac{2}{3}$. The average age of men is 37 years and that of women is 42 years. What is the average age of the inhabitants of Campo Verde?",
      "answer": "40"
    },
    {
      "question": "a) A point $M$ moves around the circumcircle of a triangle, and for each of its positions, we consider the projections of $M$ onto the sides of the triangle. We find that the points where $M$ is such that all three projections fall on the extensions of the sides (or exactly at the vertices) cover $\\frac{8}{10}$ of the circumference of the circle. What is the largest angle of the triangle?\n\nb) A point $N$ moves around the incircle of a triangle. We find that the points where $N$ is such that one of the three projections (formed in the same way as before) falls on the extension of a side cover $\\frac{8}{10}$ of the circumference of the circle. What is the largest angle of the triangle? (It is sufficient to give the angle with an accuracy of $0.1^{\\circ}$.)",
      "answer": "162"
    },
    {
      "question": "17. How many roots has the equation $\\sin x-\\log _{10} x=0$ ?",
      "answer": "3"
    },
    {
      "question": "## 114. Math Puzzle $11 / 74$\n\nIn a numerically controlled machine tool, the electronics account for $40 \\%$ of the total price.\n\nHow much does the electronics cost in relation to the actual machine?",
      "answer": "67"
    },
    {
      "question": "13.228. Two workers were assigned to manufacture a batch of identical parts. After the first worked for 2 hours and the second for 5 hours, it turned out that they had completed half of the entire work. Working together for another 3 hours, they found that they had 0.05 of the entire work left to complete. In what time interval can each of them, working separately, complete the entire work?",
      "answer": "12"
    },
    {
      "question": "3. Let $f(x)=\\frac{4^{x}}{4^{x}+2}$, then the sum\n$$\nf\\left(\\frac{1}{1001}\\right)+f\\left(\\frac{2}{1001}\\right)+f\\left(\\frac{3}{1001}\\right)+\\cdots+f\\left(\\frac{1000}{1001}\\right)\n$$\n\nis equal to . $\\qquad$",
      "answer": "500"
    },
    {
      "question": "10. (20 points) How many zeros does the number $4^{5^{6}}+6^{5^{4}}$ end with in its decimal representation?\n\n#",
      "answer": "5"
    },
    {
      "question": "1. Joey is standing at a corner of the rectangular field shown. He walks the perimeter of the field 5 times. How many metres does Joey walk?",
      "answer": "1500"
    },
    {
      "question": "$\\left.\\frac{\\text { [Inscribed and Circumscribed Circles ]}}{\\text { Rhombuses Properties and Characteristics }}\\right]$\n\nThe side of the rhombus is 8 cm, and the acute angle is $30^{\\circ}$. Find the radius of the inscribed circle.",
      "answer": "2"
    },
    {
      "question": "Determine the years of our century that exhibit the following property of their digits. Subtract the first digit from the second, the second from the third, and the third from the fourth, and form an arithmetic progression with these three differences.",
      "answer": "18811894"
    },
    {
      "question": "75. There are two identical urns. The first contains 2 black and 3 white balls, the second - 2 black and 1 white ball. First, an urn is chosen at random, and then one ball is drawn at random from it. What is the probability that a white ball will be selected?",
      "answer": "\\frac{7}{15}"
    },
    {
      "question": "João has a deck of 52 cards numbered from 1 to 52. A set of three cards is called lucky if the sum of the digits on each card is the same. What is the minimum number of cards João has to take from the deck, without looking, so that among the cards he takes there are necessarily three cards that form a set of lucky cards?",
      "answer": "26"
    },
    {
      "question": "1614. 600 corn seeds are planted with a probability of 0.9 for each seed to germinate. Find the boundary of the absolute value of the deviation of the frequency of germinated seeds from the probability $p=0.9$, if this boundary is to be guaranteed with a probability $P=0.995$.",
      "answer": "0.034"
    },
    {
      "question": "10. Lottery. (From 7th grade, 3 points) It so happened that the Absent-Minded Scholar had only 20 rubles left, and he needed to buy a bus ticket to go home. The ticket costs 45 rubles. Near the bus stop, instant lottery tickets are sold for exactly 10 rubles each. With a probability of \\( p = 0.1 \\), a ticket contains a win of 30 rubles, and there are no other prizes. The Scholar decided to take a risk, as he had nothing to lose. Find the probability that the Absent-Minded Scholar will be able to win enough money to buy a bus ticket.",
      "answer": "\\frac{14}{419}"
    },
    {
      "question": "8. On a circular track that is 660 m long, a relay race is being held, with each stage being $150 \\mathrm{~m}$. The start and finish are in the same place. What is the minimum number of stages that can be in this relay?",
      "answer": "22"
    },
    {
      "question": "## Task A-4.5.\n\nDetermine (if it exists) the largest natural number that cannot be represented as the sum of some, not necessarily distinct, elements of the set $\\{135,136,137, \\ldots, 144\\}$.",
      "answer": "2024"
    },
    {
      "question": "2. Find $x+y$, given that $x^{2}-y^{2}=10$ and $x-y=2$.",
      "answer": "5"
    },
    {
      "question": "Determine the largest constant $C>0$ satisfying the following property:\n\nFor any 2023-tuple $\\left(a_{1}, a_{2}, \\ldots, a_{2023}\\right)$ of strictly positive real numbers that are pairwise distinct, the following inequality holds:\n\n$$\n\\frac{a_{1}}{\\left|a_{2}-a_{3}\\right|}+\\frac{a_{2}}{\\left|a_{3}-a_{4}\\right|}+\\ldots+\\frac{a_{2023}}{\\left|a_{1}-a_{2}\\right|}>C\n$$",
      "answer": "1012"
    },
    {
      "question": "2 Let convex quadrilateral $A B C D$ satisfy $A B=A D=1, \\angle A=160^{\\circ}, \\angle C=$ $100^{\\circ}$, then the range of the length of diagonal $A C$ is $\\qquad$ .",
      "answer": "1"
    },
    {
      "question": "1. On Misfortune Island, a table tennis tournament was held in a round-robin format (i.e., everyone played against everyone else once). After each match, both participants approached the Chief Referee separately and reported the result. Among the participants, there were only three types of people: knights, who always tell the truth, liars, who always lie, and politicians, who always say what is advantageous to them - each politician claimed after each match that they had won.\n\nAfter the tournament, it turned out that the Chief Referee, incognito, had attended several matches, as a result of which he was lucky enough to not only be able to determine the results of all matches but also to find out that there were exactly six politicians. What is the minimum number of matches he could have attended?\n\n(There are no draws in table tennis. The Chief Referee did not know in advance which players belonged to which type, and did not receive information in any other way except as described in the condition)",
      "answer": "16"
    },
    {
      "question": "21. (10 points) When Xiao Hong was organizing her change purse, she found that there were a total of 25 coins with denominations of 1 cent, 2 cents, and 5 cents, with a total value of 0.60 yuan. Then, the maximum number of 5-cent coins is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\nNote: The instruction at the end is not part of the translation but is included to clarify the request. The actual translation is provided above.",
      "answer": "8"
    },
    {
      "question": "1. The price of a mathematics book in bookstore A at the beginning of January 2012 was higher than the price of the same book in bookstore B. On February 15, 2012, the price of the book in bookstore A decreased by $30\\%$, while the price of the same book in bookstore B increased by $30\\%$. Then, on March 15, 2012, the price of the book in bookstore A increased by $30\\%$, and the price of the book in bookstore B decreased by $30\\%$. The difference in the prices of the book after the changes on March 15, 2012, was 273 denars. By how much was the price of the mathematics book in bookstore A higher than the price of the same book in bookstore B at the beginning of January 2012?",
      "answer": "300"
    },
    {
      "question": "## Task B-1.4.\n\nDetermine the smallest natural number $n$ for which the number $2565 \\cdot n$ is a square of some natural number.",
      "answer": "285"
    },
    {
      "question": "A car and a minivan drive from Alphaville to Betatown. The car travels at a constant speed of $40 \\mathrm{~km} / \\mathrm{h}$ and the minivan travels at a constant speed of $50 \\mathrm{~km} / \\mathrm{h}$. The minivan passes the car 10 minutes before the car arrives at Betatown. How many minutes pass between the time at which the minivan arrives in Betatown and the time at which the car arrives in Betatown?",
      "answer": "2"
    },
    {
      "question": "Example 1 Calculate $\\left|\\frac{(\\sqrt{3}+\\sqrt{2} i)(\\sqrt{5}+\\sqrt{2} i)(\\sqrt{5}+\\sqrt{3} i)}{(\\sqrt{2}-\\sqrt{3} i)(\\sqrt{2}-\\sqrt{5} i)}\\right|$\n(1983 Shanghai High School Competition)",
      "answer": "8"
    },
    {
      "question": "3. On the surface of the cube $A B C D E F G H$, a broken line composed of four identical segments is constructed in the faces $A B F E, B C G F, C D H G$ and $G H E F$, which starts from vertex $A$ and ends at vertex $E$. Determine in what ratio this broken line divides the edge $C G$.",
      "answer": "\\frac{1}{2}(\\sqrt{5}-1)"
    },
    {
      "question": "8. (10 points) The difference between the minimum value of the sum of the squares of ten different odd numbers and the remainder when this minimum value is divided by 4 is $\\qquad$\n(Note: The product of the same two natural numbers is called the square of this natural number, such as $1 \\times 1=1^{2}, 2 \\times 2=2^{2}, 3 \\times 3$ $=3^{3}$, and so on)",
      "answer": "1328"
    },
    {
      "question": "(7) Given positive integers $x, y, z$ satisfy the condition $x y z=(14-x)(14-y)(14-$\n$z)$, and $x+y+z<28$, then the maximum value of $x^{2}+y^{2}+z^{2}$ is $\\qquad$.",
      "answer": "219"
    },
    {
      "question": "9. Bear Da painted the outer surface of a cubic wooden block blue, then divided it into many smaller cubic wooden blocks of the same size, among which 156 smaller cubic wooden blocks have two blue faces. Therefore, the number of smaller cubic wooden blocks with no blue faces is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "2197"
    },
    {
      "question": "$2 \\cdot 34$ Let $S$ be a subset of $\\{1,2, \\cdots, 1989\\}$, and any two numbers in $S$ do not have a difference of 4 or 7. How many elements can $S$ have at most?",
      "answer": "905"
    },
    {
      "question": "Problem 8.5. In a convex quadrilateral $A B C D$, the bisectors of angles $A$ and $C$ are parallel, and the bisectors of angles $B$ and $D$ intersect at an angle of $46^{\\circ}$, as shown in the figure. How many degrees does the acute angle between the bisectors of angles $A$ and $B$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_191a5994d6b917032c66g-03.jpg?height=490&width=616&top_left_y=94&top_left_x=419)",
      "answer": "67"
    },
    {
      "question": "3. The two diagonals of a quadrilateral are perpendicular. The lengths of the diagonals are 14 and 30 . What is the area of the quadrilateral?",
      "answer": "210"
    },
    {
      "question": "6. On the table, there are three cones standing on their bases, touching each other. The radii of their bases are 72, 28, and 28, and the angles at their vertices are $-\\frac{\\pi}{3}$, $\\frac{2 \\pi}{3}$, and $\\frac{2 \\pi}{3}$ respectively (the angle at the vertex of a cone is the angle between its generators in the axial section). A sphere is suspended above the table, touching all the cones. It turns out that the center of the sphere is equidistant from all points of contact with the cones. Find the radius of the sphere.",
      "answer": "\\frac{\\sqrt{3}+1}{2}"
    },
    {
      "question": "Question 179, As shown in the figure, in the Cartesian coordinate system, the ellipse $\\Gamma: \\frac{x^{2}}{2}+y^{2}=1$ has left and right foci $F_{1}$ and $F_{2}$, respectively. Let $P$ be a point on $\\Gamma$ in the first quadrant, and the extensions of $P F_{1}$ and $P F_{2}$ intersect $\\Gamma$ at points $Q_{1}$ and $Q_{2}$, respectively. Let $r_{1}$ and $r_{2}$ be the inradii of $\\triangle P F_{1} Q_{2}$ and $\\triangle P F_{2} Q_{1}$, respectively. Find the maximum value of $r_{1}-r_{2}$.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "5 married couples gather at a party. As they come in and greet each other, various people exchange handshakes - but, of course, people never shake hands with themselves or with their own respective spouses. At the end of the party, one woman goes around asking people how many hands they shook, and she gets nine different answers. How many hands did she herself shake?",
      "answer": "4"
    },
    {
      "question": "22. Find the value of $\\frac{\\tan 40^{\\circ} \\tan 60^{\\circ} \\tan 80^{\\circ}}{\\tan 40^{\\circ}+\\tan 60^{\\circ}+\\tan 80^{\\circ}}$.",
      "answer": "1"
    },
    {
      "question": "Fomin D.\n\nOn the board, the numbers $1, 1/2, 1/3, \\ldots, 1/100$ are written. We choose two arbitrary numbers $a$ and $b$ from those written on the board, erase them, and write the number\n\n$a + b + ab$. We perform this operation 99 times until only one number remains. What is this number? Find it and prove that it does not depend on the sequence of number selections.",
      "answer": "100"
    },
    {
      "question": "3.2. Znayka cut out a semicircle from paper. Neznaika marked a point $C$ on the diameter $A B$ of this semicircle and cut off two semicircles from Znayka's semicircle with diameters $A C$ and $C B$. Find the area of the remaining figure if the length of the part of the chord lying inside it, passing through point $C$ perpendicular to $A B$, is 8. If necessary, round your answer to two decimal places.",
      "answer": "50.27"
    },
    {
      "question": "2. At the Olympiad, for each solved problem, one could receive 3, 8, or 10 points. Vasya scored 45 points. What is the smallest number of problems he could have solved? (It is necessary to explain why he could not have solved fewer problems.)",
      "answer": "6"
    },
    {
      "question": "Point $D$ lies on side $CB$ of the right triangle $ABC\\left(\\angle C=90^{\\circ}\\right)$, and $AB=5, \\angle ADC=\\arccos$ $\\frac{1}{\\sqrt{10}}, DB=\\frac{4 \\sqrt{10}}{3}$. Find the area of triangle $ABC$.",
      "answer": "\\frac{15}{4}"
    },
    {
      "question": "\\section*{Problem 4 - 051044}\n\nCalculate the difference \\(D\\) between the sum of the squares of all even natural numbers \\(\\leq 100\\) and the sum of the squares of all odd natural numbers \\(<100\\)!",
      "answer": "5050"
    },
    {
      "question": "G8.2 From 1 to 121 , there are $b$ numbers which are not divisible by 5 nor 7 . Find $b$.",
      "answer": "83"
    },
    {
      "question": "14. 2011 cards are arranged in a row on a table. One of the numbers ' 1 ', ' 2 ' and ' 3 ' is printed on each card. It is found that there is at least one card between any two cards labelled ' 1 ', at least two cards between any two cards labelled ' 2 ', and at least three cards between any two cards labelled ' 3 '. If the smallest and greatest possible numbers of cards labelled ' 3 ' are $m$ and $M$ respectively, find the value of $m+M$.\n2011 cards are placed in a row on a table, each card has one of the numbers '1', '2', or '3' written on it. It is known that there is at least one card between any two cards labelled '1', at least two cards between any two cards labelled '2', and at least three cards between any two cards labelled '3'. If the smallest and greatest possible numbers of cards labelled '3' are $m$ and $M$ respectively, find the value of $m+M$.",
      "answer": "1005"
    },
    {
      "question": "2. First method: Since $x_{1}^{3}-2015 x_{1}+2016=0$ and $x_{2}^{3}-2015 x_{2}+2016=0$, then $x_{1}^{3}-x_{2}^{3}=2015 \\cdot\\left(x_{1}-x_{2}\\right)$. Therefore, $x_{1}^{2}+x_{1} x_{2}+x_{2}^{2}=2015$.\n\nSecond method: By Vieta's theorem (but then it needs to be justified that there are three distinct roots): $x_{1}+x_{2}+x_{3}=0, x_{1} x_{2} x_{3}=-2016$. Therefore,\n\n$$\n\\begin{aligned}\nx_{1}^{2}+x_{1} x_{2}+x_{2}^{2}=\\left(x_{1}+x_{2}\\right)^{2}-x_{1} x_{2}=\\left(-x_{3}\\right)^{2}+\\frac{2016}{x_{3}} & \\\\\n=\\frac{x_{3}^{3}+2016}{x_{3}} & =\\frac{2015 x_{3}}{x_{3}}=2015\n\\end{aligned}\n$$",
      "answer": "2015"
    },
    {
      "question": "## Task 2 - V10922\n\nAccording to our seven-year plan, the industrial production of the German Democratic Republic will increase significantly. The total industrial production will grow by 88% from 1958 to 1965. The production of means of production (i.e., raw materials, machinery, equipment for industry, agriculture, and transportation, etc.) will increase by 95%, while the production of consumer goods (i.e., goods intended for the needs of the population) will increase by 77%.\n\nWhat percentage of the total industrial production did the production of means of production account for in 1958? What percentage will it account for in 1965?",
      "answer": "61.1"
    },
    {
      "question": "To punish his class for indiscipline, Professor Zerus decided to deduct from each student's monthly grade a percentage equal to the test score, that is, whoever scored 60 will have a 60% deduction from their grade, whoever scored 20 will have a 20% deduction, and so on. The maximum monthly grade is 100.\n\n(a) Who will end up with the highest grade?\n\n(b) And the lowest?\n(c) Students who scored well complained that they will end up with the same grade as those who scored poorly. Do they have a point?",
      "answer": "25"
    },
    {
      "question": "I2.3 If $c$ is the minimum value of the quadratic function $y=x^{2}+4 x+b$, find the value of $c$.",
      "answer": "100"
    },
    {
      "question": "3. Given an isosceles triangle \\(ABC\\) with base \\(BC\\). On the extension of side \\(AC\\) beyond point \\(C\\), point \\(K\\) is marked, and a circle is inscribed in triangle \\(ABK\\) with center at point \\(I\\). A circle passing through points \\(B\\) and \\(I\\) is tangent to line \\(AB\\) at point \\(B\\). This circle intersects segment \\(BK\\) again at point \\(L\\). Find the angle between lines \\(IK\\) and \\(CL\\). Answer: \\(90^{\\circ}\\).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_c43b1f7fb831d10d42ecg-02.jpg?height=496&width=820&top_left_y=483&top_left_x=638)",
      "answer": "90"
    },
    {
      "question": "## 39. Square Plot\n\nMathieu has a square plot of land. Surrounding this plot is an alley of constant width with an area of $464 \\mathrm{~m}^{2}$. Walking around his plot, Mathieu noticed that the difference in length between the outer and inner edges of the alley is 32 m.\n\nWhat is the total area of Mathieu's plot, including the alley?",
      "answer": "1089"
    },
    {
      "question": "10. Given that point $P$ lies on the hyperbola $\\frac{x^{2}}{16}-\\frac{y^{2}}{9}=1$, and the distance from $P$ to the right directrix of this hyperbola is exactly the arithmetic mean of the distances from $P$ to the two foci of this hyperbola. Then, the x-coordinate of $P$ is $\\qquad$ .",
      "answer": "-\\frac{64}{5}"
    },
    {
      "question": "14. A rectangular prism with integer edge lengths is painted red on all its surfaces, and then it is cut into small cubes with edge lengths of 1. Among them, there are 40 small cubes with two red faces, and 66 small cubes with one red face. What is the volume of this rectangular prism?",
      "answer": "150"
    },
    {
      "question": "Problem 1. Find the least positive integer $n$ with the following property: if $n$ distinct sums of the form $x_{p}+x_{q}+x_{r}, 1 \\leq p<q<r \\leq 5$, equal 0 , then $x_{1}=x_{2}=x_{3}=x_{4}=x_{5}=0$.\n\nSava Grozdev, Svetlozar Doychev",
      "answer": "7"
    },
    {
      "question": "13 The function $f(x)=2 \\sin \\omega x(\\omega>0)$ is monotonically increasing on $\\left[0, \\frac{\\pi}{4}\\right]$, and the maximum value on this interval is $\\sqrt{3}$. Then, $\\omega=$ $\\qquad$ .",
      "answer": "\\frac{4}{3}"
    },
    {
      "question": "Mr. Krbec with his cat Kokeš were selling tickets at Kulikov Castle. On Saturday, they sold 210 children's tickets for 25 groschen each and also some adult tickets for 50 groschen each. In total, they earned 5950 groschen that day.\n\nHow many adult tickets did they sell?\n\n(M. Krejčová)\n\nHint. How much did they earn from the adult tickets?",
      "answer": "14"
    },
    {
      "question": "43.3. In a circle, mutually perpendicular diameters $A B$ and $C D$ are drawn, and an arbitrary point $M$ is taken on the arc $A C$. Find $M B + M D$, if $M A + M C = a$.\n\n$$\n\\text { (9-10 grades) }\n$$",
      "answer": "(1+\\sqrt{2})"
    },
    {
      "question": "19. Let $O$ be the origin, $A_{1}, A_{2}, A_{3}, \\ldots$ be points on the curve $y=\\sqrt{x}$ and $B_{1}, B_{2}, B_{3}, \\ldots$ be points on the positive $x$-axis such that the triangles $O B_{1} A_{1}, B_{1} B_{2} A_{2}, B_{2} B_{3} A_{3}, \\ldots$ are all equilateral, with side lengths $l_{1}, l_{2}, l_{3}, \\ldots$ respectively. Find the value of $l_{1}+l_{2}+l_{3}+\\cdots+l_{2005}$.\n(2 marks)\nLet $O$ be the origin, $A_{1}, A_{2}, A_{3}, \\ldots$ be points on the curve $y=\\sqrt{x}$ and $B_{1}, B_{2}, B_{3}, \\ldots$ be points on the positive $x$-axis such that the triangles $O B_{1} A_{1}, B_{1} B_{2} A_{2}, B_{2} B_{3} A_{3}, \\ldots$ are all equilateral, with side lengths $l_{1}, l_{2}, l_{3}, \\ldots$ respectively. Find the value of $l_{1}+l_{2}+l_{3}+\\cdots+l_{2005}$.",
      "answer": "\\frac{4022030}{3}"
    },
    {
      "question": "8. If positive integers $a, b, c$ satisfy $2017 \\geqslant 10 a \\geqslant 100 b \\geqslant 1000 c$, then the number of tuples $(a, b, c)$ is\n$\\qquad$ .",
      "answer": "574"
    },
    {
      "question": "Three concentric circles have radii of 1, 2, and 3 units, respectively. A point is chosen on each of the three circles such that they form the vertices of an equilateral triangle. What can be the side length of this equilateral triangle?",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "8. (10 points) As shown in the figure, the area of trapezoid $ABCD$ is 320, the area of parallelogram $CDEF$ is 240, and the area of triangle $OCD$ is $\\qquad$\n\n保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "45"
    },
    {
      "question": "10.5. On side $AB$ of triangle $ABC$, point $K$ is marked, and on side $AC$, point $M$ is marked. Segments $BM$ and $CK$ intersect at point $P$. It turns out that angles $APB$, $BPC$, and $CPA$ are each $120^{\\circ}$, and the area of quadrilateral $AKPM$ is equal to the area of triangle $BPC$. Find angle $BAC$.",
      "answer": "60"
    },
    {
      "question": "51. Given $\\triangle A B C$ with three interior angles $\\angle A, \\angle B, \\angle C$ and $\\alpha=\\angle A+\\angle B, \\beta=\\angle C+\\angle A, \\gamma$ $=\\angle C+\\angle B$, the maximum number of acute angles among $\\alpha, \\beta, \\gamma$ is . $\\qquad$",
      "answer": "1"
    },
    {
      "question": "10. In 2021, the difference between the digits of any two adjacent positions does not exceed 2 (subtracting the smaller number from the larger one). If all four-digit numbers where the difference between the digits of any two adjacent positions does not exceed 2 are arranged in ascending order, then 2021 is the $\\qquad$th number in this sequence.",
      "answer": "84"
    },
    {
      "question": "4. If the inequality\n$$\n\\left|a x^{2}+b x+a\\right| \\leqslant x\n$$\n\nholds for all $x \\in[1,2]$, then the maximum value of $3 a+b$ is\n$\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "176 Given the parabola $y^{2}=x, A B$ is a chord passing through the focus $F$, $A$ and $B$ are in the $\\mathrm{I}$ and $\\mathrm{N}$ quadrants respectively, if $A F-F B=1$, then the slope of $A B$ $=$ $\\qquad$",
      "answer": "\\frac{1+\\sqrt{5}}{2}"
    },
    {
      "question": "5. In a hat, there are 11 slips of paper with numbers $0,1,2,3,4,5,6,7,8,9,10$ (one with each number). Each of the eleven students takes one slip. Then they announce in alphabetical order the sentence \"so far, at least $k$ students have told a lie,\" where $k$ is the number on the drawn slip. How many of these 11 statements can be true at most?",
      "answer": "6"
    },
    {
      "question": "## Task 3 - V00803\n\nI drop a ball. It bounces up to $\\frac{2}{3}$ of its falling height. It falls again and bounces the second time $\\frac{5}{8}$ of the first bounce height.\n\nCalculate from what height I dropped the ball if it bounced $45 \\mathrm{~cm}$ less high the second time than the first time!",
      "answer": "180"
    },
    {
      "question": "1. Find the minimum value of the expression $\\frac{25 x^{2} \\sin ^{2} x+16}{x \\sin x}$ for $0<x<\\pi$.",
      "answer": "40"
    },
    {
      "question": "39. How far is it to school?\n\nAll the time while Alice and the Griffin were solving the previous problem, the Tortoise Quasi was weeping inconsolably.\n\n- Can you tell me what is sad about this problem? - the Griffin barked at him angrily. - After all, the passenger caught up with the train. Or did I misunderstand something?\n- Everything is correct, - agreed the Tortoise Quasi, - but we don't know what happened afterward! After all, judging by everything, the train could have derailed!\n- No, just think about it! - Alice got angry. - First, they come up with all sorts of nonsense, and then they make a sad story out of everything!\n\nThe Tortoise Quasi did not answer, only grabbing his head with his front paws.\n\n- Alright, - he said finally. - I will tell you a sad story. One morning, a boy had to go to school...\n- That's sad! - agreed the Griffin.\n- No, no! That's not sad, - objected the Tortoise Quasi. - The saddest part is still to come.\n\nAlice and the Griffin listened to the whole story attentively but found nothing sad in it.\n\n- So, - continued the Tortoise Quasi, - the father said to the boy: \"Hurry up, or you'll be late for school!\" - \"I won't be late, Dad! - the boy replied. - I have everything calculated precisely. If I walk at a speed of four miles per hour, I will be five minutes late for the start of classes, but if I walk at a speed of five miles per hour, I will arrive ten minutes before the start of classes.\"\n\nHow far is it to school?",
      "answer": "5"
    },
    {
      "question": "One, (25 points) Given the family of curves $2(2 \\sin \\theta-\\cos \\theta+3) x^{2}-(8 \\sin \\theta+\\cos \\theta+1) y=0, \\theta$ being the parameter, find the maximum length of the chord intercepted by the line $y=2 x$ on this curve.\n\n---\n\nThe translation retains the original text's line breaks and formatting.",
      "answer": "8\\sqrt{5}"
    },
    {
      "question": "Third question: For all real numbers $x_{1}, x_{2}, \\ldots, x_{60} \\in[-1,1]$, find the maximum value of $\\sum_{i=1}^{60} x_{i}^{2}\\left(x_{i+1}-x_{i-1}\\right)$, where $x_{0}=x_{60}, x_{61}=x_{1}$.",
      "answer": "40"
    },
    {
      "question": "2. For the number $a$, the equality $a+\\frac{1}{a}=1$ is satisfied. Calculate the value of\n\n$$\na^{5}+\\frac{1}{a^{5}}\n$$",
      "answer": "1"
    },
    {
      "question": "4. Wu Yu wrote three letters and three envelopes, and he needs to put each letter into the corresponding envelope, with only one letter per envelope. The number of all possible scenarios where at least one of the three letters is placed incorrectly is $\\qquad$ kinds.",
      "answer": "5"
    },
    {
      "question": "6 books are aligned in a library. How many ways are there to arrange them.",
      "answer": "720"
    },
    {
      "question": "## Task 25/62\n\nAn apprentice is supposed to count 1000 balls with a diameter of $1 \\mathrm{~cm}$ in a ball bearing factory. To speed up this task, he takes a container with inner dimensions of $10 \\mathrm{~cm} \\times 10 \\mathrm{~cm} \\times 10$ $\\mathrm{cm}$; he lays the first layer neatly and then continues to fill it.\n\nIn the end, he finds that, contrary to his expectations, the inner space of the container is not completely filled. He therefore counts the balls. Surprisingly, there are more than a thousand.\n\nHow many were there, and how many centimeters were missing from the top layer of balls to the edge of the container?",
      "answer": "1186"
    },
    {
      "question": "46. A U-shaped number is a special type of three-digit number where the units digit and the hundreds digit are equal and larger than the tens digit, such as 818 is a $\\cup$ shaped number. There are $\\qquad$ U-shaped numbers.",
      "answer": "45"
    },
    {
      "question": "15. Given the ellipse $E: \\frac{x^{2}}{a^{2}}+\\frac{y^{2}}{b^{2}}=1(a>b>0)$, its minor axis length is 4, and the eccentricity is $e_{1}$. The hyperbola $\\frac{x^{2}}{m}-\\frac{y^{2}}{n}=1(m>0$, $n>0)$ has asymptotes $y= \\pm x$, and the eccentricity is $e_{2}$, and $e_{1} \\cdot e_{2}=1$.\n(1) Find the equation of the ellipse $E$;\n(2) Let the right focus of the ellipse $E$ be $F$, and draw a line through the point $G(4,0)$ with a non-zero slope intersecting the ellipse $E$ at points $M, N$. Let the slopes of the lines $F M$ and $F N$ be $k_{1}, k_{2}$, respectively. Determine whether $k_{1}+k_{2}$ is a constant value. If it is a constant value, find the value; if not, explain the reason.",
      "answer": "0"
    },
    {
      "question": "Problem 6. (30 points) Ten balls of the same radius are arranged in the form of a triangular pyramid such that each ball touches at least three others. Find the radius of the sphere in which the pyramid of balls is inscribed, if the radius of the sphere inscribed in the center of the pyramid of balls, touching six identical balls, is $\\sqrt{2}-1$.\n\n#",
      "answer": "\\sqrt{6}+1"
    },
    {
      "question": "Given\n$$\nf(x y)=f(x)+f(y)\n$$\n\nand $f(2)=1$. Find the value of $f\\left(\\frac{1}{64}\\right)$.",
      "answer": "-6"
    },
    {
      "question": "7.5. The following numbers are written on the wall: $1,3,4,6,8,9,11,12,16$. Four of these numbers were written by Vova, four numbers were written by Dima, and one number was simply the house number of the local police officer. The police officer found out that the sum of the numbers written by Vova is three times the sum of the numbers written by Dima. Calculate the house number.",
      "answer": "6"
    },
    {
      "question": "Task B-3.3. Determine the fundamental period of the function $f(x)=8 \\sin ^{2} x \\cos ^{2} x-2 \\cos ^{2} 2 x$.",
      "answer": "\\frac{\\pi}{2}"
    },
    {
      "question": "Shapavaooov A.B.\n\na) On each of the fields of the top and bottom horizontal rows of an $8 \\times 8$ chessboard, there is a token: white ones at the bottom, black ones at the top. In one move, it is allowed to move any token to an adjacent free cell vertically or horizontally. What is the minimum number of moves required to achieve a configuration where all black tokens are at the bottom and all white tokens are at the top?\n\nb) The same question for a $7 \\times 7$ board.",
      "answer": "120"
    },
    {
      "question": "On a quiz, every question is solved by exactly four students, every pair of questions is solved by exactly one student, and none of the students solved all of the questions. Find the maximum possible number of questions on the quiz.",
      "answer": "13"
    },
    {
      "question": "5. (8 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $12$, and $\\cos \\alpha = \\frac{3}{4}$?",
      "answer": "18"
    },
    {
      "question": "8. (5 points) As shown in the figure, in the right triangle $\\triangle ABC$, the hypotenuse $AB=10, BC=5, \\angle ABC=60^{\\circ}$. With point $B$ as the center, $\\triangle ABC$ is rotated clockwise by $120^{\\circ}$, and points $A$ and $C$ reach points $E$ and $D$, respectively. The area swept by side $AC$ (i.e., the shaded area in the figure) is $\\qquad$. (Take $\\pi$ as 3)",
      "answer": "75"
    },
    {
      "question": "8. For a regular 2017-gon $A_{1} A_{2} \\cdots A_{2017}$ inscribed in the unit circle $O$, if any two distinct vertices $A_{i}, A_{j}$ are chosen, then the probability that $\\overrightarrow{O A_{i}} \\cdot \\overrightarrow{O A_{j}}>\\frac{1}{2}$ is $\\qquad$.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "02.2. In two bowls there are in total $N$ balls, numbered from 1 to $N$. One ball is moved from one of the bowls to the other. The average of the numbers in the bowls is increased in both of the bowls by the same amount, $x$. Determine the largest possible value of $x$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Problem 3. Let $\\triangle A B C$ with $m(\\measuredangle A)=90^{\\circ}$. Construct the square $B D E C$ in the half-plane delimited by $B C$ that does not contain $A$. The angle bisector of $\\angle A$ intersects the sides $[B C]$ and $[D E]$ at $F$ and $G$ respectively. If $|A B|=16 \\mathrm{~cm}$ and $|A C|=4 \\mathrm{~cm}$, calculate the area of the quadrilateral $B D G F$.",
      "answer": "136"
    },
    {
      "question": "As shown in the figure, in an $8 \\times 8$ square grid, there are two bugs at points $A$ and $B$ (the bug at point $A$ is called bug $a$, and the bug at point $B$ is called bug $b$). Each bug moves 1 grid (choosing one of the four directions: up, down, left, or right). If bug $b$ moves two times, and bug $a$ moves three times, the number of ways in which the distance from bug $b$ to point $A$ is less than or equal to the distance from bug $a$ to point $A$ is $\\qquad$ ways.",
      "answer": "748"
    },
    {
      "question": "5. In the rectangular prism $A B C D-E F G H$ shown in the figure, let $P$ be the center of rectangle $E F G H$, and line segment $A P$ intersects plane $B D E$ at point $Q$. If $A B=3, A D=2, A E=$ 1, then $P Q=$ $\\qquad$.",
      "answer": "\\frac{\\sqrt{17}}{4}"
    },
    {
      "question": "60. A technology company produced 17 units and 15 units of the same model of testing equipment in locations A and B, respectively, all to be transported to venues $A$ and $B$ of a major sports event, with 18 units going to venue $A$ and 14 units to venue $B$. The transportation costs to venues $A$ and $B$ are as follows:\n（1）Assume that $x$ units of equipment are transported from location A to venue $A$. Please fill in the table below and find the function relationship between the total transportation cost $y$ (yuan) and $x$ (units);\n（2）When $x$ is $\\qquad$, the total transportation cost is minimized, with the minimum value being $\\qquad$",
      "answer": "19900"
    },
    {
      "question": "2. The car traveled half of the distance at a speed of 60 km/h, then one third of the remaining distance at a speed of 120 km/h, and the remaining distance at a speed of 80 km/h.\n\nFind the average speed of the car during this trip. Give your answer in km/h.",
      "answer": "72"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$$\n\\lim _{n \\rightarrow \\infty}\\left(\\frac{1+5+9+13+\\ldots+(4 n-3)}{n+1}-\\frac{4 n+1}{2}\\right)\n$$",
      "answer": "-\\frac{7}{2}"
    },
    {
      "question": "3. (3 points) Definition: $a \\oplus b=a+b+a b$, then $(2 \\oplus 3) \\oplus 4$ is\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "59"
    },
    {
      "question": "38. A flour mill has 20 workers. To gain more profit, it adds a noodle processing project, using the flour produced by the mill to process into noodles (producing $1 \\mathrm{~kg}$ of noodles requires $1 \\mathrm{~kg}$ of flour). It is known that each worker can produce an average of $600 \\mathrm{~kg}$ of flour or process $400 \\mathrm{~kg}$ of noodles per day. Selling flour directly yields a profit of 0.2 yuan per kilogram, while processing it into noodles and selling yields a profit of 0.6 yuan per kilogram. If each worker can only do one job per day, and other factors are not considered, when $\\qquad$ workers process noodles, the mill's total profit per day is maximized, and the maximum profit is $\\qquad$ yuan.",
      "answer": "2880"
    },
    {
      "question": "On an $n \\times k$ chessboard ( $n, k \\geq 2$ ), some figures are placed. (A figure can only stand on a square, and at most one figure can stand on each square.) We know that on the 4 squares where any 2 rows and 2 columns intersect, there is always exactly 1 or 3 figures. What are the limits within which the number of figures can vary?",
      "answer": "13"
    },
    {
      "question": "3. A black box contains 60 wooden blocks of the same size and shape, with every 15 blocks painted the same color. To ensure that at least 2 blocks have the same color, one must take out at least $\\qquad$ blocks.",
      "answer": "5"
    },
    {
      "question": "Given the rectangle $A B C D$. On the rays $A B, B C, C D$, and $D A$ starting from $A, B, C$, and $D$ respectively, we mark the points $A_{1}, B_{1}, C_{1}, D_{1}$ such that\n\n$$\n\\frac{A A_{1}}{A B}=\\frac{B B_{1}}{B C}=\\frac{C C_{1}}{C D}=\\frac{D D_{1}}{D A}=k>0\n$$\n\nFor what value of $k$ will the area of the quadrilateral $A_{1} B_{1} C_{1} D_{1}$ be minimal?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "5. The function $f(x)=a^{2 x}+3 a^{x}-2(a>0, a \\neq 1)$ has a maximum value of 8 on the interval $x \\in[-1,1]$, then its minimum value on this interval is $\\qquad$",
      "answer": "-\\frac{1}{4}"
    },
    {
      "question": "10.4. Real numbers ${ }^{a}$ and ${ }^{b}$ are such that $a^{3}+b^{3}=1-3 a b$. Find all values that the sum ${ }^{a+b}$ can take.",
      "answer": "-21"
    },
    {
      "question": "2.108. $\\left(\\frac{a}{b}+\\frac{b}{a}+2\\right) \\cdot\\left(\\frac{a+b}{2 a}-\\frac{b}{a+b}\\right):\\left(\\left(a+2 b+\\frac{b^{2}}{a}\\right) \\cdot\\left(\\frac{a}{a+b}+\\frac{b}{a-b}\\right)\\right) ;$ $a=0.75 ; b=4 / 3$.",
      "answer": "-\\frac{7}{24}"
    },
    {
      "question": "Determine the value of $123456123456 \\div 10000001$.",
      "answer": "123456"
    },
    {
      "question": "The ninety-eighth problem: Given a positive integer $\\mathrm{m} \\geq 3$. There are $2 \\mathrm{~m}$ players participating in a round-robin tournament. Each round, the $2 \\mathrm{~m}$ players are divided into $\\mathrm{m}$ groups, with two players competing in each group. In the next round, the players are regrouped to compete again, for a total of $2 m-1$ rounds, ensuring that each player competes against each of the other $2 m-1$ players exactly once. Find the smallest positive integer $n$ such that there exists a feasible tournament schedule where, after $n$ rounds, for any four players, they have either not competed against each other at all or have competed against each other at least twice in total.",
      "answer": "-1"
    },
    {
      "question": "9.2. The bisectors of the external angles $B$ and $C$ of triangle $ABC$ intersect at point $M$. a) Can angle $BMC$ be obtuse? b) Find angle $BAC$ if it is known that $\\angle BMC = \\frac{\\angle BAM}{2}$.",
      "answer": "120"
    },
    {
      "question": "Aline tore a sheet of paper into three pieces. Then, she tore one of these pieces into three pieces again; this gave her a total of five pieces. She then repeated the process, tearing one of the five pieces into three pieces, and so on. Can she, at some point, end up with a total of 2020 pieces? 2023 pieces?",
      "answer": "2023"
    },
    {
      "question": "1. (8 points) Calculate $82-38+49-51=$ $\\qquad$",
      "answer": "42"
    },
    {
      "question": "What is the value of $123456^{2}-123455 \\times 123457$ ?",
      "answer": "1"
    },
    {
      "question": "15. A student, in order to plot the graph of the function $y=a x^{2}+b x+c(a \\neq 0)$, took 7 values of the independent variable: $x_{1}<$ $x_{2}<\\cdots<x_{7}$, and $x_{2}-x_{1}=x_{3}-x_{2}=\\cdots=x_{7}-x_{6}$, and calculated the corresponding $y$ values. The following table is listed.\n\\begin{tabular}{|c|c|c|c|c|c|c|c|}\n\\hline$x$ & $x_{1}$ & $x_{2}$ & $x_{3}$ & $x_{4}$ & $x_{5}$ & $x_{6}$ & $x_{7}$ \\\\\n\\hline$y$ & 51 & 107 & 185 & 285 & 407 & 549 & 717 \\\\\n\\hline\n\\end{tabular}\n\nBut due to carelessness, one of the $y$ values was calculated incorrectly. Please point out which value is incorrect and what the correct value should be. Explain your reasoning.",
      "answer": "551"
    },
    {
      "question": "# Problem 4. (3 points)\n\nPetya wrote the number 11234567 on the board, and then all the numbers obtained from it by rearranging the digits, in ascending order. What was the position of the number $46753211$?",
      "answer": "12240"
    },
    {
      "question": "3. How many diagonals in a regular 32-sided polygon are not parallel to any of its sides",
      "answer": "240"
    },
    {
      "question": "What is the largest integer $n$ for which\n\n$$\n\\frac{\\sqrt{7}+2 \\sqrt{n}}{2 \\sqrt{7}-\\sqrt{n}}\n$$\n\nis an integer?",
      "answer": "343"
    },
    {
      "question": "18. Let $x$ and $y$ be real numbers satisfying the inequality\n$$\n5 x^{2}+y^{2}-4 x y+24 \\leq 10 x-1 \\text {. }\n$$\n\nFind the value of $x^{2}+y^{2}$.",
      "answer": "125"
    },
    {
      "question": "10,11\n\nConstruct a circle equidistant from four points on a plane. How many solutions does the problem have?",
      "answer": "7"
    },
    {
      "question": "78. How many circles of radius 1 are needed to cover a square with a side length of $2$?",
      "answer": "4"
    },
    {
      "question": "## Task A-1.5.\n\nIn some language, every word is a string of letters $a$ and $b$. Each word has at least one and at most 13 letters, but not all such strings are words. It is known that by concatenating one word to another, we never get a word. Determine the maximum possible number of words in this language.",
      "answer": "16256"
    },
    {
      "question": "50th Putnam 1989 Problem B1 S is a square side 1. R is the subset of points closer to its center than to any side. Find the area of R. Solution",
      "answer": "\\frac{4\\sqrt{2}-5}{3}"
    },
    {
      "question": "8. (15 points) Density is defined as the ratio of the mass of a body to its volume. There are two cubes. The second cube is made of a material with twice the density of the first, and the side length of the second cube is 100% greater than the side length of the first. By what percentage is the mass of the second cube greater than the mass of the first?",
      "answer": "1500"
    },
    {
      "question": "37th Swedish 1997 Problem 4 A and B play a game as follows. Each throws a dice. Suppose A gets x and B gets y. If x and y have the same parity, then A wins. If not, then they make a list of all two digit numbers ab ≤ xy with 1 ≤ a, b ≤ 6. Then they take turns (starting with A) replacing two numbers on the list by their (non-negative) difference. When just one number remains, it is compared to x. If it has the same parity A wins, otherwise B wins. Find the probability that A wins.",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "5. Given $n!=n \\times(n-1) \\times(n-2) \\times \\cdots \\times 2 \\times 1$, then $10!\\div(5!\\times 2!)=$",
      "answer": "15120"
    },
    {
      "question": "Problem 11.8. Given a parallelepiped $A B C D A_{1} B_{1} C_{1} D_{1}$. A point $X$ is chosen on the edge $A_{1} D_{1}$, and a point $Y$ is chosen on the edge $B C$. It is known that $A_{1} X=5, B Y=3, B_{1} C_{1}=14$. The plane $C_{1} X Y$ intersects the ray $D A$ at point $Z$. Find $D Z$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-46.jpg?height=501&width=678&top_left_y=359&top_left_x=388)",
      "answer": "20"
    },
    {
      "question": "3. Solve the equation $2021 \\cdot \\sqrt[202]{x^{2020}}-1=2020 x$ for $x \\geq 0 \\cdot$ (10 points)",
      "answer": "1"
    },
    {
      "question": "Cut a $4 \\times 4 \\times 4$ cube into 64 $1 \\times 1 \\times 1$ small cubes, then dye 16 of the $1 \\times 1 \\times 1$ small cubes red, requiring that among any 4 small cubes parallel to any edge, exactly 1 small cube is dyed red. The number of different dyeing methods is $\\qquad$ (dyeing methods that are the same after rotation are also considered different dyeing methods).",
      "answer": "576"
    },
    {
      "question": "2. (5 points) At the World Meteorologists Conference, each participant in turn announced the average monthly temperature in their hometown. At this moment, all the others recorded the product of the temperatures in their and the speaker's cities. In total, 78 positive and 54 negative numbers were recorded. What is the smallest number of times a positive temperature could have been announced?",
      "answer": "3"
    },
    {
      "question": "Two rods - one made of steel, the other of zinc - need to be sized such that at $0^{\\circ}$ C, $a=1 \\mathrm{~m}$, and at $100^{\\circ}$ C, their combined length is $b=1.0024 \\mathrm{~m}$. What is the length of each rod at $0^{\\circ}$ C, given that the linear expansion coefficient of steel is $\\lambda^{\\prime}=0.000011$, and that of zinc is $\\lambda^{\\prime \\prime}=0.000031$?",
      "answer": "0.35"
    },
    {
      "question": "### 5.24. Investigate the function\n\n$$\nf(x)=\\frac{5^{1 / x}}{1+5^{1 / x}}\n$$\n\nfor continuity and determine the nature of the points of discontinuity.",
      "answer": "0"
    },
    {
      "question": "(11) (25 points) The sum of $n$ positive integers $x_{1}, x_{2}, \\cdots, x_{n}$ is 2009. If these $n$ numbers can be divided into 41 groups with equal sums and also into 49 groups with equal sums, find the minimum value of $n$.",
      "answer": "89"
    },
    {
      "question": "For any sequence of real numbers $A=\\left\\{a_{1}, a_{2}, \\ldots\\right\\}$, we define $\\Delta A$ as the sequence $\\left\\{a_{2}-a_{1}, a_{3}-a_{2}, \\ldots\\right\\}$.\n\nWe assume that all terms of the sequence $\\Delta(\\Delta A)$ are 1 and that $a_{19}=a_{92}=0$.\n\nDetermine $a_{1}$.",
      "answer": "819"
    },
    {
      "question": "Count the number of triangles in the figure: $\\qquad$ triangles.\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "11"
    },
    {
      "question": "A $\\square\\{1,2,3, \\ldots, 49\\}$ does not contain six consecutive integers. Find the largest possible value of |A|. How many such subsets are there (of the maximum size)?\n\n## Answer\n\n$\\max =41$; no. ways 495",
      "answer": "495"
    },
    {
      "question": "4. Determine the smallest natural number $n$ with the following property: If we choose $n$ different natural numbers less than 2006, there are two among them such that the ratio of the sum and the difference of their squares is greater than three.",
      "answer": "21"
    },
    {
      "question": "12. For any positive integer $n, Z(n)$ is the smallest positive integer $m$ such that $1+2+\\cdots+m$ is a multiple of $n$. Then the sum of all positive integers $n$ that satisfy $Z(n)=6$ is $\\qquad$.",
      "answer": "28"
    },
    {
      "question": "48. Among the 2019 natural numbers from 1 to 2019, if any 809 odd numbers are taken and multiplied together, what is the unit digit of the product?",
      "answer": "5"
    },
    {
      "question": "10. (6 points) As shown in the figure, the area of the small square is 1, then the area of the shaded part in the figure is\n\n\n\n保留了源文本的换行和格式，这里的翻译结果已经直接输出。",
      "answer": "37.5"
    },
    {
      "question": "Right triangle $A B C$ is divided by the height $C D$, drawn to the hypotenuse, into two triangles: $B C D$ and $A C D$. The radii of the circles inscribed in these triangles are 4 and 3, respectively. Find the radius of the circle inscribed in triangle $A B C$.",
      "answer": "5"
    },
    {
      "question": "8. Given $a, b \\in Z$, and $a+b$ is a root of the equation $x^{2}+a x+b=0$, then the maximum possible value of $b$ is",
      "answer": "9"
    },
    {
      "question": "1st APMO 1989 Problem 3 ABC is a triangle. X lies on the segment AB so that AX/AB = 1/4. CX intersects the median from A at A' and the median from B at B''. Points B', C', A'', C'' are defined similarly. Find the area of the triangle A''B''C'' divided by the area of the triangle A'B'C'. Solution",
      "answer": "\\frac{25}{49}"
    },
    {
      "question": "Problem 1. The probability that a shooter, upon firing a shot, will score 10 points is 0.4; 9 points - 0.3; and finally, 8 or fewer points - 0.3. Find the probability that the shooter will score no less than 9 points in one shot.",
      "answer": "0.7"
    },
    {
      "question": "2. Given a positive integer $N<10^{2020}$, when 7 is placed at the first position of $N$, the resulting number is 5 times the number formed when 7 is placed at the last position of $N$. Then the number of all different values of $N$ is $\\qquad$.",
      "answer": "336"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{h \\rightarrow 0} \\frac{a^{x+h}+a^{x-h}-2 a^{x}}{h}$",
      "answer": "0"
    },
    {
      "question": "1. (USS) Three problems $A, B$, and $C$ were given on a mathematics olympiad. All 25 students solved at least one of these problems. The number of students who solved $B$ and not $A$ is twice the number of students who solved $C$ and not $A$. The number of students who solved only $A$ is greater by 1 than the number of students who along with $A$ solved at least one other problem. Among the students who solved only one problem, half solved $A$. How many students solved only $B$ ?",
      "answer": "6"
    },
    {
      "question": "$\\begin{array}{r}\\text { I. (40 points) Given } x_{i} \\in \\mathbf{R}(1 \\leqslant i \\leqslant 2020), \\\\ \\text { and } x_{1010}=1 \\text {. Find the minimum value of } \\sum_{i, j=1}^{2020} \\min \\{i, j\\} x_{i} x_{j} \\text { . }\\end{array}$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "3. A traveler visited a village where every person either always tells the truth or always lies. The villagers stood in a circle, and each one told the traveler whether the neighbor to their right was truthful or a liar. Based on these statements, the traveler was able to determine unequivocally what fraction of the village's population consists of truthful people. Determine what this fraction is.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "10-5. For each natural number from 1 to 999, Damir subtracted the last digit from the first digit and wrote all 1000 differences on the board. For example, for the number 7, Damir wrote the number 0 on the board, for the number 105 he wrote $(-4)$, and for the number 61 he wrote 5.\n\nWhat is the sum of all the numbers on the board",
      "answer": "495"
    },
    {
      "question": "B-bank issues ATM cards with a four-digit PIN code that does not contain the digit 0. Mr. Sclerosis was afraid he would forget the PIN code of his card, so he wrote it directly on it, but in Roman numerals IIIVIIIXIV, to make it harder for a potential thief. He revealed his idea to his best friend, Mr. Odkoukal, who was also a client of B-bank. Soon, Mr. Odkoukal did the same with his PIN code and wrote IVIIIVI on his card. To his great surprise, however, he could not determine his PIN code exactly from the Roman numeral notation.\n\n1. What is the PIN code on Mr. Sclerosis's card?\n2. What PIN code could Mr. Odkoukal's card have?",
      "answer": "3794"
    },
    {
      "question": "8. The sum of 2020 consecutive natural numbers arranged in ascending order is $1010 \\times 2043$. Spider-Man adds the 2nd, 4th, 6th, ..., 2018th, and 2020th numbers, and then divides the sum by 505. The quotient is $\\qquad$ .",
      "answer": "2044"
    },
    {
      "question": "Question 11: Given the mapping f: $\\{1,2,3,4,5\\} \\rightarrow\\{1,2,3,4,5\\}$ is both injective and surjective, and satisfies $f(x)+f(f(x))=6$, then $f(1)=$ $\\qquad$ _.",
      "answer": "5"
    },
    {
      "question": "1.598 girls and 25 boys stand in a circle, with at least two boys standing between any two girls. How many different arrangements are there (arrangements that can be made to coincide by rotating the circle are considered the same)?",
      "answer": "\\frac{16!25!}{9!}"
    },
    {
      "question": "Determine a value of $n$ for which the number $2^{8}+2^{11}+2^{n}$ is a perfect square.",
      "answer": "12"
    },
    {
      "question": "2. Let $A=\\overline{a b c b a}$ be a five-digit symmetric number, $a \\neq 0$. If $1 \\leq a \\leq 8$, then the last digit of the number $A+11$ will be $a+1$, and therefore the first digit in the representation of $A+11$ should also be $a+1$. This is possible only with a carry-over from the digit, i.e., when $b=c=9$. Then $A+11=(a+1) 999(a+1)$ is a symmetric number for any $a=1,2, \\ldots, 8$. The case $a=9$ is impossible, since $A+11$ ends in zero, and thus, due to symmetry, it should start with zero. But a number cannot start with zero.\n\nThe total number of solutions is equal to the number of possible choices for the number $a$, which is eight.",
      "answer": "8"
    },
    {
      "question": "Exercise 2. How many 8-digit numbers are there whose decimal notation is of the form ab2019cd with a $>0$, and which are divisible by 360?",
      "answer": "20"
    },
    {
      "question": "4. A frog jumps over lily pads. It can jump 7 lily pads to the right or 4 lily pads to the left. What is the minimum number of jumps needed for the frog to move exactly 2024 lily pads to the right?\n\n![](https://cdn.mathpix.com/cropped/2024_06_03_1d05bbb73aad96dd359ag-3.jpg?height=285&width=1514&top_left_y=1945&top_left_x=294)",
      "answer": "297"
    },
    {
      "question": "3. Let $n$ be an integer greater than 1 . If $n$ is greater than 1200 times any one of its prime factors, find the smallest possible value of $n$.\n(1 mark)\n設 $n$ 為大於 1 的整數。若 $n$ 較它的任何一個質因數的 1200 倍為大, 求 $n$ 的最小可能值。",
      "answer": "3888"
    },
    {
      "question": "I3.2 Two bags $U_{1}$ and $U_{2}$ contain identical red and white balls. $U_{1}$ contains $A$ red balls and 2 white balls. $U_{2}$ contains 2 red balls and $B$ white balls. Take two balls out of each bag. If the probability of all four balls are red is $\\frac{1}{60}$, find the value of $B$.",
      "answer": "3"
    },
    {
      "question": "14. Two circles $C_{1}$ and $C_{2}$ with centers $A$ and $B$ are externally tangent at $T$. Let $B D$ be a segment tangent to $C_{1}$ at $D$ and let $T C$ be the segment tangent to both at $T$ with $C \\in B D$. If $A T$ is 80 units long and $B T$ is 90 units long, what is the length of $C D$?",
      "answer": "48"
    },
    {
      "question": "5. July 1, 2021 is the 100th anniversary of the founding of our Party, recorded as 20210701, where the sum of the digits in the odd positions is $1+7+1+0=9$ and the sum of the digits in the even positions is $0+0+2+2=4$, with a ratio of $9: 4$; among the 8-digit numbers formed by $0 \\sim 9$ (no repetition), how many satisfy the condition that the sum of the digits in the odd positions to the sum of the digits in the even positions is $9: 4$?",
      "answer": "2304"
    },
    {
      "question": "358. Calculate the integral $\\int_{1}^{9} \\sqrt{6 x-5} d x$ using the Newton-Leibniz formula and approximate formulas for rectangles and trapezoids, dividing the interval of integration into 8 equal parts. Estimate the error of the results.",
      "answer": "38"
    },
    {
      "question": "1. Fold the triangular paper piece $A B C$ along a line parallel to $B C$, the folded part intersects $B C$ at points $F, G$. Given $B C=9 \\mathrm{~cm}, F G=3 \\mathrm{~cm}$, then the length of the crease $D E$ is $\\qquad$ $\\mathrm{cm}$.",
      "answer": "6"
    },
    {
      "question": "10. The function $y=f(x)$ defined on $\\mathbf{R}$ has the following properties:\n（1）For any $x \\in \\mathbf{R}$, $f\\left(x^{3}\\right)=f^{3}(x)$;\n（2）For any $x_{1} 、 x_{2} \\in \\mathbf{R}, x_{1} \\neq x_{2}$, $f\\left(x_{1}\\right) \\neq f\\left(x_{2}\\right)$. Then the value of $f(0)+f(1)+f(-1)$ is $\\qquad$.",
      "answer": "0"
    },
    {
      "question": "20. 11, 12, 13 are three consecutive natural numbers, the sum of their digits is $1+1+1+2+1+3=9$. There are three consecutive natural numbers, all less than 100, the sum of their digits is 18, there are $\\qquad$ possible cases.",
      "answer": "8"
    },
    {
      "question": "5. (10 points) The 3rd term of an arithmetic sequence is 14, and the 18th term is 23. How many terms in the first 2010 terms of this sequence are integers.",
      "answer": "402"
    },
    {
      "question": "9. Given that two cars, A and B, start from points $A$ and $B$ respectively at the same time, and travel back and forth between $A$ and $B$ at a constant speed. If after the first meeting, car A continues to drive for 4 hours to reach $B$, while car B only drives for 1 hour to reach $A$, then when the two cars meet for the 15th time (excluding meetings at points $A$ and $B$), they have driven $\\qquad$ hours.",
      "answer": "86"
    },
    {
      "question": "13th Iberoamerican 1998 Problem A3 Find the smallest number n such that given any n distinct numbers from {1, 2, 3, ... , 999}, one can choose four different numbers a, b, c, d such that a + 2b + 3c = d. Solution",
      "answer": "835"
    },
    {
      "question": "Example 9 In $\\triangle A B C$, it is known that point $E$ is the midpoint of side $B C$, $D$ is on $A C$, $A C=1, \\angle B A C=$ $60^{\\circ}, \\angle A B C=10^{\\circ}, \\angle D E C=80^{\\circ}$, find the value of $S_{\\triangle A B C}+2 S_{\\triangle C D E}$.",
      "answer": "\\frac{\\sqrt{3}}{8}"
    },
    {
      "question": "37. The area of the triangle formed by the lines $y=x+2$, $y=-2x+5$ and the $x$-axis is equal to",
      "answer": "\\frac{27}{4}"
    },
    {
      "question": "7. Let $a$ be a non-zero real number, in the Cartesian coordinate system $x O y$, the focal distance of the quadratic curve $x^{2}+a y^{2}+a^{2}=0$ is 4, then the value of $a$ is $\\qquad$",
      "answer": "\\frac{1-\\sqrt{17}}{2}"
    },
    {
      "question": "Solve for real $x$ :\n\n$$\nx+\\sqrt{(x+1)(x+2)}+\\sqrt{(x+2)(x+3)}+\\sqrt{(x+3)(x+1)}=4 .\n$$",
      "answer": "-\\frac{311}{840}"
    },
    {
      "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{5 / 2} \\frac{x^{2} \\cdot d x}{\\sqrt{25-x^{2}}}\n$$",
      "answer": "\\frac{25\\pi}{12}-\\frac{25\\sqrt{3}}{8}"
    },
    {
      "question": "In the diagram, $P A D B, T B C$ and $N C D$ are straight line segments. If $\\angle T B D=110^{\\circ}$, $\\angle B C N=126^{\\circ}$, and $D C=D A$, determine the measure of $\\angle P A C$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_30_3011c729c030a190170fg-1.jpg?height=398&width=677&top_left_y=1189&top_left_x=781)",
      "answer": "152"
    },
    {
      "question": "8,9\n\nOn a circle, a point $A$ is taken, on the diameter $B C$ - points $D$ and $E$, and on its extension beyond point $B$ - point $F$. Find $B C$, if $\\angle B A D=\\angle A C D, \\angle B A F=\\angle C A E, B D=2, B E=5$ and $B F=4$.",
      "answer": "11"
    },
    {
      "question": "3. Let $A B C D$ be a regular tetrahedron with side length 2 . The plane parallel to edges $A B$ and $C D$ and lying halfway between them cuts $A B C D$ into two pieces. Find the surface area of one of these pieces.",
      "answer": "1+2\\sqrt{3}"
    },
    {
      "question": "5. Given a convex hexagon $A B C D E F$ with six equal side lengths, the interior angles $\\angle A$, $\\angle B$, and $\\angle C$ are $134^{\\circ}$, $106^{\\circ}$, and $134^{\\circ}$, respectively. Then the measure of the interior angle $\\angle E$ is $\\qquad$.",
      "answer": "134"
    },
    {
      "question": "In the pyramid $A B C D$, the dihedral angle at edge $A C$ is $90^{\\circ}, A B=B C=C D, B D=A C$. Find the dihedral angle at edge $A D$.\n\n#",
      "answer": "60"
    },
    {
      "question": "9. Given that a line passing through the focus $F$ of the parabola $y^{2}=4 x$ intersects the parabola at points $M$ and $N$, and $E(m, 0)$ is a point on the $x$-axis. The extensions of $M E$ and $N E$ intersect the parabola at points $P$ and $Q$ respectively. If the slopes $k_{1}$ and $k_{2}$ of $M N$ and $P Q$ satisfy $k_{1}=3 k_{2}$, then the value of the real number $m$ is",
      "answer": "3"
    },
    {
      "question": "4. Given nine-digit numbers $m$ and $n$, obtained from each other by writing the digits in reverse order. It turned out that the product $mn$ consists of an odd number of digits and reads the same from left to right and from right to left. Find the largest number $m$ for which this is possible.",
      "answer": "220000001"
    },
    {
      "question": "## Task B-1.2.\n\nIf $a-b=3$, and $a \\cdot b=1$, what is $\\frac{1}{a^{3}}-\\frac{1}{b^{3}}$ and $\\frac{1}{a^{6}}+\\frac{1}{b^{6}}$?",
      "answer": "-361298"
    },
    {
      "question": "There are given 100 distinct positive integers. We call a pair of integers among them good if the ratio of its elements is either 2 or 3. What is the maximum number $g$ of good pairs that these 100 numbers can form? (A same number can be used in several pairs.)",
      "answer": "180"
    },
    {
      "question": "4. A certain three-digit number is a multiple of 2, adding 1 makes it a multiple of 3, adding 2 makes it a multiple of 4, adding 3 makes it a multiple of 5, and adding 4 makes it a multiple of 6. The smallest such number is $\\qquad$ .",
      "answer": "122"
    },
    {
      "question": "Example 5 Given that $f(x)$ is a function defined on the set of real numbers, and $f(x+2)[1-f(x)]=1+f(x)$, if $f(5)$ $=2+\\sqrt{3}$, find $f(2009)$.",
      "answer": "\\sqrt{3}-2"
    },
    {
      "question": "## Task 27/77\n\nDetermine the largest natural number $z$ that consists of 10 different digits (in the decimal positional system) and is divisible by 11.",
      "answer": "9876524130"
    },
    {
      "question": "Problem 12.3. Find the maximal cardinality of a set of phone numbers satisfying the following three conditions:\n\na) all of them are five-digit numbers (the first digit can be 0 );\n\nb) each phone number contains at most two different digits;\n\nc) the deletion of an arbitrary digit in two arbitrary phone numbers (possibly in different positions) does not lead to identical sequences of digits of length 4 .\n\nIvan Landjev",
      "answer": "190"
    },
    {
      "question": "Example 8. An urn contains 8 red and 6 blue balls. Three balls are drawn sequentially without replacement from the urn. Find the probability that all 3 balls are blue.",
      "answer": "\\frac{5}{91}"
    },
    {
      "question": "5. Let the line $y=k x-2$ intersect the parabola $y^{2}=8 x$ at points $A$ and $B$. If the x-coordinate of the midpoint of segment $A B$ is 2, then the length of segment $A B$ is $|A B|=$ $\\qquad$",
      "answer": "2\\sqrt{15}"
    },
    {
      "question": "Task 1. (5 points) Find $\\frac{a^{12}-4096}{64 a^{6}}$, if $\\frac{a^{2}}{4}-\\frac{4}{a^{2}}=5$.",
      "answer": "140"
    },
    {
      "question": "7. In the company, there are elves, fairies, and gnomes. Each elf is friends with all fairies except for three, and each fairy is friends with twice as many elves. Each elf is friends with exactly three gnomes, and each fairy is friends with all gnomes. Each gnome is friends with exactly half of the elves and fairies combined. How many gnomes are there in the company?",
      "answer": "12"
    },
    {
      "question": "Three. (50 points) Find the smallest positive integer $n$, such that there exist rational numbers $a_{i}, b_{i} (i=1,2, \\cdots, n)$ satisfying\n$$\nx^{2}+x+4=\\sum_{i=1}^{n}\\left(a_{i} x+b_{i}\\right)^{2} .\n$$",
      "answer": "5"
    },
    {
      "question": "17. What is the remainder when\n$$\n16^{15}-8^{15}-4^{15}-2^{15}-1^{15}\n$$\nis divided by 96 ?",
      "answer": "31"
    },
    {
      "question": "5) A class of students is planting a batch of saplings. If each student is given k saplings, there will be 38 left; if each student is allocated 9 saplings, there will be a shortage of 3. Therefore, the class has $\\qquad$ students.",
      "answer": "41"
    },
    {
      "question": "Problem 9.5. Point $M$ is the midpoint of side $B C$ of triangle $A B C$, where $A B=17$, $A C=30, B C=19$. A circle is constructed with side $A B$ as its diameter. An arbitrary point $X$ is chosen on this circle. What is the minimum value that the length of segment $M X$ can take?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_85a336fa11c4e8eb26a1g-03.jpg?height=414&width=553&top_left_y=1517&top_left_x=448)",
      "answer": "6.5"
    },
    {
      "question": "8. Given the function $f(x)=x^{2}+a x+b, a, b \\in \\mathbf{R}$, let $M(a, b)$ be the maximum value of $|f(x)|$ on the interval $[-1,1]$. When $a, b$ satisfy $M(a, b) \\leqslant 2$, the maximum value of $|a|+|b|$ is $\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "Problem 10.6. In a convex quadrilateral $A B C D$, the midpoint of side $A D$ is marked as point $M$. Segments $B M$ and $A C$ intersect at point $O$. It is known that $\\angle A B M=55^{\\circ}, \\angle A M B=$ $70^{\\circ}, \\angle B O C=80^{\\circ}, \\angle A D C=60^{\\circ}$. How many degrees does the angle $B C A$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_564c13f715a760703913g-39.jpg?height=371&width=709&top_left_y=89&top_left_x=372)",
      "answer": "35"
    },
    {
      "question": "1. (3 points) Calculate: $19 \\times 0.125+281 \\times \\frac{1}{8}+12.5=$",
      "answer": "50"
    },
    {
      "question": "3. Find the number of positive integer solutions to the equation $x_{1}+x_{2}+x_{3}+x_{4}=23$, with the conditions $x_{1} \\leqslant 9, x_{2} \\leqslant 8, x_{3} \\leqslant 7$, $x_{4} \\leqslant 6$.",
      "answer": "115"
    },
    {
      "question": "4. For what values of the parameter $a$ does the system of equations\n\n$$\n\\left\\{\\begin{array}{l}\nx^{2}+y^{2}+z^{2}+2 y=0 \\\\\nx+a y+a z-a=0\n\\end{array}\\right.\n$$\n\nhave a unique solution?",
      "answer": "\\\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "Example 5. Calculate the double integral $I=\\int_{D} \\int(x+y) d x d y$, where $D-$ is the region bounded by the lines $x=0, y=x^{2}+x-3$, $2 y=3 x(x \\geqslant 0)$.",
      "answer": "\\frac{14}{5}"
    },
    {
      "question": "9. (10 points) In a certain exam, 153 people scored no more than 30 points, with an average of 24 points; 59 people scored no less than 80 points, with an average of 92 points; those who scored more than 30 points had an average of 62 points, and those who scored less than 80 points had an average of 54 points. How many people in total participated in this exam?",
      "answer": "1007"
    },
    {
      "question": "1. a) Solve in $Z$ the equation: $5 \\cdot(2 \\cdot|3 x-4|+4)-30=10$",
      "answer": "2"
    },
    {
      "question": "411. Find the curvature of the curve: 1) $x=t^{2}, y=2 t^{3}$ at the point where $t=1$; 2) $y=\\cos 2 x$ at the point where $x=\\frac{\\pi}{2}$.",
      "answer": "4"
    },
    {
      "question": "Problem 10.8. Given a triangle $A B C$. Let point $I$ be the center of its inscribed circle, and points $P$ and $Q$ be the midpoints of sides $A B$ and $A C$ respectively. It turns out that $\\angle P I Q + \\angle B I C = 180^{\\circ}$. Find the length of segment $B C$, if $A B = 20$ and $A C = 14$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_baaf26a33e2d9a36d9b4g-06.jpg?height=501&width=422&top_left_y=575&top_left_x=516)",
      "answer": "\\frac{34}{3}"
    },
    {
      "question": "8. (10 points) On a plane, use toothpicks of length 6 cm to form squares. To form a square with a side length of 6 cm, 4 toothpicks are needed. To form 5 such squares, at least $\\qquad$ toothpicks are needed.",
      "answer": "15"
    },
    {
      "question": "Some Christmas lights are arranged using magical wires. Each light can be green or yellow. Each wire is connected to two lights and has a magical property: when someone touches a wire connecting two lights, each of them changes color from green to yellow or from yellow to green.\n\na) In the arrangement below, each point represents a light and the segments represent the magical wires. At the beginning, all of them are yellow. What is the minimum number of wires we need to touch so that all the lights turn green? Show an example of how to make this change with the minimum number of wires.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-24.jpg?height=320&width=463&top_left_y=859&top_left_x=702)\n\nb) Consider the arrangement in the following figure where all the lights are yellow. Show that it is not possible to touch some magical wires and change the color of all the lights to green.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-24.jpg?height=323&width=685&top_left_y=1426&top_left_x=594)",
      "answer": "8"
    },
    {
      "question": "$3 \\cdot 33$ Find the sum $\\sum_{n=1}^{1994}(-1)^{n} \\frac{n^{2}+n+1}{n!}$.",
      "answer": "-1+\\frac{1995}{1994!}"
    },
    {
      "question": "We drew all the diagonals in three polygons with different numbers of sides. The second polygon has 3 more sides and 3 times as many diagonals as the first one. The third one has 7 times as many diagonals as the second one. How many more sides does the third polygon have compared to the second one?",
      "answer": "12"
    },
    {
      "question": "3. A pedestrian is walking along a path next to a steep track (in the longitudinal direction) at a speed of $6 \\mathrm{~km} / \\mathrm{h}$. He noticed that the train coming from the opposite direction passed him in 12.6 seconds. With the same train, while walking uphill at a speed of $3.6 \\mathrm{~km} / \\mathrm{h}$, he passed it in 15 seconds. What is the length of the train and what is its speed?",
      "answer": "210"
    },
    {
      "question": "7. Let positive real numbers $x, y$ satisfy $x^{2}+y^{2}+\\frac{1}{x}+\\frac{1}{y}=\\frac{27}{4}$. Then the minimum value of $P=\\frac{15}{x}-\\frac{3}{4 y}$ is",
      "answer": "6"
    },
    {
      "question": "With 5 equal digits, I can write 100 as follows: $100=111-11$, or $100=3 \\times 33+\\frac{3}{3}$, or $(5+5+5+5) \\times 5$. Based on these, let's write 100 with 9 different digits.",
      "answer": "38"
    },
    {
      "question": "2. There is a bag with 16 letters: А, А, А, А, В, В, Д, И, И, М, М, Н, Н, Н, Я, Я. Anna, Vanya, Danya, and Dima each took 4 letters from it, after which the bag was empty. How many of them could have successfully spelled out their names? Explain your answer.",
      "answer": "3"
    },
    {
      "question": "10. In Rt $\\triangle A B C$, $\\angle C=90^{\\circ}, D, E$ are points on sides $B C, A C$ respectively. Given $A D=17$, $B E=13, D E=\\sqrt{97}$, then $A B=$ $\\qquad$ .",
      "answer": "19"
    },
    {
      "question": "\\section*{Problem 6 - 061046}\n\nGive the total number of all different integer solution pairs \\((x, y)\\) of the inequality\n\n\\[\n|x|+|y| \\leq 100\n\\]\n\nwhere two solution pairs \\(\\left(x_{1}, y_{1}\\right),\\left(x_{2}, y_{2}\\right)\\) are considered the same if and only if \\(x_{1}=x_{2}\\) and \\(y_{1}=y_{2}\\).",
      "answer": "20201"
    },
    {
      "question": "G87 (47-2, 塞黑) Let $P$ be a regular 2006-gon. If an end of a diagonal of $P$ divides the boundary of $P$ into two parts, each containing an odd number of sides of $P$, then the diagonal is called a \"good edge\". It is stipulated that each side of $P$ is a \"good edge\".\n\nGiven 2003 non-intersecting diagonals inside $P$ that divide $P$ into several triangles, how many isosceles triangles with two \"good edges\" can there be in such a division?\n\n\n\nTranslating the text into English while preserving the original formatting and line breaks:\n\nG87 (47-2, 塞黑) Let $P$ be a regular 2006-gon. If an end of a diagonal of $P$ divides the boundary of $P$ into two parts, each containing an odd number of sides of $P$, then the diagonal is called a \"good edge\". It is stipulated that each side of $P$ is a \"good edge\".\n\nGiven 2003 non-intersecting diagonals inside $P$ that divide $P$ into several triangles, how many isosceles triangles with two \"good edges\" can there be in such a division?",
      "answer": "1003"
    },
    {
      "question": "11. (20 points) Given the function\n$$\nf(x)=\\left(1-x^{2}\\right)\\left(x^{2}+b x+c\\right)(x \\in[-1,1]) \\text {. }\n$$\n\nLet the maximum value of $\\mid f(x)$ be $M(b, c)$. When $b$ and $c$ vary, find the minimum value of $M(b, c)$.",
      "answer": "3-2\\sqrt{2}"
    },
    {
      "question": "Task B-1.6. How many ordered pairs of integers $x, y, 0<x<y<10^{6}$ are there such that their arithmetic mean is exactly 8 greater than their geometric mean? (The geometric mean of numbers $a$ and $b$ is the number $\\sqrt{a \\cdot b}$.)",
      "answer": "995"
    },
    {
      "question": "Ali Baba, upon returning to the cave filled with diamonds and gold, pondered the following: The treasures can only be placed in his sack. From experience, he knows that 40 pounds of diamonds and 200 pounds of gold can fit into it, but he can only carry and take away 100 pounds. In the market, 1 pound of gold can be exchanged for 20 tevets, and 1 pound of diamonds for 60 tevets. How should he fill his sack so that he can buy the maximum number of tevets?",
      "answer": "25"
    },
    {
      "question": "Example 3 In the Cartesian coordinate system $x O y$, given two points $M(-1,2)$ and $N(1,4)$, point $P$ moves on the $x$-axis. When $\\angle M P N$ takes the maximum value, find the x-coordinate of point $P$.\n\n untranslated text remains unchanged.",
      "answer": "1"
    },
    {
      "question": "2. (8 points) Xiaoming's family keeps three hens, the first hen lays one egg per day, the second hen lays one egg every two days, and the third hen lays one egg every three days. It is known that on January 1st, all three hens laid eggs, so in the thirty-one days of January, these three hens laid $\\qquad$ eggs in total,",
      "answer": "58"
    },
    {
      "question": "2. Each of the 10 people is either a knight, who always tells the truth, or a liar, who always lies. Each of them thought of some natural number. Then the first said: “My number is greater than 1”, the second said: “My number is greater than 2”, ..., the tenth said: “My number is greater than 10”. After that, they, in a different order, said (each one phrase): “My number is less than 1”, “My number is less than 2”, ..., “My number is less than 10”. What is the maximum number of knights that could have been among these 10 people? (O. Podlipsky)",
      "answer": "8"
    },
    {
      "question": "3. In a regular hexagon $A B C D E F$ with side length $a$, the extensions of sides $A F$ and $D E$ intersect at point $T$. Calculate the exact length of segment $B T$.",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "Variant 11.3.4. The polynomial $G(x)$ with real coefficients takes the value 2022 at exactly five different points $x_{1}<x_{2}<x_{3}<x_{4}<x_{5}$. It is known that the graph of the function $y=G(x)$ is symmetric with respect to the line $x=-6$.\n\n(a) (2 points) Find $x_{1}+x_{3}+x_{5}$.\n\n(b) (2 points) What is the smallest degree that $G(x)$ can have?",
      "answer": "6"
    },
    {
      "question": "Two identical equilateral triangles $A B C$ and $C D E$ with side length 1 are positioned such that they share only one common point $C$ and the angle $B C D$ is less than $60^{\\circ}$. Point $K$ is the midpoint of $A C$, point $L$ is the midpoint of $C E$, and point $M$ is the midpoint of $B D$. The area of triangle $K L M$ is $\\frac{\\sqrt{3}}{5}$. Find $B D$.",
      "answer": "\\frac{2-\\sqrt{3}}{\\sqrt{5}}"
    },
    {
      "question": "Let's determine those five-digit numbers, which, when divided by the four-digit number obtained by omitting the middle digit, result in an integer.\n\n \n\nFind the five-digit numbers such that when divided by the four-digit number obtained by omitting the middle digit, the result is an integer.",
      "answer": "8"
    },
    {
      "question": "Task 1. If to a two-digit number the digit 5 is added once at the beginning, and another time at the end, two different numbers are obtained whose difference is 252. What is that two-digit number?",
      "answer": "2783"
    },
    {
      "question": "Let $E=\\{a, b, c, d\\}$ be a set. How many functions $f$ from $E$ to $E$ satisfy, for all $x$ in $E, f(f(f(x)))=x$?",
      "answer": "9"
    },
    {
      "question": "Problem 8.1. A square was cut into four equal rectangles, and from them, a large letter P was formed, as shown in the figure, with a perimeter of 56.\n![](https://cdn.mathpix.com/cropped/2024_05_06_e7f9c87b6a37ffba3564g-27.jpg?height=356&width=720&top_left_y=274&top_left_x=366)\n\nWhat is the perimeter of the original square",
      "answer": "32"
    },
    {
      "question": "3. Let $\\left\\{a_{n}\\right\\}$ be an arithmetic sequence with the sum of the first $n$ terms denoted as $S_{n}$. If $S_{6}=26, a_{7}=2$, then the maximum value of $n S_{n}$ is $\\qquad$ .",
      "answer": "338"
    },
    {
      "question": "Problem 9.4. A prime number $p$ is such that the number $p+25$ is the seventh power of a prime number. What can $p$ be? List all possible options.",
      "answer": "103"
    },
    {
      "question": "For how many dice is the probability the greatest that exactly one six will appear when the dice are rolled simultaneously?",
      "answer": "56"
    },
    {
      "question": "In the plane, 2013 red points and 2014 blue points are marked so that no three of the marked points are collinear. One needs to draw \\( k \\) lines not passing through the marked points and dividing the plane into several regions. The goal is to do it in such a way that no region contains points of both colors. Find the minimal value of \\( k \\) such that the goal is attainable for every possible configuration of 4027 points. (Australia)",
      "answer": "2013"
    },
    {
      "question": "Problem 4.5. Hooligan Dima made a construction in the shape of a $3 \\times 5$ rectangle using 38 wooden toothpicks. Then he simultaneously set fire to two adjacent corners of this rectangle, marked in the figure.\n\nIt is known that one toothpick burns for 10 seconds. How many seconds will it take for the entire construction to burn?\n\n(Fire spreads along the toothpicks at a constant speed. The fire continues to spread from each burned toothpick to all adjacent unburned toothpicks.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_86512448fd32765ac040g-04.jpg?height=458&width=359&top_left_y=110&top_left_x=547)",
      "answer": "65"
    },
    {
      "question": "4. Person A and Person B agreed to have dinner at a restaurant. Since this restaurant is very popular, after A arrived, they took a waiting number and waited for B. B arrived at the restaurant a while later but didn't see A, so they also took a waiting number. While waiting, B saw A, and both of them showed their waiting numbers. They found that the digits of the two numbers were reverse two-digit numbers, and the sum of the digits (for example, the sum of the digits of the number 23 is $2+3=5$) of both numbers was 8, and B's number was 18 more than A's. What is A's number? $\\qquad$",
      "answer": "35"
    },
    {
      "question": "43. Without using any unit of length or other auxiliary means, how can you cut exactly half a meter from a piece of fabric that is $\\frac{8}{15}$ m?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Determine the number of all positive ten-digit integers with the following properties:\n\n- The number contains each of the digits 0, 1, 2, ..., 8, and 9 exactly once.\n- Each digit, except for the 9, has a neighboring digit that is greater than it.\n\n(Note. For example, in the number 1230, the digits 1 and 3 are the neighboring digits of 2, and 2 and 0 are the neighboring digits of 3. The digits 1 and 0 have only one neighboring digit.)\n\n(Karl Czakler)\n\nAnswer. There are 256 numbers with the required properties.",
      "answer": "256"
    },
    {
      "question": "1. Calculate the value of the numerical expression: $36+64 \\cdot 17-(502-352: 8+511)$.",
      "answer": "155"
    },
    {
      "question": "5. Calculate $\\sqrt{n}+\\sqrt{n+524}$, given that this number is rational and that $n$ is a natural number.",
      "answer": "262"
    },
    {
      "question": "5. (4 points) Let two sequences of numbers $\\left(x_{0}, x_{1}, \\ldots, x_{2009}\\right)$, $\\left(y_{0}, y_{1}, \\ldots, y_{2009}\\right)$ be constructed according to the following rules:\na) $x_{0}=12, x_{1}=\\frac{1}{3}, y_{0}=4, y_{1}=\\frac{1}{18}$,\n\nb) $x_{i+1}=x_{i-1}+4 x_{i}$ and $y_{i+1}=y_{i-1}-4 y_{i}$ for $i=1, \\ldots, 2008$.\n\nCalculate the value of $x_{2009} y_{2008}+x_{2008} y_{2009}$.",
      "answer": "2"
    },
    {
      "question": "4. let the positive divisors of the natural number $n$ be $1=d_{1}<d_{2}<\\ldots<d_{k}=n$. Determine all $n$ for which the following applies\n\n$$\n2 n=d_{5}^{2}+d_{6}^{2}-1\n$$\n\n## Solution",
      "answer": "272"
    },
    {
      "question": "1. The clock shows 00:00, at which the hour and minute hands of the clock coincide. Considering this coincidence as number 0, determine after what interval of time (in minutes) they will coincide for the 19th time. If necessary, round the answer to the hundredths place according to rounding rules.",
      "answer": "1243.64"
    },
    {
      "question": "Task 2. Determine all positive integers $n$ for which there exist positive integers $a_{1}, a_{2}, \\ldots, a_{n}$ such that\n\n$$\na_{1}+2 a_{2}+3 a_{3}+\\ldots+n a_{n}=6 n\n$$\n\nand\n\n$$\n\\frac{1}{a_{1}}+\\frac{2}{a_{2}}+\\frac{3}{a_{3}}+\\ldots+\\frac{n}{a_{n}}=2+\\frac{1}{n}\n$$",
      "answer": "3"
    },
    {
      "question": "(5) In a regular tetrahedron $ABCD$, $AO \\perp$ plane $BCD$, with the foot of the perpendicular being $O$. Let $M$ be a point on the line segment $AO$ such that $\\angle BMC=90^{\\circ}$. Then $\\frac{AM}{MO}=$ $\\qquad$ .",
      "answer": "1"
    },
    {
      "question": "Example 29 (2003 National High School Mathematics Competition Question) Let the three sides of a triangle be integers $l, m, n$, and $l>m>n$. It is known that $\\left\\{\\frac{3^{l}}{10^{4}}\\right\\}=\\left\\{\\frac{3^{m}}{10^{4}}\\right\\}=\\left\\{\\frac{3^{n}}{10^{4}}\\right\\}$, where $\\{x\\}=x-[x]$, and $[x]$ represents the greatest integer not exceeding $x$. Find the minimum perimeter of such a triangle.",
      "answer": "3003"
    },
    {
      "question": "3. In the sequence $\\left\\{a_{n}\\right\\}$, for $1 \\leqslant n \\leqslant 5$, we have $a_{n}=n^{2}$; and for all positive integers $n$, we have $a_{n+5}+a_{n+1}=a_{n+4}+a_{n}$. Then $a_{2023}=$ $\\qquad$ .",
      "answer": "17"
    },
    {
      "question": "Example 1. Find the volume of the body $\\Omega$, bounded by the surfaces\n\n$$\nx=17 \\sqrt{2 y}, \\quad x=2 \\sqrt{2 y}, \\quad z=\\frac{1}{2}-y, \\quad z=0\n$$",
      "answer": "1"
    },
    {
      "question": "Let $A$ be the following.\n\nA numerical sequence is defined by the conditions: $a_{1}=1, a_{n+1}=a_{n}+\\left[\\sqrt{a_{n}}\\right]$.\n\nHow many perfect squares occur among the first terms of this sequence, not exceeding\n\n1000000?",
      "answer": "10"
    },
    {
      "question": "1. Let the set $P_{n}=\\{1,2, \\cdots, n\\}\\left(n \\in \\mathbf{Z}_{+}\\right)$. Denote $f(n)$ as the number of sets $A$ that satisfy the following conditions:\n(1) $A \\subseteq P_{n}, \\bar{A}=P_{n} \\backslash A$;\n(2) If $x \\in A$, then $2 x \\notin A$;\n(3) If $x \\in \\bar{A}$, then $2 x \\notin \\bar{A}$.\nThen $f(2018)=$ $\\qquad$",
      "answer": "2^{1009}"
    },
    {
      "question": "5. Let $[x]$ denote the greatest integer not exceeding the real number $x$. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies\n$$\na_{1}=a_{2}=\\frac{1}{2}, a_{n+1}=2 a_{n}+a_{n-1}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nThen $\\left[\\sum_{k=2}^{2016} \\frac{1}{a_{k-1} a_{k+1}}\\right]=$ $\\qquad$ .",
      "answer": "1"
    },
    {
      "question": "2. In a triangle, the length of one median is half longer than the length of the side to which it is drawn. Find the angle between the other two medians.",
      "answer": "90"
    },
    {
      "question": "38. How many six-digit numbers are there in which only the middle two digits are the same?",
      "answer": "27216"
    },
    {
      "question": "75. Исследование пустыни. Девять участников экспедиции (каждый на автомашине) встречаются на восточной окраине пустыни. Они хотят исследовать ее внутренние районы, двигаясь все время на запад. Каждому автомобилю полного бака (содержащего 1 галлон бензина) хватает на 40 миль пути. Кроме того, он может взять с собой еще 9 канистр бензина по галлону каждая (но не больше). Целые канистры можно передавать с одного автомобиля на другой. На какое максимальное расстояние исследователи могут проникнуть в пустыню, не создавая складов топлива, необходимого для возвращения назад",
      "answer": "360"
    },
    {
      "question": "## 22. General Cleaning\n\nA solid joint-stock company occupies three floors in a tower, which has the shape of a parallelepiped and is located in Paris in the La Défense district: the 13th and 14th floors, where the company's offices are located, and the 25th floor, where its board of directors is situated.\n\nTo clean these premises, a cleaning crew is called in, who work for four hours in the offices. Then the crew splits into two parts: half goes to clean the 25th floor, while the other half continues to clean the 13th and 14th floors. After another four hours, the crew stops working: the two lower floors are already completely cleaned, while the 25th floor is scheduled to receive another cleaner the next day, who will complete the work in 8 hours.\n\nAssuming that the time required for cleaning is proportional only to the area of the cleaned premises, how many cleaners were in the crew called by the company?",
      "answer": "8"
    },
    {
      "question": "2. Find the value of the expression\n\n$$\n2^{2}+4^{2}+6^{2}+\\ldots+2018^{2}+2020^{2}-1^{2}-3^{2}-5^{2}-\\ldots-2017^{2}-2019^{2}\n$$",
      "answer": "2041210"
    },
    {
      "question": "31. Let $x, y, z$ be real numbers such that\n$$\nx+y+z=1 \\quad \\text { and } \\quad x^{2}+y^{2}+z^{2}=1\n$$\n\nLet $m$ denote the minimum value of $x^{3}+y^{3}+z^{3}$. Find $9 m$.",
      "answer": "5"
    },
    {
      "question": "Let $p$ and $q$ be different prime numbers. In how many ways can the fraction $1 / p q$ be decomposed into the sum of the reciprocals of 2 different natural numbers?",
      "answer": "4"
    },
    {
      "question": "2. On the written math test, the first problem was not solved by $12 \\%$ of the students, $32 \\%$ solved it partially, and the remaining 14 students solved it correctly. How many students were in this class?",
      "answer": "25"
    },
    {
      "question": "3 [ [sphere, circumscribed around a pyramid]\n\nA regular quadrilateral pyramid is inscribed in a sphere. The radius of the sphere is 1. The plane angle at the vertex of the pyramid is $45^{\\circ}$. Find the lateral surface area of the pyramid.\n\n#",
      "answer": "4"
    },
    {
      "question": "47. As shown in the figure, in rectangle $A B C D$, $M$ is the midpoint of $A B$, $A N$ is an arc of a circle with center $D$ and radius 3, $K N$ is an arc of a circle with center $C$, and the area of the shaded region is $10-\\frac{5 \\pi}{2}$. What is the length of the radius of $K N$?",
      "answer": "1"
    },
    {
      "question": "I2.1 Given that functions $f(x)=x^{2}+r x+s$ and $g(x)=x^{2}-9 x+6$ have the properties that the sum of roots of $f(x)$ is the product of the roots of $g(x)$, and the product of roots of $f(x)$ is the sum of roots of $g(x)$. If $f(x)$ attains its minimum at $x=a$, find the value of $a$.",
      "answer": "3"
    },
    {
      "question": "Example 2.41. $I=\\int_{0}^{1} \\ln x d x$.",
      "answer": "-1"
    },
    {
      "question": "$\\underline{\\text { Folklore }}$\n\nIn triangle $ABC$, the median drawn from vertex $A$ to side $BC$ is four times smaller than side $AB$ and forms an angle of $60^{\\circ}$ with it. Find angle $A$.",
      "answer": "150"
    },
    {
      "question": "3. If for all positive numbers $x, y$, we have $\\sqrt{x}+\\sqrt{y} \\leq a \\sqrt{x+y}$, then the minimum value of the real number $a$ is",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "Let's determine those positive integers which are 14 times as large as the sum of their digits.",
      "answer": "126"
    },
    {
      "question": "Find the largest four-digit number, all digits of which are different and which is divisible by 2, 5, 9, and 11.\n\n#",
      "answer": "8910"
    },
    {
      "question": "22. When proving the Pythagorean theorem, four congruent right triangles can be used to form a square (as shown in Figure 1), and we call such a figure a \"string figure.\"\nA similar method of constructing string figures is often used in rectangles. As shown in Figure 2, draw a line $EF$ through vertex $A$ of rectangle $ABCD$, and draw perpendiculars from points $B$ and $D$ to $EF$, with the feet of the perpendiculars being points $E$ and $F$, respectively, then $\\triangle BEA \\sim \\triangle AFD$.\nAs shown in Figure 3, in rectangle $ABCD$, $AD=5$, $AB=10$, construct congruent right triangles $ADE$ and $CBF$ with $AD$ and $BC$ as the hypotenuses, respectively. If $AE=CF=3$, then $EF^{2}=$ . $\\qquad$",
      "answer": "221"
    },
    {
      "question": "7 chocolates are more expensive than 8 packs of cookies. What is more expensive - 8 chocolates or 9 packs of cookies?\n\n#",
      "answer": "8"
    },
    {
      "question": "## Task 1 - 080621\n\nIn a 6th grade class, no student received a final grade of 5 in mathematics, every ninth student received a grade of 1, every third student received a grade of 2, and every sixth student received a grade of 4.\n\nThe number of students $n$ in this class is known to be: $20<n<40$.\n\nCalculate the number of students who received a final grade of 3!",
      "answer": "14"
    },
    {
      "question": "4. The numbers $x$ and $y$ are such that the equalities $\\sin y + \\cos x = \\sin 3x$ and $\\sin 2y - \\sin 2x = \\cos 4x - \\cos 2x$ hold. What is the smallest value that the sum $\\cos y + \\sin x$ can take?",
      "answer": "-1-\\frac{\\sqrt{2+\\sqrt{2}}}{2}"
    },
    {
      "question": "(10) There are 8 black, white, and yellow chopsticks each. Without looking, take out the chopsticks arbitrarily to ensure that there are at least two pairs of chopsticks of different colors. Then, at least $\\qquad$ chopsticks need to be taken out to achieve this.",
      "answer": "11"
    },
    {
      "question": "Seven people dine around a round table every Saturday evening. How many times is it possible to have dinner if everyone wants to have two new neighbors each time? What is the result for eight people?",
      "answer": "3"
    },
    {
      "question": "## Task 2\n\nSubtract 7 from 19!\n\na) Write down the equation!\n\nb) Justify your solution with the basic problem!\n\na) $19-7=12$; b) $9-7=2$",
      "answer": "12"
    },
    {
      "question": "2. (5 points) If twice a number minus half of the number equals 2013, then the number is $\\qquad$",
      "answer": "1342"
    },
    {
      "question": "25. Find the number that when divided by 2 gives a remainder of 1, when divided by 3 gives a remainder of 2, when divided by 4 gives a remainder of 3, and when divided by 5 gives a remainder of 4.",
      "answer": "59"
    },
    {
      "question": "7. Variant 1. Roma decided to create his own multiplication table. The rows correspond to the numbers 12, $13,14, \\ldots, 60$, and the columns - to the numbers $15,16,17, \\ldots, 40$. In the cells of the table, he wrote the products of the pairs of numbers from the row and column. How many of these products will be even numbers?",
      "answer": "962"
    },
    {
      "question": "4. \"24 Points\" is a familiar math game to many people, the game process is as follows: arbitrarily draw 4 cards from 52 cards (excluding the jokers). Use the numbers on these 4 cards (from 1 to 13, where $A=1, J=11, Q=12, K=13$) to derive 24 using the four basic arithmetic operations. The person who finds the algorithm first wins. The game rules state that all 4 cards must be used, and each card can only be used once. For example, with $2,3,4, Q$, the algorithm $2 \\times Q \\times(4-3)$ can be used to get 24.\nIf in one game, you happen to draw $2,5, \\mathrm{~J}, \\mathrm{Q}$, then your algorithm is: $\\qquad$",
      "answer": "2\\times(11-5)+12"
    },
    {
      "question": "66. A seven-digit number, from left to right, the sum of each pair of adjacent digits is 9, 7, 9, 2, 8, 11. This seven-digit number is $\\qquad$ .",
      "answer": "9072083"
    },
    {
      "question": "6. If $x^{3}-3 \\sqrt{2} x^{2}+6 x-2 \\sqrt{2}-8=0$, find the value of $x^{5}-41 x^{2}+2012$.\n(1 mark) If $x^{3}-3 \\sqrt{2} x^{2}+6 x-2 \\sqrt{2}-8=0$, find the value of $x^{5}-41 x^{2}+2012$.",
      "answer": "1998"
    },
    {
      "question": "4. The given figure is an isosceles triangle $ABC$ with base $\\overline{AB}$ of length $65 \\, \\text{cm}$ and legs $\\overline{AC}$ and $\\overline{BC}$ of length $80 \\, \\text{cm}$. On the base $\\overline{AB}$, a point $D$ is chosen such that the perimeter of triangle $ADC$ is $173 \\, \\text{cm}$, and the perimeter of triangle $DBC$ is $220 \\, \\text{cm}$. What are the lengths of segments $\\overline{CD}, \\overline{AD}$, and $\\overline{DB}$?",
      "answer": "84"
    },
    {
      "question": "What can be the maximum area of a triangle if the length of any of its sides is not greater than 2?",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "Example 2 As shown in Figure 1-10, 5 beads are strung together to form a necklace. Now, there are 3 colors of beads available for use. How many different necklaces can be made?",
      "answer": "39"
    },
    {
      "question": "Problem 3. We are given a combination lock consisting of 6 rotating discs. Each disc consists of digits $0,1,2, \\ldots, 9$, in that order (after digit 9 comes 0 ). Lock is opened by exactly one combination. A move consists of turning one of the discs one digit in any direction and the lock opens instantly if the current combination is correct. Discs are initially put in the position 000000, and we know that this combination is not correct.\n\na) What is the least number of moves necessary to ensure that we have found the correct combination?\n\nb) What is the least number of moves necessary to ensure that we have found the correct combination, if we know that none of the combinations $000000,111111,222222, \\ldots, 999999$ is correct?\n\n(Ognjen Stipetić, Grgur Valentić)",
      "answer": "10^{6}-2"
    },
    {
      "question": "[ Number of divisors and sum of divisors of a number ] equations in integers $\\quad]$\n\nFind a natural number $n$, knowing that it has two prime divisors and satisfies the conditions $\\tau(n)=6$, $\\sigma(n)=28$.",
      "answer": "12"
    },
    {
      "question": "2. Calculate $1+12+123+1234+12345+123456+1234567+12345678+123456789$, the hundreds digit of the sum is . $\\qquad$",
      "answer": "2"
    },
    {
      "question": "5. The lengths of the sides of an isosceles triangle are expressed as natural numbers (in centimeters). The product of the lengths of all sides of this triangle is 2016. What is the perimeter of this triangle?\n\n## Tasks worth 10 points:",
      "answer": "38"
    },
    {
      "question": "1.1 $\\frac{\\frac{1}{a}-\\frac{1}{b+c}}{\\frac{1}{a}+\\frac{1}{b+c}}\\left(1+\\frac{b^{2}+c^{2}-a^{2}}{2 b c}\\right): \\frac{a-b-c}{a b c}$;\n\n$$\na=0.02, b=-11.05, c=1.07\n$$\n\n$1.2\\left(\\frac{1}{t^{2}+3 t+2}+\\frac{2 t}{t^{2}+4 t+3}+\\frac{1}{t^{2}+5 t+6}\\right)^{2} \\frac{(t-3)^{2}+12 t}{2}$.",
      "answer": "0.1"
    },
    {
      "question": "14. Given that the sum of the sines of any 5 interior angles and the cosines of the remaining 4 interior angles of a convex nonagon equals a constant value $\\lambda$. If one of the nine interior angles is $120^{\\circ}$, find the value of the constant $\\lambda$.",
      "answer": "1-\\frac{\\sqrt{3}}{2}"
    },
    {
      "question": "4. Given that $n$ is a positive integer, if $\\frac{n^{2}+3 n-10}{n^{2}+6 n-16}$ is a reduced fraction, then the value of this fraction is $\\qquad$ .",
      "answer": "\\frac{8}{11}"
    },
    {
      "question": "Problem 5.8. Inside a large triangle with a perimeter of 120, several segments were drawn, dividing it into nine smaller triangles, as shown in the figure. It turned out that the perimeters of all nine small triangles are equal to each other. What can they be equal to? List all possible options.\n\nThe perimeter of a figure is the sum of the lengths of all its sides.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_5b323dccc142e96a9907g-12.jpg?height=256&width=323&top_left_y=1257&top_left_x=565)",
      "answer": "40"
    },
    {
      "question": "On the diagonal $B D$ of the rectangular trapezoid $A B C D\\left(\\angle D=90^{\\circ}, B C \\| A D\\right)$, a point $Q$ is taken such that $B Q: Q D=1: 3$. A circle with center at point $Q$ touches the line $A D$ and intersects the line $B C$ at points $P$ and $M$. Find the length of the side $A B$, if $B C=9, A D=8, P M=4$.",
      "answer": "3"
    },
    {
      "question": "5. Five pairs of twin siblings participate in the activities of $k$ groups. If the rules are: (1) Twin siblings are not in the same group; (2) Any two people who are not twins have participated in exactly one group together; (3) One person participates in only two groups. Then the minimum value of $k$ is $\\qquad$ .",
      "answer": "14"
    },
    {
      "question": "5. You and I play the following game on an $8 \\times 8$ square grid of boxes: Initially, every box is empty. On your turn, you choose an empty box and draw an $X$ in it; if any of the four adjacent boxes are empty, you mark them with an $X$ as well. (Two boxes are adjacent if they share an edge.) We alternate turns, with you moving first, and whoever draws the last $X$ wins. How many choices do you have for a first move that will enable you to guarantee a win no matter how I play?",
      "answer": "0"
    },
    {
      "question": "2. If $\\sin x+\\cos x=\\frac{\\sqrt{2}}{2}$, then $\\sin ^{3} x+\\cos ^{3} x=$ $\\qquad$",
      "answer": "\\frac{5\\sqrt{2}}{8}"
    },
    {
      "question": "I5.3 It is known that $b^{16}-1$ has four distinct prime factors, determine the largest one, denoted by $c$",
      "answer": "257"
    },
    {
      "question": "B3 In the figure, two circles with radius 6 are drawn. The centers of the circles are also 6 units apart.\n\nDetermine the area of the shaded region, which is bounded by two circular arcs and a common tangent.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2582a2b087f6953c1edag-2.jpg?height=363&width=537&top_left_y=721&top_left_x=1316)",
      "answer": "36-9\\sqrt{3}-6\\pi"
    },
    {
      "question": "A quantity of grey paint has a mass of $12 \\mathrm{~kg}$. The grey paint is a mixture of black paint and white paint, of which $80 \\%$ by mass is white paint. More white paint is added until the mixture is $90 \\%$ white paint by mass. What is the resulting total mass of the paint, in $\\mathrm{kg}$ ?",
      "answer": "24"
    },
    {
      "question": "1. Call a positive integer a hussel number if:\n\n(1) All digits are not equal to 0.\n\n(2) The number is divisible by 11.\n\n(3) The number is divisible by 12. If you rearrange the digits in any other random order, you always get a number that is divisible by 12.\n\nHow many 5-digit hussel numbers are there?",
      "answer": "2"
    },
    {
      "question": "5. In trapezoid $A B C D$ with bases $A D / / B C$, the diagonals intersect at point $E$. It is known that the areas $S(\\triangle A D E)=12$ and $S(\\triangle B C E)=3$. Find the area of the trapezoid.",
      "answer": "27"
    },
    {
      "question": "1. In a contest, $\\mathbf{30}$ problems are given. For each correct answer, 5 points are awarded, and for each wrong answer, 3 points are deducted. How many correct answers did a student give who scored 118 points?",
      "answer": "26"
    },
    {
      "question": "8. Let $a, b$ be positive integers, and $a-b \\sqrt{3}=(2-\\sqrt{3})^{100}$, then the units digit of $a b$ is",
      "answer": "2"
    },
    {
      "question": "In the picture, you can see a pool with a long step on one of its walls. We started filling the empty pool with a constant flow and observed the water level. After\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_ac63ecbffe7e6b0bc0e7g-2.jpg?height=309&width=614&top_left_y=451&top_left_x=755)\n\n8 min, the water level had risen to a height of $20 \\mathrm{~cm}$ and had not yet reached the level of the step. After $23 \\mathrm{~min}$ of filling, the water level was at $55 \\mathrm{~cm}$ and the step had already been submerged for some time. After 35.5 min of filling, the pool was filled to a height of $80 \\mathrm{~cm}$. At what height $h$ from the bottom of the pool is the step located?\n\n(L. Šimünek)",
      "answer": "45"
    },
    {
      "question": "5. [5 points] Given the numbers $\\log _{\\sqrt{\\frac{x}{3}+3}}(6 x-14), \\log _{6 x-14}(x-1)^{2}, \\log _{x-1}\\left(\\frac{x}{3}+3\\right)$. For which $x$ are two of these numbers equal, and the third one less than them by 1?",
      "answer": "3"
    },
    {
      "question": "Problem 2. In class 4-b, there were 24 students. Each student can choose whether to study the subjects of informatics and German. 15 students study informatics, 12 study German, and 8 students do not study either informatics or German. How many students from class 4-b study both informatics and German at the same time?",
      "answer": "11"
    },
    {
      "question": "Problem 2. $n$ mushroom pickers went to the forest and brought a total of 162 mushrooms (each brought at least one mushroom home). Boy Petya, upon learning this, declared: \"Some two of them must have brought the same number of mushrooms!\" For what smallest $n$ will Petya definitely be right? Don't forget to justify your answer.",
      "answer": "18"
    },
    {
      "question": "16. If a positive integer cannot be written as the difference of two square numbers, then the integer is called a \"cute\" integer. For example, 1,2 and 4 are the first three \"cute\" integers. Find the $2010^{\\text {th }}$ \"cute\" integer.\n(Note: A square number is the square of a positive integer. As an illustration, 1,4,9 and 16 are the first four square numbers.)",
      "answer": "8030"
    },
    {
      "question": "Example 4 Find the largest constant $k$, such that for $x, y, z \\in \\mathbf{R}^{+}$, we have\n$$\n\\frac{x}{\\sqrt{y+z}}+\\frac{y}{\\sqrt{z+x}}+\\frac{z}{\\sqrt{x+y}} \\leqslant k \\sqrt{x+y+z} .\n$$",
      "answer": "\\frac{\\sqrt{6}}{2}"
    },
    {
      "question": "Find all primes of the form $a^{2}-1$, with $a \\geqslant 2$ natural.",
      "answer": "3"
    },
    {
      "question": "How should a rook move across the chessboard to visit each square exactly once and make the fewest number of turns?\n\n#",
      "answer": "14"
    },
    {
      "question": "20. Let $f: \\mathbb{Q} \\backslash\\{0,1\\} \\rightarrow \\mathbb{Q}$ be a function such that\n$$\nx^{2} f(x)+f\\left(\\frac{x-1}{x}\\right)=2 x^{2}\n$$\nfor all rational numbers $x \\neq 0,1$. Here $\\mathbb{Q}$ denotes the set of rational numbers. Find the value of $f\\left(\\frac{1}{2}\\right)$.\n20. Answer: 1\nSubstituting $x=\\frac{1}{2},-1,2$, we get\n$$\n\\begin{aligned}\n\\frac{1}{4} f\\left(\\frac{1}{2}\\right)+f(-1) & =\\frac{1}{2}, \\\\\nf\\left(\\frac{1}{2}\\right)+f(-1)+f(2) & =2, \\\\\n4 f(2) & =8 .\n\\end{aligned}\n$$\n\nSolving these equations, we get $f\\left(\\frac{1}{2}\\right)=1$. In fact the same method can be used to determine $f$. Letting $x=z, \\frac{z-1}{z}, \\frac{1}{1-z}$, we get\n$$\n\\begin{array}{l}\nz^{2} f(z)+f\\left(\\frac{z-1}{z}\\right)=2 z^{2}, \\\\\n\\left(\\frac{z-1}{z}\\right)^{2} f\\left(\\frac{z-1}{z}\\right)+f\\left(\\frac{1}{11-z}\\right)=2\\left(\\frac{z-1}{z}\\right)^{2}, \\\\\nf(z)+\\frac{1}{(1-z)^{2}} f\\left(\\frac{1}{1-z}\\right)=\\frac{z}{2(1-z)^{2}} . \\\\\n\\end{array}\n$$\n\nUsing Cramer's rule, we can solve this system of linear equations in the unknowns $f(z), f\\left(\\frac{z-1}{z}\\right), f\\left(\\frac{1}{1-z}\\right)$. We obtain\n$$\nf(z)=1+\\frac{1}{(1-z)^{2}}-\\frac{1}{z^{2}} .\n$$\n\nIndeed one can easily check that it satisfies the given functional equation.",
      "answer": "1"
    },
    {
      "question": "Three. (50 points)\nIn a square with a side length of 4, 2002 points are placed. How many pairs of points are there with a distance less than 3 (find the largest specific number)?",
      "answer": "500000"
    },
    {
      "question": "The image shows a payment order for electricity to a certain energy sales company.\n\n![](https://cdn.mathpix.com/cropped/2024_05_08_22585010b2534c7ab9ebg-38.jpg?height=646&width=1380&top_left_y=1163&top_left_x=339)\n\nEach month, the client provides the company with readings from a three-tariff meter installed in the apartment. The current month's readings are subtracted from the corresponding readings of the previous month to obtain the actual consumption for the month in each of the three tariff zones (peak, night, semi-peak). The consumption in each zone is then multiplied by the price per kilowatt-hour in that zone. By summing these amounts, the client gets the total payment for the month. In this example, the client will pay 660 rubles and 72 kopecks.\n\nThe company keeps track of electricity consumption and payments using the data provided by the client. The problem is that the company sometimes confuses the six numbers, rearranging them in any order, although it ensures that the current reading remains greater than the previous one. As a result, the company's calculation can be incorrect. If the company believes that the client owes more than they paid, it demands the difference.\n\nUsing the data from the illustrated receipt, find:\n\na) the maximum possible additional payment for March 2013 that the company may demand from the client;\n\nb) the expected value of the difference between the amount the company will charge and the amount the client has paid.\n\n#",
      "answer": "397.34"
    },
    {
      "question": "The side length of the equilateral triangle $ABC$ is $6 \\, \\text{cm}$. Starting from vertex $C$, a bug moves uniformly towards vertex $A$ at a speed of $4 \\, \\text{mm/s}$. At the same time, another bug starts from vertex $B$ and moves towards vertex $C$ at a speed of $3 \\, \\text{mm/s}$. How much time after their start will they be closest to each other, and what will this distance be?",
      "answer": "34.17"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 2} \\frac{\\operatorname{arctg}\\left(x^{2}-2 x\\right)}{\\sin 3 \\pi x}$",
      "answer": "\\frac{2}{3\\pi}"
    },
    {
      "question": "We want to draw a digit on each face of two identical cubes so that by appropriately rotating the cubes and placing them next to each other, we can read any day of the month, including single-digit days with a leading zero for uniformity. How many ways can we choose the digits to be drawn on each cube? (The relative positions of the six digits on one cube are not considered in the counting.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_02_9b382d5b0c4e2d739941g-1.jpg?height=353&width=591&top_left_y=398&top_left_x=756)",
      "answer": "10"
    },
    {
      "question": "[ Inequalities Problems. Case Analysis]\n\nLong, long ago, nine identical books cost 11 rubles and some kopecks, and thirteen such books cost 15 rubles and some kopecks.\n\nHow much did one book cost\n\n#",
      "answer": "1"
    },
    {
      "question": "## Aufgabe 2\n\n$$\n\\begin{array}{ll}\n12+5-4 & 13-3+2 \\\\\n17+2+0 & 17-5-1\n\\end{array}\n$$",
      "answer": "13"
    },
    {
      "question": "1. Let $A B C D$ be a convex quadrilateral with $A B=a, B C=b, C D=c$ and $D A=d$. Suppose\n\n$$\na^{2}+b^{2}+c^{2}+d^{2}=a b+b c+c d+d a\n$$\n\nand the area of $A B C D$ is 60 square units. If the length of one of the diagonals is 30 units, determine the length of the other diagonal.",
      "answer": "4"
    },
    {
      "question": "## Problem 1\n\nAn obtuse angled triangle has integral sides and one acute angle is twice the other. Find the smallest possible perimeter.",
      "answer": "77"
    },
    {
      "question": "6. On the shore of a round island called Somewhere, there are 20 villages, each inhabited by 20 wrestlers. A tournament was held where each wrestler faced all wrestlers from all other villages. Village A is considered stronger than Village B if at least $k$ matches between wrestlers from these villages end with a victory for a wrestler from Village A. It turned out that each village is stronger than the next one in the clockwise direction. What is the maximum value that $k$ can have? (All wrestlers have different strengths, and the stronger wrestler always wins in a match.)",
      "answer": "290"
    },
    {
      "question": "Simplify the following expressions:\n\n$$\n[1+(\\operatorname{tg} x-\\operatorname{cotg} x) \\sin 2 x] \\cos x+\\cos 3 x\n$$\n\n$$\n\\frac{\\operatorname{tg}^{2} x-\\sin ^{2} x}{\\operatorname{cotg}^{2} x-\\cos ^{2} x}\n$$",
      "answer": "0"
    },
    {
      "question": "9.5. On a checkerboard of size $8 \\times 8$, 8 checkerboard ships of size $1 \\times 3$ are placed such that no two cells occupied by different ships share any points. One shot is allowed to pierce all 8 cells of one row or one column. What is the minimum number of shots needed to guarantee hitting at least one ship?",
      "answer": "2"
    },
    {
      "question": "Jóska and Gábor every summer watched the license plates of 51 cars passing by each day, and if all four digits of the number were different, Jóska got 1 point, otherwise Gábor did. After 40 days, Gábor claimed that he won because he had more points on 21 days. However, Jóska claims that if they added up the results of all 40 days, he would have more points. - In a long game, who has a better chance of accumulating more points? Based on what assumptions did we answer the question?",
      "answer": "5040"
    },
    {
      "question": "Let's find a perfect square consisting of 4 digits, knowing that the number formed by the first two digits exceeds the number formed by the last two digits by one.",
      "answer": "8281"
    },
    {
      "question": "3. In all other cases - 0 points\n\n## *Important: the numerical assessment of the free area (solution) is not the only possible one, for example, the \"gap\" can be more than 10 m.\n\n## Assignment 2 (12 points)\n\nCrocodile Gena and Old Lady Shapoklyak entered into a futures contract, according to which Gena agreed to invest in the 1st project an amount equal to $p_{1}>0$ thousand rubles, and after a year receive $x_{1}$ thousand rubles such that\n\n$$\n4 x_{1}-3 p_{1}-44=0\n$$\n\nand at the same time Gena agreed to invest in the 2nd project an amount equal to $p_{2}>0$ thousand rubles, and after a year receive $x_{2}$ thousand rubles such that\n\n$$\np_{2}^{2}-12 p_{2}+x_{2}^{2}-8 x_{2}+43=0\n$$\n\nAccording to the terms of the contract, the parameters $p_{1}, x_{1}, p_{2}, x_{2}$ were chosen so that the value\n\n$$\nd=\\sqrt{\\left(x_{1}-x_{2}\\right)^{2}+\\left(p_{1}-p_{2}\\right)^{2}}\n$$\n\nwas minimized. Determine:\n\n1) the minimum value of the quantity $d$;\n2) is the deal profitable for Gena? Find the amount of profit or loss for Gena if he agrees to this deal.",
      "answer": "13080"
    },
    {
      "question": "## Task A-4.2.\n\nExactly three interior angles of a convex polygon are obtuse. What is the maximum number of sides that this polygon can have?",
      "answer": "6"
    },
    {
      "question": "4. (3 points) The rules of a musical instrument competition stipulate: The preliminary round is divided into four rounds, and the average score of the four rounds must be no less than 96 to advance to the final. Xiao Guang's scores in the first three rounds are $95$, $97$, and $94$. Therefore, to advance to the final, his score in the fourth round must be at least $\\qquad$ points.",
      "answer": "98"
    },
    {
      "question": "7.3.1. (12 points) Find the sum of all real roots of the equation\n\n$$\n\\sin \\left(\\pi\\left(x^{2}-x+1\\right)\\right)=\\sin (\\pi(x-1))\n$$\n\nbelonging to the interval $[0 ; 2]$.",
      "answer": "4.73"
    },
    {
      "question": "I3.1 Given that $\\frac{2 x-3}{x^{2}-x}=\\frac{A}{x-1}+\\frac{B}{x}$, where $A$ and $B$ are constants. If $S=A^{2}+B^{2}$, find the value of $S$",
      "answer": "10"
    },
    {
      "question": "(5) The minimum value of the function $f(x)=\\sqrt{x^{2}+9}+\\sqrt{x^{2}-8 x+17}$ is",
      "answer": "4\\sqrt{2}"
    },
    {
      "question": "# Task 3. (12 points)\n\nEgorov decided to open a savings account to buy a car worth 900000 rubles. The initial deposit amount is 300000 rubles. After one month and subsequently every month, Egorov plans to top up his account by 15000 rubles. The bank accrues interest monthly at an annual rate of $12 \\%$. Interest accrued for the month is credited to the account, and in the following month, interest is also accrued on this interest. What is the minimum number of months required for the account to have a sufficient amount to buy the car?",
      "answer": "29"
    },
    {
      "question": "3. In the sum $(1-2-3+4)+(5-6-7+8)+\\ldots+(2009-2010-2011+2012)+$ (2013-2014-2015), all numbers that end in 0 were crossed out. Find the value of the resulting expression.",
      "answer": "-3026"
    },
    {
      "question": "1. Determine the number of pairs of positive integers $(a, b)$, with $a \\leq b$, for which $\\operatorname{lcm}(a, b)=2004$.\n\n$\\operatorname{lcm}(a, b)$ denotes the least common multiple of $a$ and $b$.\n\nExample: $\\operatorname{lcm}(18,24)=72$.",
      "answer": "23"
    },
    {
      "question": "20. A convex quadrilateral paper sheet has $n$ points inside it. Together with the 4 vertices of the quadrilateral, there are $n+4$ points, and no three points are collinear. Using these points as vertices, the quadrilateral is cut into several triangular paper pieces. The maximum number of triangles that can be cut is 1992, then $n=$",
      "answer": "9955"
    },
    {
      "question": "A sequence of numbers $1, 4, 7, 10, \\cdots, 697, 700$ follows the rule: the first number is 1, and each subsequent number is 3 more than the previous one, up to 700. If all these numbers are multiplied together, find the number of trailing zeros in the resulting product (for example, the number of trailing zeros in 12003000 is 3).",
      "answer": "60"
    },
    {
      "question": "6. A motorcyclist traveled 980 km over three days. In the first two days, he traveled 725 km, and on the second day, he traveled 123 km more than on the third day. How many kilometers did he travel each of these three days?",
      "answer": "347"
    },
    {
      "question": "$7.73 \\sqrt{\\log _{0.04} x+1}+\\sqrt{\\log _{0.2} x+3}=1$.\n\nTranslate the above text into English, keeping the original text's line breaks and format, and output the translation result directly.\n\n$7.73 \\sqrt{\\log _{0.04} x+1}+\\sqrt{\\log _{0.2} x+3}=1$.",
      "answer": "25"
    },
    {
      "question": "Problem 1. Children went to the forest to pick mushrooms. If Anya gives half of her mushrooms to Vitya, all the children will have the same number of mushrooms, and if instead Anya gives all her mushrooms to Sasha, Sasha will have as many mushrooms as all the others combined. How many children went mushroom picking?",
      "answer": "6"
    },
    {
      "question": "14. (15 points) The distance between location A and location B is 360 kilometers. A truck loaded with 6 boxes of medicine is driving from location A to location B, while at the same time, a motorcycle starts from location B and heads towards the truck. The truck's speed is 40 kilometers/hour, and the motorcycle's speed is 80 kilometers/hour. When the motorcycle meets the truck, it unloads 2 boxes of medicine and returns to location B. It then immediately turns around... Each time the motorcycle meets the truck, it unloads 2 boxes of medicine and returns to location B. How long will it take to transport all 6 boxes of medicine to location B? How far will the motorcycle have traveled in total by then? (Neglect the time for loading and unloading the medicine.)",
      "answer": "8\\frac{2}{3}"
    },
    {
      "question": "11. (5 points) As shown in the figure, the \"L\" shaped paper piece with a perimeter of 52 centimeters can be divided into two identical rectangles along the dotted line. If the longest side is 16 centimeters, then the area of the \"L\" shaped paper piece is $\\qquad$ square centimeters.",
      "answer": "120"
    },
    {
      "question": "10. (20 points) Given the function\n$$\nf(x)=x^{4}+a x^{3}+b x^{2}+a x+1(a, b \\in \\mathbf{R})\n$$\n\nhas at least one root. Find the minimum value of $a^{2}-b$.",
      "answer": "1"
    },
    {
      "question": "【Question 10】\nAs shown in the figure, in square $A B C D$, extend $B A$ to $G$ such that $A G=B D$. Then the degree measure of $\\angle B C G$ is $\\qquad$ degrees.",
      "answer": "67.5"
    },
    {
      "question": "4. The function $f: \\mathbb{N} \\rightarrow \\mathbb{N}$ is given such that $f(n)=1$ if $n$ is odd, and $f(n)=k$ for every even number $n=2^{k} l$, where $k$ is a natural number and $l$ is an odd number. Determine the largest natural number $n$ for which\n\n$$\nf(1)+f(2)+\\ldots+f(n) \\leqq 123456 .\n$$",
      "answer": "82307"
    },
    {
      "question": "27. Find the digit that has the property that if it is appended to the end of any natural number, the resulting number is equal to the sum of three terms, the first of which is the original number, the second is the number represented by the sought digit, and the third is the product of the first two terms.",
      "answer": "9"
    },
    {
      "question": "Misha thought of a number not less than 1 and not more than 1000. Vasya is allowed to ask only such questions, to which Misha can answer \"yes\" or \"no\" (Misha always tells the truth). Can Vasya determine the guessed number in 10 questions?",
      "answer": "183"
    },
    {
      "question": "11. Given that when $x \\in[0, \\sqrt{3}]$, $\\left|x^{3}+\\frac{3}{2}(1-a) x^{2}-3 a x+b\\right| \\leq 1$ always holds,\n(1) Prove: The equation $x^{3}+\\frac{3}{2}(1-a) x^{2}-3 a x+b=0$ has three real roots $x_{1}<x_{2}<x_{3}$,\n(2) Find $\\left(x_{1}^{2}-2-x_{2}\\right)\\left(x_{2}^{2}-2-x_{3}\\right)\\left(x_{3}^{2}-2-x_{1}\\right)$.",
      "answer": "-9"
    },
    {
      "question": "4. For a positive integer $n$, we denote by $P(n)$ the product of the positive divisors of $n$. For example, $P(20)=8000$. The positive divisors of 20 are 1, 2, 4, 5, 10, and 20, with the product $1 \\cdot 2 \\cdot 4 \\cdot 5 \\cdot 10 \\cdot 20=8000$.\n\n(a) Find all positive integers $n$ for which $P(n)=15 n$.\n\n(b) Show that there are no positive integers $n$ for which $P(n)=15 n^{2}$.",
      "answer": "15"
    },
    {
      "question": "1. Given a set of data consisting of seven positive integers, the only mode is 6, and the median is 4. Then the minimum value of the sum of these seven positive integers is $\\qquad$ .",
      "answer": "26"
    },
    {
      "question": "2. (5 points) On Valentine's Day, every student in the school gave each female student a valentine. It turned out that the number of valentines was 28 more than the total number of students. How many valentines were given?",
      "answer": "60"
    },
    {
      "question": "16 (12 points) Given in the cube $A B C D$ $A_{1} B_{1} C_{1} D_{1}$, $O, E, F, G$ are the midpoints of $B D$, $B B_{1}$, $A_{1} D_{1}$, $D_{1} C_{1}$ respectively, and $A B=1$. Find the volume of the tetrahedron $O E F G$.",
      "answer": "\\frac{5}{48}"
    },
    {
      "question": "2. (7 points) In Sun City, they exchange a honey cake for 6 pretzels, and for 9 pretzels, they give 4 doughnuts. How many doughnuts will they give for 3 honey cakes? Explain your answer.",
      "answer": "8"
    },
    {
      "question": "Example 1. Derive the Taylor formula for $n=2$ for the function $f(x, y)=x^{y}$ in the neighborhood of the point $M_{0}(1,1)$ and approximate $1.1^{1.02}$.",
      "answer": "1.102"
    },
    {
      "question": "Prokopenko D:\n\nAround triangle $A B C$, a circle $\\Omega$ is described. Let $L$ and $W$ be the points of intersection of the angle bisector of angle $A$ with side $B C$ and circle $\\Omega$, respectively. Point $O$ is the center of the circumcircle of triangle $A C L$. Restore triangle $A B C$ if the circle $\\Omega$ and points $W$ and $O$ are given.",
      "answer": "0"
    },
    {
      "question": "\\section*{Problem 6B - 111236B}\n\n50 white and 50 black balls are to be distributed into two indistinguishable urns in such a way that neither urn remains empty and all balls are used.\n\nHow should the balls be distributed between the two urns so that the probability of drawing a white ball when one of the two urns is chosen at random and a ball is drawn from it is as high as possible?\n\n\\section*{Hints:}\n\n1. In classical probability theory, the probability \\(p\\) of an event is defined as the quotient of the number \\(g\\) of favorable cases for this event and the total number \\(m\\) of possible cases, i.e., \\(p=\\frac{g}{m}\\).\n\n2. Therefore, the probability of drawing a white ball (blindly) from an urn that contains a total of \\(u\\) balls, of which \\(w\\) are white, is set to \\(p=\\frac{w}{u}\\).\n\n3. If there are two urns, with the probability of drawing a white ball being \\(p_{1}\\) and \\(p_{2}\\) respectively, then the probability of the combined event \"selecting one of the two urns and drawing a white ball from the selected urn\" is \\(p=\\frac{1}{2} p_{1}+\\frac{1}{2} p_{2}\\).",
      "answer": "\\frac{74}{99}"
    },
    {
      "question": "\\section*{Problem 5 - 071035}\n\nFor which real \\(a\\) does the sum of the squares of the solutions of the equation \\(x^{2}+a x+a-2=0\\) attain its smallest value?",
      "answer": "1"
    },
    {
      "question": "2. In a small fishing game, each time you catch a small fish or a small turtle. If you catch a small fish, you get 5 points, and if you catch a small turtle, you lose 2 points. Luo Xiahei played 17 times, and his score increased from 100 points to 157 points. Out of these 17 times, he caught a small fish $\\qquad$ times.",
      "answer": "13"
    },
    {
      "question": "2. $[\\mathbf{3}] A, B, C$, and $D$ are points on a circle, and segments $\\overline{A C}$ and $\\overline{B D}$ intersect at $P$, such that $A P=8$, $P C=1$, and $B D=6$. Find $B P$, given that $B P<D P$.",
      "answer": "2"
    },
    {
      "question": "42. Someone asked Uncle Wang: \"How many pots of flowers are there in your garden?\" Uncle Wang said: \"Half are roses, a quarter are lilies, one seventh are peonies, and less than 6 pots are roses.\" Therefore, Uncle Wang's garden has $\\qquad$ pots of flowers.",
      "answer": "28"
    },
    {
      "question": "4. In triangle $A B C$, points $M$ and $N$ are the midpoints of sides $A C$ and $B C$ respectively. It is known that the point of intersection of the medians of triangle $A M N$ is the point of intersection of the altitudes of triangle $A B C$. Find the angle $A B C$.",
      "answer": "45"
    },
    {
      "question": "Example 2.7 Among the integers from 1 to 1000, how many integers are divisible by 4 but not by 3 or 10?",
      "answer": "133"
    },
    {
      "question": "1. (16 points) Solve the equation $x-5=\\frac{3 \\cdot|x-2|}{x-2}$. If the equation has multiple roots, write their sum in the answer.\n\n#",
      "answer": "8"
    },
    {
      "question": "Arnaldo, Bernaldo, Cernaldo, Dernaldo, and Ernaldo are students from different parts of Brazil who were chosen to represent their country in international olympiads. After several weeks of training, some friendships were formed. We then asked each of them how many friends they had made in the group. Arnaldo, Bernaldo, Cernaldo, and Dernaldo answered, respectively, that they had made 1, 2, 3, and 4 friends within the group. How many of the group members are friends with Ernaldo?",
      "answer": "2"
    },
    {
      "question": "5. In the quadrilateral pyramid $P-ABCD$, $BC \\parallel AD$, $AD \\perp AB$, $AB=2\\sqrt{3}$, $AD=6$, $BC=4$, $PA=PB=PD=4\\sqrt{3}$. Then the surface area of the circumscribed sphere of the tetrahedron $P-BCD$ is $\\qquad$ .",
      "answer": "80\\pi"
    },
    {
      "question": "# Problem 7. (4 points)\n\nTwo cubes with an edge of $12 \\sqrt[4]{\\frac{8}{11}}$ share a common face. A section of one of these cubes by a certain plane is a triangle with an area of 16. The section of the other by the same plane is a quadrilateral. What is the maximum value that its area can take?",
      "answer": "128"
    },
    {
      "question": "$82009 \\times 1911$ chessboard's diagonal crosses $\\qquad$ squares.\n\nThe diagonal of an $82009 \\times 1911$ square grid crosses $\\qquad$ grid cells.",
      "answer": "83871"
    },
    {
      "question": "4. Let $p(x)$ be a polynomial of degree $2n$, $p(0)=p(2)=\\cdots=p(2n)=0, p(1)=p(3)=\\cdots=$ $p(2n-1)=2, p(2n+1)=-30$. Find $n$.",
      "answer": "2"
    },
    {
      "question": "## Task 1 - 100931\n\nGünter spent a number of days in a camp with his FDJ group during his vacation. Every day, exactly two students from his group were assigned to table duty in the morning and exactly two students in the afternoon. Over the days, all students in his group were assigned to this table duty equally often.\n\nFurthermore, the following is known:\n\n(1) Günter was assigned to table duty on exactly 6 days.\n\n(2) If he had table duty in the afternoon, he did not have it in the morning.\n\n(3) He did not have table duty in the afternoon on exactly 13 of these days.\n\n(4) He did not have table duty in the morning on exactly 11 of these days.\n\nHow many students were in Günter's group?",
      "answer": "10"
    },
    {
      "question": "## Task 3 - 010913\n\nTo maintain the cutting ability of tools when machining metals, cooling is often done with an emulsion of lubricated mineral oil (density $0.98 \\mathrm{~g} / \\mathrm{cm}^{3}$) and as soft water as possible (density $1.0 \\mathrm{~g} / \\mathrm{cm}^{3}$). The mixture must have a density of $0.996 \\mathrm{g} / \\mathrm{cm}^{3}$ for cutting tools of higher hardness, and a density of $0.992 \\mathrm{~g} / \\mathrm{cm}^{3}$ for grinding operations. How many liters of lubricated mineral oil and how many liters of soft water are needed for 10 liters of emulsion in each case?",
      "answer": "81"
    },
    {
      "question": "In the drawing below, $\\triangle A B C$ is an equilateral triangle and $C D$ is both a height of the triangle and a diameter of the circle. If $A B=10 \\mathrm{~cm}$, determine the shaded area.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4747cb56a5540358921fg-02.jpg?height=426&width=491&top_left_y=495&top_left_x=680)",
      "answer": "\\frac{100\\pi-75\\sqrt{3}}{4}"
    },
    {
      "question": "5. (1 mark) A positive integer is said to be a \"palindrome\" if it reads the same from left to right as from right to left. For example 2002 is a palindrome. Find the sum of all 4-digit palindromes.\n(1 分) A positive integer is said to be a \"palindrome\" if it reads the same from left to right as from right to left. For example 2002 is a palindrome. Find the sum of all 4-digit palindromes.",
      "answer": "495000"
    },
    {
      "question": "5. Let $E$ be a moving point inside square $ABCD$. It is known that the minimum value of the sum of the distances from $E$ to points $A$, $B$, and $C$ is $\\sqrt{2}+\\sqrt{6}$. Try to find the side length of this square.",
      "answer": "2"
    },
    {
      "question": "The rhombus $ABCD$ is divided into two equilateral triangles by its diagonal $BD$. A point $P$ is given on the segment $AD$, and a point $Q$ is on the segment $CD$ such that $\\angle PBQ = 60^{\\circ}$. What are the other two angles of the triangle $PBQ$?",
      "answer": "60"
    },
    {
      "question": "16. (15 points) Given the functions\n$$\nf(x)=a \\sin x(a \\in \\mathbf{R}), g(x)=\\mathrm{e}^{x} \\text {. }\n$$\n(1) If $00, m0$ always holds, find the minimum value of $k$.",
      "answer": "2"
    },
    {
      "question": "Problem 5.1. After a football match, the coach lined up the team as shown in the figure, and commanded: \"Run to the locker room, those whose number is less than that of any of their neighbors.\" After several people ran away, he repeated his command. The coach continued until only one player was left. What is Igor's number, if it is known that after he ran away, 3 people remained in the line? (After each command, one or several players ran away, after which the line closed, and there were no empty spaces between the remaining players.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_adefdf2067937cb67fc7g-08.jpg?height=386&width=1072&top_left_y=485&top_left_x=191)",
      "answer": "5"
    },
    {
      "question": "\\section*{Problem 1 - 181041}\n\nWhat are the last two digits (in the usual decimal digit notation) of the number \\(x\\) that satisfies the equation\n\n\\[\n\\log _{13}\\left[\\log _{12}\\left(\\log _{11} x\\right)\\right]=1\n\\]",
      "answer": "21"
    },
    {
      "question": "Two parallel lines are intersected by a third. Find the angle between the bisectors of the internal same-side angles.\n\n#",
      "answer": "90"
    },
    {
      "question": "31. The blackboard has the following 9 numbers written on it. Each time, the teacher arbitrarily erases two numbers, and writes their sum on the blackboard. When only one number remains on the blackboard, this number is $\\qquad$.",
      "answer": "90"
    },
    {
      "question": "9.6. In a certain company, there are 100 shareholders, and any 66 of them own no less than $50 \\%$ of the company's shares. What is the largest percentage of all shares that one shareholder can own?",
      "answer": "25"
    },
    {
      "question": "In triangle $ABC$, construct points $C_1$ on side $AB$ and $A_1$ on side $BC$ such that the lines $AA_1$ and $CC_1$ intersect at point $P$ where $AP / PA_1 = 3 / 2$ and $CP / PC_1 = 2 / 1$. In what ratio does point $C_1$ divide side $AB$?",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "10. Let $A B C D$ be a square of side length 13 . Let $E$ and $F$ be points on rays $A B$ and $A D$, respectively, so that the area of square $A B C D$ equals the area of triangle $A E F$. If $E F$ intersects $B C$ at $X$ and $B X=6$, determine $D F$.",
      "answer": "\\sqrt{13}"
    },
    {
      "question": "2. If the difference between the maximum and minimum elements of the real number set $\\{1,2,3, x\\}$ equals the sum of all elements in the set, then the value of $x$ is $\\qquad$ .",
      "answer": "-\\frac{3}{2}"
    },
    {
      "question": "2. Each card has a number on one side and a letter on the other side (two cards that have the same letter on one side do not necessarily have the same number on the other side, and vice versa). On the table, there are cards that show the following on their visible side:\n\n$$\nM, A, T, E, M, A, T, И, K, A, 2,0,1,8\n$$\n\nVlada claims the following: \"If a card has a vowel, then the other side has an even number.\" Miljan wants to check Vlada's claim. How many and which cards does he need to turn over to verify the correctness of the statement?",
      "answer": "6"
    },
    {
      "question": "Tokarev S.i.\n\nIn a line, all integers from 1 to 100 are written in an unknown order. With one question about any 50 numbers, you can find out the order of these 50 numbers relative to each other. What is the minimum number of questions needed to definitely determine the order of all 100 numbers?",
      "answer": "5"
    },
    {
      "question": "$A$ says to $B$: »I will tell you how old you are if you answer the following question. Multiply the digit in the tens place of your birth year by 5, add 2 to this product, and then multiply the resulting sum by 2. Add the units digit of your birth year to this new product and tell me the number you get!«\n\n$B$ replies: »43«, whereupon $A$ immediately exclaims: »You are 16 years old!« How did $A$ know this?",
      "answer": "16"
    },
    {
      "question": "1. A tractor is pulling a very long pipe on sled runners. Gavrila walked along the entire pipe at a constant speed in the direction of the tractor's movement and counted 210 steps. When he walked in the opposite direction at the same speed, the number of steps was 100. What is the length of the pipe if Gavrila's step is 80 cm? Round the answer to the nearest whole number of meters. The speed of the tractor is constant.",
      "answer": "108"
    },
    {
      "question": "The National Bank of Hungary withdrew the 1 and 2 forint coins from circulation on March 1, 2008. In cash purchases in stores, the prices of individual items are not rounded separately, but the final amount is rounded to the nearest five. The following table can be seen in the issued information about the rounding rules:\n\namounts ending in 1 or 2 are rounded down to 0;\n\namounts ending in 3 or 4 are rounded up to 5;\n\namounts ending in 6 or 7 are rounded down to 5;\n\namounts ending in 8 or 9 are rounded up to 0.\n\nMarci buys two croissants every morning at the local bakery. After a few days, due to rounding, he saves exactly the price of one croissant. We know that the price of a croissant is more than 10 Ft. What could be the unit price of the croissant if this saving occurs as quickly as possible?",
      "answer": "16"
    },
    {
      "question": "2. Given positive integers $a, b, c (a \\leqslant b \\leqslant c)$ and real numbers $x, y, z, w$ satisfying $a^{x}=b^{y}=c^{z}=70^{w}$, and $\\frac{1}{x}+\\frac{1}{y}+\\frac{1}{z}=\\frac{1}{w}$, the value of $c$ is $\\qquad$.",
      "answer": "7"
    },
    {
      "question": "The first and second terms of a sequence are 4 and 5 , respectively. Each term after the second is determined by increasing the previous term by one and dividing the result by the term before that. For example, the third term equals $\\frac{5+1}{4}$. What is the 1234 th term of this sequence?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "5. The sum of the approximate values you get when you round the largest four-digit number with distinct odd digits to the nearest hundred, and the smallest four-digit number with distinct even digits to the nearest ten, and then round to the nearest thousand.",
      "answer": "11000"
    },
    {
      "question": "If we want to write down all the integers from 1 to 10,000, how many times do we need to write a digit, for example, the digit 5?",
      "answer": "4000"
    },
    {
      "question": "2. (8 points) A shepherd was herding a flock of sheep to graze. After one ram ran out, he counted the number of sheep and found that the ratio of rams to ewes among the remaining sheep was 7:5. After a while, the ram that had run out returned to the flock, but then a ewe ran out. The shepherd counted the sheep again and found that the ratio of rams to ewes was 5:3. How many sheep were there in the flock originally?",
      "answer": "25"
    },
    {
      "question": "2. (5 points) Two different natural numbers end with 7 zeros and have exactly 72 divisors. Find their sum.\n\n#",
      "answer": "70000000"
    },
    {
      "question": "111. In the pyramid $S A B C$, the products of the lengths of the edges of each of the four faces are equal to the same number. The length of the height of the pyramid, dropped from $S$ to the face $A B C$, is $2 \\sqrt{\\frac{102}{55}}$, and the measure of the angle $C A B$ is $\\arccos \\left(\\frac{1}{6} \\sqrt{\\frac{17}{2}}\\right)$. Find the volume of the pyramid $S A B C$, if\n\n$$\n|S A|^{2}+|S B|^{2}-5|S C|^{2}=60\n$$",
      "answer": "\\frac{34\\sqrt{6}}{3}"
    },
    {
      "question": "5. Through the vertex $M$ of some angle, a circle is drawn, intersecting the sides of the angle at points $N$ and $K$, and the bisector of this angle at point $L$. Find the sum of the lengths of segments $M N$ and $M K$, if the area of $M N L K$ is 25, and the angle $L M N$ is $30^{\\circ}$.",
      "answer": "10\\sqrt[4]{3}"
    },
    {
      "question": "28. Let's determine the amount of sulfur substance\n\nAccording to the chemical reaction equation:\n\n$$\n\\mathrm{n}(\\mathrm{S})=\\mathrm{n}\\left(\\mathrm{SO}_{2}\\right)=1 \\text { mole }\n$$\n\n29. Let's calculate the mass of pure sulfur:\n\n$$\nm(S)=n \\cdot M=1 \\cdot 32=32 \\text { g }\n$$\n\nDetermination of the mass of the substance - 2 points\n\n30. Thus, in a sample of natural sulfur weighing 38 g, there are 32 g of pure substance.\n\nLet's calculate the mass of the impurity:\n\n$m($ impurity $)=38-32=6$ g\n\nDetermination of the mass of the impurity - 2 points",
      "answer": "6"
    },
    {
      "question": "## Task A-1.1.\n\nAna and Vanja are standing together next to a railway track and waiting for a train that is moving at a constant speed. At the moment when the front of the train reaches them, Ana starts walking at a constant speed in the direction of the train, while Vanja walks at the same speed in the opposite direction. Each of them stops at the moment when the rear of the train passes by her. Ana has walked a total of 45 meters, while Vanja has walked 30 meters. How long is the train?",
      "answer": "180"
    },
    {
      "question": "16. (6 points) If the two-digit number $\\overline{a b}$ is 24 more than $\\overline{c d}$, and the three-digit number $\\overline{1 a b}$ is 15 more than $\\overline{c d 1}$, then $\\overline{a d}=$",
      "answer": "32"
    },
    {
      "question": "8. (10 points) In the expression $(x+y+z)^{2018}+(x-y-z)^{2018}$, the brackets were expanded and like terms were combined. How many monomials $x^{a} y^{b} z^{c}$ with a non-zero coefficient were obtained?",
      "answer": "1020100"
    },
    {
      "question": "1. From Central Square to the station, there is a straight street divided by 11 intersections into 12 equal blocks. At each intersection, there is a traffic light. All traffic lights simultaneously turn green for 3 minutes, then red for 1 minute. It takes the bus two minutes to travel one block (from intersection to intersection), while a car takes one minute to travel one block. The bus and the car start from the square at the same time, at which moment all traffic lights turn green. Which vehicle will arrive at the station first, and how many minutes earlier?",
      "answer": "9"
    },
    {
      "question": "[  Angles between angle bisectors  Law of Cosines\n\n[ Area of a triangle (using semiperimeter and radius of inscribed or excircle\n\nIn triangle $A B C$, the angle at vertex $B$ is $\\frac{\\pi}{3}$, and the segments connecting the incenter to vertices $A$ and $C$ are 4 and 6, respectively. Find the radius of the circle inscribed in triangle $A B C$.\n\n#",
      "answer": "\\frac{6\\sqrt{3}}{\\sqrt{19}}"
    },
    {
      "question": "8. The smallest solution to the equation $\\left[\\frac{x}{2}\\right]+\\left[\\frac{x}{3}\\right]+\\left[\\frac{x}{7}\\right]=x$ is $\\qquad$ (where, $[x]$ denotes the greatest integer not exceeding $x$).",
      "answer": "-85"
    },
    {
      "question": "53. What is the smallest number of non-overlapping tetrahedra into which a cube can be divided?",
      "answer": "5"
    },
    {
      "question": "2. Let $a_{i}(i=1,2,3,4)$ be real numbers. If the sum of the elements of all non-empty proper subsets of the set $\\left\\{a_{1}, a_{2}, a_{3}, a_{4}\\right\\}$ is\n28, then $a_{1}+a_{2}+a_{3}+a_{4}=$ $\\qquad$",
      "answer": "4"
    },
    {
      "question": "In acute $\\triangle ABC$, $AB > AC$, $\\angle ACB - \\angle ABC = 30^{\\circ}$, $O$ and $H$ are the circumcenter and orthocenter of $\\triangle ABC$ respectively, and $\\triangle DOH$ is an equilateral triangle. Find the value of $\\angle DAO + \\angle AOH$.",
      "answer": "60"
    },
    {
      "question": "4. Let the tetrahedron $P-ABC$ satisfy $PA=PB=3, AB=BC=CA=2$, then the maximum volume of the tetrahedron is $\\qquad$ .",
      "answer": "\\frac{2\\sqrt{6}}{3}"
    },
    {
      "question": "## Task 1 - 050811\n\nA student had attached the following overview of participation in the 1st stage of the 4th Olympiad of Young Mathematicians of the GDR to the wall newspaper:\n\nClass 8a: Out of 33 students, 20 participated, which is approximately 60.6 percent.\n\nClass 8b: Out of 32 students, 21 participated, which is approximately 65.6 percent.\n\nClass 8c: Out of 27 students, 19 participated, which is approximately 70.4 percent.\n\nThe students of these classes are tasked with determining the overall percentage of participation of the 8th grade students. Some students calculate the arithmetic mean of the percentages, while others calculate the quotient of the total number of participants and the total number of students in these classes, multiplied by 100.\n\na) How large is the difference that results from the two calculations?\n\nb) Which students have calculated the percentage in the correct way?",
      "answer": "0.31"
    },
    {
      "question": "The function $f$ has the following properties:\n\n(i) its domain is all real numbers,\n\n(ii) it is an odd function (that is, $f(-x)=-f(x)$ for every real number $x$ ), and\n\n(iii) $f(2 x-3)-2 f(3 x-10)+f(x-3)=28-6 x$ for every real number $x$.\n\nDetermine the value of $f(4)$.",
      "answer": "8"
    },
    {
      "question": "【Question 2】\nThe sum of two numbers is 17, the difference between the two numbers is 7, the larger number is $\\qquad$.",
      "answer": "12"
    },
    {
      "question": "16. Let $n$ be a given positive integer, $S_{n} \\subseteq\\left\\{\\alpha \\mid \\alpha=\\left(p_{1}, p_{2}, \\cdots, p_{n}\\right), p_{k} \\in\\{0,1\\}, k=1,2, \\cdots, n\\right\\}$. For any elements $\\beta=\\left(x_{1}, x_{2}, \\cdots, x_{n}\\right)$ and $\\gamma=\\left(y_{1}, y_{2}, \\cdots, y_{n}\\right)$ in the set $S_{n}$, the following conditions are satisfied:\n(1) If $\\beta=\\gamma$, then $\\sum_{i=1}^{n} x_{i} y_{i}$ is odd;\n(2) If $\\beta \\neq \\gamma$, then $\\sum_{i=1}^{n} x_{i} y_{i}$ is even.\n(1) When $n=5$, find the maximum number of elements in the set $S_{5}$;\n(2) When $n \\geqslant 6$, find the maximum number of elements in the set $S_{n}$.",
      "answer": "5"
    },
    {
      "question": "10. In $\\triangle A B C, A B=9, B C=8$ and $A C=7$. The bisector of $\\angle A$ meets $B C$ at $D$. The circle passing through $A$ and touching $B C$ at $D$ cuts $A B$ and $A C$ at $M$ and $N$ respectively. Find $M N$.\n在 $\\triangle A B C$ 中, $A B=9, B C=8$ 及 $A C=7 \\circ \\angle A$ 的角平分線交 $B C$ 於 $D$ 。穿過 $A$且與 $B C$ 相切於 $D$ 的圓分別交 $A B$ 和 $A C$ 於 $M$ 和 $N \\circ$ 求 $M N$ 。",
      "answer": "6"
    },
    {
      "question": "714. Find the limits:\n1) $\\lim _{\\substack{x \\rightarrow 3 \\\\ y \\rightarrow 0}} \\frac{\\tan(x y)}{y}$\n2) $\\lim _{\\substack{x \\rightarrow 0 \\\\ y \\rightarrow 0}} \\frac{x}{x+y}$.",
      "answer": "3"
    },
    {
      "question": "[Example 3.3.9] Let $n$ be a fixed integer, $n \\geqslant 2$\n(I) Determine the smallest constant $c$ such that the inequality\n$$\n\\sum_{1 \\leqslant i<j \\leqslant n} x_{i} x_{j}\\left(x_{i}^{2}+x_{j}^{2}\\right) \\leqslant c\\left(\\sum_{i=1}^{n} x_{i}\\right)^{4} ;\n$$\n(II) For this constant $c$, determine the necessary and sufficient conditions for equality to hold.",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "## Subject 4.\n\nAna, Barbu, and Cristi together have an amount of money less than 100 lei, and each of them has more than 5 lei. Ana and Barbu together have three times more than Cristi, and Barbu and Cristi together have three times more than Ana. Find the largest and smallest amounts of money that the three children can have.\n\n## Note: All subjects are mandatory\n\nTime allowed: 2 hours\n\n## MATHEMATICS OLYMPIAD  LOCAL STAGE  JANUARY 18, 2014\n\n## GRADE 5  Scoring Guidelines",
      "answer": "24"
    },
    {
      "question": "7.4. Karlson counts 200 buns baked by Fräulein Bock: «one, two, three, ..., one hundred and nine, one hundred and ten, two hundred». How many words will he say in total? (Each word is counted as many times as it was said.)",
      "answer": "443"
    },
    {
      "question": "20. (6 points) If the ten-digit number $\\mathrm{a} 2016 \\mathrm{~b} 2017$ is divisible by 33, then, how many such ten-digit numbers are there?",
      "answer": "3"
    },
    {
      "question": "B1. What is the maximum possible greatest common divisor of the numbers $a-2b+3, 2a-3b-1$, and $3a+b-2$, if $a$ and $b$ are natural numbers?",
      "answer": "38"
    },
    {
      "question": "6. In the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, $A B=5, A A_{1}=4, A D=3$, points $E, F, G$ are the midpoints of edges $A A_{1}, C_{1} D_{1}$, and $B C$, respectively. Then the volume of the tetrahedron $B_{1}-E F G$ is $\\qquad$",
      "answer": "\\frac{45}{16}"
    },
    {
      "question": "Jindra collects dice, all of the same size. Yesterday he found a box and started arranging the dice in it. He managed to fill the square bottom with one layer of dice. He then filled five more layers in the same way, but ran out of dice halfway through the next layer. Today, Jindra received 18 more dice from his grandmother, which were exactly what he needed to complete this layer.\n\nHow many dice did Jindra have yesterday?\n\n(M. Petrová)\n\nHint. How many dice did Jindra have in each complete layer?",
      "answer": "234"
    },
    {
      "question": "6.2. In an isosceles triangle \\(ABC\\), one of the angles is equal to the difference of the other two, and one of the angles is twice another. The angle bisectors of angles \\(A\\), \\(B\\), and \\(C\\) intersect the circumcircle of the triangle at points \\(L\\), \\(O\\), and \\(M\\) respectively. Find the area of triangle \\(LOM\\), if the area of triangle \\(ABC\\) is 2. If the answer is not an integer, round it to the nearest integer.",
      "answer": "\\sqrt{3}+1"
    },
    {
      "question": "1. For how many natural numbers $n$ are the inequalities\n\n$$\n2019<\\sqrt{n}<2020 ?\n$$\n\nExplain your answer!",
      "answer": "4038"
    },
    {
      "question": "6. (10 points) Two cubes of different sizes are glued together to form the solid figure shown below, where the four vertices of the smaller cube's bottom face are exactly the midpoints of the edges of the larger cube's top face. If the edge length of the larger cube is 2, then the surface area of this solid figure is . $\\qquad$",
      "answer": "32"
    },
    {
      "question": "Bogognov i.i.\n\nIn some cells of a $200 \\times 200$ square, there is one chip - either red or blue; the other cells are empty. One chip sees another if they are in the same row or column. It is known that each chip sees exactly five chips of the other color (and possibly some chips of its own color). Find the maximum possible number of chips.",
      "answer": "3800"
    },
    {
      "question": "14. A four-digit number, if a comma is placed between the hundreds and tens place, can be written as two two-digit numbers $(3126 \\rightarrow 31,62)$. If the two two-digit numbers have an integer multiple relationship, we call such a four-digit number a \"clever number\". If you select 4 numbers from $1, 2, 4, 6, 8$ to form a four-digit number, how many \"clever numbers\" are there? $\\qquad$",
      "answer": "12"
    },
    {
      "question": "6. Let the hyperbola $\\frac{x^{2}}{a^{2}}-\\frac{y^{2}}{b^{2}}=1(a>0, b>0)$ have its left and right foci as $F_{1}$ and $F_{2}$, respectively, and let $A$ be a point on the asymptote of the hyperbola such that $A F_{2} \\perp F_{1} F_{2}$. The distance from the origin $O$ to the line $A F_{1}$ is $\\frac{1}{3}\\left|O F_{1}\\right|$. Then the eccentricity of the hyperbola is",
      "answer": "\\frac{\\sqrt{6}}{2}"
    },
    {
      "question": "8,9\n\nTwo tangents are drawn to a circle of radius 7 from a point located at a distance of 25 from the center.\n\nFind the distance between the points of tangency.",
      "answer": "13.44"
    },
    {
      "question": "16. Through the right focus $F$ of the ellipse $\\frac{x^{2}}{3}+\\frac{y^{2}}{2}=1$, draw two perpendicular chords $A B$ and $C D$. Let the midpoints of $A B$ and $C D$ be $M$ and $N$ respectively.\n(1) Prove that the line $M N$ must pass through a fixed point, and find this fixed point;\n(2) If the slopes of chords $A B$ and $C D$ both exist, find the maximum value of the area $S$ of $\\triangle F M N$.",
      "answer": "\\frac{4}{25}"
    },
    {
      "question": "2. Sergey wrote down the numbers from 500 to 1499 in a row in some order. Under each number, except the leftmost one, he wrote the GCD of that number and its left neighbor, obtaining a second row of 999 numbers. Then he applied the same rule to get a third row of 998 numbers, from it a fourth row of 997 numbers, and so on. He stopped when all the numbers in the next row were equal to one for the first time. What is the maximum number of rows he could have written by this moment?",
      "answer": "501"
    },
    {
      "question": "3. A cyclist is moving from place A to place B, and halfway between A and B is place C. The cyclist covered the distance from A to C in 4 hours, traveling at a speed of $18 \\mathrm{~km}$ per hour. How many hours will it take the cyclist to cover the distance from place C to B, if he travels at a speed of $12 \\mathrm{~km}$ per hour?",
      "answer": "6"
    },
    {
      "question": "2. In the Cartesian coordinate system, the area of the region formed by points $(x, y)$ satisfying $|x|+|y|+|x-2| \\leqslant 4$ is\n$\\qquad$ (Supplied by An Zhenping)",
      "answer": "12"
    },
    {
      "question": "1. Given the integer sequence $\\left\\{a_{n}\\right\\}$ satisfying $a_{n}=a_{n-1}-a_{n-2}(n \\geqslant 3)$, if the sum of the first 1492 terms is 1985, and the sum of the first 1985 terms is 1492, find the sum of the first 2001 terms.",
      "answer": "986"
    },
    {
      "question": "A hemisphere with radius $r=1$ is to be divided into two equal volume parts by a plane parallel to the base. How high is each of the two parts?",
      "answer": "0.6527"
    },
    {
      "question": "8. A station has exactly one bus arriving every $8: 00 \\sim 9: 00, 9: 00 \\sim 10: 00$, but the arrival time is random, and the arrival times of the two buses are independent, with the following pattern:\n\nA passenger arrives at the station at $8: 20$, then the expected waiting time for the bus is $\\qquad$ (to the nearest minute).",
      "answer": "27"
    },
    {
      "question": "16. Mailboxes. In a newly inhabited apartment building, workers installed a block of 80 mailboxes. Inside each mailbox, they placed a ring with a key and a tag with the apartment number, but they did not bother to ensure that each mailbox contained the key to that mailbox. They simply threw the keys into the mailboxes in a random order. The mailboxes, of course, are locked.\n\nThe Absent-Minded Scientist, who recently moved into apartment 37, was delighted to see the mailboxes, but the concierge said she could not get the keys out. One had to go to the management company, which accepts citizens every third Monday of the month from 12:00 to 12:30.\n\nFinding a stick in the trash can (someone had ordered sushi), the Scientist managed to extract a key from mailbox № 37 on the fourth try. He reasoned as follows: if this is the key to my mailbox, then everything is fine. If not, I will open the mailbox to which this key belongs, leave the key in the lock, take out the next key, open the next mailbox, and so on, until I get my key.\n\na) (for 8th grade, 3 points). Prove that the described algorithm is effective, that is, that the Absent-Minded Scientist will find his key.\n\nb) (for 9th grade, 6 points). Find the expected number of residents who will have to poke in the mailboxes with a stick if they all follow the Absent-Minded Scientist's algorithm.",
      "answer": "4.965"
    },
    {
      "question": "11. (15 points) Find the minimum value of $g(x)=\\frac{x^{3}+1}{x^{2}-x}(x>1)$.",
      "answer": "\\sqrt{9+6\\sqrt{3}}"
    },
    {
      "question": "A grid with nine cells, as shown in the image, is filled with nine consecutive natural numbers. These numbers are arranged in ascending order from left to right and from top to bottom (i.e., the smallest number is in the top left, and the largest is in the bottom right). The thick broken line divides the grid into two parts. The sum of the numbers in the left part is 100 less than the sum of the numbers in the right part.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_f53d56ddc45593eccca2g-2.jpg?height=377&width=376&top_left_y=477&top_left_x=840)\n\nWhich number is in the middle cell?\n\n(L. Šimůnek)",
      "answer": "94"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$$\n\\lim _{x \\rightarrow 0}\\left(\\frac{1+x \\cdot 3^{x}}{1+x \\cdot 7^{x}}\\right)^{\\frac{1}{x^2}}\n$$",
      "answer": "\\frac{3}{7}"
    },
    {
      "question": "3. At a rock festival, vocalists, guitarists, and drummers met, a total of 121 people. Each vocalist gave a slap on the back of each guitarist, each guitarist gave a slap on the back of each drummer, and each drummer gave a slap on the back of each vocalist. What is the maximum number of slaps on the back that the participants of the festival could have received?",
      "answer": "4880"
    },
    {
      "question": "11. (20 points) As shown in Figure 1, the ellipse $E: \\frac{x^{2}}{4}+y^{2}=1$ has its right focus at $F$, and the circle $\\odot O: x^{2}+y^{2}=1$ has a tangent line $AB$ that intersects the ellipse $E$ at points $A$ and $B$ ($A$ and $B$ are both on the right side of the $y$-axis). Find the maximum value of the inradius of $\\triangle ABF$.",
      "answer": "\\frac{3-\\sqrt{3}}{4}"
    },
    {
      "question": "11.2. Several married couples came to the New Year's Eve party, each of whom had from 1 to 10 children. Santa Claus chose one child, one mother, and one father from three different families and took them for a ride in his sleigh. It turned out that he had exactly 3630 ways to choose the required trio of people. How many children could there be in total at this party?\n\n(S. Volchonkov)",
      "answer": "33"
    },
    {
      "question": "## Task 16/65\n\nThe difference between the difference of the cubes and the difference of the squares of two consecutive natural numbers is 114. Which numbers are these?",
      "answer": "76"
    },
    {
      "question": "Podlipsky O.K.\n\nOleg drew an empty $50 \\times 50$ table and wrote a number above each column and to the left of each row. It turned out that all 100 written numbers are distinct, with 50 of them being rational and the other 50 being irrational. Then, in each cell of the table, he wrote the product of the numbers written next to its row and its column (\"multiplication table\"). What is the maximum number of products in this table that could be rational numbers?",
      "answer": "1275"
    },
    {
      "question": "A tank can be filled with water through two pipes. One pipe can fill the tank in 3 hours, and both pipes together can fill the tank in 1 hour and 12 minutes. How many hours does it take to fill the tank if only the second pipe is used?",
      "answer": "2"
    },
    {
      "question": "We randomly place points $A, B, C$, and $D$ on the circumference of a circle, independently of each other. What is the probability that the chords $AB$ and $CD$ intersect?",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "3. Andrew has two containers for carrying water. The containers are cubes without tops and have base areas of $4 \\mathrm{dm}^{2}$ and $36 \\mathrm{dm}^{2}$ respectively. Andrew has to completely fill the larger cube with pond water, which must be carried from the pond using the smaller cube. What is the smallest number of visits Andrew has to make to the pond with the smaller cube?",
      "answer": "27"
    },
    {
      "question": "2. Let there be $n$ voters, and $k$ votes were cast for a given candidate. Then $0.50332 \\leqslant$ $\\frac{k}{n} \\leqslant 0.50333$, and $1.00664 \\leqslant \\frac{2 k}{n} \\leqslant 1.00666$. If we denote $m=2 k-n$, then $0.00664 \\leqslant \\frac{m}{n} \\leqslant$ 0.00666.\n\n- If $m=1$, then $150.1<\\frac{1}{0.00666} \\leqslant n \\leqslant \\frac{1}{0.00664}<150.7$ - no integer solutions.\n- If $m=2$, then $300<\\frac{2}{0.00666} \\leqslant n \\leqslant \\frac{2}{0.00664}<302$. But if $n=301$ and $m=2$, then the corresponding value of $k$ does not exist.\n- If $m=3$, then $450<\\frac{3}{0.00666} \\leqslant n \\leqslant \\frac{3}{0.00664}<452$. Therefore, $n=451, k=227$, and all inequalities are satisfied.",
      "answer": "451"
    },
    {
      "question": "Problem 3. How many natural numbers $n>1$ exist, for which there are $n$ consecutive natural numbers, the sum of which is equal to 2016?",
      "answer": "5"
    },
    {
      "question": "Example 4: Given 1 one-yuan note, 1 two-yuan note, 1 five-yuan note, 4 ten-yuan notes, and 2 fifty-yuan notes, how many different amounts of money can be paid using these notes?\n(1986 Shanghai Competition Question)",
      "answer": "119"
    },
    {
      "question": "[ Right triangle with an angle in ]\n\nFrom the vertex of the right angle of triangle $ABC$, the median $CM$ is drawn. The circle inscribed in triangle $CAM$ touches $CM$ at its midpoint. Find the angle $BAC$.\n\n#",
      "answer": "60"
    },
    {
      "question": "7. In a $100 \\times 100$ square, 10000 cuts were made along the grid lines (each cut one cell long), and it split into 2500 four-cell figures. How many of them are $2 \\times 2$ squares?",
      "answer": "2300"
    },
    {
      "question": "1. Let $Q(x)=a_{0}+a_{1} x+\\cdots+a_{n} x^{n}$ be a polynomial with integer coefficients, and $0 \\leq a_{i}<3$ for all $0 \\leq i \\leq n$.\nGiven that $Q(\\sqrt{3})=20+17 \\sqrt{3}$, compute $Q(2)$.",
      "answer": "86"
    },
    {
      "question": "7.055. $\\log _{3}\\left(3^{x}-8\\right)=2-x$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.\n\n7.055. $\\log _{3}\\left(3^{x}-8\\right)=2-x$.",
      "answer": "2"
    },
    {
      "question": "20th Brazil 1998 Problem A2 ABC is a triangle. D is the midpoint of AB, E is a point on the side BC such that BE = 2 EC and ∠ADC = ∠BAE. Find ∠BAC.",
      "answer": "90"
    },
    {
      "question": "13.012. The working day has been reduced from 8 to 7 hours. By what percentage does labor productivity need to increase to ensure that, with the same rates, the wage increases by $5 \\%$?",
      "answer": "20"
    },
    {
      "question": "7. If $n$ is a positive integer such that $n^{6}+206$ is divisible by $n^{2}+2$, find the sum of all possible values of $n$.\n(1 mark)\n若 $n$ 是正整數, 且 $n^{6}+206$ 可被 $n^{2}+2$ 整除, 求 $n$ 所有可能值之和。\n(1 分)",
      "answer": "32"
    },
    {
      "question": "Kuuyggin A.K.\n\nAmong any five nodes of a regular square grid, there will always be two nodes such that the midpoint of the segment between them is also a node of the grid. What is the minimum number of nodes of a regular hexagonal grid that must be taken so that among them there will always be two nodes such that the midpoint of the segment between them is also a node of this grid?",
      "answer": "9"
    },
    {
      "question": "## In a convex quadrilateral $A B C D$, the length of the segment connecting the midpoints of sides $A B$ and $C D$ is 1. Lines\n\n$B C$ and $A D$ are perpendicular. Find the length of the segment connecting the midpoints of diagonals $A C$ and $B D$.",
      "answer": "1"
    },
    {
      "question": "118. Find $\\lim _{\\alpha \\rightarrow 0} \\frac{\\sin 2 \\alpha}{\\alpha}$.",
      "answer": "2"
    },
    {
      "question": "4. The rectangles $S_{1}$ and $S_{2}$ are drawn on a set of coordinate axes. $S_{1}$ has vertices at $(0, y) ;(x, y) ;(x, 4)$; and $(0,4)$, where $x<9$ and $y<4$. $S_{2}$ has vertices at $(x, 0)$; $(9,0) ;(9, y)$; and $(x, y)$. The rectangles are equal in area. What is the value of $\\frac{360 x}{y}$ ?",
      "answer": "810"
    },
    {
      "question": "6. A circle can be circumscribed around quadrilateral $A B C D$. The lengths of sides $A B$ and $A D$ are equal.\n\nOn side $C D$ is point $Q$ such that $D Q=1$, and on side $B C$ is point $P$ such that $B P=5$.\n\nFurthermore, $\\measuredangle D A B=2 \\measuredangle Q A P$. Find the length of segment $P Q$.\n\n## Solutions and answers\n\n## Variant 1\n\nProblem 1 Answer: $7-\\frac{\\sqrt{2}}{2}$",
      "answer": "7-\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "Šárka and Terezka received a box of chocolates, containing 35 chocolate candies. Each day, each of the girls ate at least one candy, and no candy was divided into parts. On the first day, Terezka ate $\\frac{2}{5}$ of what Šárka ate on the same day. On the second day, Šárka ate $\\frac{3}{4}$ of what Terezka ate on the same day. By the end of the second day, the box was empty.\n\nHow many candies did Terezka eat in total, given that the difference between the number of candies eaten by Terezka and the number of candies eaten by Šárka was the smallest possible?\n\n(L. Hozová)",
      "answer": "18"
    },
    {
      "question": "Task 7. A subscriber has forgotten the last digit of the phone number and therefore dials it at random. Determine the probability that he will need to call no more than three times to get the correct connection.",
      "answer": "0.3"
    },
    {
      "question": "7.3. In the 7th grade, there are 25 people, and each attends a dance club or a drama club (some attend both clubs). Everyone took a math test, and the teacher calculated the percentage of students who received a failing grade (2) separately among the \"dancers\" and among the \"actors.\" It turned out that the percentage is the same and equals $30 \\%$. Can the percentage of failing grades in the entire class be more than $35 \\%$?",
      "answer": "36"
    },
    {
      "question": "Three, (50 points) In an $8 \\times 8$ grid, each cell contains a knight or a knave. Knights always tell the truth, and knaves always lie. It is known that each resident in the grid says: “The number of knaves in my column is (strictly) greater than the number of knaves in my row.” Determine the number of arrangements of residents that satisfy this condition.",
      "answer": "255"
    },
    {
      "question": "[\n\nThe Tale of the Dead Princess and the Seven Bogatyrs. Once, returning home in the evening, the bogatyrs gave the princess their prey - 29 gray ducks. Each brother shot at least one duck. All of them hunted a different number of ducks: the older the brother, the more game he shot. What was the catch of the eldest brother?\n\n#",
      "answer": "8"
    },
    {
      "question": "## 10. Anniversary\n\n\"Do you remember Leonitina's fortieth birthday? It was December 28, $19 \\ldots$ Since then, Leonitina has aged. I noticed that half of her age equals twice the sum of its digits.\"\n\nPlease fill in the missing digits in the date of Leonitina's fortieth birthday.",
      "answer": "1970"
    },
    {
      "question": "1. A mathematician left point A for point B. After some time, a physicist also left point A for point B. Catching up with the mathematician after 20 km, the physicist, without stopping, continued to point B and turned back. They met again 20 km from B. Then each continued in their respective directions. Upon reaching points A and B respectively, they turned and walked towards each other again. How many kilometers from point A will they meet for the third time, if the distance between points A and B is 100 km?",
      "answer": "45"
    },
    {
      "question": "## 86. Math Puzzle $7 / 72$\n\nThe water tank of the LPG contains $1000 \\mathrm{~m}^{3}$ of water and can be filled by two pumping stations in two days, provided no water is consumed. One pump delivers $300 \\mathrm{~m}^{3}$ more water per day than the other.\n\nHow many days can the LPG withdraw $400 \\mathrm{~m}^{3}$ of water daily if the larger pumping station is out of service?",
      "answer": "3"
    },
    {
      "question": "Problem 2. Vanya thought of a two-digit number, then swapped its digits and multiplied the resulting number by itself. The result turned out to be four times larger than the number he thought of. What number did Vanya think of?",
      "answer": "81"
    },
    {
      "question": "Problem 3. Little kids were eating candies. Each one ate 7 candies less than all the others together, but still more than one candy. How many candies were eaten in total?\n\n[5 points] (A. V. Shapovalov)",
      "answer": "21"
    },
    {
      "question": "4. (15 points) The efficiency of an ideal heat engine is $50 \\%$. What will it become if the temperature of the heater is increased by $50 \\%$, and the temperature of the cooler is decreased by $50 \\%$?",
      "answer": "83"
    },
    {
      "question": "1. (5 points) The left figure below is the recently discovered Archimedes' \"Stomachion\" puzzle, dividing a square into 14 polygons: After research, experts found that the 14 pieces can be correctly drawn on a square with a side length of $12 \\mathrm{~cm}$, as shown in the figure. Question: The area of the gray piece is $\\qquad$ square centimeters.\n12.",
      "answer": "12"
    },
    {
      "question": "Sorrelkova N.P.\n\nUnder one of the cells of an $8 \\times 8$ board, a treasure is buried. Under each of the other cells, there is a sign indicating the minimum number of steps required to reach the treasure from that cell (one step allows moving from a cell to an adjacent cell by side). What is the minimum number of cells that need to be dug up to definitely retrieve the treasure?",
      "answer": "3"
    },
    {
      "question": "10. Find the maximum of the expression\n$$\n|| \\ldots|| x_{1}-x_{2}\\left|-x_{3}\\right|-\\ldots\\left|-x_{2023}\\right|,\n$$\nwhere $x_{1}, x_{2}, \\ldots, x_{2023}$ are distinct natural numbers between 1 and 2023 .",
      "answer": "2022"
    },
    {
      "question": "Compute the sum $S=\\sum_{i=0}^{101} \\frac{x_{i}^{3}}{1-3 x_{i}+3 x_{i}^{2}}$ for $x_{i}=\\frac{i}{101}$.\n\nAnswer: $S=51$.\n\n#",
      "answer": "51"
    },
    {
      "question": "Problem 9.5. Positive numbers $a, b$, and $c$ satisfy the equations\n\n$$\na^{2}+a b+b^{2}=1, \\quad b^{2}+b c+c^{2}=3, \\quad c^{2}+c a+a^{2}=4\n$$\n\nFind $a+b+c$.",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "## 12. Punctual Wife\n\nEvery Saturday between 15 and 16, I play tennis with my friend Philippe. My wife arrives to pick me up in the car precisely at 16:10. One day, Philippe fell ill. Unaware of this, I set out for tennis as usual. However, at 15:05, realizing that Philippe wasn't coming, I gathered my things and started walking home. On the way, I met my wife, who was driving to pick me up. I got into the car, and we arrived home 10 minutes earlier than usual.\n\nThis might not seem very interesting, but could you determine from this story the ratio between my walking speed and the speed at which my wife drives?",
      "answer": "12"
    },
    {
      "question": "Analyzing the 4-digit natural numbers:\n\na) How many of them have all different digits?\n\nb) How many have the digit 1 exactly once and all different digits?\n\nc) How many have the digit 1?",
      "answer": "3168"
    },
    {
      "question": "9.1. It is known that the sum of the digits of number A is 59, and the sum of the digits of number B is 77. What is the minimum sum of the digits that the number A+B can have?",
      "answer": "1"
    },
    {
      "question": "7. Each cell of a $10 \\times 10$ board is painted black or white. A cell is said to be out of place if it has at least seven neighbors of a different color than itself. (Neighbors are cells that share a common side or corner.) What is the maximum number of white cells on the board that can be out of place at the same time?",
      "answer": "26"
    },
    {
      "question": "36. Given that each of the values of $x_{1}, x_{2}, \\cdots \\cdots, x_{2018}$ is either -1 or 1, if $x_{1}+x_{2}+\\cdots+x_{2018}=4$, then the maximum value of $\\left|x_{1}-x_{2}\\right|+\\left|x_{3}-x_{4}\\right|+\\cdots+\\left|x_{2017}-x_{2018}\\right|$ is $\\qquad$",
      "answer": "2014"
    },
    {
      "question": "11. Let $A$ be a set composed of any 100 distinct positive integers, and let\n$$\nB=\\left\\{\\left.\\frac{a}{b} \\right\\rvert\\, a, b \\in A \\text { and } a \\neq b\\right\\},\n$$\n$f(A)$ denotes the number of elements in set $B$. Then the sum of the maximum and minimum values of $f(A)$ is $\\qquad$",
      "answer": "10098"
    },
    {
      "question": "6. Solve the equation $\\sqrt{\\frac{x-3}{2 x+1}}+2=3 \\sqrt{\\frac{2 x+1}{x-3}}$.",
      "answer": "-4"
    },
    {
      "question": "3. (13 points) Money placed in the 1st month will be on deposit for 6 months and will bring a nominal income of $8700\\left((1+0.06 / 12)^{\\wedge}-1\\right)=264.28$ rubles.\n\nFunds placed in the 2nd month will be on deposit for 5 months and will bring a nominal income of $8700\\left((1+0.06 / 12)^{\\wedge} 5-1\\right)=219.69$ rubles.\n\nSimilarly in the following months:\n\n$8700\\left((1+0.06 / 12)^{\\wedge} 4-1\\right)=175.31$ rubles.\n\n$8700\\left((1+0.06 / 12)^{\\wedge} 3-1\\right)=131.15$ rubles.\n\n$8700\\left((1+0.06 / 12)^{\\wedge} 2-1\\right)=87.22$ rubles.\n\n$8700((1+0.06 / 12)-1)=43.5$ rubles.\n\nThus, the nominal income for 6 months will be:\n\n$264.28+219.69+175.31+131.15+87.22+43.5=921.15$ rubles.\n\n## Maximum 25 points\n\n## Problem 5\n\nMobile applications can significantly improve the convenience of paying for services and goods. Imagine that you are developing a mobile internet banking application where users can pay for services of various organizations with their bank card.\n\nList at least 3 security measures that you should implement to protect users of the application from theft of money from their bank card.\n\n#",
      "answer": "921.15"
    },
    {
      "question": "14. From the 20 numbers 11, 12, 13, 14, ... 30, at least ( ) numbers must be taken to ensure that there are two numbers among the taken numbers whose sum is a multiple of ten.",
      "answer": "11"
    },
    {
      "question": "$7 \\cdot 8$ If a positive divisor is randomly selected from $10^{99}$, the probability that it is exactly a multiple of $10^{88}$ is $\\frac{m}{n}$, where $m$ and $n$ are coprime, find $m+n$.\n\nThe prime factorization of $10^{99}$ is $2^{99} \\cdot 5^{99}$. The total number of positive divisors of $10^{99}$ is $(99+1)(99+1) = 100 \\cdot 100 = 10000$.\n\nFor a divisor to be a multiple of $10^{88}$, it must be of the form $2^{a} \\cdot 5^{b}$ where $a \\geq 88$ and $b \\geq 88$. The number of such divisors is $(100-88)(100-88) = 12 \\cdot 12 = 144$.\n\nThus, the probability that a randomly selected divisor of $10^{99}$ is a multiple of $10^{88}$ is $\\frac{144}{10000} = \\frac{9}{625}$. Here, $m = 9$ and $n = 625$, and they are coprime.\n\nTherefore, $m+n = 9 + 625 = 634$.",
      "answer": "634"
    },
    {
      "question": "Question 39, Let $a, b$ be real numbers, and the function $f(x)=x^{3}+a x^{2}+b x$. If there exist three real numbers $x_{1}, x_{2}, x_{3}$ satisfying $x_{1}+1 \\leq x_{2} \\leq x_{3}-1$, and $f\\left(x_{1}\\right)=f\\left(x_{2}\\right)=f\\left(x_{3}\\right)$, and the minimum value of $|a|+2|b|$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "8 The function $f(x)=\\frac{x-x^{3}}{1+2 x^{2}+x^{4}}$ has a product of its maximum and minimum values of\n\n保留源文本的换行和格式，翻译结果如下：\n\n8 The function $f(x)=\\frac{x-x^{3}}{1+2 x^{2}+x^{4}}$ has a product of its maximum and minimum values of",
      "answer": "-\\frac{1}{16}"
    },
    {
      "question": "3. The triangle $A B C$ is isosceles with $A B=B C$. The point $D$ is a point on $B C$, between $B$ and $C$, so that $A C=A D=B D$.\nWhat is the size of angle $A B C$ ?",
      "answer": "36"
    },
    {
      "question": "5.5. The hedgehogs collected 65 mushrooms and divided them so that each hedgehog got at least one mushroom, but no two hedgehogs had the same number of mushrooms. What is the maximum number of hedgehogs that could be",
      "answer": "10"
    },
    {
      "question": "15. In the complex plane, the three vertices of $\\triangle A B C$ correspond to the complex numbers $z_{1}, z_{2}, \\dot{z}_{3}$. It is known that $\\left|z_{1}\\right|=3$, $z_{2}=\\bar{z}_{1}, z_{3}=\\frac{1}{z_{1}}$, find the maximum value of the area of $\\triangle A B C$.",
      "answer": "4"
    },
    {
      "question": "20、Person A and Person B start from points $A$ and $B$ respectively at the same time, heading towards each other. After they meet, A increases his speed by 10 kilometers per hour, and B decreases his speed by 5 kilometers per hour. When A reaches point B, B is exactly at the midpoint of points $A$ and $B$. If B's original speed was $\\frac{2}{2}$ of A's speed, then A's original speed was $\\qquad$ kilometers per hour.",
      "answer": "18"
    },
    {
      "question": "The kindergarten received cards for reading lessons: some have \"MA\" written on them, and others have \"NYA\". Each child took three cards and started forming words from them. It turned out that 20 children can form the word \"MAMA\", 30 children can form the word \"NYANYA\", and 40 children can form the word \"MANYA\". How many children have all three cards the same?\n\n#",
      "answer": "10"
    },
    {
      "question": "3. Find the smallest trimestral number for which all the digits of its triple are even.",
      "answer": "134"
    },
    {
      "question": "6. On the table, there are three spheres and a cone (with its base $k$ on the table), touching each other externally. The radii of the spheres are 20, 40, and 40, and the radius of the base of the cone is 21. Find the height of the cone.",
      "answer": "28"
    },
    {
      "question": "4. All the 7-digit numbers containing each of the digits $1,2,3,4,5,6,7$ exactly once, and not divisible by 5 , are arranged in the increasing order. Find the 2000-th number in this list.",
      "answer": "4315672"
    },
    {
      "question": "2・102 Let the set $S=\\{1,2, \\cdots, 50\\}$, find the smallest positive integer $n$, such that in every $n$-element subset of $S$ there are 3 numbers that can serve as the 3 side lengths of a right triangle.",
      "answer": "42"
    },
    {
      "question": "7. Bruno set a four-digit even number as the unlock code for his mobile phone. To avoid forgetting the number, he wrote down the following:\n\n- All digits are different, and the sum of all digits is 15.\n- The unit digit is three times smaller than the thousands digit.\n- The tens digit is smaller than the unit digit.\n\nDetermine the code that Bruno set for unlocking his mobile phone.",
      "answer": "6702"
    },
    {
      "question": "3. (8 points) The largest four-digit number is $\\qquad$ times larger than the largest two-digit number.",
      "answer": "100"
    },
    {
      "question": "The focus of the parabola $y^{2}=7 x$ is crossed by a line which intersects the $Y$ axis at (-1). What is the area of the parabolic segment thus obtained?",
      "answer": "66.87"
    },
    {
      "question": "# Problem 9.\n\nLet $A(n)$ denote the greatest odd divisor of the number $n$. For example, $A(21)=21$, $A(72)=9, A(64)=1$. Find the sum $A(111)+A(112)+\\ldots+A(218)+A(219)$.",
      "answer": "12045"
    },
    {
      "question": "15. English Club (from 9th grade. 6 points). Every Friday, ten gentlemen come to the club, and each hands their hat to the doorman. Each hat fits its owner perfectly, but no two hats are the same size. The gentlemen leave one by one in a random order.\n\nWhen seeing off each gentleman, the doorman tries to put on the first hat he finds. If it fits, the gentleman leaves with that hat. If it is too small, the doorman tries the next random hat from the remaining ones. If all the remaining hats are too small, the doorman says to the poor man: \"Sir, a hat does not suit you today,\" and the gentleman goes home with an uncovered head. Find the probability that next Friday the doorman will not have any hats left.\n\nIX Olympiad in Probability Theory and Statistics for Schoolchildren. Correspondence Round.\n\n© Laboratory of Probability Theory and Statistics of the MCNMO, 2015-2016\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_be2097b0254fa1f5b26ag-18.jpg?height=391&width=534&top_left_y=216&top_left_x=293)",
      "answer": "0.000516"
    },
    {
      "question": "16. Given the function $f(x)=\\left(x^{2}-4\\right)^{-\\frac{1}{2}}(x>2), f^{-1}(x)$ is the inverse function of $f(x)$.\n(1) If the sequence $\\left\\{a_{n}\\right\\}, a_{1}=1, \\frac{1}{a_{n+1}}=f^{-1}\\left(a_{n}\\right)\\left(n \\in \\mathbf{N}_{+}\\right)$, find $a_{n}$;\n(2) Let $S_{n}=a_{1}^{2}+a_{2}^{2}+\\cdots+a_{n}^{2}, b_{n}=S_{2 n+1}-S_{n}$, does there exist a largest positive integer $p$, such that for all $n \\in \\mathbf{N}_{+}$, $b_{n}<\\frac{1}{p}$ holds? Please provide an explanation or proof.",
      "answer": "3"
    },
    {
      "question": "13. (12 points) In a cage, there are chickens and rabbits, with a total of 28 heads. The total number of rabbit feet is 12 more than 3 times the total number of chicken feet. How many rabbits are there in the cage?",
      "answer": "18"
    },
    {
      "question": "$13 \\cdot 17$ There is a chain made up of 60 rings, each weighing 1 gram. What is the minimum number of rings that need to be cut so that the segments of the chain can be used to make up weights of 1 gram, 2 grams, $\\cdots, 59$ grams, 60 grams (the cut rings still weigh 1 gram)?",
      "answer": "3"
    },
    {
      "question": "Problem 1. In two tubs, there are $4 \\frac{1}{6}$ and $3 \\frac{1}{2}$ liters of water respectively. How much water needs to be transferred from the first tub to the second tub so that after the transfer, both tubs have the same amount of water? Explain your answer!",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Problem 4.3. How many rectangles exist on this picture with sides running along the grid lines? (A square is also a rectangle.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0973a8d23c1bf92cb27dg-06.jpg?height=173&width=206&top_left_y=614&top_left_x=624)",
      "answer": "24"
    },
    {
      "question": "Example 1 Connect any two vertices of a cube with a straight line, among these lines, the number of pairs of mutually perpendicular skew lines is $\\qquad$.",
      "answer": "78"
    },
    {
      "question": "Example 4 As shown in Figure 4, in $\\triangle A B C$, $A B=A C, \\angle A$ $=100^{\\circ}, I$ is the incenter, and $D$ is a point on side $A B$ such that $B D=B I$. Find the measure of $\\angle B C D$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "30"
    },
    {
      "question": "9.8. A straight rod 2 meters long was sawn into $N$ sticks, the length of each of which is expressed as an integer number of centimeters. For what smallest $N$ can it be guaranteed that, using all the resulting sticks, one can, without breaking them, form the contour of some rectangle?\n\n(A. Magazinov)",
      "answer": "102"
    },
    {
      "question": "Problem 5.8. In the country, there are 100 cities: 30 of them are located in the mountainous part of the country, and 70 are in the plain. Over three years, air routes were established between the cities. Each year, 50 new air routes were opened in the country: all cities were randomly divided into 50 pairs, and a route was opened between the cities in the same pair. After three years, it turned out that out of the 150 routes opened, exactly 21 connect pairs of \"mountainous\" cities. How many routes connect pairs of \"plain\" cities?",
      "answer": "81"
    },
    {
      "question": "You are out walking and see a group of rhinoceroses (which each have two horns and four legs) and triceratopses (which each have three horns and four legs). If you count 31 horns and 48 legs, how many triceratopses are there?",
      "answer": "7"
    },
    {
      "question": "29. Given that the real numbers $x, y$ and $z$ satisfies the condition $x+y+z=3$, find the maximum value of $f(x y z)=\\sqrt{2 x+13}+\\sqrt[3]{3 y+5}+\\sqrt[4]{8 z+12}$.",
      "answer": "8"
    },
    {
      "question": "247. $\\int_{0}^{2} 3 x^{4} d x . \\quad$ 248. $\\int_{-1}^{\\sqrt{3}} 4 t^{3} d t . \\quad$ 249. $\\int_{0}^{1} \\sqrt[3]{x} d x$.",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "4. Three people want to travel from city A to city B, which is 45 kilometers away from A. They have two bicycles. The speed of a cyclist is 15 km/h, and the speed of a pedestrian is 5 km/h. What is the minimum time they can reach B, if the bicycle can be left unattended on the road?",
      "answer": "3"
    },
    {
      "question": "6. In a $4 \\times 4$ square, cells are colored in several colors such that in any $1 \\times 3$ rectangle, there are two cells of the same color. What is the maximum number of colors that can be used?",
      "answer": "9"
    },
    {
      "question": "Standa and Jana received two three-digit numbers. Standa added a decimal point after the first digit in the first number, and after the second digit in the second number, then added the resulting decimal numbers and got the result 50.13. Jana added a decimal point after the second digit in the first number, and after the first digit in the second number, then added the resulting decimal numbers and got the result 34.02.\n\nDetermine the sum of the original three-digit numbers.\n\n(K. Pazourek)",
      "answer": "765"
    },
    {
      "question": "We know about a convex quadrilateral that if we divide it with diagonals, the area of three of the resulting four triangles is either $24 \\mathrm{~cm}^{2}$ or $25 \\mathrm{~cm}^{2}$. What is the maximum possible area of the quadrilateral?",
      "answer": "100\\frac{1}{24}"
    },
    {
      "question": "Someone wrote down the integers from 1 to 1999 next to each other. What digit stands at the 1999th position?",
      "answer": "7"
    },
    {
      "question": "3. Find the value of $\\frac{1}{1+1^{2}+1^{4}}+\\frac{2}{1+2^{2}+2^{4}}+\\frac{3}{1+3^{2}+3^{4}}+\\cdots+\\frac{100}{1+100^{2}+100^{4}}$.\n(1 mark)\nFind the value of $\\frac{1}{1+1^{2}+1^{4}}+\\frac{2}{1+2^{2}+2^{4}}+\\frac{3}{1+3^{2}+3^{4}}+\\cdots+\\frac{100}{1+100^{2}+100^{4}}$.",
      "answer": "\\frac{5050}{10101}"
    },
    {
      "question": "41. A class of natural numbers, starting from the third digit, each digit is exactly the sum of the two preceding digits, such as 134, 279, etc. The number of such three-digit numbers is $\\qquad$.",
      "answer": "45"
    },
    {
      "question": "【Question 14】\nAs shown in the figure, in rectangle $A B C D$, $A B=6, B C=8$. After folding rectangle $A B C D$ along $C E$, point $D$ lands on point $F$ on diagonal $A C$. What is the area of trapezoid $A B C E$? $\\qquad$",
      "answer": "39"
    },
    {
      "question": "10. $A 、 B$ two people bought the same number of sheets of letter paper and the same number of envelopes. $A$ put 1 sheet of letter paper in each envelope, and after using up all the envelopes, 40 sheets of letter paper were left; $B$ put 3 sheets of letter paper in each envelope, and after using up all the sheets of letter paper, 40 envelopes were left. They each bought $\\qquad$ sheets of letter paper.",
      "answer": "120"
    },
    {
      "question": "18. The diagram shows a semicircle with diameter $P Q$ inscribed in a rhombus $A B C D$. The rhombus is tangent to the arc of the semicircle in two places. Points $P$ and $Q$ lie on sides $B C$ and $C D$ of the rhombus respectively. The line of symmetry of the semicircle is coincident with the diagonal $A C$ of the rhombus. It is given that $\\angle C B A=60^{\\circ}$. The semicircle has radius 10 . The area of the rhombus can be written in the form $a \\sqrt{b}$ where $a$ and $b$ are integers and $b$ is prime. What is the value of $a b+a+b ?$",
      "answer": "603"
    },
    {
      "question": "5. Given positive numbers $a$ and $b$ satisfy $a+3b=7$. Then the minimum value of $\\frac{1}{1+a}$ $+\\frac{4}{2+b}$ is $\\qquad$ .",
      "answer": "\\frac{13+4\\sqrt{3}}{14}"
    },
    {
      "question": "3. Felix chooses a positive integer as the starting number and writes it on the board. He then repeatedly performs the following step: he replaces the number $n$ on the board with $\\frac{1}{2} n$ if $n$ is even and with $n^{2}+3$ if $n$ is odd.\n\n(a) Prove that if Felix starts with an odd number, the next two numbers he writes will be even.\n\n(b) For how many choices of starting numbers less than 2023 will Felix never write a number with more than four digits on the board?",
      "answer": "21"
    },
    {
      "question": "12. (5 points) Two people, A and B, start from points $A$ and $B$, which are 10 kilometers apart, and walk towards each other. If they start at the same time, they will meet 1 kilometer away from the midpoint of $A$ and $B$. If A starts 5 minutes later, they will meet at the midpoint of $A$ and $B$. At this time, A has walked $\\qquad$ minutes.",
      "answer": "10"
    },
    {
      "question": "4. In a charity donation event, a total of 72 people donated, and the total donation amount is a five-digit number, with the ten-thousands digit being 5, the hundreds digit being 3, the tens digit being 7, and the average donation amount per person is an integer. Therefore, the average donation per person is $\\qquad$ yuan.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。 \n\n---\n\nNote: The last sentence is a repetition of the instruction and should not be included in the translated text. Here is the corrected version:\n\n4. In a charity donation event, a total of 72 people donated, and the total donation amount is a five-digit number, with the ten-thousands digit being 5, the hundreds digit being 3, the tens digit being 7, and the average donation amount per person is an integer. Therefore, the average donation per person is $\\qquad$ yuan.",
      "answer": "783"
    },
    {
      "question": "15. Evaluate\n$$\n\\frac{678+690+702+714+\\cdots+1998+2010}{3+9+15+21+\\cdots+327+333} .\n$$",
      "answer": "16"
    },
    {
      "question": "1. (5 points) Calculate: $(2.016+201) \\times 201.7-20.16 \\times(20.17+2010)=$",
      "answer": "20.1"
    },
    {
      "question": "8.3. Toshi is traveling from point A to point B via point C. From A to C, Toshi travels at an average speed of 75 km/h, and from C to B, Toshi travels at an average speed of 145 km/h. The entire journey from A to B took Toshi 4 hours and 48 minutes. The next day, Toshi travels back at an average speed of 100 km/h. The journey from B to C took Toshi 2 hours, and the journey from C to A was completed at an average speed of 70 km/h. Find the distance between B and C.",
      "answer": "290"
    },
    {
      "question": "Let $a=\\log _{4} 9$ and $b=108 \\log _{3} 8$. What is the integer equal to $\\sqrt{a b}$ ?",
      "answer": "18"
    },
    {
      "question": "Task 2. What is the smallest natural number that is divisible by 2022 and whose notation starts with 2023?",
      "answer": "20230110"
    },
    {
      "question": "4. a) On a circle, the natural numbers from 1 to 20 are written in order in a clockwise direction. By a step, we mean to swap the positions of any two of the 20 numbers written on the circle. What is the minimum number of steps required to rearrange these numbers successively in the counterclockwise direction?\n\nb) What is the answer to the question in a) if 21 numbers are written on the circle in a clockwise direction from 1 to 21?\n\n## MATHEMATICS OLYMPIAD  SECTOR STAGE, 21.02.2016  5th GRADE  SOLUTIONS AND ORIENTATIVE SCORING GUIDELINES\n\nNote: Each problem is scored from 0 to 7 points. Only integer scores are given. Any other solution is assimilated according to the scoring guidelines.",
      "answer": "10"
    },
    {
      "question": "Problem 11.2. On each side of a triangle with angles $30^{\\circ}, 60^{\\circ}$ and $90^{\\circ}$ and hypotenuse 1 is chosen a point such that the triangle formed from these three points is a right triangle. What is the least value of the hypotenuse of this triangle?",
      "answer": "\\frac{\\sqrt{39}-\\sqrt{3}}{12}"
    },
    {
      "question": "14th USAMO 1985 Problem 3 A tetrahedron has at most one edge longer than 1. What is the maximum total length of its edges? Solution",
      "answer": "5+\\sqrt{3}"
    },
    {
      "question": "A square area of size $100 \\times 100$ is paved with $1 \\times 1$ square tiles of four colors: white, red, black, and gray - such that no two tiles of the same color touch each other (that is, they do not share a side or a vertex). How many red tiles can there be?",
      "answer": "2500"
    },
    {
      "question": "9. Given that the largest real root of the equation $x^{3}-7 x^{2}+1=0$ is $t$, then the remainder when $\\left[t^{20100}\\right]$ is divided by 7",
      "answer": "6"
    },
    {
      "question": "6. The integer part and the fractional part of $(\\sqrt{5}+2)^{2 n+1}(n \\in \\mathbf{N})$ are denoted as $P$ and $Q$, respectively, then $Q(P+Q)=$\n$\\qquad$",
      "answer": "1"
    },
    {
      "question": "[ The product of the lengths of the segments of chords and the lengths of the segments of secants ]\n\nFrom point $A$, two rays are drawn intersecting a given circle: one at points $B$ and $C$, and the other at points $D$ and $E$. It is known that $A B=7, B C=7, A D=10$. Find $D E$.\n\n#",
      "answer": "\\frac{1}{5}"
    },
    {
      "question": "If there exists a positive integer $m$ such that $m!$ ends with exactly $n$ zeros, then the positive integer $n$ is called a \"factorial tail\". How many numbers less than 1992 are not \"factorial tails\"?",
      "answer": "396"
    },
    {
      "question": "70. One year, the ages of Xiao Ming's family of four were: Father 30 years old. Mother is two years younger than Father, Brother's age is three times that of Xiao Ming, this year the total age of the four family members is 114, then Xiao Ming's age this year is $\\qquad$",
      "answer": "14"
    },
    {
      "question": "【Example 2】 6 athletes participate in the final, with no ties for any positions. Find the number of possible outcomes where Athlete No. 1 does not come in sixth place.",
      "answer": "600"
    },
    {
      "question": "10. (20 points) Let $x_{1}, x_{2}, x_{3}$ be non-negative real numbers, satisfying $x_{1}+x_{2}+x_{3}=1$. Find the minimum and maximum values of\n$$\n\\left(x_{1}+3 x_{2}+5 x_{3}\\right)\\left(x_{1}+\\frac{x_{2}}{3}+\\frac{x_{3}}{5}\\right)\n$$",
      "answer": "1"
    },
    {
      "question": "3. The first railway independently designed and constructed in our country, the \"Jingzhang Railway,\" adopted a \"human-shaped\" design. Now, a section of the human-shaped road is to be built, as shown in the figure, where the length of $\\mathrm{A} \\rightarrow \\mathrm{C}$ is 25 kilometers, the length of $\\mathrm{C} \\rightarrow \\mathrm{D}$ is 36 kilometers, and the length of $\\mathrm{A} \\rightarrow \\mathrm{B} \\rightarrow \\mathrm{D}$ is 43 kilometers. What is the length of $\\mathrm{B} \\rightarrow \\mathrm{C}$? $\\qquad$ kilometers.",
      "answer": "9"
    },
    {
      "question": "2. Which three-digit numbers are more numerous: those in which all digits have the same parity, or those in which adjacent digits have different parity",
      "answer": "225"
    },
    {
      "question": "## Task B-1.6.\n\nLuka and Pero decided to visit a friend and set off from place $A$ to place $B$, which is $170 \\mathrm{~km}$ away. Luka started walking, while Pero was driven by a motorcyclist. After some time, the motorcyclist returned to pick up Luka, and Pero continued on foot, so that both arrived at place $B$ at the same time. Both pedestrians walk at the same speed of $5 \\mathrm{~km} / \\mathrm{h}$, and the motorcyclist drives at a speed of $85 \\mathrm{~km} / \\mathrm{h}$. At what distance from place $B$ and after how much time did the motorcyclist stop and return to pick up Luka? How long did it take for Luka and Pero to travel from place $A$ to place $B$?",
      "answer": "5"
    },
    {
      "question": "11. (12 points) The three brothers, Big Mao, Second Mao, and Third Mao, Big Mao said to Third Mao: “When dad was 36 years old, my age was 4 times yours, and Second Mao's age was 3 times yours.” Second Mao said: “Yes, at that time, the sum of our ages was exactly half of dad's current age.” Third Mao said: “Now, the total age of the four of us is already 108 years old.” So, how old is Third Mao this year? $\\qquad$",
      "answer": "15"
    },
    {
      "question": "## Zadatak B-1.3.\n\nAko je $\\frac{a}{b}=\\frac{c}{d}, b \\neq 0, d \\neq 0$ koliko je\n\n$$\n\\frac{(a+c)(b+d)}{a+b+c+d}-\\frac{a b}{a+b}-\\frac{c d}{c+d} ?\n$$",
      "answer": "0"
    },
    {
      "question": "## Task 1 - 050521\n\nFrom 36 equally large squares, a rectangle is to be formed by placing them together.\n\na) How many solution possibilities are there? (In each solution, all squares should be used.)\n\nb) Which of the possible rectangles has the smallest perimeter?",
      "answer": "24"
    },
    {
      "question": "Tokarev S.i.\n\nThe set of five-digit numbers $\\{N_1, \\dots, N_k\\}$ is such that any five-digit number, all of whose digits are in increasing order, coincides in at least one digit with at least one of the numbers $N_1, \\dots, N_k$. Find the smallest possible value of $k$.",
      "answer": "1"
    },
    {
      "question": "Example 5 Given: $a, b, c$ are all positive numbers, find the maximum value of $y=\\frac{a b+2 b c}{a^{2}+b^{2}+c^{2}}$.\nAnalysis In the process of using the arithmetic mean inequality, note that the coefficient ratio of $a b$ and $b c$ should be $1: 2$, so we use parameters to determine the method of splitting $b^{2}$.",
      "answer": "\\frac{\\sqrt{5}}{2}"
    },
    {
      "question": "Example 2. The probability density of a random variable $X$ is given by the function\n\n$$\np(x)= \\begin{cases}0 & \\text { if } x \\leq 0 \\\\ x / 2 & \\text { if } 0 < x \\leq 2\\end{cases}\n$$\n\nFind the probability that in the experiment, the variable $X$ will take a value from the interval $(1,2)$.",
      "answer": "0.75"
    },
    {
      "question": "Problem 2. Duorites are creatures that have two horns, while heptorites have 7 horns. In one herd, there were both types of creatures. Together, they had 16 horns. How many duorites and heptorites were there in this herd?",
      "answer": "1"
    },
    {
      "question": "Sure, here is the translated text:\n\n```\nFind all real numbers $p$ such that the cubic equation\n$\n5 x^{3}-5(p+1) x^{2}+(71 p-1) x+1=66 p\n$\n\nhas three roots that are all natural numbers.\n```",
      "answer": "76"
    },
    {
      "question": "5. The parabolas in the figure are obtained by shifting the parabola $f(x)=x^{2}$ and are arranged such that the points of their intersections with the $O X$ axis respectively coincide in pairs, and all vertices lie on the same straight line. There are a total of 2020 parabolas. The length of the segment on the $O X$ axis enclosed between the roots of the first parabola is 1; the length of the segment enclosed between the roots of the second parabola is 2. Find the length of the segment on the $O X$ axis enclosed between the roots of the last parabola.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_eefe83f285a2477dfcd6g-2.jpg?height=357&width=539&top_left_y=1689&top_left_x=770)",
      "answer": "2020"
    },
    {
      "question": "10.4. Find the maximum value of the expression $a+b+c+d-ab-bc-cd-da$, if each of the numbers $a, b, c$ and $d$ belongs to the interval $[0 ; 1]$.",
      "answer": "2"
    },
    {
      "question": "9.1. A four-digit number $n$ was doubled and 1000 was added to the result. As a result, a four-digit number was obtained, written with the same digits as $n$, but in reverse order. Find all possible values of $n$.",
      "answer": "2996"
    },
    {
      "question": "2. Andrey, Boris, Vasily, Gennady, and Dmitry played table tennis in pairs such that every two of them played with every other pair exactly once. There were no draws in the tennis matches. It is known that Andrey lost exactly 12 times, and Boris lost exactly 6 times. How many times did Gennady win?\n\nOm vem: Gennady won 8 times.",
      "answer": "8"
    },
    {
      "question": "6. Fill in the blanks with appropriate numbers to make the vertical calculation valid. The result of this multiplication problem is $\\qquad$ .\n\n\n\nThe translation is as follows:\n\n6. Fill in the blanks with appropriate numbers to make the vertical calculation valid. The result of this multiplication problem is $\\qquad$ .",
      "answer": "726"
    },
    {
      "question": "# 4.1. Condition:\n\nIn front of the elevator stand people weighing 50, 51, 55, 57, 58, 59, 60, 63, 75, and 140 kg. The elevator's load capacity is 180 kg. What is the minimum number of trips needed to get everyone up?",
      "answer": "4"
    },
    {
      "question": "12. Let $x \\in \\mathbf{R}$, then the minimum value of the function $f(x)=|2 x-1|+|3 x-2|+|4 x-3|+|5 x-4|$ is",
      "answer": "1"
    },
    {
      "question": "7.1. Find the smallest natural number divisible by 5, with the sum of its digits being 100. Justify your answer.",
      "answer": "599999999995"
    },
    {
      "question": "7th Australian 1986 Problem A2 What is the smallest tower of 100s that exceeds a tower of 100 threes? In other words, let a 1 = 3, a 2 = 3 3 , and a n+1 = 3 a n . Similarly, b 1 = 100, b 2 = 100 100 etc. What is the smallest n for which b n > a 100 ?",
      "answer": "99"
    },
    {
      "question": "13. Fox Alice and Cat Basilio (8th grade. 2 points). Every day, Cat Basilio and Fox Alice visit all 20 courtyards of the capital of the Country of Fools, and in each courtyard, they either receive or do not receive one gold coin with a probability of $\\frac{1}{2}$. If the number of gold coins collected by the end of the day is even, they split them equally. If it is odd, they split all but one coin equally, and the last coin is taken by Cat Basilio as the commercial director of the enterprise. Find the expected number of coins received by the cat per day.",
      "answer": "5.25"
    },
    {
      "question": "2. In pentagon $A B C D E, A B=B C=C D=D E, \\angle B=96^{\\circ}$ and $\\angle C=\\angle D=108^{\\circ}$. Find $\\angle E$.\n(1 mark)\n在五邊形 $A B C D E$ 中, $A B=B C=C D=D E, \\angle B=96^{\\circ}$ 且 $\\angle C=\\angle D=108^{\\circ}$ 。\n求 $\\angle E$ 。",
      "answer": "102"
    },
    {
      "question": "11. Let $A B C D E F$ be a convex equilateral hexagon such that lines $B C, A D$, and $E F$ are parallel. Let $H$ be the orthocenter of triangle $A B D$. If the smallest interior angle of the hexagon is 4 degrees, determine the smallest angle of the triangle $H A D$ in degrees.",
      "answer": "3"
    },
    {
      "question": "19. Alice and Bob play a game in which two thousand and eleven $2011 \\times 2011$ grids are distributed between the two of them, 1 to Bob, and the other 2010 to Alice. They go behind closed doors and fill their grid(s) with the numbers $1,2, \\ldots, 2011^{2}$ so that the numbers across rows (left-to-right) and down columns (top-to-bottom) are strictly increasing. No two of Alice's grids may be filled identically. After the grids are filled, Bob is allowed to look at Alice's grids and then swap numbers on his own grid, two at a time, as long as the numbering remains legal (i.e. increasing across rows and down columns) after each swap. When he is done swapping, a grid of Alice's is selected at random. If there exist two integers in the same column of this grid that occur in the same row of Bob's grid, Bob wins. Otherwise, Alice wins. If Bob selects his initial grid optimally, what is the maximum number of swaps that Bob may need in order to guarantee victory?",
      "answer": "1"
    },
    {
      "question": "11. Find the maximum and minimum values of the function $y=\\sqrt{x+27}+\\sqrt{13-x}+\\sqrt{x}$.\n\n untranslated portion:\n \n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。\n\n---\n\n11. Find the maximum and minimum values of the function $y=\\sqrt{x+27}+\\sqrt{13-x}+\\sqrt{x}$.",
      "answer": "11"
    },
    {
      "question": "## Task 1 - 340711\n\nArmin has 100 sticks of $7 \\mathrm{~cm}$ each and 100 sticks of $12 \\mathrm{~cm}$ each. He wants to lay out a line segment of $1 \\mathrm{~m}$ in length using such sticks. The sticks should always fit seamlessly together, and no part of the segment should be covered more than once.\n\nFind all the possibilities for how many $7 \\mathrm{~cm}$ sticks and how many $12 \\mathrm{~cm}$ sticks can be combined to form such a layout!",
      "answer": "4"
    },
    {
      "question": "3. Through the point $(2,3)$ on the line $p: y=2 x-1$, a line $q$ perpendicular to the line $p$ was drawn. Find the area of the convex quadrilateral bounded by the lines $p, q$ and the coordinate axes.",
      "answer": "\\frac{19}{4}"
    },
    {
      "question": "(14) From the left focus $F$ of the hyperbola $\\frac{x^{2}}{9}-\\frac{y^{2}}{16}=1$, draw a tangent to the circle $x^{2}+y^{2}=9$, with the point of tangency being $T$. Extend $F T$ to intersect the right branch of the hyperbola at point $P$. If $M$ is the midpoint of segment $F P$, and $O$ is the origin, find the value of $|M O|-| M T|$.",
      "answer": "1"
    },
    {
      "question": "In the plane rectangular coordinate system $x O y$, the set of points\n$$\n\\begin{aligned}\nK= & \\{(x, y) \\mid(|x|+|3 y|-6) \\cdot \\\\\n& (|3 x|+|y|-6) \\leqslant 0\\}\n\\end{aligned}\n$$\n\ncorresponds to a plane region whose area is $\\qquad$",
      "answer": "24"
    },
    {
      "question": "## Task 2 - 080522\n\nIn a warehouse, there are three times as many kilograms of wheat as in a second one. After removing $85000 \\mathrm{~kg}$ from the first and $5000 \\mathrm{~kg}$ from the second, the stocks were equal.\n\nHow many tons of wheat were in the first and how many in the second warehouse before the removal?",
      "answer": "120"
    },
    {
      "question": "1. Jeanne is standing in a single line with several other people. There are four people in front of her and seven people behind her. How many people are in the line?",
      "answer": "12"
    },
    {
      "question": "11.5. For what minimum natural $n$ can $n$ distinct natural numbers $s_{1}, s_{2}, \\ldots, s_{n}$ be found such that $\\left(1-\\frac{1}{s_{1}}\\right)\\left(1-\\frac{1}{s_{2}}\\right) \\ldots\\left(1-\\frac{1}{s_{n}}\\right)=\\frac{7}{66} ?$",
      "answer": "9"
    },
    {
      "question": "## Zadatak A-1.1.\n\nOdredi $x_{1006}$ ako je\n\n$$\n\\begin{aligned}\n& \\frac{x_{1}}{x_{1}+1}= \\frac{x_{2}}{x_{2}+3}=\\frac{x_{3}}{x_{3}+5}=\\ldots=\\frac{x_{1006}}{x_{1006}+2011} \\\\\n& x_{1}+x_{2}+\\ldots+x_{1006}=503^{2}\n\\end{aligned}\n$$",
      "answer": "\\frac{2011}{4}"
    },
    {
      "question": "12 Positive integer $n \\leqslant 500$, has the following property: For any element $m$ chosen from the set $\\{1,2, \\cdots, 500\\}$, the probability that $m$ divides $n$ is $\\frac{1}{100}$, then the maximum value of $n$ is $\\qquad$ .",
      "answer": "81"
    },
    {
      "question": "1. On the board, three quadratic equations are written:\n\n$$\n\\begin{gathered}\n2020 x^{2}+b x+2021=0 \\\\\n2019 x^{2}+b x+2020=0 \\\\\nx^{2}+b x+2019=0\n\\end{gathered}\n$$\n\nFind the product of the roots of all the equations written on the board, given that each of them has two real roots.",
      "answer": "2021"
    },
    {
      "question": "G3.1 Given that the solution of the equation $\\sqrt{3 x+1}+\\sqrt{3 x+6}=\\sqrt{4 x-2}+\\sqrt{4 x+3}$ is $a$, find the value of $a$.",
      "answer": "3"
    },
    {
      "question": "7. Given the sequence $\\left\\{a_{n}\\right\\}$:\n$$\na_{n}=2^{n}+3^{n}+6^{n}+1\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nDoes there exist an integer $k \\geqslant 2$, such that $k$ is coprime with all numbers in the sequence $\\left\\{a_{n}\\right\\}$? If it exists, find the smallest integer $k$; if not, explain the reason. (Zhang Lei provided the problem)",
      "answer": "23"
    },
    {
      "question": "Fedorov R.M.\n\nIn the Banana Republic, parliamentary elections were held in which all residents participated. All those who voted for the \"Mandarin\" party love mandarins. Among those who voted for other parties, 90% do not like mandarins. What percentage of the votes did the \"Mandarin\" party receive in the elections, if exactly $46\\%$ of the residents love mandarins?",
      "answer": "40"
    },
    {
      "question": "1167. A locomotive is moving along a horizontal section of the track c) at a speed of 72 km/hour. In what time and at what distance will it be stopped by the brake, if the resistance to motion after the start of braking is equal to 0.2 of its weight.",
      "answer": "102"
    },
    {
      "question": "## SUBJECT I\n\nSolve the equation $\\frac{x-2}{x}+\\frac{x-4}{x}+\\frac{x-6}{x}+\\ldots . . .+\\frac{2}{x}=12$",
      "answer": "50"
    },
    {
      "question": "6. Let real numbers $a, b, c$ satisfy $a^{2}+b^{2}+c^{2}=1$, and let the maximum and minimum values of $a b+b c+c a$ be $M$ and $m$, respectively, then $M-m=$",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "Coloring Vertex Degree\n\n20 phones are connected by wires in such a way that each wire connects two phones, each pair of phones is connected by no more than one wire, and no more than two wires extend from each phone. It is necessary to color the wires (each wire entirely in one color) so that the wires extending from each phone are of different colors. What is the smallest number of colors sufficient for such a coloring?",
      "answer": "3"
    },
    {
      "question": "9.5. In the parliament of the island state of Promenade-and-Tornado, 2019 indigenous inhabitants were elected, who are divided into knights and liars: knights always tell the truth, liars always lie. At the first meeting, 2016 of them sat in the parliamentary seats arranged in the hall in a rectangle of $42 \\times 48$, three - in the seats of the chairman and his deputies in the presidium. During the meeting, each one stated that among their neighbors in the seat there are both knights and liars (neighbors are those who sit to the left, right, in front, behind, and diagonally: there can be from 3 to 8 in the hall and 1 or 2 in the presidium). Determine the minimum number of liars at the meeting.",
      "answer": "227"
    },
    {
      "question": "5. Given positive real numbers $a$ and $b$ satisfy $a+b=1$, then $M=$ $\\sqrt{1+a^{2}}+\\sqrt{1+2 b}$ the integer part is",
      "answer": "2"
    },
    {
      "question": "IMO 1991 Problem A3 Let S = {1, 2, 3, ... 280}. Find the smallest integer n such that each n-element subset of S contains five numbers which are pairwise relatively prime. Solution",
      "answer": "217"
    },
    {
      "question": "4. (1990 AIME Problem 8) \\( n \\) is the smallest positive integer that satisfies the following conditions:\n(1) \\( n \\) is a multiple of 75.\n(2) \\( n \\) has exactly 75 positive divisors (including 1 and itself).\nFind \\( \\frac{n}{75} \\).",
      "answer": "432"
    },
    {
      "question": "8. (10 points) In the inscribed quadrilateral $A B C D$, the degree measures of the angles are in the ratio $\\angle A: \\angle B: \\angle C=2: 3: 4$. Find the length of $A C$, if $C D=18, B C=12 \\sqrt{3}-9$.",
      "answer": "30"
    },
    {
      "question": "Anička got a cake in the shape of a rectangle for her birthday. She cut it using two straight cuts. One cut went through the longer sides of the rectangle, intersecting them at one-third of their length, and the other cut went through the shorter sides, intersecting them at one-fifth of their length. Neither cut was parallel to the sides of the rectangle, and at each vertex of the rectangle, either two shorter segments or two longer segments of the divided sides always met.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_24585cf941083de07254g-3.jpg?height=402&width=552&top_left_y=553&top_left_x=752)\n\nAnička ate the gray-marked piece of the cake. Determine what fraction of the cake this was.\n\n(A. Bohiniková)",
      "answer": "\\frac{2}{15}"
    },
    {
      "question": "G6.4 If $M=\\left(10^{2}+10 \\times 1+1^{2}\\right)\\left(10^{2}-1^{2}\\right)\\left(10^{2}-10 \\times 1+1^{2}\\right)$, find $M$.",
      "answer": "999999"
    },
    {
      "question": "10}\n\nPoints $A, B, C, D$ are marked on a sheet of paper. A recognition device can perform two types of operations with absolute accuracy: a) measure the distance in centimeters between two given points; b) compare two given numbers. What is the minimum number of operations this device needs to perform to definitely determine whether the quadrilateral $A B C D$ is a rectangle?",
      "answer": "10"
    },
    {
      "question": "10. Given positive real numbers $a, b$ satisfy: for real number $x$, when $1 \\leqslant x \\leqslant 4$, we have $a x+b-3 \\leqslant 0$. Then the minimum value of $\\frac{1}{a}-b$ is $\\qquad$",
      "answer": "1"
    },
    {
      "question": "20. Let $\\left\\{a_{n}\\right\\}$ and $\\left\\{b_{n}\\right\\}$ be sequences defined recursively by $a_{0}=2 ; b_{0}=2$, and $a_{n+1}=a_{n} \\sqrt{1+a_{n}^{2}+b_{n}^{2}}-b_{n}$; $b_{n+1}=b_{n} \\sqrt{1+a_{n}^{2}+b_{n}^{2}}+a_{n}$. Find the ternary (base 3) representation of $a_{4}$ and $b_{4}$.",
      "answer": "10000011001112222211100110000012"
    },
    {
      "question": "The lottery numbers drawn on February 28, 1969, were: 2, 34, 36, 50, 65. In connection with this, let's calculate the probability that in a lottery draw, the sum of the first two numbers, when arranged in ascending order, will be equal to the third number. What is the most frequent value of the third number in such draws?",
      "answer": "31"
    },
    {
      "question": "A1. A set $A$ is endowed with a binary operation $*$ satisfying the following four conditions:\n\n(1) If $a, b, c$ are elements of $A$, then $a *(b * c)=(a * b) * c$;\n\n(2) If $a, b, c$ are elements of $A$ such that $a * c=b * c$, then $a=b$;\n\n(3) There exists an element $e$ of $A$ such that $a * e=a$ for all $a$ in $A$; and\n\n(4) If $a$ and $b$ are distinct elements of $A \\backslash\\{e\\}$, then $a^{3} * b=b^{3} * a^{2}$, where $x^{k}=x * x^{k-1}$ for all integers $k \\geq 2$ and all $x$ in $A$.\n\nDetermine the largest cardinality $A$ may have.\n\n## BoJan Bašić, Serbia",
      "answer": "3"
    },
    {
      "question": "\\section*{Exercise 3 - 241023}\n\nDetermine all natural numbers \\(z\\) for which the following holds:\n\nIf the last digit is removed from the digit representation of \\(z\\), the resulting digit representation is a divisor of \\(z\\).",
      "answer": "1"
    },
    {
      "question": "On an $A B=8 \\mathrm{~km}$ long road, buses travel in both directions at a speed of $12 \\mathrm{~km}$ per hour. The first bus from each location departs at 6 o'clock, and a new bus departs every 10 minutes.\n\nA pedestrian starts from $A$ towards $B$ at $81 / 4$ hours; their speed is $4 \\mathrm{~km}$ per hour.\n\nDetermine graphically how many oncoming buses the pedestrian encounters, and when and where.",
      "answer": "16"
    },
    {
      "question": "14th ASU 1980 Problem 12 Some unit squares in an infinite sheet of squared paper are colored red so that every 2 x 3 and 3 x 2 rectangle contains exactly two red squares. How many red squares are there in a 9 x 11 rectangle?",
      "answer": "33"
    },
    {
      "question": "3. Petrov and Vasechkin were solving the same arithmetic problem. A certain number had to be divided by 2, multiplied by 7, and 1001 subtracted. Petrov performed all the operations correctly, while Vasechkin got everything mixed up: he divided by 8, squared the result, and also subtracted 1001. It is known that Petrov got a prime number. What number did Vasechkin get?",
      "answer": "295"
    },
    {
      "question": "26. Given any 4-digit positive integer $x$ not ending in ' 0 ', we can reverse the digits to obtain another 4 -digit integer $y$. For example if $x$ is 1234 then $y$ is 4321 . How many possible 4-digit integers $x$ are there if $y-x=3177$ ?",
      "answer": "48"
    },
    {
      "question": "## SUBJECT 2\n\nFind the cardinality of the set:\n\n$$\nA=\\left\\{\\overline{a b c} / \\frac{a \\sqrt{2}}{\\sqrt{1}+\\sqrt{2}+\\sqrt{3}}-\\frac{b \\sqrt{3}}{\\sqrt{1}+\\sqrt{3}+\\sqrt{4}}-\\frac{2 c \\sqrt{8}}{\\sqrt{4}+\\sqrt{8}+\\sqrt{12}}=\\frac{-3}{\\sqrt{2}} ; a \\neq b \\neq c\\right\\}\n$$\n\nProf. Nicolae Jurubiță",
      "answer": "4"
    },
    {
      "question": "3B. Determine all natural numbers $x$ such that $x^{3}-x+p=507$, where $p$ is a prime number.",
      "answer": "8"
    },
    {
      "question": "Pedro wants to paint a cubic box in such a way that the faces that share an edge are painted in different colors. Calculate the minimum number of colors that will be needed to paint the box in this way.",
      "answer": "3"
    },
    {
      "question": "76. There are 13 different candies. In how many ways can a set be formed containing an odd number of candies",
      "answer": "2^{12}"
    },
    {
      "question": "6. Determine all possible values of the sum $a+b+c+d$, where $a, b, c, d$ are natural numbers satisfying the equation\n\n$$\n\\left(a^{2}-b^{2}\\right)\\left(c^{2}-d^{2}\\right)+\\left(b^{2}-d^{2}\\right)\\left(c^{2}-a^{2}\\right)=2021 .\n$$\n\n(Mária Dományová, Patrik Bak)",
      "answer": "90"
    },
    {
      "question": "Let $T$ be the set of all positive integer divisors of $2004^{100}$. What is the largest possible number of elements that a subset $S$ of $T$ can have if no element of $S$ is an integer multiple of any other element of $S$?",
      "answer": "10201"
    },
    {
      "question": "## Task B-2.2.\n\nA natural number is said to be a palindrome if it reads the same from left to right and from right to left in decimal notation. We can order palindromes by size. Determine the 2010th palindrome in this order.",
      "answer": "1011101"
    },
    {
      "question": "37. We use one finger to represent a binary digit, so we can represent a binary number with our hands. A finger curled up represents the number 0, and a finger extended represents the number 1.\nFor example, in Figure 1, Guangtou Qiang's gesture represents the binary number $(10000)_2$, which is converted to a decimal number as follows: $1 \\times 16 + 0 \\times 8 + 0 \\times 4 + 0 \\times 2 + 0 \\times 1$, which equals 16.\nIn Figure 2, Guangtou Qiang's gesture represents the binary number $(1100)_2$, which is converted to a decimal number as follows: $1 \\times 8 + 1 \\times 4 + 0 \\times 2 + 0 \\times 1$, which equals 12.\n\nThen, in Figure 3, the gesture of Sailor Moon represents the binary number, which, when converted to a decimal number, equals $\\qquad$",
      "answer": "25"
    },
    {
      "question": "3. For the parabola $y^{2}=4 a x(a>0)$, the focus is $A$. With $B(a+4,0)$ as the center and $|A B|$ as the radius, a semicircle is drawn above the $x$-axis, intersecting the parabola at two different points $M, N$. Let $P$ be the midpoint of the segment $M N$. (I) Find the value of $|A M|+|A N|$; (II) Does there exist such an $a$ that $|A M|,|A P|,|A N|$ form an arithmetic sequence?",
      "answer": "8"
    },
    {
      "question": "3. Find the number of natural numbers $k$, not exceeding 291000, such that $k^{2}-1$ is divisible by 291.",
      "answer": "4000"
    },
    {
      "question": "19.3. Can the number of diagonals of a polygon be exactly twice the number of its sides?\n\n$(6-8$ grades $)$",
      "answer": "7"
    },
    {
      "question": "Example 2.52. Calculate the area of the figure bounded by one arch of the cycloid $x=2(t-\\sin t), y=$ $=2(1-\\cos t)$.",
      "answer": "12\\pi"
    },
    {
      "question": "15. [9] Let $a, b, c$ be positive integers such that\n$$\n\\frac{a}{77}+\\frac{b}{91}+\\frac{c}{143}=1 .\n$$\n\nWhat is the smallest possible value of $a+b+c$ ?",
      "answer": "79"
    },
    {
      "question": "Problem 4. Consider a $2018 \\times$ 2018 board. We call an LC tile a tile consisting of 9 congruent unit squares as shown in the figure below\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_fbc9a4a5e943c67ef798g-3.jpg?height=143&width=157&top_left_y=2341&top_left_x=961)\n\nWhat is the maximum number of LC tiles that can be placed without overlapping on the board? (Each of the 9 unit squares of the tile must overlap with one of the unit squares of the board; a tile can be rotated, mirrored, etc.)\n\nAlexandru Gîrban",
      "answer": "672^2"
    },
    {
      "question": "$10 \\cdot 10$ Given a four-digit number that satisfies the following conditions:\n(1) If the units digit and the hundreds digit, as well as the tens digit and the thousands digit, are simultaneously swapped, then its value increases by 5940;\n(2) When divided by 9, the remainder is 8.\nFind the smallest odd number among these four-digit numbers.\n(Shandong Province, China Mathematics Competition, 1979)",
      "answer": "1979"
    },
    {
      "question": "8-1. Two rectangles $8 \\times 10$ and $12 \\times 9$ are overlaid as shown in the figure. The area of the black part is 37. What is the area of the gray part? If necessary, round the answer to 0.01 or write the answer as a common fraction.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1d29d5cb430346a58eb5g-15.jpg?height=471&width=577&top_left_y=684&top_left_x=745)",
      "answer": "65"
    },
    {
      "question": "1308. Calculate the volume of the body obtained by rotating around the $O x$ axis the figure bounded by the parabolas $y=x^{2}$ and $x=y^{2}$.",
      "answer": "\\frac{3}{10}\\pi"
    },
    {
      "question": "We wrote the numbers from 1 to 2009 on a piece of paper. In the second step, we also wrote down twice each of these numbers on the paper, then erased the numbers that appeared twice. We repeat this step in such a way that in the $i$-th step, we also write down $i$ times each of the numbers $1, 2, \\ldots, 2009$ on the paper, then erase the numbers that appear twice. How many numbers will be on the paper after the 2009th step?",
      "answer": "2009"
    },
    {
      "question": "8.2. Points $A, B, C, D$ are the centers of the circles inscribed in the faces $P Q S, P R S$, $Q R S$ and $P Q R$ of the tetrahedron $P Q R S$. It is known that $P Q=R S=7, P R=Q S=8, P S=Q R=9$. Find the volume of the tetrahedron $A B C D$. Round your answer to the nearest hundredth if necessary.",
      "answer": "1.84"
    },
    {
      "question": "8.5. On an $8 \\times 8$ chessboard, $k$ rooks and $k$ knights are placed such that no figure attacks any other. What is the largest $k$ for which this is possible?",
      "answer": "5"
    },
    {
      "question": "10. In the methods for finding the zeros of a function, there is one called the Newton's method. The process of this method is: for the function $f(x)$, given $x_{1}$, and the intersection of the tangent line of $f(x)$ passing through the point $Q_{n}\\left(x_{n}, f\\left(x_{n}\\right)\\right)$ with the $x$-axis is denoted as $x_{n+1}(n=1,2,3, \\ldots \\ldots)$, when $n$ is sufficiently large, we can use $x_{n}$ as an approximate value of the zero. Let $f(x)=x^{2}-2, x_{1}=2$, find the recursive formula of the sequence $\\left\\{x_{n}\\right\\}$, and find its limit.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "What is the maximum number of kings that can be placed on an $8 \\times 8$ chessboard without any of them attacking each other?\n\n#",
      "answer": "16"
    },
    {
      "question": "Example 2 Let $S=\\{1,2,3,4\\}, n$ terms of the sequence: $a_{1}, a_{2}, \\cdots, a_{n}$ have the following property: for any non-empty subset $B$ of $S$ (the number of elements in $B$ is denoted by। $B$ ।), there are adjacent। $B$ ।terms in the sequence that exactly form the set $B$. Find the minimum value of $n$.\n(1997 Shanghai Competition Question)",
      "answer": "8"
    },
    {
      "question": "39. Junior high school student Xiao Peng was surprised to find that: his age this year (2018) is exactly the sum of the four digits of his birth year plus 10. He told this discovery to his math teacher, Mr. Wang, who thought for a moment and said: My age this year is also exactly the sum of the four digits of my birth year plus 10. So, Mr. Wang is $\\qquad$ years older than Xiao Peng.",
      "answer": "18"
    },
    {
      "question": "10. Washing a pile of vegetables to remove residual pesticides with water. The effect of washing once with a certain amount of water is assumed as follows: 1 unit of water can remove $\\frac{1}{2}$ of the residual pesticides on the vegetables. The more water used, the more pesticides are removed, but there will always be some pesticides left on the vegetables. Let the ratio of the residual pesticide amount on the vegetables after washing once with $x$ units of water to the residual pesticide amount before washing be the function $f(x)$.\n(1) Specify the value of $f(0)$ and explain its practical significance;\n(2) Write down the conditions and properties that the function $f(x)$ should satisfy based on the assumptions;\n(3) Suppose $f(x)=\\frac{1}{1+x^{2}}$. Given $a (a>0)$ units of water, you can wash once, or you can divide the water into 2 equal parts and wash twice. Which method leaves fewer residual pesticides on the vegetables? Explain your reasoning.",
      "answer": "2\\sqrt{2}"
    },
    {
      "question": "(13) As shown in the figure, in the Cartesian coordinate system, the circle $M$ with the equation $x^{2}+y^{2}+D x+E y+F=0$ has an inscribed quadrilateral $A B C D$ whose diagonals $A C$ and $B D$ are perpendicular to each other, and $A C$ and $B D$ lie on the $x$-axis and $y$-axis, respectively.\n(1) Prove that $F<0$;\n(2) If the area of quadrilateral $A B C D$ is 8, the length of diagonal $A C$ is 2, and $\\overrightarrow{A B} \\cdot \\overrightarrow{A D}=0$, find the value of $D^{2}+E^{2}-4 F$.",
      "answer": "64"
    },
    {
      "question": "12. Restore the digits. On the board, there was the product of three consecutive even numbers. During the break, Vasya erased some digits. As a result, on the board remained $87 * * * * * 8$. Help Petya find the missing digits in the product.",
      "answer": "87526608"
    },
    {
      "question": "5. The minimum value of the function $y=\\sqrt{x^{2}-2 x+5}+\\sqrt{x^{2}-4 x+13}$ is",
      "answer": "\\sqrt{26}"
    },
    {
      "question": "## Task A-4.1.\n\nFor a natural number $n$, let $s(n)$ denote the sum of its positive divisors, and $d(n)$ the number of its positive divisors. Determine all natural numbers $n$ such that\n\n$$\ns(n)=n+d(n)+1\n$$",
      "answer": "9"
    },
    {
      "question": "7. Let $S=\\left\\{(x, y) \\in \\mathbb{Z}^{2} \\mid 0 \\leq x \\leq 11,0 \\leq y \\leq 9\\right\\}$. Compute the number of sequences $\\left(s_{0}, s_{1}, \\ldots, s_{n}\\right)$ of elements in $S$ (for any positive integer $n \\geq 2$ ) that satisfy the following conditions:\n- $s_{0}=(0,0)$ and $s_{1}=(1,0)$,\n- $s_{0}, s_{1}, \\ldots, s_{n}$ are distinct,\n- for all integers $2 \\leq i \\leq n, s_{i}$ is obtained by rotating $s_{i-2}$ about $s_{i-1}$ by either $90^{\\circ}$ or $180^{\\circ}$ in the clockwise direction.",
      "answer": "646634"
    },
    {
      "question": "19. Next year's 'leap day', 29th February 2024 , may be written in 'ddmmyy' format as the number 290224. What is the sum of all the two-digit factors of 290224 ?",
      "answer": "397"
    },
    {
      "question": "3. Three brigades, working together, must complete a certain job. It is known that the first and second brigades together can complete it 36 minutes faster than the third brigade. In the time it takes for the first and third brigades to complete the job together, the second brigade can complete only half of the job. In the time it takes for the second and third brigades to complete the job together, the first brigade can complete $\\frac{2}{7}$ of the job. How long will it take for all three brigades to complete the job together?",
      "answer": "1"
    },
    {
      "question": "9.1. For non-negative numbers $a, b, c, d$, the following equalities are satisfied:\n\n$\\sqrt{a+b}+\\sqrt{c+d}=\\sqrt{a+c}+\\sqrt{b+d}=\\sqrt{a+d}+\\sqrt{b+c}$. What is the maximum number of distinct values that can be among the numbers $a, b, c, d$?",
      "answer": "2"
    },
    {
      "question": "4. Find the smallest four-digit number that is equal to the square of the sum of the numbers formed by its first two digits and its last two digits.",
      "answer": "2025"
    },
    {
      "question": "14. Garden plot. The length of a rectangular plot of land was increased by $35 \\%$, and the width was decreased by $14 \\%$. By what percentage did the area of the plot change?",
      "answer": "16.1"
    },
    {
      "question": "1. If we insert a zero between the digits of a two-digit number, we get a number that is nine times larger. Write down all such two-digit numbers.",
      "answer": "45"
    },
    {
      "question": "Problem 10. The sum of the surface areas of the polyhedra into which a parallelepiped is divided by sections is equal to the sum of the surface area of the parallelepiped and the areas of the internal surfaces. The sum of the areas of the internal surfaces is equal to twice the sum of the areas of the sections.\n\nLet's find the maximum possible area of a section passing through the diagonal $XY$ of an arbitrary parallelepiped with edges $a \\leq b \\leq c$. The section is a parallelogram $ZXTY$, whose vertices lie on opposite edges of the parallelepiped. The area of the parallelogram is the product of the length of the diagonal $XY$ and the distance from point $Z$ to $XY$.\n\nConsider the projection of the parallelepiped onto a plane perpendicular to the diagonal $XY$. From the diagram, it is clear that the distance from point $Z$ to the broken line $ABC$, and thus to the diagonal $XY$, is greatest if\n\n$Z$ coincides with one of the vertices $A, B$, or $C$.\n\nThis means that the section passes through one of the edges of the parallelepiped. Therefore, the section with the largest area is one of the diagonal sections. All these sections are rectangles. Let's find the largest of their areas:\n\n$$\nS_{1}=a \\sqrt{b^{2}+c^{2}}, \\quad S_{2}=b \\sqrt{a^{2}+c^{2}}, \\quad \\text{and} \\quad S_{3}=c \\sqrt{b^{2}+a^{2}}\n$$\n\nFrom the condition $a \\leq b \\leq c$, it follows that\n\n$a^{2} b^{2}+a^{2} c^{2} \\leq c^{2} b^{2}+a^{2} c^{2}$, and $a^{2} b^{2}+c^{2} b^{2} \\leq c^{2} b^{2}+a^{2} c^{2}$. Therefore, $S_{1} \\leq S_{3}$ and $S_{2} \\leq S_{3}$. This means that the section with the largest area passes through the largest edge. According to the condition, the largest length is the edge $AA_{1}$, so the sections $AA_{1}C_{1}C$ and $BB_{1}D_{1}D$ have the largest area:\n\n$5 \\sqrt{4^{2}+3^{2}}=25$. The sum of the surface areas of the polyhedra into which the parallelepiped is divided by these sections (see the diagram) is\n\n$$\n2\\left(A A_{1} \\cdot A B + A A_{1} \\cdot A D + A B \\cdot A D\\right) + 4 \\cdot 25 = 194\n$$",
      "answer": "194"
    },
    {
      "question": "\\section*{Task 1 - 011021}\n\nIn 1970, the Soviet Union was to produce at least 900 billion kWh and in 1980 at least 2700 billion kWh of electricity. For the USA, the Federal Energy Commission assumes 1475 billion kWh and 2230 billion kWh, respectively.\n\nWhen would the USSR have overtaken the USA in electricity production if a constant increase in energy production is assumed?",
      "answer": "1976"
    },
    {
      "question": "7-3. A secret object is a rectangle measuring $200 \\times 300$ meters. Outside the object, there is a guard at each of the four corners. An intruder approached the perimeter of the secret object from the outside, and all the guards ran to him along the shortest paths along the external perimeter (the intruder remained in place). Three of the guards ran a total of 850 meters to reach the intruder. How many meters did the fourth guard run to reach the intruder?",
      "answer": "150"
    },
    {
      "question": "1. Determine the number of pairs $(a, b)$ of natural numbers $(1 \\leqq a<b \\leqq 86)$ for which the product $a b$ is divisible by three.",
      "answer": "2002"
    },
    {
      "question": "1. (8 points) The sum of the digits of the result of the expression $999999999-88888888+7777777-666666+55555-4444+333-22+1$ is $\\qquad$ .",
      "answer": "45"
    },
    {
      "question": "## 233. Math Puzzle 10/84\n\nMatthias rides a 3 km long street at a constant speed of $v=15 \\mathrm{~km} / \\mathrm{h}$.\n\nHow often and how many city buses overtake him, which run at 5-minute intervals with $v=40 \\mathrm{~km} / \\mathrm{h}$?",
      "answer": "2"
    },
    {
      "question": "10. (6 points) Count the number of squares with \"mei\" in the figure: $\\qquad$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "10"
    },
    {
      "question": "10. $A B C D-A_{1} B_{1} C_{1} D_{1}$ is a cube with edge length 1, $D 、 E 、 F 、 G$ are the midpoints of $B A$ 、 $B C 、 B B_{1}$ and $A_{1} D_{1}$, respectively, then the volume of the tetrahedron $G-D E F$ is",
      "answer": "\\frac{1}{12}"
    },
    {
      "question": "21. In the diagram below, $A B C D$ is a convex quadrilateral such that $A C$ intersects $B D$ at the midpoint $E$ of $B D$. The point $H$ is the foot of the perpendicular from $A$ onto $D E$, and $H$ lies in the interior of the segment $D E$. Suppose $\\angle B C A=90^{\\circ}, C E=12 \\mathrm{~cm}, E H=15 \\mathrm{~cm}$, $A H=40 \\mathrm{~cm}$ and $C D=x \\mathrm{~cm}$. Find the value of $x$.\n21. Answer: 40\nLet $M$ be the midpoint of $A H$ and $N$ the midpoint of $B C$. Then $C D$ is parallel to $N E$ and $C D=2 N E$. Since $\\triangle A H E$ is similar to $\\triangle B C E$, we have $\\triangle M H E$ is similar to $\\triangle N C E$. As $M E=\\sqrt{15^{2}+20^{2}}=25$, we thus have $N E=\\frac{E O}{E H} \\cdot M E=\\frac{12 \\times 25}{15}=20$. Therefore $C D=2 N E=40$.",
      "answer": "40"
    },
    {
      "question": "5. In $\\triangle A B C$, $\\angle C=90^{\\circ}, \\angle B=30^{\\circ}$, $A C=1$, $M$ is the midpoint of $A B$. Fold $\\triangle A C M$ along $C M$ so that the distance between points $A$ and $B$ is $\\sqrt{2}$. Then the distance from point $A$ to the plane $B C M$ is $\\qquad$ .",
      "answer": "\\frac{\\sqrt{6}}{3}"
    },
    {
      "question": "1. Circle $k_{1}$ with radius 1 has an external tangency $\\mathrm{s}$ with circle $k_{2}$ with radius 2. Each of the circles $k_{1}, k_{2}$ has an internal tangency with circle $k_{3}$ with radius 3. Calculate the radius of circle $k$, which has external tangency $\\mathrm{s}$ with circles $k_{1}, k_{2}$ and internal tangency $\\mathrm{s}$ with circle $k_{3}$.",
      "answer": "\\frac{6}{7}"
    },
    {
      "question": "8,9 |\n\nFind the cosine of the angle at the base of an isosceles triangle if the point of intersection of its altitudes lies on the inscribed circle of the triangle.\n\n#",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "19. Given a quadrilateral pyramid $P-ABCD$, the base $ABCD$ is a rhombus with side length 2, $\\angle ABC=60^{\\circ}$, $E$ is the midpoint of $AB$, $PA \\perp$ plane $ABCD$, and the sine of the angle between $PC$ and plane $PAD$ is $\\frac{\\sqrt{6}}{4}$.\n(1) Find a point $F$ on edge $PD$ such that $AF \\parallel$ plane $PEC$;\n(2) Find the cosine of the dihedral angle $D-PE-A$.",
      "answer": "\\frac{4\\sqrt{31}}{31}"
    },
    {
      "question": "10. In the packaging workshop, there are 2 types of boxes: one for 20 parts and one for 27 parts. A batch of parts between 500 and 600 pieces has arrived for packaging. When the parts are packed in the first type of box, 13 parts are left unpacked, and when packed in the second type of box, 7 spaces are left unfilled. How many parts were in the batch?",
      "answer": "533"
    },
    {
      "question": "6. In $\\triangle A B C$, $a, b, c$ are the sides opposite to the internal angles $A, B, C$ respectively. If $7 b^{2}+25 c^{2}-25 a^{2}=0$. Try to find the value of $\\sin (2 A-C)$ when $A-C$ reaches its maximum value.",
      "answer": "\\frac{117}{125}"
    },
    {
      "question": "6. Let the result of a single coin toss be denoted as $A$ if it lands heads up; as $B$ if it lands tails up. When tossing the coin several times in a row, we can obtain a sequence composed of $A$ and $B$, for example, tossing six times might result in the sequence $A A B A B B$. In the resulting sequence, we can count the occurrences of patterns like $A B$. For instance, in $A A B A B B$, $A B$ appears twice. When tossing the coin 16 times in a row, among all possible sequences, there are $\\qquad$ sequences that satisfy: $A A$ appears 2 times, $B B$ appears 6 times, $A B$ appears 4 times, and $B A$ appears 4 times.",
      "answer": "840"
    },
    {
      "question": "10.4. A chord is at a distance $h$ from the center of a circle. In each of the two segments of the circle subtended by this chord, a square is inscribed such that a pair of its adjacent vertices lies on the chord, and the other pair of adjacent vertices lies on the corresponding arc of the circle. Find the difference in the lengths of the sides of the squares.",
      "answer": "\\frac{8}{5}"
    },
    {
      "question": "Exercise 3. In the figure below, \\( AB = AF = CD = DE = 18 \\), \\( BC = EF = a \\), and \\( BF = 24 \\). Additionally, the quadrilateral \\( BCEF \\) is a rectangle.\n\n![](https://cdn.mathpix.com/cropped/2024_06_04_d69c62b301e4ba44b0f0g-2.jpg?height=238&width=512&top_left_y=1949&top_left_x=796)\n\nWe assume that the area of the hexagon \\( ABCDEF \\) is equal to the area of a rectangle whose two consecutive sides have lengths \\( a \\) and 36. Find \\( a^2 \\). Justify your answer.\n\nNote. You may use the Pythagorean theorem, which states that if \\( XYZ \\) is a right triangle at \\( X \\), then \\( XY^2 + XZ^2 = YZ^2 \\).",
      "answer": "720"
    },
    {
      "question": "## Task 2\n\nAs the special train stops, a freight train passes by. Peter counts the cars: Right after the locomotive, 5 closed cars pass. Then follow 4 times as many open cars. At the end, 6 tank cars pass.\n\nHow many cars does the freight train have?",
      "answer": "31"
    },
    {
      "question": "## Task 1\n\nCalculate the sum of the numbers 6 and 4. Add the number 30 to this sum.",
      "answer": "40"
    },
    {
      "question": "18. Let $m$ and $n$ be relatively prime positive integers. If $m^{3} n^{5}$ has 209 positive divisors, then how many positive divisors does $m^{5} n^{3}$ have?",
      "answer": "217"
    },
    {
      "question": "13. Given three different prime numbers $p$, $q$, and $r$ that satisfy $p q r = 18 \\underbrace{99 \\ldots 9}_{n \\uparrow 9} 62$, define $f(n)$ as the sum of the digits of the natural number $n$. Then $f(p) + f(q) + f(r) - f(p q r) =$ $\\qquad$",
      "answer": "8"
    },
    {
      "question": "13. When the value of the natural number $n$ is taken as $1,2,3, \\cdots, 2015$, the expression $\\left[\\frac{n}{2}\\right]+\\left[\\frac{n}{3}\\right]+\\left[\\frac{n}{5}\\right]$ has $\\qquad$ different values. (Note: $[x]$ represents the largest natural number not exceeding $x$)",
      "answer": "1479"
    },
    {
      "question": "## Task 4 - 130614\n\nJörg and Claudia are arguing about whether there are more natural numbers from 0 to 1000 whose decimal representation contains (at least) one 5, or whether there are more that do not contain a 5.\n\nDetermine the correct answer to this question!",
      "answer": "271"
    },
    {
      "question": "Task B-1.3. An amount of 18200 kuna needs to be divided among three people so that each subsequent person receives $20 \\%$ more than the previous one. How much money will each of them receive?",
      "answer": "5000"
    },
    {
      "question": "2. To walk 4 km, ride 6 km on a bicycle, and drive 40 km by car, Uncle Vanya needs 2 hours and 12 minutes. If he needs to walk 5 km, ride 8 km on a bicycle, and drive 30 km by car, it will take him 2 hours and 24 minutes. How much time will Uncle Vanya need to walk 8 km, ride 10 km on a bicycle, and drive 160 km by car?",
      "answer": "5.8"
    },
    {
      "question": "Diding M.\nThere are 100 piles, each with 400 stones. In one move, Petya selects two piles, removes one stone from each, and earns as many points as the absolute difference in the number of stones in these two piles. Petya must remove all the stones. What is the maximum total number of points he can earn?",
      "answer": "3920000"
    },
    {
      "question": "8. (3 points) Mr. Wang, who teaches mathematics, is preparing to visit a friend. Before leaving, Mr. Wang calls his friend, whose phone number is 27433619. After Mr. Wang finishes the call, he realizes that this phone number is exactly the product of 4 consecutive prime numbers. What is the sum of these 4 prime numbers? $\\qquad$",
      "answer": "290"
    },
    {
      "question": "12. C5 (NZL) Let $N$ be a positive integer. Two players $A$ and $B$, taking turns, write numbers from the set $\\{1, \\ldots, N\\}$ on a blackboard. $A$ begins the game by writing 1 on his first move. Then, if a player has written $n$ on a certain move, his adversary is allowed to write $n+1$ or $2 n$ (provided the number he writes does not exceed $N$ ). The player who writes $N$ wins. We say that $N$ is of type $A$ or of type $B$ according as $A$ or $B$ has a winning strategy.\n(a) Determine whether $N=2004$ is of type $A$ or of type $B$.\n(b) Find the least $N>2004$ whose type is different from that of 2004 .",
      "answer": "2048"
    },
    {
      "question": "2. (15 points) Let $f(x)=x^{2}-5 x+2020$. Solve the equation $f(3-x)=f(3 x-1)$",
      "answer": "1.51"
    },
    {
      "question": "2. It is known that the numbers $x, y, z$ form an arithmetic progression in the given order with a common difference $\\alpha=\\arccos \\frac{2}{3}$, and the numbers $\\frac{1}{\\sin x}, \\frac{6}{\\sin y}, \\frac{1}{\\sin z}$ also form an arithmetic progression in the given order. Find $\\sin ^{2} y$.",
      "answer": "\\frac{5}{8}"
    },
    {
      "question": "4.031. It is known that the interior angles of a certain convex polygon, the smallest of which is $120^{\\circ}$, form an arithmetic progression with a difference of $5^{\\circ}$. Determine the number of sides of the polygon.",
      "answer": "9"
    },
    {
      "question": "3. How many nine-digit numbers written using nine different digits 1, 2, 3, 4, 5, 6, 7, 8, 9 are there such that each three-digit number formed by three consecutive digits of that number is divisible by 3?",
      "answer": "1296"
    },
    {
      "question": "Zhendarov R.G.\n\nIn a bank, there are 2002 employees. All employees came to the anniversary and were seated at one round table. It is known that the salaries of those sitting next to each other differ by 2 or 3 dollars. What is the greatest possible difference between two salaries of the employees of this bank, given that all salaries are different?\n\n#",
      "answer": "3002"
    },
    {
      "question": "3-4. There is a piece of chain consisting of 150 links, each weighing 1 g. What is the smallest number of links that need to be unbuckled so that from the resulting parts, all weights of 1 g, 2 g, 3 g, \\(\\ldots, 150\\) g can be formed (an unbuckled link also weighs 1 g)?",
      "answer": "4"
    },
    {
      "question": "2 [Arithmetic. Mental arithmetic, etc.]\n\nThere are nuts in the boxes. In the first box, there are 6 kg fewer nuts than in the other two together. And in the second box, there are 10 kg fewer nuts than in the other two together. How many nuts are in the third box?",
      "answer": "8"
    },
    {
      "question": "Find all real numbers $x, y, z$ such that\n\n$$\nx+y+z=3, \\quad x^{2}+y^{2}+z^{2}=3, \\quad x^{3}+y^{3}+z^{3}=3\n$$\n\nDistinction between polynomial and polynomial function Here, we explain why it is necessary to make this distinction, starting by defining a polynomial in another way. Here, $\\mathbb{K}=\\mathbb{Q}, \\mathbb{R}, \\mathbb{C}$ or $\\mathbb{Z} / p \\mathbb{Z}$ equipped with the usual addition and multiplication laws.\n\nDefinition. A polynomial with coefficients in $\\mathbb{K}$ is an infinite sequence of elements of $\\mathbb{K}$ that is zero from a certain rank.\n\nExample. For example, $(0,1,2,3,0,0, \\ldots, 0, \\ldots)$ is a polynomial, as is $(0,0, \\ldots, 0, \\ldots)$. However, $(1,1, \\ldots, 1, \\ldots)$ is not one.\n\nDefinition. Let $P=\\left(u_{n}\\right)_{n}$ and $Q=\\left(v_{n}\\right)_{n}$ be two polynomials. We define the polynomial $P+Q$ by the sequence $w_{n}=u_{n}+v_{n}$ (which is indeed zero from a certain rank) and the polynomial $P \\times Q$ by the sequence $\\left(z_{n}\\right)$, where $z_{n}=\\sum_{i+j=n} u_{n} v_{n}$ (verify that $\\left(z_{n}\\right)$ is zero from a certain rank). We identify the elements of $\\mathbb{K}$ with the constant polynomials via the application that associates to an element $\\lambda \\in \\mathbb{K}$ the polynomial $(\\lambda, 0,0, \\ldots, 0, \\ldots)$. Note that this is consistent with the intuitive notion of multiplying a polynomial by an element of $\\mathbb{K}$: if $\\left(u_{n}\\right)$ is a polynomial and $\\lambda \\in \\mathbb{K}$, then the polynomial $\\lambda \\times\\left(u_{n}\\right)$ is the polynomial $\\left(\\lambda u_{n}\\right)$.\n\nWe now introduce the indeterminate $X$.\n\nDefinition. Let us denote by $X$ the polynomial $(0,1,0,0, \\ldots)$.\n\nProposition. Every polynomial $P$ can be expressed in the form $P=\\sum_{i=0}^{n} a_{n} X^{n}$. We denote $P$ or $P(X)$ indifferently to remind that we denote $X$ the indeterminate (we could very well denote it $Y!$ ).\n\nProof. If $P=\\left(a_{0}, a_{1}, a_{2}, \\ldots\\right)$, let $N$ be an integer such that $i \\geq N$ implies $a_{i}=0$. Then $P(X)=a_{0}+a_{1} X+\\cdots+a_{N} x^{N}$. This is an immediate consequence of the definition of $X$ and the multiplication between polynomials.\n\nHere is now the link between polynomial and associated polynomial function. Recall that, for now, a polynomial is just a sequence of numbers that is zero from a certain rank and is not seen as an application.\n\nProposition. Let $P(X)=a_{0}+a_{1} X+\\cdots+a_{n} X^{n} \\in \\mathbb{K}[X]$ be a polynomial. We denote by $\\widetilde{P}$ the application defined by $\\widetilde{P}(x)=a_{0}+a_{1} x+\\cdots+a_{n} x^{n}$ for $x \\in \\mathbb{K}$, which we call the polynomial function associated with $P$. The application $P \\mapsto \\widetilde{P}$ is injective if $\\mathbb{K}$ is infinite. If $\\mathbb{K}$ is finite, this application is not necessarily injective.\n\nProof. First, consider the case where $\\mathbb{K}$ is infinite. Let $P, Q \\in \\mathbb{K}[X]$ such that $\\widetilde{P}=\\widetilde{Q}$. Write $P(X)=\\sum_{i} a_{i} X^{i}$ and $Q(X)=\\sum_{i} b_{i} X^{i}$. Then the polynomial $P(x)-Q(x)$, in the sense of the previous sections, has an infinite number of roots, so it is zero. Therefore, $a_{i}=b_{i}$ for all $i$.\n\nHowever, in the case where $\\mathbb{K}$ is finite, the previous reasoning does not apply. Moreover, let us exhibit a counterexample. Consider $\\mathbb{K}=\\mathbb{Z} / p \\mathbb{Z}$ and $P(X)=X^{p}-X$. According to Fermat's little theorem, for all $x \\in \\mathbb{K}$, we have $P(x)=0$. Thus, $P$ is not the zero polynomial, but the two associated polynomial functions are the same.\n\nIn other words, when $\\mathbb{K}$ is infinite (for example, $\\mathbb{K}=\\mathbb{Q}, \\mathbb{R}, \\mathbb{C}$, which explains why we did not lose generality in the first sections), we can talk without distinction about a polynomial or the associated polynomial function. However, in other cases, one must be very careful!\n\nElements of the answer to the exercises",
      "answer": "1"
    },
    {
      "question": "6. (10 points) Car A and Car B depart from City A to City B, which is 300 kilometers away. It is known that Car A departs 1 hour later than Car B, but arrives 1 hour earlier at City B. Then, Car A catches up with Car B at a distance of $\\qquad$ kilometers from City B.",
      "answer": "150"
    },
    {
      "question": "Problem 8-5. 8 8-5. A divisor of a natural number is called proper if it is different from 1 and the number itself. Find all natural numbers for which the difference between the sum of the two largest proper divisors and the sum of the two smallest proper divisors is a prime number.",
      "answer": "12"
    },
    {
      "question": "2. Given two linear functions $f(x)$ and $g(x)$ such that the graphs $y=f(x)$ and $y=g(x)$ are parallel lines, not parallel to the coordinate axes. Find the minimum value of the function $(g(x))^{2}+$ $2 f(x)$, if the minimum value of the function $(f(x))^{2}+2 g(x)$ is 5.",
      "answer": "-7"
    },
    {
      "question": "14. On a rectangular chessboard filled with chess pieces, two players, A and B, take turns to pick up the pieces, following these rules: in a certain row or column, take any continuously placed pieces (i.e., cannot pick across empty spaces), not allowed to skip a turn, and cannot pick from multiple rows (columns) at once. The game ends when all pieces on the board are taken. The player who takes the last piece wins.\nFaced with the chessboard as shown in the figure, the first player has a winning strategy. The first player\n\nshould take\n\n(all correct solutions), to ensure victory.",
      "answer": "4"
    },
    {
      "question": "Example 13 Find the maximum value of the function $f(x)=\\sqrt{10 x-9-x^{2}}+\\sqrt{68 x-256-x^{2}}$.",
      "answer": "3\\sqrt{35}"
    },
    {
      "question": "5. How many days have passed from March 19, 1990 to March 23, 1996 inclusive",
      "answer": "2196"
    },
    {
      "question": "7.3. How many five-digit natural numbers exist, each of which has adjacent digits with different parity",
      "answer": "5625"
    },
    {
      "question": "1. Let the right vertex of the ellipse $\\frac{x^{2}}{a^{2}}+\\frac{y^{2}}{b^{2}}=1(a>b>0)$ be $A$, the upper vertex be $B$, and the left focus be $F$. If $\\angle A B F=90^{\\circ}$, then the eccentricity of the ellipse is $\\qquad$.",
      "answer": "\\frac{\\sqrt{5}-1}{2}"
    },
    {
      "question": "6、Let $x$ be an acute angle, then the maximum value of the function $y=\\sin x \\sin 2 x$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly. \n\n6、Let $x$ be an acute angle, then the maximum value of the function $y=\\sin x \\sin 2 x$ is $\\qquad$",
      "answer": "\\frac{4\\sqrt{3}}{9}"
    },
    {
      "question": "Determine the smallest positive integer $n$ for which there exists a polynomial\n\n$$\nP(X)=a_{2 n} X^{2 n}+a_{2 n-1} X^{2 n-1}+\\ldots+a_{1} X+a_{0}\n$$\n\nwith real coefficients that satisfies the following two properties:\n\n- For $i=0,1, \\ldots, 2 n$, we have $2014 \\leq a_{i} \\leq 2015$.\n- There exists a real number $\\xi$ such that $P(\\xi)=0$.",
      "answer": "2014"
    },
    {
      "question": "12. Given a moving point $P(x, y)$ satisfies the quadratic equation $10 x-2 x y-2 y+1=0$, then the eccentricity of this quadratic curve is $\\qquad$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "2. [5] Evaluate the sum\n$$\n1-2+3-4+\\cdots+2007-2008 .\n$$",
      "answer": "-1004"
    },
    {
      "question": "## Task 3 - 020723\n\nEmil tells: \"My brother Heinz is only half as old as I am. If you multiply the number of his years by itself, you get my father's age. My mother is 5 years younger than my father. All together, we are 85 years old.\"\n\nHow old is Emil? Describe how you found the solution!",
      "answer": "12"
    },
    {
      "question": "6. Given $\\overrightarrow{O A}$ and $\\overrightarrow{O B}$ are non-zero and non-collinear vectors. Let $\\overrightarrow{O C}=\\frac{1}{1+r} \\overrightarrow{O A}+\\frac{r}{1+r} \\overrightarrow{O B}$. Define the set of points\n$$\nM=\\left\\{K \\left\\lvert\\, \\frac{\\mid \\overrightarrow{K A} \\cdot \\overrightarrow{K C}}{|\\overrightarrow{K A}|}=\\frac{\\overrightarrow{K B} \\cdot \\overrightarrow{K C}}{|\\overrightarrow{K B}|}\\right.\\right\\} .\n$$\n\nWhen $K_{1} 、 K_{2} \\in M$, if for any $r \\geqslant 2$, the inequality $\\left|\\overrightarrow{K_{1} K_{2}}\\right| \\leqslant c|\\overrightarrow{A B}|$ always holds, then the minimum value of the real number $c$ is $\\qquad$",
      "answer": "\\frac{4}{3}"
    },
    {
      "question": "7.2. Three cyclists leave the city. The speed of the first one is 12 km/h, the second one is 16 km/h, and the third one is 24 km/h. It is known that the first one was cycling exactly when the second and third were resting (standing still), and that at no time did two cyclists cycle simultaneously. It is also known that in 3 hours, all three covered the same distance. Find this distance.",
      "answer": "16"
    },
    {
      "question": "1. If the 5th term in the expansion of $\\left(x \\sqrt{x}-\\frac{1}{x}\\right)^{6}$ is $\\frac{15}{2}$, then $\\lim _{n \\rightarrow \\infty}\\left(x^{-1}+x^{-2}+\\cdots+x^{-n}\\right)=$",
      "answer": "1"
    },
    {
      "question": "1. The picture shows several circles connected by segments. Nastya chooses a natural number \\( n \\) and places different natural numbers in the circles so that for all these numbers the following property holds:\n\nif numbers \\( a \\) and \\( b \\) are not connected by a segment, then the difference \\( a - b \\) must be coprime with \\( n \\), and if they are connected, then \\( a - b \\) and \\( n \\) must have a common natural divisor greater than 1. For what smallest \\( n \\) does such an arrangement exist?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6bce8335c6b594326f89g-15.jpg?height=271&width=263&top_left_y=601&top_left_x=908)",
      "answer": "385"
    },
    {
      "question": "13. (10 points) A class of 30 students goes to the movies, and their student IDs are $1, 2, \\cdots, 30$; the movie tickets they hold are exactly the 1st, 2nd, ..., 30th seats in a row. Now, the movie tickets are to be distributed to these students according to the following requirements: for any two students, A and B, if A's student ID can be divided by B's student ID, then A's movie ticket number can also be divided by B's movie ticket number. How many different ways are there to distribute the movie tickets? $\\qquad$",
      "answer": "48"
    },
    {
      "question": "5. A product that initially contained $98\\%$ water dried over some time and began to contain $95\\%$ water. By what factor did it shrink (i.e., reduce its weight)?",
      "answer": "2.5"
    },
    {
      "question": "4. Dad, Mom, the guest, and I, four people, are sitting around a round table having tea. If we only consider the person on each person's left, how many different seating arrangements are there?",
      "answer": "6"
    },
    {
      "question": "# 3. CONDITION\n\nGiven a cube $A B C D A_{1} B_{1} C_{1} D_{1}$ with edge 1. A line $l$ passes through point $E$, the midpoint of edge $C_{1} D_{1}$, and intersects lines $A D_{1}$ and $A_{1} B$. Find the distance from point $E$ to the point of intersection of line $l$ with line $A_{1} B$.",
      "answer": "1.5"
    },
    {
      "question": "## Task B-2.3.\n\nOn the side $\\overline{A B}$ of triangle $A B C$, a point $E$ is given such that $|A E|=1$ and $|E B|=2$. Let point $D$ be on the side $\\overline{A C}$ such that the segment $\\overline{D E}$ is parallel to the segment $\\overline{B C}$, and point $F$ be on the side $\\overline{B C}$ such that the segment $\\overline{E F}$ is parallel to the segment $\\overline{A C}$. Calculate the ratio of the areas of quadrilateral $C D E F$ and triangle $A B C$.",
      "answer": "\\frac{4}{9}"
    },
    {
      "question": "G9.4 Find the sum $d$ where\n$$\nd=\\left(\\frac{1}{2}+\\frac{1}{3}+\\frac{1}{4}+\\cdots+\\frac{1}{60}\\right)+\\left(\\frac{2}{3}+\\frac{2}{4}+\\frac{2}{5}+\\cdots+\\frac{2}{60}\\right)+\\left(\\frac{3}{4}+\\frac{3}{5}+\\cdots+\\frac{3}{60}\\right)+\\cdots+\\left(\\frac{58}{59}+\\frac{58}{60}\\right)+\\frac{59}{60}\n$$",
      "answer": "885"
    },
    {
      "question": "8. A figure composed of some vertices and edges is called a graph. Coloring the vertices of a graph with different colors, requiring that two vertices connected by the same edge be colored differently, is called vertex coloring of a graph. The usual problem studied in graph vertex coloring is the minimum number of colors needed to complete the coloring, which is called the chromatic number of the graph. The graph below is called the Petersen graph, and the chromatic number of the Petersen graph is _",
      "answer": "3"
    },
    {
      "question": "G9.4 If the sum of the interior angles of an $n$-sided convex polygon is $x^{\\circ}$, find $x$.",
      "answer": "3240"
    },
    {
      "question": "6. (5 points) There are four natural numbers, their sum is 243. If the first number is increased by 8, the second number is decreased by 8, the third number is multiplied by 8, and the fourth number is divided by 8, then the four resulting numbers are equal. Therefore, the product of the largest and smallest of the original four numbers is . $\\qquad$",
      "answer": "576"
    },
    {
      "question": "Three mountains lie on the same meridian. The spherical distance between the first two is $a^{0}$, and between the second two is $b^{0}$; their heights are $h_{1}, h_{2}, h_{3}$ respectively. Does the second mountain obscure the third mountain when viewed from the first mountain?\n\nNumerical values: $a=1^{\\circ}, b_{0}=1^{\\circ} 30^{\\prime}, h_{1}=2500 \\mathrm{~m}, h_{2}=3000 \\mathrm{~m}, h_{3}=8800 \\mathrm{~m}$ and the radius of the Earth is $6366 \\mathrm{~km}$.",
      "answer": "1500"
    },
    {
      "question": "4. Let $[x]$ denote the greatest integer not exceeding the real number $x$. Calculate: $\\sum_{k=0}^{2019}\\left[\\frac{4^{k}}{5}\\right]=$ $\\qquad$ .",
      "answer": "\\frac{4^{2020}-1}{15}-1010"
    },
    {
      "question": "Consider 9 points in the plane with no alignment. What is the minimum value of $n$ such that if we color $n$ edges connecting two of the points in red or blue, we are sure to have a monochromatic triangle regardless of the coloring?\n\n(IMO 1992)",
      "answer": "33"
    },
    {
      "question": "3. (7 points) There are 8 blue, 7 red, and 12 white light bulbs. In how many ways can they be arranged into a garland (using all the bulbs) so that no two white bulbs are adjacent?",
      "answer": "11711700"
    },
    {
      "question": "The fifth question: If an infinite sequence of positive real numbers $\\left\\{x_{n}\\right\\}$ satisfies: $x_{0}=1, x_{i} \\geq x_{i+1}(i \\in N)$, then the sequence is called a “good sequence”. Find the smallest constant $c$ such that there exists a “good sequence” $\\left\\{x_{n}\\right\\}$ satisfying: $\\sum_{i=0}^{n} \\frac{x_{i}^{2}}{x_{i+1}} \\leq c$ for any positive integer $n$.",
      "answer": "4"
    },
    {
      "question": "5. Drop perpendiculars $D D_{1}, D D_{2}, D D_{3}$ from point $D$ to the planes $S B C$, $S A C$, and $S A B$ respectively. Let $D D_{1}=x, D D_{2}=y, D D_{3}=z$. According to the condition, we form the system of equations\n\n$$\n\\left\\{\\begin{array}{l}\ny^{2}+z^{2}=5 \\\\\nx^{2}+z^{2}=13 \\\\\nx^{2}+y^{2}=10\n\\end{array}\\right.\n$$\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0013fadf99b0b72b3281g-9.jpg?height=526&width=630&top_left_y=95&top_left_x=1848)\n\nFig. 6:\n\nFrom here, we find $x=3, y=1, z=2$. Let the lengths of the edges $S A, S B$, and $S C$ be $a, b$, and $c$ respectively. Since points $A, B, C$, and $D$ lie in the same plane, the relation $\\frac{3}{a}+\\frac{1}{b}+\\frac{2}{c}=1$ holds.\n\nUsing the inequality between the arithmetic mean and the geometric mean for three variables, we get:\n\n$$\n\\begin{aligned}\n& \\frac{\\frac{3}{a}+\\frac{1}{b}+\\frac{2}{c}}{3} \\geqslant \\sqrt[3]{\\frac{3}{a} \\cdot \\frac{1}{b} \\cdot \\frac{2}{c}}=\\sqrt[3]{\\frac{6}{a b c}} \\Longleftrightarrow \\\\\n& \\Longleftrightarrow 1=\\left(\\frac{3}{a}+\\frac{1}{b}+\\frac{2}{c}\\right)^{3} \\geqslant \\frac{6 \\cdot 27}{a b c} \\Longleftrightarrow a b c \\geqslant 6 \\cdot 27\n\\end{aligned}\n$$\n\nwith equality holding when $\\frac{3}{a}=\\frac{1}{b}=\\frac{2}{c}=\\frac{1}{3}$. The volume of the pyramid $V=\\frac{a b c}{6}$, so $V \\geqslant 27$. Equality holds when $a=9, b=3, c=6$.",
      "answer": "27"
    },
    {
      "question": "8. In a right trapezoid, the upper base is shorter than the lower base. If the volume of the solid of revolution obtained by rotating it around its lower base is $80 \\pi$,\n\nthe volume of the solid of revolution obtained by rotating it around its upper base is $112 \\pi$, and the volume of the solid of revolution obtained by rotating it around its right leg is $156 \\pi$, then the perimeter of the right trapezoid is",
      "answer": "16+2\\sqrt{13}"
    },
    {
      "question": "433. Gifted Paintings. A wealthy collector had 10 valuable paintings. He wanted to make a gift to a museum, but the collector couldn't figure out how many gift options he had: after all, he could give any one painting, any two, any three paintings, and so on, he could even give all ten paintings.\n\nThe reader might think that answering this question would require a long and tedious calculation; however, I will provide a small rule that allows one to answer in all similar cases without any difficulties and thankless work.",
      "answer": "1023"
    },
    {
      "question": "2. Simplify the expression $x^{(x+1)^{2}} \\cdot\\left(x^{x-1}\\right)^{x+1}: \\frac{x^{x^{2}}}{x^{-2 x}}$ and then calculate the value of the expression for $x=\\sqrt{2}$.",
      "answer": "2"
    },
    {
      "question": "3. A $4 \\times 4 \\times h$ rectangular prism can hold 8 small spheres with a radius of 1 and 1 large sphere with a radius of 2. Then the minimum value of $h$ is $\\qquad$ .",
      "answer": "2+2\\sqrt{7}"
    },
    {
      "question": "3. In triangle $ABC (\\overline{BC} > \\overline{AC})$, the angles $\\alpha = \\measuredangle BAC$ and $\\beta = \\measuredangle ABC$ differ by $30^{\\circ}$. If point $D$ is chosen on side $BC$ such that $\\overline{AC} = \\overline{CD}$, calculate $\\measuredangle DAB$.",
      "answer": "15"
    },
    {
      "question": "10. (10 points) In a math test at Sheep Village Elementary School's fourth grade, the test consists of 15 questions. If Xiaoxixi, Xiaobeibe, Xiaomeimei, and Xiaolan all answered 11, 12, 13, and 14 questions correctly, respectively, then the minimum number of questions they all answered correctly is $\\qquad$ questions.",
      "answer": "5"
    },
    {
      "question": "4. Given three non-zero real numbers $a, b, c$ such that the equations: $a x^{2}+b x+c=0, b x^{2}+c x+a=0, c x^{2}+a x+b=0$ each have two roots. How many of the roots of these equations can be negative?",
      "answer": "2"
    },
    {
      "question": "19. (6 points) Xiao Ming puts 127 Go stones into several bags. No matter how many stones a child wants (not exceeding 127 stones), Xiao Ming can satisfy the request by taking out a few bags. How many bags does Xiao Ming need to prepare at least?",
      "answer": "7"
    },
    {
      "question": "Problem 4.7. Denis threw darts at four identical dartboards: he threw exactly three darts at each board, where they landed is shown in the figure. On the first board, he scored 30 points, on the second - 38 points, on the third - 41 points. How many points did he score on the fourth board? (For hitting each specific zone - ring or central field - a certain number of points is awarded.)\n![](https://cdn.mathpix.com/cropped/2024_05_06_c49ecf27446af41a8bcdg-06.jpg?height=260&width=1136&top_left_y=960&top_left_x=156)",
      "answer": "34"
    },
    {
      "question": "4. Xiaoming formed a sequence using the four digits $2, 0, 1, 6$ (continuously adding these four digits in this order to the end of the number): $2,20,201,2016,20162,201620,2016201,20162016,201620162, \\ldots$. In this sequence, the number of prime numbers is $\\qquad$.",
      "answer": "1"
    },
    {
      "question": "1. How many positive integers less than 2019 are divisible by either 18 or 21 , but not both?",
      "answer": "176"
    },
    {
      "question": "Example 9 (1) When $0 \\leqslant x \\leqslant 1$, find the range of the function\n$f(x)=(\\sqrt{1+x}+\\sqrt{1+x}+2) \\cdot\\left(\\sqrt{1-x^{2}}+1\\right)$.\n(2) Prove that when $0 \\leqslant x \\leqslant 1$, there exists a positive number $\\beta$ such that the inequality $\\sqrt{1+x}+\\sqrt{1-x} \\leqslant 2-\\frac{x^{a}}{\\beta}$ holds for the smallest positive number $\\alpha=2$, and find the smallest positive number $\\beta$ at this time.",
      "answer": "4"
    },
    {
      "question": "15.1.1 * Let $a_{n}=6^{n}+8^{n}$, determine the remainder when $a_{83}$ is divided by 49.",
      "answer": "35"
    },
    {
      "question": "Problem 11.1. Masha lives in apartment No. 290, which is located in the 4th entrance of a 17-story building. On which floor does Masha live? (The number of apartments is the same in all entrances of the building on all 17 floors; apartment numbers start from 1.$)$",
      "answer": "7"
    },
    {
      "question": "7. Given $1 \\leqslant x^{2}+y^{2} \\leqslant 4$, then the sum of the maximum and minimum values of $x^{2}-x y+y^{2}$ is\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "6\\frac{1}{2}"
    },
    {
      "question": "## Task B-1.2.\n\nTwo pipes fill a pool in 4 hours. One of these pipes fills the pool by itself in 5 hours and 15 minutes. How long does it take for the other pipe to fill the pool by itself?",
      "answer": "16"
    },
    {
      "question": "73. Using $0, 2, 6, 8$, we can form $\\qquad$ number of three-digit numbers without repeated digits.",
      "answer": "18"
    },
    {
      "question": "In a certain kingdom, there were 32 knights. Some of them were vassals of others (a vassal could have only one suzerain, and a suzerain was always richer than his vassal). A knight who had at least four vassals bore the title of baron. What is the maximum number of barons that could be under these conditions?\n\n(The kingdom had a law: \"a vassal of my vassal is not my vassal\".)\n\n#",
      "answer": "7"
    },
    {
      "question": "3. Given a positive real number $x$ satisfies\n$$\nx^{3}+x^{-3}+x^{6}+x^{-6}=2754 \\text {. }\n$$\n\nThen $x+\\frac{1}{x}=$ $\\qquad$ .",
      "answer": "4"
    },
    {
      "question": "423. Can the sum of a finite number of terms of a geometric progression be zero?",
      "answer": "-1"
    },
    {
      "question": "1. (7 points) In an $8 \\times 8$ frame with a width of 2 cells (see figure), there are a total of 48 cells. How many cells are in a $254 \\times 254$ frame with a width of 2 cells?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_af60849ded79da38d4cfg-1.jpg?height=288&width=280&top_left_y=661&top_left_x=891)",
      "answer": "2016"
    },
    {
      "question": "Example 3 In the Cartesian coordinate system $x O y$, given two points $M(-1,2)$ and $N(1,4)$, point $P$ moves on the $x$-axis. When $\\angle M P N$ takes the maximum value, find the x-coordinate of point $P$.\n(2004 National High School Mathematics Competition Problem)",
      "answer": "1"
    },
    {
      "question": "A rectangle with sides of $15 \\mathrm{~cm}$ and $20 \\mathrm{~cm}$ is rotated about one of its diagonals. What is the volume of the resulting solid of revolution?",
      "answer": "5238.85"
    },
    {
      "question": "9. Find the maximum value of the function $f(x)=9 \\sin x+12 \\cos x$.",
      "answer": "15"
    },
    {
      "question": "54. As shown in the figure, a square is divided into two identical L-shapes. If the two L-shapes can form a rectangle with a perimeter of 26, find the area of this square.",
      "answer": "36"
    },
    {
      "question": "6. Given three points $A, B, C$ on a plane satisfying $|\\overrightarrow{A B}|=3,|\\overrightarrow{B C}|=4,|\\overrightarrow{C A}|=5$, then $\\overrightarrow{A B} \\cdot \\overrightarrow{B C}+\\overrightarrow{B C} \\cdot \\overrightarrow{C A}+\\overrightarrow{C A} \\cdot \\overrightarrow{A B}$ is equal to",
      "answer": "-25"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$$\n\\lim _{x \\rightarrow 0} \\frac{\\sqrt{1+x \\sin x}-1}{e^{x^{2}}-1}\n$$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "6. The reading on a factory's electricity meter is 52222 kilowatts. After several days, the meter reading (a five-digit number) again shows four identical digits. How much electricity, in kilowatts, did the factory use at least in these days?\n\n untranslated portion: $\\qquad$",
      "answer": "333"
    },
    {
      "question": "7. There are 5 power poles, 2 of which will leak electricity. There are 5 birds each randomly choosing one of the poles to rest (if a bird touches a leaking pole, it will be stunned by the electric shock and fall to the ground). The probability that only 2 of the poles have birds on them is $\\qquad$",
      "answer": "\\frac{342}{625}"
    },
    {
      "question": "Let's construct a triangle, given one side and two heights (2 cases).\n\n\n\n\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly. \n\nNote: The provided text is already in English, so no translation is needed for the second part of the instruction.",
      "answer": "30"
    },
    {
      "question": "145. Find $\\lim _{x \\rightarrow 0} \\frac{1-\\cos 8 x}{2 x^{2}}$.",
      "answer": "16"
    },
    {
      "question": "[ Properties and characteristics of an isosceles triangle. ]\n\nIn trapezoid $ABCD$ ($AD$ is the larger base), diagonal $AC$ is perpendicular to side $CD$ and bisects angle $BAD$. It is known that $\\angle CDA = 60^\\circ$, and the perimeter of the trapezoid is 2. Find $AD$.",
      "answer": "\\frac{4}{5}"
    },
    {
      "question": "Problem 4.8. The figure shows a road map between the houses of five children. The shortest distance by road from Asya to Galia is 12 km, from Galia to Borya - 10 km, from Asya to Borya 8 km, from Dasha to Galia - 15 km, from Vasya to Galia - 17 km. How many kilometers is the shortest distance by road from Dasha to Vasya?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f1bf0225c3b69484645g-07.jpg?height=531&width=872&top_left_y=1073&top_left_x=291)",
      "answer": "18"
    },
    {
      "question": "63. A large batch of car tires contains $1.5\\%$ defects. What should be the volume of a random sample so that the probability of finding at least one defective car tire in it would be more than $0.92?$",
      "answer": "168"
    },
    {
      "question": "Find all natural numbers $x$, satisfying the conditions: the product of the digits of the number $x$ is equal to $44x - 86868$, and the sum of the digits is a cube of a natural number.",
      "answer": "1989"
    },
    {
      "question": "11.145. The lateral faces of a triangular pyramid are mutually perpendicular, and their areas are equal to $a^{2}, b^{2}$ and $c^{2}$. Determine the volume of the pyramid.",
      "answer": "\\frac{1}{3}\\sqrt{2}"
    },
    {
      "question": "Ex. 72. The height of a right triangle, dropped to the hypotenuse, is $h$. What is the minimum length that the median, which bisects the larger leg, can have?",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "29. Odd integers starting from 1 are grouped as follows: (1), $(3,5),(7,9,11),(13,15,17,19), \\ldots$. where the $n$-th group consists of $n$ odd integers. How many odd integers are in the same group which 2007 belongs to?",
      "answer": "45"
    },
    {
      "question": "Example 2. Five fair coins are tossed. Find the probability that: exactly 2 heads appeared; more than one head appeared.",
      "answer": "0.8125"
    },
    {
      "question": "4. Determine the minimum value of the expression $E(a, b)=a+b+\\frac{1}{a b}$, where $a, b>0$ and $a+b \\leq 1$.\n\nRomeo Ilie",
      "answer": "5"
    },
    {
      "question": "5. In a circle, a diameter $A B$ and a chord $C D$ parallel to it were drawn such that the distance between them is half the radius of this circle (see figure). Find the angle $CAB$.",
      "answer": "75"
    },
    {
      "question": "504. Find the area of a right triangle, both legs of which have a length of 1.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "How many positive integer solutions does the following equation have?\n\n$$\n\\left[\\frac{x}{10}\\right]=\\left[\\frac{x}{11}\\right]+1\n$$",
      "answer": "110"
    },
    {
      "question": "4. In $\\triangle A B C$, the sides opposite to $\\angle A$, $\\angle B$, and $\\angle C$ are denoted as $a$, $b$, and $c$ respectively. If $b^{2}=a^{2}+c^{2}-a c$, and $c-a$ equals the height $h$ from vertex $A$ to side $A C$, then $\\sin \\frac{C-A}{2}=$ $\\qquad$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "2. $y=\\arcsin ^{2} x-2 \\arcsin x-2, y_{\\max }=$ $\\qquad$",
      "answer": "\\frac{\\pi^{2}}{4}+\\pi-2"
    },
    {
      "question": "1. Find the probability that the townspeople win if there are initially two townspeople and one goon.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "G4.1 In Figure $1, A B C D$ is a parallelogram, $B E \\perp C D, B F \\perp A D$, $C E=2 \\mathrm{~cm}, D F=1 \\mathrm{~cm}$ and $\\angle E B F=60^{\\circ}$.\nIf the area of the parallelogram $A B C D$ is $R \\mathrm{~cm}^{2}$, find the value of $R$.",
      "answer": "12\\sqrt{3}"
    },
    {
      "question": "11. We call a four-digit number a \"Mid-ring number\" if it has the following characteristics: (1) all four digits are different; (2) the thousand's digit is neither the largest nor the smallest of these four digits; (3) the unit's digit is not the smallest of these four digits. There are $\\qquad$ such \"Mid-ring numbers\".",
      "answer": "1680"
    },
    {
      "question": "1. Let $ABCD$ be an isosceles trapezoid ($AD \\| BC$) with an angle $\\frac{\\pi}{3}$ at the longer base $AD$ and diagonal $\\overline{AC}=\\sqrt{3}$. A point $M$ is at a distance of 1 from $A$ and at a distance of 3 from $D$. Find $\\overline{MC}$.",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "7. [6] Rachel has two indistinguishable tokens, and places them on the first and second square of a $1 \\times 6$ grid of squares, She can move the pieces in two ways:\n- If a token has free square in front of it, then she can move this token one square to the right\n- If the square immediately to the right of a token is occupied by the other token, then she can \"leapfrog\" the first token; she moves the first token two squares to the right, over the other token, so that it is on the square immediately to the right of the other token.\nIf a token reaches the 6th square, then it cannot move forward any more, and Rachel must move the other one until it reaches the 5 th square. How many different sequences of moves for the tokens can Rachel make so that the two tokens end up on the 5 th square and the 6 th square?",
      "answer": "42"
    },
    {
      "question": "In a right-angled triangle $A B C$, from vertices $B$ and $C$, one point each starts moving simultaneously towards point $A$ along the hypotenuse $B A=c=85$ m and the leg $C A=b=75$ m, respectively. The points move at speeds of $8.5 \\mathrm{~m} / \\mathrm{sec}$ and $5 \\mathrm{~m} / \\mathrm{sec}$. At what time will the distance between the two points be $26 \\mathrm{~m}$. (Without using trigonometry!)",
      "answer": "4"
    },
    {
      "question": "A square-based pyramid has every edge 1 unit long. What is the radius of the sphere inscribed in the pyramid?",
      "answer": "\\frac{\\sqrt{6}-\\sqrt{2}}{4}"
    },
    {
      "question": "We roll a die until a six comes up. What is the probability that we do not roll a five in the meantime?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Lisa considers the number\n\n$$\nx=\\frac{1}{1^{1}}+\\frac{1}{2^{2}}+\\cdots+\\frac{1}{100^{100}} .\n$$\n\nLisa wants to know what $x$ is when rounded to the nearest integer. Help her determine its value.",
      "answer": "1"
    },
    {
      "question": "We throw five dice of different colors. In how many ways can the sum of the throws be 11?",
      "answer": "205"
    },
    {
      "question": "Task B-3.7. The length of the side of an equilateral triangle is $6 \\mathrm{~cm}$. A line parallel to one of the sides has been drawn through its centroid. Calculate the volume of a right prism whose base is the resulting trapezoid, and the length of the prism's height is equal to the height of the trapezoid. Determine the cosine of the angle that the spatial diagonal of the prism forms with the base plane.",
      "answer": "\\sqrt{\\frac{28}{31}}"
    },
    {
      "question": "Example 1. Calculate the definite integral $\\int_{1}^{4} x^{2} d x$.",
      "answer": "21"
    },
    {
      "question": "Second Part (8 points per question, total 40 points)\n6. A box contains 10 black balls, 9 white balls, and 8 red balls. If you close your eyes and draw balls from the box, how many balls must you draw to ensure that you have at least one red ball and one white ball? $\\qquad$",
      "answer": "20"
    },
    {
      "question": "Problem 11.4. On a horizontal floor, there are three volleyball balls with a radius of 18, each touching the other two. Above them, a tennis ball with a radius of 6 is placed, touching all three volleyball balls. Find the distance from the top point of the tennis ball to the floor. (All balls are spherical.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_9703912434d201fc4f33g-03.jpg?height=385&width=521&top_left_y=84&top_left_x=466)",
      "answer": "36"
    },
    {
      "question": "11. There are 11 students who have signed up for the volunteer tour guide activity at the museum. The activity runs from 9 AM to 5 PM, with a public mini-lecture every hour. Each session requires 1 student to provide tour guide services to visitors. To avoid overworking the students, the museum will not schedule the same person for two consecutive sessions, and the first and last sessions must be handled by two different students. How many scheduling arrangements are possible for the museum?",
      "answer": "100000010"
    },
    {
      "question": "8. (3 points) In $17 \\square 17 \\square 17 \\square 17$, fill in the four $\\square$ with the operation symbols “+”, “-”, “$\\times$”, “$\\div$” each once. The maximum value of the resulting expression is $\\qquad$ .",
      "answer": "305"
    },
    {
      "question": "Let's determine the greatest common divisor of the numbers $A$ and $C$, as well as $B$ and $C$.\n\n$$\n\\begin{aligned}\n& A=177^{5}+30621 \\cdot 173^{3}-173^{5} \\\\\n& B=173^{5}+30621 \\cdot 177^{3}-177^{5} \\\\\n& C=173^{4}+30621^{2}+177^{4}\n\\end{aligned}\n$$",
      "answer": "30637"
    },
    {
      "question": "7. A survey was conducted involving 420 families. It was found that $95\\%$ of the families own a TV, and $80\\%$ of the families own a computer. A family is randomly selected. If it is known that each surveyed family has at least one of the mentioned devices, what is the probability that the selected family owns a computer but not a TV? Express the probability as a percentage.\n\nNo calculators or reference materials are allowed.",
      "answer": "5"
    },
    {
      "question": "A box of chocolates in the shape of a cuboid was full of chocolates arranged in rows and columns. Misha ate some of them and rearranged the remaining ones so that they filled (without gaps) up to one place three whole rows. The remaining chocolates from the next incomplete row were eaten. Then the remaining chocolates were rearranged and filled five columns without one place. The chocolates in the incomplete column were eaten again. Thus, only a third of the original number of chocolates remained in the box. Find:\n\na) how many chocolates the entire box had,\n\nb) how many chocolates Misha ate before he did the first rearrangement.\n\n(Bednářrová)",
      "answer": "25"
    },
    {
      "question": "21. Each of the integers $1,2,3, \\ldots, 9$ is assigned to each vertex of a regular 9 -sided polygon (that is, every vertex receives exactly one integer from $\\{1,2, \\ldots, 9\\}$, and two vertices receive different integers) so that the sum of the integers assigned to any three consecutive vertices does not exceed some positive integer $n$. What is the least possible value of $n$ for which this assignment can be done?",
      "answer": "16"
    },
    {
      "question": "5. During the shooting practice, each soldier fired 10 times. One of them completed the task successfully and scored 90 points. How many times did he score 7 points, if he hit the bullseye 4 times, and the results of the other hits were sevens, eights, and nines? There were no misses at all.",
      "answer": "1"
    },
    {
      "question": "8. (5 points) If two 0s are added after a number, the new number is 1980 more than the original number. What is this number? $\\qquad$",
      "answer": "20"
    },
    {
      "question": "Test $\\mathbf{G}$ Calculation:\n$$\n\\frac{\\left(2^{4}+\\frac{1}{4}\\right)\\left(4^{4}+\\frac{1}{4}\\right)\\left(6^{4}+\\frac{1}{4}\\right)\\left(8^{4}+\\frac{1}{4}\\right)\\left(10^{4}+\\frac{1}{4}\\right)}{\\left(1^{4}+\\frac{1}{4}\\right)\\left(3^{4}+\\frac{1}{4}\\right)\\left(5^{4}+\\frac{1}{4}\\right)\\left(7^{4}+\\frac{1}{4}\\right)\\left(9^{4}+\\frac{1}{4}\\right)} .\n$$\n(1991 Jiangsu Province Junior High School Mathematics Competition Question)",
      "answer": "221"
    },
    {
      "question": "2. Three candles were lit simultaneously. When the first candle burned out, $\\frac{2}{5}$ of the second candle remained, and $\\frac{3}{7}$ of the third candle remained. What part of the third candle will remain when the second candle burns out?",
      "answer": "\\frac{1}{21}"
    },
    {
      "question": "## Task B-2.7.\n\nA line passing through the vertex $C$ of the right angle of triangle $A B C$ with the shorter leg forms an angle of $30^{\\circ}$ and divides the hypotenuse in the ratio 1 : 2. If the length of the shorter leg is $\\sqrt{3}$, determine the length of the hypotenuse.",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "6- 16 Find the value of $10 \\operatorname{ctg}(\\operatorname{arcctg} 3+\\operatorname{arcctg} 7+\\operatorname{arcctg} 13+\\operatorname{arcctg} 21)$.",
      "answer": "15"
    },
    {
      "question": "3. Given real numbers $a, b, c$ satisfy\n$$\nf(x)=a \\cos x+b \\cos 2 x+c \\cos 3 x \\geqslant-1\n$$\n\nfor any real number $x$. Then the maximum value of $a+b+c$ is . $\\qquad$",
      "answer": "3"
    },
    {
      "question": "Task 2. (10 points) A circle with a radius of 10 touches two adjacent sides $AB$ and $AD$ of the square $ABCD$. On the other two sides, the circle intersects at points, cutting off segments of 4 cm and 2 cm from the vertices, respectively. Find the length of the segment that the circle cuts off from vertex $B$ at the point of tangency.",
      "answer": "8"
    },
    {
      "question": "Problem 6. (Option 1).\n\nIn the right triangle $ABC: \\angle ACB=90^{\\circ}, AC=6, BC=4$. On the line $BC$, a point $D (CD>BD)$ is marked such that $\\angle ADC=45^{\\circ}$. On the line $AD$, a point $E$ is marked such that the perimeter of triangle $CBE$ is the smallest possible. Then, on the line $DC$, a point $F$ is marked such that the perimeter of triangle $AFE$ is the smallest possible. Find $CF$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_3fdcc7ef9dfdd3dab8ecg-07.jpg?height=846&width=557&top_left_y=228&top_left_x=241)\n\n## Construction and Proof:\n\nFor convenience, construct a square ACDK (AD is its diagonal, since $\\angle ADC=45^{\\circ}$. Mark point $B_1$ on side $DK (B_1D=BD \\Rightarrow B_1B \\perp AD)$.\n\nDraw line $B_1C$, which intersects $AD$ at point $E$.\n\nThe perimeter of triangle $CBE$ is the smallest because among all possible points $E_1$ on line $AD$, the sum of the lengths of segments $B_1E + EC$ is the smallest ($B_1E + EC < B_1E_1 + E_1C$ - triangle inequality) and $B_1E = EB$. Similarly, mark point $A_1$ on side $AC$ ($A_1C = AC$). Draw line $A_1E$, which intersects $CD$ at point $F$.\n\nThe perimeter of triangle $AFE$ is the smallest because among all possible points $F_1$ on line $AD$, the sum of the lengths of segments $A_1F + EF$ is the smallest ($A_1F + EF < A_1F_1 + F_1E$ - triangle inequality) and $A_1F = FA$.",
      "answer": "3.6"
    },
    {
      "question": "【Question 12】A and B take turns marking numbers from 1 to 17, with the rules: you cannot mark a number that has already been marked; you cannot mark a number that is twice a marked number; you cannot mark a number that is half of a marked number; the player who cannot mark a number loses. Now A marks 8 first, to ensure a certain win, B should mark _ next. $\\qquad$",
      "answer": "6"
    },
    {
      "question": "Let $a, b, c, d$ be real numbers defined by\n\n$$\na=\\sqrt{4-\\sqrt{5-a}}, b=\\sqrt{4+\\sqrt{5-b}}, c=\\sqrt{4-\\sqrt{5+c}}, d=\\sqrt{4+\\sqrt{5+d}}\n$$\n\nCalculate their product.\n\n## 2 Solutions",
      "answer": "11"
    },
    {
      "question": "10.4 Find the smallest positive integer $k$, for which in any coloring of the numbers of the set $M=\\{1,2,3, \\ldots, k\\}$ in two colors, there will be ten not necessarily distinct numbers of the same color from the set $M$",
      "answer": "109"
    },
    {
      "question": "Two workers are working on identical workpieces and both are currently meeting the standard. With the application of an innovation, one completes a workpiece 5 minutes faster than the standard time, while the other completes it 3 minutes faster. As a result, their average performance increases to 137.5%. What is the standard time for completing one workpiece?",
      "answer": "15"
    },
    {
      "question": "4. What is the minimum number of cells that need to be colored on a $6 \\times 6$ board so that, for any placement (including rotations and flips) of a figure consisting of 4 cells in the shape of the letter $\\Gamma$, there is at least one colored cell?",
      "answer": "12"
    },
    {
      "question": "For real numbers $a$ and $b$, where $|a| \\neq|b|$ and $a \\neq 0$, the following holds:\n\n$$\n\\frac{a-b}{a^{2}+a b}+\\frac{a+b}{a^{2}-a b}=\\frac{3 a-b}{a^{2}-b^{2}}\n$$\n\nDetermine the value of the expression $\\frac{b}{a}$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "H2. How many nine-digit integers of the form 'pqrpqrpqr' are multiples of 24 ? (Note that $p, q$ and $r$ need not be different.)",
      "answer": "112"
    },
    {
      "question": "Extend 1 Let non-negative real numbers $a, b, c, x, y, z$ satisfy $a+b+c=x+y+z=1$.\nFind the minimum value of $\\left(a-x^{2}\\right)\\left(b-y^{2}\\right)\\left(c-z^{2}\\right)$.",
      "answer": "-\\frac{1}{4}"
    },
    {
      "question": "B2. Integers $a, b, c, d$, and $e$ satisfy the following three properties:\n(i) $2 \\leq a<b<c<d<e<100$\n(ii) $\\operatorname{gcd}(a, e)=1$\n(iii) $a, b, c, d, e$ form a geometric sequence.\nWhat is the value of $c$ ?",
      "answer": "36"
    },
    {
      "question": "3 3.7 A regular dodecahedron has 20 vertices and 30 edges. Each vertex is the intersection of 3 edges, and the other endpoints of these 3 edges are 3 other vertices of the regular dodecahedron. We call these 3 vertices adjacent to the first vertex. Place a real number at each vertex, such that the number placed at each vertex is exactly the arithmetic mean of the numbers placed at the 3 adjacent vertices. Let $M, m$ be the largest and smallest of these 20 numbers, respectively. What is the range of $M-m$?",
      "answer": "0"
    },
    {
      "question": "I1.4 In the coordinate plane, the distance from the point $(-C, 0)$ to the straight line $y=x$ is $\\sqrt{D}$, find the value of $D$.",
      "answer": "8"
    },
    {
      "question": "33rd BMO 1997 Problem 3 Find the number of polynomials of degree 5 with distinct coefficients from the set {1, 2, ... , 9} which are divisible by x 2 - x + 1. Solution",
      "answer": "624"
    },
    {
      "question": "5. Given a natural number $x=6^{n}+1$, where $n-$ is an odd natural number. It is known that $x$ has exactly three distinct prime divisors, one of which is 11. Find $x$.",
      "answer": "7777"
    },
    {
      "question": "14. (10 points) Using a batch of paper to bind a type of exercise book. If 120 books have already been bound, the remaining paper is $40 \\%$ of the total; if 185 books have been bound, then 1350 sheets of paper are left. How many sheets of paper are there in total?",
      "answer": "18000"
    },
    {
      "question": "1. Let $f(x)=x^{3}+a x+b$, with $a \\neq b$, and suppose the tangent lines to the graph of $f$ at $x=a$ and $x=b$ are parallel. Find $f(1)$.",
      "answer": "1"
    },
    {
      "question": "Problem 9.5. Let $A B C D$ be a cyclic quadrilateral with circumcircle $k$. The rays $\\overrightarrow{D A}$ and $\\overrightarrow{C B}$ meet at point $N$ and the line $N T$ is tangent to $k, T \\in k$. The diagonals $A C$ and $B D$ meet at the centroid $P$ of $\\triangle N T D$. Find the ratio $N T: A P$.\n\nIvailo Kortezov",
      "answer": "3"
    },
    {
      "question": "## problem statement\n\nApproximately calculate using the differential.\n\n$y=\\sqrt{x^{2}+5}, x=1.97$",
      "answer": "2.98"
    },
    {
      "question": "Find the remainder when $10^{2^{0}}+10^{2^{1}}+\\cdots+10^{2^{2021}}$ is divided by $\\underbrace{44 \\cdots 44}_{44}$.",
      "answer": "2020202020202020202020202020202020202020110"
    },
    {
      "question": "35. Let $x$ be a positive integer, and write $a=\\left\\lfloor\\log _{10} x\\right\\rfloor$ and $b=\\left\\lfloor\\log _{10} \\frac{100}{x}\\right\\rfloor$. Here $\\lfloor c\\rfloor$ denotes the greatest integer less than or equal to $c$. Find the largest possible value of $2 a^{2}-3 b^{2}$.",
      "answer": "24"
    },
    {
      "question": "6. In tetrahedron $ABCD$, $AD=2\\sqrt{3}$, $\\angle BAC=60^{\\circ}$, $\\angle BAD=\\angle CAD=45^{\\circ}$. If a sphere passing through point $D$ is tangent to plane $ABC$ and internally tangent to the circumscribed sphere of the tetrahedron with a radius of 1, then the radius of the circumscribed sphere of tetrahedron $ABCD$ is . $\\qquad$",
      "answer": "3"
    },
    {
      "question": "3. $f(x)$ is an odd function, $g(x)$ is an even function, and $f(x)+g(x)+f(x) g(x)=\\left(e^{x}+\\sin x\\right) \\cdot \\cos x$, then $f(x)$ has $\\qquad$ zeros in $[0,4 \\pi)$.",
      "answer": "5"
    },
    {
      "question": "Example 3. Find $\\lim _{x \\rightarrow \\frac{\\pi}{2}} \\frac{\\tan x}{\\tan 3 x}$.",
      "answer": "3"
    },
    {
      "question": "2.2. Gavriila found out that the front tires of the car last for 24000 km, while the rear tires last for 36000 km. Therefore, he decided to swap them at some point to maximize the distance the car can travel. Find this distance (in km).",
      "answer": "28800"
    },
    {
      "question": "1. (12th \"Hope Cup\" Invitational Training Question) In the quadratic trinomial $a x^{2}+b x+c$, $a>100$, how many integer values of $x$ at most can make the absolute value of the quadratic trinomial not exceed 50?",
      "answer": "2"
    },
    {
      "question": "7.6. Let $a, b, c$ be natural numbers, and the product $ab$ is divisible by $5c$, the product $c$ is divisible by $13a$, and the product $ca$ is divisible by $31b$. Find the smallest possible value of the product $abc$. Justify your answer.",
      "answer": "4060225"
    },
    {
      "question": "Example 2. Find the smallest positive integer $n$, such that in any 9-vertex and $n$-edge graph with edges colored in two colors, there must exist a monochromatic triangle.\n\n untranslated portion: \n(There is no additional text to translate, as the provided text has been fully translated.)",
      "answer": "33"
    },
    {
      "question": "2. Find the smallest natural number that has exactly 12 different natural divisors, the largest prime divisor of which is the number 101, and the last digit is zero.",
      "answer": "2020"
    },
    {
      "question": "10. (20 points) Given $O$ as the origin, $F$ as the right focus of the ellipse $C: \\frac{x^{2}}{a^{2}}+\\frac{y^{2}}{b^{2}}=1(a>b>0)$, a line $l$ passing through point $F$ intersects the ellipse $C$ at points $A$ and $B$. Points $P$ and $Q$ on the ellipse satisfy\n$$\n\\overrightarrow{O P}+\\overrightarrow{O A}+\\overrightarrow{O B}=\\overrightarrow{O P}+\\overrightarrow{O Q}=0 \\text {, }\n$$\n\nand points $P, A, Q, B$ are concyclic. Find the eccentricity of the ellipse $C$.",
      "answer": "\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "$$\n[\\sqrt{2-\\sqrt{3}}(2-\\sqrt{3})+\\sqrt{2+\\sqrt{3}}(2+\\sqrt{3})]^{2}=?\n$$",
      "answer": "54"
    },
    {
      "question": "97. The Fibonacci sequence. Consider the Fibonacci sequence $1,1,2,3,5,13, \\ldots$, whose terms $F_{n}$ (Fibonacci numbers) satisfy the relation $F_{n+2}=F_{n}+F_{n+1}$, $F_{1}=F_{2}=1$. Now consider the sequence of digits in the units place of the Fibonacci numbers. Will this sequence be cyclic, that is, can it be obtained by the unlimited repetition of the same finite set of digits, as is true, for example, in the case of the sequence $055055 \\ldots$.",
      "answer": "60"
    },
    {
      "question": "Task 1. 15 points\n\nFind the minimum value of the expression\n\n$$\n\\sqrt{x^{2}-\\sqrt{3} \\cdot|x|+1}+\\sqrt{x^{2}+\\sqrt{3} \\cdot|x|+3}\n$$\n\nas well as the values of $x$ at which it is achieved.",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "$\\underline{\\text { Folklore }}$\n\nFind the largest natural $n$, for which $n^{200}<5^{300}$.",
      "answer": "11"
    },
    {
      "question": "1. Calculate: $25 \\times 13 \\times 2 + 15 \\times 13 \\times 7=$",
      "answer": "2015"
    },
    {
      "question": "1. (5 points) If $0.142857^{+} x=1.5$, then $x=$",
      "answer": "1\\frac{5}{14}"
    },
    {
      "question": "11. Real numbers $x, y, z, w$ satisfy $x+y+z+w=1$, then the maximum value of $M=x w+2 y w+3 x y+3 z w+4 x z+5 y z$ is $\\qquad$ .",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "19. In the triangle $A B C, A B=A C, \\angle A=90^{\\circ}, D$ is the midpoint of $B C, E$ is the midpoint of $A C$ and $F$ is a point on $A B$ such that $B E$ intersects $C F$ at $P$ and $B, D, P, F$ lie on a circle. Let $A D$ intersect $C P$ at $H$. Given $A P=\\sqrt{5}+2$, find the length of $P H$.",
      "answer": "1"
    },
    {
      "question": "47 (1289). Two brothers walk together from school to home at the same speed. One day, 15 minutes after leaving school, the first brother ran back to school and, upon reaching it, immediately set off to catch up with the second. Left alone, the second continued walking home at half his usual speed. When the first brother caught up with the second, they resumed their initial speed and arrived home 6 minutes later than usual. How many times faster is the first brother's running speed compared to their usual walking speed?",
      "answer": "3"
    },
    {
      "question": "18. (6 points) Given the four-digit number $\\overline{\\mathrm{ABCD}}$, the conclusions of three people, Jia, Yi, and Bing, are as follows:\n\nJia: “The unit digit is half of the hundred digit”;\nYi: “The tens digit is 1.5 times the hundred digit”;\nBing: “The average of the four digits is 4”.\nAccording to the above information, we can get: $\\overline{\\mathrm{ABCD}}=$",
      "answer": "4462"
    },
    {
      "question": "6. A line with slope $k$ is drawn through one of the foci of the ellipse $x^{2}+2 y^{2}=3$, intersecting the ellipse at points $A, B$. If $|A B|=2$, then $|k|=$ $\\qquad$ .",
      "answer": "\\sqrt{1+\\sqrt{3}}"
    },
    {
      "question": "Exercise 4. We are given 20 points in the plane such that no three of them are collinear. How many lines can be formed passing through two of these points?",
      "answer": "190"
    },
    {
      "question": "(11) As shown in the figure, let $S-ABCD$ be a pyramid with a height of 3, where the base $ABCD$ is a square with side length 2, and the projection of vertex $S$ onto the base is the center of the square $ABCD$. $K$ is the midpoint of edge $SC$, and a plane through $AK$ intersects segments $SB$ and $SD$ at points $M$ and $N$ respectively. ($M$ and $N$ can be the endpoints of the segments).\n(1) Find the sine of the angle between line $AK$ and plane $SBC$;\n(2) Prove that $\\frac{V_{S-MKN}}{V_{S-BCD}}=\\frac{SM \\cdot SK \\cdot SN}{SB \\cdot SC \\cdot SD}$, and find the volume of the pyramid $S-AMKN$ when $M$ is the midpoint of $SB$.",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "11. The selling price of a commodity is an integer number of yuan. 100 yuan can buy at most 3 pieces. Person A and Person B each brought several hundred-yuan bills. The money A brought can buy at most 7 pieces of this commodity, and the money B brought can buy at most 14 pieces. When the two people's money is combined, they can buy 1 more piece. The price of each piece of this commodity is yuan.",
      "answer": "27"
    },
    {
      "question": "Question 42: A set of 4027 points on a plane is called a \"Colombian point set\" if no three points are collinear, and 2013 points are red, 2014 points are blue. A set of lines can divide the plane into several regions. If a set of lines for a Colombian point set satisfies the following two conditions, it is called a \"good set of lines\":\n(1) These lines do not pass through any of the points in the Colombian point set;\n(2) No region contains points of both colors.\nFind the minimum value of $\\mathrm{k}$ such that for any Colombian point set, there exists a good set of lines consisting of $\\mathrm{k}$ lines.",
      "answer": "2013"
    },
    {
      "question": "## Task 2\n\nA Pioneer group goes to the castle pond in Karl-Marx-City to go boating. 23 Pioneers get into the boats. The other 5 Pioneers wait on the shore and play with a ball.\n\nHow many Pioneers are in the group?",
      "answer": "28"
    },
    {
      "question": "12. What is the value of $A^{2}+B^{3}+C^{5}$, given that:\n$$\n\\begin{array}{l}\nA=\\sqrt[3]{16 \\sqrt{2}} \\\\\nB=\\sqrt{9 \\sqrt[3]{9}} \\\\\nC=\\left[(\\sqrt[5]{2})^{2}\\right]^{2}\n\\end{array}\n$$",
      "answer": "105"
    },
    {
      "question": "11.1. Find all positive integer solutions of the equation $(n+2)!-(n+1)!-(n)!=n^{2}+n^{4}$. Answer. $n=3$.",
      "answer": "3"
    },
    {
      "question": "For the vertices of an $n$-sided polygon, we wrote different real numbers such that any written number is the product of the numbers written at the two adjacent vertices.\n\nDetermine the value of $n$!",
      "answer": "6"
    },
    {
      "question": "Which is the three-digit (integer) number that, when increased or decreased by the sum of its digits, results in a number consisting of the same digit repeated?",
      "answer": "105"
    },
    {
      "question": "Example 1 Calculate the sum $S=\\sum_{n=1}^{509}\\left[\\frac{511 n}{509}\\right]$.",
      "answer": "130051"
    },
    {
      "question": "3. Pasha and Igor are flipping a coin. If it lands on heads, Pasha wins; if tails, Igor wins. The first time the loser pays the winner 1 ruble, the second time - 2 rubles, then - 4, and so on (each time the loser pays twice as much as the previous time). At the beginning of the game, Pasha had a single-digit amount of money, and Igor had a four-digit amount. By the end, Igor had a two-digit amount, and Pasha had a three-digit amount. What is the minimum number of games Pasha could have won? The players cannot go into debt.\n\n(L. S. Korechkova, A. A. Tessler)",
      "answer": "7"
    },
    {
      "question": "$1 \\cdot 30$ Find the largest natural number $x$, such that for every natural number $y, x$ divides\n$$\n7^{y}+12 y-1 \\text {. }\n$$\n(\"Friendship Cup\" International Mathematics Competition, 1992)",
      "answer": "18"
    },
    {
      "question": "4. On a plane, there are 10 different points. We considered the midpoints of all segments connecting all pairs of points. What is the smallest number of midpoints that could have arisen?",
      "answer": "17"
    },
    {
      "question": "1. [3] How many different values can $\\angle A B C$ take, where $A, B, C$ are distinct vertices of a cube?",
      "answer": "5"
    },
    {
      "question": "## Task 1\n\nParents of class 2a want to host guests for the festival. 8 parents each host 3 guests, 7 parents each host 2 guests, and 9 parents each host 1 guest.\n\nHow many guests will be hosted by the parents of class 2a?",
      "answer": "47"
    },
    {
      "question": "6. (10 points) The sequence $1,1,2,3,5,8, \\cdots$ is such that each term from the second onward is equal to the sum of the two preceding terms, forming the Fibonacci sequence. Each term in this sequence is called a Fibonacci number. It can be proven that \"any positive integer $n$ can be expressed as the sum of several different Fibonacci numbers.\" How many ways are there to express 100 as the sum of several different Fibonacci numbers? (Different orders of the addends are considered the same method)",
      "answer": "9"
    },
    {
      "question": "Exercise 2. Anne drove her car for an integer (and non-zero) number of hours, and traveled an integer number of kilometers, at a speed of $55 \\mathrm{~km} / \\mathrm{h}$. At the beginning of the trip, the odometer read abc kilometers, where abc is a 3-digit number such that $a \\geqslant 1$ and $a+b+c \\leqslant 7$. At the end of the trip, the odometer read cba kilometers. Determine all possible values of the number abc.",
      "answer": "106"
    },
    {
      "question": "A magpie cooked porridge and fed her chicks. The third chick received as much porridge as the first two combined. The fourth one received as much as the second and third. The fifth one received as much as the third and fourth. The sixth one received as much as the fourth and fifth. The seventh one did not get any - the porridge ran out! It is known that the fifth chick received 10 grams of porridge. How much porridge did the magpie cook?\n\n#",
      "answer": "40"
    },
    {
      "question": "PROBLEM 4. Let the sequence $\\left(x_{n}\\right)_{n \\geq 1}$, defined by $x_{1}=3$ and $x_{n}=x_{n-1}+2 n+1$, for $n \\geq 2$.\n\na) Determine the general term of the sequence;\n\nb) Calculate $\\lim _{n \\rightarrow \\infty} n\\left(\\ln 2+\\ln \\left(\\sum_{k=1}^{n} \\frac{1}{x_{2 k-1}}\\right)\\right)$.[^0]\n\n## NATIONAL MATHEMATICS OLYMPIAD  Local stage - 14.02. 2015  GRADING SCALE - 11th Grade",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "23. In $\\triangle \\mathrm{ABC}, \\angle \\mathrm{CAB}=30^{\\circ}$ and $\\angle \\mathrm{ABC}=80^{\\circ}$. The point $\\mathrm{M}$ lies inside the triangle such that $\\angle \\mathrm{MAC}=10^{\\circ}$ and $\\angle \\mathrm{MCA}=30^{\\circ}$. Find $\\angle \\mathrm{BMC}$ in degrees.",
      "answer": "110"
    },
    {
      "question": "Example 3. Find $\\lim _{x \\rightarrow 4} \\frac{x^{2}-6 x+8}{x-4}$.",
      "answer": "2"
    },
    {
      "question": "11. (20 points) Given the quadratic functions\n$$\ny_{1}=a x^{2}+4 x+b \\text { and } y_{2}=b x^{2}+4 x+a\n$$\n\nboth have minimum values, denoted as $m$ and $n$ for $y_{1}$ and $y_{2}$, respectively.\n(1) If $m+n=0$, prove that for any real number $x$, $y_{1}+y_{2} \\geqslant 0$;\n(2) If $m$ and $n$ are both greater than 0, and $m n=2$, let $M$ be the larger of $m$ and $n$. Find the minimum value of $M$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "4-7. Along a straight alley, 100 lamps are placed at equal intervals, numbered sequentially from 1 to 100. At the same time, from different ends of the alley, Petya and Vasya started walking towards each other at different constant speeds (Petya from the first lamp, Vasya from the hundredth). When Petya was at the 22nd lamp, Vasya was at the 88th lamp. At which lamp will their meeting occur? If the meeting occurs between two lamps, indicate the smaller number of these two.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7d113eccc6d3ed32fb40g-03.jpg?height=485&width=1865&top_left_y=1131&top_left_x=107)",
      "answer": "64"
    },
    {
      "question": "G4.4 If $x_{1}, x_{2}, x_{3}, x_{4}$ and $x_{5}$ are positive integers that satisfy $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=x_{1} x_{2} x_{3} x_{4} x_{5}$, that is the sum is the product, find the maximum value of $x_{5}$.",
      "answer": "5"
    },
    {
      "question": "## Task 18/76\n\nGiven an annulus and $n$ circles within it such that each of them touches the inner and outer boundary circles and two other circles. How large is $n$ if the ratio of the annulus area $A_{1}$ to the sum of the $n$ circle areas $A_{2}$ is $\\sqrt{2}$?",
      "answer": "4"
    },
    {
      "question": "12.1.1. In a box, there are 6 identical pairs of black gloves and 4 identical pairs of beige gloves. Find the probability that two randomly drawn gloves form a pair.",
      "answer": "\\frac{47}{95}"
    },
    {
      "question": "6. $P$ is the midpoint of the height $V O$ of the regular quadrilateral pyramid $V-A B C D$. If the distance from point $P$ to the side face is 3, and the distance from point $P$ to the base is 5, then the volume of the regular quadrilateral pyramid is",
      "answer": "750"
    },
    {
      "question": "Task B-4.4. Determine all natural numbers $a$ for which the number $a^{3}+1$ is a power of 3.",
      "answer": "2"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nSuppose that $O$ is the origin. Points $P(a, b)$ and $Q(c, 1)$ are in the first quadrant with $a=2 c$. If the slope of $O P$ is $t$ and the slope of $O Q$ is 1 , what is the slope of $P Q$ ?",
      "answer": "21"
    },
    {
      "question": "## 258. Math Puzzle $11 / 86$\n\nHans is supposed to nail a $90 \\mathrm{~cm}$ wide gap in a wooden fence. He has eight $10 \\mathrm{~cm}$ wide and sixteen $8 \\mathrm{~cm}$ wide boards of suitable length. A saw, with which he could split the wood lengthwise, is not available to him.\n\nHow many and which options does Hans have to close the opening without any gaps?",
      "answer": "1)5"
    },
    {
      "question": "Example 6 (2008 National High School Mathematics Joint Competition, First Test) Let $f(x)$ be a function defined on $\\mathbf{R}$. If $f(0)=2008$, and for any $x \\in \\mathbf{R}$, it satisfies $f(x+2)-f(x) \\leqslant 3 \\cdot 2^{x}, f(x+6)-f(x) \\geqslant 63 \\cdot 2^{x}$, then $f(2008)=$ $\\qquad$ .",
      "answer": "2^{2008}+2007"
    },
    {
      "question": "9.3. How many solutions in integers $x, y$ does the inequality $|y-x|+|3 x-2 y| \\leq a$ have a) for $a=2$; b) for $a=20$?",
      "answer": "13"
    },
    {
      "question": "6. Several cells on a $14 \\times 14$ board are marked. It is known that no two of the marked cells are in the same row and the same column, and also that a knight can, starting from any marked cell, reach any other marked cell via marked cells. What is the maximum possible number of marked cells?",
      "answer": "14"
    },
    {
      "question": "## Zadatak B-1.6.\n\nRiješite jednadžbu\n\n$$\n\\frac{x^{4}-4 x^{3}+8 x^{2}-16 x+16}{x^{4}-16}=\\frac{1}{1 \\cdot 2}+\\frac{1}{2 \\cdot 3}+\\frac{1}{3 \\cdot 4}+\\cdots+\\frac{1}{25 \\cdot 26}\n$$",
      "answer": "102"
    },
    {
      "question": "8. The value of the algebraic expression $\\frac{1}{1 \\cdot 3}+\\frac{1}{3 \\cdot 5}+\\cdots+\\frac{1}{(2 n-1)(2 n+1)}+\\cdots+\\frac{1}{255 \\cdot 257}$ is",
      "answer": "\\frac{128}{257}"
    },
    {
      "question": "From point $A$, a passenger train travels towards $B$ at a speed of $30 \\mathrm{~km}$ per hour. A fast train, also departing from $A$ but later, travels at a speed of $60 \\mathrm{~km}$ per hour and catches up with the passenger train in $B$ under normal conditions. The passenger train, after covering ${ }^{2} / 3$ of its journey, is forced to continue at half its original speed due to a technical failure, which results in the fast train catching up $27^{1} / 9 \\mathrm{~km}$ before $B$. What is the distance between $A$ and $B$?",
      "answer": "122"
    },
    {
      "question": "6. (3 points) Xiaofang participated in an English speaking competition, and the scores from ten judges are: 95 points, 94 points, 91 points, 88 points, 91 points, 90 points, 94 points, 93 points, 91 points, 92 points. If the highest score and the lowest score are removed, Xiaofang's final average score is $\\qquad$ points.",
      "answer": "92"
    },
    {
      "question": "9. For what values of the parameter a does the equation $f(x)=p(x)$ have one solution, if $f(x)=$ $\\left|\\frac{2 x^{3}-5 x^{2}-2 x+5}{(1.5 x-3)^{2}-(0.5 x-2)^{2}}\\right|, \\quad p(x)=|2 x+5|+a$. If there are more than one value of the parameter, then in the answer, specify their sum.\n\n#",
      "answer": "-10"
    },
    {
      "question": "## Task 4 - 340734\n\nA quadrilateral is called convex if and only if all its diagonals lie entirely within the area of the polygon.\n\nHow many diagonals does a convex 1995-gon have in total? Justify the number you provide!",
      "answer": "1987020"
    },
    {
      "question": "Given is an unknown sequence $a_{1}, a_{2}, a_{3}, \\ldots$ of integers that satisfies the following property: for every prime number $p$ and every positive integer $k$ it holds that\n\n$$\na_{p k+1}=p a_{k}-3 a_{p}+13\n$$\n\nDetermine all possible values of $a_{2013}$.",
      "answer": "2016"
    },
    {
      "question": "Task 1. We have two jugs, one holds exactly 3 liters, and the other holds exactly 5 liters. Can we measure exactly 4 liters using the two jugs?",
      "answer": "4"
    },
    {
      "question": "8. Draw a line with an inclination angle of $\\frac{\\pi}{4}$ through the point $P(11,0)$, intersecting the parabola $y^{2}=4 x$ at points $R$ and $Q$. Then draw a line parallel to $R Q$ intersecting the parabolic arc $\\overparen{R O Q}$ at points $M$ and $N$ (as shown in the figure). Find the maximum value of the area of $\\triangle P M N$.",
      "answer": "22"
    },
    {
      "question": "Problem 7.6.1. In a row, there are 10 boxes. These boxes contain balls of two colors: red and blue. In some boxes, all the balls may be of the same color; there are no empty boxes. It is known that in each subsequent box (from left to right), there are no fewer balls than in the previous one. It is also known that there are no two boxes with the same set of red and blue balls. How many blue and how many red balls are in the rightmost box, if the total number of red balls in all the boxes is 11 and the total number of blue balls is 13?",
      "answer": "1"
    },
    {
      "question": "Ha az előbbi feladatban azt is tudni akarjuk, hogy a hamis pénz könnyebb-e, vagy nehezebb a többinél, hány pénzdarab a legtöbb, amennyi közül még három mérés elegendő ennek a kérdésnek az eldöntéséhez?\n\nIf in the previous task we also want to know whether the counterfeit coin is lighter or heavier than the others, how many coins is the maximum number from which three measurements are still sufficient to determine the answer to this question?",
      "answer": "12"
    },
    {
      "question": "## Task 1 - 040831\n\nIf the digits of a two-digit number $n$ are reversed, the resulting number is $\\frac{8}{3}$ times as large as $n$. The number $n$ is to be determined.",
      "answer": "27"
    },
    {
      "question": "10. Let the function $f(x)=4 x^{3}+b x+1(b \\in \\mathbf{R})$, for any $x \\in[-1,1]$, it holds that $f(x) \\geqslant 0$. Find the range of the real number $b$.",
      "answer": "-3"
    },
    {
      "question": "8. If positive integers $a, b, c$ satisfy\n$$\n2019 \\geqslant 10 a \\geqslant 100 b \\geqslant 1000 c \\text {, }\n$$\n\nthen the number of arrays $(a, b, c)$ is $\\qquad$ .",
      "answer": "574"
    },
    {
      "question": "9. (10 points) The teacher in the kindergarten brought 55 apples, 114 cookies, 83 chocolates to the class, and after distributing them equally, there were 3 apples, 10 cookies, and 5 chocolates left. The class has at most $\\qquad$ children.",
      "answer": "26"
    },
    {
      "question": "2. Two squirrels had the same number of pine cones and the same number of cedar cones. In total, each squirrel had fewer than 25 cones. The first squirrel gathered as many pine cones as it already had and 26 cedar cones. It ended up with more pine cones than cedar cones. The second squirrel gathered as many cedar cones as it already had and ate 4 pine cones. It ended up with more cedar cones than pine cones. How many pine and cedar cones did each squirrel have initially?",
      "answer": "17"
    },
    {
      "question": "4. Calculate: $2017 \\times 2016 + 2016 \\times 2014 - 2015 \\times 2016 - 2015 \\times 2017$.",
      "answer": "1"
    },
    {
      "question": "## Task A-2.4.\n\nAll vertices of the hexagon $A B C D E F$ lie on a circle with diameter $\\overline{A D}$. The line $B F$ intersects the lines $A D$ and $C E$ at points $G$ and $H$ respectively. If $\\varangle F E H=56^{\\circ}, \\varangle D G B=124^{\\circ}$ and $\\varangle D E C=34^{\\circ}$, determine $\\varangle C E B$.",
      "answer": "22"
    },
    {
      "question": "56. In the class, 12 subjects are studied. In how many ways can 5 different subjects be scheduled for Monday?",
      "answer": "9504"
    },
    {
      "question": "## [ Rectangles and Squares. Properties and Characteristics  Method of Coordinates\n\nOn the plane, a square $A B C D$ is given. Find the minimum of the ratio $\\frac{O A+O C}{O B+O D}$, where $O-$ is an arbitrary point on the plane.",
      "answer": "\\frac{1}{\\sqrt{2}}"
    },
    {
      "question": "14. (15 points) As shown in the figure, in a square $P Q R S$ with a side length greater than $20 \\mathrm{~cm}$, there is a largest circle $O$. If a point $T$ on the circumference of the circle is $8 \\mathrm{~cm}$ away from $P S$ and $9 \\mathrm{~cm}$ away from $P Q$, what is the radius of circle $O$ in centimeters?",
      "answer": "29"
    },
    {
      "question": "9. (16 points) Find the integer part of $\\sqrt{100+\\sqrt{101+\\cdots+\\sqrt{999}}}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "10"
    },
    {
      "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a_{\\text {and }} b$.\n\n\\[\n\\begin{aligned}\n& a=4 p-q \\\\\n& b=p+2 q \\\\\n& |p|=5 \\\\\n& |q|=4 \\\\\n& (\\widehat{p, q})=\\frac{\\pi}{4}\n\\end{aligned}\n\\]",
      "answer": "90\\sqrt{2}"
    },
    {
      "question": "Point $A$ lies in the plane $\\alpha$, the orthogonal projection of segment $A B$ onto this plane is $1, A B=2$. Find the distance from point $B$ to the plane $\\alpha$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "11.32. The area of a trapezoid is 1. What is the smallest value that the largest diagonal of this trapezoid can have?",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "276. Solve the equation:\n\n$$\n\\left(x^{2}-16\\right)(x-3)^{2}+9 x^{2}=0\n$$",
      "answer": "-1\\\\sqrt{7}"
    },
    {
      "question": "# Task 8.5\n\nOn the Island of Liars and Knights, a circular arrangement is called correct if each person standing in the circle can say that among their two neighbors, there is a representative of their tribe. Once, 2019 natives formed a correct arrangement in a circle. A liar approached them and said: \"Now we can also form a correct arrangement in a circle together.\" How many knights could there have been in the initial arrangement?\n\n## Number of points 7",
      "answer": "1346"
    },
    {
      "question": "4. On an $8 \\times 8$ board, there is a beetle on each square. At a certain moment, the distribution of the beetles on the board changes: each beetle crawls either one square to the left or one square diagonally to the right and down. If a beetle cannot make either move without falling off the board, it stays on its square.\n\nHow many squares can become unoccupied at most due to this change?",
      "answer": "29"
    },
    {
      "question": "10.8. A numismatist has 100 coins that look identical. He knows that among them, 30 are genuine and 70 are counterfeit. Moreover, he knows that the weights of all genuine coins are the same, while the weights of all counterfeit coins are different, and any counterfeit coin is heavier than a genuine one; however, the exact weights of the coins are unknown. There are balance scales without weights, on which it is possible to compare the weights of two groups consisting of the same number of coins in one weighing. What is the minimum number of weighings on these scales that the numismatist can guarantee to find at least one genuine coin?\n\n(C. Berlov, I. Bogdanov)",
      "answer": "70"
    },
    {
      "question": "13.\n\nA box of hawthorns has more than one hundred grains. When counted in groups of 3, there is 1 extra; when counted in groups of 4, there are 2 extra; when counted in groups of 5, there are 3 extra; and when counted in groups of 6, there are 4 extra. The box of hawthorns has at most ( ) grains.",
      "answer": "178"
    },
    {
      "question": "9. (16 points) Let $x \\in [0, 2\\pi]$. Find the maximum value of the function\n$$\n\\begin{aligned}\nf(x)= & \\sqrt{4 \\cos ^{2} x + 4 \\sqrt{6} \\cos x + 6} + \\\\\n& \\sqrt{4 \\cos ^{2} x - 8 \\sqrt{6} \\cos x + 4 \\sqrt{2} \\sin x + 22}\n\\end{aligned}\n$$",
      "answer": "2(\\sqrt{6}+\\sqrt{2})"
    },
    {
      "question": "Task 3. (15 points) An educational center \"Young Geologist\" received an object for research consisting of about 300 monoliths (a container designed for 300 monoliths, which was almost completely filled). Each monolith has a specific name (sandy loam or clayey loam) and genesis (marine or lake-glacial deposits). The relative frequency (statistical probability) that a randomly selected monolith will be sandy loam is $\\frac{1}{8}$. The relative frequency that a randomly selected monolith will be marine clayey loam is $\\frac{22}{37}$. How many monoliths of lake-glacial genesis does the object contain, if there are no marine sandy loams among the sandy loams?",
      "answer": "120"
    },
    {
      "question": "9. A gardener is preparing to plant a row of 20 trees, with two types of trees available: maple trees or sycamore trees. The number of trees between any two maple trees (not including these two maple trees) cannot be equal to 3. How many maple trees can there be at most among the 20 trees? $\\qquad$",
      "answer": "12"
    },
    {
      "question": "2. Graphically solve the equation $\\log _{2}(x-1)+1=-\\frac{3}{2} x+4$. Verify the solution by calculation.",
      "answer": "2"
    },
    {
      "question": "10.3. The sum $1+\\frac{1}{2}+\\frac{1}{3}+\\ldots+\\frac{1}{45}$ is represented as a fraction with the denominator $45!=1 \\cdot 2 \\cdots 45$. How many zeros (in decimal notation) does the numerator of this fraction end with?",
      "answer": "8"
    },
    {
      "question": "10. Consider the expression $1 * 2 * 3 * 4 * 5 * 6$.\nEach star in the expression is to be replaced with either ' + ' or ' $x$ '.\n$N$ is the largest possible value of the expression. What is the largest prime factor of $N$ ?",
      "answer": "103"
    },
    {
      "question": "Example 8 Color each vertex of a quadrilateral pyramid with one color, and make the endpoints of the same edge have different colors. If only 5 colors are available, how many different coloring methods are there?",
      "answer": "420"
    },
    {
      "question": "14. A ball was added to an urn containing one white ball - either white or black (with equal probabilities of selection). After this, one ball was randomly drawn from the urn. It turned out to be white. What is the conditional probability that the remaining ball in the urn is also white?",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "65. Find two numbers whose sum is 20 and product is 96.",
      "answer": "128"
    },
    {
      "question": "29.1. Find the three-digit number $\\overline{a b c}$, if it is known that\n\n$$\n\\begin{gathered}\n8 a+5 b+c=100 \\\\\na+b+c=20\n\\end{gathered}\n$$",
      "answer": "866"
    },
    {
      "question": "2B. In a certain populated place, all telephone numbers consist of six digits arranged in strictly ascending or strictly descending order, with the first digit in the number not being 0. What is the maximum number of telephone numbers that can exist in this place?",
      "answer": "294"
    },
    {
      "question": "12 different items are distributed among 3 people so that each person gets 4 items. In how many ways is this possible?",
      "answer": "34650"
    },
    {
      "question": "29. When from the numbers from 1 to 333 Tanya excluded all numbers divisible by 3 but not divisible by 7, and all numbers divisible by 7 but not divisible by 3, she ended up with 215 numbers. Did she solve the problem correctly?",
      "answer": "205"
    },
    {
      "question": "Three, (25 points) Find all positive integers such that it is 224 times the sum of its digits.\n\n\n\nThe above text has been translated into English, retaining the original text's line breaks and format.",
      "answer": "2016"
    },
    {
      "question": "3. Point $O$ is the center of the circumcircle of triangle $ABC$. On the circumcircle of triangle $BOC$, outside triangle $ABC$, point $X$ is chosen. On the rays $XB$ and $XC$ beyond points $B$ and $C$, points $Y$ and $Z$ are chosen respectively such that $XY = XZ$. The circumcircle of triangle $ABY$ intersects side $AC$ at point $T$. Find the angle $\\angle Y T Z$.",
      "answer": "180"
    },
    {
      "question": "25. One day, the owner put more than 60 identical chess pieces into some identical boxes, with only one box being empty. The mischievous Tom cat took one piece from each of the boxes that had pieces and put them into the empty box. The clever Jerry mouse rearranged the boxes, and as a result, the position of each box and the number of pieces in each box looked the same as before. Therefore, there are $\\qquad$ boxes.",
      "answer": "12"
    },
    {
      "question": "2. The minimum value of the function $f(x)=\\frac{(\\sin x+1)(\\cos x+1)}{\\sin x \\cos x}$ on $\\left(0, \\frac{\\pi}{2}\\right)$ is",
      "answer": "3+2\\sqrt{2}"
    },
    {
      "question": "\\section*{Problem 2 - 121232}\n\nIn space, four points \\(P_{1}, P_{2}, P_{3}\\) and \\(P_{4}\\) are given, which do not lie in the same plane. Determine the number of all those planes that are equidistant from these four points.",
      "answer": "7"
    },
    {
      "question": "[ Tiling with domino bones and tiles ] ..... ]\n[ Examples and counterexamples. Constructions ]\n\nOn a grid sheet of size 100×100, several pairwise non-overlapping cardboard isosceles right triangles with a leg of 1 were placed; each triangle occupies exactly half of one of the cells. It turned out that each unit segment of the grid (including boundary segments) is covered by exactly one leg of a triangle. Find the maximum possible number of cells that do not contain any triangle.",
      "answer": "2450"
    },
    {
      "question": "7. Given that each carbon pen costs 1 yuan 8 jiao, each notebook costs 3 yuan 5 jiao, and each stationery box costs 4 yuan 2 jiao, Jingjing bought these three stationery items and spent exactly 20 yuan. How many notebooks did she buy? $\\qquad$",
      "answer": "4"
    },
    {
      "question": "6. Find the largest constant $K \\in \\mathbb{R}$ with the following property: if $a_{1}, a_{2}, a_{3}, a_{4}>0$ are such that for all $i, j, k \\in \\mathbb{N}, 1 \\leqslant i<j<k \\leqslant 4$, we have $a_{i}^{2}+a_{j}^{2}+a_{k}^{2} \\geqslant 2\\left(a_{i} a_{j}+a_{j} a_{k}+a_{k} a_{i}\\right)$, then\n\n$$\na_{1}^{2}+a_{2}^{2}+a_{3}^{2}+a_{4}^{2} \\geqslant K\\left(a_{1} a_{2}+a_{1} a_{3}+a_{1} a_{4}+a_{2} a_{3}+a_{2} a_{4}+a_{3} a_{4}\\right)\n$$\n\n(Dušan Đukić)",
      "answer": "\\frac{11}{7}"
    },
    {
      "question": "Calculate $\\frac{1}{3}$ in $\\mathbb{Z}_{5}$.\n\n untranslated text:\nCalculer $\\frac{1}{3}$ dans $\\mathbb{Z}_{5}$.",
      "answer": "2"
    },
    {
      "question": "6.8 In the school football championship, each team must play a match against every other team. Each match's winning team gets 2 points, a draw gives each team 1 point, and the losing team gets 0 points. It is known that one team scored the most points (every other team scored fewer points than this team), but it won fewer matches than any other team. How many teams participated at the minimum?",
      "answer": "6"
    },
    {
      "question": "A sphere of radius 4 with center at point $Q$ touches three parallel lines at points $F, G$, and $H$. It is known that the area of triangle $Q G H$ is $4 \\sqrt{2}$, and the area of triangle $F G H$ is greater than 16. Find the angle $G F H$.",
      "answer": "67.5"
    },
    {
      "question": "11. Given the quadratic function $y=a x^{2}+b x+c$ whose graph intersects the $x$-axis at points $A$ and $B$, and whose vertex is $C$.\n(1) If $\\triangle A B C$ is a right triangle, find the value of $b^{2}-4 a c$.\n(2) Consider the quadratic function\n$$\ny=x^{2}-(2 m+2) x+m^{2}+5 m+3\n$$\n\nwhose graph intersects the $x$-axis at points $E$ and $F$, and intersects the graph of the linear function $y=3 x-1$ at two points, with the point having the smaller $y$-coordinate denoted as point $G$.\n(i) Express the coordinates of point $G$ in terms of $m$;\n(ii) If $\\triangle E F G$ is a right triangle, find the value of $m$.",
      "answer": "-1"
    },
    {
      "question": "[ Tangent lines and tangent circles ] Described quadrilaterals $\\quad]$\n\nTwo circles of radii 4 and 3 touch each other externally. Common external tangents $P Q$ and $R S$ are drawn to these circles such that points $P$ and $S$ lie on the circle of the larger radius, and points $Q$ and $R$ lie on the circle of the smaller radius. Find the radius of the circle that touches segments $R S$, $S P$, and $P Q$.",
      "answer": "\\frac{24}{7}"
    },
    {
      "question": "Task B-3.5. (20 points) Two equal chessboards ($8 \\times 8$ squares) have a common center and one completely covers the other. One of them is rotated around the center by $45^{\\circ}$. Determine the area of the intersection of all black squares of one board with all black squares of the other board if the area of one square is 1.",
      "answer": "32(\\sqrt{2}-1)"
    },
    {
      "question": "24.3.5 $\\star$ * 64 points arranged in an 8 by 8 grid, how many lines are needed at minimum to separate them?",
      "answer": "14"
    },
    {
      "question": "## Problem Statement\n\nBased on the definition of the derivative, find $f^{\\prime}(0)$:\n\n$$\nf(x)=\\left\\{\\begin{array}{c}\ne^{\\sin \\left(x^{\\frac{3}{2}} \\sin \\frac{2}{x}\\right)}-1+x^{2}, x \\neq 0 \\\\\n0, x=0\n\\end{array}\\right.\n$$",
      "answer": "0"
    },
    {
      "question": "19. (3 points) The Hope Cup Committee in a certain area is arranging examination rooms for candidates participating in the Hope Cup examination. If each examination room is arranged for 30 candidates, there will be one examination room with 26 candidates; if each examination room is arranged for 26 candidates, there will be one examination room with 20 candidates, and it will require 9 more examination rooms than the previous arrangement. The number of candidates participating in the examination in this area is $\\qquad$.",
      "answer": "1736"
    },
    {
      "question": "## Task 2\n\nMultiply the numbers 7 and 6. Subtract 5 from the product.\n\nWrite the equation and solve.",
      "answer": "37"
    },
    {
      "question": "10. Evaluate\n$$\n\\int_{-\\infty}^{\\infty} \\frac{1-x^{2}}{1+x^{4}} d x\n$$",
      "answer": "0"
    },
    {
      "question": "3. Given that $a, b$ are real numbers, if the quadratic function $f(x)=x^{2}+a x+b$ satisfies $f(f(0))=f(f(1))=0$, and $f(0) \\neq f(1)$, then the value of $f(2)$ is $\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "7. Given a tetrahedron $ABCD$ where one edge length is 3, and the other five edge lengths are all 2. Then the radius of its circumscribed sphere is $\\qquad$",
      "answer": "\\frac{\\sqrt{21}}{3}"
    },
    {
      "question": "Exercise 8. Let $x, y, z$ be three real numbers satisfying $x+y+z=2$ and $xy+yz+zx=1$. Determine the maximum value that $x-y$ can take.",
      "answer": "\\frac{2}{\\sqrt{3}}"
    },
    {
      "question": "Example 8 Find the maximum value of the function\n$$\nf(x)=\\sqrt{-x^{2}+10 x-9}+\\sqrt{-x^{2}+68 x-256}\n$$",
      "answer": "3\\sqrt{35}"
    },
    {
      "question": "8. The function $f(x)$ defined on the set $\\left\\{x \\in \\mathbf{Z}_{+} \\mid 1 \\leqslant x \\leqslant 12\\right\\}$ satisfies\n$$\n|f(x+1)-f(x)|=1(x=1,2, \\cdots, 11),\n$$\n\nand $f(1) 、 f(6) 、 f(12)$ form a geometric sequence. If $f(1)$ $=1$, then the number of different functions $f(x)$ that satisfy the conditions is $\\qquad$",
      "answer": "155"
    },
    {
      "question": "6. (48th Slovenian Mathematical Olympiad) Find all prime numbers $p$ such that $p+28$ and $p+56$ are also prime.",
      "answer": "3"
    },
    {
      "question": "10. As shown in the figure, $\\triangle A B C$ is an isosceles right triangle, $E$ is a point on side $B C$ such that $B E=3 C E$, and points $D, A, F$ are collinear. Let the height from $D$ to $B E$ in $\\triangle D B E$ be $h_{1}$, and the height from $F$ to $E C$ in $\\triangle F E C$ be $h_{2}$. We have $3 h_{1}+h_{2}=3$ cm. The sum of the areas of $\\triangle D B E$ and $\\triangle F E C$ is 6 square cm, then the area of $\\triangle A B C$ is $\\qquad$ square cm.",
      "answer": "64"
    },
    {
      "question": "Problem 5. $DABC$ is a triangular pyramid (Fig. 5). $\\overline{AK}=\\overline{KD}, \\overline{BP}=\\overline{PC}, \\overline{DM}=0.4 \\overline{DC}$. Find the area $S$ of the section of the pyramid by the plane $KMP$, if the vertex $A$ is at a distance $h=1$ from the plane $KMP$ and the volume of the pyramid $DABC$ is 5.",
      "answer": "3"
    },
    {
      "question": "3.11 Apples of the first grade for a total of 228 rubles and apples of the second grade for a total of 180 rubles were delivered to the store. During unloading, the delivered apples were accidentally mixed. It was found that if all the apples are now sold at one price - 90 kopecks lower than the price per kilogram of first-grade apples, the planned amount will be received. How many kilograms of apples were delivered, if it is known that there were 5 kg more of second-grade apples than first-grade apples?\n\n## Ratios between natural numbers",
      "answer": "85"
    },
    {
      "question": "13. Find the remainder when $(1!\\times 1)+(2!\\times 2)+(3!\\times 3)+\\cdots+(286!\\times 286)$ is divided by 2009 .",
      "answer": "2008"
    },
    {
      "question": "15.5. (GDR, 74). Determine into how many parts a regular tetrahedron is divided by six planes, each of which passes through one edge and the midpoint of the opposite edge of the tetrahedron. Find the volume of each of the parts if the volume of the tetrahedron is 1.",
      "answer": "\\frac{1}{24}"
    },
    {
      "question": "$\\underline{\\text { Folklore }}$\n\nFind the smallest natural $n$, for which the number $A=n^{3}+12 n^{2}+15 n+180$ is divisible by 23.",
      "answer": "10"
    },
    {
      "question": "8.1. Lesha wrote down a number, and then replaced the same digits with the same letters, and different digits with different letters. He ended up with the word NOVOSIBIRSK. Could the original number be divisible by 9?",
      "answer": "10203454638"
    },
    {
      "question": "Create a section on a sphere whose area is equal to one-third of the great circle's area, and on the intersecting surface, place a cone whose apex falls into the center of the sphere; if the volume of the cone is $480 \\mathrm{~m}^{3}$, what will be the radius of the sphere?",
      "answer": "11.90"
    },
    {
      "question": "2. Given $a>0, x, y$ satisfy the constraint conditions $\\left\\{\\begin{array}{l}x \\geqslant 1, \\\\ x+y \\leqslant 3, \\\\ y \\geqslant a(x-3) .\\end{array}\\right.$ If the minimum value of $z=2 x+y$ is 1, then $a=$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "8. (3 points) A large cube, four medium cubes, and four small cubes are assembled into a three-dimensional figure as shown in the diagram. It is known that the edge lengths of the large, medium, and small cubes are 5 cm, 2 cm, and 1 cm, respectively. What is the surface area of this three-dimensional figure in square centimeters?",
      "answer": "230"
    },
    {
      "question": "10th Mexico 1996 Problem A2 64 tokens are numbered 1, 2, ... , 64. The tokens are arranged in a circle around 1996 lamps which are all turned off. Each minute the tokens are all moved. Token number n is moved n places clockwise. More than one token is allowed to occupy the same place. After each move we count the number of tokens which occupy the same place as token 1 and turn on that number of lamps. Where is token 1 when the last lamp is turned on?",
      "answer": "32"
    },
    {
      "question": "4. To build a batch of identical houses with a total area of $2500 \\mathrm{~m}^{2}$, the cost of a $a \\mathrm{~m}^{2}$ house is the sum of material cost $100 p_{1} a^{\\frac{3}{2}}$ yuan, construction cost $100 p_{2} a$ yuan, and other various expenses $100 p_{3} a^{\\frac{1}{2}}$ yuan, where the numbers $p_{1} 、 p_{2} 、 p_{3}$ are three consecutive terms of a geometric sequence, their sum is 21, and their product is 64. If 63 such houses are built, the material cost will be less than the sum of the construction cost and other various expenses. To minimize the total cost, what is the maximum number of houses that can be built?",
      "answer": "156"
    },
    {
      "question": "3. (3 points) In a certain country, there are 50 cities. The Ministry of Aviation requires that every two cities be connected by a two-way flight operated by exactly one airline, and that it should be possible to travel from any city to any other using the flights of each airline (possibly with layovers). What is the maximum number of airlines for which this is possible?\n\n## Answer: 25",
      "answer": "25"
    },
    {
      "question": "In how many ways can the nine digits from 1 to 9 be placed in a $3 \\times 3$ grid so that the sums of the rows and the sums of the columns are all equal?",
      "answer": "72"
    },
    {
      "question": "4. In the rebus\n\n$$\n\\mathbf{K}\\mathbf{O}>\\mathbf{H}>\\mathbf{A}>\\mathbf{B}>\\mathbf{U}>\\mathbf{P}>\\mathbf{y}>\\mathbf{C}\n$$\n\ndifferent letters represent different digits. How many solutions does the rebus have?",
      "answer": "0"
    },
    {
      "question": "113. How many two-digit natural numbers are there in which the first digit is greater than the second?",
      "answer": "45"
    },
    {
      "question": "3. Given a trapezoid $A B C D$ with bases $A B$ and $C D$. Let $E$ be the midpoint of side $A B$, $F$ the midpoint of segment $D E$, and $G$ the intersection of segments $B D$ and $C E$. Express the area of trapezoid $A B C D$ in terms of its height $v$ and the length of segment $F G$, assuming that points $A, F, C$ lie on a straight line.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_1d8fc6c4c0f260a9dad0g-3.jpg?height=517&width=648&top_left_y=1723&top_left_x=710)",
      "answer": "3"
    },
    {
      "question": "8. Let the integer sequence $a_{1}, a_{2}, \\cdots, a_{10}$ satisfy $a_{10}=3 a_{1}, a_{2}+a_{8}=2 a_{5}$, and\n$$\na_{i+1} \\in\\left\\{1+a_{i}, 2+a_{i}\\right\\}, i=1,2, \\cdots, 9 \\text {, }\n$$\n\nthen the number of such sequences is $\\qquad$ .",
      "answer": "80"
    },
    {
      "question": "Question 223, Find the largest positive integer $n$, such that there exists a set with $n$ elements, where the set contains exactly 1 element divisible by $n$, exactly 2 elements divisible by $\\mathrm{n}-1$, $\\cdots$, exactly $\\mathrm{n}-1$ elements divisible by 2, and $\\mathrm{n}$ elements divisible by 1.",
      "answer": "5"
    },
    {
      "question": "5. (6 points) If $a<\\frac{1}{\\frac{1}{2011}+\\frac{1}{2012}+\\frac{1}{2013}+\\frac{1}{2014}+\\frac{1}{2015}}<a+1$, then the natural number $a=$ $\\qquad$",
      "answer": "402"
    },
    {
      "question": "8.4. On the Island of Liars and Knights, a circular arrangement is called correct if each person standing in the circle can say that among their two neighbors, there is a representative of their tribe. Once, 2019 natives formed a correct circular arrangement. A liar approached them and said: \"Now we can also form a correct circular arrangement together.\" How many knights could there have been in the initial arrangement?",
      "answer": "1346"
    },
    {
      "question": "[ Dirichlet's Principle (continued). ] [Partitions into pairs and groups; bijections]\n\nBefore a clairvoyant lies a deck of 36 cards face down (4 suits, 9 cards of each suit). He names the suit of the top card, after which the card is revealed to him. Then he names the suit of the next card, and so on. The task of the clairvoyant is to guess the suit as many times as possible. The backs of the cards are asymmetrical, and the clairvoyant can see which of the two positions the top card is in. The clairvoyant's assistant knows the order of the cards in the deck, cannot change it, but can arrange the back of each card in one way or another. Could the clairvoyant have agreed with the assistant, when the latter still did not know the order of the cards, to ensure guessing the suit of at least\n\na) 19 cards;\n\nb) 23 cards",
      "answer": "23"
    },
    {
      "question": "(4) The parabola $y^{2}=2 p x(p>0)$ has a focus $F$, and a directrix $l$. $A$ and $B$ are two moving points on the parabola, and they satisfy $\\angle A F B=\\frac{\\pi}{3}$. Let the midpoint of segment $A B$ be $M$, and the projection of $M$ on $l$ be $N$. Then the maximum value of $\\frac{|M N|}{|A B|}$ is $\\qquad$.",
      "answer": "1"
    },
    {
      "question": "9. A, B, C, and D are comparing their heights. Among them, the sum of the heights of two people is the same as the sum of the heights of the other two. The average height of A and B is 4 cm more than the average height of A and C. D is 10 cm taller than A, and the sum of the heights of B and C is 288 cm. A's height is $\\qquad$ cm.",
      "answer": "139"
    },
    {
      "question": "# Problem 4.\n\nGiven a square with a side length of 5 cm. On its four sides, the vertices of a second square are located; on the four sides of the second square, the vertices of a third square, and so on. For what smallest natural $n$ will the sum of the areas of the first $n$ squares be guaranteed to be greater than 49 cm $^{2}$?",
      "answer": "6"
    },
    {
      "question": "9. Let $F(x)=|f(x) \\cdot g(x)|$, where $f(x)=a x^{2}+b x+c, g(x)=c x^{2}+b x+a, x \\in[-1,1]$. For any $a, b, c$ satisfying $|f(x)| \\leqslant 1$. When $a, b, c$ vary, find the maximum value of $F(x)$.",
      "answer": "2"
    },
    {
      "question": "9.3. Vasya must write one digit on each face of several dice so that any ordered combination of three digits from 000 to 999 inclusive can be obtained by selecting some three different dice and placing them with the appropriate sides up in the correct order. At the same time, the digits 6 and 9 do not transform into each other when rotated 180 degrees. What is the minimum number of dice Vasya should use?",
      "answer": "5"
    },
    {
      "question": "7. If $\\cot \\alpha=2, \\tan (\\alpha-\\beta)=-\\frac{2}{5}$, then the value of $\\tan (\\beta-2 \\alpha)$ is",
      "answer": "-\\frac{1}{12}"
    },
    {
      "question": "97. The distance between locations A and B is 300 kilometers. A car was originally planned to travel from A to B in 6 hours. The car traveled half the distance and then stopped for 30 minutes due to an issue. If it is to arrive at B on schedule, the car should increase its speed by $\\qquad$ kilometers/hour for the second half of the journey.",
      "answer": "10"
    },
    {
      "question": "14. There are 40 red points, 30 blue points, and 20 green points marked on a circle. The circle is divided into 90 arcs, and each arc is labeled with a number based on the colors of its endpoints: a red-blue arc is labeled 1, a red-green arc is labeled 2, a blue-green arc is labeled 3, and arcs with endpoints of the same color are labeled 0. Find the minimum and maximum values of the sum of all these numbers.",
      "answer": "6140"
    },
    {
      "question": "[ Rectangles and squares. Properties and characteristics ]\n\nSide $A B$ of rectangle $A B C D$ is equal to 12, and side $A D$ is equal to 5. The diagonals of the rectangle intersect at point $E$. Find the ratio of the distance from point $E$ to the center of the circle inscribed in triangle $A E D$ to the distance from point $E$ to the center of the circle inscribed in triangle $D E C$.\n\n#",
      "answer": "\\frac{10}{3}"
    },
    {
      "question": "*end Let $S=\\{1,2, \\cdots, 2005\\}$. If any set of $n$ pairwise coprime numbers in $S$ contains at least one prime number, find the minimum value of $n$.",
      "answer": "16"
    },
    {
      "question": "8-1. Two rectangles $8 \\times 10$ and $12 \\times 9$ are overlaid as shown in the figure. The area of the black part is 37. What is the area of the gray part? If necessary, round the answer to 0.01 or write the answer as a common fraction.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7d113eccc6d3ed32fb40g-15.jpg?height=471&width=577&top_left_y=684&top_left_x=745)",
      "answer": "65"
    },
    {
      "question": "Example 3 Let $a$ be an integer such that $x^{2}-x+a$ divides $x^{13}+x+90$. Find the value of $a$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "2"
    },
    {
      "question": "..... ]\n\n[ Auxiliary area. The area helps to solve the problem ]\n[ Thales' theorem and the theorem of proportional segments ]\n\nIn a right triangle with legs of 6 and 8, a square is inscribed, sharing a right angle with the triangle. Find the side of the square.\n\n#",
      "answer": "\\frac{24}{7}"
    },
    {
      "question": "6. Find the minimum value of the expression $x^{2}+y^{2}+3 z^{2}+2 x y$ given that $x y+z=1$ and $x, y, z>0$.",
      "answer": "\\frac{8}{3}"
    },
    {
      "question": "## Task A-1.5.\n\nHow many numbers at least need to be removed from the set $\\{1,2,3, \\ldots, 2020\\}$ so that the resulting set does not contain the product of any two of its distinct elements?",
      "answer": "44"
    },
    {
      "question": "7.2. Vasya left the village of Vasilki and walked to the village of Romashki. At the same time, Roma left the village of Romashki for the village of Vasilki. The distance between the villages is 36 km, Vasya's speed is 5 km/h, and Roma's speed is 4 km/h. At the same time as Vasya, Dima left Vasilki on a bicycle at a speed equal to the sum of the boys' speeds and headed towards Roma. When he met Roma, he greeted him and immediately turned back. Meeting Vasya, he greeted him and immediately turned back. And so he rode back and forth between the boys until they met. How many kilometers did the cyclist travel during this time?",
      "answer": "36"
    },
    {
      "question": "Problem T-7. The numbers from 1 to $2013^{2}$ are written row by row into a table consisting of $2013 \\times 2013$ cells. Afterwards, all columns and all rows containing at least one of the perfect squares $1,4,9, \\ldots, 2013^{2}$ are simultaneously deleted.\n\nHow many cells remain?",
      "answer": "825423"
    },
    {
      "question": "10. Taking the vertex $C(0,1)$ of the ellipse $x^{2}+a^{2} y^{2}=a^{2}(a>1)$ as the right-angle vertex, construct an inscribed isosceles right-angled triangle $A B C$ in this ellipse. The maximum number of such triangles that can be constructed is $\\qquad$.",
      "answer": "3"
    },
    {
      "question": "1. In a computer program written in Turbo Pascal, the function $\\operatorname{Random}(x)$ is used, generating integer random numbers from 1 to $x$. What is the probability that when this function is executed, a number divisible by 5 will appear if $x=100?$",
      "answer": "0.2"
    },
    {
      "question": "11.43*. In a city, there are 10 streets parallel to each other, and 10 streets intersecting them at right angles. What is the minimum number of turns a closed bus route can have, passing through all intersections?",
      "answer": "20"
    },
    {
      "question": "6. In a perfectly competitive market, the demand function for a certain good is $\\mathrm{Q}_{\\mathrm{d}}(\\mathrm{p})=150-\\mathrm{p}$, and the supply function for this good is: $\\mathrm{Q}_{\\mathrm{s}}(\\mathrm{p})=3 \\mathrm{p}-10$. As a result of a sharp increase in the number of consumers of this good, under all other equal conditions, the demand for it increased by a factor of $\\alpha$ at every possible price level. As a result, the price of the good increased by $25 \\%$. Find the value of $\\alpha$. (8 points)",
      "answer": "1.4"
    },
    {
      "question": "Find all odd integers $n \\geqslant 1$ such that $n$ divides $3^{n}+1$.\n\n untranslated text remains unchanged.",
      "answer": "1"
    },
    {
      "question": "10. In $\\triangle A B C$, $A B=\\sqrt{2}, A C=\\sqrt{3}$, $\\angle B A C=30^{\\circ}$, and $P$ is any point in the plane of $\\triangle A B C$. Then the minimum value of $\\mu=\\overrightarrow{P A} \\cdot \\overrightarrow{P B}+\\overrightarrow{P B} \\cdot \\overrightarrow{P C}+\\overrightarrow{P C} \\cdot \\overrightarrow{P A}$ is . $\\qquad$",
      "answer": "\\frac{\\sqrt{2}}{2}-\\frac{5}{3}"
    },
    {
      "question": "Suppose that $A B C D$ is a square with side length 4 and that $0<k<4$. Let points $P, Q, R$, and $S$ be on $B C, C D, D A$, and $A P$, respectively, so that\n\n$$\n\\frac{B P}{P C}=\\frac{C Q}{Q D}=\\frac{D R}{R A}=\\frac{A S}{S P}=\\frac{k}{4-k}\n$$\n\nWhat is the value of $k$ which minimizes the area of quadrilateral $P Q R S$ ?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_bceb232fd49f7f0368beg-2.jpg?height=330&width=333&top_left_y=2082&top_left_x=1357)\n\n## PART B\n\nFor each question in Part B, your solution must be well-organized and contain words of explanation or justification. Marks are awarded for completeness, clarity, and style of presentation. A correct solution, poorly presented, will not earn full marks.",
      "answer": "\\frac{8}{3}"
    },
    {
      "question": "39. Find the smallest integer starting with the digit 7 that decreases threefold when this digit is moved to the end. Find all such numbers.",
      "answer": "7241379310344827586206896551"
    },
    {
      "question": "In a group of $n$ people, each one had a different ball. They performed a sequence of swaps; in each swap, two people swapped the ball they had at that moment. Each pair of people performed at least one swap. In the end each person had the ball he/she had at the start. Find the least possible number of swaps, if: $a$ ) $n=5$; b) $n=6$.",
      "answer": "16"
    },
    {
      "question": "6・65 Let positive integers $a, b$ be such that $15a + 16b$ and $16a - 15b$ are both squares of positive integers. Find the smallest value that the smaller of these two squares can take.\n(37th International Mathematical Olympiad, 1996)",
      "answer": "481^2"
    },
    {
      "question": "16. 7 (US MO 16) In the plane, there are three circles $C_{i}(i=1,2,3)$, where the diameter of $C_{1}$ is $A B=1$; $C_{2}$ is concentric with $C_{1}$, has a diameter of $k$, and satisfies $1<k<3$; $C_{3}$ has $A$ as its center and $2 k$ as its diameter ($k$ is a constant). Consider all line segments $X Y$, one end $X$ on $C_{2}$, the other end $Y$ on $C_{3}$, and $X Y$ contains point $B$. For what ratio $X B: B Y$ is the length of segment $X Y$ minimized?",
      "answer": "1"
    },
    {
      "question": "5. The teacher invited a group of children to sit around a round table. There were three times as many boys as girls. She walked around the table and observed pairs of children sitting next to each other. She found that the number of pairs where both children were of the same gender was twice the number of pairs where the children were of different genders. What is the minimum number of children sitting around the table?\n\n## 53rd Mathematical Competition for High School Students in Slovenia\n\nSelection Competition, April 1, 2009\n\n## Problems for 4th Year",
      "answer": "12"
    },
    {
      "question": "1. (mathematics) There are scales with two pans, 4 weights of 2 kg each, 3 weights of 3 kg each, and two weights of 5 kg each. In how many different ways can a 12 kg load be balanced on the scales, if the weights are allowed to be placed on both pans?",
      "answer": "7"
    },
    {
      "question": "3. (7 points) The number 20 is written on the board. In one move, it is allowed to either double the number or erase its last digit. Is it possible to get the number 25 in several moves?",
      "answer": "25"
    },
    {
      "question": "3.1. Find the sum of all two-digit numbers for each of which the sum of the squares of the digits is 37 more than the product of the same digits.",
      "answer": "231"
    },
    {
      "question": "[ $\\quad$ Number of divisors and their sum of a number [Fundamental Theorem of Arithmetic. Factorization into prime factors ]\n\na) Find the number $k$, which is divisible by 2 and 9 and has exactly 14 divisors (including 1 and $k$ ).\n\nb) Prove that if 14 is replaced by 15, the problem will have multiple solutions, and if 14 is replaced by 17, there will be no solutions at all.",
      "answer": "1458"
    },
    {
      "question": "[ Algebraic inequalities (miscellaneous).] [ Case analysis $]$\n\n$x, y>0$. Let $S$ denote the smallest of the numbers $x, 1 / y, y+1 / x$. What is the maximum value that $S$ can take?",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "208. Random variables $X$ and $Y$ are independent. Find the variance of the random variable $Z=3X+2Y$, given that $D(X)=5, D(Y)=6$.",
      "answer": "69"
    },
    {
      "question": "11. (20 points) Let 2018 real numbers $a_{1}, a_{2}, \\cdots, a_{2018}$ satisfy\n$$\n\\sum_{i=1}^{2018} a_{i}=0, \\sum_{i=1}^{2018} a_{i}^{2}=2018 .\n$$\n\nFind the maximum possible value of the minimum product of any two of these 2018 real numbers.",
      "answer": "-1"
    },
    {
      "question": "8. (10 points) Given $n!=1 \\times 2 \\times 3 \\times \\cdots \\times n$, what is the result of the expression $\\frac{(2014!-1)!\\times 2015!}{(2014!)!}$? $\\qquad$ .",
      "answer": "2015"
    },
    {
      "question": "8. Given $f(x)=\\sum_{k=1}^{2017} \\frac{\\cos k x}{\\cos ^{k} x}$. Then $f\\left(\\frac{\\pi}{2018}\\right)=$",
      "answer": "-1"
    },
    {
      "question": "Problem 8.2. In the chat of students from one of the schools, a vote was held: \"On which day to hold the disco: October 22 or October 29?\"\n\nThe graph shows how the votes were distributed an hour after the start of the voting.\n\nThen, 80 more people participated in the voting, voting only for October 22. After that, the voting ended. The final distribution of votes is also shown on the graph.\n\nHow many people participated in the voting?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0323bbf84409a1adeb34g-24.jpg?height=494&width=747&top_left_y=1295&top_left_x=339)",
      "answer": "260"
    },
    {
      "question": "A caravan of camels, 1 km long, moves uniformly across the desert. A rider rode from the end of the caravan to the beginning and returned to the end of the caravan. During this time, the caravan traveled 1 km. What distance did the rider travel if his speed was constant?",
      "answer": "1+\\sqrt{2}"
    },
    {
      "question": "5. Determine the unknown number $x$ and explain.\n\n| 9 | 22 | 13 | 16 |\n| :---: | :---: | :---: | :---: |\n| 144 | 176 | 143 | 192 |\n| 16 | 8 | 11 | $x$ |\n\nTasks worth 10 points",
      "answer": "12"
    },
    {
      "question": "2. (2 points) Find the minimum value of the expression $x^{2}-6 x \\sin y-9 \\cos ^{2} y$.",
      "answer": "-9"
    },
    {
      "question": "Problem 4. The bases $AB$ and $CD$ of trapezoid $ABCD$ are equal to 65 and 31, respectively, and its diagonals are perpendicular to each other. Find the scalar product of vectors $\\overrightarrow{AD}$ and $\\overrightarrow{BC}$.",
      "answer": "2015"
    },
    {
      "question": "[\n\nA jeweler was commissioned to make a gold ring of width $h$, having the shape of a body bounded by the surface of a sphere with center $O$ and the surface of a cylinder of radius $r$, the axis of which passes through point $O$. The master made such a ring, but chose $r$ too small. How much gold will he have to add if $r$ needs to be increased by $k$ times, while keeping the width $h$ the same?\n\n#",
      "answer": "\\pi/6^{3}"
    },
    {
      "question": "Six bags contain 18, 19, 21, 23, 25, and 34 coins respectively, one of which is entirely filled with fake coins, while the other five bags do not contain any fake coins. Xiao Cong took three bags, and Xiao Min took the other two bags, leaving the bag with the fake coins. If the total number of coins Xiao Cong got is twice that of Xiao Min, then how many fake coins are there?",
      "answer": "23"
    },
    {
      "question": "| $A$ | $B$ | $C$ | $D$ | $E$ |\n| :--- | :--- | :--- | :--- | :--- |\n| $E$ | $D$ | $C$ | $B$ | $A$ |\n| $F$ | $F$ | $F$ | $F$ | $F$ |$+$\n\nA fragment of a conversation on the beach:\n\nFeri: Don't rush me now, I need to write digits here instead of the letters (Fig. 1), so that the addition is correct, the same digit for the same letter, and different digits for different letters. - Gyurka (after looking at the problem): What's the fuss about this? I can write down 77 solutions for you. - F: I doubt that. - Gy: I'll buy you as many ice creams as the number of solutions you miss, just come and swim now. - F: I'll go, but is the bet on? - How did the bet turn out?",
      "answer": "32"
    },
    {
      "question": "1. Let $f(x)$ be a function defined on $R$, for any real number $x$ we have $f(x+3) \\cdot f(x-4)=-1$. Also, when $0 \\leq x<7$, $f(x)=\\log _{2}(9-x)$, then the value of $f(-100)$ is . $\\qquad$",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "3. A three-stage launch vehicle consists of stages in the form of cylinders. All these cylinders are similar to each other. The length of the middle stage is two times less than the sum of the lengths of the first and third stages. In the fueled state, the mass of the middle stage is $13 / 6$ times less than the total mass of the fueled first and third stages. Find the ratio of the lengths of the first and third stages. The masses of the engines, instruments, and the stages' shells can be neglected.",
      "answer": "\\frac{7}{5}"
    },
    {
      "question": "# 7. Variant 1\n\nUnit cubes were used to assemble a large parallelepiped with sides greater than 4. Two cubes will be called adjacent if they touch by faces. Thus, one cube can have up to 6 neighbors. It is known that the number of cubes that have exactly 6 neighbors is 836. Find the number of cubes that have no more than four neighbors.",
      "answer": "144"
    },
    {
      "question": "2. The general solution of the equation $\\cos \\frac{x}{4}-\\cos x$ is ( ), within $(0,24 \\pi)$, there are ( ) distinct solutions.",
      "answer": "20"
    },
    {
      "question": "14. Four problems were attempted by 100 contestants in a Mathematics competition. The first problem was solved by 90 contestants, the second by 85 contestants, the third by 80 contestants and the fourth by 75 contestants. What is the smallest possible number of contestants who solved all four problems?",
      "answer": "30"
    },
    {
      "question": "7. In a $4 \\times 4$ grid of 16 small squares, fill in 2 $a$s and 2 $b$s, with at most one letter in each small square. If the same letter is neither in the same row nor in the same column, there are $\\qquad$ ways to arrange them.",
      "answer": "3960"
    },
    {
      "question": "5. Dima has 25 identical bricks of size $5 \\times 14 \\times 17$. Dima wants to build a tower from all his bricks, each time adding one more brick on top (each new brick adds 5, 14, or 17 to the current height of the tower). We will call a number $n$ constructible if Dima can build a tower of height exactly $n$. How many constructible numbers exist?",
      "answer": "98"
    },
    {
      "question": "A $5 \\times 5$ chessboard consists of unit squares, 7 of which are red and 18 are blue. Two of the red squares are located on the edge of the board. The segments that separate two adjacent red squares are also colored red. The segments that separate two adjacent blue squares are colored blue. All other segments, including the edges of the board, are black. In this way, a total of 35 black segments are formed. How many red segments are there?",
      "answer": "5"
    },
    {
      "question": "4. Dina has a calculating machine, labelled $f$, that takes one number as input and calculates an output. The machine $f$ calculates its output by multiplying its input by 2 and then subtracting 3. For example, if Dina inputs 2.16 into $f$, the output is 1.32. If Dina inputs a number $x$ into $f$, she gets a first output which she then inputs back into $f$ to obtain a second output, which is -35 . What is the value of $x$ ?",
      "answer": "-\\frac{13}{2}"
    },
    {
      "question": "G6.1 If $\\log _{2} a-2 \\log _{a} 2=1$, find $a$.",
      "answer": "4"
    },
    {
      "question": "The center of a circle touching the legs $A C$ and $B C$ of a right triangle $A B C$ lies on the hypotenuse $A B$. Find the radius of the circle, if it is six times smaller than the sum of the legs, and the area of triangle $A B C$ is 27.",
      "answer": "3"
    },
    {
      "question": "Example 4.34. Compute the integral $\\int_{0}^{1} \\frac{\\sin t}{t} d t$ with accuracy $\\varepsilon=10^{-4}$.",
      "answer": "0.9444"
    },
    {
      "question": "Consider 2018 lines in the plane, no two of which are parallel and no three of which are concurrent. Let $E$ be the set of their intersection points. We want to assign a color to each point in $E$ such that any two points on the same line, whose segment connecting them contains no other point of $E$, are of different colors.\n\nHow many colors are needed at a minimum to achieve such a coloring?",
      "answer": "3"
    },
    {
      "question": "【Question 13】There are 50 students in the third grade, all of whom choose to subscribe to one, two, or all three of the three magazines A, B, and C. Then at least $\\qquad$ students have the same subscription type.",
      "answer": "8"
    },
    {
      "question": "XXVI - I - Problem 11\n\nA ship is moving at a constant speed along a straight line with an east-west direction. Every $ T $ minutes, the direction of movement is randomly chosen: with probability $ p $, the ship moves in the eastern direction for the next $ T $ minutes, and with probability $ q= 1-p $, it moves in the western direction. At a point outside the line, there is a submarine whose task is to torpedo the ship. The travel time of the torpedo from the point of firing to any point on the ship's track is $ 2T $. The captain of the submarine knows the value of $ p $ and aims to maximize the probability of hitting the ship. How should $ p $ be chosen to minimize the probability of the ship being torpedoed?",
      "answer": "\\frac{4}{9}"
    },
    {
      "question": "1. Find all real numbers $x$ for which\n\n$$\n|||| x|-2|-20|-200|=2007\n$$",
      "answer": "2229-2229"
    },
    {
      "question": "9. (6 points) If $\\overline{\\mathrm{ab}} \\times 65=\\overline{48 \\mathrm{ab}}$, then $\\overline{\\mathrm{ab}}=$",
      "answer": "75"
    },
    {
      "question": "[Perpendicular Planes]\n\nThe base of the pyramid $M B K H E$ is a convex quadrilateral $M B K H$, where the angle at vertex $M$ is $\\pi / 2$, the angle formed by the diagonal $B H$ and the edge $B K$ is $\\pi / 4$, the length of the edge $M B$ is 1. The area of triangle $B K H$ is twice the area of triangle $M B H$. The sum of the lengths of edges $B E$ and $H E$ is $\\sqrt{3}$. The volume of the pyramid is 1/4. Find the radius of the sphere with the largest volume among all spheres that can fit inside the pyramid $M B K H E$.",
      "answer": "2-\\sqrt{2}"
    },
    {
      "question": "2. You have a list of real numbers, whose sum is 40 . If you replace every number $x$ on the list by $1-x$, the sum of the new numbers will be 20 . If instead you had replaced every number $x$ by $1+x$, what would the sum then be?",
      "answer": "100"
    },
    {
      "question": "7. Given real numbers $a, b, c$ satisfy\n$$\n\\begin{array}{l}\na b c \\neq 0, a+b+c=a^{2}+b^{2}+c^{2}=2 . \\\\\n\\text { Then } \\frac{(1-a)^{2}}{b c}+\\frac{(1-b)^{2}}{c a}+\\frac{(1-c)^{2}}{a b}=\n\\end{array}\n$$\n$\\qquad$",
      "answer": "3"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 1}\\left(\\frac{2 x-1}{x}\\right)^{\\frac{\\ln (3+2 x)}{\\ln (2-x)}}$",
      "answer": "\\frac{1}{5}"
    },
    {
      "question": "## Task 24/85\n\nLet there be a rectangle $A B C D$ with an internal point $P$. Determine $P A$ in terms of $P B, P C$, and $P D$.\n\nWhat value is obtained for $P B=33 \\mathrm{LE}, P C=28 \\mathrm{LE}, P D=41 \\mathrm{LE}$?",
      "answer": "\\sqrt{1996}"
    },
    {
      "question": "## Task 2 - 040712\n\nA freight train traveled $35 \\frac{3}{4} \\mathrm{~km}$ in the first hour and in the following $2 \\frac{1}{2}$ hours, it covered an additional 92.7 $\\mathrm{km}$. For the return trip on the same route, it took three hours and 12 minutes. Calculate the average speed for the entire trip! Round to one decimal place!",
      "answer": "38.3"
    },
    {
      "question": "[Example 2.2.1] Simplify: $M=\\frac{2}{\\sqrt{4-3 \\sqrt[4]{5}+2 \\sqrt[4]{25}-\\sqrt[4]{125}}}$.",
      "answer": "\\sqrt[4]{5}+1"
    },
    {
      "question": "A two-digit number has 3 as its tens digit, and when a digit is inserted in the middle to form a three-digit number, this three-digit number is exactly 9 times the original two-digit number. The new three-digit number is $\\qquad$",
      "answer": "315"
    },
    {
      "question": "Problem 8.3. In the grove, there are 140 chameleons - blue and red. One day, several blue chameleons changed their color to red. As a result, the number of blue chameleons decreased by 5 times, and the number of red chameleons increased by 3 times. How many chameleons changed their color?",
      "answer": "80"
    },
    {
      "question": "8. (10 points) On the right is an equation, where 9 Chinese characters represent the numbers 1 to 9, and different characters represent different numbers. The maximum possible value of the equation is $\\qquad$.\n\nHope $\\times$ Longing + Tree $\\times$ Green + Sky $\\times$ Blue",
      "answer": "8569"
    },
    {
      "question": "$4 \\cdot 64$ Find the product of the real roots of the equation $x^{2}+18 x+30=2 \\sqrt{x^{2}+18 x+45}$.",
      "answer": "20"
    },
    {
      "question": "6. For any positive integers $x, y, z$, if $x \\mid y^{3}$, $y \\mid z^{3}$, $z \\mid x^{3}$, and $x y z \\mid (x+y+z)^{n}$, find the smallest positive integer $n$ that satisfies the condition.",
      "answer": "13"
    },
    {
      "question": "21st BMO 1985 Problem 3 Let { x } denote the nearest integer to x, so that x - 1/2 ≤ { x } < x + 1/2. Define the sequence u 1 , u 2 , u 3 , ... by u 1 = 1. u n+1 = u n + { u n √2 }. So, for example, u 2 = 2, u 3 = 5, u 4 = 12. Find the units digit of u 1985 .",
      "answer": "9"
    },
    {
      "question": "2. Vasya and Petya simultaneously ran out from the starting point of a circular running track and ran in opposite directions at constant speeds. At some point, they met. Vasya ran a full circle and, continuing to run in the same direction, reached the place of their first meeting at the moment when Petya ran a full circle. Find the ratio of Vasya's and Petya's speeds.",
      "answer": "\\frac{1+\\sqrt{5}}{2}"
    },
    {
      "question": "5. Given that $x, y, z$ are 3 real numbers greater than or equal to 1, then\n$$\n\\left(\\frac{\\sqrt{x^{2}(y-1)^{2}+y^{2}}}{x y}+\\frac{\\sqrt{y^{2}(z-1)^{2}+z^{2}}}{y z}+\\frac{\\sqrt{z^{2}(x-1)^{2}+x^{2}}}{z x}\\right)^{2}\n$$\n\nthe sum of the numerator and denominator of the minimum value written as a simplified fraction is $\\qquad$ .",
      "answer": "11"
    },
    {
      "question": "6. In a rectangle $A B C D$ (including the boundary) with an area of 1, there are 5 points, among which no three points are collinear. Find the minimum number of triangles, with these 5 points as vertices, whose area is not greater than $\\frac{1}{4}$.",
      "answer": "2"
    },
    {
      "question": "3. Point $M$ is the midpoint of side $AB$ of triangle $ABC$. A circle $\\omega_{1}$ is drawn through points $A$ and $M$, tangent to line $AC$, and a circle $\\omega_{2}$ is drawn through points $B$ and $M$, tangent to line $BC$. Circles $\\omega_{1}$ and $\\omega_{2}$ intersect again at point $D$. Point $E$ lies inside triangle $ABC$ and is symmetric to point $D$ with respect to line $AB$. Find the angle $CEM$.",
      "answer": "180"
    },
    {
      "question": "In a board game played with a die, our pawn is four spaces away from the finish. If we roll at least a four, we reach the finish. If we roll a three, we will definitely be in with the next roll.\n\nWhat is the probability that we will reach the finish in only two more rolls?",
      "answer": "\\frac{1}{12}"
    },
    {
      "question": "15. In a $1 \\times 5$ grid, there are four chips, each with one side white and the other side black. Each operation allows you to select a chip to jump over one, two, or three chips to an empty position, but not to move it directly. The chips that are jumped over must be flipped, but the jumping chip remains the same. Now, it is desired to transform the situation on the left into the situation on the right in six operations. If the positions of the chips before they jump are recorded in sequence, a six-digit number can be formed. Please provide a possible six-digit number that can accomplish the task. $\\qquad$ (Provide one possible answer only).",
      "answer": "251425"
    },
    {
      "question": "62. Use the six digits $2,3,4,5,6,7$ to form two three-digit numbers, so that the greatest common divisor (GCD) of these two three-digit numbers and 540 is as large as possible. The largest possible GCD is $\\qquad$.",
      "answer": "108"
    },
    {
      "question": "KOVSHIN=BOTTLE+GLASS; $\\quad$ TWO KOVSHINS=SEVEN GLASSES; BOTTLE=CUP+TWO GLASSES; $\\quad$ BOTTLE=how many CUPS?",
      "answer": "5"
    },
    {
      "question": "Problem 8.3. How many lines exist that pass through the point $(0,2019)$ and intersect the parabola $y=x^{2}$ at two points with integer coordinates on the $y$-axis?\n\nAnswer: 9.",
      "answer": "9"
    },
    {
      "question": "8. (10 points) In the equation, $\\overline{\\mathrm{ABCD}}+\\overline{\\mathrm{EFG}}=2020$, different letters represent different digits. Therefore, $A+B+C+D+E+F+G=$ $\\qquad$ .",
      "answer": "31"
    },
    {
      "question": "Problem 6.6. Ksyusha runs twice as fast as she walks (both speeds are constant).\n\nOn Tuesday, when she left home for school, she first walked, and then, when she realized she was running late, she started running. The distance she walked was twice the distance she ran. In the end, it took her exactly 30 minutes to get from home to school.\n\nOn Wednesday, Ksyusha left home even later, so she had to run twice the distance she walked. How many minutes did it take her to get from home to school on Wednesday?",
      "answer": "24"
    },
    {
      "question": "Example 2-55 In the permutations of the numbers $1,2,3, \\cdots, 9$, find the number of derangements where even numbers are in their original positions, and the rest are not in their original positions.",
      "answer": "44"
    },
    {
      "question": "Biyankov Yu.A.\n\nIn an isosceles triangle \\(ABC\\), a point \\(M\\) is marked on the lateral side \\(BC\\) such that the segment \\(MC\\) is equal to the height of the triangle dropped to this side, and a point \\(K\\) is marked on the lateral side \\(AB\\) such that the angle \\(KMC\\) is a right angle. Find the angle \\(ACK\\).",
      "answer": "45"
    },
    {
      "question": "3. Calculate:\n\n$$\n\\int_{0}^{\\frac{\\pi}{2}} \\frac{\\sin ^{2} x+\\sin x}{\\sin x+\\cos x+1} d x\n$$",
      "answer": "\\frac{\\pi}{4}"
    },
    {
      "question": "We are creating an infinite table of integers with the following procedure: The first column contains the terms of an arithmetic sequence in ascending order, whose first term is 4 and the common difference is 3. For every $k \\geq 1$, the $k$-th row is the arithmetic sequence whose common difference is $2k+1$. Where can the 1994 and the $1995$ be found in this table?",
      "answer": "1995"
    }
  ],
  "test": [
    {
      "question": "The graph of $x^{4}=x^{2} y^{2}$ is a union of $n$ different lines. What is the value of $n$?",
      "answer": "3"
    },
    {
      "question": "In how many ways can the positive integers from 1 to 100 be arranged in a circle such that the sum of every two integers placed opposite each other is the same? (Arrangements that are rotations of each other count as the same.)",
      "answer": "49!\\cdot 2^{49}"
    },
    {
      "question": "Let $A=H_{1}, B=H_{6}+1$. A real number $x$ is chosen randomly and uniformly in the interval $[A, B]$. Find the probability that $x^{2}>x^{3}>x$.",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Suppose Harvard Yard is a $17 \\times 17$ square. There are 14 dorms located on the perimeter of the Yard. If $s$ is the minimum distance between two dorms, the maximum possible value of $s$ can be expressed as $a-\\sqrt{b}$ where $a, b$ are positive integers. Compute $100a+b$.",
      "answer": "602"
    },
    {
      "question": "Find the smallest positive integer $n$ such that, if there are initially $n+1$ townspeople and $n$ goons, then the probability the townspeople win is less than $1\\%$.",
      "answer": "6"
    },
    {
      "question": "How many integers $n>1$ are there such that $n$ divides $x^{13}-x$ for every positive integer $x$?",
      "answer": "31"
    },
    {
      "question": "A domino is a 1-by-2 or 2-by-1 rectangle. A domino tiling of a region of the plane is a way of covering it (and only it) completely by nonoverlapping dominoes. For instance, there is one domino tiling of a 2-by-1 rectangle and there are 2 tilings of a 2-by-2 rectangle (one consisting of two horizontal dominoes and one consisting of two vertical dominoes). How many domino tilings are there of a 2-by-10 rectangle?",
      "answer": "89"
    },
    {
      "question": "Consider an infinite grid of unit squares. An $n$-omino is a subset of $n$ squares that is connected. Below are depicted examples of 8 -ominoes. Two $n$-ominoes are considered equivalent if one can be obtained from the other by translations and rotations. What is the number of distinct 15 -ominoes? Your score will be equal to $25-13|\\ln (A)-\\ln (C)|$.",
      "answer": "3426576"
    },
    {
      "question": "What is the quantity equivalent to '2% of 1'?",
      "answer": "\\frac{2}{100}"
    },
    {
      "question": "Suppose that there are real numbers $a, b, c \\geq 1$ and that there are positive reals $x, y, z$ such that $$\\begin{aligned} a^{x}+b^{y}+c^{z} & =4 \\\\ x a^{x}+y b^{y}+z c^{z} & =6 \\\\ x^{2} a^{x}+y^{2} b^{y}+z^{2} c^{z} & =9 \\end{aligned}$$ What is the maximum possible value of $c$ ?",
      "answer": "\\sqrt[3]{4}"
    },
    {
      "question": "For all $n \\geq 1$, let \\[ a_n = \\sum_{k=1}^{n-1} \\frac{\\sin \\left( \\frac{(2k-1)\\pi}{2n} \\right)}{\\cos^2 \\left( \\frac{(k-1)\\pi}{2n} \\right) \\cos^2 \\left( \\frac{k\\pi}{2n} \\right)}. \\] Determine \\[ \\lim_{n \\to \\infty} \\frac{a_n}{n^3}. \\]",
      "answer": "\\frac{8}{\\pi^3}"
    },
    {
      "question": "Ken is the best sugar cube retailer in the nation. Trevor, who loves sugar, is coming over to make an order. Ken knows Trevor cannot afford more than 127 sugar cubes, but might ask for any number of cubes less than or equal to that. Ken prepares seven cups of cubes, with which he can satisfy any order Trevor might make. How many cubes are in the cup with the most sugar?",
      "answer": "64"
    },
    {
      "question": "Emilia writes down the numbers $5, x$, and 9. Valentin calculates the mean (average) of each pair of these numbers and obtains 7, 10, and 12. What is the value of $x$?",
      "answer": "15"
    },
    {
      "question": "Define a sequence of polynomials as follows: let $a_{1}=3 x^{2}-x$, let $a_{2}=3 x^{2}-7 x+3$, and for $n \\geq 1$, let $a_{n+2}=\\frac{5}{2} a_{n+1}-a_{n}$. As $n$ tends to infinity, what is the limit of the sum of the roots of $a_{n}$ ?",
      "answer": "\\frac{13}{3}"
    },
    {
      "question": "Three integers from the list $1,2,4,8,16,20$ have a product of 80. What is the sum of these three integers?",
      "answer": "25"
    },
    {
      "question": "Let $f(n)$ be the number of distinct digits of $n$ when written in base 10. Compute the sum of $f(n)$ as $n$ ranges over all positive 2019-digit integers.",
      "answer": "9\\left(10^{2019}-9^{2019}\\right)"
    },
    {
      "question": "A fat coin is one which, when tossed, has a $2 / 5$ probability of being heads, $2 / 5$ of being tails, and $1 / 5$ of landing on its edge. Mr. Fat starts at 0 on the real line. Every minute, he tosses a fat coin. If it's heads, he moves left, decreasing his coordinate by 1; if it's tails, he moves right, increasing his coordinate by 1. If the coin lands on its edge, he moves back to 0. If Mr. Fat does this ad infinitum, what fraction of his time will he spend at 0?",
      "answer": "\\[\n\\frac{1}{3}\n\\]"
    },
    {
      "question": "If \\( 8 + 6 = n + 8 \\), what is the value of \\( n \\)?",
      "answer": "6"
    },
    {
      "question": "Let \\(ABCD\\) be a square of side length 2. Let points \\(X, Y\\), and \\(Z\\) be constructed inside \\(ABCD\\) such that \\(ABX, BCY\\), and \\(CDZ\\) are equilateral triangles. Let point \\(W\\) be outside \\(ABCD\\) such that triangle \\(DAW\\) is equilateral. Let the area of quadrilateral \\(WXYZ\\) be \\(a+\\sqrt{b}\\), where \\(a\\) and \\(b\\) are integers. Find \\(a+b\\).",
      "answer": "10"
    },
    {
      "question": "Given that $a, b, c$ are integers with $a b c=60$, and that complex number $\\omega \\neq 1$ satisfies $\\omega^{3}=1$, find the minimum possible value of $\\left|a+b \\omega+c \\omega^{2}\\right|$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "Fifteen freshmen are sitting in a circle around a table, but the course assistant (who remains standing) has made only six copies of today's handout. No freshman should get more than one handout, and any freshman who does not get one should be able to read a neighbor's. If the freshmen are distinguishable but the handouts are not, how many ways are there to distribute the six handouts subject to the above conditions?",
      "answer": "125"
    },
    {
      "question": "Let $\\triangle A B C$ be a triangle inscribed in a unit circle with center $O$. Let $I$ be the incenter of $\\triangle A B C$, and let $D$ be the intersection of $B C$ and the angle bisector of $\\angle B A C$. Suppose that the circumcircle of $\\triangle A D O$ intersects $B C$ again at a point $E$ such that $E$ lies on $I O$. If $\\cos A=\\frac{12}{13}$, find the area of $\\triangle A B C$.",
      "answer": "\\frac{15}{169}"
    },
    {
      "question": "$A B C$ is a triangle with points $E, F$ on sides $A C, A B$, respectively. Suppose that $B E, C F$ intersect at $X$. It is given that $A F / F B=(A E / E C)^{2}$ and that $X$ is the midpoint of $B E$. Find the ratio $C X / X F$.",
      "answer": "\\sqrt{5}"
    },
    {
      "question": "Let $ABCDEF$ be a regular hexagon. Let $P$ be the circle inscribed in $\\triangle BDF$. Find the ratio of the area of circle $P$ to the area of rectangle $ABDE$.",
      "answer": "\\frac{\\pi \\sqrt{3}}{12}"
    },
    {
      "question": "The side lengths of a triangle are distinct positive integers. One of the side lengths is a multiple of 42, and another is a multiple of 72. What is the minimum possible length of the third side?",
      "answer": "7"
    },
    {
      "question": "A regular octahedron has a side length of 1. What is the distance between two opposite faces?",
      "answer": "\\sqrt{6} / 3"
    },
    {
      "question": "Let $f(x)=x^{2}+a x+b$ and $g(x)=x^{2}+c x+d$ be two distinct real polynomials such that the $x$-coordinate of the vertex of $f$ is a root of $g$, the $x$-coordinate of the vertex of $g$ is a root of $f$ and both $f$ and $g$ have the same minimum value. If the graphs of the two polynomials intersect at the point (2012, - 2012), what is the value of $a+c$ ?",
      "answer": "-8048"
    },
    {
      "question": "There are 5 students on a team for a math competition. The math competition has 5 subject tests. Each student on the team must choose 2 distinct tests, and each test must be taken by exactly two people. In how many ways can this be done?",
      "answer": "2040"
    },
    {
      "question": "There are $F$ fractions $\\frac{m}{n}$ with the properties: $m$ and $n$ are positive integers with $m<n$, $\\frac{m}{n}$ is in lowest terms, $n$ is not divisible by the square of any integer larger than 1, and the shortest sequence of consecutive digits that repeats consecutively and indefinitely in the decimal equivalent of $\\frac{m}{n}$ has length 6. We define $G=F+p$, where the integer $F$ has $p$ digits. What is the sum of the squares of the digits of $G$?",
      "answer": "181"
    },
    {
      "question": "A counter begins at 0 . Then, every second, the counter either increases by 1 or resets back to 0 with equal probability. The expected value of the counter after ten seconds can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "103324"
    },
    {
      "question": "Let $f(n)$ and $g(n)$ be polynomials of degree 2014 such that $f(n)+(-1)^{n} g(n)=2^{n}$ for $n=1,2, \\ldots, 4030$. Find the coefficient of $x^{2014}$ in $g(x)$.",
      "answer": "\\frac{3^{2014}}{2^{2014} \\cdot 2014!}"
    },
    {
      "question": "Evaluate $1201201_{-4}$.",
      "answer": "2017"
    },
    {
      "question": "The real numbers $x, y$ and $z$ satisfy the three equations $x+y=7$, $xz=-180$, and $(x+y+z)^{2}=4$. If $S$ is the sum of the two possible values of $y$, what is $-S$?",
      "answer": "42"
    },
    {
      "question": "Calculate the value of the expression $\\left(2 \\times \\frac{1}{3}\\right) \\times \\left(3 \\times \\frac{1}{2}\\right)$.",
      "answer": "1"
    },
    {
      "question": "A collection of $n$ squares on the plane is called tri-connected if the following criteria are satisfied: (i) All the squares are congruent. (ii) If two squares have a point $P$ in common, then $P$ is a vertex of each of the squares. (iii) Each square touches exactly three other squares. How many positive integers $n$ are there with $2018 \\leq n \\leq 3018$, such that there exists a collection of $n$ squares that is tri-connected?",
      "answer": "501"
    },
    {
      "question": "How many of the 20 perfect squares $1^{2}, 2^{2}, 3^{2}, \\ldots, 19^{2}, 20^{2}$ are divisible by 9?",
      "answer": "6"
    },
    {
      "question": "A class of 10 students took a math test. Each problem was solved by exactly 7 of the students. If the first nine students each solved 4 problems, how many problems did the tenth student solve?",
      "answer": "6"
    },
    {
      "question": "Let $A_{10}$ denote the answer to problem 10. Two circles lie in the plane; denote the lengths of the internal and external tangents between these two circles by $x$ and $y$, respectively. Given that the product of the radii of these two circles is $15 / 2$, and that the distance between their centers is $A_{10}$, determine $y^{2}-x^{2}$.",
      "answer": "30"
    },
    {
      "question": "Given a point $p$ and a line segment $l$, let $d(p, l)$ be the distance between them. Let $A, B$, and $C$ be points in the plane such that $A B=6, B C=8, A C=10$. What is the area of the region in the $(x, y)$-plane formed by the ordered pairs $(x, y)$ such that there exists a point $P$ inside triangle $A B C$ with $d(P, A B)+x=d(P, B C)+y=d(P, A C)$?",
      "answer": "\\frac{288}{5}"
    },
    {
      "question": "For the sequence of numbers $n_{1}, n_{2}, n_{3}, \\ldots$, the relation $n_{i}=2 n_{i-1}+a$ holds for all $i>1$. If $n_{2}=5$ and $n_{8}=257$, what is $n_{5}$ ?",
      "answer": "33"
    },
    {
      "question": "When $(3 + 2x + x^{2})(1 + mx + m^{2}x^{2})$ is expanded and fully simplified, the coefficient of $x^{2}$ is equal to 1. What is the sum of all possible values of $m$?",
      "answer": "-\\frac{2}{3}"
    },
    {
      "question": "Let $ABCD$ be a regular tetrahedron, and let $O$ be the centroid of triangle $BCD$. Consider the point $P$ on $AO$ such that $P$ minimizes $PA+2(PB+PC+PD)$. Find $\\sin \\angle PBO$.",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "Reimu and Sanae play a game using 4 fair coins. Initially both sides of each coin are white. Starting with Reimu, they take turns to color one of the white sides either red or green. After all sides are colored, the 4 coins are tossed. If there are more red sides showing up, then Reimu wins, and if there are more green sides showing up, then Sanae wins. However, if there is an equal number of red sides and green sides, then neither of them wins. Given that both of them play optimally to maximize the probability of winning, what is the probability that Reimu wins?",
      "answer": "\\frac{5}{16}"
    },
    {
      "question": "When a single number is added to each member of the sequence 20, 50, 100, the sequence becomes expressible as $x, a x, a^{2} x$. Find $a$.",
      "answer": "\\frac{5}{3}"
    },
    {
      "question": "The integers $1,2, \\ldots, 64$ are written in the squares of a $8 \\times 8$ chess board, such that for each $1 \\leq i<64$, the numbers $i$ and $i+1$ are in squares that share an edge. What is the largest possible sum that can appear along one of the diagonals?",
      "answer": "432"
    },
    {
      "question": "Let a sequence $\\left\\{a_{n}\\right\\}_{n=0}^{\\infty}$ be defined by $a_{0}=\\sqrt{2}, a_{1}=2$, and $a_{n+1}=a_{n} a_{n-1}^{2}$ for $n \\geq 1$. The sequence of remainders when $a_{0}, a_{1}, a_{2}, \\cdots$ are divided by 2014 is eventually periodic with some minimal period $p$ (meaning that $a_{m}=a_{m+p}$ for all sufficiently large integers $m$, and $p$ is the smallest such positive integer). Find $p$.",
      "answer": "12"
    },
    {
      "question": "Fran writes the numbers \\(1,2,3, \\ldots, 20\\) on a chalkboard. Then she erases all the numbers by making a series of moves; in each move, she chooses a number \\(n\\) uniformly at random from the set of all numbers still on the chalkboard, and then erases all of the divisors of \\(n\\) that are still on the chalkboard (including \\(n\\) itself). What is the expected number of moves that Fran must make to erase all the numbers?",
      "answer": "\\frac{131}{10}"
    },
    {
      "question": "Find the number of subsets $S$ of $\\{1,2, \\ldots 63\\}$ the sum of whose elements is 2008.",
      "answer": "66"
    },
    {
      "question": "Digits are placed in the two boxes of $2 \\square \\square$, with one digit in each box, to create a three-digit positive integer. In how many ways can this be done so that the three-digit positive integer is larger than 217?",
      "answer": "82"
    },
    {
      "question": "For each positive integer $n$, let $k(n)$ be the number of ones in the binary representation of $2023 \\cdot n$. What is the minimum value of $k(n)$?",
      "answer": "3"
    },
    {
      "question": "Ash and Gary independently come up with their own lineups of 15 fire, grass, and water monsters. Then, the first monster of both lineups will fight, with fire beating grass, grass beating water, and water beating fire. The defeated monster is then substituted with the next one from their team's lineup; if there is a draw, both monsters get defeated. Gary completes his lineup randomly, with each monster being equally likely to be any of the three types. Without seeing Gary's lineup, Ash chooses a lineup that maximizes the probability $p$ that his monsters are the last ones standing. Compute $p$.",
      "answer": "1-\\frac{2^{15}}{3^{15}}"
    },
    {
      "question": "The operation $\\nabla$ is defined by $g \\nabla h=g^{2}-h^{2}$. If $g>0$ and $g \\nabla 6=45$, what is the value of $g$?",
      "answer": "9"
    },
    {
      "question": "Find the smallest integer $k \\geq 2$ such that for every partition of the set $\\{2, 3,\\hdots, k\\}$ into two parts, at least one of these parts contains (not necessarily distinct) numbers $a$, $b$ and $c$ with $ab = c$.",
      "answer": "32"
    },
    {
      "question": "Harriet ran a 1000 m course in 380 seconds. She ran the first 720 m of the course at a constant speed of 3 m/s. What was her speed for the remaining part of the course?",
      "answer": "2"
    },
    {
      "question": "An acute triangle $ABC$ is inscribed in a circle of radius 1 with centre $O;$ all the angles of $ABC$ are greater than $45^\\circ.$ \n$B_{1}$ is the foot of perpendicular from $B$ to $CO,$ $B_{2}$ is the foot of perpendicular from $B_{1}$ to $AC.$ \nSimilarly, $C_{1}$ is the foot of perpendicular from $C$ to $BO,$ $C_{2}$ is the foot of perpendicular from $C_{1}$ to $AB.$ \nThe lines $B_{1}B_{2}$ and $C_{1}C_{2}$ intersect at $A_{3}.$ The points $B_{3}$ and $C_{3}$ are defined in the same way. \nFind the circumradius of triangle $A_{3}B_{3}C_{3}.$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Each of given $100$ numbers was increased by $1$. Then each number was increased by $1$ once more. Given that the fi\frst time the sum of the squares of the numbers was not changed \ffind how this sum was changed the second time.",
      "answer": "200"
    },
    {
      "question": "An ordered pair $(a, b)$ of positive integers is called spicy if $\\operatorname{gcd}(a+b, ab+1)=1$. Compute the probability that both $(99, n)$ and $(101, n)$ are spicy when $n$ is chosen from $\\{1,2, \\ldots, 2024\\}$ uniformly at random.",
      "answer": "\\frac{96}{595}"
    },
    {
      "question": "Determine the smallest positive real number $r$ such that there exist differentiable functions $f\\colon \\mathbb{R} \\to \\mathbb{R}$ and $g\\colon \\mathbb{R} \\to \\mathbb{R}$ satisfying \\begin{enumerate} \\item[(a)] $f(0) > 0$, \\item[(b)] $g(0) = 0$, \\item[(c)] $|f'(x)| \\leq |g(x)|$ for all $x$, \\item[(d)] $|g'(x)| \\leq |f(x)|$ for all $x$, and \\item[(e)] $f(r) = 0$. \\end{enumerate}",
      "answer": "\\frac{\\pi}{2}"
    },
    {
      "question": "How many six-digit multiples of 27 have only 3, 6, or 9 as their digits?",
      "answer": "51"
    },
    {
      "question": "What number did Janet pick if she added 7 to the number, multiplied the sum by 2, subtracted 4, and the final result was 28?",
      "answer": "9"
    },
    {
      "question": "Alice starts with the number 0. She can apply 100 operations on her number. In each operation, she can either add 1 to her number, or square her number. After applying all operations, her score is the minimum distance from her number to any perfect square. What is the maximum score she can attain?",
      "answer": "94"
    },
    {
      "question": "Let $\\triangle A B C$ be an acute triangle, with $M$ being the midpoint of $\\overline{B C}$, such that $A M=B C$. Let $D$ and $E$ be the intersection of the internal angle bisectors of $\\angle A M B$ and $\\angle A M C$ with $A B$ and $A C$, respectively. Find the ratio of the area of $\\triangle D M E$ to the area of $\\triangle A B C$.",
      "answer": "\\frac{2}{9}"
    },
    {
      "question": "Two positive integers \\( x \\) and \\( y \\) have \\( xy=24 \\) and \\( x-y=5 \\). What is the value of \\( x+y \\)?",
      "answer": "11"
    },
    {
      "question": "Let $a, b, c$ be non-negative numbers with $a+b+c = 3$. Prove the inequality\n\\[\\frac{a}{b^2+1}+\\frac{b}{c^2+1}+\\frac{c}{a^2+1} \\geq \\frac 32.\\]",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "How many ways are there to arrange the numbers $1,2,3,4,5,6$ on the vertices of a regular hexagon such that exactly 3 of the numbers are larger than both of their neighbors? Rotations and reflections are considered the same.",
      "answer": "8"
    },
    {
      "question": "Regular tetrahedron $A B C D$ is projected onto a plane sending $A, B, C$, and $D$ to $A^{\\prime}, B^{\\prime}, C^{\\prime}$, and $D^{\\prime}$ respectively. Suppose $A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}$ is a convex quadrilateral with $A^{\\prime} B^{\\prime}=A^{\\prime} D^{\\prime}$ and $C^{\\prime} B^{\\prime}=C^{\\prime} D^{\\prime}$, and suppose that the area of $A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}=4$. Given these conditions, the set of possible lengths of $A B$ consists of all real numbers in the interval $[a, b)$. Compute $b$.",
      "answer": "2 \\sqrt[4]{6}"
    },
    {
      "question": "Given that $a, b, c$ are positive integers satisfying $$a+b+c=\\operatorname{gcd}(a, b)+\\operatorname{gcd}(b, c)+\\operatorname{gcd}(c, a)+120$$ determine the maximum possible value of $a$.",
      "answer": "240"
    },
    {
      "question": "Positive integers $a, b$, and $c$ have the property that $a^{b}, b^{c}$, and $c^{a}$ end in 4, 2, and 9, respectively. Compute the minimum possible value of $a+b+c$.",
      "answer": "17"
    },
    {
      "question": "Let $A B C D$ be an isosceles trapezoid with parallel bases $A B=1$ and $C D=2$ and height 1. Find the area of the region containing all points inside $A B C D$ whose projections onto the four sides of the trapezoid lie on the segments formed by $A B, B C, C D$ and $D A$.",
      "answer": "\\frac{5}{8}"
    },
    {
      "question": "Let $S$ be the set of $3^{4}$ points in four-dimensional space where each coordinate is in $\\{-1,0,1\\}$. Let $N$ be the number of sequences of points $P_{1}, P_{2}, \\ldots, P_{2020}$ in $S$ such that $P_{i} P_{i+1}=2$ for all $1 \\leq i \\leq 2020$ and $P_{1}=(0,0,0,0)$. (Here $P_{2021}=P_{1}$.) Find the largest integer $n$ such that $2^{n}$ divides $N$.",
      "answer": "4041"
    },
    {
      "question": "For each positive integer $n$, define $S(n)$ to be the smallest positive integer divisible by each of the positive integers $1, 2, 3, \\ldots, n$. How many positive integers $n$ with $1 \\leq n \\leq 100$ have $S(n) = S(n+4)$?",
      "answer": "11"
    },
    {
      "question": "There are 2018 frogs in a pool and there is 1 frog on the shore. In each time-step thereafter, one random frog moves position. If it was in the pool, it jumps to the shore, and vice versa. Find the expected number of time-steps before all frogs are in the pool for the first time.",
      "answer": "2^{2018}-1"
    },
    {
      "question": "Consider the parabola consisting of the points $(x, y)$ in the real plane satisfying $(y+x)=(y-x)^{2}+3(y-x)+3$. Find the minimum possible value of $y$.",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "How many ways are there to choose 2010 functions $f_{1}, \\ldots, f_{2010}$ from \\{0,1\\} to \\{0,1\\} such that $f_{2010} \\circ f_{2009} \\circ \\cdots \\circ f_{1}$ is constant? Note: a function $g$ is constant if $g(a)=g(b)$ for all $a, b$ in the domain of $g$.",
      "answer": "$4^{2010}-2^{2010}$"
    },
    {
      "question": "A set of positive integers is called [i]fragrant[/i] if it contains at least two elements and each of its elements has a prime factor in common with at least one of the other elements.  Let $P(n)=n^2+n+1$.  What is the least possible positive integer value of $b$ such that there exists a non-negative integer $a$ for which the set $$\\{P(a+1),P(a+2),\\ldots,P(a+b)\\}$$ is fragrant?",
      "answer": "6"
    },
    {
      "question": "The set $S$ consists of 9 distinct positive integers. The average of the two smallest integers in $S$ is 5. The average of the two largest integers in $S$ is 22. What is the greatest possible average of all of the integers of $S$?",
      "answer": "16"
    },
    {
      "question": "How many multiples of 7 between $10^{6}$ and $10^{9}$ are perfect squares?",
      "answer": "4375"
    },
    {
      "question": "In $\\triangle PQR, \\angle RPQ=90^{\\circ}$ and $S$ is on $PQ$. If $SQ=14, SP=18$, and $SR=30$, what is the area of $\\triangle QRS$?",
      "answer": "168"
    },
    {
      "question": "Determine the smallest positive integer $A$ with an odd number of digits and this property, that both $A$ and the number $B$ created by removing the middle digit of the number $A$ are divisible by $2018$.",
      "answer": "100902018"
    },
    {
      "question": "A regular hexagon $A B C D E F$ has side length 1 and center $O$. Parabolas $P_{1}, P_{2}, \\ldots, P_{6}$ are constructed with common focus $O$ and directrices $A B, B C, C D, D E, E F, F A$ respectively. Let $\\chi$ be the set of all distinct points on the plane that lie on at least two of the six parabolas. Compute $$\\sum_{X \\in \\chi}|O X|$$ (Recall that the focus is the point and the directrix is the line such that the parabola is the locus of points that are equidistant from the focus and the directrix.)",
      "answer": "35 \\sqrt{3}"
    },
    {
      "question": "Six men and their wives are sitting at a round table with 12 seats. These men and women are very jealous - no man will allow his wife to sit next to any man except for himself, and no woman will allow her husband to sit next to any woman except for herself. In how many distinct ways can these 12 people be seated such that these conditions are satisfied?",
      "answer": "288000"
    },
    {
      "question": "Suppose $a$ and $b$ are positive integers for which $8 a^{a} b^{b}=27 a^{b} b^{a}$. Find $a^{2}+b^{2}$.",
      "answer": "117"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=9, B C=10$, and $C A=17$. Let $B^{\\prime}$ be the reflection of the point $B$ over the line $C A$. Let $G$ be the centroid of triangle $A B C$, and let $G^{\\prime}$ be the centroid of triangle $A B^{\\prime} C$. Determine the length of segment $G G^{\\prime}$.",
      "answer": "\\frac{48}{17}"
    },
    {
      "question": "Let $T$ be the set of ordered triples $(x,y,z)$, where $x,y,z$ are integers with $0\\leq x,y,z\\leq9$. Players $A$ and $B$ play the following guessing game. Player $A$ chooses a triple $(x,y,z)$ in $T$, and Player $B$ has to discover $A$[i]'s[/i] triple in as few moves as possible.  A [i]move[/i] consists of the following: $B$ gives $A$ a triple $(a,b,c)$ in $T$, and $A$ replies by giving $B$ the number $\\left|x+y-a-b\\right |+\\left|y+z-b-c\\right|+\\left|z+x-c-a\\right|$. Find the minimum number of moves that $B$ needs to be sure of determining $A$[i]'s[/i]  triple.",
      "answer": "3"
    },
    {
      "question": "Consider a circular cone with vertex $V$, and let $ABC$ be a triangle inscribed in the base of the cone, such that $AB$ is a diameter and $AC=BC$. Let $L$ be a point on $BV$ such that the volume of the cone is 4 times the volume of the tetrahedron $ABCL$. Find the value of $BL/LV$.",
      "answer": "\\frac{\\pi}{4-\\pi}"
    },
    {
      "question": "Let $r_{1}, r_{2}, \\ldots, r_{7}$ be the distinct complex roots of the polynomial $P(x)=x^{7}-7$. Let $$K=\\prod_{1 \\leq i<j \\leq 7}\\left(r_{i}+r_{j}\\right)$$ that is, the product of all numbers of the form $r_{i}+r_{j}$, where $i$ and $j$ are integers for which $1 \\leq i<j \\leq 7$. Determine the value of $K^{2}$.",
      "answer": "117649"
    },
    {
      "question": "Let $f(x)=x^{3}+3 x-1$ have roots $a, b, c$. Given that $$\\frac{1}{a^{3}+b^{3}}+\\frac{1}{b^{3}+c^{3}}+\\frac{1}{c^{3}+a^{3}}$$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$, find $100 m+n$.",
      "answer": "3989"
    },
    {
      "question": "Two different numbers are randomly selected from the set $\\{-3, -1, 0, 2, 4\\}$ and then multiplied together. What is the probability that the product of the two numbers chosen is 0?",
      "answer": "\\frac{2}{5}"
    },
    {
      "question": "Let $a \\star b=ab-2$. Compute the remainder when $(((579 \\star 569) \\star 559) \\star \\cdots \\star 19) \\star 9$ is divided by 100.",
      "answer": "29"
    },
    {
      "question": "Consider parallelogram $A B C D$ with $A B>B C$. Point $E$ on $\\overline{A B}$ and point $F$ on $\\overline{C D}$ are marked such that there exists a circle $\\omega_{1}$ passing through $A, D, E, F$ and a circle $\\omega_{2}$ passing through $B, C, E, F$. If $\\omega_{1}, \\omega_{2}$ partition $\\overline{B D}$ into segments $\\overline{B X}, \\overline{X Y}, \\overline{Y D}$ in that order, with lengths $200,9,80$, respectively, compute $B C$.",
      "answer": "51"
    },
    {
      "question": "A path of length $n$ is a sequence of points $\\left(x_{1}, y_{1}\\right),\\left(x_{2}, y_{2}\\right), \\ldots,\\left(x_{n}, y_{n}\\right)$ with integer coordinates such that for all $i$ between 1 and $n-1$ inclusive, either (1) $x_{i+1}=x_{i}+1$ and $y_{i+1}=y_{i}$ (in which case we say the $i$th step is rightward) or (2) $x_{i+1}=x_{i}$ and $y_{i+1}=y_{i}+1$ (in which case we say that the $i$th step is upward). This path is said to start at $\\left(x_{1}, y_{1}\\right)$ and end at $\\left(x_{n}, y_{n}\\right)$. Let $P(a, b)$, for $a$ and $b$ nonnegative integers, be the number of paths that start at $(0,0)$ and end at $(a, b)$. Find $\\sum_{i=0}^{10} P(i, 10-i)$.",
      "answer": "1024"
    },
    {
      "question": "Elena earns $\\$ 13.25$ per hour working at a store. How much does Elena earn in 4 hours?",
      "answer": "\\$53.00"
    },
    {
      "question": "Yang has the sequence of integers $1,2, \\ldots, 2017$. He makes 2016 swaps in order, where a swap changes the positions of two integers in the sequence. His goal is to end with $2,3, \\ldots, 2017,1$. How many different sequences of swaps can Yang do to achieve his goal?",
      "answer": "2017^{2015}"
    },
    {
      "question": "Each one of 2009 distinct points in the plane is coloured in blue or red, so that on every blue-centered unit circle there are exactly two red points. Find the gratest possible number of blue points.",
      "answer": "45"
    },
    {
      "question": "What is the 18 th digit after the decimal point of $\\frac{10000}{9899}$ ?",
      "answer": "5"
    },
    {
      "question": "A mass of 15 kg is halfway between 10 kg and 20 kg on the vertical axis. What is the age of the cod when its mass is 15 kg?",
      "answer": "7"
    },
    {
      "question": "Let $a$ and $b$ be real numbers, and let $r, s$, and $t$ be the roots of $f(x)=x^{3}+a x^{2}+b x-1$. Also, $g(x)=x^{3}+m x^{2}+n x+p$ has roots $r^{2}, s^{2}$, and $t^{2}$. If $g(-1)=-5$, find the maximum possible value of $b$.",
      "answer": "1+\\sqrt{5}"
    },
    {
      "question": "Let $$\\begin{aligned} & A=(1+2 \\sqrt{2}+3 \\sqrt{3}+6 \\sqrt{6})(2+6 \\sqrt{2}+\\sqrt{3}+3 \\sqrt{6})(3+\\sqrt{2}+6 \\sqrt{3}+2 \\sqrt{6})(6+3 \\sqrt{2}+2 \\sqrt{3}+\\sqrt{6}) \\\\ & B=(1+3 \\sqrt{2}+2 \\sqrt{3}+6 \\sqrt{6})(2+\\sqrt{2}+6 \\sqrt{3}+3 \\sqrt{6})(3+6 \\sqrt{2}+\\sqrt{3}+2 \\sqrt{6})(6+2 \\sqrt{2}+3 \\sqrt{3}+\\sqrt{6}) \\end{aligned}$$ Compute the value of $A / B$.",
      "answer": "1"
    },
    {
      "question": "Gustave has 15 steel bars of masses $1 \\mathrm{~kg}, 2 \\mathrm{~kg}, 3 \\mathrm{~kg}, \\ldots, 14 \\mathrm{~kg}, 15 \\mathrm{~kg}$. He also has 3 bags labelled $A, B, C$. He places two steel bars in each bag so that the total mass in each bag is equal to $M \\mathrm{~kg}$. How many different values of $M$ are possible?",
      "answer": "19"
    },
    {
      "question": "Find the number of ordered triples of integers $(a, b, c)$ with $1 \\leq a, b, c \\leq 100$ and $a^{2} b+b^{2} c+c^{2} a=a b^{2}+b c^{2}+c a^{2}$",
      "answer": "29800"
    },
    {
      "question": "Let \\(ABCDEF\\) be a regular hexagon and let point \\(O\\) be the center of the hexagon. How many ways can you color these seven points either red or blue such that there doesn't exist any equilateral triangle with vertices of all the same color?",
      "answer": "6"
    },
    {
      "question": "Alicia starts a sequence with $m=3$. What is the fifth term of her sequence following the algorithm: Step 1: Alicia writes down the number $m$ as the first term. Step 2: If $m$ is even, Alicia sets $n=\frac{1}{2} m$. If $m$ is odd, Alicia sets $n=m+1$. Step 3: Alicia writes down the number $m+n+1$ as the next term. Step 4: Alicia sets $m$ equal to the value of the term that she just wrote down in Step 3. Step 5: Alicia repeats Steps 2, 3, 4 until she has five terms, at which point she stops.",
      "answer": "43"
    },
    {
      "question": "In the figure, if $A E=3, C E=1, B D=C D=2$, and $A B=5$, find $A G$.",
      "answer": "3\\sqrt{66} / 7"
    },
    {
      "question": "Find the number of positive integer solutions to $n^{x}+n^{y}=n^{z}$ with $n^{z}<2001$.",
      "answer": "10"
    },
    {
      "question": "Find the largest real number $\\lambda$ with the following property: for any positive real numbers $p,q,r,s$ there exists a complex number $z=a+bi$($a,b\\in \\mathbb{R})$  such that $$ |b|\\ge \\lambda |a| \\quad \\text{and} \\quad (pz^3+2qz^2+2rz+s) \\cdot (qz^3+2pz^2+2sz+r) =0.$$",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "Cyclic quadrilateral $A B C D$ has side lengths $A B=1, B C=2, C D=3$ and $D A=4$. Points $P$ and $Q$ are the midpoints of $\\overline{B C}$ and $\\overline{D A}$. Compute $P Q^{2}$.",
      "answer": "\\frac{116}{35}"
    },
    {
      "question": "Let $x_{1}, \\ldots, x_{100}$ be defined so that for each $i, x_{i}$ is a (uniformly) random integer between 1 and 6 inclusive. Find the expected number of integers in the set $\\{x_{1}, x_{1}+x_{2}, \\ldots, x_{1}+x_{2}+\\ldots+x_{100}\\}$ that are multiples of 6.",
      "answer": "\\frac{50}{3}"
    },
    {
      "question": "What is the largest real number $\\theta$ less than $\\pi$ (i.e. $\\theta<\\pi$ ) such that $\\prod_{k=0}^{10} \\cos \\left(2^{k} \\theta\\right) \\neq 0$ and $\\prod_{k=0}^{10}\\left(1+\\frac{1}{\\cos \\left(2^{k} \\theta\\right)}\\right)=1 ?",
      "answer": "\\frac{2046 \\pi}{2047}"
    },
    {
      "question": "Let $P$ be a polynomial with integer coefficients such that $P(0)+P(90)=2018$. Find the least possible value for $|P(20)+P(70)|$.",
      "answer": "\\[ 782 \\]"
    },
    {
      "question": "The pairwise greatest common divisors of five positive integers are $2,3,4,5,6,7,8, p, q, r$ in some order, for some positive integers $p, q, r$. Compute the minimum possible value of $p+q+r$.",
      "answer": "9"
    },
    {
      "question": "Bob is coloring lattice points in the coordinate plane. Find the number of ways Bob can color five points in $\\{(x, y) \\mid 1 \\leq x, y \\leq 5\\}$ blue such that the distance between any two blue points is not an integer.",
      "answer": "80"
    },
    {
      "question": "If \\( x=2 \\), what is the value of \\( (x+2-x)(2-x-2) \\)?",
      "answer": "-4"
    },
    {
      "question": "Let $ABC$ be an acute triangle.  Let $DAC,EAB$, and $FBC$ be isosceles triangles exterior to $ABC$, with $DA=DC, EA=EB$, and $FB=FC$, such that\n\\[\n\\angle ADC = 2\\angle BAC, \\quad \\angle BEA= 2 \\angle ABC, \\quad\n\\angle CFB = 2 \\angle ACB.\n\\]\nLet $D'$  be the intersection of lines $DB$ and $EF$, let $E'$ be the intersection of $EC$ and $DF$, and let $F'$   be the intersection of $FA$ and $DE$. Find, with proof, the value of the sum\n\\[\n\\frac{DB}{DD'}+\\frac{EC}{EE'}+\\frac{FA}{FF'}.\n\\]",
      "answer": "4"
    },
    {
      "question": "Suppose a real number \\(x>1\\) satisfies \\(\\log _{2}\\left(\\log _{4} x\\right)+\\log _{4}\\left(\\log _{16} x\\right)+\\log _{16}\\left(\\log _{2} x\\right)=0\\). Compute \\(\\log _{2}\\left(\\log _{16} x\\right)+\\log _{16}\\left(\\log _{4} x\\right)+\\log _{4}\\left(\\log _{2} x\\right)\\).",
      "answer": "-\\frac{1}{4}"
    },
    {
      "question": "Let $x$ be a real number. Find the maximum value of $2^{x(1-x)}$.",
      "answer": "\\sqrt[4]{2}"
    },
    {
      "question": "Let $a, b, c, d$ be real numbers such that $\\min (20 x+19,19 x+20)=(a x+b)-|c x+d|$ for all real numbers $x$. Find $a b+c d$.",
      "answer": "380"
    },
    {
      "question": "There are 2 runners on the perimeter of a regular hexagon, initially located at adjacent vertices. Every second, each of the runners independently moves either one vertex to the left, with probability $\\frac{1}{2}$, or one vertex to the right, also with probability $\\frac{1}{2}$. Find the probability that after a 2013 second run (in which the runners switch vertices 2013 times each), the runners end up at adjacent vertices once again.",
      "answer": "\\frac{2}{3}+\\frac{1}{3}\\left(\\frac{1}{4}\\right)^{2013}"
    },
    {
      "question": "Barry picks infinitely many points inside a unit circle, each independently and uniformly at random, $P_{1}, P_{2}, \\ldots$ Compute the expected value of $N$, where $N$ is the smallest integer such that $P_{N+1}$ is inside the convex hull formed by the points $P_{1}, P_{2}, \\ldots, P_{N}$. Submit a positive real number $E$. If the correct answer is $A$, you will receive $\\lfloor 100 \\cdot \\max (0.2099-|E-A|, 0)\\rfloor$ points.",
      "answer": "6.54"
    },
    {
      "question": "The sequence $a_{1}, a_{2}, a_{3}, \\ldots$ of real numbers satisfies the recurrence $a_{n+1}=\\frac{a_{n}^{2}-a_{n-1}+2 a_{n}}{a_{n-1}+1}$. Given that $a_{1}=1$ and $a_{9}=7$, find $a_{5}$.",
      "answer": "3"
    },
    {
      "question": "A tetrahedron has all its faces triangles with sides $13,14,15$. What is its volume?",
      "answer": "42 \\sqrt{55}"
    },
    {
      "question": "A lame king is a chess piece that can move from a cell to any cell that shares at least one vertex with it, except for the cells in the same column as the current cell. A lame king is placed in the top-left cell of a $7 \\times 7$ grid. Compute the maximum number of cells it can visit without visiting the same cell twice (including its starting cell).",
      "answer": "43"
    },
    {
      "question": "A complex quartic polynomial $Q$ is quirky if it has four distinct roots, one of which is the sum of the other three. There are four complex values of $k$ for which the polynomial $Q(x)=x^{4}-k x^{3}-x^{2}-x-45$ is quirky. Compute the product of these four values of $k$.",
      "answer": "720"
    },
    {
      "question": "The function $f$ satisfies $f(x)+f(2 x+y)+5 x y=f(3 x-y)+2 x^{2}+1$ for all real numbers $x, y$. Determine the value of $f(10)$.",
      "answer": "-49"
    },
    {
      "question": "What is the value of the expression $\frac{3}{10}+\frac{3}{100}+\frac{3}{1000}$?",
      "answer": "0.333"
    },
    {
      "question": "Nine fair coins are flipped independently and placed in the cells of a 3 by 3 square grid. Let $p$ be the probability that no row has all its coins showing heads and no column has all its coins showing tails. If $p=\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
      "answer": "8956"
    },
    {
      "question": "Compute $$\\sum_{n=0}^{\\infty} \\frac{n}{n^{4}+n^{2}+1}$$",
      "answer": "1/2"
    },
    {
      "question": "Three $1 \times 1 \times 1$ cubes are joined side by side. What is the surface area of the resulting prism?",
      "answer": "14"
    },
    {
      "question": "What is the greatest possible value of $n$ if Juliana chooses three different numbers from the set $\\{-6,-4,-2,0,1,3,5,7\\}$ and multiplies them together to obtain the integer $n$?",
      "answer": "168"
    },
    {
      "question": "The taxicab distance between points $\\left(x_{1}, y_{1}\\right)$ and $\\left(x_{2}, y_{2}\\right)$ is $\\left|x_{2}-x_{1}\\right|+\\left|y_{2}-y_{1}\\right|$. A regular octagon is positioned in the $x y$ plane so that one of its sides has endpoints $(0,0)$ and $(1,0)$. Let $S$ be the set of all points inside the octagon whose taxicab distance from some octagon vertex is at most \\frac{2}{3}$. The area of $S$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "2309"
    },
    {
      "question": "How many sequences of integers $(a_{1}, \\ldots, a_{7})$ are there for which $-1 \\leq a_{i} \\leq 1$ for every $i$, and $a_{1} a_{2}+a_{2} a_{3}+a_{3} a_{4}+a_{4} a_{5}+a_{5} a_{6}+a_{6} a_{7}=4$?",
      "answer": "38"
    },
    {
      "question": "Compute the sum of all positive real numbers \\(x \\leq 5\\) satisfying \\(x=\\frac{\\left\\lceil x^{2}\\right\\rceil+\\lceil x\\rceil \\cdot\\lfloor x\\rfloor}{\\lceil x\\rceil+\\lfloor x\\rfloor}\\).",
      "answer": "85"
    },
    {
      "question": "Let $a, b, c$ be not necessarily distinct integers between 1 and 2011, inclusive. Find the smallest possible value of $\\frac{a b+c}{a+b+c}$.",
      "answer": "$\\frac{2}{3}$"
    },
    {
      "question": "We call a positive integer $t$ good if there is a sequence $a_{0}, a_{1}, \\ldots$ of positive integers satisfying $a_{0}=15, a_{1}=t$, and $a_{n-1} a_{n+1}=\\left(a_{n}-1\\right)\\left(a_{n}+1\\right)$ for all positive integers $n$. Find the sum of all good numbers.",
      "answer": "296"
    },
    {
      "question": "Let $ f(n)$ be a function defined on the set of all positive integers and having its values in the same set. Suppose that $ f(f(n) \\plus{} f(m)) \\equal{} m \\plus{} n$ for all positive integers $ n,m.$ Find the possible value for $ f(1988).$",
      "answer": "1988"
    },
    {
      "question": "If $x$ and $y$ are positive real numbers with $\\frac{1}{x+y}=\\frac{1}{x}-\\frac{1}{y}$, what is the value of $\\left(\\frac{x}{y}+\\frac{y}{x}\\right)^{2}$?",
      "answer": "5"
    },
    {
      "question": "The country of HMMTLand has 8 cities. Its government decides to construct several two-way roads between pairs of distinct cities. After they finish construction, it turns out that each city can reach exactly 3 other cities via a single road, and from any pair of distinct cities, either exactly 0 or 2 other cities can be reached from both cities by a single road. Compute the number of ways HMMTLand could have constructed the roads.",
      "answer": "875"
    },
    {
      "question": "For how many ordered triplets $(a, b, c)$ of positive integers less than 10 is the product $a \\times b \\times c$ divisible by 20?",
      "answer": "102"
    },
    {
      "question": "A collection $\\mathcal{S}$ of 10000 points is formed by picking each point uniformly at random inside a circle of radius 1. Let $N$ be the expected number of points of $\\mathcal{S}$ which are vertices of the convex hull of the $\\mathcal{S}$. (The convex hull is the smallest convex polygon containing every point of $\\mathcal{S}$.) Estimate $N$.",
      "answer": "72.8"
    },
    {
      "question": "Let $A B C$ be an acute triangle with $A$-excircle $\\Gamma$. Let the line through $A$ perpendicular to $B C$ intersect $B C$ at $D$ and intersect $\\Gamma$ at $E$ and $F$. Suppose that $A D=D E=E F$. If the maximum value of $\\sin B$ can be expressed as $\\frac{\\sqrt{a}+\\sqrt{b}}{c}$ for positive integers $a, b$, and $c$, compute the minimum possible value of $a+b+c$.",
      "answer": "705"
    },
    {
      "question": "Find the smallest positive integer $b$ such that $1111_{b}$ ( 1111 in base $b$) is a perfect square. If no such $b$ exists, write \"No solution\".",
      "answer": "7"
    },
    {
      "question": "Suppose $P(x)$ is a polynomial such that $P(1)=1$ and $$\\frac{P(2 x)}{P(x+1)}=8-\\frac{56}{x+7}$$ for all real $x$ for which both sides are defined. Find $P(-1)$.",
      "answer": "-5/21"
    },
    {
      "question": "For positive integers $m, n$, let \\operatorname{gcd}(m, n) denote the largest positive integer that is a factor of both $m$ and $n$. Compute $$\\sum_{n=1}^{91} \\operatorname{gcd}(n, 91)$$",
      "answer": "325"
    },
    {
      "question": "Let $ABCD$ be a square with side length $1$. How many points $P$ inside the square (not on its sides) have the property that the square can be cut into $10$ triangles of equal area such that all of them have $P$ as a vertex?",
      "answer": "16"
    },
    {
      "question": "Call a positive integer $N \\geq 2$ \"special\" if for every $k$ such that $2 \\leq k \\leq N, N$ can be expressed as a sum of $k$ positive integers that are relatively prime to $N$ (although not necessarily relatively prime to each other). How many special integers are there less than $100$?",
      "answer": "50"
    },
    {
      "question": "Five equally skilled tennis players named Allen, Bob, Catheryn, David, and Evan play in a round robin tournament, such that each pair of people play exactly once, and there are no ties. In each of the ten games, the two players both have a $50 \\%$ chance of winning, and the results of the games are independent. Compute the probability that there exist four distinct players $P_{1}, P_{2}, P_{3}, P_{4}$ such that $P_{i}$ beats $P_{i+1}$ for $i=1,2,3,4$. (We denote $P_{5}=P_{1}$ ).",
      "answer": "\\frac{49}{64}"
    },
    {
      "question": "Let $A B C D$ be a quadrilateral with an inscribed circle $\\omega$ that has center $I$. If $I A=5, I B=7, I C=4, I D=9$, find the value of $\\frac{A B}{C D}$.",
      "answer": "\\frac{35}{36}"
    },
    {
      "question": "Let $ABC$ be a triangle with circumcenter $O$ such that $AC=7$. Suppose that the circumcircle of $AOC$ is tangent to $BC$ at $C$ and intersects the line $AB$ at $A$ and $F$. Let $FO$ intersect $BC$ at $E$. Compute $BE$.",
      "answer": "\\frac{7}{2}"
    },
    {
      "question": "Find the smallest positive integer $n$ such that $\\frac{5^{n+1}+2^{n+1}}{5^{n}+2^{n}}>4.99$.",
      "answer": "7"
    },
    {
      "question": "Let $T$ be the set of numbers of the form $2^{a} 3^{b}$ where $a$ and $b$ are integers satisfying $0 \\leq a, b \\leq 5$. How many subsets $S$ of $T$ have the property that if $n$ is in $S$ then all positive integer divisors of $n$ are in $S$ ?",
      "answer": "924"
    },
    {
      "question": "How many ordered pairs $(a, b)$ of positive integers satisfy $a^{2}+b^{2}=50$?",
      "answer": "3"
    },
    {
      "question": "A positive integer $n$ is $inverosimil$ if there exists $n$ integers not necessarily distinct such that the sum and the product of this integers are equal to $n$. How many positive integers less than or equal to $2022$ are $inverosimils$?",
      "answer": "1010"
    },
    {
      "question": "Franklin has four bags, numbered 1 through 4. Initially, the first bag contains fifteen balls, numbered 1 through 15 , and the other bags are empty. Franklin randomly pulls a pair of balls out of the first bag, throws away the ball with the lower number, and moves the ball with the higher number into the second bag. He does this until there is only one ball left in the first bag. He then repeats this process in the second and third bag until there is exactly one ball in each bag. What is the probability that ball 14 is in one of the bags at the end?",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "Suppose $a, b, c, d$, and $e$ are objects that we can multiply together, but the multiplication doesn't necessarily satisfy the associative law, i.e. ( $x y) z$ does not necessarily equal $x(y z)$. How many different ways are there to interpret the product abcde?",
      "answer": "14"
    },
    {
      "question": "Two concentric circles have radii $r$ and $R>r$. Three new circles are drawn so that they are each tangent to the big two circles and tangent to the other two new circles. Find $\\frac{R}{r}$.",
      "answer": "3"
    },
    {
      "question": "What is the value of $\frac{8+4}{8-4}$?",
      "answer": "3"
    },
    {
      "question": "An integer $n$ is decreased by 2 and then multiplied by 5. If the result is 85, what is the value of $n$?",
      "answer": "19"
    },
    {
      "question": "If $x=18$ is one of the solutions of the equation $x^{2}+12x+c=0$, what is the other solution of this equation?",
      "answer": "-30"
    },
    {
      "question": "If $a(x+b)=3 x+12$ for all values of $x$, what is the value of $a+b$?",
      "answer": "7"
    },
    {
      "question": "The Cayley Corner Store sells three types of toys: Exes, Wyes and Zeds. All Exes are identical, all Wyes are identical, and all Zeds are identical. The mass of 2 Exes equals the mass of 29 Wyes. The mass of 1 Zed equals the mass of 16 Exes. The mass of 1 Zed equals the mass of how many Wyes?",
      "answer": "232"
    },
    {
      "question": "Reimu has 2019 coins $C_{0}, C_{1}, \\ldots, C_{2018}$, one of which is fake, though they look identical to each other (so each of them is equally likely to be fake). She has a machine that takes any two coins and picks one that is not fake. If both coins are not fake, the machine picks one uniformly at random. For each $i=1,2, \\ldots, 1009$, she puts $C_{0}$ and $C_{i}$ into the machine once, and machine picks $C_{i}$. What is the probability that $C_{0}$ is fake?",
      "answer": "\\frac{2^{1009}}{2^{1009}+1009}"
    },
    {
      "question": "In the $5 \\times 5$ grid shown, 15 cells contain X's and 10 cells are empty. What is the smallest number of X's that must be moved so that each row and each column contains exactly three X's?",
      "answer": "2"
    },
    {
      "question": "A function $f:\\{1,2,3,4,5\\} \\rightarrow\\{1,2,3,4,5\\}$ is said to be nasty if there do not exist distinct $a, b \\in\\{1,2,3,4,5\\}$ satisfying $f(a)=b$ and $f(b)=a$. How many nasty functions are there?",
      "answer": "1950"
    },
    {
      "question": "A polynomial $P$ has four roots, $\\frac{1}{4}, \\frac{1}{2}, 2,4$. The product of the roots is 1, and $P(1)=1$. Find $P(0)$.",
      "answer": "\\frac{8}{9}"
    },
    {
      "question": "Let $S_{0}=0$ and let $S_{k}$ equal $a_{1}+2 a_{2}+\\ldots+k a_{k}$ for $k \\geq 1$. Define $a_{i}$ to be 1 if $S_{i-1}<i$ and -1 if $S_{i-1} \\geq i$. What is the largest $k \\leq 2010$ such that $S_{k}=0$?",
      "answer": "1092"
    },
    {
      "question": "Find the number of ordered quadruples of positive integers $(a, b, c, d)$ such that $a, b, c$, and $d$ are all (not necessarily distinct) factors of 30 and $abcd>900$.",
      "answer": "1940"
    },
    {
      "question": "Given a permutation $\\pi$ of the set $\\{1,2, \\ldots, 10\\}$, define a rotated cycle as a set of three integers $i, j, k$ such that $i<j<k$ and $\\pi(j)<\\pi(k)<\\pi(i)$. What is the total number of rotated cycles over all permutations $\\pi$ of the set $\\{1,2, \\ldots, 10\\}$ ?",
      "answer": "72576000"
    },
    {
      "question": "Let $\\mathcal{P}$ be a parabola with focus $F$ and directrix $\\ell$. A line through $F$ intersects $\\mathcal{P}$ at two points $A$ and $B$. Let $D$ and $C$ be the feet of the altitudes from $A$ and $B$ onto $\\ell$, respectively. Given that $AB=20$ and $CD=14$, compute the area of $ABCD$.",
      "answer": "140"
    },
    {
      "question": "$A B C$ is an acute triangle with incircle $\\omega$. $\\omega$ is tangent to sides $\\overline{B C}, \\overline{C A}$, and $\\overline{A B}$ at $D, E$, and $F$ respectively. $P$ is a point on the altitude from $A$ such that $\\Gamma$, the circle with diameter $\\overline{A P}$, is tangent to $\\omega$. $\\Gamma$ intersects $\\overline{A C}$ and $\\overline{A B}$ at $X$ and $Y$ respectively. Given $X Y=8, A E=15$, and that the radius of $\\Gamma$ is 5, compute $B D \\cdot D C$.",
      "answer": "\\frac{675}{4}"
    },
    {
      "question": "Let $f$ be a monic cubic polynomial satisfying $f(x)+f(-x)=0$ for all real numbers $x$. For all real numbers $y$, define $g(y)$ to be the number of distinct real solutions $x$ to the equation $f(f(x))=y$. Suppose that the set of possible values of $g(y)$ over all real numbers $y$ is exactly $\\{1,5,9\\}$. Compute the sum of all possible values of $f(10)$.",
      "answer": "970"
    },
    {
      "question": "Let $\\triangle ABC$ be an equilateral triangle of side length 1. Let $D,E,F$ be points on $BC,AC,AB$ respectively, such that $\\frac{DE}{20} = \\frac{EF}{22} = \\frac{FD}{38}$. Let $X,Y,Z$ be on lines $BC,CA,AB$ respectively, such that $XY\\perp DE, YZ\\perp EF, ZX\\perp FD$. Find all possible values of $\\frac{1}{[DEF]} + \\frac{1}{[XYZ]}$.",
      "answer": "\\frac{97 \\sqrt{2} + 40 \\sqrt{3}}{15}"
    },
    {
      "question": "Count how many 8-digit numbers there are that contain exactly four nines as digits.",
      "answer": "433755"
    },
    {
      "question": "A four-digit positive integer is called [i]virtual[/i] if it has the form $\\overline{abab}$, where $a$ and $b$ are digits and $a \\neq 0$. For example 2020, 2121 and 2222 are virtual numbers, while 2002 and 0202 are not. Find all virtual numbers of the form $n^2+1$, for some positive integer $n$.",
      "answer": "8282"
    },
    {
      "question": "Let $a,b,c$ be positive real numbers such that $a+b+c = 3$.  Find the minimum value of the expression \\[A=\\dfrac{2-a^3}a+\\dfrac{2-b^3}b+\\dfrac{2-c^3}c.\\]",
      "answer": "3"
    },
    {
      "question": "An electric car is charged 3 times per week for 52 weeks. The cost to charge the car each time is $0.78. What is the total cost to charge the car over these 52 weeks?",
      "answer": "\\$121.68"
    },
    {
      "question": "Triangle $ABC$ has an area 1.  Points $E,F,G$ lie, respectively, on sides $BC$, $CA$, $AB$ such that $AE$ bisects $BF$ at point $R$, $BF$ bisects $CG$ at point $S$, and $CG$ bisects $AE$ at point $T$. Find the area of the triangle $RST$.",
      "answer": "\\frac{7 - 3 \\sqrt{5}}{4}"
    },
    {
      "question": "Let $A B C D$ be a square of side length 5. A circle passing through $A$ is tangent to segment $C D$ at $T$ and meets $A B$ and $A D$ again at $X \\neq A$ and $Y \\neq A$, respectively. Given that $X Y=6$, compute $A T$.",
      "answer": "\\sqrt{30}"
    },
    {
      "question": "Let $f(x)=x^{4}+14 x^{3}+52 x^{2}+56 x+16$. Let $z_{1}, z_{2}, z_{3}, z_{4}$ be the four roots of $f$. Find the smallest possible value of $|z_{a} z_{b}+z_{c} z_{d}|$ where $\\{a, b, c, d\\}=\\{1,2,3,4\\}$.",
      "answer": "8"
    },
    {
      "question": "For an integer $x \\geq 1$, let $p(x)$ be the least prime that does not divide $x$, and define $q(x)$ to be the product of all primes less than $p(x)$. In particular, $p(1) = 2.$ For $x$ having $p(x) = 2$, define $q(x) = 1$. Consider the sequence $x_0, x_1, x_2, \\ldots$ defined by $x_0 = 1$ and \\[ x_{n+1} = \\frac{x_n p(x_n)}{q(x_n)} \\] for $n \\geq 0$. Find all $n$ such that $x_n = 1995$.",
      "answer": "142"
    },
    {
      "question": "Let \\(ABC\\) be a triangle with \\(AB=13, BC=14\\), and \\(CA=15\\). Pick points \\(Q\\) and \\(R\\) on \\(AC\\) and \\(AB\\) such that \\(\\angle CBQ=\\angle BCR=90^{\\circ}\\). There exist two points \\(P_{1} \\neq P_{2}\\) in the plane of \\(ABC\\) such that \\(\\triangle P_{1}QR, \\triangle P_{2}QR\\), and \\(\\triangle ABC\\) are similar (with vertices in order). Compute the sum of the distances from \\(P_{1}\\) to \\(BC\\) and \\(P_{2}\\) to \\(BC\\).",
      "answer": "48"
    },
    {
      "question": "A $5 \\times 5$ square grid has the number -3 written in the upper-left square and the number 3 written in the lower-right square. In how many ways can the remaining squares be filled in with integers so that any two adjacent numbers differ by 1, where two squares are adjacent if they share a common edge (but not if they share only a corner)?",
      "answer": "250"
    },
    {
      "question": "Let $S = \\{(x,y) | x = 1, 2, \\ldots, 1993, y = 1, 2, 3, 4\\}$. If $T \\subset S$ and there aren't any squares in $T.$ Find the maximum possible value of $|T|.$ The squares in T use points in S as vertices.",
      "answer": "5183"
    },
    {
      "question": "There is a grid of height 2 stretching infinitely in one direction. Between any two edge-adjacent cells of the grid, there is a door that is locked with probability $\\frac{1}{2}$ independent of all other doors. Philip starts in a corner of the grid (in the starred cell). Compute the expected number of cells that Philip can reach, assuming he can only travel between cells if the door between them is unlocked.",
      "answer": "\\frac{32}{7}"
    },
    {
      "question": "Let $\\Omega$ be a circle of radius 8 centered at point $O$, and let $M$ be a point on $\\Omega$. Let $S$ be the set of points $P$ such that $P$ is contained within $\\Omega$, or such that there exists some rectangle $A B C D$ containing $P$ whose center is on $\\Omega$ with $A B=4, B C=5$, and $B C \\| O M$. Find the area of $S$.",
      "answer": "164+64 \\pi"
    },
    {
      "question": "Let $AD, BE$, and $CF$ be segments sharing a common midpoint, with $AB < AE$ and $BC < BF$. Suppose that each pair of segments forms a $60^{\\circ}$ angle, and that $AD=7, BE=10$, and $CF=18$. Let $K$ denote the sum of the areas of the six triangles $\\triangle ABC, \\triangle BCD, \\triangle CDE, \\triangle DEF, \\triangle EFA$, and $\\triangle FAB$. Compute $K \\sqrt{3}$.",
      "answer": "141"
    },
    {
      "question": "There are 12 students in a classroom; 6 of them are Democrats and 6 of them are Republicans. Every hour the students are randomly separated into four groups of three for political debates. If a group contains students from both parties, the minority in the group will change his/her political alignment to that of the majority at the end of the debate. What is the expected amount of time needed for all 12 students to have the same political alignment, in hours?",
      "answer": "\\frac{341}{55}"
    },
    {
      "question": "Two random points are chosen on a segment and the segment is divided at each of these two points. Of the three segments obtained, find the probability that the largest segment is more than three times longer than the smallest segment.",
      "answer": "\\frac{27}{35}"
    },
    {
      "question": "Find the maximum number of points $X_{i}$ such that for each $i$, $\\triangle A B X_{i} \\cong \\triangle C D X_{i}$.",
      "answer": "4"
    },
    {
      "question": "The area of the triangular region bounded by the $x$-axis, the $y$-axis and the line with equation $y=2x-6$ is one-quarter of the area of the triangular region bounded by the $x$-axis, the line with equation $y=2x-6$ and the line with equation $x=d$, where $d>0$. What is the value of $d$?",
      "answer": "9"
    },
    {
      "question": "Triangle $ABC$ has side lengths $AB=19, BC=20$, and $CA=21$. Points $X$ and $Y$ are selected on sides $AB$ and $AC$, respectively, such that $AY=XY$ and $XY$ is tangent to the incircle of $\\triangle ABC$. If the length of segment $AX$ can be written as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers, compute $100 a+b$.",
      "answer": "6710"
    },
    {
      "question": "Aaron has 144 identical cubes, each with edge length 1 cm. He uses all of the cubes to construct a solid rectangular prism, which he places on a flat table. If the perimeter of the base of the prism is 20 cm, what is the sum of all possible heights of the prism?",
      "answer": "31"
    },
    {
      "question": "Ten distinct positive real numbers are given and the sum of each pair is written (So 45 sums). Between these sums there are 5 equal numbers. If we calculate product of each pair, find the biggest number $k$ such that there may be $k$ equal numbers between them.",
      "answer": "4"
    },
    {
      "question": "How many different graphs with 9 vertices exist where each vertex is connected to 2 others?",
      "answer": "4"
    },
    {
      "question": "How many values of $x,-19<x<98$, satisfy $$\\cos ^{2} x+2 \\sin ^{2} x=1 ?$$",
      "answer": "38"
    },
    {
      "question": "Let $S$ be a set of intervals defined recursively as follows: Initially, $[1,1000]$ is the only interval in $S$. If $l \\neq r$ and $[l, r] \\in S$, then both $\\left[l,\\left\\lfloor\\frac{l+r}{2}\\right\\rfloor\\right],\\left[\\left\\lfloor\\frac{l+r}{2}\\right\\rfloor+1, r\\right] \\in S$. An integer $i$ is chosen uniformly at random from the range $[1,1000]$. What is the expected number of intervals in $S$ which contain $i$?",
      "answer": "10.976"
    },
    {
      "question": "Suppose that $P(x, y, z)$ is a homogeneous degree 4 polynomial in three variables such that $P(a, b, c)=P(b, c, a)$ and $P(a, a, b)=0$ for all real $a, b$, and $c$. If $P(1,2,3)=1$, compute $P(2,4,8)$.",
      "answer": "56"
    },
    {
      "question": "A tightrope walker stands in the center of a rope of length 32 meters. Every minute she walks forward one meter with probability $3 / 4$ and backward one meter with probability $1 / 4$. What is the probability that she reaches the end in front of her before the end behind her?",
      "answer": "3^{16} / (3^{16}+1)"
    },
    {
      "question": "Knot is ready to face Gammadorf in a card game. In this game, there is a deck with twenty cards numbered from 1 to 20. Each player starts with a five card hand drawn from this deck. In each round, Gammadorf plays a card in his hand, then Knot plays a card in his hand. Whoever played a card with greater value gets a point. At the end of five rounds, the player with the most points wins. If Gammadorf starts with a hand of $1,5,10,15,20$, how many five-card hands of the fifteen remaining cards can Knot draw which always let Knot win (assuming he plays optimally)?",
      "answer": "2982"
    },
    {
      "question": "Find the smallest positive real $\\alpha$, such that $$\\frac{x+y} {2}\\geq \\alpha\\sqrt{xy}+(1 - \\alpha)\\sqrt{\\frac{x^2+y^2}{2}}$$ for all positive reals $x, y$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Tetrahedron $A B C D$ has side lengths $A B=6, B D=6 \\sqrt{2}, B C=10, A C=8, C D=10$, and $A D=6$. The distance from vertex $A$ to face $B C D$ can be written as $\\frac{a \\sqrt{b}}{c}$, where $a, b, c$ are positive integers, $b$ is square-free, and $\\operatorname{gcd}(a, c)=1$. Find $100 a+10 b+c$.",
      "answer": "2851"
    },
    {
      "question": "Let $x$ and $y$ be positive real numbers. Define $a=1+\\frac{x}{y}$ and $b=1+\\frac{y}{x}$. If $a^{2}+b^{2}=15$, compute $a^{3}+b^{3}$.",
      "answer": "50"
    },
    {
      "question": "Augustin has six $1 \\times 2 \\times \\pi$ bricks. He stacks them, one on top of another, to form a tower six bricks high. Each brick can be in any orientation so long as it rests flat on top of the next brick below it (or on the floor). How many distinct heights of towers can he make?",
      "answer": "28"
    },
    {
      "question": "Compute the number of functions $f:\\{1,2, \\ldots, 9\\} \\rightarrow\\{1,2, \\ldots, 9\\}$ which satisfy $f(f(f(f(f(x)))))=$ $x$ for each $x \\in\\{1,2, \\ldots, 9\\}$.",
      "answer": "3025"
    },
    {
      "question": "For how many positive integers $n \\leq 100$ is it true that $10 n$ has exactly three times as many positive divisors as $n$ has?",
      "answer": "28"
    },
    {
      "question": "Let $n$ be a positive integer. Claudio has $n$ cards, each labeled with a different number from 1 to n. He takes a subset of these cards, and multiplies together the numbers on the cards. He remarks that, given any positive integer $m$, it is possible to select some subset of the cards so that the difference between their product and $m$ is divisible by 100. Compute the smallest possible value of $n$.",
      "answer": "17"
    },
    {
      "question": "Each cell of a $2 \\times 5$ grid of unit squares is to be colored white or black. Compute the number of such colorings for which no $2 \\times 2$ square is a single color.",
      "answer": "634"
    },
    {
      "question": "Find the number of ordered pairs of integers $(a, b) \\in\\{1,2, \\ldots, 35\\}^{2}$ (not necessarily distinct) such that $a x+b$ is a \"quadratic residue modulo $x^{2}+1$ and 35 \", i.e. there exists a polynomial $f(x)$ with integer coefficients such that either of the following equivalent conditions holds: - there exist polynomials $P, Q$ with integer coefficients such that $f(x)^{2}-(a x+b)=\\left(x^{2}+1\\right) P(x)+35 Q(x)$ - or more conceptually, the remainder when (the polynomial) $f(x)^{2}-(a x+b)$ is divided by (the polynomial) $x^{2}+1$ is a polynomial with (integer) coefficients all divisible by 35 .",
      "answer": "225"
    },
    {
      "question": "16 students took part in a competition. All problems were multiple choice style. Each problem had four choices. It was said that any two students had at most one answer in common, find the maximum number of problems.",
      "answer": "5"
    },
    {
      "question": "You are given an unlimited supply of red, blue, and yellow cards to form a hand. Each card has a point value and your score is the sum of the point values of those cards. The point values are as follows: the value of each red card is 1 , the value of each blue card is equal to twice the number of red cards, and the value of each yellow card is equal to three times the number of blue cards. What is the maximum score you can get with fifteen cards?",
      "answer": "168"
    },
    {
      "question": "Kevin writes down the positive integers $1,2, \\ldots, 15$ on a blackboard. Then, he repeatedly picks two random integers $a, b$ on the blackboard, erases them, and writes down $\\operatorname{gcd}(a, b)$ and $\\operatorname{lcm}(a, b)$. He does this until he is no longer able to change the set of numbers written on the board. Find the maximum sum of the numbers on the board after this process.",
      "answer": "360864"
    },
    {
      "question": "The integer 119 is a multiple of which number?",
      "answer": "7"
    },
    {
      "question": "Let $x, y$, and $z$ be positive real numbers such that $(x \\cdot y)+z=(x+z) \\cdot(y+z)$. What is the maximum possible value of $x y z$?",
      "answer": "1/27"
    },
    {
      "question": "An infinite sequence $ \\,x_{0},x_{1},x_{2},\\ldots \\,$ of real numbers is said to be [b]bounded[/b] if there is a constant $ \\,C\\,$ such that $ \\, \\vert x_{i} \\vert \\leq C\\,$ for every $ \\,i\\geq 0$. Given any real number $ \\,a > 1,\\,$ construct a bounded infinite sequence $ x_{0},x_{1},x_{2},\\ldots \\,$ such that\n\\[ \\vert x_{i} \\minus{} x_{j} \\vert \\vert i \\minus{} j \\vert^{a}\\geq 1\n\\]\nfor every pair of distinct nonnegative integers $ i, j$.",
      "answer": "1"
    },
    {
      "question": "How many integers are greater than $\\sqrt{15}$ and less than $\\sqrt{50}$?",
      "answer": "4"
    },
    {
      "question": "Find the number of terms $n \\leq 2012$ such that $a_{n}=\\frac{3^{n+1}-1}{2}$ is divisible by 7.",
      "answer": "335"
    },
    {
      "question": "There are two buildings facing each other, each 5 stories high. How many ways can Kevin string ziplines between the buildings so that: (a) each zipline starts and ends in the middle of a floor. (b) ziplines can go up, stay flat, or go down, but can't touch each other (this includes touching at their endpoints). Note that you can't string a zipline between two floors of the same building.",
      "answer": "252"
    },
    {
      "question": "A repunit is a positive integer, all of whose digits are 1s. Let $a_{1}<a_{2}<a_{3}<\\ldots$ be a list of all the positive integers that can be expressed as the sum of distinct repunits. Compute $a_{111}$.",
      "answer": "1223456"
    },
    {
      "question": "Quadrilateral $A B C D$ satisfies $A B=8, B C=5, C D=17, D A=10$. Let $E$ be the intersection of $A C$ and $B D$. Suppose $B E: E D=1: 2$. Find the area of $A B C D$.",
      "answer": "60"
    },
    {
      "question": "Lucas writes two distinct positive integers on a whiteboard. He decreases the smaller number by 20 and increases the larger number by 23 , only to discover the product of the two original numbers is equal to the product of the two altered numbers. Compute the minimum possible sum of the original two numbers on the board.",
      "answer": "321"
    },
    {
      "question": "A $3 \\times 3$ table starts with every entry equal to 0 and is modified using the following steps: (i) adding 1 to all three numbers in any row; (ii) adding 2 to all three numbers in any column. After step (i) has been used a total of $a$ times and step (ii) has been used a total of $b$ times, the table appears as \\begin{tabular}{|l|l|l|} \\hline 7 & 1 & 5 \\\\ \\hline 9 & 3 & 7 \\\\ \\hline 8 & 2 & 6 \\\\ \\hline \\end{tabular} shown. What is the value of $a+b$?",
      "answer": "11"
    },
    {
      "question": "Fisica and Ritmo discovered a piece of Notalium shaped like a rectangular box, and wanted to find its volume. To do so, Fisica measured its three dimensions using a ruler with infinite precision, multiplied the results and rounded the product to the nearest cubic centimeter, getting a result of 2017 cubic centimeters. Ritmo, on the other hand, measured each dimension to the nearest centimeter and multiplied the rounded measurements, getting a result of $V$ cubic centimeters. Find the positive difference between the least and greatest possible positive values for $V$.",
      "answer": "7174"
    },
    {
      "question": "A sequence has 101 terms, each of which is a positive integer. If a term, $n$, is even, the next term is equal to $\\frac{1}{2}n+1$. If a term, $n$, is odd, the next term is equal to $\\frac{1}{2}(n+1)$. If the first term is 16, what is the 101st term?",
      "answer": "2"
    },
    {
      "question": "Let $a$ and $b$ be positive integers for which $45a+b=2021$. What is the minimum possible value of $a+b$?",
      "answer": "85"
    },
    {
      "question": "In the future, MIT has attracted so many students that its buildings have become skyscrapers. Ben and Jerry decide to go ziplining together. Ben starts at the top of the Green Building, and ziplines to the bottom of the Stata Center. After waiting $a$ seconds, Jerry starts at the top of the Stata Center, and ziplines to the bottom of the Green Building. The Green Building is 160 meters tall, the Stata Center is 90 meters tall, and the two buildings are 120 meters apart. Furthermore, both zipline at 10 meters per second. Given that Ben and Jerry meet at the point where the two ziplines cross, compute $100 a$.",
      "answer": "740"
    },
    {
      "question": "You are the general of an army. You and the opposing general both have an equal number of troops to distribute among three battlefields. Whoever has more troops on a battlefield always wins (you win ties). An order is an ordered triple of non-negative real numbers $(x, y, z)$ such that $x+y+z=1$, and corresponds to sending a fraction $x$ of the troops to the first field, $y$ to the second, and $z$ to the third. Suppose that you give the order $\\left(\\frac{1}{4}, \\frac{1}{4}, \\frac{1}{2}\\right)$ and that the other general issues an order chosen uniformly at random from all possible orders. What is the probability that you win two out of the three battles?",
      "answer": "\\sqrt[5]{8}"
    },
    {
      "question": "Suppose that $a$ and $b$ are integers with $4<a<b<22$. If the average (mean) of the numbers $4, a, b, 22$ is 13, how many possible pairs $(a, b)$ are there?",
      "answer": "8"
    },
    {
      "question": "Knowing that the system\n\\[x + y + z = 3,\\]\\[x^3 + y^3 + z^3 = 15,\\]\\[x^4 + y^4 + z^4 = 35,\\]\nhas a real solution $x, y, z$ for which $x^2 + y^2 + z^2 < 10$, find the value of $x^5 + y^5 + z^5$ for that solution.",
      "answer": "83"
    },
    {
      "question": "Find the smallest integer $n$ such that each subset of $\\{1,2,\\ldots, 2004\\}$ with $n$ elements has two distinct elements $a$ and $b$ for which $a^2-b^2$ is a multiple of $2004$.",
      "answer": "1003"
    },
    {
      "question": "Compute the number of permutations $\\pi$ of the set $\\{1,2, \\ldots, 10\\}$ so that for all (not necessarily distinct) $m, n \\in\\{1,2, \\ldots, 10\\}$ where $m+n$ is prime, $\\pi(m)+\\pi(n)$ is prime.",
      "answer": "4"
    },
    {
      "question": "What is the least number of gumballs that Wally must buy to guarantee that he receives 3 gumballs of the same colour?",
      "answer": "8"
    },
    {
      "question": "The expression $\\left(1+\\frac{1}{2}\\right)\\left(1+\\frac{1}{3}\\right)\\left(1+\\frac{1}{4}\\right)\\left(1+\\frac{1}{5}\\right)\\left(1+\\frac{1}{6}\\right)\\left(1+\\frac{1}{7}\\right)\\left(1+\\frac{1}{8}\\right)\\left(1+\\frac{1}{9}\\right)$ is equal to what?",
      "answer": "5"
    },
    {
      "question": "The function $f(n)$ is defined on the positive integers and takes non-negative integer values. $f(2)=0,f(3)>0,f(9999)=3333$ and for all $m,n:$ \\[ f(m+n)-f(m)-f(n)=0 \\text{ or } 1. \\] Determine $f(1982)$.",
      "answer": "660"
    },
    {
      "question": "Alain and Louise are driving on a circular track with radius 25 km. Alain leaves the starting line first, going clockwise at 80 km/h. Fifteen minutes later, Louise leaves the same starting line, going counterclockwise at 100 km/h. For how many hours will Louise have been driving when they pass each other for the fourth time?",
      "answer": "\\\\frac{10\\\\pi-1}{9}"
    },
    {
      "question": "Find the number of real zeros of $x^{3}-x^{2}-x+2$.",
      "answer": "1"
    },
    {
      "question": "Find the volume of the three-dimensional solid given by the inequality $\\sqrt{x^{2}+y^{2}}+$ $|z| \\leq 1$.",
      "answer": "2 \\pi / 3"
    },
    {
      "question": "A basket contains 12 apples and 15 bananas. If 3 more bananas are added to the basket, what fraction of the fruit in the basket will be bananas?",
      "answer": "\\frac{3}{5}"
    },
    {
      "question": "An $n$-string is a string of digits formed by writing the numbers $1,2, \\ldots, n$ in some order (in base ten). For example, one possible 10-string is $$35728910461$$ What is the smallest $n>1$ such that there exists a palindromic $n$-string?",
      "answer": "19"
    },
    {
      "question": "In a $16 \\times 16$ table of integers, each row and column contains at most 4 distinct integers. What is the maximum number of distinct integers that there can be in the whole table?",
      "answer": "49"
    },
    {
      "question": "We colour all the sides and diagonals of a regular polygon $P$ with $43$ vertices either\nred or blue in such a way that every vertex is an endpoint of $20$ red segments and $22$ blue segments.\nA triangle formed by vertices of $P$ is called monochromatic if all of its sides have the same colour.\nSuppose that there are $2022$ blue monochromatic triangles. How many red monochromatic triangles\nare there?",
      "answer": "859"
    },
    {
      "question": "Let $S$ be the sum of all the real coefficients of the expansion of $(1+i x)^{2009}$. What is $\\log _{2}(S)$ ?",
      "answer": "1004"
    },
    {
      "question": "Four points are chosen uniformly and independently at random in the interior of a given circle. Find the probability that they are the vertices of a convex quadrilateral.",
      "answer": "1 - \\frac{35}{12 \\pi^2}"
    },
    {
      "question": "A set consists of five different odd positive integers, each greater than 2. When these five integers are multiplied together, their product is a five-digit integer of the form $AB0AB$, where $A$ and $B$ are digits with $A \\neq 0$ and $A \\neq B$. (The hundreds digit of the product is zero.) For example, the integers in the set $\\{3,5,7,13,33\\}$ have a product of 45045. In total, how many different sets of five different odd positive integers have these properties?",
      "answer": "24"
    },
    {
      "question": "Let $ABC$ be a right triangle with $\\angle A=90^{\\circ}$. Let $D$ be the midpoint of $AB$ and let $E$ be a point on segment $AC$ such that $AD=AE$. Let $BE$ meet $CD$ at $F$. If $\\angle BFC=135^{\\circ}$, determine $BC/AB$.",
      "answer": "\\frac{\\sqrt{13}}{2}"
    },
    {
      "question": "Find the number of 7 -tuples $\\left(n_{1}, \\ldots, n_{7}\\right)$ of integers such that $$\\sum_{i=1}^{7} n_{i}^{6}=96957$$",
      "answer": "2688"
    },
    {
      "question": "Sherry is waiting for a train. Every minute, there is a $75 \\%$ chance that a train will arrive. However, she is engrossed in her game of sudoku, so even if a train arrives she has a $75 \\%$ chance of not noticing it (and hence missing the train). What is the probability that Sherry catches the train in the next five minutes?",
      "answer": "1-\\left(\\frac{13}{16}\\right)^{5}"
    },
    {
      "question": "The numbers $1,2, \\ldots, 10$ are written in a circle. There are four people, and each person randomly selects five consecutive integers (e.g. $1,2,3,4,5$, or $8,9,10,1,2$). If the probability that there exists some number that was not selected by any of the four people is $p$, compute $10000p$.",
      "answer": "3690"
    },
    {
      "question": "Let $p, q, r$ be primes such that $2 p+3 q=6 r$. Find $p+q+r$.",
      "answer": "7"
    },
    {
      "question": "A 24-hour digital clock shows times $h: m: s$, where $h, m$, and $s$ are integers with $0 \\leq h \\leq 23$, $0 \\leq m \\leq 59$, and $0 \\leq s \\leq 59$. How many times $h: m: s$ satisfy $h+m=s$?",
      "answer": "1164"
    },
    {
      "question": "In triangle $A B C$, let the parabola with focus $A$ and directrix $B C$ intersect sides $A B$ and $A C$ at $A_{1}$ and $A_{2}$, respectively. Similarly, let the parabola with focus $B$ and directrix $C A$ intersect sides $B C$ and $B A$ at $B_{1}$ and $B_{2}$, respectively. Finally, let the parabola with focus $C$ and directrix $A B$ intersect sides $C A$ and $C B$ at $C_{1}$ and $C_{2}$, respectively. If triangle $A B C$ has sides of length 5,12, and 13, find the area of the triangle determined by lines $A_{1} C_{2}, B_{1} A_{2}$ and $C_{1} B_{2}$.",
      "answer": "\\frac{6728}{3375}"
    },
    {
      "question": "Let $A B C D$ be a parallelogram with $A B=8, A D=11$, and $\\angle B A D=60^{\\circ}$. Let $X$ be on segment $C D$ with $C X / X D=1 / 3$ and $Y$ be on segment $A D$ with $A Y / Y D=1 / 2$. Let $Z$ be on segment $A B$ such that $A X, B Y$, and $D Z$ are concurrent. Determine the area of triangle $X Y Z$.",
      "answer": "\\frac{19 \\sqrt{3}}{2}"
    },
    {
      "question": "Carl is on a vertex of a regular pentagon. Every minute, he randomly selects an adjacent vertex (each with probability $\\frac{1}{2}$ ) and walks along the edge to it. What is the probability that after 10 minutes, he ends up where he had started?",
      "answer": "\\frac{127}{512}"
    },
    {
      "question": "Find the smallest positive integer $n$ such that the $73$ fractions $\\frac{19}{n+21}, \\frac{20}{n+22},\\frac{21}{n+23},...,\\frac{91}{n+93}$ are all irreducible.",
      "answer": "95"
    },
    {
      "question": "Find the probability that a monkey typing randomly on a typewriter will type the string 'abc' before 'aaa'.",
      "answer": "\\frac{3}{7}"
    },
    {
      "question": "Let $\\omega_{1}, \\omega_{2}, \\ldots, \\omega_{100}$ be the roots of $\\frac{x^{101}-1}{x-1}$ (in some order). Consider the set $S=\\left\\{\\omega_{1}^{1}, \\omega_{2}^{2}, \\omega_{3}^{3}, \\ldots, \\omega_{100}^{100}\\right\\}$. Let $M$ be the maximum possible number of unique values in $S$, and let $N$ be the minimum possible number of unique values in $S$. Find $M-N$.",
      "answer": "98"
    },
    {
      "question": "A perfect power is an integer $n$ that can be represented as $a^{k}$ for some positive integers $a \\geq 1$ and $k \\geq 2$. Find the sum of all prime numbers $0<p<50$ such that $p$ is 1 less than a perfect power.",
      "answer": "41"
    },
    {
      "question": "Calculate the probability of the Alphas winning given the probability of the Reals hitting 0, 1, 2, 3, or 4 singles.",
      "answer": "\\frac{224}{243}"
    },
    {
      "question": "Find the number of pentominoes (5-square polyominoes) that span a 3-by-3 rectangle, where polyominoes that are flips or rotations of each other are considered the same polyomino.",
      "answer": "6"
    },
    {
      "question": "Define a number to be an anti-palindrome if, when written in base 3 as $a_{n} a_{n-1} \\ldots a_{0}$, then $a_{i}+a_{n-i}=2$ for any $0 \\leq i \\leq n$. Find the number of anti-palindromes less than $3^{12}$ such that no two consecutive digits in base 3 are equal.",
      "answer": "126"
    },
    {
      "question": "Complex number $\\omega$ satisfies $\\omega^{5}=2$. Find the sum of all possible values of $\\omega^{4}+\\omega^{3}+\\omega^{2}+\\omega+1$.",
      "answer": "5"
    },
    {
      "question": "A sequence of real numbers $a_{0}, a_{1}, \\ldots, a_{9}$ with $a_{0}=0, a_{1}=1$, and $a_{2}>0$ satisfies $$a_{n+2} a_{n} a_{n-1}=a_{n+2}+a_{n}+a_{n-1}$$ for all $1 \\leq n \\leq 7$, but cannot be extended to $a_{10}$. In other words, no values of $a_{10} \\in \\mathbb{R}$ satisfy $$a_{10} a_{8} a_{7}=a_{10}+a_{8}+a_{7}$$ Compute the smallest possible value of $a_{2}$.",
      "answer": "\\sqrt{2}-1"
    },
    {
      "question": "Consider an $8 \\times 8$ grid of squares. A rook is placed in the lower left corner, and every minute it moves to a square in the same row or column with equal probability (the rook must move; i.e. it cannot stay in the same square). What is the expected number of minutes until the rook reaches the upper right corner?",
      "answer": "70"
    },
    {
      "question": "How many sequences of 0s and 1s are there of length 10 such that there are no three 0s or 1s consecutively anywhere in the sequence?",
      "answer": "178"
    },
    {
      "question": "Rosencrantz plays $n \\leq 2015$ games of question, and ends up with a win rate (i.e. $\\frac{\\# \\text { of games won }}{\\# \\text { of games played }}$ ) of $k$. Guildenstern has also played several games, and has a win rate less than $k$. He realizes that if, after playing some more games, his win rate becomes higher than $k$, then there must have been some point in time when Rosencrantz and Guildenstern had the exact same win-rate. Find the product of all possible values of $k$.",
      "answer": "\\frac{1}{2015}"
    },
    {
      "question": "Let $S=\\{1,2, \\ldots 2016\\}$, and let $f$ be a randomly chosen bijection from $S$ to itself. Let $n$ be the smallest positive integer such that $f^{(n)}(1)=1$, where $f^{(i)}(x)=f\\left(f^{(i-1)}(x)\\right)$. What is the expected value of $n$?",
      "answer": "\\frac{2017}{2}"
    },
    {
      "question": "Triangle $\\triangle P Q R$, with $P Q=P R=5$ and $Q R=6$, is inscribed in circle $\\omega$. Compute the radius of the circle with center on $\\overline{Q R}$ which is tangent to both $\\omega$ and $\\overline{P Q}$.",
      "answer": "\\frac{20}{9}"
    },
    {
      "question": "Seven little children sit in a circle. The teacher distributes pieces of candy to the children in such a way that the following conditions hold. - Every little child gets at least one piece of candy. - No two little children have the same number of pieces of candy. - The numbers of candy pieces given to any two adjacent little children have a common factor other than 1. - There is no prime dividing every little child's number of candy pieces. What is the smallest number of pieces of candy that the teacher must have ready for the little children?",
      "answer": "44"
    },
    {
      "question": "An up-right path between two lattice points $P$ and $Q$ is a path from $P$ to $Q$ that takes steps of length 1 unit either up or to the right. How many up-right paths from $(0,0)$ to $(7,7)$, when drawn in the plane with the line $y=x-2.021$, enclose exactly one bounded region below that line?",
      "answer": "637"
    },
    {
      "question": "Let $z_{1}, z_{2}, z_{3}, z_{4}$ be the solutions to the equation $x^{4}+3 x^{3}+3 x^{2}+3 x+1=0$. Then $\\left|z_{1}\\right|+\\left|z_{2}\\right|+\\left|z_{3}\\right|+\\left|z_{4}\\right|$ can be written as $\\frac{a+b \\sqrt{c}}{d}$, where $c$ is a square-free positive integer, and $a, b, d$ are positive integers with $\\operatorname{gcd}(a, b, d)=1$. Compute $1000 a+100 b+10 c+d$.",
      "answer": "7152"
    },
    {
      "question": "Call an positive integer almost-square if it can be written as $a \\cdot b$, where $a$ and $b$ are integers and $a \\leq b \\leq \\frac{4}{3} a$. How many almost-square positive integers are less than or equal to 1000000 ? Your score will be equal to $25-65 \\frac{|A-C|}{\\min (A, C)}$.",
      "answer": "130348"
    },
    {
      "question": "For any finite set $S$, let $f(S)$ be the sum of the elements of $S$ (if $S$ is empty then $f(S)=0$). Find the sum over all subsets $E$ of $S$ of $\\frac{f(E)}{f(S)}$ for $S=\\{1,2, \\ldots, 1999\\}$.",
      "answer": "2^{1998}"
    },
    {
      "question": "Let $f(x)=x^{2}+6 x+7$. Determine the smallest possible value of $f(f(f(f(x))))$ over all real numbers $x$.",
      "answer": "23"
    },
    {
      "question": "The numbers $1,2, \\ldots, 10$ are randomly arranged in a circle. Let $p$ be the probability that for every positive integer $k<10$, there exists an integer $k^{\\prime}>k$ such that there is at most one number between $k$ and $k^{\\prime}$ in the circle. If $p$ can be expressed as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
      "answer": "1390"
    },
    {
      "question": "Find the number of positive integer divisors of 12 ! that leave a remainder of 1 when divided by 3.",
      "answer": "66"
    },
    {
      "question": "In a certain country, there are 100 senators, each of whom has 4 aides. These senators and aides serve on various committees. A committee may consist either of 5 senators, of 4 senators and 4 aides, or of 2 senators and 12 aides. Every senator serves on 5 committees, and every aide serves on 3 committees. How many committees are there altogether?",
      "answer": "160"
    },
    {
      "question": "If \\( 3x + 4 = x + 2 \\), what is the value of \\( x \\)?",
      "answer": "-1"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=A C=5$ and $B C=6$. Denote by $\\omega$ the circumcircle of $A B C$. We draw a circle $\\Omega$ which is externally tangent to $\\omega$ as well as to the lines $A B$ and $A C$ (such a circle is called an $A$-mixtilinear excircle). Find the radius of $\\Omega$.",
      "answer": "\\frac{75}{8}"
    },
    {
      "question": "Let $r = \\sqrt{\\frac{\\sqrt{53}}{2} + \\frac{3}{2}}$. There is a unique triple of positive integers $(a, b, c)$ such that $r^{100} = 2r^{98} + 14r^{96} + 11r^{94} - r^{50} + ar^{46} + br^{44} + cr^{40}$. What is the value of $a^{2} + b^{2} + c^{2}$?",
      "answer": "15339"
    },
    {
      "question": "You have 128 teams in a single elimination tournament. The Engineers and the Crimson are two of these teams. Each of the 128 teams in the tournament is equally strong, so during each match, each team has an equal probability of winning. Now, the 128 teams are randomly put into the bracket. What is the probability that the Engineers play the Crimson sometime during the tournament?",
      "answer": "\\frac{1}{64}"
    },
    {
      "question": "In each cell of a $4 \\times 4$ grid, one of the two diagonals is drawn uniformly at random. Compute the probability that the resulting 32 triangular regions can be colored red and blue so that any two regions sharing an edge have different colors.",
      "answer": "\\frac{1}{512}"
    },
    {
      "question": "Consider an equilateral triangle and a square both inscribed in a unit circle such that one side of the square is parallel to one side of the triangle. Compute the area of the convex heptagon formed by the vertices of both the triangle and the square.",
      "answer": "\\frac{3+\\sqrt{3}}{2}"
    },
    {
      "question": "A restricted path of length $n$ is a path of length $n$ such that for all $i$ between 1 and $n-2$ inclusive, if the $i$th step is upward, the $i+1$st step must be rightward. Find the number of restricted paths that start at $(0,0)$ and end at $(7,3)$.",
      "answer": "56"
    },
    {
      "question": "Define $\\phi^{!}(n)$ as the product of all positive integers less than or equal to $n$ and relatively prime to $n$. Compute the remainder when $$ \\sum_{\\substack{2 \\leq n \\leq 50 \\\\ \\operatorname{gcd}(n, 50)=1}} \\phi^{!}(n) $$ is divided by 50 .",
      "answer": "12"
    },
    {
      "question": "Let $N$ be the number of triples of positive integers $(a, b, c)$ satisfying $a \\leq b \\leq c, \\quad \\operatorname{gcd}(a, b, c)=1, \\quad a b c=6^{2020}$. Compute the remainder when $N$ is divided by 1000.",
      "answer": "602"
    },
    {
      "question": "Evaluate $\\sum_{n=2}^{17} \\frac{n^{2}+n+1}{n^{4}+2 n^{3}-n^{2}-2 n}$.",
      "answer": "\\frac{592}{969}"
    },
    {
      "question": "We have a polyhedron such that an ant can walk from one vertex to another, traveling only along edges, and traversing every edge exactly once. What is the smallest possible total number of vertices, edges, and faces of this polyhedron?",
      "answer": "20"
    },
    {
      "question": "When $x=3$ and $y=4$, what is the value of the expression $xy-x$?",
      "answer": "9"
    },
    {
      "question": "If \\( x \\) and \\( y \\) are positive integers with \\( x>y \\) and \\( x+x y=391 \\), what is the value of \\( x+y \\)?",
      "answer": "39"
    },
    {
      "question": "On each cell of a $200 \\times 200$ grid, we place a car, which faces in one of the four cardinal directions. In a move, one chooses a car that does not have a car immediately in front of it, and slides it one cell forward. If a move would cause a car to exit the grid, the car is removed instead. The cars are placed so that there exists a sequence of moves that eventually removes all the cars from the grid. Across all such starting configurations, determine the maximum possible number of moves to do so.",
      "answer": "6014950"
    },
    {
      "question": "Compute the number of integers \\(n \\in\\{1,2, \\ldots, 300\\}\\) such that \\(n\\) is the product of two distinct primes, and is also the length of the longest leg of some nondegenerate right triangle with integer side lengths.",
      "answer": "13"
    },
    {
      "question": "What is the value of $x$ if the three numbers $2, x$, and 10 have an average of $x$?",
      "answer": "6"
    },
    {
      "question": "For dessert, Melinda eats a spherical scoop of ice cream with diameter 2 inches. She prefers to eat her ice cream in cube-like shapes, however. She has a special machine which, given a sphere placed in space, cuts it through the planes $x=n, y=n$, and $z=n$ for every integer $n$ (not necessarily positive). Melinda centers the scoop of ice cream uniformly at random inside the cube $0 \\leq x, y, z \\leq 1$, and then cuts it into pieces using her machine. What is the expected number of pieces she cuts the ice cream into?",
      "answer": "7+\\frac{13 \\pi}{3}"
    },
    {
      "question": "In how many different places in the $xy$-plane can a third point, $R$, be placed so that $PQ = QR = PR$ if points $P$ and $Q$ are two distinct points in the $xy$-plane?",
      "answer": "2"
    },
    {
      "question": "Let $a, b, c, d, e, f$ be integers selected from the set $\\{1,2, \\ldots, 100\\}$, uniformly and at random with replacement. Set $M=a+2 b+4 c+8 d+16 e+32 f$. What is the expected value of the remainder when $M$ is divided by 64?",
      "answer": "\\frac{63}{2}"
    },
    {
      "question": "Convex quadrilateral $A B C D$ has right angles $\\angle A$ and $\\angle C$ and is such that $A B=B C$ and $A D=C D$. The diagonals $A C$ and $B D$ intersect at point $M$. Points $P$ and $Q$ lie on the circumcircle of triangle $A M B$ and segment $C D$, respectively, such that points $P, M$, and $Q$ are collinear. Suppose that $m \\angle A B C=160^{\\circ}$ and $m \\angle Q M C=40^{\\circ}$. Find $M P \\cdot M Q$, given that $M C=6$.",
      "answer": "36"
    },
    {
      "question": "The unknown real numbers $x, y, z$ satisfy the equations $$\\frac{x+y}{1+z}=\\frac{1-z+z^{2}}{x^{2}-x y+y^{2}} ; \\quad \\frac{x-y}{3-z}=\\frac{9+3 z+z^{2}}{x^{2}+x y+y^{2}}$$ Find $x$.",
      "answer": "\\sqrt[3]{14}"
    },
    {
      "question": "At Barker High School, a total of 36 students are on either the baseball team, the hockey team, or both. If there are 25 students on the baseball team and 19 students on the hockey team, how many students play both sports?",
      "answer": "8"
    },
    {
      "question": "A triple of positive integers $(a, b, c)$ is tasty if $\\operatorname{lcm}(a, b, c) \\mid a+b+c-1$ and $a<b<c$. Find the sum of $a+b+c$ across all tasty triples.",
      "answer": "44"
    },
    {
      "question": "Caroline starts with the number 1, and every second she flips a fair coin; if it lands heads, she adds 1 to her number, and if it lands tails she multiplies her number by 2. Compute the expected number of seconds it takes for her number to become a multiple of 2021.",
      "answer": "4040"
    },
    {
      "question": "Divide an $m$-by-$n$ rectangle into $m n$ nonoverlapping 1-by-1 squares. A polyomino of this rectangle is a subset of these unit squares such that for any two unit squares $S, T$ in the polyomino, either (1) $S$ and $T$ share an edge or (2) there exists a positive integer $n$ such that the polyomino contains unit squares $S_{1}, S_{2}, S_{3}, \\ldots, S_{n}$ such that $S$ and $S_{1}$ share an edge, $S_{n}$ and $T$ share an edge, and for all positive integers $k<n, S_{k}$ and $S_{k+1}$ share an edge. We say a polyomino of a given rectangle spans the rectangle if for each of the four edges of the rectangle the polyomino contains a square whose edge lies on it. What is the minimum number of unit squares a polyomino can have if it spans a 128-by343 rectangle?",
      "answer": "470"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=5, A C=4, B C=6$. The angle bisector of $C$ intersects side $A B$ at $X$. Points $M$ and $N$ are drawn on sides $B C$ and $A C$, respectively, such that $\\overline{X M} \\| \\overline{A C}$ and $\\overline{X N} \\| \\overline{B C}$. Compute the length $M N$.",
      "answer": "\\frac{3 \\sqrt{14}}{5}"
    },
    {
      "question": "Compute $\\sum_{n=2009}^{\\infty} \\frac{1}{\\binom{n}{2009}}$",
      "answer": "\\frac{2009}{2008}"
    },
    {
      "question": "The point \\((p, q)\\) is on the line \\(y=\\frac{2}{5} x\\). Also, the area of the rectangle shown is 90. What is the value of \\(p\\)?",
      "answer": "15"
    },
    {
      "question": "Suppose that $f$ is a function from $\\mathbb{R}$ to $\\mathbb{R}$ such that \\[ f(x) + f\\left( 1 - \\frac{1}{x} \\right) = \\arctan x \\] for all real $x \\neq 0$. (As usual, $y = \\arctan x$ means $-\\pi/2 < y < \\pi/2$ and $\\tan y = x$.) Find \\[ \\int_0^1 f(x)\\,dx. \\]",
      "answer": "\\frac{3\\pi}{8}"
    },
    {
      "question": "How many elements are in the set obtained by transforming $\\{(0,0),(2,0)\\} 14$ times?",
      "answer": "477"
    },
    {
      "question": "For some integers $m$ and $n$, the expression $(x+m)(x+n)$ is equal to a quadratic expression in $x$ with a constant term of -12. Which of the following cannot be a value of $m$?",
      "answer": "5"
    },
    {
      "question": "A robotic grasshopper jumps 1 cm to the east, then 2 cm to the north, then 3 cm to the west, then 4 cm to the south. After every fourth jump, the grasshopper restarts the sequence of jumps. After a total of $n$ jumps, the position of the grasshopper is 162 cm to the west and 158 cm to the south of its original position. What is the sum of the squares of the digits of $n$?",
      "answer": "22"
    },
    {
      "question": "A rectangle has positive integer side lengths and an area of 24. What perimeter of the rectangle cannot be?",
      "answer": "36"
    },
    {
      "question": "Real numbers $x, y$, and $z$ are chosen from the interval $[-1,1]$ independently and uniformly at random. What is the probability that $|x|+|y|+|z|+|x+y+z|=|x+y|+|y+z|+|z+x|$?",
      "answer": "\\frac{3}{8}"
    },
    {
      "question": "How many solutions in nonnegative integers $(a, b, c)$ are there to the equation $2^{a}+2^{b}=c!\\quad ?$",
      "answer": "5"
    },
    {
      "question": "Consider the following flowchart: INPUT $\\rightarrow$ Subtract $8 \\rightarrow \\square \\rightarrow$ Divide by $2 \\rightarrow \\square$ Add $16 \\rightarrow$ OUTPUT. If the OUTPUT is 32, what was the INPUT?",
      "answer": "40"
    },
    {
      "question": "In 12 years, Janice will be 8 times as old as she was 2 years ago. How old is Janice now?",
      "answer": "4"
    },
    {
      "question": "For each \\(i \\in\\{1, \\ldots, 10\\}, a_{i}\\) is chosen independently and uniformly at random from \\([0, i^{2}]\\). Let \\(P\\) be the probability that \\(a_{1}<a_{2}<\\cdots<a_{10}\\). Estimate \\(P\\).",
      "answer": "0.003679"
    },
    {
      "question": "Let $\\omega_{1}$ be a circle of radius 5, and let $\\omega_{2}$ be a circle of radius 2 whose center lies on $\\omega_{1}$. Let the two circles intersect at $A$ and $B$, and let the tangents to $\\omega_{2}$ at $A$ and $B$ intersect at $P$. If the area of $\\triangle ABP$ can be expressed as $\\frac{a \\sqrt{b}}{c}$, where $b$ is square-free and $a, c$ are relatively prime positive integers, compute $100a+10b+c$.",
      "answer": "19285"
    },
    {
      "question": "If $2x + 6 = 16$, what is the value of $x + 4$?",
      "answer": "9"
    },
    {
      "question": "Suppose $$h \\cdot a \\cdot r \\cdot v \\cdot a \\cdot r \\cdot d=m \\cdot i \\cdot t=h \\cdot m \\cdot m \\cdot t=100$$ Find $(r \\cdot a \\cdot d) \\cdot(t \\cdot r \\cdot i \\cdot v \\cdot i \\cdot a)$.",
      "answer": "10000"
    },
    {
      "question": "The integer 48178 includes the block of digits 178. How many integers between 10000 and 100000 include the block of digits 178?",
      "answer": "280"
    },
    {
      "question": "How many pairs of integers $(a, b)$, with $1 \\leq a \\leq b \\leq 60$, have the property that $b$ is divisible by $a$ and $b+1$ is divisible by $a+1$?",
      "answer": "106"
    },
    {
      "question": "Define the sequences $(a_n),(b_n)$ by\n\\begin{align*}\n& a_n, b_n > 0, \\forall n\\in\\mathbb{N_+} \\\\ \n& a_{n+1} = a_n - \\frac{1}{1+\\sum_{i=1}^n\\frac{1}{a_i}} \\\\ \n& b_{n+1} = b_n + \\frac{1}{1+\\sum_{i=1}^n\\frac{1}{b_i}}\n\\end{align*}\n1) If $a_{100}b_{100} = a_{101}b_{101}$, find the value of $a_1-b_1$;\n2) If $a_{100} = b_{99}$, determine which is larger between $a_{100}+b_{100}$ and $a_{101}+b_{101}$.",
      "answer": "199"
    },
    {
      "question": "Regular octagon $C H I L D R E N$ has area 1. Find the area of pentagon $C H I L D$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Eight points are chosen on the circumference of a circle, labelled $P_{1}, P_{2}, \\ldots, P_{8}$ in clockwise order. A route is a sequence of at least two points $P_{a_{1}}, P_{a_{2}}, \\ldots, P_{a_{n}}$ such that if an ant were to visit these points in their given order, starting at $P_{a_{1}}$ and ending at $P_{a_{n}}$, by following $n-1$ straight line segments (each connecting each $P_{a_{i}}$ and $P_{a_{i+1}}$ ), it would never visit a point twice or cross its own path. Find the number of routes.",
      "answer": "8744"
    },
    {
      "question": "Let $A B C D$ be a quadrilateral inscribed in a unit circle with center $O$. Suppose that $\\angle A O B=\\angle C O D=135^{\\circ}, B C=1$. Let $B^{\\prime}$ and $C^{\\prime}$ be the reflections of $A$ across $B O$ and $C O$ respectively. Let $H_{1}$ and $H_{2}$ be the orthocenters of $A B^{\\prime} C^{\\prime}$ and $B C D$, respectively. If $M$ is the midpoint of $O H_{1}$, and $O^{\\prime}$ is the reflection of $O$ about the midpoint of $M H_{2}$, compute $O O^{\\prime}$.",
      "answer": "\\frac{1}{4}(8-\\sqrt{6}-3 \\sqrt{2})"
    },
    {
      "question": "Let $S=\\{1,2, \\ldots, 2008\\}$. For any nonempty subset $A \\subset S$, define $m(A)$ to be the median of $A$ (when $A$ has an even number of elements, $m(A)$ is the average of the middle two elements). Determine the average of $m(A)$, when $A$ is taken over all nonempty subsets of $S$.",
      "answer": "\\frac{2009}{2}"
    },
    {
      "question": "Let $A B C$ be an equilateral triangle with side length 1. Points $D, E, F$ lie inside triangle $A B C$ such that $A, E, F$ are collinear, $B, F, D$ are collinear, $C, D, E$ are collinear, and triangle $D E F$ is equilateral. Suppose that there exists a unique equilateral triangle $X Y Z$ with $X$ on side $\\overline{B C}, Y$ on side $\\overline{A B}$, and $Z$ on side $\\overline{A C}$ such that $D$ lies on side $\\overline{X Z}, E$ lies on side $\\overline{Y Z}$, and $F$ lies on side $\\overline{X Y}$. Compute $A Z$.",
      "answer": "\\frac{1}{1+\\sqrt[3]{2}}"
    },
    {
      "question": "Let $x, y$, and $z$ be distinct real numbers that sum to 0. Find the maximum possible value of $$\\frac{x y+y z+z x}{x^{2}+y^{2}+z^{2}}$$",
      "answer": "-1/2"
    },
    {
      "question": "Find the area of the region between a circle of radius 100 and a circle of radius 99.",
      "answer": "199 \\pi"
    },
    {
      "question": "A pebble is shaped as the intersection of a cube of side length 1 with the solid sphere tangent to all of the cube's edges. What is the surface area of this pebble?",
      "answer": "\\frac{6 \\sqrt{2}-5}{2} \\pi"
    },
    {
      "question": "In a magic square, what is the sum \\( a+b+c \\)?",
      "answer": "47"
    },
    {
      "question": "How many different numbers are obtainable from five 5s by first concatenating some of the 5s, then multiplying them together? For example, we could do $5 \\cdot 55 \\cdot 55,555 \\cdot 55$, or 55555, but not $5 \\cdot 5$ or 2525.",
      "answer": "7"
    },
    {
      "question": "Two players play a game where they are each given 10 indistinguishable units that must be distributed across three locations. (Units cannot be split.) At each location, a player wins at that location if the number of units they placed there is at least 2 more than the units of the other player. If both players distribute their units randomly (i.e. there is an equal probability of them distributing their units for any attainable distribution across the 3 locations), the probability that at least one location is won by one of the players can be expressed as $\\frac{a}{b}$, where $a, b$ are relatively prime positive integers. Compute $100a+b$.",
      "answer": "1011"
    },
    {
      "question": "Let $P R O B L E M Z$ be a regular octagon inscribed in a circle of unit radius. Diagonals $M R, O Z$ meet at $I$. Compute $L I$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "Find the largest possible integer $k$, such that the following statement is true:  \nLet $2009$ arbitrary non-degenerated triangles be given. In every triangle the three sides are coloured, such that one is blue, one is red and one is white. Now, for every colour separately, let us sort the lengths of the sides. We obtain\n\\[ \\left. \\begin{array}{rcl}\n & b_1 \\leq b_2\\leq\\ldots\\leq b_{2009} & \\textrm{the lengths of the blue sides }\\\\\n & r_1 \\leq r_2\\leq\\ldots\\leq r_{2009} & \\textrm{the lengths of the red sides }\\\\\n \\textrm{and } & w_1 \\leq w_2\\leq\\ldots\\leq w_{2009} & \\textrm{the lengths of the white sides }\\\\\n \\end{array}\\right.\\]\nThen there exist $k$ indices $j$ such that we can form a non-degenerated triangle with side lengths $b_j$, $r_j$, $w_j$.\n\n[i]",
      "answer": "1"
    },
    {
      "question": "An equilateral hexagon with side length 1 has interior angles $90^{\\circ}, 120^{\\circ}, 150^{\\circ}, 90^{\\circ}, 120^{\\circ}, 150^{\\circ}$ in that order. Find its area.",
      "answer": "\\frac{3+\\sqrt{3}}{2}"
    },
    {
      "question": "Find the largest positive integer $m$ which makes it possible to color several cells of a $70\\times 70$ table red such that [list] [*] There are no two red cells satisfying: the two rows in which they are have the same number of red cells, while the two columns in which they are also have the same number of red cells; [*] There are two rows with exactly $m$ red cells each. [/list]",
      "answer": "32"
    },
    {
      "question": "Alice, Bob, and Charlie are playing a game with 6 cards numbered 1 through 6. Each player is dealt 2 cards uniformly at random. On each player's turn, they play one of their cards, and the winner is the person who plays the median of the three cards played. Charlie goes last, so Alice and Bob decide to tell their cards to each other, trying to prevent him from winning whenever possible. Compute the probability that Charlie wins regardless.",
      "answer": "\\frac{2}{15}"
    },
    {
      "question": "Five people take a true-or-false test with five questions. Each person randomly guesses on every question. Given that, for each question, a majority of test-takers answered it correctly, let $p$ be the probability that every person answers exactly three questions correctly. Suppose that $p=\\frac{a}{2^{b}}$ where $a$ is an odd positive integer and $b$ is a nonnegative integer. Compute 100a+b.",
      "answer": "25517"
    },
    {
      "question": "Let $d$ be a randomly chosen divisor of 2016. Find the expected value of $\\frac{d^{2}}{d^{2}+2016}$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "A certain lottery has tickets labeled with the numbers $1,2,3, \\ldots, 1000$. The lottery is run as follows: First, a ticket is drawn at random. If the number on the ticket is odd, the drawing ends; if it is even, another ticket is randomly drawn (without replacement). If this new ticket has an odd number, the drawing ends; if it is even, another ticket is randomly drawn (again without replacement), and so forth, until an odd number is drawn. Then, every person whose ticket number was drawn (at any point in the process) wins a prize. You have ticket number 1000. What is the probability that you get a prize?",
      "answer": "1/501"
    },
    {
      "question": "Mr. Canada chooses a positive real $a$ uniformly at random from $(0,1]$, chooses a positive real $b$ uniformly at random from $(0,1]$, and then sets $c=a /(a+b)$. What is the probability that $c$ lies between $1 / 4$ and $3 / 4$ ?",
      "answer": "2 / 3"
    },
    {
      "question": "What is the smallest positive integer that is a multiple of each of 3, 5, 7, and 9?",
      "answer": "315"
    },
    {
      "question": "The numbers $a_{1}, a_{2}, \\ldots, a_{100}$ are a permutation of the numbers $1,2, \\ldots, 100$. Let $S_{1}=a_{1}$, $S_{2}=a_{1}+a_{2}, \\ldots, S_{100}=a_{1}+a_{2}+\\ldots+a_{100}$. What maximum number of perfect squares can be among the numbers $S_{1}, S_{2}, \\ldots, S_{100}$?",
      "answer": "60"
    },
    {
      "question": "Real numbers $a, b, c$ satisfy the equations $a+b+c=26,1 / a+1 / b+1 / c=28$. Find the value of $$\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{a}+\\frac{a}{c}+\\frac{c}{b}+\\frac{b}{a}$$",
      "answer": "725"
    },
    {
      "question": "A beaver walks from $(0,0)$ to $(4,4)$ in the plane, walking one unit in the positive $x$ direction or one unit in the positive $y$ direction at each step. Moreover, he never goes to a point $(x, y)$ with $y>x$. How many different paths can he walk?",
      "answer": "14"
    },
    {
      "question": "Let \\omega=\\cos \\frac{2 \\pi}{727}+i \\sin \\frac{2 \\pi}{727}$. The imaginary part of the complex number $$\\prod_{k=8}^{13}\\left(1+\\omega^{3^{k-1}}+\\omega^{2 \\cdot 3^{k-1}}\\right)$$ is equal to $\\sin \\alpha$ for some angle $\\alpha$ between $-\\frac{\\pi}{2}$ and $\\frac{\\pi}{2}$, inclusive. Find $\\alpha$.",
      "answer": "\\frac{12 \\pi}{727}"
    },
    {
      "question": "Victor has a drawer with 6 socks of 3 different types: 2 complex socks, 2 synthetic socks, and 2 trigonometric socks. He repeatedly draws 2 socks at a time from the drawer at random, and stops if the socks are of the same type. However, Victor is 'synthetic-complex type-blind', so he also stops if he sees a synthetic and a complex sock. What is the probability that Victor stops with 2 socks of the same type? Assume Victor returns both socks to the drawer after each step.",
      "answer": "\\frac{3}{7}"
    },
    {
      "question": "Compute the sum of all positive integers $n$ for which $9 \\sqrt{n}+4 \\sqrt{n+2}-3 \\sqrt{n+16}$ is an integer.",
      "answer": "18"
    },
    {
      "question": "Define a power cycle to be a set $S$ consisting of the nonnegative integer powers of an integer $a$, i.e. $S=\\left\\{1, a, a^{2}, \\ldots\\right\\}$ for some integer $a$. What is the minimum number of power cycles required such that given any odd integer $n$, there exists some integer $k$ in one of the power cycles such that $n \\equiv k$ $(\\bmod 1024) ?$",
      "answer": "10"
    },
    {
      "question": "It is midnight on April 29th, and Abigail is listening to a song by her favorite artist while staring at her clock, which has an hour, minute, and second hand. These hands move continuously. Between two consecutive midnights, compute the number of times the hour, minute, and second hands form two equal angles and no two hands overlap.",
      "answer": "5700"
    },
    {
      "question": "Find the smallest positive integer $n$ such that the divisors of $n$ can be partitioned into three sets with equal sums.",
      "answer": "120"
    },
    {
      "question": "Spencer is making burritos, each of which consists of one wrap and one filling. He has enough filling for up to four beef burritos and three chicken burritos. However, he only has five wraps for the burritos; in how many orders can he make exactly five burritos?",
      "answer": "25"
    },
    {
      "question": "On a given circle, six points $A$ , $B$ , $C$ , $D$ , $E$ , and $F$ are chosen at random, independently and uniformly with respect to arc length. Determine the probability that the two triangles $ABC$ and $DEF$ are disjoint, i.e., have no common points.",
      "answer": "\\[\n\\frac{3}{10}\n\\]"
    },
    {
      "question": "Let $D$ be a regular ten-sided polygon with edges of length 1. A triangle $T$ is defined by choosing three vertices of $D$ and connecting them with edges. How many different (non-congruent) triangles $T$ can be formed?",
      "answer": "8"
    },
    {
      "question": "Let $f(x)=c x(x-1)$, where $c$ is a positive real number. We use $f^{n}(x)$ to denote the polynomial obtained by composing $f$ with itself $n$ times. For every positive integer $n$, all the roots of $f^{n}(x)$ are real. What is the smallest possible value of $c$?",
      "answer": "2"
    },
    {
      "question": "Define $\\varphi^{k}(n)$ as the number of positive integers that are less than or equal to $n / k$ and relatively prime to $n$. Find $\\phi^{2001}\\left(2002^{2}-1\\right)$. (Hint: $\\phi(2003)=2002$.)",
      "answer": "1233"
    },
    {
      "question": "The average (mean) of a list of 10 numbers is 17. When one number is removed from the list, the new average is 16. What number was removed?",
      "answer": "26"
    },
    {
      "question": "Find the sum of every even positive integer less than 233 not divisible by 10.",
      "answer": "10812"
    },
    {
      "question": "How many real triples $(a, b, c)$ are there such that the polynomial $p(x)=x^{4}+a x^{3}+b x^{2}+a x+c$ has exactly three distinct roots, which are equal to $\\tan y, \\tan 2 y$, and $\\tan 3 y$ for some real $y$ ?",
      "answer": "18"
    },
    {
      "question": "A cylinder of base radius 1 is cut into two equal parts along a plane passing through the center of the cylinder and tangent to the two base circles. Suppose that each piece's surface area is $m$ times its volume. Find the greatest lower bound for all possible values of $m$ as the height of the cylinder varies.",
      "answer": "3"
    },
    {
      "question": "Plot points $A, B, C$ at coordinates $(0,0),(0,1)$, and $(1,1)$ in the plane, respectively. Let $S$ denote the union of the two line segments $A B$ and $B C$. Let $X_{1}$ be the area swept out when Bobby rotates $S$ counterclockwise 45 degrees about point $A$. Let $X_{2}$ be the area swept out when Calvin rotates $S$ clockwise 45 degrees about point $A$. Find $\\frac{X_{1}+X_{2}}{2}$.",
      "answer": "\\frac{\\pi}{4}"
    },
    {
      "question": "Let $S=\\{1,2, \\ldots, 2014\\}$. For each non-empty subset $T \\subseteq S$, one of its members is chosen as its representative. Find the number of ways to assign representatives to all non-empty subsets of $S$ so that if a subset $D \\subseteq S$ is a disjoint union of non-empty subsets $A, B, C \\subseteq S$, then the representative of $D$ is also the representative of at least one of $A, B, C$.",
      "answer": "\\[ 108 \\cdot 2014! \\]"
    },
    {
      "question": "A conical flask contains some water. When the flask is oriented so that its base is horizontal and lies at the bottom (so that the vertex is at the top), the water is 1 inch deep. When the flask is turned upside-down, so that the vertex is at the bottom, the water is 2 inches deep. What is the height of the cone?",
      "answer": "\\frac{1}{2}+\\frac{\\sqrt{93}}{6}"
    },
    {
      "question": "Two $4 \\times 4$ squares are randomly placed on an $8 \\times 8$ chessboard so that their sides lie along the grid lines of the board. What is the probability that the two squares overlap?",
      "answer": "529/625"
    },
    {
      "question": "For any positive integer $x$, define $\\operatorname{Accident}(x)$ to be the set of ordered pairs $(s, t)$ with $s \\in \\{0,2,4,5,7,9,11\\}$ and $t \\in\\{1,3,6,8,10\\}$ such that $x+s-t$ is divisible by 12. For any nonnegative integer $i$, let $a_{i}$ denote the number of $x \\in\\{0,1, \\ldots, 11\\}$ for which $|\\operatorname{Accident}(x)|=i$. Find $$a_{0}^{2}+a_{1}^{2}+a_{2}^{2}+a_{3}^{2}+a_{4}^{2}+a_{5}^{2}$$",
      "answer": "26"
    },
    {
      "question": "How many different combinations of 4 marbles can be made from 5 indistinguishable red marbles, 4 indistinguishable blue marbles, and 2 indistinguishable black marbles?",
      "answer": "12"
    },
    {
      "question": "The walls of a room are in the shape of a triangle $A B C$ with $\\angle A B C=90^{\\circ}, \\angle B A C=60^{\\circ}$, and $A B=6$. Chong stands at the midpoint of $B C$ and rolls a ball toward $A B$. Suppose that the ball bounces off $A B$, then $A C$, then returns exactly to Chong. Find the length of the path of the ball.",
      "answer": "3\\sqrt{21}"
    },
    {
      "question": "If $x+\\sqrt{81}=25$, what is the value of $x$?",
      "answer": "16"
    },
    {
      "question": "The Fibonacci sequence is defined as follows: $F_{0}=0, F_{1}=1$, and $F_{n}=F_{n-1}+F_{n-2}$ for all integers $n \\geq 2$. Find the smallest positive integer $m$ such that $F_{m} \\equiv 0(\\bmod 127)$ and $F_{m+1} \\equiv 1(\\bmod 127)$.",
      "answer": "256"
    },
    {
      "question": "For positive integers $n$ and $k$, let $\\mho(n, k)$ be the number of distinct prime divisors of $n$ that are at least $k$. Find the closest integer to $$\\sum_{n=1}^{\\infty} \\sum_{k=1}^{\\infty} \\frac{\\mho(n, k)}{3^{n+k-7}}$$",
      "answer": "167"
    },
    {
      "question": "A positive integer is called fancy if it can be expressed in the form $$2^{a_{1}}+2^{a_{2}}+\\cdots+2^{a_{100}}$$ where $a_{1}, a_{2}, \\ldots, a_{100}$ are non-negative integers that are not necessarily distinct. Find the smallest positive integer $n$ such that no multiple of $n$ is a fancy number.",
      "answer": "2^{101}-1"
    },
    {
      "question": "Find the minimum positive integer $k$ such that there exists a function $f$ from the set $\\Bbb{Z}$ of all integers to $\\{1, 2, \\ldots k\\}$ with the property that $f(x) \\neq f(y)$ whenever $|x-y| \\in \\{5, 7, 12\\}$.",
      "answer": "4"
    },
    {
      "question": "Somewhere in the universe, $n$ students are taking a 10-question math competition. Their collective performance is called laughable if, for some pair of questions, there exist 57 students such that either all of them answered both questions correctly or none of them answered both questions correctly. Compute the smallest $n$ such that the performance is necessarily laughable.",
      "answer": "253"
    },
    {
      "question": "Farmer Bill's 1000 animals - ducks, cows, and rabbits - are standing in a circle. In order to feel safe, every duck must either be standing next to at least one cow or between two rabbits. If there are 600 ducks, what is the least number of cows there can be for this to be possible?",
      "answer": "201"
    },
    {
      "question": "Let \\mathcal{V} be the volume enclosed by the graph $x^{2016}+y^{2016}+z^{2}=2016$. Find \\mathcal{V} rounded to the nearest multiple of ten.",
      "answer": "360"
    },
    {
      "question": "Five people of heights $65,66,67,68$, and 69 inches stand facing forwards in a line. How many orders are there for them to line up, if no person can stand immediately before or after someone who is exactly 1 inch taller or exactly 1 inch shorter than himself?",
      "answer": "14"
    },
    {
      "question": "What is the value of $1^{3}+2^{3}+3^{3}+4^{3}$?",
      "answer": "10^{2}"
    },
    {
      "question": "$A B C D$ is a rectangle with $A B=20$ and $B C=3$. A circle with radius 5, centered at the midpoint of $D C$, meets the rectangle at four points: $W, X, Y$, and $Z$. Find the area of quadrilateral $W X Y Z$.",
      "answer": "27"
    },
    {
      "question": "Evaluate the expression where the digit 2 appears 2013 times.",
      "answer": "\\frac{2013}{2014}"
    },
    {
      "question": "Suppose that $x, y$, and $z$ are complex numbers of equal magnitude that satisfy $$x+y+z=-\\frac{\\sqrt{3}}{2}-i \\sqrt{5}$$ and $$x y z=\\sqrt{3}+i \\sqrt{5}.$$ If $x=x_{1}+i x_{2}, y=y_{1}+i y_{2}$, and $z=z_{1}+i z_{2}$ for real $x_{1}, x_{2}, y_{1}, y_{2}, z_{1}$, and $z_{2}$, then $$\\left(x_{1} x_{2}+y_{1} y_{2}+z_{1} z_{2}\\right)^{2}$$ can be written as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
      "answer": "1516"
    },
    {
      "question": "Let $w, x, y, z$ be real numbers such that $w+x+y+z =5$, $2 w+4 x+8 y+16 z =7$, $3 w+9 x+27 y+81 z =11$, $4 w+16 x+64 y+256 z =1$. What is the value of $5 w+25 x+125 y+625 z ?$",
      "answer": "-60"
    },
    {
      "question": "Undecillion years ago in a galaxy far, far away, there were four space stations in the three-dimensional space, each pair spaced 1 light year away from each other. Admiral Ackbar wanted to establish a base somewhere in space such that the sum of squares of the distances from the base to each of the stations does not exceed 15 square light years. (The sizes of the space stations and the base are negligible.) Determine the volume, in cubic light years, of the set of all possible locations for the Admiral's base.",
      "answer": "\\frac{27 \\sqrt{6}}{8} \\pi"
    },
    {
      "question": "What is the minimum total number of boxes that Carley could have bought if each treat bag contains exactly 1 chocolate, 1 mint, and 1 caramel, and chocolates come in boxes of 50, mints in boxes of 40, and caramels in boxes of 25?",
      "answer": "17"
    },
    {
      "question": "Dizzy Daisy is standing on the point $(0,0)$ on the $xy$-plane and is trying to get to the point $(6,6)$. She starts facing rightward and takes a step 1 unit forward. On each subsequent second, she either takes a step 1 unit forward or turns 90 degrees counterclockwise then takes a step 1 unit forward. She may never go on a point outside the square defined by $|x| \\leq 6,|y| \\leq 6$, nor may she ever go on the same point twice. How many different paths may Daisy take?",
      "answer": "131922"
    },
    {
      "question": "Find the value of $1006 \\sin \\frac{\\pi}{1006}$. Approximating directly by $\\pi=3.1415 \\ldots$ is worth only 3 points.",
      "answer": "3.1415875473"
    },
    {
      "question": "Let $\\mathcal{P}_{1}, \\mathcal{P}_{2}, \\mathcal{P}_{3}$ be pairwise distinct parabolas in the plane. Find the maximum possible number of intersections between two or more of the $\\mathcal{P}_{i}$. In other words, find the maximum number of points that can lie on two or more of the parabolas $\\mathcal{P}_{1}, \\mathcal{P}_{2}, \\mathcal{P}_{3}$.",
      "answer": "12"
    },
    {
      "question": "Points $A, B, C, D$ lie on a circle in that order such that $\\frac{A B}{B C}=\\frac{D A}{C D}$. If $A C=3$ and $B D=B C=4$, find $A D$.",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "A certain state issues license plates consisting of six digits (from 0 through 9). The state requires that any two plates differ in at least two places. (Thus the plates $\\boxed{027592}$ and $\\boxed{020592}$ cannot both be used.) Determine, with proof, the maximum number of distinct license plates that the state can use.",
      "answer": "\\[ 10^5 \\]"
    },
    {
      "question": "In an equilateral triangle $\\triangle PRS$, if $QS=QT$ and $\\angle QTS=40^\\circ$, what is the value of $x$?",
      "answer": "80"
    },
    {
      "question": "For distinct positive integers $a, b<2012$, define $f(a, b)$ to be the number of integers $k$ with $1\\le k<2012$ such that the remainder when $ak$ divided by $2012$ is greater than that of $bk$ divided by $2012$. Let $S$ be the minimum value of $f(a, b)$, where $a$ and $b$ range over all pairs of distinct positive integers less than $2012$. Determine $S$.",
      "answer": "502"
    },
    {
      "question": "Let $\\lfloor x\\rfloor$ denote the largest integer less than or equal to $x$, and let $\\{x\\}$ denote the fractional part of $x$. For example, $\\lfloor\\pi\\rfloor=3$, and $\\{\\pi\\}=0.14159 \\ldots$, while $\\lfloor 100\\rfloor=100$ and $\\{100\\}=0$. If $n$ is the largest solution to the equation $\\frac{\\lfloor n\\rfloor}{n}=\\frac{2015}{2016}$, compute $\\{n\\}$.",
      "answer": "\\frac{2014}{2015}"
    },
    {
      "question": "We are given triangle $A B C$, with $A B=9, A C=10$, and $B C=12$, and a point $D$ on $B C . B$ and $C$ are reflected in $A D$ to $B^{\\prime}$ and $C^{\\prime}$, respectively. Suppose that lines $B C^{\\prime}$ and $B^{\\prime} C$ never meet (i.e., are parallel and distinct). Find $B D$.",
      "answer": "6"
    },
    {
      "question": "A positive integer $n$ is a multiple of 7. The square root of $n$ is between 17 and 18. How many possible values of $n$ are there?",
      "answer": "5"
    },
    {
      "question": "Solve the equation $$\\sqrt{x+\\sqrt{4x+\\sqrt{16x+\\sqrt{\\ldots+\\sqrt{4^{2008}x+3}}}}}-\\sqrt{x}=1$$",
      "answer": "\\frac{1}{2^{4016}}"
    },
    {
      "question": "Let $S$ be the set of all positive factors of 6000. What is the probability of a random quadruple $(a, b, c, d) \\in S^{4}$ satisfies $$\\operatorname{lcm}(\\operatorname{gcd}(a, b), \\operatorname{gcd}(c, d))=\\operatorname{gcd}(\\operatorname{lcm}(a, b), \\operatorname{lcm}(c, d)) ?$$",
      "answer": "\\frac{41}{512}"
    },
    {
      "question": "The Fibonacci numbers are defined by $F_{0}=0, F_{1}=1$, and $F_{n}=F_{n-1}+F_{n-2}$ for $n \\geq 2$. There exist unique positive integers $n_{1}, n_{2}, n_{3}, n_{4}, n_{5}, n_{6}$ such that $\\sum_{i_{1}=0}^{100} \\sum_{i_{2}=0}^{100} \\sum_{i_{3}=0}^{100} \\sum_{i_{4}=0}^{100} \\sum_{i_{5}=0}^{100} F_{i_{1}+i_{2}+i_{3}+i_{4}+i_{5}}=F_{n_{1}}-5 F_{n_{2}}+10 F_{n_{3}}-10 F_{n_{4}}+5 F_{n_{5}}-F_{n_{6}}$. Find $n_{1}+n_{2}+n_{3}+n_{4}+n_{5}+n_{6}$.",
      "answer": "1545"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=13, B C=14, C A=15$. The altitude from $A$ intersects $B C$ at $D$. Let $\\omega_{1}$ and $\\omega_{2}$ be the incircles of $A B D$ and $A C D$, and let the common external tangent of $\\omega_{1}$ and $\\omega_{2}$ (other than $B C$) intersect $A D$ at $E$. Compute the length of $A E$.",
      "answer": "7"
    },
    {
      "question": "Two positive rational numbers $x$ and $y$, when written in lowest terms, have the property that the sum of their numerators is 9 and the sum of their denominators is 10 . What is the largest possible value of $x+y$ ?",
      "answer": "73 / 9"
    },
    {
      "question": "Let the sequence $\\left\\{a_{i}\\right\\}_{i=0}^{\\infty}$ be defined by $a_{0}=\\frac{1}{2}$ and $a_{n}=1+\\left(a_{n-1}-1\\right)^{2}$. Find the product $$\\prod_{i=0}^{\\infty} a_{i}=a_{0} a_{1} a_{2}$$",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "How many real solutions are there to the equation $|||| x|-2|-2|-2|=|||| x|-3|-3|-3|$?",
      "answer": "6"
    },
    {
      "question": "Point $P$ is on the $y$-axis with $y$-coordinate greater than 0 and less than 100. A circle is drawn through $P, Q(4,4)$ and $O(0,0)$. How many possible positions for $P$ are there so that the radius of this circle is an integer?",
      "answer": "66"
    },
    {
      "question": "While waiting for their next class on Killian Court, Alesha and Belinda both write the same sequence $S$ on a piece of paper, where $S$ is a 2020-term strictly increasing geometric sequence with an integer common ratio $r$. Every second, Alesha erases the two smallest terms on her paper and replaces them with their geometric mean, while Belinda erases the two largest terms in her paper and replaces them with their geometric mean. They continue this process until Alesha is left with a single value $A$ and Belinda is left with a single value $B$. Let $r_{0}$ be the minimal value of $r$ such that $\\frac{A}{B}$ is an integer. If $d$ is the number of positive factors of $r_{0}$, what is the closest integer to $\\log _{2} d$ ?",
      "answer": "\\[\n\\boxed{2018}\n\\]"
    },
    {
      "question": "The vertices of a regular hexagon are labeled $\\cos (\\theta), \\cos (2 \\theta), \\ldots, \\cos (6 \\theta)$. For every pair of vertices, Bob draws a blue line through the vertices if one of these functions can be expressed as a polynomial function of the other (that holds for all real $\\theta$ ), and otherwise Roberta draws a red line through the vertices. In the resulting graph, how many triangles whose vertices lie on the hexagon have at least one red and at least one blue edge?",
      "answer": "14"
    },
    {
      "question": "How many 3-digit positive integers have exactly one even digit?",
      "answer": "350"
    },
    {
      "question": "There are two values of $k$ for which the equation $x^{2}+2kx+7k-10=0$ has two equal real roots (that is, has exactly one solution for $x$). What is the sum of these values of $k$?",
      "answer": "7"
    },
    {
      "question": "Mark and William are playing a game with a stored value. On his turn, a player may either multiply the stored value by 2 and add 1 or he may multiply the stored value by 4 and add 3. The first player to make the stored value exceed $2^{100}$ wins. The stored value starts at 1 and Mark goes first. Assuming both players play optimally, what is the maximum number of times that William can make a move?",
      "answer": "33"
    },
    {
      "question": "How many times does 24 divide into 100! (factorial)?",
      "answer": "32"
    },
    {
      "question": "Today, Ivan the Confessor prefers continuous functions \\(f:[0,1] \\rightarrow \\mathbb{R}\\) satisfying \\(f(x)+f(y) \\geq|x-y|\\) for all pairs \\(x, y \\in[0,1]\\). Find the minimum of \\(\\int_{0}^{1} f\\) over all preferred functions.",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Let $A,B,C,D$ denote four points in space such that at most one of the distances $AB,AC,AD,BC,BD,CD$ is greater than $1$ . Determine the maximum value of the sum of the six distances.",
      "answer": "\\[ 5 + \\sqrt{3} \\]"
    },
    {
      "question": "You have infinitely many boxes, and you randomly put 3 balls into them. The boxes are labeled $1,2, \\ldots$. Each ball has probability $1 / 2^{n}$ of being put into box $n$. The balls are placed independently of each other. What is the probability that some box will contain at least 2 balls?",
      "answer": "5 / 7"
    },
    {
      "question": "In triangle \\(A B C, A B=6, B C=7\\) and \\(C A=8\\). Let \\(D, E, F\\) be the midpoints of sides \\(B C\\), \\(A C, A B\\), respectively. Also let \\(O_{A}, O_{B}, O_{C}\\) be the circumcenters of triangles \\(A F D, B D E\\), and \\(C E F\\), respectively. Find the area of triangle \\(O_{A} O_{B} O_{C}\\).",
      "answer": "\\frac{21 \\sqrt{15}}{16}"
    },
    {
      "question": "How many ways are there to color the vertices of a triangle red, green, blue, or yellow such that no two vertices have the same color? Rotations and reflections are considered distinct.",
      "answer": "24"
    },
    {
      "question": "Let $S_{7}$ denote all the permutations of $1,2, \\ldots, 7$. For any \\pi \\in S_{7}$, let $f(\\pi)$ be the smallest positive integer $i$ such that \\pi(1), \\pi(2), \\ldots, \\pi(i)$ is a permutation of $1,2, \\ldots, i$. Compute \\sum_{\\pi \\in S_{7}} f(\\pi)$.",
      "answer": "29093"
    },
    {
      "question": "How many integers $n$ in the set $\\{4,9,14,19, \\ldots, 2014\\}$ have the property that the sum of the decimal digits of $n$ is even?",
      "answer": "201"
    },
    {
      "question": "Kelvin the frog currently sits at $(0,0)$ in the coordinate plane. If Kelvin is at $(x, y)$, either he can walk to any of $(x, y+1),(x+1, y)$, or $(x+1, y+1)$, or he can jump to any of $(x, y+2),(x+2, y)$ or $(x+1, y+1)$. Walking and jumping from $(x, y)$ to $(x+1, y+1)$ are considered distinct actions. Compute the number of ways Kelvin can reach $(6,8)$.",
      "answer": "1831830"
    },
    {
      "question": "$x, y$ are positive real numbers such that $x+y^{2}=x y$. What is the smallest possible value of $x$?",
      "answer": "4"
    },
    {
      "question": "Let $ABCDE$ be a convex pentagon such that $AB=AE=CD=1$ , $\\angle ABC=\\angle DEA=90^\\circ$ and $BC+DE=1$ . Compute the area of the pentagon.",
      "answer": "\\[ 1 \\]"
    },
    {
      "question": "Sylvia chose positive integers $a, b$ and $c$. Peter determined the value of $a + \\frac{b}{c}$ and got an answer of 101. Paul determined the value of $\\frac{a}{c} + b$ and got an answer of 68. Mary determined the value of $\\frac{a + b}{c}$ and got an answer of $k$. What is the value of $k$?",
      "answer": "13"
    },
    {
      "question": "There are 20 students in a class. In total, 10 of them have black hair, 5 of them wear glasses, and 3 of them both have black hair and wear glasses. How many of the students have black hair but do not wear glasses?",
      "answer": "7"
    },
    {
      "question": "Determine the number of unordered triples of distinct points in the $4 \\times 4 \\times 4$ lattice grid $\\{0,1,2,3\\}^{3}$ that are collinear in $\\mathbb{R}^{3}$ (i.e. there exists a line passing through the three points).",
      "answer": "376"
    },
    {
      "question": "Ten gangsters are standing on a flat surface, and the distances between them are all distinct. At twelve o’clock, when the church bells start chiming, each of them fatally shoots the one among the other nine gangsters who is the nearest. At least how many gangsters will be killed?",
      "answer": "7"
    },
    {
      "question": "Find all integers $n$, $n \\ge 1$, such that $n \\cdot 2^{n+1}+1$ is a perfect square.",
      "answer": "3"
    },
    {
      "question": "Which of the following is equal to $9^{4}$?",
      "answer": "3^{8}"
    },
    {
      "question": "Matt has somewhere between 1000 and 2000 pieces of paper he's trying to divide into piles of the same size (but not all in one pile or piles of one sheet each). He tries $2,3,4,5,6,7$, and 8 piles but ends up with one sheet left over each time. How many piles does he need?",
      "answer": "41"
    },
    {
      "question": "A sequence $s_{0}, s_{1}, s_{2}, s_{3}, \\ldots$ is defined by $s_{0}=s_{1}=1$ and, for every positive integer $n, s_{2 n}=s_{n}, s_{4 n+1}=s_{2 n+1}, s_{4 n-1}=s_{2 n-1}+s_{2 n-1}^{2} / s_{n-1}$. What is the value of $s_{1000}$?",
      "answer": "720"
    },
    {
      "question": "Mike rides his bicycle at a constant speed of $30 \\mathrm{~km} / \\mathrm{h}$. How many kilometres does Mike travel in 20 minutes?",
      "answer": "10"
    },
    {
      "question": "Erin walks $\\frac{3}{5}$ of the way home in 30 minutes. If she continues to walk at the same rate, how many minutes will it take her to walk the rest of the way home?",
      "answer": "20"
    },
    {
      "question": "On an $8 \\times 8$ chessboard, 6 black rooks and $k$ white rooks are placed on different cells so that each rook only attacks rooks of the opposite color. Compute the maximum possible value of $k$.",
      "answer": "14"
    },
    {
      "question": "If $x = -3$, what is the value of $(x-3)^{2}$?",
      "answer": "36"
    },
    {
      "question": "A circle with center at $O$ has radius 1. Points $P$ and $Q$ outside the circle are placed such that $P Q$ passes through $O$. Tangent lines to the circle through $P$ hit the circle at $P_{1}$ and $P_{2}$, and tangent lines to the circle through $Q$ hit the circle at $Q_{1}$ and $Q_{2}$. If $\\angle P_{1} P P_{2}=45^{\\circ}$ and angle $Q_{1} Q Q_{2}=30^{\\circ}$, find the minimum possible length of arc $P_{2} Q_{2}$.",
      "answer": "\\frac{\\pi}{12}"
    },
    {
      "question": "In acute $\\triangle A B C$ with centroid $G, A B=22$ and $A C=19$. Let $E$ and $F$ be the feet of the altitudes from $B$ and $C$ to $A C$ and $A B$ respectively. Let $G^{\\prime}$ be the reflection of $G$ over $B C$. If $E, F, G$, and $G^{\\prime}$ lie on a circle, compute $B C$.",
      "answer": "13"
    },
    {
      "question": "Calculate the value of $\\sqrt{\\frac{\\sqrt{81} + \\sqrt{81}}{2}}$.",
      "answer": "3"
    },
    {
      "question": "Let $A B C D$ be a rectangle with $A B=20$ and $A D=23$. Let $M$ be the midpoint of $C D$, and let $X$ be the reflection of $M$ across point $A$. Compute the area of triangle $X B D$.",
      "answer": "575"
    },
    {
      "question": "Consider a $6 \\times 6$ grid of squares. Edmond chooses four of these squares uniformly at random. What is the probability that the centers of these four squares form a square?",
      "answer": "\\frac{1}{561}"
    },
    {
      "question": "Five cards labeled A, B, C, D, and E are placed consecutively in a row. How many ways can they be re-arranged so that no card is moved more than one position away from where it started?",
      "answer": "8"
    },
    {
      "question": "A regular tetrahedron has a square shadow of area 16 when projected onto a flat surface (light is shone perpendicular onto the plane). Compute the sidelength of the regular tetrahedron.",
      "answer": "4 \\sqrt{2}"
    },
    {
      "question": "How many pairs of positive integers $(x, y)$ have the property that the ratio $x: 4$ equals the ratio $9: y$?",
      "answer": "9"
    },
    {
      "question": "Suppose $x$ satisfies $x^{3}+x^{2}+x+1=0$. What are all possible values of $x^{4}+2 x^{3}+2 x^{2}+2 x+1 ?$",
      "answer": "0"
    },
    {
      "question": "Chords $\\overline{A B}$ and $\\overline{C D}$ of circle $\\omega$ intersect at $E$ such that $A E=8, B E=2, C D=10$, and $\\angle A E C=90^{\\circ}$. Let $R$ be a rectangle inside $\\omega$ with sides parallel to $\\overline{A B}$ and $\\overline{C D}$, such that no point in the interior of $R$ lies on $\\overline{A B}, \\overline{C D}$, or the boundary of $\\omega$. What is the maximum possible area of $R$?",
      "answer": "26+6 \\sqrt{17}"
    },
    {
      "question": "Fifty numbers have an average of 76. Forty of these numbers have an average of 80. What is the average of the other ten numbers?",
      "answer": "60"
    },
    {
      "question": "Evaluate $\\sum_{i=1}^{\\infty} \\frac{(i+1)(i+2)(i+3)}{(-2)^{i}}$.",
      "answer": "96"
    },
    {
      "question": "A $5 \\times 5$ table is called regular if each of its cells contains one of four pairwise distinct real numbers, such that each of them occurs exactly once in every $2 \\times 2$ subtable.The sum of all numbers of a regular table is called the total sum of the table. With any four numbers, one constructs all possible regular tables, computes their total sums, and counts the distinct outcomes. Determine the maximum possible count.",
      "answer": "\\boxed{60}"
    },
    {
      "question": "Mary and Sally were once the same height. Since then, Sally grew \\( 20\\% \\) taller and Mary's height increased by half as many centimetres as Sally's height increased. Sally is now 180 cm tall. How tall, in cm, is Mary now?",
      "answer": "165"
    },
    {
      "question": "Suppose that $x$ and $y$ are real numbers with $-4 \\leq x \\leq -2$ and $2 \\leq y \\leq 4$. What is the greatest possible value of $\\frac{x+y}{x}$?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "What are the last 8 digits of $$11 \\times 101 \\times 1001 \\times 10001 \\times 100001 \\times 1000001 \\times 111 ?$$",
      "answer": "19754321"
    },
    {
      "question": "Rachel has the number 1000 in her hands. When she puts the number $x$ in her left pocket, the number changes to $x+1$. When she puts the number $x$ in her right pocket, the number changes to $x^{-1}$. Each minute, she flips a fair coin. If it lands heads, she puts the number into her left pocket, and if it lands tails, she puts it into her right pocket. She then takes the new number out of her pocket. If the expected value of the number in Rachel's hands after eight minutes is $E$, then compute $\\left\\lfloor\\frac{E}{10}\\right\\rfloor$.",
      "answer": "13"
    },
    {
      "question": "Kelvin the Frog likes numbers whose digits strictly decrease, but numbers that violate this condition in at most one place are good enough. In other words, if $d_{i}$ denotes the $i$ th digit, then $d_{i} \\leq d_{i+1}$ for at most one value of $i$. For example, Kelvin likes the numbers 43210, 132, and 3, but not the numbers 1337 and 123. How many 5-digit numbers does Kelvin like?",
      "answer": "14034"
    },
    {
      "question": "Carrie sends five text messages to her brother each Saturday and Sunday, and two messages on other days. Over four weeks, how many text messages does Carrie send?",
      "answer": "80"
    },
    {
      "question": "How many 48-tuples of positive integers \\(\\left(a_{1}, a_{2}, \\ldots, a_{48}\\right)\\) between 0 and 100 inclusive have the property that for all \\(1 \\leq i<j \\leq 48, a_{i} \\notin\\left\\{a_{j}, a_{j}+1\\right\\}\\) ?",
      "answer": "54^{48}"
    },
    {
      "question": "Compute the surface area of a cube inscribed in a sphere of surface area $\\pi$.",
      "answer": "2"
    },
    {
      "question": "What is the area of the region \\( \\mathcal{R} \\) formed by all points \\( L(r, t) \\) with \\( 0 \\leq r \\leq 10 \\) and \\( 0 \\leq t \\leq 10 \\) such that the area of triangle \\( JKL \\) is less than or equal to 10, where \\( J(2,7) \\) and \\( K(5,3) \\)?",
      "answer": "\\frac{313}{4}"
    },
    {
      "question": "A random binary string of length 1000 is chosen. Let \\(L\\) be the expected length of its longest (contiguous) palindromic substring. Estimate \\(L\\).",
      "answer": "23.120"
    },
    {
      "question": "A knight begins on the lower-left square of a standard chessboard. How many squares could the knight end up at after exactly 2009 legal knight's moves?",
      "answer": "32"
    },
    {
      "question": "Let $a,b,c,d$ be real numbers such that $b-d \\ge 5$ and all zeros $x_1, x_2, x_3,$ and $x_4$ of the polynomial $P(x)=x^4+ax^3+bx^2+cx+d$ are real. Find the smallest value the product $(x_1^2+1)(x_2^2+1)(x_3^2+1)(x_4^2+1)$ can take.",
      "answer": "\\boxed{16}"
    },
    {
      "question": "Let $ABC$ be an equilateral triangle of side length 6 inscribed in a circle $\\omega$. Let $A_{1}, A_{2}$ be the points (distinct from $A$) where the lines through $A$ passing through the two trisection points of $BC$ meet $\\omega$. Define $B_{1}, B_{2}, C_{1}, C_{2}$ similarly. Given that $A_{1}, A_{2}, B_{1}, B_{2}, C_{1}, C_{2}$ appear on $\\omega$ in that order, find the area of hexagon $A_{1}A_{2}B_{1}B_{2}C_{1}C_{2}$.",
      "answer": "\\frac{846\\sqrt{3}}{49}"
    },
    {
      "question": "Let $A B C$ be an isosceles triangle with $A B=A C$. Let $D$ and $E$ be the midpoints of segments $A B$ and $A C$, respectively. Suppose that there exists a point $F$ on ray $\\overrightarrow{D E}$ outside of $A B C$ such that triangle $B F A$ is similar to triangle $A B C$. Compute $\\frac{A B}{B C}$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "Now a ball is launched from a vertex of an equilateral triangle with side length 5. It strikes the opposite side after traveling a distance of $\\sqrt{19}$. How many times does the ball bounce before it returns to a vertex? (The final contact with a vertex does not count as a bounce.)",
      "answer": "7"
    },
    {
      "question": "Given that $\\sin A+\\sin B=1$ and $\\cos A+\\cos B=3 / 2$, what is the value of $\\cos (A-B)$?",
      "answer": "5/8"
    },
    {
      "question": "Which of the following numbers is less than $\\frac{1}{20}$?",
      "answer": "\\frac{1}{25}"
    },
    {
      "question": "The game of Penta is played with teams of five players each, and there are five roles the players can play. Each of the five players chooses two of five roles they wish to play. If each player chooses their roles randomly, what is the probability that each role will have exactly two players?",
      "answer": "\\frac{51}{2500}"
    },
    {
      "question": "Evaluate $\\frac{2016!^{2}}{2015!2017!}$. Here $n$ ! denotes $1 \\times 2 \\times \\cdots \\times n$.",
      "answer": "\\frac{2016}{2017}"
    },
    {
      "question": "Compute the value of $\\sqrt{105^{3}-104^{3}}$, given that it is a positive integer.",
      "answer": "181"
    },
    {
      "question": "Ali wants to move from point $A$ to point $B$. He cannot walk inside the black areas but he is free to move in any direction inside the white areas (not only the grid lines but the whole plane). Help Ali to find the shortest path between $A$ and $B$. Only draw the path and write its length.\n[img]https://1.bp.blogspot.com/-nZrxJLfIAp8/W1RyCdnhl3I/AAAAAAAAIzQ/NM3t5EtJWMcWQS0ig0IghSo54DQUBH5hwCK4BGAYYCw/s1600/igo%2B2016.el1.png[/img]\nby Morteza Saghafian",
      "answer": "7 + 5\\sqrt{2}"
    },
    {
      "question": "A rectangle has length 8 cm and width $\\pi$ cm. A semi-circle has the same area as the rectangle. What is its radius?",
      "answer": "4"
    },
    {
      "question": "Let $P$ be a point selected uniformly at random in the cube $[0,1]^{3}$. The plane parallel to $x+y+z=0$ passing through $P$ intersects the cube in a two-dimensional region $\\mathcal{R}$. Let $t$ be the expected value of the perimeter of $\\mathcal{R}$. If $t^{2}$ can be written as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers, compute $100 a+b$.",
      "answer": "12108"
    },
    {
      "question": "How many of the first 1000 positive integers can be written as the sum of finitely many distinct numbers from the sequence $3^{0}, 3^{1}, 3^{2}, \\ldots$?",
      "answer": "105"
    },
    {
      "question": "Consider five-dimensional Cartesian space $\\mathbb{R}^{5}=\\left\\{\\left(x_{1}, x_{2}, x_{3}, x_{4}, x_{5}\\right) \\mid x_{i} \\in \\mathbb{R}\\right\\}$ and consider the hyperplanes with the following equations: - $x_{i}=x_{j}$ for every $1 \\leq i<j \\leq 5$; - $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=-1$ - $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=0$ - $x_{1}+x_{2}+x_{3}+x_{4}+x_{5}=1$. Into how many regions do these hyperplanes divide $\\mathbb{R}^{5}$ ?",
      "answer": "480"
    },
    {
      "question": "Compute the number of ordered pairs of positive integers $(a, b)$ satisfying the equation $\\operatorname{gcd}(a, b) \\cdot a+b^{2}=10000$",
      "answer": "99"
    },
    {
      "question": "On Monday, Mukesh travelled \\(x \\mathrm{~km}\\) at a constant speed of \\(90 \\mathrm{~km} / \\mathrm{h}\\). On Tuesday, he travelled on the same route at a constant speed of \\(120 \\mathrm{~km} / \\mathrm{h}\\). His trip on Tuesday took 16 minutes less than his trip on Monday. What is the value of \\(x\\)?",
      "answer": "96"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=8, B C=15$, and $A C=17$. Point $X$ is chosen at random on line segment $A B$. Point $Y$ is chosen at random on line segment $B C$. Point $Z$ is chosen at random on line segment $C A$. What is the expected area of triangle $X Y Z$ ?",
      "answer": "15"
    },
    {
      "question": "Let $\\omega$ be a circle of radius 1 centered at $O$. Let $B$ be a point on $\\omega$, and let $l$ be the line tangent to $\\omega$ at $B$. Let $A$ be on $l$ such that $\\angle A O B=60^{\\circ}$. Let $C$ be the foot of the perpendicular from $B$ to $O A$. Find the length of line segment $O C$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Call a positive integer $n$ weird if $n$ does not divide $(n-2)$!. Determine the number of weird numbers between 2 and 100 inclusive.",
      "answer": "26"
    },
    {
      "question": "Let $a$ be the proportion of teams that correctly answered problem 1 on the Guts round. Estimate $A=\\lfloor 10000a\\rfloor$. An estimate of $E$ earns $\\max (0,\\lfloor 20-|A-E| / 20\\rfloor)$ points. If you have forgotten, question 1 was the following: Two hexagons are attached to form a new polygon $P$. What is the minimum number of sides that $P$ can have?",
      "answer": "2539"
    },
    {
      "question": "In a math test, there are easy and hard questions. The easy questions worth 3 points and the hard questions worth D points.\\\\\nIf all the questions begin to worth 4 points, the total punctuation of the test increases 16 points.\\\\\nInstead, if we exchange the questions scores, scoring D points for the easy questions and 3 for the hard ones, the total punctuation of the test is multiplied by $\\frac{3}{2}$.\\\\\nKnowing that the number of easy questions is 9 times bigger the number of hard questions, find the number of questions in this test.",
      "answer": "20"
    },
    {
      "question": "What is the largest positive integer $n$ that satisfies $n^{200}<3^{500}$?",
      "answer": "15"
    },
    {
      "question": "Four teams play in a tournament in which each team plays exactly one game against each of the other three teams. At the end of each game, either the two teams tie or one team wins and the other team loses. A team is awarded 3 points for a win, 0 points for a loss, and 1 point for a tie. If $S$ is the sum of the points of the four teams after the tournament is complete, which of the following values can $S$ not equal?",
      "answer": "11"
    },
    {
      "question": "Two circles have radii 13 and 30, and their centers are 41 units apart. The line through the centers of the two circles intersects the smaller circle at two points; let $A$ be the one outside the larger circle. Suppose $B$ is a point on the smaller circle and $C$ a point on the larger circle such that $B$ is the midpoint of $A C$. Compute the distance $A C$.",
      "answer": "12 \\sqrt{13}"
    },
    {
      "question": "Solve the system of equations: $20=4a^{2}+9b^{2}$ and $20+12ab=(2a+3b)^{2}$. Find $ab$.",
      "answer": "\\frac{20}{3}"
    },
    {
      "question": "The elevator buttons in Harvard's Science Center form a $3 \\times 2$ grid of identical buttons, and each button lights up when pressed. One day, a student is in the elevator when all the other lights in the elevator malfunction, so that only the buttons which are lit can be seen, but one cannot see which floors they correspond to. Given that at least one of the buttons is lit, how many distinct arrangements can the student observe? (For example, if only one button is lit, then the student will observe the same arrangement regardless of which button it is.)",
      "answer": "44"
    },
    {
      "question": "A monomial term $x_{i_{1}} x_{i_{2}} \\ldots x_{i_{k}}$ in the variables $x_{1}, x_{2}, \\ldots x_{8}$ is square-free if $i_{1}, i_{2}, \\ldots i_{k}$ are distinct. (A constant term such as 1 is considered square-free.) What is the sum of the coefficients of the squarefree terms in the following product? $$\\prod_{1 \\leq i<j \\leq 8}\\left(1+x_{i} x_{j}\\right)$$",
      "answer": "764"
    },
    {
      "question": "Let $P$ be a regular $2006$-gon. A diagonal is called [i]good[/i] if its endpoints divide the boundary of $P$ into two parts, each composed of an odd number of sides of $P$. The sides of $P$ are also called [i]good[/i].\nSuppose $P$ has been dissected into triangles by $2003$ diagonals, no two of which have a common point in the interior of $P$. Find the maximum number of isosceles triangles having two good sides that could appear in such a configuration.",
      "answer": "1003"
    },
    {
      "question": "A peacock is a ten-digit positive integer that uses each digit exactly once. Compute the number of peacocks that are exactly twice another peacock.",
      "answer": "184320"
    },
    {
      "question": "Determine the value of the sum $$\\frac{3}{1^{2} \\cdot 2^{2}}+\\frac{5}{2^{2} \\cdot 3^{2}}+\\frac{7}{3^{2} \\cdot 4^{2}}+\\cdots+\\frac{29}{14^{2} \\cdot 15^{2}}$$",
      "answer": "\\frac{224}{225}"
    },
    {
      "question": "Find the total number of different integer values the function $$f(x)=[x]+[2 x]+\\left[\\frac{5 x}{3}\\right]+[3 x]+[4 x]$$ takes for real numbers $x$ with $0 \\leq x \\leq 100$. Note: $[t]$ is the largest integer that does not exceed $t$.",
      "answer": "734"
    },
    {
      "question": "For $a$ a positive real number, let $x_{1}, x_{2}, x_{3}$ be the roots of the equation $x^{3}-a x^{2}+a x-a=0$. Determine the smallest possible value of $x_{1}^{3}+x_{2}^{3}+x_{3}^{3}-3 x_{1} x_{2} x_{3}$.",
      "answer": "-4"
    },
    {
      "question": "Let $n$ be a fixed positive integer. Determine the smallest possible rank of an $n \\times n$ matrix that has zeros along the main diagonal and strictly positive real numbers off the main diagonal.",
      "answer": "3"
    },
    {
      "question": "In equilateral triangle $ABC$ with side length 2, let the parabola with focus $A$ and directrix $BC$ intersect sides $AB$ and $AC$ at $A_{1}$ and $A_{2}$, respectively. Similarly, let the parabola with focus $B$ and directrix $CA$ intersect sides $BC$ and $BA$ at $B_{1}$ and $B_{2}$, respectively. Finally, let the parabola with focus $C$ and directrix $AB$ intersect sides $CA$ and $CB$ at $C_{1}$ and $C_{2}$, respectively. Find the perimeter of the triangle formed by lines $A_{1}A_{2}, B_{1}B_{2}, C_{1}C_{2}$.",
      "answer": "66-36\\sqrt{3}"
    },
    {
      "question": "Points $X$ and $Y$ are inside a unit square. The score of a vertex of the square is the minimum distance from that vertex to $X$ or $Y$. What is the minimum possible sum of the scores of the vertices of the square?",
      "answer": "\\frac{\\sqrt{6}+\\sqrt{2}}{2}"
    },
    {
      "question": "Over all real numbers $x$ and $y$ such that $$x^{3}=3 x+y \\quad \\text { and } \\quad y^{3}=3 y+x$$ compute the sum of all possible values of $x^{2}+y^{2}$.",
      "answer": "15"
    },
    {
      "question": "How many functions $f:\\{0,1\\}^{3} \\rightarrow\\{0,1\\}$ satisfy the property that, for all ordered triples \\left(a_{1}, a_{2}, a_{3}\\right) and \\left(b_{1}, b_{2}, b_{3}\\right) such that $a_{i} \\geq b_{i}$ for all $i, f\\left(a_{1}, a_{2}, a_{3}\\right) \\geq f\\left(b_{1}, b_{2}, b_{3}\\right)$?",
      "answer": "20"
    },
    {
      "question": "For how many integer values of $b$ does there exist a polynomial function with integer coefficients such that $f(2)=2010$ and $f(b)=8$?",
      "answer": "32"
    },
    {
      "question": "Violet has one-half of the money she needs to buy her mother a necklace. After her sister gives her $\\$30$, she has three-quarters of the amount she needs. How much will Violet's father give her?",
      "answer": "$30"
    },
    {
      "question": "If $x+\\sqrt{25}=\\sqrt{36}$, what is the value of $x$?",
      "answer": "1"
    },
    {
      "question": "If \\( N \\) is the smallest positive integer whose digits have a product of 2700, what is the sum of the digits of \\( N \\)?",
      "answer": "27"
    },
    {
      "question": "Consider the set \\(S\\) of all complex numbers \\(z\\) with nonnegative real and imaginary part such that \\(\\left|z^{2}+2\\right| \\leq|z|\\). Across all \\(z \\in S\\), compute the minimum possible value of \\(\\tan \\theta\\), where \\(\\theta\\) is the angle formed between \\(z\\) and the real axis.",
      "answer": "\\sqrt{7}"
    },
    {
      "question": "Originally, every square of $8 \\times 8$ chessboard contains a rook. One by one, rooks which attack an odd number of others are removed. Find the maximal number of rooks that can be removed. (A rook attacks another rook if they are on the same row or column and there are no other rooks between them.)",
      "answer": "59"
    },
    {
      "question": "In the list 7, 9, 10, 11, 18, which number is the average (mean) of the other four numbers?",
      "answer": "11"
    },
    {
      "question": "$A B C D$ is a cyclic quadrilateral in which $A B=4, B C=3, C D=2$, and $A D=5$. Diagonals $A C$ and $B D$ intersect at $X$. A circle $\\omega$ passes through $A$ and is tangent to $B D$ at $X . \\omega$ intersects $A B$ and $A D$ at $Y$ and $Z$ respectively. Compute $Y Z / B D$.",
      "answer": "\\frac{115}{143}"
    },
    {
      "question": "Determine the number of quadratic polynomials $P(x)=p_{1} x^{2}+p_{2} x-p_{3}$, where $p_{1}, p_{2}, p_{3}$ are not necessarily distinct (positive) prime numbers less than 50, whose roots are distinct rational numbers.",
      "answer": "31"
    },
    {
      "question": "$A B C D$ is a parallelogram satisfying $A B=7, B C=2$, and $\\angle D A B=120^{\\circ}$. Parallelogram $E C F A$ is contained in $A B C D$ and is similar to it. Find the ratio of the area of $E C F A$ to the area of $A B C D$.",
      "answer": "\\frac{39}{67}"
    },
    {
      "question": "Let $ABC$ be a triangle in the plane with $AB=13, BC=14, AC=15$. Let $M_{n}$ denote the smallest possible value of $\\left(AP^{n}+BP^{n}+CP^{n}\\right)^{\\frac{1}{n}}$ over all points $P$ in the plane. Find $\\lim _{n \\rightarrow \\infty} M_{n}$.",
      "answer": "\\frac{65}{8}"
    },
    {
      "question": "Let \\(p\\) be the answer to this question. If a point is chosen uniformly at random from the square bounded by \\(x=0, x=1, y=0\\), and \\(y=1\\), what is the probability that at least one of its coordinates is greater than \\(p\\)?",
      "answer": "\\frac{\\sqrt{5}-1}{2}"
    },
    {
      "question": "Find the smallest positive integer  $n$ such that if $n$ squares of a $1000 \\times 1000$ chessboard are colored, then there will exist three colored squares whose centers form a right triangle with sides parallel to the edges of the board.",
      "answer": "\\boxed{1999}"
    },
    {
      "question": "A certain cafeteria serves ham and cheese sandwiches, ham and tomato sandwiches, and tomato and cheese sandwiches. It is common for one meal to include multiple types of sandwiches. On a certain day, it was found that 80 customers had meals which contained both ham and cheese; 90 had meals containing both ham and tomatoes; 100 had meals containing both tomatoes and cheese. 20 customers' meals included all three ingredients. How many customers were there?",
      "answer": "230"
    },
    {
      "question": "Let $ABC$ be a right triangle with hypotenuse $AC$. Let $B^{\\prime}$ be the reflection of point $B$ across $AC$, and let $C^{\\prime}$ be the reflection of $C$ across $AB^{\\prime}$. Find the ratio of $[BCB^{\\prime}]$ to $[BC^{\\prime}B^{\\prime}]$.",
      "answer": "1"
    },
    {
      "question": "Farmer James wishes to cover a circle with circumference $10 \\pi$ with six different types of colored arcs. Each type of arc has radius 5, has length either $\\pi$ or $2 \\pi$, and is colored either red, green, or blue. He has an unlimited number of each of the six arc types. He wishes to completely cover his circle without overlap, subject to the following conditions: Any two adjacent arcs are of different colors. Any three adjacent arcs where the middle arc has length $\\pi$ are of three different colors. Find the number of distinct ways Farmer James can cover his circle. Here, two coverings are equivalent if and only if they are rotations of one another. In particular, two colorings are considered distinct if they are reflections of one another, but not rotations of one another.",
      "answer": "93"
    },
    {
      "question": "In $\\triangle A B C$, the incircle centered at $I$ touches sides $A B$ and $B C$ at $X$ and $Y$, respectively. Additionally, the area of quadrilateral $B X I Y$ is $\\frac{2}{5}$ of the area of $A B C$. Let $p$ be the smallest possible perimeter of a $\\triangle A B C$ that meets these conditions and has integer side lengths. Find the smallest possible area of such a triangle with perimeter $p$.",
      "answer": "2 \\sqrt{5}"
    },
    {
      "question": "Which number is closest to \\(-3.4\\) on a number line?",
      "answer": "-3"
    },
    {
      "question": "Three fair six-sided dice, each numbered 1 through 6 , are rolled. What is the probability that the three numbers that come up can form the sides of a triangle?",
      "answer": "37/72"
    },
    {
      "question": "If $x + 2y = 30$, what is the value of $\\frac{x}{5} + \\frac{2y}{3} + \\frac{2y}{5} + \\frac{x}{3}$?",
      "answer": "16"
    },
    {
      "question": "There are 30 people in a room, 60\\% of whom are men. If no men enter or leave the room, how many women must enter the room so that 40\\% of the total number of people in the room are men?",
      "answer": "15"
    },
    {
      "question": "Kimothy starts in the bottom-left square of a 4 by 4 chessboard. In one step, he can move up, down, left, or right to an adjacent square. Kimothy takes 16 steps and ends up where he started, visiting each square exactly once (except for his starting/ending square). How many paths could he have taken?",
      "answer": "12"
    },
    {
      "question": "Consider a $9 \\times 9$ grid of squares. Haruki fills each square in this grid with an integer between 1 and 9 , inclusive. The grid is called a super-sudoku if each of the following three conditions hold: - Each column in the grid contains each of the numbers $1,2,3,4,5,6,7,8,9$ exactly once. - Each row in the grid contains each of the numbers $1,2,3,4,5,6,7,8,9$ exactly once. - Each $3 \\times 3$ subsquare in the grid contains each of the numbers $1,2,3,4,5,6,7,8,9$ exactly once. How many possible super-sudoku grids are there?",
      "answer": "0"
    },
    {
      "question": "Number $a$ is such that $\\forall a_1, a_2, a_3, a_4 \\in \\mathbb{R}$, there are integers $k_1, k_2, k_3, k_4$ such that $\\sum_{1 \\leq i < j \\leq 4} ((a_i - k_i) - (a_j - k_j))^2 \\leq a$. Find the minimum of $a$.",
      "answer": "1.25"
    },
    {
      "question": "For how many pairs of sequences of nonnegative integers $\\left(b_{1}, b_{2}, \\ldots, b_{2018}\\right)$ and $\\left(c_{1}, c_{2}, \\ldots, c_{2018}\\right)$ does there exist a sequence of nonnegative integers $\\left(a_{0}, \\ldots, a_{2018}\\right)$ with the following properties: For $0 \\leq i \\leq 2018, a_{i}<2^{2018}$; For $1 \\leq i \\leq 2018, b_{i}=a_{i-1}+a_{i}$ and $c_{i}=a_{i-1} \\mid a_{i}$ where $\\mid$ denotes the bitwise or operation?",
      "answer": "\\left(2^{2019}-1\\right)^{2018}"
    },
    {
      "question": "The number $$316990099009901=\\frac{32016000000000001}{101}$$ is the product of two distinct prime numbers. Compute the smaller of these two primes.",
      "answer": "4002001"
    },
    {
      "question": "Mattis is hosting a badminton tournament for $40$ players on $20$ courts numbered from $1$ to $20$. The players are distributed with $2$ players on each court. In each round a winner is determined on each court. Afterwards, the player who lost on court $1$, and the player who won on court $20$ stay in place. For the remaining $38$ players, the winner on court $i$ moves to court $i + 1$ and the loser moves to court $i - 1$. The tournament continues until every player has played every other player at least once. What is the minimal number of rounds the tournament can last?",
      "answer": "39"
    },
    {
      "question": "How many words are there in a language that are 10 letters long and begin with a vowel, given that the language uses only the letters A, B, C, D, and E, where A and E are vowels, and B, C, and D are consonants, and a word does not include the same letter twice in a row or two vowels in a row?",
      "answer": "199776"
    },
    {
      "question": "Find the largest integer $n$ such that the following holds: there exists a set of $n$ points in the plane such that, for any choice of three of them, some two are unit distance apart.",
      "answer": "7"
    },
    {
      "question": "What is the smallest integer $n$ , greater than one, for which the root-mean-square of the first $n$ positive integers is an integer?\n$\\mathbf{Note.}$ The root-mean-square of $n$ numbers $a_1, a_2, \\cdots, a_n$ is defined to be \\[\\left[\\frac{a_1^2 + a_2^2 + \\cdots + a_n^2}n\\right]^{1/2}\\]",
      "answer": "\\(\\boxed{337}\\)"
    },
    {
      "question": "What number is placed in the shaded circle if each of the numbers $1,5,6,7,13,14,17,22,26$ is placed in a different circle, the numbers 13 and 17 are placed as shown, and Jen calculates the average of the numbers in the first three circles, the average of the numbers in the middle three circles, and the average of the numbers in the last three circles, and these three averages are equal?",
      "answer": "7"
    },
    {
      "question": "Six consecutive integers are written on a blackboard. When one of them is erased, the sum of the remaining five integers is 2012. What is the sum of the digits of the integer that was erased?",
      "answer": "7"
    },
    {
      "question": "All subscripts in this problem are to be considered modulo 6 , that means for example that $\\omega_{7}$ is the same as $\\omega_{1}$. Let $\\omega_{1}, \\ldots \\omega_{6}$ be circles of radius $r$, whose centers lie on a regular hexagon of side length 1 . Let $P_{i}$ be the intersection of $\\omega_{i}$ and $\\omega_{i+1}$ that lies further from the center of the hexagon, for $i=1, \\ldots 6$. Let $Q_{i}, i=1 \\ldots 6$, lie on $\\omega_{i}$ such that $Q_{i}, P_{i}, Q_{i+1}$ are colinear. Find the number of possible values of $r$.",
      "answer": "5"
    },
    {
      "question": "Consider the sequence: $x_1=19,x_2=95,x_{n+2}=\\text{lcm} (x_{n+1},x_n)+x_n$, for $n>1$, where $\\text{lcm} (a,b)$ means the least common multiple of $a$ and $b$. Find the greatest common divisor of $x_{1995}$ and $x_{1996}$.",
      "answer": "19"
    },
    {
      "question": "Admiral Ackbar needs to send a 5-character message through hyperspace to the Rebels. Each character is a lowercase letter, and the same letter may appear more than once in a message. When the message is beamed through hyperspace, the characters come out in a random order. Ackbar chooses his message so that the Rebels have at least a $\\frac{1}{2}$ chance of getting the same message he sent. How many distinct messages could he send?",
      "answer": "26"
    },
    {
      "question": "Compute the number of ordered quintuples of nonnegative integers $(a_{1}, a_{2}, a_{3}, a_{4}, a_{5})$ such that $0 \\leq a_{1}, a_{2}, a_{3}, a_{4}, a_{5} \\leq 7$ and 5 divides $2^{a_{1}}+2^{a_{2}}+2^{a_{3}}+2^{a_{4}}+2^{a_{5}}$.",
      "answer": "6528"
    },
    {
      "question": "Let $A B C$ be an acute isosceles triangle with orthocenter $H$. Let $M$ and $N$ be the midpoints of sides $\\overline{A B}$ and $\\overline{A C}$, respectively. The circumcircle of triangle $M H N$ intersects line $B C$ at two points $X$ and $Y$. Given $X Y=A B=A C=2$, compute $B C^{2}$.",
      "answer": "2(\\sqrt{17}-1)"
    },
    {
      "question": "In the game of projective set, each card contains some nonempty subset of six distinguishable dots. A projective set deck consists of one card for each of the 63 possible nonempty subsets of dots. How many collections of five cards have an even number of each dot? The order in which the cards appear does not matter.",
      "answer": "109368"
    },
    {
      "question": "Points $E, F, G, H$ are chosen on segments $A B, B C, C D, D A$, respectively, of square $A B C D$. Given that segment $E G$ has length 7 , segment $F H$ has length 8 , and that $E G$ and $F H$ intersect inside $A B C D$ at an acute angle of $30^{\\circ}$, then compute the area of square $A B C D$.",
      "answer": "\\frac{784}{19}"
    },
    {
      "question": "Given a positive integer $k$, let \\|k\\| denote the absolute difference between $k$ and the nearest perfect square. For example, \\|13\\|=3 since the nearest perfect square to 13 is 16. Compute the smallest positive integer $n$ such that $\\frac{\\|1\\|+\\|2\\|+\\cdots+\\|n\\|}{n}=100$.",
      "answer": "89800"
    },
    {
      "question": "Suppose $x^{3}-a x^{2}+b x-48$ is a polynomial with three positive roots $p, q$, and $r$ such that $p<q<r$. What is the minimum possible value of $1 / p+2 / q+3 / r$ ?",
      "answer": "3 / 2"
    },
    {
      "question": "Compute \\( \\frac{2^{3}-1}{2^{3}+1} \\cdot \\frac{3^{3}-1}{3^{3}+1} \\cdot \\frac{4^{3}-1}{4^{3}+1} \\cdot \\frac{5^{3}-1}{5^{3}+1} \\cdot \\frac{6^{3}-1}{6^{3}+1} \\).",
      "answer": "43/63"
    },
    {
      "question": "When 542 is multiplied by 3, what is the ones (units) digit of the result?",
      "answer": "6"
    },
    {
      "question": "Two 18-24-30 triangles in the plane share the same circumcircle as well as the same incircle. What's the area of the region common to both the triangles?",
      "answer": "132"
    },
    {
      "question": "Allen and Yang want to share the numbers \\(1,2,3,4,5,6,7,8,9,10\\). How many ways are there to split all ten numbers among Allen and Yang so that each person gets at least one number, and either Allen's numbers or Yang's numbers sum to an even number?",
      "answer": "1022"
    },
    {
      "question": "Kevin starts with the vectors \\((1,0)\\) and \\((0,1)\\) and at each time step, he replaces one of the vectors with their sum. Find the cotangent of the minimum possible angle between the vectors after 8 time steps.",
      "answer": "987"
    },
    {
      "question": "A tourist is learning an incorrect way to sort a permutation $(p_{1}, \\ldots, p_{n})$ of the integers $(1, \\ldots, n)$. We define a fix on two adjacent elements $p_{i}$ and $p_{i+1}$, to be an operation which swaps the two elements if $p_{i}>p_{i+1}$, and does nothing otherwise. The tourist performs $n-1$ rounds of fixes, numbered $a=1,2, \\ldots, n-1$. In round $a$ of fixes, the tourist fixes $p_{a}$ and $p_{a+1}$, then $p_{a+1}$ and $p_{a+2}$, and so on, up to $p_{n-1}$ and $p_{n}$. In this process, there are $(n-1)+(n-2)+\\cdots+1=\\frac{n(n-1)}{2}$ total fixes performed. How many permutations of $(1, \\ldots, 2018)$ can the tourist start with to obtain $(1, \\ldots, 2018)$ after performing these steps?",
      "answer": "1009! \\cdot 1010!"
    },
    {
      "question": "There are 101 people participating in a Secret Santa gift exchange. As usual each person is randomly assigned another person for whom (s)he has to get a gift, such that each person gives and receives exactly one gift and no one gives a gift to themself. What is the probability that the first person neither gives gifts to or receives gifts from the second or third person? Express your answer as a decimal rounded to five decimal places.",
      "answer": "0.96039"
    },
    {
      "question": "What is the average (mean) number of hamburgers eaten per student if 12 students ate 0 hamburgers, 14 students ate 1 hamburger, 8 students ate 2 hamburgers, 4 students ate 3 hamburgers, and 2 students ate 4 hamburgers?",
      "answer": "1.25"
    },
    {
      "question": "Find the sum of the reciprocals of all the (positive) divisors of 144.",
      "answer": "403/144"
    },
    {
      "question": "Today, Ivan the Confessor prefers continuous functions $f:[0,1]\\to\\mathbb{R}$ satisfying $f(x)+f(y)\\geq |x-y|$ for all pairs $x,y\\in [0,1]$. Find the minimum of $\\int_0^1 f$ over all preferred functions.\n\n(",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Suppose $A B C$ is a triangle with circumcenter $O$ and orthocenter $H$ such that $A, B, C, O$, and $H$ are all on distinct points with integer coordinates. What is the second smallest possible value of the circumradius of $A B C$ ?",
      "answer": "\\sqrt{10}"
    },
    {
      "question": "You are standing at a pole and a snail is moving directly away from the pole at $1 \\mathrm{~cm} / \\mathrm{s}$. When the snail is 1 meter away, you start 'Round 1'. In Round $n(n \\geq 1)$, you move directly toward the snail at $n+1 \\mathrm{~cm} / \\mathrm{s}$. When you reach the snail, you immediately turn around and move back to the starting pole at $n+1 \\mathrm{~cm} / \\mathrm{s}$. When you reach the pole, you immediately turn around and Round $n+1$ begins. At the start of Round 100, how many meters away is the snail?",
      "answer": "5050"
    },
    {
      "question": "Let $n$ be the smallest positive integer with exactly 2015 positive factors. What is the sum of the (not necessarily distinct) prime factors of $n$?",
      "answer": "116"
    },
    {
      "question": "Quadrilateral $ABCD$ has $\\angle BCD=\\angle DAB=90^{\\circ}$. The perimeter of $ABCD$ is 224 and its area is 2205. One side of $ABCD$ has length 7. The remaining three sides have integer lengths. What is the integer formed by the rightmost two digits of the sum of the squares of the side lengths of $ABCD$?",
      "answer": "60"
    },
    {
      "question": "For an integer $n$, let $f(n)$ denote the number of pairs $(x, y)$ of integers such that $x^{2}+x y+y^{2}=n$. Compute the sum $\\sum_{n=1}^{10^{6}} n f(n)$",
      "answer": "1.813759629294 \\cdot 10^{12}"
    },
    {
      "question": "One integer is selected at random from the following list of 15 integers: $1,2,2,3,3,3,4,4,4,4,5,5,5,5,5$. The probability that the selected integer is equal to $n$ is $\\frac{1}{3}$. What is the value of $n$?",
      "answer": "5"
    },
    {
      "question": "Let $\\omega_{1}$ and $\\omega_{2}$ be two non-intersecting circles. Suppose the following three conditions hold: - The length of a common internal tangent of $\\omega_{1}$ and $\\omega_{2}$ is equal to 19 . - The length of a common external tangent of $\\omega_{1}$ and $\\omega_{2}$ is equal to 37 . - If two points $X$ and $Y$ are selected on $\\omega_{1}$ and $\\omega_{2}$, respectively, uniformly at random, then the expected value of $X Y^{2}$ is 2023 . Compute the distance between the centers of $\\omega_{1}$ and $\\omega_{2}$.",
      "answer": "38"
    },
    {
      "question": "Find the maximum possible value of $H \\cdot M \\cdot M \\cdot T$ over all ordered triples $(H, M, T)$ of integers such that $H \\cdot M \\cdot M \\cdot T=H+M+M+T$.",
      "answer": "8"
    },
    {
      "question": "Denote $\\phi=\\frac{1+\\sqrt{5}}{2}$ and consider the set of all finite binary strings without leading zeroes. Each string $S$ has a \"base-$\\phi$ \" value $p(S)$. For example, $p(1101)=\\phi^{3}+\\phi^{2}+1$. For any positive integer $n$, let $f(n)$ be the number of such strings $S$ that satisfy $p(S)=\\frac{\\phi^{48 n}-1}{\\phi^{48}-1}$. The sequence of fractions $\\frac{f(n+1)}{f(n)}$ approaches a real number $c$ as $n$ goes to infinity. Determine the value of $c$.",
      "answer": "\\frac{25+3 \\sqrt{69}}{2}"
    },
    {
      "question": "A rectangular pool table has vertices at $(0,0)(12,0)(0,10)$, and $(12,10)$. There are pockets only in the four corners. A ball is hit from $(0,0)$ along the line $y=x$ and bounces off several walls before eventually entering a pocket. Find the number of walls that the ball bounces off of before entering a pocket.",
      "answer": "9"
    },
    {
      "question": "The line with equation $y = x$ is translated 3 units to the right and 2 units down. What is the $y$-intercept of the resulting line?",
      "answer": "-5"
    },
    {
      "question": "A regular hexagon has one side along the diameter of a semicircle, and the two opposite vertices on the semicircle. Find the area of the hexagon if the diameter of the semicircle is 1.",
      "answer": "3 \\sqrt{3} / 26"
    },
    {
      "question": "Tetrahedron $A B C D$ with volume 1 is inscribed in circumsphere $\\omega$ such that $A B=A C=A D=2$ and $B C \\cdot C D \\cdot D B=16$. Find the radius of $\\omega$.",
      "answer": "\\frac{5}{3}"
    },
    {
      "question": "Find the number of positive integers $j \\leq 3^{2013}$ such that $$j=\\sum_{k=0}^{m}\\left((-1)^{k} \\cdot 3^{a_{k}}\\right)$$ for some strictly increasing sequence of nonnegative integers $\\left\\{a_{k}\\right\\}$. For example, we may write $3=3^{1}$ and $55=3^{0}-3^{3}+3^{4}$, but 4 cannot be written in this form.",
      "answer": "2^{2013}"
    },
    {
      "question": "Nasim buys trading cards in packages of 5 cards and in packages of 8 cards. He can purchase exactly 18 cards by buying two 5-packs and one 8-pack, but he cannot purchase exactly 12 cards with any combination of packages. For how many of the integers $n=24,25,26,27,28,29$ can he buy exactly $n$ cards?",
      "answer": "5"
    },
    {
      "question": "The Red Sox play the Yankees in a best-of-seven series that ends as soon as one team wins four games. Suppose that the probability that the Red Sox win Game $n$ is $\\frac{n-1}{6}$. What is the probability that the Red Sox will win the series?",
      "answer": "1/2"
    },
    {
      "question": "In a 3 by 3 grid of unit squares, an up-right path is a path from the bottom left corner to the top right corner that travels only up and right in steps of 1 unit. For such a path $p$, let $A_{p}$ denote the number of unit squares under the path $p$. Compute the sum of $A_{p}$ over all up-right paths $p$.",
      "answer": "90"
    },
    {
      "question": "Each square in the following hexomino has side length 1. Find the minimum area of any rectangle that contains the entire hexomino.",
      "answer": "\\frac{21}{2}"
    },
    {
      "question": "If you roll four fair 6-sided dice, what is the probability that at least three of them will show the same value?",
      "answer": "\\frac{7}{72}"
    },
    {
      "question": "Let \\(A B C\\) be a triangle with \\(\\angle A=18^{\\circ}, \\angle B=36^{\\circ}\\). Let \\(M\\) be the midpoint of \\(A B, D\\) a point on ray \\(C M\\) such that \\(A B=A D ; E\\) a point on ray \\(B C\\) such that \\(A B=B E\\), and \\(F\\) a point on ray \\(A C\\) such that \\(A B=A F\\). Find \\(\\angle F D E\\).",
      "answer": "27"
    },
    {
      "question": "Let $\\pi$ be a randomly chosen permutation of the numbers from 1 through 2012. Find the probability that $\\pi(\\pi(2012))=2012$.",
      "answer": "\\frac{1}{1006}"
    },
    {
      "question": "Suppose $A B C D$ is a convex quadrilateral with $\\angle A B D=105^{\\circ}, \\angle A D B=15^{\\circ}, A C=7$, and $B C=C D=5$. Compute the sum of all possible values of $B D$.",
      "answer": "\\sqrt{291}"
    },
    {
      "question": "Let $n$ be square with 4 digits, such that all its digits are less than 6. If we add 1 to each digit the resulting number is another square. Find $n$",
      "answer": "2025"
    },
    {
      "question": "Pascal has a triangle. In the $n$th row, there are $n+1$ numbers $a_{n, 0}, a_{n, 1}, a_{n, 2}, \\ldots, a_{n, n}$ where $a_{n, 0}=a_{n, n}=1$. For all $1 \\leq k \\leq n-1, a_{n, k}=a_{n-1, k}-a_{n-1, k-1}$. Let $N$ be the value of the sum $$\\sum_{k=0}^{2018} \\frac{\\left|a_{2018, k}\\right|}{\\binom{2018}{k}}$$ Estimate $N$.",
      "answer": "780.9280674537"
    },
    {
      "question": "John lists the integers from 1 to 20 in increasing order. He then erases the first half of the integers in the list and rewrites them in order at the end of the second half of the list. Which integer in the new list has exactly 12 integers to its left?",
      "answer": "3"
    },
    {
      "question": "Evaluate\n\\[\\left \\lfloor  \\ \\prod_{n=1}^{1992} \\frac{3n+2}{3n+1} \\  \\right \\rfloor\\]",
      "answer": "12"
    },
    {
      "question": "An isosceles right triangle $A B C$ has area 1. Points $D, E, F$ are chosen on $B C, C A, A B$ respectively such that $D E F$ is also an isosceles right triangle. Find the smallest possible area of $D E F$.",
      "answer": "\\frac{1}{5}"
    },
    {
      "question": "Unit squares $A B C D$ and $E F G H$ have centers $O_{1}$ and $O_{2}$ respectively, and are originally situated such that $B$ and $E$ are at the same position and $C$ and $H$ are at the same position. The squares then rotate clockwise about their centers at the rate of one revolution per hour. After 5 minutes, what is the area of the intersection of the two squares?",
      "answer": "\\frac{2-\\sqrt{3}}{4}"
    },
    {
      "question": "Sarah stands at $(0,0)$ and Rachel stands at $(6,8)$ in the Euclidean plane. Sarah can only move 1 unit in the positive $x$ or $y$ direction, and Rachel can only move 1 unit in the negative $x$ or $y$ direction. Each second, Sarah and Rachel see each other, independently pick a direction to move at the same time, and move to their new position. Sarah catches Rachel if Sarah and Rachel are ever at the same point. Rachel wins if she is able to get to $(0,0)$ without being caught; otherwise, Sarah wins. Given that both of them play optimally to maximize their probability of winning, what is the probability that Rachel wins?",
      "answer": "\\[\n\\frac{63}{64}\n\\]"
    },
    {
      "question": "A tournament among 2021 ranked teams is played over 2020 rounds. In each round, two teams are selected uniformly at random among all remaining teams to play against each other. The better ranked team always wins, and the worse ranked team is eliminated. Let $p$ be the probability that the second best ranked team is eliminated in the last round. Compute $\\lfloor 2021 p \\rfloor$.",
      "answer": "674"
    },
    {
      "question": "A sequence of real numbers $a_{0}, a_{1}, \\ldots$ is said to be good if the following three conditions hold. (i) The value of $a_{0}$ is a positive integer. (ii) For each non-negative integer $i$ we have $a_{i+1}=2 a_{i}+1$ or $a_{i+1}=\\frac{a_{i}}{a_{i}+2}$. (iii) There exists a positive integer $k$ such that $a_{k}=2014$. Find the smallest positive integer $n$ such that there exists a good sequence $a_{0}, a_{1}, \\ldots$ of real numbers with the property that $a_{n}=2014$.",
      "answer": "60"
    },
    {
      "question": "If a positive integer multiple of 864 is picked randomly, with each multiple having the same probability of being picked, what is the probability that it is divisible by 1944?",
      "answer": "\\frac{1}{9}"
    },
    {
      "question": "Let $A B C D$ be a convex quadrilateral so that all of its sides and diagonals have integer lengths. Given that $\\angle A B C=\\angle A D C=90^{\\circ}, A B=B D$, and $C D=41$, find the length of $B C$.",
      "answer": "580"
    },
    {
      "question": "Two circles with equal radii intersect as shown. The area of the shaded region equals the sum of the areas of the two unshaded regions. If the area of the shaded region is $216\\pi$, what is the circumference of each circle?",
      "answer": "36\\pi"
    },
    {
      "question": "The operation \\( \\otimes \\) is defined by \\( a \\otimes b = \\frac{a}{b} + \\frac{b}{a} \\). What is the value of \\( 4 \\otimes 8 \\)?",
      "answer": "\\frac{5}{2}"
    },
    {
      "question": "If $x=11$, $y=-8$, and $2x-3z=5y$, what is the value of $z$?",
      "answer": "\\frac{62}{3}"
    },
    {
      "question": "Let $P_1P_2\\ldots P_{24}$ be a regular $24$-sided polygon inscribed in a circle $\\omega$ with circumference $24$. Determine the number of ways to choose sets of eight distinct vertices from these $24$ such that none of the arcs has length $3$ or $8$.",
      "answer": "258"
    },
    {
      "question": "Joshua chooses five distinct numbers. In how many different ways can he assign these numbers to the variables $p, q, r, s$, and $t$ so that $p<s, q<s, r<t$, and $s<t$?",
      "answer": "8"
    },
    {
      "question": "Abbot writes the letter $A$ on the board. Every minute, he replaces every occurrence of $A$ with $A B$ and every occurrence of $B$ with $B A$, hence creating a string that is twice as long. After 10 minutes, there are $2^{10}=1024$ letters on the board. How many adjacent pairs are the same letter?",
      "answer": "341"
    },
    {
      "question": "Let $\\omega$ be a fixed circle with radius 1, and let $B C$ be a fixed chord of $\\omega$ such that $B C=1$. The locus of the incenter of $A B C$ as $A$ varies along the circumference of $\\omega$ bounds a region $\\mathcal{R}$ in the plane. Find the area of $\\mathcal{R}$.",
      "answer": "\\pi\\left(\\frac{3-\\sqrt{3}}{3}\\right)-1"
    },
    {
      "question": "A robotic grasshopper jumps 1 cm to the east, then 2 cm to the north, then 3 cm to the west, then 4 cm to the south. After every fourth jump, the grasshopper restarts the sequence of jumps: 1 cm to the east, then 2 cm to the north, then 3 cm to the west, then 4 cm to the south. After a total of $n$ jumps, the position of the grasshopper is 162 cm to the west and 158 cm to the south of its original position. What is the sum of the squares of the digits of $n$?",
      "answer": "22"
    },
    {
      "question": "Among citizens of Cambridge there exist 8 different types of blood antigens. In a crowded lecture hall are 256 students, each of whom has a blood type corresponding to a distinct subset of the antigens; the remaining of the antigens are foreign to them. Quito the Mosquito flies around the lecture hall, picks a subset of the students uniformly at random, and bites the chosen students in a random order. After biting a student, Quito stores a bit of any antigens that student had. A student bitten while Quito had $k$ blood antigen foreign to him/her will suffer for $k$ hours. What is the expected total suffering of all 256 students, in hours?",
      "answer": "\\frac{2^{135}-2^{128}+1}{2^{119} \\cdot 129}"
    },
    {
      "question": "The integers -5 and 6 are shown on a number line. What is the distance between them?",
      "answer": "11"
    },
    {
      "question": "Let $f(x)=x^{3}+x+1$. Suppose $g$ is a cubic polynomial such that $g(0)=-1$, and the roots of $g$ are the squares of the roots of $f$. Find $g(9)$.",
      "answer": "899"
    },
    {
      "question": "Let $n$ be the answer to this problem. An urn contains white and black balls. There are $n$ white balls and at least two balls of each color in the urn. Two balls are randomly drawn from the urn without replacement. Find the probability, in percent, that the first ball drawn is white and the second is black.",
      "answer": "19"
    },
    {
      "question": "Suppose that $d$ is an odd integer and $e$ is an even integer. How many of the following expressions are equal to an odd integer? $d+d, (e+e) \times d, d \times d, d \times(e+d)$",
      "answer": "2"
    },
    {
      "question": "How many of the positive divisors of 128 are perfect squares larger than 1?",
      "answer": "3"
    },
    {
      "question": "Consider a $5 \\times 5$ grid of squares. Vladimir colors some of these squares red, such that the centers of any four red squares do not form an axis-parallel rectangle (i.e. a rectangle whose sides are parallel to those of the squares). What is the maximum number of squares he could have colored red?",
      "answer": "12"
    },
    {
      "question": "Owen spends $\\$ 1.20$ per litre on gasoline. He uses an average of 1 L of gasoline to drive 12.5 km. How much will Owen spend on gasoline to drive 50 km?",
      "answer": "\\$ 4.80"
    },
    {
      "question": "How many orderings $(a_{1}, \\ldots, a_{8})$ of $(1,2, \\ldots, 8)$ exist such that $a_{1}-a_{2}+a_{3}-a_{4}+a_{5}-a_{6}+a_{7}-a_{8}=0$ ?",
      "answer": "4608"
    },
    {
      "question": "During the weekends, Eli delivers milk in the complex plane. On Saturday, he begins at $z$ and delivers milk to houses located at $z^{3}, z^{5}, z^{7}, \\ldots, z^{2013}$, in that order; on Sunday, he begins at 1 and delivers milk to houses located at $z^{2}, z^{4}, z^{6}, \\ldots, z^{2012}$, in that order. Eli always walks directly (in a straight line) between two houses. If the distance he must travel from his starting point to the last house is $\\sqrt{2012}$ on both days, find the real part of $z^{2}$.",
      "answer": "\\frac{1005}{1006}"
    },
    {
      "question": "Find the number of digits in the decimal representation of $2^{41}$.",
      "answer": "13"
    },
    {
      "question": "Let $a, b, c, x, y$, and $z$ be complex numbers such that $a=\\frac{b+c}{x-2}, \\quad b=\\frac{c+a}{y-2}, \\quad c=\\frac{a+b}{z-2}$. If $x y+y z+z x=67$ and $x+y+z=2010$, find the value of $x y z$.",
      "answer": "-5892"
    },
    {
      "question": "Compute $$100^{2}+99^{2}-98^{2}-97^{2}+96^{2}+95^{2}-94^{2}-93^{2}+\\ldots+4^{2}+3^{2}-2^{2}-1^{2}$$",
      "answer": "10100"
    },
    {
      "question": "In $\\triangle A B C, \\omega$ is the circumcircle, $I$ is the incenter and $I_{A}$ is the $A$-excenter. Let $M$ be the midpoint of arc $\\widehat{B A C}$ on $\\omega$, and suppose that $X, Y$ are the projections of $I$ onto $M I_{A}$ and $I_{A}$ onto $M I$, respectively. If $\\triangle X Y I_{A}$ is an equilateral triangle with side length 1, compute the area of $\\triangle A B C$.",
      "answer": "\\frac{\\sqrt{6}}{7}"
    },
    {
      "question": "Evaluate \\[ \\sum_{k=1}^\\infty \\frac{(-1)^{k-1}}{k} \\sum_{n=0}^\\infty \\frac{1}{k2^n + 1}. \\]",
      "answer": "1"
    },
    {
      "question": "How many subsets $S$ of the set $\\{1,2, \\ldots, 10\\}$ satisfy the property that, for all $i \\in[1,9]$, either $i$ or $i+1$ (or both) is in $S$?",
      "answer": "144"
    },
    {
      "question": "Let $x<y$ be positive real numbers such that $\\sqrt{x}+\\sqrt{y}=4$ and $\\sqrt{x+2}+\\sqrt{y+2}=5$. Compute $x$.",
      "answer": "\\frac{49}{36}"
    },
    {
      "question": "Let $x, y, z$ be real numbers satisfying $$\\frac{1}{x}+y+z=x+\\frac{1}{y}+z=x+y+\\frac{1}{z}=3$$ The sum of all possible values of $x+y+z$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "6106"
    },
    {
      "question": "Let $A, B, C$ be points in that order along a line, such that $A B=20$ and $B C=18$. Let $\\omega$ be a circle of nonzero radius centered at $B$, and let $\\ell_{1}$ and $\\ell_{2}$ be tangents to $\\omega$ through $A$ and $C$, respectively. Let $K$ be the intersection of $\\ell_{1}$ and $\\ell_{2}$. Let $X$ lie on segment $\\overline{K A}$ and $Y$ lie on segment $\\overline{K C}$ such that $X Y \\| B C$ and $X Y$ is tangent to $\\omega$. What is the largest possible integer length for $X Y$?",
      "answer": "35"
    },
    {
      "question": "Let $a_{1}, a_{2}, \\ldots, a_{n}$ be a sequence of distinct positive integers such that $a_{1}+a_{2}+\\cdots+a_{n}=2021$ and $a_{1} a_{2} \\cdots a_{n}$ is maximized. If $M=a_{1} a_{2} \\cdots a_{n}$, compute the largest positive integer $k$ such that $2^{k} \\mid M$.",
      "answer": "62"
    },
    {
      "question": "For dinner, Priya is eating grilled pineapple spears. Each spear is in the shape of the quadrilateral PINE, with $P I=6 \\mathrm{~cm}, I N=15 \\mathrm{~cm}, N E=6 \\mathrm{~cm}, E P=25 \\mathrm{~cm}$, and \\angle N E P+\\angle E P I=60^{\\circ}$. What is the area of each spear, in \\mathrm{cm}^{2}$ ?",
      "answer": "\\frac{100 \\sqrt{3}}{3}"
    },
    {
      "question": "Compute the radius of the inscribed circle of a triangle with sides 15,16 , and 17 .",
      "answer": "\\sqrt{21}"
    },
    {
      "question": "For $x$ a real number, let $f(x)=0$ if $x<1$ and $f(x)=2 x-2$ if $x \\geq 1$. How many solutions are there to the equation $f(f(f(f(x))))=x ?$",
      "answer": "2"
    },
    {
      "question": "Consider an isosceles triangle $T$ with base 10 and height 12. Define a sequence $\\omega_{1}, \\omega_{2}, \\ldots$ of circles such that $\\omega_{1}$ is the incircle of $T$ and $\\omega_{i+1}$ is tangent to $\\omega_{i}$ and both legs of the isosceles triangle for $i>1$. Find the ratio of the radius of $\\omega_{i+1}$ to the radius of $\\omega_{i}$.",
      "answer": "\\frac{4}{9}"
    },
    {
      "question": "Let real  $a$, $b$, and $c$ satisfy $$abc+a+b+c=ab+bc+ca+5.$$ Find the least possible value of $a^2+b^2+c^2$.",
      "answer": "6"
    },
    {
      "question": "A frog is at the point $(0,0)$. Every second, he can jump one unit either up or right. He can only move to points $(x, y)$ where $x$ and $y$ are not both odd. How many ways can he get to the point $(8,14)$?",
      "answer": "330"
    },
    {
      "question": "How many positive integers $n \\leq 20000$ have the properties that $2n$ has 64 positive divisors including 1 and $2n$, and $5n$ has 60 positive divisors including 1 and $5n$?",
      "answer": "4"
    },
    {
      "question": "Given positive integers $k, m, n$ such that $1 \\leq  k \\leq  m \\leq  n$. Evaluate\n\n\\[\\sum^{n}_{i=0} \\frac{(-1)^i}{n+k+i} \\cdot \\frac{(m+n+i)!}{i!(n-i)!(m+i)!}.\\]",
      "answer": "0"
    },
    {
      "question": "Suppose that there are initially eight townspeople and one goon. One of the eight townspeople is named Jester. If Jester is sent to jail during some morning, then the game ends immediately in his sole victory. (However, the Jester does not win if he is sent to jail during some night.) Find the probability that only the Jester wins.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Find the smallest possible value of $x+y$ where $x, y \\geq 1$ and $x$ and $y$ are integers that satisfy $x^{2}-29y^{2}=1$",
      "answer": "11621"
    },
    {
      "question": "Let $x$ be a complex number such that $x+x^{-1}$ is a root of the polynomial $p(t)=t^{3}+t^{2}-2 t-1$. Find all possible values of $x^{7}+x^{-7}$.",
      "answer": "2"
    },
    {
      "question": "Points $A, C$, and $B$ lie on a line in that order such that $A C=4$ and $B C=2$. Circles $\\omega_{1}, \\omega_{2}$, and $\\omega_{3}$ have $\\overline{B C}, \\overline{A C}$, and $\\overline{A B}$ as diameters. Circle $\\Gamma$ is externally tangent to $\\omega_{1}$ and $\\omega_{2}$ at $D$ and $E$ respectively, and is internally tangent to $\\omega_{3}$. Compute the circumradius of triangle $C D E$.",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "A teacher must divide 221 apples evenly among 403 students. What is the minimal number of pieces into which she must cut the apples? (A whole uncut apple counts as one piece.)",
      "answer": "611"
    },
    {
      "question": "Let $\\triangle X Y Z$ be a right triangle with $\\angle X Y Z=90^{\\circ}$. Suppose there exists an infinite sequence of equilateral triangles $X_{0} Y_{0} T_{0}, X_{1} Y_{1} T_{1}, \\ldots$ such that $X_{0}=X, Y_{0}=Y, X_{i}$ lies on the segment $X Z$ for all $i \\geq 0, Y_{i}$ lies on the segment $Y Z$ for all $i \\geq 0, X_{i} Y_{i}$ is perpendicular to $Y Z$ for all $i \\geq 0, T_{i}$ and $Y$ are separated by line $X Z$ for all $i \\geq 0$, and $X_{i}$ lies on segment $Y_{i-1} T_{i-1}$ for $i \\geq 1$. Let $\\mathcal{P}$ denote the union of the equilateral triangles. If the area of $\\mathcal{P}$ is equal to the area of $X Y Z$, find $\\frac{X Y}{Y Z}$.",
      "answer": "1"
    },
    {
      "question": "A sphere is centered at a point with integer coordinates and passes through the three points $(2,0,0)$, $(0,4,0),(0,0,6)$, but not the origin $(0,0,0)$. If $r$ is the smallest possible radius of the sphere, compute $r^{2}$.",
      "answer": "51"
    },
    {
      "question": "Turbo the snail plays a game on a board with $2024$ rows and $2023$ columns. There are hidden monsters in $2022$ of the cells. Initially, Turbo does not know where any of the monsters are, but he knows that there is exactly one monster in each row except the first row and the last row, and that each column contains at most one monster.\n\nTurbo makes a series of attempts to go from the first row to the last row. On each attempt, he chooses to start on any cell in the first row, then repeatedly moves to an adjacent cell sharing a common side. (He is allowed to return to a previously visited cell.) If he reaches a cell with a monster, his attempt ends and he is transported back to the first row to start a new attempt. The monsters do not move, and Turbo remembers whether or not each cell he has visited contains a monster. If he reaches any cell in the last row, his attempt ends and the game is over.\n\nDetermine the minimum value of $n$ for which Turbo has a strategy that guarantees reaching the last row on the $n$-th attempt or earlier, regardless of the locations of the monsters.\n\n[i]",
      "answer": "3"
    },
    {
      "question": "Starting with an empty string, we create a string by repeatedly appending one of the letters $H, M, T$ with probabilities $\\frac{1}{4}, \\frac{1}{2}, \\frac{1}{4}$, respectively, until the letter $M$ appears twice consecutively. What is the expected value of the length of the resulting string?",
      "answer": "6"
    },
    {
      "question": "For any integer $n$, define $\\lfloor n\\rfloor$ as the greatest integer less than or equal to $n$. For any positive integer $n$, let $$f(n)=\\lfloor n\\rfloor+\\left\\lfloor\\frac{n}{2}\\right\\rfloor+\\left\\lfloor\\frac{n}{3}\\right\\rfloor+\\cdots+\\left\\lfloor\\frac{n}{n}\\right\\rfloor.$$ For how many values of $n, 1 \\leq n \\leq 100$, is $f(n)$ odd?",
      "answer": "55"
    },
    {
      "question": "There are 100 houses in a row on a street. A painter comes and paints every house red. Then, another painter comes and paints every third house (starting with house number 3) blue. Another painter comes and paints every fifth house red (even if it is already red), then another painter paints every seventh house blue, and so forth, alternating between red and blue, until 50 painters have been by. After this is finished, how many houses will be red?",
      "answer": "52"
    },
    {
      "question": "Suppose that $x, y$, and $z$ are non-negative real numbers such that $x+y+z=1$. What is the maximum possible value of $x+y^{2}+z^{3}$ ?",
      "answer": "1"
    },
    {
      "question": "Consider a cube $A B C D E F G H$, where $A B C D$ and $E F G H$ are faces, and segments $A E, B F, C G, D H$ are edges of the cube. Let $P$ be the center of face $E F G H$, and let $O$ be the center of the cube. Given that $A G=1$, determine the area of triangle $A O P$.",
      "answer": "$\\frac{\\sqrt{2}}{24}$"
    },
    {
      "question": "Five points are chosen uniformly at random on a segment of length 1. What is the expected distance between the closest pair of points?",
      "answer": "\\frac{1}{24}"
    },
    {
      "question": "Shuxin begins with 10 red candies, 7 yellow candies, and 3 blue candies. After eating some of the candies, there are equal numbers of red, yellow, and blue candies remaining. What is the smallest possible number of candies that Shuxin ate?",
      "answer": "11"
    },
    {
      "question": "Bob's Rice ID number has six digits, each a number from 1 to 9, and any digit can be used any number of times. The ID number satisfies the following property: the first two digits is a number divisible by 2, the first three digits is a number divisible by 3, etc. so that the ID number itself is divisible by 6. One ID number that satisfies this condition is 123252. How many different possibilities are there for Bob's ID number?",
      "answer": "324"
    },
    {
      "question": "Let $A B C$ be a triangle where $A B=9, B C=10, C A=17$. Let $\\Omega$ be its circumcircle, and let $A_{1}, B_{1}, C_{1}$ be the diametrically opposite points from $A, B, C$, respectively, on $\\Omega$. Find the area of the convex hexagon with the vertices $A, B, C, A_{1}, B_{1}, C_{1}$.",
      "answer": "\\frac{1155}{4}"
    },
    {
      "question": "Let $N$ be a positive integer whose decimal representation contains 11235 as a contiguous substring, and let $k$ be a positive integer such that $10^{k}>N$. Find the minimum possible value of $$ \\frac{10^{k}-1}{\\operatorname{gcd}\\left(N, 10^{k}-1\\right)} $$",
      "answer": "89"
    },
    {
      "question": "The cafeteria in a certain laboratory is open from noon until 2 in the afternoon every Monday for lunch. Two professors eat 15 minute lunches sometime between noon and 2. What is the probability that they are in the cafeteria simultaneously on any given Monday?",
      "answer": "15"
    },
    {
      "question": "A fair coin is flipped eight times in a row. Let $p$ be the probability that there is exactly one pair of consecutive flips that are both heads and exactly one pair of consecutive flips that are both tails. If $p=\\frac{a}{b}$, where $a, b$ are relatively prime positive integers, compute $100a+b$.",
      "answer": "1028"
    },
    {
      "question": "In the plane we consider rectangles whose sides are parallel to the coordinate axes and have positive length. Such a rectangle will be called a [i]box[/i]. Two boxes [i]intersect[/i] if they have a common point in their interior or on their boundary. Find the largest $ n$ for which there exist $ n$ boxes $ B_1$, $ \\ldots$, $ B_n$ such that $ B_i$ and $ B_j$ intersect if and only if $ i\\not\\equiv j\\pm 1\\pmod n$.\n\n[i]",
      "answer": "6"
    },
    {
      "question": "In how many ways can you rearrange the letters of \"HMMTHMMT\" such that the consecutive substring \"HMMT\" does not appear?",
      "answer": "361"
    },
    {
      "question": "A palindrome is a string that does not change when its characters are written in reverse order. Let S be a 40-digit string consisting only of 0's and 1's, chosen uniformly at random out of all such strings. Let $E$ be the expected number of nonempty contiguous substrings of $S$ which are palindromes. Compute the value of $\\lfloor E\\rfloor$.",
      "answer": "113"
    },
    {
      "question": "Consider a sequence $x_{n}$ such that $x_{1}=x_{2}=1, x_{3}=\\frac{2}{3}$. Suppose that $x_{n}=\\frac{x_{n-1}^{2} x_{n-2}}{2 x_{n-2}^{2}-x_{n-1} x_{n-3}}$ for all $n \\geq 4$. Find the least $n$ such that $x_{n} \\leq \\frac{1}{10^{6}}$.",
      "answer": "13"
    },
    {
      "question": "Betty has a $3 \\times 4$ grid of dots. She colors each dot either red or maroon. Compute the number of ways Betty can color the grid such that there is no rectangle whose sides are parallel to the grid lines and whose vertices all have the same color.",
      "answer": "408"
    },
    {
      "question": "$A B C$ is a triangle with $A B=15, B C=14$, and $C A=13$. The altitude from $A$ to $B C$ is extended to meet the circumcircle of $A B C$ at $D$. Find $A D$.",
      "answer": "\\frac{63}{4}"
    },
    {
      "question": "For each positive integer $n$, let $a_{n}$ be the smallest nonnegative integer such that there is only one positive integer at most $n$ that is relatively prime to all of $n, n+1, \\ldots, n+a_{n}$. If $n<100$, compute the largest possible value of $n-a_{n}$.",
      "answer": "16"
    },
    {
      "question": "In square $PQRS$ with side length 2, each of $P, Q, R$, and $S$ is the centre of a circle with radius 1. What is the area of the shaded region?",
      "answer": "4-\\pi"
    },
    {
      "question": "64 people are in a single elimination rock-paper-scissors tournament, which consists of a 6-round knockout bracket. Each person has a different rock-paper-scissors skill level, and in any game, the person with the higher skill level will always win. For how many players $P$ is it possible that $P$ wins the first four rounds that he plays?",
      "answer": "49"
    },
    {
      "question": "How many perfect squares divide $10^{10}$?",
      "answer": "36"
    },
    {
      "question": "Let $\\left(x_{1}, y_{1}\\right), \\ldots,\\left(x_{k}, y_{k}\\right)$ be the distinct real solutions to the equation $$\\left(x^{2}+y^{2}\\right)^{6}=\\left(x^{2}-y^{2}\\right)^{4}=\\left(2 x^{3}-6 x y^{2}\\right)^{3}$$ Then $\\sum_{i=1}^{k}\\left(x_{i}+y_{i}\\right)$ can be expressed as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
      "answer": "516"
    },
    {
      "question": "How many lines pass through exactly two points in the following hexagonal grid?",
      "answer": "60"
    },
    {
      "question": "In the game of rock-paper-scissors-lizard-Spock, rock defeats scissors and lizard, paper defeats rock and Spock, scissors defeats paper and lizard, lizard defeats paper and Spock, and Spock defeats rock and scissors. If three people each play a game of rock-paper-scissors-lizard-Spock at the same time by choosing one of the five moves at random, what is the probability that one player beats the other two?",
      "answer": "\\frac{12}{25}"
    },
    {
      "question": "$O$ is the center of square $A B C D$, and $M$ and $N$ are the midpoints of $\\overline{B C}$ and $\\overline{A D}$, respectively. Points $A^{\\prime}, B^{\\prime}, C^{\\prime}, D^{\\prime}$ are chosen on $\\overline{A O}, \\overline{B O}, \\overline{C O}, \\overline{D O}$, respectively, so that $A^{\\prime} B^{\\prime} M C^{\\prime} D^{\\prime} N$ is an equiangular hexagon. The ratio $\\frac{\\left[A^{\\prime} B^{\\prime} M C^{\\prime} D^{\\prime} N\\right]}{[A B C D]}$ can be written as $\\frac{a+b \\sqrt{c}}{d}$, where $a, b, c, d$ are integers, $d$ is positive, $c$ is square-free, and $\\operatorname{gcd}(a, b, d)=1$. Find $1000 a+100 b+10 c+d$.",
      "answer": "8634"
    },
    {
      "question": "Point $P$ is inside a square $A B C D$ such that $\\angle A P B=135^{\\circ}, P C=12$, and $P D=15$. Compute the area of this square.",
      "answer": "123+6\\sqrt{119}"
    },
    {
      "question": "Suppose that $A, B, C, D$ are four points in the plane, and let $Q, R, S, T, U, V$ be the respective midpoints of $A B, A C, A D, B C, B D, C D$. If $Q R=2001, S U=2002, T V=$ 2003, find the distance between the midpoints of $Q U$ and $R V$.",
      "answer": "2001"
    },
    {
      "question": "Brian has a 20-sided die with faces numbered from 1 to 20, and George has three 6-sided dice with faces numbered from 1 to 6. Brian and George simultaneously roll all their dice. What is the probability that the number on Brian's die is larger than the sum of the numbers on George's dice?",
      "answer": "\\frac{19}{40}"
    },
    {
      "question": "Let $\\mathcal{A}$ denote the set of all polynomials in three variables $x, y, z$ with integer coefficients. Let $\\mathcal{B}$ denote the subset of $\\mathcal{A}$ formed by all polynomials which can be expressed as\n\\begin{align*}\n(x + y + z)P(x, y, z) + (xy + yz + zx)Q(x, y, z) + xyzR(x, y, z)\n\\end{align*}\nwith $P, Q, R \\in \\mathcal{A}$.  Find the smallest non-negative integer $n$ such that $x^i y^j z^k \\in \\mathcal{B}$ for all non-negative integers $i, j, k$ satisfying $i + j + k \\geq n$.",
      "answer": "4"
    },
    {
      "question": "Denote $S$ as the subset of $\\{1,2,3,\\dots,1000\\}$ with the property that none of the sums of two different elements in $S$ is in $S$. Find the maximum number of elements in $S$.",
      "answer": "501"
    },
    {
      "question": "The equation $x^{2}+2 x=i$ has two complex solutions. Determine the product of their real parts.",
      "answer": "\\frac{1-\\sqrt{2}}{2}"
    },
    {
      "question": "Find the greatest common divisor of the numbers $2002+2,2002^{2}+2,2002^{3}+2, \\ldots$.",
      "answer": "6"
    },
    {
      "question": "Let $Q(x)=x^{2}+2x+3$, and suppose that $P(x)$ is a polynomial such that $P(Q(x))=x^{6}+6x^{5}+18x^{4}+32x^{3}+35x^{2}+22x+8$. Compute $P(2)$.",
      "answer": "2"
    },
    {
      "question": "Mark writes the expression $\\sqrt{d}$ for each positive divisor $d$ of 8 ! on the board. Seeing that these expressions might not be worth points on HMMT, Rishabh simplifies each expression to the form $a \\sqrt{b}$, where $a$ and $b$ are integers such that $b$ is not divisible by the square of a prime number. Compute the sum of $a+b$ across all expressions that Rishabh writes.",
      "answer": "3480"
    },
    {
      "question": "How many times does the letter \"e\" occur in all problem statements in this year's HMMT February competition?",
      "answer": "1661"
    },
    {
      "question": "John ate a total of 120 peanuts over four consecutive nights. Each night he ate 6 more peanuts than the night before. How many peanuts did he eat on the fourth night?",
      "answer": "39"
    },
    {
      "question": "Compute $\\sqrt[4]{5508^{3}+5625^{3}+5742^{3}}$, given that it is an integer.",
      "answer": "855"
    },
    {
      "question": "Michael is playing basketball. He makes $10 \\%$ of his shots, and gets the ball back after $90 \\%$ of his missed shots. If he does not get the ball back he stops playing. What is the probability that Michael eventually makes a shot?",
      "answer": "\\frac{10}{19}"
    },
    {
      "question": "Let $$ A=\\lim _{n \\rightarrow \\infty} \\sum_{i=0}^{2016}(-1)^{i} \\cdot \\frac{\\binom{n}{i}\\binom{n}{i+2}}{\\binom{n}{i+1}^{2}} $$ Find the largest integer less than or equal to $\\frac{1}{A}$.",
      "answer": "1"
    },
    {
      "question": "A stacking of circles in the plane consists of a base, or some number of unit circles centered on the $x$-axis in a row without overlap or gaps, and circles above the $x$-axis that must be tangent to two circles below them (so that if the ends of the base were secured and gravity were applied from below, then nothing would move). How many stackings of circles in the plane have 4 circles in the base?",
      "answer": "14"
    },
    {
      "question": "In circle $\\omega$, two perpendicular chords intersect at a point $P$. The two chords have midpoints $M_{1}$ and $M_{2}$ respectively, such that $P M_{1}=15$ and $P M_{2}=20$. Line $M_{1} M_{2}$ intersects $\\omega$ at points $A$ and $B$, with $M_{1}$ between $A$ and $M_{2}$. Compute the largest possible value of $B M_{2}-A M_{1}$.",
      "answer": "7"
    },
    {
      "question": "How many ways are there to cover a $3 \\times 8$ rectangle with 12 identical dominoes?",
      "answer": "153"
    },
    {
      "question": "Find the smallest number $n$ such that there exist polynomials $f_1, f_2, \\ldots , f_n$ with rational coefficients satisfying \\[x^2+7 = f_1\\left(x\\right)^2 + f_2\\left(x\\right)^2 + \\ldots + f_n\\left(x\\right)^2.\\]\n\n[i]",
      "answer": "5"
    },
    {
      "question": "Suppose $x, y$, and $z$ are real numbers greater than 1 such that $$\\begin{aligned} x^{\\log _{y} z} & =2, \\\\ y^{\\log _{z} x} & =4, \\text { and } \\\\ z^{\\log _{x} y} & =8 \\end{aligned}$$ Compute $\\log _{x} y$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "One writes 268 numbers around a circle, such that the sum of 20 consectutive numbers is always equal to 75. The number 3, 4 and 9 are written in positions 17, 83 and 144 respectively. Find the number in position 210.",
      "answer": "-1"
    },
    {
      "question": "For a positive integer $n$, let, $\\tau(n)$ be the number of positive integer divisors of $n$. How many integers $1 \\leq n \\leq 50$ are there such that $\\tau(\\tau(n))$ is odd?",
      "answer": "17"
    },
    {
      "question": "Given a permutation $\\sigma$ of $\\{1,2, \\ldots, 2013\\}$, let $f(\\sigma)$ to be the number of fixed points of $\\sigma$ - that is, the number of $k \\in\\{1,2, \\ldots, 2013\\}$ such that $\\sigma(k)=k$. If $S$ is the set of all possible permutations $\\sigma$, compute $$\\sum_{\\sigma \\in S} f(\\sigma)^{4}$$ (Here, a permutation $\\sigma$ is a bijective mapping from $\\{1,2, \\ldots, 2013\\}$ to $\\{1,2, \\ldots, 2013\\}$.)",
      "answer": "15(2013!)"
    },
    {
      "question": "In triangle $ABC, AB=32, AC=35$, and $BC=x$. What is the smallest positive integer $x$ such that $1+\\cos^{2}A, \\cos^{2}B$, and $\\cos^{2}C$ form the sides of a non-degenerate triangle?",
      "answer": "48"
    },
    {
      "question": "Compute the number of positive four-digit multiples of 11 whose sum of digits (in base ten) is divisible by 11.",
      "answer": "72"
    },
    {
      "question": "Compute $$\\sum_{n_{60}=0}^{2} \\sum_{n_{59}=0}^{n_{60}} \\cdots \\sum_{n_{2}=0}^{n_{3}} \\sum_{n_{1}=0}^{n_{2}} \\sum_{n_{0}=0}^{n_{1}} 1$$",
      "answer": "1953"
    },
    {
      "question": "Two distinct squares on a $4 \\times 4$ chessboard are chosen, with each pair of squares equally likely to be chosen. A knight is placed on one of the squares. The expected value of the minimum number of moves it takes for the knight to reach the other squarecan be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "1205"
    },
    {
      "question": "At a nursery, 2006 babies sit in a circle. Suddenly each baby pokes the baby immediately to either its left or its right, with equal probability. What is the expected number of unpoked babies?",
      "answer": "\\frac{1003}{2}"
    },
    {
      "question": "The pairwise products $a b, b c, c d$, and $d a$ of positive integers $a, b, c$, and $d$ are $64,88,120$, and 165 in some order. Find $a+b+c+d$.",
      "answer": "42"
    },
    {
      "question": "Jeff has a 50 point quiz at 11 am . He wakes up at a random time between 10 am and noon, then arrives at class 15 minutes later. If he arrives on time, he will get a perfect score, but if he arrives more than 30 minutes after the quiz starts, he will get a 0 , but otherwise, he loses a point for each minute he's late (he can lose parts of one point if he arrives a nonintegral number of minutes late). What is Jeff's expected score on the quiz?",
      "answer": "\\frac{55}{2}"
    },
    {
      "question": "How many lattice points are enclosed by the triangle with vertices $(0,99),(5,100)$, and $(2003,500) ?$ Don't count boundary points.",
      "answer": "0"
    },
    {
      "question": "Let $n$ be the answer to this problem. Suppose square $ABCD$ has side-length 3. Then, congruent non-overlapping squares $EHGF$ and $IHJK$ of side-length $\\frac{n}{6}$ are drawn such that $A, C$, and $H$ are collinear, $E$ lies on $BC$ and $I$ lies on $CD$. Given that $AJG$ is an equilateral triangle, then the area of $AJG$ is $a+b\\sqrt{c}$, where $a, b, c$ are positive integers and $c$ is not divisible by the square of any prime. Find $a+b+c$.",
      "answer": "48"
    },
    {
      "question": "Convex quadrilateral $ ABCD$ is inscribed in a circle, $ \\angle{A}\\equal{}60^o$, $ BC\\equal{}CD\\equal{}1$, rays $ AB$ and $ DC$ intersect at point $ E$, rays $ BC$ and $ AD$ intersect each other at point $ F$. It is given that the perimeters of triangle $ BCE$ and triangle $ CDF$ are both integers. Find the perimeter of quadrilateral $ ABCD$.",
      "answer": "\\frac{38}{7}"
    },
    {
      "question": "Suppose that $x$ and $y$ are real numbers that satisfy the two equations: $x^{2} + 3xy + y^{2} = 909$ and $3x^{2} + xy + 3y^{2} = 1287$. What is a possible value for $x+y$?",
      "answer": "27"
    },
    {
      "question": "The $GEB$ sequence $1,3,7,12, \\ldots$ is defined by the following properties: (i) the GEB sequence is increasing (that is, each term is larger than the previous term), (ii) the sequence formed using the differences between each pair of consecutive terms in the GEB sequence (namely, the sequence $2,4,5, \\ldots$) is increasing, and (iii) each positive integer that does not occur in the GEB sequence occurs exactly once in the sequence of differences in (ii). What is the 100th term of the GEB sequence?",
      "answer": "5764"
    },
    {
      "question": "Let $S$ be the set of integers of the form $2^{x}+2^{y}+2^{z}$, where $x, y, z$ are pairwise distinct non-negative integers. Determine the 100th smallest element of $S$.",
      "answer": "577"
    },
    {
      "question": "Find the sum $$\\frac{2^{1}}{4^{1}-1}+\\frac{2^{2}}{4^{2}-1}+\\frac{2^{4}}{4^{4}-1}+\\frac{2^{8}}{4^{8}-1}+\\cdots$$",
      "answer": "1"
    },
    {
      "question": "Let $n \\ge 3$ be an integer. What is the largest possible number of interior angles greater than $180^\\circ$ in an $n$-gon in the plane, given that the $n$-gon does not intersect itself and all its sides have the same length?",
      "answer": "0"
    },
    {
      "question": "If $f(x)$ is a monic quartic polynomial such that $f(-1)=-1, f(2)=-4, f(-3)=-9$, and $f(4)=-16$, find $f(1)$.",
      "answer": "23"
    },
    {
      "question": "Bob the bomb-defuser has stumbled upon an active bomb. He opens it up, and finds the red and green wires conveniently located for him to cut. Being a seasoned member of the bomb-squad, Bob quickly determines that it is the green wire that he should cut, and puts his wirecutters on the green wire. But just before he starts to cut, the bomb starts to count down, ticking every second. Each time the bomb ticks, starting at time $t=15$ seconds, Bob panics and has a certain chance to move his wirecutters to the other wire. However, he is a rational man even when panicking, and has a $\\frac{1}{2 t^{2}}$ chance of switching wires at time $t$, regardless of which wire he is about to cut. When the bomb ticks at $t=1$, Bob cuts whatever wire his wirecutters are on, without switching wires. What is the probability that Bob cuts the green wire?",
      "answer": "\\frac{23}{30}"
    },
    {
      "question": "Chords $A B$ and $C D$ of a circle are perpendicular and intersect at a point $P$. If $A P=6, B P=12$, and $C D=22$, find the area of the circle.",
      "answer": "130 \\pi"
    },
    {
      "question": "Suppose there are initially 1001 townspeople and two goons. What is the probability that, when the game ends, there are exactly 1000 people in jail?",
      "answer": "\\frac{3}{1003}"
    },
    {
      "question": "Find the number of eight-digit positive integers that are multiples of 9 and have all distinct digits.",
      "answer": "181440"
    },
    {
      "question": "Points $P$ and $Q$ are 3 units apart. A circle centered at $P$ with a radius of $\\sqrt{3}$ units intersects a circle centered at $Q$ with a radius of 3 units at points $A$ and $B$. Find the area of quadrilateral APBQ.",
      "answer": "\\frac{3 \\sqrt{11}}{2}"
    },
    {
      "question": "Consider an unusual biased coin, with probability $p$ of landing heads, probability $q \\leq p$ of landing tails, and probability \\frac{1}{6}$ of landing on its side (i.e. on neither face). It is known that if this coin is flipped twice, the likelihood that both flips will have the same result is \\frac{1}{2}$. Find $p$.",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "There are 2017 jars in a row on a table, initially empty. Each day, a nice man picks ten consecutive jars and deposits one coin in each of the ten jars. Later, Kelvin the Frog comes back to see that $N$ of the jars all contain the same positive integer number of coins (i.e. there is an integer $d>0$ such that $N$ of the jars have exactly $d$ coins). What is the maximum possible value of $N$?",
      "answer": "2014"
    },
    {
      "question": "Let $A, B, C, D$ be points chosen on a circle, in that order. Line $B D$ is reflected over lines $A B$ and $D A$ to obtain lines $\\ell_{1}$ and $\\ell_{2}$ respectively. If lines $\\ell_{1}, \\ell_{2}$, and $A C$ meet at a common point and if $A B=4, B C=3, C D=2$, compute the length $D A$.",
      "answer": "\\sqrt{21}"
    },
    {
      "question": "What is the value of $m$ if Tobias downloads $m$ apps, each app costs $\\$ 2.00$ plus $10 \\%$ tax, and he spends $\\$ 52.80$ in total on these $m$ apps?",
      "answer": "24"
    },
    {
      "question": "2500 chess kings have to be placed on a $100 \\times 100$ chessboard so that\n\n[b](i)[/b] no king can capture any other one (i.e. no two kings are placed in two squares sharing a common vertex);\n[b](ii)[/b] each row and each column contains exactly 25 kings.\n\nFind the number of such arrangements. (Two arrangements differing by rotation or symmetry are supposed to be different.)\n\n[i]",
      "answer": "2"
    },
    {
      "question": "The numbers $1,2 \\cdots 11$ are arranged in a line from left to right in a random order. It is observed that the middle number is larger than exactly one number to its left. Find the probability that it is larger than exactly one number to its right.",
      "answer": "\\frac{10}{33}"
    },
    {
      "question": "Find the number of ways to distribute 4 pieces of candy to 12 children such that no two consecutive children receive candy.",
      "answer": "105"
    },
    {
      "question": "If \\( x=2 \\) and \\( v=3x \\), what is the value of \\((2v-5)-(2x-5)\\)?",
      "answer": "8"
    },
    {
      "question": "Let $S$ be the set of all points in the plane whose coordinates are positive integers less than or equal to 100 (so $S$ has $100^{2}$ elements), and let $\\mathcal{L}$ be the set of all lines $\\ell$ such that $\\ell$ passes through at least two points in $S$. Find, with proof, the largest integer $N \\geq 2$ for which it is possible to choose $N$ distinct lines in $\\mathcal{L}$ such that every two of the chosen lines are parallel.",
      "answer": "4950"
    },
    {
      "question": "If the system of equations $$\\begin{aligned} & |x+y|=99 \\\\ & |x-y|=c \\end{aligned}$$ has exactly two real solutions $(x, y)$, find the value of $c$.",
      "answer": "0"
    },
    {
      "question": "How many roots does $\\arctan x=x^{2}-1.6$ have, where the arctan function is defined in the range $-\\frac{p i}{2}<\\arctan x<\\frac{p i}{2}$ ?",
      "answer": "2"
    },
    {
      "question": "A real number \\(x\\) is chosen uniformly at random from the interval \\([0,1000]\\). Find the probability that \\(\\left\\lfloor\\frac{\\left\\lfloor\\frac{x}{2.5}\\right\\rfloor}{2.5}\\right\\rfloor=\\left\\lfloor\\frac{x}{6.25}\\right\\rfloor\\).",
      "answer": "\\frac{9}{10}"
    },
    {
      "question": "At a university dinner, there are 2017 mathematicians who each order two distinct entrées, with no two mathematicians ordering the same pair of entrées. The cost of each entrée is equal to the number of mathematicians who ordered it, and the university pays for each mathematician's less expensive entrée (ties broken arbitrarily). Over all possible sets of orders, what is the maximum total amount the university could have paid?",
      "answer": "127009"
    },
    {
      "question": "If $\\frac{1}{9}$ of 60 is 5, what is $\\frac{1}{20}$ of 80?",
      "answer": "6"
    },
    {
      "question": "A rubber band is 4 inches long. An ant begins at the left end. Every minute, the ant walks one inch along rightwards along the rubber band, but then the band is stretched (uniformly) by one inch. For what value of $n$ will the ant reach the right end during the $n$th minute?",
      "answer": "7"
    },
    {
      "question": "In the $x-y$ plane, draw a circle of radius 2 centered at $(0,0)$. Color the circle red above the line $y=1$, color the circle blue below the line $y=-1$, and color the rest of the circle white. Now consider an arbitrary straight line at distance 1 from the circle. We color each point $P$ of the line with the color of the closest point to $P$ on the circle. If we pick such an arbitrary line, randomly oriented, what is the probability that it contains red, white, and blue points?",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "Every second, Andrea writes down a random digit uniformly chosen from the set $\\{1,2,3,4\\}$. She stops when the last two numbers she has written sum to a prime number. What is the probability that the last number she writes down is 1?",
      "answer": "15/44"
    },
    {
      "question": "Amina and Bert alternate turns tossing a fair coin. Amina goes first and each player takes three turns. The first player to toss a tail wins. If neither Amina nor Bert tosses a tail, then neither wins. What is the probability that Amina wins?",
      "answer": "\\frac{21}{32}"
    },
    {
      "question": "An ordered pair of sets $(A, B)$ is good if $A$ is not a subset of $B$ and $B$ is not a subset of $A$. How many ordered pairs of subsets of $\\{1,2, \\ldots, 2017\\}$ are good?",
      "answer": "4^{2017}-2 \\cdot 3^{2017}+2^{2017}"
    },
    {
      "question": "If $2 \\times 2 \\times 3 \\times 3 \\times 5 \\times 6=5 \\times 6 \\times n \\times n$, what is a possible value of $n$?",
      "answer": "6"
    },
    {
      "question": "Paul fills in a $7 \\times 7$ grid with the numbers 1 through 49 in a random arrangement. He then erases his work and does the same thing again (to obtain two different random arrangements of the numbers in the grid). What is the expected number of pairs of numbers that occur in either the same row as each other or the same column as each other in both of the two arrangements?",
      "answer": "147 / 2"
    },
    {
      "question": "Let $n$ be the answer to this problem. Hexagon $ABCDEF$ is inscribed in a circle of radius 90. The area of $ABCDEF$ is $8n$, $AB=BC=DE=EF$, and $CD=FA$. Find the area of triangle $ABC$.",
      "answer": "2592"
    },
    {
      "question": "The integers $a, b$ and $c$ satisfy the equations $a+5=b$, $5+b=c$, and $b+c=a$. What is the value of $b$?",
      "answer": "-10"
    },
    {
      "question": "The three numbers $5, a, b$ have an average (mean) of 33. What is the average of $a$ and $b$?",
      "answer": "47"
    },
    {
      "question": "Victor has a drawer with two red socks, two green socks, two blue socks, two magenta socks, two lavender socks, two neon socks, two mauve socks, two wisteria socks, and 2000 copper socks, for a total of 2016 socks. He repeatedly draws two socks at a time from the drawer at random, and stops if the socks are of the same color. However, Victor is red-green colorblind, so he also stops if he sees a red and green sock. What is the probability that Victor stops with two socks of the same color? Assume Victor returns both socks to the drawer at each step.",
      "answer": "\\frac{1999008}{1999012}"
    },
    {
      "question": "Let $p=2^{24036583}-1$, the largest prime currently known. For how many positive integers $c$ do the quadratics \\pm x^{2} \\pm p x \\pm c all have rational roots?",
      "answer": "0"
    },
    {
      "question": "How many functions $f:\\{1,2,3,4,5\\} \\rightarrow\\{1,2,3,4,5\\}$ have the property that $f(\\{1,2,3\\})$ and $f(f(\\{1,2,3\\}))$ are disjoint?",
      "answer": "94"
    },
    {
      "question": "What is the earliest row in which the number 2004 may appear?",
      "answer": "12"
    },
    {
      "question": "Find the number of solutions in positive integers $(k ; a_{1}, a_{2}, \\ldots, a_{k} ; b_{1}, b_{2}, \\ldots, b_{k})$ to the equation $$a_{1}(b_{1})+a_{2}(b_{1}+b_{2})+\\cdots+a_{k}(b_{1}+b_{2}+\\cdots+b_{k})=7$$",
      "answer": "15"
    },
    {
      "question": "Given a $9 \\times 9$ chess board, we consider all the rectangles whose edges lie along grid lines (the board consists of 81 unit squares, and the grid lines lie on the borders of the unit squares). For each such rectangle, we put a mark in every one of the unit squares inside it. When this process is completed, how many unit squares will contain an even number of marks?",
      "answer": "56"
    },
    {
      "question": "How many ways are there to insert +'s between the digits of 111111111111111 (fifteen 1's) so that the result will be a multiple of 30?",
      "answer": "2002"
    },
    {
      "question": "Several positive integers are written on a blackboard. The sum of any two of them is some power of two (for example, $2, 4, 8,...$). What is the maximal possible number of different integers on the blackboard?",
      "answer": "2"
    },
    {
      "question": "For each prime $p$, a polynomial $P(x)$ with rational coefficients is called $p$-good if and only if there exist three integers $a, b$, and $c$ such that $0 \\leq a<b<c<\\frac{p}{3}$ and $p$ divides all the numerators of $P(a)$, $P(b)$, and $P(c)$, when written in simplest form. Compute the number of ordered pairs $(r, s)$ of rational numbers such that the polynomial $x^{3}+10x^{2}+rx+s$ is $p$-good for infinitely many primes $p$.",
      "answer": "12"
    },
    {
      "question": "Consider the cube whose vertices are the eight points $(x, y, z)$ for which each of $x, y$, and $z$ is either 0 or 1 . How many ways are there to color its vertices black or white such that, for any vertex, if all of its neighbors are the same color then it is also that color? Two vertices are neighbors if they are the two endpoints of some edge of the cube.",
      "answer": "118"
    },
    {
      "question": "The average of 1, 3, and \\( x \\) is 3. What is the value of \\( x \\)?",
      "answer": "5"
    },
    {
      "question": "Call a string of letters $S$ an almost palindrome if $S$ and the reverse of $S$ differ in exactly two places. Find the number of ways to order the letters in $H M M T T H E M E T E A M$ to get an almost palindrome.",
      "answer": "2160"
    },
    {
      "question": "The sequence $\\left\\{a_{n}\\right\\}_{n \\geq 1}$ is defined by $a_{n+2}=7 a_{n+1}-a_{n}$ for positive integers $n$ with initial values $a_{1}=1$ and $a_{2}=8$. Another sequence, $\\left\\{b_{n}\\right\\}$, is defined by the rule $b_{n+2}=3 b_{n+1}-b_{n}$ for positive integers $n$ together with the values $b_{1}=1$ and $b_{2}=2$. Find \\operatorname{gcd}\\left(a_{5000}, b_{501}\\right).",
      "answer": "89"
    },
    {
      "question": "Trapezoid $A B C D$, with bases $A B$ and $C D$, has side lengths $A B=28, B C=13, C D=14$, and $D A=15$. Let diagonals $A C$ and $B D$ intersect at $P$, and let $E$ and $F$ be the midpoints of $A P$ and $B P$, respectively. Find the area of quadrilateral $C D E F$.",
      "answer": "112"
    },
    {
      "question": "Simplify: $2 \\sqrt{1.5+\\sqrt{2}}-(1.5+\\sqrt{2})$.",
      "answer": "1/2"
    },
    {
      "question": "For any positive real number $x, \\lfloor x \\rfloor$ denotes the largest integer less than or equal to $x$. If $\\lfloor x \\rfloor \\cdot x = 36$ and $\\lfloor y \\rfloor \\cdot y = 71$ where $x, y > 0$, what is $x + y$ equal to?",
      "answer": "\\frac{119}{8}"
    },
    {
      "question": "Suppose $A_1,A_2,\\cdots ,A_n \\subseteq \\left \\{ 1,2,\\cdots ,2018 \\right \\}$ and $\\left | A_i \\right |=2, i=1,2,\\cdots ,n$, satisfying that $$A_i + A_j, \\; 1 \\le i \\le j \\le n ,$$ are distinct from each other. $A + B = \\left \\{ a+b|a\\in A,\\,b\\in B \\right \\}$. Determine the maximal value of $n$.",
      "answer": "4033"
    },
    {
      "question": "If $x, y, z$ are real numbers such that $xy=6, x-z=2$, and $x+y+z=9$, compute $\\frac{x}{y}-\\frac{z}{x}-\\frac{z^{2}}{xy}$.",
      "answer": "2"
    },
    {
      "question": "Create a cube $C_{1}$ with edge length 1. Take the centers of the faces and connect them to form an octahedron $O_{1}$. Take the centers of the octahedron's faces and connect them to form a new cube $C_{2}$. Continue this process infinitely. Find the sum of all the surface areas of the cubes and octahedrons.",
      "answer": "\\frac{54+9 \\sqrt{3}}{8}"
    },
    {
      "question": "Jeffrey writes the numbers 1 and $100000000=10^{8}$ on the blackboard. Every minute, if $x, y$ are on the board, Jeffrey replaces them with $\\frac{x+y}{2} \\text{ and } 2\\left(\\frac{1}{x}+\\frac{1}{y}\\right)^{-1}$. After 2017 minutes the two numbers are $a$ and $b$. Find $\\min (a, b)$ to the nearest integer.",
      "answer": "10000"
    },
    {
      "question": "There are functions $f(x)$ with the following properties: $f(x)=ax^{2}+bx+c$ for some integers $a, b$ and $c$ with $a>0$, and $f(p)=f(q)=17$ and $f(p+q)=47$ for some prime numbers $p$ and $q$ with $p<q$. For each such function, the value of $f(pq)$ is calculated. The sum of all possible values of $f(pq)$ is $S$. What are the rightmost two digits of $S$?",
      "answer": "71"
    },
    {
      "question": "For each positive integer $n$, let $s(n)$ be the sum of the squares of the digits of $n$. For example, $s(15)=1^2+5^2=26$. Determine all integers $n\\geq 1$ such that $s(n)=n$.",
      "answer": "1"
    },
    {
      "question": "In Rad's garden there are exactly 30 red roses, exactly 19 yellow roses, and no other roses. How many of the yellow roses does Rad need to remove so that $\\frac{2}{7}$ of the roses in the garden are yellow?",
      "answer": "7"
    },
    {
      "question": "Find a real number $t$ such that for any set of 120 points $P_1, \\ldots P_{120}$ on the boundary of a unit square, there exists a point $Q$ on this boundary with $|P_1Q| + |P_2Q| + \\cdots + |P_{120}Q| = t$.",
      "answer": "30(1 + \\sqrt{5})"
    },
    {
      "question": "Find the area of a triangle with side lengths 14, 48, and 50.",
      "answer": "336"
    },
    {
      "question": "A circle is tangent to both branches of the hyperbola $x^{2}-20y^{2}=24$ as well as the $x$-axis. Compute the area of this circle.",
      "answer": "504\\pi"
    },
    {
      "question": "An infinite sequence of positive real numbers is defined by \\( a_{0}=1 \\) and \\( a_{n+2}=6 a_{n}-a_{n+1} \\) for \\( n=0,1,2, \\cdots \\) Find the possible value(s) of \\( a_{2007} \\).",
      "answer": "2^{2007}"
    },
    {
      "question": "Find the largest positive integer \\(n\\) for which there exist \\(n\\) finite sets \\(X_{1}, X_{2}, \\ldots, X_{n}\\) with the property that for every \\(1 \\leq a<b<c \\leq n\\), the equation \\(\\left|X_{a} \\cup X_{b} \\cup X_{c}\\right|=\\lceil\\sqrt{a b c}\\rceil\\) holds.",
      "answer": "4"
    },
    {
      "question": "FIx positive integer $n$. Prove: For any positive integers $a,b,c$ not exceeding $3n^2+4n$, there exist integers $x,y,z$ with absolute value not exceeding $2n$ and not all $0$, such that $ax+by+cz=0$",
      "answer": "0"
    },
    {
      "question": "Compute the sum of all positive integers $n$ such that $50 \\leq n \\leq 100$ and $2 n+3$ does not divide $2^{n!}-1$.",
      "answer": "222"
    },
    {
      "question": "There are 21 competitors with distinct skill levels numbered $1,2, \\ldots, 21$. They participate in a pingpong tournament as follows. First, a random competitor is chosen to be \"active\", while the rest are \"inactive.\" Every round, a random inactive competitor is chosen to play against the current active one. The player with the higher skill will win and become (or remain) active, while the loser will be eliminated from the tournament. The tournament lasts for 20 rounds, after which there will only be one player remaining. Alice is the competitor with skill 11. What is the expected number of games that she will get to play?",
      "answer": "\\frac{47}{42}"
    },
    {
      "question": "How many integers between 1 and 2000 inclusive share no common factors with 2001?",
      "answer": "1232"
    },
    {
      "question": "Chris received a mark of $50 \\%$ on a recent test. Chris answered 13 of the first 20 questions correctly. Chris also answered $25 \\%$ of the remaining questions on the test correctly. If each question on the test was worth one mark, how many questions in total were on the test?",
      "answer": "32"
    },
    {
      "question": "Let $ABCD$ be a trapezoid with $AB \\parallel CD$ and $\\angle D=90^{\\circ}$. Suppose that there is a point $E$ on $CD$ such that $AE=BE$ and that triangles $AED$ and $CEB$ are similar, but not congruent. Given that $\\frac{CD}{AB}=2014$, find $\\frac{BC}{AD}$.",
      "answer": "\\sqrt{4027}"
    },
    {
      "question": "Determine the smallest positive integer $n \\geq 3$ for which $$A \\equiv 2^{10 n}\\left(\\bmod 2^{170}\\right)$$ where $A$ denotes the result when the numbers $2^{10}, 2^{20}, \\ldots, 2^{10 n}$ are written in decimal notation and concatenated (for example, if $n=2$ we have $A=10241048576$).",
      "answer": "14"
    },
    {
      "question": "A real number $x$ satisfies $9^{x}+3^{x}=6$. Compute the value of $16^{1 / x}+4^{1 / x}$.",
      "answer": "90"
    },
    {
      "question": "The remainder when 111 is divided by 10 is 1. The remainder when 111 is divided by the positive integer $n$ is 6. How many possible values of $n$ are there?",
      "answer": "5"
    },
    {
      "question": "In the figure below, how many ways are there to select 5 bricks, one in each row, such that any two bricks in adjacent rows are adjacent?",
      "answer": "61"
    },
    {
      "question": "Let $a_{1}, a_{2}, \\ldots$ be an arithmetic sequence and $b_{1}, b_{2}, \\ldots$ be a geometric sequence. Suppose that $a_{1} b_{1}=20$, $a_{2} b_{2}=19$, and $a_{3} b_{3}=14$. Find the greatest possible value of $a_{4} b_{4}$.",
      "answer": "\\frac{37}{4}"
    },
    {
      "question": "Call a triangle nice if the plane can be tiled using congruent copies of this triangle so that any two triangles that share an edge (or part of an edge) are reflections of each other via the shared edge. How many dissimilar nice triangles are there?",
      "answer": "4"
    },
    {
      "question": "Pablo has 27 solid $1 \\times 1 \\times 1$ cubes that he assembles in a larger $3 \\times 3 \\times 3$ cube. If 10 of the smaller cubes are red, 9 are blue, and 8 are yellow, what is the smallest possible surface area of the larger cube that is red?",
      "answer": "12"
    },
    {
      "question": "Determine the number of 8-tuples of nonnegative integers $\\left(a_{1}, a_{2}, a_{3}, a_{4}, b_{1}, b_{2}, b_{3}, b_{4}\\right)$ satisfying $0 \\leq a_{k} \\leq k$, for each $k=1,2,3,4$, and $a_{1}+a_{2}+a_{3}+a_{4}+2 b_{1}+3 b_{2}+4 b_{3}+5 b_{4}=19$.",
      "answer": "1540"
    },
    {
      "question": "We have an $n$-gon, and each of its vertices is labeled with a number from the set $\\{1, \\ldots, 10\\}$. We know that for any pair of distinct numbers from this set there is at least one side of the polygon whose endpoints have these two numbers. Find the smallest possible value of $n$.",
      "answer": "50"
    },
    {
      "question": "In the sum shown, $P, Q$ and $R$ represent three different single digits. What is the value of $P+Q+R$?\n\n\\begin{tabular}{r}\n$P 7 R$ \\\\\n$+\\quad 39 R$ \\\\\n\\hline$R Q 0$\n\\end{tabular}",
      "answer": "13"
    },
    {
      "question": "Mona has 12 match sticks of length 1, and she has to use them to make regular polygons, with each match being a side or a fraction of a side of a polygon, and no two matches overlapping or crossing each other. What is the smallest total area of the polygons Mona can make?",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "For any positive integer $n$, let \\langle n\\rangle denote the closest integer to \\sqrt{n}.  Evaluate\n\\[\\sum_{n=1}^\\infty \\frac{2^{\\langle n\\rangle}+2^{-\\langle n\\rangle}}{2^n}.\\]",
      "answer": "3"
    },
    {
      "question": "Find the sum of the ages of everyone who wrote a problem for this year's HMMT November contest. If your answer is $X$ and the actual value is $Y$, your score will be $\\max (0,20-|X-Y|)$",
      "answer": "258"
    },
    {
      "question": "Given that three roots of $f(x) = x^{4} + ax^{2} + bx + c$ are $2, -3$, and $5$, what is the value of $a + b + c$?",
      "answer": "79"
    },
    {
      "question": "Vivek is painting three doors numbered 1, 2, and 3. Each door is to be painted either black or gold. How many different ways can the three doors be painted?",
      "answer": "8"
    },
    {
      "question": "Let $n$ be the answer to this problem. Box $B$ initially contains $n$ balls, and Box $A$ contains half as many balls as Box $B$. After 80 balls are moved from Box $A$ to Box $B$, the ratio of balls in Box $A$ to Box $B$ is now $\\frac{p}{q}$, where $p, q$ are positive integers with $\\operatorname{gcd}(p, q)=1$. Find $100p+q$.",
      "answer": "720"
    },
    {
      "question": "For positive integers $n$, let $L(n)$ be the largest factor of $n$ other than $n$ itself. Determine the number of ordered pairs of composite positive integers $(m, n)$ for which $L(m) L(n)=80$.",
      "answer": "12"
    },
    {
      "question": "Alex chose positive integers $a, b, c, d, e, f$ and completely multiplied out the polynomial product $(1-x)^{a}(1+x)^{b}\\left(1-x+x^{2}\\right)^{c}\\left(1+x^{2}\\right)^{d}\\left(1+x+x^{2}\\right)^{e}\\left(1+x+x^{2}+x^{3}+x^{4}\\right)^{f}$. After she simplified her result, she discarded any term involving $x$ to any power larger than 6 and was astonished to see that what was left was $1-2 x$. If $a>d+e+f$ and $b>c+d$ and $e>c$, what value of $a$ did she choose?",
      "answer": "23"
    },
    {
      "question": "Find all positive integers $n$ such that the inequality $$\\left( \\sum\\limits_{i=1}^n a_i^2\\right) \\left(\\sum\\limits_{i=1}^n a_i \\right) -\\sum\\limits_{i=1}^n a_i^3 \\geq 6 \\prod\\limits_{i=1}^n a_i$$ holds for any $n$ positive numbers $a_1, \\dots, a_n$.",
      "answer": "3"
    },
    {
      "question": "Let $\\zeta=\\cos \\frac{2 \\pi}{13}+i \\sin \\frac{2 \\pi}{13}$. Suppose $a>b>c>d$ are positive integers satisfying $$\\left|\\zeta^{a}+\\zeta^{b}+\\zeta^{c}+\\zeta^{d}\\right|=\\sqrt{3}$$ Compute the smallest possible value of $1000 a+100 b+10 c+d$.",
      "answer": "7521"
    },
    {
      "question": "Call a set of positive integers good if there is a partition of it into two sets $S$ and $T$, such that there do not exist three elements $a, b, c \\in S$ such that $a^{b}=c$ and such that there do not exist three elements $a, b, c \\in T$ such that $a^{b}=c$ ( $a$ and $b$ need not be distinct). Find the smallest positive integer $n$ such that the set $\\{2,3,4, \\ldots, n\\}$ is not good.",
      "answer": "65536"
    },
    {
      "question": "Michael picks a random subset of the complex numbers \\(\\left\\{1, \\omega, \\omega^{2}, \\ldots, \\omega^{2017}\\right\\}\\) where \\(\\omega\\) is a primitive \\(2018^{\\text {th }}\\) root of unity and all subsets are equally likely to be chosen. If the sum of the elements in his subset is \\(S\\), what is the expected value of \\(|S|^{2}\\)? (The sum of the elements of the empty set is 0.)",
      "answer": "\\frac{1009}{2}"
    },
    {
      "question": "A triangle in the $x y$-plane is such that when projected onto the $x$-axis, $y$-axis, and the line $y=x$, the results are line segments whose endpoints are $(1,0)$ and $(5,0),(0,8)$ and $(0,13)$, and $(5,5)$ and $(7.5,7.5)$, respectively. What is the triangle's area?",
      "answer": "\\frac{17}{2}"
    },
    {
      "question": "Bob knows that Alice has 2021 secret positive integers $x_{1}, \\ldots, x_{2021}$ that are pairwise relatively prime. Bob would like to figure out Alice's integers. He is allowed to choose a set $S \\subseteq\\{1,2, \\ldots, 2021\\}$ and ask her for the product of $x_{i}$ over $i \\in S$. Alice must answer each of Bob's queries truthfully, and Bob may use Alice's previous answers to decide his next query. Compute the minimum number of queries Bob needs to guarantee that he can figure out each of Alice's integers.",
      "answer": "11"
    },
    {
      "question": "Each of $a, b$ and $c$ is equal to a number from the list $3^{1}, 3^{2}, 3^{3}, 3^{4}, 3^{5}, 3^{6}, 3^{7}, 3^{8}$. There are $N$ triples $(a, b, c)$ with $a \\leq b \\leq c$ for which each of $\\frac{ab}{c}, \\frac{ac}{b}$ and $\\frac{bc}{a}$ is equal to an integer. What is the value of $N$?",
      "answer": "86"
    },
    {
      "question": "Suppose $A B C$ is a triangle with incircle $\\omega$, and $\\omega$ is tangent to $\\overline{B C}$ and $\\overline{C A}$ at $D$ and $E$ respectively. The bisectors of $\\angle A$ and $\\angle B$ intersect line $D E$ at $F$ and $G$ respectively, such that $B F=1$ and $F G=G A=6$. Compute the radius of $\\omega$.",
      "answer": "\\frac{2 \\sqrt{5}}{5}"
    },
    {
      "question": "For some positive real $\\alpha$, the set $S$ of positive real numbers $x$ with $\\{x\\}>\\alpha x$ consists of the union of several intervals, with total length 20.2. The value of $\\alpha$ can be expressed as $\\frac{a}{b}$, where $a, b$ are relatively prime positive integers. Compute $100a+b$. (Here, $\\{x\\}=x-\\lfloor x\\rfloor$ is the fractional part of $x$.)",
      "answer": "4633"
    },
    {
      "question": "There exist unique nonnegative integers $A, B$ between 0 and 9, inclusive, such that $(1001 \\cdot A+110 \\cdot B)^{2}=57,108,249$. Find $10 \\cdot A+B$.",
      "answer": "75"
    },
    {
      "question": "What is the minimum value of the product $\\prod_{i=1}^{6} \\frac{a_{i}-a_{i+1}}{a_{i+2}-a_{i+3}}$ given that $\\left(a_{1}, a_{2}, a_{3}, a_{4}, a_{5}, a_{6}\\right)$ is a permutation of $(1,2,3,4,5,6)$?",
      "answer": "1"
    },
    {
      "question": "Carina is in a tournament in which no game can end in a tie. She continues to play games until she loses 2 games, at which point she is eliminated and plays no more games. The probability of Carina winning the first game is $\frac{1}{2}$. After she wins a game, the probability of Carina winning the next game is $\frac{3}{4}$. After she loses a game, the probability of Carina winning the next game is $\frac{1}{3}$. What is the probability that Carina wins 3 games before being eliminated from the tournament?",
      "answer": "23"
    },
    {
      "question": "A point $P$ lies at the center of square $A B C D$. A sequence of points $\\left\\{P_{n}\\right\\}$ is determined by $P_{0}=P$, and given point $P_{i}$, point $P_{i+1}$ is obtained by reflecting $P_{i}$ over one of the four lines $A B, B C, C D, D A$, chosen uniformly at random and independently for each $i$. What is the probability that $P_{8}=P$ ?",
      "answer": "\\frac{1225}{16384}"
    },
    {
      "question": "Find the sum of the coefficients of the polynomial $P(x)=x^{4}-29 x^{3}+a x^{2}+b x+c$, given that $P(5)=11, P(11)=17$, and $P(17)=23$.",
      "answer": "-3193"
    },
    {
      "question": "The sequence $\\{a_n\\}_{n\\geq 0}$ of real numbers satisfies the relation:\n\\[ a_{m+n} + a_{m-n} - m + n -1 = \\frac12 (a_{2m} + a_{2n})  \\]\nfor all non-negative integers $m$ and $n$, $m \\ge n$. If $a_1 = 3$ find $a_{2004}$.",
      "answer": "4018021"
    },
    {
      "question": "2019 points are chosen independently and uniformly at random on the interval $[0,1]$. Tairitsu picks 1000 of them randomly and colors them black, leaving the remaining ones white. Hikari then computes the sum of the positions of the leftmost white point and the rightmost black point. What is the probability that this sum is at most 1 ?",
      "answer": "\\frac{1019}{2019}"
    },
    {
      "question": "A regular octahedron $A B C D E F$ is given such that $A D, B E$, and $C F$ are perpendicular. Let $G, H$, and $I$ lie on edges $A B, B C$, and $C A$ respectively such that \\frac{A G}{G B}=\\frac{B H}{H C}=\\frac{C I}{I A}=\\rho. For some choice of $\\rho>1, G H, H I$, and $I G$ are three edges of a regular icosahedron, eight of whose faces are inscribed in the faces of $A B C D E F$. Find $\\rho$.",
      "answer": "(1+\\sqrt{5}) / 2"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=13, B C=14, C A=15$. Company XYZ wants to locate their base at the point $P$ in the plane minimizing the total distance to their workers, who are located at vertices $A, B$, and $C$. There are 1,5 , and 4 workers at $A, B$, and $C$, respectively. Find the minimum possible total distance Company XYZ's workers have to travel to get to $P$.",
      "answer": "69"
    },
    {
      "question": "A cylinder with radius 15 and height 16 is inscribed in a sphere. Three congruent smaller spheres of radius $x$ are externally tangent to the base of the cylinder, externally tangent to each other, and internally tangent to the large sphere. What is the value of $x$?",
      "answer": "\\frac{15 \\sqrt{37}-75}{4}"
    },
    {
      "question": "Let $\\otimes$ be a binary operation that takes two positive real numbers and returns a positive real number. Suppose further that $\\otimes$ is continuous, commutative $(a \\otimes b=b \\otimes a)$, distributive across multiplication $(a \\otimes(b c)=(a \\otimes b)(a \\otimes c))$, and that $2 \\otimes 2=4$. Solve the equation $x \\otimes y=x$ for $y$ in terms of $x$ for $x>1$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "A moth starts at vertex $A$ of a certain cube and is trying to get to vertex $B$, which is opposite $A$, in five or fewer \"steps,\" where a step consists in traveling along an edge from one vertex to another. The moth will stop as soon as it reaches $B$. How many ways can the moth achieve its objective?",
      "answer": "48"
    },
    {
      "question": "The first four terms of a sequence are $1,4,2$, and 3. Beginning with the fifth term in the sequence, each term is the sum of the previous four terms. What is the eighth term?",
      "answer": "66"
    },
    {
      "question": "Suppose that $m$ and $n$ are positive integers with $m<n$ such that the interval $[m, n)$ contains more multiples of 2021 than multiples of 2000. Compute the maximum possible value of $n-m$.",
      "answer": "191999"
    },
    {
      "question": "Petya bought one cake, two cupcakes and three bagels, Apya bought three cakes and a bagel, and Kolya bought six cupcakes. They all paid the same amount of money for purchases. Lena bought two cakes and two bagels. And how many cupcakes could be bought for the same amount spent to her?",
      "answer": "$\\frac{13}{4}$"
    },
    {
      "question": "Meg started with the number 100. She increased this number by $20\\%$ and then increased the resulting number by $50\\%$. What was her final result?",
      "answer": "180"
    },
    {
      "question": "Wendy eats sushi for lunch. She wants to eat six pieces of sushi arranged in a $2 \\times 3$ rectangular grid, but sushi is sticky, and Wendy can only eat a piece if it is adjacent to (not counting diagonally) at most two other pieces. In how many orders can Wendy eat the six pieces of sushi, assuming that the pieces of sushi are distinguishable?",
      "answer": "360"
    },
    {
      "question": "The rational numbers $x$ and $y$, when written in lowest terms, have denominators 60 and 70 , respectively. What is the smallest possible denominator of $x+y$ ?",
      "answer": "84"
    },
    {
      "question": "A graph $G(V,E)$ is triangle-free, but adding any edges to the graph will form a triangle. It's given that $|V|=2019$, $|E|>2018$, find the minimum of $|E|$ .",
      "answer": "4033"
    },
    {
      "question": "In a single-elimination tournament consisting of $2^{9}=512$ teams, there is a strict ordering on the skill levels of the teams, but Joy does not know that ordering. The teams are randomly put into a bracket and they play out the tournament, with the better team always beating the worse team. Joy is then given the results of all 511 matches and must create a list of teams such that she can guarantee that the third-best team is on the list. What is the minimum possible length of Joy's list?",
      "answer": "45"
    },
    {
      "question": "Point P_{1} is located 600 miles West of point P_{2}. At 7:00 AM a car departs from P_{1} and drives East at a speed of 50 miles per hour. At 8:00 AM another car departs from P_{2} and drives West at a constant speed of x miles per hour. If the cars meet each other exactly halfway between P_{1} and P_{2}, what is the value of x?",
      "answer": "60"
    },
    {
      "question": "Circles $C_{1}, C_{2}, C_{3}$ have radius 1 and centers $O, P, Q$ respectively. $C_{1}$ and $C_{2}$ intersect at $A, C_{2}$ and $C_{3}$ intersect at $B, C_{3}$ and $C_{1}$ intersect at $C$, in such a way that $\\angle A P B=60^{\\circ}, \\angle B Q C=36^{\\circ}$, and $\\angle C O A=72^{\\circ}$. Find angle $A B C$ (degrees).",
      "answer": "90"
    },
    {
      "question": "For any positive integer $n, S_{n}$ be the set of all permutations of \\{1,2,3, \\ldots, n\\}. For each permutation $\\pi \\in S_{n}$, let $f(\\pi)$ be the number of ordered pairs $(j, k)$ for which $\\pi(j)>\\pi(k)$ and $1 \\leq j<k \\leq n$. Further define $g(\\pi)$ to be the number of positive integers $k \\leq n$ such that $\\pi(k) \\equiv k \\pm 1(\\bmod n)$. Compute $$\\sum_{\\pi \\in S_{999}}(-1)^{f(\\pi)+g(\\pi)}$$",
      "answer": "995 \\times 2^{998}"
    },
    {
      "question": "Six standard fair six-sided dice are rolled and arranged in a row at random. Compute the expected number of dice showing the same number as the sixth die in the row.",
      "answer": "\\frac{11}{6}"
    },
    {
      "question": "If $3n=9+9+9$, what is the value of $n$?",
      "answer": "9"
    },
    {
      "question": "It takes Pearl 7 days to dig 4 holes. It takes Miguel 3 days to dig 2 holes. If they work together and each continues digging at these same rates, how many holes in total will they dig in 21 days?",
      "answer": "26"
    },
    {
      "question": "If $\\frac{1}{3}$ of $x$ is equal to 4, what is $\\frac{1}{6}$ of $x$?",
      "answer": "2"
    },
    {
      "question": "Kelvin the Frog and 10 of his relatives are at a party. Every pair of frogs is either friendly or unfriendly. When 3 pairwise friendly frogs meet up, they will gossip about one another and end up in a fight (but stay friendly anyway). When 3 pairwise unfriendly frogs meet up, they will also end up in a fight. In all other cases, common ground is found and there is no fight. If all $\\binom{11}{3}$ triples of frogs meet up exactly once, what is the minimum possible number of fights?",
      "answer": "28"
    },
    {
      "question": "Three ants begin on three different vertices of a tetrahedron. Every second, they choose one of the three edges connecting to the vertex they are on with equal probability and travel to the other vertex on that edge. They all stop when any two ants reach the same vertex at the same time. What is the probability that all three ants are at the same vertex when they stop?",
      "answer": "\\frac{1}{16}"
    },
    {
      "question": "Neo has an infinite supply of red pills and blue pills. When he takes a red pill, his weight will double, and when he takes a blue pill, he will lose one pound. If Neo originally weighs one pound, what is the minimum number of pills he must take to make his weight 2015 pounds?",
      "answer": "13"
    },
    {
      "question": "Say that a polynomial with real coefficients in two variables, $x,y$, is \\emph{balanced} if\nthe average value of the polynomial on each circle centered at the origin is $0$.\nThe balanced polynomials of degree at most $2009$ form a vector space $V$ over $\\mathbb{R}$.\nFind the dimension of $V$.",
      "answer": "2020050"
    },
    {
      "question": "If \\( a=\\frac{2}{3}b \\) and \\( b \\neq 0 \\), what is \\( \\frac{9a+8b}{6a} \\) equal to?",
      "answer": "\\frac{7}{2}"
    },
    {
      "question": "In how many ways can the numbers $1,2, \\ldots, 2002$ be placed at the vertices of a regular 2002-gon so that no two adjacent numbers differ by more than 2? (Rotations and reflections are considered distinct.)",
      "answer": "4004"
    },
    {
      "question": "Determine the largest of all integers $n$ with the property that $n$ is divisible by all positive integers that are less than $\\sqrt[3]{n}$.",
      "answer": "420"
    },
    {
      "question": "In her last basketball game, Jackie scored 36 points. These points raised the average number of points that she scored per game from 20 to 21. To raise this average to 22 points, how many points must Jackie score in her next game?",
      "answer": "38"
    },
    {
      "question": "How many triples \\((a, b, c)\\) of positive integers satisfy the conditions \\( 6ab = c^2 \\) and \\( a < b < c \\leq 35 \\)?",
      "answer": "8"
    },
    {
      "question": "How many integers are greater than $\frac{5}{7}$ and less than $\frac{28}{3}$?",
      "answer": "9"
    },
    {
      "question": "Carl only eats food in the shape of equilateral pentagons. Unfortunately, for dinner he receives a piece of steak in the shape of an equilateral triangle. So that he can eat it, he cuts off two corners with straight cuts to form an equilateral pentagon. The set of possible perimeters of the pentagon he obtains is exactly the interval $[a, b)$, where $a$ and $b$ are positive real numbers. Compute $\\frac{a}{b}$.",
      "answer": "4 \\sqrt{3}-6"
    },
    {
      "question": "Karim has 23 candies. He eats $n$ candies and divides the remaining candies equally among his three children so that each child gets an integer number of candies. Which of the following is not a possible value of $n$?",
      "answer": "9"
    },
    {
      "question": "Find the number of triangulations of a general convex 7-gon into 5 triangles by 4 diagonals that do not intersect in their interiors.",
      "answer": "42"
    },
    {
      "question": "Let $G$ be a simple graph with 100 vertices such that for each vertice $u$, there exists a vertice $v \\in N \\left ( u \\right )$ and $ N \\left ( u \\right ) \\cap  N \\left ( v \\right ) = \\o $. Try to find the maximal possible number of edges in $G$. The $ N \\left ( . \\right )$  refers to the neighborhood.",
      "answer": "3822"
    },
    {
      "question": "Let $k$ be the answer to this problem. The probability that an integer chosen uniformly at random from $\\{1,2, \\ldots, k\\}$ is a multiple of 11 can be written as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
      "answer": "1000"
    },
    {
      "question": "Compute the number of ordered pairs of integers $(a, b)$, with $2 \\leq a, b \\leq 2021$, that satisfy the equation $$a^{\\log _{b}\\left(a^{-4}\\right)}=b^{\\log _{a}\\left(b a^{-3}\\right)}.$$",
      "answer": "43"
    },
    {
      "question": "Let $L$ be the number formed by $2022$ digits equal to $1$, that is, $L=1111\\dots 111$.\nCompute the sum of the digits of the number $9L^2+2L$.",
      "answer": "4044"
    },
    {
      "question": "Let $A$ be a $n\\times n$ matrix such that $A_{ij} = i+j$. Find the rank of $A$.\n\n[hide=\"Remark\"]Not asked in the contest: $A$ is diagonalisable since real symetric matrix it is not difficult to find its eigenvalues.[/hide]",
      "answer": "2"
    },
    {
      "question": "What is the radius of the smallest sphere in which 4 spheres of radius 1 will fit?",
      "answer": "\\frac{2+\\sqrt{6}}{2}"
    },
    {
      "question": "You have a twig of length 1. You repeatedly do the following: select two points on the twig independently and uniformly at random, make cuts on these two points, and keep only the largest piece. After 2012 repetitions, what is the expected length of the remaining piece?",
      "answer": "\\left(\\frac{11}{18}\\right)^{2012}"
    },
    {
      "question": "Let $x_{1}, x_{2}, \\ldots, x_{2022}$ be nonzero real numbers. Suppose that $x_{k}+\\frac{1}{x_{k+1}}<0$ for each $1 \\leq k \\leq 2022$, where $x_{2023}=x_{1}$. Compute the maximum possible number of integers $1 \\leq n \\leq 2022$ such that $x_{n}>0$.",
      "answer": "1010"
    },
    {
      "question": "A point $P$ is chosen uniformly at random inside a square of side length 2. If $P_{1}, P_{2}, P_{3}$, and $P_{4}$ are the reflections of $P$ over each of the four sides of the square, find the expected value of the area of quadrilateral $P_{1} P_{2} P_{3} P_{4}$.",
      "answer": "8"
    },
    {
      "question": "What is the perimeter of the triangle formed by the points of tangency of the incircle of a 5-7-8 triangle with its sides?",
      "answer": "\\frac{9 \\sqrt{21}}{7}+3"
    },
    {
      "question": "For which value of \\( x \\) is \\( x^3 < x^2 \\)?",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "Kelvin the Frog is trying to hop across a river. The river has 10 lilypads on it, and he must hop on them in a specific order (the order is unknown to Kelvin). If Kelvin hops to the wrong lilypad at any point, he will be thrown back to the wrong side of the river and will have to start over. Assuming Kelvin is infinitely intelligent, what is the minimum number of hops he will need to guarantee reaching the other side?",
      "answer": "176"
    },
    {
      "question": "In right triangle $A B C$, a point $D$ is on hypotenuse $A C$ such that $B D \\perp A C$. Let $\\omega$ be a circle with center $O$, passing through $C$ and $D$ and tangent to line $A B$ at a point other than $B$. Point $X$ is chosen on $B C$ such that $A X \\perp B O$. If $A B=2$ and $B C=5$, then $B X$ can be expressed as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
      "answer": "8041"
    },
    {
      "question": "(Caos) A cao [sic] has 6 legs, 3 on each side. A walking pattern for the cao is defined as an ordered sequence of raising and lowering each of the legs exactly once (altogether 12 actions), starting and ending with all legs on the ground. The pattern is safe if at any point, he has at least 3 legs on the ground and not all three legs are on the same side. Estimate $N$, the number of safe patterns.",
      "answer": "1416528"
    },
    {
      "question": "A $10 \\times 10$ table consists of 100 unit cells. A block is a $2 \\times 2$ square consisting of 4 unit cells of the table. A set $C$ of $n$ blocks covers the table (i.e. each cell of the table is covered by some block of $C$ ) but no $n-1$ blocks of $C$ cover the table. Find the largest possible value of n.",
      "answer": "39"
    },
    {
      "question": "Let $A B C D$ be a quadrilateral inscribed in a circle with diameter $\\overline{A D}$. If $A B=5, A C=6$, and $B D=7$, find $C D$.",
      "answer": "\\sqrt{38}"
    },
    {
      "question": "$P$ is a point inside triangle $A B C$, and lines $A P, B P, C P$ intersect the opposite sides $B C, C A, A B$ in points $D, E, F$, respectively. It is given that $\\angle A P B=90^{\\circ}$, and that $A C=B C$ and $A B=B D$. We also know that $B F=1$, and that $B C=999$. Find $A F$.",
      "answer": "499 / 500"
    },
    {
      "question": "In three-dimensional space, let $S$ be the region of points $(x, y, z)$ satisfying $-1 \\leq z \\leq 1$. Let $S_{1}, S_{2}, \\ldots, S_{2022}$ be 2022 independent random rotations of $S$ about the origin ( $0,0,0$). The expected volume of the region $S_{1} \\cap S_{2} \\cap \\cdots \\cap S_{2022}$ can be expressed as $\\frac{a \\pi}{b}$, for relatively prime positive integers $a$ and $b$. Compute $100 a+b$.",
      "answer": "271619"
    },
    {
      "question": "Farmer James invents a new currency, such that for every positive integer $n \\leq 6$, there exists an $n$-coin worth $n$ ! cents. Furthermore, he has exactly $n$ copies of each $n$-coin. An integer $k$ is said to be nice if Farmer James can make $k$ cents using at least one copy of each type of coin. How many positive integers less than 2018 are nice?",
      "answer": "210"
    },
    {
      "question": "For a positive integer $n$, and a non empty subset $A$ of $\\{1,2,...,2n\\}$, call $A$ good if the set $\\{u\\pm v|u,v\\in A\\}$ does not contain the set $\\{1,2,...,n\\}$. Find the smallest real number $c$, such that for any positive integer $n$, and any good subset $A$ of  $\\{1,2,...,2n\\}$, $|A|\\leq cn$.",
      "answer": "\\frac{6}{5}"
    },
    {
      "question": "Four points, $A, B, C$, and $D$, are chosen randomly on the circumference of a circle with independent uniform probability. What is the expected number of sides of triangle $A B C$ for which the projection of $D$ onto the line containing the side lies between the two vertices?",
      "answer": "3/2"
    },
    {
      "question": "Three points are chosen inside a unit cube uniformly and independently at random. What is the probability that there exists a cube with side length $\\frac{1}{2}$ and edges parallel to those of the unit cube that contains all three points?",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "Let $p$ be a real number between 0 and 1. Jocelin has a coin that lands heads with probability $p$ and tails with probability $1-p$; she also has a number written on a blackboard. Each minute, she flips the coin, and if it lands heads, she replaces the number $x$ on the blackboard with $3 x+1$; if it lands tails she replaces it with $x / 2$. Given that there are constants $a, b$ such that the expected value of the value written on the blackboard after $t$ minutes can be written as $a t+b$ for all positive integers $t$, compute $p$.",
      "answer": "\\frac{1}{5}"
    },
    {
      "question": "What is the value of $n$ if $2^{n}=8^{20}$?",
      "answer": "60"
    },
    {
      "question": "Given right triangle $ABC$, with $AB=4, BC=3$, and $CA=5$. Circle $\\omega$ passes through $A$ and is tangent to $BC$ at $C$. What is the radius of $\\omega$?",
      "answer": "\\frac{25}{8}"
    },
    {
      "question": "Let $A_{12}$ denote the answer to problem 12. There exists a unique triple of digits $(B, C, D)$ such that $10>A_{12}>B>C>D>0$ and $$\\overline{A_{12} B C D}-\\overline{D C B A_{12}}=\\overline{B D A_{12} C}$$ where $\\overline{A_{12} B C D}$ denotes the four digit base 10 integer. Compute $B+C+D$.",
      "answer": "11"
    },
    {
      "question": "Let $a_{1}, a_{2}, \\ldots, a_{2005}$ be real numbers such that $$\\begin{array}{ccccccccccc} a_{1} \\cdot 1 & + & a_{2} \\cdot 2 & + & a_{3} \\cdot 3 & + & \\cdots & + & a_{2005} \\cdot 2005 & = & 0 \\\\ a_{1} \\cdot 1^{2} & + & a_{2} \\cdot 2^{2} & + & a_{3} \\cdot 3^{2} & + & \\cdots & + & a_{2005} \\cdot 2005^{2} & = & 0 \\\\ a_{1} \\cdot 1^{3} & + & a_{2} \\cdot 2^{3} & + & a_{3} \\cdot 3^{3} & + & \\cdots & + & a_{2005} \\cdot 2005^{3} & = & 0 \\\\ \\vdots & & \\vdots & & \\vdots & & & & \\vdots & & \\vdots \\\\ a_{1} \\cdot 1^{2004} & + & a_{2} \\cdot 2^{2004} & + & a_{3} \\cdot 3^{2004} & + & \\cdots & + & a_{2005} \\cdot 2005^{2004} & = & 0 \\end{array}$$ and $$a_{1} \\cdot 1^{2005}+a_{2} \\cdot 2^{2005}+a_{3} \\cdot 3^{2005}+\\cdots+a_{2005} \\cdot 2005^{2005}=1$$ What is the value of $a_{1}$?",
      "answer": "1 / 2004!"
    },
    {
      "question": "If $2 x^{2}=9 x-4$ and $x \neq 4$, what is the value of $2 x$?",
      "answer": "1"
    },
    {
      "question": "We wish to color the integers $1,2,3, \\ldots, 10$ in red, green, and blue, so that no two numbers $a$ and $b$, with $a-b$ odd, have the same color. (We do not require that all three colors be used.) In how many ways can this be done?",
      "answer": "186"
    },
    {
      "question": "Consider a three-person game involving the following three types of fair six-sided dice. - Dice of type $A$ have faces labelled $2,2,4,4,9,9$. - Dice of type $B$ have faces labelled $1,1,6,6,8,8$. - Dice of type $C$ have faces labelled $3,3,5,5,7,7$. All three players simultaneously choose a die (more than one person can choose the same type of die, and the players don't know one another's choices) and roll it. Then the score of a player $P$ is the number of players whose roll is less than $P$ 's roll (and hence is either 0,1 , or 2 ). Assuming all three players play optimally, what is the expected score of a particular player?",
      "answer": "\\frac{8}{9}"
    },
    {
      "question": "How many ways are there to place four points in the plane such that the set of pairwise distances between the points consists of exactly 2 elements? (Two configurations are the same if one can be obtained from the other via rotation and scaling.)",
      "answer": "6"
    },
    {
      "question": "Find the sum of the infinite series $\\frac{1}{3^{2}-1^{2}}\\left(\\frac{1}{1^{2}}-\\frac{1}{3^{2}}\\right)+\\frac{1}{5^{2}-3^{2}}\\left(\\frac{1}{3^{2}}-\\frac{1}{5^{2}}\\right)+\\frac{1}{7^{2}-5^{2}}\\left(\\frac{1}{5^{2}}-\\frac{1}{7^{2}}\\right)+$",
      "answer": "1"
    },
    {
      "question": "Wendy is playing darts with a circular dartboard of radius 20. Whenever she throws a dart, it lands uniformly at random on the dartboard. At the start of her game, there are 2020 darts placed randomly on the board. Every turn, she takes the dart farthest from the center, and throws it at the board again. What is the expected number of darts she has to throw before all the darts are within 10 units of the center?",
      "answer": "6060"
    },
    {
      "question": "In the $3 \times 3$ grid shown, the central square contains the integer 5. The remaining eight squares contain $a, b, c, d, e, f, g, h$, which are each to be replaced with an integer from 1 to 9, inclusive. Integers can be repeated. There are $N$ ways to complete the grid so that the sums of the integers along each row, along each column, and along the two main diagonals are all divisible by 5. What are the rightmost two digits of $N$?",
      "answer": "73"
    },
    {
      "question": "Suppose there exists a convex $n$-gon such that each of its angle measures, in degrees, is an odd prime number. Compute the difference between the largest and smallest possible values of $n$.",
      "answer": "356"
    },
    {
      "question": "20 players are playing in a Super Smash Bros. Melee tournament. They are ranked $1-20$, and player $n$ will always beat player $m$ if $n<m$. Out of all possible tournaments where each player plays 18 distinct other players exactly once, one is chosen uniformly at random. Find the expected number of pairs of players that win the same number of games.",
      "answer": "4"
    },
    {
      "question": "Let $W, S$ be as in problem 32. Let $A$ be the least positive integer such that an acute triangle with side lengths $S, A$, and $W$ exists. Find $A$.",
      "answer": "7"
    },
    {
      "question": "Consider a sequence $\\{a_n\\}$ of integers, satisfying $a_1=1, a_2=2$ and $a_{n+1}$ is the largest prime divisor of $a_1+a_2+\\ldots+a_n$. Find $a_{100}$.",
      "answer": "53"
    },
    {
      "question": "Which of the following fractions has the greatest value: $\\frac{3}{10}$, $\\frac{4}{7}$, $\\frac{5}{23}$, $\\frac{2}{3}$, $\\frac{1}{2}$?",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "How many different-looking arrangements are possible when four balls are selected at random from six identical red balls and three identical green balls and then arranged in a line?",
      "answer": "15"
    },
    {
      "question": "Compute the number of triples $(f, g, h)$ of permutations on $\\{1,2,3,4,5\\}$ such that $$ \\begin{aligned} & f(g(h(x)))=h(g(f(x)))=g(x), \\\\ & g(h(f(x)))=f(h(g(x)))=h(x), \\text { and } \\\\ & h(f(g(x)))=g(f(h(x)))=f(x) \\end{aligned} $$ for all $x \\in\\{1,2,3,4,5\\}$.",
      "answer": "146"
    },
    {
      "question": "Let $a=256$. Find the unique real number $x>a^{2}$ such that $$\\log _{a} \\log _{a} \\log _{a} x=\\log _{a^{2}} \\log _{a^{2}} \\log _{a^{2}} x$$",
      "answer": "2^{32}"
    },
    {
      "question": "Alice is bored in class, so she thinks of a positive integer. Every second after that, she subtracts from her current number its smallest prime divisor, possibly itself. After 2022 seconds, she realizes that her number is prime. Find the sum of all possible values of her initial number.",
      "answer": "8093"
    },
    {
      "question": "How many polynomials of degree exactly 5 with real coefficients send the set \\{1,2,3,4,5,6\\} to a permutation of itself?",
      "answer": "714"
    },
    {
      "question": "If \\( 50\\% \\) of \\( N \\) is 16, what is \\( 75\\% \\) of \\( N \\)?",
      "answer": "24"
    },
    {
      "question": "If $3^{x}=5$, what is the value of $3^{x+2}$?",
      "answer": "45"
    },
    {
      "question": "There are 100 positive integers written on a board. At each step, Alex composes 50 fractions using each number written on the board exactly once, brings these fractions to their irreducible form, and then replaces the 100 numbers on the board with the new numerators and denominators to create 100 new numbers. \n\nFind the smallest positive integer $n{}$ such that regardless of the values of the initial 100 numbers, after $n{}$ steps Alex can arrange to have on the board only pairwise coprime numbers.",
      "answer": "99"
    },
    {
      "question": "Consider positive integers $a \\leq b \\leq c \\leq d \\leq e$. There are $N$ lists $a, b, c, d, e$ with a mean of 2023 and a median of 2023, in which the integer 2023 appears more than once, and in which no other integer appears more than once. What is the sum of the digits of $N$?",
      "answer": "28"
    },
    {
      "question": "Let $a_{0}=-2, b_{0}=1$, and for $n \\geq 0$, let $$\\begin{aligned} & a_{n+1}=a_{n}+b_{n}+\\sqrt{a_{n}^{2}+b_{n}^{2}} \\\\ & b_{n+1}=a_{n}+b_{n}-\\sqrt{a_{n}^{2}+b_{n}^{2}} \\end{aligned}$$ Find $a_{2012}$.",
      "answer": "2^{1006} \\sqrt{2^{2010}+2}-2^{2011}"
    },
    {
      "question": "$101$ people,  sitting at a round table in any order,  had  $1,2,... , 101$ cards,  respectively.  \nA transfer is someone give  one  card  to one of  the two people  adjacent to him.\nFind the  smallest positive integer $k$ such that   there always  can  through  no more than  $ k $  times transfer,  each person  hold cards of the same number, regardless of  the sitting order.",
      "answer": "42925"
    },
    {
      "question": "The numbers $4x, 2x-3, 4x-3$ are three consecutive terms in an arithmetic sequence. What is the value of $x$?",
      "answer": "-\\frac{3}{4}"
    },
    {
      "question": "Find the smallest real constant $\\alpha$ such that for all positive integers $n$ and real numbers $0=y_{0}<$ $y_{1}<\\cdots<y_{n}$, the following inequality holds: $\\alpha \\sum_{k=1}^{n} \\frac{(k+1)^{3 / 2}}{\\sqrt{y_{k}^{2}-y_{k-1}^{2}}} \\geq \\sum_{k=1}^{n} \\frac{k^{2}+3 k+3}{y_{k}}$.",
      "answer": "\\frac{16 \\sqrt{2}}{9}"
    },
    {
      "question": "Compute $\\frac{\\tan ^{2}\\left(20^{\\circ}\\right)-\\sin ^{2}\\left(20^{\\circ}\\right)}{\\tan ^{2}\\left(20^{\\circ}\\right) \\sin ^{2}\\left(20^{\\circ}\\right)}$.",
      "answer": "1"
    },
    {
      "question": "Find the value of $$\\sum_{1 \\leq a<b<c} \\frac{1}{2^{a} 3^{b} 5^{c}}$$ (i.e. the sum of $\\frac{1}{2^{a} 3^{b} 5^{c}}$ over all triples of positive integers $(a, b, c)$ satisfying $a<b<c$).",
      "answer": "\\frac{1}{1624}"
    },
    {
      "question": "How many four-digit numbers are there in which at least one digit occurs more than once?",
      "answer": "4464"
    },
    {
      "question": "Circle $\\Omega$ has radius 5. Points $A$ and $B$ lie on $\\Omega$ such that chord $A B$ has length 6. A unit circle $\\omega$ is tangent to chord $A B$ at point $T$. Given that $\\omega$ is also internally tangent to $\\Omega$, find $A T \\cdot B T$.",
      "answer": "2"
    },
    {
      "question": "In $\\triangle ABC, D$ and $E$ are the midpoints of $BC$ and $CA$, respectively. $AD$ and $BE$ intersect at $G$. Given that $GEC$D is cyclic, $AB=41$, and $AC=31$, compute $BC$.",
      "answer": "49"
    },
    {
      "question": "Consider a $2 \\times 2$ grid of squares. David writes a positive integer in each of the squares. Next to each row, he writes the product of the numbers in the row, and next to each column, he writes the product of the numbers in each column. If the sum of the eight numbers he writes down is 2015, what is the minimum possible sum of the four numbers he writes in the grid?",
      "answer": "88"
    },
    {
      "question": "$A B C D E$ is a cyclic convex pentagon, and $A C=B D=C E . A C$ and $B D$ intersect at $X$, and $B D$ and $C E$ intersect at $Y$. If $A X=6, X Y=4$, and $Y E=7$, then the area of pentagon $A B C D E$ can be written as $\\frac{a \\sqrt{b}}{c}$, where $a, b, c$ are integers, $c$ is positive, $b$ is square-free, and $\\operatorname{gcd}(a, c)=1$. Find $100 a+10 b+c$.",
      "answer": "2852"
    },
    {
      "question": "Let x; y; z be real numbers, satisfying the relations\n$x \\ge 20$\n$y \u0015\\ge 40$\n$z \u0015\\ge 1675$\nx + y + z = 2015\nFind the greatest value of the product P = $x\u0001y z$",
      "answer": "\\frac{721480000}{27}"
    },
    {
      "question": "A group of friends are sharing a bag of candy. On the first day, they eat $\frac{1}{2}$ of the candies in the bag. On the second day, they eat $\frac{2}{3}$ of the remaining candies. On the third day, they eat $\frac{3}{4}$ of the remaining candies. On the fourth day, they eat $\frac{4}{5}$ of the remaining candies. On the fifth day, they eat $\frac{5}{6}$ of the remaining candies. At the end of the fifth day, there is 1 candy remaining in the bag. How many candies were in the bag before the first day?",
      "answer": "720"
    },
    {
      "question": "Let $a, b$, and $c$ be the 3 roots of $x^{3}-x+1=0$. Find $\\frac{1}{a+1}+\\frac{1}{b+1}+\\frac{1}{c+1}$.",
      "answer": "-2"
    },
    {
      "question": "Let $N=30^{2015}$. Find the number of ordered 4-tuples of integers $(A, B, C, D) \\in\\{1,2, \\ldots, N\\}^{4}$ (not necessarily distinct) such that for every integer $n, A n^{3}+B n^{2}+2 C n+D$ is divisible by $N$.",
      "answer": "24"
    },
    {
      "question": "An 8 by 8 grid of numbers obeys the following pattern: 1) The first row and first column consist of all 1s. 2) The entry in the $i$th row and $j$th column equals the sum of the numbers in the $(i-1)$ by $(j-1)$ sub-grid with row less than $i$ and column less than $j$. What is the number in the 8th row and 8th column?",
      "answer": "2508"
    },
    {
      "question": "In a survey, 100 students were asked if they like lentils and were also asked if they like chickpeas. A total of 68 students like lentils. A total of 53 like chickpeas. A total of 6 like neither lentils nor chickpeas. How many of the 100 students like both lentils and chickpeas?",
      "answer": "27"
    },
    {
      "question": "What is the largest possible value for $n$ if the average of the two positive integers $m$ and $n$ is 5?",
      "answer": "9"
    },
    {
      "question": "We call an even positive integer $n$ [i]nice[/i] if the set $\\{1, 2, \\dots, n\\}$ can be partitioned into $\\frac{n}{2}$ two-element subsets, such that the sum of the elements in each subset is a power of $3$. For example, $6$ is nice, because the set $\\{1, 2, 3, 4, 5, 6\\}$ can be partitioned into subsets $\\{1, 2\\}$, $\\{3, 6\\}$, $\\{4, 5\\}$. Find the number of nice positive integers which are smaller than $3^{2022}$.",
      "answer": "2^{2022} - 1"
    },
    {
      "question": "Suppose $m>n>1$ are positive integers such that there exist $n$ complex numbers $x_{1}, x_{2}, \\ldots, x_{n}$ for which - $x_{1}^{k}+x_{2}^{k}+\\cdots+x_{n}^{k}=1$ for $k=1,2, \\ldots, n-1$ - $x_{1}^{n}+x_{2}^{n}+\\cdots+x_{n}^{n}=2$; and - $x_{1}^{m}+x_{2}^{m}+\\cdots+x_{n}^{m}=4$. Compute the smallest possible value of $m+n$.",
      "answer": "34"
    },
    {
      "question": "Let $A_{1}, A_{2}, A_{3}$ be three points in the plane, and for convenience, let $A_{4}=A_{1}, A_{5}=A_{2}$. For $n=1,2$, and 3, suppose that $B_{n}$ is the midpoint of $A_{n} A_{n+1}$, and suppose that $C_{n}$ is the midpoint of $A_{n} B_{n}$. Suppose that $A_{n} C_{n+1}$ and $B_{n} A_{n+2}$ meet at $D_{n}$, and that $A_{n} B_{n+1}$ and $C_{n} A_{n+2}$ meet at $E_{n}$. Calculate the ratio of the area of triangle $D_{1} D_{2} D_{3}$ to the area of triangle $E_{1} E_{2} E_{3}$.",
      "answer": "\\frac{25}{49}"
    },
    {
      "question": "Given any positive integer, we can write the integer in base 12 and add together the digits of its base 12 representation. We perform this operation on the number $7^{6^{5^{3^{2^{1}}}}}$ repeatedly until a single base 12 digit remains. Find this digit.",
      "answer": "4"
    },
    {
      "question": "At the Lacsap Hospital, Emily is a doctor and Robert is a nurse. Not including Emily, there are five doctors and three nurses at the hospital. Not including Robert, there are $d$ doctors and $n$ nurses at the hospital. What is the product of $d$ and $n$?",
      "answer": "12"
    },
    {
      "question": "There are $n$ students in the math club. When grouped in 4s, there is one incomplete group. When grouped in 3s, there are 3 more complete groups than with 4s, and one incomplete group. When grouped in 2s, there are 5 more complete groups than with 3s, and one incomplete group. What is the sum of the digits of $n^{2}-n$?",
      "answer": "12"
    },
    {
      "question": "Let $A$ be the area of the largest semicircle that can be inscribed in a quarter-circle of radius 1. Compute $\\frac{120 A}{\\pi}$.",
      "answer": "20"
    },
    {
      "question": "What is the sum of the positive divisors of 1184?",
      "answer": "2394"
    },
    {
      "question": "A cube has side length 1. Find the product of the lengths of the diagonals of this cube (a diagonal is a line between two vertices that is not an edge).",
      "answer": "576"
    },
    {
      "question": "How many ordered triples of positive integers $(a, b, c)$ are there for which $a^{4} b^{2} c=54000$ ?",
      "answer": "16"
    },
    {
      "question": "Kevin has four red marbles and eight blue marbles. He arranges these twelve marbles randomly, in a ring. Determine the probability that no two red marbles are adjacent.",
      "answer": "\\frac{7}{33}"
    },
    {
      "question": "Let $f(x)=x^{2}+2 x+1$. Let $g(x)=f(f(\\cdots f(x)))$, where there are $2009 f \\mathrm{~s}$ in the expression for $g(x)$. Then $g(x)$ can be written as $g(x)=x^{2^{2009}}+a_{2^{2009}-1} x^{2^{2009}-1}+\\cdots+a_{1} x+a_{0}$ where the $a_{i}$ are constants. Compute $a_{2^{2009}-1}$.",
      "answer": "2^{2009}"
    },
    {
      "question": "Let $A B C D$ be a unit square. A circle with radius $\\frac{32}{49}$ passes through point $D$ and is tangent to side $A B$ at point $E$. Then $D E=\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "807"
    },
    {
      "question": "Calculate the sum: $\\sum_{n=1}^{99} \\left(n^{3}+3n^{2}+3n\\right)$.",
      "answer": "25502400"
    },
    {
      "question": "A ball inside a rectangular container of width 7 and height 12 is launched from the lower-left vertex of the container. It first strikes the right side of the container after traveling a distance of $\\sqrt{53}$ (and strikes no other sides between its launch and its impact with the right side). How many times does the ball bounce before it returns to a vertex? (The final contact with a vertex does not count as a bounce.)",
      "answer": "5"
    },
    {
      "question": "Find $\\prod_{n=2}^{\\infty}\\left(1-\\frac{1}{n^{2}}\\right)$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Let $P$ be a polyhedron where every face is a regular polygon, and every edge has length 1. Each vertex of $P$ is incident to two regular hexagons and one square. Choose a vertex $V$ of the polyhedron. Find the volume of the set of all points contained in $P$ that are closer to $V$ than to any other vertex.",
      "answer": "\\frac{\\sqrt{2}}{3}"
    },
    {
      "question": "There are $2022$ equally spaced points on a circular track $\\gamma$ of circumference $2022$. The points are labeled $A_1, A_2, \\ldots, A_{2022}$ in some order, each label used once. Initially, Bunbun the Bunny begins at $A_1$. She hops along $\\gamma$ from $A_1$ to $A_2$, then from $A_2$ to $A_3$, until she reaches $A_{2022}$, after which she hops back to $A_1$. When hopping from $P$ to $Q$, she always hops along the shorter of the two arcs $\\widehat{PQ}$ of $\\gamma$; if $\\overline{PQ}$ is a diameter of $\\gamma$, she moves along either semicircle.\n\nDetermine the maximal possible sum of the lengths of the $2022$ arcs which Bunbun traveled, over all possible labellings of the $2022$ points.\n\n[i]Kevin Cong[/i]",
      "answer": "2042222"
    },
    {
      "question": "A number is called [i]Norwegian[/i] if it has three distinct positive divisors whose sum is equal to $2022$. Determine the smallest Norwegian number.\n(Note: The total number of positive divisors of a Norwegian number is allowed to be larger than $3$.)",
      "answer": "1344"
    },
    {
      "question": "A cuboctahedron is a polyhedron whose faces are squares and equilateral triangles such that two squares and two triangles alternate around each vertex. What is the volume of a cuboctahedron of side length 1?",
      "answer": "5 \\sqrt{2} / 3"
    },
    {
      "question": "Three distinct integers $a, b,$ and $c$ satisfy the following three conditions: $abc=17955$, $a, b,$ and $c$ form an arithmetic sequence in that order, and $(3a+b), (3b+c),$ and $(3c+a)$ form a geometric sequence in that order. What is the value of $a+b+c$?",
      "answer": "-63"
    },
    {
      "question": "Find the number of arrangements of 4 beads (2 red, 2 green, 2 blue) in a circle such that the two red beads are not adjacent.",
      "answer": "11"
    },
    {
      "question": "Your math friend Steven rolls five fair icosahedral dice (each of which is labelled $1,2, \\ldots, 20$ on its sides). He conceals the results but tells you that at least half of the rolls are 20. Assuming that Steven is truthful, what is the probability that all three remaining concealed dice show $20 ?$",
      "answer": "\\frac{1}{58}"
    },
    {
      "question": "In the octagon COMPUTER exhibited below, all interior angles are either $90^{\\circ}$ or $270^{\\circ}$ and we have $C O=O M=M P=P U=U T=T E=1$. Point $D$ (not to scale in the diagram) is selected on segment $R E$ so that polygons COMPUTED and $C D R$ have the same area. Find $D R$.",
      "answer": "2"
    },
    {
      "question": "Let $N=2^{(2^{2})}$ and $x$ be a real number such that $N^{(N^{N})}=2^{(2^{x})}$. Find $x$.",
      "answer": "66"
    },
    {
      "question": "How many of the four integers $222, 2222, 22222$, and $222222$ are multiples of 3?",
      "answer": "2"
    },
    {
      "question": "Let $m$ and $n$ be positive integers with $m\\le 2000$ and $k=3-\\frac{m}{n}$. Find the smallest positive value of $k$.",
      "answer": "$\\boxed{ \\frac{1}{667}} .$"
    },
    {
      "question": "Square \\(ABCD\\) is inscribed in circle \\(\\omega\\) with radius 10. Four additional squares are drawn inside \\(\\omega\\) but outside \\(ABCD\\) such that the lengths of their diagonals are as large as possible. A sixth square is drawn by connecting the centers of the four aforementioned small squares. Find the area of the sixth square.",
      "answer": "144"
    },
    {
      "question": "Suppose $a, b$, and $c$ are distinct positive integers such that $\\sqrt{a \\sqrt{b \\sqrt{c}}}$ is an integer. Compute the least possible value of $a+b+c$.",
      "answer": "7"
    },
    {
      "question": "What is the probability that Robbie will win if he and Francine each roll a special six-sided die three times, and after two rolls each, Robbie has a score of 8 and Francine has a score of 10?",
      "answer": "\\frac{55}{441}"
    },
    {
      "question": "How many points $(x, y)$, with $x$ and $y$ both integers, are on the line with equation $y=4x+3$ and inside the region bounded by $x=25, x=75, y=120$, and $y=250$?",
      "answer": "32"
    },
    {
      "question": "Andrea flips a fair coin repeatedly, continuing until she either flips two heads in a row (the sequence $H H$ ) or flips tails followed by heads (the sequence $T H$ ). What is the probability that she will stop after flipping $H H$ ?",
      "answer": "1/4"
    },
    {
      "question": "Compute the sum of all integers $1 \\leq a \\leq 10$ with the following property: there exist integers $p$ and $q$ such that $p, q, p^{2}+a$ and $q^{2}+a$ are all distinct prime numbers.",
      "answer": "20"
    },
    {
      "question": "Let $a_0 = 5/2$ and $a_k = a_{k-1}^2 - 2$ for $k \\geq 1$. Compute \\[ \\prod_{k=0}^\\infty \\left(1 - \\frac{1}{a_k} \\right) \\] in closed form.",
      "answer": "\\frac{3}{7}"
    },
    {
      "question": "Which of the following divisions is not equal to a whole number: $\\frac{60}{12}$, $\\frac{60}{8}$, $\\frac{60}{5}$, $\\frac{60}{4}$, $\\frac{60}{3}$?",
      "answer": "7.5"
    },
    {
      "question": "Let $n$ be a positive integer. Let there be $P_{n}$ ways for Pretty Penny to make exactly $n$ dollars out of quarters, dimes, nickels, and pennies. Also, let there be $B_{n}$ ways for Beautiful Bill to make exactly $n$ dollars out of one dollar bills, quarters, dimes, and nickels. As $n$ goes to infinity, the sequence of fractions \\frac{P_{n}}{B_{n}}$ approaches a real number $c$. Find $c$.",
      "answer": "20"
    },
    {
      "question": "The product \\( \\left(1-\\frac{1}{3}\\right)\\left(1-\\frac{1}{4}\\right)\\left(1-\\frac{1}{5}\\right) \\) is equal to what?",
      "answer": "\\frac{2}{5}"
    },
    {
      "question": "Charlie folds an $\\frac{17}{2}$-inch by 11-inch piece of paper in half twice, each time along a straight line parallel to one of the paper's edges. What is the smallest possible perimeter of the piece after two such folds?",
      "answer": "$\\frac{39}{2}$"
    },
    {
      "question": "12 points are placed around the circumference of a circle. How many ways are there to draw 6 non-intersecting chords joining these points in pairs?",
      "answer": "132"
    },
    {
      "question": "James is standing at the point $(0,1)$ on the coordinate plane and wants to eat a hamburger. For each integer $n \\geq 0$, the point $(n, 0)$ has a hamburger with $n$ patties. There is also a wall at $y=2.1$ which James cannot cross. In each move, James can go either up, right, or down 1 unit as long as he does not cross the wall or visit a point he has already visited. Every second, James chooses a valid move uniformly at random, until he reaches a point with a hamburger. Then he eats the hamburger and stops moving. Find the expected number of patties that James eats on his burger.",
      "answer": "\\frac{7}{3}"
    },
    {
      "question": "A permutation of the set of positive integers $[n] = \\{1, 2, \\ldots, n\\}$ is a sequence $(a_1, a_2, \\ldots, a_n)$ such that each element of $[n]$ appears precisely one time as a term of the sequence. For example, $(3, 5, 1, 2, 4)$ is a permutation of $[5]$ . Let $P(n)$ be the number of permutations of $[n]$ for which $ka_k$ is a perfect square for all $1\\leq k\\leq n$ . Find with proof the smallest $n$ such that $P(n)$ is a multiple of $2010$ .",
      "answer": "\\(\\boxed{4489}\\)"
    },
    {
      "question": "Suppose that $(a_1,b_1),$ $(a_2,b_2),$ $\\dots,$ $(a_{100},b_{100})$ are distinct ordered pairs of nonnegative integers. Let $N$ denote the number of pairs of integers $(i,j)$ satisfying $1\\leq i<j\\leq 100$ and $|a_ib_j-a_jb_i|=1$. Determine the largest possible value of $N$ over all possible choices of the $100$ ordered pairs.",
      "answer": "197"
    },
    {
      "question": "Let $A$ be a set of integers such that for each integer $m$, there exists an integer $a \\in A$ and positive integer $n$ such that $a^{n} \\equiv m(\\bmod 100)$. What is the smallest possible value of $|A|$?",
      "answer": "41"
    },
    {
      "question": "Pyramid $E A R L Y$ is placed in $(x, y, z)$ coordinates so that $E=(10,10,0), A=(10,-10,0), R=(-10,-10,0), L=(-10,10,0)$, and $Y=(0,0,10)$. Tunnels are drilled through the pyramid in such a way that one can move from $(x, y, z)$ to any of the 9 points $(x, y, z-1),(x \\pm 1, y, z-1)$, $(x, y \\pm 1, z-1),(x \\pm 1, y \\pm 1, z-1)$. Sean starts at $Y$ and moves randomly down to the base of the pyramid, choosing each of the possible paths with probability $\\frac{1}{9}$ each time. What is the probability that he ends up at the point $(8,9,0)$?",
      "answer": "\\frac{550}{9^{10}}"
    },
    {
      "question": "An empty $2020 \\times 2020 \\times 2020$ cube is given, and a $2020 \\times 2020$ grid of square unit cells is drawn  on each of its six faces. A [i]beam[/i] is a $1 \\times 1 \\times 2020$ rectangular prism. Several beams are placed inside the cube subject to the following conditions:\n[list=]\n[*]The two $1 \\times 1$ faces of each beam coincide with unit cells lying on opposite faces of the cube. (Hence, there are $3 \\cdot {2020}^2$ possible positions for a beam.)\n[*]No two beams have intersecting interiors.\n[*]The interiors of each of the four $1 \\times 2020$ faces of each beam touch either a face of the cube or the interior of the face of another beam.\n[/list]\nWhat is the smallest positive number of beams that can be placed to satisfy these conditions?",
      "answer": "3030"
    },
    {
      "question": "Calculate the value of the expression $(8 \\times 6)-(4 \\div 2)$.",
      "answer": "46"
    },
    {
      "question": "For how many of the given drawings can the six dots be labelled to represent the links between suspects?",
      "answer": "2"
    },
    {
      "question": "For breakfast, Milan is eating a piece of toast shaped like an equilateral triangle. On the piece of toast rests a single sesame seed that is one inch away from one side, two inches away from another side, and four inches away from the third side. He places a circular piece of cheese on top of the toast that is tangent to each side of the triangle. What is the area of this piece of cheese?",
      "answer": "\\frac{49 \\pi}{9}"
    },
    {
      "question": "How many perfect squares divide $2^{3} \\cdot 3^{5} \\cdot 5^{7} \\cdot 7^{9}$?",
      "answer": "120"
    },
    {
      "question": "Consider triangle $A B C$ with side lengths $A B=4, B C=7$, and $A C=8$. Let $M$ be the midpoint of segment $A B$, and let $N$ be the point on the interior of segment $A C$ that also lies on the circumcircle of triangle $M B C$. Compute $B N$.",
      "answer": "\\frac{\\sqrt{210}}{4}"
    },
    {
      "question": "Compute the number of even positive integers $n \\leq 2024$ such that $1,2, \\ldots, n$ can be split into $\\frac{n}{2}$ pairs, and the sum of the numbers in each pair is a multiple of 3.",
      "answer": "675"
    },
    {
      "question": "On a chessboard, a queen attacks every square it can reach by moving from its current square along a row, column, or diagonal without passing through a different square that is occupied by a chess piece. Find the number of ways in which three indistinguishable queens can be placed on an $8 \\times 8$ chess board so that each queen attacks both others.",
      "answer": "864"
    },
    {
      "question": "The equation\n$$(x-1)(x-2)\\cdots(x-2016)=(x-1)(x-2)\\cdots (x-2016)$$\nis written on the board, with $2016$ linear factors on each side. What is the least possible value of $k$ for which it is possible to erase exactly $k$ of these $4032$ linear factors so that at least one factor remains on each side and the resulting equation has no real solutions?",
      "answer": "2016"
    },
    {
      "question": "Let $A B C D$ be a rectangle with $A B=6$ and $B C=4$. Let $E$ be the point on $B C$ with $B E=3$, and let $F$ be the point on segment $A E$ such that $F$ lies halfway between the segments $A B$ and $C D$. If $G$ is the point of intersection of $D F$ and $B C$, find $B G$.",
      "answer": "1"
    },
    {
      "question": "Tessa has a unit cube, on which each vertex is labeled by a distinct integer between 1 and 8 inclusive. She also has a deck of 8 cards, 4 of which are black and 4 of which are white. At each step she draws a card from the deck, and if the card is black, she simultaneously replaces the number on each vertex by the sum of the three numbers on vertices that are distance 1 away from this vertex; if the card is white, she simultaneously replaces the number on each vertex by the sum of the three numbers on vertices that are distance \\sqrt{2} away from this vertex. When Tessa finishes drawing all cards of the deck, what is the maximum possible value of a number that is on the cube?",
      "answer": "42648"
    },
    {
      "question": "How many nonempty subsets of $\\{1,2,3, \\ldots, 12\\}$ have the property that the sum of the largest element and the smallest element is 13?",
      "answer": "1365"
    },
    {
      "question": "How many positive integers less than 100 are relatively prime to 200?",
      "answer": "40"
    },
    {
      "question": "A pizza is cut into 10 pieces. Two of the pieces are each \\(\\frac{1}{24}\\) of the whole pizza, four are each \\(\\frac{1}{12}\\), two are each \\(\\frac{1}{8}\\), and two are each \\(\\frac{1}{6}\\). A group of \\(n\\) friends share the pizza by distributing all of these pieces. They do not cut any of these pieces. Each of the \\(n\\) friends receives, in total, an equal fraction of the whole pizza. What is the sum of the values of \\(n\\) with \\(2 \\leq n \\leq 10\\) for which this is not possible?",
      "answer": "39"
    },
    {
      "question": "If two fair dice are tossed, what is the probability that their sum is divisible by 5 ?",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "$ S$ is a non-empty subset of the set $ \\{ 1, 2, \\cdots, 108 \\}$, satisfying:\r\n\r\n(1) For any two numbers $ a,b \\in S$ ( may not distinct), there exists $ c \\in S$, such that $ \\gcd(a,c)\\equal{}\\gcd(b,c)\\equal{}1$.\r\n\r\n(2) For any two numbers $ a,b \\in S$ ( may not distinct), there exists $ c' \\in S$, $ c' \\neq a$, $ c' \\neq b$, such that $ \\gcd(a, c') > 1$, $ \\gcd(b,c') >1$.\r\n\r\nFind the largest possible value of $ |S|$.",
      "answer": "79"
    },
    {
      "question": "Let $a, b, c$ be nonzero real numbers such that $a+b+c=0$ and $a^{3}+b^{3}+c^{3}=a^{5}+b^{5}+c^{5}$. Find the value of $a^{2}+b^{2}+c^{2}$.",
      "answer": "\\frac{6}{5}"
    },
    {
      "question": "Define a sequence $a_{i, j}$ of integers such that $a_{1, n}=n^{n}$ for $n \\geq 1$ and $a_{i, j}=a_{i-1, j}+a_{i-1, j+1}$ for all $i, j \\geq 1$. Find the last (decimal) digit of $a_{128,1}$.",
      "answer": "4"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=23, B C=24$, and $C A=27$. Let $D$ be the point on segment $A C$ such that the incircles of triangles $B A D$ and $B C D$ are tangent. Determine the ratio $C D / D A$.",
      "answer": "$\\frac{14}{13}$"
    },
    {
      "question": "In a number line, point $P$ is at 3 and $V$ is at 33. The number line between 3 and 33 is divided into six equal parts by the points $Q, R, S, T, U$. What is the sum of the lengths of $PS$ and $TV$?",
      "answer": "25"
    },
    {
      "question": "Compute the sum of all two-digit positive integers $x$ such that for all three-digit (base 10) positive integers \\underline{a} \\underline{b} \\underline{c}, if \\underline{a} \\underline{b} \\underline{c} is a multiple of $x$, then the three-digit (base 10) number \\underline{b} \\underline{c} \\underline{a} is also a multiple of $x$.",
      "answer": "64"
    },
    {
      "question": "Suppose \\(\\triangle A B C\\) has lengths \\(A B=5, B C=8\\), and \\(C A=7\\), and let \\(\\omega\\) be the circumcircle of \\(\\triangle A B C\\). Let \\(X\\) be the second intersection of the external angle bisector of \\(\\angle B\\) with \\(\\omega\\), and let \\(Y\\) be the foot of the perpendicular from \\(X\\) to \\(B C\\). Find the length of \\(Y C\\).",
      "answer": "\\frac{13}{2}"
    },
    {
      "question": "Let $ S \\equal{} \\{1,2,3,\\cdots ,280\\}$. Find the smallest integer $ n$ such that each $ n$-element subset of $ S$ contains five numbers which are pairwise relatively prime.",
      "answer": "217"
    },
    {
      "question": "Let $A B C$ be an equilateral triangle with $A B=3$. Circle $\\omega$ with diameter 1 is drawn inside the triangle such that it is tangent to sides $A B$ and $A C$. Let $P$ be a point on $\\omega$ and $Q$ be a point on segment $B C$. Find the minimum possible length of the segment $P Q$.",
      "answer": "\\frac{3 \\sqrt{3}-3}{2}"
    },
    {
      "question": "Cyclic pentagon $ABCDE$ has side lengths $AB=BC=5, CD=DE=12$, and $AE=14$. Determine the radius of its circumcircle.",
      "answer": "\\frac{225\\sqrt{11}}{88}"
    },
    {
      "question": "A triangle with side lengths $5,7,8$ is inscribed in a circle $C$. The diameters of $C$ parallel to the sides of lengths 5 and 8 divide $C$ into four sectors. What is the area of either of the two smaller ones?",
      "answer": "\\frac{49}{18} \\pi"
    },
    {
      "question": "The sum of the digits of the time 19 minutes ago is two less than the sum of the digits of the time right now. Find the sum of the digits of the time in 19 minutes. (Here, we use a standard 12-hour clock of the form hh:mm.)",
      "answer": "11"
    },
    {
      "question": "Compute the product of all positive integers $b \\geq 2$ for which the base $b$ number $111111_{b}$ has exactly $b$ distinct prime divisors.",
      "answer": "24"
    },
    {
      "question": "Find the shortest distance between the lines $\\frac{x+2}{2}=\\frac{y-1}{3}=\\frac{z}{1}$ and $\\frac{x-3}{-1}=\\frac{y}{1}=\\frac{z+1}{2}$",
      "answer": "\\frac{5 \\sqrt{3}}{3}"
    },
    {
      "question": "Last summer, Pat worked at a summer camp. For each day that he worked, he earned \\$100 and he was not charged for food. For each day that he did not work, he was not paid and he was charged \\$20 for food. After 70 days, the money that he earned minus his food costs equalled \\$5440. On how many of these 70 days did Pat work?",
      "answer": "57"
    },
    {
      "question": "Let $\\pi$ be a permutation of the numbers from 1 through 2012. What is the maximum possible number of integers $n$ with $1 \\leq n \\leq 2011$ such that $\\pi(n)$ divides $\\pi(n+1)$?",
      "answer": "1006"
    },
    {
      "question": "The numbers $5,6,10,17$, and 21 are rearranged so that the sum of the first three numbers is equal to the sum of the last three numbers. Which number is in the middle of this rearrangement?",
      "answer": "5"
    },
    {
      "question": "All positive integers whose binary representations (excluding leading zeroes) have at least as many 1's as 0's are put in increasing order. Compute the number of digits in the binary representation of the 200th number.",
      "answer": "9"
    },
    {
      "question": "Let $b(x)=x^{2}+x+1$. The polynomial $x^{2015}+x^{2014}+\\cdots+x+1$ has a unique \"base $b(x)$ \" representation $x^{2015}+x^{2014}+\\cdots+x+1=\\sum_{k=0}^{N} a_{k}(x) b(x)^{k}$ where each \"digit\" $a_{k}(x)$ is either the zero polynomial or a nonzero polynomial of degree less than $\\operatorname{deg} b=2$; and the \"leading digit $a_{N}(x)$ \" is nonzero. Find $a_{N}(0)$.",
      "answer": "-1006"
    },
    {
      "question": "The three sides of a right triangle form a geometric sequence. Determine the ratio of the length of the hypotenuse to the length of the shorter leg.",
      "answer": "\\frac{1+\\sqrt{5}}{2}"
    },
    {
      "question": "Two fair six-sided dice are rolled. What is the probability that their sum is at least 10?",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "Find the largest real number $k$ such that there exists a sequence of positive reals $\\left\\{a_{i}\\right\\}$ for which $\\sum_{n=1}^{\\infty} a_{n}$ converges but $\\sum_{n=1}^{\\infty} \\frac{\\sqrt{a_{n}}}{n^{k}}$ does not.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "In a game show, Bob is faced with 7 doors, 2 of which hide prizes. After he chooses a door, the host opens three other doors, of which one is hiding a prize. Bob chooses to switch to another door. What is the probability that his new door is hiding a prize?",
      "answer": "\\frac{5}{21}"
    },
    {
      "question": "Find the sum of $\\frac{1}{n}$ over all positive integers $n$ with the property that the decimal representation of $\\frac{1}{n}$ terminates.",
      "answer": "\\sqrt{\\frac{5}{2}}"
    },
    {
      "question": "If $x$, $y$, $z$ are positive numbers satisfying \n\\[x+\\frac{y}{z}=y+\\frac{z}{x}=z+\\frac{x}{y}=2.\\]\nFind all the possible values of $x+y+z$.",
      "answer": "3"
    },
    {
      "question": "Find $a_{2012}$ if $a_{n} \\equiv a_{n-1}+n(\\bmod 2012)$ and $a_{1}=1$.",
      "answer": "1006"
    },
    {
      "question": "If each of Bill's steps is $\frac{1}{2}$ metre long, how many steps does Bill take to walk 12 metres in a straight line?",
      "answer": "24"
    },
    {
      "question": "Let $a_{0}, a_{1}, a_{2}, \\ldots$ be an infinite sequence where each term is independently and uniformly random in the set $\\{1,2,3,4\\}$. Define an infinite sequence $b_{0}, b_{1}, b_{2}, \\ldots$ recursively by $b_{0}=1$ and $b_{i+1}=a_{i}^{b_{i}}$. Compute the expected value of the smallest positive integer $k$ such that $b_{k} \\equiv 1(\\bmod 5)$.",
      "answer": "\\frac{35}{16}"
    },
    {
      "question": "Compute the remainder when 10002000400080016003200640128025605121024204840968192 is divided by 100020004000800160032.",
      "answer": "40968192"
    },
    {
      "question": "Compute the largest positive integer such that $\\frac{2007!}{2007^{n}}$ is an integer.",
      "answer": "9"
    },
    {
      "question": "Another professor enters the same room and says, 'Each of you has to write down an integer between 0 and 200. I will then compute $X$, the number that is 3 greater than half the average of all the numbers that you will have written down. Each student who writes down the number closest to $X$ (either above or below $X$) will receive a prize.' One student, who misunderstood the question, announces to the class that he will write the number 107. If among the other 99 students it is common knowledge that all 99 of them will write down the best response, and there is no further communication between students, what single integer should each of the 99 students write down?",
      "answer": "7"
    },
    {
      "question": "Alice and Bob are playing in the forest. They have six sticks of length $1,2,3,4,5,6$ inches. Somehow, they have managed to arrange these sticks, such that they form the sides of an equiangular hexagon. Compute the sum of all possible values of the area of this hexagon.",
      "answer": "33 \\sqrt{3}"
    },
    {
      "question": "Let $f: \\mathbb{N} \\rightarrow \\mathbb{N}$ be a function satisfying the following conditions: (a) $f(1)=1$ (b) $f(a) \\leq f(b)$ whenever $a$ and $b$ are positive integers with $a \\leq b$. (c) $f(2a)=f(a)+1$ for all positive integers $a$. How many possible values can the 2014-tuple $(f(1), f(2), \\ldots, f(2014))$ take?",
      "answer": "1007"
    },
    {
      "question": "If $a$ and $b$ are positive integers that can each be written as a sum of two squares, then $a b$ is also a sum of two squares. Find the smallest positive integer $c$ such that $c=a b$, where $a=x^{3}+y^{3}$ and $b=x^{3}+y^{3}$ each have solutions in integers $(x, y)$, but $c=x^{3}+y^{3}$ does not.",
      "answer": "4"
    },
    {
      "question": "Determine the number of triples $0 \\leq k, m, n \\leq 100$ of integers such that $$ 2^{m} n-2^{n} m=2^{k} $$",
      "answer": "22"
    },
    {
      "question": "Joe has written 5 questions of different difficulties for a test with problems numbered 1 though 5. He wants to make sure that problem $i$ is harder than problem $j$ whenever $i-j \\geq 3$. In how many ways can he order the problems for his test?",
      "answer": "25"
    },
    {
      "question": "Ainsley and Buddy play a game where they repeatedly roll a standard fair six-sided die. Ainsley wins if two multiples of 3 in a row are rolled before a non-multiple of 3 followed by a multiple of 3, and Buddy wins otherwise. If the probability that Ainsley wins is $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100a+b$.",
      "answer": "109"
    },
    {
      "question": "Circle $\\Omega$ has radius 13. Circle $\\omega$ has radius 14 and its center $P$ lies on the boundary of circle $\\Omega$. Points $A$ and $B$ lie on $\\Omega$ such that chord $A B$ has length 24 and is tangent to $\\omega$ at point $T$. Find $A T \\cdot B T$.",
      "answer": "56"
    },
    {
      "question": "Find the smallest positive integer $n$ such that $1^{2}+2^{2}+3^{2}+4^{2}+\\cdots+n^{2}$ is divisible by 100.",
      "answer": "24"
    },
    {
      "question": "A particular coin has a $\\frac{1}{3}$ chance of landing on heads (H), $\\frac{1}{3}$ chance of landing on tails (T), and $\\frac{1}{3}$ chance of landing vertically in the middle (M). When continuously flipping this coin, what is the probability of observing the continuous sequence HMMT before HMT?",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "What is the value of \\( z \\) in the carpet installation cost chart?",
      "answer": "1261.40"
    },
    {
      "question": "Solve $x=\\sqrt{x-\\frac{1}{x}}+\\sqrt{1-\\frac{1}{x}}$ for $x$.",
      "answer": "\\frac{1+\\sqrt{5}}{2}"
    },
    {
      "question": "Suppose that $ABC$ is an isosceles triangle with $AB=AC$. Let $P$ be the point on side $AC$ so that $AP=2CP$. Given that $BP=1$, determine the maximum possible area of $ABC$.",
      "answer": "\\frac{9}{10}"
    },
    {
      "question": "Find the greatest constant $\\lambda$ such that for any doubly stochastic matrix of order 100, we can pick $150$ entries such that if the other $9850$ entries were replaced by $0$, the sum of entries in each row and each column is at least $\\lambda$.\n\nNote: A doubly stochastic matrix of order $n$ is a $n\\times n$ matrix, all entries are nonnegative reals, and the sum of entries in each row and column is equal to 1.",
      "answer": "\\frac{17}{1900}"
    },
    {
      "question": "Find the smallest positive number $\\lambda $ , such that for any complex numbers ${z_1},{z_2},{z_3}\\in\\{z\\in C\\big| |z|<1\\}$ ,if  $z_1+z_2+z_3=0$, then $$\\left|z_1z_2 +z_2z_3+z_3z_1\\right|^2+\\left|z_1z_2z_3\\right|^2 <\\lambda .$$",
      "answer": "1"
    },
    {
      "question": "Victoria wants to order at least 550 donuts from Dunkin' Donuts for the HMMT 2014 November contest. However, donuts only come in multiples of twelve. Assuming every twelve donuts cost \\$7.49, what is the minimum amount Victoria needs to pay, in dollars?",
      "answer": "344.54"
    },
    {
      "question": "Find the area of triangle $EFC$ given that $[EFC]=\\left(\\frac{5}{6}\\right)[AEC]=\\left(\\frac{5}{6}\\right)\\left(\\frac{4}{5}\\right)[ADC]=\\left(\\frac{5}{6}\\right)\\left(\\frac{4}{5}\\right)\\left(\\frac{2}{3}\\right)[ABC]$ and $[ABC]=20\\sqrt{3}$.",
      "answer": "\\frac{80\\sqrt{3}}{9}"
    },
    {
      "question": "Call a polygon normal if it can be inscribed in a unit circle. How many non-congruent normal polygons are there such that the square of each side length is a positive integer?",
      "answer": "14"
    },
    {
      "question": "Let $a_{0}, a_{1}, a_{2}, \\ldots$ be a sequence of real numbers defined by $a_{0}=21, a_{1}=35$, and $a_{n+2}=4 a_{n+1}-4 a_{n}+n^{2}$ for $n \\geq 2$. Compute the remainder obtained when $a_{2006}$ is divided by 100.",
      "answer": "0"
    },
    {
      "question": "At the round table, $10$ people are sitting, some of them are knights, and the rest are liars (knights always say pride, and liars always lie) .  It is clear thath I have at least one knight and at least one liar. \nWhat is the largest number of those sitting at the table can say: ''Both of my neighbors are knights '' ?\n(A statement that is at least partially false is considered false.)",
      "answer": "9"
    },
    {
      "question": "For positive integers $n$, let the numbers $c(n)$ be determined by the rules $c(1) = 1$, $c(2n) = c(n)$, and $c(2n+1) = (-1)^n c(n)$. Find the value of \\[ \\sum_{n=1}^{2013} c(n) c(n+2). \\]",
      "answer": "-1"
    },
    {
      "question": "Peter has $2022$ pieces of magnetic railroad cars, which are of two types: some have the front with north and the rear with south magnetic polarity, and some have the rear with north and the rear with south magnetic polarity (on these railroad cars the front and the rear can be distinguished). Peter wants to decide whether there is the same number of both types of cars. He can try to fit together two cars in one try. What is the least number of tries needed?",
      "answer": "2021"
    },
    {
      "question": "A circle passes through the points $(2,0)$ and $(4,0)$ and is tangent to the line $y=x$. Find the sum of all possible values for the $y$-coordinate of the center of the circle.",
      "answer": "-6"
    },
    {
      "question": "Let $ABC$ be a triangle with $AB=5, BC=4$ and $AC=3$. Let $\\mathcal{P}$ and $\\mathcal{Q}$ be squares inside $ABC$ with disjoint interiors such that they both have one side lying on $AB$. Also, the two squares each have an edge lying on a common line perpendicular to $AB$, and $\\mathcal{P}$ has one vertex on $AC$ and $\\mathcal{Q}$ has one vertex on $BC$. Determine the minimum value of the sum of the areas of the two squares.",
      "answer": "\\frac{144}{49}"
    },
    {
      "question": "If $\\frac{1}{6} + \\frac{1}{3} = \\frac{1}{x}$, what is the value of $x$?",
      "answer": "2"
    },
    {
      "question": "Through a point in the interior of a triangle $A B C$, three lines are drawn, one parallel to each side. These lines divide the sides of the triangle into three regions each. Let $a, b$, and $c$ be the lengths of the sides opposite $\\angle A, \\angle B$, and $\\angle C$, respectively, and let $a^{\\prime}, b^{\\prime}$, and $c^{\\prime}$ be the lengths of the middle regions of the sides opposite $\\angle A, \\angle B$, and $\\angle C$, respectively. Find the numerical value of $a^{\\prime} / a+b^{\\prime} / b+c^{\\prime} / c$.",
      "answer": "1"
    },
    {
      "question": "You are trapped in a room with only one exit, a long hallway with a series of doors and land mines. To get out you must open all the doors and disarm all the mines. In the room is a panel with 3 buttons, which conveniently contains an instruction manual. The red button arms a mine, the yellow button disarms two mines and closes a door, and the green button opens two doors. Initially 3 doors are closed and 3 mines are armed. The manual warns that attempting to disarm two mines or open two doors when only one is armed/closed will reset the system to its initial state. What is the minimum number of buttons you must push to get out?",
      "answer": "9"
    },
    {
      "question": "A polygon \\(\\mathcal{P}\\) is drawn on the 2D coordinate plane. Each side of \\(\\mathcal{P}\\) is either parallel to the \\(x\\) axis or the \\(y\\) axis (the vertices of \\(\\mathcal{P}\\) do not have to be lattice points). Given that the interior of \\(\\mathcal{P}\\) includes the interior of the circle \\(x^{2}+y^{2}=2022\\), find the minimum possible perimeter of \\(\\mathcal{P}\\).",
      "answer": "8 \\sqrt{2022}"
    },
    {
      "question": "The set of points $\\left(x_{1}, x_{2}, x_{3}, x_{4}\\right)$ in $\\mathbf{R}^{4}$ such that $x_{1} \\geq x_{2} \\geq x_{3} \\geq x_{4}$ is a cone (or hypercone, if you insist). Into how many regions is this cone sliced by the hyperplanes $x_{i}-x_{j}=1$ for $1 \\leq i<j \\leq n$ ?",
      "answer": "14"
    },
    {
      "question": "An HMMT party has $m$ MIT students and $h$ Harvard students for some positive integers $m$ and $h$, For every pair of people at the party, they are either friends or enemies. If every MIT student has 16 MIT friends and 8 Harvard friends, and every Harvard student has 7 MIT enemies and 10 Harvard enemies, compute how many pairs of friends there are at the party.",
      "answer": "342"
    },
    {
      "question": "What is the sum of all of the possibilities for Sam's number if Sam thinks of a 5-digit number, Sam's friend Sally tries to guess his number, Sam writes the number of matching digits beside each of Sally's guesses, and a digit is considered \"matching\" when it is the correct digit in the correct position?",
      "answer": "526758"
    },
    {
      "question": "In Middle-Earth, nine cities form a 3 by 3 grid. The top left city is the capital of Gondor and the bottom right city is the capital of Mordor. How many ways can the remaining cities be divided among the two nations such that all cities in a country can be reached from its capital via the grid-lines without passing through a city of the other country?",
      "answer": "30"
    },
    {
      "question": "Many of the students in M. Gamache's class brought a skateboard or a bicycle to school yesterday. The ratio of the number of skateboards to the number of bicycles was $7:4$. There were 12 more skateboards than bicycles. How many skateboards and bicycles were there in total?",
      "answer": "44"
    },
    {
      "question": "What is 25% of 60?",
      "answer": "15"
    },
    {
      "question": "A Vandal and a Moderator are editing a Wikipedia article. The article originally is error-free. Each day, the Vandal introduces one new error into the Wikipedia article. At the end of the day, the moderator checks the article and has a $2 / 3$ chance of catching each individual error still in the article. After 3 days, what is the probability that the article is error-free?",
      "answer": "\\frac{416}{729}"
    },
    {
      "question": "Let $S$ be the set of $10$-tuples of non-negative integers that have sum $2019$. For any tuple in $S$, if one of the numbers in the tuple is $\\geq 9$, then we can subtract $9$ from it, and add $1$ to the remaining numbers in the tuple. Call thus one operation. If for $A,B\\in S$ we can get from $A$ to $B$ in finitely many operations, then denote $A\\rightarrow B$.\n\n(1) Find the smallest integer $k$, such that if the minimum number in $A,B\\in S$ respectively are both $\\geq k$, then $A\\rightarrow B$ implies $B\\rightarrow A$.\n\n(2) For the $k$ obtained in (1), how many tuples can we pick from $S$, such that any two of these tuples $A,B$ that are distinct, $A\\not\\rightarrow B$.",
      "answer": "10^8"
    },
    {
      "question": "What is the smallest integer that can be placed in the box so that $\\frac{1}{2} < \\frac{\\square}{9}$?",
      "answer": "5"
    },
    {
      "question": "Acute triangle $A B C$ has circumcenter $O$. The bisector of $\\angle A B C$ and the altitude from $C$ to side $A B$ intersect at $X$. Suppose that there is a circle passing through $B, O, X$, and $C$. If $\\angle B A C=n^{\\circ}$, where $n$ is a positive integer, compute the largest possible value of $n$.",
      "answer": "67"
    },
    {
      "question": "Glen, Hao, Ioana, Julia, Karla, and Levi participated in the 2023 Canadian Team Mathematics Contest. On their team uniforms, each had a different number chosen from the list $11,12,13,14,15,16$. Hao's and Julia's numbers were even. Karla's and Levi's numbers were prime numbers. Glen's number was a perfect square. What was Ioana's number?",
      "answer": "15"
    },
    {
      "question": "Find the value of $$ \\binom{2003}{1}+\\binom{2003}{4}+\\binom{2003}{7}+\\cdots+\\binom{2003}{2002} $$",
      "answer": "\\left(2^{2003}-2\\right) / 3"
    },
    {
      "question": "Calculate $\\sum_{n=1}^{2001} n^{3}$.",
      "answer": "4012013006001"
    },
    {
      "question": "$100$ children stand in a line each having $100$ candies. In one move, one of them may take some of their candies and distribute them to a non-empty set of the remaining children. After what least number of moves can it happen that no two children have the same number of candies?",
      "answer": "30"
    },
    {
      "question": "In 2019, a team, including professor Andrew Sutherland of MIT, found three cubes of integers which sum to 42: $42=\\left(-8053873881207597 \\_\\right)^{3}+(80435758145817515)^{3}+(12602123297335631)^{3}$. One of the digits, labeled by an underscore, is missing. What is that digit?",
      "answer": "4"
    },
    {
      "question": "Anna thinks of an integer that is not a multiple of three, not a perfect square, and the sum of its digits is a prime number. What could the integer be?",
      "answer": "14"
    },
    {
      "question": "How much money does Roman give Dale if Roman wins a contest with a prize of $\\$ 200$, gives $30 \\%$ of the prize to Jackie, and then splits $15 \\%$ of what remains equally between Dale and Natalia?",
      "answer": "\\$ 10.50"
    },
    {
      "question": "Let $x<0.1$ be a positive real number. Let the foury series be $4+4 x+4 x^{2}+4 x^{3}+\\ldots$, and let the fourier series be $4+44 x+444 x^{2}+4444 x^{3}+\\ldots$ Suppose that the sum of the fourier series is four times the sum of the foury series. Compute $x$.",
      "answer": "\\frac{3}{40}"
    },
    {
      "question": "I have five different pairs of socks. Every day for five days, I pick two socks at random without replacement to wear for the day. Find the probability that I wear matching socks on both the third day and the fifth day.",
      "answer": "\\frac{1}{63}"
    },
    {
      "question": "Let $s(n)$ denote the number of 1's in the binary representation of $n$. Compute $$\\frac{1}{255} \\sum_{0 \\leq n<16} 2^{n}(-1)^{s(n)}$$",
      "answer": "45"
    },
    {
      "question": "Tessa has a figure created by adding a semicircle of radius 1 on each side of an equilateral triangle with side length 2, with semicircles oriented outwards. She then marks two points on the boundary of the figure. What is the greatest possible distance between the two points?",
      "answer": "3"
    },
    {
      "question": "Determine the greatest possible value of \\(\\sum_{i=1}^{10} \\cos(3x_i)\\) for real numbers $x_1,x_2,\\dots,x_{10}$ satisfying \\(\\sum_{i=1}^{10} \\cos(x_i) = 0\\).",
      "answer": "\\frac{480}{49}"
    },
    {
      "question": "A sequence of positive integers is defined by $a_{0}=1$ and $a_{n+1}=a_{n}^{2}+1$ for each $n \\geq 0$. Find $\\operatorname{gcd}(a_{999}, a_{2004})$.",
      "answer": "677"
    },
    {
      "question": "Let $r_{1}, r_{2}, r_{3}, r_{4}$ be the four roots of the polynomial $x^{4}-4 x^{3}+8 x^{2}-7 x+3$. Find the value of $\\frac{r_{1}^{2}}{r_{2}^{2}+r_{3}^{2}+r_{4}^{2}}+\\frac{r_{2}^{2}}{r_{1}^{2}+r_{3}^{2}+r_{4}^{2}}+\\frac{r_{3}^{2}}{r_{1}^{2}+r_{2}^{2}+r_{4}^{2}}+\\frac{r_{4}^{2}}{r_{1}^{2}+r_{2}^{2}+r_{3}^{2}}$",
      "answer": "-4"
    },
    {
      "question": "Find the number of nonempty sets $\\mathcal{F}$ of subsets of the set $\\{1, \\ldots, 2014\\}$ such that: (a) For any subsets $S_{1}, S_{2} \\in \\mathcal{F}, S_{1} \\cap S_{2} \\in \\mathcal{F}$. (b) If $S \\in \\mathcal{F}, T \\subseteq\\{1, \\ldots, 2014\\}$, and $S \\subseteq T$, then $T \\in \\mathcal{F}$.",
      "answer": "2^{2014}"
    },
    {
      "question": "Brave NiuNiu (a milk drink company) organizes a promotion during the Chinese New Year: one gets a red packet when buying a carton of milk of their brand, and there is one of the following characters in the red packet \"虎\"(Tiger), \"生\"(Gain), \"威\"(Strength). If one collects two \"虎\", one \"生\" and one \"威\", then they form a Chinese phrases \"虎虎生威\" (Pronunciation: hu hu sheng wei), which means \"Have the courage and strength of the tiger\". This is a nice blessing because the Chinese zodiac sign for the year 2022 is tiger. Soon, the product of Brave NiuNiu becomes quite popular and people hope to get a collection of \"虎虎生威\". Suppose that the characters in every packet are independently random, and each character has probability $\\frac{1}{3}$. What is the expectation of cartons of milk to collect \"虎虎生威\" (i.e. one collects at least 2 copies of \"虎\", 1 copy of \"生\", 1 copy of \"威\")? Options: (A) $6 \\frac{1}{3}$, (B) $7 \\frac{1}{3}$, (C) $8 \\frac{1}{3}$, (D) $9 \\frac{1}{3}$, (E) None of the above.",
      "answer": "7 \\frac{1}{3}"
    },
    {
      "question": "The mean (average) of 5 consecutive integers is 9. What is the smallest of these 5 integers?",
      "answer": "7"
    },
    {
      "question": "Find the area of triangle $QCD$ given that $Q$ is the intersection of the line through $B$ and the midpoint of $AC$ with the plane through $A, C, D$ and $N$ is the midpoint of $CD$.",
      "answer": "\\frac{3 \\sqrt{3}}{20}"
    },
    {
      "question": "What is the perimeter of the shaded region in a \\( 3 \\times 3 \\) grid where some \\( 1 \\times 1 \\) squares are shaded?",
      "answer": "10"
    },
    {
      "question": "Let $a,b,c,d$ be real numbers such that $a^2+b^2+c^2+d^2=1$. Determine the minimum value of $(a-b)(b-c)(c-d)(d-a)$ and determine all values of $(a,b,c,d)$ such that the minimum value is achived.",
      "answer": "-\\frac{1}{8}"
    },
    {
      "question": "How many pairs of real numbers $(x, y)$ satisfy the equation $y^{4}-y^{2}=x y^{3}-x y=x^{3} y-x y=x^{4}-x^{2}=0$?",
      "answer": "9"
    },
    {
      "question": "How many positive integers $n \\leq 2009$ have the property that $\\left\\lfloor\\log _{2}(n)\\right\\rfloor$ is odd?",
      "answer": "682"
    },
    {
      "question": "On a party with 99 guests, hosts Ann and Bob play a game (the hosts are not regarded as guests). There are 99 chairs arranged in a circle; initially, all guests hang around those chairs. The hosts take turns alternately. By a turn, a host orders any standing guest to sit on an unoccupied chair $c$. If some chair adjacent to $c$ is already occupied, the same host orders one guest on such chair to stand up (if both chairs adjacent to $c$ are occupied, the host chooses exactly one of them). All orders are carried out immediately. Ann makes the first move; her goal is to fulfill, after some move of hers, that at least $k$ chairs are occupied. Determine the largest $k$ for which Ann can reach the goal, regardless of Bob's play.",
      "answer": "34"
    },
    {
      "question": "Find the largest integer $n$ such that $3^{512}-1$ is divisible by $2^{n}$.",
      "answer": "11"
    },
    {
      "question": "$H O W, B O W$, and $D A H$ are equilateral triangles in a plane such that $W O=7$ and $A H=2$. Given that $D, A, B$ are collinear in that order, find the length of $B A$.",
      "answer": "11"
    },
    {
      "question": "Consider a $7 \\times 7$ grid of squares. Let $f:\\{1,2,3,4,5,6,7\\} \\rightarrow\\{1,2,3,4,5,6,7\\}$ be a function; in other words, $f(1), f(2), \\ldots, f(7)$ are each (not necessarily distinct) integers from 1 to 7 . In the top row of the grid, the numbers from 1 to 7 are written in order; in every other square, $f(x)$ is written where $x$ is the number above the square. How many functions have the property that the bottom row is identical to the top row, and no other row is identical to the top row?",
      "answer": "1470"
    },
    {
      "question": "Pentagon $J A M E S$ is such that $A M=S J$ and the internal angles satisfy $\\angle J=\\angle A=\\angle E=90^{\\circ}$, and $\\angle M=\\angle S$. Given that there exists a diagonal of $J A M E S$ that bisects its area, find the ratio of the shortest side of $J A M E S$ to the longest side of $J A M E S$.",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Find the largest real $C$ such that for all pairwise distinct positive real $a_{1}, a_{2}, \\ldots, a_{2019}$ the following inequality holds $$\\frac{a_{1}}{\\left|a_{2}-a_{3}\\right|}+\\frac{a_{2}}{\\left|a_{3}-a_{4}\\right|}+\\ldots+\\frac{a_{2018}}{\\left|a_{2019}-a_{1}\\right|}+\\frac{a_{2019}}{\\left|a_{1}-a_{2}\\right|}>C$$",
      "answer": "1010"
    },
    {
      "question": "The cells of a $8 \\times 8$ table are initially white. Alice and Bob play a game. First Alice paints $n$ of the fields in red. Then Bob chooses $4$ rows and $4$ columns from the table and paints all fields in them in black. Alice wins if there is at least one red field left. Find the least value of $n$ such that Alice can win the game no matter how Bob plays.",
      "answer": "13"
    },
    {
      "question": "Let $V=\\{1, \\ldots, 8\\}$. How many permutations $\\sigma: V \\rightarrow V$ are automorphisms of some tree?",
      "answer": "30212"
    },
    {
      "question": "Let $S=\\{-100,-99,-98, \\ldots, 99,100\\}$. Choose a 50-element subset $T$ of $S$ at random. Find the expected number of elements of the set $\\{|x|: x \\in T\\}$.",
      "answer": "\\frac{8825}{201}"
    },
    {
      "question": "Let $(x, y)$ be a pair of real numbers satisfying $$56x+33y=\\frac{-y}{x^{2}+y^{2}}, \\quad \\text { and } \\quad 33x-56y=\\frac{x}{x^{2}+y^{2}}$$ Determine the value of $|x|+|y|$.",
      "answer": "\\frac{11}{65}"
    },
    {
      "question": "The integers $1,2,4,5,6,9,10,11,13$ are to be placed in the circles and squares below with one number in each shape. Each integer must be used exactly once and the integer in each circle must be equal to the sum of the integers in the two neighbouring squares. If the integer $x$ is placed in the leftmost square and the integer $y$ is placed in the rightmost square, what is the largest possible value of $x+y$?",
      "answer": "20"
    },
    {
      "question": "What is the value of $k$ if the side lengths of four squares are shown, and the area of the fifth square is $k$?",
      "answer": "36"
    },
    {
      "question": "What is the probability that a randomly selected set of 5 numbers from the set of the first 15 positive integers has a sum divisible by 3?",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "For a real number $x$, let $[x]$ be $x$ rounded to the nearest integer and $\\langle x\\rangle$ be $x$ rounded to the nearest tenth. Real numbers $a$ and $b$ satisfy $\\langle a\\rangle+[b]=98.6$ and $[a]+\\langle b\\rangle=99.3$. Compute the minimum possible value of $[10(a+b)]$.",
      "answer": "988"
    },
    {
      "question": "The set $S=\\{1,2,3, \\ldots, 49,50\\}$ contains the first 50 positive integers. After the multiples of 2 and the multiples of 3 are removed, how many integers remain in the set $S$?",
      "answer": "17"
    },
    {
      "question": "Let $f: \\mathbb{Z} \\rightarrow \\mathbb{Z}$ be a function such that for any integers $x, y$, we have $f\\left(x^{2}-3 y^{2}\\right)+f\\left(x^{2}+y^{2}\\right)=2(x+y) f(x-y)$. Suppose that $f(n)>0$ for all $n>0$ and that $f(2015) \\cdot f(2016)$ is a perfect square. Find the minimum possible value of $f(1)+f(2)$.",
      "answer": "246"
    },
    {
      "question": "On the Cartesian grid, Johnny wants to travel from $(0,0)$ to $(5,1)$, and he wants to pass through all twelve points in the set $S=\\{(i, j) \\mid 0 \\leq i \\leq 1,0 \\leq j \\leq 5, i, j \\in \\mathbb{Z}\\}$. Each step, Johnny may go from one point in $S$ to another point in $S$ by a line segment connecting the two points. How many ways are there for Johnny to start at $(0,0)$ and end at $(5,1)$ so that he never crosses his own path?",
      "answer": "252"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=5, B C=4$, and $C A=3$. Initially, there is an ant at each vertex. The ants start walking at a rate of 1 unit per second, in the direction $A \\rightarrow B \\rightarrow C \\rightarrow A$ (so the ant starting at $A$ moves along ray $\\overrightarrow{A B}$, etc.). For a positive real number $t$ less than 3, let $A(t)$ be the area of the triangle whose vertices are the positions of the ants after $t$ seconds have elapsed. For what positive real number $t$ less than 3 is $A(t)$ minimized?",
      "answer": "\\frac{47}{24}"
    },
    {
      "question": "What is the value of the expression \\( 4 + \\frac{3}{10} + \\frac{9}{1000} \\)?",
      "answer": "4.309"
    },
    {
      "question": "If $2^{x}=16$, what is the value of $2^{x+3}$?",
      "answer": "128"
    },
    {
      "question": "The digits $1,2,3,4,5,6$ are randomly chosen (without replacement) to form the three-digit numbers $M=\\overline{A B C}$ and $N=\\overline{D E F}$. For example, we could have $M=413$ and $N=256$. Find the expected value of $M \\cdot N$.",
      "answer": "143745"
    },
    {
      "question": "A rectangle has length 13 and width 10. The length and the width of the rectangle are each increased by 2. By how much does the area of the rectangle increase?",
      "answer": "50"
    },
    {
      "question": "What is the value of $\frac{(20-16) \times (12+8)}{4}$?",
      "answer": "20"
    },
    {
      "question": "A solid rectangular prism has dimensions 4 by 2 by 2. A 1 by 1 by 1 cube is cut out of the corner creating the new solid shown. What is the surface area of the new solid?",
      "answer": "40"
    },
    {
      "question": "Mrs. Toad has a class of 2017 students, with unhappiness levels $1,2, \\ldots, 2017$ respectively. Today in class, there is a group project and Mrs. Toad wants to split the class in exactly 15 groups. The unhappiness level of a group is the average unhappiness of its members, and the unhappiness of the class is the sum of the unhappiness of all 15 groups. What's the minimum unhappiness of the class Mrs. Toad can achieve by splitting the class into 15 groups?",
      "answer": "1121"
    },
    {
      "question": "Ten numbers have an average (mean) of 87. Two of those numbers are 51 and 99. What is the average of the other eight numbers?",
      "answer": "90"
    },
    {
      "question": "A math professor stands up in front of a room containing 100 very smart math students and says, 'Each of you has to write down an integer between 0 and 100, inclusive, to guess 'two-thirds of the average of all the responses.' Each student who guesses the highest integer that is not higher than two-thirds of the average of all responses will receive a prize.' If among all the students it is common knowledge that everyone will write down the best response, and there is no communication between students, what single integer should each of the 100 students write down?",
      "answer": "0"
    },
    {
      "question": "We are given an infinite deck of cards, each with a real number on it. For every real number $x$, there is exactly one card in the deck that has $x$ written on it. Now two players draw disjoint sets $A$ and $B$ of $100$ cards each from this deck. We would like to define a rule that declares one of them a winner. This rule should satisfy the following conditions:\n   1. The winner only depends on the relative order of the $200$ cards: if the cards are laid down in increasing order face down and we are told which card belongs to which player, but not what numbers are written on them, we can still decide the winner.\n   2. If we write the elements of both sets in increasing order as $A =\\{ a_1 , a_2 , \\ldots, a_{100} \\}$ and $B= \\{ b_1 , b_2 , \\ldots , b_{100} \\}$, and $a_i > b_i$ for all $i$, then $A$ beats $B$.\n   3. If three players draw three disjoint sets $A, B, C$ from the deck, $A$ beats $B$ and $B$ beats $C$  then $A$ also beats $C$.\nHow many ways are there to define such a rule? Here, we consider two rules as different if there exist two sets $A$ and $B$ such that $A$ beats $B$ according to one rule, but $B$ beats $A$ according to the other.\n\n[i]",
      "answer": "100"
    },
    {
      "question": "Euler's Bridge: The following figure is the graph of the city of Konigsburg in 1736 - vertices represent sections of the cities, edges are bridges. An Eulerian path through the graph is a path which moves from vertex to vertex, crossing each edge exactly once. How many ways could World War II bombers have knocked out some of the bridges of Konigsburg such that the Allied victory parade could trace an Eulerian path through the graph? (The order in which the bridges are destroyed matters.)",
      "answer": "13023"
    },
    {
      "question": "If $x \\%$ of 60 is 12, what is $15 \\%$ of $x$?",
      "answer": "3"
    },
    {
      "question": "The cookies in a cookie jar contain a total of 100 raisins. All but one of the cookies are the same size and contain the same number of raisins. One cookie is larger and contains one more raisin than each of the others. The number of cookies in the jar is between 5 and 10, inclusive. How many raisins are in the larger cookie?",
      "answer": "12"
    },
    {
      "question": "Alice and the Cheshire Cat play a game. At each step, Alice either (1) gives the cat a penny, which causes the cat to change the number of (magic) beans that Alice has from $n$ to $5n$ or (2) gives the cat a nickel, which causes the cat to give Alice another bean. Alice wins (and the cat disappears) as soon as the number of beans Alice has is greater than 2008 and has last two digits 42. What is the minimum number of cents Alice can spend to win the game, assuming she starts with 0 beans?",
      "answer": "35"
    },
    {
      "question": "Suppose that there are 16 variables $\\left\\{a_{i, j}\\right\\}_{0 \\leq i, j \\leq 3}$, each of which may be 0 or 1 . For how many settings of the variables $a_{i, j}$ do there exist positive reals $c_{i, j}$ such that the polynomial $$f(x, y)=\\sum_{0 \\leq i, j \\leq 3} a_{i, j} c_{i, j} x^{i} y^{j}$$ $(x, y \\in \\mathbb{R})$ is bounded below?",
      "answer": "126"
    },
    {
      "question": "The integer 636405 may be written as the product of three 2-digit positive integers. What is the sum of these three integers?",
      "answer": "259"
    },
    {
      "question": "Compute the number of distinct pairs of the form (first three digits of $x$, first three digits of $x^{4}$ ) over all integers $x>10^{10}$. For example, one such pair is $(100,100)$ when $x=10^{10^{10}}$.",
      "answer": "4495"
    },
    {
      "question": "In a game of rock-paper-scissors with $n$ people, the following rules are used to determine a champion: (a) In a round, each person who has not been eliminated randomly chooses one of rock, paper, or scissors to play. (b) If at least one person plays rock, at least one person plays paper, and at least one person plays scissors, then the round is declared a tie and no one is eliminated. If everyone makes the same move, then the round is also declared a tie. (c) If exactly two moves are represented, then everyone who made the losing move is eliminated from playing in all further rounds (for example, in a game with 8 people, if 5 people play rock and 3 people play scissors, then the 3 who played scissors are eliminated). (d) The rounds continue until only one person has not been eliminated. That person is declared the champion and the game ends. If a game begins with 4 people, what is the expected value of the number of rounds required for a champion to be determined?",
      "answer": "\\frac{45}{14}"
    },
    {
      "question": "A convex polyhedron has $n$ faces that are all congruent triangles with angles $36^{\\circ}, 72^{\\circ}$, and $72^{\\circ}$. Determine, with proof, the maximum possible value of $n$.",
      "answer": "\\[ 36 \\]"
    },
    {
      "question": "Three real numbers \\( x, y, \\) and \\( z \\) are such that \\( (x+4) / 2=(y+9) /(z-3)=(x+5) /(z-5) \\). Determine the value of \\( x / y \\).",
      "answer": "1/2"
    },
    {
      "question": "$A B C D$ is a regular tetrahedron of volume 1. Maria glues regular tetrahedra $A^{\\prime} B C D, A B^{\\prime} C D$, $A B C^{\\prime} D$, and $A B C D^{\\prime}$ to the faces of $A B C D$. What is the volume of the tetrahedron $A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}$?",
      "answer": "\\frac{125}{27}"
    },
    {
      "question": "Suppose that $a$ and $b$ are real numbers such that the line $y=a x+b$ intersects the graph of $y=x^{2}$ at two distinct points $A$ and $B$. If the coordinates of the midpoint of $A B$ are $(5,101)$, compute $a+b$.",
      "answer": "61"
    },
    {
      "question": "Find the minimum positive integer $n\\ge 3$, such that there exist $n$ points $A_1,A_2,\\cdots, A_n$ satisfying no three points are collinear and for any $1\\le i\\le n$, there exist $1\\le j \\le n (j\\neq i)$, segment $A_jA_{j+1}$ pass through the midpoint of segment $A_iA_{i+1}$, where $A_{n+1}=A_1$",
      "answer": "6"
    },
    {
      "question": "A bug is on a corner of a cube. A healthy path for the bug is a path along the edges of the cube that starts and ends where the bug is located, uses no edge multiple times, and uses at most two of the edges adjacent to any particular face. Find the number of healthy paths.",
      "answer": "6"
    },
    {
      "question": "Ellie's drawer of hair clips contains 4 red clips, 5 blue clips, and 7 green clips. Each morning, she randomly chooses one hair clip to wear for the day. She returns this clip to the drawer each evening. One morning, Kyne removes $k$ hair clips before Ellie can make her daily selection. As a result, the probability that Ellie chooses a red clip is doubled. What is a possible value of $k$?",
      "answer": "12"
    },
    {
      "question": "Suppose that a polynomial of the form $p(x)=x^{2010} \\pm x^{2009} \\pm \\cdots \\pm x \\pm 1$ has no real roots. What is the maximum possible number of coefficients of -1 in $p$?",
      "answer": "1005"
    },
    {
      "question": "There are 100 people in a room with ages $1,2, \\ldots, 100$. A pair of people is called cute if each of them is at least seven years older than half the age of the other person in the pair. At most how many pairwise disjoint cute pairs can be formed in this room?",
      "answer": "43"
    },
    {
      "question": "The $y$-intercepts of three parallel lines are 2, 3, and 4. The sum of the $x$-intercepts of the three lines is 36. What is the slope of these parallel lines?",
      "answer": "-\\frac{1}{4}"
    },
    {
      "question": "(Lucas Numbers) The Lucas numbers are defined by $L_{0}=2, L_{1}=1$, and $L_{n+2}=L_{n+1}+L_{n}$ for every $n \\geq 0$. There are $N$ integers $1 \\leq n \\leq 2016$ such that $L_{n}$ contains the digit 1 . Estimate $N$.",
      "answer": "1984"
    },
    {
      "question": "A subset $S$ of the set $\\{1,2, \\ldots, 10\\}$ is chosen randomly, with all possible subsets being equally likely. Compute the expected number of positive integers which divide the product of the elements of $S$. (By convention, the product of the elements of the empty set is 1.)",
      "answer": "\\frac{375}{8}"
    },
    {
      "question": "If $N$ is a positive integer between 1000000 and 10000000, inclusive, what is the maximum possible value for the sum of the digits of $25 \\times N$?",
      "answer": "67"
    },
    {
      "question": "If $a(x+2)+b(x+2)=60$ and $a+b=12$, what is the value of $x$?",
      "answer": "3"
    },
    {
      "question": "Points $A, B, C$, and $D$ are on a line in that order. The distance from $A$ to $D$ is 24. The distance from $B$ to $D$ is 3 times the distance from $A$ to $B$. Point $C$ is halfway between $B$ and $D$. What is the distance from $A$ to $C$?",
      "answer": "15"
    },
    {
      "question": "Let \\(ABCD\\) be an isosceles trapezoid with \\(AB=1, BC=DA=5, CD=7\\). Let \\(P\\) be the intersection of diagonals \\(AC\\) and \\(BD\\), and let \\(Q\\) be the foot of the altitude from \\(D\\) to \\(BC\\). Let \\(PQ\\) intersect \\(AB\\) at \\(R\\). Compute \\(\\sin \\angle RPD\\).",
      "answer": "\\frac{4}{5}"
    },
    {
      "question": "How many 5-digit numbers $\\overline{a b c d e}$ exist such that digits $b$ and $d$ are each the sum of the digits to their immediate left and right? (That is, $b=a+c$ and $d=c+e$.)",
      "answer": "330"
    },
    {
      "question": "Let $A B C D E$ be a convex pentagon such that $\\angle A B C=\\angle A C D=\\angle A D E=90^{\\circ}$ and $A B=B C=C D=D E=1$. Compute $A E$.",
      "answer": "2"
    },
    {
      "question": "A line of soldiers 1 mile long is jogging. The drill sergeant, in a car, moving at twice their speed, repeatedly drives from the back of the line to the front of the line and back again. When each soldier has marched 15 miles, how much mileage has been added to the car, to the nearest mile?",
      "answer": "30"
    },
    {
      "question": "Triangle $A B C$ is given in the plane. Let $A D$ be the angle bisector of $\\angle B A C$; let $B E$ be the altitude from $B$ to $A D$, and let $F$ be the midpoint of $A B$. Given that $A B=28, B C=33, C A=37$, what is the length of $E F$ ?",
      "answer": "14"
    },
    {
      "question": "Let $f(x)$ be a quotient of two quadratic polynomials. Given that $f(n)=n^{3}$ for all $n \\in\\{1,2,3,4,5\\}$, compute $f(0)$.",
      "answer": "\\frac{24}{17}"
    },
    {
      "question": "Alison is eating 2401 grains of rice for lunch. She eats the rice in a very peculiar manner: every step, if she has only one grain of rice remaining, she eats it. Otherwise, she finds the smallest positive integer $d>1$ for which she can group the rice into equal groups of size $d$ with none left over. She then groups the rice into groups of size $d$, eats one grain from each group, and puts the rice back into a single pile. How many steps does it take her to finish all her rice?",
      "answer": "17"
    },
    {
      "question": "Chris and Paul each rent a different room of a hotel from rooms $1-60$. However, the hotel manager mistakes them for one person and gives \"Chris Paul\" a room with Chris's and Paul's room concatenated. For example, if Chris had 15 and Paul had 9, \"Chris Paul\" has 159. If there are 360 rooms in the hotel, what is the probability that \"Chris Paul\" has a valid room?",
      "answer": "\\frac{153}{1180}"
    },
    {
      "question": "679 contestants participated in HMMT February 2017. Let \\(N\\) be the number of these contestants who performed at or above the median score in at least one of the three individual tests. Estimate \\(N\\). An estimate of \\(E\\) earns \\(\\left\\lfloor 20-\\frac{|E-N|}{2}\\right\\rfloor\\) or 0 points, whichever is greater.",
      "answer": "516"
    },
    {
      "question": "Let \\(a \\star b=\\sin a \\cos b\\) for all real numbers \\(a\\) and \\(b\\). If \\(x\\) and \\(y\\) are real numbers such that \\(x \\star y-y \\star x=1\\), what is the maximum value of \\(x \\star y+y \\star x\\)?",
      "answer": "1"
    },
    {
      "question": "The squares of a $3 \\times 3$ grid are filled with positive integers such that 1 is the label of the upperleftmost square, 2009 is the label of the lower-rightmost square, and the label of each square divides the one directly to the right of it and the one directly below it. How many such labelings are possible?",
      "answer": "2448"
    },
    {
      "question": "Let the sequence $a_{i}$ be defined as $a_{i+1}=2^{a_{i}}$. Find the number of integers $1 \\leq n \\leq 1000$ such that if $a_{0}=n$, then 100 divides $a_{1000}-a_{1}$.",
      "answer": "50"
    },
    {
      "question": "Shanille O'Keal shoots free throws on a basketball court. She hits the first and misses the second, and thereafter the probability that she hits the next shot is equal to the proportion of shots she has hit so far. What is the probability she hits exactly 50 of her first 100 shots?",
      "answer": "\\(\\frac{1}{99}\\)"
    },
    {
      "question": "If $n$ is a positive integer, let $s(n)$ denote the sum of the digits of $n$. We say that $n$ is zesty if there exist positive integers $x$ and $y$ greater than 1 such that $x y=n$ and $s(x) s(y)=s(n)$. How many zesty two-digit numbers are there?",
      "answer": "34"
    },
    {
      "question": "What is the difference between the largest and smallest numbers in the list $0.023,0.302,0.203,0.320,0.032$?",
      "answer": "0.297"
    },
    {
      "question": "Which of the following integers cannot be written as a product of two integers, each greater than 1: 6, 27, 53, 39, 77?",
      "answer": "53"
    },
    {
      "question": "The Antarctican language has an alphabet of just 16 letters. Interestingly, every word in the language has exactly 3 letters, and it is known that no word's first letter equals any word's last letter (for instance, if the alphabet were $\\{a, b\\}$ then $a a b$ and aaa could not both be words in the language because $a$ is the first letter of a word and the last letter of a word; in fact, just aaa alone couldn't be in the language). Given this, determine the maximum possible number of words in the language.",
      "answer": "1024"
    },
    {
      "question": "Pick a random digit in the decimal expansion of $\\frac{1}{99999}$. What is the probability that it is 0?",
      "answer": "\\frac{4}{5}"
    },
    {
      "question": "Let $z_{0}+z_{1}+z_{2}+\\cdots$ be an infinite complex geometric series such that $z_{0}=1$ and $z_{2013}=\\frac{1}{2013^{2013}}$. Find the sum of all possible sums of this series.",
      "answer": "\\frac{2013^{2014}}{2013^{2013}-1}"
    },
    {
      "question": "The top section of an 8 cm by 6 cm rectangular sheet of paper is folded along a straight line so that when the top section lies flat on the bottom section, corner $P$ lies on top of corner $R$. What is the length of the crease?",
      "answer": "7.5"
    },
    {
      "question": "Assume the quartic $x^{4}-a x^{3}+b x^{2}-a x+d=0$ has four real roots $\\frac{1}{2} \\leq x_{1}, x_{2}, x_{3}, x_{4} \\leq 2$. Find the maximum possible value of $\\frac{\\left(x_{1}+x_{2}\\right)\\left(x_{1}+x_{3}\\right) x_{4}}{\\left(x_{4}+x_{2}\\right)\\left(x_{4}+x_{3}\\right) x_{1}}$ (over all valid choices of $\\left.a, b, d\\right)$.",
      "answer": "\\frac{5}{4}"
    },
    {
      "question": "Let $V$ be a rectangular prism with integer side lengths. The largest face has area 240 and the smallest face has area 48. A third face has area $x$, where $x$ is not equal to 48 or 240. What is the sum of all possible values of $x$?",
      "answer": "260"
    },
    {
      "question": "The gas tank in Catherine's car is $\\frac{1}{8}$ full. When 30 litres of gas are added, the tank becomes $\\frac{3}{4}$ full. If the gas costs Catherine $\\$ 1.38$ per litre, how much will it cost her to fill the remaining quarter of the tank?",
      "answer": "\\$16.56"
    },
    {
      "question": "How many ways, without taking order into consideration, can 2002 be expressed as the sum of 3 positive integers (for instance, $1000+1000+2$ and $1000+2+1000$ are considered to be the same way)?",
      "answer": "334000"
    },
    {
      "question": "In triangle $A B C, \\angle A B C$ is obtuse. Point $D$ lies on side $A C$ such that \\angle A B D$ is right, and point $E$ lies on side $A C$ between $A$ and $D$ such that $B D$ bisects \\angle E B C$. Find $C E$, given that $A C=35, B C=7$, and $B E=5$.",
      "answer": "10"
    },
    {
      "question": "Consider the addition problem: \\begin{tabular}{ccccc} & C & A & S & H \\\\ + & & & M & E \\\\ \\hline O & S & I & D & E \\end{tabular} where each letter represents a base-ten digit, and $C, M, O \\neq 0$. (Distinct letters are allowed to represent the same digit) How many ways are there to assign values to the letters so that the addition problem is true?",
      "answer": "0"
    },
    {
      "question": "Suppose that $a, b$ and $c$ are integers with $(x-a)(x-6)+3=(x+b)(x+c)$ for all real numbers $x$. What is the sum of all possible values of $b$?",
      "answer": "-24"
    },
    {
      "question": "We call the pair $(m, n)$ of positive integers a happy pair if the greatest common divisor of $m$ and $n$ is a perfect square. For example, $(20, 24)$ is a happy pair because the greatest common divisor of 20 and 24 is 4. Suppose that $k$ is a positive integer such that $(205800, 35k)$ is a happy pair. What is the number of possible values of $k$ with $k \\leq 2940$?",
      "answer": "30"
    },
    {
      "question": "Let $a_{1}, a_{2}, a_{3}, \\ldots$ be a sequence of positive integers where $a_{1}=\\sum_{i=0}^{100} i$! and $a_{i}+a_{i+1}$ is an odd perfect square for all $i \\geq 1$. Compute the smallest possible value of $a_{1000}$.",
      "answer": "7"
    },
    {
      "question": "A positive integer is called [i]fancy[/i] if it can be expressed in the form $$2^{a_1}+2^{a_2}+ \\cdots+ 2^{a_{100}},$$ where $a_1,a_2, \\cdots, a_{100}$ are non-negative integers that are not necessarily distinct. Find the smallest positive integer $n$ such that no multiple of $n$ is a [i]fancy[/i] number.",
      "answer": "2^{101} - 1"
    },
    {
      "question": "On a $3 \\times 3$ chessboard, each square contains a knight with $\\frac{1}{2}$ probability. What is the probability that there are two knights that can attack each other? (In chess, a knight can attack any piece which is two squares away from it in a particular direction and one square away in a perpendicular direction.)",
      "answer": "\\frac{209}{256}"
    },
    {
      "question": "Given $30$ students such that each student has at most $5$ friends and for every $5$ students there is a pair of students that are not friends, determine the maximum $k$ such that for all such possible configurations, there exists $k$ students who are all not friends.",
      "answer": "6"
    },
    {
      "question": "Bernie has 2020 marbles and 2020 bags labeled $B_{1}, \\ldots, B_{2020}$ in which he randomly distributes the marbles (each marble is placed in a random bag independently). If $E$ the expected number of integers $1 \\leq i \\leq 2020$ such that $B_{i}$ has at least $i$ marbles, compute the closest integer to $1000E$.",
      "answer": "1000"
    },
    {
      "question": "Find the maximum possible number of diagonals of equal length in a convex hexagon.",
      "answer": "7"
    },
    {
      "question": "Let $n$ be the second smallest integer that can be written as the sum of two positive cubes in two different ways. Compute $n$.",
      "answer": "4104"
    },
    {
      "question": "What fraction of the area of a regular hexagon of side length 1 is within distance $\\frac{1}{2}$ of at least one of the vertices?",
      "answer": "\\pi \\sqrt{3} / 9"
    },
    {
      "question": "What is the smallest number of regular hexagons of side length 1 needed to completely cover a disc of radius 1 ?",
      "answer": "3"
    },
    {
      "question": "Steph Curry is playing the following game and he wins if he has exactly 5 points at some time. Flip a fair coin. If heads, shoot a 3-point shot which is worth 3 points. If tails, shoot a free throw which is worth 1 point. He makes \\frac{1}{2} of his 3-point shots and all of his free throws. Find the probability he will win the game. (Note he keeps flipping the coin until he has exactly 5 or goes over 5 points)",
      "answer": "\\frac{140}{243}"
    },
    {
      "question": "The UEFA Champions League playoffs is a 16-team soccer tournament in which Spanish teams always win against non-Spanish teams. In each of 4 rounds, each remaining team is randomly paired against one other team; the winner advances to the next round, and the loser is permanently knocked out of the tournament. If 3 of the 16 teams are Spanish, what is the probability that there are 2 Spanish teams in the final round?",
      "answer": "$\\frac{4}{5}$"
    },
    {
      "question": "In a cafeteria line, the number of people ahead of Kaukab is equal to two times the number of people behind her. There are $n$ people in the line. What is a possible value of $n$?",
      "answer": "25"
    },
    {
      "question": "On a blackboard a stranger writes the values of $s_{7}(n)^{2}$ for $n=0,1, \\ldots, 7^{20}-1$, where $s_{7}(n)$ denotes the sum of digits of $n$ in base 7 . Compute the average value of all the numbers on the board.",
      "answer": "3680"
    },
    {
      "question": "The numbers $1,2,\\ldots,64$ are written in the squares of an $8\\times 8$ chessboard, one number to each square. Then $2\\times 2$ tiles are placed on the chessboard (without overlapping) so that each tile covers exactly four squares whose numbers sum to less than $100$. Find, with proof, the maximum number of tiles that can be placed on the chessboard, and give an example of a distribution of the numbers $1,2,\\ldots,64$ into the squares of the chessboard that admits this maximum number of tiles.",
      "answer": "12"
    },
    {
      "question": "The function $f: \\mathbb{Z}^{2} \\rightarrow \\mathbb{Z}$ satisfies - $f(x, 0)=f(0, y)=0$, and - $f(x, y)=f(x-1, y)+f(x, y-1)+x+y$ for all nonnegative integers $x$ and $y$. Find $f(6,12)$.",
      "answer": "77500"
    },
    {
      "question": "The train schedule in Hummut is hopelessly unreliable. Train A will enter Intersection X from the west at a random time between 9:00 am and 2:30 pm; each moment in that interval is equally likely. Train B will enter the same intersection from the north at a random time between 9:30 am and 12:30 pm, independent of Train A; again, each moment in the interval is equally likely. If each train takes 45 minutes to clear the intersection, what is the probability of a collision today?",
      "answer": "\\frac{13}{48}"
    },
    {
      "question": "What is the last digit of $1^{1}+2^{2}+3^{3}+\\cdots+100^{100}$?",
      "answer": "0"
    },
    {
      "question": "For a finite graph $G$, let $f(G)$ be the number of triangles and $g(G)$ the number of tetrahedra formed by edges of $G$. Find the least constant $c$ such that \\[g(G)^3\\le c\\cdot f(G)^4\\] for every graph $G$.\n\n[i]",
      "answer": "\\frac{3}{32}"
    },
    {
      "question": "Suppose that $x$ and $y$ are real numbers that satisfy the two equations $3x+2y=6$ and $9x^2+4y^2=468$. What is the value of $xy$?",
      "answer": "-36"
    },
    {
      "question": "What is the probability that exactly one person gets their hat back when 6 people randomly pick hats?",
      "answer": "\\frac{11}{30}"
    },
    {
      "question": "The attached figure is an undirected graph. The circled numbers represent the nodes, and the numbers along the edges are their lengths (symmetrical in both directions). An Alibaba Hema Xiansheng carrier starts at point A and will pick up three orders from merchants B_{1}, B_{2}, B_{3} and deliver them to three customers C_{1}, C_{2}, C_{3}, respectively. The carrier drives a scooter with a trunk that holds at most two orders at any time. All the orders have equal size. Find the shortest travel route that starts at A and ends at the last delivery. To simplify this question, assume no waiting time during each pickup and delivery.",
      "answer": "16"
    },
    {
      "question": "The classrooms at MIT are each identified with a positive integer (with no leading zeroes). One day, as President Reif walks down the Infinite Corridor, he notices that a digit zero on a room sign has fallen off. Let $N$ be the original number of the room, and let $M$ be the room number as shown on the sign. The smallest interval containing all possible values of $\\frac{M}{N}$ can be expressed as $\\left[\\frac{a}{b}, \\frac{c}{d}\\right)$ where $a, b, c, d$ are positive integers with $\\operatorname{gcd}(a, b)=\\operatorname{gcd}(c, d)=1$. Compute $1000 a+100 b+10 c+d$.",
      "answer": "2031"
    },
    {
      "question": "Let $A B C D$ be a parallelogram with $A B=480, A D=200$, and $B D=625$. The angle bisector of $\\angle B A D$ meets side $C D$ at point $E$. Find $C E$.",
      "answer": "280"
    },
    {
      "question": "A [i]site[/i] is any point $(x, y)$ in the plane such that $x$ and $y$ are both positive integers less than or equal to 20.\n\nInitially, each of the 400 sites is unoccupied. Amy and Ben take turns placing stones with Amy going first. On her turn, Amy places a new red stone on an unoccupied site such that the distance between any two sites occupied by red stones is not equal to $\\sqrt{5}$. On his turn, Ben places a new blue stone on any unoccupied site. (A site occupied by a blue stone is allowed to be at any distance from any other occupied site.) They stop as soon as a player cannot place a stone.\n\nFind the greatest $K$ such that Amy can ensure that she places at least $K$ red stones, no matter how Ben places his blue stones.\n\n[i]",
      "answer": "100"
    },
    {
      "question": "A contest has six problems worth seven points each. On any given problem, a contestant can score either 0,1 , or 7 points. How many possible total scores can a contestant achieve over all six problems?",
      "answer": "28"
    },
    {
      "question": "Consider all functions $f: \\mathbb{Z} \\rightarrow \\mathbb{Z}$ satisfying $$f(f(x)+2 x+20)=15$$ Call an integer $n$ good if $f(n)$ can take any integer value. In other words, if we fix $n$, for any integer $m$, there exists a function $f$ such that $f(n)=m$. Find the sum of all good integers $x$.",
      "answer": "-35"
    },
    {
      "question": "If $x, y$, and $z$ are real numbers such that $2 x^{2}+y^{2}+z^{2}=2 x-4 y+2 x z-5$, find the maximum possible value of $x-y+z$.",
      "answer": "4"
    },
    {
      "question": "The line with equation $y=3x+6$ is reflected in the $y$-axis. What is the $x$-intercept of the new line?",
      "answer": "2"
    },
    {
      "question": "What is the smallest possible value of $n$ if a solid cube is made of white plastic and has dimensions $n \\times n \\times n$, the six faces of the cube are completely covered with gold paint, the cube is then cut into $n^{3}$ cubes, each of which has dimensions $1 \\times 1 \\times 1$, and the number of $1 \\times 1 \\times 1$ cubes with 0 gold faces is strictly greater than the number of $1 \\times 1 \\times 1$ cubes with exactly 1 gold face?",
      "answer": "9"
    },
    {
      "question": "While waiting for their food at a restaurant in Harvard Square, Ana and Banana draw 3 squares $\\square_{1}, \\square_{2}, \\square_{3}$ on one of their napkins. Starting with Ana, they take turns filling in the squares with integers from the set $\\{1,2,3,4,5\\}$ such that no integer is used more than once. Ana's goal is to minimize the minimum value $M$ that the polynomial $a_{1} x^{2}+a_{2} x+a_{3}$ attains over all real $x$, where $a_{1}, a_{2}, a_{3}$ are the integers written in $\\square_{1}, \\square_{2}, \\square_{3}$ respectively. Banana aims to maximize $M$. Assuming both play optimally, compute the final value of $100 a_{1}+10 a_{2}+a_{3}$.",
      "answer": "\\[ 541 \\]"
    },
    {
      "question": "If $x, y, k$ are positive reals such that $$3=k^{2}\\left(\\frac{x^{2}}{y^{2}}+\\frac{y^{2}}{x^{2}}\\right)+k\\left(\\frac{x}{y}+\\frac{y}{x}\\right)$$ find the maximum possible value of $k$.",
      "answer": "(-1+\\sqrt{7})/2"
    },
    {
      "question": "Find the total number of solutions to the equation $(a-b)(a+b)+(a-b)(c)=(a-b)(a+b+c)=2012$ where $a, b, c$ are positive integers.",
      "answer": "1755"
    },
    {
      "question": "Descartes's Blackjack: How many integer lattice points (points of the form $(m, n)$ for integers $m$ and $n$) lie inside or on the boundary of the disk of radius 2009 centered at the origin?",
      "answer": "12679605"
    },
    {
      "question": "Arturo has an equal number of $\\$5$ bills, of $\\$10$ bills, and of $\\$20$ bills. The total value of these bills is $\\$700$. How many $\\$5$ bills does Arturo have?",
      "answer": "20"
    },
    {
      "question": "For positive integers $n$, let $c_{n}$ be the smallest positive integer for which $n^{c_{n}}-1$ is divisible by 210, if such a positive integer exists, and $c_{n}=0$ otherwise. What is $c_{1}+c_{2}+\\cdots+c_{210}$?",
      "answer": "329"
    },
    {
      "question": "For the specific example $M=5$, find a value of $k$, not necessarily the smallest, such that $\\sum_{n=1}^{k} \\frac{1}{n}>M$. Justify your answer.",
      "answer": "256"
    },
    {
      "question": "Define a sequence $\\left\\{a_{n}\\right\\}$ by $a_{1}=1$ and $a_{n}=\\left(a_{n-1}\\right)!+1$ for every $n>1$. Find the least $n$ for which $a_{n}>10^{10}$.",
      "answer": "6"
    },
    {
      "question": "What is the smallest value that the sum of the digits of the number $3n^2+n+1,$ $n\\in\\mathbb{N}$ can take?",
      "answer": "3"
    },
    {
      "question": "A positive integer will be called \"sparkly\" if its smallest (positive) divisor, other than 1, equals the total number of divisors (including 1). How many of the numbers $2,3, \\ldots, 2003$ are sparkly?",
      "answer": "3"
    },
    {
      "question": "Let $a, b, c$ be the three roots of $p(x)=x^{3}+x^{2}-333 x-1001$. Find $a^{3}+b^{3}+c^{3}$.",
      "answer": "2003"
    },
    {
      "question": "What is the area of rectangle \\( PQRS \\) if the perimeter of rectangle \\( TVWY \\) is 60?",
      "answer": "600"
    },
    {
      "question": "In rectangle $PQRS$, $PS=6$ and $SR=3$. Point $U$ is on $QR$ with $QU=2$. Point $T$ is on $PS$ with $\\angle TUR=90^{\\circ}$. What is the length of $TR$?",
      "answer": "5"
    },
    {
      "question": "We have a calculator with two buttons that displays an integer $x$. Pressing the first button replaces $x$ by $\\left\\lfloor\\frac{x}{2}\\right\\rfloor$, and pressing the second button replaces $x$ by $4 x+1$. Initially, the calculator displays 0. How many integers less than or equal to 2014 can be achieved through a sequence of arbitrary button presses? (It is permitted for the number displayed to exceed 2014 during the sequence. Here, $\\lfloor y\\rfloor$ denotes the greatest integer less than or equal to the real number $y$.)",
      "answer": "233"
    },
    {
      "question": "For how many odd integers $k$ between 0 and 100 does the equation $2^{4m^{2}}+2^{m^{2}-n^{2}+4}=2^{k+4}+2^{3m^{2}+n^{2}+k}$ have exactly two pairs of positive integers $(m, n)$ that are solutions?",
      "answer": "18"
    },
    {
      "question": "Six soccer teams are competing in a tournament in Waterloo. Every team is to play three games, each against a different team. How many different schedules are possible?",
      "answer": "70"
    },
    {
      "question": "Start by writing the integers $1,2,4,6$ on the blackboard. At each step, write the smallest positive integer $n$ that satisfies both of the following properties on the board. - $n$ is larger than any integer on the board currently. - $n$ cannot be written as the sum of 2 distinct integers on the board. Find the 100-th integer that you write on the board. Recall that at the beginning, there are already 4 integers on the board.",
      "answer": "388"
    },
    {
      "question": "A $5 \\times 5$ grid of squares is filled with integers. Call a rectangle corner-odd if its sides are grid lines and the sum of the integers in its four corners is an odd number. What is the maximum possible number of corner-odd rectangles within the grid?",
      "answer": "60"
    },
    {
      "question": "Find all integers $n$ satisfying $n \\geq 2$ and $\\dfrac{\\sigma(n)}{p(n)-1} = n$, in which $\\sigma(n)$ denotes the sum of all positive divisors of $n$, and $p(n)$ denotes the largest prime divisor of $n$.",
      "answer": "6"
    },
    {
      "question": "Consider the paths from \\((0,0)\\) to \\((6,3)\\) that only take steps of unit length up and right. Compute the sum of the areas bounded by the path, the \\(x\\)-axis, and the line \\(x=6\\) over all such paths.",
      "answer": "756"
    },
    {
      "question": "A circle $\\omega_{1}$ of radius 15 intersects a circle $\\omega_{2}$ of radius 13 at points $P$ and $Q$. Point $A$ is on line $P Q$ such that $P$ is between $A$ and $Q$. $R$ and $S$ are the points of tangency from $A$ to $\\omega_{1}$ and $\\omega_{2}$, respectively, such that the line $A S$ does not intersect $\\omega_{1}$ and the line $A R$ does not intersect $\\omega_{2}$. If $P Q=24$ and $\\angle R A S$ has a measure of $90^{\\circ}$, compute the length of $A R$.",
      "answer": "14+\\sqrt{97}"
    },
    {
      "question": "Let $\\omega$ be a circle, and let $ABCD$ be a quadrilateral inscribed in $\\omega$. Suppose that $BD$ and $AC$ intersect at a point $E$. The tangent to $\\omega$ at $B$ meets line $AC$ at a point $F$, so that $C$ lies between $E$ and $F$. Given that $AE=6, EC=4, BE=2$, and $BF=12$, find $DA$.",
      "answer": "2 \\sqrt{42}"
    },
    {
      "question": "Matilda has a summer job delivering newspapers. She earns \\$6.00 an hour plus \\$0.25 per newspaper delivered. Matilda delivers 30 newspapers per hour. How much money will she earn during a 3-hour shift?",
      "answer": "\\$40.50"
    },
    {
      "question": "The polynomial \\( f(x)=x^{2007}+17 x^{2006}+1 \\) has distinct zeroes \\( r_{1}, \\ldots, r_{2007} \\). A polynomial \\( P \\) of degree 2007 has the property that \\( P\\left(r_{j}+\\frac{1}{r_{j}}\\right)=0 \\) for \\( j=1, \\ldots, 2007 \\). Determine the value of \\( P(1) / P(-1) \\).",
      "answer": "289/259"
    },
    {
      "question": "For a positive integer $n$, let $\\theta(n)$ denote the number of integers $0 \\leq x<2010$ such that $x^{2}-n$ is divisible by 2010. Determine the remainder when $\\sum_{n=0}^{2009} n \\cdot \\theta(n)$ is divided by 2010.",
      "answer": "335"
    },
    {
      "question": "The digits in a two-digit positive integer are reversed. The new two-digit integer minus the original integer equals 54. What is the positive difference between the two digits of the original integer?",
      "answer": "6"
    },
    {
      "question": "A ball inside a rectangular container of width 7 and height 12 is launched from the lower-left vertex of the container. It first strikes the right side of the container after traveling a distance of $\\sqrt{53}$ (and strikes no other sides between its launch and its impact with the right side). Find the height at which the ball first contacts the right side.",
      "answer": "2"
    },
    {
      "question": "Suppose \\(x\\) and \\(y\\) are positive real numbers such that \\(x+\\frac{1}{y}=y+\\frac{2}{x}=3\\). Compute the maximum possible value of \\(xy\\).",
      "answer": "3+\\sqrt{7}"
    },
    {
      "question": "The value of $\\frac{x}{2}$ is less than the value of $x^{2}$. The value of $x^{2}$ is less than the value of $x$. Which of the following could be a value of $x$?",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "Let $W$ be the hypercube $\\left\\{\\left(x_{1}, x_{2}, x_{3}, x_{4}\\right) \\mid 0 \\leq x_{1}, x_{2}, x_{3}, x_{4} \\leq 1\\right\\}$. The intersection of $W$ and a hyperplane parallel to $x_{1}+x_{2}+x_{3}+x_{4}=0$ is a non-degenerate 3-dimensional polyhedron. What is the maximum number of faces of this polyhedron?",
      "answer": "8"
    },
    {
      "question": "An ant starts at the origin of a coordinate plane. Each minute, it either walks one unit to the right or one unit up, but it will never move in the same direction more than twice in the row. In how many different ways can it get to the point $(5,5)$ ?",
      "answer": "84"
    },
    {
      "question": "Let $G$ be the number of Google hits of \"guts round\" at 10:31PM on October 31, 2011. Let $B$ be the number of Bing hits of \"guts round\" at the same time. Determine $B / G$. Your score will be $$\\max (0,\\left\\lfloor 20\\left(1-\\frac{20|a-k|}{k}\\right)\\right\\rfloor)$$ where $k$ is the actual answer and $a$ is your answer.",
      "answer": ".82721"
    },
    {
      "question": "Let $x$ and $y$ be non-negative real numbers that sum to 1. Compute the number of ordered pairs $(a, b)$ with $a, b \\in\\{0,1,2,3,4\\}$ such that the expression $x^{a} y^{b}+y^{a} x^{b}$ has maximum value $2^{1-a-b}$.",
      "answer": "17"
    },
    {
      "question": "Let $S(x)$ denote the sum of the digits of a positive integer $x$. Find the maximum possible value of $S(x+2019)-S(x)$.",
      "answer": "12"
    },
    {
      "question": "A particular coin can land on heads $(H)$, on tails $(T)$, or in the middle $(M)$, each with probability $\\frac{1}{3}$. Find the expected number of flips necessary to observe the contiguous sequence HMMTHMMT...HMMT, where the sequence HMMT is repeated 2016 times.",
      "answer": "\\frac{3^{8068}-81}{80}"
    },
    {
      "question": "A line with a slope of 2 and a line with a slope of -4 each have a $y$-intercept of 6. What is the distance between the $x$-intercepts of these lines?",
      "answer": "\\frac{9}{2}"
    },
    {
      "question": "Herbert rolls 6 fair standard dice and computes the product of all of his rolls. If the probability that the product is prime can be expressed as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
      "answer": "2692"
    },
    {
      "question": "An ordered list of four numbers is called a quadruple. A quadruple $(p, q, r, s)$ of integers with $p, q, r, s \\geq 0$ is chosen at random such that $2 p+q+r+s=4$. What is the probability that $p+q+r+s=3$?",
      "answer": "\\frac{3}{11}"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=3, B C=4$, and $C A=5$. Let $A_{1}, A_{2}$ be points on side $B C$, $B_{1}, B_{2}$ be points on side $C A$, and $C_{1}, C_{2}$ be points on side $A B$. Suppose that there exists a point $P$ such that $P A_{1} A_{2}, P B_{1} B_{2}$, and $P C_{1} C_{2}$ are congruent equilateral triangles. Find the area of convex hexagon $A_{1} A_{2} B_{1} B_{2} C_{1} C_{2}$.",
      "answer": "\\frac{12+22 \\sqrt{3}}{15}"
    },
    {
      "question": "A polynomial $P$ of degree 2015 satisfies the equation $P(n)=\\frac{1}{n^{2}}$ for $n=1,2, \\ldots, 2016$. Find \\lfloor 2017 P(2017)\\rfloor.",
      "answer": "-9"
    },
    {
      "question": "In a photograph, Aristotle, David, Flora, Munirah, and Pedro are seated in a random order in a row of 5 chairs. If David is seated in the middle of the row, what is the probability that Pedro is seated beside him?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Let \\{a_{n}\\}_{n \\geq 1}$ be an arithmetic sequence and \\{g_{n}\\}_{n \\geq 1}$ be a geometric sequence such that the first four terms of \\{a_{n}+g_{n}\\}$ are $0,0,1$, and 0 , in that order. What is the 10th term of \\{a_{n}+g_{n}\\}$ ?",
      "answer": "-54"
    },
    {
      "question": "The real numbers $x, y, z$ satisfy $0 \\leq x \\leq y \\leq z \\leq 4$. If their squares form an arithmetic progression with common difference 2, determine the minimum possible value of $|x-y|+|y-z|$.",
      "answer": "4-2\\sqrt{3}"
    },
    {
      "question": "In preparation for a game of Fish, Carl must deal 48 cards to 6 players. For each card that he deals, he runs through the entirety of the following process: 1. He gives a card to a random player. 2. A player Z is randomly chosen from the set of players who have at least as many cards as every other player (i.e. Z has the most cards or is tied for having the most cards). 3. A player D is randomly chosen from the set of players other than Z who have at most as many cards as every other player (i.e. D has the fewest cards or is tied for having the fewest cards). 4. Z gives one card to D. He repeats steps 1-4 for each card dealt, including the last card. After all the cards have been dealt, what is the probability that each player has exactly 8 cards?",
      "answer": "\\frac{5}{6}"
    },
    {
      "question": "A wealthy king has his blacksmith fashion him a large cup, whose inside is a cone of height 9 inches and base diameter 6 inches. At one of his many feasts, he orders the mug to be filled to the brim with cranberry juice. For each positive integer $n$, the king stirs his drink vigorously and takes a sip such that the height of fluid left in his cup after the sip goes down by $\\frac{1}{n^{2}}$ inches. Shortly afterwards, while the king is distracted, the court jester adds pure Soylent to the cup until it's once again full. The king takes sips precisely every minute, and his first sip is exactly one minute after the feast begins. As time progresses, the amount of juice consumed by the king (in cubic inches) approaches a number $r$. Find $r$.",
      "answer": "\\frac{216 \\pi^{3}-2187 \\sqrt{3}}{8 \\pi^{2}}"
    },
    {
      "question": "In triangle $A B C$ with altitude $A D, \\angle B A C=45^{\\circ}, D B=3$, and $C D=2$. Find the area of triangle $A B C$.",
      "answer": "15"
    },
    {
      "question": "There are 2008 distinct points on a circle. If you connect two of these points to form a line and then connect another two points (distinct from the first two) to form another line, what is the probability that the two lines intersect inside the circle?",
      "answer": "1/3"
    },
    {
      "question": "Simplify $\frac{1}{2+\frac{2}{3}}$.",
      "answer": "\\frac{3}{8}"
    },
    {
      "question": "Let $A=\\{a_{1}, b_{1}, a_{2}, b_{2}, \\ldots, a_{10}, b_{10}\\}$, and consider the 2-configuration $C$ consisting of \\( \\{a_{i}, b_{i}\\} \\) for all \\( 1 \\leq i \\leq 10, \\{a_{i}, a_{i+1}\\} \\) for all \\( 1 \\leq i \\leq 9 \\), and \\( \\{b_{i}, b_{i+1}\\} \\) for all \\( 1 \\leq i \\leq 9 \\). Find the number of subsets of $C$ that are consistent of order 1.",
      "answer": "89"
    },
    {
      "question": "In some squares of a $2012\\times 2012$ grid there are some beetles, such that no square contain more than one beetle. At one moment, all the beetles fly off the grid and then land on the grid again, also satisfying the condition that there is at most one beetle standing in each square. The vector from the centre of the square from which a beetle $B$ flies to the centre of the square on which it lands is called the [i]translation vector[/i] of beetle $B$. \nFor all possible starting and ending configurations, find the maximum length of the sum of the  [i]translation vectors[/i] of all beetles.",
      "answer": "\\frac{2012^3}{4}"
    },
    {
      "question": "In \\(\\triangle ABC\\), the external angle bisector of \\(\\angle BAC\\) intersects line \\(BC\\) at \\(D\\). \\(E\\) is a point on ray \\(\\overrightarrow{AC}\\) such that \\(\\angle BDE=2 \\angle ADB\\). If \\(AB=10, AC=12\\), and \\(CE=33\\), compute \\(\\frac{DB}{DE}\\).",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "Let $x, y, z$ be real numbers satisfying $$\\begin{aligned} 2 x+y+4 x y+6 x z & =-6 \\\\ y+2 z+2 x y+6 y z & =4 \\\\ x-z+2 x z-4 y z & =-3 \\end{aligned}$$ Find $x^{2}+y^{2}+z^{2}$.",
      "answer": "29"
    },
    {
      "question": "A group of 101 Dalmathians participate in an election, where they each vote independently on either candidate \\(A\\) or \\(B\\) with equal probability. If \\(X\\) Dalmathians voted for the winning candidate, the expected value of \\(X^{2}\\) can be expressed as \\(\\frac{a}{b}\\) for positive integers \\(a, b\\) with \\(\\operatorname{gcd}(a, b)=1\\). Find the unique positive integer \\(k \\leq 103\\) such that \\(103 \\mid a-bk\\).",
      "answer": "51"
    },
    {
      "question": "Consider an equilateral triangular grid $G$ with 20 points on a side, where each row consists of points spaced 1 unit apart. More specifically, there is a single point in the first row, two points in the second row, ..., and 20 points in the last row, for a total of 210 points. Let $S$ be a closed non-selfintersecting polygon which has 210 vertices, using each point in $G$ exactly once. Find the sum of all possible values of the area of $S$.",
      "answer": "52 \\sqrt{3}"
    },
    {
      "question": "How many ways are there to label the faces of a regular octahedron with the integers 18, using each exactly once, so that any two faces that share an edge have numbers that are relatively prime? Physically realizable rotations are considered indistinguishable, but physically unrealizable reflections are considered different.",
      "answer": "12"
    },
    {
      "question": "Alex picks his favorite point $(x, y)$ in the first quadrant on the unit circle $x^{2}+y^{2}=1$, such that a ray from the origin through $(x, y)$ is $\\theta$ radians counterclockwise from the positive $x$-axis. He then computes $\\cos ^{-1}\\left(\\frac{4 x+3 y}{5}\\right)$ and is surprised to get $\\theta$. What is $\\tan (\\theta)$?",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Dorothea has a $3 \\times 4$ grid of dots. She colors each dot red, blue, or dark gray. Compute the number of ways Dorothea can color the grid such that there is no rectangle whose sides are parallel to the grid lines and whose vertices all have the same color.",
      "answer": "284688"
    },
    {
      "question": "Let $a_{0}, a_{1}, \\ldots$ and $b_{0}, b_{1}, \\ldots$ be geometric sequences with common ratios $r_{a}$ and $r_{b}$, respectively, such that $$\\sum_{i=0}^{\\infty} a_{i}=\\sum_{i=0}^{\\infty} b_{i}=1 \\quad \\text { and } \\quad\\left(\\sum_{i=0}^{\\infty} a_{i}^{2}\\right)\\left(\\sum_{i=0}^{\\infty} b_{i}^{2}\\right)=\\sum_{i=0}^{\\infty} a_{i} b_{i}$$ Find the smallest real number $c$ such that $a_{0}<c$ must be true.",
      "answer": "\\frac{4}{3}"
    },
    {
      "question": "An icosidodecahedron is a convex polyhedron with 20 triangular faces and 12 pentagonal faces. How many vertices does it have?",
      "answer": "30"
    },
    {
      "question": "Horizontal parallel segments $A B=10$ and $C D=15$ are the bases of trapezoid $A B C D$. Circle $\\gamma$ of radius 6 has center within the trapezoid and is tangent to sides $A B, B C$, and $D A$. If side $C D$ cuts out an arc of $\\gamma$ measuring $120^{\\circ}$, find the area of $A B C D$.",
      "answer": "\\frac{225}{2}"
    },
    {
      "question": "Let $n$ be a positive integer. Given that $n^{n}$ has 861 positive divisors, find $n$.",
      "answer": "20"
    },
    {
      "question": "Two fair coins are simultaneously flipped. This is done repeatedly until at least one of the coins comes up heads, at which point the process stops. What is the probability that the other coin also came up heads on this last flip?",
      "answer": "1/3"
    },
    {
      "question": "Rahul has ten cards face-down, which consist of five distinct pairs of matching cards. During each move of his game, Rahul chooses one card to turn face-up, looks at it, and then chooses another to turn face-up and looks at it. If the two face-up cards match, the game ends. If not, Rahul flips both cards face-down and keeps repeating this process. Initially, Rahul doesn't know which cards are which. Assuming that he has perfect memory, find the smallest number of moves after which he can guarantee that the game has ended.",
      "answer": "4"
    },
    {
      "question": "If a line segment joins the points $(-9,-2)$ and $(6,8)$, how many points on the line segment have coordinates that are both integers?",
      "answer": "6"
    },
    {
      "question": "Suppose that $a,b,c,d$ are positive real numbers satisfying $(a+c)(b+d)=ac+bd$. Find the smallest possible value of\n$$\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{d}+\\frac{d}{a}.$$\n[i]Israel[/i]",
      "answer": "8"
    },
    {
      "question": "A jar contains 8 red balls and 2 blue balls. Every minute, a ball is randomly removed. The probability that there exists a time during this process where there are more blue balls than red balls in the jar can be expressed as $\\frac{a}{b}$ for relatively prime integers $a$ and $b$. Compute $100 a+b$.",
      "answer": "209"
    },
    {
      "question": "Find the smallest $n$ such that $n$! ends in 290 zeroes.",
      "answer": "1170"
    },
    {
      "question": "There are four people in a room. For every two people, there is a $50 \\%$ chance that they are friends. Two people are connected if they are friends, or a third person is friends with both of them, or they have different friends who are friends of each other. What is the probability that every pair of people in this room is connected?",
      "answer": "\\frac{19}{32}"
    },
    {
      "question": "Compute \\(\\sum_{k=0}^{100}\\left\\lfloor\\frac{2^{100}}{2^{50}+2^{k}}\\right\\rfloor\\). (Here, if \\(x\\) is a real number, then \\(\\lfloor x\\rfloor\\) denotes the largest integer less than or equal to \\(x\\).)",
      "answer": "101 \\cdot 2^{49}-50"
    },
    {
      "question": "Alice and Bob take turns removing balls from a bag containing 10 black balls and 10 white balls, with Alice going first. Alice always removes a black ball if there is one, while Bob removes one of the remaining balls uniformly at random. Once all balls have been removed, the expected number of black balls which Bob has can be expressed as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
      "answer": "4519"
    },
    {
      "question": "The time on a cell phone is $3:52$. How many minutes will pass before the phone next shows a time using each of the digits 2, 3, and 5 exactly once?",
      "answer": "91"
    },
    {
      "question": "A sequence of 11 positive real numbers, $a_{1}, a_{2}, a_{3}, \\ldots, a_{11}$, satisfies $a_{1}=4$ and $a_{11}=1024$ and $a_{n}+a_{n-1}=\\frac{5}{2} \\sqrt{a_{n} \\cdot a_{n-1}}$ for every integer $n$ with $2 \\leq n \\leq 11$. For example when $n=7, a_{7}+a_{6}=\\frac{5}{2} \\sqrt{a_{7} \\cdot a_{6}}$. There are $S$ such sequences. What are the rightmost two digits of $S$?",
      "answer": "20"
    },
    {
      "question": "If a bag contains only green, yellow, and red marbles in the ratio $3: 4: 2$ and 63 of the marbles are not red, how many red marbles are in the bag?",
      "answer": "18"
    },
    {
      "question": "When 100 is divided by a positive integer $x$, the remainder is 10. When 1000 is divided by $x$, what is the remainder?",
      "answer": "10"
    },
    {
      "question": "Let $A B C$ be an acute scalene triangle with circumcenter $O$ and centroid $G$. Given that $A G O$ is a right triangle, $A O=9$, and $B C=15$, let $S$ be the sum of all possible values for the area of triangle $A G O$. Compute $S^{2}$.",
      "answer": "288"
    },
    {
      "question": "Let $S$ be the set of points $(a, b)$ with $0 \\leq a, b \\leq 1$ such that the equation $x^{4}+a x^{3}-b x^{2}+a x+1=0$ has at least one real root. Determine the area of the graph of $S$.",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "What is the tens digit of the smallest positive integer that is divisible by each of 20, 16, and 2016?",
      "answer": "8"
    },
    {
      "question": "What is the median of the numbers in the list $19^{20}, \\frac{20}{19}, 20^{19}, 2019, 20 \\times 19$?",
      "answer": "2019"
    },
    {
      "question": "If \\( 10^{x} \\cdot 10^{5}=100^{4} \\), what is the value of \\( x \\)?",
      "answer": "3"
    },
    {
      "question": "The numbers $2^{0}, 2^{1}, \\cdots, 2^{15}, 2^{16}=65536$ are written on a blackboard. You repeatedly take two numbers on the blackboard, subtract one from the other, erase them both, and write the result of the subtraction on the blackboard. What is the largest possible number that can remain on the blackboard when there is only one number left?",
      "answer": "131069"
    },
    {
      "question": "We want to design a new chess piece, the American, with the property that (i) the American can never attack itself, and (ii) if an American $A_{1}$ attacks another American $A_{2}$, then $A_{2}$ also attacks $A_{1}$. Let $m$ be the number of squares that an American attacks when placed in the top left corner of an 8 by 8 chessboard. Let $n$ be the maximal number of Americans that can be placed on the 8 by 8 chessboard such that no Americans attack each other, if one American must be in the top left corner. Find the largest possible value of $m n$.",
      "answer": "1024"
    },
    {
      "question": "The real function $f$ has the property that, whenever $a, b, n$ are positive integers such that $a+b=2^{n}$, the equation $f(a)+f(b)=n^{2}$ holds. What is $f(2002)$?",
      "answer": "96"
    },
    {
      "question": "Anders is solving a math problem, and he encounters the expression $\\sqrt{15!}$. He attempts to simplify this radical by expressing it as $a \\sqrt{b}$ where $a$ and $b$ are positive integers. The sum of all possible distinct values of $ab$ can be expressed in the form $q \\cdot 15!$ for some rational number $q$. Find $q$.",
      "answer": "4"
    },
    {
      "question": "Let $a_0,a_1,a_2,\\dots $ be a sequence of real numbers such that $a_0=0, a_1=1,$ and for every $n\\geq 2$ there exists $1 \\leq k \\leq n$ satisfying \\[ a_n=\\frac{a_{n-1}+\\dots + a_{n-k}}{k}. \\]Find the maximum possible value of $a_{2018}-a_{2017}$.",
      "answer": "\\frac{2016}{2017^2}"
    },
    {
      "question": "The list $p, q, r, s$ consists of four consecutive integers listed in increasing order. If $p + s = 109$, what is the value of $q + r$?",
      "answer": "109"
    },
    {
      "question": "If $2x-3=10$, what is the value of $4x$?",
      "answer": "26"
    },
    {
      "question": "Svitlana writes the number 147 on a blackboard. Then, at any point, if the number on the blackboard is $n$, she can perform one of the following three operations: - if $n$ is even, she can replace $n$ with $\\frac{n}{2}$; - if $n$ is odd, she can replace $n$ with $\\frac{n+255}{2}$; and - if $n \\geq 64$, she can replace $n$ with $n-64$. Compute the number of possible values that Svitlana can obtain by doing zero or more operations.",
      "answer": "163"
    },
    {
      "question": "Let $\\mathbb{Q}$ be the set of rational numbers. A function $f: \\mathbb{Q} \\to \\mathbb{Q}$ is called aquaesulian if the following property holds: for every $x,y \\in \\mathbb{Q}$,\n\\[ f(x+f(y)) = f(x) + y \\quad \\text{or} \\quad f(f(x)+y) = x + f(y). \\]\nShow that there exists an integer $c$ such that for any aquaesulian function $f$ there are at most $c$ different rational numbers of the form $f(r) + f(-r)$ for some rational number $r$, and find the smallest possible value of $c$.",
      "answer": "1"
    },
    {
      "question": "There are $N$ lockers, labeled from 1 to $N$, placed in clockwise order around a circular hallway. Initially, all lockers are open. Ansoon starts at the first locker and always moves clockwise. When she is at locker $n$ and there are more than $n$ open lockers, she keeps locker $n$ open and closes the next $n$ open lockers, then repeats the process with the next open locker. If she is at locker $n$ and there are at most $n$ lockers still open, she keeps locker $n$ open and closes all other lockers. She continues this process until only one locker is left open. What is the smallest integer $N>2021$ such that the last open locker is locker 1?",
      "answer": "2046"
    },
    {
      "question": "When three positive integers are added in pairs, the resulting sums are 998, 1050, and 1234. What is the difference between the largest and smallest of the three original positive integers?",
      "answer": "236"
    },
    {
      "question": "Points $G$ and $N$ are chosen on the interiors of sides $E D$ and $D O$ of unit square $D O M E$, so that pentagon GNOME has only two distinct side lengths. The sum of all possible areas of quadrilateral $N O M E$ can be expressed as $\\frac{a-b \\sqrt{c}}{d}$, where $a, b, c, d$ are positive integers such that $\\operatorname{gcd}(a, b, d)=1$ and $c$ is square-free (i.e. no perfect square greater than 1 divides $c$ ). Compute $1000 a+100 b+10 c+d$.",
      "answer": "10324"
    },
    {
      "question": "The roots of $z^{6}+z^{4}+z^{2}+1=0$ are the vertices of a convex polygon in the complex plane. Find the sum of the squares of the side lengths of the polygon.",
      "answer": "12-4 \\sqrt{2}"
    },
    {
      "question": "If $a, b, c>0$, what is the smallest possible value of $\\left\\lfloor\\frac{a+b}{c}\\right\\rfloor+\\left\\lfloor\\frac{b+c}{a}\\right\\rfloor+\\left\\lfloor\\frac{c+a}{b}\\right\\rfloor$? (Note that $\\lfloor x\\rfloor$ denotes the greatest integer less than or equal to $x$.)",
      "answer": "4"
    },
    {
      "question": "A real number $x$ is chosen uniformly at random from the interval $(0,10)$. Compute the probability that $\\sqrt{x}, \\sqrt{x+7}$, and $\\sqrt{10-x}$ are the side lengths of a non-degenerate triangle.",
      "answer": "\\frac{22}{25}"
    },
    {
      "question": "Let $x$ and $y$ be positive real numbers such that $x^{2}+y^{2}=1$ and \\left(3 x-4 x^{3}\\right)\\left(3 y-4 y^{3}\\right)=-\\frac{1}{2}$. Compute $x+y$.",
      "answer": "\\frac{\\sqrt{6}}{2}"
    },
    {
      "question": "Mike has two containers. One container is a rectangular prism with width 2 cm, length 4 cm, and height 10 cm. The other is a right cylinder with radius 1 cm and height 10 cm. Both containers sit on a flat surface. Water has been poured into the two containers so that the height of the water in both containers is the same. If the combined volume of the water in the two containers is $80 \\mathrm{~cm}^{3}$, what is the height of the water in each container?",
      "answer": "7.2"
    },
    {
      "question": "If $3^{2x}=64$, what is the value of $3^{-x}$?",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "Solve in positive real numbers: $n+ \\lfloor \\sqrt{n} \\rfloor+\\lfloor \\sqrt[3]{n} \\rfloor=2014$",
      "answer": "1958"
    },
    {
      "question": "In $\\triangle Q R S$, point $T$ is on $Q S$ with $\\angle Q R T=\\angle S R T$. Suppose that $Q T=m$ and $T S=n$ for some integers $m$ and $n$ with $n>m$ and for which $n+m$ is a multiple of $n-m$. Suppose also that the perimeter of $\\triangle Q R S$ is $p$ and that the number of possible integer values for $p$ is $m^{2}+2 m-1$. What is the value of $n-m$?",
      "answer": "4"
    },
    {
      "question": "Let $a_{1}=1$, and let $a_{n}=\\left\\lfloor n^{3} / a_{n-1}\\right\\rfloor$ for $n>1$. Determine the value of $a_{999}$.",
      "answer": "999"
    },
    {
      "question": "Tac is dressing his cat to go outside. He has four indistinguishable socks, four indistinguishable shoes, and 4 indistinguishable show-shoes. In a hurry, Tac randomly pulls pieces of clothing out of a door and tries to put them on a random one of his cat's legs; however, Tac never tries to put more than one of each type of clothing on each leg of his cat. What is the probability that, after Tac is done, the snow-shoe on each of his cat's legs is on top of the shoe, which is on top of the sock?",
      "answer": "$\\frac{1}{1296}$"
    },
    {
      "question": "Suppose $m$ and $n$ are positive integers for which the sum of the first $m$ multiples of $n$ is 120, and the sum of the first $m^{3}$ multiples of $n^{3}$ is 4032000. Determine the sum of the first $m^{2}$ multiples of $n^{2}$.",
      "answer": "20800"
    },
    {
      "question": "An up-right path from $(a, b) \\in \\mathbb{R}^{2}$ to $(c, d) \\in \\mathbb{R}^{2}$ is a finite sequence $(x_{1}, y_{1}), \\ldots,(x_{k}, y_{k})$ of points in $\\mathbb{R}^{2}$ such that $(a, b)=(x_{1}, y_{1}),(c, d)=(x_{k}, y_{k})$, and for each $1 \\leq i<k$ we have that either $(x_{i+1}, y_{i+1})=(x_{i}+1, y_{i})$ or $(x_{i+1}, y_{i+1})=(x_{i}, y_{i}+1)$. Let $S$ be the set of all up-right paths from $(-400,-400)$ to $(400,400)$. What fraction of the paths in $S$ do not contain any point $(x, y)$ such that $|x|,|y| \\leq 10$? Express your answer as a decimal number between 0 and 1.",
      "answer": "0.2937156494680644"
    },
    {
      "question": "How many pairs $(x, y)$ of non-negative integers with $0 \\leq x \\leq y$ satisfy the equation $5x^{2}-4xy+2x+y^{2}=624$?",
      "answer": "7"
    },
    {
      "question": "Calculate the value of the expression $2 \\times 0 + 2 \\times 4$.",
      "answer": "8"
    },
    {
      "question": "Given that $a,b,c,d,e$ are real numbers such that\n$a+b+c+d+e=8$ ,\n$a^2+b^2+c^2+d^2+e^2=16$ .\nDetermine the maximum value of $e$ .",
      "answer": "\\[\n\\frac{16}{5}\n\\]"
    },
    {
      "question": "Seven students shared the cost of a $\\$26.00$ pizza. Each student paid either $\\$3.71$ or $\\$3.72$. How many students paid $\\$3.72$?",
      "answer": "3"
    },
    {
      "question": "Find the number of triples of sets $(A, B, C)$ such that: (a) $A, B, C \\subseteq\\{1,2,3, \\ldots, 8\\}$. (b) $|A \\cap B|=|B \\cap C|=|C \\cap A|=2$. (c) $|A|=|B|=|C|=4$. Here, $|S|$ denotes the number of elements in the set $S$.",
      "answer": "45360"
    },
    {
      "question": "Begining at a vertex, an ant crawls between the vertices of a regular octahedron. After reaching a vertex, it randomly picks a neighboring vertex (sharing an edge) and walks to that vertex along the adjoining edge (with all possibilities equally likely.) What is the probability that after walking along 2006 edges, the ant returns to the vertex where it began?",
      "answer": "\\frac{2^{2005}+1}{3 \\cdot 2^{2006}}"
    },
    {
      "question": "Compute the number of real solutions $(x, y, z, w)$ to the system of equations: $$\\begin{array}{rlrl} x & =z+w+z w x & z & =x+y+x y z \\\\ y & =w+x+w x y & w & =y+z+y z w \\end{array}$$",
      "answer": "5"
    },
    {
      "question": "Express, as concisely as possible, the value of the product $$\\left(0^{3}-350\\right)\\left(1^{3}-349\\right)\\left(2^{3}-348\\right)\\left(3^{3}-347\\right) \\cdots\\left(349^{3}-1\\right)\\left(350^{3}-0\\right)$$",
      "answer": "0"
    },
    {
      "question": "Find the number of integers $n$ such that $$ 1+\\left\\lfloor\\frac{100 n}{101}\\right\\rfloor=\\left\\lceil\\frac{99 n}{100}\\right\\rceil $$",
      "answer": "10100"
    },
    {
      "question": "Let $x$ and $y$ be positive real numbers and $\\theta$ an angle such that $\\theta \\neq \\frac{\\pi}{2} n$ for any integer $n$. Suppose $$\\frac{\\sin \\theta}{x}=\\frac{\\cos \\theta}{y}$$ and $$\\frac{\\cos ^{4} \\theta}{x^{4}}+\\frac{\\sin ^{4} \\theta}{y^{4}}=\\frac{97 \\sin 2 \\theta}{x^{3} y+y^{3} x}$$ Compute $\\frac{x}{y}+\\frac{y}{x}$.",
      "answer": "4"
    },
    {
      "question": "Let $ABC$ be an acute triangle and let $M$ be the midpoint of $AC$. A circle $\\omega$ passing through $B$ and $M$ meets the sides $AB$ and $BC$ at points $P$ and $Q$ respectively. Let $T$ be the point such that $BPTQ$ is a parallelogram. Suppose that $T$ lies on the circumcircle of $ABC$. Determine all possible values of $\\frac{BT}{BM}$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "Natalie has a copy of the unit interval $[0,1]$ that is colored white. She also has a black marker, and she colors the interval in the following manner: at each step, she selects a value $x \\in[0,1]$ uniformly at random, and (a) If $x \\leq \\frac{1}{2}$ she colors the interval $[x, x+\\frac{1}{2}]$ with her marker. (b) If $x>\\frac{1}{2}$ she colors the intervals $[x, 1]$ and $[0, x-\\frac{1}{2}]$ with her marker. What is the expected value of the number of steps Natalie will need to color the entire interval black?",
      "answer": "5"
    },
    {
      "question": "Bob is writing a sequence of letters of the alphabet, each of which can be either uppercase or lowercase, according to the following two rules: If he had just written an uppercase letter, he can either write the same letter in lowercase after it, or the next letter of the alphabet in uppercase. If he had just written a lowercase letter, he can either write the same letter in uppercase after it, or the preceding letter of the alphabet in lowercase. For instance, one such sequence is $a A a A B C D d c b B C$. How many sequences of 32 letters can he write that start at (lowercase) $a$ and end at (lowercase) $z$?",
      "answer": "376"
    },
    {
      "question": "Ewan writes out a sequence where he counts by 11s starting at 3. The resulting sequence is $3, 14, 25, 36, \\ldots$. What is a number that will appear in Ewan's sequence?",
      "answer": "113"
    },
    {
      "question": "Find the least possible area of a convex set in the plane that intersects both branches of the hyperbola $xy = 1$ and both branches of the hyperbola $xy = -1$. (A set $S$ in the plane is called \\emph{convex} if for any two points in $S$ the line segment connecting them is contained in $S$.)",
      "answer": "4"
    },
    {
      "question": "An isosceles trapezoid $A B C D$ with bases $A B$ and $C D$ has $A B=13, C D=17$, and height 3. Let $E$ be the intersection of $A C$ and $B D$. Circles $\\Omega$ and $\\omega$ are circumscribed about triangles $A B E$ and $C D E$. Compute the sum of the radii of $\\Omega$ and $\\omega$.",
      "answer": "39"
    },
    {
      "question": "Isabella writes the expression $\\sqrt{d}$ for each positive integer $d$ not exceeding 8 ! on the board. Seeing that these expressions might not be worth points on HMMT, Vidur simplifies each expression to the form $a \\sqrt{b}$, where $a$ and $b$ are integers such that $b$ is not divisible by the square of a prime number. (For example, $\\sqrt{20}, \\sqrt{16}$, and $\\sqrt{6}$ simplify to $2 \\sqrt{5}, 4 \\sqrt{1}$, and $1 \\sqrt{6}$, respectively.) Compute the sum of $a+b$ across all expressions that Vidur writes.",
      "answer": "534810086"
    },
    {
      "question": "Last Thursday, each of the students in M. Fermat's class brought one piece of fruit to school. Each brought an apple, a banana, or an orange. In total, $20\\%$ of the students brought an apple and $35\\%$ brought a banana. If 9 students brought oranges, how many students were in the class?",
      "answer": "20"
    },
    {
      "question": "Roger the ant is traveling on a coordinate plane, starting at $(0,0)$. Every second, he moves from one lattice point to a different lattice point at distance 1, chosen with equal probability. He will continue to move until he reaches some point $P$ for which he could have reached $P$ more quickly had he taken a different route. For example, if he goes from $(0,0)$ to $(1,0)$ to $(1,1)$ to $(1,2)$ to $(0,2)$, he stops at $(0,2)$ because he could have gone from $(0,0)$ to $(0,1)$ to $(0,2)$ in only 2 seconds. The expected number of steps Roger takes before he stops can be expressed as $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
      "answer": "1103"
    },
    {
      "question": "Suppose $X$ is a random variable that takes on only nonnegative integer values, with $E\\left[ X \\right] = 1$, $E\\left[ X^2 \\right] = 2$, and $E \\left[ X^3 \\right] = 5$. Determine the smallest possible value of the probability of the event $X=0$.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "When $5^{35}-6^{21}$ is evaluated, what is the units (ones) digit?",
      "answer": "9"
    },
    {
      "question": "Find the minimum possible value of the largest of $x y, 1-x-y+x y$, and $x+y-2 x y$ if $0 \\leq x \\leq y \\leq 1$.",
      "answer": "\\frac{4}{9}"
    },
    {
      "question": "During the regular season, Washington Redskins achieve a record of 10 wins and 6 losses. Compute the probability that their wins came in three streaks of consecutive wins, assuming that all possible arrangements of wins and losses are equally likely. (For example, the record LLWWWWWLWWLWWWLL contains three winning streaks, while WWWWWWWLLLLLLWWW has just two.)",
      "answer": "\\frac{315}{2002}"
    },
    {
      "question": "Eight coins are arranged in a circle heads up. A move consists of flipping over two adjacent coins. How many different sequences of six moves leave the coins alternating heads up and tails up?",
      "answer": "7680"
    },
    {
      "question": "There is one odd integer \\( N \\) between 400 and 600 that is divisible by both 5 and 11. What is the sum of the digits of \\( N \\)?",
      "answer": "18"
    },
    {
      "question": "The number 3003 is the only number known to appear eight times in Pascal's triangle, at positions $\\binom{3003}{1},\\binom{3003}{3002},\\binom{a}{2},\\binom{a}{a-2},\\binom{15}{b},\\binom{15}{15-b},\\binom{14}{6},\\binom{14}{8}$. Compute $a+b(15-b)$.",
      "answer": "128"
    },
    {
      "question": "Circle $O$ has chord $A B$. A circle is tangent to $O$ at $T$ and tangent to $A B$ at $X$ such that $A X=2 X B$. What is \\frac{A T}{B T}?",
      "answer": "2"
    },
    {
      "question": "Evaluate $$\\sin \\left(1998^{\\circ}+237^{\\circ}\\right) \\sin \\left(1998^{\\circ}-1653^{\\circ}\\right)$$",
      "answer": "-\\frac{1}{4}"
    },
    {
      "question": "Find the minimum value of $| \\sin x + \\cos x + \\tan x + \\cot x + \\sec x + \\csc x |$ for real numbers $x$.",
      "answer": "2\\sqrt{2} - 1"
    },
    {
      "question": "Let $a, b, c$ be non-negative real numbers such that $ab+bc+ca=3$. Suppose that $a^{3}b+b^{3}c+c^{3}a+2abc(a+b+c)=\\frac{9}{2}$. What is the minimum possible value of $ab^{3}+bc^{3}+ca^{3}$?",
      "answer": "18"
    },
    {
      "question": "If $n$ is a positive integer such that $n^{3}+2 n^{2}+9 n+8$ is the cube of an integer, find $n$.",
      "answer": "7"
    },
    {
      "question": "The polynomial $x^{3}-3 x^{2}+1$ has three real roots $r_{1}, r_{2}$, and $r_{3}$. Compute $\\sqrt[3]{3 r_{1}-2}+\\sqrt[3]{3 r_{2}-2}+\\sqrt[3]{3 r_{3}-2}$.",
      "answer": "0"
    },
    {
      "question": "There are two prime numbers $p$ so that $5 p$ can be expressed in the form $\\left\\lfloor\\frac{n^{2}}{5}\\right\\rfloor$ for some positive integer $n$. What is the sum of these two prime numbers?",
      "answer": "52"
    },
    {
      "question": "Suppose rectangle $F O L K$ and square $L O R E$ are on the plane such that $R L=12$ and $R K=11$. Compute the product of all possible areas of triangle $R K L$.",
      "answer": "414"
    },
    {
      "question": "How many solid $1 \times 1 \times 1$ cubes are required to make a solid $2 \times 2 \times 2$ cube?",
      "answer": "8"
    },
    {
      "question": "Let $C_{k, n}$ denote the number of paths on the Cartesian plane along which you can travel from $(0,0)$ to $(k, n)$, given the following rules: 1) You can only travel directly upward or directly rightward 2) You can only change direction at lattice points 3) Each horizontal segment in the path must be at most 99 units long. Find $\\sum_{j=0}^{\\infty} C_{100 j+19,17}$",
      "answer": "100^{17}"
    },
    {
      "question": "What is the maximum number of colours that can be used to paint an $8 \\times 8$ chessboard so that every square is painted in a single colour, and is adjacent , horizontally, vertically but not diagonally, to at least two other squares of its own colour?\n\n(A Shapovalov)",
      "answer": "16"
    },
    {
      "question": "If $u=-6$ and $x=\frac{1}{3}(3-4 u)$, what is the value of $x$?",
      "answer": "9"
    },
    {
      "question": "The set of $\\{1,2,3,...,63\\}$  was divided  into three non-empty disjoint sets $A,B$. Let $a,b,c$ be the product of all numbers in each set $A,B,C$ respectively  and finally we have determined the greatest common divisor of these three products. What was the biggest result we could get?",
      "answer": "$\\boxed{2^{19} \\cdot 3^{10} \\cdot 5^{4} \\cdot 3^{3} \\cdot 11\\cdot 13\\cdot 17\\cdot 19}$"
    },
    {
      "question": "For each nonnegative integer $n$ we define $A_n = 2^{3n}+3^{6n+2}+5^{6n+2}$. Find the greatest common divisor of the numbers $A_0,A_1,\\ldots, A_{1999}$.\n\n[i]Romania[/i]",
      "answer": "7"
    },
    {
      "question": "If the line that passes through the points $(2,7)$ and $(a, 3a)$ has a slope of 2, what is the value of $a$?",
      "answer": "3"
    },
    {
      "question": "Triangle $A B C$ has $A B=4, B C=3$, and a right angle at $B$. Circles $\\omega_{1}$ and $\\omega_{2}$ of equal radii are drawn such that $\\omega_{1}$ is tangent to $A B$ and $A C, \\omega_{2}$ is tangent to $B C$ and $A C$, and $\\omega_{1}$ is tangent to $\\omega_{2}$. Find the radius of $\\omega_{1}$.",
      "answer": "\\frac{5}{7}"
    },
    {
      "question": "Find out the maximum value of the numbers of edges of a solid regular octahedron that we can see from a point out of the regular octahedron.(We define we can see an edge $AB$ of the regular octahedron from point $P$ outside if and only if the intersection of non degenerate triangle $PAB$ and the solid regular octahedron is exactly edge $AB$.",
      "answer": "9"
    },
    {
      "question": "Let $A B C$ be an isosceles triangle with apex $A$. Let $I$ be the incenter. If $A I=3$ and the distance from $I$ to $B C$ is 2 , then what is the length of $B C$ ?",
      "answer": "4\\sqrt{5}"
    },
    {
      "question": "Dewa writes down a list of four integers. He calculates the average of each group of three of the four integers. These averages are $32,39,40,44$. What is the largest of the four integers?",
      "answer": "59"
    },
    {
      "question": "A function, $f$, has $f(2)=5$ and $f(3)=7$. In addition, $f$ has the property that $f(m)+f(n)=f(mn)$ for all positive integers $m$ and $n$. What is the value of $f(12)$?",
      "answer": "17"
    },
    {
      "question": "The lines $y=x, y=2 x$, and $y=3 x$ are the three medians of a triangle with perimeter 1. Find the length of the longest side of the triangle.",
      "answer": "\\sqrt{\\frac{\\sqrt{58}}{2+\\sqrt{34}+\\sqrt{58}}}"
    },
    {
      "question": "Find the value of $$\\sum_{a=1}^{\\infty} \\sum_{b=1}^{\\infty} \\sum_{c=1}^{\\infty} \\frac{a b(3 a+c)}{4^{a+b+c}(a+b)(b+c)(c+a)}$$",
      "answer": "\\frac{1}{54}"
    },
    {
      "question": "Let $S$ be a set, $|S|=35$. A set $F$ of mappings from $S$ to itself is called to be satisfying property $P(k)$, if for any $x,y\\in S$, there exist $f_1, \\cdots, f_k \\in F$ (not necessarily different), such that $f_k(f_{k-1}(\\cdots (f_1(x))))=f_k(f_{k-1}(\\cdots (f_1(y))))$.\nFind the least positive integer $m$, such that if $F$ satisfies property $P(2019)$, then it also satisfies property $P(m)$.",
      "answer": "595"
    },
    {
      "question": "You are given a $10 \\times 2$ grid of unit squares. Two different squares are adjacent if they share a side. How many ways can one mark exactly nine of the squares so that no two marked squares are adjacent?",
      "answer": "36"
    },
    {
      "question": "If the perimeter of a square is 28, what is the side length of the square?",
      "answer": "7"
    },
    {
      "question": "Consider sequences \\(a\\) of the form \\(a=\\left(a_{1}, a_{2}, \\ldots, a_{20}\\right)\\) such that each term \\(a_{i}\\) is either 0 or 1. For each such sequence \\(a\\), we can produce a sequence \\(b=\\left(b_{1}, b_{2}, \\ldots, b_{20}\\right)\\), where \\(b_{i}= \\begin{cases}a_{i}+a_{i+1} & i=1 \\\\ a_{i-1}+a_{i}+a_{i+1} & 1<i<20 \\\\ a_{i-1}+a_{i} & i=20\\end{cases}\\). How many sequences \\(b\\) are there that can be produced by more than one distinct sequence \\(a\\)?",
      "answer": "64"
    },
    {
      "question": "Let $a, b, c, d, e$ be nonnegative integers such that $625 a+250 b+100 c+40 d+16 e=15^{3}$. What is the maximum possible value of $a+b+c+d+e$ ?",
      "answer": "153"
    },
    {
      "question": "In the game of Galactic Dominion, players compete to amass cards, each of which is worth a certain number of points. Say you are playing a version of this game with only two kinds of cards, planet cards and hegemon cards. Each planet card is worth 2010 points, and each hegemon card is worth four points per planet card held. You start with no planet cards and no hegemon cards, and, on each turn, starting at turn one, you take either a planet card or a hegemon card, whichever is worth more points given the hand you currently hold. Define a sequence $\\left\\{a_{n}\\right\\}$ for all positive integers $n$ by setting $a_{n}$ to be 0 if on turn $n$ you take a planet card and 1 if you take a hegemon card. What is the smallest value of $N$ such that the sequence $a_{N}, a_{N+1}, \\ldots$ is necessarily periodic (meaning that there is a positive integer $k$ such that $a_{n+k}=a_{n}$ for all $\\left.n \\geq N\\right)$ ?",
      "answer": "503"
    },
    {
      "question": "Sindy writes down the positive integers less than 200 in increasing order, but skips the multiples of 10. She then alternately places + and - signs before each of the integers, yielding an expression $+1-2+3-4+5-6+7-8+9-11+12-\\cdots-199$. What is the value of the resulting expression?",
      "answer": "-100"
    },
    {
      "question": "Let $d_n$ be the determinant of the $n \\times n$ matrix whose entries, from\nleft to right and then from top to bottom, are $\\cos 1, \\cos 2, \\dots, \\cos\nn^2$. Evaluate\n$\\lim_{n\\to\\infty} d_n$.",
      "answer": "0"
    },
    {
      "question": "A point in three-space has distances $2,6,7,8,9$ from five of the vertices of a regular octahedron. What is its distance from the sixth vertex?",
      "answer": "\\sqrt{21}"
    },
    {
      "question": "For any positive integer $n$, let $\\tau(n)$ denote the number of positive divisors of $n$. If $n$ is a positive integer such that $\\frac{\\tau\\left(n^{2}\\right)}{\\tau(n)}=3$, compute $\\frac{\\tau\\left(n^{7}\\right)}{\\tau(n)}$.",
      "answer": "29"
    },
    {
      "question": "For how many ordered triples $(a, b, c)$ of positive integers are the equations $abc+9=ab+bc+ca$ and $a+b+c=10$ satisfied?",
      "answer": "21"
    },
    {
      "question": "A [i]permutation[/i] of the set of positive integers $[n] = \\{1, 2, . . . , n\\}$ is a sequence $(a_1 , a_2 , \\ldots, a_n ) $ such that each element of $[n]$ appears precisely one time as a term of the sequence. For example, $(3, 5, 1, 2, 4)$ is a permutation of $[5]$. Let $P (n)$ be the number of permutations of $[n]$ for which $ka_k$ is a perfect square for all $1 \\leq k \\leq n$. Find with proof the smallest $n$ such that $P (n)$ is a multiple of $2010$.",
      "answer": "4489"
    },
    {
      "question": "A set of 6 distinct lattice points is chosen uniformly at random from the set $\\{1,2,3,4,5,6\\}^{2}$. Let $A$ be the expected area of the convex hull of these 6 points. Estimate $N=\\left\\lfloor 10^{4} A\\right\\rfloor$. An estimate of $E$ will receive $\\max \\left(0,\\left\\lfloor 20-20\\left(\\frac{|E-N|}{10^{4}}\\right)^{1 / 3}\\right\\rfloor\\right)$ points.",
      "answer": "104552"
    },
    {
      "question": "Let $a=\\sqrt{17}$ and $b=i \\sqrt{19}$, where $i=\\sqrt{-1}$. Find the maximum possible value of the ratio $|a-z| /|b-z|$ over all complex numbers $z$ of magnitude 1 (i.e. over the unit circle $|z|=1$ ).",
      "answer": "\\frac{4}{3}"
    },
    {
      "question": "Hagrid has 100 animals. Among these animals, each is either striped or spotted but not both, each has either wings or horns but not both, there are 28 striped animals with wings, there are 62 spotted animals, and there are 36 animals with horns. How many of Hagrid's spotted animals have horns?",
      "answer": "26"
    },
    {
      "question": "Let $\\mathcal{P}$ be a parabola, and let $V_{1}$ and $F_{1}$ be its vertex and focus, respectively. Let $A$ and $B$ be points on $\\mathcal{P}$ so that $\\angle AV_{1}B=90^{\\circ}$. Let $\\mathcal{Q}$ be the locus of the midpoint of $AB$. It turns out that $\\mathcal{Q}$ is also a parabola, and let $V_{2}$ and $F_{2}$ denote its vertex and focus, respectively. Determine the ratio $F_{1}F_{2}/V_{1}V_{2}$.",
      "answer": "\\frac{7}{8}"
    },
    {
      "question": "Patrick and Anderson are having a snowball fight. Patrick throws a snowball at Anderson which is shaped like a sphere with a radius of 10 centimeters. Anderson catches the snowball and uses the snow from the snowball to construct snowballs with radii of 4 centimeters. Given that the total volume of the snowballs that Anderson constructs cannot exceed the volume of the snowball that Patrick threw, how many snowballs can Anderson construct?",
      "answer": "15"
    },
    {
      "question": "If $a, b$, and $c$ are random real numbers from 0 to 1, independently and uniformly chosen, what is the average (expected) value of the smallest of $a, b$, and $c$?",
      "answer": "1/4"
    },
    {
      "question": "Suppose that $k>0$ and that the line with equation $y=3kx+4k^{2}$ intersects the parabola with equation $y=x^{2}$ at points $P$ and $Q$. If $O$ is the origin and the area of $\triangle OPQ$ is 80, then what is the slope of the line?",
      "answer": "6"
    },
    {
      "question": "In the Year 0 of Cambridge there is one squirrel and one rabbit. Both animals multiply in numbers quickly. In particular, if there are $m$ squirrels and $n$ rabbits in Year $k$, then there will be $2 m+2019$ squirrels and $4 n-2$ rabbits in Year $k+1$. What is the first year in which there will be strictly more rabbits than squirrels?",
      "answer": "13"
    },
    {
      "question": "Let $A B C D$ be a convex quadrilateral whose diagonals $A C$ and $B D$ meet at $P$. Let the area of triangle $A P B$ be 24 and let the area of triangle $C P D$ be 25 . What is the minimum possible area of quadrilateral $A B C D ?$",
      "answer": "49+20 \\sqrt{6}"
    },
    {
      "question": "Two players play alternately on a $ 5 \\times 5$ board. The first player always enters a $ 1$ into an empty square and the second player always enters a $ 0$ into an empty square. When the board is full, the sum of the numbers in each of the nine $ 3 \\times 3$ squares is calculated and the first player's score is the largest such sum. What is the largest score the first player can make, regardless of the responses of the second player?",
      "answer": "6"
    },
    {
      "question": "Mayar and Rosie are 90 metres apart. Starting at the same time, they run towards each other. Mayar runs twice as fast as Rosie. How far has Mayar run when they meet?",
      "answer": "60"
    },
    {
      "question": "For any positive integer $n$, let $f(n)$ denote the number of 1's in the base-2 representation of $n$. For how many values of $n$ with $1 \\leq n \\leq 2002$ do we have $f(n)=f(n+1)$?",
      "answer": "501"
    },
    {
      "question": "Find the number of ways to choose two nonempty subsets $X$ and $Y$ of $\\{1,2, \\ldots, 2001\\}$, such that $|Y|=1001$ and the smallest element of $Y$ is equal to the largest element of $X$.",
      "answer": "2^{2000}"
    },
    {
      "question": "If $10^n = 1000^{20}$, what is the value of $n$?",
      "answer": "60"
    },
    {
      "question": "Suppose there are 100 cookies arranged in a circle, and 53 of them are chocolate chip, with the remainder being oatmeal. Pearl wants to choose a contiguous subsegment of exactly 67 cookies and wants this subsegment to have exactly \\(k\\) chocolate chip cookies. Find the sum of the \\(k\\) for which Pearl is guaranteed to succeed regardless of how the cookies are arranged.",
      "answer": "71"
    },
    {
      "question": "Let $P(x)$ be the monic polynomial with rational coefficients of minimal degree such that $\\frac{1}{\\sqrt{2}}$, $\\frac{1}{\\sqrt{3}}, \\frac{1}{\\sqrt{4}}, \\ldots, \\frac{1}{\\sqrt{1000}}$ are roots of $P$. What is the sum of the coefficients of $P$?",
      "answer": "\\frac{1}{16000}"
    },
    {
      "question": "Calculate the value of the expression $\\frac{1+(3 \\times 5)}{2}$.",
      "answer": "8"
    },
    {
      "question": "Determine which of the following numbers is smallest in value: $54 \\sqrt{3}, 144,108 \\sqrt{6}-108 \\sqrt{2}$.",
      "answer": "$54 \\sqrt{3}$"
    },
    {
      "question": "David and Evan each repeatedly flip a fair coin. David will stop when he flips a tail, and Evan will stop once he flips 2 consecutive tails. Find the probability that David flips more total heads than Evan.",
      "answer": "\\frac{1}{5}"
    },
    {
      "question": "A rectangle is divided into four smaller rectangles, labelled W, X, Y, and Z. The perimeters of rectangles W, X, and Y are 2, 3, and 5, respectively. What is the perimeter of rectangle Z?",
      "answer": "6"
    },
    {
      "question": "Marisa has a collection of $2^{8}-1=255$ distinct nonempty subsets of $\\{1,2,3,4,5,6,7,8\\}$. For each step she takes two subsets chosen uniformly at random from the collection, and replaces them with either their union or their intersection, chosen randomly with equal probability. (The collection is allowed to contain repeated sets.) She repeats this process $2^{8}-2=254$ times until there is only one set left in the collection. What is the expected size of this set?",
      "answer": "\\frac{1024}{255}"
    },
    {
      "question": "Let $t_{n}$ equal the integer closest to $\\sqrt{n}$. What is the sum $\\frac{1}{t_{1}}+\\frac{1}{t_{2}}+\\frac{1}{t_{3}}+\\frac{1}{t_{4}}+\\cdots+\\frac{1}{t_{2008}}+\\frac{1}{t_{2009}}+\\frac{1}{t_{2010}}$?",
      "answer": "88 \\frac{2}{3}"
    },
    {
      "question": "In a square of side length 4 , a point on the interior of the square is randomly chosen and a circle of radius 1 is drawn centered at the point. What is the probability that the circle intersects the square exactly twice?",
      "answer": "\\frac{\\pi+8}{16}"
    },
    {
      "question": "Let $m, n > 2$ be integers. One of the angles of a regular $n$-gon is dissected into $m$ angles of equal size by $(m-1)$ rays. If each of these rays intersects the polygon again at one of its vertices, we say $n$ is $m$-cut. Compute the smallest positive integer $n$ that is both 3-cut and 4-cut.",
      "answer": "14"
    },
    {
      "question": "Let $\\mathcal{H}$ be a regular hexagon with side length one. Peter picks a point $P$ uniformly and at random within $\\mathcal{H}$, then draws the largest circle with center $P$ that is contained in $\\mathcal{H}$. What is this probability that the radius of this circle is less than $\\frac{1}{2}$?",
      "answer": "\\frac{2 \\sqrt{3}-1}{3}"
    },
    {
      "question": "In a gumball machine containing 13 red, 5 blue, 1 white, and 9 green gumballs, what is the least number of gumballs that must be bought to guarantee receiving 3 gumballs of the same color?",
      "answer": "8"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=13, B C=14, C A=15$. Let $I_{A}, I_{B}, I_{C}$ be the $A, B, C$ excenters of this triangle, and let $O$ be the circumcenter of the triangle. Let $\\gamma_{A}, \\gamma_{B}, \\gamma_{C}$ be the corresponding excircles and $\\omega$ be the circumcircle. $X$ is one of the intersections between $\\gamma_{A}$ and $\\omega$. Likewise, $Y$ is an intersection of $\\gamma_{B}$ and $\\omega$, and $Z$ is an intersection of $\\gamma_{C}$ and $\\omega$. Compute $$\\cos \\angle O X I_{A}+\\cos \\angle O Y I_{B}+\\cos \\angle O Z I_{C}$$",
      "answer": "-\\frac{49}{65}"
    },
    {
      "question": "An empty $2020 \\times 2020 \\times 2020$ cube is given, and a $2020 \\times 2020$ grid of square unit cells is drawn on each of its six faces. A beam is a $1 \\times 1 \\times 2020$ rectangular prism. Several beams are placed inside the cube subject to the following conditions:\nThe two faces of each beam coincide with unit cells lying on opposite faces of the cube. (Hence, there are possible positions for a beam.)  No two beams have intersecting interiors.  The interiors of each of the four faces of each beam touch either a face of the cube or the interior of the face of another beam.\nWhat is the smallest positive number of beams that can be placed to satisfy these conditions?",
      "answer": "\\[\n3030\n\\]"
    },
    {
      "question": "Lily has a $300 \\times 300$ grid of squares. She now removes $100 \\times 100$ squares from each of the four corners and colors each of the remaining 50000 squares black and white. Given that no $2 \\times 2$ square is colored in a checkerboard pattern, find the maximum possible number of (unordered) pairs of squares such that one is black, one is white and the squares share an edge.",
      "answer": "49998"
    },
    {
      "question": "What is the value of \\((-1)^{3}+(-1)^{2}+(-1)\\)?",
      "answer": "-1"
    },
    {
      "question": "Compute $$\\lim _{A \\rightarrow+\\infty} \\frac{1}{A} \\int_{1}^{A} A^{\\frac{1}{x}} \\mathrm{~d} x$$",
      "answer": "1"
    },
    {
      "question": "A palindrome is a positive integer that reads the same backwards as forwards, such as 82328. What is the smallest 5 -digit palindrome that is a multiple of 99 ?",
      "answer": "54945"
    },
    {
      "question": "If $x=2018$, what is the value of the expression $x^{2}+2x-x(x+1)$?",
      "answer": "2018"
    },
    {
      "question": "An ant starts at one vertex of a tetrahedron. Each minute it walks along a random edge to an adjacent vertex. What is the probability that after one hour the ant winds up at the same vertex it started at?",
      "answer": "\\frac{3^{59}+1}{4 \\cdot 3^{59}}"
    },
    {
      "question": "Let $p_{i}$ be the $i$th prime. Let $$f(x)=\\sum_{i=1}^{50} p_{i} x^{i-1}=2+3x+\\cdots+229x^{49}$$ If $a$ is the unique positive real number with $f(a)=100$, estimate $A=\\lfloor 100000a\\rfloor$. An estimate of $E$ will earn $\\max (0,\\lfloor 20-|A-E| / 250\\rfloor)$ points.",
      "answer": "83601"
    },
    {
      "question": "The value of $\\sqrt{3^{3}+3^{3}+3^{3}}$ is what?",
      "answer": "9"
    },
    {
      "question": "Snacks are purchased for 17 soccer players. Juice boxes come in packs of 3 and cost $2.00 per pack. Apples come in bags of 5 and cost $4.00 per bag. What is the minimum amount of money that Danny spends to ensure every player gets a juice box and an apple?",
      "answer": "\\$28.00"
    },
    {
      "question": "In Mrs. Warner's class, there are 30 students. Strangely, 15 of the students have a height of 1.60 m and 15 of the students have a height of 1.22 m. Mrs. Warner lines up \\(n\\) students so that the average height of any four consecutive students is greater than 1.50 m and the average height of any seven consecutive students is less than 1.50 m. What is the largest possible value of \\(n\\)?",
      "answer": "9"
    },
    {
      "question": "Compute the number of complex numbers $z$ with $|z|=1$ that satisfy $$1+z^{5}+z^{10}+z^{15}+z^{18}+z^{21}+z^{24}+z^{27}=0$$",
      "answer": "11"
    },
    {
      "question": "A 50-card deck consists of 4 cards labeled \" $i$ \" for $i=1,2, \\ldots, 12$ and 2 cards labeled \" 13 \". If Bob randomly chooses 2 cards from the deck without replacement, what is the probability that his 2 cards have the same label?",
      "answer": "\\frac{73}{1225}"
    },
    {
      "question": "Find the sum $\\sum_{d=1}^{2012}\\left\\lfloor\\frac{2012}{d}\\right\\rfloor$.",
      "answer": "15612"
    },
    {
      "question": "A hockey team has 6 more red helmets than blue helmets. The ratio of red helmets to blue helmets is $5:3$. What is the total number of red helmets and blue helmets?",
      "answer": "24"
    },
    {
      "question": "Define the sequence $\\{x_{i}\\}_{i \\geq 0}$ by $x_{0}=2009$ and $x_{n}=-\\frac{2009}{n} \\sum_{k=0}^{n-1} x_{k}$ for all $n \\geq 1$. Compute the value of $\\sum_{n=0}^{2009} 2^{n} x_{n}$",
      "answer": "2009"
    },
    {
      "question": "Compute the number of ways to color the vertices of a regular heptagon red, green, or blue (with rotations and reflections distinct) such that no isosceles triangle whose vertices are vertices of the heptagon has all three vertices the same color.",
      "answer": "294"
    },
    {
      "question": "For a positive integer $n$, denote by $\\tau(n)$ the number of positive integer divisors of $n$, and denote by $\\phi(n)$ the number of positive integers that are less than or equal to $n$ and relatively prime to $n$. Call a positive integer $n$ good if $\\varphi(n)+4 \\tau(n)=n$. For example, the number 44 is good because $\\varphi(44)+4 \\tau(44)=44$. Find the sum of all good positive integers $n$.",
      "answer": "172"
    },
    {
      "question": "Consider a permutation $\\left(a_{1}, a_{2}, a_{3}, a_{4}, a_{5}\\right)$ of $\\{1,2,3,4,5\\}$. We say the tuple $\\left(a_{1}, a_{2}, a_{3}, a_{4}, a_{5}\\right)$ is flawless if for all $1 \\leq i<j<k \\leq 5$, the sequence $\\left(a_{i}, a_{j}, a_{k}\\right)$ is not an arithmetic progression (in that order). Find the number of flawless 5-tuples.",
      "answer": "20"
    },
    {
      "question": "A sequence is defined by $a_{0}=1$ and $a_{n}=2^{a_{n-1}}$ for $n \\geq 1$. What is the last digit (in base 10) of $a_{15}$?",
      "answer": "6"
    },
    {
      "question": "Find the sum of squares of all distinct complex numbers $x$ satisfying the equation $0=4 x^{10}-7 x^{9}+5 x^{8}-8 x^{7}+12 x^{6}-12 x^{5}+12 x^{4}-8 x^{3}+5 x^{2}-7 x+4$",
      "answer": "-\\frac{7}{16}"
    },
    {
      "question": "The function $f(x)$ is of the form $a x^{2}+b x+c$ for some integers $a, b$, and $c$. Given that $\\{f(177883), f(348710), f(796921), f(858522)\\} = \\{1324754875645,1782225466694,1984194627862,4388794883485\\}$ compute $a$.",
      "answer": "23"
    },
    {
      "question": "What is the 7th oblong number?",
      "answer": "56"
    },
    {
      "question": "Suppose $x$ is a real number such that $\\sin \\left(1+\\cos ^{2} x+\\sin ^{4} x\\right)=\\frac{13}{14}$. Compute $\\cos \\left(1+\\sin ^{2} x+\\cos ^{4} x\\right)$.",
      "answer": "-\\frac{3 \\sqrt{3}}{14}"
    },
    {
      "question": "What is 30% of 200?",
      "answer": "60"
    },
    {
      "question": "What is \\( 110\\% \\) of 500?",
      "answer": "550"
    },
    {
      "question": "On the blackboard, Amy writes 2017 in base-$a$ to get $133201_{a}$. Betsy notices she can erase a digit from Amy's number and change the base to base-$b$ such that the value of the number remains the same. Catherine then notices she can erase a digit from Betsy's number and change the base to base-$c$ such that the value still remains the same. Compute, in decimal, $a+b+c$.",
      "answer": "22"
    },
    {
      "question": "Let $w, x, y$, and $z$ be positive real numbers such that $0 \\neq \\cos w \\cos x \\cos y \\cos z$, $2 \\pi =w+x+y+z$, $3 \\tan w =k(1+\\sec w)$, $4 \\tan x =k(1+\\sec x)$, $5 \\tan y =k(1+\\sec y)$, $6 \\tan z =k(1+\\sec z)$. Find $k$.",
      "answer": "\\sqrt{19}"
    },
    {
      "question": "Find the number of subsets $S$ of $\\{1,2, \\ldots, 48\\}$ satisfying both of the following properties: - For each integer $1 \\leq k \\leq 24$, exactly one of $2 k-1$ and $2 k$ is in $S$. - There are exactly nine integers $1 \\leq m \\leq 47$ so that both $m$ and $m+1$ are in $S$.",
      "answer": "177100"
    },
    {
      "question": "If Alex does not sing on Saturday, then she has a $70 \\%$ chance of singing on Sunday; however, to rest her voice, she never sings on both days. If Alex has a $50 \\%$ chance of singing on Sunday, find the probability that she sings on Saturday.",
      "answer": "\\frac{2}{7}"
    },
    {
      "question": "Carl and Bob can demolish a building in 6 days, Anne and Bob can do it in 3, Anne and Carl in 5. How many days does it take all of them working together if Carl gets injured at the end of the first day and can't come back? Express your answer as a fraction in lowest terms.",
      "answer": "\\frac{59}{20}"
    },
    {
      "question": "Compute the number of nonempty subsets $S \\subseteq\\{-10,-9,-8, \\ldots, 8,9,10\\}$ that satisfy $|S|+\\min (S)$. $\\max (S)=0$.",
      "answer": "335"
    },
    {
      "question": "What is the largest possible radius of a circle contained in a 4-dimensional hypercube of side length 1?",
      "answer": "\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "Suppose that $R, S$ and $T$ are digits and that $N$ is the four-digit positive integer $8 R S T$. That is, $N$ has thousands digit 8, hundreds digit $R$, tens digits $S$, and ones (units) digit $T$, which means that $N=8000+100 R+10 S+T$. Suppose that the following conditions are all true: - The two-digit integer $8 R$ is divisible by 3. - The three-digit integer $8 R S$ is divisible by 4. - The four-digit integer $8 R S T$ is divisible by 5. - The digits of $N$ are not necessarily all different. What is the number of possible values for the integer $N$?",
      "answer": "14"
    },
    {
      "question": "Find the volume of the set of points $(x, y, z)$ satisfying $$\\begin{array}{r} x, y, z \\geq 0 \\\\ x+y \\leq 1 \\\\ y+z \\leq 1 \\\\ z+x \\leq 1 \\end{array}$$",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Which of the following is equal to $110 \\%$ of 500?",
      "answer": "550"
    },
    {
      "question": "On the number line, points $M$ and $N$ divide $L P$ into three equal parts. What is the value at $M$?",
      "answer": "\\frac{1}{9}"
    },
    {
      "question": "Let $a, b, c, d$ be real numbers such that $a^{2}+b^{2}+c^{2}+d^{2}=1$. Determine the minimum value of $(a-b)(b-c)(c-d)(d-a)$ and determine all values of $(a, b, c, d)$ such that the minimum value is achieved.",
      "answer": "-\\frac{1}{8}"
    },
    {
      "question": "Compute the positive real number $x$ satisfying $x^{\\left(2 x^{6}\\right)}=3$",
      "answer": "\\sqrt[6]{3}"
    },
    {
      "question": "The quadrilateral $ABCD$ has the following equality $\\angle ABC=\\angle BCD=150^{\\circ}$. Moreover, $AB=18$ and $BC=24$, the equilateral triangles $\\triangle APB,\\triangle BQC,\\triangle CRD$ are drawn outside the quadrilateral. If $P(X)$ is the perimeter of the polygon $X$, then the following equality is true $P(APQRD)=P(ABCD)+32$. Determine the length of the side $CD$.",
      "answer": "10"
    },
    {
      "question": "In triangle $BCD$, $\\angle CBD=\\angle CDB$ because $BC=CD$. If $\\angle BCD=80+50+30=160$, find $\\angle CBD=\\angle CDB$.",
      "answer": "10"
    },
    {
      "question": "Kelvin and 15 other frogs are in a meeting, for a total of 16 frogs. During the meeting, each pair of distinct frogs becomes friends with probability $\\frac{1}{2}$. Kelvin thinks the situation after the meeting is cool if for each of the 16 frogs, the number of friends they made during the meeting is a multiple of 4. Say that the probability of the situation being cool can be expressed in the form $\\frac{a}{b}$, where $a$ and $b$ are relatively prime. Find $a$.",
      "answer": "1167"
    },
    {
      "question": "How many of the integers from 1 to 100, inclusive, have at least one digit equal to 6?",
      "answer": "19"
    },
    {
      "question": "Several positive integers are given, not necessarily all different. Their sum is 2003. Suppose that $n_{1}$ of the given numbers are equal to $1, n_{2}$ of them are equal to $2, \\ldots, n_{2003}$ of them are equal to 2003. Find the largest possible value of $$n_{2}+2 n_{3}+3 n_{4}+\\cdots+2002 n_{2003}$$",
      "answer": "2002"
    },
    {
      "question": "Determine all positive integers relatively prime to all the terms of the infinite sequence \\[ a_n=2^n+3^n+6^n -1,\\ n\\geq 1. \\]",
      "answer": "1"
    },
    {
      "question": "Find the minimum possible value of $\\sqrt{58-42 x}+\\sqrt{149-140 \\sqrt{1-x^{2}}}$ where $-1 \\leq x \\leq 1$",
      "answer": "\\sqrt{109}"
    },
    {
      "question": "Which number from the set $\\{1,2,3,4,5,6,7,8,9,10,11\\}$ must be removed so that the mean (average) of the numbers remaining in the set is 6.1?",
      "answer": "5"
    },
    {
      "question": "At Wednesday's basketball game, the Cayley Comets scored 90 points. At Friday's game, they scored $80\\%$ as many points as they scored on Wednesday. How many points did they score on Friday?",
      "answer": "72"
    },
    {
      "question": "Find the number of ways in which the nine numbers $$1,12,123,1234, \\ldots, 123456789$$ can be arranged in a row so that adjacent numbers are relatively prime.",
      "answer": "0"
    },
    {
      "question": "Pentagon $S P E A K$ is inscribed in triangle $N O W$ such that $S$ and $P$ lie on segment $N O, K$ and $A$ lie on segment $N W$, and $E$ lies on segment $O W$. Suppose that $N S=S P=P O$ and $N K=K A=A W$. Given that $E P=E K=5$ and $E A=E S=6$, compute $O W$.",
      "answer": "\\frac{3 \\sqrt{610}}{5}"
    },
    {
      "question": "Consider pairs $(f,g)$ of functions from the set of nonnegative integers to itself such that \n[list]\n[*]$f(0) \\geq f(1) \\geq f(2) \\geq \\dots \\geq f(300) \\geq 0$\n[*]$f(0)+f(1)+f(2)+\\dots+f(300) \\leq 300$\n[*]for any 20 nonnegative integers $n_1, n_2, \\dots, n_{20}$, not necessarily distinct, we have $$g(n_1+n_2+\\dots+n_{20}) \\leq f(n_1)+f(n_2)+\\dots+f(n_{20}).$$\n[/list]\nDetermine the maximum possible value of $g(0)+g(1)+\\dots+g(6000)$ over all such pairs of functions.\n\n[i]Sean Li[/i]",
      "answer": "115440"
    },
    {
      "question": "Let $N$ be the number of functions $f$ from $\\{1,2, \\ldots, 101\\} \\rightarrow\\{1,2, \\ldots, 101\\}$ such that $f^{101}(1)=2$. Find the remainder when $N$ is divided by 103.",
      "answer": "43"
    },
    {
      "question": "Find the sum of all positive integers $n$ such that there exists an integer $b$ with $|b| \\neq 4$ such that the base -4 representation of $n$ is the same as the base $b$ representation of $n$.",
      "answer": "1026"
    },
    {
      "question": "Find the sum of the absolute values of the roots of $x^{4}-4 x^{3}-4 x^{2}+16 x-8=0$.",
      "answer": "2+2\\sqrt{2}+2\\sqrt{3}"
    },
    {
      "question": "Suppose $(a_{1}, a_{2}, a_{3}, a_{4})$ is a 4-term sequence of real numbers satisfying the following two conditions: - $a_{3}=a_{2}+a_{1}$ and $a_{4}=a_{3}+a_{2}$ - there exist real numbers $a, b, c$ such that $a n^{2}+b n+c=\\cos \\left(a_{n}\\right)$ for all $n \\in\\{1,2,3,4\\}$. Compute the maximum possible value of $\\cos \\left(a_{1}\\right)-\\cos \\left(a_{4}\\right)$ over all such sequences $(a_{1}, a_{2}, a_{3}, a_{4})$.",
      "answer": "-9+3\\sqrt{13}"
    },
    {
      "question": "Frank and Joe are playing ping pong. For each game, there is a $30 \\%$ chance that Frank wins and a $70 \\%$ chance Joe wins. During a match, they play games until someone wins a total of 21 games. What is the expected value of number of games played per match?",
      "answer": "30"
    },
    {
      "question": "After viewing the John Harvard statue, a group of tourists decides to estimate the distances of nearby locations on a map by drawing a circle, centered at the statue, of radius $\\sqrt{n}$ inches for each integer $2020 \\leq n \\leq 10000$, so that they draw 7981 circles altogether. Given that, on the map, the Johnston Gate is 10 -inch line segment which is entirely contained between the smallest and the largest circles, what is the minimum number of points on this line segment which lie on one of the drawn circles? (The endpoint of a segment is considered to be on the segment.)",
      "answer": "49"
    },
    {
      "question": "Manya has a stack of $85=1+4+16+64$ blocks comprised of 4 layers (the $k$ th layer from the top has $4^{k-1}$ blocks). Each block rests on 4 smaller blocks, each with dimensions half those of the larger block. Laura removes blocks one at a time from this stack, removing only blocks that currently have no blocks on top of them. Find the number of ways Laura can remove precisely 5 blocks from Manya's stack (the order in which they are removed matters).",
      "answer": "3384"
    },
    {
      "question": "Alex and Bob have 30 matches. Alex picks up somewhere between one and six matches (inclusive), then Bob picks up somewhere between one and six matches, and so on. The player who picks up the last match wins. How many matches should Alex pick up at the beginning to guarantee that he will be able to win?",
      "answer": "2"
    },
    {
      "question": "Sally the snail sits on the $3 \\times 24$ lattice of points $(i, j)$ for all $1 \\leq i \\leq 3$ and $1 \\leq j \\leq 24$. She wants to visit every point in the lattice exactly once. In a move, Sally can move to a point in the lattice exactly one unit away. Given that Sally starts at $(2,1)$, compute the number of possible paths Sally can take.",
      "answer": "4096"
    },
    {
      "question": "3. Consider all 100-digit positive integers such that each decimal digit of these equals $2,3,4,5,6$, or 7 . How many of these integers are divisible by $2^{100}$ ?\n\t\n\tPavel Kozhevnikov",
      "answer": "3^{100}"
    },
    {
      "question": "The Athenas are playing a 44 game season. They have 20 wins and 15 losses so far. What is the smallest number of their remaining games that they must win to make the playoffs, given they must win at least 60% of all of their games?",
      "answer": "7"
    },
    {
      "question": "Determine the greatest real number $ C $, such that for every positive integer $ n\\ge 2 $, there exists $ x_1, x_2,..., x_n  \\in [-1,1]$, so that\n$$\\prod_{1\\le i<j\\le n}(x_i-x_j) \\ge C^{\\frac{n(n-1)}{2}}$$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Alice is thinking of a positive real number $x$, and Bob is thinking of a positive real number $y$. Given that $x^{\\sqrt{y}}=27$ and $(\\sqrt{x})^{y}=9$, compute $x y$.",
      "answer": "16 \\sqrt[4]{3}"
    },
    {
      "question": "Our next object up for bid is an arithmetic progression of primes. For example, the primes 3,5, and 7 form an arithmetic progression of length 3. What is the largest possible length of an arithmetic progression formed of positive primes less than 1,000,000? Be prepared to justify your answer.",
      "answer": "12"
    },
    {
      "question": "Farmer James has some strange animals. His hens have 2 heads and 8 legs, his peacocks have 3 heads and 9 legs, and his zombie hens have 6 heads and 12 legs. Farmer James counts 800 heads and 2018 legs on his farm. What is the number of animals that Farmer James has on his farm?",
      "answer": "203"
    },
    {
      "question": "Let $P$ be a polynomial such that $P(x)=P(0)+P(1) x+P(2) x^{2}$ and $P(-1)=1$. Compute $P(3)$.",
      "answer": "5"
    },
    {
      "question": "Find the smallest integer $n \\geq 5$ for which there exists a set of $n$ distinct pairs $\\left(x_{1}, y_{1}\\right), \\ldots,\\left(x_{n}, y_{n}\\right)$ of positive integers with $1 \\leq x_{i}, y_{i} \\leq 4$ for $i=1,2, \\ldots, n$, such that for any indices $r, s \\in\\{1,2, \\ldots, n\\}$ (not necessarily distinct), there exists an index $t \\in\\{1,2, \\ldots, n\\}$ such that 4 divides $x_{r}+x_{s}-x_{t}$ and $y_{r}+y_{s}-y_{t}$.",
      "answer": "8"
    },
    {
      "question": "Let acute triangle $ABC$ have circumcenter $O$, and let $M$ be the midpoint of $BC$. Let $P$ be the unique point such that $\\angle BAP=\\angle CAM, \\angle CAP=\\angle BAM$, and $\\angle APO=90^{\\circ}$. If $AO=53, OM=28$, and $AM=75$, compute the perimeter of $\\triangle BPC$.",
      "answer": "192"
    },
    {
      "question": "Let $P$ be the set of points $$\\{(x, y) \\mid 0 \\leq x, y \\leq 25, x, y \\in \\mathbb{Z}\\}$$ and let $T$ be the set of triangles formed by picking three distinct points in $P$ (rotations, reflections, and translations count as distinct triangles). Compute the number of triangles in $T$ that have area larger than 300.",
      "answer": "436"
    },
    {
      "question": "An equilateral triangle lies in the Cartesian plane such that the $x$-coordinates of its vertices are pairwise distinct and all satisfy the equation $x^{3}-9 x^{2}+10 x+5=0$. Compute the side length of the triangle.",
      "answer": "2 \\sqrt{17}"
    },
    {
      "question": "There are $2022$ users on a social network called Mathbook, and some of them are Mathbook-friends. (On Mathbook, friendship is always mutual and permanent.) \n\nStarting now, Mathbook will only allow a new friendship to be formed between two users if they have [i]at least two[/i] friends in common. What is the minimum number of friendships that must already exist so that every user could eventually become friends with every other user?",
      "answer": "3031"
    },
    {
      "question": "Find the smallest $n$ such that $n!$ ends with 10 zeroes.",
      "answer": "45"
    },
    {
      "question": "There are 10 people who want to choose a committee of 5 people among them. They do this by first electing a set of $1,2,3$, or 4 committee leaders, who then choose among the remaining people to complete the 5-person committee. In how many ways can the committee be formed, assuming that people are distinguishable? (Two committees that have the same members but different sets of leaders are considered to be distinct.)",
      "answer": "7560"
    },
    {
      "question": "There is a unique quadruple of positive integers $(a, b, c, k)$ such that $c$ is not a perfect square and $a+\\sqrt{b+\\sqrt{c}}$ is a root of the polynomial $x^{4}-20 x^{3}+108 x^{2}-k x+9$. Compute $c$.",
      "answer": "7"
    },
    {
      "question": "Compute the number of quadruples $(a, b, c, d)$ of positive integers satisfying $12a+21b+28c+84d=2024$.",
      "answer": "2024"
    },
    {
      "question": "In how many ways can the set of ordered pairs of integers be colored red and blue such that for all $a$ and $b$, the points $(a, b),(-1-b, a+1)$, and $(1-b, a-1)$ are all the same color?",
      "answer": "16"
    },
    {
      "question": "Jerry has ten distinguishable coins, each of which currently has heads facing up. He chooses one coin and flips it over, so it now has tails facing up. Then he picks another coin (possibly the same one as before) and flips it over. How many configurations of heads and tails are possible after these two flips?",
      "answer": "46"
    },
    {
      "question": "Determine the greatest positive integer $k$ that satisfies the following property: The set of positive integers can be partitioned into $k$ subsets $A_1, A_2, \\ldots, A_k$ such that for all integers $n \\geq 15$ and all $i \\in \\{1, 2, \\ldots, k\\}$ there exist two distinct elements of $A_i$ whose sum is $n.$\n\n[i]",
      "answer": "3"
    },
    {
      "question": "Let $f(n)$ be the number of times you have to hit the $\\sqrt{ }$ key on a calculator to get a number less than 2 starting from $n$. For instance, $f(2)=1, f(5)=2$. For how many $1<m<2008$ is $f(m)$ odd?",
      "answer": "242"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=4, B C=8$, and $C A=5$. Let $M$ be the midpoint of $B C$, and let $D$ be the point on the circumcircle of $A B C$ so that segment $A D$ intersects the interior of $A B C$, and $\\angle B A D=\\angle C A M$. Let $A D$ intersect side $B C$ at $X$. Compute the ratio $A X / A D$.",
      "answer": "$\\frac{9}{41}$"
    },
    {
      "question": "If $\\left(a+\\frac{1}{a}\\right)^{2}=3$, find $\\left(a+\\frac{1}{a}\\right)^{3}$ in terms of $a$.",
      "answer": "0"
    },
    {
      "question": "Let $A B C$ be a triangle with $\\angle B=90^{\\circ}$. Given that there exists a point $D$ on $A C$ such that $A D=D C$ and $B D=B C$, compute the value of the ratio $\\frac{A B}{B C}$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "A cube has an edge length of 30. A rectangular solid has edge lengths 20, 30, and $L$. If the cube and the rectangular solid have equal surface areas, what is the value of $L$?",
      "answer": "42"
    },
    {
      "question": "The point $P$ is inside of an equilateral triangle with side length $10$ so that the distance from $P$ to two of the sides are $1$ and $3$. Find the distance from $P$ to the third side.",
      "answer": "5\\sqrt{3} - 4"
    },
    {
      "question": "Evaluate the expression $2x^{2}+3x^{2}$ when $x=2$.",
      "answer": "20"
    },
    {
      "question": "The average of a set of distinct primes is 27. What is the largest prime that can be in this set?",
      "answer": "139"
    },
    {
      "question": "If the number of zeros in the integer equal to $(10^{100}) \times (100^{10})$ is sought, what is this number?",
      "answer": "120"
    },
    {
      "question": "Let $x=2001^{1002}-2001^{-1002}$ and $y=2001^{1002}+2001^{-1002}$. Find $x^{2}-y^{2}$.",
      "answer": "-4"
    },
    {
      "question": "If $\frac{x-y}{z-y}=-10$, what is the value of $\frac{x-z}{y-z}$?",
      "answer": "11"
    },
    {
      "question": "Alice draws three cards from a standard 52-card deck with replacement. Ace through 10 are worth 1 to 10 points respectively, and the face cards King, Queen, and Jack are each worth 10 points. The probability that the sum of the point values of the cards drawn is a multiple of 10 can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "26597"
    },
    {
      "question": "Let $A B C D$ be a rectangle, and let $E$ and $F$ be points on segment $A B$ such that $A E=E F=F B$. If $C E$ intersects the line $A D$ at $P$, and $P F$ intersects $B C$ at $Q$, determine the ratio of $B Q$ to $C Q$.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Let $a_1,a_2,a_3,\\cdots$ be a non-decreasing sequence of positive integers. For $m\\ge1$ , define $b_m=\\min\\{n: a_n \\ge m\\}$ , that is, $b_m$ is the minimum value of $n$ such that $a_n\\ge m$ . If $a_{19}=85$ , determine the maximum value of $a_1+a_2+\\cdots+a_{19}+b_1+b_2+\\cdots+b_{85}$ .",
      "answer": "\\boxed{1700}"
    },
    {
      "question": "Contessa is taking a random lattice walk in the plane, starting at $(1,1)$. (In a random lattice walk, one moves up, down, left, or right 1 unit with equal probability at each step.) If she lands on a point of the form $(6 m, 6 n)$ for $m, n \\in \\mathbb{Z}$, she ascends to heaven, but if she lands on a point of the form $(6 m+3,6 n+3)$ for $m, n \\in \\mathbb{Z}$, she descends to hell. What is the probability that she ascends to heaven?",
      "answer": "\\frac{13}{22}"
    },
    {
      "question": "Let $b(n)$ be the number of digits in the base -4 representation of $n$. Evaluate $\\sum_{i=1}^{2013} b(i)$.",
      "answer": "12345"
    },
    {
      "question": "Let $a_{1}, a_{2}, \\ldots$ be a sequence defined by $a_{1}=a_{2}=1$ and $a_{n+2}=a_{n+1}+a_{n}$ for $n \\geq 1$. Find $$\\sum_{n=1}^{\\infty} \\frac{a_{n}}{4^{n+1}}$$",
      "answer": "\\frac{1}{11}"
    },
    {
      "question": "Calculate the sum of the coefficients of $P(x)$ if $\\left(20 x^{27}+2 x^{2}+1\\right) P(x)=2001 x^{2001}$.",
      "answer": "87"
    },
    {
      "question": "The smallest of nine consecutive integers is 2012. These nine integers are placed in the circles to the right. The sum of the three integers along each of the four lines is the same. If this sum is as small as possible, what is the value of $u$?",
      "answer": "2015"
    },
    {
      "question": "The numbers $1,2, \\ldots, 20$ are put into a hat. Claire draws two numbers from the hat uniformly at random, $a<b$, and then puts them back into the hat. Then, William draws two numbers from the hat uniformly at random, $c<d$. Let $N$ denote the number of integers $n$ that satisfy exactly one of $a \\leq n \\leq b$ and $c \\leq n \\leq d$. Compute the probability $N$ is even.",
      "answer": "\\frac{181}{361}"
    },
    {
      "question": "Determine the maximum value of $m^2+n^2$, where $m$ and $n$ are integers in the range $1,2,\\ldots,1981$ satisfying $(n^2-mn-m^2)^2=1$.",
      "answer": "\\boxed{987^2+1597^2}"
    },
    {
      "question": "You are trapped in ancient Japan, and a giant enemy crab is approaching! You must defeat it by cutting off its two claws and six legs and attacking its weak point for massive damage. You cannot cut off any of its claws until you cut off at least three of its legs, and you cannot attack its weak point until you have cut off all of its claws and legs. In how many ways can you defeat the giant enemy crab?",
      "answer": "14400"
    },
    {
      "question": "Find $\\sum_{k=0}^{\\infty}\\left\\lfloor\\frac{1+\\sqrt{\\frac{2000000}{4^{k}}}}{2}\\right\\rfloor$ where $\\lfloor x\\rfloor$ denotes the largest integer less than or equal to $x$.",
      "answer": "1414"
    },
    {
      "question": "If $n$ is a positive integer, the notation $n$! (read \" $n$ factorial\") is used to represent the product of the integers from 1 to $n$. That is, $n!=n(n-1)(n-2) \\cdots(3)(2)(1)$. For example, $4!=4(3)(2)(1)=24$ and $1!=1$. If $a$ and $b$ are positive integers with $b>a$, what is the ones (units) digit of $b!-a$! that cannot be?",
      "answer": "7"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=7, B C=9$, and $C A=4$. Let $D$ be the point such that $A B \\| C D$ and $C A \\| B D$. Let $R$ be a point within triangle $B C D$. Lines $\\ell$ and $m$ going through $R$ are parallel to $C A$ and $A B$ respectively. Line $\\ell$ meets $A B$ and $B C$ at $P$ and $P^{\\prime}$ respectively, and $m$ meets $C A$ and $B C$ at $Q$ and $Q^{\\prime}$ respectively. If $S$ denotes the largest possible sum of the areas of triangles $B P P^{\\prime}, R P^{\\prime} Q^{\\prime}$, and $C Q Q^{\\prime}$, determine the value of $S^{2}$.",
      "answer": "180"
    },
    {
      "question": "Let $q(x)=q^{1}(x)=2x^{2}+2x-1$, and let $q^{n}(x)=q(q^{n-1}(x))$ for $n>1$. How many negative real roots does $q^{2016}(x)$ have?",
      "answer": "\\frac{2017+1}{3}"
    },
    {
      "question": "Suppose $E, I, L, V$ are (not necessarily distinct) nonzero digits in base ten for which the four-digit number $\\underline{E} \\underline{V} \\underline{I} \\underline{L}$ is divisible by 73 , and the four-digit number $\\underline{V} \\underline{I} \\underline{L} \\underline{E}$ is divisible by 74 . Compute the four-digit number $\\underline{L} \\underline{I} \\underline{V} \\underline{E}$.",
      "answer": "9954"
    },
    {
      "question": "Let $C$ be a circle with two diameters intersecting at an angle of 30 degrees. A circle $S$ is tangent to both diameters and to $C$, and has radius 1. Find the largest possible radius of $C$.",
      "answer": "1+\\sqrt{2}+\\sqrt{6}"
    },
    {
      "question": "What is the result of subtracting eighty-seven from nine hundred forty-three?",
      "answer": "856"
    },
    {
      "question": "Let $A B C$ be a triangle with $C A=C B=5$ and $A B=8$. A circle $\\omega$ is drawn such that the interior of triangle $A B C$ is completely contained in the interior of $\\omega$. Find the smallest possible area of $\\omega$.",
      "answer": "16 \\pi"
    },
    {
      "question": "Jacob flipped a fair coin five times. In the first three flips, the coin came up heads exactly twice. In the last three flips, the coin also came up heads exactly twice. What is the probability that the third flip was heads?",
      "answer": "\\frac{4}{5}"
    },
    {
      "question": "Evaluate the infinite sum $\\sum_{n=0}^{\\infty}\\binom{2 n}{n} \\frac{1}{5^{n}}$.",
      "answer": "\\sqrt{5}"
    },
    {
      "question": "Let $a$ and $b$ be positive real numbers. Determine the minimum possible value of $$\\sqrt{a^{2}+b^{2}}+\\sqrt{(a-1)^{2}+b^{2}}+\\sqrt{a^{2}+(b-1)^{2}}+\\sqrt{(a-1)^{2}+(b-1)^{2}}$$",
      "answer": "2 \\sqrt{2}"
    },
    {
      "question": "In how many ways can we fill the cells of a $4\\times4$ grid such that each cell contains exactly one positive integer and the product of the numbers in each row and each column is $2020$?",
      "answer": "576"
    },
    {
      "question": "A ten-level 2-tree is drawn in the plane: a vertex $A_{1}$ is marked, it is connected by segments with two vertices $B_{1}$ and $B_{2}$, each of $B_{1}$ and $B_{2}$ is connected by segments with two of the four vertices $C_{1}, C_{2}, C_{3}, C_{4}$ (each $C_{i}$ is connected with one $B_{j}$ exactly); and so on, up to 512 vertices $J_{1}, \\ldots, J_{512}$. Each of the vertices $J_{1}, \\ldots, J_{512}$ is coloured blue or golden. Consider all permutations $f$ of the vertices of this tree, such that (i) if $X$ and $Y$ are connected with a segment, then so are $f(X)$ and $f(Y)$, and (ii) if $X$ is coloured, then $f(X)$ has the same colour. Find the maximum $M$ such that there are at least $M$ permutations with these properties, regardless of the colouring.",
      "answer": "2^{2^{7}}"
    },
    {
      "question": "In the decimal representation of $\frac{1}{7}$, the 100th digit to the right of the decimal is?",
      "answer": "8"
    },
    {
      "question": "Yann writes down the first $n$ consecutive positive integers, $1,2,3,4, \\ldots, n-1, n$. He removes four different integers $p, q, r, s$ from the list. At least three of $p, q, r, s$ are consecutive and $100<p<q<r<s$. The average of the integers remaining in the list is 89.5625. What is the number of possible values of $s$?",
      "answer": "22"
    },
    {
      "question": "Count the number of sequences $a_{1}, a_{2}, a_{3}, a_{4}, a_{5}$ of integers such that $a_{i} \\leq 1$ for all $i$ and all partial sums $\\left(a_{1}, a_{1}+a_{2}\\right.$, etc.) are non-negative.",
      "answer": "132"
    },
    {
      "question": "A rectangular prism has a volume of $12 \\mathrm{~cm}^{3}$. A new prism is formed by doubling the length, doubling the width, and tripling the height of the original prism. What is the volume of this new prism?",
      "answer": "144"
    },
    {
      "question": "Find the number of positive integers less than 1000000 which are less than or equal to the sum of their proper divisors. If your answer is $X$ and the actual value is $Y$, your score will be $\\max \\left(0,20-80\\left|1-\\frac{X}{Y}\\right|\\right)$ rounded to the nearest integer.",
      "answer": "247548"
    },
    {
      "question": "Let $f(n)$ be the largest prime factor of $n$. Estimate $$N=\\left\\lfloor 10^{4} \\cdot \\frac{\\sum_{n=2}^{10^{6}} f\\left(n^{2}-1\\right)}{\\sum_{n=2}^{10^{6}} f(n)}\\right\\rfloor$$ An estimate of $E$ will receive $\\max \\left(0,\\left\\lfloor 20-20\\left(\\frac{|E-N|}{10^{3}}\\right)^{1 / 3}\\right\\rfloor\\right)$ points.",
      "answer": "18215"
    },
    {
      "question": "Find the smallest positive integer $n$ such that $\\underbrace{2^{2 \\cdot 2}}_{n}>3^{3^{3^{3}}}$. (The notation $\\underbrace{2^{2^{2}}}_{n}$, is used to denote a power tower with $n 2$ 's. For example, $\\underbrace{2^{22^{2}}}_{n}$ with $n=4$ would equal $2^{2^{2^{2}}}$.)",
      "answer": "6"
    },
    {
      "question": "For each positive integer $n$ let $S_{n}$ denote the set $\\{1,2,3, \\ldots, n\\}$. Compute the number of triples of subsets $A, B, C$ of $S_{2006}$ (not necessarily nonempty or proper) such that $A$ is a subset of $B$ and $S_{2006}-A$ is a subset of $C$.",
      "answer": "2^{4012}"
    },
    {
      "question": "The English alphabet, which has 26 letters, is randomly permuted. Let \\(p_{1}\\) be the probability that \\(\\mathrm{AB}, \\mathrm{CD}\\), and \\(\\mathrm{EF}\\) all appear as contiguous substrings. Let \\(p_{2}\\) be the probability that \\(\\mathrm{ABC}\\) and \\(\\mathrm{DEF}\\) both appear as contiguous substrings. Compute \\(\\frac{p_{1}}{p_{2}}\\).",
      "answer": "23"
    },
    {
      "question": "All the sequences consisting of five letters from the set $\\{T, U, R, N, I, P\\}$ (with repetitions allowed) are arranged in alphabetical order in a dictionary. Two sequences are called \"anagrams\" of each other if one can be obtained by rearranging the letters of the other. How many pairs of anagrams are there that have exactly 100 other sequences between them in the dictionary?",
      "answer": "0"
    },
    {
      "question": "Sean is a biologist, and is looking at a string of length 66 composed of the letters $A, T, C, G$. A substring of a string is a contiguous sequence of letters in the string. For example, the string $AGTC$ has 10 substrings: $A, G, T, C, AG, GT, TC, AGT, GTC, AGTC$. What is the maximum number of distinct substrings of the string Sean is looking at?",
      "answer": "2100"
    },
    {
      "question": "Let $A$ be the number of unordered pairs of ordered pairs of integers between 1 and 6 inclusive, and let $B$ be the number of ordered pairs of unordered pairs of integers between 1 and 6 inclusive. (Repetitions are allowed in both ordered and unordered pairs.) Find $A-B$.",
      "answer": "225"
    },
    {
      "question": "Kermit the frog enjoys hopping around the infinite square grid in his backyard. It takes him 1 Joule of energy to hop one step north or one step south, and 1 Joule of energy to hop one step east or one step west. He wakes up one morning on the grid with 100 Joules of energy, and hops till he falls asleep with 0 energy. How many different places could he have gone to sleep?",
      "answer": "10201"
    },
    {
      "question": "Find the largest integer $n$ such that $n$ is divisible by all positive integers less than $\\sqrt[3]{n}$.",
      "answer": "420"
    },
    {
      "question": "Compute the prime factorization of 1007021035035021007001.",
      "answer": "7^{7} \\cdot 11^{7} \\cdot 13^{7}"
    },
    {
      "question": "Find the total area of the region outside of an equilateral triangle but inside three circles each with radius 1, centered at the vertices of the triangle.",
      "answer": "\\frac{2 \\pi-\\sqrt{3}}{2}"
    },
    {
      "question": "Determine the value of $$2002+\\frac{1}{2}\\left(2001+\\frac{1}{2}\\left(2000+\\cdots+\\frac{1}{2}\\left(3+\\frac{1}{2} \\cdot 2\\right)\\right) \\cdots\\right)$$",
      "answer": "4002"
    },
    {
      "question": "You are trying to cross a 400 foot wide river. You can jump at most 4 feet, but you have many stones you can throw into the river. You will stop throwing stones and cross the river once you have placed enough stones to be able to do so. You can throw straight, but you can't judge distance very well, so each stone ends up being placed uniformly at random along the width of the river. Estimate the expected number $N$ of stones you must throw before you can get across the river. An estimate of $E$ will earn $\\left\\lfloor 20 \\min \\left(\\frac{N}{E}, \\frac{E}{N}\\right)^{3}\\right\\rfloor$ points.",
      "answer": "712.811"
    },
    {
      "question": "A two-digit positive integer $x$ has the property that when 109 is divided by $x$, the remainder is 4. What is the sum of all such two-digit positive integers $x$?",
      "answer": "71"
    },
    {
      "question": "For odd primes $p$, let $f(p)$ denote the smallest positive integer $a$ for which there does not exist an integer $n$ satisfying $p \\mid n^{2}-a$. Estimate $N$, the sum of $f(p)^{2}$ over the first $10^{5}$ odd primes $p$. An estimate of $E>0$ will receive $\\left\\lfloor 22 \\min (N / E, E / N)^{3}\\right\\rfloor$ points.",
      "answer": "2266067"
    },
    {
      "question": "A committee of 5 is to be chosen from a group of 9 people. How many ways can it be chosen, if Bill and Karl must serve together or not at all, and Alice and Jane refuse to serve with each other?",
      "answer": "41"
    },
    {
      "question": "Find the number of ways in which the letters in \"HMMTHMMT\" can be rearranged so that each letter is adjacent to another copy of the same letter. For example, \"MMMMTTHH\" satisfies this property, but \"HHTMMMTM\" does not.",
      "answer": "12"
    },
    {
      "question": "Let $N$ be the number of sequences of positive integers $\\left(a_{1}, a_{2}, a_{3}, \\ldots, a_{15}\\right)$ for which the polynomials $$x^{2}-a_{i} x+a_{i+1}$$ each have an integer root for every $1 \\leq i \\leq 15$, setting $a_{16}=a_{1}$. Estimate $N$. An estimate of $E$ will earn $\\left\\lfloor 20 \\min \\left(\\frac{N}{E}, \\frac{E}{N}\\right)^{2}\\right\\rfloor$ points.",
      "answer": "1409"
    },
    {
      "question": "Suppose that the plane is tiled with an infinite checkerboard of unit squares. If another unit square is dropped on the plane at random with position and orientation independent of the checkerboard tiling, what is the probability that it does not cover any of the corners of the squares of the checkerboard?",
      "answer": "2 - \\frac{6}{\\pi}"
    },
    {
      "question": "Regular hexagon $A B C D E F$ has side length 2. A laser beam is fired inside the hexagon from point $A$ and hits $\\overline{B C}$ at point $G$. The laser then reflects off $\\overline{B C}$ and hits the midpoint of $\\overline{D E}$. Find $B G$.",
      "answer": "\\frac{2}{5}"
    },
    {
      "question": "Square $P Q R S$ has an area of 900. $M$ is the midpoint of $P Q$ and $N$ is the midpoint of $P S$. What is the area of triangle $P M N$?",
      "answer": "112.5"
    },
    {
      "question": "A five-digit positive integer is created using each of the odd digits $1, 3, 5, 7, 9$ once so that the thousands digit is larger than the hundreds digit, the thousands digit is larger than the ten thousands digit, the tens digit is larger than the hundreds digit, and the tens digit is larger than the units digit. How many such five-digit positive integers are there?",
      "answer": "16"
    },
    {
      "question": "Find the sum of the $x$-coordinates of the distinct points of intersection of the plane curves given by $x^{2}=x+y+4$ and $y^{2}=y-15 x+36$.",
      "answer": "0"
    },
    {
      "question": "Compute the number of positive integers less than 10! which can be expressed as the sum of at most 4 (not necessarily distinct) factorials.",
      "answer": "648"
    },
    {
      "question": "An omino is a 1-by-1 square or a 1-by-2 horizontal rectangle. An omino tiling of a region of the plane is a way of covering it (and only it) by ominoes. How many omino tilings are there of a 2-by-10 horizontal rectangle?",
      "answer": "7921"
    },
    {
      "question": "Let $r_{k}$ denote the remainder when $\\binom{127}{k}$ is divided by 8. Compute $r_{1}+2 r_{2}+3 r_{3}+\\cdots+63 r_{63}$.",
      "answer": "8096"
    },
    {
      "question": "If $y=1$ and $4x-2y+3=3x+3y$, what is the value of $x$?",
      "answer": "2"
    },
    {
      "question": "To survive the coming Cambridge winter, Chim Tu doesn't wear one T-shirt, but instead wears up to FOUR T-shirts, all in different colors. An outfit consists of three or more T-shirts, put on one on top of the other in some order, such that two outfits are distinct if the sets of T-shirts used are different or the sets of T-shirts used are the same but the order in which they are worn is different. Given that Chim Tu changes his outfit every three days, and otherwise never wears the same outfit twice, how many days of winter can Chim Tu survive? (Needless to say, he only has four t-shirts.)",
      "answer": "144"
    },
    {
      "question": "If \\( 3^x = 5 \\), what is the value of \\( 3^{x+2} \\)?",
      "answer": "45"
    },
    {
      "question": "Let $\\alpha, \\beta$, and $\\gamma$ be three real numbers. Suppose that $\\cos \\alpha+\\cos \\beta+\\cos \\gamma =1$ and $\\sin \\alpha+\\sin \\beta+\\sin \\gamma =1$. Find the smallest possible value of $\\cos \\alpha$.",
      "answer": "\\frac{-1-\\sqrt{7}}{4}"
    },
    {
      "question": "Paul and Sara are playing a game with integers on a whiteboard, with Paul going first. When it is Paul's turn, he can pick any two integers on the board and replace them with their product; when it is Sara's turn, she can pick any two integers on the board and replace them with their sum. Play continues until exactly one integer remains on the board. Paul wins if that integer is odd, and Sara wins if it is even. Initially, there are 2021 integers on the board, each one sampled uniformly at random from the set \\{0,1,2,3, \\ldots, 2021\\}. Assuming both players play optimally, the probability that Paul wins is $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find the remainder when $m+n$ is divided by 1000.",
      "answer": "\\[\n383\n\\]"
    },
    {
      "question": "David and Evan are playing a game. Evan thinks of a positive integer $N$ between 1 and 59, inclusive, and David tries to guess it. Each time David makes a guess, Evan will tell him whether the guess is greater than, equal to, or less than $N$. David wants to devise a strategy that will guarantee that he knows $N$ in five guesses. In David's strategy, each guess will be determined only by Evan's responses to any previous guesses (the first guess will always be the same), and David will only guess a number which satisfies each of Evan's responses. How many such strategies are there?",
      "answer": "36440"
    },
    {
      "question": "An unfair coin has the property that when flipped four times, it has the same probability of turning up 2 heads and 2 tails (in any order) as 3 heads and 1 tail (in any order). What is the probability of getting a head in any one flip?",
      "answer": "\\frac{3}{5}"
    },
    {
      "question": "A plane $P$ slices through a cube of volume 1 with a cross-section in the shape of a regular hexagon. This cube also has an inscribed sphere, whose intersection with $P$ is a circle. What is the area of the region inside the regular hexagon but outside the circle?",
      "answer": "(3 \\sqrt{3}-\\pi) / 4"
    },
    {
      "question": "In how many ways can 6 purple balls and 6 green balls be placed into a $4 \\times 4$ grid of boxes such that every row and column contains two balls of one color and one ball of the other color? Only one ball may be placed in each box, and rotations and reflections of a single configuration are considered different.",
      "answer": "5184"
    },
    {
      "question": "What is the value of $2^{4}-2^{3}$?",
      "answer": "2^{3}"
    },
    {
      "question": "The sum of four different positive integers is 100. The largest of these four integers is $n$. What is the smallest possible value of $n$?",
      "answer": "27"
    },
    {
      "question": "Triangle $A B C$ has $A B=1, B C=\\sqrt{7}$, and $C A=\\sqrt{3}$. Let $\\ell_{1}$ be the line through $A$ perpendicular to $A B, \\ell_{2}$ the line through $B$ perpendicular to $A C$, and $P$ the point of intersection of $\\ell_{1}$ and $\\ell_{2}$. Find $P C$.",
      "answer": "3"
    },
    {
      "question": "Which of the following integers is equal to a perfect square: $2^{3}$, $3^{5}$, $4^{7}$, $5^{9}$, $6^{11}$?",
      "answer": "4^{7}"
    },
    {
      "question": "Call the pentominoes found in the last problem square pentominoes. Just like dominos and ominos can be used to tile regions of the plane, so can square pentominoes. In particular, a square pentomino tiling of a region of the plane is a way of covering it (and only it) completely by nonoverlapping square pentominoes. How many square pentomino tilings are there of a 12-by-12 rectangle?",
      "answer": "0"
    },
    {
      "question": "Anna walked at a constant rate. If she walked 600 metres in 4 minutes, how far did she walk in 6 minutes?",
      "answer": "900"
    },
    {
      "question": "What is the smallest possible perimeter of a triangle whose side lengths are all squares of distinct positive integers?",
      "answer": "77"
    },
    {
      "question": "Triangle $A B C$ has side lengths $A B=15, B C=18, C A=20$. Extend $C A$ and $C B$ to points $D$ and $E$ respectively such that $D A=A B=B E$. Line $A B$ intersects the circumcircle of $C D E$ at $P$ and $Q$. Find the length of $P Q$.",
      "answer": "37"
    },
    {
      "question": "Rachelle picks a positive integer \\(a\\) and writes it next to itself to obtain a new positive integer \\(b\\). For instance, if \\(a=17\\), then \\(b=1717\\). To her surprise, she finds that \\(b\\) is a multiple of \\(a^{2}\\). Find the product of all the possible values of \\(\\frac{b}{a^{2}}\\).",
      "answer": "77"
    },
    {
      "question": "Convex quadrilateral $B C D E$ lies in the plane. Lines $E B$ and $D C$ intersect at $A$, with $A B=2$, $A C=5, A D=200, A E=500$, and $\\cos \\angle B A C=\\frac{7}{9}$. What is the largest number of nonoverlapping circles that can lie in quadrilateral $B C D E$ such that all of them are tangent to both lines $B E$ and $C D$ ?",
      "answer": "5"
    },
    {
      "question": "What fraction of the entire wall is painted red if Matilda paints half of her section red and Ellie paints one third of her section red?",
      "answer": "\\frac{5}{12}"
    },
    {
      "question": "A subset of a student group is called an [i]ideal company[/i] if\n1) in this subset, all girls are liked by all young men,\n2) no one can be added to this subset without violating condition $1$.\nIn a certain group, $9$ female students and $15$ students study. Warden of the group made a list of all kinds of ideal companies in this group. What is the largest number of companies on this list?",
      "answer": "512"
    },
    {
      "question": "There are six empty slots corresponding to the digits of a six-digit number. Claire and William take turns rolling a standard six-sided die, with Claire going first. They alternate with each roll until they have each rolled three times. After a player rolls, they place the number from their die roll into a remaining empty slot of their choice. Claire wins if the resulting six-digit number is divisible by 6, and William wins otherwise. If both players play optimally, compute the probability that Claire wins.",
      "answer": "\\frac{43}{192}"
    },
    {
      "question": "Let $a, b$ be integers chosen independently and uniformly at random from the set $\\{0,1,2, \\ldots, 80\\}$. Compute the expected value of the remainder when the binomial coefficient $\\binom{a}{b}=\\frac{a!}{b!(a-b)!}$ is divided by 3.",
      "answer": "\\frac{1816}{6561}"
    },
    {
      "question": "For how many positive integers $k$ do the lines with equations $9x+4y=600$ and $kx-4y=24$ intersect at a point whose coordinates are positive integers?",
      "answer": "7"
    },
    {
      "question": "We are given some similar triangles. Their areas are $1^{2}, 3^{2}, 5^{2} \\ldots$, and $49^{2}$. If the smallest triangle has a perimeter of 4, what is the sum of all the triangles' perimeters?",
      "answer": "2500"
    },
    {
      "question": "Given complex number $z$, define sequence $z_{0}, z_{1}, z_{2}, \\ldots$ as $z_{0}=z$ and $z_{n+1}=2 z_{n}^{2}+2 z_{n}$ for $n \\geq 0$. Given that $z_{10}=2017$, find the minimum possible value of $|z|$.",
      "answer": "\\frac{\\sqrt[1024]{4035}-1}{2}"
    },
    {
      "question": "Katie has a fair 2019-sided die with sides labeled $1,2, \\ldots, 2019$. After each roll, she replaces her $n$-sided die with an $(n+1)$-sided die having the $n$ sides of her previous die and an additional side with the number she just rolled. What is the probability that Katie's $2019^{\\text {th }}$ roll is a 2019?",
      "answer": "\\frac{1}{2019}"
    },
    {
      "question": "An equiangular hexagon has side lengths $1,1, a, 1,1, a$ in that order. Given that there exists a circle that intersects the hexagon at 12 distinct points, we have $M<a<N$ for some real numbers $M$ and $N$. Determine the minimum possible value of the ratio $\\frac{N}{M}$.",
      "answer": "\\frac{3 \\sqrt{3}+3}{2}"
    },
    {
      "question": "Kelvin the Frog was bored in math class one day, so he wrote all ordered triples $(a, b, c)$ of positive integers such that $a b c=2310$ on a sheet of paper. Find the sum of all the integers he wrote down. In other words, compute $$\\sum_{\\substack{a b c=2310 \\\\ a, b, c \\in \\mathbb{N}}}(a+b+c)$$ where $\\mathbb{N}$ denotes the positive integers.",
      "answer": "49140"
    },
    {
      "question": "Suppose that $m$ and $n$ are positive integers with $\\sqrt{7+\\sqrt{48}}=m+\\sqrt{n}$. What is the value of $m^{2}+n^{2}$?",
      "answer": "13"
    },
    {
      "question": "Estimate $A$, the number of times an 8-digit number appears in Pascal's triangle. An estimate of $E$ earns $\\max (0,\\lfloor 20-|A-E| / 200\\rfloor)$ points.",
      "answer": "180020660"
    },
    {
      "question": "If $x$ and $y$ are positive integers with $xy = 6$, what is the sum of all possible values of $\\frac{2^{x+y}}{2^{x-y}}$?",
      "answer": "4180"
    },
    {
      "question": "The polynomial $f(x)=x^{3}-3 x^{2}-4 x+4$ has three real roots $r_{1}, r_{2}$, and $r_{3}$. Let $g(x)=x^{3}+a x^{2}+b x+c$ be the polynomial which has roots $s_{1}, s_{2}$, and $s_{3}$, where $s_{1}=r_{1}+r_{2} z+r_{3} z^{2}$, $s_{2}=r_{1} z+r_{2} z^{2}+r_{3}, s_{3}=r_{1} z^{2}+r_{2}+r_{3} z$, and $z=\\frac{-1+i \\sqrt{3}}{2}$. Find the real part of the sum of the coefficients of $g(x)$.",
      "answer": "-26"
    },
    {
      "question": "Find the smallest positive integer $n$ for which $$1!2!\\cdots(n-1)!>n!^{2}$$",
      "answer": "8"
    },
    {
      "question": "Four points are independently chosen uniformly at random from the interior of a regular dodecahedron. What is the probability that they form a tetrahedron whose interior contains the dodecahedron's center?",
      "answer": "\\[\n\\frac{1}{8}\n\\]"
    },
    {
      "question": "Compute the smallest positive integer $n$ for which $\\sqrt{100+\\sqrt{n}}+\\sqrt{100-\\sqrt{n}}$ is an integer.",
      "answer": "6156"
    },
    {
      "question": "Alberto, Bernardo, and Carlos are collectively listening to three different songs. Each is simultaneously listening to exactly two songs, and each song is being listened to by exactly two people. In how many ways can this occur?",
      "answer": "6"
    },
    {
      "question": "The number $27,000,001$ has exactly four prime factors. Find their sum.",
      "answer": "652"
    },
    {
      "question": "A cafe has 3 tables and 5 individual counter seats. People enter in groups of size between 1 and 4, inclusive, and groups never share a table. A group of more than 1 will always try to sit at a table, but will sit in counter seats if no tables are available. Conversely, a group of 1 will always try to sit at the counter first. One morning, $M$ groups consisting of a total of $N$ people enter and sit down. Then, a single person walks in, and realizes that all the tables and counter seats are occupied by some person or group. What is the minimum possible value of $M+N$?",
      "answer": "16"
    },
    {
      "question": "Connie has a number of gold bars, all of different weights. She gives the 24 lightest bars, which weigh $45 \\%$ of the total weight, to Brennan. She gives the 13 heaviest bars, which weigh $26 \\%$ of the total weight, to Maya. How many bars did Blair receive?",
      "answer": "15"
    },
    {
      "question": "A semicircle with radius 2021 has diameter $AB$ and center $O$. Points $C$ and $D$ lie on the semicircle such that $\\angle AOC < \\angle AOD = 90^{\\circ}$. A circle of radius $r$ is inscribed in the sector bounded by $OA$ and $OC$ and is tangent to the semicircle at $E$. If $CD=CE$, compute $\\lfloor r \\rfloor$.",
      "answer": "673"
    },
    {
      "question": "Let $S$ be a set of consecutive positive integers such that for any integer $n$ in $S$, the sum of the digits of $n$ is not a multiple of 11. Determine the largest possible number of elements of $S$.",
      "answer": "38"
    },
    {
      "question": "In convex quadrilateral $ ABCD$, $ AB\\equal{}a$, $ BC\\equal{}b$, $ CD\\equal{}c$, $ DA\\equal{}d$, $ AC\\equal{}e$, $ BD\\equal{}f$. If $ \\max \\{a,b,c,d,e,f \\}\\equal{}1$, then find the maximum value of $ abcd$.",
      "answer": "2 - \\sqrt{3}"
    },
    {
      "question": "Let $X$ be the collection of all functions $f:\\{0,1, \\ldots, 2016\\} \\rightarrow\\{0,1, \\ldots, 2016\\}$. Compute the number of functions $f \\in X$ such that $$\\max _{g \\in X}\\left(\\min _{0 \\leq i \\leq 2016}(\\max (f(i), g(i)))-\\max _{0 \\leq i \\leq 2016}(\\min (f(i), g(i)))\\right)=2015$$",
      "answer": "2 \\cdot\\left(3^{2017}-2^{2017}\\right)"
    },
    {
      "question": "Milan has a bag of 2020 red balls and 2021 green balls. He repeatedly draws 2 balls out of the bag uniformly at random. If they are the same color, he changes them both to the opposite color and returns them to the bag. If they are different colors, he discards them. Eventually the bag has 1 ball left. Let $p$ be the probability that it is green. Compute $\\lfloor 2021 p \\rfloor$.",
      "answer": "2021"
    },
    {
      "question": "As shown below, there is a $40\\times30$ paper with a filled $10\\times5$ rectangle inside of it. We want to cut out the filled rectangle from the paper using four straight cuts. Each straight cut is a straight line that divides the paper into two pieces, and we keep the piece containing the filled rectangle. The goal is to minimize the total length of the straight cuts. How to achieve this goal, and what is that minimized length? Show the correct cuts and write the final answer. There is no need to prove the answer.",
      "answer": "65"
    },
    {
      "question": "Lucy starts by writing $s$ integer-valued $2022$-tuples on a blackboard. After doing that, she can take any two (not necessarily distinct) tuples $\\mathbf{v}=(v_1,\\ldots,v_{2022})$ and $\\mathbf{w}=(w_1,\\ldots,w_{2022})$ that she has already written, and apply one of the following operations to obtain a new tuple:\n\\begin{align*}\n\\mathbf{v}+\\mathbf{w}&=(v_1+w_1,\\ldots,v_{2022}+w_{2022}) \\\\\n\\mathbf{v} \\lor \\mathbf{w}&=(\\max(v_1,w_1),\\ldots,\\max(v_{2022},w_{2022}))\n\\end{align*}\nand then write this tuple on the blackboard.\n\nIt turns out that, in this way, Lucy can write any integer-valued $2022$-tuple on the blackboard after finitely many steps. What is the smallest possible number $s$ of tuples that she initially wrote?",
      "answer": "3"
    },
    {
      "question": "Elisenda has a piece of paper in the shape of a triangle with vertices $A, B$, and $C$ such that $A B=42$. She chooses a point $D$ on segment $A C$, and she folds the paper along line $B D$ so that $A$ lands at a point $E$ on segment $B C$. Then, she folds the paper along line $D E$. When she does this, $B$ lands at the midpoint of segment $D C$. Compute the perimeter of the original unfolded triangle.",
      "answer": "168+48 \\sqrt{7}"
    },
    {
      "question": "Let \\mathbb{N} denote the natural numbers. Compute the number of functions $f: \\mathbb{N} \\rightarrow\\{0,1, \\ldots, 16\\}$ such that $$f(x+17)=f(x) \\quad \\text { and } \\quad f\\left(x^{2}\\right) \\equiv f(x)^{2}+15 \\quad(\\bmod 17)$$ for all integers $x \\geq 1$",
      "answer": "12066"
    },
    {
      "question": "What is the probability that in a randomly chosen arrangement of the numbers and letters in \"HMMT2005,\" one can read either \"HMMT\" or \"2005\" from left to right?",
      "answer": "23/144"
    },
    {
      "question": "Find the shortest distance from the line $3 x+4 y=25$ to the circle $x^{2}+y^{2}=6 x-8 y$.",
      "answer": "7 / 5"
    },
    {
      "question": "Compute the number of ways to select 99 cells of a $19 \\times 19$ square grid such that no two selected cells share an edge or vertex.",
      "answer": "1000"
    },
    {
      "question": "Niffy's favorite number is a positive integer, and Stebbysaurus is trying to guess what it is. Niffy tells her that when expressed in decimal without any leading zeros, her favorite number satisfies the following: - Adding 1 to the number results in an integer divisible by 210 . - The sum of the digits of the number is twice its number of digits. - The number has no more than 12 digits. - The number alternates in even and odd digits. Given this information, what are all possible values of Niffy's favorite number?",
      "answer": "1010309"
    },
    {
      "question": "Let $X$ be the number of sequences of integers $a_{1}, a_{2}, \\ldots, a_{2047}$ that satisfy all of the following properties: - Each $a_{i}$ is either 0 or a power of 2 . - $a_{i}=a_{2 i}+a_{2 i+1}$ for $1 \\leq i \\leq 1023$ - $a_{1}=1024$. Find the remainder when $X$ is divided by 100 .",
      "answer": "15"
    },
    {
      "question": "We have 10 points on a line A_{1}, A_{2} \\cdots A_{10} in that order. Initially there are n chips on point A_{1}. Now we are allowed to perform two types of moves. Take two chips on A_{i}, remove them and place one chip on A_{i+1}, or take two chips on A_{i+1}, remove them, and place a chip on A_{i+2} and A_{i}. Find the minimum possible value of n such that it is possible to get a chip on A_{10} through a sequence of moves.",
      "answer": "46"
    },
    {
      "question": "There is a $6 \\times 6$ grid of lights. There is a switch at the top of each column and on the left of each row. A light will only turn on if the switches corresponding to both its column and its row are in the \"on\" position. Compute the number of different configurations of lights.",
      "answer": "3970"
    },
    {
      "question": "Let $f(n)$ be the number of distinct prime divisors of $n$ less than 6. Compute $$\\sum_{n=1}^{2020} f(n)^{2}$$",
      "answer": "3431"
    },
    {
      "question": "Two unit squares $S_{1}$ and $S_{2}$ have horizontal and vertical sides. Let $x$ be the minimum distance between a point in $S_{1}$ and a point in $S_{2}$, and let $y$ be the maximum distance between a point in $S_{1}$ and a point in $S_{2}$. Given that $x=5$, the difference between the maximum and minimum possible values for $y$ can be written as $a+b \\sqrt{c}$, where $a, b$, and $c$ are integers and $c$ is positive and square-free. Find $100 a+10 b+c$.",
      "answer": "472"
    },
    {
      "question": "Marcus and four of his relatives are at a party. Each pair of the five people are either friends or enemies. For any two enemies, there is no person that they are both friends with. In how many ways is this possible?",
      "answer": "52"
    },
    {
      "question": "Find the remainder when $1^{2}+3^{2}+5^{2}+\\cdots+99^{2}$ is divided by 1000.",
      "answer": "650"
    },
    {
      "question": "New this year at HMNT: the exciting game of $R N G$ baseball! In RNG baseball, a team of infinitely many people play on a square field, with a base at each vertex; in particular, one of the bases is called the home base. Every turn, a new player stands at home base and chooses a number $n$ uniformly at random from \\{0,1,2,3,4\\}. Then, the following occurs: - If $n>0$, then the player and everyone else currently on the field moves (counterclockwise) around the square by $n$ bases. However, if in doing so a player returns to or moves past the home base, he/she leaves the field immediately and the team scores one point. - If $n=0$ (a strikeout), then the game ends immediately; the team does not score any more points. What is the expected number of points that a given team will score in this game?",
      "answer": "\\frac{409}{125}"
    },
    {
      "question": "Find all real numbers $x$ such that $$x^{2}+\\left\\lfloor\\frac{x}{2}\\right\\rfloor+\\left\\lfloor\\frac{x}{3}\\right\\rfloor=10$$",
      "answer": "-\\sqrt{14}"
    },
    {
      "question": "Consider the function $z(x, y)$ describing the paraboloid $z=(2 x-y)^{2}-2 y^{2}-3 y$. Archimedes and Brahmagupta are playing a game. Archimedes first chooses $x$. Afterwards, Brahmagupta chooses $y$. Archimedes wishes to minimize $z$ while Brahmagupta wishes to maximize $z$. Assuming that Brahmagupta will play optimally, what value of $x$ should Archimedes choose?",
      "answer": "-\\frac{3}{8}"
    },
    {
      "question": "A bag contains nine blue marbles, ten ugly marbles, and one special marble. Ryan picks marbles randomly from this bag with replacement until he draws the special marble. He notices that none of the marbles he drew were ugly. Given this information, what is the expected value of the number of total marbles he drew?",
      "answer": "\\frac{20}{11}"
    },
    {
      "question": "At the beginning of the first day, a box contains 1 black ball, 1 gold ball, and no other balls. At the end of each day, for each gold ball in the box, 2 black balls and 1 gold ball are added to the box. If no balls are removed from the box, how many balls are in the box at the end of the seventh day?",
      "answer": "383"
    },
    {
      "question": "Regular hexagon $P_{1} P_{2} P_{3} P_{4} P_{5} P_{6}$ has side length 2. For $1 \\leq i \\leq 6$, let $C_{i}$ be a unit circle centered at $P_{i}$ and $\\ell_{i}$ be one of the internal common tangents of $C_{i}$ and $C_{i+2}$, where $C_{7}=C_{1}$ and $C_{8}=C_{2}$. Assume that the lines $\\{\\ell_{1}, \\ell_{2}, \\ell_{3}, \\ell_{4}, \\ell_{5}, \\ell_{6}\\}$ bound a regular hexagon. The area of this hexagon can be expressed as $\\sqrt{\\frac{a}{b}}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
      "answer": "1603"
    },
    {
      "question": "Let $a \\geq b \\geq c$ be real numbers such that $$\\begin{aligned} a^{2} b c+a b^{2} c+a b c^{2}+8 & =a+b+c \\\\ a^{2} b+a^{2} c+b^{2} c+b^{2} a+c^{2} a+c^{2} b+3 a b c & =-4 \\\\ a^{2} b^{2} c+a b^{2} c^{2}+a^{2} b c^{2} & =2+a b+b c+c a \\end{aligned}$$ If $a+b+c>0$, then compute the integer nearest to $a^{5}$.",
      "answer": "1279"
    },
    {
      "question": "If $x=3$, $y=2x$, and $z=3y$, what is the value of $z$?",
      "answer": "18"
    },
    {
      "question": "Compute the number of ways a non-self-intersecting concave quadrilateral can be drawn in the plane such that two of its vertices are $(0,0)$ and $(1,0)$, and the other two vertices are two distinct lattice points $(a, b),(c, d)$ with $0 \\leq a, c \\leq 59$ and $1 \\leq b, d \\leq 5$.",
      "answer": "366"
    },
    {
      "question": "Find the smallest value that the expression takes $x^4 + y^4 - x^2y - xy^2$, for positive numbers $x$ and $y$ satisfying $x + y \\le 1$.",
      "answer": "-\\frac{1}{8}"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=2, C A=3, B C=4$. Let $D$ be the point diametrically opposite $A$ on the circumcircle of $A B C$, and let $E$ lie on line $A D$ such that $D$ is the midpoint of $\\overline{A E}$. Line $l$ passes through $E$ perpendicular to $\\overline{A E}$, and $F$ and $G$ are the intersections of the extensions of $\\overline{A B}$ and $\\overline{A C}$ with $l$. Compute $F G$.",
      "answer": "\\frac{1024}{45}"
    },
    {
      "question": "Determine the number of four-digit integers $n$ such that $n$ and $2n$ are both palindromes.",
      "answer": "20"
    },
    {
      "question": "A digital clock shows the time $4:56$. How many minutes will pass until the clock next shows a time in which all of the digits are consecutive and are in increasing order?",
      "answer": "458"
    },
    {
      "question": "Suppose that $PQRS TUVW$ is a regular octagon. There are 70 ways in which four of its sides can be chosen at random. If four of its sides are chosen at random and each of these sides is extended infinitely in both directions, what is the probability that they will meet to form a quadrilateral that contains the octagon?",
      "answer": "\\frac{19}{35}"
    },
    {
      "question": "A class of 30 students was asked what they did on their winter holiday. 20 students said that they went skating. 9 students said that they went skiing. Exactly 5 students said that they went skating and went skiing. How many students did not go skating and did not go skiing?",
      "answer": "6"
    },
    {
      "question": "A rectangle has width $x$ and length $y$. The rectangle is cut along the horizontal and vertical dotted lines to produce four smaller rectangles. The sum of the perimeters of these four rectangles is 24. What is the value of $x+y$?",
      "answer": "6"
    },
    {
      "question": "Rachel has two indistinguishable tokens, and places them on the first and second square of a $1 \\times 6$ grid of squares. She can move the pieces in two ways: If a token has a free square in front of it, then she can move this token one square to the right. If the square immediately to the right of a token is occupied by the other token, then she can 'leapfrog' the first token; she moves the first token two squares to the right, over the other token, so that it is on the square immediately to the right of the other token. If a token reaches the 6th square, then it cannot move forward any more, and Rachel must move the other one until it reaches the 5th square. How many different sequences of moves for the tokens can Rachel make so that the two tokens end up on the 5th square and the 6th square?",
      "answer": "42"
    },
    {
      "question": "If $a_{1}=1, a_{2}=0$, and $a_{n+1}=a_{n}+\\frac{a_{n+2}}{2}$ for all $n \\geq 1$, compute $a_{2004}$.",
      "answer": "-2^{1002}"
    },
    {
      "question": "Let $A B C$ be a triangle whose incircle has center $I$ and is tangent to $\\overline{B C}, \\overline{C A}, \\overline{A B}$, at $D, E, F$. Denote by $X$ the midpoint of major arc $\\widehat{B A C}$ of the circumcircle of $A B C$. Suppose $P$ is a point on line $X I$ such that $\\overline{D P} \\perp \\overline{E F}$. Given that $A B=14, A C=15$, and $B C=13$, compute $D P$.",
      "answer": "\\frac{4 \\sqrt{5}}{5}"
    },
    {
      "question": "Two integers are relatively prime if they don't share any common factors, i.e. if their greatest common divisor is 1. Define $\\varphi(n)$ as the number of positive integers that are less than $n$ and relatively prime to $n$. Define $\\varphi_{d}(n)$ as the number of positive integers that are less than $d n$ and relatively prime to $n$. What is the least $n$ such that $\\varphi_{x}(n)=64000$, where $x=\\varphi_{y}(n)$, where $y=\\varphi(n)$?",
      "answer": "41"
    },
    {
      "question": "What is the remainder when the integer equal to \\( QT^2 \\) is divided by 100, given that \\( QU = 9 \\sqrt{33} \\) and \\( UT = 40 \\)?",
      "answer": "9"
    },
    {
      "question": "Three different numbers from the list $2, 3, 4, 6$ have a sum of 11. What is the product of these numbers?",
      "answer": "36"
    },
    {
      "question": "The skeletal structure of coronene, a hydrocarbon with the chemical formula $\\mathrm{C}_{24} \\mathrm{H}_{12}$, is shown below. Each line segment between two atoms is at least a single bond. However, since each carbon (C) requires exactly four bonds connected to it and each hydrogen $(\\mathrm{H})$ requires exactly one bond, some of the line segments are actually double bonds. How many arrangements of single/double bonds are there such that the above requirements are satisfied?",
      "answer": "20"
    },
    {
      "question": "Find the minimum positive integer $k$ such that $f(n+k) \\equiv f(n)(\\bmod 23)$ for all integers $n$.",
      "answer": "2530"
    },
    {
      "question": "The product of $N$ consecutive four-digit positive integers is divisible by $2010^{2}$. What is the least possible value of $N$?",
      "answer": "5"
    },
    {
      "question": "Given two distinct points $A, B$ and line $\\ell$ that is not perpendicular to $A B$, what is the maximum possible number of points $P$ on $\\ell$ such that $A B P$ is an isosceles triangle?",
      "answer": "5"
    },
    {
      "question": "For each positive integer $n$, define $s(n)$ to equal the sum of the digits of $n$. The number of integers $n$ with $100 \\leq n \\leq 999$ and $7 \\leq s(n) \\leq 11$ is $S$. What is the integer formed by the rightmost two digits of $S$?",
      "answer": "24"
    },
    {
      "question": "Let \\(a_{1}, a_{2}, \\ldots\\) be an infinite sequence of integers such that \\(a_{i}\\) divides \\(a_{i+1}\\) for all \\(i \\geq 1\\), and let \\(b_{i}\\) be the remainder when \\(a_{i}\\) is divided by 210. What is the maximal number of distinct terms in the sequence \\(b_{1}, b_{2}, \\ldots\\)?",
      "answer": "127"
    },
    {
      "question": "Charlie is making a necklace with yellow beads and green beads. She has already used 4 green beads and 0 yellow beads. How many yellow beads will she have to add so that $\frac{4}{5}$ of the total number of beads are yellow?",
      "answer": "16"
    },
    {
      "question": "The mathematician John is having trouble remembering his girlfriend Alicia's 7-digit phone number. He remembers that the first four digits consist of one 1, one 2, and two 3s. He also remembers that the fifth digit is either a 4 or 5. While he has no memory of the sixth digit, he remembers that the seventh digit is 9 minus the sixth digit. If this is all the information he has, how many phone numbers does he have to try if he is to make sure he dials the correct number?",
      "answer": "240"
    },
    {
      "question": "Compute the number of positive integers that divide at least two of the integers in the set $\\{1^{1}, 2^{2}, 3^{3}, 4^{4}, 5^{5}, 6^{6}, 7^{7}, 8^{8}, 9^{9}, 10^{10}\\}$.",
      "answer": "22"
    },
    {
      "question": "If Kai will celebrate his 25th birthday in March 2020, in what year was Kai born?",
      "answer": "1995"
    },
    {
      "question": "Compute $$\\sum_{\\substack{a+b+c=12 \\\\ a \\geq 6, b, c \\geq 0}} \\frac{a!}{b!c!(a-b-c)!}$$ where the sum runs over all triples of nonnegative integers $(a, b, c)$ such that $a+b+c=12$ and $a \\geq 6$.",
      "answer": "2731"
    },
    {
      "question": "For any finite sequence of positive integers \\pi, let $S(\\pi)$ be the number of strictly increasing subsequences in \\pi with length 2 or more. For example, in the sequence $\\pi=\\{3,1,2,4\\}$, there are five increasing sub-sequences: $\\{3,4\\},\\{1,2\\},\\{1,4\\},\\{2,4\\}$, and $\\{1,2,4\\}$, so $S(\\pi)=5$. In an eight-player game of Fish, Joy is dealt six cards of distinct values, which she puts in a random order \\pi from left to right in her hand. Determine $\\sum_{\\pi} S(\\pi)$ where the sum is taken over all possible orders \\pi of the card values.",
      "answer": "8287"
    },
    {
      "question": "Compute the number of tuples $\\left(a_{0}, a_{1}, a_{2}, a_{3}, a_{4}, a_{5}\\right)$ of (not necessarily positive) integers such that $a_{i} \\leq i$ for all $0 \\leq i \\leq 5$ and $$a_{0}+a_{1}+\\cdots+a_{5}=6$$",
      "answer": "2002"
    },
    {
      "question": "Suppose that point $D$ lies on side $B C$ of triangle $A B C$ such that $A D$ bisects $\\angle B A C$, and let $\\ell$ denote the line through $A$ perpendicular to $A D$. If the distances from $B$ and $C$ to $\\ell$ are 5 and 6 , respectively, compute $A D$.",
      "answer": "\\frac{60}{11}"
    },
    {
      "question": "What is the 3-digit number formed by the $9998^{\\text {th }}$ through $10000^{\\text {th }}$ digits after the decimal point in the decimal expansion of \\frac{1}{998}$ ?",
      "answer": "042"
    },
    {
      "question": "Wesyu is a farmer, and she's building a cao (a relative of the cow) pasture. She starts with a triangle $A_{0} A_{1} A_{2}$ where angle $A_{0}$ is $90^{\\circ}$, angle $A_{1}$ is $60^{\\circ}$, and $A_{0} A_{1}$ is 1. She then extends the pasture. First, she extends $A_{2} A_{0}$ to $A_{3}$ such that $A_{3} A_{0}=\\frac{1}{2} A_{2} A_{0}$ and the new pasture is triangle $A_{1} A_{2} A_{3}$. Next, she extends $A_{3} A_{1}$ to $A_{4}$ such that $A_{4} A_{1}=\\frac{1}{6} A_{3} A_{1}$. She continues, each time extending $A_{n} A_{n-2}$ to $A_{n+1}$ such that $A_{n+1} A_{n-2}=\\frac{1}{2^{n}-2} A_{n} A_{n-2}$. What is the smallest $K$ such that her pasture never exceeds an area of $K$?",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "Suppose $a$ and $b$ are positive integers. Isabella and Vidur both fill up an $a \\times b$ table. Isabella fills it up with numbers $1,2, \\ldots, a b$, putting the numbers $1,2, \\ldots, b$ in the first row, $b+1, b+2, \\ldots, 2 b$ in the second row, and so on. Vidur fills it up like a multiplication table, putting $i j$ in the cell in row $i$ and column $j$. Isabella sums up the numbers in her grid, and Vidur sums up the numbers in his grid; the difference between these two quantities is 1200. Compute $a+b$.",
      "answer": "21"
    },
    {
      "question": "Consider a $4 \\times 4$ grid of squares, each of which are originally colored red. Every minute, Piet can jump on one of the squares, changing the color of it and any adjacent squares (two squares are adjacent if they share a side) to blue. What is the minimum number of minutes it will take Piet to change the entire grid to blue?",
      "answer": "4"
    },
    {
      "question": "For how many integers $m$, with $1 \\leq m \\leq 30$, is it possible to find a value of $n$ so that $n!$ ends with exactly $m$ zeros?",
      "answer": "24"
    },
    {
      "question": "An $E$-shape is a geometric figure in the two-dimensional plane consisting of three rays pointing in the same direction, along with a line segment such that the endpoints of the rays all lie on the segment, the segment is perpendicular to all three rays, both endpoints of the segment are endpoints of rays. Suppose two $E$-shapes intersect each other $N$ times in the plane for some positive integer $N$. Compute the maximum possible value of $N$.",
      "answer": "11"
    },
    {
      "question": "Regular octagon CHILDREN has area 1. Determine the area of quadrilateral LINE.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Compute the value of $1^{25}+2^{24}+3^{23}+\\ldots+24^{2}+25^{1}$. If your answer is $A$ and the correct answer is $C$, then your score on this problem will be $\\left\\lfloor 25 \\mathrm{~min}\\left(\\left(\\frac{A}{C}\\right)^{2},\\left(\\frac{C}{A}\\right)^{2}\\right)\\right\\rfloor$.",
      "answer": "66071772829247409"
    },
    {
      "question": "You have a length of string and 7 beads in the 7 colors of the rainbow. You place the beads on the string as follows - you randomly pick a bead that you haven't used yet, then randomly add it to either the left end or the right end of the string. What is the probability that, at the end, the colors of the beads are the colors of the rainbow in order? (The string cannot be flipped, so the red bead must appear on the left side and the violet bead on the right side.)",
      "answer": "\\frac{1}{5040}"
    },
    {
      "question": "If $x=11, y=8$, and $2x+3z=5y$, what is the value of $z$?",
      "answer": "6"
    },
    {
      "question": "Compute the maximum number of sides of a polygon that is the cross-section of a regular hexagonal prism.",
      "answer": "8"
    },
    {
      "question": "Find the number of pairs $(a, b)$ of positive integers with the property that the greatest common divisor of $a$ and $ b$ is equal to $1\\cdot 2 \\cdot 3\\cdot ... \\cdot50$, and the least common multiple of $a$ and $ b$ is $1^2 \\cdot 2^2 \\cdot 3^2\\cdot ... \\cdot 50^2$.",
      "answer": "32768"
    },
    {
      "question": "Morgan uses a spreadsheet to create a table of values. In the first column, she lists the positive integers from 1 to 400. She then puts integers in the second column in the following way: if the integer in the first column of a given row is $n$, the number in the second column of that row is $3 n+1$. Which of the following integers does not appear in the second column: 31, 94, 131, 331, 907?",
      "answer": "131"
    },
    {
      "question": "Ben \"One Hunna Dolla\" Franklin is flying a kite KITE such that $I E$ is the perpendicular bisector of $K T$. Let $I E$ meet $K T$ at $R$. The midpoints of $K I, I T, T E, E K$ are $A, N, M, D$, respectively. Given that $[M A K E]=18, I T=10,[R A I N]=4$, find $[D I M E]$.",
      "answer": "16"
    },
    {
      "question": "Let $A B C$ be a triangle with $A B=13, B C=14$, and $C A=15$. Let $D$ be the foot of the altitude from $A$ to $B C$. The inscribed circles of triangles $A B D$ and $A C D$ are tangent to $A D$ at $P$ and $Q$, respectively, and are tangent to $B C$ at $X$ and $Y$, respectively. Let $P X$ and $Q Y$ meet at $Z$. Determine the area of triangle $X Y Z$.",
      "answer": "\\frac{25}{4}"
    },
    {
      "question": "Flat Albert and his buddy Mike are watching the game on Sunday afternoon. Albert is drinking lemonade from a two-dimensional cup which is an isosceles triangle whose height and base measure 9 cm and 6 cm; the opening of the cup corresponds to the base, which points upwards. Every minute after the game begins, the following takes place: if $n$ minutes have elapsed, Albert stirs his drink vigorously and takes a sip of height $\\frac{1}{n^{2}} \\mathrm{~cm}$. Shortly afterwards, while Albert is busy watching the game, Mike adds cranberry juice to the cup until it's once again full in an attempt to create Mike's cranberry lemonade. Albert takes sips precisely every minute, and his first sip is exactly one minute after the game begins. After an infinite amount of time, let $A$ denote the amount of cranberry juice that has been poured (in square centimeters). Find the integer nearest $\\frac{27}{\\pi^{2}} A$.",
      "answer": "26"
    },
    {
      "question": "Find the largest positive integer $k{}$ for which there exists a convex polyhedron $\\mathcal{P}$ with 2022 edges, which satisfies the following properties:\n[list]\n[*]The degrees of the vertices of $\\mathcal{P}$ don’t differ by more than one, and\n[*]It is possible to colour the edges of $\\mathcal{P}$ with $k{}$ colours such that for every colour $c{}$, and every pair of vertices $(v_1, v_2)$ of $\\mathcal{P}$, there is a monochromatic path between $v_1$ and $v_2$ in the colour $c{}$.\n[/list]\n[i]Viktor Simjanoski, Macedonia[/i]",
      "answer": "2"
    },
    {
      "question": "Let $f(x)=x^{2}-2 x$. How many distinct real numbers $c$ satisfy $f(f(f(f(c))))=3$ ?",
      "answer": "9"
    },
    {
      "question": "How many of the numbers in Grace's sequence, starting from 43 and each number being 4 less than the previous one, are positive?",
      "answer": "11"
    },
    {
      "question": "Sam rolls a fair four-sided die containing the numbers $1,2,3$, and 4. Tyler rolls a fair six-sided die containing the numbers $1,2,3,4,5$, and 6. What is the probability that Sam rolls a larger number than Tyler?",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Let $G_{1} G_{2} G_{3}$ be a triangle with $G_{1} G_{2}=7, G_{2} G_{3}=13$, and $G_{3} G_{1}=15$. Let $G_{4}$ be a point outside triangle $G_{1} G_{2} G_{3}$ so that ray $\\overrightarrow{G_{1} G_{4}}$ cuts through the interior of the triangle, $G_{3} G_{4}=G_{4} G_{2}$, and $\\angle G_{3} G_{1} G_{4}=30^{\\circ}$. Let $G_{3} G_{4}$ and $G_{1} G_{2}$ meet at $G_{5}$. Determine the length of segment $G_{2} G_{5}$.",
      "answer": "\\frac{169}{23}"
    },
    {
      "question": "Let $S=\\left\\{p_{1} p_{2} \\cdots p_{n} \\mid p_{1}, p_{2}, \\ldots, p_{n}\\right.$ are distinct primes and $\\left.p_{1}, \\ldots, p_{n}<30\\right\\}$. Assume 1 is in $S$. Let $a_{1}$ be an element of $S$. We define, for all positive integers $n$ : $$ \\begin{gathered} a_{n+1}=a_{n} /(n+1) \\quad \\text { if } a_{n} \\text { is divisible by } n+1 \\\\ a_{n+1}=(n+2) a_{n} \\quad \\text { if } a_{n} \\text { is not divisible by } n+1 \\end{gathered} $$ How many distinct possible values of $a_{1}$ are there such that $a_{j}=a_{1}$ for infinitely many $j$ 's?",
      "answer": "512"
    },
    {
      "question": "Let $\\Delta A_{1} B_{1} C$ be a triangle with $\\angle A_{1} B_{1} C=90^{\\circ}$ and $\\frac{C A_{1}}{C B_{1}}=\\sqrt{5}+2$. For any $i \\geq 2$, define $A_{i}$ to be the point on the line $A_{1} C$ such that $A_{i} B_{i-1} \\perp A_{1} C$ and define $B_{i}$ to be the point on the line $B_{1} C$ such that $A_{i} B_{i} \\perp B_{1} C$. Let $\\Gamma_{1}$ be the incircle of $\\Delta A_{1} B_{1} C$ and for $i \\geq 2, \\Gamma_{i}$ be the circle tangent to $\\Gamma_{i-1}, A_{1} C, B_{1} C$ which is smaller than $\\Gamma_{i-1}$. How many integers $k$ are there such that the line $A_{1} B_{2016}$ intersects $\\Gamma_{k}$ ?",
      "answer": "4030"
    },
    {
      "question": "Let $a, b$ and $c$ be positive real numbers such that $$\\begin{aligned} a^{2}+a b+b^{2} & =9 \\\\ b^{2}+b c+c^{2} & =52 \\\\ c^{2}+c a+a^{2} & =49 \\end{aligned}$$ Compute the value of $\\frac{49 b^{2}-33 b c+9 c^{2}}{a^{2}}$.",
      "answer": "52"
    },
    {
      "question": "Let $\\{ z_n \\}_{n \\ge 1}$ be a sequence of complex numbers, whose odd terms are real, even terms are purely imaginary, and for every positive integer $k$, $|z_k z_{k+1}|=2^k$. Denote $f_n=|z_1+z_2+\\cdots+z_n|,$ for $n=1,2,\\cdots$\n(1) Find the minimum of $f_{2020}$.\n(2) Find the minimum of $f_{2020} \\cdot f_{2021}$.",
      "answer": "2"
    },
    {
      "question": "Mario is once again on a quest to save Princess Peach. Mario enters Peach's castle and finds himself in a room with 4 doors. This room is the first in a sequence of 6 indistinguishable rooms. In each room, 1 door leads to the next room in the sequence (or, for the last room, Bowser's level), while the other 3 doors lead to the first room. Now what is the expected number of doors through which Mario will pass before he reaches Bowser's level?",
      "answer": "5460"
    },
    {
      "question": "Find the value of $\\frac{\\sin^{2}B+\\sin^{2}C-\\sin^{2}A}{\\sin B \\sin C}$ given that $\\frac{\\sin B}{\\sin C}=\\frac{AC}{AB}$, $\\frac{\\sin C}{\\sin B}=\\frac{AB}{AC}$, and $\\frac{\\sin A}{\\sin B \\sin C}=\\frac{BC}{AC \\cdot AB}$.",
      "answer": "\\frac{83}{80}"
    },
    {
      "question": "Let $f(x)=-x^{2}+10 x-20$. Find the sum of all $2^{2010}$ solutions to $\\underbrace{f(f(\\ldots(x) \\ldots))}_{2010 f \\mathrm{~s}}=2$.",
      "answer": "5 \\cdot 2^{2010}"
    },
    {
      "question": "Consider a $2 \\times 2$ grid of squares. Each of the squares will be colored with one of 10 colors, and two colorings are considered equivalent if one can be rotated to form the other. How many distinct colorings are there?",
      "answer": "2530"
    },
    {
      "question": "Consider triangle $A B C$ where $B C=7, C A=8$, and $A B=9$. $D$ and $E$ are the midpoints of $B C$ and $C A$, respectively, and $A D$ and $B E$ meet at $G$. The reflection of $G$ across $D$ is $G^{\\prime}$, and $G^{\\prime} E$ meets $C G$ at $P$. Find the length $P G$.",
      "answer": "\\frac{\\sqrt{145}}{9}"
    },
    {
      "question": "In each square of a garden shaped like a $2022 \\times 2022$ board, there is initially a tree of height $0$. A gardener and a lumberjack alternate turns playing the following game, with the gardener taking the first turn:\n[list]\n[*] The gardener chooses a square in the garden. Each tree on that square and all the surrounding squares (of which there are at most eight) then becomes one unit taller.\n[*] The lumberjack then chooses four different squares on the board. Each tree of positive height on those squares then becomes one unit shorter.\n[/list]\nWe say that a tree is [i]majestic[/i] if its height is at least $10^6$. Determine the largest $K$ such that the gardener can ensure there are eventually $K$ majestic trees on the board, no matter how the lumberjack plays.",
      "answer": "2271380"
    },
    {
      "question": "Real numbers $x, y, z$ satisfy $$x+x y+x y z=1, \\quad y+y z+x y z=2, \\quad z+x z+x y z=4$$ The largest possible value of $x y z$ is $\\frac{a+b \\sqrt{c}}{d}$, where $a, b, c, d$ are integers, $d$ is positive, $c$ is square-free, and $\\operatorname{gcd}(a, b, d)=1$. Find $1000 a+100 b+10 c+d$.",
      "answer": "5272"
    },
    {
      "question": "Complex numbers $a, b, c$ form an equilateral triangle with side length 18 in the complex plane. If $|a+b+c|=36$, find $|b c+c a+a b|$.",
      "answer": "432"
    },
    {
      "question": "Jude repeatedly flips a coin. If he has already flipped $n$ heads, the coin lands heads with probability $\\frac{1}{n+2}$ and tails with probability $\\frac{n+1}{n+2}$. If Jude continues flipping forever, let $p$ be the probability that he flips 3 heads in a row at some point. Compute $\\lfloor 180 p\\rfloor$.",
      "answer": "47"
    },
    {
      "question": "A mathematician $M^{\\prime}$ is called a descendent of mathematician $M$ if there is a sequence of mathematicians $M=M_{1}, M_{2}, \\ldots, M_{k}=M^{\\prime}$ such that $M_{i}$ was $M_{i+1}$ 's doctoral advisor for all $i$. Estimate the number of descendents that the mathematician who has had the largest number of descendents has had, according to the Mathematical Genealogy Project. Note that the Mathematical Genealogy Project has records dating back to the 1300s. If the correct answer is $X$ and you write down $A$, your team will receive $\\max \\left(25-\\left\\lfloor\\frac{|X-A|}{100}\\right\\rfloor, 0\\right)$ points, where $\\lfloor x\\rfloor$ is the largest integer less than or equal to $x$.",
      "answer": "82310"
    },
    {
      "question": "Three real numbers $x, y, z$ are chosen randomly, and independently of each other, between 0 and 1, inclusive. What is the probability that each of $x-y$ and $x-z$ is greater than $-\\frac{1}{2}$ and less than $\\frac{1}{2}$?",
      "answer": "\\frac{7}{12}"
    },
    {
      "question": "Seven lattice points form a convex heptagon with all sides having distinct lengths. Find the minimum possible value of the sum of the squares of the sides of the heptagon.",
      "answer": "42"
    },
    {
      "question": "Zebadiah has 3 red shirts, 3 blue shirts, and 3 green shirts in a drawer. Without looking, he randomly pulls shirts from his drawer one at a time. What is the minimum number of shirts that Zebadiah has to pull out to guarantee that he has a set of shirts that includes either 3 of the same colour or 3 of different colours?",
      "answer": "5"
    },
    {
      "question": "Calculate the value of $\\frac{2 \\times 3 + 4}{2 + 3}$.",
      "answer": "2"
    },
    {
      "question": "A candy company makes 5 colors of jellybeans, which come in equal proportions. If I grab a random sample of 5 jellybeans, what is the probability that I get exactly 2 distinct colors?",
      "answer": "\\frac{12}{125}"
    },
    {
      "question": "Compute the nearest integer to $$100 \\sum_{n=1}^{\\infty} 3^{n} \\sin ^{3}\\left(\\frac{\\pi}{3^{n}}\\right)$$",
      "answer": "236"
    },
    {
      "question": "Find the number of strictly increasing sequences of nonnegative integers with the following properties: - The first term is 0 and the last term is 12. In particular, the sequence has at least two terms. - Among any two consecutive terms, exactly one of them is even.",
      "answer": "144"
    },
    {
      "question": "Among a group of 120 people, some pairs are friends. A [i]weak quartet[/i] is a set of four people containing exactly one pair of friends. What is the maximum possible number of weak quartets ?",
      "answer": "4769280"
    },
    {
      "question": "Suppose $P(x)$ is a polynomial with real coefficients such that $P(t)=P(1) t^{2}+P(P(1)) t+P(P(P(1)))$ for all real numbers $t$. Compute the largest possible value of $P(P(P(P(1))))$.",
      "answer": "\\frac{1}{9}"
    },
    {
      "question": "Compute $\\sum_{k=1}^{2009} k\\left(\\left\\lfloor\\frac{2009}{k}\\right\\rfloor-\\left\\lfloor\\frac{2008}{k}\\right\\rfloor\\right)$.",
      "answer": "2394"
    },
    {
      "question": "Find the number of quadruples $(a, b, c, d)$ of integers with absolute value at most 5 such that $\\left(a^{2}+b^{2}+c^{2}+d^{2}\\right)^{2}=(a+b+c+d)(a-b+c-d)\\left((a-c)^{2}+(b-d)^{2}\\right)$",
      "answer": "49"
    },
    {
      "question": "Three faces $\\mathcal{X}, \\mathcal{Y}, \\mathcal{Z}$ of a unit cube share a common vertex. Suppose the projections of $\\mathcal{X}, \\mathcal{Y}, \\mathcal{Z}$ onto a fixed plane $\\mathcal{P}$ have areas $x, y, z$, respectively. If $x: y: z=6: 10: 15$, then $x+y+z$ can be written as $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "3119"
    },
    {
      "question": "In the base 10 arithmetic problem $H M M T+G U T S=R O U N D$, each distinct letter represents a different digit, and leading zeroes are not allowed. What is the maximum possible value of $R O U N D$?",
      "answer": "16352"
    },
    {
      "question": "Suppose $a$ and $b$ be positive integers not exceeding 100 such that $$a b=\\left(\\frac{\\operatorname{lcm}(a, b)}{\\operatorname{gcd}(a, b)}\\right)^{2}$$ Compute the largest possible value of $a+b$.",
      "answer": "78"
    },
    {
      "question": "The operation $\\nabla$ is defined by $a \\nabla b=4 a+b$. What is the value of $(5 \\nabla 2) \\nabla 2$?",
      "answer": "90"
    },
    {
      "question": "Let $N$ be the number of distinct roots of \\prod_{k=1}^{2012}\\left(x^{k}-1\\right)$. Give lower and upper bounds $L$ and $U$ on $N$. If $0<L \\leq N \\leq U$, then your score will be \\left[\\frac{23}{(U / L)^{1.7}}\\right\\rfloor$. Otherwise, your score will be 0 .",
      "answer": "1231288"
    },
    {
      "question": "A Sudoku matrix is defined as a $9 \\times 9$ array with entries from \\{1,2, \\ldots, 9\\} and with the constraint that each row, each column, and each of the nine $3 \\times 3$ boxes that tile the array contains each digit from 1 to 9 exactly once. A Sudoku matrix is chosen at random (so that every Sudoku matrix has equal probability of being chosen). We know two of squares in this matrix, as shown. What is the probability that the square marked by ? contains the digit 3 ?",
      "answer": "\\frac{2}{21}"
    },
    {
      "question": "The integers \\(1,2,3,4,5,6,7,8,9,10\\) are written on a blackboard. Each day, a teacher chooses one of the integers uniformly at random and decreases it by 1. Let \\(X\\) be the expected value of the number of days which elapse before there are no longer positive integers on the board. Estimate \\(X\\). An estimate of \\(E\\) earns \\(\\left\\lfloor 20 \\cdot 2^{-|X-E| / 8}\\right\\rfloor\\) points.",
      "answer": "120.75280458176904"
    },
    {
      "question": "Let $A_{1} A_{2} \\ldots A_{100}$ be the vertices of a regular 100-gon. Let $\\pi$ be a randomly chosen permutation of the numbers from 1 through 100. The segments $A_{\\pi(1)} A_{\\pi(2)}, A_{\\pi(2)} A_{\\pi(3)}, \\ldots, A_{\\pi(99)} A_{\\pi(100)}, A_{\\pi(100)} A_{\\pi(1)}$ are drawn. Find the expected number of pairs of line segments that intersect at a point in the interior of the 100-gon.",
      "answer": "\\frac{4850}{3}"
    },
    {
      "question": "Find the number of sequences $a_{1}, a_{2}, \\ldots, a_{10}$ of positive integers with the property that $a_{n+2}=a_{n+1}+a_{n}$ for $n=1,2, \\ldots, 8$, and $a_{10}=2002$.",
      "answer": "3"
    },
    {
      "question": "Given that $a, b, c$ are positive real numbers and $\\log _{a} b+\\log _{b} c+\\log _{c} a=0$, find the value of $\\left(\\log _{a} b\\right)^{3}+\\left(\\log _{b} c\\right)^{3}+\\left(\\log _{c} a\\right)^{3}$.",
      "answer": "3"
    },
    {
      "question": "Two numbers $a$ and $b$ with $0 \\leq a \\leq 1$ and $0 \\leq b \\leq 1$ are chosen at random. The number $c$ is defined by $c=2a+2b$. The numbers $a, b$ and $c$ are each rounded to the nearest integer to give $A, B$ and $C$, respectively. What is the probability that $2A+2B=C$?",
      "answer": "\\frac{7}{16}"
    },
    {
      "question": "A box contains 5 black ties, 7 gold ties, and 8 pink ties. What is the probability that Stephen randomly chooses a pink tie?",
      "answer": "\\frac{2}{5}"
    },
    {
      "question": "Lauren plays basketball with her friends. She makes 10 baskets. Each of these baskets is worth either 2 or 3 points. Lauren scores a total of 26 points. How many 3 point baskets did she make?",
      "answer": "6"
    },
    {
      "question": "Let $D$ be the set of divisors of 100. Let $Z$ be the set of integers between 1 and 100, inclusive. Mark chooses an element $d$ of $D$ and an element $z$ of $Z$ uniformly at random. What is the probability that $d$ divides $z$?",
      "answer": "\\frac{217}{900}"
    },
    {
      "question": "Let $S$ be the set of ordered pairs $(a, b)$ of positive integers such that \\operatorname{gcd}(a, b)=1$. Compute $$\\sum_{(a, b) \\in S}\\left\\lfloor\\frac{300}{2 a+3 b}\\right\\rfloor$$",
      "answer": "7400"
    },
    {
      "question": "Given any two positive real numbers $x$ and $y$, then $x \\diamond y$ is a positive real number defined in terms of $x$ and $y$ by some fixed rule. Suppose the operation $x \\diamond y$ satisfies the equations \\((x \\cdot y) \\diamond y=x(y \\diamond y)\\) and \\((x \\diamond 1) \\diamond x=x \\diamond 1\\) for all $x, y>0$. Given that $1 \\diamond 1=1$, find $19 \\diamond 98$.",
      "answer": "19"
    },
    {
      "question": "The expression $(5 \\times 5)+(5 \\times 5)+(5 \\times 5)+(5 \\times 5)+(5 \\times 5)$ is equal to what?",
      "answer": "125"
    },
    {
      "question": "For how many integers $a(1 \\leq a \\leq 200)$ is the number $a^{a}$ a square?",
      "answer": "107"
    },
    {
      "question": "Let $a_{1}=3$, and for $n \\geq 1$, let $a_{n+1}=(n+1) a_{n}-n$. Find the smallest $m \\geq 2005$ such that $a_{m+1}-1 \\mid a_{m}^{2}-1$.",
      "answer": "2010"
    },
    {
      "question": "If points $P, Q, R$, and $S$ are arranged in order on a line segment with $P Q=1, Q R=2 P Q$, and $R S=3 Q R$, what is the length of $P S$?",
      "answer": "9"
    },
    {
      "question": "Find the number of pairs of integers $(x, y)$ such that $x^{2}+2y^{2}<25$.",
      "answer": "55"
    },
    {
      "question": "Let $\\varphi(n)$ denote the number of positive integers less than or equal to $n$ which are relatively prime to $n$. Let $S$ be the set of positive integers $n$ such that $\\frac{2 n}{\\varphi(n)}$ is an integer. Compute the sum $\\sum_{n \\in S} \\frac{1}{n}$.",
      "answer": "\\frac{10}{3}"
    },
    {
      "question": "Side $\\overline{A B}$ of $\\triangle A B C$ is the diameter of a semicircle, as shown below. If $A B=3+\\sqrt{3}, B C=3 \\sqrt{2}$, and $A C=2 \\sqrt{3}$, then the area of the shaded region can be written as $\\frac{a+(b+c \\sqrt{d}) \\pi}{e}$, where $a, b, c, d, e$ are integers, $e$ is positive, $d$ is square-free, and $\\operatorname{gcd}(a, b, c, e)=1$. Find $10000 a+1000 b+100 c+10 d+e$.",
      "answer": "147938"
    },
    {
      "question": "Determine the least possible value of $f(1998),$ where $f:\\Bbb{N}\\to \\Bbb{N}$ is a function such that for all $m,n\\in {\\Bbb N}$, \n\n\\[f\\left( n^{2}f(m)\\right) =m\\left( f(n)\\right) ^{2}. \\]",
      "answer": "120"
    },
    {
      "question": "If $4^{n}=64^{2}$, what is the value of $n$?",
      "answer": "6"
    },
    {
      "question": "Let $n$ be an integer and $$m=(n-1001)(n-2001)(n-2002)(n-3001)(n-3002)(n-3003)$$ Given that $m$ is positive, find the minimum number of digits of $m$.",
      "answer": "11"
    },
    {
      "question": "Pascal High School organized three different trips. Fifty percent of the students went on the first trip, $80 \\%$ went on the second trip, and $90 \\%$ went on the third trip. A total of 160 students went on all three trips, and all of the other students went on exactly two trips. How many students are at Pascal High School?",
      "answer": "800"
    },
    {
      "question": "A man named Juan has three rectangular solids, each having volume 128. Two of the faces of one solid have areas 4 and 32. Two faces of another solid have areas 64 and 16. Finally, two faces of the last solid have areas 8 and 32. What is the minimum possible exposed surface area of the tallest tower Juan can construct by stacking his solids one on top of the other, face to face? (Assume that the base of the tower is not exposed).",
      "answer": "688"
    },
    {
      "question": "Determine the remainder when $$2^{\\frac{1 \\cdot 2}{2}}+2^{\\frac{2 \\cdot 3}{2}}+\\cdots+2^{\\frac{2011 \\cdot 2012}{2}}$$ is divided by 7.",
      "answer": "1"
    },
    {
      "question": "Meghal is playing a game with 2016 rounds $1,2, \\cdots, 2016$. In round $n$, two rectangular double-sided mirrors are arranged such that they share a common edge and the angle between the faces is $\\frac{2 \\pi}{n+2}$. Meghal shoots a laser at these mirrors and her score for the round is the number of points on the two mirrors at which the laser beam touches a mirror. What is the maximum possible score Meghal could have after she finishes the game?",
      "answer": "1019088"
    },
    {
      "question": "Define the sequence $a_{1}, a_{2} \\ldots$ as follows: $a_{1}=1$ and for every $n \\geq 2$, $a_{n}= \\begin{cases}n-2 & \\text { if } a_{n-1}=0 \\\\ a_{n-1}-1 & \\text { if } a_{n-1} \\neq 0\\end{cases}$. A non-negative integer $d$ is said to be jet-lagged if there are non-negative integers $r, s$ and a positive integer $n$ such that $d=r+s$ and that $a_{n+r}=a_{n}+s$. How many integers in $\\{1,2, \\ldots, 2016\\}$ are jet-lagged?",
      "answer": "51"
    },
    {
      "question": "Triangle $A B C$ has incircle $\\omega$ which touches $A B$ at $C_{1}, B C$ at $A_{1}$, and $C A$ at $B_{1}$. Let $A_{2}$ be the reflection of $A_{1}$ over the midpoint of $B C$, and define $B_{2}$ and $C_{2}$ similarly. Let $A_{3}$ be the intersection of $A A_{2}$ with $\\omega$ that is closer to $A$, and define $B_{3}$ and $C_{3}$ similarly. If $A B=9, B C=10$, and $C A=13$, find \\left[A_{3} B_{3} C_{3}\\right] /[A B C].",
      "answer": "14/65"
    },
    {
      "question": "When $x=-2$, what is the value of $(x+1)^{3}$?",
      "answer": "-1"
    },
    {
      "question": "A classroom consists of a $5 \\times 5$ array of desks, to be filled by anywhere from 0 to 25 students, inclusive. No student will sit at a desk unless either all other desks in its row or all others in its column are filled (or both). Considering only the set of desks that are occupied (and not which student sits at each desk), how many possible arrangements are there?",
      "answer": "962"
    },
    {
      "question": "Max repeatedly throws a fair coin in a hurricane. For each throw, there is a $4 \\%$ chance that the coin gets blown away. He records the number of heads $H$ and the number of tails $T$ before the coin is lost. (If the coin is blown away on a toss, no result is recorded for that toss.) What is the expected value of $|H-T|$?",
      "answer": "\\frac{24}{7}"
    },
    {
      "question": "The vertices of a regular nonagon are colored such that 1) adjacent vertices are different colors and 2) if 3 vertices form an equilateral triangle, they are all different colors. Let m be the minimum number of colors needed for a valid coloring, and n be the total number of colorings using m colors. Determine mn. (Assume each vertex is distinguishable.)",
      "answer": "54"
    },
    {
      "question": "Let $m \\circ n=(m+n) /(m n+4)$. Compute $((\\cdots((2005 \\circ 2004) \\circ 2003) \\circ \\cdots \\circ 1) \\circ 0)$.",
      "answer": "1/12"
    },
    {
      "question": "A right triangle has area 5 and a hypotenuse of length 5. Find its perimeter.",
      "answer": "5+3 \\sqrt{5}"
    },
    {
      "question": "Determine the number of subsets $S$ of $\\{1,2,3, \\ldots, 10\\}$ with the following property: there exist integers $a<b<c$ with $a \\in S, b \\notin S, c \\in S$.",
      "answer": "968"
    },
    {
      "question": "Solve for \\(x\\): \\(x\\lfloor x\\lfloor x\\lfloor x\\lfloor x\\rfloor\\rfloor\\rfloor\\rfloor=122\\).",
      "answer": "\\frac{122}{41}"
    },
    {
      "question": "Let $S$ be a subset of $\\{1,2,3, \\ldots, 12\\}$ such that it is impossible to partition $S$ into $k$ disjoint subsets, each of whose elements sum to the same value, for any integer $k \\geq 2$. Find the maximum possible sum of the elements of $S$.",
      "answer": "77"
    },
    {
      "question": "How many positive integers $n$ with $n \\leq 100$ can be expressed as the sum of four or more consecutive positive integers?",
      "answer": "63"
    },
    {
      "question": "Unit circle $\\Omega$ has points $X, Y, Z$ on its circumference so that $X Y Z$ is an equilateral triangle. Let $W$ be a point other than $X$ in the plane such that triangle $W Y Z$ is also equilateral. Determine the area of the region inside triangle $W Y Z$ that lies outside circle $\\Omega$.",
      "answer": "$\\frac{3 \\sqrt{3}-\\pi}{3}$"
    },
    {
      "question": "Find the sum of the series $\\sum_{n=1}^{\\infty} \\frac{1}{n^{2}+2n}$.",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "Let $a_{1}=3$, and for $n>1$, let $a_{n}$ be the largest real number such that $$4\\left(a_{n-1}^{2}+a_{n}^{2}\\right)=10 a_{n-1} a_{n}-9$$ What is the largest positive integer less than $a_{8}$ ?",
      "answer": "335"
    },
    {
      "question": "Sherry and Val are playing a game. Sherry has a deck containing 2011 red cards and 2012 black cards, shuffled randomly. Sherry flips these cards over one at a time, and before she flips each card over, Val guesses whether it is red or black. If Val guesses correctly, she wins 1 dollar; otherwise, she loses 1 dollar. In addition, Val must guess red exactly 2011 times. If Val plays optimally, what is her expected profit from this game?",
      "answer": "\\frac{1}{4023}"
    },
    {
      "question": "Let $E$ be a three-dimensional ellipsoid. For a plane $p$, let $E(p)$ be the projection of $E$ onto the plane $p$. The minimum and maximum areas of $E(p)$ are $9 \\pi$ and $25 \\pi$, and there exists a $p$ where $E(p)$ is a circle of area $16 \\pi$. If $V$ is the volume of $E$, compute $V / \\pi$.",
      "answer": "75"
    },
    {
      "question": "Let $n$ be the 200th smallest positive real solution to the equation $x-\\frac{\\pi}{2}=\\tan x$. Find the greatest integer that does not exceed $\\frac{n}{2}$.",
      "answer": "314"
    },
    {
      "question": "Find the largest real number $c$ such that $$\\sum_{i=1}^{101} x_{i}^{2} \\geq c M^{2}$$ whenever $x_{1}, \\ldots, x_{101}$ are real numbers such that $x_{1}+\\cdots+x_{101}=0$ and $M$ is the median of $x_{1}, \\ldots, x_{101}$.",
      "answer": "\\frac{5151}{50}"
    },
    {
      "question": "Find the largest integer that divides $m^{5}-5 m^{3}+4 m$ for all $m \\geq 5$.",
      "answer": "120"
    },
    {
      "question": "There are $2018$ players sitting around a round table. At the beginning of the game we arbitrarily deal all the cards from a deck of $K$ cards to the players (some players may receive no cards). In each turn we choose a player who draws one card from each of the two neighbors. It is only allowed to choose a player whose each neighbor holds a nonzero number of cards. The game terminates when there is no such player. Determine the largest possible value of $K$ such that, no matter how we deal the cards and how we choose the players, the game always terminates after a finite number of turns.",
      "answer": "2017"
    },
    {
      "question": "You want to arrange the numbers $1,2,3, \\ldots, 25$ in a sequence with the following property: if $n$ is divisible by $m$, then the $n$th number is divisible by the $m$ th number. How many such sequences are there?",
      "answer": "24"
    },
    {
      "question": "Let $\\triangle A B C$ be a scalene triangle. Let $h_{a}$ be the locus of points $P$ such that $|P B-P C|=|A B-A C|$. Let $h_{b}$ be the locus of points $P$ such that $|P C-P A|=|B C-B A|$. Let $h_{c}$ be the locus of points $P$ such that $|P A-P B|=|C A-C B|$. In how many points do all of $h_{a}, h_{b}$, and $h_{c}$ concur?",
      "answer": "2"
    },
    {
      "question": "For how many $n$ with $1 \\leq n \\leq 100$ can a unit square be divided into $n$ congruent figures?",
      "answer": "100"
    },
    {
      "question": "A set of six edges of a regular octahedron is called Hamiltonian cycle if the edges in some order constitute a single continuous loop that visits each vertex exactly once. How many ways are there to partition the twelve edges into two Hamiltonian cycles?",
      "answer": "6"
    },
    {
      "question": "Boris was given a Connect Four game set for his birthday, but his color-blindness makes it hard to play the game. Still, he enjoys the shapes he can make by dropping checkers into the set. If the number of shapes possible modulo (horizontal) flips about the vertical axis of symmetry is expressed as $9(1+2+\\cdots+n)$, find $n$.",
      "answer": "729"
    },
    {
      "question": "The game of rock-scissors is played just like rock-paper-scissors, except that neither player is allowed to play paper. You play against a poorly-designed computer program that plays rock with $50 \\%$ probability and scissors with $50 \\%$ probability. If you play optimally against the computer, find the probability that after 8 games you have won at least 4.",
      "answer": "\\frac{163}{256}"
    },
    {
      "question": "A circle inscribed in a square has two chords as shown in a pair. It has radius 2, and $P$ bisects $T U$. The chords' intersection is where? Answer the question by giving the distance of the point of intersection from the center of the circle.",
      "answer": "2\\sqrt{2} - 2"
    },
    {
      "question": "A jar contains 97 marbles that are either red, green, or blue. Neil draws two marbles from the jar without replacement and notes that the probability that they would be the same color is $\\frac{5}{12}$. After Neil puts his marbles back, Jerry draws two marbles from the jar with replacement. Compute the probability that the marbles that Jerry draws are the same color.",
      "answer": "\\frac{41}{97}"
    },
    {
      "question": "On a computer screen is the single character a. The computer has two keys: c (copy) and p (paste), which may be pressed in any sequence. Pressing p increases the number of a's on screen by the number that were there the last time c was pressed. c doesn't change the number of a's on screen. Determine the fewest number of keystrokes required to attain at least 2018 a's on screen. (Note: pressing p before the first press of c does nothing).",
      "answer": "21"
    },
    {
      "question": "A standard $n$-sided die has $n$ sides labeled 1 to $n$. Luis, Luke, and Sean play a game in which they roll a fair standard 4-sided die, a fair standard 6-sided die, and a fair standard 8-sided die, respectively. They lose the game if Luis's roll is less than Luke's roll, and Luke's roll is less than Sean's roll. Compute the probability that they lose the game.",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "$M$ is an $8 \\times 8$ matrix. For $1 \\leq i \\leq 8$, all entries in row $i$ are at least $i$, and all entries on column $i$ are at least $i$. What is the minimum possible sum of the entries of $M$ ?",
      "answer": "372"
    },
    {
      "question": "A total of $n$ points are equally spaced around a circle and are labelled with the integers 1 to $n$, in order. Two points are called diametrically opposite if the line segment joining them is a diameter of the circle. If the points labelled 7 and 35 are diametrically opposite, then what is the value of $n$?",
      "answer": "56"
    },
    {
      "question": "Consider the system \\begin{align*}x + y  &= z + u,\\\\2xy & = zu.\\end{align*} Find the greatest value of the real constant $m$ such that $m \\leq x/y$ for any positive integer solution $(x,y,z,u)$ of the system, with $x \\geq y$.",
      "answer": "3 + 2\\sqrt{2}"
    },
    {
      "question": "A loonie is a $\\$ 1$ coin and a dime is a $\\$ 0.10$ coin. One loonie has the same mass as 4 dimes. A bag of dimes has the same mass as a bag of loonies. The coins in the bag of loonies are worth $\\$ 400$ in total. How much are the coins in the bag of dimes worth?",
      "answer": "160"
    },
    {
      "question": "Each unit square of a $4 \\times 4$ square grid is colored either red, green, or blue. Over all possible colorings of the grid, what is the maximum possible number of L-trominos that contain exactly one square of each color?",
      "answer": "18"
    },
    {
      "question": "Compute the positive integer less than 1000 which has exactly 29 positive proper divisors.",
      "answer": "720"
    },
    {
      "question": "The product of the roots of the equation \\((x-4)(x-2)+(x-2)(x-6)=0\\) is",
      "answer": "10"
    },
    {
      "question": "Let $a$ be a positive integer such that $2a$ has units digit 4. What is the sum of the possible units digits of $3a$?",
      "answer": "7"
    },
    {
      "question": "Three tanks contain water. The number of litres in each is shown in the table: Tank A: 3600 L, Tank B: 1600 L, Tank C: 3800 L. Water is moved from each of Tank A and Tank C into Tank B so that each tank contains the same volume of water. How many litres of water are moved from Tank A to Tank B?",
      "answer": "600"
    },
    {
      "question": "The value of \\( \\frac{1}{2} + \\frac{2}{4} + \\frac{4}{8} + \\frac{8}{16} \\) is what?",
      "answer": "2"
    },
    {
      "question": "For each positive integer $k$, let $A(k)$ be the number of odd divisors of $k$ in the interval $[1, \\sqrt{2k})$. Evaluate\n\\[\n\\sum_{k=1}^\\infty (-1)^{k-1} \\frac{A(k)}{k}.\n\\]",
      "answer": "\\frac{\\pi^2}{16}"
    },
    {
      "question": "Suppose $a, b$ and $c$ are integers such that the greatest common divisor of $x^{2}+a x+b$ and $x^{2}+b x+c$ is $x+1$ (in the ring of polynomials in $x$ with integer coefficients), and the least common multiple of $x^{2}+a x+b$ and $x^{2}+b x+c$ is $x^{3}-4 x^{2}+x+6$. Find $a+b+c$.",
      "answer": "-6"
    },
    {
      "question": "Each of the following 15 cards has a letter on one side and a positive integer on the other side. What is the minimum number of cards that need to be turned over to check if the following statement is true? 'If a card has a lower case letter on one side, then it has an odd integer on the other side.'",
      "answer": "3"
    },
    {
      "question": "Compute the number of ordered pairs of integers $(x, y)$ such that $x^{2}+y^{2}<2019$ and $$x^{2}+\\min (x, y)=y^{2}+\\max (x, y)$$",
      "answer": "127"
    },
    {
      "question": "Compute the number of positive real numbers $x$ that satisfy $\\left(3 \\cdot 2^{\\left\\lfloor\\log _{2} x\\right\\rfloor}-x\\right)^{16}=2022 x^{13}$.",
      "answer": "9"
    },
    {
      "question": "Two real numbers $x$ and $y$ are such that $8 y^{4}+4 x^{2} y^{2}+4 x y^{2}+2 x^{3}+2 y^{2}+2 x=x^{2}+1$. Find all possible values of $x+2 y^{2}$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Over all real numbers $x$ and $y$, find the minimum possible value of $$ (x y)^{2}+(x+7)^{2}+(2 y+7)^{2} $$",
      "answer": "45"
    },
    {
      "question": "The symbol $\\diamond$ is defined so that $a \\diamond b=\\frac{a+b}{a \\times b}$. What is the value of $3 \\diamond 6$?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Knot is on an epic quest to save the land of Hyruler from the evil Gammadorf. To do this, he must collect the two pieces of the Lineforce, then go to the Temple of Lime. As shown on the figure, Knot starts on point $K$, and must travel to point $T$, where $O K=2$ and $O T=4$. However, he must first reach both solid lines in the figure below to collect the pieces of the Lineforce. What is the minimal distance Knot must travel to do so?",
      "answer": "2 \\sqrt{5}"
    },
    {
      "question": "A number is chosen uniformly at random from the set of all positive integers with at least two digits, none of which are repeated. Find the probability that the number is even.",
      "answer": "\\frac{41}{81}"
    },
    {
      "question": "Lily and Sarah are playing a game. They each choose a real number at random between -1 and 1. They then add the squares of their numbers together. If the result is greater than or equal to 1, Lily wins, and if the result is less than 1, Sarah wins. What is the probability that Sarah wins?",
      "answer": "\\frac{\\pi}{4}"
    },
    {
      "question": "Suppose $a_{1}, a_{2}, \\ldots, a_{100}$ are positive real numbers such that $$a_{k}=\\frac{k a_{k-1}}{a_{k-1}-(k-1)}$$ for $k=2,3, \\ldots, 100$. Given that $a_{20}=a_{23}$, compute $a_{100}$.",
      "answer": "215"
    },
    {
      "question": "Consider $9$ points in space, no four of which are coplanar. Each pair of points is joined by an edge (that is, a line segment) and each edge is either colored blue or red or left uncolored. Find the smallest value of  $\\,n\\,$ such that whenever exactly $\\,n\\,$ edges are colored, the set of colored edges necessarily contains a triangle all of whose edges have the same color.",
      "answer": "33"
    },
    {
      "question": "Define the sequence \\left\\{x_{i}\\right\\}_{i \\geq 0} by $x_{0}=x_{1}=x_{2}=1$ and $x_{k}=\\frac{x_{k-1}+x_{k-2}+1}{x_{k-3}}$ for $k>2$. Find $x_{2013}$.",
      "answer": "9"
    },
    {
      "question": "The minute hand on a clock points at the 12. After rotating $120^{\\circ}$ clockwise, which number will it point at?",
      "answer": "4"
    },
    {
      "question": "Ava and Tiffany participate in a knockout tournament consisting of a total of 32 players. In each of 5 rounds, the remaining players are paired uniformly at random. In each pair, both players are equally likely to win, and the loser is knocked out of the tournament. The probability that Ava and Tiffany play each other during the tournament is $\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
      "answer": "116"
    },
    {
      "question": "Allen and Brian are playing a game in which they roll a 6-sided die until one of them wins. Allen wins if two consecutive rolls are equal and at most 3. Brian wins if two consecutive rolls add up to 7 and the latter is at most 3. What is the probability that Allen wins?",
      "answer": "\\frac{5}{12}"
    },
    {
      "question": "Find the largest number $n$ such that $(2004!)!$ is divisible by $((n!)!)!$.",
      "answer": "6"
    },
    {
      "question": "Find the total number of occurrences of the digits $0,1 \\ldots, 9$ in the entire guts round. If your answer is $X$ and the actual value is $Y$, your score will be $\\max \\left(0,20-\\frac{|X-Y|}{2}\\right)$",
      "answer": "559"
    },
    {
      "question": "For each positive integer $n$ and non-negative integer $k$, define $W(n, k)$ recursively by $$ W(n, k)= \\begin{cases}n^{n} & k=0 \\\\ W(W(n, k-1), k-1) & k>0\\end{cases} $$ Find the last three digits in the decimal representation of $W(555,2)$.",
      "answer": "875"
    },
    {
      "question": "$A B C D$ is a cyclic quadrilateral in which $A B=3, B C=5, C D=6$, and $A D=10 . M, I$, and $T$ are the feet of the perpendiculars from $D$ to lines $A B, A C$, and $B C$ respectively. Determine the value of $M I / I T$.",
      "answer": "\\frac{25}{9}"
    },
    {
      "question": "Determine the positive real value of $x$ for which $$\\sqrt{2+A C+2 C x}+\\sqrt{A C-2+2 A x}=\\sqrt{2(A+C) x+2 A C}$$",
      "answer": "4"
    },
    {
      "question": "There are three pairs of real numbers \\left(x_{1}, y_{1}\\right),\\left(x_{2}, y_{2}\\right), and \\left(x_{3}, y_{3}\\right) that satisfy both $x^{3}-3 x y^{2}=2005$ and $y^{3}-3 x^{2} y=2004$. Compute \\left(1-\\frac{x_{1}}{y_{1}}\\right)\\left(1-\\frac{x_{2}}{y_{2}}\\right)\\left(1-\\frac{x_{3}}{y_{3}}\\right).",
      "answer": "1/1002"
    },
    {
      "question": "Urn A contains 4 white balls and 2 red balls. Urn B contains 3 red balls and 3 black balls. An urn is randomly selected, and then a ball inside of that urn is removed. We then repeat the process of selecting an urn and drawing out a ball, without returning the first ball. What is the probability that the first ball drawn was red, given that the second ball drawn was black?",
      "answer": "7/15"
    },
    {
      "question": "Calvin has a bag containing 50 red balls, 50 blue balls, and 30 yellow balls. Given that after pulling out 65 balls at random (without replacement), he has pulled out 5 more red balls than blue balls, what is the probability that the next ball he pulls out is red?",
      "answer": "\\frac{9}{26}"
    },
    {
      "question": "Compute the number of ways to color 3 cells in a $3 \\times 3$ grid so that no two colored cells share an edge.",
      "answer": "22"
    },
    {
      "question": "Sixteen wooden Cs are placed in a 4-by-4 grid, all with the same orientation, and each is to be colored either red or blue. A quadrant operation on the grid consists of choosing one of the four two-by-two subgrids of Cs found at the corners of the grid and moving each C in the subgrid to the adjacent square in the subgrid that is 90 degrees away in the clockwise direction, without changing the orientation of the C. Given that two colorings are the considered same if and only if one can be obtained from the other by a series of quadrant operations, determine the number of distinct colorings of the Cs.",
      "answer": "1296"
    },
    {
      "question": "Pick a random integer between 0 and 4095, inclusive. Write it in base 2 (without any leading zeroes). What is the expected number of consecutive digits that are not the same (that is, the expected number of occurrences of either 01 or 10 in the base 2 representation)?",
      "answer": "\\frac{20481}{4096}"
    },
    {
      "question": "Find the expected value of the number formed by rolling a fair 6-sided die with faces numbered 1, 2, 3, 5, 7, 9 infinitely many times.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Amy and Ben need to eat 1000 total carrots and 1000 total muffins. The muffins can not be eaten until all the carrots are eaten. Furthermore, Amy can not eat a muffin within 5 minutes of eating a carrot and neither can Ben. If Amy eats 40 carrots per minute and 70 muffins per minute and Ben eats 60 carrots per minute and 30 muffins per minute, what is the minimum number of minutes it will take them to finish the food?",
      "answer": "23.5"
    },
    {
      "question": "Jessica has three marbles colored red, green, and blue. She randomly selects a non-empty subset of them (such that each subset is equally likely) and puts them in a bag. You then draw three marbles from the bag with replacement. The colors you see are red, blue, red. What is the probability that the only marbles in the bag are red and blue?",
      "answer": "\\frac{27}{35}"
    },
    {
      "question": "A student at Harvard named Kevin was counting his stones by 11. He messed up $n$ times and instead counted 9s and wound up at 2007. How many values of $n$ could make this limerick true?",
      "answer": "21"
    },
    {
      "question": "Let $ a_1 \\equal{} 11^{11}, \\, a_2 \\equal{} 12^{12}, \\, a_3 \\equal{} 13^{13}$, and $ a_n \\equal{} |a_{n \\minus{} 1} \\minus{} a_{n \\minus{} 2}| \\plus{} |a_{n \\minus{} 2} \\minus{} a_{n \\minus{} 3}|, n \\geq 4.$ Determine $ a_{14^{14}}$.",
      "answer": "1"
    },
    {
      "question": "Let $S$ be a subset of the set $\\{1,2,3, \\ldots, 2015\\}$ such that for any two elements $a, b \\in S$, the difference $a-b$ does not divide the sum $a+b$. Find the maximum possible size of $S$.",
      "answer": "672"
    },
    {
      "question": "Meghana writes two (not necessarily distinct) primes $q$ and $r$ in base 10 next to each other on a blackboard, resulting in the concatenation of $q$ and $r$ (for example, if $q=13$ and $r=5$, the number on the blackboard is now 135). She notices that three more than the resulting number is the square of a prime $p$. Find all possible values of $p$.",
      "answer": "5"
    },
    {
      "question": "If $x=1$ is a solution of the equation $x^{2} + ax + 1 = 0$, what is the value of $a$?",
      "answer": "-2"
    },
    {
      "question": "Find the number of 10-digit numbers $\\overline{a_{1} a_{2} \\cdots a_{10}}$ which are multiples of 11 such that the digits are non-increasing from left to right, i.e. $a_{i} \\geq a_{i+1}$ for each $1 \\leq i \\leq 9$.",
      "answer": "2001"
    },
    {
      "question": "A chord is drawn on a circle by choosing two points uniformly at random along its circumference. This is done two more times to obtain three total random chords. The circle is cut along these three lines, splitting it into pieces. The probability that one of the pieces is a triangle is $\\frac{m}{n}$, where $m, n$ are positive integers and $\\operatorname{gcd}(m, n)=1$. Find $100 m+n$.",
      "answer": "115"
    },
    {
      "question": "Define $a$ ? $=(a-1) /(a+1)$ for $a \\neq-1$. Determine all real values $N$ for which $(N ?)$ ?=\\tan 15.",
      "answer": "-2-\\sqrt{3}"
    },
    {
      "question": "Let $N$ be a three-digit integer such that the difference between any two positive integer factors of $N$ is divisible by 3 . Let $d(N)$ denote the number of positive integers which divide $N$. Find the maximum possible value of $N \\cdot d(N)$.",
      "answer": "5586"
    },
    {
      "question": "For which maximal $N$ there exists an $N$-digit number with the following property: among any sequence of its consecutive decimal digits some digit is present once only?\n\t\n\tAlexey Glebov",
      "answer": "1023"
    },
    {
      "question": "Regular polygons $I C A O, V E N T I$, and $A L B E D O$ lie on a plane. Given that $I N=1$, compute the number of possible values of $O N$.",
      "answer": "2"
    },
    {
      "question": "I have written a strictly increasing sequence of six positive integers, such that each number (besides the first) is a multiple of the one before it, and the sum of all six numbers is 79 . What is the largest number in my sequence?",
      "answer": "48"
    },
    {
      "question": "Compute the sum of all positive integers $a \\leq 26$ for which there exist integers $b$ and $c$ such that $a+23 b+15 c-2$ and $2 a+5 b+14 c-8$ are both multiples of 26.",
      "answer": "31"
    },
    {
      "question": "Let $a_{1}, a_{2}, a_{3}, a_{4}, a_{5}$ be real numbers whose sum is 20. Determine with proof the smallest possible value of \\(\\sum_{1 \\leq i<j \\leq 5}\\left\\lfloor a_{i}+a_{j}\\right\\rfloor\\).",
      "answer": "\\[\n72\n\\]"
    },
    {
      "question": "Given a acute triangle $PA_1B_1$ is inscribed in the circle $\\Gamma$ with radius $1$. for all  integers $n \\ge 1$ are defined:\n$C_n$ the foot of the perpendicular from $P$ to $A_nB_n$\n$O_n$ is the center of $\\odot (PA_nB_n)$\n$A_{n+1}$ is the foot of the perpendicular from $C_n$ to $PA_n$\n$B_{n+1} \\equiv PB_n \\cap O_nA_{n+1}$\n\nIf $PC_1 =\\sqrt{2}$, find the length of $PO_{2015}$ \n\n[hide=Source]Cono Sur Olympiad - 2015 - Day 1 - Problem 3[/hide]",
      "answer": "\\frac{1}{2^{1007}}"
    },
    {
      "question": "Let $s(n)$ denote the sum of the digits (in base ten) of a positive integer $n$. Compute the number of positive integers $n$ at most $10^{4}$ that satisfy $$s(11 n)=2 s(n)$$",
      "answer": "2530"
    },
    {
      "question": "To set up for a Fourth of July party, David is making a string of red, white, and blue balloons. He places them according to the following rules: - No red balloon is adjacent to another red balloon. - White balloons appear in groups of exactly two, and groups of white balloons are separated by at least two non-white balloons. - Blue balloons appear in groups of exactly three, and groups of blue balloons are separated by at least three non-blue balloons. If David uses over 600 balloons, determine the smallest number of red balloons that he can use.",
      "answer": "99"
    },
    {
      "question": "Let $P_{1}, P_{2}, \\ldots, P_{8}$ be 8 distinct points on a circle. Determine the number of possible configurations made by drawing a set of line segments connecting pairs of these 8 points, such that: (1) each $P_{i}$ is the endpoint of at most one segment and (2) two no segments intersect. (The configuration with no edges drawn is allowed.)",
      "answer": "323"
    },
    {
      "question": "For a string of $P$ 's and $Q$ 's, the value is defined to be the product of the positions of the $P$ 's. For example, the string $P P Q P Q Q$ has value $1 \\cdot 2 \\cdot 4=8$. Also, a string is called antipalindromic if writing it backwards, then turning all the $P$ 's into $Q$ 's and vice versa, produces the original string. For example, $P P Q P Q Q$ is antipalindromic. There are $2^{1002}$ antipalindromic strings of length 2004. Find the sum of the reciprocals of their values.",
      "answer": "2005^{1002} / 2004!"
    },
    {
      "question": "Suppose that a positive integer $N$ can be expressed as the sum of $k$ consecutive positive integers \\[ N = a + (a+1) +(a+2) + \\cdots + (a+k-1) \\] for $k=2017$ but for no other values of $k>1$. Considering all positive integers $N$ with this property, what is the smallest positive integer $a$ that occurs in any of these expressions?",
      "answer": "16"
    },
    {
      "question": "Calculate the value of $(3,1) \\nabla (4,2)$ using the operation ' $\\nabla$ ' defined by $(a, b) \\nabla (c, d)=ac+bd$.",
      "answer": "14"
    },
    {
      "question": "Consider a square, inside which is inscribed a circle, inside which is inscribed a square, inside which is inscribed a circle, and so on, with the outermost square having side length 1. Find the difference between the sum of the areas of the squares and the sum of the areas of the circles.",
      "answer": "2 - \\frac{\\pi}{2}"
    },
    {
      "question": "How many students chose Greek food if 200 students were asked to choose between pizza, Thai food, or Greek food, and the circle graph shows the results?",
      "answer": "100"
    },
    {
      "question": "For each integer $x$ with $1 \\leq x \\leq 10$, a point is randomly placed at either $(x, 1)$ or $(x,-1)$ with equal probability. What is the expected area of the convex hull of these points? Note: the convex hull of a finite set is the smallest convex polygon containing it.",
      "answer": "\\frac{1793}{128}"
    },
    {
      "question": "Turbo the snail sits on a point on a circle with circumference $1$. Given an infinite sequence of positive real numbers $c_1, c_2, c_3, \\dots$, Turbo successively crawls distances $c_1, c_2, c_3, \\dots$ around the circle, each time choosing to crawl either clockwise or counterclockwise.\nDetermine the largest constant $C > 0$ with the following property: for every sequence of positive real numbers $c_1, c_2, c_3, \\dots$ with $c_i < C$ for all $i$, Turbo can (after studying the sequence) ensure that there is some point on the circle that it will never visit or crawl across.",
      "answer": "0.5"
    },
    {
      "question": "Let $A B C D$ be a convex quadrilateral inscribed in a circle with shortest side $A B$. The ratio $[B C D] /[A B D]$ is an integer (where $[X Y Z]$ denotes the area of triangle $X Y Z$.) If the lengths of $A B, B C, C D$, and $D A$ are distinct integers no greater than 10, find the largest possible value of $A B$.",
      "answer": "5"
    },
    {
      "question": "Let \\(\\triangle A B C\\) be a right triangle with right angle \\(C\\). Let \\(I\\) be the incenter of \\(A B C\\), and let \\(M\\) lie on \\(A C\\) and \\(N\\) on \\(B C\\), respectively, such that \\(M, I, N\\) are collinear and \\(\\overline{M N}\\) is parallel to \\(A B\\). If \\(A B=36\\) and the perimeter of \\(C M N\\) is 48, find the area of \\(A B C\\).",
      "answer": "252"
    },
    {
      "question": "A function $f: \\mathbb{Z} \\rightarrow \\mathbb{Z}$ satisfies: $f(0)=0$ and $$\\left|f\\left((n+1) 2^{k}\\right)-f\\left(n 2^{k}\\right)\\right| \\leq 1$$ for all integers $k \\geq 0$ and $n$. What is the maximum possible value of $f(2019)$?",
      "answer": "4"
    },
    {
      "question": "Karen has seven envelopes and seven letters of congratulations to various HMMT coaches. If she places the letters in the envelopes at random with each possible configuration having an equal probability, what is the probability that exactly six of the letters are in the correct envelopes?",
      "answer": "0"
    },
    {
      "question": "$S$ is a set of complex numbers such that if $u, v \\in S$, then $u v \\in S$ and $u^{2}+v^{2} \\in S$. Suppose that the number $N$ of elements of $S$ with absolute value at most 1 is finite. What is the largest possible value of $N$ ?",
      "answer": "13"
    },
    {
      "question": "Two mathematicians, Kelly and Jason, play a cooperative game. The computer selects some secret positive integer $n<60$ (both Kelly and Jason know that $n<60$, but that they don't know what the value of $n$ is). The computer tells Kelly the unit digit of $n$, and it tells Jason the number of divisors of $n$. Then, Kelly and Jason have the following dialogue: Kelly: I don't know what $n$ is, and I'm sure that you don't know either. However, I know that $n$ is divisible by at least two different primes. Jason: Oh, then I know what the value of $n$ is. Kelly: Now I also know what $n$ is. Assuming that both Kelly and Jason speak truthfully and to the best of their knowledge, what are all the possible values of $n$?",
      "answer": "10"
    },
    {
      "question": "An integer $n$ is chosen uniformly at random from the set $\\{1,2,3, \\ldots, 2023!\\}$. Compute the probability that $$\\operatorname{gcd}\\left(n^{n}+50, n+1\\right)=1$$",
      "answer": "\\frac{265}{357}"
    },
    {
      "question": "Ewan writes out a sequence where he counts by 11s starting at 3. Which number will appear in Ewan's sequence?",
      "answer": "113"
    },
    {
      "question": "A number $n$ is $b a d$ if there exists some integer $c$ for which $x^{x} \\equiv c(\\bmod n)$ has no integer solutions for $x$. Find the number of bad integers between 2 and 42 inclusive.",
      "answer": "25"
    },
    {
      "question": "In unit square $A B C D$, points $E, F, G$ are chosen on side $B C, C D, D A$ respectively such that $A E$ is perpendicular to $E F$ and $E F$ is perpendicular to $F G$. Given that $G A=\\frac{404}{1331}$, find all possible values of the length of $B E$.",
      "answer": "\\frac{9}{11}"
    },
    {
      "question": "As part of his effort to take over the world, Edward starts producing his own currency. As part of an effort to stop Edward, Alex works in the mint and produces 1 counterfeit coin for every 99 real ones. Alex isn't very good at this, so none of the counterfeit coins are the right weight. Since the mint is not perfect, each coin is weighed before leaving. If the coin is not the right weight, then it is sent to a lab for testing. The scale is accurate $95 \\%$ of the time, $5 \\%$ of all the coins minted are sent to the lab, and the lab's test is accurate $90 \\%$ of the time. If the lab says a coin is counterfeit, what is the probability that it really is?",
      "answer": "\\frac{19}{28}"
    },
    {
      "question": "Define the polynomials $P_0, P_1, P_2 \\cdots$ by:\n\\[ P_0(x)=x^3+213x^2-67x-2000 \\]\n\\[ P_n(x)=P_{n-1}(x-n), n \\in N \\]\nFind the coefficient of $x$ in $P_{21}(x)$.",
      "answer": "61610"
    },
    {
      "question": "There are 42 stepping stones in a pond, arranged along a circle. You are standing on one of the stones. You would like to jump among the stones so that you move counterclockwise by either 1 stone or 7 stones at each jump. Moreover, you would like to do this in such a way that you visit each stone (except for the starting spot) exactly once before returning to your initial stone for the first time. In how many ways can you do this?",
      "answer": "63"
    },
    {
      "question": "Determine the remainder when $$\\sum_{i=0}^{2015}\\left\\lfloor\\frac{2^{i}}{25}\\right\\rfloor$$ is divided by 100, where $\\lfloor x\\rfloor$ denotes the largest integer not greater than $x$.",
      "answer": "14"
    },
    {
      "question": "Determine the sum of all distinct real values of $x$ such that $|||\\cdots||x|+x|\\cdots|+x|+x|=1$ where there are 2017 $x$ 's in the equation.",
      "answer": "-\\frac{2016}{2017}"
    },
    {
      "question": "Let $n$ be a positive integer, and let $s$ be the sum of the digits of the base-four representation of $2^{n}-1$. If $s=2023$ (in base ten), compute $n$ (in base ten).",
      "answer": "1349"
    },
    {
      "question": "Let $A B C D E$ be a convex pentagon such that $$\\begin{aligned} & A B+B C+C D+D E+E A=64 \\text { and } \\\\ & A C+C E+E B+B D+D A=72 \\end{aligned}$$ Compute the perimeter of the convex pentagon whose vertices are the midpoints of the sides of $A B C D E$.",
      "answer": "36"
    },
    {
      "question": "For how many pairs $(m, n)$ with $m$ and $n$ integers satisfying $1 \\leq m \\leq 100$ and $101 \\leq n \\leq 205$ is $3^{m}+7^{n}$ divisible by 10?",
      "answer": "2625"
    },
    {
      "question": "In convex quadrilateral \\(ABCD\\) with \\(AB=11\\) and \\(CD=13\\), there is a point \\(P\\) for which \\(\\triangle ADP\\) and \\(\\triangle BCP\\) are congruent equilateral triangles. Compute the side length of these triangles.",
      "answer": "7"
    },
    {
      "question": "Alex has a $20 \\times 16$ grid of lightbulbs, initially all off. He has 36 switches, one for each row and column. Flipping the switch for the $i$th row will toggle the state of each lightbulb in the $i$th row (so that if it were on before, it would be off, and vice versa). Similarly, the switch for the $j$th column will toggle the state of each bulb in the $j$th column. Alex makes some (possibly empty) sequence of switch flips, resulting in some configuration of the lightbulbs and their states. How many distinct possible configurations of lightbulbs can Alex achieve with such a sequence? Two configurations are distinct if there exists a lightbulb that is on in one configuration and off in another.",
      "answer": "2^{35}"
    },
    {
      "question": "For what value of $k$ is the line through the points $(3,2k+1)$ and $(8,4k-5)$ parallel to the $x$-axis?",
      "answer": "3"
    },
    {
      "question": "Let $f(n)$ be the number of ways to write $n$ as a sum of powers of $2$ , where we keep track of the order of the summation.  For example, $f(4)=6$ because $4$ can be written as $4$ , $2+2$ , $2+1+1$ , $1+2+1$ , $1+1+2$ , and $1+1+1+1$ .  Find the smallest $n$ greater than $2013$ for which $f(n)$ is odd.",
      "answer": "\\[ 2047 \\]"
    },
    {
      "question": "Let $z$ be a complex number and $k$ a positive integer such that $z^{k}$ is a positive real number other than 1. Let $f(n)$ denote the real part of the complex number $z^{n}$. Assume the parabola $p(n)=an^{2}+bn+c$ intersects $f(n)$ four times, at $n=0,1,2,3$. Assuming the smallest possible value of $k$, find the largest possible value of $a$.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Let $S$ be a set of positive integers satisfying the following two conditions: - For each positive integer $n$, at least one of $n, 2 n, \\ldots, 100 n$ is in $S$. - If $a_{1}, a_{2}, b_{1}, b_{2}$ are positive integers such that $\\operatorname{gcd}\\left(a_{1} a_{2}, b_{1} b_{2}\\right)=1$ and $a_{1} b_{1}, a_{2} b_{2} \\in S$, then $a_{2} b_{1}, a_{1} b_{2} \\in S$ Suppose that $S$ has natural density $r$. Compute the minimum possible value of $\\left\\lfloor 10^{5} r\\right\\rfloor$. Note: $S$ has natural density $r$ if $\\frac{1}{n}|S \\cap\\{1, \\ldots, n\\}|$ approaches $r$ as $n$ approaches $\\infty$.",
      "answer": "396"
    },
    {
      "question": "What is the value of \\( \\frac{5-2}{2+1} \\)?",
      "answer": "1"
    },
    {
      "question": "Determine which of the following expressions has the largest value: $4^2$, $4 \\times 2$, $4 - 2$, $\\frac{4}{2}$, or $4 + 2$.",
      "answer": "16"
    },
    {
      "question": "For each real number $x$, let\n\\[\nf(x) = \\sum_{n\\in S_x} \\frac{1}{2^n},\n\\]\nwhere $S_x$ is the set of positive integers $n$ for which $\\lfloor nx \\rfloor$ is even. What is the largest real number $L$ such that $f(x) \\geq L$ for all $x \\in [0,1)$? (As usual, $\\lfloor z \\rfloor$ denotes the greatest integer less than or equal to $z$.)",
      "answer": "4/7"
    },
    {
      "question": "For positive integers $a$ and $b$ such that $a$ is coprime to $b$, define $\\operatorname{ord}_{b}(a)$ as the least positive integer $k$ such that $b \\mid a^{k}-1$, and define $\\varphi(a)$ to be the number of positive integers less than or equal to $a$ which are coprime to $a$. Find the least positive integer $n$ such that $$\\operatorname{ord}_{n}(m)<\\frac{\\varphi(n)}{10}$$ for all positive integers $m$ coprime to $n$.",
      "answer": "240"
    },
    {
      "question": "Count the number of functions $f: \\mathbb{Z} \\rightarrow\\{$ 'green', 'blue' $\\}$ such that $f(x)=f(x+22)$ for all integers $x$ and there does not exist an integer $y$ with $f(y)=f(y+2)=$ 'green'.",
      "answer": "39601"
    },
    {
      "question": "Given a positive integer $n$, let $M(n)$ be the largest integer $m$ such that \\[ \\binom{m}{n-1} > \\binom{m-1}{n}. \\] Evaluate \\[ \\lim_{n \\to \\infty} \\frac{M(n)}{n}. \\]",
      "answer": "\\frac{3+\\sqrt{5}}{2}"
    },
    {
      "question": "FemtoPravis is walking on an $8 \\times 8$ chessboard that wraps around at its edges (so squares on the left edge of the chessboard are adjacent to squares on the right edge, and similarly for the top and bottom edges). Each femtosecond, FemtoPravis moves in one of the four diagonal directions uniformly at random. After 2012 femtoseconds, what is the probability that FemtoPravis is at his original location?",
      "answer": "\\left(\\frac{1+2^{1005}}{2^{1007}}\\right)^{2}"
    },
    {
      "question": "Compute the smallest positive integer $n$ for which $$0<\\sqrt[4]{n}-\\lfloor\\sqrt[4]{n}\\rfloor<\\frac{1}{2015}$$",
      "answer": "4097"
    },
    {
      "question": "Write 1 as a sum of 4 distinct unit fractions.",
      "answer": "\\frac{1}{2}+\\frac{1}{3}+\\frac{1}{7}+\\frac{1}{42}"
    },
    {
      "question": "How many regions of the plane are bounded by the graph of $$x^{6}-x^{5}+3 x^{4} y^{2}+10 x^{3} y^{2}+3 x^{2} y^{4}-5 x y^{4}+y^{6}=0 ?$$",
      "answer": "5"
    },
    {
      "question": "Let $N$ be the largest positive integer that can be expressed as a 2013-digit base -4 number. What is the remainder when $N$ is divided by 210?",
      "answer": "51"
    },
    {
      "question": "What is the integer formed by the rightmost two digits of the integer equal to \\(4^{127} + 5^{129} + 7^{131}\\)?",
      "answer": "52"
    },
    {
      "question": "Kristoff is planning to transport a number of indivisible ice blocks with positive integer weights from the north mountain to Arendelle. He knows that when he reaches Arendelle, Princess Anna and Queen Elsa will name an ordered pair $(p, q)$ of nonnegative integers satisfying $p+q \\leq 2016$. Kristoff must then give Princess Anna exactly $p$ kilograms of ice. Afterward, he must give Queen Elsa exactly $q$ kilograms of ice. What is the minimum number of blocks of ice Kristoff must carry to guarantee that he can always meet Anna and Elsa's demands, regardless of which $p$ and $q$ are chosen?",
      "answer": "18"
    },
    {
      "question": "There are $2n$ students in a school $(n \\in \\mathbb{N}, n \\geq 2)$. Each week $n$ students go on a trip. After several trips the following condition was fulfilled: every two students were together on at least one trip. What is the minimum number of trips needed for this to happen?",
      "answer": "6"
    },
    {
      "question": "Find the largest real number $\\lambda$ such that $a^{2}+b^{2}+c^{2}+d^{2} \\geq a b+\\lambda b c+c d$ for all real numbers $a, b, c, d$.",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "What is the perimeter of the figure shown if $x=3$?",
      "answer": "23"
    },
    {
      "question": "Let $\\Gamma$ denote the circumcircle of triangle $A B C$. Point $D$ is on $\\overline{A B}$ such that $\\overline{C D}$ bisects $\\angle A C B$. Points $P$ and $Q$ are on $\\Gamma$ such that $\\overline{P Q}$ passes through $D$ and is perpendicular to $\\overline{C D}$. Compute $P Q$, given that $B C=20, C A=80, A B=65$.",
      "answer": "4 \\sqrt{745}"
    },
    {
      "question": "Find $PB$ given that $AP$ is a tangent to $\\Omega$, $\\angle PAB=\\angle PCA$, and $\\frac{PB}{PA}=\\frac{4}{7}=\\frac{PA}{PB+6}$.",
      "answer": "\\frac{32}{11}"
    },
    {
      "question": "The altitudes of a triangle are $12$, $15$, and $20$.  What is the area of this triangle?",
      "answer": "150"
    },
    {
      "question": "A right triangle has side lengths $a, b$, and $\\sqrt{2016}$ in some order, where $a$ and $b$ are positive integers. Determine the smallest possible perimeter of the triangle.",
      "answer": "48+\\sqrt{2016}"
    },
    {
      "question": "In this final problem, a ball is again launched from the vertex of an equilateral triangle with side length 5. In how many ways can the ball be launched so that it will return again to a vertex for the first time after 2009 bounces?",
      "answer": "502"
    },
    {
      "question": "Sergio recently opened a store. One day, he determined that the average number of items sold per employee to date was 75. The next day, one employee sold 6 items, one employee sold 5 items, and one employee sold 4 items. The remaining employees each sold 3 items. This made the new average number of items sold per employee to date equal to 78.3. How many employees are there at the store?",
      "answer": "20"
    },
    {
      "question": "A square grid $100 \\times 100$ is tiled in two ways - only with dominoes and only with squares $2 \\times 2$. What is the least number of dominoes that are entirely inside some square $2 \\times 2$?",
      "answer": "100"
    },
    {
      "question": "Each of the four digits of the integer 2024 is even. How many integers between 1000 and 9999, inclusive, have the property that all four of their digits are even?",
      "answer": "500"
    },
    {
      "question": "A string has been cut into 4 pieces, all of different lengths. The length of each piece is 2 times the length of the next smaller piece. What fraction of the original string is the longest piece?",
      "answer": "\\frac{8}{15}"
    },
    {
      "question": "A positive integer \\overline{A B C}, where $A, B, C$ are digits, satisfies $\\overline{A B C}=B^{C}-A$. Find $\\overline{A B C}$.",
      "answer": "127"
    },
    {
      "question": "A sequence of positive integers is given by $a_{1}=1$ and $a_{n}=\\operatorname{gcd}\\left(a_{n-1}, n\\right)+1$ for $n>1$. Calculate $a_{2002}$.",
      "answer": "3"
    },
    {
      "question": "A hotel consists of a $2 \\times 8$ square grid of rooms, each occupied by one guest. All the guests are uncomfortable, so each guest would like to move to one of the adjoining rooms (horizontally or vertically). Of course, they should do this simultaneously, in such a way that each room will again have one guest. In how many different ways can they collectively move?",
      "answer": "1156"
    },
    {
      "question": "Jim wrote a sequence of symbols a total of 50 times. How many more of one symbol than another did he write?",
      "answer": "150"
    },
    {
      "question": "The L shape made by adjoining three congruent squares can be subdivided into four smaller L shapes. Each of these can in turn be subdivided, and so forth. If we perform 2005 successive subdivisions, how many of the $4^{2005}$ L's left at the end will be in the same orientation as the original one?",
      "answer": "4^{2004}+2^{2004}"
    },
    {
      "question": "Find the number of integers $n$ with $1 \\leq n \\leq 2017$ so that $(n-2)(n-0)(n-1)(n-7)$ is an integer multiple of 1001.",
      "answer": "99"
    },
    {
      "question": "In acute triangle $ABC$, let $H$ be the orthocenter and $D$ the foot of the altitude from $A$. The circumcircle of triangle $BHC$ intersects $AC$ at $E \\neq C$, and $AB$ at $F \\neq B$. If $BD=3, CD=7$, and $\\frac{AH}{HD}=\\frac{5}{7}$, the area of triangle $AEF$ can be expressed as $\\frac{a}{b}$, where $a, b$ are relatively prime positive integers. Compute $100a+b$.",
      "answer": "12017"
    },
    {
      "question": "For each continuous function $f: [0,1] \\to \\mathbb{R}$, let $I(f) = \\int_0^1 x^2 f(x)\\,dx$ and $J(x) = \\int_0^1 x \\left(f(x)\\right)^2\\,dx$. Find the maximum value of $I(f) - J(f)$ over all such functions $f$.",
      "answer": "1/16"
    },
    {
      "question": "A subset $S$ of the nonnegative integers is called supported if it contains 0, and $k+8, k+9 \\in S$ for all $k \\in S$. How many supported sets are there?",
      "answer": "1430"
    },
    {
      "question": "Let $x$ be a positive real number. Find the maximum possible value of $$\\frac{x^{2}+2-\\sqrt{x^{4}+4}}{x}$$",
      "answer": "2 \\sqrt{2}-2"
    },
    {
      "question": "The area of the largest regular hexagon that can fit inside of a rectangle with side lengths 20 and 22 can be expressed as $a \\sqrt{b}-c$, for positive integers $a, b$, and $c$, where $b$ is squarefree. Compute $100 a+10 b+c$.",
      "answer": "134610"
    },
    {
      "question": "For integers $a, b, c, d$, let $f(a, b, c, d)$ denote the number of ordered pairs of integers $(x, y) \\in \\{1,2,3,4,5\\}^{2}$ such that $a x+b y$ and $c x+d y$ are both divisible by 5. Find the sum of all possible values of $f(a, b, c, d)$.",
      "answer": "31"
    },
    {
      "question": "Crisp All, a basketball player, is dropping dimes and nickels on a number line. Crisp drops a dime on every positive multiple of 10 , and a nickel on every multiple of 5 that is not a multiple of 10. Crisp then starts at 0 . Every second, he has a $\\frac{2}{3}$ chance of jumping from his current location $x$ to $x+3$, and a $\\frac{1}{3}$ chance of jumping from his current location $x$ to $x+7$. When Crisp jumps on either a dime or a nickel, he stops jumping. What is the probability that Crisp stops on a dime?",
      "answer": "\\frac{20}{31}"
    },
    {
      "question": "In a party with $1982$ people, among any group of four there is at least one person who knows each of the other three. What is the minimum number of people in the party who know everyone else?",
      "answer": "\\[ 1979 \\]"
    },
    {
      "question": "Let $ ABP, BCQ, CAR$ be three non-overlapping triangles erected outside of acute triangle $ ABC$. Let $ M$ be the midpoint of segment $ AP$. Given that $ \\angle PAB \\equal{} \\angle CQB \\equal{} 45^\\circ$, $ \\angle ABP \\equal{} \\angle QBC \\equal{} 75^\\circ$, $ \\angle RAC \\equal{} 105^\\circ$, and $ RQ^2 \\equal{} 6CM^2$, compute $ AC^2/AR^2$.\r\n\r\n[i]Zuming Feng.[/i]",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "For non-negative real numbers $x_1, x_2, \\ldots, x_n$ which satisfy $x_1 + x_2 + \\cdots + x_n = 1$, find the largest possible value of $\\sum_{j = 1}^{n} (x_j^{4} - x_j^{5})$.",
      "answer": "\\frac{1}{12}"
    },
    {
      "question": "In the sum shown, each letter represents a different digit with $T \\neq 0$ and $W \\neq 0$. How many different values of $U$ are possible? \n\n\\begin{tabular}{rrrrr} \n& $W$ & $X$ & $Y$ & $Z$ \\\\ \n+ & $W$ & $X$ & $Y$ & $Z$ \\\\ \n\\hline & $W$ & $U$ & $Y$ & $V$ \n\\end{tabular}",
      "answer": "3"
    },
    {
      "question": "Rosencrantz and Guildenstern are playing a game where they repeatedly flip coins. Rosencrantz wins if 1 heads followed by 2009 tails appears. Guildenstern wins if 2010 heads come in a row. They will flip coins until someone wins. What is the probability that Rosencrantz wins?",
      "answer": "$\\sqrt{\\frac{2^{2009}-1}{3 \\cdot 2^{2008}-1}}$"
    },
    {
      "question": "Determine the number of integers $D$ such that whenever $a$ and $b$ are both real numbers with $-1 / 4<a, b<1 / 4$, then $\\left|a^{2}-D b^{2}\\right|<1$.",
      "answer": "32"
    },
    {
      "question": "Stan has a stack of 100 blocks and starts with a score of 0, and plays a game in which he iterates the following two-step procedure: (a) Stan picks a stack of blocks and splits it into 2 smaller stacks each with a positive number of blocks, say $a$ and $b$. (The order in which the new piles are placed does not matter.) (b) Stan adds the product of the two piles' sizes, $a b$, to his score. The game ends when there are only 1-block stacks left. What is the expected value of Stan's score at the end of the game?",
      "answer": "4950"
    },
    {
      "question": "Let $z$ be a non-real complex number with $z^{23}=1$. Compute $$ \\sum_{k=0}^{22} \\frac{1}{1+z^{k}+z^{2 k}} $$",
      "answer": "46 / 3"
    },
    {
      "question": "Let $f$ be a polynomial with integer coefficients such that the greatest common divisor of all its coefficients is 1. For any $n \\in \\mathbb{N}, f(n)$ is a multiple of 85. Find the smallest possible degree of $f$.",
      "answer": "17"
    },
    {
      "question": "If $x$ and $y$ are positive integers with $x+y=31$, what is the largest possible value of $x y$?",
      "answer": "240"
    },
    {
      "question": "How many positive integers \\( n \\) between 10 and 1000 have the property that the sum of the digits of \\( n \\) is 3?",
      "answer": "9"
    },
    {
      "question": "The real numbers $x, y, z, w$ satisfy $$\\begin{aligned} & 2 x+y+z+w=1 \\\\ & x+3 y+z+w=2 \\\\ & x+y+4 z+w=3 \\\\ & x+y+z+5 w=25 \\end{aligned}$$ Find the value of $w$.",
      "answer": "11/2"
    },
    {
      "question": "A positive integer is called primer if it has a prime number of distinct prime factors. A positive integer is called primest if it has a primer number of distinct primer factors. A positive integer is called prime-minister if it has a primest number of distinct primest factors. Let $N$ be the smallest prime-minister number. Estimate $N$.",
      "answer": "378000"
    },
    {
      "question": "The spikiness of a sequence $a_{1}, a_{2}, \\ldots, a_{n}$ of at least two real numbers is the sum $\\sum_{i=1}^{n-1}\\left|a_{i+1}-a_{i}\\right|$. Suppose $x_{1}, x_{2}, \\ldots, x_{9}$ are chosen uniformly and randomly from the interval $[0,1]$. Let $M$ be the largest possible value of the spikiness of a permutation of $x_{1}, x_{2}, \\ldots, x_{9}$. Compute the expected value of $M$.",
      "answer": "\\frac{79}{20}"
    },
    {
      "question": "In-Young generates a string of $B$ zeroes and ones using the following method:\n- First, she flips a fair coin. If it lands heads, her first digit will be a 0, and if it lands tails, her first digit will be a 1.\n- For each subsequent bit, she flips an unfair coin, which lands heads with probability $A$. If the coin lands heads, she writes down the number (zero or one) different from previous digit, while if the coin lands tails, she writes down the previous digit again.\n\nWhat is the expected value of the number of zeroes in her string?",
      "answer": "2"
    },
    {
      "question": "Let $x$ be a real number such that $2^{x}=3$. Determine the value of $4^{3 x+2}$.",
      "answer": "11664"
    },
    {
      "question": "Find the smallest positive integer $n$ such that the polynomial $(x+1)^{n}-1$ is \"divisible by $x^{2}+1$ modulo 3\", or more precisely, either of the following equivalent conditions holds: there exist polynomials $P, Q$ with integer coefficients such that $(x+1)^{n}-1=\\left(x^{2}+1\\right) P(x)+3 Q(x)$; or more conceptually, the remainder when (the polynomial) $(x+1)^{n}-1$ is divided by (the polynomial) $x^{2}+1$ is a polynomial with (integer) coefficients all divisible by 3.",
      "answer": "8"
    },
    {
      "question": "On a game show, Merble will be presented with a series of 2013 marbles, each of which is either red or blue on the outside. Each time he sees a marble, he can either keep it or pass, but cannot return to a previous marble; he receives 3 points for keeping a red marble, loses 2 points for keeping a blue marble, and gains 0 points for passing. All distributions of colors are equally likely and Merble can only see the color of his current marble. If his goal is to end with exactly one point and he plays optimally, what is the probability that he fails?",
      "answer": "\\frac{1}{2^{2012}}"
    },
    {
      "question": "A parking lot consists of 2012 parking spots equally spaced in a line, numbered 1 through 2012. One by one, 2012 cars park in these spots under the following procedure: the first car picks from the 2012 spots uniformly randomly, and each following car picks uniformly randomly among all possible choices which maximize the minimal distance from an already parked car. What is the probability that the last car to park must choose spot 1?",
      "answer": "\\frac{1}{2062300}"
    },
    {
      "question": "Let $r=H_{1}$ be the answer to this problem. Given that $r$ is a nonzero real number, what is the value of $r^{4}+4 r^{3}+6 r^{2}+4 r ?$",
      "answer": "-1"
    },
    {
      "question": "For positive integers $x$, let $g(x)$ be the number of blocks of consecutive 1's in the binary expansion of $x$. For example, $g(19)=2$ because $19=10011_{2}$ has a block of one 1 at the beginning and a block of two 1's at the end, and $g(7)=1$ because $7=111_{2}$ only has a single block of three 1's. Compute $g(1)+g(2)+g(3)+\\cdots+g(256)$.",
      "answer": "577"
    },
    {
      "question": "One bag contains 2 red marbles and 2 blue marbles. A second bag contains 2 red marbles, 2 blue marbles, and $g$ green marbles, with $g>0$. For each bag, Maria calculates the probability of randomly drawing two marbles of the same colour in two draws from that bag, without replacement. If these two probabilities are equal, what is the value of $g$?",
      "answer": "5"
    },
    {
      "question": "Let $\\{a_{i}\\}_{i \\geq 0}$ be a sequence of real numbers defined by $a_{n+1}=a_{n}^{2}-\\frac{1}{2^{2020 \\cdot 2^{n}-1}}$ for $n \\geq 0$. Determine the largest value for $a_{0}$ such that $\\{a_{i}\\}_{i \\geq 0}$ is bounded.",
      "answer": "1+\\frac{1}{2^{2020}}"
    },
    {
      "question": "A ladder is leaning against a house with its lower end 15 feet from the house. When the lower end is pulled 9 feet farther from the house, the upper end slides 13 feet down. How long is the ladder (in feet)?",
      "answer": "25"
    },
    {
      "question": "John has a 1 liter bottle of pure orange juice. He pours half of the contents of the bottle into a vat, fills the bottle with water, and mixes thoroughly. He then repeats this process 9 more times. Afterwards, he pours the remaining contents of the bottle into the vat. What fraction of the liquid in the vat is now water?",
      "answer": "\\frac{5}{6}"
    },
    {
      "question": "A positive integer $n$ is infallible if it is possible to select $n$ vertices of a regular 100-gon so that they form a convex, non-self-intersecting $n$-gon having all equal angles. Find the sum of all infallible integers $n$ between 3 and 100, inclusive.",
      "answer": "262"
    },
    {
      "question": "There are $n \\geq 2$ coins, each with a different positive integer value. Call an integer $m$ sticky if some subset of these $n$ coins have total value $m$. We call the entire set of coins a stick if all the sticky numbers form a consecutive range of integers. Compute the minimum total value of a stick across all sticks containing a coin of value 100.",
      "answer": "199"
    },
    {
      "question": "Suppose we keep rolling a fair 2014-sided die (whose faces are labelled 1, 2, .., 2014) until we obtain a value less than or equal to the previous roll. Let $E$ be the expected number of times we roll the die. Find the nearest integer to $100 E$.",
      "answer": "272"
    },
    {
      "question": "Find the number of pairs of integers \\((a, b)\\) with \\(1 \\leq a<b \\leq 57\\) such that \\(a^{2}\\) has a smaller remainder than \\(b^{2}\\) when divided by 57.",
      "answer": "738"
    },
    {
      "question": "Find the roots of the equation  $(x-a)(x-b)=(x-c)(x-d)$, if you know that $a+d=b+c=2015$ and $a \\ne c$ (numbers $a, b, c, d$ are not given).",
      "answer": "\\frac{2015}{2}"
    },
    {
      "question": "A function $f(x, y)$ is linear in $x$ and in $y . f(x, y)=\\frac{1}{x y}$ for $x, y \\in\\{3,4\\}$. What is $f(5,5)$?",
      "answer": "\\frac{1}{36}"
    },
    {
      "question": "Let $n$ be the maximum number of bishops that can be placed on the squares of a $6 \\times 6$ chessboard such that no two bishops are attacking each other. Let $k$ be the number of ways to put $n$ bishops on an $6 \\times 6$ chessboard such that no two bishops are attacking each other. Find $n+k$. (Two bishops are considered to be attacking each other if they lie on the same diagonal. Equivalently, if we label the squares with coordinates $(x, y)$, with $1 \\leq x, y \\leq 6$, then the bishops on $(a, b)$ and $(c, d)$ are attacking each other if and only if $|a-c|=|b-d|$.)",
      "answer": "74"
    },
    {
      "question": "Find the number of sets of composite numbers less than 23 that sum to 23.",
      "answer": "4"
    },
    {
      "question": "Find the sum of all positive integers $n$ such that $1+2+\\cdots+n$ divides $15\\left[(n+1)^{2}+(n+2)^{2}+\\cdots+(2 n)^{2}\\right]$",
      "answer": "64"
    },
    {
      "question": "Ten positive integers are arranged around a circle. Each number is one more than the greatest common divisor of its two neighbors. What is the sum of the ten numbers?",
      "answer": "28"
    },
    {
      "question": "In an acute scalene triangle $ABC$, points $D,E,F$ lie on sides $BC, CA, AB$, respectively, such that $AD \\perp BC, BE \\perp CA, CF \\perp AB$. Altitudes $AD, BE, CF$ meet at orthocenter $H$. Points $P$ and $Q$ lie on segment $EF$ such that $AP \\perp EF$ and $HQ \\perp EF$. Lines $DP$ and $QH$ intersect at point $R$. Compute $HQ/HR$.",
      "answer": "1"
    },
    {
      "question": "Let $f$ be a function that takes in a triple of integers and outputs a real number. Suppose that $f$ satisfies the equations $f(a, b, c) =\\frac{f(a+1, b, c)+f(a-1, b, c)}{2}$, $f(a, b, c) =\\frac{f(a, b+1, c)+f(a, b-1, c)}{2}$, $f(a, b, c) =\\frac{f(a, b, c+1)+f(a, b, c-1)}{2}$ for all integers $a, b, c$. What is the minimum number of triples at which we need to evaluate $f$ in order to know its value everywhere?",
      "answer": "8"
    },
    {
      "question": "Define $x \\star y=\\frac{\\sqrt{x^{2}+3 x y+y^{2}-2 x-2 y+4}}{x y+4}$. Compute $$((\\cdots((2007 \\star 2006) \\star 2005) \\star \\cdots) \\star 1)$$",
      "answer": "\\frac{\\sqrt{15}}{9}"
    },
    {
      "question": "If $x y=5$ and $x^{2}+y^{2}=21$, compute $x^{4}+y^{4}$.",
      "answer": "391"
    },
    {
      "question": "What is the value of the expression $2 \\times 3 + 2 \\times 3$?",
      "answer": "12"
    },
    {
      "question": "Let $A, E, H, L, T$, and $V$ be chosen independently and at random from the set $\\left\\{0, \\frac{1}{2}, 1\\right\\}$. Compute the probability that $\\lfloor T \\cdot H \\cdot E\\rfloor=L \\cdot A \\cdot V \\cdot A$.",
      "answer": "\\frac{55}{81}"
    },
    {
      "question": "Find the smallest positive integer $n$ such that, if there are initially $2n$ townspeople and 1 goon, then the probability the townspeople win is greater than $50\\%$.",
      "answer": "3"
    },
    {
      "question": "Given a set $A$ with 10 elements, find the number of consistent 2-configurations of $A$ of order 2 with exactly 2 cells.",
      "answer": "99144"
    },
    {
      "question": "In a $k \\times k$ chessboard, a set $S$ of 25 cells that are in a $5 \\times 5$ square is chosen uniformly at random. The probability that there are more black squares than white squares in $S$ is $48 \\%$. Find $k$.",
      "answer": "9"
    },
    {
      "question": "If $wxyz$ is a four-digit positive integer with $w \\neq 0$, the layer sum of this integer equals $wxyz + xyz + yz + z$. If the layer sum of $wxyz$ equals 2014, what is the value of $w + x + y + z$?",
      "answer": "13"
    },
    {
      "question": "A 5 by 5 grid of unit squares is partitioned into 5 pairwise incongruent rectangles with sides lying on the gridlines. Find the maximum possible value of the product of their areas.",
      "answer": "2304"
    },
    {
      "question": "Find the sum of all real numbers $x$ for which $$\\lfloor\\lfloor\\cdots\\lfloor\\lfloor\\lfloor x\\rfloor+x\\rfloor+x\\rfloor \\cdots\\rfloor+x\\rfloor=2017 \\text { and }\\{\\{\\cdots\\{\\{\\{x\\}+x\\}+x\\} \\cdots\\}+x\\}=\\frac{1}{2017}$$ where there are $2017 x$ 's in both equations. ( $\\lfloor x\\rfloor$ is the integer part of $x$, and $\\{x\\}$ is the fractional part of $x$.) Express your sum as a mixed number.",
      "answer": "3025 \\frac{1}{2017}"
    },
    {
      "question": "Suppose that $\\sqrt{\\frac{1}{2} \\times \\frac{2}{3} \\times \\frac{3}{4} \\times \\frac{4}{5} \\times \\cdots \\times \\frac{n-1}{n}} = \\frac{1}{8}$. What is the value of $n$?",
      "answer": "64"
    },
    {
      "question": "Compute the smallest positive integer that does not appear in any problem statement on any round at HMMT November 2023.",
      "answer": "22"
    },
    {
      "question": "What is the smallest positive integer $t$ such that there exist integers $x_1,x_2,\\ldots,x_t$ with  \\[x^3_1+x^3_2+\\,\\ldots\\,+x^3_t=2002^{2002}\\,?\\]",
      "answer": "4"
    },
    {
      "question": "The numbers $1-10$ are written in a circle randomly. Find the expected number of numbers which are at least 2 larger than an adjacent number.",
      "answer": "\\frac{17}{3}"
    },
    {
      "question": "The line with equation $y=2x-6$ is translated upwards by 4 units. What is the $x$-intercept of the resulting line?",
      "answer": "1"
    },
    {
      "question": "Let $S=\\{1,2, \\ldots, 2021\\}$, and let $\\mathcal{F}$ denote the set of functions $f: S \\rightarrow S$. For a function $f \\in \\mathcal{F}$, let $$T_{f}=\\left\\{f^{2021}(s): s \\in S\\right\\}$$ where $f^{2021}(s)$ denotes $f(f(\\cdots(f(s)) \\cdots))$ with 2021 copies of $f$. Compute the remainder when $$\\sum_{f \\in \\mathcal{F}}\\left|T_{f}\\right|$$ is divided by the prime 2017, where the sum is over all functions $f$ in $\\mathcal{F}$.",
      "answer": "255"
    },
    {
      "question": "A rectangle with dimensions 100 cm by 150 cm is tilted so that one corner is 20 cm above a horizontal line, as shown. To the nearest centimetre, the height of vertex $Z$ above the horizontal line is $(100+x) \\mathrm{cm}$. What is the value of $x$?",
      "answer": "67"
    },
    {
      "question": "What is the smallest integer greater than 10 such that the sum of the digits in its base 17 representation is equal to the sum of the digits in its base 10 representation?",
      "answer": "153"
    },
    {
      "question": "In a group of people, there are 13 who like apples, 9 who like blueberries, 15 who like cantaloupe, and 6 who like dates. (A person can like more than 1 kind of fruit.) Each person who likes blueberries also likes exactly one of apples and cantaloupe. Each person who likes cantaloupe also likes exactly one of blueberries and dates. Find the minimum possible number of people in the group.",
      "answer": "22"
    },
    {
      "question": "There are three video game systems: the Paystation, the WHAT, and the ZBoz2 \\pi, and none of these systems will play games for the other systems. Uncle Riemann has three nephews: Bernoulli, Galois, and Dirac. Bernoulli owns a Paystation and a WHAT, Galois owns a WHAT and a ZBoz2 \\pi, and Dirac owns a ZBoz2 \\pi and a Paystation. A store sells 4 different games for the Paystation, 6 different games for the WHAT, and 10 different games for the ZBoz2 \\pi. Uncle Riemann does not understand the difference between the systems, so he walks into the store and buys 3 random games (not necessarily distinct) and randomly hands them to his nephews. What is the probability that each nephew receives a game he can play?",
      "answer": "\\frac{7}{25}"
    },
    {
      "question": "What is the smallest positive integer $n$ which cannot be written in any of the following forms? - $n=1+2+\\cdots+k$ for a positive integer $k$. - $n=p^{k}$ for a prime number $p$ and integer $k$ - $n=p+1$ for a prime number $p$. - $n=p q$ for some distinct prime numbers $p$ and $q$",
      "answer": "40"
    },
    {
      "question": "A function $f$ satisfies, for all nonnegative integers $x$ and $y$: - $f(0, x)=f(x, 0)=x$ - If $x \\geq y \\geq 0, f(x, y)=f(x-y, y)+1$ - If $y \\geq x \\geq 0, f(x, y)=f(x, y-x)+1$ Find the maximum value of $f$ over $0 \\leq x, y \\leq 100$.",
      "answer": "101"
    },
    {
      "question": "The sum of five consecutive odd integers is 125. What is the smallest of these integers?",
      "answer": "21"
    },
    {
      "question": "A sequence is defined by $A_{0}=0, A_{1}=1, A_{2}=2$, and, for integers $n \\geq 3$, $$A_{n}=\\frac{A_{n-1}+A_{n-2}+A_{n-3}}{3}+\\frac{1}{n^{4}-n^{2}}$$ Compute $\\lim _{N \\rightarrow \\infty} A_{N}$.",
      "answer": "\\frac{13}{6}-\\frac{\\pi^{2}}{12}"
    },
    {
      "question": "Reduce the number $\\sqrt[3]{2+\\sqrt{5}}+\\sqrt[3]{2-\\sqrt{5}}$.",
      "answer": "1"
    },
    {
      "question": "The expression $\\lfloor x\\rfloor$ denotes the greatest integer less than or equal to $x$. Find the value of $$\\left\\lfloor\\frac{2002!}{2001!+2000!+1999!+\\cdots+1!}\\right\\rfloor.$$",
      "answer": "2000"
    },
    {
      "question": "In the diagram below, how many distinct paths are there from January 1 to December 31, moving from one adjacent dot to the next either to the right, down, or diagonally down to the right?",
      "answer": "372"
    },
    {
      "question": "Let $P$ and $A$ denote the perimeter and area respectively of a right triangle with relatively prime integer side-lengths. Find the largest possible integral value of $\\frac{P^{2}}{A}$.",
      "answer": "45"
    },
    {
      "question": "At a recent math contest, Evan was asked to find $2^{2016}(\\bmod p)$ for a given prime number $p$ with $100<p<500$. Evan has forgotten what the prime $p$ was, but still remembers how he solved it: - Evan first tried taking 2016 modulo $p-1$, but got a value $e$ larger than 100. - However, Evan noted that $e-\\frac{1}{2}(p-1)=21$, and then realized the answer was $-2^{21}(\\bmod p)$. What was the prime $p$?",
      "answer": "211"
    },
    {
      "question": "The average of $a, b$ and $c$ is 16. The average of $c, d$ and $e$ is 26. The average of $a, b, c, d$, and $e$ is 20. What is the value of $c$?",
      "answer": "26"
    },
    {
      "question": "Evaluate the infinite sum $\\sum_{n=1}^{\\infty} \\frac{n}{n^{4}+4}$.",
      "answer": "\\frac{3}{8}"
    },
    {
      "question": "How many integers between 100 and 300 are multiples of both 5 and 7, but are not multiples of 10?",
      "answer": "3"
    },
    {
      "question": "Estimate the number of positive integers $n \\leq 10^{6}$ such that $n^{2}+1$ has a prime factor greater than $n$. Submit a positive integer $E$. If the correct answer is $A$, you will receive $\\max \\left(0,\\left\\lfloor 20 \\cdot \\min \\left(\\frac{E}{A}, \\frac{10^{6}-E}{10^{6}-A}\\right)^{5}+0.5\\right\\rfloor\\right)$ points.",
      "answer": "757575"
    },
    {
      "question": "Two reals \\( x \\) and \\( y \\) are such that \\( x-y=4 \\) and \\( x^{3}-y^{3}=28 \\). Compute \\( x y \\).",
      "answer": "-3"
    },
    {
      "question": "An infinite sequence of real numbers $a_{1}, a_{2}, \\ldots$ satisfies the recurrence $$a_{n+3}=a_{n+2}-2 a_{n+1}+a_{n}$$ for every positive integer $n$. Given that $a_{1}=a_{3}=1$ and $a_{98}=a_{99}$, compute $a_{1}+a_{2}+\\cdots+a_{100}$.",
      "answer": "3"
    },
    {
      "question": "If $x=2$, what is the value of $4x^2 - 3x^2$?",
      "answer": "4"
    },
    {
      "question": "Let $a, b, c, n$ be positive real numbers such that $\\frac{a+b}{a}=3, \\frac{b+c}{b}=4$, and $\\frac{c+a}{c}=n$. Find $n$.",
      "answer": "\\frac{7}{6}"
    },
    {
      "question": "Bob Barker went back to school for a PhD in math, and decided to raise the intellectual level of The Price is Right by having contestants guess how many objects exist of a certain type, without going over. The number of points you will get is the percentage of the correct answer, divided by 10, with no points for going over (i.e. a maximum of 10 points). Let's see the first object for our contestants...a table of shape (5,4,3,2,1) is an arrangement of the integers 1 through 15 with five numbers in the top row, four in the next, three in the next, two in the next, and one in the last, such that each row and each column is increasing (from left to right, and top to bottom, respectively). For instance: \\begin{tabular}{lcccc} 1 & 2 & 3 & 4 & 5 \\\\ 6 & 7 & 8 & 9 & \\\\ 10 & 11 & 12 & & \\\\ 13 & 14 & & & \\\\ 15 & & & & \\end{tabular} is one table. How many tables are there?",
      "answer": "292864"
    },
    {
      "question": "For each positive integer $n$, there is a circle around the origin with radius $n$. Rainbow Dash starts off somewhere on the plane, but not on a circle. She takes off in some direction in a straight path. She moves \\frac{\\sqrt{5}}{5}$ units before crossing a circle, then \\sqrt{5}$ units, then \\frac{3 \\sqrt{5}}{5}$ units. What distance will she travel before she crosses another circle?",
      "answer": "\\frac{2 \\sqrt{170}-9 \\sqrt{5}}{5}"
    },
    {
      "question": "Let the circles $k_1$ and $k_2$ intersect at two points $A$ and $B$ , and let $t$ be a common tangent of $k_1$ and $k_2$ that touches $k_1$ and $k_2$ at $M$ and $N$ respectively. If $t\\perp AM$ and $MN=2AM$ , evaluate the angle $NMB$ .",
      "answer": "\\[\n\\boxed{\\frac{\\pi}{4}}\n\\]"
    },
    {
      "question": "Daniel and Scott are playing a game where a player wins as soon as he has two points more than his opponent. Both players start at par, and points are earned one at a time. If Daniel has a $60 \\%$ chance of winning each point, what is the probability that he will win the game?",
      "answer": "9 / 13"
    },
    {
      "question": "A regular $n$-gon $P_{1} P_{2} \\ldots P_{n}$ satisfies $\\angle P_{1} P_{7} P_{8}=178^{\\circ}$. Compute $n$.",
      "answer": "630"
    },
    {
      "question": "Trodgor the dragon is burning down a village consisting of 90 cottages. At time $t=0$ an angry peasant arises from each cottage, and every 8 minutes (480 seconds) thereafter another angry peasant spontaneously generates from each non-burned cottage. It takes Trodgor 5 seconds to either burn a peasant or to burn a cottage, but Trodgor cannot begin burning cottages until all the peasants around him have been burned. How many seconds does it take Trodgor to burn down the entire village?",
      "answer": "1920"
    },
    {
      "question": "Find the number of solutions to the equation $x+y+z=525$ where $x$ is a multiple of 7, $y$ is a multiple of 5, and $z$ is a multiple of 3.",
      "answer": "21"
    },
    {
      "question": "Let $\\mathbb{R}$ be the set of real numbers. Let $f: \\mathbb{R} \\rightarrow \\mathbb{R}$ be a function such that for all real numbers $x$ and $y$, we have $$f\\left(x^{2}\\right)+f\\left(y^{2}\\right)=f(x+y)^{2}-2 x y$$ Let $S=\\sum_{n=-2019}^{2019} f(n)$. Determine the number of possible values of $S$.",
      "answer": "2039191"
    },
    {
      "question": "In the addition problem shown, $m, n, p$, and $q$ represent positive digits. What is the value of $m+n+p+q$?",
      "answer": "24"
    },
    {
      "question": "The International Mathematical Olympiad is being organized in Japan, where a folklore belief is that the number $4$ brings bad luck. The opening ceremony takes place at the Grand Theatre where each row has the capacity of $55$ seats. What is the maximum number of contestants that can be seated in a single row with the restriction that no two of them are $4$ seats apart (so that bad luck during the competition is avoided)?",
      "answer": "30"
    },
    {
      "question": "How many non-empty subsets of $\\{1,2,3,4,5,6,7,8\\}$ have exactly $k$ elements and do not contain the element $k$ for some $k=1,2, \\ldots, 8$.",
      "answer": "127"
    },
    {
      "question": "Zlatan has 2017 socks of various colours. He wants to proudly display one sock of each of the colours, and he counts that there are $N$ ways to select socks from his collection for display. Given this information, what is the maximum value of $N$?",
      "answer": "3^{671} \\cdot 4"
    },
    {
      "question": "Equilateral triangle $ABC$ has circumcircle $\\Omega$. Points $D$ and $E$ are chosen on minor arcs $AB$ and $AC$ of $\\Omega$ respectively such that $BC=DE$. Given that triangle $ABE$ has area 3 and triangle $ACD$ has area 4, find the area of triangle $ABC$.",
      "answer": "\\frac{37}{7}"
    },
    {
      "question": "Will stands at a point \\(P\\) on the edge of a circular room with perfectly reflective walls. He shines two laser pointers into the room, forming angles of \\(n^{\\circ}\\) and \\((n+1)^{\\circ}\\) with the tangent at \\(P\\), where \\(n\\) is a positive integer less than 90. The lasers reflect off of the walls, illuminating the points they hit on the walls, until they reach \\(P\\) again. (\\(P\\) is also illuminated at the end.) What is the minimum possible number of illuminated points on the walls of the room?",
      "answer": "28"
    },
    {
      "question": "In $\\triangle A B C, A B=2019, B C=2020$, and $C A=2021$. Yannick draws three regular $n$-gons in the plane of $\\triangle A B C$ so that each $n$-gon shares a side with a distinct side of $\\triangle A B C$ and no two of the $n$-gons overlap. What is the maximum possible value of $n$?",
      "answer": "11"
    },
    {
      "question": "Let $X Y Z$ be an equilateral triangle, and let $K, L, M$ be points on sides $X Y, Y Z, Z X$, respectively, such that $X K / K Y=B, Y L / L Z=1 / C$, and $Z M / M X=1$. Determine the ratio of the area of triangle $K L M$ to the area of triangle $X Y Z$.",
      "answer": "$\\frac{1}{5}$"
    },
    {
      "question": "Evaluate the sum $$\\frac{1}{2\\lfloor\\sqrt{1}\\rfloor+1}+\\frac{1}{2\\lfloor\\sqrt{2}\\rfloor+1}+\\frac{1}{2\\lfloor\\sqrt{3}\\rfloor+1}+\\cdots+\\frac{1}{2\\lfloor\\sqrt{100}\\rfloor+1}$$",
      "answer": "190/21"
    },
    {
      "question": "In a chess-playing club, some of the players take lessons from other players. It is possible (but not necessary) for two players both to take lessons from each other. It so happens that for any three distinct members of the club, $A, B$, and $C$, exactly one of the following three statements is true: $A$ takes lessons from $B ; B$ takes lessons from $C ; C$ takes lessons from $A$. What is the largest number of players there can be?",
      "answer": "4"
    },
    {
      "question": "Find the number of ordered 2012-tuples of integers $\\left(x_{1}, x_{2}, \\ldots, x_{2012}\\right)$, with each integer between 0 and 2011 inclusive, such that the sum $x_{1}+2 x_{2}+3 x_{3}+\\cdots+2012 x_{2012}$ is divisible by 2012.",
      "answer": "2012^{2011}"
    },
    {
      "question": "A deck of 100 cards is labeled $1,2, \\ldots, 100$ from top to bottom. The top two cards are drawn; one of them is discarded at random, and the other is inserted back at the bottom of the deck. This process is repeated until only one card remains in the deck. Compute the expected value of the label of the remaining card.",
      "answer": "\\frac{467}{8}"
    },
    {
      "question": "Find the sum of the digits of \\(11 \\cdot 101 \\cdot 111 \\cdot 110011\\).",
      "answer": "48"
    },
    {
      "question": "Let $P(x)=x^{3}+a x^{2}+b x+2015$ be a polynomial all of whose roots are integers. Given that $P(x) \\geq 0$ for all $x \\geq 0$, find the sum of all possible values of $P(-1)$.",
      "answer": "9496"
    },
    {
      "question": "Lil Wayne, the rain god, determines the weather. If Lil Wayne makes it rain on any given day, the probability that he makes it rain the next day is $75 \\%$. If Lil Wayne doesn't make it rain on one day, the probability that he makes it rain the next day is $25 \\%$. He decides not to make it rain today. Find the smallest positive integer $n$ such that the probability that Lil Wayne makes it rain $n$ days from today is greater than $49.9 \\%$.",
      "answer": "9"
    },
    {
      "question": "Given that the area of a rectangle is 192 and its length is 24, what is the perimeter of the rectangle?",
      "answer": "64"
    },
    {
      "question": "Determine the value of $$1 \\cdot 2-2 \\cdot 3+3 \\cdot 4-4 \\cdot 5+\\cdots+2001 \\cdot 2002$$",
      "answer": "2004002"
    },
    {
      "question": "If $(2)(3)(4) = 6x$, what is the value of $x$?",
      "answer": "4"
    },
    {
      "question": "The Evil League of Evil is plotting to poison the city's water supply. They plan to set out from their headquarters at $(5,1)$ and put poison in two pipes, one along the line $y=x$ and one along the line $x=7$. However, they need to get the job done quickly before Captain Hammer catches them. What's the shortest distance they can travel to visit both pipes and then return to their headquarters?",
      "answer": "4 \\sqrt{5}"
    },
    {
      "question": "What is the sum of all numbers $q$ which can be written in the form $q=\\frac{a}{b}$ where $a$ and $b$ are positive integers with $b \\leq 10$ and for which there are exactly 19 integers $n$ that satisfy $\\sqrt{q}<n<q$?",
      "answer": "777.5"
    },
    {
      "question": "In how many ways can you fill a $3 \\times 3$ table with the numbers 1 through 9 (each used once) such that all pairs of adjacent numbers (sharing one side) are relatively prime?",
      "answer": "2016"
    },
    {
      "question": "If $x+2 y-3 z=7$ and $2 x-y+2 z=6$, determine $8 x+y$.",
      "answer": "32"
    },
    {
      "question": "Massachusetts Avenue is ten blocks long. One boy and one girl live on each block. They want to form friendships such that each boy is friends with exactly one girl and vice versa. Nobody wants a friend living more than one block away (but they may be on the same block). How many pairings are possible?",
      "answer": "89"
    },
    {
      "question": "Jack, Jill, and John play a game in which each randomly picks and then replaces a card from a standard 52 card deck, until a spades card is drawn. What is the probability that Jill draws the spade? (Jack, Jill, and John draw in that order, and the game repeats if no spade is drawn.)",
      "answer": "\\frac{12}{37}"
    },
    {
      "question": "Two standard six-sided dice are rolled. What is the probability that the product of the two numbers rolled is 12?",
      "answer": "\\frac{4}{36}"
    },
    {
      "question": "Consider the two hands of an analog clock, each of which moves with constant angular velocity. Certain positions of these hands are possible (e.g. the hour hand halfway between the 5 and 6 and the minute hand exactly at the 6), while others are impossible (e.g. the hour hand exactly at the 5 and the minute hand exactly at the 6). How many different positions are there that would remain possible if the hour and minute hands were switched?",
      "answer": "143"
    },
    {
      "question": "A rectangular piece of paper $P Q R S$ has $P Q=20$ and $Q R=15$. The piece of paper is glued flat on the surface of a large cube so that $Q$ and $S$ are at vertices of the cube. What is the shortest distance from $P$ to $R$, as measured through the cube?",
      "answer": "18.4"
    },
    {
      "question": "How many ways can you mark 8 squares of an $8 \\times 8$ chessboard so that no two marked squares are in the same row or column, and none of the four corner squares is marked? (Rotations and reflections are considered different.)",
      "answer": "21600"
    },
    {
      "question": "Suppose $a, b$, and $c$ are real numbers such that $$\\begin{aligned} a^{2}-b c & =14 \\\\ b^{2}-c a & =14, \\text { and } \\\\ c^{2}-a b & =-3 \\end{aligned}$$ Compute $|a+b+c|$.",
      "answer": "\\frac{17}{5}"
    },
    {
      "question": "Danielle picks a positive integer $1 \\leq n \\leq 2016$ uniformly at random. What is the probability that \\operatorname{gcd}(n, 2015)=1?",
      "answer": "\\frac{1441}{2016}"
    },
    {
      "question": "Find $AB + AC$ in triangle $ABC$ given that $D$ is the midpoint of $BC$, $E$ is the midpoint of $DC$, and $BD = DE = EA = AD$.",
      "answer": "1+\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "There are two red, two black, two white, and a positive but unknown number of blue socks in a drawer. It is empirically determined that if two socks are taken from the drawer without replacement, the probability they are of the same color is $\\frac{1}{5}$. How many blue socks are there in the drawer?",
      "answer": "4"
    },
    {
      "question": "What is $x-y$ if a town has 2017 houses, 1820 have a dog, 1651 have a cat, 1182 have a turtle, $x$ is the largest possible number of houses that have a dog, a cat, and a turtle, and $y$ is the smallest possible number of houses that have a dog, a cat, and a turtle?",
      "answer": "563"
    },
    {
      "question": "In how many distinct ways can you color each of the vertices of a tetrahedron either red, blue, or green such that no face has all three vertices the same color? (Two colorings are considered the same if one coloring can be rotated in three dimensions to obtain the other.)",
      "answer": "6"
    },
    {
      "question": "In how many different ways can André form exactly \\( \\$10 \\) using \\( \\$1 \\) coins, \\( \\$2 \\) coins, and \\( \\$5 \\) bills?",
      "answer": "10"
    },
    {
      "question": "Azmi has four blocks, each in the shape of a rectangular prism and each with dimensions $2 \times 3 \times 6$. She carefully stacks these four blocks on a flat table to form a tower that is four blocks high. What is the number of possible heights for this tower?",
      "answer": "14"
    },
    {
      "question": "Given $\\frac{e}{f}=\\frac{3}{4}$ and $\\sqrt{e^{2}+f^{2}}=15$, find $ef$.",
      "answer": "108"
    },
    {
      "question": "Let $M$ denote the number of positive integers which divide 2014!, and let $N$ be the integer closest to $\\ln (M)$. Estimate the value of $N$. If your answer is a positive integer $A$, your score on this problem will be the larger of 0 and $\\left\\lfloor 20-\\frac{1}{8}|A-N|\\right\\rfloor$. Otherwise, your score will be zero.",
      "answer": "439"
    },
    {
      "question": "If $m, n$ and $p$ are positive integers with $m+\\frac{1}{n+\\frac{1}{p}}=\\frac{17}{3}$, what is the value of $n$?",
      "answer": "1"
    },
    {
      "question": "Find all positive integers $n$ for which all positive divisors of $n$ can be put into the cells of a rectangular table under the following constraints:\n[list]\n[*]each cell contains a distinct divisor;\n[*]the sums of all rows are equal; and\n[*]the sums of all columns are equal.\n[/list]",
      "answer": "1"
    },
    {
      "question": "Find the number of ordered triples of divisors $(d_{1}, d_{2}, d_{3})$ of 360 such that $d_{1} d_{2} d_{3}$ is also a divisor of 360.",
      "answer": "800"
    },
    {
      "question": "Compute $\\frac{x}{w}$ if $w \\neq 0$ and $\\frac{x+6 y-3 z}{-3 x+4 w}=\\frac{-2 y+z}{x-w}=\\frac{2}{3}$.",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "A positive integer $n$ is picante if $n$ ! ends in the same number of zeroes whether written in base 7 or in base 8 . How many of the numbers $1,2, \\ldots, 2004$ are picante?",
      "answer": "4"
    },
    {
      "question": "Find the number of ordered triples of positive integers $(a, b, c)$ such that $6a+10b+15c=3000$.",
      "answer": "4851"
    },
    {
      "question": "Let $A B C D$ be an isosceles trapezoid with $A D=B C=255$ and $A B=128$. Let $M$ be the midpoint of $C D$ and let $N$ be the foot of the perpendicular from $A$ to $C D$. If $\\angle M B C=90^{\\circ}$, compute $\\tan \\angle N B M$.",
      "answer": "\\frac{120}{353}"
    },
    {
      "question": "In a rectangle $P Q R S$ with $P Q=5$ and $Q R=3$, $P R$ is divided into three segments of equal length by points $T$ and $U$. What is the area of quadrilateral $S T Q U$?",
      "answer": "5"
    },
    {
      "question": "Let $A$ denote the set of all integers $n$ such that $1 \\leq n \\leq 10000$, and moreover the sum of the decimal digits of $n$ is 2. Find the sum of the squares of the elements of $A$.",
      "answer": "7294927"
    },
    {
      "question": "In the country of Francisca, there are 2010 cities, some of which are connected by roads. Between any two cities, there is a unique path which runs along the roads and which does not pass through any city twice. What is the maximum possible number of cities in Francisca which have at least 3 roads running out of them?",
      "answer": "1004"
    },
    {
      "question": "One million bucks (i.e. one million male deer) are in different cells of a $1000 \\times 1000$ grid. The left and right edges of the grid are then glued together, and the top and bottom edges of the grid are glued together, so that the grid forms a doughnut-shaped torus. Furthermore, some of the bucks are honest bucks, who always tell the truth, and the remaining bucks are dishonest bucks, who never tell the truth. Each of the million bucks claims that \"at most one of my neighboring bucks is an honest buck.\" A pair of neighboring bucks is said to be buckaroo if exactly one of them is an honest buck. What is the minimum possible number of buckaroo pairs in the grid?",
      "answer": "1200000"
    },
    {
      "question": "For how many pairs of nonzero integers $(c, d)$ with $-2015 \\leq c, d \\leq 2015$ do the equations $c x=d$ and $d x=c$ both have an integer solution?",
      "answer": "8060"
    },
    {
      "question": "Find the largest integer less than 2012 all of whose divisors have at most two 1's in their binary representations.",
      "answer": "1536"
    },
    {
      "question": "Let $P$ and $Q$ be points on line $l$ with $P Q=12$. Two circles, $\\omega$ and $\\Omega$, are both tangent to $l$ at $P$ and are externally tangent to each other. A line through $Q$ intersects $\\omega$ at $A$ and $B$, with $A$ closer to $Q$ than $B$, such that $A B=10$. Similarly, another line through $Q$ intersects $\\Omega$ at $C$ and $D$, with $C$ closer to $Q$ than $D$, such that $C D=7$. Find the ratio $A D / B C$.",
      "answer": "\\frac{8}{9}"
    },
    {
      "question": "Roger initially has 20 socks in a drawer, each of which is either white or black. He chooses a sock uniformly at random from the drawer and throws it away. He repeats this action until there are equal numbers of white and black socks remaining. Suppose that the probability he stops before all socks are gone is $p$. If the sum of all distinct possible values of $p$ over all initial combinations of socks is $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$",
      "answer": "20738"
    },
    {
      "question": "A positive integer $n$ is magical if $\\lfloor\\sqrt{\\lceil\\sqrt{n}\\rceil}\\rfloor=\\lceil\\sqrt{\\lfloor\\sqrt{n}\\rfloor}\\rceil$ where $\\lfloor\\cdot\\rfloor$ and $\\lceil\\cdot\\rceil$ represent the floor and ceiling function respectively. Find the number of magical integers between 1 and 10,000, inclusive.",
      "answer": "1330"
    },
    {
      "question": "There are $10$ birds on the ground. For any $5$ of them, there are at least $4$ birds on a circle. Determine the least possible number of birds on the circle with the most birds.",
      "answer": "9"
    },
    {
      "question": "What is the tens digit of the smallest six-digit positive integer that is divisible by each of $10,11,12,13,14$, and 15?",
      "answer": "2"
    },
    {
      "question": "Eugene swam on Sunday, Monday, and Tuesday. On Monday, he swam for 30 minutes. On Tuesday, he swam for 45 minutes. His average swim time over the three days was 34 minutes. For how many minutes did he swim on Sunday?",
      "answer": "27"
    },
    {
      "question": "A numerical value is assigned to each letter of the alphabet. The value of a word is determined by adding up the numerical values of each of its letters. The value of SET is 2, the value of HAT is 7, the value of TASTE is 3, and the value of MAT is 4. What is the value of the word MATH?",
      "answer": "10"
    },
    {
      "question": "Jarris is a weighted tetrahedral die with faces $F_{1}, F_{2}, F_{3}, F_{4}$. He tosses himself onto a table, so that the probability he lands on a given face is proportional to the area of that face. Let $k$ be the maximum distance any part of Jarris is from the table after he rolls himself. Given that Jarris has an inscribed sphere of radius 3 and circumscribed sphere of radius 10, find the minimum possible value of the expected value of $k$.",
      "answer": "12"
    },
    {
      "question": "What is the number of positive integers $p$ for which $-1<\\sqrt{p}-\\sqrt{100}<1$?",
      "answer": "39"
    },
    {
      "question": "A point is equidistant from the coordinate axes if the vertical distance from the point to the $x$-axis is equal to the horizontal distance from the point to the $y$-axis. The point of intersection of the vertical line $x = a$ with the line with equation $3x + 8y = 24$ is equidistant from the coordinate axes. What is the sum of all possible values of $a$?",
      "answer": "-\\frac{144}{55}"
    },
    {
      "question": "Find the sum of all real numbers $x$ such that $5 x^{4}-10 x^{3}+10 x^{2}-5 x-11=0$.",
      "answer": "1"
    },
    {
      "question": "Let a positive integer \\(n\\) be called a cubic square if there exist positive integers \\(a, b\\) with \\(n=\\operatorname{gcd}\\left(a^{2}, b^{3}\\right)\\). Count the number of cubic squares between 1 and 100 inclusive.",
      "answer": "13"
    },
    {
      "question": "The very hungry caterpillar lives on the number line. For each non-zero integer $i$, a fruit sits on the point with coordinate $i$. The caterpillar moves back and forth; whenever he reaches a point with food, he eats the food, increasing his weight by one pound, and turns around. The caterpillar moves at a speed of $2^{-w}$ units per day, where $w$ is his weight. If the caterpillar starts off at the origin, weighing zero pounds, and initially moves in the positive $x$ direction, after how many days will he weigh 10 pounds?",
      "answer": "9217"
    },
    {
      "question": "Let $Z$ be as in problem 15. Let $X$ be the greatest integer such that $|X Z| \\leq 5$. Find $X$.",
      "answer": "2"
    },
    {
      "question": "The cells of a $5 \\times 5$ grid are each colored red, white, or blue. Sam starts at the bottom-left cell of the grid and walks to the top-right cell by taking steps one cell either up or to the right. Thus, he passes through 9 cells on his path, including the start and end cells. Compute the number of colorings for which Sam is guaranteed to pass through a total of exactly 3 red cells, exactly 3 white cells, and exactly 3 blue cells no matter which route he takes.",
      "answer": "1680"
    },
    {
      "question": "Compute the number of dates in the year 2023 such that when put in MM/DD/YY form, the three numbers are in strictly increasing order. For example, $06 / 18 / 23$ is such a date since $6<18<23$, while today, $11 / 11 / 23$, is not.",
      "answer": "186"
    },
    {
      "question": "Altitudes $B E$ and $C F$ of acute triangle $A B C$ intersect at $H$. Suppose that the altitudes of triangle $E H F$ concur on line $B C$. If $A B=3$ and $A C=4$, then $B C^{2}=\\frac{a}{b}$, where $a$ and $b$ are relatively prime positive integers. Compute $100 a+b$.",
      "answer": "33725"
    },
    {
      "question": "What is the probability that the arrow stops on a shaded region if a circular spinner is divided into six regions, four regions each have a central angle of $x^{\\circ}$, and the remaining regions have central angles of $20^{\\circ}$ and $140^{\\circ}$?",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "Compute the number of positive integers $n \\leq 1000$ such that \\operatorname{lcm}(n, 9)$ is a perfect square.",
      "answer": "43"
    },
    {
      "question": "In the Cartesian plane, a perfectly reflective semicircular room is bounded by the upper half of the unit circle centered at $(0,0)$ and the line segment from $(-1,0)$ to $(1,0)$. David stands at the point $(-1,0)$ and shines a flashlight into the room at an angle of $46^{\\circ}$ above the horizontal. How many times does the light beam reflect off the walls before coming back to David at $(-1,0)$ for the first time?",
      "answer": "65"
    },
    {
      "question": "Let $f(x)=x^{3}-3x$. Compute the number of positive divisors of $$\\left\\lfloor f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(f\\left(\\frac{5}{2}\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\right)\\rfloor$$ where $f$ is applied 8 times.",
      "answer": "6562"
    },
    {
      "question": "There is a heads up coin on every integer of the number line. Lucky is initially standing on the zero point of the number line facing in the positive direction. Lucky performs the following procedure: he looks at the coin (or lack thereof) underneath him, and then, - If the coin is heads up, Lucky flips it to tails up, turns around, and steps forward a distance of one unit. - If the coin is tails up, Lucky picks up the coin and steps forward a distance of one unit facing the same direction. - If there is no coin, Lucky places a coin heads up underneath him and steps forward a distance of one unit facing the same direction. He repeats this procedure until there are 20 coins anywhere that are tails up. How many times has Lucky performed the procedure when the process stops?",
      "answer": "6098"
    },
    {
      "question": "Find the smallest constant $C$ such that for every real polynomial $P(x)$ of degree 3 that has a root in the interval $[0,1]$, \\[ \\int_0^1 \\left| P(x) \\right|\\,dx \\leq C \\max_{x \\in [0,1]} \\left| P(x) \\right|. \\]",
      "answer": "\\frac{5}{6}"
    },
    {
      "question": "A sequence consists of 2010 terms. Each term after the first is 1 larger than the previous term. The sum of the 2010 terms is 5307. What is the sum when every second term is added up, starting with the first term and ending with the second last term?",
      "answer": "2151"
    },
    {
      "question": "Find the 6-digit number beginning and ending in the digit 2 that is the product of three consecutive even integers.",
      "answer": "287232"
    },
    {
      "question": "For a positive integer $n$, let $p(n)$ denote the product of the positive integer factors of $n$. Determine the number of factors $n$ of 2310 for which $p(n)$ is a perfect square.",
      "answer": "27"
    },
    {
      "question": "Let $S$ be a subset with four elements chosen from \\{1,2, \\ldots, 10\\}$. Michael notes that there is a way to label the vertices of a square with elements from $S$ such that no two vertices have the same label, and the labels adjacent to any side of the square differ by at least 4 . How many possibilities are there for the subset $S$ ?",
      "answer": "36"
    },
    {
      "question": "For what single digit $n$ does 91 divide the 9-digit number $12345 n 789$?",
      "answer": "7"
    },
    {
      "question": "For each positive integer $1 \\leq m \\leq 10$, Krit chooses an integer $0 \\leq a_{m}<m$ uniformly at random. Let $p$ be the probability that there exists an integer $n$ for which $n \\equiv a_{m}(\\bmod m)$ for all $m$. If $p$ can be written as $\\frac{a}{b}$ for relatively prime positive integers $a$ and $b$, compute $100 a+b$.",
      "answer": "1540"
    },
    {
      "question": "For how many positive integers $a$ does the polynomial $x^{2}-a x+a$ have an integer root?",
      "answer": "1"
    },
    {
      "question": "Reimu has a wooden cube. In each step, she creates a new polyhedron from the previous one by cutting off a pyramid from each vertex of the polyhedron along a plane through the trisection point on each adjacent edge that is closer to the vertex. For example, the polyhedron after the first step has six octagonal faces and eight equilateral triangular faces. How many faces are on the polyhedron after the fifth step?",
      "answer": "974"
    },
    {
      "question": "Let $P$ be the number to partition 2013 into an ordered tuple of prime numbers? What is $\\log _{2}(P)$? If your answer is $A$ and the correct answer is $C$, then your score on this problem will be $\\left\\lfloor\\frac{125}{2}\\left(\\min \\left(\\frac{C}{A}, \\frac{A}{C}\\right)-\\frac{3}{5}\\right)\\right\\rfloor$ or zero, whichever is larger.",
      "answer": "614.519..."
    },
    {
      "question": "Let $A B C$ be a triangle and $D, E$, and $F$ be the midpoints of sides $B C, C A$, and $A B$ respectively. What is the maximum number of circles which pass through at least 3 of these 6 points?",
      "answer": "17"
    },
    {
      "question": "Let $A$ be as in problem 33. Let $W$ be the sum of all positive integers that divide $A$. Find $W$.",
      "answer": "8"
    },
    {
      "question": "In quadrilateral $ABCD$, there exists a point $E$ on segment $AD$ such that $\\frac{AE}{ED}=\\frac{1}{9}$ and $\\angle BEC$ is a right angle. Additionally, the area of triangle $CED$ is 27 times more than the area of triangle $AEB$. If $\\angle EBC=\\angle EAB, \\angle ECB=\\angle EDC$, and $BC=6$, compute the value of $AD^{2}$.",
      "answer": "320"
    },
    {
      "question": "Compute the side length of the largest cube contained in the region $\\{(x, y, z): x^{2}+y^{2}+z^{2} \\leq 25 \\text{ and } x \\geq 0\\}$ of three-dimensional space.",
      "answer": "\\frac{5 \\sqrt{6}}{3}"
    },
    {
      "question": "Ten points are equally spaced on a circle. A graph is a set of segments (possibly empty) drawn between pairs of points, so that every two points are joined by either zero or one segments. Two graphs are considered the same if we can obtain one from the other by rearranging the points. Let $N$ denote the number of graphs with the property that for any two points, there exists a path from one to the other among the segments of the graph. Estimate the value of $N$. If your answer is a positive integer $A$, your score on this problem will be the larger of 0 and $\\lfloor 20-5|\\ln (A / N)|\\rfloor$. Otherwise, your score will be zero.",
      "answer": "11716571"
    },
    {
      "question": "Mary has a sequence $m_{2}, m_{3}, m_{4}, \\ldots$, such that for each $b \\geq 2, m_{b}$ is the least positive integer $m$ for which none of the base-$b$ logarithms $\\log _{b}(m), \\log _{b}(m+1), \\ldots, \\log _{b}(m+2017)$ are integers. Find the largest number in her sequence.",
      "answer": "2188"
    },
    {
      "question": "Let $ \\left(a_{n}\\right)$ be the sequence of reals defined by $ a_{1}=\\frac{1}{4}$ and the recurrence $ a_{n}= \\frac{1}{4}(1+a_{n-1})^{2}, n\\geq 2$. Find the minimum real $ \\lambda$ such that for any non-negative reals $ x_{1},x_{2},\\dots,x_{2002}$, it holds\r\n\\[ \\sum_{k=1}^{2002}A_{k}\\leq \\lambda a_{2002}, \\]\r\nwhere $  A_{k}= \\frac{x_{k}-k}{(x_{k}+\\cdots+x_{2002}+\\frac{k(k-1)}{2}+1)^{2}}, k\\geq 1$.",
      "answer": "\\frac{1}{2005004}"
    },
    {
      "question": "Abigail chooses an integer at random from the set $\\{2,4,6,8,10\\}$. Bill chooses an integer at random from the set $\\{2,4,6,8,10\\}$. Charlie chooses an integer at random from the set $\\{2,4,6,8,10\\}$. What is the probability that the product of their three integers is not a power of 2?",
      "answer": "\\frac{98}{125}"
    },
    {
      "question": "For how many values of $n$ with $3 \\leq n \\leq 12$ can a Fano table be created?",
      "answer": "3"
    },
    {
      "question": "Distinct prime numbers $p, q, r$ satisfy the equation $2 p q r+50 p q=7 p q r+55 p r=8 p q r+12 q r=A$ for some positive integer $A$. What is $A$ ?",
      "answer": "1980"
    },
    {
      "question": "Let $a$ and $b$ be positive real numbers such that $3a^2 + 2b^2 = 3a + 2b$. Find the minimum value of $A =\\sqrt{\\frac{a}{b(3a+2)}} + \\sqrt{\\frac{b}{a(2b+3)}} $",
      "answer": "\\frac{2}{\\sqrt{5}}"
    },
    {
      "question": "Given that the 32-digit integer 64312311692944269609355712372657 is the product of 6 consecutive primes, compute the sum of these 6 primes.",
      "answer": "1200974"
    },
    {
      "question": "The Dyslexian alphabet consists of consonants and vowels. It so happens that a finite sequence of letters is a word in Dyslexian precisely if it alternates between consonants and vowels (it may begin with either). There are 4800 five-letter words in Dyslexian. How many letters are in the alphabet?",
      "answer": "12"
    },
    {
      "question": "How many functions $f:\\{1,2, \\ldots, 10\\} \\rightarrow\\{1,2, \\ldots, 10\\}$ satisfy the property that $f(i)+f(j)=11$ for all values of $i$ and $j$ such that $i+j=11$.",
      "answer": "100000"
    },
    {
      "question": "We consider positive integers $n$ having at least six positive divisors. Let the positive divisors of $n$ be arranged in a sequence $(d_i)_{1\\le i\\le k}$ with $$1=d_1<d_2<\\dots <d_k=n\\quad (k\\ge 6).$$\nFind all positive integers $n$ such that $$n=d_5^2+d_6^2.$$",
      "answer": "500"
    },
    {
      "question": "A lock code is made up of four digits that satisfy the following rules: - At least one digit is a 4, but neither the second digit nor the fourth digit is a 4. - Exactly one digit is a 2, but the first digit is not 2. - Exactly one digit is a 7. - The code includes a 1, or the code includes a 6, or the code includes two 4s. How many codes are possible?",
      "answer": "22"
    },
    {
      "question": "Our third and final item comes to us from Germany, I mean Geometry. It is known that a regular $n$-gon can be constructed with straightedge and compass if $n$ is a prime that is 1 plus a power of 2. It is also possible to construct a $2 n$-gon whenever an $n$-gon is constructible, or a $p_{1} p_{2} \\cdots p_{m}$-gon where the $p_{i}$ 's are distinct primes of the above form. What is really interesting is that these conditions, together with the fact that we can construct a square, is that they give us all constructible regular $n$-gons. What is the largest $n$ less than $4,300,000,000$ such that a regular $n$-gon is constructible?",
      "answer": "2^{32}"
    },
    {
      "question": "Cyclic pentagon $ABCDE$ has a right angle $\\angle ABC=90^{\\circ}$ and side lengths $AB=15$ and $BC=20$. Supposing that $AB=DE=EA$, find $CD$.",
      "answer": "7"
    },
    {
      "question": "Let $P(n)=\\left(n-1^{3}\\right)\\left(n-2^{3}\\right) \\ldots\\left(n-40^{3}\\right)$ for positive integers $n$. Suppose that $d$ is the largest positive integer that divides $P(n)$ for every integer $n>2023$. If $d$ is a product of $m$ (not necessarily distinct) prime numbers, compute $m$.",
      "answer": "48"
    },
    {
      "question": "Triangle $A B C$ has $A B=4, B C=5$, and $C A=6$. Points $A^{\\prime}, B^{\\prime}, C^{\\prime}$ are such that $B^{\\prime} C^{\\prime}$ is tangent to the circumcircle of $\\triangle A B C$ at $A, C^{\\prime} A^{\\prime}$ is tangent to the circumcircle at $B$, and $A^{\\prime} B^{\\prime}$ is tangent to the circumcircle at $C$. Find the length $B^{\\prime} C^{\\prime}$.",
      "answer": "\\frac{80}{3}"
    },
    {
      "question": "Mark and William are playing a game. Two walls are placed 1 meter apart, with Mark and William each starting an orb at one of the walls. Simultaneously, they release their orbs directly toward the other. Both orbs are enchanted such that, upon colliding with each other, they instantly reverse direction and go at double their previous speed. Furthermore, Mark has enchanted his orb so that when it collides with a wall it instantly reverses direction and goes at double its previous speed (William's reverses direction at the same speed). Initially, Mark's orb is moving at \\frac{1}{1000} meters/s, and William's orb is moving at 1 meter/s. Mark wins when his orb passes the halfway point between the two walls. How fast, in meters/s, is his orb going when this first happens?",
      "answer": "\\frac{2^{17}}{125}"
    },
    {
      "question": "Let P_{1}, P_{2}, \\ldots, P_{6} be points in the complex plane, which are also roots of the equation x^{6}+6 x^{3}-216=0. Given that P_{1} P_{2} P_{3} P_{4} P_{5} P_{6} is a convex hexagon, determine the area of this hexagon.",
      "answer": "9 \\sqrt{3}"
    },
    {
      "question": "Numbers $m$ and $n$ are on the number line. What is the value of $n-m$?",
      "answer": "55"
    },
    {
      "question": "Find all prime numbers $p$ such that there exists a unique $a \\in \\mathbb{Z}_p$ for which $a^3 - 3a + 1 = 0.$",
      "answer": "3"
    },
    {
      "question": "If $a$ and $b$ are positive integers, the operation $\nabla$ is defined by $a \nabla b=a^{b} \times b^{a}$. What is the value of $2 \nabla 3$?",
      "answer": "72"
    },
    {
      "question": "If $a$ , $b$ are integers and $s=a^3+b^3-60ab(a+b)\\geq 2012$ , find the least possible value of $s$.",
      "answer": "2015"
    },
    {
      "question": "A polynomial $f \\in \\mathbb{Z}[x]$ is called splitty if and only if for every prime $p$, there exist polynomials $g_{p}, h_{p} \\in \\mathbb{Z}[x]$ with $\\operatorname{deg} g_{p}, \\operatorname{deg} h_{p}<\\operatorname{deg} f$ and all coefficients of $f-g_{p} h_{p}$ are divisible by $p$. Compute the sum of all positive integers $n \\leq 100$ such that the polynomial $x^{4}+16 x^{2}+n$ is splitty.",
      "answer": "\\[ 693 \\]"
    },
    {
      "question": "How many of the integers \\(19, 21, 23, 25, 27\\) can be expressed as the sum of two prime numbers?",
      "answer": "3"
    },
    {
      "question": "If $a$ and $b$ are randomly selected real numbers between 0 and 1, find the probability that the nearest integer to $\\frac{a-b}{a+b}$ is odd.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "A regular dodecahedron is projected orthogonally onto a plane, and its image is an $n$-sided polygon. What is the smallest possible value of $n$ ?",
      "answer": "6"
    },
    {
      "question": "Find all primes $p$ such that $p^2-p+1$ is a perfect cube.",
      "answer": "19"
    },
    {
      "question": "Let $n$ be the product of the first 10 primes, and let $$S=\\sum_{x y \\mid n} \\varphi(x) \\cdot y$$ where $\\varphi(x)$ denotes the number of positive integers less than or equal to $x$ that are relatively prime to $x$, and the sum is taken over ordered pairs $(x, y)$ of positive integers for which $x y$ divides $n$. Compute $\\frac{S}{n}$.",
      "answer": "1024"
    },
    {
      "question": "Let rectangle $A B C D$ have lengths $A B=20$ and $B C=12$. Extend ray $B C$ to $Z$ such that $C Z=18$. Let $E$ be the point in the interior of $A B C D$ such that the perpendicular distance from $E$ to \\overline{A B}$ is 6 and the perpendicular distance from $E$ to \\overline{A D}$ is 6 . Let line $E Z$ intersect $A B$ at $X$ and $C D$ at $Y$. Find the area of quadrilateral $A X Y D$.",
      "answer": "72"
    },
    {
      "question": "If $4x + 14 = 8x - 48$, what is the value of $2x$?",
      "answer": "31"
    },
    {
      "question": "Two identical smaller cubes are stacked next to a larger cube. Each of the two smaller cubes has a volume of 8. What is the volume of the larger cube?",
      "answer": "64"
    },
    {
      "question": "A right triangle and a circle are drawn such that the circle is tangent to the legs of the right triangle. The circle cuts the hypotenuse into three segments of lengths 1,24 , and 3 , and the segment of length 24 is a chord of the circle. Compute the area of the triangle.",
      "answer": "192"
    },
    {
      "question": "In a classroom, 34 students are seated in 5 rows of 7 chairs. The place at the center of the room is unoccupied. A teacher decides to reassign the seats such that each student will occupy a chair adjacent to his/her present one (i.e. move one desk forward, back, left or right). In how many ways can this reassignment be made?",
      "answer": "0"
    },
    {
      "question": "Pascal has a triangle. In the $n$th row, there are $n+1$ numbers $a_{n, 0}, a_{n, 1}, a_{n, 2}, \\ldots, a_{n, n}$ where $a_{n, 0}=a_{n, n}=1$. For all $1 \\leq k \\leq n-1, a_{n, k}=a_{n-1, k}-a_{n-1, k-1}$. What is the sum of all numbers in the 2018th row?",
      "answer": "2"
    }
  ],
  "anchor_pool": [
    {
      "question": "14. Let the complex numbers $z_{1}, z_{2}$ satisfy\n$$\n\\begin{array}{l}\n\\left|z_{1}\\right|=\\left|z_{2}\\right|=1 \\text {, and } z_{1}+z_{2}-1=0 \\text {. } \\\\\n\\text { Then }\\left(z_{1}-z_{2}\\right)^{2}=\n\\end{array}\n$$",
      "answer": "-3"
    },
    {
      "question": "## 53. Math Puzzle 10/69\n\nAt the beginning of a group meeting, the Pioneers greet each other with handshakes. A total of 66 handshakes are exchanged.\n\nHow many friends attended the meeting?",
      "answer": "12"
    },
    {
      "question": "A1. At a bazaar, you can win a prize by guessing the exact number of ping pong balls in a glass jar. Arie guesses there are 90, Bea guesses there are 97, Cor guesses there are 99, and Dirk guesses there are 101. None of the four win the prize. It turns out that one of the four is off by 7, one is off by 4, and one is off by 3.\n\nHow many ping pong balls are in the jar?",
      "answer": "94"
    },
    {
      "question": "## Task A-1.6.\n\nGiven is a triangle $A B C$ with an area of 1. Point $D$ is the midpoint of segment $\\overline{B C}$, point $E$ is the midpoint of segment $\\overline{A D}$, point $F$ is the midpoint of segment $\\overline{B E}$, and point $G$ is the midpoint of segment $\\overline{C F}$. Determine the area of triangle $E F G$.",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "18. Find the remainder when $2^{55}+1$ is divided by 33 .",
      "answer": "0"
    },
    {
      "question": "4. (2004 National College Entrance Examination - National Paper) The maximum value of the function $y=\\sin x-\\frac{1}{2} \\cos x(x \\in \\mathbf{R})$ is",
      "answer": "\\frac{\\sqrt{5}}{2}"
    },
    {
      "question": "72. Find the product:\n\n$$\n\\left(1-\\frac{1}{4}\\right) \\cdot\\left(1-\\frac{1}{9}\\right) \\cdot\\left(1-\\frac{1}{16}\\right) \\ldots\\left(1-\\frac{1}{225}\\right)\n$$",
      "answer": "\\frac{8}{15}"
    },
    {
      "question": "5. For the distinct real numbers $a, b, c$, it holds that\n\n$$\n\\frac{a^{2}}{(b-c)^{2}}+\\frac{b^{2}}{(c-a)^{2}}+\\frac{c^{2}}{(a-b)^{2}}=2\n$$\n\nCalculate\n\n$$\n\\frac{a}{b-c}+\\frac{b}{c-a}+\\frac{c}{a-b}\n$$",
      "answer": "0"
    },
    {
      "question": "3. The residents of the Immortal Island celebrate their birthdays every 3 years. Xiaohuaxian had her first birthday at the age of 1, and her second birthday at the age of 4. She receives a number of gifts equal to her age each birthday. By the time she finished celebrating her birthday this year, Xiaohuaxian had received a total of 70 birthday gifts. How old is Xiaohuaxian this year? $\\qquad$ years old.",
      "answer": "19"
    },
    {
      "question": "8. Given that the three sides of $\\triangle A B C$ are $13,14,15$. There are 4 circles $O, O_{1}, O_{2}, O_{3}$ with the same radius $r$ placed inside $\\triangle A B C$, and circle $O_{1}$ is tangent to sides $A B$ and $A C$, circle $O_{2}$ is tangent to sides $B A$ and $B C$, circle $O_{3}$ is tangent to sides $C B$ and $C A$, and circle $O$ is tangent to circles $O_{1}, O_{2}, O_{3}$. Then $r=$ $\\qquad$ .",
      "answer": "\\frac{260}{129}"
    },
    {
      "question": "【Question 10】\nThe school is to distribute 90 storybooks to third-grade students. If distributed at a rate of 1 book per student, there will be leftovers; if the remaining books are then distributed at a rate of 1 book for every two students, they will be exactly distributed. Therefore, the number of third-grade students is $\\qquad$ people.",
      "answer": "60"
    },
    {
      "question": "2. For $x, y \\in[1,3]$ find the minimum value of the expression\n\n$$\nA=\\frac{\\left(3 x y+x^{2}\\right) \\sqrt{3 x y+x-3 y}+\\left(3 x y+y^{2}\\right) \\sqrt{3 x y+y-3 x}}{x^{2} y+y^{2} x}\n$$",
      "answer": "4"
    },
    {
      "question": "(7) Set $A=\\left\\{x \\left\\lvert\\, x=\\left[\\frac{5 k}{6}\\right]\\right., k \\in \\mathbf{Z}, 100 \\leqslant k \\leqslant 999\\right\\}$, where $[x]$ denotes the greatest integer not greater than $x$. Then the number of elements in set $A$ is $\\qquad$ .",
      "answer": "750"
    },
    {
      "question": "1. Find all prime numbers $p$ for which the number $7^{p-2}+9 p^{4}$ is a perfect square.",
      "answer": "7"
    },
    {
      "question": "3. In $\\triangle A B C$, the lengths of the sides opposite to $\\angle A$, $\\angle B$, and $\\angle C$ are $a$, $b$, and $c$ respectively. Point $G$ satisfies\n$$\n\\overrightarrow{G A}+\\overrightarrow{G B}+\\overrightarrow{G C}=\\mathbf{0}, \\overrightarrow{G A} \\cdot \\overrightarrow{G B}=0 \\text {. }\n$$\n\nIf $(\\tan A+\\tan B) \\tan C=m \\tan A \\cdot \\tan B$, then $m=$ . $\\qquad$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "In the picture, there is a structure composed of twelve identical cubes. To how many different places can we move the dark cube so that the surface area of the assembled body does not change?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_22cb57ce78fb8d5dd5c3g-5.jpg?height=334&width=394&top_left_y=1758&top_left_x=837)\n\nJust like in the original structure, the cubes in the new structure must touch each other with entire faces. The positions of the light cubes cannot be changed.\n\n(D. Reichmann)\n\nHint. How will the surface area of the body change after removing the dark cube?",
      "answer": "3"
    },
    {
      "question": "Let's divide a solid cube with an edge of $n \\mathrm{~cm}$ into smaller cubes with an edge of $1 \\mathrm{~cm}$ ( $n$ is a natural number). For which $n$ can a cube with an edge of $2 n \\mathrm{~cm}$ be assembled from the resulting small cubes, if we allow there to be a void inside the cube?",
      "answer": "23"
    },
    {
      "question": "6. Find the sum of the integers that belong to the set of values of the function $f(x)=\\log _{3}(40 \\cos 2 x+41)$ for $x \\in[(5 / 3)(\\operatorname{arctg}(1 / 5)) \\cos (\\pi-\\arcsin (-0.8)) ; \\operatorname{arctg} 3]$ (10 points)",
      "answer": "9"
    },
    {
      "question": "5. Option 1.\n\nPetya is the oldest child in the family. He has two sisters, Anya and Katya, and a brother Vasya. Petya calculated that Anya and Katya together are 19 years old, and Anya and Vasya together are 14 years old. Determine how old Katya is, if it is known that the two youngest children together are 7 years old.",
      "answer": "6"
    },
    {
      "question": "## Task 3 - 190833\n\nGiven a square $A B C D$ with side length $a$.\n\nA line parallel to $A B$ intersects the sides $B C$ and $A D$ at points $E$ and $F$, respectively. A line parallel to $B C$ intersects $A B$ and $E F$ at points $G$ and $H$, respectively, and a line parallel to $A B$ intersects the segments $B E$ and $G H$ at points $J$ and $K$, respectively.\n\na) Determine the perimeter of the rectangle $K J E H$ in terms of $a$ under the condition that the rectangles $A G H F, G B J K, K J E H$, and $F E C D$ are equal in area!\n\nb) Determine the area of the rectangle $K J E H$ in terms of $a$ under the condition that the rectangles $A G H F, G B J K, K J E H$, and $F E C D$ are equal in perimeter!",
      "answer": "\\frac{3}{8}"
    },
    {
      "question": "A straight prism has a rhombus as its base, with an acute angle of $60^{\\circ}$. What angle does a plane form with the base if the intersection of this plane with the prism's lateral surface is a square?",
      "answer": "54.7"
    },
    {
      "question": "4. In the tetrahedron $P-ABC$, it is known that $BC=3$, $CA=4$, $AB=5$. If the dihedral angles between the three lateral faces and the base are all $45^{\\circ}$, then the volume of the tetrahedron $P-ABC$ is $\\qquad$",
      "answer": "2"
    },
    {
      "question": "Example 5 Let the function $f(x)$ be defined for all $x>0$, and satisfy the following conditions:\n(1) For $x>0$, $f(x) f\\left[f(x)+\\frac{1}{x}\\right]=1$; (2) $f(x)$ is increasing on $(0,+\\infty)$. Find $f(1)$.",
      "answer": "\\frac{1-\\sqrt{5}}{2}"
    },
    {
      "question": "1. Given the quadratic function\n$$\nf(x)=x^{2}+b x+8(b \\neq 0)\n$$\n\nhas two distinct real roots $x_{1}, x_{2}$, and $x_{1}+\\frac{1}{x_{2}}, x_{2}+\\frac{1}{x_{1}}$ are the two zeros of a quadratic function $g(x)$ with a leading coefficient of 1. If $g(1)=f(1)$, find all possible values of $g(1)$.",
      "answer": "-8"
    },
    {
      "question": "## Task 4\n\nIt is known about a three-digit number: It is less than 200 and ends with the digit 5. The middle digit is the smallest natural number. What is the number?",
      "answer": "105"
    },
    {
      "question": "9. Find the smallest natural number that is greater than the sum of its digits by 1755 (the year of the founding of Moscow University).",
      "answer": "1770"
    },
    {
      "question": "3.18. Calculate\n\n$$\n\\int_{L}\\left(z^{2}+2 z \\bar{z}\\right) d z\n$$\n\nwhere $L-$ is the arc of the circle $|z|=1, \\arg z \\in[0, \\pi]$.",
      "answer": "-\\frac{14}{3}"
    },
    {
      "question": "18. A collection of 2011 circles divide the plane into $N$ regions in such a way that any pair of circles intersects at two points and no point lies on three circles. Find the last four digits of $N$.",
      "answer": "2112"
    },
    {
      "question": "## Task 2\n\nCalculate the difference between the product and the sum of the numbers 9 and 2.",
      "answer": "7"
    },
    {
      "question": "4. For each pair of numbers $x$, let us denote by $s(x, y)$ the smallest of the numbers $x, 1-y, y-x$. What is the largest value that the number $s(x, y)$ can take?",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "1. Find the maximum possible area of a quadrilateral in which the product of any two adjacent sides is 1.",
      "answer": "1"
    },
    {
      "question": "$3+$ [ Law of Sines\n\nIn an isosceles trapezoid, the bases are given as $a=21, b=9$ and the height $h=8$. Find the radius of the circumscribed circle.\n\n#",
      "answer": "\\frac{85}{8}"
    },
    {
      "question": "How many obtuse triangles are there in which the measures of the angles are three different integers?",
      "answer": "1936"
    },
    {
      "question": "7. Given a complex number $z$ satisfying $|z|=1$. Then the maximum value of $\\left|z^{3}-3 z-2\\right|$ is $\\qquad$ .\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "3\\sqrt{3}"
    },
    {
      "question": "3. (10 points) The simplest fraction $\\frac{\\mathrm{a}}{\\mathrm{b}}$ satisfies $\\frac{1}{5}<\\frac{\\mathrm{a}}{\\mathrm{b}}<\\frac{1}{4}$, and $b$ does not exceed 19, then the product of the maximum possible value and the minimum possible value of $a+b$ is",
      "answer": "253"
    },
    {
      "question": "11. In the Cartesian coordinate system $x O y$, the circle $\\odot M$ with center $M(0,1)$ intersects the parabola $y=x^{2}$ at four points $A$, $B$, $C$, and $D$ in sequence.\n(1) Find the range of the radius $r$ of $\\odot M$;\n(2) Find the maximum area of quadrilateral $A B C D$ (accurate to $10^{-4}$).",
      "answer": "1.0887"
    },
    {
      "question": "34. [20] Let $f(n)$ be the largest prime factor of $n$. Estimate\n$$\nN=\\left\\lfloor 10^{4} \\cdot \\frac{\\sum_{n=2}^{10^{6}} f\\left(n^{2}-1\\right)}{\\sum_{n=2}^{10^{6}} f(n)}\\right\\rfloor .\n$$\n\nAn estimate of $E$ will receive $\\max \\left(0,\\left\\lfloor 20-20\\left(\\frac{|E-N|}{10^{3}}\\right)^{1 / 3}\\right\\rfloor\\right)$ points.",
      "answer": "18215"
    },
    {
      "question": "1.1. (2 points) On two bookshelves, there are mathematics books, with an equal number on each. If 5 books are moved from the first shelf to the second, then the second shelf will have twice as many books as the first. How many books are there in total on both shelves?",
      "answer": "30"
    },
    {
      "question": "5. Let $f(x)=a_{0}+a_{1} x+a_{2} x^{2}+a_{4} x^{3}+a_{4} x^{4}+a_{5} x^{5}$ be a polynomial in $x$ where $a_{0}, a_{1}, a_{2}, a_{3}, a_{4}$ are constants and $a_{5}=7$. When divided by $x-2004, x-2005, x-2006, x-2007$ and $x-2008$, $\\mathrm{f}(x)$ leaves a remainder of $72,-30,32,-24$ and 24 respectively. Find the value of $f(2009)$.",
      "answer": "1742"
    },
    {
      "question": "Example 9. The probability of an event occurring in each of 900 independent trials is 0.5. Find the probability that the relative frequency of the event's occurrence deviates from its probability in absolute value by no more than 0.02.",
      "answer": "0.7698"
    },
    {
      "question": "## Task 2\n\nIn a district, there were 1685 apartments. In the last five years, 275 apartments were built there. How many apartments are there now in this district?",
      "answer": "1960"
    },
    {
      "question": "As shown in the figure, in quadrilateral $A B C D$, $\\angle B=90^{\\circ}, \\angle D A B=\\angle D=96^{\\circ}$, $\\angle B C D=78^{\\circ}$, and $D A=2 A B$, find $\\angle C A B$.",
      "answer": "66"
    },
    {
      "question": "14. (6 points) A group of PLA soldiers formed a three-layer hollow square formation with 9 soldiers left over. If another layer was added inside the hollow part, 7 soldiers would be short. The total number of soldiers in this group is $\\qquad$ people.",
      "answer": "105"
    },
    {
      "question": "7. Let $m$ and $n$ both be positive integers greater than 1. Given that $\\frac{(n+m)!}{n!}=17297280$. Then $\\frac{n}{m}=$ $\\qquad$",
      "answer": "\\frac{31}{2}"
    },
    {
      "question": "16. (5 points)\nThe Reuleaux triangle is a famous shape. By using the vertices of an equilateral triangle as the centers and its side length as the radius to draw three circular arcs, a Reuleaux triangle can be formed. In the figure below, the perimeter of the Reuleaux triangle is $314 \\mathrm{~cm}$, then the side length of the central equilateral triangle is $\\qquad$ $\\mathrm{cm}$. ( $\\pi$ is taken as 3.14)",
      "answer": "100"
    },
    {
      "question": "Problem 9.5. On the base $AC$ of isosceles triangle $ABC (AB = BC)$, a point $M$ is marked. It is known that $AM = 7, MB = 3, \\angle BMC = 60^\\circ$. Find the length of segment $AC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_ddcc40423d5e6a42b0ccg-33.jpg?height=240&width=711&top_left_y=86&top_left_x=369)",
      "answer": "17"
    },
    {
      "question": "I1.1 The given figure shows a circle of radius $18 \\mathrm{~cm}$, centre $O$.\nIf $\\angle A O B=\\frac{\\pi}{3}$ and the length of arc $A P B$ is $a \\pi \\mathrm{cm}$, find $a$.",
      "answer": "6"
    },
    {
      "question": "12.55*. In a right triangle $ABC$ with a right angle at $A$, a circle is constructed on the altitude $AD$ as a diameter, intersecting side $AB$ at point $K$ and side $AC$ at point $M$. Segments $AD$ and $KM$ intersect at point $L$. Find the acute angles of triangle $ABC$, given that $AK: AL = AL: AM$.",
      "answer": "15"
    },
    {
      "question": "$\\begin{array}{l}\\text { 1. Let } p, q \\in \\mathbf{R}_{+} \\text {, and satisfy } \\\\ \\log _{9} p=\\log _{12} q=\\log _{16}(p+q) \\text {. } \\\\ \\text { Then } \\frac{q}{p}=\\end{array}$",
      "answer": "\\frac{1+\\sqrt{5}}{2}"
    },
    {
      "question": "$10 、$ Use the four numbers $8 、 8 、 8 、 10$, and the four basic arithmetic operations to form an expression $\\qquad$ that equals 24 (parentheses can be added)",
      "answer": "24"
    },
    {
      "question": "12. Given the line $a x+b y+c=0$ where $a, b, c$ are 3 different elements taken from the set $\\{-3,-2$, $-1,0,1,2,3\\}$, and the inclination angle of the line is acute. Then the number of such lines is $\\qquad$",
      "answer": "43"
    },
    {
      "question": "## Task A-4.2.\n\nLet $S$ be the set of all natural numbers less than 1000 whose all digits in decimal notation are even. Let $\\omega$ be a complex number such that $\\omega^{2}+\\omega+1=0$.\n\nCalculate the sum $\\sum_{k \\in S} \\omega^{k}$, i.e., the sum of the values $\\omega^{k}$ for all $k$ in the set $S$.",
      "answer": "-2"
    },
    {
      "question": "A mother distributes walnuts from a basket among her 4 children. Tamás receives one walnut and a quarter of the remaining walnuts; then Erzsi receives one walnut and a quarter of the remaining walnuts; Béla also receives one walnut and a quarter of the remaining walnuts; finally, Juliska receives one walnut and a quarter of the remaining walnuts. It turned out that the boys received 100 more walnuts than the girls. How many walnuts were there originally, and how many were left in the basket at the end?",
      "answer": "1021"
    },
    {
      "question": "Find the smallest natural number such that its half is divisible by three, its third is divisible by four, its quarter is divisible by eleven, and its half gives a remainder of five when divided by seven.\n\n(E. Patáková)",
      "answer": "528"
    },
    {
      "question": "[Algebraic equations and systems of equations (other)]\nFormulas for abbreviated multiplication (other)\n\nThe number $a$ is a root of the equation $x^{11} + x^{7} + x^{3} = 1$. For which natural values of $n$ does the equality $a^{4} + a^{3} = a^{n} + 1$ hold?",
      "answer": "15"
    },
    {
      "question": "3. Solve the equation $2021 x=2022 \\cdot \\sqrt[2022]{x^{2021}}-1 .(10$ points $)$",
      "answer": "1"
    },
    {
      "question": "3.5. Znayka cut out a semicircle from paper. Neznaika marked a point $C$ on the diameter $A B$ of this semicircle and cut off two semicircles from Znayka's semicircle with diameters $A C$ and $C B$. The area of the remaining figure turned out to be $10 \\pi$. Find the length of the part of the chord lying inside it, passing through point $C$ perpendicular to $A B$. If necessary, round the answer to two decimal places.",
      "answer": "2\\sqrt{10}"
    },
    {
      "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{\\pi} 2^{4} \\cdot \\cos ^{8} x d x\n$$",
      "answer": "\\frac{35\\pi}{8}"
    },
    {
      "question": "8. (12 points) Fill the ten digits from $0-9$ into the circles in the figure so that the sums of the numbers on the five lines form an arithmetic sequence, and the sum of the terms of this arithmetic sequence is 55. The number of possible values for the common difference of this arithmetic sequence is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "3"
    },
    {
      "question": "1. (6 points) Calculate: $\\frac{2 \\frac{1}{4}+0.25}{2 \\frac{3}{4}-\\frac{1}{2}}+\\frac{2 \\times 0.5}{2 \\frac{1}{5}-\\frac{2}{5}}=$",
      "answer": "\\frac{5}{3}"
    },
    {
      "question": "16. (6 points) A road is 400 meters long, and on both sides of the road, a trash can is placed every 20 meters, with the start and end points being bus stops where no trash cans are placed. How many trash cans are placed in total?",
      "answer": "38"
    },
    {
      "question": "14. Let $A=\\{1,2, \\ldots, 2011\\}$. Find the number of functions $f$ from $A$ to $A$ that satisfy $f(n) \\leq n$ for all $n$ in $A$ and attain exactly 2010 distinct values.",
      "answer": "2^{2011}-2012"
    },
    {
      "question": "3. (7 points) Four mole burrows $A, B, C, D$ are connected sequentially by three tunnels. Each minute, the mole runs through a tunnel to one of the adjacent burrows. In how many ways can the mole get from burrow $A$ to $C$ in 20 minutes?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d5d5613c0f147d056827g-22.jpg?height=103&width=834&top_left_y=288&top_left_x=608)",
      "answer": "6765"
    },
    {
      "question": "The geometric progressions $a_{1}, a_{2}, a_{3}, \\ldots$ and $b_{1}, b_{2}, b_{3}, \\ldots$ have the same ratio, with $a_{1}=$ $27, b_{1}=99$ and $a_{15}=b_{11}$. Find the value of $a_{9}$.\n\n#",
      "answer": "363"
    },
    {
      "question": "The reverse of a two-digit integer is the number obtained by reversing the order of its digits. For example, 34 is the reverse of 43. How many numbers exist such that when added to their reverse, they give a perfect square?",
      "answer": "8"
    },
    {
      "question": "The equation  \n$$ (x-1)(x-2) \\cdots(x-2016)=(x-1)(x-2) \\cdots(x-2016) $$  \nis written on the board. One tries to erase some linear factors from both sides so that each side still has at least one factor, and the resulting equation has no real roots. Find the least number of linear factors one needs to erase to achieve this.",
      "answer": "2016"
    },
    {
      "question": "3. (3 points) There are two cylinders with a volume ratio of 5:8, and the lateral surfaces of these cylinders unfold into the same rectangle. If the length and width of this rectangle are both increased by 6, its area increases by 114. What is the area of this rectangle $\\qquad$ .",
      "answer": "40"
    },
    {
      "question": "On a circle of radius 1, a point $O$ is marked, and from it, a notch is made to the right with a radius of $l$. From the resulting point $O_{1}$, another notch is made in the same direction with the same radius, and this is repeated 1968 times. After this, the circle is cut at all 1968 notches, resulting in 1968 arcs. How many different lengths of arcs can be obtained in this way?",
      "answer": "3"
    },
    {
      "question": "5. Given is $\\triangle A B C, \\measuredangle C=90^{\\circ}$. Squares $A E D C$ and $C F G B$ are constructed externally on its sides. Segment $E B$ intersects segments $A C$ and $A G$ at points $H$ and $I$, respectively. Segment $A G$ intersects segment $B C$ at point $J$. If the area of $\\triangle A I B$ is 2017, find the area of quadrilateral $H C J I$.\n\n## SOLUTIONS TO THE MUNICIPAL COMPETITION IN MATHEMATICS FOR HIGH SCHOOL STUDENTS, 21. 1. 2017.\n\n## First Grade - A category",
      "answer": "2017"
    },
    {
      "question": "Let $x$ be a non-zero real number such that $x+\\frac{1}{x}=\\sqrt{2019}$. What is the value of $x^{2}+\\frac{1}{x^{2}}$?",
      "answer": "2017"
    },
    {
      "question": "In the plane, there are 2020 points, some of which are black and the rest are green.\n\nFor each black point, there are exactly two green points that are at a distance of 2020 from this black point.\n\nDetermine the smallest possible number of green points.\n\n(Walther Janous)\n\nAnswer. The smallest possible number is 45 green points.",
      "answer": "45"
    },
    {
      "question": "71. A three-digit number leaves a remainder of 2 when divided by 4, 5, and 6. If three digits are added to the end of this number to make it a six-digit number, and this six-digit number is divisible by $4, 5, 6$, then the smallest six-digit number that meets the condition is",
      "answer": "122040"
    },
    {
      "question": "$4.35 \\operatorname{tg} 435^{\\circ}+\\operatorname{tg} 375^{\\circ}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.\n\n$4.35 \\tan 435^{\\circ}+\\tan 375^{\\circ}$.",
      "answer": "4"
    },
    {
      "question": "# Problem 3. (3 points)\n\nAll residents of the island are either blondes or brunettes with green or blue eyes. The proportion of brunettes among blue-eyed residents is $65 \\%$. The proportion of blue-eyed residents among blondes is $70 \\%$. Finally, the proportion of blondes among green-eyed residents is 10\\%. What percentage of the island's population are green-eyed brunettes?",
      "answer": "54"
    },
    {
      "question": "99. In a regular $n$-gon ( $n>5$ ), the difference between the largest and smallest diagonals is equal to the side. Find $n$.",
      "answer": "9"
    },
    {
      "question": "5. Let the mass of the new alloy be 1, and the relative masses of the 1st, 2nd, and 3rd alloys in it be \\( x, y, z \\) respectively; the content of chromium in the new alloy is \\( k \\). We obtain the system of equations:\n\n$$\n\\left\\{\\begin{array}{c}\nx+y+z=1 \\\\\n0.9 x+0.3 z=0.45 \\\\\n0.4 x+0.1 y+0.5 z=k\n\\end{array}\\right.\n$$\n\nFrom the 2nd equation: \\( y=\\frac{1}{2}-\\frac{1}{3} z \\). Substituting into the 1st equation, we get: \\( x=\\frac{1}{2}-\\frac{2}{3} z \\).\n\nSubstituting into the 3rd equation: \\( k=0.25+0.2 z \\).\n\nSince \\( z \\geq 0 \\), then \\( k_{\\min }=0.25=25 \\% \\).\n\n\\( x=\\frac{1}{2}-\\frac{2}{3} z \\Leftrightarrow z=\\frac{3}{4}-\\frac{3}{2} x \\)\n\n\\( k=0.25+0.2 z=0.4-0.3 x \\).\n\nSince \\( x \\geq 0 \\), then \\( k_{\\max }=0.4=40 \\% \\).",
      "answer": "2540"
    },
    {
      "question": "(19) Let $P$ be a moving point on the major axis of the ellipse $\\frac{x^{2}}{25}+\\frac{y^{2}}{16}=1$. A line passing through $P$ with slope $k$ intersects the ellipse at points $A$ and $B$. If $|P A|^{2}+|P B|^{2}$ depends only on $k$ and not on $P$, find the value of $k$.\n\nLet $P$ be a moving point on the major axis of the ellipse $\\frac{x^{2}}{25}+\\frac{y^{2}}{16}=1$. A line passing through $P$ with slope $k$ intersects the ellipse at points $A$ and $B$. If $|P A|^{2}+|P B|^{2}$ depends only on $k$ and not on $P$, find the value of $k$.",
      "answer": "\\\\frac{4}{5}"
    },
    {
      "question": "117. Two Ferries. Two ferries travel between two opposite banks of a river at constant speeds. Upon reaching a bank, each immediately starts moving in the opposite direction. The ferries departed from opposite banks simultaneously, met for the first time 700 meters from one of the banks, continued to their respective banks, then turned around and met again 400 meters from the other bank. Determine the width of the river in your mind.",
      "answer": "1700"
    },
    {
      "question": "8. For a finite set\n$$\nA=\\left\\{a_{i} \\mid 1 \\leqslant i \\leqslant n, i \\in \\mathbf{Z}_{+}\\right\\}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {, }\n$$\n\nlet $S=\\sum_{i=1}^{n} a_{i}$, then $S$ is called the \"sum\" of set $A$, denoted as $|A|$. Given the set $P=\\{2 n-1 \\mid n=1,2, \\cdots, 10\\}$, all subsets of $P$ containing three elements are $P_{1}, P_{2}, \\cdots, P_{k}$. Then $\\sum_{i=1}^{k}\\left|P_{i}\\right|=$ $\\qquad$ .",
      "answer": "3600"
    },
    {
      "question": "11.2. For the angles of triangle $ABC$, it is known that $\\sin A + \\cos B = \\sqrt{2}$ and $\\cos A + \\sin B = \\sqrt{2}$. Find the measure of angle $C$.",
      "answer": "90"
    },
    {
      "question": "4. Out of 43 coins arranged in a row, 8 coins are facing \"tails\" up, and 35 are facing \"heads\" up. In one step, any 20 coins can be flipped. Is it possible after a finite number of steps for all the coins to be facing \"heads\" up? In the minimum number of steps, how is this possible? Explain your answer!",
      "answer": "2"
    },
    {
      "question": "## 10. Four colors\n\nEach of the squares in a $4 \\times 4$ grid, as shown in the figure, must be colored with one of four colors. How many ways can this be done such that any four squares sharing a common vertex are colored with different colors?\n\n![](https://cdn.mathpix.com/cropped/2024_05_30_1fa462ef9f831ff2e2d1g-6.jpg?height=306&width=309&top_left_y=492&top_left_x=1512)\n\n## Result: $\\quad 168$",
      "answer": "168"
    },
    {
      "question": "## Task 2 - 340612\n\nThe tile pattern in the figure is composed of 14 white and 10 patterned triangular tiles. One can find several squares and triangles, each made up of more than one tile.\n\nHow many such squares can be found in total?\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=274&width=262&top_left_y=1391&top_left_x=1428)\n\nThere are exactly 4 squares made up of 2 tiles each, 5 squares made up of 4 tiles each, 4 squares made up of 7 tiles each, 1 square made up of 8 tiles (see examples in Fig. a, b, c, d), making a total of 14 squares of the desired type.\na)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=105&width=117&top_left_y=1935&top_left_x=655)\nb)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=106&width=111&top_left_y=1940&top_left_x=824)\nc)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=197&width=194&top_left_y=1849&top_left_x=997)\nd)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=195&width=194&top_left_y=1850&top_left_x=1251)\ne)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=102&width=106&top_left_y=2142&top_left_x=467)\nf)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=103&width=186&top_left_y=2147&top_left_x=632)\ng)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=103&width=168&top_left_y=2144&top_left_x=887)\nh)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=188&width=174&top_left_y=2062&top_left_x=1135)\ni)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0292.jpg?height=148&width=260&top_left_y=2102&top_left_x=1366)\n\nThere are exactly 20 triangles made up of 2 tiles each, 8 triangles made up of 3 tiles each, 8 triangles made up of 4 tiles each, 4 triangles made up of 8 tiles each, 4 triangles made up of 9 tiles each (see examples in Fig. e, f, g, h, i), making a total of 44 triangles of the desired type.",
      "answer": "32"
    },
    {
      "question": "2. Let $a>0$, if the solution set of the inequality $\\sqrt{x+a}+\\sqrt{x-a} \\leqslant \\sqrt{2(x+1)}$ corresponds to an interval length of $\\frac{1}{2}$ on the number line, then the value of $a$ that satisfies the condition is . $\\qquad$",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "5. Once Valera left home, walked to the cottage, painted 11 fence boards there, and returned home 2 hours after leaving. Another time, Valera and Olga went to the cottage together, painted 9 fence boards (without helping or hindering each other), and returned home together 3 hours after leaving. How many boards will Olga be able to paint alone if she needs to return home 1 hour after leaving? The physical abilities, diligence, and working conditions of Valera and Olga remain unchanged.\n\n(V. P. Fedorov)",
      "answer": "0"
    },
    {
      "question": "8. Let $g(x)=\\sqrt{x(1-x)}$ be a function defined on the interval $[0,1]$, then the area of the figure enclosed by the graph of the function $x g(x)$ and the $x$-axis is $\\qquad$ .",
      "answer": "\\frac{\\pi}{16}"
    },
    {
      "question": "20. Given a finite number of polygons in the plane, if for any two of them, there is a line passing through the origin that intersects both, then these polygons are said to be properly placed.\n\nFind the smallest positive integer $m$, such that for any set of properly placed polygons, it is possible to draw $m$ lines passing through the origin, so that each of these polygons intersects at least one of these $m$ lines.",
      "answer": "2"
    },
    {
      "question": "Determine the gcd of all numbers of the form\n\n$$\n(a-b)(b-c)(c-d)(d-a)(a-c)(b-d)\n$$\n\nwhere $a, b, c, d$ are integers.",
      "answer": "12"
    },
    {
      "question": "4. In the picture, there is a special playing die. The numbers on opposite sides always add up to the same sum. The numbers we don't see in the picture are prime. What number is opposite the number 14, and which number is opposite the number 18?\n\n![](https://cdn.mathpix.com/cropped/2024_06_03_46703489c20da379fcf1g-1.jpg?height=223&width=234&top_left_y=1873&top_left_x=911)",
      "answer": "2319"
    },
    {
      "question": "8. One or two? Let's take all natural numbers from 1 to 1000000 and for each of them, calculate the sum of its digits. For all the resulting numbers, we will again find the sum of their digits. We will continue this process until all the resulting numbers are single-digit. Among the million resulting numbers, 1 and 2 will appear. Which number will be more frequent: 1 or 2?",
      "answer": "1"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nSuppose that\n\n$$\n\\frac{1}{2^{12}}+\\frac{1}{2^{11}}+\\frac{1}{2^{10}}+\\cdots+\\frac{1}{2^{t+1}}+\\frac{1}{2^{t}}=\\frac{n}{2^{12}}\n$$\n\n(The sum on the left side consists of $13-t$ terms.)\n\nWhat is the value of $n$ ?",
      "answer": "127"
    },
    {
      "question": "Solve the following equation:\n\n$$\n\\frac{3 x+3}{\\sqrt{x}}=4+\\frac{x+1}{\\sqrt{x^{2}-x+1}} .\n$$",
      "answer": "1"
    },
    {
      "question": "## 42. Twins\n\nAnna and Brigitte, two little girls, are twins; they look like each other as much as two drops of water. At school, they sit at the same desk; one of them always sits on the left, and the other on the right. Each of them claims that she is Brigitte. The teacher knows from experience that, generally speaking, the girl on the right lies one time out of four, while the girl on the left lies one time out of five. But the teacher is interested in the probability that Anna is sitting on the left. Help her.",
      "answer": "\\frac{3}{7}"
    },
    {
      "question": "10. In the tetrahedron $P-ABC$, $PA=BC=\\sqrt{6}$, $PB=AC=\\sqrt{8}$, $PC=AB=\\sqrt{10}$. Then the radius of the circumscribed sphere of the tetrahedron is $\\qquad$.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "7 Given complex numbers $Z_{1}$ and $Z_{2}$ satisfy $\\left|Z_{1}\\right|=2,\\left|Z_{2}\\right|=3$. If the angle between the vectors they correspond to is $60^{\\circ}$, then $\\left|\\frac{Z_{1}+Z_{2}}{Z_{1}-Z_{2}}\\right|=$ $\\qquad$ .",
      "answer": "\\frac{\\sqrt{133}}{7}"
    },
    {
      "question": "In Anchuria, a unified state exam is taking place. The probability of guessing the correct answer to each question on the exam is 0.25. In 2011, to obtain a certificate, one needed to answer three questions correctly out of 20. In 2012, the School Administration of Anchuria decided that three questions were too few. Now, one needs to answer six questions correctly out of 40. The question is, if one knows nothing and simply guesses the answers, in which year is the probability of obtaining an Anchurian certificate higher - in 2011 or in 2012?",
      "answer": "6"
    },
    {
      "question": "$9.17 C_{x}^{x-1}+C_{x}^{x-2}+C_{x}^{x-3}+\\ldots+C_{x}^{x-9}+C_{x}^{x-10}=1023$.",
      "answer": "10"
    },
    {
      "question": "57. The number $123456789(10)(11)(12)(13)(14)$ is written in the base-15 numeral system, i.e., this number is equal to\n\n(14) $+(13) \\cdot 15+(12) \\cdot 15^{2}+(11) \\cdot 15^{3}+\\ldots+2 \\cdot 15^{12}+15^{13}$. What remainder does it give when divided by 7?",
      "answer": "0"
    },
    {
      "question": "6. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies:\n$$\na_{1}=1, a_{n+1}=\\frac{(n+1) a_{n}}{2 n+a_{n}}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nThen $\\sum_{k=1}^{2021} \\frac{k}{a_{k}}=$ . $\\qquad$",
      "answer": "2^{2022}-2023"
    },
    {
      "question": "5. The numbers from 1 to 8 are arranged at the vertices of a cube such that the sum of the numbers in any three vertices lying on the same face is at least 10. What is the smallest possible sum of the numbers at the vertices of one face?",
      "answer": "16"
    },
    {
      "question": "16. Arrange $a, b, c, d, e, f$ such that the letter $b$ always appears before the letter $e$. How many such arrangements are there?",
      "answer": "360"
    },
    {
      "question": "8th Mexico 1994 Problem B1 A capricious mathematician writes a book with pages numbered from 2 to 400. The pages are to be read in the following order. Take the last unread page (400), then read (in the usual order) all pages which are not relatively prime to it and which have not been read before. Repeat until all pages are read. So, the order would be 2, 4, 5, ... , 400, 3, 7, 9, ... , 399, ... . What is the last page to be read?",
      "answer": "37"
    },
    {
      "question": "10. Determine the maximum value attained by\n$$\n\\frac{x^{4}-x^{2}}{x^{6}+2 x^{3}-1}\n$$\nover real numbers $x>1$.",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "8. (10 points) Divide a cubic wooden block with an edge length of 1 meter into small cubic building blocks with an edge length of 1 centimeter. Imagine that Sun Wukong uses his divine power to stack all the small building blocks one by one, forming a rectangular \"magic rod\" pointing to the sky. Knowing that the altitude of Mount Everest is 8844 meters, then the height of the \"magic rod\" exceeds the altitude of Mount Everest by meters.",
      "answer": "1156"
    },
    {
      "question": "G47 (23-5, Netherlands) Given that $A C, C E$ are two diagonals of a regular hexagon $A B C D E F$, and points $M, N$ internally divide $A C, C E$ respectively, such that $\\frac{A M}{A C}=\\frac{C N}{C E}=r$. If $B, M, N$ are collinear, find the value of $r$.\n\n---\n\nThe problem is to find the value of $r$ given that points $M$ and $N$ divide the diagonals $A C$ and $C E$ of a regular hexagon $A B C D E F$ in the ratio $r$, and that points $B, M, N$ are collinear.",
      "answer": "\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "39th Putnam 1978 Problem B1 A convex octagon inscribed in a circle has 4 consecutive sides length 3 and the remaining sides length 2. Find its area.",
      "answer": "13+12\\sqrt{2}"
    },
    {
      "question": "11.3. How many points are there on the hyperbola $y=\\frac{2013}{x}$ such that the tangent at these points intersects both coordinate axes at points with integer coordinates",
      "answer": "48"
    },
    {
      "question": "## Problem 2\n\nFind the smallest positive integer $k$ for which there exist a colouring of the positive integers $\\mathbb{Z}_{>0}$ with $k$ colours and a function $f: \\mathbb{Z}_{>0} \\rightarrow \\mathbb{Z}_{>0}$ with the following two properties:\n\n(i) For all positive integers $m, n$ of the same colour, $f(m+n)=f(m)+f(n)$.\n\n(ii) There are positive integers $m, n$ such that $f(m+n) \\neq f(m)+f(n)$.\n\nIn a colouring of $\\mathbb{Z}_{>0}$ with $k$ colours, every integer is coloured in exactly one of the $k$ colours. In both (i) and (ii) the positive integers $m, n$ are not necessarily different.\n\nMerlijn Staps, the Netherlands",
      "answer": "3"
    },
    {
      "question": "7.4. How to cut a $5 \\times 5$ square with straight lines so that the resulting pieces can be used to form 50 equal squares? It is not allowed to leave unused pieces or overlap them.",
      "answer": "50"
    },
    {
      "question": "4. In the Olympionov family, it is a tradition to especially celebrate the day when a person turns as many years old as the sum of the digits of their birth year. Kolya Olympionov had such a celebration in 2013, and Tolya Olympionov had one in 2014. Who is older and by how many years?",
      "answer": "4"
    },
    {
      "question": "1.97 Find the largest integer $n$ such that $\\frac{(n-2)^{2}(n+1)}{2 n-1}$ is an integer.\n(17th American Invitational Mathematics Examination, 1999)",
      "answer": "14"
    },
    {
      "question": "1. Calculate: $3752 \\div(39 \\times 2)+5030 \\div(39 \\times 10)=$",
      "answer": "61"
    },
    {
      "question": "10. As shown in the figure, two squares $\\mathrm{ABEG}, \\mathrm{GECD}$, point $\\mathrm{H}$ is the midpoint of $\\mathrm{GE}$, $\\frac{D F}{D C}=\\frac{1}{3}$. Connect $\\mathrm{DH}$, $\\mathrm{CH}$, $\\mathrm{AF}$, $\\mathrm{BF}$. The area of square $\\mathrm{ABEG}$ is $\\mathrm{m}$ square centimeters, and the area of the shaded part is $\\mathrm{n}$ square centimeters. Given that $\\mathrm{m}$ and $\\mathrm{n}$ are positive integers, and $\\mathrm{m}$ has 9 divisors, then the side length of square $\\mathrm{ABEG}$ is $\\qquad$ centimeters.",
      "answer": "10"
    },
    {
      "question": "12. In the tetrahedron $ABCD$, there is a point $O$ inside such that the lines $AO, BO, CO, DO$ intersect the faces $BCD, ACD, ABD, ABC$ at points $A_1, B_1, C_1, D_1$ respectively, and $\\frac{AO}{OA_1}=\\frac{BO}{OB_1}=\\frac{CO}{OC_1}=\\frac{DO}{OD_1}=k$. Then $k=$ $\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "Example 8 Given an integer $n \\geqslant 2$, try to determine the smallest constant $c$, such that for any non-negative real numbers $x_{i}(1 \\leqslant i \\leqslant n)$, we have $\\sum_{1 \\leqslant i<j \\leqslant n} x_{i} x_{j}\\left(x_{i}^{2}+x_{j}^{2}\\right) \\leqslant c\\left(\\sum_{i=1}^{n} x_{i}\\right)^{4}$. And for this constant $c$, determine the necessary and sufficient conditions for equality to hold.",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "11.4 If $\\sqrt{3-2 \\sqrt{2}}=\\sqrt{c}-\\sqrt{d}$, find the value of $d$.",
      "answer": "1"
    },
    {
      "question": "11.9. In a class, there are $m$ students. During September, each of them went to the swimming pool several times; no one went twice on the same day. On October 1st, it turned out that all the numbers of swimming pool visits by the students were different. Moreover, for any two of them, there was definitely a day when the first one was at the pool and the second one was not, and a day when, conversely, the second one was at the pool and the first one was not. Find the maximum possible value of $m$. (September has 30 days.)",
      "answer": "28"
    },
    {
      "question": "1. (2 points) Does there exist a four-digit natural number with the sum of its digits being 21, which is divisible by $14 ?$",
      "answer": "6384"
    },
    {
      "question": "## Task B-3.4.\n\nA circle with a diameter of $5 \\mathrm{~cm}$ is inscribed in an isosceles trapezoid with an area of $36 \\mathrm{~cm}^{2}$. Determine the perimeter of the trapezoid.",
      "answer": "28.8"
    },
    {
      "question": "In a certain right-angled triangle, the sum of the hypotenuse and one of the legs is $c+b=14 \\text{~cm}, \\operatorname{tg} \\frac{\\alpha}{2}=\\frac{1}{2}$, where $\\alpha$ is the angle opposite the leg $a$. Calculate the lengths of the sides of the triangle without using trigonometric tables.",
      "answer": "7"
    },
    {
      "question": "3. What can the value of the expression $p^{4}-3 p^{3}-5 p^{2}+16 p+2015$ be if $p$ is a root of the equation $x^{3}-5 x+1=0$?\n\nAnswer: 2018",
      "answer": "2018"
    },
    {
      "question": "19. (7 points) When Wang Ming is 300 meters away from home after school, his sister and the puppy run towards him at the same time. Wang Ming and his sister both run at a speed of 50 meters per minute, while the puppy runs at a speed of 200 meters per minute. The puppy meets Wang Ming and then keeps running back and forth between Wang Ming and his sister at the same speed. When Wang Ming and his sister are 10 meters apart, how many meters has the puppy run in total?",
      "answer": "580"
    },
    {
      "question": "4. Given a tetrahedron $S-ABC$ with the base being an isosceles right triangle with $AB$ as the hypotenuse, $SA=SB=SC=2, AB=$ 2, and suppose points $S, A, B, C$ all lie on a sphere with center $O$. Then the distance from point $O$ to the plane $ABC$ is $\\qquad$",
      "answer": "\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "## Task 5\n\nPeter calculates the sum of three numbers $a, b$, and $c$.\n\nThe number $a$ is 100 greater than the difference between the numbers 50700 and 30200.\n\nThe number $b$ is the successor of seven times 583.\n\nThe number $c$ is the third part of the predecessor of 2101.\n\nGive the numbers $a, b$, and $c$ and their sum.",
      "answer": "25382"
    },
    {
      "question": "12. Nick and Tim are playing a game, in which there are no ties in each round. In a round of the game, the losing side must take at least one candy, and the winning side takes more candies than the losing side. They also agreed in advance on the number of candies (two given positive integers) that the losing and winning sides should take in each round. After several rounds, Nick has a total of 30 candies, and Tim has a total of 25 candies. It is known that Tim won 2 rounds. Question: How many candies should the winning side take in each round?\n(2004 Slovenian Olympiad Problem)",
      "answer": "8"
    },
    {
      "question": "1. (3 points) Xiaohui started from point $A$ and walked 15 meters west to reach point $B$, then walked 23 meters east from point $B$ to reach point $C$. The distance from point $C$ to point $A$ is $\\qquad$ meters.",
      "answer": "8"
    },
    {
      "question": "3. Schoolchildren were given 7 sheets of paper and asked to cut some of them into 7 pieces. The resulting pieces of paper were mixed and they were asked again to cut some of them into 7 pieces. This was repeated several times. How many pieces of paper need to be cut so that in the end there are 331 pieces.",
      "answer": "54"
    },
    {
      "question": "Russian problems Problem 92 ABCD is a unit square. One vertex of a rhombus lies on side AB, another on side BC, and a third on side AD. Find the area of the set of all possible locations for the fourth vertex of the rhombus. Solution",
      "answer": "2\\frac{1}{3}"
    },
    {
      "question": "Let $n$ be a two-digit number written in the decimal system, and let $s$ be the sum of the squares of its digits. What is the smallest and largest value of $n-s$?",
      "answer": "25-63"
    },
    {
      "question": "18. A positive integer is said to be 'good' if each digit is 1 or 2 and there is neither four consecutive 1 's nor three consecutive 2 's. Let $a_{n}$ denote the number of $n$-digit positive integers that are 'good'. Find the value of $\\frac{a_{10}-a_{8}-a_{5}}{a_{7}+a_{6}}$.\n(2 marks)\n若某正整數的每個數字均為 1 或 2 , 且當中既沒有四個連續的 $\\ulcorner 1$ 亦沒有三個連續的 22 , 便稱為 「好數」。設 $a_{n}$ 表示 $n$ 位「好數」的數目。求 $\\frac{a_{10}-a_{8}-a_{5}}{a_{7}+a_{6}}$ 的值。",
      "answer": "2"
    },
    {
      "question": "585. Find the last non-zero digit of the product of all natural numbers from 1 to 40.",
      "answer": "6"
    },
    {
      "question": "9.2. a) Given the quadratic equation $x^{2}-9 x-10=0$. Let $a$ be its smallest root. Find $a^{4}-909 a$. b) For the quadratic equation $x^{2}-9 x+10=0$, where $b$ is the smallest root, find $b^{4}-549 b$.",
      "answer": "910"
    },
    {
      "question": "8. For a four-digit number $\\overline{a b c d}(1 \\leqslant a \\leqslant 9,0 \\leqslant b, c, d \\leqslant 9)$, if $a>b, bd$, then $\\overline{a b c d}$ is called a $P$ class number; if $ac, c<d$, then $\\overline{a b c d}$ is called a $Q$ class number. Let $N(P)$ and $N(Q)$ represent the number of $P$ class numbers and $Q$ class numbers, respectively, then the value of $N(P)-N(Q)$ is $\\qquad$.",
      "answer": "285"
    },
    {
      "question": "5. If $a>b>c>d$, and $\\frac{1}{a-b}+\\frac{1}{b-c}+\\frac{1}{c-d} \\geq \\frac{n}{a-d}$, then the maximum value of the integer $n$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "9"
    },
    {
      "question": "6. A circle of radius 2, inscribed in triangle $A B C$, touches side $B C$ at point $D$. A circle of radius 4 touches the extensions of sides $A B$ and $A C$, as well as side $B C$ at point $E$. Find the area of triangle $A B C$, if the measure of angle $A C B$ is $120^{\\circ}$.",
      "answer": "\\frac{56}{\\sqrt{3}}"
    },
    {
      "question": "7. (4 points) With the number written on the board, one of the following operations is allowed:\n1) If there is a digit in the original number that is not equal to 9 and has two neighboring digits greater than 0, you can increase this digit by 1, and decrease the neighboring digits by 1.\n2) Subtract 1 from any non-zero digit except the last one, and add 3 to the next digit.\n3) Decrease any sufficiently large digit by 7.\n\nIf, as a result of any of these operations, there are zeros in the number on one or more of the leading positions, they are automatically discarded.\n\nInitially, the number consisting of one hundred nines was written on the board. In the end, a single-digit number remained. Which one exactly?",
      "answer": "3"
    },
    {
      "question": "6. (1995 National High School Mathematics Competition) Find all real numbers $p$ such that the cubic equation $5 x^{3}-5(p+1) x^{2}+(71 p-1) x+1=66 p$ has three roots that are all natural numbers.",
      "answer": "76"
    },
    {
      "question": "Example 12. A factory sent 5000 good-quality items to a base. The probability that an item will be damaged during transportation is 0.0002. What is the probability that 3 defective items will arrive at the base?\n\n$\\mathrm{Pe} \\mathrm{sh} \\mathbf{e n i e . ~ F r o m ~ t h e ~ c o n d i t i o n ~ i t ~ f o l l o w s ~ t h a t ~} n=5000, p=0.0002, \\text{ and } np=5000 \\cdot 0.0002=1$. According to formula (3.3.1) we have\n\n$$\nP_{5000}(3)=\\frac{1^{3}}{3!} e^{-1}=\\frac{1}{6 e} \\approx 0.0613\n$$\n\nExample 13. A radio apparatus consists of 1000 electrical elements. The probability of failure of one element in one year of operation is 0.001 and is independent of the state of other elements. What is the probability of the failure of two elements? What is the probability of the failure of at least two elements in a year?",
      "answer": "0"
    },
    {
      "question": "## Task B-1.7.\n\nIn a right-angled triangle $ABC$, the right angle is at vertex $C$, $|AB|=8$, and $\\angle ABC=60^{\\circ}$. The triangle $ABC$ is rotated around vertex $C$ by $30^{\\circ}$ counterclockwise, mapping it to triangle $A'B'C$. What is the area of the common part of triangles $ABC$ and $A'B'C$?",
      "answer": "24-10\\sqrt{3}"
    },
    {
      "question": "36. The number of students in grades V-VI of one of the schools is expressed by a three-digit number. If you find the product of the digits of this number, and then the product of the digits of the resulting number, then all these three numbers can be written as:\n\n$$\n\\triangle \\bigcirc O ; \\triangle \\square ; \\square\n$$\n\n(Identical shapes correspond to identical digits.) How many students are there in grades V-VI in the school?",
      "answer": "144"
    },
    {
      "question": "Problem 15. Solve the equation\n\n$$\n\\frac{5}{\\sqrt{1-x^{2}}}=\\frac{1}{x}+\\frac{85}{12}\n$$\n\n104",
      "answer": "0.8"
    },
    {
      "question": "Do there exist positive integers $x, y, z$ such that $28 x+30 y+31 z=365$ ? What is the largest positive integer $n$ such that the equation $28 x+30 y+31 z=n$ cannot be solved in positive integers $x, y, z$ ?",
      "answer": "370"
    },
    {
      "question": "## [\n\nIn an isosceles trapezoid with bases 1 and 4, two circles are placed, each touching the other circle, the two lateral sides, and one of the bases. Find the area of the trapezoid.\n\n#",
      "answer": "\\frac{15\\sqrt{2}}{2}"
    },
    {
      "question": "Example 12 Let $X=\\{1,2, \\cdots, 2001\\}$. Find the smallest positive integer $m$, such that for any $m$-element subset $W$ of $X$, there exist $u, v \\in W$ (where $u$ and $v$ can be the same), such that $u+v$ is a power of 2.\n(2001 China Mathematical Olympiad)",
      "answer": "999"
    },
    {
      "question": "B2. Alice places a coin, heads up, on a table then turns off the light and leaves the room. Bill enters the room with 2 coins and flips them onto the table and leaves. Carl enters the room, in the dark, and removes a coin at random. Alice reenters the room, turns on the light and notices that both coins are heads. What is the probability that the coin Carl removed was also heads?",
      "answer": "\\frac{3}{5}"
    },
    {
      "question": "Exercise 14. Let $x$ be a rational number such that $\\sqrt{x}+\\sqrt{x+13}=\\sqrt{4 x+17}$. We write $x$ in the form $x=\\frac{a}{b}$ where $a$ and $b$ are natural numbers with no common divisors. Determine the value of $a+b$.\n\n---\n\nThe translation is provided as requested, maintaining the original formatting and structure.",
      "answer": "13"
    },
    {
      "question": "11.4. A set of composite numbers from the set $\\{1,2,3,4, \\ldots, 2016\\}$ is called good if any two numbers in this set do not have common divisors (other than 1). What is the maximum number of numbers that a good set can have?",
      "answer": "14"
    },
    {
      "question": "[ Work Problems.]\n\nThree workers are digging a pit. They work in turns, and each of them works as long as it takes the other two to dig half of the pit. Working this way, they dug the pit. How many times faster would three workers dig the same pit if they worked simultaneously?",
      "answer": "2.5"
    },
    {
      "question": "10.218. A line parallel to the bases of a trapezoid passes through the point of intersection of its diagonals. Find the length of the segment of this line, enclosed between the lateral sides of the trapezoid, if the bases of the trapezoid are 4 and 12 cm.",
      "answer": "6"
    },
    {
      "question": "For the elements of the sequence $a_{n}$, it holds that $a_{1}=1337$, and furthermore, that $a_{2 n+1}=a_{2 n}=n-a_{n}$ for every positive integer $n$. Determine the value of $a_{2004}$.",
      "answer": "2004"
    },
    {
      "question": "Robots Robert and Hubert assemble and disassemble coffee grinders. Each of them assembles a coffee grinder four times faster than they can disassemble it. When they came to the workshop in the morning, several coffee grinders were already assembled there.\n\nAt 7:00, Hubert started assembling and Robert started disassembling, and precisely at 12:00, Hubert finished assembling a coffee grinder and Robert finished disassembling another one. In total, 70 coffee grinders were added during this shift.\n\nAt 13:00, Robert started assembling and Hubert started disassembling, and precisely at 22:00, Robert finished assembling the last coffee grinder and Hubert finished disassembling another one. In total, 36 coffee grinders were added during this shift.\n\nHow long would it take to assemble 360 coffee grinders if Robert and Hubert were assembling together? (K. Pazourek)\n\nHint. How many coffee grinders will be added per hour in each shift?",
      "answer": "15"
    },
    {
      "question": "1. (2 points) In trapezoid $A B C D$ with bases $A D=16$ and $B C=10$, the circles constructed on sides $A B, B C$ and $C D$ as diameters intersect at one point. The length of diagonal $A C$ is 10. Find the length of $B D$.",
      "answer": "24"
    },
    {
      "question": "If the Earth were to suddenly stop its orbit around the Sun and fall into the Sun, how long would it take for the Earth to reach the Sun?",
      "answer": "64.5"
    },
    {
      "question": "For positive real numbers $x$ and $y$, the following holds:\n\n$$\n2 \\log (x-2 y)=\\log x+\\log y\n$$\n\nWhat is $\\frac{x}{y} ?$",
      "answer": "4"
    },
    {
      "question": "Example 1 As shown in Figure 13-4, in $\\triangle A B C$, $O$ is the circumcenter of $\\triangle A B C$, and $I$ is its incenter. If $\\angle B O C=$ $\\angle B I C$, find $\\angle A$.\n\n保留了原文的格式和换行。",
      "answer": "60"
    },
    {
      "question": "Problem 7.4.1. Brothers Lёsha and Sasha decided to get to the skate park. They left at the same time, but Lёsha walked with the skateboard in his hands, while Sasha rode on the skateboard. It is known that Sasha rides the skateboard 3 times faster than Lёsha walks with the skateboard. After some time, they simultaneously switched their modes of transportation: Lёsha started riding the skateboard, and Sasha started walking. As a result, each of their speeds changed by a factor of 2: Lёsha's speed increased, and Sasha's speed decreased. It turned out that they arrived at the skate park at the same time. How many meters did Sasha ride on the skateboard, if the distance from home to the skate park is 3300 meters?",
      "answer": "1100"
    },
    {
      "question": "4. Given real numbers $x, y$ satisfy\n$$\n2 x=\\ln (x+y-1)+\\ln (x-y-1)+4 \\text {. }\n$$\n\nThen the value of $2015 x^{2}+2016 y^{3}$ is $\\qquad$",
      "answer": "8060"
    },
    {
      "question": "Someone says that seven times their birth year, when divided by 13, leaves a remainder of 11, and thirteen times their birth year, when divided by 11, leaves a remainder of 7. In which year of their life will this person be in 1954?",
      "answer": "1868"
    },
    {
      "question": "Example 3. Solve the equation\n\n$$\n5^{2 x+1}-3 \\cdot 5^{2 x-1}=550\n$$",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "1. In triangle $A B C, A B=6, B C=10$, and $C A=14$. If $D, E$, and $F$ are the midpoints of sides $B C$, $C A$, and $A B$, respectively, find $A D^{2}+B E^{2}+C F^{2}$.",
      "answer": "249"
    },
    {
      "question": "GS. 2 If $R$ is the remainder of $1^{6}+2^{6}+3^{6}+4^{6}+5^{6}+6^{6}$ divided by 7 , find the value of $R$.",
      "answer": "6"
    },
    {
      "question": "6.58. $\\lim _{x \\rightarrow+0} x^{x}$.",
      "answer": "1"
    },
    {
      "question": "225. Fruit Garden. The gardener decided to create a new fruit garden. He planted young trees in rows so that a square was formed. In doing so, he had 146 extra seedlings left. But to increase the square by adding an extra row, the gardener had to buy 31 more trees.\n\nHow many trees are there in the garden at the end of the work?",
      "answer": "7921"
    },
    {
      "question": "4. Let $X=\\{1,2,3, \\ldots, 12\\}$. Find the the number of pairs $\\{A, B\\}$ such that $A \\subseteq X$, $B \\subseteq X, A \\neq B$ and $A \\cap B=\\{2,3,5,7,8\\}$.",
      "answer": "3^{7}-1"
    },
    {
      "question": "In how many ways can we remove 8 letters from the 16 letters in Figure 1 so that two are missing from each row and each column?\n\n| $A$ | $B$ | $C$ | $D$ |\n| :---: | :---: | :---: | :---: |\n| $E$ | $F$ | $G$ | $H$ |\n| $I$ | $J$ | $K$ | $L$ |\n| $M$ | $N$ | $O$ | $P$ |",
      "answer": "90"
    },
    {
      "question": "8. It is given that $\\sqrt{a}-\\sqrt{b}=20$, where $a$ and $b$ are real numbers. Find the maximum possible value of $a-5 b$.",
      "answer": "500"
    },
    {
      "question": "1. The solution to the equation $\\frac{16}{3} x^{4}+\\frac{1}{6 x^{2}}=\\sin \\pi x$ is",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "1. [5] Solve for $x$ in the equation $20 \\cdot 14+x=20+14 \\cdot x$.",
      "answer": "20"
    },
    {
      "question": "5. What is the maximum number of rooks that can be placed on the cells of a $300 \\times 300$ board so that each rook attacks no more than one other rook? (A rook attacks all cells it can reach according to chess rules, without passing through other pieces.)\n\n#",
      "answer": "400"
    },
    {
      "question": "6.1. On 19 cards, the numbers $15,16,17, \\ldots, 33$ are written respectively (one number per card). Members of the math club Vasya, Petya, and Misha decided to divide all these cards among themselves so that each of them gets at least one card and no one ends up with a pair of cards where the difference between the numbers is odd. How many ways are there to distribute the cards?",
      "answer": "4596"
    },
    {
      "question": "Problem 10.4. Consider the sequence\n\n$$\na_{n}=\\cos (\\underbrace{100 \\ldots 0^{\\circ}}_{n-1})\n$$\n\nFor example, $a_{1}=\\cos 1^{\\circ}, a_{6}=\\cos 100000^{\\circ}$.\n\nHow many of the numbers $a_{1}, a_{2}, \\ldots, a_{100}$ are positive?",
      "answer": "99"
    },
    {
      "question": "4. In the store \"Everything for School,\" chalk is sold in packages of three grades: regular, unusual, and superior. Initially, the ratio of the quantities by grade was 3:4:6. As a result of sales and deliveries from the warehouse, this ratio changed to 2:5:8. It is known that the number of packages of superior chalk increased by $80\\%$, and the number of packages of regular chalk decreased by no more than 10 packages. How many packages of chalk were there in the store initially?\n\n(7 points)",
      "answer": "260"
    },
    {
      "question": "The Vaskapu rising above the city of Esztergom is seen from point $A$ near the railway station at an angle of $6^{\\circ} 50^{\\prime} 33^{\\prime \\prime}$, and from 200 meters closer at an angle of $7^{\\circ} 25^{\\prime} 52^{\\prime \\prime}$; from the same point $A$, the Sashegy is seen at an angle of $5^{\\circ} 24^{\\prime} 52^{\\prime \\prime}$, and from 200 meters closer at an angle of $5^{\\circ} 55^{\\prime} 36^{\\prime \\prime}$; the angle formed by the horizontal direction lines starting from point $A$ is $31^{\\circ} 4^{\\prime} 34^{\\prime \\prime}$. How far apart are the two mountain peaks from each other?",
      "answer": "1303"
    },
    {
      "question": "G3.4 Let $x \\geq 0$ and $y \\geq 0$. Given that $x+y=18$. If the maximum value of $\\sqrt{x}+\\sqrt{y}$ is $d$, find the value of $d$. (Reference: 1999 FGS.2)",
      "answer": "6"
    },
    {
      "question": "Draw two circles with the same center, one with a radius of $1 \\mathrm{~cm}$ and the other with a radius of $3 \\mathrm{~cm}$. In the region outside the circle with a radius of $1 \\mathrm{~cm}$ and inside the circle with a radius of $3 \\mathrm{~cm}$, draw circles that are simultaneously tangent to both circles, as shown in the figure below.\n\n(a) What should be the radius of these circles?\n\n(b) What is the maximum number of these circles, assuming they do not overlap?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_eead07372ad548e635bag-05.jpg?height=250&width=252&top_left_y=1214&top_left_x=1424)",
      "answer": "6"
    },
    {
      "question": "## Problem 10.3. (Problem for the Atanas Radev award)\n\nFind all natural numbers $n$ such that: If $a$ and $b$ are natural numbers and $n$ divides $a^{2} b+1$, then $n$ divides $a^{2}+b$.",
      "answer": "240"
    },
    {
      "question": "13.429 First, the motorboat traveled $a$ km across the lake, and then half of this distance up the river flowing into the lake. The entire trip lasted 1 hour. Find the own speed of the motorboat if the speed of the river current is $c$ km/h.",
      "answer": "\\frac{3+2+\\sqrt{9^{2}-4+4^{2}}}{4}"
    },
    {
      "question": "3. Find the sum\n$$\n\\frac{2^{1}}{4^{1}-1}+\\frac{2^{2}}{4^{2}-1}+\\frac{2^{4}}{4^{4}-1}+\\frac{2^{8}}{4^{8}-1}+\\cdots .\n$$",
      "answer": "1"
    },
    {
      "question": "2. (1992 Ukrainian Mathematical Olympiad) Find all prime numbers $p$ not exceeding 1000 such that $2p+1$ is a power of a natural number (i.e., there exist natural numbers $m$ and $n, n \\geqslant 2$, such that $2p+1=m^n$).",
      "answer": "13"
    },
    {
      "question": "1. Janez spent $\\frac{3}{5}$ of the money he had, then $\\frac{5}{9}$ of the remainder, and finally $\\frac{3}{8}$ of the new remainder. He was left with 80 euros. How much money did he have at the beginning?",
      "answer": "720"
    },
    {
      "question": "For positive numbers $a, b, c$ such that $a^{2}+b^{2}+c^{2}=1$. Determine the smallest possible value of\n\n$$\nS=\\frac{a b}{c}+\\frac{b c}{a}+\\frac{c a}{b}\n$$",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "# Task 2.\n\nWhile a lion cub, who was 6 minutes away from the water hole, set off to drink, a second one, having already quenched his thirst, started heading back along the same path at 1.5 times the speed of the first. At the same time, a turtle, who was 32 minutes away from the water hole, set off along the same path to drink. After some time, the first lion cub stepped on the turtle, and 2 minutes and 24 seconds later, the second lion cub stepped on her. How many minutes after the second incident did the turtle reach the water hole, given that all three moved at constant speeds?",
      "answer": "28.8"
    },
    {
      "question": "3. (10 points) The distance between $A$ and $B$ is 500 kilometers. Two people, Jia and Yi, start cycling from $A$ to $B$ at the same time. Jia rides 30 kilometers per day, and Yi rides 50 kilometers per day but rests every other day. At the end of the $\\qquad$th day, the distance Yi has left to $B$ is twice the distance Jia has left to $B$.",
      "answer": "15"
    },
    {
      "question": "Pista received the following task. \"Write the\n\n$$\n\\frac{16 e-10}{10 e-3}\n$$\n\nexpression, substituting integers for $e$, and simplify the resulting fraction if possible. Collect all the numbers with which you simplified!\"\n\nAfter several attempts, Pista found such a number and then proved that there are no more such numbers. Let's solve the problem too.",
      "answer": "13"
    },
    {
      "question": "11. (IMO-32 Shortlist) Find the largest positive integer $k$, such that\n$$\n1991^{k} \\mid 1990^{19911992}+1992^{19911990} \\text {. }\n$$",
      "answer": "1991"
    },
    {
      "question": "Let $f: \\mathbb{N} \\rightarrow \\mathbb{N}$ satisfy\n\n$f(1)=2, f(2)=1, f(3 n)=3 f(n), f(3 n+1)=3 f(n)+2, f(3 n+2)=3 f(n)+1$.\n\nFind how many integers $n \\leq 2014$ such that $f(n)=2 n$.",
      "answer": "128"
    },
    {
      "question": "During a school dance performance, a total of 430 different pairs danced. The first girl danced with 12 different boys, the second girl with 13 different boys, and so on, until the last girl danced with every boy. How many girls and how many boys attended the performance?",
      "answer": "20"
    },
    {
      "question": "4. With one mower, it is possible to mow $2 \\frac{1}{2}$ ha of meadow in 2 hours, and with another, smaller mower, $1 \\frac{1}{4}$ ha can be mowed in 1 hour and 30 minutes. How much meadow area will both mowers mow together in 3 hours and 36 minutes?",
      "answer": "7\\frac{1}{2}"
    },
    {
      "question": "Example 22 (2002 National High School Mathematics Competition Question) Before the World Cup, Coach of Country $F$ is evaluating seven players, $A_{1}, A_{2}, \\cdots, A_{7}$, and plans to have them all play in three training matches (each 90 minutes long). Assuming that at any moment during the matches, exactly one of these players is on the field, and the total playing time (in minutes) of $A_{1}, A_{2}, A_{3}, A_{4}$ is divisible by 13. If there are no limits on the number of substitutions per match, how many different scenarios are there for the total playing time of each player?",
      "answer": "42244"
    },
    {
      "question": "Ali-Baba came to a cave where there is gold, diamonds, and a chest in which he can carry them away. A full chest of gold weighs 200 kg, a full chest of diamonds - 40 kg, an empty chest weighs nothing. A kilogram of gold is worth 20 dinars on the market, a kilogram of diamonds - 60 dinars. Ali-Baba can lift and carry no more than 100 kg. What is the maximum amount of money he can get for the treasures he brings out of the cave in one go?",
      "answer": "3000"
    },
    {
      "question": "Problem 3. How many odd three-digit numbers exist where the digit in the tens place is 4?",
      "answer": "45"
    },
    {
      "question": "10. As shown in the figure, it is a chessboard. The chess piece \"Knight\" can move from one square, jumping 2 squares horizontally or vertically, and at the same time jumping 1 square in the other direction, forming an \"L\" shape. For example, from point A in the figure, it can reach any of the B positions in one step. Then, for a chess piece \"Knight\" to jump from the bottom-left corner of the chessboard to the top-right corner, the minimum number of steps required is $\\qquad$.",
      "answer": "6"
    },
    {
      "question": "9. Consider a triangle $A B C$ and a point $D$ on the side $A C$. If $\\overline{A B}=\\overline{D C}=1, \\angle D B C=30^{\\circ}$ and $\\angle A B D=90^{\\circ}$, calculate the value of $\\overline{A D}$.",
      "answer": "\\sqrt[3]{2}"
    },
    {
      "question": "15. Each letter in the figure represents a number, and the sum of the three numbers in each circle is 21, and the sum of all the numbers is 69. Then $b=$ $\\qquad$ .",
      "answer": "10"
    },
    {
      "question": "Problem 7.8. Carlson and Little Man have several jars of jam, each weighing an integer number of pounds.\n\nThe total weight of all the jars of jam that Carlson has is 13 times the total weight of all the jars that Little Man has. Carlson gave Little Man the jar with the smallest weight (from those he had), after which the total weight of his jars turned out to be 8 times the total weight of Little Man's jars.\n\nWhat is the maximum number of jars of jam that Carlson could have had initially?",
      "answer": "23"
    },
    {
      "question": "2. Let $a$ and $b$ be real numbers such that $a>b>0$ and $a^{2}+b^{2}=6ab$. Determine the value of the expression $\\frac{a+b}{a-b}$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "5. First-grader (6th grade. 2 points). Maksim will go to the first grade on September 1, 2014, at the age of 6 years, and this day will not be his birthday. What is the probability that he was born in 2008?",
      "answer": "\\frac{244}{365}"
    },
    {
      "question": "37. In two boxes, there are 70 coins. It is known that in the first box, $\\frac{5}{9}$ of the total number of coins are gold, and the rest are silver, in the second box, $\\frac{7}{17}$ of the number of coins are silver, and the rest are gold. How many coins are in each box",
      "answer": "36"
    },
    {
      "question": "4・183 Find the number of real number pairs $(a, b)$ that make the system of equations $\\left\\{\\begin{array}{l}a x+b y=1, \\\\ x^{2}+y^{2}=50 .\\end{array}\\right.$ have at least one solution, and all solutions are integer solutions.",
      "answer": "72"
    },
    {
      "question": "4. Among the integers from 1 to 2020, there are 1616 numbers that are not multiples of 5. Divide these 1616 numbers into several groups (the numbers in each group do not have to be the same), such that the difference (larger number minus smaller number) between any two numbers in each group is a prime number. What is the minimum number of groups needed?",
      "answer": "404"
    },
    {
      "question": "3. Calculate: $\\frac{1}{1+\\frac{2}{1+\\frac{3}{1+\\frac{4}{5}+1}+1}+1}$.",
      "answer": "\\frac{43}{114}"
    },
    {
      "question": "8. (15 points) A primary school is conducting a height statistics survey. There are 99 students whose height does not exceed $130 \\mathrm{~cm}$, with an average height of $122 \\mathrm{~cm}$. There are 72 students whose height is not less than $160 \\mathrm{~cm}$, with an average height of $163 \\mathrm{~cm}$. The average height of students whose height exceeds $130 \\mathrm{~cm}$ is $155 \\mathrm{~cm}$, and the average height of students whose height is below $160 \\mathrm{~cm}$ is $148 \\mathrm{~cm}$. How many students are there in total? $\\qquad$",
      "answer": "621"
    },
    {
      "question": "3.040. $\\frac{1-\\cos 4 \\alpha}{\\cos ^{-2} 2 \\alpha-1}+\\frac{1+\\cos 4 \\alpha}{\\sin ^{-2} 2 \\alpha-1}=2$.",
      "answer": "2"
    },
    {
      "question": "(Example 3 Given that the function $y=f(x)$ has an inverse function within its domain $(-\\infty, 0]$, and $f(x-1)=x^{2}-2 x$, find the value of $f^{-1}\\left(-\\frac{1}{2}\\right)$.",
      "answer": "-\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "Example 5. Expand the function\n\n$$\nf(z)=\\frac{z}{z^{2}-2 z-3}\n$$\n\ninto a Taylor series in the neighborhood of the point $z_{0}=0$ using expansion (12), and find the radius of convergence of the series.",
      "answer": "1"
    },
    {
      "question": "## Task B-2.1.\n\nFor which $n \\in \\mathbb{N}$ does the following hold\n\n$$\n1 i^{1}+2 i^{2}+3 i^{3}+4 i^{4}+\\cdots+n i^{n}=48+49 i, \\text { if } i=\\sqrt{-1} ?\n$$",
      "answer": "97"
    },
    {
      "question": "## 7. Gold Coins\n\nAnte, Branimir, Celestin, and Dubravko are sitting around a round table (in that order). Together, they have 1600 gold coins. First, Ante divides half of his gold coins into two equal parts and gives one part to his left neighbor and the other part to his right neighbor, while keeping the other half for himself. Then, Branimir does the same, followed by Celestin, and finally Dubravko. In the end, all four of them have an equal number of gold coins. How many gold coins did Branimir have at the beginning?\n\nResult: $\\quad 575$",
      "answer": "575"
    },
    {
      "question": "5. In $\\triangle A B C$, $\\angle B A C=90^{\\circ}, A C=A B=4$, point $D$ is inside $\\triangle A B C$, and $A D=\\sqrt{2}$. Then the minimum value of $B D+C D$ is $\\qquad$ .",
      "answer": "2\\sqrt{10}"
    },
    {
      "question": "Given an equilateral triangle $ABC$ with side length 4. Points $D$, $E$, and $F$ are on $BC$, $CA$, and $AB$ respectively, and $|AE| = |BF| = |CD| = 1$. Connect $AD$, $BE$, and $CF$, intersecting to form $\\triangle RQS$. Point $P$ moves within $\\triangle RQS$ and on its sides. The distances from $P$ to the three sides of $\\triangle ABC$ are denoted as $x$, $y$, and $z$ respectively.\n(1) Prove that when point $P$ is at the vertices of $\\triangle RQS$, the product $xyz$ has a minimum value; $\\square$\n(2) Find the minimum value of the product $xyz$.",
      "answer": "\\frac{648}{2197}\\sqrt{3}"
    },
    {
      "question": "7.1. A cyclist was initially traveling at a speed of 20 (km/h). After covering one-third of the distance, he glanced at his watch and decided to increase his speed by 20%. He continued the rest of the journey at the new speed. What was the cyclist's average speed?",
      "answer": "22.5"
    },
    {
      "question": "Example 2 Use $1 \\times 1, 2 \\times 2, 3 \\times 3$ tiles to cover a $23 \\times 23$ floor (no overlapping, no gaps), how many $1 \\times 1$ tiles are needed at least?",
      "answer": "1"
    },
    {
      "question": "12. The ordered pairs $(2011,2),(2010,3),(2009,4), \\ldots,(1008,1005),(1007,1006)$ are written from left to right on a blackboard. Every minute, Elizabeth selects a pair of adjacent pairs $\\left(x_{i}, y_{i}\\right)$ and $\\left(x_{j}, y_{j}\\right)$, with $\\left(x_{i}, y_{i}\\right)$ left of $\\left(x_{j}, y_{j}\\right)$, erases them, and writes $\\left(\\frac{x_{i} y_{i} x_{j}}{y_{j}}, \\frac{x_{i} y_{i} y_{j}}{x_{j}}\\right)$ in their place. Elizabeth continues this process until only one ordered pair remains. How many possible ordered pairs $(x, y)$ could appear on the blackboard after the process has come to a conclusion?",
      "answer": "504510"
    },
    {
      "question": "7.2. Find the GCD of all numbers obtained by all possible permutations of the digits of the number 202120222023\n\nSolution. By the divisibility rule, all these numbers are divisible by 9 (the sum of the digits is 18). A sufficient condition to prove that there are no other numbers is that the difference between any two such numbers is also divisible by the GCD. For example, 222222100032 - 222222100023 = 9 is divisible by 9 and cannot be greater than 9.",
      "answer": "9"
    },
    {
      "question": "## Task 4.\n\nDetermine all natural numbers $n$ for which there exist distinct divisors $a$ and $b$ of $n$ such that there are no other divisors of $n$ between them and that\n\n$$\nn=a^{2}-b\n$$",
      "answer": "8"
    },
    {
      "question": "3 [ Volume of a Tetrahedron and Pyramid $\\quad$]\n\nConsider a rectangle $A B C D$, where $A B=2, B C=3$. Segment $K M$ is parallel to $A B$ (see figure), located 1 unit away from the plane $A B C D$ and $K M=5$. Find the volume of the polyhedron $A B C D K M$.\n\n#",
      "answer": "\\frac{9}{2}"
    },
    {
      "question": "Example 4 Find the largest constant $c$ such that for all real numbers $x, y$ satisfying $x>0, y>0, x^{2}+y^{2}=1$, we always have $x^{6}+y^{6} \\geqslant c x y$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "5. (3 points) In a regular triangular pyramid (not a regular tetrahedron), the area of the base is half the area of a lateral face. The height of the pyramid is 70 cm.\n\nConsider the following (infinite) sequence of spheres. Let \\( S_{1} \\) be the inscribed sphere of this pyramid. Then \\( S_{2} \\) is the sphere that is tangent to the lateral faces of the pyramid and to the sphere \\( S_{1} \\); \\( S_{3} \\) is the sphere that is tangent to the lateral faces of the pyramid and to the sphere \\( S_{2} \\), not equal to \\( S_{1} \\), and so on; \\( S_{n+1} \\) is the sphere that is tangent to the lateral faces of the pyramid and to the sphere \\( S_{n} \\), not equal to \\( S_{n-1} \\).\n\nFind the total volume of all these spheres.",
      "answer": "\\frac{686}{327}\\pi^3"
    },
    {
      "question": "Beatriz adores odd numbers. How many numbers between 0 and 1000 can she write using only odd digits?",
      "answer": "155"
    },
    {
      "question": "1. (8 points) The calculation result of the expression $2016 \\times\\left(\\frac{8}{7 \\times 9}-\\frac{1}{8}\\right)$ is",
      "answer": "4"
    },
    {
      "question": "[ Numerical inequalities. Comparing numbers.]\n\nWhich number is greater: $100^{100}$ or $50^{50} \\cdot 150^{50}$ ?",
      "answer": "100^{100}"
    },
    {
      "question": "Example 3.28. Compute the double integral $\\iint_{D}(x+2 y) d x d y$, where the region of integration is bounded by the parabolas $y=x-x^{2}$, $y=1-x^{2}$, and the $O y$ axis (Fig. 48).\n\n![](https://cdn.mathpix.com/cropped/2024_05_22_e915d089cc29fd681be5g-177.jpg?height=450&width=534&top_left_y=175&top_left_x=441)\n\nFig. 48",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "3.2. A chocolate bar weighed 400 g and cost 150 rubles. Recently, to save money, the manufacturer reduced the weight of the bar to 300 g, and increased its price to 180 rubles. By what percentage did the manufacturer's revenue increase?",
      "answer": "60"
    },
    {
      "question": "Problem 6. Calculate $2 \\operatorname{arctg} 5+\\arcsin \\frac{5}{13}$.",
      "answer": "\\pi"
    },
    {
      "question": "A $20 \\times 20 \\times 20$ cube is divided into 8000 unit cubes. A number is written in each unit cube. In each row and in each column of 20 small cubes, parallel to one of the cube's edges, the sum of the numbers is 1. In one of the small cubes, the number written is 10. Through this small cube pass three layers $1 \\times 20 \\times 20$ parallel to the faces of the cube. Find the sum of all the numbers outside these three layers.\n\n## - Solution of the exercises -\n\n untranslated portion: \n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。\n\n---\n\nA $20 \\times 20 \\times 20$ cube is divided into 8000 unit cubes. A number is written in each unit cube. In each row and in each column of 20 small cubes, parallel to one of the cube's edges, the sum of the numbers is 1. In one of the small cubes, the number written is 10. Through this small cube pass three layers $1 \\times 20 \\times 20$ parallel to the faces of the cube. Find the sum of all the numbers outside these three layers.\n\n## - Solution of the exercises -",
      "answer": "333"
    },
    {
      "question": "11. For a positive integer $n$, select several positive divisors of $n$ such that these divisors do not have an integer multiple relationship with each other. Let the maximum number of divisors that can be selected be $f(n)$, then $f\\left(2^{2020} \\times 2020\\right)+f\\left(3^{2021} \\times 2021\\right)+f\\left(5^{2022} \\times 2022\\right)=$ $\\qquad$ .",
      "answer": "16"
    },
    {
      "question": "2. Find all natural numbers $n$ for which the value of the expression $\\left[\\frac{n^{2}}{4}\\right]$ is a square of an integer.\n\nNOTE. $\\mathrm{S}[x]$ denotes the integer part of the number $x$, which is the greatest integer less than or equal to $x$.",
      "answer": "1"
    },
    {
      "question": "$4 \\cdot 34$ An integer has exactly 4 prime factors, the sum of whose squares is 476. Find this integer.\n(China National Training Team Practice Question, 1990)",
      "answer": "1989"
    },
    {
      "question": "[ Arithmetic. Mental calculation, etc.]\n\nOn the mailbox, it is written: \"Mail is collected five times a day from 7 to 19 hours.\" Indeed, the first time the postman collects the mail is at 7 in the morning, and the last time is at 7 in the evening. At what intervals are the letters taken out of the box?",
      "answer": "3"
    },
    {
      "question": "131. A natural number $n$ is the product of four consecutive natural numbers, each greater than 5. Determine the maximum number of possible last digits of $n$, given that its last digit is not 0.",
      "answer": "4"
    },
    {
      "question": "1. (8 points) Calculate: $6 \\times\\left(\\frac{1}{2}-\\frac{1}{3}\\right)+12 \\times\\left(\\frac{1}{3}+\\frac{1}{4}\\right)+19-33+21-7+22=$ $\\qquad$",
      "answer": "30"
    },
    {
      "question": "I2.2 In the trapezium $A B C D, A B / / D C . A C$ and $B D$ intersect at $O$. The areas of triangles $A O B$ and $C O D$ are $P$ and 25 respectively. Given that the area of the trapezium is $Q$, find the value of $Q$.",
      "answer": "81"
    },
    {
      "question": "## Task B-4.6.\n\nIn an isosceles triangle with a base length of $14 \\mathrm{~cm}$ and legs of length $25 \\mathrm{~cm}$, a circle is inscribed. Above it, another circle is inscribed which touches both legs and the first circle, above this one a third circle is inscribed, and so on, continuing the process of inscribing circles. If all the inscribed circles are painted with a certain color, what part of the triangle's area will remain uncolored?",
      "answer": "168-\\frac{1008}{25}\\pi"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0}\\left(\\tan\\left(\\frac{\\pi}{4}-x\\right)\\right)^{\\left(e^{x}-1\\right) / x}$",
      "answer": "1"
    },
    {
      "question": "76(1010). A collective farm team must harvest a potato crop by a certain deadline. After 60% of the potatoes were harvested, a combine was sent to assist the team, which reduced the harvesting time by 5 days. How many days would it have taken to harvest the potatoes without the combine's help, if it is known that the combine could have completed the entire job 8 days faster than the team?",
      "answer": "20"
    },
    {
      "question": "20. Xiaoling, while calculating division, wrote the divisor 65 as 56, resulting in a quotient of 13 and a remainder of 52. The correct quotient should be $\\qquad$.",
      "answer": "12"
    },
    {
      "question": "2. A clock takes 12 seconds to strike 7 times, how many seconds does it take to strike 4 times?",
      "answer": "6"
    },
    {
      "question": "84. As shown in the figure, Jia and Yi are two cylindrical containers, with the ratio of their base areas being $2: 3$. In the Jia container, there is an iron ball with a volume of 30 cubic centimeters, at which point the water levels in the two containers differ by $x$ centimeters; if the iron ball is moved from the Jia container to the Yi container, the water levels in the two containers still differ by $x$ centimeters. If the base area of the Jia container is 25 square centimeters, $x=$ $\\qquad$ cm.",
      "answer": "1"
    },
    {
      "question": "Three. (This question is worth 50 points) Given 10 points in space, where no four points lie on the same plane. Connect some of the points with line segments. If the resulting figure contains no triangles and no spatial quadrilaterals, determine the maximum number of line segments that can be drawn.\n\nConnect some of the points with line segments, if the resulting figure contains no triangles and no spatial quadrilaterals, determine the maximum number of line segments that can be drawn.",
      "answer": "15"
    },
    {
      "question": "Task B-4.3. Find all solutions to the equation\n\n$$\n\\frac{1}{\\binom{4}{x}}-\\frac{1}{\\binom{5}{x}}=\\frac{1}{\\binom{6}{x}}\n$$",
      "answer": "2"
    },
    {
      "question": "Example 6 Find all prime numbers $p$ such that $4 p^{2}+1$ and $6 p^{2}+1$ are also prime.",
      "answer": "5"
    },
    {
      "question": "5. Solve the equation $\\frac{\\sqrt{(-x)^{2}}+(\\sqrt{-x})^{2}}{x^{2}+(-x)^{2}}=\\frac{1}{2016}$ ANSWER: $x=-2016$.",
      "answer": "-2016"
    },
    {
      "question": "6.2. On the plate, there were 15 doughnuts. Karlson took three times more doughnuts than Little Man, and Little Man's dog Bibbo took three times fewer than Little Man. How many doughnuts are left on the plate? Explain your answer.",
      "answer": "2"
    },
    {
      "question": "11. Let $a \\in \\mathbf{R}$, the equation ||$x-a|-a|=2$ has exactly three distinct roots, then $a=$",
      "answer": "2"
    },
    {
      "question": "20. As shown in the figure, with the three sides of the isosceles right triangle $A B C$ as sides, squares are constructed outward. Given that the area of triangle $A B C$ is 4, then the area of the hexagon $E F G H I J$ formed by the vertices of the squares is $\\qquad$",
      "answer": "48"
    },
    {
      "question": "11. In the arithmetic expression below, each square represents a digit. Question: What is the sum of the 6 digits in the squares?\n$$\n\\begin{array}{r}\n\\square \\square \\square \\\\\n+\\square \\square \\square \\\\\n\\hline 1991\n\\end{array}\n$$",
      "answer": "47"
    },
    {
      "question": "[ Ratio of areas of similar triangles ] Law of Sines\n\nIn an acute-angled triangle $A B C$, heights $A P$ and $C Q$ are dropped from vertices $A$ and $C$ to sides $B C$ and $A B$. It is known that the area of triangle $A B C$ is 18, the area of triangle $B P Q$ is 2, and $P Q=2 \\sqrt{2}$. Find the radius of the circumcircle of triangle $A B C$.",
      "answer": "\\frac{9}{2}"
    },
    {
      "question": "Booin d.A.\n\nGiven two sequences: $2,4,8,16,14,10,2$ and 3, 6, 12. In each of them, each number is obtained from the previous one according to the same rule.\n\na) Find this rule.\n\nb) Find all natural numbers that transform into themselves (according to this rule).\n\nc) Prove that the number $2^{1991}$ will become a single-digit number after several transformations.",
      "answer": "18"
    },
    {
      "question": "G1.3 Determine the remainder of the 81 -digit integer $\\overline{111 \\cdots 1}$ divided by 81 .",
      "answer": "0"
    },
    {
      "question": "14. A, B, C, and D are competing in a table tennis tournament, where every two individuals play one match against each other. As a result, A defeated D, and A, B, and C have the same number of wins. How many matches did D win?",
      "answer": "0"
    },
    {
      "question": "Task 2. Angelica wants to choose a three-digit code for her suitcase. To make it easier to remember, Angelica wants all the digits in her code to be in non-decreasing order. How many different options does Angelica have for choosing the code?",
      "answer": "220"
    },
    {
      "question": "Task 1. Represent in the form of an irreducible fraction\n\n$$\n6 \\frac{7}{2015} \\times 4 \\frac{5}{2016}-7 \\frac{2008}{2015} \\times 2 \\frac{2011}{2016}-7 \\times \\frac{7}{2015}\n$$",
      "answer": "\\frac{5}{144}"
    },
    {
      "question": "Exercise 3. Let $x, y$ and $z$ be three real numbers such that $x^{2}+y^{2}+z^{2}=1$. Find the minimum and maximum possible values of the real number $x y+y z-z x$.",
      "answer": "-1"
    },
    {
      "question": "2k* Given that the base of the tetrahedron $S-ABC$ is an equilateral triangle, the projection of point $A$ on the side face $SBC$ is $H$, which is the orthocenter of $\\triangle SBC$, the plane angle of the dihedral angle $H-AB-C$ is $30^{\\circ}$, and $SA=2\\sqrt{3}$. Then, the volume of the tetrahedron $S-ABC$ is $\\qquad$ .",
      "answer": "\\frac{9}{4}\\sqrt{3}"
    },
    {
      "question": "8. Priti is learning a new language called Tedio. During her one hour lesson, which started at midday, she looks at the clock and notices that the hour hand and the minute hand make exactly the same angle with the vertical, as shown in the diagram. How many whole seconds remain until the end of the lesson?",
      "answer": "276"
    },
    {
      "question": "4. In the regular tetrahedron \\(ABCD\\), consider point \\(M\\) - the midpoint of \\([AB]\\) and \\(N\\) - the midpoint of \\([AC]\\). If the edge of the tetrahedron is \\(8 \\, \\text{cm}\\), determine:\n\n(3 p) a) The perimeter of quadrilateral \\(BCNM\\)\n\n(2 p) b) The measure of the angle formed by line \\(MN\\) with \\(AB\\)\n\n(2 p) c) The distance from point \\(C\\) to the plane \\((ABD)\\).\n\nNote: All subtasks are mandatory,\n\nTime allowed: 3 hours\n\nEach subtask is graded from 0 to 7 points.\n\n## GRADING AND SCORING CRITERIA FOR THE MATHEMATICS OLYMPIAD LOCAL PHASE",
      "answer": "20"
    },
    {
      "question": "## Task 5 - 040935\n\nAt a puzzle afternoon, the best young mathematician in the class is given the task of guessing a certain real number. His classmates take turns naming properties of this number:\n\nKlaus: \"The number is divisible by 4 without a remainder.\"\n\nInge: \"The number is the radius of a circle whose circumference is 2.\"\n\nGünter: \"The number is less than 3.\"\n\nMonika: \"The number is the length of the diagonal of a square whose side has a length of 2.\"\n\nBärbel: \"The number is irrational.\"\n\nPeter: \"The number is the area of an equilateral triangle whose side has a length of 2.\" Furthermore, he learns that of the students Klaus and Inge, Günter and Monika, and Bärbel and Peter, exactly one of each pair spoke the truth.\n\nWhat is the number?",
      "answer": "\\frac{1}{\\pi}"
    },
    {
      "question": "Example 3 Let the complex numbers $z_{1}, z_{2}, z_{3}$ satisfy:\n$$\n\\begin{array}{l}\n\\left|z_{1}\\right|=\\left|z_{2}\\right|=\\sqrt{3}\\left|z_{3}\\right|, z_{1}+z_{3}=z_{2} . \\\\\n\\text { Find the value of } \\frac{z_{1} z_{2}}{z_{3}} \\text { . }\n\\end{array}\n$$",
      "answer": "-5"
    },
    {
      "question": "A fly and $\\mathrm{k}$ spiders move on a $2019 \\times 2019$ grid. On their turn, the fly can move 1 square and the $k$ spiders can each move 1 square. What is the minimal $k$ for which the spiders are sure to catch the fly.",
      "answer": "2"
    },
    {
      "question": "Example 2 The equation $x^{10}+(13 x-1)^{10}=0$ has 10 complex roots $r_{1}, \\overline{r_{1}}, \\overline{r_{2}}, \\overline{r_{2}}, \\overline{r_{3}}, \\overline{r_{3}}, \\overline{r_{4}}, \\overline{r_{4}}, \\overline{r_{5}}, \\overline{r_{5}}$, where $\\overline{r_{i}}$ is the conjugate of $r_{i}$ $(i=1,2,3,4,5)$, find the value of $\\frac{1}{r_{1} \\overline{r_{1}}}+\\frac{1}{r_{2} \\overline{r_{2}}}+\\frac{1}{r_{3}} \\overline{r_{3}}+\\frac{1}{r_{4} \\overline{r_{4}}}+\\frac{1}{r_{5} r_{5}}$.",
      "answer": "850"
    },
    {
      "question": "2. [5 points] Consider all possible tetrahedra $ABCD$ in which $AB=2$, $AC=CB=7$, and $AD=DB=8$. Each such tetrahedron is inscribed in a cylinder such that all vertices lie on its lateral surface, and the edge $CD$ is parallel to the axis of the cylinder. Choose the tetrahedron for which the radius of the cylinder is the smallest among those obtained. What values can the length of $CD$ take in such a tetrahedron?",
      "answer": "\\sqrt{62}\\\\sqrt{47}"
    },
    {
      "question": "Problem 9.4. A line $\\ell$ is drawn through vertex $A$ of rectangle $ABCD$, as shown in the figure. Perpendiculars $BX$ and $DY$ are dropped from points $B$ and $D$ to line $\\ell$. Find the length of segment $XY$, given that $BX=4$, $DY=10$, and $BC=2AB$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_3890a5f9667fd1ab5160g-30.jpg?height=426&width=515&top_left_y=1495&top_left_x=469)",
      "answer": "13"
    },
    {
      "question": "$7 \\cdot 1$ Given $n$ points on a plane, any 3 of which are the 3 vertices of a right-angled triangle, find the maximum value of $n$.\n\n untranslated text retained as requested.",
      "answer": "4"
    },
    {
      "question": "4. First solution. Let $a_{1}, a_{2}, \\ldots, a_{15}$ be the days of the month that were sunny. Then Andrey Stepanovich will not drink a single drop from the 1st to the $a_{1}$-th day, he will drink one drop per day from the $a_{1}$-th day to the $a_{2}$-th day (including the $a_{1}$-th day but not the $a_{2}$-th day), and so on. In total, he will drink\n\n$$\n\\begin{aligned}\n& 1 \\cdot\\left(a_{2}-a_{1}\\right)+2 \\cdot\\left(a_{3}-a_{2}\\right)+\\ldots \\\\\n& \\quad \\ldots+14 \\cdot\\left(a_{15}-a_{14}\\right)+15 \\cdot\\left(30-a_{15}+1\\right)= \\\\\n& \\quad=15 \\cdot 31-\\left(a_{1}+a_{2}+\\ldots+a_{15}\\right)\n\\end{aligned}\n$$\n\nIvan Petrovich, on the other hand, will drink a number of drops equal to the sum of the numbers of all days except $a_{i}$:\n\n$$\n\\begin{aligned}\n(1+2+\\ldots+30)-\\left(a_{1}+a_{2}+\\ldots\\right. & \\left.+a_{15}\\right) \\\\\n& =15 \\cdot 31-\\left(a_{1}+a_{2}+\\ldots+a_{15}\\right)\n\\end{aligned}\n$$",
      "answer": "120"
    },
    {
      "question": "C4. An integer container $(x, y, z)$ is a rectangular prism with positive integer side lengths $x, y, z$, where $x \\leq y \\leq z$. A stick has $x=y=1$; a flat has $x=1$ and $y>1$; and a box has $x>1$. There are 5 integer containers with volume 30: one stick $(1,1,30)$, three flats $(1,2,15),(1$, $3,10),(1,5,6)$ and one box $(2,3,5)$.\na. How many sticks, flats and boxes are there among the integer containers with volume 36 ?\nb. How many flats and boxes are there among the integer containers with volume 210 ?\nc. Suppose $n=p_{1}^{e_{1}} \\cdots p_{k}^{e_{k}}$ has $k$ distinct prime factors $p_{1}, p_{2}, \\ldots, p_{k}$, each with integer exponent $e_{1} \\geq 1, e_{2} \\geq 1, \\ldots, e_{k} \\geq 1$ and $k \\geq 3$. How many boxes are there among the integer containers with volume $n$ ? Express your answer in terms of $e_{1}, e_{2}, \\ldots, e_{k}$. How many boxes with volume $n=8!$ are there?",
      "answer": "280"
    },
    {
      "question": "13. (5 points)\nThe distance from Earth to Planet Sadala is 80 light years (Note: A light year is a unit of length). Vegeta and Goku depart from Earth to Planet Sadala. Vegeta departs 1 day earlier than Goku. If Vegeta and Goku fly in a straight line, they can both fly 10 light years per day. How many days after Goku's departure will Vegeta be exactly in the middle of Goku and Planet Sadala?",
      "answer": "6"
    },
    {
      "question": "1. Compute the sum:\n\n$$\n\\sqrt{2018-\\sqrt{2019 \\cdot 2017}}+\\sqrt{2016-\\sqrt{2017 \\cdot 2015}}+\\cdots+\\sqrt{2-\\sqrt{3 \\cdot 1}}\n$$",
      "answer": "\\frac{\\sqrt{2019}-1}{\\sqrt{2}}"
    },
    {
      "question": "22 、A triangle with a perimeter of 36 has three sides that are all composite numbers, then the number of such triangles is. $\\qquad$",
      "answer": "9"
    },
    {
      "question": "9.1. At the school for slackers, a competition on cheating and giving hints was organized. It is known that $75 \\%$ of the students did not show up for the competition at all, and all the rest participated in at least one of the competitions. When the results were announced, it turned out that $10 \\%$ of all those who showed up participated in both competitions, and that those who participated in the hinting competition were one and a half times more than those who participated in the cheating competition. Find the smallest possible number of students in the school for slackers.",
      "answer": "200"
    },
    {
      "question": "8. Find the value of $\\frac{1}{3^{2}+1}+\\frac{1}{4^{2}+2}+\\frac{1}{5^{2}+3}+\\cdots$.",
      "answer": "\\frac{13}{36}"
    },
    {
      "question": "14. $[8]$ Danny has a set of 15 pool balls, numbered $1,2, \\ldots, 15$. In how many ways can he put the balls in 8 indistinguishable bins such that the sum of the numbers of the balls in each bin is 14,15 , or 16 ?",
      "answer": "122"
    },
    {
      "question": "3.337. $\\sin ^{2} 70^{\\circ} \\sin ^{2} 50^{\\circ} \\sin ^{2} 10^{\\circ}=\\frac{1}{64}$.",
      "answer": "\\frac{1}{64}"
    },
    {
      "question": "9. In the number table below, both the upper and lower rows of numbers are arithmetic sequences. For the corresponding numbers in the upper and lower rows, the difference between the larger number and the smaller number, the smallest is $\\qquad$ _",
      "answer": "4"
    },
    {
      "question": "Let $P$ be a regular polygon with 2026 sides. A diagonal of $P$ is said to be good if its endpoints divide the perimeter of $P$ into two parts, each having an odd number of sides of $P$. The sides of $P$ are also considered good. Suppose that $P$ has been subdivided into triangles by 2023 diagonals that do not intersect each other inside $P$. How many, at most, such a subdivision can contain isosceles triangles with two good sides?\n\n## Solutions to the exercises\n\nTranslate the above text into English, preserving the original text's line breaks and format, and output the translation directly.",
      "answer": "1013"
    },
    {
      "question": "9.5 In some cells of a $10 \\times 10$ table, crosses are placed such that each of them is the only one either in its row or in its column. What is the maximum number of crosses that can be in such a table? Justify your answer.",
      "answer": "18"
    },
    {
      "question": "4. There is a four-digit integer. By adding a decimal point in front of one of its digits, and then adding this number to the four-digit integer, the sum is 2000.81. Find this four-digit number.",
      "answer": "1981"
    },
    {
      "question": "5. (8 points) Xiao Bing participated in a strange math exam. There were a total of 100 questions, with 1 point for each correct answer, -3 points for each wrong answer, and -2 points for each unanswered question. $\\square$ It is known that Xiao Bing scored a total of 50 points. Then, the maximum number of questions Xiao Bing could have answered correctly is $\\qquad$.",
      "answer": "87"
    },
    {
      "question": "11. (15 points) Let the system of inequalities $\\left\\{\\begin{array}{l}x+y>0, \\\\ x-y<0\\end{array}\\right.$ represent the plane region $D$. The product of the distances from a moving point $P$ in region $D$ to the lines $x+y=0$ and $x-y=0$ is 2. The trajectory of point $P$ is denoted as curve $C$. A line $l$ passing through the point $F(2 \\sqrt{2}, 0)$ intersects curve $C$ at points $A$ and $B$. If the circle with diameter $A B$ is tangent to the $y$-axis, find the slope of line $l$.",
      "answer": "-\\sqrt{\\sqrt{2}-1}"
    },
    {
      "question": "For each positive integer $n$, define the set\n$$\nP_{n}=\\left\\{n^{k} \\mid k=0,1, \\cdots\\right\\} \\text {. }\n$$\n\nFor positive integers $a$, $b$, and $c$, if there exists a positive integer $m$ such that the numbers $a-1$, $ab-12$, and $abc-2015$ (not necessarily distinct) all belong to the set $P_{m}$, then the positive integer triplet $(a, b, c)$ is called \"lucky\". Find the number of all lucky positive integer triplets.\n（Supplied by He Yijie）",
      "answer": "25"
    },
    {
      "question": "Marta is playing a game where she guesses a five-digit number consisting of different digits. The course of the first three rounds is as follows:\n\n| 1st round | 2 | 6 | 1 | 3 | 8 |\n| :--- | :--- | :--- | :--- | :--- | :--- |\n| 2nd round | 4 | 1 | 9 | 6 | 2 |\n| 3rd round | 8 | 1 | 0 | 2 | 5 |\n\nThe color of the cell reveals something about the digit it contains:\n\n- a green cell means the digit is present in the guessed number and is in the correct position,\n- a yellow cell means the digit is present in the guessed number but in a different position,\n- a gray cell means the digit is not present in the guessed number.\n\nExplain whether Marta can or cannot guess the number with certainty in the next round.\n\n(J. Tkadlec)\n\nHint. Which digits can appear in the number?",
      "answer": "71284"
    },
    {
      "question": "What is the minimum length of the edge of a cube that can fit 2 unit-radius spheres, which do not have any common interior points?",
      "answer": "2+\\frac{2}{\\sqrt{3}}"
    },
    {
      "question": "\\section*{Problem 9}\n\nEvery city in a certain state is directly connected by air with at most three other cities in the state, but one can get from any city to any other city with at most one change of plane. What is the maximum possible number of cities?",
      "answer": "10"
    },
    {
      "question": "5. At each vertex of a regular $n$-gon, a number 1 or -1 is inscribed, with not all $n$ numbers being equal. The product of the numbers inscribed in any 3 consecutive vertices is -1. Determine the sum of all inscribed numbers for:\na) $n=6$;\nb) $n=2019$.\n\n## Fourth grade - B category",
      "answer": "2"
    },
    {
      "question": "4. Early in the morning, the pump was turned on and they started filling the pool. At 10 am, another pump was connected and by 12 pm the pool was half full. By 5 pm, the pool was full. What could be the latest time the first pump was turned on?",
      "answer": "7"
    },
    {
      "question": "6. In triangle $A B C$ with angle $\\angle B=120^{\\circ}$, the angle bisectors $A A_{1}, B B_{1}, C C_{1}$ are drawn. Find the degree measure of angle $C_{1} B_{1} A_{1}$.",
      "answer": "90"
    },
    {
      "question": "9-2. The school volleyball team played several matches. After they won another match, the share of victories increased by $1 / 6$. To increase the share of victories by another 1/6, the volleyball players had to win two more consecutive matches. What is the minimum number of victories the team needs to achieve to increase the share of wins by another 1/6?",
      "answer": "6"
    },
    {
      "question": "14. (15 points) In the following addition problem, different Chinese characters can represent the same digit. How many different equations satisfy the requirement?\n\n\n\nThe translation is provided while preserving the original text's line breaks and format.",
      "answer": "100"
    },
    {
      "question": "Example 5 (4th American Invitational Mathematics Examination) The increasing sequence $1,3,4,9,10,12,13, \\cdots$ consists of some positive integers, which are either powers of 3 or sums of several different powers of 3. Find the 100th term of this sequence.",
      "answer": "981"
    },
    {
      "question": "Example 8 Find the largest positive integer $x$, such that for every positive integer $y$, $x$ divides $7^{y}+12 y-1$.\n\n---\n\nThe translation maintains the original text's format and line breaks as requested.",
      "answer": "18"
    },
    {
      "question": "7. [5] Find $p$ so that $\\lim _{x \\rightarrow \\infty} x^{p}(\\sqrt[3]{x+1}+\\sqrt[3]{x-1}-2 \\sqrt[3]{x})$ is some non-zero real number.",
      "answer": "\\frac{5}{3}"
    },
    {
      "question": "1. As shown in the figure, divide a line segment of length 1 into segments $x$ and $y$. Then, bend the segment of length $x$ into a semicircle $A C B$, and fold the segment of length $y$ into three sides $(B D, D E, E A)$ of a rectangle $A B D E$ to form a closed \"curved polygon\" $A C B D E A$. The maximum value of the area of this curved polygon is . $\\qquad$",
      "answer": "\\frac{1}{2(\\pi+4)}"
    },
    {
      "question": "Example 1. Calculate the surface area of $z=\\frac{x y}{a}$, located inside the cylinder $x^{2}+y^{2}=a^{2}, a>0$.",
      "answer": "\\frac{2\\pi}{3}^{2}(2\\sqrt{2}-1)"
    },
    {
      "question": "13.354. When multiplying two numbers, one of which is 10 more than the other, a student made a mistake by reducing the tens digit of the product by 4. When dividing the obtained product by the smaller multiplier to check the answer, he got a quotient of 39 and a remainder of 22. Find the multipliers.",
      "answer": "3141"
    },
    {
      "question": "34. Find the smallest positive integer $n$ such that $n(n+1)(n+2)$ is divisible by 247 .",
      "answer": "37"
    },
    {
      "question": "16. (1982 American High School Mathematics Examination) Figure 12-6 is a partial map of a city. Small rectangles represent houses and buildings, and the spaces between them represent streets. There is a student who walks from point $A$ to point $B$ every morning along the streets shown in the figure, and he can only walk east or south. At each intersection, he has a probability of $\\frac{1}{2}$ of choosing to go east or south (each choice is independent of the others). What is the probability that the student will walk through point $C$?",
      "answer": "\\frac{21}{32}"
    },
    {
      "question": "8. Given that 5 students are proficient in Chinese, Mathematics, Physics, Chemistry, and History respectively, there are 5 test papers (one each for Chinese, Mathematics, Physics, Chemistry, and History). The teacher randomly distributes one test paper to each student. The probability that at least 4 students receive a test paper that does not match their proficient subject is $\\qquad$ .",
      "answer": "\\frac{89}{120}"
    },
    {
      "question": "5. In $\\triangle A B C$, if $\\overrightarrow{A B} \\cdot \\overrightarrow{A C}=7,|\\overrightarrow{A B}-\\overrightarrow{A C}|=6$, then the maximum value of the area of $\\triangle A B C$ is",
      "answer": "12"
    },
    {
      "question": "2. For the quadratic trinomial $p(x)=(a+1) x^{2}-(a+1) x+2022$, it is known that $-2022 \\leq p(x) \\leq 2022$ for $x \\in[0 ; 1]$. Find the greatest possible value of $a$.",
      "answer": "16175"
    },
    {
      "question": "1. To climb from the valley to the mountain top, one must walk 4 hours on the road, and then -4 hours on the path. On the mountain top, two fire-breathing dragons live. The first dragon spews fire for 1 hour, then sleeps for 17 hours, then spews fire for 1 hour again, and so on. The second dragon spews fire for 1 hour, then sleeps for 9 hours, then spews fire for 1 hour again, and so on. It is dangerous to walk on both the road and the path during the first dragon's fire-spewing, while it is dangerous to walk only on the path during the second dragon's fire-spewing. Both dragons start spewing fire simultaneously at midnight. Is it possible to safely climb from the valley to the mountain top and return back? (8 points)",
      "answer": "38"
    },
    {
      "question": "3-2. Two identical polygons were cut out of cardboard, aligned, and pierced with a pin at some point. When one of the polygons is rotated around this \"axis\" by $25^{\\circ} 30^{\\prime}$, it aligns again with the second polygon. What is the smallest possible number of sides of such polygons?",
      "answer": "240"
    },
    {
      "question": "4. If $a+b+c=1$, and $a, b, c \\in \\mathbf{R}^{+}$, then the minimum value of $\\sqrt{a^{2}+b^{2}}+\\sqrt{b^{2}+c^{2}}+\\sqrt{c^{2}+a^{2}}$ is $\\qquad$ .",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "99. Person A departs from point A to point B, while at the same time, person B departs from point B to point A. After 12 minutes, the two have passed their meeting point and are 100 meters apart. It is known that person A takes 20 minutes to complete the journey, and person B walks at a speed of 65 meters per minute. Therefore, the distance between points A and B is ______ meters. $\\qquad$",
      "answer": "1700"
    },
    {
      "question": "8. A certain intelligence station has four different passwords $A, B, C, D$, and uses one of them each week. Each week, one of the three passwords not used in the previous week is chosen with equal probability. If the first week uses password $A$, what is the probability that the seventh week also uses password $A$? (Express your answer as a simplified fraction)\n\nFill in the blank: $\\qquad$ .",
      "answer": "\\frac{61}{243}"
    },
    {
      "question": "16. Fill in each square in $\\square \\square+\\square \\square=\\square \\square \\square$ with a digit from $0, 1, 2, \\cdots \\cdots, 9$ (digits in the squares can be the same, and the leading digit of any number cannot be 0), so that the equation holds. \n\nThere are",
      "answer": "4860"
    },
    {
      "question": "51. In a box, there are 250 light bulbs, of which 100 are 100W, 50 are 60W, 50 are 25W, and 50 are 15W. Calculate the probability that the power of any randomly selected light bulb will not exceed 60W.",
      "answer": "\\frac{3}{5}"
    },
    {
      "question": "5. Let $C D$ be a chord of a circle $\\Gamma_{1}$ and $A B$ a diameter of $\\Gamma_{1}$ perpendicular to $C D$ at $N$ with $A N>N B$. A circle $\\Gamma_{2}$ centred at $C$ with radius $C N$ intersects $\\Gamma_{1}$ at points $P$ and $Q$, and the segments $P Q$ and $C D$ intersect at $M$. Given that the radii of $\\Gamma_{1}$ and $\\Gamma_{2}$ are 61 and 60 respectively, find the length of $A M$.",
      "answer": "78"
    },
    {
      "question": "7. (10 points) The calculation result of the expression $1007 \\times \\frac{1 \\frac{3}{4} \\div \\frac{3}{4}+3 \\div 2 \\frac{1}{4}+\\frac{1}{3}}{(1+2+3+4+5) \\times 5-22} \\div 19$ is",
      "answer": "4"
    },
    {
      "question": "1. (7 points) 1. Calculate: $\\left(7 \\frac{4}{25}+8.6\\right) \\div\\left[\\left(4 \\frac{5}{7}-0.005 \\times 900\\right) \\div \\frac{6}{7}\\right]=$",
      "answer": "63.04"
    },
    {
      "question": "19. Given an arbitrary triangle. On each side of the triangle, 10 points are marked. Each vertex of the triangle is connected by segments to all the marked points on the opposite side. Into what maximum number of parts could the segments divide the triangle?",
      "answer": "331"
    },
    {
      "question": "## Task A-2.5.\n\nIn a coordinate system, all integer points $(x, y)$ are marked where $1 \\leqslant x \\leqslant$ 200 and $1 \\leqslant y \\leqslant 100$, a total of 20000 points. How many line segments of length $\\sqrt{5}$ have their endpoints at the marked points?",
      "answer": "78208"
    },
    {
      "question": "10. The graph on the right shows the fuel consumption per 100 kilometers for four different types of cars: $U, V, W, X$. If each car has 50 liters of fuel, what is the maximum total distance that these four cars can travel in kilometers?",
      "answer": "1950"
    },
    {
      "question": "A10. If you open a book and add the numbers that mark the left and right page, you get 21. What number do you get if you multiply the same two numbers?\n(A) 100\n(B) 121\n(C) 420\n(D) 110\n(E) 200\n\n# 4th Regional Mathematics Knowledge Competition for Vocational School Students\n\nMarch 31, 2004\n\n## II. Part: LONGER PROBLEMS\n\nInstruction: Solve problems B1 to B4 of the second part on the provided paper, where you write down the complete calculations. Read each problem carefully and answer the questions asked. The solution to each of these problems will be rated from 0 to 5 points.\n\nNote that you have 90 minutes to solve the problems of the first and second parts.",
      "answer": "110"
    },
    {
      "question": "1. In the wallet, there are six different coins of cents and euros (5 cents, 10 cents, 20 cents, 50 cents, $1 €$ and $2 €$). Three coins are drawn simultaneously and without looking. In how many different ways can the coins be drawn so that their total value is greater than 85 cents?",
      "answer": "16"
    },
    {
      "question": "3. Aroon's PIN has four digits. When the first digit (reading from the left) is moved to the end of the PIN, the resulting integer is 6 less than 3 times Aroon's PIN. What could Aroon's PIN be?",
      "answer": "2856"
    },
    {
      "question": "11. (12 points) Fill in the blanks with numbers $1-6$, so that the numbers in each row and column do not repeat, the numbers on both sides of the black dot are in a two-fold relationship, and the numbers on both sides of the white dot have a difference of 1. Then the five-digit number formed by the first 5 numbers from left to right in the fourth row is $\\qquad$\n\nFill in the blanks with numbers $1-6$, so that the numbers in each row and column do not repeat, the numbers on both sides of the black dot are in a two-fold relationship, and the numbers on both sides of the white dot have a difference of 1. Then the five-digit number formed by the first 5 numbers from left to right in the fourth row is $\\qquad$",
      "answer": "21436"
    },
    {
      "question": "10. Let the integer $n \\geqslant 4, A=\\left\\{a_{1}, a_{2}, \\cdots, a_{n}\\right\\}$ be a set of integers, satisfying $2023 \\in A$, and the absolute difference between any two elements of $A$ is a prime number. Then the number of such sets $A$ is $\\qquad$.",
      "answer": "4"
    },
    {
      "question": "49. A ship sails downstream from Port A to Port B, and upon arrival, immediately returns upstream to Port A, taking a total of 12 hours. It is known that the ship travels 16 kilometers more per hour downstream than upstream, and it also travels 80 kilometers more in the first 6 hours than in the last 6 hours. Therefore, the distance between Port A and Port B is $\\qquad$ kilometers.",
      "answer": "280"
    },
    {
      "question": "9. Two people, A and B, agree to meet at a certain place within 10 days. They agree that the one who arrives first will wait for the other, but can leave after waiting for 3 days. If they are equally likely to arrive at the destination within the limit, then the probability that these two people will meet is $\\qquad$ .",
      "answer": "\\frac{51}{100}"
    },
    {
      "question": "37. Find the remainder when $1^{2017}+2^{2017}+3^{2017}+4^{2017}+5^{2017}$ is divided by 5. (where $a^{2017}$ denotes 2017 instances of $a$ multiplied together)",
      "answer": "0"
    },
    {
      "question": "7. Let $A B C$ be an obtuse triangle with circumcenter $O$ such that $\\angle A B C=15^{\\circ}$ and $\\angle B A C>90^{\\circ}$. Suppose that $A O$ meets $B C$ at $D$, and that $O D^{2}+O C \\cdot D C=O C^{2}$. Find $\\angle C$.",
      "answer": "35"
    },
    {
      "question": "1. Calculate $2015-5 \\cdot(25 \\cdot 13+13+3 \\cdot 25-10)$.",
      "answer": "0"
    },
    {
      "question": "6.31 Team A and Team B each send out 7 players to participate in a Go relay match according to a pre-arranged order. Both sides start with the No. 1 player competing, the loser is eliminated, and the winner then competes with the No. 2 player from the losing side, $\\cdots$, until all players from one side are eliminated, and the other side wins, forming a match process. How many different possible match processes are there?",
      "answer": "3432"
    },
    {
      "question": "Task 2. Lёnya has cards with digits from 1 to 7. How many ways are there to glue them into two three-digit numbers (one card will not be used) so that their product is divisible by 81, and their sum is divisible by 9?",
      "answer": "36"
    },
    {
      "question": "Example 2 Given that $x, y$ are both in the interval $(-2,2)$, and $x y=-1$, then what is the minimum value of the function $u=\\frac{4}{4-x^{2}}+\\frac{9}{9-y^{2}}$?",
      "answer": "\\frac{12}{5}"
    },
    {
      "question": "5. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies $a_{1}=a>2, a_{2017}=$ 2017, and for any positive integer $n, a_{n+1}=a_{n}^{2}-2$. Then $\\left[\\frac{\\sqrt{a-2}}{10^{6}} a_{1} a_{2} \\cdots a_{2017}\\right]=$ $\\qquad$, where $[x]$ denotes the greatest integer not exceeding the real number $x$.",
      "answer": "2"
    },
    {
      "question": "\\section*{Problem 1 - V01001}\n\nThe digit sum of a two-digit number is 9. If you multiply the number by 5 and subtract 9 from the product, you get a two-digit number with the same digits in reverse order. What is the two-digit number?",
      "answer": "18"
    },
    {
      "question": "Problem 2. Anya, Borya, Vika, and Gena are on duty at school for 20 days. It is known that exactly three of them are on duty each day. Anya was on duty 15 times, Borya - 14 times, Vika - 18 times. How many times was Gena on duty",
      "answer": "13"
    },
    {
      "question": "8. Let $x_{i} \\in\\{1,2, \\cdots, n\\}, i=1,2, \\cdots, n$ satisfy $\\sum_{i=1}^{n} x_{i}=\\frac{n(n+1)}{2}, x_{1} x_{2} \\cdots x_{n}=n$!. Then, the largest positive integer $n$ for which $x_{1}, x_{2}, \\cdots, x_{n}$ must be a permutation of $1,2, \\cdots, n$ is $\\qquad$.",
      "answer": "8"
    },
    {
      "question": "G3.1 Simplify $\\sqrt{3+\\sqrt{5}}+\\sqrt{3-\\sqrt{5}}$.",
      "answer": "\\sqrt{10}"
    },
    {
      "question": "## Task B-1.6.\n\nA student has passed 31 exams over the course of a five-year study. Each year, he took more exams than the previous year, and in the fifth year, he took three times as many exams as in the first year. How many exams did the student take in the fourth year?",
      "answer": "8"
    },
    {
      "question": "Task 1. In the first week, a squirrel collected 84 nuts, in the second week 96 nuts, and in the third week 65 nuts. Another squirrel in the fourth week collected three times more nuts than the first squirrel collected in the second and third weeks combined. How many nuts did the two squirrels collect in total over the four weeks?",
      "answer": "728"
    },
    {
      "question": "Let's determine the largest of the $A$ numbers for which the following statement holds. No matter how we write down the first 100 positive integers, there will always be ten consecutive numbers in the sequence whose sum is at least $A$.",
      "answer": "505"
    },
    {
      "question": "# Problem 3.\n\nB-1\n\nAt the time when a lion cub, located 6 minutes away, set off for a drink, the second, having already quenched its thirst, headed back along the same path at 1.5 times the speed of the first. At the same time, a tortoise, located 32 minutes away, set off along the same path for a drink. After some time, the first lion cub stepped on the tortoise, and some time later, the second lion cub did the same. 28 minutes and 48 seconds after the second incident, the tortoise reached the water hole. How many minutes passed between the two incidents, given that all three moved at constant speeds?",
      "answer": "2.4"
    },
    {
      "question": "8. A mob of 2015 kangaroos contains only red and grey kangaroos. One grey kangaroo is taller than exactly one red kangaroo, one grey kangaroo is taller than exactly three red kangaroos, one grey kangaroo is taller than exactly five red kangaroos and so on with each successive grey kangaroo being taller than exactly two more red kangaroos than the previous grey kangaroo. The final grey kangaroo is taller than all the red kangaroos. How many grey kangaroos are in the mob?",
      "answer": "672"
    },
    {
      "question": "2. (8 points) There is a special calculator. When a number is input, the calculator first multiplies this number by 3, then reverses the digits of the result, and finally adds 2 to display the final result. Xiao Ming input a four-digit number, and the displayed result was 2015. What is the four-digit number that Xiao Ming input? $\\qquad$ .",
      "answer": "1034"
    },
    {
      "question": "Twelve points are marked on a sheet of graph paper, as shown in the figure. What is the maximum number of squares that can be formed by connecting four of these points?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_8935ac0233b397f8e8f1g-2.jpg?height=320&width=326&top_left_y=768&top_left_x=1393)",
      "answer": "11"
    },
    {
      "question": "Subject 2. Solve in integers the equation: $2^{x+1}=x^{2}+x+1$.\n\nMathematical Gazette 1987",
      "answer": "1"
    },
    {
      "question": "I1.2 If $a$ copies of a right-angled isosceles triangle with hypotenuse $\\sqrt{2} \\mathrm{~cm}$ can be assembled to form a trapezium with perimeter equal to $b \\mathrm{~cm}$, find the least possible value of $b$. (give the answer in surd form).",
      "answer": "2+4\\sqrt{2}"
    },
    {
      "question": "G5.4 If $\\cos ^{6} \\theta+\\sin ^{6} \\theta=0.4$ and $d=2+5 \\cos ^{2} \\theta \\sin ^{2} \\theta$, find the value of $d$.",
      "answer": "3"
    },
    {
      "question": "The bases of the trapezoid are 8 and 2. The angles adjacent to the larger base are each $45^{\\circ}$. Find the volume of the solid formed by rotating the trapezoid around the larger base.",
      "answer": "36\\pi"
    },
    {
      "question": "7.2. A bus with programmers left Novosibirsk for Pavlodar. When it had traveled 70 km, Pavel Viktorovich set off from Novosibirsk in a car along the same route, and caught up with the programmers in Karasuk. After that, Pavel drove another 40 km, while the bus traveled only 20 km in the same time. Find the distance from Novosibirsk to Karasuk, if both the car and the bus traveled at constant speeds. (Provide a complete solution, not just the answer.)",
      "answer": "140"
    },
    {
      "question": "G4.2 If $x=\\frac{\\sqrt{5}+1}{2}$ and $y=\\frac{\\sqrt{5}-1}{2}$, determine the value of $x^{3} y+2 x^{2} y^{2}+x y^{3}$.",
      "answer": "5"
    },
    {
      "question": "25. The area of a square is 40 square centimeters, then the area of its circumscribed circle is $\\qquad$ square centimeters. ( $\\pi$ is taken as 3.14)",
      "answer": "62.8"
    },
    {
      "question": "133. Digits and Prime Numbers. Using each of the nine digits once and only once, form prime numbers (numbers that do not divide evenly by any integer except 1 and themselves) whose sum is the smallest.\n\nFor example, four prime numbers\n\n61\n\n+283\n\n47\n\n59\n\n450\n\ncontain all nine digits once and only once, and their sum is 450, but it can be significantly reduced. This is a very simple puzzle.",
      "answer": "207"
    },
    {
      "question": "II. (40 points) Let $p$ be a prime number, and the sequence $\\left\\{a_{n}\\right\\}(n \\geqslant 0)$ satisfies $a_{0}=0, a_{1}=1$, and for any non-negative integer $n$, $a_{n+2}=2 a_{n+1}-p a_{n}$. If -1 is a term in the sequence $\\left\\{a_{n}\\right\\}$, find all possible values of $p$.\n\n保留了原文的换行和格式。",
      "answer": "5"
    },
    {
      "question": "Four consecutive integers have a product that is a perfect square. What are these numbers?",
      "answer": "0"
    },
    {
      "question": "We drew a 13 cm and a $35 \\mathrm{~cm}$ side square on a grid paper, with each side lying on a grid line of the paper, and the distance between the lines is $1 \\mathrm{~cm}$ in both directions. We want to cut these two squares into pieces so that we can form three new squares with side lengths of 17, 23, and $24 \\mathrm{~cm}$. The cutting must be done along the grid lines. Provide a dissection that results in as few pieces as possible. (For example, it is sufficient to submit a $1: 4$ scale drawing of the dissection and the reassembly, indicating the number of pieces and briefly explaining why the given solution is better than other attempts.)\n\n\n\n\n\nTranslating the text as requested, while preserving the original formatting and line breaks.",
      "answer": "7"
    },
    {
      "question": "4. Let $A=\\{(\\alpha, \\beta, \\gamma) \\mid \\alpha+\\beta \\sin x+\\gamma \\cos x=0, x \\in(-\\infty,+\\infty)\\}$, then the number of elements in the set $A$ is $m=$\n$\\qquad$ .",
      "answer": "1"
    },
    {
      "question": "329. Find the value of the arbitrary function $f(x)=\\sin ^{4} x-$ $-\\cos ^{4} x$ at $x=\\pi / 12$.",
      "answer": "1"
    },
    {
      "question": "## Problem 2\n\nFind all real roots of the quartic $\\mathrm{x}^{4}-(2 \\mathrm{~N}+1) \\mathrm{x}^{2}-x+\\mathrm{N}^{2}+N-1=0$ correct to 4 decimal places, where $\\mathrm{N}=$ $10^{10}$.",
      "answer": "99999.9984100000.0016"
    },
    {
      "question": "10,11\n\nFind the length of the shortest path on the surface of a unit regular tetrahedron between the midpoints of its opposite edges.\n\n#",
      "answer": "1"
    },
    {
      "question": "7.1. Given two different geometric progressions, the first terms of which are equal to 1, and the sum of the denominators is 3. Find the sum of the fifth terms of these progressions, if the sum of the sixth terms is 573. If the answer to the question is not unique, specify the sum of all possible values of the desired quantity.",
      "answer": "161"
    },
    {
      "question": "5. If $a, b$ are nonzero real numbers such that $a^{2}+b^{2}=8 a b$, find the value of $\\left|\\frac{a+b}{a-b}\\right|$.",
      "answer": "\\frac{\\sqrt{15}}{3}"
    },
    {
      "question": "All the points on the line with equation $y=3 x+6$ are translated up 3 units, then translated left 4 units, and then reflected in the line with equation $y=x$. Determine the $y$-intercept of the resulting line.",
      "answer": "-7"
    },
    {
      "question": "10.3. In the distant times of stagnation in the Soviet Union, 15 and 20 kopeck coins were in circulation. Schoolboy Valera had a certain amount of money only in such coins. Moreover, the number of 20 kopeck coins was greater than the number of 15 kopeck coins. Valera spent one-fifth of all his money, paying two coins for a movie ticket. He then spent half of the remaining money on lunch, paying with three coins. How many coins of each denomination did Valera have initially? Justify your answer.",
      "answer": "2"
    },
    {
      "question": "## Problem Statement\n\nFind the distance from point $M_{0}$ to the plane passing through three points $M_{1}, M_{2}, M_{3}$.\n\n$M_{1}(7 ; 2 ; 4)$\n\n$M_{2}(7 ;-1 ;-2)$\n\n$M_{3}(-5 ;-2 ;-1)$\n\n$M_{0}(10 ; 1 ; 8)$",
      "answer": "3"
    },
    {
      "question": "1.2. Find the greatest negative root of the equation $\\sin \\pi x = -\\sqrt{3} \\cos \\frac{\\pi x}{2}$.",
      "answer": "-\\frac{2}{3}"
    },
    {
      "question": "The function $f(x)$ satisfies for any positive numbers $x, y$: $f(x y)=f(x)+f(y)$, and $f(2)=1$, then the value of $f\\left(\\frac{1}{64}\\right)$ is $\\qquad$ .",
      "answer": "-6"
    },
    {
      "question": "## Task B-4.4.\n\nGrandpa Ante, looking for a way to entertain his grandchildren Iva and Mato, found three cards. On the first card, one side has the number 1, and the other side has the number 4. On the second card, one side has the number 2, and the other side has the number 4. On the third card, one side has the number 3, and the other side also has the number 4. Grandpa Ante immediately knew how to keep the children occupied for a long time. He told the younger one, Mato, to arrange the three cards in a row, and the schoolboy Iva had to write down the three-digit numbers that Mato forms by arranging the cards and then calculate the sum of all the recorded numbers. Mato randomly picks a card and its side. How many different three-digit numbers can Ivo write down at most, and what is the sum of these numbers?",
      "answer": "10434"
    },
    {
      "question": "2. Find the number of positive integer solutions $(x, y, z)$ that satisfy the following system of equations:\n\\[\n\\left\\{\\begin{array}{l}\nx^{2}+y^{2}=2 z^{2}, \\\\\nz<y \\leqslant z+50\n\\end{array}\\right.\n\\]",
      "answer": "131"
    },
    {
      "question": "Example 2. Into how many parts should the interval of integration be divided to calculate $\\int_{2}^{7} \\frac{d x}{\\sqrt{x+2}}$ with an accuracy of 0.1?",
      "answer": "8"
    },
    {
      "question": "8. The number of different shapes of ellipses with eccentricity $e=\\log _{p} q$ (where $p, q$ are positive integers not exceeding 9) is\n\n保留了源文本的换行和格式。",
      "answer": "26"
    },
    {
      "question": "3. Let $A, B, C$ be three distinct points on a circle, $P$ be the area of triangle $ABC$, and $P_{1}$ be the area of the triangle determined by the tangents to the circle at points $A, B, C$. Determine the limit of the ratio $P_{1}: P$ as point $A$ is fixed and points $B$ and $C$ approach $A$ along the circle such that $B \\neq C$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "2.145. $\\frac{1-a x}{1+a x} \\cdot \\sqrt{\\frac{1+b x}{1-b x}} ; \\quad x=\\frac{1}{a} \\cdot \\sqrt{\\frac{2 a-b}{b}} ; \\quad 0<\\frac{b}{2}<a<b$.",
      "answer": "1"
    },
    {
      "question": "10.16 There is a five-digit positive odd number $x$. In $x$, all 2s are replaced with 5s, and all 5s are replaced with 2s, while other digits remain unchanged, resulting in a new five-digit number, denoted as $y$. If $x$ and $y$ satisfy the equation $y=2(x+1)$, what is the value of $x$?\n(China Junior High School Mathematics League, 1987)",
      "answer": "29995"
    },
    {
      "question": "3. In tetrahedron $ABCD$, $\\triangle ADB$ is an isosceles right triangle, $AD=1, \\angle ADB=90^{\\circ}, \\angle BDC=$ $\\angle ADC=60^{\\circ}$. Then the distance between the skew lines $AB$ and $CD$ is $\\qquad$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "1. Find the maximum value of the function $f(x)=(\\sin x)^{\\cos x}$ on the interval $\\left[0, \\frac{\\pi}{2}\\right]$.",
      "answer": "1"
    },
    {
      "question": "6. 10 students (one is the team leader, 9 are team members) formed a team to participate in a math competition and won the first place. The organizing committee decided to award each team member 200 yuan, and the team leader received 90 yuan more than the average prize of the entire team of 10 members. Therefore, the team leader's prize is $\\qquad$ yuan.",
      "answer": "300"
    },
    {
      "question": "1. From a square steel sheet with a side of 1 meter, identical rectangular isosceles triangles are cut from each of the four corners, leaving a smaller square. Determine its mass if the thickness of the sheet is 3 mm and the density of steel is 7.8 $g / \\mathrm{cm}^{3}$.",
      "answer": "11.7"
    },
    {
      "question": "In a right trapezoid $A B C D(A B \\| C D)$ the angle at vertex $B$ measures $75^{\\circ}$. Point $H$ is the foot of the perpendicular from point $A$ to the line $B C$. If $B H=D C$ and $A D+A H=8$, find the area of $A B C D$.",
      "answer": "8"
    },
    {
      "question": "81. Determine the sum of the coefficients of the polynomial that results from expanding and combining like terms in the expression $\\left(1+x-3 x^{2}\\right)^{1965}$.",
      "answer": "-1"
    },
    {
      "question": "14. (15 points) On the blackboard, there are 5 natural numbers: 1, 3, 5, 7, 9. One operation involves arbitrarily selecting two numbers from the blackboard, erasing them, and writing their sum on the blackboard, while writing their product on a piece of paper. After 4 such operations, there is only 1 number left on the blackboard, and 4 numbers are written on the paper. Find the sum of these 4 numbers.",
      "answer": "230"
    },
    {
      "question": "13.031. A group of students went on a hike through the Moscow region during their vacation. The first 30 km they walked, $20 \\%$ of the remaining part of the route they traveled by raft along the river, and then they walked again, covering a distance 1.5 times greater than the distance they traveled by raft. The remaining part of the journey was covered in 1 hour 30 minutes by hitching a ride on a passing truck, which was traveling at a speed of 40 km/h. What is the total length of the route?",
      "answer": "150"
    },
    {
      "question": "12. Let $f(x)=x^{2}+6 x+c$ for all real numbers $x$, where $c$ is some real number. For what values of $c$ does $f(f(x))$ have exactly 3 distinct real roots?",
      "answer": "\\frac{11-\\sqrt{13}}{2}"
    },
    {
      "question": "15. Given the sequence $\\left\\{a_{n}\\right\\}$ satisfies: $a_{1}=1, a_{n+1}=\\frac{1}{8} a_{n}^{2}+m\\left(n \\in \\mathbf{N}^{*}\\right)$, if for any positive integer $n$, we have $a_{n}<4$, find the maximum value of the real number $m$.",
      "answer": "2"
    },
    {
      "question": "## 268. Math Puzzle $9 / 87$\n\nAt the closing ceremony of a Pioneer camp, a 150-liter soda keg was tapped. Unfortunately, it was a bit cooler than expected that day, so at the end of the event, 20 liters more soda remained in the keg than had been served.\n\nHow many liters did the Pioneers drink?",
      "answer": "65"
    },
    {
      "question": "1. The minimum value of the function $f(x)=\\sin 2022 x+|x-11|+|x-27|$ is",
      "answer": "15"
    },
    {
      "question": "3. (3 points) Wang Qiang bought 6 notebooks and 4 pencils for a total of 9.2 yuan, Zhou Jun bought the same 3 notebooks and 1 pencil, for a total of 3.8 yuan. Then, the cost of buying one notebook and one pencil should be $\\qquad$ yuan.",
      "answer": "1.8"
    },
    {
      "question": "1. Fill two $a$s and two $b$s into the 16 cells shown in Figure 3, with at most one letter per cell. If the same letters must not be in the same row or column, find the number of different ways to fill the cells.\n(2007, National High School Mathematics Competition)",
      "answer": "3960"
    },
    {
      "question": "## Task B-2.1.\n\nSolve the equation $\\sqrt{3-\\frac{1}{2020 x}}=1-\\frac{1}{2020 x}$.",
      "answer": "-\\frac{1}{2020}"
    },
    {
      "question": "243. Christmas Cryptarithm. The Christmas greeting $A M E R R Y X M A S T O A L L^{*}$ is a cryptarithm where each letter represents a specific decimal digit, and each word replaces a square number. Decode this message, given that, in addition, the sum of the digits in each word also matches some square.\n\nTranslate the above text into English, please retain the line breaks and format of the source text, and output the translation result directly.",
      "answer": "934225739681900"
    },
    {
      "question": "2. Given the lines $l_{1}: y=x, l_{2}: y=2 x$, then the slope of the line $l$ that forms equal angles with $l_{1}, l_{2}$ is",
      "answer": "\\frac{1\\\\sqrt{10}}{3}"
    },
    {
      "question": "12. (6 points) There are 9 steel balls, 8 of which weigh the same, and one is slightly lighter than the other 8. Using a balance scale, what is the minimum number of weighings required to find the lighter steel ball?",
      "answer": "2"
    },
    {
      "question": "8. (10th \"Hope Cup\" Invitational Competition Question) Let $x_{1}, x_{2}$ be the roots of the equation $x^{2}+x+1=0$, then the value of the sum $\\frac{x_{1}}{x_{2}}+\\left(\\frac{x_{1}}{x_{2}}\\right)^{2}+\\left(\\frac{x_{1}}{x_{2}}\\right)^{3}+\\cdots+\\left(\\frac{x_{1}}{x_{2}}\\right)^{1998}$ is $\\qquad$ $ـ$.",
      "answer": "0"
    },
    {
      "question": "2. There is a fraction, when the numerator is reduced by 2021 and the denominator is increased by 2021, it simplifies to $\\frac{1}{2021}$. Then the minimum value of the sum of the original fraction's numerator and denominator is",
      "answer": "4044"
    },
    {
      "question": "8. (5 points) A fast train and a slow train are traveling towards each other. The length of the fast train is 315 meters, and the length of the slow train is 300 meters. A person sitting on the slow train sees the fast train pass by in 21 seconds. How many seconds does it take for a person sitting on the fast train to see the slow train pass by?",
      "answer": "20"
    },
    {
      "question": "\\section*{Task 1 - V11231}\n\nIn a large state-owned enterprise in the electrical industry, 12,000 units of a certain semi-finished product are annually procured from a supplier at a price of 1.00 DM per unit. The orders were previously placed twice a year, on January 1 and July 1.\n\nThe administrative costs for each order (issuing and sending the order, monitoring the delivery date, invoice review, booking, etc.) amount to 30.00 DM. The storage costs (space costs, administration, \"shrinkage\" due to spoilage and damage, etc.) amount to 20% of the value of the average inventory. The costs for ordering and storage were therefore:\n\n2 orders ... 60.00 DM\n\nStorage costs 20% of the average inventory (3000 units), i.e., 20% of 3000.00 DM, which is 600.00 DM\n\nTotal 660.00 DM\n\nIn a production meeting, it is suggested to reduce costs by ordering the quantity needed for each quarter (3000 units) four times a year.\n\na) How high are the costs according to this suggestion?\n\nb) For how many orders do the lowest costs occur? How high are the costs in this case?\n\nHint: The term \"Deutsche Mark\" was only changed to \"Mark der Deutschen Notenbank\" (MDN) in 1964 in East Germany, and subsequently to \"Mark\" in 1968.",
      "answer": "380"
    },
    {
      "question": "## Problem 3\n\nA tetrahedron has at most one edge longer than 1 . What is the maximum total length of its edges?",
      "answer": "5+\\sqrt{3}"
    },
    {
      "question": "A cube is divided by planes that are perpendicular to each of its body diagonals and pass through their midpoints. How many parts does these 4 planes divide the surface of the cube into? - How many parts are created if perpendicular planes are placed at both one-third points of each body diagonal? (8 planes).",
      "answer": "14"
    },
    {
      "question": "2B. On Monday, three bananas cost as much as a lemon and an orange together. On Tuesday, the prices of all fruits were reduced by the same amount of money, two oranges cost as much as three bananas and one lemon, and the price of half a lemon was 5 denars.\n\nWhat was the price of one orange on Monday?",
      "answer": "20"
    },
    {
      "question": "A mathematical worm starts from point $A$ and reaches point $B$ in the figure below.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-04.jpg?height=426&width=436&top_left_y=453&top_left_x=772)\n\nB\n\nThis mathematical worm always moves along the black lines in the drawing above, and never passes over a place it has already been to. Additionally, this worm can move down, up, and to the right, but not to the left. For example, a possible path for the mathematical worm to go from point $A$ to point $B$ could be:\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-04.jpg?height=420&width=419&top_left_y=1298&top_left_x=790)\n\na) In how many different ways can the mathematical worm go from point $A$ to point $B$ through paths contained in the segments shown in the figure below? (following the rules described above).\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_cc59d970f26105cc4ca1g-04.jpg?height=466&width=440&top_left_y=2020&top_left_x=751)\n\nb) What is the total number of ways the mathematical worm can go from point $A$ to point $B$? (following the previous rules, for any path, not just those in part a)).",
      "answer": "117649"
    },
    {
      "question": "2. The octahedron $A B C D E F$ has a square $A B C D$ as its base, while the line $E F$ is normal to the plane determined by the square $A B C D$ and passes through its center. It is known that the sphere that touches all faces of the octahedron and the sphere that touches all lateral edges (i.e., $E A, E B, E C, E D, F A, F B, F C$ and $F D$) have the same center, and that the sphere that touches the lateral edges has a surface area that is $50\\%$ larger. Find the ratio $\\frac{E F}{A B}$.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "3. Grey Wolf needs to cut a 20-meter long steel pipe into 4-meter long segments, and it takes 2 minutes for each cut. How many minutes will it take in total?",
      "answer": "8"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$\\lim _{n \\rightarrow \\infty} \\frac{4 n^{2}-\\sqrt[4]{n^{3}}}{\\sqrt[3]{n^{6}+n^{3}+1}-5 n}$",
      "answer": "4"
    },
    {
      "question": "Anthroov A.\n\nSeveral powers of two are written in a row on the board in some order. For each pair of adjacent numbers, Petya wrote down in his notebook the power to which the left number must be raised to get the right number. The first number in the row on the board was 2, and the last number was 1024. Vasya claims that this is enough to find the product of all the numbers in the notebook. Is Vasya right?",
      "answer": "10"
    },
    {
      "question": "In the tin, there were red and green candies. Čeněk ate $\\frac{2}{5}$ of all the red candies and Zuzka ate $\\frac{3}{5}$ of all the green candies. Now, the red candies make up $\\frac{3}{8}$ of all the candies in the tin.\n\nHow many candies were there in the tin at least originally?\n\n(L. Růžičková)\n\nHint. How many candies of each color could, or could not, have been originally in the tin?",
      "answer": "35"
    },
    {
      "question": "9. $[7]$ Let $A B C$ be a triangle, and $I$ its incenter. Let the incircle of $A B C$ touch side $B C$ at $D$, and let lines $B I$ and $C I$ meet the circle with diameter $A I$ at points $P$ and $Q$, respectively. Given $B I=$ $6, C I=5, D I=3$, determine the value of $(D P / D Q)^{2}$.",
      "answer": "\\frac{75}{64}"
    },
    {
      "question": "5. How many five-digit natural numbers are there in which all digits are different, and among which the digits 1 and 2 appear in adjacent decimal places?\n\nThe use of a pocket calculator or any reference materials is not allowed.\n\n# Ministry of Science and Education of the Republic of Croatia Agency for Education and Education Croatian Mathematical Society \n\nCOUNTY COMPETITION IN MATHEMATICS February 26, 2024. 6th grade - elementary school\n\nEach task is worth 10 points. In addition to the final result, the process is also graded. To earn all points, it is necessary to find all solutions, determine that there are no others, document the process, and justify your conclusions.",
      "answer": "2436"
    },
    {
      "question": "A circle is inscribed in a right angle. A chord connecting the points of tangency is equal to 2. Find the distance from the center of the circle to this chord.\n\n#",
      "answer": "1"
    },
    {
      "question": "1. Determine the number of distinct values of the expression\n\n$$\n\\frac{n^{2}-2}{n^{2}-n+2}\n$$\n\nwhere $n \\in\\{1,2, \\ldots, 100\\}$.",
      "answer": "98"
    },
    {
      "question": "The 83 interns from the Animath internship each choose an activity for the free afternoon from 5 proposed activities. We know that:\n\n$\\triangleright$ Shopping has been at least as popular as Laser Game;\n\n$\\Delta$ Movie tickets are sold in lots of 6;\n\n$\\Delta$ At most 5 students go to the beach;\n\n$\\triangleright$ At most one student has visited a Romanesque church.\n\nWhen writing the list of activities in alphabetical order, one writes the number of students corresponding to each activity. How many different lists of numbers could have been written?",
      "answer": "3570"
    },
    {
      "question": "14. (POL 2) For any positive integer $x$ define  $$ \\begin{aligned} g(x) & =\\text { greatest odd divisor of } x, \\\\ f(x) & =\\left\\{\\begin{array}{l} x / 2+x / g(x), \\text { if } x \\text { is even; } \\\\ 2^{(x+1) / 2}, \\text { if } x \\text { is odd. } \\end{array}\\right. \\end{aligned} $$  Construct the sequence $x_{1}=1, x_{n+1}=f\\left(x_{n}\\right)$. Show that the number 1992 appears in this sequence, determine the least $n$ such that $x_{n}=1992$, and determine whether $n$ is unique.",
      "answer": "8253"
    },
    {
      "question": "13.022. On the mathematics entrance exam, $15 \\%$ of the applicants did not solve a single problem, 144 people solved problems with errors, and the number of those who solved all problems correctly is to the number of those who did not solve any at all as 5:3. How many people took the mathematics exam that day",
      "answer": "240"
    },
    {
      "question": "3. The price of one pencil is a whole number of denars. The price of 9 such pencils is greater than 1100 denars, but less than 1200 denars. The price of 13 pencils is greater than 1500 denars, but less than 1600 denars. How much does one pencil cost?",
      "answer": "123"
    },
    {
      "question": "9. A9 (POL) Let the sequence \\(a(n), n=1,2,3, \\ldots\\), be generated as follows: \\(a(1)=0\\), and for \\(n>1\\), \n\\[ a(n)=a([n / 2])+(-1)^{\\frac{n(n+1)}{2}} . \\quad(\\text{Here }[t]=\\text{ the greatest integer } \\leq t .) \\]\n(a) Determine the maximum and minimum value of \\(a(n)\\) over \\(n \\leq 1996\\) and find all \\(n \\leq 1996\\) for which these extreme values are attained.\n(b) How many terms \\(a(n), n \\leq 1996\\), are equal to 0?",
      "answer": "346"
    },
    {
      "question": "12. (10 points) There are four colors of light bulbs (with an ample supply of each color), and one light bulb needs to be installed at each vertex of the triangular prism $A B C-A_{1} B_{1} C_{1}$. The requirement is that the light bulbs at the two endpoints of the same edge must be of different colors, and each color of light bulb must be used at least once. How many installation methods are there?",
      "answer": "216"
    },
    {
      "question": "## Task B-3.6.\n\nInside the square $A B C D$ there is a point $M$ such that $|M A|=7 \\mathrm{~cm}, |M B|=13 \\mathrm{~cm}$ and $|M C|=17 \\mathrm{~cm}$. Calculate the area of the square $A B C D$.",
      "answer": "288"
    },
    {
      "question": "4. If the positive integer $n$ makes $\\sqrt{3}$ always lie between $1+\\frac{3}{n}$ and $1+\\frac{3}{n+1}$, then $n=$",
      "answer": "4"
    },
    {
      "question": "11. Use 100 RMB to buy items, with the rule that you can only buy items in one of the following three ways each day: (1) buy candy for 1 RMB; (2) buy cookies for 2 RMB; (3) buy fruit for 2 RMB, and you are not allowed to buy nothing. There are $\\qquad$ ways to spend all 100 RMB.",
      "answer": "\\frac{1}{3}(2^{101}+1)"
    },
    {
      "question": "8. [30] Consider the following two-player game. Player 1 starts with a number, $N$. He then subtracts a proper divisor of $N$ from $N$ and gives the result to player 2 (a proper divisor of $N$ is a positive divisor of $N$ that is not equal to 1 or $N$ ). Player 2 does the same thing with the number she gets from player 1 , and gives the result back to player 1. The two players continue until a player is given a prime number, at which point that player loses. For how many values of $N$ between 2 and 100 inclusive does player 1 have a winning strategy?",
      "answer": "47"
    },
    {
      "question": "2. Let $P$ be the intersection of the diagonals of a convex quadrilateral $A B C D$. Calculate its area if the areas of triangles $A B C, B C D$ and $D A P$ are $8 \\mathrm{~cm}^{2}, 9 \\mathrm{~cm}^{2}$, $10 \\mathrm{~cm}^{2}$, respectively.",
      "answer": "24"
    },
    {
      "question": ". Let $ABCD$ be an isosceles trapezoid, with the larger base $[AB]$. Suppose the diagonals intersect at a point $O$ such that $\\frac{OA}{OC}=2$. Given that the area of triangle $BOC$ is 10, what is the area of trapezoid $ABCD$?",
      "answer": "45"
    },
    {
      "question": "2. (2 points) Find the minimum value of the expression $4 x^{2}+4 x \\sin y-\\cos ^{2} y$.",
      "answer": "-1"
    },
    {
      "question": "I3.4 $x, y$ are real numbers. If $x+y=c-195$ and $d$ is the maximum value of $x y$, find $d$.",
      "answer": "4"
    },
    {
      "question": "## Task 6B - 171246B\n\na) Given a natural number $n \\geq 2$. Let $u$ be the circumcircle of a regular $2 n$-gon $A_{0} A_{1} \\ldots A_{2 n-1}$.\n\nA set of three vertices $A_{i}, A_{j}, A_{k}$ of this $2 n$-gon is called one-sided if there is a semicircular arc $h$ on the circumference $u$, including its two endpoints, that contains $A_{i}, A_{j}$, and $A_{k}$.\n\nDetermine the probability $w$ that a randomly chosen set $M=\\left\\{A_{i}, A_{j}, A_{k}\\right\\}$ of three vertices is one-sided.\n\nb) Determine the limit $\\lim _{n \\rightarrow \\infty} w_{n}$ if it exists.\n\nHint:\n\nIf $m_{n}$ is the number of all sets that can be formed from three vertices $A_{i}, A_{j}, A_{k}$ of the $2 n$-gon, and $g_{n}$ is the number of one-sided sets among them, then the probability sought in a) is defined as $w_{n}=\\frac{g_{n}}{m_{n}}$.",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "Example 13 Find the minimum value of the bivariate function $f(x, y)=(x-y)^{2}+\\left(x+\\frac{1}{y}+1\\right)^{2}$.\n(1998 \"Hope Cup\" Competition Question)",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "45a * If positive numbers $a, b, c$ and constant $k$ satisfy the inequality $\\frac{k a b c}{a+b+c} \\leqslant(a+b)^{2}+(a+b+$ $4 c)^{2}$, find the maximum value of the constant $k$.",
      "answer": "100"
    },
    {
      "question": "$4 \\cdot 218$ In a middle school mathematics competition, three problems $A$, $B$, and $C$ were given. Among the 25 students who participated in the competition, each student solved at least one problem. Among those who did not solve $A$, the number of students who solved $B$ is twice the number of students who solved $C$. The number of students who solved only $A$ is one more than the number of students who solved $A$ among the remaining students. Among the students who solved only one problem, half did not solve $A$. How many students solved only $B$?",
      "answer": "6"
    },
    {
      "question": "68. For the red clover variety Perm local, on average $84\\%$ of the plants are late maturing. What is the probability that 52 plants out of 60 clover plants, randomly selected, will be late maturing?",
      "answer": "0.1201"
    },
    {
      "question": "For which values of $m \\in \\mathbb{R}$ does the equation:\n\n$$\n\\left(x^{2}-2 m x-4\\left(m^{2}+1\\right)\\right)\\left(x^{2}-4 x-2 m\\left(m^{2}+1\\right)\\right)=0\n$$\n\nhave exactly three distinct roots?",
      "answer": "3"
    },
    {
      "question": "3. The diagonals of the inscribed quadrilateral $ABCD$ intersect at point $O$. Inside triangle $AOB$, a point $K$ is chosen such that line $KO$ is the bisector of angle $CK$. Ray $DK$ intersects the circumcircle of triangle $COK$ again at point $L$, and ray $CK$ intersects the circumcircle of triangle $DOK$ again at point $M$. Find the ratio of the areas of triangles $ALO$ and $BMO$.",
      "answer": "1"
    },
    {
      "question": "3.1. A number was multiplied by its first digit and the result was 494, by its second digit - 988, by its third digit - 1729. Find this number.",
      "answer": "247"
    },
    {
      "question": "25. Form a five-digit number without repeating digits using $1,3,5,7,9$, and form another five-digit number without repeating digits using $0,2,4,6,8$. Subtract the smaller number from the larger number among these two five-digit numbers, the smallest possible difference is $\\qquad$",
      "answer": "517"
    },
    {
      "question": "4. For a finite arithmetic sequence $\\left\\{a_{n}\\right\\}$ with the first term $a_{1}=1$ and a common difference of 2, the arithmetic mean of all its terms is 2015. If one term is deleted, the arithmetic mean of the remaining terms is an integer. Then the number of ways to delete a term is $\\qquad$.",
      "answer": "3"
    },
    {
      "question": "1. Let the numbers be $m=(1+2+3)^{4}+5 \\cdot 6 \\cdot(7+8+9)$ and $n=1+2+3+\\ldots+63$.\n\na) (3p) Calculate $2016^{m-n}$.\n\nb) (4p) Determine the largest and the smallest non-zero natural number which, when divided by $n$, gives a quotient equal to a quarter of the remainder.\n\nProf. Stela Boghian",
      "answer": "10160602020"
    },
    {
      "question": "Folkpor\n\nFive identical balls are moving in one direction in a straight line at some distance from each other, and five other identical balls are moving towards them. The speeds of all the balls are the same. When any two balls collide, they fly apart in opposite directions with the same speed they had before the collision. How many total collisions will occur between the balls?\n\n#",
      "answer": "25"
    },
    {
      "question": "8. (10 points) Through a point $P$ inside an equilateral triangle $A B C$, perpendiculars are drawn to the three sides, with the feet of the perpendiculars being $D, E, F$ respectively. Connect $A P$, $B P$, $C P$. If the area of the equilateral triangle $A B C$ is 2028 square centimeters, and the areas of triangles $P A D$ and $P B E$ are both 192 square centimeters, then the area of triangle $P C F$ is $\\qquad$ square centimeters.",
      "answer": "630"
    },
    {
      "question": "Podolsky A.\n\nChichikov is playing with Nozdryov. First, Nozdryov distributes 1001 nuts among three small boxes. After looking at the distribution, Chichikov names any integer \\( N \\) from 1 to 1001. Then Nozdryov must, if necessary, move one or several nuts to an empty fourth box and show Chichikov one or several boxes where the total number of nuts is exactly \\( N \\). As a result, Chichikov will receive as many dead souls as the number of nuts Nozdryov moved. What is the maximum number of souls Chichikov can guarantee for himself, regardless of how Nozdryov plays?",
      "answer": "71"
    },
    {
      "question": "1.58. Perpendiculars are drawn from the vertex of the acute angle of a rhombus to the lines containing the sides of the rhombus to which this vertex does not belong. The length of each perpendicular is 3 cm, and the distance between their bases is $3 \\sqrt{3}$ cm. Calculate the lengths of the diagonals of the rhombus.",
      "answer": "2\\sqrt{3}"
    },
    {
      "question": "3. Determine the number of all triples of distinct three-digit natural numbers, the sum of which is divisible by each of the three addends.",
      "answer": "234"
    },
    {
      "question": "29. [16] Let $A B C$ be a triangle such that $A B=A C=182$ and $B C=140$. Let $X_{1}$ lie on $A C$ such that $C X_{1}=130$. Let the line through $X_{1}$ perpendicular to $B X_{1}$ at $X_{1}$ meet $A B$ at $X_{2}$. Define $X_{2}, X_{3}, \\ldots$, as follows: for $n$ odd and $n \\geq 1$, let $X_{n+1}$ be the intersection of $A B$ with the perpendicular to $X_{n-1} X_{n}$ through $X_{n}$; for $n$ even and $n \\geq 2$, let $X_{n+1}$ be the intersection of $A C$ with the perpendicular to $X_{n-1} X_{n}$ through $X_{n}$. Find $B X_{1}+X_{1} X_{2}+X_{2} X_{3}+\\ldots$",
      "answer": "\\frac{1106}{5}"
    },
    {
      "question": "An arithmetic sequence has a common difference, $d$, that is a positive integer and is greater than 1. The sequence includes the three terms 3, 468 and 2018. What is the sum of all of the possible values of $d$ ?\n\n(An arithmetic sequence is a sequence in which each term after the first is obtained from the previous term by adding a constant, called the common difference. For example, $3,5,7,9$ are the first four terms of an arithmetic sequence with common difference 2.)",
      "answer": "191"
    },
    {
      "question": "4. Given a parallelogram $A B C D$. A line passing through point $D$ intersects segment $A C$ at point $G$, segment $B C$ at point $F$, and ray $A B$ at point $E$ such that triangles $BEF$ and $CGF$ have the same area. Determine the ratio $|A G|:|G C|$.",
      "answer": "\\frac{1}{2}(1+\\sqrt{5})"
    },
    {
      "question": "3-5. Determine the maximum value of the ratio of a three-digit number to the number equal to the sum of the digits of this number.",
      "answer": "100"
    },
    {
      "question": "11.1. Does there exist a natural number $n$, greater than 1, such that the value of the expression $\\sqrt{n \\sqrt{n \\sqrt{n}}}$ is a natural number?",
      "answer": "128"
    },
    {
      "question": "B1. Determine the smallest possible value of the expression $\\left|25^{m}-36^{n}\\right|$, where $m$ and $n$ are natural numbers.",
      "answer": "11"
    },
    {
      "question": "14. Let $[x]$ denote the greatest integer not exceeding $x$, e.g. $[\\pi]=3,[5.31]=5$ and $[2010]=2010$. Given $f(0)=0$ and $f(n)=f\\left(\\left[\\frac{n}{2}\\right]\\right)+n-2\\left[\\frac{n}{2}\\right]$ for any positive integer $n$. If $m$ is a positive integer not exceeding 2010, find the greatest possible value of $f(m)$.\n(2 marks)\n設 $[x]$ 表示不超過 $x$ 的最大整數, 例如： $[\\pi]=3 、[5.31]=5 、[2010]=2010$ 。已知 $f(0)=0$, 且對任意正整數 $n$ 皆有 $f(n)=f\\left(\\left[\\frac{n}{2}\\right]\\right)+n-2\\left[\\frac{n}{2}\\right]$ 。若 $m$ 是不超過 2010 的正整數, 求 $f(m)$ 的最大可能值。\n(2 分)",
      "answer": "10"
    },
    {
      "question": "Bororin P.A.\n\nOn the surface of a spherical planet, there are four continents separated from each other by an ocean. We will call a point in the ocean special if for it there are at least three nearest (equidistant from it) land points, and all are on different continents. What is the maximum number of special points that can exist on this planet?",
      "answer": "4"
    },
    {
      "question": "Petrov booked an apartment in a new building with five identical entrances. Initially, the entrances were numbered from left to right, and Petrov's apartment had the number 636. Then the developer changed the numbering to the opposite direction (from right to left, see the figure). As a result, Petrov's apartment became number 242. How many apartments are in the building? (The order of apartment numbering within each entrance did not change.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d3cd76ad0dc53ae543bdg-43.jpg?height=89&width=734&top_left_y=184&top_left_x=662)",
      "answer": "985"
    },
    {
      "question": "8.5. Each digit of the natural number $N$ is strictly greater than the one to its left. What is the sum of the digits of the number $9 N$?",
      "answer": "9"
    },
    {
      "question": "## Task 1 - 310711\n\nA department store received a shipment of red, blue, and green balls, totaling 675 pieces. During a certain period, the following were sold:\n\nhalf of the red balls, two-thirds of the blue balls, and one-quarter of the green balls.\n\nIt turned out that after this, the same number of balls remained for each of the three colors.\n\nDetermine from these data,\n\na) how many balls of each of the three colors were sold during this time.\n\nb) how many balls were still available in total afterward!",
      "answer": "225"
    },
    {
      "question": "10. Xiaocuxin made a mistake when calculating the division, writing the dividend 2002 as 2020, resulting in a quotient of 106 and a remainder of 6. The correct quotient should be $\\qquad$ .",
      "answer": "105"
    },
    {
      "question": "8,9 [ Inscribed Circle in an Angle\n\nIn an isosceles trapezoid, there are two circles. One of them, with a radius of 1, is inscribed in the trapezoid, and the second circle touches two sides of the trapezoid and the first circle. The distance from the vertex of the angle formed by the two sides of the trapezoid that touch the second circle to the point of tangency of the circles is twice the diameter of the second circle. Find the area of the trapezoid.\n\n#",
      "answer": "\\frac{9\\sqrt{2}}{2}"
    },
    {
      "question": "In a right-angled triangle, $s_{a}$ and $s_{b}$ are the medians to the legs, and $s_{c}$ is the median to the hypotenuse. Determine the maximum value of the expression $\\frac{s_{a}+s_{b}}{s_{c}}$.",
      "answer": "\\sqrt{10}"
    },
    {
      "question": "On sides $A B$ and $B C$ of an equilateral triangle $A B C$, points $D$ and $K$ are marked, respectively, and on side $A C$, points $E$ and $M$ are marked such that $D A + A E = K C + C M = A B$. Segments $D M$ and $K E$ intersect. Find the angle between them.",
      "answer": "60"
    },
    {
      "question": "6. Among all natural numbers from 1 to 20 inclusive, some 10 numbers were painted blue, and the other 10 - red, then all possible sums of pairs of numbers, one of which is blue and the other is red, were counted. What is the maximum number of different sums that can be among the hundred obtained numbers?",
      "answer": "35"
    },
    {
      "question": "7.3. In the notebook, all irreducible fractions with the numerator 15 are written down, which are greater than $\\frac{1}{16}$ and less than $\\frac{1}{15}$. How many such fractions are written down in the notebook?",
      "answer": "8"
    },
    {
      "question": "9. Si Si wrote $1, 2, 3, \\cdots \\cdots, 99$ on the blackboard. Rui Rui doesn't like the number 2, so he erased the following numbers:\n(1) multiples of 2;\n(2) numbers containing the digit 2 (such as 92);\n(3) numbers whose sum of digits is divisible by 2 (such as 31).\nRui Rui erased $\\qquad$ numbers in total.",
      "answer": "79"
    },
    {
      "question": "5. On one side of the road, some power poles are arranged at equal distances. Xiao Ming rides his bike along the road. He took 3 minutes to ride from the 1st power pole to the 10th power pole. At this speed, in another 3 minutes, Xiao Ming can ride to the $\\qquad$th power pole.",
      "answer": "19"
    },
    {
      "question": "7. Use 4 colors to color a $4 \\times 4$ grid, requiring that the 4 cells in the same row and the 4 cells in the same column have different colors. There are $\\qquad$ different coloring methods.",
      "answer": "576"
    },
    {
      "question": "Problem 5.8. Inside a large triangle with a perimeter of 120, several segments were drawn, dividing it into nine smaller triangles, as shown in the figure. It turned out that the perimeters of all nine small triangles are equal to each other. What can they be equal to? List all possible options.\n\nThe perimeter of a figure is the sum of the lengths of all its sides.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_86512448fd32765ac040g-12.jpg?height=256&width=323&top_left_y=1257&top_left_x=565)",
      "answer": "40"
    },
    {
      "question": "2. In triangle $ABC$, the lengths of the sides are known: $\\overline{AB}=4, \\overline{BC}=7$ and $\\overline{AC}=5$. Let $\\alpha=\\angle BAC$. Calculate the value of the expression $\\sin ^{6} \\frac{\\alpha}{2}+\\cos ^{6} \\frac{\\alpha}{2}$.",
      "answer": "\\frac{7}{25}"
    },
    {
      "question": "8.68. Using a compass and a ruler, divide an angle of $19^{\\circ}$ into 19 equal parts.",
      "answer": "1"
    },
    {
      "question": "Task 1. Dad and Mom are cooking pancakes on the stove, while their children Petya and Vasya are eating them. Dad can cook 70 pancakes in an hour, and Mom can cook 100 pancakes. Petya, if he tries hard, can eat 10 pancakes in 15 minutes, and Vasya can eat twice as much. After how much time will there be no less than 20 uneaten pancakes left on the table?",
      "answer": "24"
    },
    {
      "question": "5. In the Cartesian coordinate system, with the point $(1,0)$ as the center and $r$ as the radius, a circle intersects the parabola $y^{2}=x$ at four points $A$, $B, C, D$. If the intersection point $F$ of $A C$ and $B D$ is exactly the focus of the parabola, then $r=$ $\\qquad$ .",
      "answer": "\\frac{\\sqrt{15}}{4}"
    },
    {
      "question": "15. Find the sum: $1+2+3+\\ldots+1999$.",
      "answer": "1999000"
    },
    {
      "question": "6. Xiao Ming plans to memorize 40 English words from Monday to Sunday over 7 days. He requires himself to memorize a few words each day, with a different number of words memorized each day, and the most words are to be memorized on Sunday. How many words can Xiao Ming memorize at most on Sunday? $\\qquad$",
      "answer": "19"
    },
    {
      "question": "## Subject II\n\na) Write as an irreducible fraction the sum: $S=\\frac{1}{1+2}+\\frac{1}{1+2+3}+\\ldots+\\frac{1}{1+2+\\cdots+2015}$\n\nb) Show that: $\\frac{1}{2} \\cdot\\left(\\frac{1}{2}+\\frac{1}{2+4}\\right)\\left(\\frac{1}{2}+\\frac{1}{2+4}+\\frac{1}{2+4+6}\\right) \\cdot \\ldots \\cdot\\left(\\frac{1}{2}+\\frac{1}{2+4}+\\cdots+\\frac{1}{2+4+6+\\cdots+4028}\\right)=\\frac{1}{2015}$",
      "answer": "\\frac{1}{2015}"
    },
    {
      "question": "3. (5 points) Think of a number, add 5 to it, multiply the result by 2, subtract 4 from the product, divide the difference by 2, and then subtract the number you initially thought of. The final calculation result is $\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "2. $a \\in \\mathbf{R}$, the complex number $\\omega=1+a \\mathrm{i}$, and the complex number $z$ satisfies $\\bar{\\omega} z-\\omega=0$. For what value of $a$ does $|z^{2}-z+2|$ have the minimum value, and find this minimum value.",
      "answer": "\\frac{\\sqrt{14}}{4}"
    },
    {
      "question": "Problem 5. (6 points) Peter has $n^{3}$ unit white cubes, from which he wants to assemble a cube $n \\times n \\times n$ with a completely white surface. How many of the faces of the unit cubes does Emil need to color to prevent Peter? Solve the problem for:\na) (2 points) $n=2$;\nb) (4 points) $n=3$.",
      "answer": "12"
    },
    {
      "question": "$2.19 \\frac{\\sqrt{x}+\\sqrt[3]{x}}{\\sqrt{x}-\\sqrt[3]{x}}=3$.",
      "answer": "64"
    },
    {
      "question": "Calculate $\\sum_{i=1}^{n} \\prod_{j \\neq i} \\frac{x-a_{j}}{a_{i}-a_{j}}$ for $a_{1}, \\ldots, a_{n} \\in \\mathbb{R}$ pairwise distinct.",
      "answer": "1"
    },
    {
      "question": "## Task 2.\n\nA natural number $n$ is good if we can color each side and diagonal of a regular $n$-gon in some color so that for any two vertices $A$ and $B$, there is exactly one vertex $C$, different from $A$ and $B$, such that the segments $\\overline{A B}, \\overline{B C}$, and $\\overline{C A}$ are colored the same color.\n\nDetermine which of the numbers $7,8,9,10,11$, and 12 are good, and which are not.",
      "answer": "7"
    },
    {
      "question": "Let $P(X)$ be a monic polynomial of degree 2017 such that $P(1)=1, P(2)=2, \\ldots, P(2017)=$ 2017. What is the value of $P(2018)$?",
      "answer": "2017!+2018"
    },
    {
      "question": "9. Stubborn Squares (from 7th grade. 2 points). Given 100 numbers. 2 was added to each of them. The sum of the squares of the numbers did not change. 2 was added to each of the resulting numbers again. How did the sum of the squares change now?",
      "answer": "800"
    },
    {
      "question": "Problem 5.4. The school principal, the caretaker, and the parent committee, failing to agree with each other, each bought a carpet for the school auditorium, which is $10 \\times 10$. After thinking about what to do, they decided to place all three carpets as shown in the picture: the first carpet $6 \\times 8$ - in one corner, the second carpet $6 \\times 6$ - in the opposite corner, and the third carpet $5 \\times 7$ - in one of the remaining corners (all dimensions are in meters).\n\nFind the area of the part of the hall covered by carpets in three layers (give the answer in square meters).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2fdcf97aa9799d0d4cd6g-10.jpg?height=337&width=1003&top_left_y=304&top_left_x=225)",
      "answer": "6"
    },
    {
      "question": "## Subject III. (20 points)\n\nLet $E_{1}=\\frac{a(2 a+8)+8}{(a+4)(2 a+12)-16}$ and $E_{2}=\\frac{a(a+4)+4}{a(a+1)-2}$.\n\na) Determine $a \\in \\mathbb{Z}$ such that the expressions make sense;\n\nb) Determine $a \\in \\mathbb{Z}$ for which $\\frac{1}{E_{1}}+\\frac{1}{E_{2}} \\in \\mathbb{N}$;\n\nc) Calculate $a \\in \\mathbb{Z}$ such that $\\frac{1}{E_{1}}+\\frac{1}{E_{2}}=\\frac{3 a+10}{a+2}$.\n\nProf. Simona Pop, Colegiul Augustin Maior Cluj-Napoca",
      "answer": "-3"
    },
    {
      "question": "G8.3 There are exactly $N$ values of $\\alpha$ satisfying the equation $\\cos ^{3} \\alpha-\\cos \\alpha=0$, where $0^{\\circ} \\leq \\alpha \\leq 360^{\\circ}$. Find $N$.\n\nG8.4 If the $N^{\\text {th }}$ day of May in a year is Thursday and the $K^{\\text {th }}$ day of May in the same year is Monday, where $10<K<20$, find $K$.",
      "answer": "16"
    },
    {
      "question": "$\\left[\\begin{array}{ll}\\text { Ratios of areas } \\\\ \\text { P } & \\text { Product of lengths of chord segments and lengths of secant segments }]\\end{array}\\right]$ [ Ratio of areas of triangles with a common angle ]\n\nIn triangle $A B C$, points $B_{1}$ and $C_{1}$ are chosen on sides $A B$ and $A C$ respectively, such that $A B_{1}: A B=1: 3$ and $A C_{1}: A C=1: 2$. A circle is drawn through points $A, B_{1}$, and $C_{1}$. A line is drawn through point $B_{1}$, intersecting segment $A C_{1}$ at point $D$, and the circle at point $E$. Find the area of triangle $B_{1} C_{1} E$, if $A C_{1}=4, A D=1$, $D E=2$, and the area of triangle $A B C$ is 12.",
      "answer": "\\frac{7}{2}"
    },
    {
      "question": "【Question 8】\nThe sum of five consecutive even numbers is a multiple of 7. The smallest possible sum of these five numbers is\n\n保留了源文本的换行和格式。",
      "answer": "70"
    },
    {
      "question": "Task 3. (15 points) Laboratory engineer Sergei received an object for research consisting of about 200 monoliths (a container designed for 200 monoliths, which was almost completely filled). Each monolith has a specific name (sandy loam or clayey loam) and genesis (marine or lake-glacial deposits). The relative frequency (statistical probability) that a randomly selected monolith will be sandy loam is $\\frac{1}{9}$. The relative frequency that a randomly selected\nmonolith will be marine clayey loam is $\\frac{11}{18}$. How many monoliths of lake-glacial genesis does the object contain, if there are no marine sandy loams among the sandy loams?",
      "answer": "77"
    },
    {
      "question": "7. (10 points) In a class of 40 students all facing forward, they stand in a line from front to back, and report numbers in the order of $1, 2, 3, 4, 1, 2, 3,$\n4, 1, 2, 3, 4, and so on, with each student reporting a number once. Students who report 3 turn around. Afterwards, if two adjacent students face each other, they will shake hands once and then turn around simultaneously, until no students are facing each other. How many times do all the students shake hands in the entire process? $\\qquad$ times.",
      "answer": "145"
    },
    {
      "question": "1. Find the sum of all three-digit numbers that can be written using the digits $1,2,3$ such that all digits in each number are different.",
      "answer": "1332"
    },
    {
      "question": "Task 5. (20 points) At the first deposit, equipment of the highest class was used, and at the second deposit, equipment of the first class was used, with the highest class being less than the first. Initially, $40 \\%$ of the equipment from the first deposit was transferred to the second. Then, $20 \\%$ of the equipment that ended up on the second deposit was transferred back to the first, with half of the transferred equipment being of the first class. After this, the equipment of the highest class on the first deposit was 26 units more than on the second, and the total amount of equipment on the second deposit increased by more than $5 \\%$ compared to the initial amount. Find the total amount of equipment of the first class.",
      "answer": "60"
    },
    {
      "question": "5. Among the numbers between 4000 and 7000, there are $\\qquad$ numbers that are even and have four distinct digits.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "728"
    },
    {
      "question": "12. Each edge of a regular tetrahedron is trisected, and through each division point, two planes are drawn, each parallel to the two faces of the tetrahedron that do not pass through this division point. How many parts do these planes divide the tetrahedron into?\n(26th Moscow Competition Problem)",
      "answer": "15"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0}\\left(\\frac{1+x \\cdot 2^{x}}{1+x \\cdot 3^{x}}\\right)^{\\frac{1}{x^{2}}}$",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "Let $a=\\sqrt{4+\\sqrt{5-a}}, b=\\sqrt{4+\\sqrt{5+b}}, c=\\sqrt{4-\\sqrt{5-c}}$ and $d=\\sqrt{4-\\sqrt{5+d}}$.\n\nCalculate $a b c d$.",
      "answer": "11"
    },
    {
      "question": "Task 3. (15 points) At the research institute, a scientific employee, Tatyana Vasilyevna, received an object for research containing about 150 oil samples (a container designed for 150 samples, which was almost completely filled). Each sample has certain characteristics in terms of sulfur content - either low-sulfur or high-sulfur, and density - either light or heavy. The relative frequency (statistical probability) that a randomly selected sample will be a heavy oil sample is $\\frac{2}{11}$. The relative frequency that a randomly selected sample will be a light low-sulfur oil sample is $\\frac{7}{13}$. How many high-sulfur oil samples does the object contain, if there were no low-sulfur samples among the heavy oil samples?",
      "answer": "66"
    },
    {
      "question": "4. The numbers $x$ and $y$ are such that the equalities $\\operatorname{tg} x+\\operatorname{tg} y=4$ and $3 \\sin (2 x+2 y)=\\sin 2 x \\sin 2 y$ hold. Find $\\operatorname{ctg} x \\operatorname{ctg} y$.",
      "answer": "\\frac{7}{6}"
    },
    {
      "question": "2. How many ordered pairs of integers $(a, b)$ are there such that $|a|+|b|<2019$?",
      "answer": "8148685"
    },
    {
      "question": "11. [8] Isosceles trapezoid $A B C D$ with bases $A B$ and $C D$ has a point $P$ on $A B$ with $A P=11, B P=27$, $C D=34$, and $\\angle C P D=90^{\\circ}$. Compute the height of isosceles trapezoid $A B C D$.",
      "answer": "15"
    },
    {
      "question": "30. $\\mathrm{P}$ is the set $\\{1,2,3, \\ldots, 14,15\\}$. If $\\mathrm{A}=\\left\\{a_{1}, a_{2}, a_{3}\\right\\}$ is a subset of $\\mathrm{P}$ where $a_{1}<a_{2}<a_{3}$ such that $a_{1}+6 \\leq a_{2}+3 \\leq a_{3}$. How many such subsets are there of P?",
      "answer": "165"
    },
    {
      "question": "Problem 10.7. On an island, there are 23 knights and 200 liars; all residents have different names. A tourist who knows this asked each of the 223 residents to write down 200 names of liars on a piece of paper. Each knight correctly wrote down 200 names of liars, while each liar wrote down an arbitrary list of 200 names, which definitely does not include their own name. What is the maximum number of liars the tourist can definitely identify from these data?",
      "answer": "16"
    },
    {
      "question": "91. On a long strip of paper, consecutive natural numbers $1, 2, 3, \\cdots \\cdots n$ are written in sequence. The long strip is cut into five segments, each containing some numbers (the digits of a single number will not be cut into different segments). The average of the numbers in these five segments are $1234, 345, 128, 19$, and $9.5$ (the order of these five numbers is scrambled). Therefore, $n=$",
      "answer": "2014"
    },
    {
      "question": "How many multiples of 3 between 1 and 2015 have a units digit of 2 in their decimal representation?",
      "answer": "67"
    },
    {
      "question": "4. From the 2022 natural numbers 1 to 2022, select several distinct numbers such that the sum of any 2 of them is not a multiple of 5. The maximum number of numbers that can be selected is $\\qquad$.",
      "answer": "811"
    },
    {
      "question": "Let's determine the pairs of squares of two-digit (integer) numbers that differ only in the hundreds place (for example, 2401 and 2601).",
      "answer": "15"
    },
    {
      "question": "5. (10 points) On the blackboard, there is a sequence of numbers: $1,2,3, \\cdots, 2011$. If every time the first 16 numbers are erased, and the sum of the erased 16 numbers is written at the end of the sequence, until only one number remains, then:\n(1) What is the last remaining number?\n(2) What is the total sum of all numbers that have appeared on the blackboard?",
      "answer": "7822326"
    },
    {
      "question": "7. Given that the complex number $z$ satisfies $|z|=1$, then the maximum value of $\\left|z^{3}-3 z-2\\right|$ is $\\qquad$ .",
      "answer": "3\\sqrt{3}"
    },
    {
      "question": "The third-grade students who participated in the 11th \"Math Walk\" and entered the finals were arranged in a square formation, with 15 remaining, and at least 18 more people are needed to form a larger square formation. There are $\\qquad$ third-grade students participating in the 11th \"Math Walk\" competition.",
      "answer": "271"
    },
    {
      "question": "7. The commission consists of 11 people. The materials that the commission works on are stored in a safe. How many locks should the safe have, and how many keys should each member of the commission be provided with, so that access to the safe is possible when a majority of the commission members gather, but not possible if fewer than half of the members gather?",
      "answer": "252"
    },
    {
      "question": "4.8 There is a four-digit number less than 2000, which has exactly 14 positive divisors (including 1 and itself), and one of its prime divisors ends with the digit 1. Find this four-digit number.\n(Shanghai Junior High School Mathematics Competition, 1984)",
      "answer": "1984"
    },
    {
      "question": "8. 确定 $\\sqrt{25^{\\frac{1}{\\log _{6} 5}}+49^{\\frac{1}{\\log _{8} 7}}}$ 的值.",
      "answer": "10"
    },
    {
      "question": "1. The product of all positive divisors of a natural number $n$ is $20^{15}$. Determine $n$.",
      "answer": "400"
    },
    {
      "question": "## Task Condition\n\nCalculate the area of the figure bounded by the graphs of the functions:\n\n$$\nx=4-(y-1)^{2}, x=y^{2}-4 y+3\n$$",
      "answer": "9"
    },
    {
      "question": "31. Let's call a number complex if it has at least two different prime divisors. Find the largest natural number that cannot be represented as the sum of two complex numbers.",
      "answer": "23"
    },
    {
      "question": "4. In the rectangular prism $A^{\\prime} C$, $A B=5, B C=4, B^{\\prime} B=6$, and $E$ is the midpoint of $A A^{\\prime}$. Find the distance between the skew lines $B E$ and $A^{\\prime} C^{\\prime}$.",
      "answer": "\\frac{60}{\\sqrt{769}}"
    },
    {
      "question": "8. Jon created a unique encryption method, where each digit of the plaintext number is transformed into a digit of the ciphertext number by \"multiplying by 3, adding 1, and taking the unit digit,\" for example, the plaintext \"27\" is encrypted to the ciphertext \"72\". If a four-digit plaintext number, after two rounds of encryption, becomes the ciphertext \"2019\", then this four-digit plaintext number is $\\qquad$",
      "answer": "2435"
    },
    {
      "question": "Solve the following equation:\n\n$$\n2^{x+3}+4^{x+1}=320 .\n$$",
      "answer": "3"
    },
    {
      "question": "Raskin M.A.\n\nMasha has two-ruble and five-ruble coins. If she takes all her two-ruble coins, she will be 60 rubles short to buy four pies. If all five-ruble coins - she will be 60 rubles short for five pies. And in total, she is 60 rubles short to buy six pies. How much does a pie cost?",
      "answer": "20"
    },
    {
      "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{-\\pi / 2}^{0} 2^{8} \\cdot \\cos ^{8} x d x\n$$",
      "answer": "35\\pi"
    },
    {
      "question": "2. The cold water tap fills the bathtub in 19 minutes, while the hot water tap fills it in 23 minutes. The hot water tap was opened. After how many minutes should the cold water tap be opened so that by the time the bathtub is completely filled, there is an equal amount of cold and hot water in it?",
      "answer": "2"
    },
    {
      "question": "8. Given that the side faces of a regular tetrahedron are right triangles with an area of 1, then the volume of the regular tetrahedron is",
      "answer": "\\frac{\\sqrt{2}}{3}"
    },
    {
      "question": "2. We consider number sequences $a_{1}, a_{2}, a_{3}, \\ldots$ such that $a_{n+1}=\\frac{a_{n}+a_{1}}{a_{n}+1}$ holds for all $n \\geqslant 1$.\n\n(a) Suppose that $a_{1}=-3$. Compute $a_{2020}$.\n\n(b) Suppose that $a_{1}=2$. Prove that $\\frac{4}{3} \\leqslant a_{n} \\leqslant \\frac{3}{2}$ holds for all $n \\geqslant 2$.",
      "answer": "-3"
    },
    {
      "question": "10. Suppose $\\omega$ is a circle centered at $O$ with radius 8 . Let $A C$ and $B D$ be perpendicular chords of $\\omega$. Let $P$ be a point inside quadrilateral $A B C D$ such that the circumcircles of triangles $A B P$ and $C D P$ are tangent, and the circumcircles of triangles $A D P$ and $B C P$ are tangent. If $A C=2 \\sqrt{61}$ and $B D=6 \\sqrt{7}$, then $O P$ can be expressed as $\\sqrt{a}-\\sqrt{b}$ for positive integers $a$ and $b$. Compute $100 a+b$.",
      "answer": "103360"
    },
    {
      "question": "## Problem Statement\n\nCalculate the lengths of the arcs of the curves given by the parametric equations.\n\n$$\n\\begin{aligned}\n& \\left\\{\\begin{array}{l}\nx=\\left(t^{2}-2\\right) \\sin t+2 t \\cos t \\\\\ny=\\left(2-t^{2}\\right) \\cos t+2 t \\sin t\n\\end{array}\\right. \\\\\n& 0 \\leq t \\leq 3 \\pi\n\\end{aligned}\n$$",
      "answer": "9\\pi^{3}"
    },
    {
      "question": "7. Find the sum of all possible different two-digit numbers, all digits of which are odd.",
      "answer": "1375"
    },
    {
      "question": "2. Batman wants to crack Edward Nigma's code. He knows that the code represents some permutation of the letters in the expression TRICKORTREAT, and that the first and last letters of the code are the same. How many possibilities are there for such a code?",
      "answer": "\\frac{10!}{3}"
    },
    {
      "question": "Megrajzoltuk a koordinátarendszerben az $f(x)=\\frac{1}{x}$ függvény grafikonját. Mekkorának válasszuk az új, egymással továbbra is egyenlő egységeket a tengelyeken, ha azt akarjuk, hogy a görbe a $g(x)=\\frac{2}{x}$ függvény grafikonja legyen?",
      "answer": "\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "4. (10 points) A school has two classes each in the third and fourth grades. Class 3-1 has 4 more students than Class 3-2, Class 4-1 has 5 fewer students than Class 4-2, and the third grade has 17 fewer students than the fourth grade. Therefore, Class 3-1 has fewer students than Class 4-2 by ____.",
      "answer": "9"
    },
    {
      "question": "8. If a positive integer $n$ makes the equation $x^{3}+y^{3}=z^{n}$ have positive integer solutions $(x, y, z)$, then $n$ is called a \"good number\". Then the number of \"good numbers\" not exceeding 2019 is $\\qquad$ .",
      "answer": "1346"
    },
    {
      "question": "2. (5 points) Xiao Fou buys 3 erasers and 5 pencils for 10.6 yuan. If he buys 4 erasers and 4 pencils of the same type, he needs to pay 12 yuan. Then the price of one eraser is $\\qquad$ yuan.",
      "answer": "2.2"
    },
    {
      "question": "The distance from a vertex of the base of a regular tetrahedron to the centroid of the opposite side is 4. The maximum volume of this regular tetrahedron is\n$\\qquad$",
      "answer": "11.18"
    },
    {
      "question": "4. The base of a quadrilateral pyramid is a square $ABCD$, with a side length of 2, and the lateral edge $SA$ is perpendicular to the base plane and also equals 2. A plane is drawn through the lateral edge $SC$ and a point on the side $AB$, such that the resulting cross-section of the pyramid has the smallest perimeter. Find the area of the cross-section.",
      "answer": "\\sqrt{6}"
    },
    {
      "question": "(15) Let $x, y$ be real numbers, then $\\max _{5 x^{2}+4 y^{2}=10 x}\\left(x^{2}+y^{2}\\right)=$ $\\qquad$",
      "answer": "4"
    },
    {
      "question": "## Task 1 - 210711\n\nThe FDJ members of a school have decided to redesign the grounds of their school. A rectangular lawn area measuring $45 \\mathrm{~m}$ in length and $26 \\mathrm{~m}$ in width is to be surrounded by a path that is $2 \\mathrm{~m}$ wide.\n\nThe path is to run outside the lawn and adjoin it all around. The area occupied by the lawn and the path together is to form a rectangle again.\n\na) Calculate the area of the planned path!\n\nb) How many paving stones are needed for the path in total if it is to be completely covered with paving stones and if exactly 16 stones are required for each square meter of the path?",
      "answer": "4800"
    },
    {
      "question": "Problem 4.5. Hooligan Dima laid out a structure in the shape of a $3 \\times 5$ rectangle using 38 wooden toothpicks. Then he simultaneously set fire to two adjacent corners of this rectangle, marked in the diagram.\n\nIt is known that one toothpick burns for 10 seconds. How many seconds will it take for the entire structure to burn?\n\n(Fire spreads along the toothpicks at a constant speed. The fire continues to spread from each burned toothpick to all adjacent unburned toothpicks.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_df4c0436ab68970fab15g-04.jpg?height=458&width=359&top_left_y=110&top_left_x=547)",
      "answer": "65"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0}\\left(\\frac{\\arcsin ^{2} x}{\\arcsin ^{2} 4 x}\\right)^{2 x+1}$",
      "answer": "\\frac{1}{16}"
    },
    {
      "question": "Example 1. Calculate approximately the value of the function $f(x)=$ $=\\sqrt{x^{2}+x+3}$ at $x=1.97$",
      "answer": "2.975"
    },
    {
      "question": "In the following figure-not drawn to scale!- $E$ is the midpoint of $B C$, triangle $F E C$ has area 7, and quadrilateral $D B E G$ has area 27. Triangles $A D G$ and $G E F$ have the same area, $x$. Find $x$.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_6cb05c45fe7c141f3316g-4.jpg?height=661&width=700&top_left_y=1209&top_left_x=688)",
      "answer": "8"
    },
    {
      "question": "On each kilometer of the highway between the villages of Yolki and Palki, there is a post with a sign. On one side of the sign, it says how many kilometers it is to Yolki, and on the other side - to Palki. A Thoughtful Observer noticed that on each post, the sum of these numbers is 13. What is the distance from Yolki to Palki?\n\n#",
      "answer": "13"
    },
    {
      "question": "1. When $3 x^{2}$ was added to the quadratic trinomial $f(x)$, its minimum value increased by 9, and when $x^{2}$ was subtracted from it, its minimum value decreased by 9. How will the minimum value of $f(x)$ change if $x^{2}$ is added to it?",
      "answer": "\\frac{9}{2}"
    },
    {
      "question": "In a magic square, the sum of the numbers in each row, column, and diagonal is the same. In the magic square below, what is the value of $a+b+c$?\n\n| 16 | 2 | $a$ |\n| :---: | :---: | :---: |\n| $c$ | 10 | $d$ |\n| $b$ | $e$ | 4 |",
      "answer": "26"
    },
    {
      "question": "7. (5 points) This year is 2017, the sum of the digits of the year is 10, then within this century, the sum of all years whose digit sum is 10\n\n保留源文本的换行和格式，翻译结果如下：\n\n7. (5 points) This year is 2017, the sum of the digits of the year is 10, then within this century, the sum of all years whose digit sum is 10",
      "answer": "18396"
    },
    {
      "question": "In a quadrilateral $A B C D$, a circle of radius 2 is inscribed. The angle $\\angle D A B$ is a right angle. Side $A B$ is 5, and side $B C$ is 6. Find the area of the quadrilateral $A B C D$.",
      "answer": "17\\frac{11}{17}"
    },
    {
      "question": "1.59 (1) $f(x)$ is a monovariate quartic polynomial with integer coefficients, i.e.,\n$$\nf(x)=c_{4} x^{4}+c_{3} x^{3}+c_{2} x^{2}+c_{1} x+c_{0},\n$$\n\nwhere $c_{0}, c_{1}, c_{2}, c_{3}, c_{4}$ are all integers. Prove that if $a, b$ are integers and $a > b$, then $a - b$ divides $f(a) - f(b)$.\n(2) Two people, A and B, are doing homework in the classroom.\nA asks B: “How many problems have you finished?”\nB replies: “The number of problems I have finished is a positive integer and is a root of a monovariate quartic polynomial with integer coefficients. Can you guess it?”\nThen, A tries substituting 7 into the polynomial and gets a value of 77. At this point, B says: “The number of problems I have finished is more than 7.”\nA says: “Alright, I will try a larger integer.” So A substitutes $B$ into the polynomial and gets a value of 85.\nB looks and says: “The number of problems I have finished is more than $B$.” After some thought and calculation, A finally determines the number of problems B has finished.\nTry to find the number of problems B has finished based on the conversation.\n(China Hebei Province Mathematics Competition, 1979)",
      "answer": "14"
    },
    {
      "question": "Seven teams, divided into two groups, participated in the football tournament in my neighborhood. Group 1 consisted of the teams Avaqui, Botágua, and Corinense. Group 2 consisted of the teams Dinossaurs, Esquisitos, Flurinthians, and Guaraná.\n\n$\\mathrm{In}$ the first round of the tournament, each team played against each of the other teams in its group exactly once. In the second round of the tournament, each team played against each of the teams in the other group exactly once.\n\n(a) How many matches were played in the first round in Group 1?\n\n(b) How many matches were played in the first round in Group 2?\n\n(c) How many matches were played in the second round?",
      "answer": "12"
    },
    {
      "question": "9. Given $F_{1}$ and $F_{2}$ are the two foci of the ellipse $\\frac{x^{2}}{4}+y^{2}=1$, $A$ and $B$ are the left vertex and the upper vertex of the ellipse, respectively, and point $P$ lies on the line segment $A B$. Then the minimum value of $\\overrightarrow{P F_{1}} \\cdot \\overrightarrow{P F_{2}}$ is .",
      "answer": "-\\frac{11}{5}"
    },
    {
      "question": "2. Painting walls in a new building 10 workers would complete in 15 days. However, at the beginning, there were only 6 workers, and after 5 days, 2 more workers joined, and after another 3 days, 4 more workers joined. In how many days was the job finally completed?",
      "answer": "16"
    },
    {
      "question": "I4.4 Let $D$ be the sum of the last three digits of $2401 \\times 7^{C}$ (in the denary system). Find the value of $D$.",
      "answer": "7"
    },
    {
      "question": "11. [15] Find the largest positive integer $n$ such that $1!+2!+3!+\\cdots+n$ ! is a perfect square. Prove that your answer is correct.",
      "answer": "3"
    },
    {
      "question": "2. The sequence $\\left(a_{n}\\right)$ is defined by the following relations: $a_{1}=1, a_{2}=2, a_{n}=a_{n-1}-a_{n-2}+n$ (for $n \\geqslant 3$). Find $a_{2019}$.",
      "answer": "2020"
    },
    {
      "question": "Pedro has a $6 \\times 2$ board, with the two rightmost cells painted gray as illustrated below:\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_d1e5bbfbdffc7558a598g-11.jpg?height=222&width=616&top_left_y=1882&top_left_x=797)\n\nHe must fill all the cells of his board with the numbers from 1 to 12 such that:\n\n- in each row, the 6 numbers, read from left to right, are in increasing order; and\n- in each column, the number on top is less than the number below.\na) Show that the largest sum Pedro can achieve in the gray cells is 23. Also show how he can achieve this sum.\n\nb) Show that the smallest sum Pedro can achieve in the gray cells is 18. Also show how he can achieve this sum.\n\n#",
      "answer": "18"
    },
    {
      "question": "Subject 2. The arithmetic mean of the distinct natural numbers $a_{1}, a_{2}, a_{3}, \\ldots, a_{2014}$, 2014, and 2016 is 2015.\n\na. Find the arithmetic mean of the numbers $a_{1}, a_{2}, a_{3}, \\ldots, a_{2014}$.\n\nb. If the numbers $a_{1}, a_{2}, a_{3}, \\ldots, a_{2014}$ are even, show that $a_{1} \\cdot a_{2} \\cdot a_{3} \\cdot \\ldots \\cdot a_{2014}=0$",
      "answer": "0"
    },
    {
      "question": "14. (6th \"Hope Cup\" Senior High School Competition) The function $y=f(x)$ is an even function and is a periodic function with a period of 2. When $x \\in [2,3]$, $f(x)=x-1$. On the graph of $y=f(x)$, there are two points $A$ and $B$, whose y-coordinates are equal ($A$ is to the left of $B$), and their x-coordinates are in the interval $[1,3]$. The fixed point $C$ has coordinates $(0, a)$, where $a>2$. Find the maximum value of the area of $\\triangle ABC$ (expressed in terms of $a$).",
      "answer": "-2"
    },
    {
      "question": "1B. If the digit 1 is appended to the right of a three-digit number, a four-digit number is obtained which is 3 times larger than the number obtained by appending the digit 2 to the left of the given three-digit number. What is that three-digit number?",
      "answer": "857"
    },
    {
      "question": "Auxiliary similar triangles Pythagorean theorem (direct and inverse).\n\nIn triangle $ABC$, altitudes $AA_1$ and $BB_1$ are drawn. Find $AC$, if\na) $AA_1=4, BB_1=5, BC=6$\n\nb) $A_1C=8, B_1C=5, BB_1=12$.",
      "answer": "4.8"
    },
    {
      "question": "1. The engines of a rocket launched vertically upward from the Earth's surface, providing the rocket with an acceleration of $20 \\mathrm{~m} / \\mathrm{c}^{2}$, suddenly stopped working 40 seconds after launch. To what maximum height will the rocket rise? Can this rocket pose a danger to an object located at an altitude of 45 km? Air resistance is not taken into account, and the acceleration due to gravity is considered to be $10 \\mathrm{~m} / \\mathrm{c}^{2}$.",
      "answer": "48"
    },
    {
      "question": "## Task 2 - 330622\n\nImagine all the different numbers formed by arranging the five digits 1, 2, 3, 4, 5 in every possible order.\n\nWhat is the sum of all these five-digit numbers?",
      "answer": "3999960"
    },
    {
      "question": "4. (10 points) In a lottery, 6 numbers out of 36 are drawn each week. What is the probability that at least one of the numbers drawn this week will be the same as one of the numbers drawn last week? Round the result to the nearest thousandth.",
      "answer": "0.695"
    },
    {
      "question": "2. (5 points) Calculate: $898+9898+99898+999898=$",
      "answer": "1110592"
    },
    {
      "question": "Task B-1.1. Solve the equation:\n\n$$\n|x-| x-|x+1|||=x\n$$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "132. For which $x$ and $y$ is the number $x x y y$ a square of a natural number?",
      "answer": "7744"
    },
    {
      "question": "Example 12 Find the minimum value of $\\frac{3}{\\cos x}+\\frac{2}{\\sin x}\\left(0<x<\\frac{\\pi}{2}\\right)$.",
      "answer": "\\sqrt{(\\sqrt[3]{9}+\\sqrt[3]{4})^{3}}"
    },
    {
      "question": "Problem 4. Given is a square $ABCD$ with side length $9 \\, \\text{cm}$. The square is folded as shown in the diagram on the right, so that vertex $B$ coincides with point $E$, which lies on side $\\overline{CD}$. In this way, three triangles are formed: $CEF$, $EDG$, and $GAH$. What is the sum of the perimeters of these three triangles?\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_e235c78af50361de0ec1g-04.jpg?height=272&width=348&top_left_y=1859&top_left_x=1114)",
      "answer": "36"
    },
    {
      "question": "3. (17th Japan Mathematical Olympiad) Find the tens digit of $11^{12^{13}}$ (where $11^{12^{3}}$ represents 11 to the power of $12^{13}$)",
      "answer": "2"
    },
    {
      "question": "It is said that an integer $n \\geqslant 1$ is Japanese if it admits three divisors $d_{1}, d_{2}$, and $d_{3}$ such that $1 \\leqslant d_{1}<d_{2}<d_{3}$ and $d_{1}+d_{2}+d_{3}=2022$.\nWhat is the smallest Japanese integer?",
      "answer": "1344"
    },
    {
      "question": "Task A-1.4. (4 points)\n\nIf we delete the units digit of a number, we will get a number that is 2010 less than the original number. What is the original number?",
      "answer": "2233"
    },
    {
      "question": "5. (8 points) In $\\triangle A B C, A B=86$, and $A C=97$. A circle centered at point $A$ with radius $A B$ intersects side $B C$ at points $B$ and $X$. Additionally, $B X$ and $C X$ have integer lengths. What is the length of $B C ?$",
      "answer": "61"
    },
    {
      "question": "In a picture trading, the price of the frames is directly proportional to the value of the paintings inside them. To reduce the price difference between certain paintings, the dealer exchanges the frames of two paintings. In one case, a painting that cost five times as much as the other, after the frames were exchanged, only cost three times as much. How will the price ratio of the paintings titled \"Winter Landscape\" and \"Village Road\" change if, before the frames were exchanged, the \"Winter Landscape\" cost nine times as much as the \"Village Road\"?",
      "answer": "4"
    },
    {
      "question": "5. Sabine has a very large collection of shells. She decides to give part of her collection to her sister.\n\nOn the first day, she lines up all her shells. She takes the shells that are in a position that is a perfect square (the first, fourth, ninth, sixteenth, etc. shell), and gives them to her sister. On the second day, she lines up her remaining shells. Again, she takes the shells that are in a position that is a perfect square, and gives them to her sister. She repeats this process every day.\n\nThe 27th day is the first day that she ends up with fewer than 1000 shells. The 28th day she ends up with a number of shells that is a perfect square for the tenth time.\n\nWhat are the possible numbers of shells that Sabine could have had in the very beginning?",
      "answer": "2020"
    },
    {
      "question": "5. If the six edges of a tetrahedron are all powers of 2, then in this tetrahedron, the lengths of the edges can have at most $\\qquad$ values.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "3"
    },
    {
      "question": "3. Calculate the value of the expression $\\sin ^{8} 75^{\\circ}-\\cos ^{8} 75^{\\circ}$.",
      "answer": "\\frac{7\\sqrt{3}}{16}"
    },
    {
      "question": "In a large box, 10 smaller boxes were placed. In each of the nested boxes, either 10 even smaller ones were placed, or nothing was placed. In each of the smaller ones, either 10 or none were placed again, and so on. After this, it turned out that there were exactly 2006 boxes with contents. How many are empty? #",
      "answer": "18055"
    },
    {
      "question": "Anya, Borya, and Vasya decided to go to the \"Fir Tree\". They agreed to meet at the bus stop, but they don't know who will arrive at what time. Each of them can arrive at a random time between 15:00 and 16:00. Vasya is the most patient of all: if he arrives and there is neither Anya nor Borya at the bus stop, he will wait for 15 minutes, and if he doesn't wait for anyone, he will go to the \"Fir Tree\" alone. Borya is less patient: he will wait only 10 minutes. Anya is the most impatient: she won't wait at all. However, if Borya and Vasya meet, they will wait for Anya until 16:00. What is the probability that they will all go to the \"Fir Tree\" together?",
      "answer": "0.124"
    },
    {
      "question": "3. Find the smallest natural number $N>9$, which is not divisible by 7, but if any of its digits is replaced by 7, the resulting number is divisible by 7.",
      "answer": "13264513"
    },
    {
      "question": "Example 12 Let $x, y, z$ be real numbers, not all zero, find the maximum value of $\\frac{x y+2 y z}{x^{2}+y^{2}+z^{2}}$.",
      "answer": "\\frac{\\sqrt{5}}{2}"
    },
    {
      "question": "Example 11. Find the mathematical expectation of the random variable\n$X$, the density of which is determined by the function $p(x)=0.2 e^{-0.2 x}$ for $x \\geq 0$.",
      "answer": "5"
    },
    {
      "question": "Example 4 Let the set $A=\\{1,2, \\cdots, 10\\},$ and the mapping $f$ from $A$ to $A$ satisfies the following two conditions:\n(1) For any $x \\in A, f_{30}(x)=x$.\n(2) For each $k \\in \\mathbf{N}_{+}, 1 \\leqslant k \\leqslant 29$, there exists at least one $a \\in A$ such that $f_{k}(a) \\neq a$.\n\nFind the number of such mappings $f$ (it is agreed that $f_{1}(x)=f(x), f_{k+1}(x)=f\\left(f_{k}(x)\\right), k=1,2, \\cdots$).",
      "answer": "120960"
    },
    {
      "question": "35. Suppose the three sides of a triangular field are all integers, and its area equals the perimeter (in numbers). What is the largest possible area of the field?",
      "answer": "60"
    },
    {
      "question": "11. In the multiplication equation in the figure, different Chinese characters represent different digits, and the same Chinese characters represent the same digit. Fill in the appropriate digits in the squares of the equation to make the equation true. Then, the three-digit number represented by \"中环杯\" is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "124"
    },
    {
      "question": "[Example 3.4.6] Let $x_{1}, x_{2}, x_{3}, x_{4}$ be positive real numbers, and $x_{1}+x_{2}+x_{3}+x_{4}=$ $\\pi$. Find the minimum value of the expression\n$$\n\\begin{array}{l}\n\\left(2 \\sin ^{2} x_{1}+\\frac{1}{\\sin ^{2} x_{1}}\\right) \\cdot\\left(2 \\sin ^{2} x_{2}+\\frac{1}{\\sin ^{2} x_{2}}\\right) . \\\\\n\\left(2 \\sin ^{2} x_{3}+\\frac{1}{\\sin ^{2} x_{3}}\\right) \\cdot\\left(2 \\sin ^{2} x_{4}+\\frac{1}{\\sin ^{2} x_{4}}\\right)\n\\end{array}\n$$",
      "answer": "81"
    },
    {
      "question": "Three of the positive divisors of 24 are 1, 8, and 24 . What is the sum of all of the positive divisors of 24 ?\n\n##",
      "answer": "60"
    },
    {
      "question": "[Example 3.5.6] Given that $p$ is a prime number, $r$ is the remainder when $p$ is divided by 210. If $r$ is a composite number and can be expressed as the sum of two perfect squares, find $r$.\n\n保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "169"
    },
    {
      "question": "3. This year, the sum of Piplu and Luluxi's ages is 22 years old, 1 year later Piplu's age will be exactly twice Luluxi's age. This year, Luluxi is $\\qquad$ years old.",
      "answer": "7"
    },
    {
      "question": "$a b c$\nExample 3.1.1 Three-digit addition $\\frac{+d e f}{g h i}$ where $a, b, c, d, e, f, g, h, i$ are exactly composed of 1, $2,3,4,5,6,7,8,9$, find the solutions for the above addition that make $g h i$ the largest and smallest.\n\nAnalysis: One-digit addition has two cases: with carry and without carry. Suppose in the three-digit addition, there are $k$ carries $(k=0,1,2)$. First, explore the necessary conditions that the addition satisfying the conditions should meet.",
      "answer": "981459"
    },
    {
      "question": "## Task $2 / 83$\n\nWe are looking for all natural numbers $n$ for which $3^{n}$, when written in decimal notation, has exactly $0.5 n$ digits.",
      "answer": "42"
    },
    {
      "question": "Tokarev S.I.\n\nOn the first horizontal line of the chessboard, there are 8 black queens, and on the last - 8 white queens. In what minimum number of moves can the white queens exchange places with the black ones? The white and black queens take turns, moving one queen per move.",
      "answer": "23"
    },
    {
      "question": "[Problems on Mixtures and Concentrations]\n\nFrom two pieces of alloys (with different lead content) weighing 6 and 12 kg, pieces of equal mass were cut. Each of the cut pieces was melted with the remainder of the other piece, after which the percentage of lead in both alloys became the same. What are the masses of each of the cut pieces?",
      "answer": "4"
    },
    {
      "question": "\\section*{Problem 3}\n\nA circle radius 100 is drawn on squared paper with unit squares. It does not touch any of the grid lines or pass through any of the lattice points. What is the maximum number of squares can it pass through?",
      "answer": "800"
    },
    {
      "question": "11. A quadrilateral contains an inscribed circle (i.e. a circle tangent to the four sides of the quadrilateral). The ratio of the perimeter of the quadrilateral to that of the circle is $4: 3$. The ratio of the area of the quadrilateral to that of the circle, expressed in its lowest terms, is $a: b$. What is the value of $100 a+49 b$ ?",
      "answer": "547"
    },
    {
      "question": "3.1. Find the sum\n\n$$\n\\sqrt[7]{(-7)^{7}}+\\sqrt[8]{(-8)^{8}}+\\sqrt[9]{(-9)^{9}}+\\ldots+\\sqrt[100]{(-100)^{100}}\n$$\n\n(Each term is of the form $\\left.\\sqrt[k]{(-k)^{k}}\\right)$",
      "answer": "47"
    },
    {
      "question": "(5) Given the sequence $\\left\\{a_{n}\\right\\}$ with the general term formula $a_{n}=\\frac{1}{(n+1) \\sqrt{n}+n \\sqrt{n+1}}$ $\\left(n \\in \\mathbf{N}^{*}\\right)$, and its first $n$ terms sum as $S_{n}$, then in the sequence $S_{1}, S_{2}, \\cdots, S_{2009}$, the number of rational terms is $\\qquad$ terms.",
      "answer": "43"
    },
    {
      "question": "【Question 6】\nThere are two sets of numbers, the sum of the first set of 7 numbers is 84, the average of the second set is 21, and the average of all numbers in both sets is 18, then the second set has\n$\\qquad$ numbers.",
      "answer": "14"
    },
    {
      "question": "7. Xiao Qiang, while calculating a division problem, wrote the divisor 76 as 67, resulting in a quotient of 15 and a remainder of 5. The correct quotient should be $\\qquad$ _.",
      "answer": "13"
    },
    {
      "question": "In the coordinate plane consider the set $S$ of all points with integer coordinates. For a positive integer $k$, two distinct points $A, B \\in S$ will be called $k$-friends if there is a point $C \\in S$ such that the area of the triangle $A B C$ is equal to $k$. A set $T \\subset S$ will be called a $k$-clique if every two points in $T$ are $k$-friends. Find the least positive integer $k$ for which there exists a $k$-clique with more than 200 elements.",
      "answer": "180180"
    },
    {
      "question": "3.1. Every hour, between two adjacent nettle bushes in a row, two more of the same grow. How many bushes do you need to plant initially so that after three hours, the total number of bushes together is 190?",
      "answer": "8"
    },
    {
      "question": "6. Let $A B C$ be a triangle with incenter $I$, incircle $\\gamma$ and circumcircle $\\Gamma$. Let $M, N, P$ be the midpoints of sides $\\overline{B C}, \\overline{C A}, \\overline{A B}$ and let $E, F$ be the tangency points of $\\gamma$ with $\\overline{C A}$ and $\\overline{A B}$, respectively. Let $U$, $V$ be the intersections of line $E F$ with line $M N$ and line $M P$, respectively, and let $X$ be the midpoint of arc $\\widehat{B A C}$ of $\\Gamma$. Given that $A B=5, A C=8$, and $\\angle A=60^{\\circ}$, compute the area of triangle $X U V$.",
      "answer": "\\frac{21\\sqrt{3}}{8}"
    },
    {
      "question": "10.2. In the bottom left corner of a $7 \\times 7$ chessboard, there is a king. In one move, he can move one square to the right, or one square up, or one square diagonally to the right and up. In how many different ways can the king travel to the top right corner of the board, if he is forbidden to visit the central cell?",
      "answer": "5020"
    },
    {
      "question": "6. To lift a load, it is attached to the hook of a crane using slings made of steel cable. The calculated mass of the load is $M=20$ t, the number of slings $n=3$. Each sling forms an angle $\\alpha=30^{\\circ}$ with the vertical. All slings carry the same load during the lifting of the cargo. According to safety requirements, the tensile strength of the cable should exceed the calculated load by a factor of $k=6$. The cable consists of a very large number of steel threads, each of which can withstand a maximum load of $q=10^{3} \\mathrm{H} / \\mathrm{mm}^{2}$. Indicate the smallest diameter of the cable that can be used to manufacture the slings, if the available cables have any integer diameter in millimeters. The acceleration due to gravity $g$ is considered to be $10 \\mathrm{~m} / \\mathrm{s}^{2}$.",
      "answer": "26"
    },
    {
      "question": "3. Given $\\angle A, \\angle B, \\angle C$ are the three interior angles of $\\triangle ABC$, and the vector $\\alpha=\\left(\\cos \\frac{A-B}{2}, \\sqrt{3} \\sin \\frac{A+B}{2}\\right)$, $|\\alpha|=\\sqrt{2}$. If when $\\angle C$ is maximized, there exists a moving point $M$ such that $|M A|, |A B|, |M B|$ form an arithmetic sequence, the maximum value of $|A B|$ is $\\qquad$",
      "answer": "\\frac{2\\sqrt{3}+\\sqrt{2}}{4}"
    },
    {
      "question": "2. (10 points) The little rabbit and the little turtle start from location $A$ to the forest amusement park at the same time. The little rabbit jumps forward 36 meters per minute, and after every 3 minutes of jumping, it plays on the spot. The first time it plays for 0.5 minutes, the second time for 1 minute, the third time for 1.5 minutes, $\\cdots$, the $k$-th time for $0.5 k$ minutes. The little turtle does not rest or play on the way. It is known that the little turtle arrives at the forest amusement park 3 minutes and 20 seconds earlier than the little rabbit. The distance from $A$ to the forest amusement park is 2640 meters. Then, the little turtle crawls $\\qquad$ meters per minute.",
      "answer": "12"
    },
    {
      "question": "2.15. Find the lateral surface of a regular triangular prism with height $h$, if a line passing through the center of the upper base and the midpoint of the side of the lower base is inclined to the plane of the base at an angle of $60^{\\circ}$.",
      "answer": "6^{2}"
    },
    {
      "question": "5. Triangle $A B C$ has side lengths $A B=2 \\sqrt{5}, B C=1$, and $C A=5$. Point $D$ is on side $A C$ such that $C D=1$, and $F$ is a point such that $B F=2$ and $C F=3$. Let $E$ be the intersection of lines $A B$ and $D F$. Find the area of $C D E B$.",
      "answer": "\\frac{22}{35}"
    },
    {
      "question": "18. (3 points) When Xiaoming goes from home to school, he walks the first half of the distance and takes a ride for the second half; when he returns from school to home, he rides for the first $\\frac{1}{3}$ of the time and walks for the last $\\frac{2}{3}$ of the time. As a result, the time it takes to go to school is 2 hours more than the time it takes to return home. It is known that Xiaoming walks at a speed of 5 kilometers per hour and rides at a speed of 15 kilometers per hour. Therefore, the distance from Xiaoming's home to school is $\\qquad$ kilometers.",
      "answer": "150"
    },
    {
      "question": "410. The distribution function of a two-dimensional random variable is given by\n\n$$\nF(x, y)=\\left\\{\\begin{array}{c}\n\\sin x \\cdot \\sin y \\text { for } 0 \\leqslant x \\leqslant \\pi / 2,0 \\leqslant y \\leqslant \\pi / 2 \\\\\n0 \\quad \\text { for } x<0 \\text { or } y<0\n\\end{array}\\right.\n$$\n\nFind the probability that a random point $(X, Y)$ falls within the rectangle bounded by the lines $x=0, x=\\pi / 4$, $\\boldsymbol{y}=\\pi / 6, y=\\pi / 3$.",
      "answer": "\\frac{(\\sqrt{6}-\\sqrt{2})}{4}"
    },
    {
      "question": "12. Given an integer $k$ satisfying $1819<k<2019$, and making the solution of the system of linear equations in two variables $\\left\\{\\begin{array}{l}4 x-5 y=6 \\\\ 5 x+4 y=k\\end{array}\\right.$\n\nall integers, then the number of $k$ that satisfies the condition is $\\qquad$.",
      "answer": "5"
    },
    {
      "question": "8. (10 points) In the expression $(x+y+z)^{2020}+(x-y-z)^{2020}$, the brackets were expanded and like terms were combined. How many monomials $x^{a} y^{b} z^{c}$ with a non-zero coefficient were obtained?",
      "answer": "1022121"
    },
    {
      "question": "7.091. $\\log _{2}\\left(4 \\cdot 3^{x}-6\\right)-\\log _{2}\\left(9^{x}-6\\right)=1$.",
      "answer": "1"
    },
    {
      "question": "1 A square grid of 16 dots (see the figure) contains the corners of nine $1 \\times 1$ squares, four $2 \\times 2$ squares, and one $3 \\times 3$ square, for a total of 14 squares whose sides are parallel to the sides of the grid. What is the smallest possible number of dots you can remove so that, after removing those dots, each of the 14 squares is missing at least one corner?\nJustify your answer by showing both that the number of dots you claim is sufficient and by explaining why no smaller number of dots will work.",
      "answer": "4"
    },
    {
      "question": "4. Let $A, B, C$ be the centers of the given circles, $A B=2, A C=2, B C=3$ (see Fig. 1).\n\nDenote by $a, b$, and $c$ the radii of the circles centered at points $A, B$, and $C$ respectively, and find their values by solving the system of equations. We get:\n\n$$\n\\left\\{\\begin{array}{l}\na + b = 2 \\\\\na + c = 2 \\\\\nb + c = 3\n\\end{array} \\Leftrightarrow \\left\\{\\begin{array}{l}\na=\\frac{1}{2} \\\\\nb=\\frac{3}{2} \\\\\nc=\\frac{3}{2}\n\\end{array}\\right.\\right.\n$$\n\nLet $S$ and $L$ be the centers of the small and large circles, respectively.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_56ffd9f61e4fedda1946g-2.jpg?height=617&width=625&top_left_y=157&top_left_x=1178)\n\nFig. 1\n\nDenote by $r$ and $R$ the radii of the small and large circles that are tangent to the three given circles, respectively.\n\nLet $\\angle A S B=2 \\gamma, \\angle A S C=2 \\gamma$, and $\\angle B S C=2 \\alpha$. Then, by the cosine rule for triangle $A S B$, we get:\n\n$$\n\\cos 2 \\gamma=\\frac{(a+r)^{2}+(b+r)^{2}-(a+b)^{2}}{2(a+r)(b+r)}=\\frac{r^{2}+(a+b) r-a b}{(a+r)(b+r)}\n$$\n\nFrom this, using the double-angle formulas, we find:\n\n$$\n\\cos ^{2} \\gamma=\\frac{r(a+b+r)}{(a+r)(b+r)}, \\sin ^{2} \\gamma=\\frac{a b}{(a+r)(b+r)}\n$$\n\nSimilarly, for triangle $B S C$, we get:\n\n$$\n\\sin ^{2} \\alpha=\\frac{b^{2}}{(b+r)^{2}}\n$$\n\nSince $\\alpha+2 \\gamma=\\pi$, then $\\sin ^{2} \\alpha=\\sin ^{2} 2 \\gamma=4 \\sin ^{2} \\gamma \\cos ^{2} \\gamma$.\n\nFrom this, we have:\n\n$$\n\\begin{gathered}\n\\frac{b^{2}}{(b+r)^{2}}=4 \\cdot \\frac{a b}{(a+r)(b+r)} \\cdot \\frac{r(a+b+r)}{(a+r)(b+r)} \\\\\nb^{2}(a+r)^{2}=4 a b(a+b+r) \\\\\n(b-4 a) r^{2}+\\left(2 a b^{2}-4 a^{2}-4 a b\\right) r+a^{2} b=0\n\\end{gathered}\n$$\n\nSubstituting the found values of the radii of the original circles, we get the equation for determining the value of $r$:\n\n$$\n\\begin{aligned}\n& r^{2}+5 r-\\frac{3}{4}=0 \\\\\n& r=\\frac{1}{2}(2 \\sqrt{7}-5)\n\\end{aligned}\n$$\n\nNote that to find the value of the radius of the large circle $R$, in all the equations for finding the radius of the small circle, we can replace $r$ with $-R$. Then we get the equation:\n\n$$\n\\begin{aligned}\n& R^{2}-5 R-\\frac{3}{4}=0 \\\\\n& R=\\frac{1}{2}(2 \\sqrt{7}+5)\n\\end{aligned}\n$$\n\nWe find the area of the desired figure, we get: $S=\\pi R^{2}-\\pi r^{2}=10 \\sqrt{7} \\pi$.\n\nNote. The radii of the original circles are found - 1 point. The radius of one of the circles is found - 4 points.",
      "answer": "10\\sqrt{7}\\pi"
    },
    {
      "question": "Example 8 As shown in Figure 1.4.9, in trapezoid $A B C D$, $A B / / D C, C E$ is the bisector of $\\angle B C D$, $C E \\perp A D$ at $E, D E=2 A E, C E$ divides the trapezoid into two parts with areas $S_{1}$ and $S_{2}$, if $S_{1}=1$, find $S_{2}$.",
      "answer": "\\frac{7}{8}"
    },
    {
      "question": "11. (12 points) There is a new type of problem-solving robot that can solve problems, but it has an IQ balance limit. Each time it solves a problem, it subtracts the score of the problem from its IQ balance, consuming an amount of IQ balance equal to the score. When it solves a problem correctly, its IQ balance increases by 1. If its IQ balance is less than the score of the problem it is solving, it will fail to solve the problem. If Xiaopeng uses a problem-solving robot with an initial IQ limit of 25 to solve a set of problems with scores ranging from $1-10$, what is the maximum score he can achieve? $\\qquad$",
      "answer": "31"
    },
    {
      "question": "2. For a rectangle, the sum of two sides is 11, and the sum of three sides is 19.5. Find the product of all possible different values of the perimeter of such a rectangle.",
      "answer": "15400"
    },
    {
      "question": "2. (2004 College Entrance Examination - Zhejiang Paper) In $\\triangle A B C$, the sides opposite to angles $A, B, C$ are $a, b, c$ respectively, and $\\cos A=\\frac{1}{3}$.\n(1) Find the value of $\\sin ^{2} \\frac{B+C}{2}+\\cos 2 A$;\n(2) If $a=\\sqrt{3}$, find the maximum value of $b c$.\n\n保留了原文的换行和格式，如上所示。",
      "answer": "\\frac{9}{4}"
    },
    {
      "question": "Problem 6. A row of new recruits stood facing the sergeant. On the command \"left,\" some turned left, while the rest turned right. It turned out that six times more soldiers were looking at the back of their neighbor than in the face. Then, on the command \"about face,\" everyone turned in the opposite direction. Now, seven times more soldiers were looking at the back of their neighbor than in the face. How many soldiers are in the row?\n\n$[8$ points] (A. V. Shipovalov)",
      "answer": "98"
    },
    {
      "question": "18. (6 points) In a certain exam, the average score of 11 students, rounded to the first decimal place, is 85.3. It is known that each student's score is an integer. Therefore, the total score of these 11 students is $\\qquad$ points.",
      "answer": "938"
    },
    {
      "question": "2. $x, y \\in \\mathbf{R}$, and satisfy $\\left\\{\\begin{array}{l}(x+1)^{\\frac{3}{5}}+2023(x+1)=-2023, \\\\ (y+1)^{\\frac{3}{5}}+2023(y+1)=2023\\end{array}\\right.$, then $x+y=$",
      "answer": "-2"
    },
    {
      "question": "2. In $\\triangle A B C$, $M, N$ are points on line segments $A B, A C$ respectively, and $A M=\\frac{1}{2} A B, A N=\\frac{1}{3} A C, E, F, G$ are three moving points on line segment $B C$, and $\\overrightarrow{A E}+\\overrightarrow{A F}+\\overrightarrow{A G}=x \\overrightarrow{A M}+y \\overrightarrow{A N}(x, y \\in \\mathbf{R})$, then the minimum value of $\\frac{2}{x}+\\frac{3}{y}$ is $\\qquad$",
      "answer": "\\frac{4}{3}"
    },
    {
      "question": "36. [25] A prime number $p$ is twin if at least one of $p+2$ or $p-2$ is prime and sexy if at least one of $p+6$ and $p-6$ is prime.\nHow many sexy twin primes (i.e. primes that are both twin and sexy) are there less than $10^{9}$ ? Express your answer as a positive integer $N$ in decimal notation; for example, 521495223. If your answer is in this form, your score for this problem will be $\\max \\left\\{0,25-\\left\\lfloor\\frac{1}{10000}|A-N|\\right\\rfloor\\right\\}$, where $A$ is the actual answer to this problem. Otherwise, your score will be zero.",
      "answer": "1462105"
    },
    {
      "question": "Problem 6. Plot the figure on the coordinate plane defined by the system of inequalities and find its area $\\left\\{\\begin{array}{c}|x+5|+\\sqrt{3}|y-1| \\leq 3 \\\\ y \\leq \\sqrt{4-4 x-x^{2}}+1 \\\\ |2 y-1| \\leq 5\\end{array}\\right.$.",
      "answer": "\\frac{2}{3}\\pi+\\frac{4}{3}\\sqrt{3}"
    },
    {
      "question": "16. [10] If $p$ and $q$ are positive integers and $\\frac{2008}{2009}<\\frac{p}{q}<\\frac{2009}{2010}$, what is the minimum value of $p$ ?",
      "answer": "4017"
    },
    {
      "question": "9. From the 10 numbers $0,1,2,3,4,5,6,7,8,9$, choose 3 numbers such that their sum is an even number not less than 10. The number of different ways to do this is $\\qquad$.",
      "answer": "51"
    },
    {
      "question": "## Task 1 - 090931\n\nLet $A B C D E F G H$ be a regular octagon. Consider all triangles formed by any three of the points $A, B, C, D, E, F, G, H$.\n\nSomeone wants to list some of these triangles, with the condition that no two of the listed triangles are congruent to each other.\n\nDetermine the maximum number of triangles he can list under this condition!",
      "answer": "5"
    },
    {
      "question": "## Task B-1.2.\n\nGiven the expression $\\quad|x-1|+|x-2|+|x-3|+\\ldots+|x-2019|+|x-2020|$.\n\nDetermine the largest interval of real numbers $[a, b]$ on which the given expression has a constant value $k$ for all $x \\in[a, b]$. What is the value of the constant $k$?",
      "answer": "1010^2"
    },
    {
      "question": "7. In a trapezoid, the diagonals of which intersect at a right angle, it is known that the midline is 6.5 and one of the diagonals is 12. Find the second diagonal\n\n#",
      "answer": "5"
    },
    {
      "question": "10. (12 points) The figure is composed of 5 identical squares. The number of triangles that can be formed using the 12 points in the figure as vertices is.\n\n保留了原文的换行和格式，以下是翻译结果：\n```\n10. (12 points) The figure is composed of 5 identical squares. The number of triangles that can be formed using the 12 points in the figure as vertices is.\n```",
      "answer": "200"
    },
    {
      "question": "In the box, there are blue, red, and green pencils. In total, there are 20 pieces. Blue pencils are 6 times more than green ones, and there are fewer red pencils than blue ones.\n\nHow many red pencils are in the box?\n\n#",
      "answer": "6"
    },
    {
      "question": "5. On the cells of a chessboard of size $8 \\times 8$, 6 identical pieces are randomly placed. Find the probability that five of them will be located either on the same row, or on the same column, or on one of the two main diagonals.",
      "answer": "\\frac{339}{446276}"
    },
    {
      "question": "Trushin 5.\n100 thimbles are arranged in a circle. Under one of them, a coin is hidden. In one move, it is allowed to flip over four thimbles and check if the coin is under one of them. After this, they are returned to their original position, and the coin is moved to one of the adjacent thimbles. What is the minimum number of moves required to definitely find the coin?",
      "answer": "33"
    },
    {
      "question": "G4.2 Suppose $P(x)=a_{0}+a_{1} x+a_{2} x^{2}+\\ldots+a_{8} x^{8}$ is a polynomial of degree 8 with real coefficients $a_{0}, a_{1}, \\ldots, a_{8}$. If $P(k)=\\frac{1}{k}$ when $k=1,2, \\ldots, 9$, and $b=P(10)$, find the value of $b$.",
      "answer": "\\frac{1}{5}"
    },
    {
      "question": "Problem 8.4. In a class, there are 20 students. Thinking about which girls to send a Valentine's card to on February 14, each boy made a list of all the girls in the class he finds attractive (possibly an empty list). It is known that there do not exist three boys whose lists have the same number of girls. What is the minimum number of girls that can be in the class?",
      "answer": "6"
    },
    {
      "question": "5. All edges of a regular quadrilateral pyramid $S A B C D$ with base $A B C D$ have equal length. Plane $\\alpha$ is perpendicular to line $S A$, and plane $\\beta$ is parallel to line $C D$. Determine the smallest possible angle between planes $\\alpha$ and $\\beta$. (7 points) Answer: $\\frac{\\pi}{6}$.",
      "answer": "\\frac{\\pi}{6}"
    },
    {
      "question": "1. (8 points) The calculation result of the expression $(19 \\times 19-12 \\times 12) \\div\\left(\\frac{19}{12}-\\frac{12}{19}\\right)$ is",
      "answer": "228"
    },
    {
      "question": "5. Given the sequence $\\left\\{a_{n}\\right\\}$, the first two terms are $a_{1}=2, a_{2}=4$. Let $b_{n}=a_{n+1}-a_{n}$. If the sequence $\\left\\{b_{n}\\right\\}$ is an arithmetic sequence with a common difference of 1, then $a_{2020}=$ $\\qquad$",
      "answer": "2041211"
    },
    {
      "question": "What is the smallest two-digit positive integer that is a multiple of 6 but is not a multiple of 4 ?",
      "answer": "18"
    },
    {
      "question": "Task 15. (8 points)\n\nAnnouncement: \"Have free time and want to earn money? Write now and earn up to 2500 rubles a day working as a courier with the service 'Food.There-Here!'. Delivery of food from stores, cafes, and restaurants.\n\nHow much will a school student working as a courier with the service 'Food.There-Here!' earn in a month (4 weeks), working 4 days a week with a minimum load for 1250 rubles a day after the income tax deduction?\n\nIn your answer, provide only the number without units of measurement! #",
      "answer": "17400"
    },
    {
      "question": "Question 215, Let the sum of all positive divisors of the positive integer $\\mathrm{n}$ be $\\sigma(\\mathrm{n})$, find the value: $\\left[\\sqrt[3]{\\sum_{\\mathrm{n}=1}^{2020} \\frac{\\sigma(\\mathrm{n}}{\\mathrm{n}}}\\right]$, here $[\\mathrm{x}]$ represents the greatest integer not exceeding $x$.\n\n保留源文本的换行和格式，翻译结果如下：\n\nQuestion 215, Let the sum of all positive divisors of the positive integer $\\mathrm{n}$ be $\\sigma(\\mathrm{n})$, find the value: $\\left[\\sqrt[3]{\\sum_{\\mathrm{n}=1}^{2020} \\frac{\\sigma(\\mathrm{n}}{\\mathrm{n}}}\\right]$, here $[\\mathrm{x}]$ represents the greatest integer not exceeding $x$.",
      "answer": "14"
    },
    {
      "question": "3. Translator (from 6th grade. 1 point). Vasily Petrov is completing an English language assignment. In this assignment, there are 10 English expressions and their Russian translations in random order. The task is to establish the correct correspondences between the expressions and their translations. 1 point is given for each correctly established correspondence. Thus, one can score from 0 to 10 points. Vasya knows nothing, so he chooses the options at random. Find the probability that he will score exactly 9 points.",
      "answer": "0"
    },
    {
      "question": "1. Given a finite set of cards. On each of them, either the number 1 or the number -1 is written (exactly one number on each card), and there are 100 more cards with -1 than cards with 1. If for each pair of different cards, the product of the numbers on them is found, and all these products are summed, the result is 1000. How many cards have the number 1?",
      "answer": "3950"
    },
    {
      "question": "19. Given 7 points on a circle, connect each pair of points with line segments. What is the minimum number of triangles in the resulting figure?\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "284"
    },
    {
      "question": "Task 1. Find the largest four-digit number in which all digits are different, and moreover, no two of them can be swapped to form a smaller number.",
      "answer": "7089"
    },
    {
      "question": "Let $A_{1} A_{2} A_{3}$ be a triangle. We construct the following points:\n\n- $B_{1}, B_{2}$, and $B_{3}$ are the midpoints of $A_{1} A_{2}, A_{2} A_{3}$, and $A_{3} A_{1}$.\n\n- $C_{1}, C_{2}$, and $C_{3}$ are the midpoints of $A_{1} B_{1}, A_{2} B_{2}$, and $A_{3} B_{3}$.\n\n- $D_{1}$ is the intersection of $\\left(A_{1} C_{2}\\right)$ and $\\left(B_{1} A_{3}\\right)$, $D_{2}$ and $D_{3}$ are defined cyclically in the same way.\n- $E_{1}$ is the intersection of $\\left(A_{1} B_{2}\\right)$ and $\\left(C_{1} A_{3}\\right)$, $E_{2}$ and $E_{3}$ are defined cyclically in the same way.\n\nCalculate the quotient of the area of $\\mathrm{D}_{1} \\mathrm{D}_{2} \\mathrm{D}_{3}$ by the area of $\\mathrm{E}_{1} \\mathrm{E}_{2} \\mathrm{E}_{3}$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_10_59677aa68548da326d08g-214.jpg?height=632&width=1371&top_left_y=1465&top_left_x=366)\n\n## 2 Solutions",
      "answer": "\\frac{25}{49}"
    },
    {
      "question": "13.9 Given several sets with identical contents, each set consists of 4 objects with integer gram weights. Now, we want to use these objects as weights to measure every integer gram weight from 1 gram to 1985 grams (including 1985 grams). To minimize the total weight of the objects used as weights, how many different ways are there to select the weights of the 4 objects?",
      "answer": "1"
    },
    {
      "question": "32. How many five-digit numbers are there that read the same from left to right and right to left?",
      "answer": "900"
    },
    {
      "question": "75. In quadrilateral $A B C D$, it is given that $\\widehat{D A B}=150^{\\circ}$, $\\widehat{D A C}+\\widehat{A B D}=120^{\\circ}, \\widehat{D B C}-\\widehat{A B \\bar{D}}=60^{\\circ}$. Find $\\widehat{B D C}$.",
      "answer": "30"
    },
    {
      "question": "A four-digit number, which is a perfect square of a number, has its first two and last two digits equal; find this number.",
      "answer": "7744"
    },
    {
      "question": "Someone said in the summer: \"My birth year is divisible by my current age.\" A friend replied: \"Although I am older than you, I can still say the same about myself.\" - How old are the speakers?",
      "answer": "3753"
    },
    {
      "question": "【Question 13】\nIn a city's archery competition, after ranking the scores, the average score of the top seven is 3 points less than the average score of the top four, and the average score of the top ten is 4 points less than the average score of the top seven. The total score of the 5th, 6th, and 7th place is $\\qquad$ points more than the total score of the 8th, 9th, and 10th place.",
      "answer": "28"
    },
    {
      "question": "1. Calculate: $\\frac{9 \\times 11+31}{2015} \\times \\frac{31+6 \\frac{1}{5}+4 \\frac{3}{7}}{1+\\frac{1}{5}+\\frac{1}{7}}=$",
      "answer": "2"
    },
    {
      "question": "(4) There are three sets of cards, red, yellow, and blue, each set containing five cards, marked with the letters $A, B, C, D$, and $E$. If 5 cards are drawn from these 15 cards, with the requirement that the letters are all different and all three colors are included, then the number of different ways to draw the cards is $\\qquad$ kinds.",
      "answer": "150"
    },
    {
      "question": "Example 16 (28th Brazilian Mathematical Olympiad) If a positive integer has 8 positive divisors, and the sum of these 8 positive divisors is 3240, then this positive integer is called a \"good number\". For example, 2006 is a good number, because the sum of its divisors 1, 2, 17, 34, 59, 118, 1003, 2006 is 3240. Find the smallest good number.",
      "answer": "1614"
    },
    {
      "question": "12. The teacher is buying souvenirs for the students. There are three different types of souvenirs in the store (souvenirs of the same type are identical), priced at 1 yuan, 2 yuan, and 4 yuan respectively. The teacher plans to spend 101 yuan, and at least one of each type of souvenir must be purchased. There are $\\qquad$ different purchasing schemes.",
      "answer": "600"
    },
    {
      "question": "$4 \\cdot 41$ Solve the equation $5 x^{2}+x-x \\cdot \\sqrt{5 x^{2}-1}-2=0$.",
      "answer": "\\\\frac{\\sqrt{10}}{5}"
    },
    {
      "question": "11.2. How much greater is one of two positive numbers than the other if their arithmetic mean is $2 \\sqrt{3}$ and the geometric mean is $\\sqrt{3}$?\n\n(Hint: the geometric mean of two numbers $m$ and $n$ is the number $p=\\sqrt{\\mathrm{mn}}$).",
      "answer": "6"
    },
    {
      "question": "1. Calculate the remainder obtained when dividing $2^{4 n+2}+2^{4}$ by $2^{2 n+1}+2^{n+1}+1$ for every natural number $n \\geq 2$.",
      "answer": "15"
    },
    {
      "question": "## 212. Math Puzzler $1 / 83$\n\nIn a shoe factory, 100 workers produce 500 pairs of shoes daily. During a Subbotnik, this daily performance is exceeded by 10 percent of the workers by 10 percent.\n\nWhat percentage did the company bill at the end of the voluntary work effort? How many pairs of shoes were produced? What did the production director need to pay attention to?",
      "answer": "101"
    },
    {
      "question": "Problem 2. Along the path between the houses of Nезнайка (Nезнayka) and Синеглазка (Sineglazka), there were 15 peonies and 15 tulips growing in a row, mixed together.\n\nSetting out from home to visit Nезнайка, Синеглазка watered all the flowers in a row. After the 10th tulip, the water ran out, and 10 flowers remained unwatered.\n\nThe next day, setting out from home to visit Синеглазка, Незнайка picked flowers for her one by one. After picking the 6th tulip, he decided that it was enough for the bouquet. How many flowers remained growing along the path?",
      "answer": "19"
    },
    {
      "question": "## SUBJECT 4\n\nLet the sequence of real numbers $\\left(x_{n}\\right)_{n \\geq 1}$, given by $x_{1}=\\alpha$, with $\\alpha \\in \\mathrm{R}^{*}, \\alpha \\neq 2$ and $x_{n+1}=\\frac{x_{n}^{2}+2 x_{n}+2}{x_{n}^{2}+1},(\\forall) n \\geq 1$. Calculate:\na) $\\lim _{n \\rightarrow \\infty} x_{n}$,\nb) $\\lim _{n \\rightarrow \\infty}\\left(x_{n}-1\\right)^{n}$.\n\nProf. Dorin Arventiev\n\n## Note:\n\nWorking time: 3 hours.\n\nAll problems are mandatory.\n\nEach problem is graded from 0 to 7.\n\nNo points are given ex officio.\n\n## MATHEMATICS OLYMPIAD\n\nLocal stage - Constanța, 21.02.2016\n\n11th Grade\n\n## Grading and marking rubric\n\nNo points are given ex officio.",
      "answer": "2"
    },
    {
      "question": "6. Given that $P$ is a point on the hyperbola $C: \\frac{x^{2}}{4}-\\frac{y^{2}}{12}=1$, $F_{1}$ and $F_{2}$ are the left and right foci of the hyperbola $C$, and $M$ and $I$ are the centroid and incenter of $\\triangle P F_{1} F_{2}$, respectively. If $M I \\perp x$-axis, then the radius of the incircle of $\\triangle P F_{1} F_{2}$ is $\\qquad$",
      "answer": "\\sqrt{6}"
    },
    {
      "question": "Problem 2. On the side $AB$ of triangle $ABC$, a point $M$ is taken. It starts moving parallel to $BC$ until it intersects with $AC$, then it moves parallel to $AB$ until it intersects with $BC$, and so on. Is it true that after a certain number of such steps, point $M$ will return to its original position? If this is true, what is the minimum number of steps sufficient for the return?",
      "answer": "6"
    },
    {
      "question": "17*. From the formula for the circumference: $C=2 \\pi r$, we get: $\\pi=$ $=\\frac{C}{2 r}$ or $\\frac{1}{\\pi}=\\frac{2 r}{C}$. Consider a square with a perimeter of 2 and sequentially find the values of $h_{p}$ and $r_{p}$ using the formulas from the previous problem. By repeatedly performing the calculations, achieve that the values of $h_{p}$ and $r_{p}$ coincide in the first five decimal places. This will find the value of $\\frac{1}{\\pi}=\\frac{2 r}{\\bar{c}} \\approx \\frac{2 r}{2}=r$ with the corresponding accuracy. (This method of calculating $\\pi$ is called the method of isoperimeters.)",
      "answer": "0.3183098"
    },
    {
      "question": "15. The famous Goldbach Conjecture can be stated as: Any even number greater than 2 can be expressed as the sum of two prime numbers. Express the even number 88 as the sum of two prime numbers, how many ways are there? $(a+b$ and $b+a$ are considered the same representation method)",
      "answer": "4"
    },
    {
      "question": "79. A small boat sails upstream from downstream port $A$, and at a point 4 kilometers away from port $A$, it encounters a log drifting with the current. The boat continues its journey to reach upstream port $B$ and immediately turns around, arriving at port $A$ at the same time as the log. Given that the speed of the current is 2 meters/second, and the speed of the boat in still water is 8 meters/second, the distance between ports $A$ and $B$ is $\\qquad$ kilometers.",
      "answer": "8"
    },
    {
      "question": "5. As shown in Figure 2, in $\\triangle A B C$, $\\angle A B C=120^{\\circ}$,\n$A B=B C=2$. Take a point $D$ on side\n$A C$ (excluding $A$ and $C$), and fold $\\triangle A B D$\nalong line segment $B D$, to get\n$\\triangle P B D$. When plane\n$P B D \\perp$ plane $A B C$, the maximum distance from point $P$ to plane $A B C$ is $\\qquad$",
      "answer": "2"
    },
    {
      "question": "1. The wolf saw a roe deer several meters away from him and chased after her along a straight forest path. The wolf's jump is $22\\%$ shorter than the roe deer's jump. Both animals jump at a constant speed. All the roe deer's jumps are of the same length, and the wolf's jumps are also equal to each other. There is a period of time during which both the wolf and the roe deer make a certain whole number of jumps. Each time, it turns out that the wolf has made $t\\%$ more jumps than the roe deer. Find the greatest integer value of $\\mathrm{t}$, for which the wolf will not be able to catch the roe deer.",
      "answer": "28"
    },
    {
      "question": "Bogdanov I.I.\n\nIn the language of the AU tribe, there are two letters - \"a\" and \"u\". Some sequences of these letters are words, and each word contains no fewer than one and no more than 13 letters. It is known that if you write down any two words in a row, the resulting sequence of letters will not be a word. Find the maximum possible number of words in such a language.",
      "answer": "16256"
    },
    {
      "question": "4. The random variable $a$ is uniformly distributed on the interval $[-2 ; 1]$. Find the probability that all roots of the quadratic equation $a x^{2}-x-4 a+1=0$ in absolute value exceed 1.",
      "answer": "\\frac{7}{9}"
    },
    {
      "question": "5-1. Solve the inequality\n\n$$\n\\sqrt{3 x-7}-\\sqrt{3 x^{2}-13 x+13} \\geqslant 3 x^{2}-16 x+20\n$$\n\nIn your answer, specify the sum of all integer values of $x$ that satisfy the inequality.",
      "answer": "3"
    },
    {
      "question": "4. How many natural numbers are there that are less than or equal to the number 2019, and which are not divisible by any of the numbers 6, 9, and 15?",
      "answer": "1526"
    },
    {
      "question": "3.1.3 * Let $\\left\\{a_{n}\\right\\}$ be an arithmetic sequence, $a_{1}=19, a_{26}=-1$. Let $A=a_{n}+a_{n+1}+\\cdots+a_{n+6}$, where $n$ is a positive integer, find the minimum value of $|A|$.",
      "answer": "\\frac{7}{5}"
    },
    {
      "question": "## Task 3 - 290913\n\nIn a counting-out game, 11 children stand in a circle. One of these children recites the counting-out rhyme; with each syllable, one child is counted in a clockwise direction. The player who recites the counting-out rhyme is also included in the counting. The counting-out rhyme has 15 syllables. The child on whom the last syllable falls leaves the circle; the counting resumes with the next child, starting again from the beginning of the counting-out rhyme. This counting and elimination continues until only one player remains in the circle; this player is the winner.\n\na) At which child should the counting player start if he wants to win himself? (To formulate the answer, number the children and assign, for example, the counting player the number 1.)\n\nb) If you have the opportunity to work on a (small) computer, you should write a program that solves task a) for counting-out games with $k$ children and a counting-out rhyme of $s$ syllables.",
      "answer": "3"
    },
    {
      "question": "7. The function $f(x)$ satisfies for all real numbers $x$:\n$$\nf(2-x)=f(2+x), f(5+x)=f(5-x) \\text {, and } f(0)=0\n$$\n\nThen $f(x)$ has at least $\\qquad$ zeros in the interval $[-21,21]$.",
      "answer": "14"
    },
    {
      "question": "75. The three-digit numbers $\\overline{a b c}$ and $\\overline{c b a}$ form a pair of reverse-order numbers (such as 123 and 321, 778 and 877). If the difference between the two numbers in a reverse-order pair is 297, how many such reverse-order pairs are there?",
      "answer": "60"
    },
    {
      "question": "4. The base of a right quadrilateral prism is a rhombus with side length 2 and an angle of $30^{\\circ}$, and the height is 1. A section is made through the base edge at an angle of $60^{\\circ}$ to the base. The area of this section is $\\qquad$ .",
      "answer": "\\frac{4\\sqrt{3}}{3}"
    },
    {
      "question": "$9 \\cdot 4$ can be divided by 3, and the digits of each number are limited to $1,2,3$ (1,2,3 may not all be used) of all different natural numbers less than 200000, how many are there?\n(Shanghai, China High School Mathematics Competition, 1988)",
      "answer": "202"
    },
    {
      "question": "10,11\n\nThe edge of a regular tetrahedron is $a$. A plane $P$ passes through vertex $B$ and the midpoints of edges $A C$ and $A D$. A sphere is tangent to the lines $A B, A C, A D$ and that part of the plane $P$ which is inside the tetrahedron. Find the radius of the sphere. (Find all solutions).",
      "answer": "\\frac{\\sqrt{2}}{5\\\\sqrt{11}}"
    },
    {
      "question": "[ [Divisibility of numbers. General properties]\n\nAuthors: Binkov A.d,, Raskina I.v.\n\nSeveral whole heads of cheese were stored in the warehouse. At night, rats came and ate 10 heads, and all of them ate equally. Several rats got sick from overeating. The remaining seven rats the next night finished off the remaining cheese, but each rat was able to eat only half as much cheese as the night before. How much cheese was originally in the warehouse?\n\n#",
      "answer": "11"
    },
    {
      "question": "78. The Hope Primary School is 48 km away from the county town, part of which is an uphill road, and the rest is a downhill road. Principal Zhang rides a bicycle from the school to the county town, taking 4.2 hours to go and 3.8 hours to return. It is known that Principal Zhang rides a bicycle uphill at a speed of 10 km per hour, so he rides a bicycle downhill at a speed of $\\qquad$ km per hour.",
      "answer": "15"
    },
    {
      "question": "8. Let $f(x)=\\frac{4^{x}}{4^{x}+2}$, then $f\\left(\\frac{1}{1001}\\right)+f\\left(\\frac{2}{1001}\\right)+\\cdots+f\\left(\\frac{1000}{1001}\\right)=$",
      "answer": "500"
    },
    {
      "question": "463. From two ports A and B, two steamships set off simultaneously towards each other across the sea. The speed of each is constant. The first steamship arrived at B 16 hours after the meeting, while the second arrived at A 25 hours after the meeting. How long does it take for each steamship to travel the entire distance?",
      "answer": "36"
    },
    {
      "question": "At the mountain cabin, our coach said: \"If we continue at this comfortable pace of $4 \\mathrm{~km}$ per hour, we will arrive at the station 45 minutes after our train has left.\"\n\nThen he pointed to a group that was just passing us: \"They are using poles, and thus achieve an average speed of $6 \\mathrm{~km}$ per hour. They will be at the station half an hour before our train leaves.\"\n\nHow far was the station from the mountain cabin?\n\n(M. Volfová)\n\nHint. Recall the relationships between average speed, total distance, and required time.",
      "answer": "15"
    },
    {
      "question": "## Problem Statement\n\nCalculate the definite integral:\n\n$$\n\\int_{0}^{1} x^{2} \\sqrt{1-x^{2}} d x\n$$",
      "answer": "\\frac{\\pi}{16}"
    },
    {
      "question": "34. A company gathered for a meeting. Let's call a person sociable if in this company they have at least 20 acquaintances, and at least two of them are acquainted with each other. Let's call a person shy if in this company they have at least 20 strangers, and at least two of them are strangers to each other. It turned out that in the gathered company there are neither sociable nor shy people. What is the maximum number of people that can be in it\n\n$\\left(\\right.$ Folklore $^{(2)}$",
      "answer": "40"
    },
    {
      "question": "7.6. Find the volume of a regular triangular pyramid, the midpoint of the height of which is at a distance of 2 and $\\sqrt{5}$ from the lateral face and the lateral edge, respectively. Round your answer to two decimal places if necessary.",
      "answer": "485.42"
    },
    {
      "question": "13. Let $n$ be the smallest positive integer of 4 digits greater than or equal to 2016 that has the following property: there exists a positive integer $S$ such that\n\n$$\nS=\\sqrt{a+\\sqrt{b+\\sqrt{c+\\sqrt{d+S}}}}\n$$\n\nwhere $a, b, c, d$ are, in order, the thousands, hundreds, tens, and units digits of $n$. What is the value of $n$?",
      "answer": "2167"
    },
    {
      "question": "11. Given real numbers $x_{1}, x_{2}, x_{3}$ satisfy $x_{1}^{2}+x_{2}^{2}+x_{3}^{2}+x_{1} x_{2}+x_{2} x_{3}=2$. Then the maximum value of $\\left|x_{2}\\right|$ is $\\qquad$.",
      "answer": "2"
    },
    {
      "question": "2. In an airplane with 62 rows, each having 6 seats, passengers have seated themselves in such a way that in no 2 rows are seats occupied in the same positions. What is the maximum number of passengers that can be in the airplane?",
      "answer": "191"
    },
    {
      "question": "Task 2. On an island, there live knights who always tell the truth, and liars who always lie. One day, 30 inhabitants of this island sat around a round table. Each of them said one of two phrases: \"My left neighbor is a liar\" or \"My right neighbor is a liar.\" What is the smallest number of knights that can be at the table?",
      "answer": "10"
    },
    {
      "question": "Touching Circles\n\nTwo circles touch each other externally at point $A$. Through point $B$ on their common tangent $A B$, two lines are drawn, one intersecting the first circle at points $M$ and $N$, and the other intersecting the second circle at points $P$ and $Q$. It is known that $A B=6, B M=9, B P=5$. Find the ratio of the areas of triangles $M N O$ and $P Q O$, where point $O$ is the intersection of lines $M P$ and $N Q$.",
      "answer": "\\frac{625}{121}"
    },
    {
      "question": "10.055. A circle is inscribed in an isosceles triangle with a vertex angle of $120^{\\circ}$ and a lateral side of $a$ (Fig. 10.55). Find the radius of this circle.",
      "answer": "\\frac{\\sqrt{3}(2-\\sqrt{3})}{2}"
    },
    {
      "question": "7.5. Vasya had 101 real coins. One of them was replaced with a fake one (it differs in weight, but it is unknown whether it is lighter or heavier than a real coin). Vasya wants to find as many real coins as possible in one weighing on a balance scale without weights. What should he do and how many real coins will he be able to find?",
      "answer": "50"
    },
    {
      "question": "4. Friends Petya, Vasya, and Kolya live in the same entrance. Climbing from the 1st to the 3rd floor, Petya passes a number of steps that is a multiple of four. Vasya, climbing from the 1st to the 6th floor, passes a number of steps that is a multiple of nine. How many steps will Kolya pass, who climbs from the 1st to the 9th floor, if the number of steps between floors is the same and less than $22?$",
      "answer": "144"
    },
    {
      "question": "H6. The diagram shows four identical white triangles symmetrically placed on a grey square. Each triangle is isosceles and right-angled.\nThe total area of the visible grey regions is equal to the total area of the white triangles.\nWhat is the smallest angle in each of the (identical) grey triangles in the diagram?",
      "answer": "15"
    },
    {
      "question": "10.2.1. (12 points) Find the greatest integer value of $a$ for which the equation\n\n$$\n\\sqrt[3]{x^{2}-(a+7) x+7 a}+\\sqrt[3]{3}=0\n$$\n\nhas at least one integer root.",
      "answer": "11"
    },
    {
      "question": "3. In a cube $A B C D-$ $A_{1} B_{1} C_{1} D_{1}$ with edge length 2, $M$ and $N$ are the midpoints of edges $B B_{1}$ and $B_{1} C_{1}$, respectively. If $P$ is a moving point in the plane $D M N$, and the distance from point $P$ to the plane $B C C_{1} B_{1}$ is equal to the length of $P D$, the eccentricity of the trajectory of point $P$ is $\\qquad$.",
      "answer": "\\frac{2\\sqrt{34}}{17}"
    },
    {
      "question": "7. The volume of a cube in cubic metres and its surface area in square metres is numerically equal to four-thirds of the sum of the lengths of its edges in metres.\nWhat is the total volume in cubic metres of twenty-seven such cubes?",
      "answer": "216"
    },
    {
      "question": "## Task 4 - 330834\n\nOn a route $A B$, a cyclist $X$ travels from $A$ to $B$, and a second cyclist $Y$ travels from $B$ to $A$. Both start at the same time. Upon reaching $B$ or $A$, they immediately turn around and travel the same route back to $A$ or $B$, respectively, and then end their journey. It is assumed that each cyclist maintains a constant speed and that the time used for turning around can be neglected.\n\nOn the outbound trip, they met 30 minutes after the start at a point that is $7.5 \\mathrm{~km}$ from $A$. Another 30 minutes later, the cyclists were together again at a point between $A$ and $B$.\n\nDetermine all possibilities for how, according to this description,\n\na) the length of the route $A B$,\n\nb) the speeds of the cyclists $X$ and $Y$ can be!",
      "answer": "30"
    },
    {
      "question": "In the plane, consider the set $S$ of points with coordinates $(x, y)$, where $x, y \\in$ $\\{1,2, \\ldots, 2013\\}$. Two points in $S$ are called neighbors if they are at a distance of 1 from each other. Every second, a fly and some spiders move on the points of $S$ as follows: first, the fly either stays still or moves to a neighboring point of the one it is on. Then, each spider either stays still or moves to a neighboring point of the one it is on. Several spiders can be on the same point at the same time, and the fly as well as the spiders know each other's positions.\na) Determine the smallest integer $k$ for which $k$ spiders can always catch the fly in a finite time, regardless of the initial positions of the fly and the spiders on the points of $S$.\nb) Answer the same question if we assume this time that we are in the usual space and that $S$ is the set of points with coordinates $(x, y, z)$ where $x, y, z \\in\\{1,2, \\ldots, 2013\\}$.",
      "answer": "2"
    },
    {
      "question": "1. (Option 1) The decimal representation of the natural number $n$ contains sixty-three digits. Among these digits, there are twos, threes, and fours. No other digits are present. The number of twos is 22 more than the number of fours. Find the remainder when $n$ is divided by 9.",
      "answer": "8"
    },
    {
      "question": "Konyagin S.V.\n\nGiven a square with side length 1. From it, four corners are cut off - four triangles, each of which has two sides along the sides of the square and these sides are $1 / 3$ of their lengths. With the resulting 8-sided polygon, the same operation is performed: from each vertex, a triangle is cut off, two sides of which are $1 / 3$ of the corresponding sides of the 8-sided polygon, and so on. A sequence of polygons is obtained (each contained in the previous one). Find the area of the figure that is the intersection of all these polygons (i.e., formed by the points belonging to all polygons).",
      "answer": "\\frac{5}{7}"
    },
    {
      "question": "9. (40 points) For real numbers $a, b$ and $c$ it is known that $a b + b c + c a = 3$. What values can the expression $\\frac{a\\left(b^{2}+3\\right)}{a+b}+\\frac{b\\left(c^{2}+3\\right)}{b+c}+\\frac{c\\left(a^{2}+3\\right)}{c+a}$ take?",
      "answer": "6"
    },
    {
      "question": "6. Let $S$ be the set of all rational numbers in $\\left(0, \\frac{5}{8}\\right)$, and for a simplified fraction $\\frac{q}{p} \\in S, (p, q)=1$, define the function $f(q p)=\\frac{q+1}{p}$. Then the number of roots of $f(x)=\\frac{2}{3}$ in $S$ is $\\qquad$.",
      "answer": "5"
    },
    {
      "question": "19. The mammal with the longest lifespan on Earth is the bowhead whale. There are 10 bowhead whales, and this year their ages are consecutive natural numbers starting from 209 to 218. Then, when the youngest bowhead whale is $\\qquad$ years old, the sum of the ages of these 10 bowhead whales will be 2020 years less than the sum of their ages this year.",
      "answer": "7"
    },
    {
      "question": "[ Doubling the Median ] [ Law of Cosines ]\n\nDetermine the angle $A$ between the sides of lengths 2 and 4, if the median drawn from vertex $A$ is $\\sqrt{7}$.",
      "answer": "60"
    },
    {
      "question": "5. Let two regions on the Cartesian plane be $M=\\left\\{(x, y) \\in \\mathbf{R}^{2} \\mid 0 \\leqslant y \\leqslant \\min (2 x, 3-x)\\right\\}, N=$ $\\left\\{(x, y) \\in \\mathbf{R}^{2} \\mid t \\leqslant x \\leqslant t+2\\right\\}$, and let the area of the intersection of $M$ and $N$ be $f(t)$. When $0 \\leqslant t \\leqslant 1$, the expression for $f(t)$ is $\\qquad$.",
      "answer": "-\\frac{3}{2}^{2}++\\frac{5}{2}"
    },
    {
      "question": "3. Solve the equation $2^{1-2|x|}+2 \\cdot 4^{1+|x|}=17$.",
      "answer": "\\\\frac{1}{2}"
    },
    {
      "question": "240. Given a cube $A B C D A_{1} B_{1} C_{1} D_{1}$ with edge $a$. Find the radius of the smallest sphere that touches the lines $A B_{1}, B_{1} C$, $C D$ and $D A$.",
      "answer": "\\sqrt{1-\\frac{\\sqrt{2}}{2}}"
    },
    {
      "question": "57 Let $f(x)=2^{x}, g(x)=\\log _{\\sqrt{2}} 8 x$, then the value of $x$ that satisfies $f[g(x)]=g[f(x)]$ is\n\nLet's translate the problem and solution step by step.\n\n### Problem:\nLet \\( f(x) = 2^x \\) and \\( g(x) = \\log_{\\sqrt{2}} (8x) \\). Find the value of \\( x \\) that satisfies \\( f[g(x)] = g[f(x)] \\).\n\n### Solution:\n1. **Express \\( f[g(x)] \\):**\n   \\[\n   f[g(x)] = f\\left( \\log_{\\sqrt{2}} (8x) \\right)\n   \\]\n   Since \\( f(x) = 2^x \\), we have:\n   \\[\n   f\\left( \\log_{\\sqrt{2}} (8x) \\right) = 2^{\\log_{\\sqrt{2}} (8x)}\n   \\]\n\n2. **Simplify \\( 2^{\\log_{\\sqrt{2}} (8x)} \\):**\n   Recall that \\( \\log_{\\sqrt{2}} (8x) = \\frac{\\log_2 (8x)}{\\log_2 (\\sqrt{2})} \\). Since \\( \\log_2 (\\sqrt{2}) = \\frac{1}{2} \\), we get:\n   \\[\n   \\log_{\\sqrt{2}} (8x) = 2 \\log_2 (8x)\n   \\]\n   Therefore:\n   \\[\n   2^{\\log_{\\sqrt{2}} (8x)} = 2^{2 \\log_2 (8x)} = (2^{\\log_2 (8x)})^2 = (8x)^2 = 64x^2\n   \\]\n\n3. **Express \\( g[f(x)] \\):**\n   \\[\n   g[f(x)] = g(2^x)\n   \\]\n   Since \\( g(x) = \\log_{\\sqrt{2}} (8x) \\), we have:\n   \\[\n   g(2^x) = \\log_{\\sqrt{2}} (8 \\cdot 2^x) = \\log_{\\sqrt{2}} (8 \\cdot 2^x)\n   \\]\n\n4. **Simplify \\( \\log_{\\sqrt{2}} (8 \\cdot 2^x) \\):**\n   \\[\n   \\log_{\\sqrt{2}} (8 \\cdot 2^x) = \\log_{\\sqrt{2}} (2^3 \\cdot 2^x) = \\log_{\\sqrt{2}} (2^{3+x})\n   \\]\n   Using the change of base formula:\n   \\[\n   \\log_{\\sqrt{2}} (2^{3+x}) = \\frac{\\log_2 (2^{3+x})}{\\log_2 (\\sqrt{2})} = \\frac{3 + x}{\\frac{1}{2}} = 2(3 + x) = 6 + 2x\n   \\]\n\n5. **Set \\( f[g(x)] \\) equal to \\( g[f(x)] \\):**\n   \\[\n   64x^2 = 6 + 2x\n   \\]\n\n6. **Solve the equation \\( 64x^2 = 6 + 2x \\):**\n   \\[\n   64x^2 - 2x - 6 = 0\n   \\]\n   This is a quadratic equation. We can solve it using the quadratic formula \\( x = \\frac{-b \\pm \\sqrt{b^2 - 4ac}}{2a} \\), where \\( a = 64 \\), \\( b = -2 \\), and \\( c = -6 \\):\n   \\[\n   x = \\frac{-(-2) \\pm \\sqrt{(-2)^2 - 4 \\cdot 64 \\cdot (-6)}}{2 \\cdot 64} = \\frac{2 \\pm \\sqrt{4 + 1536}}{128} = \\frac{2 \\pm \\sqrt{1540}}{128} = \\frac{2 \\pm 2\\sqrt{385}}{128} = \\frac{1 \\pm \\sqrt{385}}{64}\n   \\]\n\n7. **Check the solutions:**\n   \\[\n   x = \\frac{1 + \\sqrt{385}}{64} \\quad \\text{and} \\quad x = \\frac{1 - \\sqrt{385}}{64}\n   \\]\n   Since \\( x \\) must be positive, we discard \\( x = \\frac{1 - \\sqrt{385}}{64} \\).\n\nThus, the value of \\( x \\) that satisfies the equation is:\n\\[\n\\boxed{\\frac{1 + \\sqrt{385}}{64}}\n\\]",
      "answer": "\\frac{1+\\sqrt{385}}{64}"
    },
    {
      "question": ". For what value of $k$ do we have the identity (true for all $a, b, c$):\n\n$$\n(a+b)(b+c)(c+a)=(a+b+c)(a b+b c+c a)+k a b c ?\n$$",
      "answer": "-1"
    },
    {
      "question": "Bakayev E.V.\n\nA secret base is surrounded by a transparent winding fence in the shape of a non-convex polygon, with a swamp outside. A straight power line consisting of 36 poles runs through the swamp, some of which are outside the base, and some are inside. (The power line does not pass through the vertices of the fence.) A spy walks around the base along the fence from the outside, keeping the fence always to his right. Each time he reaches the power line, he counts how many poles are to his left (he can see all of them). By the time the spy has walked around the entire fence, he has counted a total of 2015 poles. How many poles are inside the base?",
      "answer": "1"
    },
    {
      "question": "G5.4 Let $S_{1}, S_{2}, \\ldots, S_{10}$ be the first ten terms of an A.P., which consists of positive integers.\nIf $\\mathrm{S}_{1}+\\mathrm{S}_{2}+\\ldots+\\mathrm{S}_{10}=55$ and $\\left(\\mathrm{S}_{10}-\\mathrm{S}_{8}\\right)+\\left(\\mathrm{S}_{9}-\\mathrm{S}_{7}\\right)+\\ldots+\\left(\\mathrm{S}_{3}-\\mathrm{S}_{1}\\right)=d$, find $d$.",
      "answer": "16"
    },
    {
      "question": "## Task 2.\n\nLet $a \\geqslant 2018$ be a real number. In each of 2018 jars, there is a finite number of balls, with the mass of each ball being of the form $a^{k}$, where $k \\in \\mathbb{Z}$. The total mass of the balls in each jar is the same. What is the minimum number of balls of the same mass among the balls in the jars?",
      "answer": "2018"
    },
    {
      "question": "G3.2 In Figure $3, A$ and $B$ are two cones inside a cylindrical tube with length of 20 and diameter of 6 . If the volumes of $A$ and $B$ are in the ratio $3: 1$ and $b$ is the height of the cone $\\mathrm{B}$, find the value of $b$.",
      "answer": "5"
    },
    {
      "question": "14. A number $N$ is the product of three distinct primes. How many distinct factors does $N^{5}$ have?",
      "answer": "216"
    },
    {
      "question": "## Task 1 - 050911\n\nA turner needs half an hour to manufacture a certain part. Since several identical parts need to be produced, he is considering building a fixture that would allow each such part to be manufactured in 20 minutes. The construction of this fixture would take 4 hours.\n\nHow many parts at least need to be produced for the construction of the fixture to result in time savings?",
      "answer": "24"
    },
    {
      "question": "5. By arranging sticks in a hexagonal grid, students create shapes called hexes. In the image, a hex is shown that has five hexagons in the center (highlighted in the image). Sticks are sold in boxes containing 45 sticks each, and the price of one box is 1 euro and 25 cents.\n\nWhat is the minimum amount that students must spend to buy enough sticks to form a hex that has 13 hexagons in the center?\n\n![](https://cdn.mathpix.com/cropped/2024_05_30_e5ca9b3e8971fb159cf9g-5.jpg?height=483&width=448&top_left_y=110&top_left_x=1455)",
      "answer": "12.50"
    },
    {
      "question": "[ [rectangular parallelepipeds]\n\nThe diagonals of the faces of a rectangular parallelepiped are $\\sqrt{3}, \\sqrt{5}$, and 2. Find its volume.",
      "answer": "\\sqrt{6}"
    },
    {
      "question": "4. Let $n>1$ be a natural number. An equilateral triangle with side length $n$ is divided by lines parallel to its sides into congruent equilateral smaller triangles with side length 1. The number of smaller triangles that have at least one side on the side of the original triangle is 1 less than the number of all other smaller triangles. Determine all such $n$.",
      "answer": "5"
    },
    {
      "question": "12. A complicated choreography involves a row of 7 dancers equally spaced on a flat stage; in front of the row of dancers, there is an identical, parallel row of the same number of male dancers. The choreographer wants to assign to each dancer some male dancers in such a way that, by drawing on the stage the segment connecting the position of each dancer to that of the male dancers assigned to her, there are no segments that intersect except at the ends. She also wants the total number of segments drawn to be 13. How many ways are there to make the assignments that satisfy these properties?\n(A) 128\n(B) 343\n(C) 486\n(D) 924\n(E) 1716\n\n## Numerical answer problems -5 points",
      "answer": "924"
    },
    {
      "question": "5. As shown in the figure, $\\angle B=30^{\\circ}, \\angle A=60^{\\circ}, \\angle D=20^{\\circ}$, then the degree of $\\angle B C D$ (the angle with the arc part in the figure) is\n$\\qquad$ $\\circ$ .",
      "answer": "250"
    },
    {
      "question": "At the end of the quarter, Vovochka wrote down his current grades for singing in a row and placed a multiplication sign between some of them. The product of the resulting numbers turned out to be 2007. What grade does Vovochka get for singing in the quarter? (\"Kol\" is not given by the singing teacher.)\n\n#",
      "answer": "3"
    },
    {
      "question": "9. Let the real number $a<-1$, and the variable $x$ satisfies $x^{2}+a x \\leqslant-x$, and the minimum value of $x^{2}+a x$ is $-\\frac{1}{2}$. Then $a=$ $\\qquad$ .",
      "answer": "-\\frac{3}{2}"
    },
    {
      "question": "6. On domino tiles, there are pairs of numbers from 0 to 6 (including numbers 0 and 6) located on two fields. All numbers, except zero, are represented by the corresponding number of dots. An empty field without dots represents zero. In a complete set of domino tiles, there are also tiles with the same values on both fields (for example: 1 - 1, ...). How many dots are there in total in the entire set of domino tiles?",
      "answer": "168"
    },
    {
      "question": "On Mr. Chernick's farm, they had three hens. The first laid one egg every day, the second laid one egg every other day, and the third laid one egg every third day. Mr. Chernick bought two new hens at the market, which also laid eggs completely regularly - always one after a few days, with one of them laying twice as often as the other.\n\nMrs. Chernick calculated that all five hens laid a total of 155 eggs in 60 days. How often did the new hens lay eggs?\n\n(L. Hozová)",
      "answer": "42"
    },
    {
      "question": "13th Australian 1992 Problem A1 P, Q, R are adjacent vertices of a regular 9-gon with center O. M is the midpoint of the segment joining O to the midpoint of QR and N is the midpoint of PQ. Find the ∠ONM.",
      "answer": "30"
    },
    {
      "question": "14. Suppose there are 3 distinct green balls, 4 distinct red balls, and 5 distinct blue balls in an urn. The balls are to be grouped into pairs such that the balls in any pair have different colors. How many sets of six pairs can be formed?",
      "answer": "1440"
    },
    {
      "question": "## 33. Reservoir\n\nThe reservoir has the shape of a rectangular parallelepiped, the width of which is half the length. It is filled to three eighths of its height. When 76 hl [1 hl (hectoliter) = 0.1 m³] is added, the water level rises by 0.38 m. After this, two sevenths of the reservoir remain to be filled.\n\nWhat is the height of the reservoir?",
      "answer": "1.12"
    },
    {
      "question": "A triangle's three side lines are tangent to four circles whose radii are four consecutive elements of a geometric sequence. What is the largest angle of the triangle?",
      "answer": "90"
    },
    {
      "question": "## Task 4\n\nAdd the numbers 1 to 10.",
      "answer": "55"
    },
    {
      "question": "1. Let $x, y \\in \\mathbf{R}$, and the sets are\n$$\n\\begin{array}{l}\nA=\\left\\{(x, y) \\left\\lvert\\, x^{2}-3 x y+4 y^{2}=\\frac{7}{2}\\right.\\right\\}, \\\\\nB=\\{(x, y) \\mid k x+y=2\\}(k>0) .\n\\end{array}\n$$\n\nWhen $A \\cap B$ has only one element, $k=$ $\\qquad$",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "34th Eötvös 1930 Problem 2 A straight line is drawn on an 8 x 8 chessboard. What is the largest possible number of the unit squares with interior points on the line?",
      "answer": "15"
    },
    {
      "question": "4. If the positive integer $a$ makes the maximum value of the function $f(x)=x+\\sqrt{13-2 a x}$ also a positive integer, then this maximum value is equal to . $\\qquad$",
      "answer": "7"
    },
    {
      "question": "4. The function $y=f(x)$ is defined on the set $(0,+\\infty)$ and takes positive values on this set. It is known that for any points $A$ and $B$ on the graph of the function, the areas of the triangle $A O B$ and the trapezoid $A B H_{B} H_{A}$ are equal to each other $\\left(H_{A}, H_{B} - \\text{the bases of the perpendiculars dropped from points } A \\text{ and } B \\text{ to the x-axis}\\right)$;\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_12c6f2bb73fb9dd0ba03g-1.jpg?height=380&width=469&top_left_y=2317&top_left_x=1453)\n\n## Interregional Olympiad for Schoolchildren Based on Departmental Educational Organizations in Mathematics\n\n$O$ - the origin). Find all such functions. Justify your solution. Given that $f(1)=4$, write the number $f(2)$ in the answer.",
      "answer": "2"
    },
    {
      "question": "4. According to the 2002 Government Work Report at the Fifth Session of the Ninth National People's Congress on March 5, 2002: \"In 2001, the Gross Domestic Product (GDP) reached 959.33 billion yuan, an increase of $7.3 \\%$ over the previous year.\" If the GDP grows at this rate every year during the Tenth Five-Year Plan period (2001-2005), then by the end of the Tenth Five-Year Plan, China's annual GDP will be approximately $\\qquad$ .",
      "answer": "127165"
    },
    {
      "question": "A circular plate with a weight of $P=1 \\mathrm{~kg}$ is supported at three points along its circumference, which form an isosceles triangle. The vertex angle of this triangle is $\\alpha=30^{\\circ}$. What load rests on each support point?",
      "answer": "0.46411"
    },
    {
      "question": "## Task 4 - 220834\n\nA helicopter took off at 4:30 AM from a city $A$ and flew with a speed of $250 \\frac{\\mathrm{km}}{\\mathrm{h}}$ to a city $B$. It stayed there for 30 minutes and then flew back to $A$ on the same route with a speed of $200 \\frac{\\mathrm{km}}{\\mathrm{h}}$, arriving at $A$ on the same day at 11:45 AM.\n\nDetermine the length of the route from $A$ to $B$!",
      "answer": "750"
    },
    {
      "question": "1. Let the sum of the digits of the natural number $x$ be $S(x)$. Then the solution set of the equation $x+S(x)+S(S(x))+S(S(S(x)))=2016$ is $\\qquad$ .",
      "answer": "1980"
    },
    {
      "question": "$10 \\cdot 98$ Find the maximum positive integer $n$, such that there exist $n$ integer points in the coordinate plane, where any 3 points form a triangle, and the centroid of the triangle is not an integer point.",
      "answer": "8"
    },
    {
      "question": "## Task A-4.1.\n\nFor $n \\in \\mathbb{N}$, we define the complex number\n\n$$\na_{n}=(1+i)\\left(1+\\frac{i}{\\sqrt{2}}\\right)\\left(1+\\frac{i}{\\sqrt{3}}\\right) \\cdots\\left(1+\\frac{i}{\\sqrt{n}}\\right)\n$$\n\nCalculate\n\n$$\n\\left|a_{1}-a_{2}\\right|+\\left|a_{2}-a_{3}\\right|+\\cdots+\\left|a_{2019}-a_{2020}\\right| \\text {. }\n$$",
      "answer": "2019"
    },
    {
      "question": "Birgit has a list of four numbers. Luciano adds these numbers together, three at a time, and gets the sums 415, 442, 396, and 325. What is the sum of Birgit's four numbers?",
      "answer": "526"
    },
    {
      "question": "1. In the set $M=\\{1,2, \\cdots, 2018\\}$, the sum of elements whose last digit is 8 is $\\qquad$",
      "answer": "204626"
    },
    {
      "question": "1. Given positive integers $a, b, c (a<b<c)$ form a geometric sequence, and\n$$\n\\log _{2016} a+\\log _{2016} b+\\log _{2016} c=3 \\text {. }\n$$\n\nThen the maximum value of $a+b+c$ is $\\qquad$ .",
      "answer": "4066273"
    },
    {
      "question": "89. A white ball is placed into an urn containing two balls, after which one ball is randomly drawn from it. Find the probability that the drawn ball will be white, if all possible assumptions about the initial composition of the balls (by color) are equally likely.",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "5. In the sum $1+3+5+\\ldots+k$ of consecutive odd natural numbers, determine the largest addend $k$ such that $1+3+5+\\ldots+k=40000$.\n\n## Tasks worth 10 points:",
      "answer": "399"
    },
    {
      "question": "* Given the quadratic equation in $x$: $a(1+\\mathrm{i}) x^{2}+\\left(1+a^{2} \\mathrm{i}\\right) x+a^{2}+\\mathrm{i}=0$ has real roots, find the value of the real number $a$.",
      "answer": "-1"
    },
    {
      "question": "## problem statement\n\nLet $k$ be the coefficient of similarity transformation with the center at the origin. Is it true that point $A$ belongs to the image of plane $a$?\n\n$A(2; -5; 4)$\n\n$a: 5x + 2y - z + 3 = 0$\n\n$k = \\frac{4}{3}$",
      "answer": "0"
    },
    {
      "question": "10. Given a sequence $\\left\\{a_{n}\\right\\}$ with 100 terms, satisfying $a_{1}=0, a_{100}=475$,\nand $\\left|a_{k+1}-a_{k}\\right|=5(k=1,2, \\cdots, 99)$.\nThen the number of different sequences that meet the conditions is $\\qquad$.",
      "answer": "4851"
    },
    {
      "question": "10. (12 points) 1 kilogram of soybeans can be made into 3 kilograms of tofu, and 1 kilogram of soybean oil requires 6 kilograms of soybeans. Tofu sells for 3 yuan per kilogram, and soybean oil sells for 15 yuan per kilogram. A batch of soybeans weighs 460 kilograms, and after being made into tofu or soybean oil and sold, it yields 1800 yuan. In this batch, $\\qquad$ kilograms of soybeans were made into soybean oil.",
      "answer": "360"
    },
    {
      "question": "Folklore\n\nPetya wants to make an unusual die, which, as usual, should have the shape of a cube, with dots drawn on its faces (different numbers of dots on different faces), but at the same time, on any two adjacent faces, the number of dots must differ by at least two (at the same time, it is allowed for some faces to have more than six dots). How many dots in total are needed to be drawn for this?",
      "answer": "27"
    },
    {
      "question": "Pechkovskiy A.n.\n\nOn a chessboard of size $8 \\times 8$, 64 points are marked - the centers of all cells. Is it possible to separate all the points from each other by drawing 13 lines that do not pass through these points?",
      "answer": "14"
    },
    {
      "question": "4. At Ivanov's house, there stands an odd three-digit number. The digit in the tens place is twice as large as the digit in the units place. The digit in the hundreds place is the sum of the digits in the tens and units places. Determine all possibilities for Ivanov's house number that is divisible by 9.",
      "answer": "963"
    },
    {
      "question": "Example 7. Calculate the integral $I=\\int_{\\overparen{A B}}\\left(x^{2}-2 x y\\right) d x+\\left(y^{2}-\\right.$ $-2 x y) d y$, where $\\overparen{A B}$ is the arc of a parabola connecting points $A(-1,1)$ and $B(1,1)$ and passing through the origin.",
      "answer": "-\\frac{14}{15}"
    },
    {
      "question": "6. (10 points) Given that the natural number $n$ has 10 divisors, $2 n$ has 20 divisors, and $3 n$ has 15 divisors, then $6 n$ has $\\qquad$ divisors.",
      "answer": "30"
    },
    {
      "question": "5. The cross-section of a regular hexagonal pyramid SABCDEF is formed by a plane passing through the center of the base $A B C D E F$ and parallel to the median $C M$ of the lateral face $S C D$ and the apothem $S N$ of the lateral face $S A F$. The side of the base of the pyramid is 8, and the distance from the vertex $S$ to the cutting plane is $3 \\sqrt{13 / 7}$. Find the cosine of the angle between the cutting plane and the base plane. (20 points)",
      "answer": "\\frac{\\sqrt{3}}{4}"
    },
    {
      "question": "6. Let $a_{0}=-2, b_{0}=1$, and for $n \\geq 0$, let\n$$\n\\begin{array}{l}\na_{n+1}=a_{n}+b_{n}+\\sqrt{a_{n}^{2}+b_{n}^{2}}, \\\\\nb_{n+1}=a_{n}+b_{n}-\\sqrt{a_{n}^{2}+b_{n}^{2}} .\n\\end{array}\n$$\n\nFind $a_{2012}$.",
      "answer": "2^{1006}\\sqrt{2^{2010}+2}-2^{2011}"
    },
    {
      "question": "39. (5 points)\nFour pirates are dividing 15 identical gold coins. The first pirate must get at least 4 coins, the second pirate can get none, the third pirate must get at least 2 coins, and the fourth pirate must get at least 1 coin. There are $\\qquad$ different ways to divide the coins.",
      "answer": "165"
    },
    {
      "question": "8. The URL of the Hua Cup Competition is www.huabeisai.cn, and the letters form the following equation:\nwww+hua+bei+sai+cn=2008.\n\nIf each letter represents a digit from $0 \\sim 9$, the same letter represents the same digit, different letters represent different digits, and $\\mathrm{w}=8, \\mathrm{~h}=6, \\mathrm{a}=9, \\mathrm{c}=7$, then the minimum value of the three-digit number $\\overline{b e} i$ is _. $\\qquad$",
      "answer": "103"
    },
    {
      "question": "7. [4] My friend and I are playing a game with the following rules: If one of us says an integer $n$, the opponent then says an integer of their choice between $2 n$ and $3 n$, inclusive. Whoever first says 2007 or greater loses the game, and their opponent wins. I must begin the game by saying a positive integer less than 10 . With how many of them can I guarantee a win?",
      "answer": "6"
    },
    {
      "question": "5. Divide $1,2,3,4,5,6,7,8,9$ into 3 groups, each containing three numbers, then the probability that the sum of the numbers in each of these three groups is a prime number is",
      "answer": "\\frac{1}{140}"
    },
    {
      "question": "18. a) Into how many parts can a plane be divided by two closed curves, one of which is a circle and the other is the boundary of a square?\n\nb) $* *$ Into how many parts can space be divided by two closed surfaces, one of which is a sphere and the other is the surface of a cube?",
      "answer": "16"
    },
    {
      "question": "6-101. Let $x, y$ be positive numbers, and $S$ be the smallest number among $x, y+\\frac{1}{x}, \\frac{1}{y}$. Find the possible maximum value of $S$, and the values of $x, y$ that achieve this maximum value.",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "Pooolish A.\n\nChichikov is playing with Nozdryov. First, Nozdryov distributes 1001 nuts into three boxes. After looking at the distribution, Chichikov names any integer \\( N \\) from 1 to 1001. Then Nozdryov must, if necessary, move one or several nuts to an empty fourth box and present Chichikov with one or several boxes, where the total number of nuts is exactly \\( N \\). As a result, Chichikov will receive as many dead souls as the number of nuts Nozdryov moved. What is the maximum number of souls Chichikov can guarantee for himself, regardless of how Nozdryov plays?",
      "answer": "71"
    },
    {
      "question": "2.044. $\\left(\\frac{\\sqrt{x-a}}{\\sqrt{x+a}+\\sqrt{x-a}}+\\frac{x-a}{\\sqrt{x^{2}-a^{2}}-x+a}\\right): \\sqrt{\\frac{x^{2}}{a^{2}}-1} ; x>a>0$.\n\n2.044. $\\left(\\frac{\\sqrt{x-a}}{\\sqrt{x+a}+\\sqrt{x-a}}+\\frac{x-a}{\\sqrt{x^{2}-a^{2}}-x+a}\\right): \\sqrt{\\frac{x^{2}}{a^{2}}-1} ; x>a>0$.",
      "answer": "1"
    },
    {
      "question": "10. (20 points) Through the focus $F$ of the parabola $\\Gamma: x^{2}=4 y$, draw two different lines $l_{1}$ and $l_{2}$ with slopes $k_{1}$ and $k_{2}$ respectively, and $k_{1}+k_{2}=2$. Line $l_{1}$ intersects the parabola $\\Gamma$ at points $A$ and $B$, and line $l_{2}$ intersects the parabola $\\Gamma$ at points $C$ and $D$. The circles $\\odot M$ and $\\odot N$ have diameters $AB$ and $CD$ respectively. The line containing the common chord of these circles is denoted as $l$. Find the minimum distance from point $M$ to the line $l$.",
      "answer": "\\frac{3\\sqrt{5}}{4}"
    },
    {
      "question": "A Barber (friseur) from Béts moved to Debretzen to continue his trade; but he became impoverished within 3 years, although he had some money. In the first year, he was forced to spend half of his money, in the second year, a third of what he had brought; in the third year, 200 forints, when he only had $50 \\mathrm{Ft}$ left for the return trip: - how many forints did he have when he moved; and how much did he spend each year?",
      "answer": "1500"
    },
    {
      "question": "4.4. To the fraction $\\frac{1}{6}$, some fraction was added, and the result turned out to be a proper fraction with a denominator less than 8. What is the largest fraction that could have been added?",
      "answer": "\\frac{29}{42}"
    },
    {
      "question": "(2) Let $a<-1$, and the variable $x$ satisfies $x^{2}+a x \\leqslant-x$, and the minimum value of $x^{2}+a x$ is $-\\frac{1}{2}$, then $a=$ $\\qquad$.",
      "answer": "-\\frac{3}{2}"
    },
    {
      "question": "14. In the quadrilateral pyramid $P-ABCD$, the side face $PAD$ is an equilateral triangle, and it is perpendicular to the base $ABCD$, the base $ABCD$ is a rhombus with a side length of 2, $\\angle BAD=60^{\\circ}$, $N$ is the midpoint of $PB$, and the section $DAN$ intersects $PC$ at $M$. Find the size of the dihedral angle $P-AD-M$.",
      "answer": "45"
    },
    {
      "question": "6. Next to the number 2022 on the board, an unknown positive number less than 2022 was written. Then one of the numbers on the board was replaced by their arithmetic mean. This replacement was performed 9 more times, and the arithmetic mean was always an integer. Find the smaller of the numbers originally written on the board.",
      "answer": "998"
    },
    {
      "question": "$9.0 .18 \\times 0.81+0.18+0.81=(\\quad)$ .\n\n\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly. \n\nHowever, the provided text is already in a mathematical expression format and does not contain any non-English words or phrases to translate. The expression remains the same in English:\n\n$9.0 .18 \\times 0.81+0.18+0.81=(\\quad)$ .",
      "answer": "\\frac{139}{121}"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$\\lim _{n \\rightarrow \\infty} \\frac{(2 n+1)^{3}-(2 n+3)^{3}}{(2 n+1)^{2}+(2 n+3)^{2}}$",
      "answer": "-3"
    },
    {
      "question": "9.2. In a kindergarten, each child was given three cards, each of which had either \"MA\" or \"NYA\" written on it. It turned out that 20 children could form the word \"MAMA\" from their cards, 30 children could form the word \"NYANYA,\" and 40 children could form the word \"MANYA.\" How many children had all three cards the same?",
      "answer": "10"
    },
    {
      "question": "A point $D$ is chosen on side $B C$ of triangle $A B C$. The line tangent to the incircles of triangles $A B D$ and $A D C$ and different from $B C$ and $A D$ intersects segment $A D$ at $T$. If $A B=40 \\text{ cm}, A C=50 \\text{ cm}$, and $B C=60 \\text{ cm}$, determine the value of the length of $A T$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4a19aea8e08094ae376fg-07.jpg?height=539&width=1108&top_left_y=1079&top_left_x=546)",
      "answer": "15"
    },
    {
      "question": "Example 12 A deck of tri-color cards consists of 32 cards in total, with 10 cards of each color: red, yellow, and blue, numbered $1,2, \\cdots, 10$; there are also a big joker and a small joker, each numbered 0. Several cards are drawn from this deck, and their scores are calculated according to the following rule: each card numbered $k$ is worth $2^{k}$ points. If the total score of these cards is 2004, they are called a \"good\" card group. Find the number of \"good\" card groups.\n(2004 National Girls' Olympiad)",
      "answer": "1006009"
    },
    {
      "question": "\\section*{Exercise 2 - 330932}\n\nFor each positive integer \\(n\\), imagine another number \\(n^{\\prime}\\) formed according to the following rule: From the digit representation of \\(n\\) in the decimal system, the first digit is removed and appended to the end of the last digit.\n\nThen let \\(n^{\\prime}\\) be the number with the resulting digit representation. Investigate whether there are numbers \\(n\\) divisible by 7 for which \\(n^{\\prime}=n / 7\\) holds!",
      "answer": "10^{21}+\\frac{10^{21}-7}{69}"
    },
    {
      "question": "33. For $n=1,2,3, \\ldots$, let $a_{n}=n^{2}+100$, and let $d_{n}$ denote the greatest common divisor of $a_{n}$ and $a_{n+1}$. Find the maximum value of $d_{n}$ as $n$ ranges over all positive integers.",
      "answer": "401"
    },
    {
      "question": "4. In each of the following 7-digit natural numbers:\n1001011,5550000,3838383,7777777,\nevery digit in the number appears at least 3 times. Find the number of such 7 -digit natural numbers.",
      "answer": "2844"
    },
    {
      "question": "5. Maximum 15 points. The company \"Intelligence, Inc\" has developed a robot with artificial intelligence. To manufacture it, a special machine is required, which can produce 1 robot in 1 hour. The company owns a large number of such machines, but the created robot is so intelligent that it can produce an exact copy of itself in exactly 2 hours. \"Intelligence, Inc\" has received an urgent order for 3250 robots, which need to be manufactured in just 10 hours. The company's expenses for creating one robot (regardless of the time of its production and whether it was created by another robot or a special machine) are constant and amount to 100 monetary units, and the maintenance of one special machine during its operation to produce a robot costs the company 70 monetary units. \"Intelligence, Inc\" does not incur any other expenses for creating robots. What are the minimum costs the company will incur in fulfilling the order?",
      "answer": "328710"
    },
    {
      "question": "Problem 5.3. By September 1st, Vlad bought himself several ballpoint and gel pens. He noticed that if all the pens he bought were gel pens, he would have paid 4 times more than he actually did. And if all the pens were ballpoint pens, the purchase would have cost him 2 times less than the actual price. How many times more expensive is a gel pen compared to a ballpoint pen?",
      "answer": "8"
    },
    {
      "question": "## Problem Statement\n\nBased on the definition of the derivative, find $f^{\\prime}(0)$:\n\n$$\nf(x)=\\left\\{\\begin{array}{c}\nx+\\arcsin \\left(x^{2} \\sin \\frac{6}{x}\\right), x \\neq 0 \\\\\n0, x=0\n\\end{array}\\right.\n$$",
      "answer": "1"
    },
    {
      "question": "## Task 4 - 240814\n\nFrom three congruent isosceles right triangles with given leg length $a$, a trapezoid can be formed.\n\na) Draw such a trapezoid!\n\nb) Such a trapezoid can be divided into four congruent trapezoids. Draw such a division!\n\nc) Determine the lengths of the parallel sides, the height, and the area of one of these smaller trapezoids in terms of $a$!",
      "answer": "\\frac{3}{8}^{2}"
    },
    {
      "question": "19. [11] Let $P$ be a polynomial with $P(1)=P(2)=\\cdots=P(2007)=0$ and $P(0)=2009$ !. $P(x)$ has leading coefficient 1 and degree 2008. Find the largest root of $P(x)$.",
      "answer": "4034072"
    },
    {
      "question": "A passenger left their belongings in an automated storage locker, and when they came to retrieve their items, they realized they had forgotten the number. They only remember that the numbers 23 and 37 were in the code. To open the locker, the correct five-digit number must be entered. What is the minimum number of codes that need to be tried to definitely open the locker?\n\n#",
      "answer": "356"
    },
    {
      "question": "$\\begin{array}{l}\\text { II. ( } 50 \\text { points) } \\\\ \\text { Infinitely many non-degenerate } \\triangle \\text {, summing the shortest side, the second longest side, and the longest side of each } \\\\ \\text { to form } 1 \\text { triangle, what is the minimum value of the cosine of the largest angle of the resulting triangle? }\\end{array}$",
      "answer": "1-\\sqrt{2}"
    },
    {
      "question": "13.196. Two hours after leaving the factory, the driver looked at the speedometer and noticed that he had only traveled 112 km. He mentally calculated that if he continued at the same speed, he would be 30 minutes late in delivering the cargo to the station. Therefore, the driver increased his speed and arrived at the station even 30 minutes early. Determine the initial and subsequent speeds of the car, if the distance from the factory to the station according to the speedometer is 280 km.",
      "answer": "56"
    },
    {
      "question": "8 The line $\\sqrt{3} x-y-\\sqrt{3}=0$ intersects the parabola $y^{2}=4 x$ at points $A$ and $B$ ($A$ is above the x-axis), and intersects the x-axis at point $F$. If $\\overrightarrow{O F}=\\lambda \\overrightarrow{O A}+\\mu \\overrightarrow{O B}$, then $\\mu^{2}-$ $\\lambda^{2}=$ $\\qquad$ .",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "4. Throughout the year, Marko took several exams and scored a certain number of points on them. If Marko were to score 89 points on his next exam, his average score on the exams would be 91. However, if he were to score 64 points on the next exam, his average score on the exams would then be 86. How many exams has Marko taken so far?",
      "answer": "4"
    },
    {
      "question": "3. Solve the equation\n\n$$\n\\sqrt{\\frac{x-3}{11}}+\\sqrt{\\frac{x-4}{10}}+\\sqrt{\\frac{x-5}{9}}=\\sqrt{\\frac{x-11}{3}}+\\sqrt{\\frac{x-10}{4}}+\\sqrt{\\frac{x-9}{5}}\n$$",
      "answer": "14"
    },
    {
      "question": "Bogdanov I.I.\n\nThe Tsar summoned two sages. He gave the first 100 blank cards and ordered him to write a positive number on each (the numbers do not have to be different), without showing them to the second. Then the first can tell the second several different numbers, each of which is either written on some card or equal to the sum of numbers on some cards (without specifying how each number was obtained). The second must determine which 100 numbers are written on the cards. If he cannot, both will have their heads cut off; otherwise, a number of hairs will be plucked from each beard equal to the number of numbers the first told the second. How can the sages, without communicating, stay alive and lose the minimum number of hairs?",
      "answer": "101"
    },
    {
      "question": "## Task Condition\n\nApproximately calculate using the differential.\n\n$y=x^{7}, x=2,002$",
      "answer": "128.896"
    },
    {
      "question": "6. (8 points) On the board, 34 ones are written. Every minute, Karlson erases two arbitrary numbers and writes their sum on the board, and then eats a number of candies equal to the product of the two erased numbers. What is the maximum number of candies he could have eaten in 34 minutes?",
      "answer": "561"
    },
    {
      "question": "1. If the solution set of the inequality $|x+a|<b$ with respect to $x$ is $\\{x \\mid 2<x<4\\}$, then the value of $a b$ is $\\qquad$",
      "answer": "-3"
    },
    {
      "question": "[ Cutting into parts with special properties ]\n\nFor what smallest $n$ can a square $n \\times n$ be cut into squares $40 \\times 40$ and $49 \\times 49$ so that squares of both types are present?\n\n#",
      "answer": "2000"
    },
    {
      "question": "A jar contains 6 crayons, of which 3 are red, 2 are blue, and 1 is green. Jakob reaches into the jar and randomly removes 2 of the crayons. What is the probability that both of these crayons are red?",
      "answer": "\\frac{1}{5}"
    },
    {
      "question": "21 Find the integer part of\n$$\n\\frac{1}{\\frac{1}{2003}+\\frac{1}{2004}+\\frac{1}{2005}+\\frac{1}{2006}+\\frac{1}{2007}+\\frac{1}{2008}+\\frac{1}{2009}} .\n$$",
      "answer": "286"
    },
    {
      "question": "6.032. $\\sqrt{x+\\sqrt{x+11}}+\\sqrt{x-\\sqrt{x+11}}=4$.\n\n6.032. $\\sqrt{x+\\sqrt{x+11}}+\\sqrt{x-\\sqrt{x+11}}=4$.",
      "answer": "5"
    },
    {
      "question": "9. The common ratio of the geometric sequence $a+\\log _{2} 3, a+\\log _{4} 3, a+\\log _{8} 3$ is",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Shapovalov A.V.\n\nA paper rectangular triangle was folded along a straight line so that the vertex of the right angle coincided with another vertex.\n\na) In what ratio are the diagonals of the resulting quadrilateral divided by their point of intersection?\n\nb) The resulting quadrilateral was cut along the diagonal coming out of the third vertex of the original triangle. Find the area of the smallest piece of paper formed.",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "14th USAMO 1985 Problem 5 0 < a 1 ≤ a 2 ≤ a 3 ≤ ... is an unbounded sequence of integers. Let b n = m if a m is the first member of the sequence to equal or exceed n. Given that a 19 = 85, what is the maximum possible value of a 1 + a 2 + ... + a 19 + b 1 + b 2 + ... + b 85 ?",
      "answer": "1700"
    },
    {
      "question": "Aerith bakes some cookes. On the first day, she gives away 1 cookie and then $1 / 8$ of the remaining cookies; on the second day, she gives away 2 cookies and then $1 / 8$ of the remaining cookies, and so on. On the 7th day, she gives away 7 cookies and then there are none left. How many cookies did she bake?",
      "answer": "49"
    },
    {
      "question": "Exercise 4. A group of 20 dwarves coming out of the mine sit around a round table to count the gold nuggets each has mined. They make the following observations:\n\n$\\triangleright$ The difference in the number of nuggets between two neighboring dwarves is always 2 or 3.\n\n$\\triangleright$ All the dwarves have mined a different number of nuggets.\n\nWhat is the greatest possible difference in nuggets between the dwarf who found the most and the one who found the least?",
      "answer": "29"
    },
    {
      "question": "10. (10 points) Simplify each of the fractions $\\frac{1}{2014}, \\frac{2}{2014}, \\cdots, \\frac{2012}{2014}, \\frac{2013}{2014}$ to their simplest form, and find the sum of all the fractions that still have 2014 as their denominator after simplification.",
      "answer": "468"
    },
    {
      "question": "8. The equation $5 z^{4}+10 z^{3}+10 z^{2}+5 z+1=0$ has four complex roots, the real part of the complex root with the largest modulus is $\\qquad$ _.",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "6. A bag contains $m$ red balls and $n$ white balls, $m>n \\geq 4$. Now, two balls are drawn at random. If the probability of drawing two balls of the same color is equal to the probability of drawing two balls of different colors, then the number of arrays $(m, n)$ that satisfy the relation $m+n \\leq 40$ is $\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "Through every point $M$ of the plane, two hyperbolas can be drawn, whose equations, referred to the rectangular coordinate axes $O x, O y$, are\n\n$$\na^{2} x y+a y+x=0\n$$\n\nwhere $a$ is a variable parameter; on which curve $(C)$ must the point $M$ lie so that the hyperbolas intersect at right angles at this point?\n\nHow many points does the curve $(C)$ intersect the hyperbolas with the equation $a^{2} x y+a y+x=0$. Only those points should be considered which are neither at infinity nor coincide with the origin of the coordinate system.\n\nWhat relationship must hold between $a$ and $b$ so that the following two hyperbolas, whose equations are\n\n$$\n\\begin{aligned}\n& a^{2} x y+a y+x=0 \\\\\n& b^{2} x y+b y+x=0\n\\end{aligned}\n$$\n\nintersect at right angles at a point different from the origin of the coordinate system. This relationship defines a curve $(C')$; the curve must be constructed. How many finite and non-origin points does the previously defined curve $(C)$ intersect the curve $(C')$?\n\n(Licentiatus stipendium competition written paper, 1894.)",
      "answer": "1"
    },
    {
      "question": "16. (1990 IMO Japan First Round Selection Test) Let $x, y, z > 0$, and $x + y + z = 1$, find the minimum value of $\\frac{1}{x} + \\frac{4}{y} + \\frac{9}{z}$.",
      "answer": "36"
    },
    {
      "question": "33. Today is January 18, 2020, and it is displayed on the electronic calendar as 20200118. If there is a multi-digit number $202001188462 \\cdots \\cdots$, starting from the 9th digit, each digit is the unit digit of the product of the three preceding digits. According to this rule, what is the 2020th digit of this multi-digit number? $\\qquad$ .",
      "answer": "2"
    },
    {
      "question": "5. The number of 4-digit numbers whose sum of digits equals 12 is. $\\qquad$",
      "answer": "342"
    },
    {
      "question": "7. (10 points) 6 small circles of equal radius and 1 large circle are arranged as shown in the figure. The area of the large circle is 120. Therefore, the area of one \n\nsmall circle is $\\qquad$ .",
      "answer": "40"
    },
    {
      "question": "1. (6 points) Calculate: $\\frac{(2015-201.5-20.15)}{2.015}=$",
      "answer": "890"
    },
    {
      "question": "2. (7 points) The kids were given the task to convert the turtle's speed from centimeters per second to meters per minute. Masha got an answer of 25 m/min, but she thought there were 60 cm in a meter and 100 seconds in a minute. Help Masha find the correct answer.",
      "answer": "9"
    },
    {
      "question": "5. On the side $\\overline{A B}$ of the rectangle $A B C D$, several points have been chosen, on the sides $\\overline{B C}$ and $\\overline{D A}$ four points each, and on the side $\\overline{C D}$ five points. None of the chosen points is a vertex of the given rectangle. Using the chosen points as vertices, exactly 4040 quadrilaterals can be drawn such that at most one vertex lies on the segment $\\overline{A B}$. How many points were chosen on the side $\\overline{A B}$?",
      "answer": "13"
    },
    {
      "question": "1.041. $\\frac{\\left(4-3.5 \\cdot\\left(2 \\frac{1}{7}-1 \\frac{1}{5}\\right)\\right): 0.16}{X}=\\frac{3 \\frac{2}{7}-\\frac{3}{14}: \\frac{1}{6}}{41 \\frac{23}{84}-40 \\frac{49}{60}}$.",
      "answer": "1"
    },
    {
      "question": "Example 11 For real number $x$, find the maximum value of the function $f(x)=$ $\\sqrt{8 x-x^{2}}-\\sqrt{14 x-x^{2}-48}$.",
      "answer": "2\\sqrt{3}"
    },
    {
      "question": "## Task B-4.2.\n\nDetermine the natural number $n$, so that the ratio of the seventh term counting from the beginning to the seventh term counting from the end, in the expansion of the binomial $\\left(\\sqrt[3]{2}+\\frac{1}{\\sqrt[3]{3}}\\right)^{n}$, is equal to $\\frac{1}{6}$.",
      "answer": "9"
    },
    {
      "question": "5. (8 points) Multiply 4.02 by a two-digit integer, and the product is an integer. Then, 10 times this product is $\\qquad$ .",
      "answer": "2010"
    },
    {
      "question": "6. Calculate with an accuracy of one-tenth the value of the expression $\\sqrt{86+41 \\sqrt{86+41 \\sqrt{86+\\ldots}}}$",
      "answer": "43"
    },
    {
      "question": "9. Let $O, O_{1}, O_{2}, O_{3}, O_{4}$ be points such that $O_{1}, O, O_{3}$ and $O_{2}, O, O_{4}$ are collinear in that order, $O O_{1}=$ $1, O O_{2}=2, O O_{3}=\\sqrt{2}, O O_{4}=2$, and $\\measuredangle O_{1} O O_{2}=45^{\\circ}$. Let $\\omega_{1}, \\omega_{2}, \\omega_{3}, \\omega_{4}$ be the circles with respective centers $O_{1}, O_{2}, O_{3}, O_{4}$ that go through $O$. Let $A$ be the intersection of $\\omega_{1}$ and $\\omega_{2}, B$ be the intersection of $\\omega_{2}$ and $\\omega_{3}, C$ be the intersection of $\\omega_{3}$ and $\\omega_{4}$, and $D$ be the intersection of $\\omega_{4}$ and $\\omega_{1}$, with $A, B, C, D$ all distinct from $O$. What is the largest possible area of a convex quadrilateral $P_{1} P_{2} P_{3} P_{4}$ such that $P_{i}$ lies on $O_{i}$ and that $A, B, C, D$ all lie on its perimeter?",
      "answer": "8+4\\sqrt{2}"
    },
    {
      "question": "Shen A.H.\n\nThere is an infinite one-way strip of cells, numbered with natural numbers, and a bag with ten stones. Initially, there are no stones in the cells of the strip. The following actions can be performed:\n\n- move a stone from the bag to the first cell of the strip or vice versa;\n- if there is a stone in the cell with number \\$\\$, then a stone can be moved from the bag to the cell with number $\\$ \\mathrm{i}+1 \\$$ or vice versa.\n\nIs it possible, acting according to these rules, to place a stone in the cell with number 1000?",
      "answer": "1023"
    },
    {
      "question": "Example 13. Find the flux of the vector field $\\mathbf{v}=(x-2 y+1) \\mathbf{i}+(2 x+ y-3 z) \\mathbf{j}+(3 y+z) \\mathbf{k}$ through the part of the sphere $x^{2}+y^{2}+z^{2}$, located in the first octant, into the region where $x^{2}+y^{2}+z^{2}>1$.",
      "answer": "\\frac{3}{4}\\pi"
    },
    {
      "question": "Problem 3. In a certain company, there are 100 shareholders, and any 66 of them collectively own at least $50\\%$ of the company's shares. What is the largest percentage of all shares that one shareholder can own? (The percentage of shares in the company owned by a shareholder can be non-integer.)",
      "answer": "25"
    },
    {
      "question": "1st ASU 1967 Problem 9 ABCD is a unit square. One vertex of a rhombus lies on side AB, another on side BC, and a third on side AD. Find the area of the set of all possible locations for the fourth vertex of the rhombus. Solution",
      "answer": "2\\frac{1}{3}"
    },
    {
      "question": "I4.4 Find the value of $1+3^{c}+\\left(3^{c}\\right)^{2}+\\left(3^{c}\\right)^{3}+\\left(3^{c}\\right)^{4}$ divided by $1+3+3^{2}+3^{3}+3^{4}$, denoted as $d$.",
      "answer": "0"
    },
    {
      "question": "G1.3 Given that $\\tan \\alpha=-\\frac{1}{2}$. If $c=\\frac{2 \\cos \\alpha-\\sin \\alpha}{\\sin \\alpha+\\cos \\alpha}$, find the value of $c$.",
      "answer": "5"
    },
    {
      "question": "1. In the cube $A B C D-A_{1} B_{1} C_{1} D_{1}$, $O$ is the center of face $A B C D$, and $O_{1}$ is the center of face $A D D_{1} A_{1}$. Find the cosine value of the angle formed by the skew lines $D_{1} O$ and $B O_{1}$.",
      "answer": "\\frac{5}{6}"
    },
    {
      "question": "3. The sides $B C$ and $A D$ of a quadrilateral $A B C D$ are parallel and the diagonals intersect in $O$. For this quadrilateral $|C D|=|A O|$ and $|B C|=|O D|$ hold. Furthermore $C A$ is the angular bisector of angle $B C D$.\n\nDetermine the size of angle $A B C$.\n\nAttention: the figure is not drawn to scale.\n\nYou have to write down your reasoning step by step in text and formulas. No points will be awarded for annotations in a picture\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_e07875fe43bd53054a56g-1.jpg?height=414&width=505&top_left_y=1483&top_left_x=1392)\nalone.",
      "answer": "126"
    },
    {
      "question": "Example 10. At an enterprise, products of a certain type are manufactured on three production lines. The first line produces $30 \\%$ of the products from the total production volume, the second line - $25 \\%$, and the third line produces the remaining part of the products. Each line is characterized by the following percentages of good products: $97 \\%, 98 \\%, 96 \\%$. Determine the probability that a randomly selected product, produced by the enterprise, will be defective.",
      "answer": "0.032"
    },
    {
      "question": "10.208. Perpendiculars are drawn from the vertex of the acute angle of a rhombus to the lines containing the sides of the rhombus to which this vertex does not belong. The length of each perpendicular is 3 cm, and the distance between their bases is $3 \\sqrt{3}$ cm. Calculate the lengths of the diagonals of the rhombus.",
      "answer": "6"
    },
    {
      "question": "14. Use 6 white beads, 8 black beads, and 1 red bead to string into a necklace. How many different ways are there to do this?",
      "answer": "1519"
    },
    {
      "question": "[ Unusual Constructions (Miscellaneous). ] [ Measuring the lengths of segments and measures of angles. Adjacent angles. $]$\n\nThere is an angle ruler with an angle of $70^{\\circ}$. How can you construct an angle of $40^{\\circ}$ with it?\n\n#",
      "answer": "40"
    },
    {
      "question": "17. From a square with a side length of 20, remove a rectangle with an area of 36, where the two side lengths of the rectangle are integers, and one side of the rectangle is part of one side of the square. The maximum perimeter of the remaining shape is $\\qquad$.",
      "answer": "116"
    },
    {
      "question": "1. What is the maximum number of lattice points (i.e. points with integer coordinates) in the plane that can be contained strictly inside a circle of radius 1 ?",
      "answer": "4"
    },
    {
      "question": "[ Decimal numeral system ] $[\\quad$ Case enumeration $\\quad]$\n\n## Author: Raskina I.V.\n\nThe year 2009 has the property that by rearranging the digits of the number 2009, it is impossible to obtain a smaller four-digit number (numbers do not start with zero). In which year will this property reappear for the first time?",
      "answer": "2022"
    },
    {
      "question": "6. (10 points) For a natural number $N$, if at least seven of the nine natural numbers from 1 to 9 are factors of $N$, then $N$ is called a \"Seven-Star Number\". Among the natural numbers greater than 2000, the smallest \"Seven-Star Number\" is $\\qquad$",
      "answer": "2016"
    },
    {
      "question": "7, 10 consecutive four-digit numbers whose sum is a multiple of 37, the smallest sum of these 10 numbers is",
      "answer": "10175"
    },
    {
      "question": "## SUBJECT III\n\nDetermine the measure of the angle formed by the bisectors of two consecutive angles in a parallelogram.",
      "answer": "90"
    },
    {
      "question": "Task 3. Determine the largest five-digit number such that the product of its digits is 2520?",
      "answer": "98751"
    },
    {
      "question": "2. Through the right focus of the hyperbola $x^{2}-\\frac{y^{2}}{2}=1$, a line $l$ intersects the hyperbola at points $A$ and $B$. If a real number $\\lambda$ makes $|A B|=$ $\\lambda$, and there are exactly 3 such lines $l$, then $\\lambda=$ $\\qquad$.",
      "answer": "4"
    },
    {
      "question": "The dandelion blooms in the morning, flowers yellow for two days, turns white on the third morning, and by evening it has scattered. Yesterday afternoon, there were 20 yellow and 14 white dandelions on the meadow, and today there are 15 yellow and 11 white.\n\na) How many yellow dandelions were there on the meadow the day before yesterday?\n\nb) How many white dandelions will there be on the meadow tomorrow?",
      "answer": "25"
    },
    {
      "question": "II. (50 points) There are 26 stamp collectors from different countries who want to exchange the latest commemorative stamps issued by their respective countries through correspondence. To ensure that each of the 26 individuals ends up with a complete set of the latest commemorative stamps from all 26 countries, what is the minimum number of letters they need to send?",
      "answer": "50"
    },
    {
      "question": "16 Let $x>0$, then the minimum value of $f(x)=\\frac{\\left(x+\\frac{1}{x}\\right)^{4}-\\left(x^{4}+\\frac{1}{x^{4}}\\right)}{\\left(x+\\frac{1}{x}\\right)^{3}-\\left(x^{3}+\\frac{1}{x^{3}}\\right)}$ is $\\qquad$ .",
      "answer": "\\frac{7}{3}"
    },
    {
      "question": "9. (12 points) Select 8 different digits from the numbers $1-9$ to fill in the squares below, so that the vertical equation holds. The largest possible four-digit number is $\\qquad$ .\n\n\n\nThe translation is provided while preserving the original text's line breaks and format.",
      "answer": "1769"
    },
    {
      "question": "641. Study the symmetric properties of an equilateral triangle.",
      "answer": "3"
    },
    {
      "question": "7. Given $F_{1}, F_{2}$ are the left and right foci of the hyperbola $C: \\frac{x^{2}}{4}-\\frac{y^{2}}{12}=1$, point $P$ is on the hyperbola $C$, and $G, I$ are the centroid and incenter of $\\triangle F_{1} P F_{2}$, respectively. If $G I \\| x$-axis, then the circumradius $R$ of $\\triangle F_{1} P F_{2}$ is $\\qquad$ ـ.",
      "answer": "5"
    },
    {
      "question": "283. Find a number of the form $7 x 36 y 5$, divisible by 1375.",
      "answer": "713625"
    },
    {
      "question": "148. A plane intersecting the surface of a triangular pyramid divides the medians of the faces emanating from one vertex in the ratios $2: 1, 1: 2, 4: 1$ respectively (counting from the vertex). In what ratio does this plane divide the volume of the pyramid?",
      "answer": "\\frac{7123}{16901}"
    },
    {
      "question": "11. Master Wang drives from location A to location B to deliver goods. If he travels at a speed of 60 kilometers per hour both ways, he can return to location A on time. However, upon arriving at location B, he finds that his speed from A to B was only 55 kilometers per hour. If he wants to return to location A on time, at what speed should he drive back?",
      "answer": "66"
    },
    {
      "question": "3. Given $13a - 4b = 11$, then $2013 - (20b - 65a) =$",
      "answer": "2068"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the numerical sequence:\n\n$$\n\\lim _{n \\rightarrow \\infty} \\frac{2^{n+1}+3^{n+1}}{2^{n}+3^{n}}\n$$",
      "answer": "3"
    },
    {
      "question": "Example 3 Given real numbers $x_{1}, x_{2}, \\cdots, x_{10}$ satisfy $\\sum_{i=1}^{10}\\left|x_{i}-1\\right| \\leqslant 4, \\sum_{i=1}^{10}\\left|x_{i}-2\\right| \\leqslant 6$. Find the average value $\\bar{x}$ of $x_{1}, x_{2}, \\cdots, x_{10}$. (2012, Zhejiang Province High School Mathematics Competition)",
      "answer": "1.4"
    },
    {
      "question": "Sure, here is the translation:\n\n---\n\na) What is the value of the following fraction?\n\n$$\n\\frac{423134 \\cdot 846267 - 423133}{423133 \\cdot 846267 + 423134}\n$$\n\nb) What number should be subtracted from the numerator and added to the denominator of the fraction $\\frac{52367}{47633}$ so that after simplification, the value of the fraction is $\\frac{17}{83}$?\n\nc) What is the sum of the following fractions?\n\n$$\n\\frac{7}{2 \\cdot 9} + \\frac{7}{9 \\cdot 16} + \\frac{7}{16 \\cdot 23} + \\cdots + \\frac{7}{65 \\cdot 72}\n$$",
      "answer": "\\frac{35}{72}"
    },
    {
      "question": "15. Minimum Sum (9th grade, 4 points). There are a lot of symmetric dice. They are thrown simultaneously. With some probability $p>0$, the sum of the points can be 2022. What is the smallest sum of points that can fall when these dice are thrown with the same probability $p$?",
      "answer": "337"
    },
    {
      "question": "7. In the Cartesian coordinate system, it is known that the parabola $y=k x^{2}(k>0)$ intersects the circle $(x-a)^{2}+(y-b)^{2}=r^{2}$ at least at 3 points, one of which is the origin, and the other two lie on the line $y=k x+b$. What is the minimum value of the real number $b$? $\\qquad$ .",
      "answer": "2"
    },
    {
      "question": "Task 1. On the faces of a six-faced dice, the numbers 1, 2, 3, 4, 5, and 6 are placed. The dice is thrown and falls on the table. After this, the numbers on all faces except one are visible. The numbers on the five visible faces are multiplied. Find the probability that this product is divisible by 16.\n\nAnswer: $0.5$.",
      "answer": "0.5"
    },
    {
      "question": "Problem 9.4. For positive numbers $a, b, c$, it is known that\n\n$$\n\\frac{a+b+c}{a+b-c}=7, \\quad \\frac{a+b+c}{a+c-b}=1.75\n$$\n\nWhat is the value of $\\frac{a+b+c}{b+c-a}$?",
      "answer": "3.5"
    },
    {
      "question": "Problem 1. The sum of the first four terms of an arithmetic progression, as well as the sum of the first nine terms, are natural numbers. In addition, its first term $b_{1}$ satisfies the inequality $b_{1} \\leqslant \\frac{3}{4}$. What is the greatest value that $b_{1}$ can take?",
      "answer": "\\frac{11}{15}"
    },
    {
      "question": "2. Find the number of natural numbers $k$, not exceeding 454500, such that $k^{2}-k$ is divisible by 505. Answer: 3600.",
      "answer": "3600"
    },
    {
      "question": "Example 17 Real numbers $x_{1}, x_{2}, \\cdots, x_{2001}$ satisfy $\\sum_{k=1}^{2000}\\left|x_{k}-x_{k+1}\\right|=2001$, let $y_{k}=\\frac{1}{k}\\left(x_{1}+x_{2}+\\cdots+\\right.$ $\\left.x_{k}\\right), k=1,2, \\cdots, 2001$. Find the maximum possible value of $\\sum_{k=1}^{2000} \\mid y_{k}-y_{k+1}$ . (2001 Shanghai Competition Problem)",
      "answer": "2000"
    },
    {
      "question": "9.18*. How many sides can a convex polygon have if all its diagonals have the same length?",
      "answer": "5"
    },
    {
      "question": "A computer program evaluates expressions without parentheses in the following way:\n\n1) First, it performs the multiplications and divisions from left to right one by one.\n2) Then, it performs the additions and subtractions again from left to right.\n\nFor example, the value of the expression $1-2 / 3-4$ is $-3 \\frac{2}{3}$. How many different results can we get if in the following expression we write one of the signs $+- /, \\times$ independently in each $*$ position?\n\n$1 * 1 * 1 * 1 * 1 * 1 * 1 * 1$",
      "answer": "15"
    },
    {
      "question": "Mumchov D:\n\nOn a circle of length 2013, 2013 points are marked, dividing it into equal arcs. A chip is placed at each marked point. We define the distance between two points as the length of the shorter arc between them. For what largest $n$ can the chips be rearranged so that there is again one chip at each marked point, and the distance between any two chips that were initially no more than $n$ apart has increased?",
      "answer": "670"
    },
    {
      "question": "Problem 11.3. The teacher wrote the number 1818 on the board. Vasya noticed that if a multiplication sign is written between the hundreds and tens digits, the value of the resulting expression is a perfect square $\\left(18 \\times 18=324=18^{2}\\right)$. What is the next four-digit number after 1818 that has the same property?",
      "answer": "1832"
    },
    {
      "question": "Example 6. Calculate $\\int_{1}^{e} \\ln x d x$.",
      "answer": "1"
    },
    {
      "question": "4. The numbers $x$ and $y$ are such that the equalities $\\operatorname{tg} x - \\operatorname{tg} y = 7$ and $2 \\sin (2 x - 2 y) = \\sin 2 x \\sin 2 y$ hold. Find $\\operatorname{tg} x \\operatorname{tg} y$.",
      "answer": "-\\frac{7}{6}"
    },
    {
      "question": "Given a $3 \\times 3$ table (like for playing tic-tac-toe). Four chips are placed randomly in four randomly chosen cells.\n\nFind the probability that among these four chips, there will be three that are in a row either vertically, horizontally, or diagonally.",
      "answer": "\\frac{8}{21}"
    },
    {
      "question": "20. Piplu is learning long division, and he finds that when a three-digit number is divided by 19, the quotient is $a$, and the remainder is $b$ ($a, b$ are natural numbers), then the maximum value of $a+b$ is . $\\qquad$",
      "answer": "69"
    },
    {
      "question": "5. Let $f(x)=x^{3}+3\\left(x^{2}+x+\\sin \\pi x\\right)$, then $f(1-\\pi)+f(\\pi-3)=$",
      "answer": "-2"
    },
    {
      "question": "Example 4-11 The side chain of methane $\\mathrm{CH}_{4}$ is , if the 4 H bonds are replaced by one of $\\mathrm{H}, \\mathrm{Cl}, \\mathrm{CH}_{3}, \\mathrm{C}_{2} \\mathrm{H}_{5}$, how many different chemical structures are there?",
      "answer": "36"
    },
    {
      "question": "18. Traffic Light (from 10th grade. 2 points). A traffic light at a pedestrian crossing allows pedestrians to cross the street for one minute and prohibits crossing for two minutes. Find the average waiting time for the green light for a pedestrian who approaches the intersection.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2e6196530ac8ddeae41bg-13.jpg?height=314&width=331&top_left_y=1308&top_left_x=291)",
      "answer": "40"
    },
    {
      "question": "Péter took one of each of the stamps worth $1, 2, 3, \\ldots, 37$ forints from his stamp collection. He would like to group these stamps so that the sum of the face values of the stamps in each group is the same. In how many different ways can he do this?",
      "answer": "2"
    },
    {
      "question": "## Task 1 - 070611\n\nA certain large city's tram station has a total of 83 trams. Of these, exactly 46 are trailers. At a certain point in time, a total of 8 motor cars, each with two trailers, and 23 motor cars, each with one trailer, are in service.\n\nWhat is the number of all motor cars and trailers that are not in service at this time?",
      "answer": "7"
    },
    {
      "question": "B3 Alpine Expedition has 420 kg of luggage.\n\nA What is the minimum number of porters they need to hire if each porter can carry a maximum of 35 kg of load?\n\nB On average, how much weight should each porter carry if there are 15 of them?\n\nC In how many ways can the first three porters choose their loads if they are choosing from ten different loads?\n\nD One of the loads is in the shape of a cuboid, and the other is in the shape of a cylinder with a diameter of $1 \\mathrm{~m}$ and a height of $0.8 \\mathrm{~m}$. The cuboid is $1.57 \\mathrm{~m}$ long and $4 \\mathrm{dm}$ wide. How high is it if both loads have the same volume?",
      "answer": "1"
    },
    {
      "question": "7. Consider the set $X=\\{1,2,3, \\ldots, 9,10\\}$. Find two disjoint nonempty subsets $A$ and $B$ of $X$ such that\n\n(a) $A \\cup B=X$;\n\n(b) $\\operatorname{prod}(A)$ is divisible by $\\operatorname{prod}(B)$, where for any finite set of numbers $C, \\operatorname{prod}(C)$ denotes the product of all numbers in $C$;\n\n(c) the quotient $\\operatorname{prod}(A) / \\operatorname{prod}(B)$ is as small as possible.",
      "answer": "7"
    },
    {
      "question": "5. As shown in the figure, a large triangle $A B C$ is divided into seven parts by three line segments, four of which are triangles and the other three are quadrilaterals. The sum of the perimeters of the three quadrilaterals is 25 cm, the sum of the perimeters of the four triangles is 20 cm, and the perimeter of triangle $A B C$ is 19 cm. Then $A D+B E+C F=$ $\\qquad$ cm",
      "answer": "13"
    },
    {
      "question": "6.3. At a round table sit 10 elves, each with a basket of nuts in front of them. Each was asked, \"How many nuts do your two neighbors have together?\" and, going around the circle, the answers received were 110, 120, 130, 140, 150, 160, 170, 180, 190, and 200. How many nuts does the elf who answered 160 have?",
      "answer": "55"
    },
    {
      "question": "1. Given a linear function $f(x)$. It is known that the distance between the points of intersection of the graphs $y=$ $x^{2}-1$ and $y=f(x)$ is $\\sqrt{30}$, and the distance between the points of intersection of the graphs $y=x^{2}$ and $y=f(x)+3$ is $\\sqrt{46}$. Find the distance between the points of intersection of the graphs of the functions $y=x^{2}-1$ and $y=f(x)+1$.",
      "answer": "\\sqrt{38}"
    },
    {
      "question": "12. Let real numbers $x, y, z$ satisfy $xy \\neq 0, x+y \\neq 0$. Find\n$$\n\\left(\\frac{y+z}{x}+2\\right)^{2}+\\left(\\frac{z}{y}+2\\right)^{2}+\\left(\\frac{z}{x+y}-1\\right)^{2}\n$$\n\nthe minimum value.",
      "answer": "5"
    },
    {
      "question": "8. Let $x, y$ satisfy the constraint conditions $\\left\\{\\begin{array}{l}x-y-2 \\geqslant 0, \\\\ x-5 \\leqslant 0, \\\\ y+2 \\geqslant 0 .\\end{array}\\right.$ Then the sum of the minimum and maximum values of $z=x^{2}+y^{2}$ is $\\qquad$",
      "answer": "36"
    },
    {
      "question": "3. (17 points) In triangle $A B C \\quad A B=4, B C=6$, angle $A B C$ is $30^{\\circ}, B D-$ is the bisector of triangle $A B C$. Find the area of triangle $A B D$.",
      "answer": "2.4"
    },
    {
      "question": "1. Buses from Moscow to Oryol depart at the beginning of each hour (at 00 minutes). Buses from Oryol to Moscow depart in the middle of each hour (at 30 minutes). The journey between the cities takes 5 hours. How many buses from Oryol will the bus that left from Moscow meet on its way?",
      "answer": "10"
    },
    {
      "question": "29.1.3 Lin has a $9 \\times 2004$ grid, in which the positive integers from 1 to 2004 are each filled in 9 times. And in each column, the sum of the numbers filled in is no more than 3. Find the minimum possible value of the sum of the numbers in the first row.",
      "answer": "2005004"
    },
    {
      "question": "Maruška wrote two different natural numbers on the board. Marta took a card and on one side of it, she wrote the sum of Maruška's numbers, and on the other side, their product. One of the numbers on the card was a prime number, and the sum of the numbers on both sides of the card was 97.\n\nWhich numbers did Maruška write on the board?\n\n(L. Hozová)",
      "answer": "613"
    },
    {
      "question": "Question 114, Given the equation $\\mathrm{x}^{10}+(13 \\mathrm{x}-1)^{10}=0$ has 5 pairs of conjugate complex roots $\\mathrm{r}_{\\mathrm{k}} 、 \\overline{\\mathrm{r}_{\\mathrm{k}}}(1 \\leq \\mathrm{k} \\leq 5)$, try to find the value of $\\sum_{\\mathrm{k}=1}^{5} \\frac{1}{\\mathrm{r}_{\\mathrm{k}} \\cdot \\mathrm{r}_{\\mathrm{k}}}$.",
      "answer": "850"
    },
    {
      "question": "What is the last digit of $2019^{2020^{2021}}$?",
      "answer": "1"
    },
    {
      "question": "4. Find four real numbers $x_{1}, x_{2}, x_{3}, x_{4}$, such that each, when added to the product of the others, equals two.",
      "answer": "1"
    },
    {
      "question": "20. Given that $[x]$ represents the greatest integer not exceeding $x$, such as $[1.5]=1,[2]=2$, then the last two digits of $\\left[\\frac{10^{2040}}{10^{60}-3}\\right]$ are . $\\qquad$",
      "answer": "23"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nIf $2 t+2 x-t-3 x+4 x+2 t=30$, what is the value of $x$ ?",
      "answer": "4"
    },
    {
      "question": "\\section*{Problem 6 - 021016}\n\nDetermine the smallest natural number \\(n\\) that has the following properties:\n\na) its decimal representation ends with the digit 6;\n\nb) if this last digit 6 is removed and written as the first digit before the other unchanged digits, the result is four times the number \\(n\\).",
      "answer": "153846"
    },
    {
      "question": "8. A coin collector has 100 coins that look the same. The collector knows that 30 of them are genuine, 70 are fake, and that all genuine coins weigh the same, while all fake coins weigh different and are heavier than the genuine ones. The collector has a balance scale that can be used to compare the weight of two groups of coins of the same number. To ensure finding at least one genuine coin, how many weighings are needed at minimum?",
      "answer": "70"
    },
    {
      "question": "85. In a triangle, two sides are given. At what angle between them will the area of the triangle be the greatest?",
      "answer": "90"
    },
    {
      "question": "11.1. When composing the options for the district mathematics olympiad for grades $7, 8, 9, 10, 11$, the jury aims to ensure that in the option for each grade, there are exactly 7 problems, of which exactly 4 do not appear in any other option. What is the maximum number of problems that can be included in the olympiad?",
      "answer": "27"
    },
    {
      "question": "5-1. A square with a side of 100 was cut into two equal rectangles. They were placed next to each other as shown in the figure. Find the perimeter of the resulting figure.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d2d35e627535cd91d6ebg-04.jpg?height=277&width=594&top_left_y=684&top_left_x=731)",
      "answer": "500"
    },
    {
      "question": "3. On three islands, there are 2019 cities, and some two-way air routes have been opened between them. Each route connects three cities on different islands. Any two cities are on at most one route. Find the maximum number of routes.",
      "answer": "673^2"
    },
    {
      "question": "44. Given 5 points, no three of which lie on the same line. Find the probability that, by randomly selecting two points, a student will get the required line.",
      "answer": "\\frac{1}{10}"
    },
    {
      "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a$ and $b$.\n\n$a=2 p-q$\n\n$b=p+3 q$\n\n$|p|=3$\n\n$|q|=2$\n\n$(\\widehat{p, q})=\\frac{\\pi}{2}$",
      "answer": "42"
    },
    {
      "question": "Let's calculate the following sum:\n\n$$\n\\sqrt{7-\\sqrt{48}}+\\sqrt{5-\\sqrt{24}}+\\sqrt{3-\\sqrt{8}}\n$$",
      "answer": "1"
    },
    {
      "question": "1. Distribute 5 different books to 3 students, with each student getting at least one book. How many different ways are there to do this?",
      "answer": "150"
    },
    {
      "question": "Vasilev N.B.\n\nA cherry, a sphere of radius $r$, is dropped into a round glass, the axial cross-section of which is the graph of the function $y=x^{4}$. For what largest $r$ will the sphere touch the lowest point of the bottom? (In other words, what is the maximum radius $r$ of a circle lying in the region $y \\geq x^{4}$ and containing the origin?)",
      "answer": "\\frac{3\\sqrt[3]{2}}{4}"
    },
    {
      "question": "11. On an elliptical track with a perimeter of 400 meters, two people, A and B, start cycling from two points 300 meters apart and head towards each other along the track. After meeting, they continue to move forward. It is known that A's cycling speed is 4 meters/second, and B's cycling speed is 6 meters/second. When they meet for the 6th time, the two have cycled for at least (\n) seconds",
      "answer": "210"
    },
    {
      "question": "A store was selling each unit of a toy for $R \\$ 13.00$. To sell its entire stock, which was not more than 100 units, the store management decided to lower the price by an integer number of reais, thus obtaining $\\mathrm{R} \\$ 781.00$ for the entire stock. What was the price reduction, per unit?",
      "answer": "2"
    },
    {
      "question": "11. (20 points) Let point $Z$ move on the circle $|z|=3$ in the complex plane, and let $w=\\frac{1}{2}\\left(z+\\frac{1}{z}\\right)$. The trajectory of $w$ is the curve $\\Gamma$. A line $l$ passes through the point $P(1,0)$ and intersects the curve $\\Gamma$ at points $A$ and $B$, and intersects the imaginary axis at point $M$. If $\\overrightarrow{M A}=t \\overrightarrow{A P}$ and $\\overrightarrow{M B}=s \\overrightarrow{B P}$, find the value of $t+s$.",
      "answer": "-\\frac{25}{8}"
    },
    {
      "question": "1.1. Arina wrote down a number and the number of the month of her birthday, multiplied them and got 248. In which month was Arina born? Write the number of the month in your answer.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6c94238ddc8917ebef5dg-1.jpg?height=142&width=874&top_left_y=266&top_left_x=585)",
      "answer": "8"
    },
    {
      "question": "5. “Xiong Da” $\\times$ “Xiong Er” = “Xiong Xiong Di” . If the same Chinese character represents the same digit from 0 to 9, different Chinese characters represent different digits, and “Da” > “Er” , then the sum of all three-digit numbers represented by “Xiong Xiong Di” that satisfy the conditions is $\\qquad$ . “Da”+“Er” $<9$;",
      "answer": "686"
    },
    {
      "question": "14. From the 20 numbers 11, 12, 13, 14, ... 30, at least ( ) numbers must be taken to ensure that there are two numbers whose sum is a multiple of ten.",
      "answer": "11"
    },
    {
      "question": "1. Find all positive integers $n$, such that $3^{n}+n^{2}+2019$ is a perfect square.\n（Zou Jin, contributed）",
      "answer": "4"
    },
    {
      "question": "1. For some real number $c$, the graphs of the equation $y=|x-20|+|x+18|$ and the line $y=x+c$ intersect at exactly one point. What is $c$ ?",
      "answer": "18"
    },
    {
      "question": "In triangle $ABC$, the height $BD$ is 6, the median $CE$ is 5, and the distance from point $K$ of intersection of segments $BD$ and $CE$ to side $AC$ is 1.\n\nFind the side $AB$.",
      "answer": "\\frac{2\\sqrt{145}}{3}"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nEach of the three lines having equations $x+t y+8=0,5 x-t y+4=0$, and $3 x-k y+1=0$ passes through the same point. What is the value of $k$ ?\n\n##",
      "answer": "5"
    },
    {
      "question": "[ [Assist\n\nIn triangle $ABC$, the height $BD$ is 11.2 and the height $AE$ is 12. Point $E$ lies on side $BC$ and $BE$ : $EC=5: 9$. Find the side $AC$.\n\n#",
      "answer": "15"
    },
    {
      "question": "1. Calculate: $1325: 5+15: 5-(99-9: 9) \\cdot 2-35-30$.",
      "answer": "7"
    },
    {
      "question": "1. Determine the smallest natural number that is divisible by 63 and whose sum of digits is equal to 63.",
      "answer": "19899999"
    },
    {
      "question": "Task 4.1. Assign each letter a digit $1,2,3,4,5$ so that all inequalities are satisfied.\n\n$$\n3>\\mathrm{A}>\\mathrm{M}<\\mathrm{E}<\\mathrm{H}<\\mathrm{A}\n$$\n\nDifferent letters must correspond to different digits. Write the number ZAMENA as your answer.",
      "answer": "541234"
    },
    {
      "question": "1. a) Verify the identity $\\sqrt{1+\\frac{1}{n^{2}}+\\frac{1}{(n+1)^{2}}}=1+\\frac{1}{n}-\\frac{1}{n+1}, \\forall n \\in \\mathbb{N}^{*}$.\n\nb) Determine the integer part of the number\n\n$$\na=\\sqrt{1+\\frac{1}{1^{2}}+\\frac{1}{2^{2}}}+\\sqrt{1+\\frac{1}{2^{2}}+\\frac{1}{3^{2}}}+\\ldots+\\sqrt{1+\\frac{1}{2012^{2}}+\\frac{1}{2013^{2}}}\n$$",
      "answer": "2012"
    },
    {
      "question": "Draw an acute isosceles triangle inside a circle. Over one of the sides of this triangle, construct another isosceles triangle such that its apex also lies on the circumference of the circle. Continuing this process, determine the limit of the angles at the vertices of the triangles.",
      "answer": "60"
    },
    {
      "question": "5. Segment $A B=8$ intersects plane $\\alpha$ at an angle of $30^{\\circ}$ and is divided by this plane in the ratio $1: 3$. Find the radius of the sphere passing through points $A$ and $B$ and intersecting plane $\\alpha$ in a circle of the smallest radius.",
      "answer": "2\\sqrt{7}"
    },
    {
      "question": "Variant 7.8.2. Real numbers $a, b, c$ are such that\n\n$$\n\\left\\{\\begin{array}{l}\n\\frac{a b}{a+b}=3 \\\\\n\\frac{b c}{b+c}=5 \\\\\n\\frac{c a}{c+a}=8\n\\end{array}\\right.\n$$\n\nFind the value of the expression $\\frac{a b c}{a b+b c+c a}$.",
      "answer": "\\frac{240}{79}"
    },
    {
      "question": "5. On a long stick, there are three types of graduation lines. The first type divides the stick into 10 equal parts;\n\nThe second type divides the stick into 12 equal parts; The third type divides the stick into 15 equal parts. If each graduation line cuts the stick, how many pieces will the stick be cut into?",
      "answer": "28"
    },
    {
      "question": "9. The function $f(x)=$ $\\frac{x+1}{x}+\\frac{x+2}{x+1}+\\cdots+\\frac{x+2023}{x+2022}$ has a center of symmetry at $(a, b)$, then $2 a+b=$ $\\qquad$ _.",
      "answer": "1"
    },
    {
      "question": "42nd Putnam 1981 Problem A1 Let the largest power of 5 dividing 1 1 2 2 3 3 ... n n be 5 f(n) . What is lim n→∞ f(n)/n 2 ? Solution",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "14. Find the value of $\\sin \\frac{\\pi}{7} \\sin \\frac{2 \\pi}{7} \\sin \\frac{3 \\pi}{7}$.",
      "answer": "\\frac{\\sqrt{7}}{8}"
    },
    {
      "question": "KöMaL Survey on a total of 14 questions, with 4 possible answers for each question. The best result was 10 correct answers. What is the probability that we achieve this result by random filling out?",
      "answer": "0.00030205"
    },
    {
      "question": "6. Let's call the distance between numbers the absolute value of their difference. It is known that the sum of the distances from twenty-five consecutive natural numbers to some number $a$ is 1270, and the sum of the distances from these same twenty-five numbers to the number $a^2$ is 1234. Find all possible values of $a$.",
      "answer": "-\\frac{4}{5}"
    },
    {
      "question": "On a $2014 \\times 2014$ board, each of the $2014^{2}$ squares contains a lamp. Lamps can be on or off. In the initial situation, some of the lamps are on. In a move, you choose a row or column in which at least 1007 lamps are on and change the status of all 2014 lamps in that row or column (from on to off and from off to on). Find the smallest non-negative integer $k$ such that the following holds: from any initial situation, you can reach a situation in a finite number of steps where at most $k$ lamps are on.",
      "answer": "1006 \\cdot 2014"
    },
    {
      "question": "248. Cats and Mice. One morning at Professor Rackbrain's table, the lively discussion was about the extermination of rodents when suddenly the professor said:\n\n- If a certain number of cats ate a total of 999,919 mice, and all the cats ate the same number of mice, how many cats were there in total?\n\nSomeone suggested that perhaps one cat ate all the mice, but Rackbrain objected, saying he had said \"cats.\" Then someone else ventured to suggest that each cat had eaten one mouse, to which the professor remarked that he had said \"mice.\" He also added, to help those present, that each cat had eaten more mice than there were cats.\n\nWhat, then, would be the correct answer?",
      "answer": "991"
    },
    {
      "question": "Problem 5.3. A cuckoo clock is hanging on the wall. When a new hour begins, the cuckoo says \"cuckoo\" a number of times equal to the number the hour hand points to (for example, at 19:00, \"cuckoo\" sounds 7 times). One morning, Maxim approached the clock when it was 9:05. He started turning the minute hand until he advanced the clock by 7 hours. How many times did \"cuckoo\" sound during this time?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_038330058549e9b1ba1cg-10.jpg?height=523&width=311&top_left_y=522&top_left_x=565)",
      "answer": "43"
    },
    {
      "question": "The ninety-sixth question: Given $S=\\{(\\mathrm{i}, \\mathrm{j}) \\mid \\mathrm{i}, \\mathrm{j}=1, 2, \\ldots, 100\\}$ as the set of $100 \\times 100$ integer points on the Cartesian plane. Each point in $S$ is colored with one of four given colors. Find the maximum possible number of rectangles with vertices of four different colors and sides parallel to the coordinate axes.",
      "answer": "9375000"
    },
    {
      "question": "$7 \\cdot 67$ Given 11 sets $M_{1}, M_{2}, \\cdots, M_{11}$, where each set has exactly 5 elements and for all $i, j, 1 \\leqslant i<j \\leqslant 11$, there is $M_{i} \\cap M_{j} \\neq \\varnothing$, find the minimum possible value of the maximum number of sets among these sets whose intersection is non-empty.",
      "answer": "4"
    },
    {
      "question": "How many digits are needed to write down the numbers from 1 to 6786?",
      "answer": "26037"
    },
    {
      "question": "8. Aladdin found 20 necklaces in the treasure room, some of which are set with 2 diamonds, and some are set with 5 diamonds, with a total of 79 diamonds. So, the number of necklaces set with 5 diamonds is $\\qquad$ pieces",
      "answer": "13"
    },
    {
      "question": "1. [5] A positive integer is called primer if it has a prime number of distinct prime factors. Find the smallest primer number.",
      "answer": "6"
    },
    {
      "question": "8.5. Kolya drew 10 segments and marked all their intersection points in red. Counting the red points, he noticed the following property: on each segment, there are three red points. a) Provide an example of the arrangement of 10 segments with this property. b) What is the maximum number of red points for 10 segments with this property?",
      "answer": "15"
    },
    {
      "question": "11. (12 points) Today is December 19, we place the numbers $1 、 2 、 1 、 9$ composed of small shaded squares with a side length of 1 in an $8 \\times 5$ large rectangle. When the large rectangle is rotated $180^{\\circ}$, it becomes “6121”. If these two $8 \\times$ 5 large rectangles are overlapped, then the number of overlapping shaded squares is $\\qquad$.",
      "answer": "30"
    },
    {
      "question": "2. At a cactus lovers' meeting, 80 cactus enthusiasts presented their collections, each consisting of cacti of different species. It turned out that no species of cactus is present in all collections at once, but any 15 people have cacti of the same species. What is the smallest total number of cactus species that can be in all the collections?",
      "answer": "16"
    },
    {
      "question": "[Ratio of areas of similar triangles]\n\nFind the area of a triangle whose medians are 12, 15, and 21.\n\n#",
      "answer": "48\\sqrt{6}"
    },
    {
      "question": "[ Coordinate method on the plane [ The ratio of the areas of triangles with a common base or common height]\n\n ] On the coordinate plane, points $A(1 ; 3), B(1 ; 9), C(6 ; 8)$, and $E(5 ; 1)$ are given. Find the area of the pentagon $A B C D E$, where $D$ is the point of intersection of the lines $A C$ and $B E$.",
      "answer": "21"
    },
    {
      "question": "3B. Let $a, b, c$ be the sides of a triangle. Determine the angle opposite side $c$, if\n\n$$\n(a+b+c)(a+b-c)=3 a b\n$$",
      "answer": "60"
    },
    {
      "question": "3. [5] How many six-digit multiples of 27 have only 3,6 , or 9 as their digits?",
      "answer": "51"
    },
    {
      "question": "5. Given four points $A, B, C, D$ on a unit sphere, satisfying $AB=AC=AD, BC=CD=DB$. Then the maximum volume of tetrahedron $ABCD$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "\\frac{8\\sqrt{3}}{27}"
    },
    {
      "question": "6. Find the smallest natural number $n$ such that for any coloring of the numbers 1, 2, 3, ..., $n$ with three colors, there exist two numbers of the same color whose difference is a square of a natural number.\n\n(Vojtech Bálint, Michal Rolínek, Josef Tkadlec)",
      "answer": "29"
    },
    {
      "question": "1. A robot is located in one of the cells of an infinite grid and can be given the following commands:\n\n- up (the robot moves to the adjacent cell above);\n- down (the robot moves to the adjacent cell below);\n- left (the robot moves to the adjacent cell to the left);\n- right (the robot moves to the adjacent cell to the right).\n\nFor example, if the robot executes a sequence of four commands (up, right, down, left), it will obviously return to its initial position, i.e., it will end up in the same cell from which it started. How many different sequences of 4 commands exist that return the robot to its initial position?",
      "answer": "36"
    },
    {
      "question": "Problem 9.5. Point $M$ is the midpoint of side $B C$ of triangle $A B C$, where $A B=17$, $A C=30, B C=19$. A circle is constructed with side $A B$ as its diameter. An arbitrary point $X$ is chosen on this circle. What is the minimum value that the length of segment $M X$ can take?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2befe970655743580344g-03.jpg?height=414&width=553&top_left_y=1517&top_left_x=448)",
      "answer": "6.5"
    },
    {
      "question": "Problem 3. Given an acute triangle $ABC$. Its altitudes $AE$ and $CD$ intersect at point $H$, and $\\overline{AB} = \\overline{CH}$. Calculate the angle $\\measuredangle ACB$.",
      "answer": "45"
    },
    {
      "question": "3. Let $A B C$ be a triangle with $\\angle B A C=90^{\\circ}$. Points $D, E$ on its hypotenuse $B C$ satisfy $C D=C A, B E=B A$. Let $F$ be such a point inside $\\triangle A B C$ that $D E F$ is an isosceles right triangle with hypotenuse DE. Find $\\angle B F C$. (Patrik Bak)",
      "answer": "135"
    },
    {
      "question": "30. [20] Let $A B C$ be a triangle with circumcenter $O$, incenter $I, \\angle B=45^{\\circ}$, and $O I \\| B C$. Find $\\cos \\angle C$.",
      "answer": "1-\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "## Problema 4\n\nFie $\\mathrm{A}, \\mathrm{B} \\in M_{2}(\\mathrm{Q})$ astfel încât $\\mathrm{AB}=\\mathrm{BA}$, $\\operatorname{det} \\mathrm{A}=-3$ și $\\operatorname{det}(\\mathrm{A}+\\sqrt{3} \\mathrm{~B})=0$.\n\nSă se calculeze $\\operatorname{det}\\left(A^{2}+B^{2}-A B\\right)$\n\nGMB 12/2011\n\nProbleme selectate de Prof. Ursan Rodica\n\nNotă: a) Timpul efectiv de lucru este de 3 ore.\n\nb) Toate problemele sunt obligatorii.\n\nc) Fiecare problemă se notează de la 0 la 7 .\n\n## OLIMPIADA NAȚIONALĂ DE MATEMATICĂ\n\nEtapa locală - 15.02.2014\n\n## Clasa a XI-a\n\n## Barem de corectare",
      "answer": "13"
    },
    {
      "question": "9.6. Find the minimum value of the expression $(\\sqrt{2(1+\\cos 2 x)}-\\sqrt{36-4 \\sqrt{5}} \\sin x+2) \\cdot(3+2 \\sqrt{10-\\sqrt{5}} \\cos y-\\cos 2 y) \\cdot$ If the answer is not an integer, round it to the nearest integer.",
      "answer": "-27"
    },
    {
      "question": "Task B-2.4. Let $f(x)=x^{2}+b x+c, b, c \\in \\mathbb{R}$. If $f(0)+f(1)=\\frac{1}{2}$, calculate $f\\left(\\frac{1}{2}\\right)$.",
      "answer": "0"
    },
    {
      "question": "Problem 3. Five different natural numbers are written in a circle on the board. Each of them, Petya divided by the next one in the clockwise direction, and then wrote down the 5 resulting numbers (not necessarily integers) on a piece of paper. Can the sum of the 5 numbers on the paper be an integer?",
      "answer": "18"
    },
    {
      "question": "9. Let $p$ and $q$ be integers such that $p+q=2010$. If both roots of the equation $10 x^{2}+p x+q=0$ are positive integers, find the sum of all possible values of $p$.\n(1 mark)\n設 $p 、 q$ 為整數, 且 $p+q=2010$ 。若方程 $10 x^{2}+p x+q=0$ 的兩根均為正整數, 求 $p$ 所有可能值之和。\n（1 分）",
      "answer": "-3100"
    },
    {
      "question": "Example 7. Calculate the area of the figure bounded above by the lines $x-y+2=0, y=2-\\frac{1}{2} x$, and below by the parabola $y=x^{2}-2 x-$ $-8$.",
      "answer": "42"
    },
    {
      "question": "5. The zoo keeps some sika deer and ostriches. The number of ostrich feet is 20 more than the number of sika deer feet. When each ostrich lifts one foot, the number of ostrich feet lifted is 10 less than the number of sika deer feet. The zoo has $\\qquad$ sika deer.",
      "answer": "10"
    },
    {
      "question": "Problem 11.5. Quadrilateral $ABCD$ is inscribed in a circle. It is known that $BC=CD, \\angle BCA=$ $64^{\\circ}, \\angle ACD=70^{\\circ}$. A point $O$ is marked on segment $AC$ such that $\\angle ADO=32^{\\circ}$. How many degrees does the angle $BOC$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_ddcc40423d5e6a42b0ccg-43.jpg?height=448&width=519&top_left_y=234&top_left_x=465)",
      "answer": "58"
    },
    {
      "question": "4. In the group, there are 6 Varaždinians, 6 Zadrians, and 6 Osijekians. In how many ways can they be divided into:\na) 6 groups of 3 people so that each group contains one Varaždinian, one Zadrian, and one Osijekian?\nb) 3 groups of 6 people so that each group contains two Varaždinians, two Zadrians, and two Osijekians?",
      "answer": "518400"
    },
    {
      "question": "135. A troop of monkeys was having fun; the square of one eighth of them was frolicking in the forest, the remaining twelve were shouting at the top of a hillock. Tell me: how many monkeys were there in total?",
      "answer": "4816"
    },
    {
      "question": "1. Find the number of triples $(a, b, c)$ of positive integers such that $a+a b+a b c=11$.",
      "answer": "3"
    },
    {
      "question": "$:$ Folklore\n\nLёsha thought of a two-digit number (from 10 to 99). Grisha tries to guess it by naming two-digit numbers. If Grisha correctly names the number, or one digit is correct and the other is off by no more than one, Lёsha responds with \"warm\"; in all other cases, Lёsha responds with \"cold\". (For example, if the number 65 is thought of, then by naming 65, 64, 66, 55, or 75, Grisha will hear \"warm,\" and in all other cases, \"cold.\")\n\na) Show that there is no way for Grisha to guarantee finding the number using 18 attempts.\n\nb) Come up with a way for Grisha to guarantee finding the number using 24 attempts (regardless of which number Lёsha thinks of).\n\nc) Can it be done in 22 attempts?",
      "answer": "24"
    },
    {
      "question": "9. Circles $\\omega_{a}, \\omega_{b}, \\omega_{c}$ have centers $A, B, C$, respectively and are pairwise externally tangent at points $D, E, F$ (with $D \\in B C, E \\in C A, F \\in A B$ ). Lines $B E$ and $C F$ meet at $T$. Given that $\\omega_{a}$ has radius 341 , there exists a line $\\ell$ tangent to all three circles, and there exists a circle of radius 49 tangent to all three circles, compute the distance from $T$ to $\\ell$.",
      "answer": "294"
    },
    {
      "question": "5. In the Cartesian coordinate system $x O y$, there are 50 different parabolas $y=a x^{2}+b x+c$ and another 50 different parabolas $x=a_{1} y^{2}+b_{1} y+c_{1}$. These 100 parabolas divide the coordinate plane into a maximum of $\\qquad$ parts.",
      "answer": "15052"
    },
    {
      "question": "## Task 4 - 271214\n\nDetermine the remainder when the sum $s=1^{5}+2^{5}+3^{5}+\\ldots+1987^{5}$ is divided by 25!",
      "answer": "8"
    },
    {
      "question": "Example 12 The sports meet lasted for $n$ days $(n>1)$, and a total of $m$ medals were awarded. On the first day, 1 medal was awarded, and then $\\frac{1}{7}$ of the remaining $m-1$ medals. On the second day, 2 medals were awarded, and then $\\frac{1}{7}$ of the remaining medals, and so on, until on the $n$-th day exactly $n$ medals were awarded, and all were distributed. How many days did the sports meet last? How many medals were awarded in total?\n(IMO - 9 problem)",
      "answer": "6"
    },
    {
      "question": "5. AFS International Cultural Exchange Organization plans to allocate 18 slots for a high school exchange program to 4 schools, with the requirement that each school gets at least one slot and the number of slots allocated to each school must be different. The number of different allocation methods is $\\qquad$ .",
      "answer": "360"
    },
    {
      "question": "Example 11 (23rd All-Union Mathematical Olympiad, 1989) A natural number $N$ has exactly 12 positive divisors (including 1 and $N$), which are numbered in increasing order: $d_{1}<d_{2}<\\cdots<d_{12}$. It is known that the positive divisor with index $d_{4}-1$ is equal to $\\left(d_{1}+d_{2}+d_{4}\\right) \\cdot d_{8}$. Find the natural number $N$.",
      "answer": "1989"
    },
    {
      "question": "146. In a regular quadrilateral pyramid, the center of the circumscribed sphere lies on the surface of the inscribed sphere. Find the measure of the plane angle at the vertex of the pyramid.",
      "answer": "\\frac{\\pi}{2}"
    },
    {
      "question": "5. Each football is sewn from pieces of leather in the shape of regular pentagons and regular hexagons. The ball has a total of 32 leather pieces. Each piece of leather in the shape of a pentagon is connected along its sides only to pieces of leather in the shape of hexagons. Each piece of leather in the shape of a hexagon is connected along its sides to 3 pentagons and 3 hexagons. How many pieces of leather are in the shape of a pentagon, and how many in the shape of a hexagon?",
      "answer": "20"
    },
    {
      "question": "9-1. Segment $P Q$ is divided into several smaller segments. On each of them, a square is constructed (see figure).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_7c4dd4d2d6040292f233g-20.jpg?height=619&width=1194&top_left_y=593&top_left_x=431)\n\nWhat is the length of the path along the arrows if the length of segment $P Q$ is 73? If necessary, round the answer to 0.01 or write the answer as a common fraction.",
      "answer": "219"
    },
    {
      "question": "Given a regular tetrahedron $S-ABC$ with height $SO=3$, and the side length of the base is 6. A perpendicular line is drawn from point $A$ to the opposite face $SBC$, with the foot of the perpendicular being $O'$. On $AO'$, take a point $P$ such that $\\frac{AP}{PO'}=8$. Find the area of the section passing through point $P$ and parallel to the base.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "$7 \\cdot 115$ Let $S=\\{1,2,3,4\\}, a_{1}, a_{2}, \\cdots, a_{k}$ be a sequence composed of numbers from $S$, and it includes all permutations of $S$ that do not end with 1, i.e., for any permutation $\\left(b_{1}, b_{2}, b_{3}, b_{4}\\right)$ of the 4 numbers in $S$, where $b_{4} \\neq 1$, there exist $i_{1}, i_{2}, i_{3}, i_{4}$, such that\n$$\na_{i_{j}}=b_{j}, j=1,2,3,4 \\text { and } 1 \\leqslant i_{1}<i_{2}<i_{3}<i_{4} \\leqslant k .\n$$\n\nFind the minimum value of the number of terms $k$ in the sequence.",
      "answer": "11"
    },
    {
      "question": "11.1. In triangle $\\mathrm{ABC}$, the bisectors of angles $\\mathrm{BAC}$ and $\\mathrm{BCA}$ intersect sides ВС and АВ at points К and Р, respectively. It is known that the length of side АС is equal to the sum of the lengths of segments АР and СК. Find the measure of angle $\\mathrm{ABC}$.",
      "answer": "60"
    },
    {
      "question": "9.2. In triangle $ABC$, the median $BM$ was drawn. It turned out that $\\angle ABM=40^{\\circ}$, $\\angle MBC=70^{\\circ}$. Find the ratio $AB: BM$. Justify your answer.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_94f1ab38259b8995cf4bg-4.jpg?height=494&width=708&top_left_y=1763&top_left_x=137)\n\nTo solve problem 9.2 Solution: Extend the median $BM$ beyond point $M$ and lay off a segment $MD = BM$ (see the figure). In quadrilateral $ABCD$, the diagonals bisect each other, so this quadrilateral is a parallelogram. Then $\\angle ABC + \\angle BCD = 180^{\\circ}$, from which $\\angle BCD = 70^{\\circ} = \\angle MBC = \\angle DBC$. This means that triangle $BCD$ is isosceles, as is the congruent triangle $BAD$. Therefore,\n\n$$\nAB: BM = AB: \\frac{BD}{2} = AB: \\frac{AB}{2} = 2\n$$",
      "answer": "2"
    },
    {
      "question": "7. Given in $\\triangle A B C$, $a=2 b, \\cos B=\\frac{2 \\sqrt{2}}{3}$, then $\\sin \\frac{A-B}{2}+\\sin \\frac{C}{2}=$ $\\qquad$",
      "answer": "\\frac{\\sqrt{10}}{3}"
    },
    {
      "question": "[ Trigonometric ratios in a right triangle [ Theorem of the length of a tangent and a secant; the product of the entire secant and its external part [ Two tangents drawn from the same point\n\nIn parallelogram $K L M N$, side $K L$ is equal to 8. A circle, tangent to sides $N K$ and $N M$, passes through point $L$ and intersects sides $K L$ and $M L$ at points $C$ and $D$ respectively. It is known that $K C: L C=4: 5$ and $L D: M D=8: 1$. Find side $K N$.",
      "answer": "10"
    },
    {
      "question": "10.1. Different positive numbers $x, y, z$ satisfy the equations\n\n$$\nx y z=1 \\quad \\text{and} \\quad x+y+z=\\frac{1}{x}+\\frac{1}{y}+\\frac{1}{z}\n$$\n\nFind the median of them. Justify your answer.",
      "answer": "1"
    },
    {
      "question": "$5 \\cdot 71$ A city has $n$ high schools, the $i$-th high school sends $C_{i}$ students to the stadium to watch a ball game. It is known that $0 \\leqslant C_{i} \\leqslant 39, i=1,2, \\cdots, n, C_{1}+C_{2}+\\cdots+C_{n}=1990$, each row of seats in the stand has 199 seats. It is required that students from the same school must sit in the same row. How many rows at least should the stadium arrange to ensure that all students can be seated according to the requirements?",
      "answer": "12"
    },
    {
      "question": "10. If $x=\\frac{1}{2}\\left(\\sqrt[3]{2009}-\\frac{1}{\\sqrt[3]{2009}}\\right)$, find the value of $\\left(x+\\sqrt{1+x^{2}}\\right)^{3}$.",
      "answer": "2009"
    },
    {
      "question": "2. Tine was at a competition where he was supposed to solve 20 problems. For each correctly solved problem, he received 8 points, and for each incorrect solution, 5 points were deducted. For a problem he did not attempt, he received 0 points. He found out that he had scored 13 points. How many problems did he attempt?",
      "answer": "13"
    },
    {
      "question": "16. A polygon is said to be friendly if it is regular and it also has angles that when measured in degrees are either integers or half-integers (i.e. have a decimal part of exactly 0.5 ). How many different friendly polygons are there?",
      "answer": "28"
    },
    {
      "question": "7. (10 points) Uncle Cook's hat fell in front of the gate, still smoking. It turned out that someone had thrown a firecracker from the window, and the falling firecracker had set the hat on fire. When the accident happened, 5 boys all poked their heads out, and of course, none of them were willing to admit it. Now, 4 of the boys are telling the truth, and one person is lying, and the liar is the one who threw the firecracker. So, the liar's room number is $\\qquad$Buster: “It wasn't me, I only found out what happened when Uncle Cook shouted.” Oak: “It wasn't me, Marco can vouch for me, I didn't throw anything.” Marco: “It wasn't Oak, it didn't come from above, I didn't see anything, and I didn't throw anything.” Corobi: “But I saw it, someone threw something from above.”\n\nMarv: “Yes, someone threw something from above, it flew over my head, just grazing my hair.”",
      "answer": "302"
    },
    {
      "question": "3. For a given positive constant $k$, define $f_{1}(k)$ as the square of the sum of the digits of $k$, and let $f_{n+1}(k)=f_{1}\\left(f_{n}(k)\\right)$, find the value of $f_{1991}\\left(2^{1990}\\right)$.\n(IMO - 31 Shortlist)",
      "answer": "169"
    },
    {
      "question": "4.1. (14 points) How many six-digit numbers exist whose sum of digits equals 51?",
      "answer": "56"
    },
    {
      "question": "On the number line, a flea is jumping. It starts from 0, the length of each jump is 1, and the next jump is in the same direction as the previous one with probability $p$, and in the opposite direction with probability $1-p$. What is the probability that it will return to 0?",
      "answer": "1"
    },
    {
      "question": "1. In a $4 \\times 4$ table, positive real numbers are written such that the product in each pentad of the shape 母 is equal to 1. Determine the maximum number of different numbers written in the table.\n\nNote: The shape 母 refers to a specific arrangement of five cells in the table, resembling the Chinese character 母. For clarity, it typically involves a central cell surrounded by four adjacent cells.",
      "answer": "3"
    },
    {
      "question": "8. Let $M=\\{1,2, \\cdots, 2017\\}$ be the set of the first 2017 positive integers. If one element is removed from the set $M$, and the sum of the remaining elements is exactly a perfect square, then the removed element is",
      "answer": "1677"
    },
    {
      "question": "18 Given a regular triangular prism $A B C-A_{1} B_{1} C_{1}$, where $A A_{1}=2 A C=4$, extend $C B$ to $D$ such that $C B=B D$.\n(1) Prove that line $C_{1} B$ // plane $A B_{1} D$;\n(2) Find the sine value of the angle formed by plane $A B_{1} D$ and plane $A C B$.",
      "answer": "\\frac{4\\sqrt{17}}{17}"
    },
    {
      "question": "[ The angle between two chords and two secants ] [ The inscribed angle is half of the central angle ]\n\nChords $A C$ and $B D$ are drawn from the ends of arc $A B$, which contains $m^{\\circ}$, such that the angle $D M C$, formed by their intersection, is equal to the inscribed angle $D N C$ in arc $C D$. Find the degree measure of this arc.",
      "answer": "180-\\frac{}{2}"
    },
    {
      "question": "3. Given a convex $n$-sided polygon, each of its diagonals is the perpendicular bisector of at least one other diagonal. Find all possible positive integers $n(n>3)$.\n\n untranslated text remains in its original format and line breaks are preserved.",
      "answer": "4"
    },
    {
      "question": "44th Putnam 1983 Problem B1 Let C be a cube side 4, center O. Let S be the sphere center O radius 2. Let A be one of the vertices of the cube. Let R be the set of points in C but not S, which are closer to A than to any other vertex of C. Find the volume of R. Solution",
      "answer": "8-\\frac{4\\pi}{3}"
    },
    {
      "question": "12. The mid-term exam score statistics for Chinese, Mathematics, and English in a class are as follows: 18 students have at least one subject with an excellent grade; 9 students have an excellent grade in Chinese, 11 students have an excellent grade in Mathematics, and 8 students have an excellent grade in English; 5 students have an excellent grade in both Chinese and Mathematics, 3 students have an excellent grade in both Mathematics and English, and 4 students have an excellent grade in both Chinese and English. Ask:\n(1) How many students have at least one excellent grade in Chinese or Mathematics?\n(2) How many students have an excellent grade in all three subjects: Chinese, Mathematics, and English?",
      "answer": "15"
    },
    {
      "question": "2. There is a sequence of numbers, starting from the 3rd number, each number is the sum of all the numbers before it. If the 1st number of this sequence is 1, and the 2nd number is 2, then the remainder when the 2022nd number is divided by 7 is $\\qquad$ -",
      "answer": "3"
    },
    {
      "question": "## Problem Statement\n\nFind the distance from point $M_{0}$ to the plane passing through three points $M_{1}, M_{2}, M_{3}$.\n\n$$\n\\begin{aligned}\n& M_{1}(14 ; 4 ; 5) \\\\\n& M_{2}(-5 ;-3 ; 2) \\\\\n& M_{3}(-2 ;-6 ;-3) \\\\\n& M_{0}(-1 ;-8 ; 7)\n\\end{aligned}\n$$",
      "answer": "3\\sqrt{\\frac{13}{2}}"
    },
    {
      "question": "## Task 1\n\nThe fifteenfold of a number is greater than 299 and less than 301.\n\na) Write the information as an inequality.\n\nb) What is the number?",
      "answer": "20"
    },
    {
      "question": "14. $A B C D$ is a square with side length 9 . Let $P$ be a point on $A B$ such that $A P: P B=7: 2$. Using $C$ as centre and $C B$ as radius, a quarter circle is drawn inside the square. The tangent from $P$ meets the circle at $E$ and $A D$ at $Q$. The segments $C E$ and $D B$ meet at $K$, while $A K$ and $P Q$ meet at $M$. Find the length of $A M$.\n$A B C D$ 是一個邊長爲 9 的正方形。設 $P$ 爲 $A B$ 上的一點, 使得 $A P: P B=7: 2 \\circ$ 以 $C$ 緺圓心 、 $C B$ 爲牛徑在正方形內作一個四分一圓, 從 $P$ 點到圓的切線與圓相交於 $E$, 與 $A D$ 相交於 $Q$ 。 $C E$ 和 $D B$ 交於 $K$, 而 $A K$ 和 $P Q$ 則交於 $M$ 。求 $A M$ 的長度。",
      "answer": "\\frac{85}{22}"
    },
    {
      "question": "2. Determine all positive integers that are equal to 300 times the sum of their digits.",
      "answer": "2700"
    },
    {
      "question": "3. The area of the closed figure enclosed by the curve $x^{2}+2 y^{2} \\leqslant 4|y|$ is . $\\qquad$",
      "answer": "2\\sqrt{2}\\pi"
    },
    {
      "question": "Problem 5.3. A cuckoo clock is hanging on the wall. When a new hour begins, the cuckoo says \"cuckoo\" a number of times equal to the number the hour hand points to (for example, at 19:00, \"cuckoo\" sounds 7 times). One morning, Maxim approached the clock when it was 9:05. He started turning the minute hand until he advanced the clock by 7 hours. How many times did \"cuckoo\" sound during this time?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_4984b677816e04f7f1c9g-10.jpg?height=523&width=311&top_left_y=522&top_left_x=565)",
      "answer": "43"
    },
    {
      "question": "B4. Calculate exactly, without using a pocket calculator: $\\frac{(-2)^{-3}}{(-0.2)^{3}}-\\left(\\frac{2}{5}\\right)^{-3} \\cdot(-3)^{-2} \\cdot 0.1^{-1}$.\n\n## PROBLEMS FOR THE THIRD YEAR\n\nBefore you are two sets of problems. Problems 1 to 6 of the first set are solved by selecting the correct answer from the five proposed answers and writing it in the table below the corresponding number. Only one answer is correct. A correct answer will be scored with two points, while a wrong answer will result in the deduction of one point.\n\nProblems 1 to 4 of the second set are solved on the attached paper. The solution to each of these problems will be evaluated on a scale from 0 to 6 points. Write only your code on the sheets where you will solve the problems. Write your solution clearly and legibly with a pen.\n\nThe time allowed for solving is 90 minutes.\n\nTHE NATIONAL COMPETITION COMMITTEE WISHES YOU GREAT SUCCESS.\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_ce887fced8b2a2bf0894g-06.jpg?height=168&width=485&top_left_y=904&top_left_x=800)\n\n## PART I",
      "answer": "-\\frac{125}{72}"
    },
    {
      "question": "B2 The point $S$ lies on the chord $A B$ of a circle such that $S A=3$ and $S B=5$. The radius of the circle from the center $M$ through $S$ intersects the circle at $C$. Given $C S=1$.\n\nCalculate the length of the radius of the circle.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_3de5ef26943dd9bef1e9g-2.jpg?height=411&width=640&top_left_y=1279&top_left_x=1116)",
      "answer": "8"
    },
    {
      "question": "[ equations in integers ]\n\nThe number 1047, when divided by $A$, gives a remainder of 23, and when divided by $A+1$, it gives a remainder of 7. Find $A$.\n\n#",
      "answer": "64"
    },
    {
      "question": "9.4 In the thirtieth kingdom, there are three types of coins in circulation: bronze rubles, silver coins worth 9 rubles, and gold coins worth 81 rubles. From the treasury, which contains an unlimited supply of each type of coin, a certain amount was issued with 23 coins, which is less than 700 rubles. Find this amount, given that it is impossible to issue it with a smaller number of coins.",
      "answer": "647"
    },
    {
      "question": "9.6. There are 2013 cards with the digit 1 written on them, and 2013 cards with the digit 2 written on them. Vasya forms a 4026-digit number from these cards. In one move, Petya can swap two cards and pay Vasya 1 ruble. The process ends when Petya gets a number divisible by 11. What is the maximum amount Vasya can earn if Petya aims to pay as little as possible?\n\n(P. Kozhevnikov)",
      "answer": "5"
    },
    {
      "question": "The sequence of numbers $t_{1}, t_{2}, t_{3}, \\ldots$ is defined by\n\n$$\n\\left\\{\\begin{array}{l}\nt_{1}=2 \\\\\nt_{n+1}=\\frac{t_{n}-1}{t_{n}+1}\n\\end{array}\\right.\n$$\n\nSuggestion: Calculate the first five terms of the sequence.\n\nfor each positive integer $n$. Find $\\mathrm{t}_{2011}$.",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "6.68. Using Taylor and Maclaurin formulas, compute the limits\n\na) $\\lim _{x \\rightarrow 0} \\frac{\\sin x-x}{x^{3}}$\n\nb) $\\lim _{x \\rightarrow 0} \\frac{\\ln (1+x)-x+x^{2} / 2}{x^{3}}$.",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "(3) Let $M$ be a set of $n$ points in the plane, satisfying:\n(1) There exist 7 points in $M$ that are the 7 vertices of a convex heptagon;\n(2) For any 5 points in $M$, if these 5 points are the 5 vertices of a convex pentagon, then this convex pentagon contains at least one point from $M$ inside it. Find the minimum value of $n$.",
      "answer": "11"
    },
    {
      "question": "The angles of a triangle are $\\alpha, \\beta, \\gamma$. What is the maximum possible value of\n\n$$\n\\sin \\alpha \\sin \\beta \\cos \\gamma+\\sin ^{2} \\gamma\n$$",
      "answer": "\\frac{9}{8}"
    },
    {
      "question": "47. Find the limit of the function:\n1) $f(x)=x^{3}-5 x^{2}+2 x+4$ as $x \\rightarrow-3$;\n2) $\\varphi(t)=t \\sqrt{t^{2}-20}-\\lg \\left(t+\\sqrt{t^{2}-20}\\right)$ as $t \\rightarrow 6$.",
      "answer": "23"
    },
    {
      "question": "8. (10 points) Three households, A, B, and C, plan to subscribe to newspapers. There are 7 different newspapers available. It is known that each household subscribes to 3 different newspapers, and any two households have exactly one newspaper in common. Therefore, the three households have $\\qquad$ different subscription methods.",
      "answer": "5670"
    },
    {
      "question": "Task B-1.7. Ivan and Nenad collect 2-kuna and 5-kuna coins in their respective piggy banks. After a year, they opened their piggy banks and counted the coins. Together, they collected 280 2-kuna coins. In Ivan's piggy bank, $60\\%$ of all the coins are 2-kuna coins, and in Nenad's piggy bank, there are twice as many 2-kuna coins as 5-kuna coins. The total value of the money from both piggy banks together is 1360 kuna. What is the value of the money from Ivan's piggy bank, and what is the value from Nenad's?",
      "answer": "640"
    },
    {
      "question": "4. (8 points) In the following horizontal equation, the same Chinese characters represent the same digits, different Chinese characters represent different digits, and no Chinese character represents 7. \"迎\", \"春\", and \"杯\" are not equal to 1. Therefore, the sum of the three digits represented by \"迎\", \"春\", and \"杯\" is $\\qquad$\nLearn math $\\times 7 \\times$ Welcome $\\times$ Spring $\\times$ Cup $=$ Cheer up cheer up $\\times$ Bar.",
      "answer": "15"
    },
    {
      "question": "5. By definition, $n!=1 \\cdot 2 \\cdot 3 \\cdot \\ldots \\cdot n$. Which factor should be removed from the product $1! \\cdot 2! \\cdot 3! \\cdot \\ldots \\cdot 20!$ so that the remaining product becomes a square of some natural number?",
      "answer": "10!"
    },
    {
      "question": "8-1. Kolya made a figure from four identical blocks as shown in the picture. What is the surface area of this figure? Express your answer in square centimeters.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1421239a8d2e12a5ec5fg-01.jpg?height=450&width=1295&top_left_y=786&top_left_x=355)",
      "answer": "64"
    },
    {
      "question": "Problem 2. In a square of grid paper containing an integer number of cells, a hole in the shape of a square, also consisting of an integer number of cells, was cut out. How many cells did the large square contain if 209 cells remained after the cutout?",
      "answer": "225"
    },
    {
      "question": "4. Given a tetrahedron $P-ABC$ with three lateral edges $PA, PB, PC$ that are pairwise perpendicular, let the dihedral angles $P-AB-C, P-BC-A$, $P-CA-B$ be $\\alpha, \\beta, \\gamma$ respectively,\nthen $\\frac{\\sin ^{2} \\alpha+\\sin ^{2} \\beta+\\sin ^{2} \\gamma}{\\cos ^{2} \\alpha+\\cos ^{2} \\beta+\\cos ^{2} \\gamma}=$ $\\qquad$ .",
      "answer": "2"
    },
    {
      "question": "6. (10 points) $n$ pieces of paper, each written with 3 different positive integers not greater than $n$, any 2 pieces of paper have exactly one number in common, find the sum of all numbers written on the pieces of paper.",
      "answer": "84"
    },
    {
      "question": "João has, in his garden, a cistern where he stores rainwater and takes water to water his flowers. At midnight on December 31, 2005, the cistern contained 156 liters of water. João has the habit of writing on a board, every day, the number of liters of water he spends to water the flowers and the amount of rainwater collected.\n\nBelow we see part of the board referring to the first 8 days of January 2006. How many liters of water were in João's cistern at midnight on January 8, 2006?\n\n| Jan | flowers(l) | rain(l) |\n| :---: | :---: | :---: |\n| 1 | 6 | 2.5 |\n| 2 | 9 | 0 |\n| 3 | 0 | 5 |\n| 4 | 4 | 0 |\n| 5 | 9 | 3 |\n| 6 | 0 | 0 |\n| 7 | 11 | 4.5 |\n| 8 | 0 | 0 |",
      "answer": "132"
    },
    {
      "question": "Variant 9.2.4. On the sides $AB$ and $AD$ of rectangle $ABCD$, points $M$ and $N$ are marked, respectively. It is known that $AN=9$, $NC=39$, $AM=10$, $MB=5$.\n\n(a) (1 point) Find the area of rectangle $ABCD$.\n\n(b) (3 points) Find the area of triangle $MNC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f4981519479532effc1g-10.jpg?height=428&width=603&top_left_y=1134&top_left_x=419)",
      "answer": "675"
    },
    {
      "question": "Problem 5. A group of tourists is dividing cookies. If they evenly distribute two identical packs, one extra cookie will remain. But if they evenly distribute three such packs, 13 extra cookies will remain. How many tourists are in the group? [7 points] (I.V. Raskina)",
      "answer": "23"
    },
    {
      "question": "5-0. A certain quadratic trinomial $x^{2}-p x+q$ has integer roots $x_{1}$ and $x_{2}$. It turns out that the numbers $x_{1}$, $x_{2}$, and $q$ form a decreasing arithmetic progression. Find the sum of all possible values of $x_{2}$.",
      "answer": "-5"
    },
    {
      "question": "## Problem Statement\n\nFind the distance from point $M_{0}$ to the plane passing through three points $M_{1}, M_{2}, M_{3}$.\n\n$M_{1}(1 ; 3 ; 6)$\n\n$M_{2}(2 ; 2 ; 1)$\n\n$M_{3}(-1 ; 0 ; 1)$\n\n$M_{0}(5 ;-4 ; 5)$",
      "answer": "2\\sqrt{14}"
    },
    {
      "question": "Example 17 (1998 \"Hope Cup\" National Invitational Competition for Senior High School Training Question) Given $x, y, z \\in \\mathbf{R}^{+}$, find the maximum value of $\\frac{x y + y z}{x^{2} + y^{2} + z^{2}}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "(8) Let $p$ and $q$ be two different prime numbers, then the remainder when $p^{q-1}+q^{p-1}$ is divided by $p \\cdot q$ is $\\qquad$",
      "answer": "1"
    },
    {
      "question": "31st IMO 1990 shortlist Problem 9 ABC is a triangle with incenter I. M is the midpoint of AC and N is the midpoint of AB. The lines NI and AC meet at X, and the lines MI and AB meet at Y. The triangles AXY and ABC have the same area. Find angle A. Solution Solution by Vivek Kumar Mehra",
      "answer": "60"
    },
    {
      "question": "14th ASU 1980 Problem 17 S is a set of integers. Its smallest element is 1 and its largest element is 100. Every element of S except 1 is the sum of two distinct members of the set or double a member of the set. What is the smallest possible number of integers in S?",
      "answer": "9"
    },
    {
      "question": "11. If one side of the square $A B C D$ lies on the line $y=2 x-17$, and the other two vertices are on the parabola $y=x^{2}$, then the minimum value of the area of the square is $\\qquad$ .",
      "answer": "80"
    },
    {
      "question": "A1 Determine the positive integer $n$ such that $8^{4}=4^{n}$.",
      "answer": "6"
    },
    {
      "question": "Problem 8.1. In a $5 \\times 5$ square, some cells have been painted black as shown in the figure. Consider all possible squares whose sides lie along the grid lines. In how many of them is the number of black and white cells the same?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d43b1f0a7bd77fffa87ag-24.jpg?height=311&width=321&top_left_y=91&top_left_x=566)",
      "answer": "16"
    },
    {
      "question": "4. Let's call a natural number $n$ squareable if the numbers from 1 to $n$ can be arranged in such an order that each member of the sequence, when added to its position, results in a perfect square. For example, the number 5 is squareable, as the numbers can be arranged as: 32154, in which $3+1=2+2=1+3=4$ and $5+4=4+5=9$. Determine which of the numbers $7,9,11,15$ are squareable.",
      "answer": "915"
    },
    {
      "question": "Problem 7.8. On a rectangular sheet of paper, a picture in the shape of a \"cross\" was drawn from two rectangles $A B C D$ and $E F G H$, the sides of which are parallel to the edges of the sheet. It is known that $A B=9, B C=5, E F=3, F G=10$. Find the area of the quadrilateral $A F C H$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_2fdcf97aa9799d0d4cd6g-23.jpg?height=422&width=507&top_left_y=92&top_left_x=469)",
      "answer": "52.5"
    },
    {
      "question": "10. Given a fixed circle $\\odot P$ with radius 1, the distance from the center $P$ to a fixed line $l$ is 2, $Q$ is a moving point on $l$, $\\odot Q$ is externally tangent to $\\odot P$, $\\odot Q$ intersects $l$ at points $M$ and $N$, for any diameter $MN$, there is a fixed point $A$ in the plane such that $\\angle MAN$ is a constant, find the degree measure of $\\angle MAN$.",
      "answer": "60"
    },
    {
      "question": "IMO 1976 Problem A1 A plane convex quadrilateral has area 32, and the sum of two opposite sides and a diagonal is 16. Determine all possible lengths for the other diagonal.",
      "answer": "8\\sqrt{2}"
    },
    {
      "question": "Problem 7.8. On a rectangular sheet of paper, a picture in the shape of a \"cross\" was drawn from two rectangles $A B C D$ and $E F G H$, the sides of which are parallel to the edges of the sheet. It is known that $A B=9, B C=5, E F=3, F G=10$. Find the area of the quadrilateral $A F C H$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-23.jpg?height=422&width=507&top_left_y=92&top_left_x=469)",
      "answer": "52.5"
    },
    {
      "question": "9. A large rectangle is divided into four identical smaller rectangles by slicing parallel to one of its sides. The perimeter of the large rectangle is 18 metres more than the perimeter of each of the smaller rectangles. The area of the large rectangle is $18 \\mathrm{~m}^{2}$ more than the area of each of the smaller rectangles. What is the perimeter in metres of the large rectangle?",
      "answer": "28"
    },
    {
      "question": "4. The integer $n$ satisfies the inequality $n+(n+1)+(n+2)+\\cdots+(n+20)>2019$. What is the minimum possible value of $n$ ?",
      "answer": "87"
    },
    {
      "question": "12 A space station consists of 25 chambers, and each two chambers are connected by a tunnel. There are a total of 50 main tunnels that can be used in both directions, the rest are all one-way tunnels. A group of four chambers is called connected if you can get from any of these chambers to any other by using only the six tunnels that connect these chambers to each other. Determine the largest possible number of connected groups of four.\n\n## Solution",
      "answer": "9650"
    },
    {
      "question": "4. For a positive integer $n$, if $(x y-5 x+3 y-15)^{n}$ is expanded and like terms are combined, $x^{i} y^{j}(i, j=0,1$, $\\cdots, n)$ after combining have at least 2021 terms, then the minimum value of $n$ is $\\qquad$.",
      "answer": "44"
    },
    {
      "question": "3. A triangle $ABC$ with side lengths $AB=7 \\text{~cm}, BC=6 \\text{~cm}, CA=5 \\text{~cm}$ has been cut out of paper. It was folded along a line so that vertex $C$ ended up at point $C_{1}$ on side $AB$. Moreover, in the resulting quadrilateral $AKMB$, two angles adjacent to the fold line $KM$ turned out to be equal. Find $AC_{1}$ and $C_{1}B$.",
      "answer": "\\frac{42}{11}"
    },
    {
      "question": "Let $P$ be a polynomial with integer coefficients such that: $P(-1)=-4, P(-3)=-40$ and $P(-5)=$\n$-156$.\n\nWhat is the maximum number of integers $x$ for which: $P(P(x))=x^{2}$?",
      "answer": "0"
    },
    {
      "question": "Example 1 Write three integers on a blackboard, then erase one of them and replace it with the difference of 1 and the sum of the other two numbers. After repeating this process several times, the result is $(17,1999,2105)$. Could the three numbers initially written on the blackboard have been $(2,2,2)$ or $(3,3,3)$?",
      "answer": "132"
    },
    {
      "question": "2. A rectangle with sides $6 \\mathrm{~cm}$ and $3 \\mathrm{~cm}$ was cut into three rectangles of equal perimeter. What can the perimeter of these rectangles be? Find all possible answers.",
      "answer": "14"
    },
    {
      "question": "8. [7] Triangle $A B C$ has side lengths $A B=231, B C=160$, and $A C=281$. Point $D$ is constructed on the opposite side of line $A C$ as point $B$ such that $A D=178$ and $C D=153$. Compute the distance from $B$ to the midpoint of segment $A D$.",
      "answer": "208"
    },
    {
      "question": "In triangle $ABC$, the sides are known: $AB=6$, $BC=4$, $AC=8$. The bisector of angle $C$ intersects side $AB$ at point $D$. A circle is drawn through points $A$, $D$, and $C$, intersecting side $BC$ at point $E$. Find the area of triangle $ADE$.\n\n#",
      "answer": "\\frac{3\\sqrt{15}}{2}"
    },
    {
      "question": "9. In how many ways can the cells of a $4 \\times 4$ table be filled in with the digits $1,2, \\ldots, 9$ so that each of the 4 -digit numbers formed by the columns is divisible by each of the 4-digit numbers formed by the rows?",
      "answer": "9"
    },
    {
      "question": "6. As shown in the figure, from a rectangular cardboard with a length of 13 cm and a width of 9 cm, squares with a side length of 2 cm are cut from each corner, and then, the cardboard is folded along the dotted lines to form a rectangular container. What is the volume of this container in cubic centimeters?",
      "answer": "90"
    },
    {
      "question": "24. The League of Red Death. During a raid on the headquarters of a secret organization, the police found a scrap of paper, depicted in the image.\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_56f37ca27ac3d928838dg-015.jpg?height=255&width=494&top_left_y=1957&top_left_x=787)\n\n- \"I've been working on this piece of paper for three days now,\" the detective said. \"It shows the total amount of membership fees for this year: $3007.37, but the number of members (I know there are no more than 500) and the amount of one fee are obscured so that they cannot be read. How many members are there in the League of Red Death, and what is the amount of the membership fee?\"\n\nOf course, the fee cannot contain fractional cents.",
      "answer": "311"
    },
    {
      "question": "Task 1. From 3 deciliters of cough syrup, which contained $10 \\%$ distilled water, the pharmacist prepared a syrup for a child, which was supposed to contain $40 \\%$ water. How many deciliters of water did she add to the three deciliters of syrup?",
      "answer": "1.5"
    },
    {
      "question": "10. (6 points) On her birthday, Grandma told Xiaoming: \"I have only had 18 birthdays since I was born.\" Grandma should be   years old this year.",
      "answer": "72"
    },
    {
      "question": "18. Distribution of Money. Nine friends $A, B, C, D, E, F, G$, $H, K$, gathering one evening to spend time together, did the following. First, $A$ gave each of the other eight people as much money as they already had. Then $B, C$, and so on up to $K$ did the same. After this, it turned out that all nine friends had the same amount of money.\n\nCan you find the smallest sum in cents that each of the participants could have had initially?",
      "answer": "512"
    },
    {
      "question": "53. Find a three-digit number, every integer power of which ends in the three digits that make up the original number.",
      "answer": "625376"
    },
    {
      "question": "Write in ascending order the multiples of 3 that, when added to 1, are perfect squares, that is, $3,15,24,48, \\ldots$ What is the multiple of 3 in the $2006^{\\mathrm{th}}$ position?",
      "answer": "9060099"
    },
    {
      "question": "2. Joško added the largest three-digit number divisible by 9 and the smallest three-digit number not divisible by 9, while Fran added the largest three-digit number not divisible by 9 and the smallest three-digit number divisible by 9. Which sum is greater and by how much?",
      "answer": "7"
    },
    {
      "question": "In the city of Oropis, there is a triangular lake with each of its three sides being part of the perimeter of a square-shaped land with areas of $370 \\mathrm{~m}^{2}$, $116 \\mathrm{~m}^{2}$, and $74 \\mathrm{~m}^{2}$, as shown in the first figure below. The mayor of Oropis, Arnaldo, wants to calculate the area of the lake but doesn't know how. The mayor's assistant, Bernaldo, has an idea. He says that it is enough to find integer values of $a, b, c$, and $d$ that satisfy the geometric conditions of the second figure, where the shaded area is congruent to the lake. He claims that after this, the task will become much simpler.\n![](https://cdn.mathpix.com/cropped/2024_05_01_10107d2fba0ec26bc845g-33.jpg?height=892&width=1442&top_left_y=766&top_left_x=404)\n\n(a) Determine integer values of $a, b, c$, and $d$ that satisfy the geometric conditions of Bernaldo's figure.\n\n(b) Determine the area of the lake.",
      "answer": "11"
    },
    {
      "question": "7. If the set $S=\\{1,2, \\cdots, n\\}$ has three-element subsets $A_{1}, A_{2}, \\cdots, A_{m}(m>1)$ satisfying:\n(1) Each two-element subset of $S$ is contained in exactly one $A_{i}$ $(1 \\leqslant i \\leqslant m)$;\n(2) $A_{i}$ and $A_{j}(i \\neq j, 1 \\leqslant i, j \\leqslant m)$ have exactly one common element.\nThen $m=$ . $\\qquad$",
      "answer": "7"
    },
    {
      "question": "3. The weight is made of an alloy of four metals and weighs 16.3 kg. The weight of the first metal in this alloy is one and a half times greater than that of the second, the weight of the second metal is to the weight of the third as $3: 4$, and the weight of the third to the weight of the fourth - as $5: 6$. Determine the weight of the fourth metal.",
      "answer": "4.8"
    },
    {
      "question": "22. Yura left the house for school 5 minutes later than Lena, but walked twice as fast as she did. How many minutes after leaving will Yura catch up to Lena?",
      "answer": "5"
    },
    {
      "question": "12. (12 points) The figure below is a $6 \\times 6$ grid. Fill in the blank cells with the numbers $1 \\sim 6$ such that each row and each column contains the numbers $1 \\sim 6$ exactly once. The grid is also divided into 6 regions by bold lines, and each region must contain the numbers $1 \\sim 6$ exactly once. What is the four-digit number $\\overline{\\mathrm{ABCD}}$ formed by the first 4 numbers in the bottom row? $\\qquad$",
      "answer": "2413"
    },
    {
      "question": "Right-angled triangle $ABC$ has legs $|AB|=5 \\text{ cm}$ and $|BC|=12 \\text{ cm}$. The vertices of triangle $ABC$ are mapped such that\n\n- point $A$ is mapped to point $A^{\\prime}$ symmetrically with respect to the angle bisector of $\\angle ABC$,\n- point $B$ is mapped to point $B^{\\prime}$ symmetrically with respect to the midpoint of segment $BC$,\n- point $C$ is mapped to point $C^{\\prime}$ symmetrically with respect to the perpendicular bisector of segment $AB$.\n\nDetermine the area of triangle $A^{\\prime} B^{\\prime} C^{\\prime}$.\n\n(M. Dillingerová)",
      "answer": "17.5"
    },
    {
      "question": "$4 \\cdot 202$ City A has 157 tons of goods to be transported to City B, large trucks have a load capacity of 5 tons, small trucks have a load capacity of 2 tons, large trucks consume 20 liters of fuel, small trucks consume 10 liters of fuel, how many large trucks and small trucks should be used to minimize fuel consumption?",
      "answer": "31"
    },
    {
      "question": "4. If the numerator of a fraction is reduced by $8\\%$, and the denominator is increased by $8\\%$, the new fraction will be 2 less than the original fraction. Determine the value of the original fraction.",
      "answer": "\\frac{27}{2}"
    },
    {
      "question": "20. (3 points) The PLA soldiers need to repair a dam while the flood continuously destroys it. If 10 people need 45 minutes, and 20 people need 20 minutes, then 14 people need $\\qquad$ minutes to repair the dam.",
      "answer": "30"
    },
    {
      "question": "Problem 10.6. In a convex quadrilateral $A B C D$, the midpoint of side $A D$ is marked as point $M$. Segments $B M$ and $A C$ intersect at point $O$. It is known that $\\angle A B M=55^{\\circ}, \\angle A M B=$ $70^{\\circ}, \\angle B O C=80^{\\circ}, \\angle A D C=60^{\\circ}$. How many degrees does the angle $B C A$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_ddcc40423d5e6a42b0ccg-39.jpg?height=371&width=709&top_left_y=89&top_left_x=372)",
      "answer": "35"
    },
    {
      "question": "3. (8 points) In rectangle $A B C D$, $B E=5, E C=4, C F=4, F D=1$, as shown in the figure, then the area of $\\triangle A E F$ is $\\qquad$ ;",
      "answer": "20"
    },
    {
      "question": "Kirienko d:\n\nSasha and Masha each thought of a natural number and told them to Vasya. Vasya wrote the sum of the numbers on one piece of paper and their product on another, then hid one of the papers and showed the other (which had the number 2002 written on it) to Sasha and Masha. Seeing this number, Sasha said that he did not know what number Masha had thought of. Hearing this, Masha said that she did not know what number Sasha had thought of. What number did Masha think of?",
      "answer": "1001"
    },
    {
      "question": "Ggrmanova T.A.\n\nTanya sequentially wrote down numbers of the form $\\{n \\wedge 7-1\\}$ for natural numbers $n=2,3, \\dots$ and noticed that for $n=8$ the resulting number is divisible by 337. For what smallest $n>1$ will she get a number divisible by $2022?$",
      "answer": "79"
    },
    {
      "question": "In triangle $A B C$, $A B=A C$ and $\\angle B A C=20^{\\circ}$. Let $E$ and $F$ be the points on sides $A B$ and $A C$ such that $\\angle B C E=50^{\\circ}$ and $\\angle C B F=60^{\\circ}$. What angle do the lines $B C$ and $E F$ form? (H)",
      "answer": "30"
    },
    {
      "question": "6. For $n \\in \\mathbf{N}^{*}$, let $S_{n}$ be the minimum value of $\\sum_{k=1}^{n} \\sqrt{(2 k-1)^{2}+a_{k}^{2}}$, where $a_{1}, a_{2}, \\cdots, a_{n} \\in \\mathbf{R}^{+}$, and $\\sum_{k=1}^{n} a_{k}=17$. If there exists $n$ such that $S_{n}$ is also an integer, find all values of $n$.",
      "answer": "12"
    },
    {
      "question": "3. In the diagram, $O A B$ is a quarter circle, $O E$ and $C D$ are parallel, $B C$ and $E A$ are parallel, and $\\angle B C D=$ $4 \\times \\angle O B C$.\nWhat is the size of $\\angle O B C$ ?\n\nQuadrilateral $F C G E$ must be a parallelogram, as $O E$ is parallel to $C D$ and $B C$ is parallel to $E A$ (given in the question).\n\nWe have that $\\angle B C D=\\angle A E O=4 x^{\\circ}$, as opposite angles in a parallelogram are equal and we are told in the question that $\\angle B C D=4 \\times \\angle O B C$.\n\nTriangle $O A E$ is isosceles as $O A=O E$ because both are radii of the quarter circle $O A B$, thus $\\angle O A E=\\angle A E O=$ $4 x^{\\circ}$.\n\nAlso $\\angle O C B=\\angle O A E$, as $B C$ is parallel to $E A$ and hence they are corresponding angles.\n\nAs $O A B$ is a quarter circle then $\\angle B O A=90^{\\circ}$.\nConsidering the sum of the interior angles of the right-angled triangle $O C B$,\n$$\n180^{\\circ}=90^{\\circ}+4 x^{\\circ}+x^{\\circ}\n$$\n\nFinally, the previous equation simplifies to $90^{\\circ}=5 x^{\\circ}$ and so $\\angle O B C=x^{\\circ}=18^{\\circ}$.",
      "answer": "18"
    },
    {
      "question": "B3. The diagram shows the distribution of bronze, silver, and gold medals at a competition.\n\n(a) What is the value of $x$? (see image)\n\n(b) Write the ratio of gold, silver, and bronze medals in the smallest possible natural numbers.\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_b93fa657b92bc897cb7bg-05.jpg?height=417&width=423&top_left_y=126&top_left_x=1496)\n\n(c) At the competition, 84 medals were awarded. How many bronze, how many silver, and how many gold medals were awarded?\n\n(d) How many grams does a bronze medal weigh if it has the shape of a cylinder with a diameter of $7 \\mathrm{~cm}$ and a height of 3 $\\mathrm{mm}$? The density of bronze is $8700 \\frac{\\mathrm{kg}}{\\mathrm{m}^{3}}$.\n\n## 21st Mathematics Knowledge Competition for Vocational School Students National Competition, May 15, 2021\n\n## Problems for 3rd Year\n\nTime for solving: 90 minutes. In section A, we will award two points for a correct answer, while we will deduct half a point for an incorrect answer. Enter the answers for section A in the left table. In section B, we will award up to seven points for a correct answer.\n![](https://cdn.mathpix.com/cropped/2024_06_07_b93fa657b92bc897cb7bg-06.jpg?height=162&width=1198&top_left_y=590&top_left_x=444)",
      "answer": "100"
    },
    {
      "question": "7.208. $2^{x-1}+2^{x-4}+2^{x-2}=6.5+3.25+1.625+\\ldots$ (the expression on the right side is an infinite geometric progression).",
      "answer": "4"
    },
    {
      "question": "9. As shown in the figure, $A B C D$ and $E G F O$ are both squares, where point $O$ is the center of square $A B C D$, and $E F / / B C$. If the lengths of $B C$ and $E F$ are both positive integers, and the area of quadrilateral $B C E F$ is 3.25, then $\\mathrm{S}_{A B C D}-\\mathrm{S}_{E G F O}=$ $\\qquad$ ($\\mathrm{S}_{E G F O}$ represents the area of $E G F O$, and so on).",
      "answer": "11.5"
    },
    {
      "question": "6.5. The arithmetic mean of four numbers is 10. If one of these numbers is erased, the arithmetic mean of the remaining three increases by 1; if instead another number is erased, the arithmetic mean of the remaining numbers increases by 2; and if only the third number is erased, the arithmetic mean of the remaining increases by 3. By how much will the arithmetic mean of the remaining three numbers change if the fourth number is erased?",
      "answer": "6"
    },
    {
      "question": "52. Given a triangle $T$.\n\na) Place a centrally symmetric polygon $m$ of the largest possible area inside $T$.\n\nWhat is the area of $m$ if the area of $T$ is 1?\n\nb) Enclose $T$ in a convex centrally symmetric polygon $M$ of the smallest possible area.\n\nWhat is the area of $M$ if the area of $T$ is 1?",
      "answer": "2"
    },
    {
      "question": "Task B-1.6. Solve the equation\n\n$$\n\\frac{x+3}{12(x+1)}:\\left(\\frac{2 x-3}{3 x-3}-\\frac{3 x-1}{4 x+4}+\\frac{x^{2}-7 x+14}{12 x^{2}-12}\\right)=2015\n$$",
      "answer": "2012"
    },
    {
      "question": "9. On the table, there are param 1 externally identical coins. It is known that among them, there are exactly param 2 counterfeit ones. You are allowed to point to any two coins and ask whether it is true that both these coins are counterfeit. What is the minimum number of questions needed to guarantee getting at least one \"Yes\" answer?\n\n| param1 | param2 |  |\n| :---: | :---: | :---: |\n| 120 | 60 |  |",
      "answer": "63"
    },
    {
      "question": "A1 Determine the positive integer $n$ that satisfies the following equation:\n$$\n\\frac{1}{2^{10}}+\\frac{1}{2^{9}}+\\frac{1}{2^{8}}=\\frac{n}{2^{10}} .\n$$",
      "answer": "7"
    },
    {
      "question": "74. A school uses a drug fumigation method to disinfect classrooms. It is known that when the drug is burning, the content of the drug $y(\\mathrm{mg})$ per cubic meter of air is directly proportional to the time $x$ (min); after the drug burns out, $y$ is inversely proportional to $x$ (as shown in the figure). It is measured that the drug burns out in $8 \\mathrm{~min}$, at which point the content of the drug in the air per cubic meter is $6 \\mathrm{mg}$. Please answer the following questions based on the information provided in the question:\n(1) When the drug is burning, the function relationship of $y$ with respect to $x$ is: $\\qquad$, the range of the independent variable $x$ is: $\\qquad$, after the drug burns out, the function relationship of $y$ with respect to $x$ is: $\\qquad$;\n(2) Research indicates that students can return to the classroom only when the content of the drug in the air per cubic meter is below $1.6 \\mathrm{mg}$. How many minutes at least from the start of disinfection must pass before students can return to the classroom? $\\qquad$ $\\min$;\n(3) Research indicates that the air-borne pathogens can be effectively killed only when the content of the drug in the air per cubic meter is no less than $3 \\mathrm{mg}$ and lasts for no less than $10 \\mathrm{~min}$. Is this disinfection effective? Answer: $\\qquad$",
      "answer": "30"
    },
    {
      "question": "How many rectangles can be seen in the figure, and how many of them are squares? (There are only two different distances between adjacent parallel lines, their ratio is $1: 2$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_02_f7d7d7aa1e130da3e5a8g-1.jpg?height=696&width=702&top_left_y=270&top_left_x=703)",
      "answer": "6975"
    },
    {
      "question": "11. (5 points) Arrange the natural numbers $1,2,3,4$, in ascending order without any gaps, to get: 1234567891011121314. In this sequence of digits, when the first set of 5 consecutive even digits appears, the position of the first (even) digit from the left is the how many-th digit?",
      "answer": "490"
    },
    {
      "question": "4. (10 points) The product of three consecutive natural numbers is 120, their sum is $\\qquad$",
      "answer": "15"
    },
    {
      "question": "4. (5 points) Definition: If $a: b=b: c$, then $b$ is called the mean proportional between $a$ and $c$; for example, $1: 2=2: 4$, so 2 is the mean proportional between 1 and 4. $\\qquad$",
      "answer": "0.48"
    },
    {
      "question": "8. (10 points) The integer $n$ has a total of 10 factors. These factors, arranged in ascending order, have the 8th one as $\\frac{\\mathrm{n}}{3}$. What is the maximum value of the integer $n$? $\\qquad$ .",
      "answer": "162"
    },
    {
      "question": "Problem 10.1. Find the least natural number $a$ such that the equation $\\cos ^{2} \\pi(a-x)-2 \\cos \\pi(a-x)+\\cos \\frac{3 \\pi x}{2 a} \\cdot \\cos \\left(\\frac{\\pi x}{2 a}+\\frac{\\pi}{3}\\right)+2=0$ has a root.",
      "answer": "6"
    },
    {
      "question": "5. A square $A B C D$ has side-length 2 , and $M$ is the midpoint of $B C$. The circle $S$ inside the quadrilateral $A M C D$ touches the three sides $A M, C D$ and $D A$. What is its radius?",
      "answer": "3-\\sqrt{5}"
    },
    {
      "question": "1.3. Replace the ellipsis with such a natural number $p$ so that the answer to the following question is unique: how many lines are drawn on the plane, if it is known that they intersect at ... different points?",
      "answer": "2"
    },
    {
      "question": "1. In a recipe for soup, it says that for 5 servings of this soup, 30 dag of carrots are needed. If the cook wants to cook soup for 60 people and prepare 2 servings of soup per person, how many carrots should be put into this soup?",
      "answer": "720"
    },
    {
      "question": "5. Consider a $2008 \\times 2008$ chess board. Let $M$ be the smallest number of rectangles that can be drawn on the chess board so that the sides of every cell of the board is contained in the sides of one of the rectangles. Find the value of $M$. (For example, for the $2 \\times 3$ chess board, the value of $M$ is 3 .)",
      "answer": "2009"
    },
    {
      "question": "A positive integer that is the same when read forwards or backwards is called a palindrome. A table is created, as shown, in which the first column contains all palindromes between 1000 and 10000 , listed in\n\n| Palindrome | Difference |\n| :---: | :---: |\n| 1001 | 110 |\n| 1111 | 110 |\n| 1221 | 110 |\n| 1331 | 110 |\n| 1441 | $\\vdots$ |\n| $\\vdots$ |  |\n\nporitive diference between each pair of consecutive palindromes. For example, 1001 and 1111 are consecutive palindromes and their positive difference is $1111-1001=110$.\n\n(a) What are the eighth and ninth palindromes in the first column of the table?\n\n(b) The second column of the table contains only two different numbers: 110 and $x$. Determine the value of $x$.\n\n(c) When the table is complete, there are $N$ palindromes in the first column. What is the value of $N$ ?\n\n(d) Determine the average of all of the $N-1$ numbers in the second column of the table.",
      "answer": "\\frac{8998}{89}"
    },
    {
      "question": "9.8 The commission consists of a chairman, his deputy, and five other people. In how many ways can the commission members distribute among themselves the duties of chairman and deputy?",
      "answer": "42"
    },
    {
      "question": "The 25th question: In country A with 2011 cities, there is a unique bidirectional flight route operated by a certain airline between any city in A and any city in country B, which also has 2011 cities. There are no flight routes between cities within the same country, and at most 19 different airlines operate these routes. Find the maximum integer $\\mathrm{k}$, such that no matter how the routes are arranged, there are always $\\mathrm{k}$ cities that are connected (one can travel from one city to another, not necessarily directly), and all the routes are operated by a fixed airline.",
      "answer": "212"
    },
    {
      "question": "4. Let $x$ and $y$ be positive numbers whose sum is 2. Find the maximum value of the expression $x^{2} y^{2}\\left(x^{2}+y^{2}\\right)$.",
      "answer": "2"
    },
    {
      "question": "1.3.7 ** Let the quadratic function $f(x)=a x^{2}+b x+c(a, b, c \\in \\mathbf{R}, a \\neq 0)$ satisfy the following conditions:\n（1）For $x \\in \\mathbf{R}$, $f(x-4)=f(2-x)$, and $f(x) \\geqslant x$;\n（2）For $x \\in(0,2)$, $f(x) \\leqslant\\left(\\frac{x+1}{2}\\right)^{2}$;\n(3) The minimum value of $f(x)$ on $\\mathbf{R}$ is 0.\n\nFind the largest $m(m>1)$ such that there exists $t \\in \\mathbf{R}$, for any $x \\in[1, m]$, we have $f(x+t) \\leqslant x$.",
      "answer": "9"
    },
    {
      "question": "## Task 2.\n\nOn each field of an $n \\times n$ board, a dragon is sleeping. Dragons are neighbors if they are on fields that share a side. Every hour, Mini wakes up one dragon that has at least one living neighbor, and Maks directs that dragon towards one of its living neighbors. The awakened dragon breathes fire and destroys the chosen neighbor, then goes back to sleep.\n\nMini wants to reduce the number of dragons, while Maks wants as many dragons to remain as possible. If both choose the best they can, how many dragons will remain on the board for\n\n(a) $n=4$ ?\n\n(b) $n=5$ ?",
      "answer": "4"
    },
    {
      "question": "We write down all the numbers from 1 to one million successively. Then we replace each number with the sum of its digits. Then we repeat this process until only single-digit numbers remain. Which digit appears most frequently?",
      "answer": "1"
    },
    {
      "question": "The sum of two positive integers is 60 and their least common multiple is 273 . What are the two integers?\n\n(The least common multiple of two positive integers is the smallest positive integer which is a multiple of these two integers.)",
      "answer": "2139"
    },
    {
      "question": "Let $a_{1}, a_{2}, \\ldots$ be a sequence of integers that has infinitely many positive and infinitely many negative elements, and for every $n$, it holds that $a_{1}, a_{2}, \\ldots, a_{n}$ leave pairwise distinct remainders when divided by $n$. How many times does $2008$ appear in the sequence?",
      "answer": "1"
    },
    {
      "question": "8. Given that the parabola $P$ has the center of the ellipse $E$ as its focus, $P$ passes through the two foci of $E$, and $P$ intersects $E$ at exactly three points, then the eccentricity of the ellipse is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "\\frac{2\\sqrt{5}}{5}"
    },
    {
      "question": "G6.4 If $r, s, t, u$ are positive integers and $r^{5}=s^{4}, t^{3}=u^{2}, t-r=19$ and $d=u-s$, find the value of $d$.",
      "answer": "757"
    },
    {
      "question": "8,9 |\n\nThe diagonal of a rectangular trapezoid and its lateral side are equal.\n\nFind the midline of the trapezoid if the height of the trapezoid is 2 and the lateral side is 4.\n\n#",
      "answer": "3\\sqrt{3}"
    },
    {
      "question": "80. In $\\triangle A B C$, $C P=\\frac{1}{3} B C, C Q=\\frac{1}{4} A C, B Q$ intersects $A P$ at point $N$. If the area of $\\triangle A B C$ is $12$, the area of $\\triangle A B N$ is $\\qquad$",
      "answer": "\\frac{72}{11}"
    },
    {
      "question": "Example 5 Two periodic functions $y_{1}$ and $y_{2}$ have the smallest positive periods of $a$ and $b$ respectively, and $b=n a$, where $n \\geqslant 2, n \\in \\mathbf{N}$. If the function $y=y_{1}+y_{2}$ has the smallest positive period of $t$, then among the following 5 scenarios: (1) $t<b$;\n(5) $a<t<b$, the possible scenarios are $\\qquad$ (fill in the sequence number).",
      "answer": "(1)(3)(5)"
    },
    {
      "question": "3. In how many ways can you partition the set $\\{1,2, \\ldots, 12\\}$ into six mutually disjoint two-element sets in such a way that the two elements in any set are coprime?\n\n(Martin Panák)",
      "answer": "252"
    },
    {
      "question": "9. (15 points) As shown in the figure, two squares $A B F E$ and $C D E F$ with side lengths of 6 are combined to form rectangle $A B C D$. $G$ is the midpoint of $D E$. Connect $B G$ and it intersects $E F$ at $H$. Find the area of pentagon $C D G H F$ in the figure.",
      "answer": "33"
    },
    {
      "question": "6. The minor axis of the ellipse $\\rho=\\frac{1}{2-\\cos \\theta}$ is equal to\n\n保留了源文本的换行和格式。",
      "answer": "\\frac{2}{3}\\sqrt{3}"
    },
    {
      "question": "1. Solve the equation $3 \\cdot 9^{x}+2 \\cdot 3^{x}=1$.",
      "answer": "-1"
    },
    {
      "question": "7. Let $A B C D E$ be a square pyramid of height $\\frac{1}{2}$ with square base $A B C D$ of side length $A B=12$ (so $E$ is the vertex of the pyramid, and the foot of the altitude from $E$ to $A B C D$ is the center of square $A B C D)$. The faces $A D E$ and $C D E$ meet at an acute angle of measure $\\alpha$ (so that $0^{\\circ}<\\alpha<90^{\\circ}$ ). Find $\\tan \\alpha$.",
      "answer": "\\frac{17}{144}"
    },
    {
      "question": "3.21 A motorcyclist set off from point $A$ to point $B$, which is 120 km away from $A$. On the way back, he set off at the same speed, but had to stop for 10 minutes one hour after departure. After this stop, he continued his journey to $A$, increasing his speed by 6 km/h. What was the initial speed of the motorcyclist, given that he spent as much time on the return trip as he did on the trip from $A$ to $B$?",
      "answer": "48"
    },
    {
      "question": "3. The circle $(x+3)^{2}+(y-4)^{2}=50$ and the line $y=2 x-5$ intersect at two points. Determine the distance between these two points.",
      "answer": "2\\sqrt{5}"
    },
    {
      "question": "5. A five-digit number with all distinct digits, if the sum of any two of its digits is not 7, is called a “hidden seven number”. There are $\\qquad$ such “hidden seven numbers”.",
      "answer": "7008"
    },
    {
      "question": "28. At the meeting, there are 25 people present. In how many ways can four people be elected from them to fill 4 different positions",
      "answer": "303600"
    },
    {
      "question": "1. Several boys and girls gathered around a round table. It is known that exactly for 7 girls, the person sitting to their left is a girl, and for 12 - a boy. It is also known that for $75\\%$ of the boys, the person sitting to their right is a girl. How many people are sitting at the table?",
      "answer": "35"
    },
    {
      "question": "2. Let\n$$\nX=\\left\\{\\left(x_{1}, x_{2}, \\cdots, x_{n}\\right) \\mid x_{1}, x_{2}, \\cdots, x_{n} \\in \\mathbf{Z}\\right\\}\n$$\n\nbe the set of all lattice points in $n$-dimensional space. For any two lattice points $A\\left(x_{1}, x_{2}, \\cdots, x_{n}\\right)$ and $B\\left(y_{1}, y_{2}, \\cdots, y_{n}\\right)$ in $X$, define their distance as\n$$\n|A-B|=\\left|x_{1}-y_{1}\\right|+\\left|x_{2}-y_{2}\\right|+\\cdots+\\left|x_{n}-y_{n}\\right| .\n$$\n\nFind the minimum value of $r$ such that the points in $X$ can be $r$-colored, satisfying the following conditions:\n(1) Any two points of the same color $A$ and $B$ satisfy\n$$\n|A-B| \\geqslant 2 \\text {; }\n$$\n(2) Any two points of the same color $A$ and $B$ satisfy\n$$\n|A-B| \\geqslant 3 \\text {. }\n$$",
      "answer": "2"
    },
    {
      "question": "$3.457 A=\\operatorname{ctg}\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\operatorname{arcctg}\\left(-\\frac{1}{2}\\right)\\right)$\n\n$3.457 A=\\cot\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\operatorname{arcctg}\\left(-\\frac{1}{2}\\right)\\right)$",
      "answer": "-2"
    },
    {
      "question": "3. Given that $\\triangle A B C$ is an equilateral triangle, the ellipse $\\Gamma$ has one focus at $A$, and the other focus $F$ lies on the line segment $B C$. If the ellipse $\\Gamma$ passes exactly through points $B$ and $C$, then its eccentricity is $\\qquad$.",
      "answer": "\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "A number appears on a computer screen. We know that if $x$ appears on the screen, the number $x^{2}-2 x+1$ appears right after. If the first number to appear is 2, what is the 2020-th number to appear",
      "answer": "1"
    },
    {
      "question": "3A. Given the set of quadratic functions $f(x)=a x^{2}+b x+c$, for which $a<b$ and $f(x) \\geq 0$ for every $x \\in \\mathbb{R}$. Determine the smallest possible value of the expression $A=\\frac{a+b+c}{b-a}$.",
      "answer": "3"
    },
    {
      "question": "7.2. In Solar City, 6 dwarfs eat donuts daily, 8 dwarfs eat donuts every other day, and the rest do not eat donuts at all. Yesterday, 11 dwarfs ate donuts. How many dwarfs will eat donuts today?",
      "answer": "9"
    },
    {
      "question": "9. (1990 Hungarian Mathematical Olympiad) For any positive integer $q_{0}$, consider the sequence $q_{1}, q_{2}, \\cdots, q_{n}$ defined by $q_{i}=\\left(q_{i-1}-1\\right)^{3}+$ $3(i=1,2, \\cdots, n)$. If each $q_{i}(i=1,2, \\cdots, n)$ is a prime, find the largest possible value of $n$.",
      "answer": "2"
    },
    {
      "question": "Task 2 - 130912 Someone wants to create a new mixture with a water content of $98 \\%$ from a mixture that consists of $99 \\%$ water, by removing water from the original mixture.\n\nDetermine how much percent of the water content in the original mixture he must remove for this purpose in total.",
      "answer": "51"
    },
    {
      "question": "153. a) Find all three-digit numbers that are equal to the sum of the factorials of their digits.\n\nb) Find all integers that are equal to the sum of the squares of their digits.",
      "answer": "1"
    },
    {
      "question": "Some unit-radius circles are drawn on the plane, and we color the center of each circle blue. We mark some points on the circumference of the circles in red such that exactly 2 red points lie on each circumference. What is the maximum number of blue points if there are a total of 25 colored points?",
      "answer": "20"
    },
    {
      "question": "3. Write down all three-digit numbers whose sum of digits is equal to 9. How many are there?",
      "answer": "45"
    },
    {
      "question": "[ Sum of angles in a triangle. Theorem about the exterior angle. ] [ Criteria and properties of an isosceles triangle. ]\n\nIn triangle $A B C$, angle $C$ is three times angle $A$. A point $D$ is taken on side $A B$ such that $B D=B C$.\n\nFind $C D$, if $A D=4$.",
      "answer": "4"
    },
    {
      "question": "[ Arithmetic. Mental calculation, etc.]\n\nThe sum of the subtrahend, minuend, and difference is 2016. Find the minuend.\n\n#",
      "answer": "1008"
    },
    {
      "question": "6. Find all positive integers $n$ such that $3^{2 n}+3 n^{2}+7$ is a perfect square.",
      "answer": "2"
    },
    {
      "question": "6.53. $\\lim _{x \\rightarrow \\infty} \\frac{x}{e^{x}}$.",
      "answer": "0"
    },
    {
      "question": "74. In a rectangular coordinate system, there are four points $A(-1,1)$, $B(5,0)$, $C(3,-3)$, and $D(-3,-2)$. After moving point $A$ down by one unit and moving point $C$ up by one unit, the area of the new quadrilateral $ABCD$ is . $\\qquad$",
      "answer": "12"
    },
    {
      "question": "4. Let $p(x)=2 x^{3}-3 x^{2}+1$. How many squares of integers are among the numbers $p(1), p(2), \\ldots, p(2016)$?",
      "answer": "32"
    },
    {
      "question": "5. Let for all real numbers $x$, the function $f(x)$ satisfies $x f(x)=2 f(1-x)+1$.\nThen $f(5)=$ $\\qquad$",
      "answer": "\\frac{1}{12}"
    },
    {
      "question": "15. Five Bunchkins sit in a horizontal field. No three of the Bunchkins are sitting in a straight line. Each Bunchkin knows the four distances between her and each of the others. Each Bunchkin calculates and then announces the total of these distances. These totals are $17,43,56,66$ and 76. A straight line is painted joining each pair of Bunchkins. What is the total length of paint required?",
      "answer": "129"
    },
    {
      "question": "3. Let the function $f(x)=\\frac{(x+1)^{2}+\\sin x}{x^{2}+1}$ have the maximum value and minimum value as $M$ and $N$, respectively. Then $M+N=$ $\\qquad$",
      "answer": "2"
    },
    {
      "question": "I1.2 Let $20112011=a P^{5}+b P^{4}+c P^{3}+d P^{2}+e P+f$, where $a, b, c, d, e$ and $f$ are integers and $0 \\leq a$, $b, c, d, e, f<P$. If $Q=a+b+c+d+e+f$, find the value of $Q$.",
      "answer": "36"
    },
    {
      "question": "19. (Mathematics for Middle School, Issue 4, 2004, Olympiad Problem 137) Try to find the maximum number of elements in a set $S$ that satisfies the following conditions:\n(1) Each element in $S$ is a positive integer not exceeding 100;\n(2) For any two different elements $a$ and $b$ in $S$, there exists another element $c$ in $S$ such that the greatest common divisor of $a+b$ and $c$ is 1;\n(3) For any two different elements $a$ and $b$ in $S$, there exists another element $c$ in $S$ such that the greatest common divisor of $a+b$ and $c$ is greater than 1.",
      "answer": "50"
    },
    {
      "question": "6. Let $S$ be the set of all rational numbers in the interval $\\left(0, \\frac{5}{8}\\right)$, for the fraction $\\frac{q}{p} \\in S,(p, q)=1$, define the function $f\\left(\\frac{q}{p}\\right)=\\frac{q+1}{p}$. Then the number of roots of $f(x)=\\frac{2}{3}$ in the set $S$ is $\\qquad$",
      "answer": "5"
    },
    {
      "question": "10. For what value of $a$ does the inequality $\\log _{\\frac{1}{4}}\\left(\\sqrt{x^{2}+a x+5}+1\\right) \\cdot \\log _{5}\\left(x^{2}+a x+6\\right)+\\log _{a} 3 \\geqslant 0$ have exactly one solution?",
      "answer": "2"
    },
    {
      "question": "4B. If $\\cos x+\\cos ^{2} x+\\cos ^{3} x=1$, calculate the value of the expression\n\n$$\n\\sin ^{6} x-4 \\sin ^{4} x+8 \\sin ^{2} x\n$$",
      "answer": "4"
    },
    {
      "question": "Example 5 Let the set $A=\\{1,2,3,4,5,6\\}$, and the mapping $f: A \\rightarrow A$, such that its third composition $f \\cdot f \\cdot f$ is the identity mapping. How many such $f$ are there?\n(1996 Japan Mathematical Olympiad Preliminary)",
      "answer": "81"
    },
    {
      "question": "8. (10 points) Place seven consecutive natural numbers at the intersection points $A, B, C, D, E, F, G$ of five circles, such that the sum of the numbers on each circle is equal. If all the numbers filled in are greater than 0 and no more than 10, then the number filled at point $G$ is.",
      "answer": "6"
    },
    {
      "question": "Task 1. (5 points) Find $\\frac{a^{12}+4096}{64 a^{6}}$, if $\\frac{a}{2}-\\frac{2}{a}=3$.\n\n#",
      "answer": "1298"
    },
    {
      "question": "13. (12 points) $A$ and $B$ are two places where a bus is dispatched every 5 minutes, traveling at a uniform speed towards each other, and all buses travel at the same speed; two people, Jia and Yi, start from $A$ and $B$ respectively, traveling towards each other at a uniform speed; 5 minutes after Jia and Yi start, the first bus departs from both places; when Jia and Yi meet, Jia is caught up by the 9th bus from $A$, and Yi is also caught up by the 6th bus from $B$; when Yi arrives at $A$, he is caught up by the 8th bus from $B$, and at this time, Jia is still 21 kilometers away from $B$. What is Yi's speed in kilometers per hour?",
      "answer": "27"
    },
    {
      "question": "The distinct prime factors of 18 are 2 and 3 . What is the sum of all of the distinct prime factors of $4446 ?$",
      "answer": "37"
    },
    {
      "question": "2. From $1,2,3, \\cdots, 14$ select $a_{1}, a_{2}, a_{3}$ in ascending order, such that $a_{2}-a_{1} \\geqslant 3$, $a_{3}-a_{2} \\geqslant 3$ are simultaneously satisfied. The number of all different selection methods that meet the above requirements is $\\qquad$.",
      "answer": "120"
    },
    {
      "question": "We start with 5000 forints in our pocket to buy gifts. We enter three stores. In each store, we like a gift item, which we will buy if we can afford it. The prices of the items in each store, independently of each other, are 1000, 1500, or 2000 Ft with a probability of $\\frac{1}{3}$. What is the probability that we will manage to buy three gift items and still have money left?",
      "answer": "\\frac{17}{27}"
    },
    {
      "question": "1. All gnomes are divided into liars and knights. Liars always lie, and knights always tell the truth. On each cell of a $4 \\times 4$ board, there is a gnome. It is known that among them, there are both liars and knights. Each gnome stated: “Among my neighbors (by side) there are an equal number of liars and knights.” How many liars are there in total?",
      "answer": "12"
    },
    {
      "question": "6. A rectangle $3 \\times 100$ consists of 300 squares $1 \\times 1$. What is the maximum number of diagonals that can be drawn in the squares so that no two diagonals share a common endpoint? (In one square, two diagonals can be drawn without sharing endpoints. Common internal points are allowed.)",
      "answer": "200"
    },
    {
      "question": "[ Percentage and ratio problems ]\n\nHow many pounds of grain need to be milled, so that after paying for the work at $-10 \\%$ of the milling - exactly 100 pounds of flour remain? There are no losses during milling.\n\n#",
      "answer": "111+\\frac{1}{9}"
    },
    {
      "question": "## Subject IV. (20 points)\n\nAmaya and Keiko are two girls from Osaka. They live on the same floor in a building with two staircases, each with 5 apartments per floor. At the ground floor of the building are shops.\n\nThe apartments are numbered in ascending order, starting from the first floor. Amaya lives in apartment 28, and Keiko in apartment 164. How many floors does the building have?\n\nProf. Sorin Borodi, \"Alexandru Papiu Ilarian\" Theoretical High School, Dej\n\nAll subjects are mandatory. 10 points are awarded by default.\n\nEffective working time - 2 hours.\n\n## Grading Scale for Grade 5 (OLM 2013 - Local Stage)\n\n## Of. $10 \\mathrm{p}$",
      "answer": "27"
    },
    {
      "question": "## Task $8 / 90$\n\nLet $n$ be a natural number with exactly $m$ digits in decimal notation $(m \\geq 2)$, which is divisible by 11. By reversing the sequence of digits, the natural number $n^{\\prime}$ (not necessarily with exactly $m$ digits) is formed. How many sums $n+n^{\\prime}$ are divisible by 11?",
      "answer": "9\\cdot\\frac{10^{-1}+(-1)^{}}{11}"
    },
    {
      "question": "6. 15 cards, each card has 3 different Chinese characters, any 2 cards do not have exactly the same characters; among any 6 cards, there must be 2 cards that have a common character. Question: What is the maximum number of different Chinese characters on these 15 cards?",
      "answer": "35"
    },
    {
      "question": "41. The road from home to school takes Seryozha 30 minutes. Once on the way, he remembered that he had forgotten a pen at home. Seryozha knew that if he continued on to school at the same speed, he would arrive there 9 minutes before the bell, but if he returned home for the pen, he would, walking at the same speed, be 11 minutes late for the start of the lesson. What part of the way had he walked?",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "In a right-angled triangle, the height drawn to the hypotenuse is $12 \\mathrm{~cm}$; the difference between the two segments of the hypotenuse is $7 \\mathrm{~cm}$. What are the lengths of the sides of the triangle?",
      "answer": "15"
    },
    {
      "question": "1.2. In triangle $ABC$ with angle $BAC$ equal to $24^{\\circ}$, points $X$ and $Y$ are taken on sides $AB$ and $AC$ respectively. The circle centered at $Y$ and passing through $A$ also passes through $X$, and the circle centered at $X$ and passing through $B$ also passes through $C$ and $Y$. Find $\\angle ABC$.",
      "answer": "54"
    },
    {
      "question": "1. The quadratic trinomial $f(x)=a x^{2}+b x+c$ has exactly one root, the quadratic trinomial $f(3 x+2)-2 f(2 x-1)$ also has exactly one root. Find the root of the trinomial $f(x)$.",
      "answer": "-7"
    },
    {
      "question": "10. [7] Let $\\omega=\\cos \\frac{2 \\pi}{727}+i \\sin \\frac{2 \\pi}{727}$. The imaginary part of the complex number\n$$\n\\prod_{k=8}^{13}\\left(1+\\omega^{3^{k-1}}+\\omega^{2 \\cdot 3^{k-1}}\\right)\n$$\nis equal to $\\sin \\alpha$ for some angle $\\alpha$ between $-\\frac{\\pi}{2}$ and $\\frac{\\pi}{2}$, inclusive. Find $\\alpha$.",
      "answer": "\\frac{12\\pi}{727}"
    },
    {
      "question": "A pyramid has a base that is a right-angled quadrilateral, one side of which is $a=6 \\mathrm{~cm}$, the other side is $b=8 \\mathrm{~cm}$; each edge of the pyramid is $c=12 \\mathrm{~cm}$. At what distance from the apex must the pyramid be cut by a plane parallel to the base so that the volumes of the resulting parts are equal?",
      "answer": "8.658"
    },
    {
      "question": "## Task 2\n\nTwo birds are sitting on a branch $8 \\mathrm{~m}$ apart.\n\nNow one bird hops 1 meter towards the other. The other bird then hops 2 meters towards the first.\n\nHow far apart are the two birds now?",
      "answer": "5"
    },
    {
      "question": "Simplify the following expression as much as possible:\n\n$$\n\\left(\\frac{\\sqrt{1+a}}{\\sqrt{1+a}-\\sqrt{1-a}}+\\frac{1-a}{\\sqrt{1-a^{2}}-1+a}\\right)\\left(\\sqrt{\\frac{1}{a^{2}}-1}-\\frac{1}{a}\\right)\n$$\n\nwhere $0<a<1$.",
      "answer": "-1"
    },
    {
      "question": "13. A. For any real number $x$, let $\\lfloor x\\rfloor$ denote the greatest integer not exceeding the real number $x$, $\\{x\\}=x-\\lfloor x\\rfloor$, and $\\lceil x\\rceil$ denote the smallest integer not less than the real number $x$. If $x_{1}, x_{2}, \\cdots, x_{m}$ $\\left(0 \\leqslant x_{1}<x_{2}<\\cdots<x_{m} \\leqslant 10\\right)$ are all the real numbers between 0 and 10 that satisfy the equation $\\lfloor x\\rfloor\\{x\\}\\lceil x\\rceil=1$, find the value of $x_{1}+$ $x_{2}+\\cdots+x_{m}$.",
      "answer": "\\frac{459}{10}"
    },
    {
      "question": "(1) Given the function $f(x)=x^{3}+a x^{2}+x+1(a \\in \\mathbf{R})$ is decreasing in the interval $\\left(-\\frac{2}{3},-\\frac{1}{3}\\right)$ and increasing in the interval $\\left(-\\frac{1}{3},+\\infty\\right)$, then $a=$",
      "answer": "2"
    },
    {
      "question": "6. The solution to the equation $2^{x^{2}-2 x}+3^{x^{2}-2 x}+x^{2}-2 x+\\frac{1}{6}=0$ is",
      "answer": "1"
    },
    {
      "question": "Problem 8.6. For quadrilateral $ABCD$, it is known that $AB=BD, \\angle ABD=\\angle DBC, \\angle BCD=$ $90^{\\circ}$. A point $E$ is marked on segment $BC$ such that $AD=DE$. What is the length of segment $BD$, if it is known that $BE=7, EC=5$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_c49ecf27446af41a8bcdg-30.jpg?height=476&width=517&top_left_y=91&top_left_x=468)",
      "answer": "17"
    },
    {
      "question": "5. [5 points] On a plane with a given rectangular Cartesian coordinate system, a square is drawn with vertices at points $(0 ; 0),(0 ; 69),(69 ; 69)$, and ( $69 ; 0)$. Find the number of ways to choose two grid nodes inside this square (not including its boundary) such that at least one of these nodes lies on one of the lines $y=x$ or $y=69-x$, but neither of the chosen nodes lies on any line parallel to any of the coordinate axes.",
      "answer": "601460"
    },
    {
      "question": "Determine all positive real numbers $x$ and $y$ satisfying the equation\n\n$$\nx+y+\\frac{1}{x}+\\frac{1}{y}+4=2 \\cdot(\\sqrt{2 x+1}+\\sqrt{2 y+1}) .\n$$",
      "answer": "1+\\sqrt{2}"
    },
    {
      "question": "Andjans A.\n\nIn a table with $m$ rows and $n$ columns, a horizontal move is defined as a permutation of the elements in the table such that each element remains in the same row it was in before the permutation; similarly, a vertical move is defined (\"row\" in the previous definition is replaced with \"column\"). Determine such a $k$ that any permutation of the elements in the table can be achieved in $k$ moves (of any kind), but there exists a permutation that cannot be achieved in fewer moves.",
      "answer": "3"
    },
    {
      "question": "3. If $z$ is a complex number with a non-zero real part, then the minimum value of $\\frac{\\operatorname{Re}\\left(z^{4}\\right)}{(\\operatorname{Re}(z))^{4}}$ is $\\qquad$",
      "answer": "-8"
    },
    {
      "question": "18. There are three types of people, A, B, and C, totaling 25 people. A type people always tell the truth, B type people always lie, and C type people alternate between telling the truth and lying (for example, if a C type person tells the truth this time, the next thing they say will definitely be a lie, and the one after that will be the truth again).\nThe priest asked everyone, “Are you an A type person?” 17 people answered “yes”.\nThe priest then asked everyone, “Are you a C type person?” 12 people answered “yes”.\nThe priest then asked everyone, “Are you a B type person?” 8 people answered “yes”.\n\nAmong these 25 people, ( ) are C type people.",
      "answer": "16"
    },
    {
      "question": "3. [4] Let $A B C$ be a triangle with $\\angle B A C=90^{\\circ}$. A circle is tangent to the sides $A B$ and $A C$ at $X$ and $Y$ respectively, such that the points on the circle diametrically opposite $X$ and $Y$ both lie on the side $B C$. Given that $A B=6$, find the area of the portion of the circle that lies outside the triangle.",
      "answer": "\\pi-2"
    },
    {
      "question": "I3.1 Given that $\\frac{4}{\\sqrt{6}+\\sqrt{2}}-\\frac{1}{\\sqrt{3}+\\sqrt{2}}=\\sqrt{a}-\\sqrt{b}$. If $m=a-b$, find the value of $m$.",
      "answer": "3"
    },
    {
      "question": "Task 1. A grass path surrounds a rectangular asphalt playground with dimensions $36 \\mathrm{dm}$ and $12 \\mathrm{~m}$ (as shown in the diagram). The width $d$ of the path is equal to a quarter of the semiperimeter of the playground. How many meters of wire are needed to fence the grass path around the playground, if it is surrounded by 5 rows of wire? How many posts are needed to secure the fence, if the distance between any two adjacent posts is $6 \\mathrm{dm}$?",
      "answer": "468"
    },
    {
      "question": "I2.4 The sum of $2 C$ consecutive even numbers is 1170 . If $D$ is the largest of them, find $D$.",
      "answer": "68"
    },
    {
      "question": "Booting D..\n\nKnower came to visit the twin brothers Vintik and Shpuntik, knowing that one of them never tells the truth, and asked one of them: \"Are you Vintik?\" \"Yes,\" — answered the one. When Knower asked the same question of the second, he received just as clear an answer and immediately determined who was who.\n\nWho was called Vintik?",
      "answer": "1"
    },
    {
      "question": "12. If: (1) $a, b, c, d$ all belong to $\\{1,2,3,4\\}$;\n(2) $a \\neq b, b \\neq c, c \\neq d, d \\neq a$;\n(3) $a$ is the smallest value among $a, b, c, d$,\n\nthen, the number of different four-digit numbers $\\overline{a b c d}$ that can be formed is",
      "answer": "28"
    },
    {
      "question": "6. (10 points) The distance between $A$ and $B$ is 100 kilometers. Car A takes $m$ hours to travel from $A$ to $B$, and Car B takes $n$ hours to travel from $A$ to $B$, where $m, n$ are integers. Now, Car A starts from $A$ and Car B starts from $B$ at the same time, heading towards each other, and they meet at point $C$ after 5 hours. If Car A has already traveled half of its distance, then the distance from $C$ to $A$ is $\\qquad$ kilometers.",
      "answer": "83\\frac{1}{3}"
    },
    {
      "question": "78 Given $x \\geqslant 0, y \\geqslant 0$, and $x+y \\leqslant 2 \\pi$, try to find the maximum and minimum values of the function $f(x, y)=\\sin x+\\sin y- \\sin (x+y)$.",
      "answer": "\\frac{3\\sqrt{3}}{2}"
    },
    {
      "question": "Example 3 Given that $f(x)$ is a function defined on $\\mathbf{R}$, $f(1)=1$. And for any $x \\in \\mathbf{R}$, there are $f(x+5) \\geqslant f(x)+5, f(x+1) \\leqslant f(x)+1$. If $g(x)=f(x)+1-x$, find $g(2002)$.\n(2002 National High School League Question)",
      "answer": "1"
    },
    {
      "question": "## Task 12\n\nIn the three schools of a city, there are a total of 63 Pioneer groups. From each group, 4 Pioneers can go to the Pioneer meeting. They are picked up by 6 equally large buses.\n\nHow many Pioneers are in each bus?",
      "answer": "42"
    },
    {
      "question": "7. Consider a positive integer,\n$$\n\\mathrm{N}=9+99+999+\\ldots \\ldots+\\underset{2018}{999 \\ldots 9}\n$$\n\nHow many times does the digit 1 occur in its decimal representation?",
      "answer": "2014"
    },
    {
      "question": "2. Determine the maximum value of the expression\n\n$$\n\\log _{2}^{4} x+12 \\log _{2}^{2} x \\cdot \\log _{2}\\left(\\frac{8}{x}\\right)\n$$\n\nif the variable $x$ varies on the interval $[1,64]$.",
      "answer": "81"
    },
    {
      "question": "Example 6. Find $\\lim _{x \\rightarrow 3} \\frac{\\ln \\left(x^{2}-5 x+7\\right)}{x-3}$.",
      "answer": "1"
    },
    {
      "question": "6. A $41 \\times 41$ battlefield is divided into 1681 unit squares. A tank is hidden in one of these unit squares. A bomber aircraft bombs the battlefield. Each bombing targets one unit square. If the tank's square is not bombed, the tank remains in place; if it is bombed, the tank must move to an adjacent unit square (two squares are considered \"adjacent\" if they share a common edge). Destroying the tank requires hitting it twice. Throughout the process, the bomber pilot has no knowledge of the tank's location or whether it has been hit. Determine the minimum number of bombings required to ensure the tank is destroyed.",
      "answer": "2521"
    },
    {
      "question": "3. Solve the equation $\\frac{\\sqrt{(-x)^{2}}+(\\sqrt{-x})^{2}}{x^{2}+(-x)^{2}}=\\frac{1}{2016}$\n\nANSWER: $x=-2016$.",
      "answer": "-2016"
    },
    {
      "question": "12. There are 8 black, 8 white, and 8 yellow chopsticks mixed together. In the dark, you want to take out two pairs of chopsticks of different colors. How many chopsticks do you need to take out to ensure you meet the requirement?",
      "answer": "11"
    },
    {
      "question": "3.1. Alina and Masha wanted to create an interesting version of the school tour olympiad. Masha proposed several problems and rejected every second problem of Alina's (exactly half), Alina also proposed several problems and did not reject only every third problem of Masha's (exactly a third). In the end, there were 10 problems, and initially, 27 were proposed. How many more problems did Masha propose than Alina?",
      "answer": "15"
    },
    {
      "question": "B1 Give the smallest positive integer that is divisible by 26, ends in 26, and for which the sum of the digits equals 26.",
      "answer": "46826"
    },
    {
      "question": "2、Given $A ※ B=(A+3) \\times(B-2)$. Then $12 ※ 17=$ $\\qquad$ -",
      "answer": "225"
    },
    {
      "question": "Problem 8.7. Given an isosceles triangle $A B C$, where $A B=A C$ and $\\angle A B C=53^{\\circ}$. Point $K$ is such that $C$ is the midpoint of segment $A K$. Point $M$ is chosen such that:\n\n- $B$ and $M$ are on the same side of line $A C$;\n- $K M=A B$\n- angle $M A K$ is the maximum possible.\n\nHow many degrees does angle $B A M$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_4984b677816e04f7f1c9g-27.jpg?height=400&width=301&top_left_y=224&top_left_x=576)",
      "answer": "44"
    },
    {
      "question": "3. (7 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $15$, and $\\sin \\alpha = \\frac{\\sqrt{21}}{5}$?",
      "answer": "12"
    },
    {
      "question": "626. Find $f^{\\prime}(0)$, if $f(x)=e^{x} \\arcsin x+\\operatorname{arctg} x$.",
      "answer": "2"
    },
    {
      "question": "7. Given positive rational numbers $a, b$, (both not 0, and $a+b+c=abc$ then $\\frac{a}{b}+\\frac{a}{c}+\\frac{b}{a}+$ $\\frac{b}{c}+\\frac{c}{a}+\\frac{c}{b}-ab-bc-ca=$ $\\qquad$ .",
      "answer": "-3"
    },
    {
      "question": "## Task B-4.6.\n\nAn elevator can transport a maximum of eight people in one ride. Eleven residents, including two who are feuding, are waiting for the elevator, and they are all going to the same floor. What is the probability that the feuding residents will be in the elevator if all residents must be transported by the elevator in the minimum number of rides?",
      "answer": "\\frac{155}{319}"
    },
    {
      "question": "Now we have three types of salt water, A, B, and C, with concentrations of $5\\%$, $8\\%$, and $9\\%$ respectively, and weights of 60 grams, 60 grams, and 47 grams. We need to prepare 100 grams of salt water with a concentration of $7\\%$. Try to find out the maximum and minimum amount of salt water A that can be used. Please write down the sum of these two numbers as the answer to this question.",
      "answer": "84"
    },
    {
      "question": "Example 8 A deck of tri-color cards consists of 32 cards in total, with 10 cards of each color: red, yellow, and blue, numbered $1, 2, \\cdots, 10$; there are also a big joker and a small joker, each numbered 0. Several cards are drawn from this deck, and the score is calculated according to the following rule: each card numbered $k$ is worth $2^{k}$ points. If the total score of these cards is 2004, they are called a \"good\" card group. Find the number of \"good\" card groups.",
      "answer": "1006009"
    },
    {
      "question": "11.076. The height of the cone is equal to the diameter of its base. Find the ratio of the area of its base to the lateral surface area.",
      "answer": "\\frac{\\sqrt{5}}{5}"
    },
    {
      "question": "5. (6 points) There is a wall clock that strikes once at every hour, and it strikes as many times as the hour. At four o'clock, the clock takes 12 seconds to finish striking; then at twelve o'clock, it will take $\\qquad$ seconds to finish striking.",
      "answer": "44"
    },
    {
      "question": "7.51 There are 1988 identical unit cubes, and they (all or part) are used to form three \"squares\" $A, B, C$ (i.e., three one-layer-high rectangular prisms with dimensions $a \\times a \\times 1, b \\times b \\times 1, c \\times c \\times 1$ where $a \\leqslant b \\leqslant c$). Now, place square $C$ on a flat surface, then place $B$ on top of $C$ so that each small block of $B$ is exactly on top of a small block of $C$, but the boundary of $B$ does not align with the sides of $C$. Finally, place $A$ on top of $B$ following the same principle, thus forming a \"three-story building\". When $a, b, c$ take what values, can the number of different \"three-story buildings\" that can be formed be maximized?",
      "answer": "345^2"
    },
    {
      "question": "1-191 Write down all natural numbers from 1 to 1988 on a blackboard. Perform operations $A$ and $B$ on these numbers repeatedly: first $A$, then $B$, followed by $A$ again, and then $B$, and so on. Operation $A$ involves subtracting the same natural number from each number written on the blackboard (the number subtracted can be different for different instances of operation $A$). Operation $B$ involves erasing two numbers from the blackboard and writing down their sum. Operations $A$ and $B$ are performed in this sequence until, after a certain operation $B$, only one number remains on the blackboard, and it is non-negative. What is this number?",
      "answer": "1"
    },
    {
      "question": "# Task № 7.3\n\n## Condition:\n\nArtist Ivan Konstantinovich decided to sell several of his paintings at the Broken Auction. The rules of the Broken Auction are as follows: first, Ivan Konstantinovich names a certain starting price for his painting, after which the participants who want to buy this painting begin to bid on it - increasing the price, but only by two times, - that is, the new price can only be twice the last one offered; the participant after whom no one dares to raise the price by another two times gets the painting; if no one bids on the painting, that is, no one offers a price higher than the initial one, the painting remains unsold. For each of his paintings, Ivan Konstantinovich decided to set the starting price at 1000 rubles. How many paintings did he sell at the Broken Auction if all his paintings were bought for different prices, and his total revenue from all the sold paintings amounted to 350000 rubles?",
      "answer": "6"
    },
    {
      "question": "9. Given the line $l: x=m y+q, m \\neq 0$ intersects the ellipse $\\Gamma: 3 x^{2}+4 y^{2}=12$ at two distinct points $\\mathrm{A}$ and $\\mathrm{B}$. Let $\\mathrm{A}$'s reflection about the major axis of the ellipse be $A_{1}$, and $F$ be the right focus of the ellipse. Find the necessary and sufficient condition for $A_{1} 、 F 、 B$ to be collinear.",
      "answer": "4"
    },
    {
      "question": "A spinner was created by drawing five radii from the centre of a circle. The first four radii divide the circle into four equal wedges. The fifth radius divides one of the wedges into two parts, one having twice the area of the other. The five wedges are labelled as pictured with the wedge labeled by 2 having twice the area of the wedge labeled by 1 . Determine the probability of spinning an odd number.\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_d49dd011068fc0fd28d1g-1.jpg?height=333&width=320&top_left_y=1685&top_left_x=1466)",
      "answer": "\\frac{7}{12}"
    },
    {
      "question": "88. Given that $x, y, z$ are three distinct non-zero natural numbers, if\n$$\n\\overline{x y y y y}+\\overline{x y y y}+\\overline{x y y}+\\overline{x y}+y=\\overline{y y y y z} .\n$$\n\nwhere $\\overline{x y y y y}$ and $\\overline{y y y y z}$ are both five-digit numbers, when $x+y$ is maximized, the corresponding value of $x+y+z$ is",
      "answer": "22"
    },
    {
      "question": "## Task 2.\n\nAnte wrote down a sequence $a_{1}, a_{2}, \\ldots, a_{2020}$ in which each of the numbers $1,2, \\ldots, 2020$ appears exactly once. Barbara wants to find out this sequence, and Ante gives her information through a series of exchanges.\n\nIn one exchange, Barbara writes down the numbers from 1 to 2020 on a piece of paper and connects some of them with lines, ensuring that each number is connected to at most one other number, and then hands the paper to Ante. Ante then writes down the numbers from 1 to 2020 on another piece of paper and for each pair of numbers $i$ and $j$ that Barbara connected with a line, he connects the numbers $a_{i}$ and $a_{j}$ with a line, and finally hands the paper to Barbara.\n\nHow many exchanges are needed at minimum for Barbara to determine Ante's sequence with certainty?",
      "answer": "2"
    },
    {
      "question": "8.2. Find the largest four-digit number in which all digits are different and which is divisible by any of its digits (don't forget to explain why it is the largest).",
      "answer": "9864"
    },
    {
      "question": "7. When asked how old he was, a mathematician replied: \"If you subtract 5 from my age, divide the result by 5, and then subtract 5 again, you will get the number 5.\" How old is he?",
      "answer": "55"
    },
    {
      "question": "3. Given an isosceles triangle \\(ABC\\) with base \\(BC\\). On the extension of side \\(AC\\) beyond point \\(C\\), point \\(K\\) is marked, and a circle is inscribed in triangle \\(ABK\\) with center at point \\(I\\). A circle passing through points \\(B\\) and \\(I\\) is tangent to line \\(AB\\) at point \\(B\\). This circle intersects segment \\(BK\\) again at point \\(L\\). Find the angle between lines \\(IK\\) and \\(CL\\). Answer: \\(90^{\\circ}\\).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b7830eaf581def5094e2g-02.jpg?height=496&width=820&top_left_y=483&top_left_x=638)",
      "answer": "90"
    },
    {
      "question": "5. (8 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $6$ and $\\cos \\alpha = \\frac{2}{3}$?",
      "answer": "8"
    },
    {
      "question": "1.2 Mishа noticed that the tram passed by him in 4 seconds, and a tunnel 64 meters long in 12 seconds. Find the length of the tram (in meters), assuming that its speed remains constant throughout the entire observation period.",
      "answer": "32"
    },
    {
      "question": "Legyen\n\n$$\n\\begin{aligned}\na+b+c & =0 \\\\\nA+B+C & =0 \\\\\n\\frac{a}{A}+\\frac{b}{B}+\\frac{c}{C} & =0\n\\end{aligned}\n$$\n\nMennyi az\n\n$$\na A^{2}+b B^{2}+c C^{2}\n$$\n\nkifejezés értéke?",
      "answer": "0"
    },
    {
      "question": "## Task 5 - 301235\n\nInvestigate whether the sequence $\\left(x_{n}\\right)$ defined by\n\n$$\nx_{1}=1, \\quad x_{n+1}=\\frac{1}{x_{n}+1} \\quad(n=1,2,3, \\ldots)\n$$\n\nis convergent, and determine its limit if it is convergent.",
      "answer": "\\frac{\\sqrt{5}-1}{2}"
    },
    {
      "question": "1. Write a million: a) using three hundreds and operation signs; b) using six tens and operation signs.",
      "answer": "1000000"
    },
    {
      "question": "Evdokimov M.A.\n\nA convex polyhedron with vertices at the midpoints of the edges of a certain cube is called a cuboctahedron. When a cuboctahedron is intersected by a plane, a regular polygon is obtained. What is the maximum number of sides it can have?",
      "answer": "6"
    },
    {
      "question": "5、A, B, and C each have some apples, with the total being less than 100. If A gives $\\frac{1}{6}$ and $\\frac{1}{4}$ of his apples to B and C respectively, the three will have the same number of apples. Therefore, A originally had the most $\\qquad$ apples, and originally B had  more apples than C. apples.",
      "answer": "48"
    },
    {
      "question": "## 34. Alsatian Village\n\nThe main street of the village (it is called Main) is perpendicular to General de Gaulle Street; they intersect at the Town Hall Square. The Catholic cathedral faces the main street, while the Protestant one faces General de Gaulle Street. The school is located on the street connecting these two cathedrals, at a bird's flight distance (500 m) from each of the mentioned streets. All streets are straight. One day, the teacher asked his students[^9] to calculate the sum of the reciprocals of the distances (in meters) from the town hall to each of the cathedrals. The students did not know trigonometry or analytic geometry.\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_fe999c0fe2ad81fc5164g-061.jpg?height=654&width=1213&top_left_y=455&top_left_x=433)\n\nWhat did they answer their teacher?",
      "answer": "0.002"
    },
    {
      "question": "32nd IMO 1991 shortlist Problem 18 What is the largest power of 1991 which divides 1990 m + 1992 n , where m = 1991 1992 and n = 1991 1990 ?",
      "answer": "1991^{1991}"
    },
    {
      "question": "167. To transport 60 tons of cargo from one place to another, a certain number of trucks were required. Due to the poor condition of the road, each truck had to carry 0.5 tons less than originally planned, which is why 4 additional trucks were required. How many trucks were initially required?",
      "answer": "20"
    },
    {
      "question": "13. (2004 National College Entrance Examination - Jiangsu Paper) Given: $0<\\alpha<\\frac{\\pi}{2}, \\tan \\frac{\\alpha}{2}+\\cot \\frac{\\alpha}{2}=\\frac{5}{2}$. Find the value of $\\sin \\left(\\alpha-\\frac{\\pi}{3}\\right)$.",
      "answer": "\\frac{1}{10}(4-3\\sqrt{3})"
    },
    {
      "question": "7. [4] Five guys are eating hamburgers. Each one puts a top half and a bottom half of a hamburger bun on the grill. When the buns are toasted, each guy randomly takes two pieces of bread off of the grill. What is the probability that each guy gets a top half and a bottom half?",
      "answer": "\\frac{8}{63}"
    },
    {
      "question": "7. Remove the joker cards from a deck of playing cards, and randomly draw 5 cards from the remaining 52 cards. The probability that at least two of the cards have the same number (or letter $J, Q, K, A$) is $\\qquad$ (calculate this probability value, accurate to 0.01).",
      "answer": "0.49"
    },
    {
      "question": "17. An old emperor placed his will in an iron box, which was equipped with several locks, each having several keys. The keys were distributed among his five sons. The old emperor required that after his death, the iron box could be opened only if at least three of his sons were present. How many locks are needed at least? And how should the keys be distributed among his sons?",
      "answer": "10"
    },
    {
      "question": "[ $\\underline{\\text { Projections of the bases, sides, or vertices of a trapezoid ] }}$ [ Pythagorean Theorem (direct and inverse) ]\n\nThe lateral side of an isosceles trapezoid is 41, the height is 40, and the midline is 45. Find the bases.",
      "answer": "3654"
    },
    {
      "question": "7. Let $P(n)$ denote the product of all non-zero digits of the positive integer $n$, for example: $P(123)=1 \\times 2 \\times 3=6$, $P(206)=2 \\times 6=12$. Then $P(1)+P(2)+\\cdots+P(999)=$ $\\qquad$ —.",
      "answer": "97335"
    },
    {
      "question": "István exchanged two car lottery tickets worth $5000 \\mathrm{Ft}$ each. On the 20th day of the lottery quarter, he said to Kálmán: \"Take over one of my books. If we win, we will share the prize as interest, in proportion to our stakes and the time elapsed.\" Kálmán gave him 4000 Ft the next day. - A month later, István borrowed 2500 Ft from László under the same conditions. - At the same time, Kálmán said to Miklós: \"Give me 2000 Ft. I have 4000 Ft in a joint car lottery ticket. If it wins, we will split my share as if you had joined me at the same time.\" Miklós accepted the offer.\n\nIn the quarterly draw, both tickets won. One car was appraised at $60000 \\mathrm{Ft}$, the other at $48000 \\mathrm{Ft}$. During the distribution, István wanted to keep the first car. The others acknowledged that they would share the proceeds from the second car. It was only then that Miklós learned that Kálmán's stake had not been in the bet for the entire quarter, which would make their prize smaller. As compensation, Kálmán agreed to calculate Miklós's share as if his money had been in the bet from the beginning of the quarter.\n\nHowever, after the second car was sold, only $42000 \\mathrm{Ft}$ remained for distribution. How much did each of the four participants receive from this?",
      "answer": "2100"
    },
    {
      "question": "Condition of the problem\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow 0} \\sqrt{\\left(e^{\\sin x}-1\\right) \\cos \\left(\\frac{1}{x}\\right)+4 \\cos x}$",
      "answer": "2"
    },
    {
      "question": "## Task Condition\n\nCalculate the area of the parallelogram constructed on vectors $a$ and $b$.\n\n$a=3 p+2 q$\n\n$b=2 p-q$\n\n$|p|=4$\n\n$|q|=3$\n\n$(\\widehat{p, q})=\\frac{3 \\pi}{4}$",
      "answer": "42\\sqrt{2}"
    },
    {
      "question": "In a quadrilateral, $a: b: c: d=9: 8: 7: 6$. The angle $\\alpha=72^{\\circ} 36^{\\prime}$ is formed by the sides $a$ and $d$, and the area of the quadrilateral is $225 \\mathrm{~cm}^{2}$. What are the lengths of the sides and the angles of the quadrilateral?",
      "answer": "18.567"
    },
    {
      "question": "## SUBJECT I (7p)\n\nConsider the sequence $3,7,11,15,19$,\n\na) Complete the sequence with 3 more terms;\n\nb) Determine the 2014th term;\n\nc) Show that the 2014th term is not a perfect square;\n\nd) Calculate the sum of the first 2014 terms.",
      "answer": "8114406"
    },
    {
      "question": "Example 16 (from the 2nd American Mathematical Invitational Competition) The function $f(x)$ is defined on the real number domain and satisfies the following conditions: for any real number $x$, $f(2+x)=f(2-x)$, and $f(7+x)=f(7-x)$. If $x=0$ is a root of the equation $f(x)=0$, how many roots should the equation $f(x)=0$ have at least in the interval $-1000 \\leqslant x \\leqslant 1000$?",
      "answer": "401"
    },
    {
      "question": "5. Let $A$ and $B$ be two moving points on the ellipse $x^{2}+3 y^{2}=1$, and $O A \\perp O B$ ($O$ is the origin). Then the product of the maximum and minimum values of $|A B|$ is $\\qquad$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "\\frac{2\\sqrt{3}}{3}"
    },
    {
      "question": "1.1. (2 points) From the digits 1, 3, and 5, different three-digit numbers are formed, each of which has all distinct digits. Find the sum of all such three-digit numbers.",
      "answer": "1998"
    },
    {
      "question": "2. When a five-digit number is divided by 100, we get a quotient $k$ and a remainder $o$. For how many five-digit numbers is the sum $k+o$ divisible by 11?",
      "answer": "8181"
    },
    {
      "question": "11.166. The lengths of the edges of the parallelepiped are $a, b$, and $c$. The edges of lengths $a$ and $b$ are mutually perpendicular, and the edge of length $c$ forms an angle of $60^{\\circ}$ with each of them. Determine the volume of the parallelepiped.",
      "answer": "\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "6. (8 points) As shown in the figure, two circles with a radius of 4 cm are placed as shown, and the two shaded areas in the rectangle are equal. Points $A$ and $B$ are the centers of the two circles. Therefore, the length of $AB$ is $\\qquad$ cm. $(\\pi$ is taken as 3 )\n\n\n\nThe translation is provided as requested, maintaining the original formatting and structure.",
      "answer": "6"
    },
    {
      "question": "4. How many nonempty subsets of $\\{1,2, \\ldots, 10\\}$ have the property that the sum of its largest element and its smallest element is 11 ?",
      "answer": "341"
    },
    {
      "question": "3. Find the value: $\\frac{\\sqrt{3}-4 \\sin 20^{\\circ}+8 \\sin ^{3} 20^{\\circ}}{2 \\sin 20^{\\circ} \\sin 480^{\\circ}}$.",
      "answer": "\\frac{2\\sqrt{3}}{3}"
    },
    {
      "question": "Five, let $\\overline{a b c d}$ be a four-digit number, where the first two digits and the last two digits form two-digit numbers $\\overline{a b}$ and $\\overline{c d}$, respectively. If the following conditions are met: $\\overline{a b} \\neq \\overline{c d}$, $\\overline{a b}$ divides $\\overline{c d}$, and $\\overline{c d}$ divides $\\overline{a b c d}$, then such a four-digit number is called a “lucky number”.\n(1) Find the total number of all “lucky numbers”;\n(2) Find the largest odd “lucky number”.",
      "answer": "65"
    },
    {
      "question": "4. Find the probability that a number randomly taken from the interval $[0 ; 5]$ is a solution to the equation $\\sin (x+|x-\\pi|)+2 \\sin ^{2}(x-|x|)=0$.\n\n#",
      "answer": "\\frac{\\pi}{5}"
    },
    {
      "question": "12. B6 (FIN 3) Four distinct circles \\( C, C_{1}, C_{2}, C_{3} \\) and a line \\( L \\) are given in the plane such that \\( C \\) and \\( L \\) are disjoint and each of the circles \\( C_{1}, C_{2}, C_{3} \\) touches the other two, as well as \\( C \\) and \\( L \\). Assuming the radius of \\( C \\) to be 1, determine the distance between its center and \\( L \\).",
      "answer": "7"
    },
    {
      "question": "Find the number of permutations $\\mathrm{a}_{1}, \\mathrm{a}_{2}, \\ldots, \\mathrm{a}_{10}$ of the numbers $1,2, \\ldots, 10$, such that $\\mathrm{a}_{\\mathrm{i}+1}$ is not less than $\\mathrm{a}_{\\mathrm{i}}-1$ (for $\\mathrm{i}=1,2, \\ldots, 9)$.",
      "answer": "512"
    },
    {
      "question": "10.017. Given a point $P$, which is 7 cm away from the center of a circle with a radius of $11 \\mathrm{~cm}$. A chord of length 18 cm is drawn through this point. What are the lengths of the segments into which the chord is divided by point $P$?",
      "answer": "12"
    },
    {
      "question": "Last year, there were 30 more boys than girls in our scout troop. This year, the number of children in the troop increased by $10 \\%$, while the number of boys increased by $5 \\%$ and the number of girls increased by $20 \\%$.\n\nHow many children do we have in the troop this year?\n\n(L. Hozová)",
      "answer": "99"
    },
    {
      "question": "A diagonal of a regular 2006-gon is called odd if its endpoints divide the boundary into two parts, each composed of an odd number of sides. Sides are also regarded as odd diagonals. Suppose the 2006-gon has been dissected into triangles by 2003 nonintersecting diagonals. Find the maximum possible number of isosceles triangles with two odd sides. (Serbia)",
      "answer": "1003"
    },
    {
      "question": "1.1. The Eiffel Tower has a height of 324 m and weighs 10000 tons. How many kilograms will a copy with a height of 1.62 m weigh?",
      "answer": "1.25"
    },
    {
      "question": "2.4 In a closed bag lie apples. Three friends tried to lift the bag and guess how many fruits were inside. The first one said that, in his opinion, there were 18 apples, the second thought there were 21, and the third said 22. When the friends opened the bag, it turned out that one of them was off by 2, another by 3, and the third by 6. How many apples were in the bag? Find all possible answers.",
      "answer": "24"
    },
    {
      "question": "5. Given 9 points in space, where no four points are coplanar, line segments are drawn between every pair of points. These line segments can be colored blue or red, or left uncolored. Find the minimum value of $n$ such that, by arbitrarily coloring any $n$ of these line segments with one of the two colors (red or blue), the set of these $n$ line segments will necessarily contain a triangle with all sides of the same color.",
      "answer": "33"
    },
    {
      "question": "## Task Condition\n\nCalculate the limit of the numerical sequence:\n\n$$\n(6-n)^{2}-(6+n)^{2}\n$$",
      "answer": "-\\frac{12}{7}"
    },
    {
      "question": "(9) If the real numbers $x, \\alpha, \\beta$ satisfy $x=\\log _{3} \\tan \\alpha=-\\log _{3} \\tan \\beta$, and $\\alpha-\\beta=$ $\\frac{\\pi}{6}$, then the value of $x$ is $\\qquad$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "1. Given non-negative real numbers $a_{1}, a_{2}, \\cdots, a_{2022}, b_{1}, b_{2}, \\cdots, b_{2022}, c_{1}, c_{2}, \\cdots, c_{2022}$ satisfy $\\sum_{i=1}^{2022} a_{i}=1$, $\\sum_{i=1}^{2022} b_{i}^{2}=2, \\sum_{i=1}^{2022} c_{i}^{3}=3$. Find the maximum value of $\\sum_{i=1}^{2022} a_{i} b_{i+1} c_{i+2}+\\sum_{i=3}^{2022} a_{i} b_{i-1} c_{i-2}$.",
      "answer": "\\sqrt[3]{12}"
    },
    {
      "question": "10.357. In trapezoid $ABCD$, the lengths of the bases $AD=24$ cm and $BC=8$ cm, and the diagonals $AC=13$ cm, $BD=5\\sqrt{17}$ cm are known. Calculate the area of the trapezoid.",
      "answer": "80"
    },
    {
      "question": "(11) The line $l:(2 m+1) x+(m+1) y-7 m-4=0$ is intersected by the circle $C:(x-1)^{2}+$ $(y-2)^{2}=25$ to form the shortest chord length of $\\qquad$ .",
      "answer": "4\\sqrt{5}"
    },
    {
      "question": "2. Find the eleventh term of the arithmetic progression if the sum of the first seven terms $S_{7}=77$, and the first term $a_{1}=5$.",
      "answer": "25"
    },
    {
      "question": "a) In the figure below, $A D=D C, A E=B D, \\angle A E C=90^{\\circ}$. Determine the value of the angle $\\angle C B D$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4747cb56a5540358921fg-18.jpg?height=463&width=674&top_left_y=2047&top_left_x=588)\nb) In the triangle $\\triangle A B C$ below, $B D$ is the bisector of the angle $\\angle A B C, E$ is the midpoint of $A C$, and $\\angle A D B=90^{\\circ}$. If $A B=12 \\mathrm{~cm}$ and $B C=20 \\mathrm{~cm}$, calculate the length of the segment $D E$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4747cb56a5540358921fg-19.jpg?height=534&width=990&top_left_y=424&top_left_x=630)",
      "answer": "30"
    },
    {
      "question": "## Task 1 - 100811\n\nDetermine the number of all six-digit natural numbers in which the digit sequence 1970 (i.e., the base digits 1, 9, 7, 0 in this order and without any other digits in between) occurs!\n\nWhat is the smallest and what is the largest of these six-digit numbers?",
      "answer": "280"
    },
    {
      "question": "5、Let $a, b, c (c > a, b)$ be the lengths of the three sides of a right triangle, and the point $(m, n)$ lies on the line $a x + b y + c = 0$. Then the minimum value of $m^{2} + n^{2}$ is $\\qquad$ .",
      "answer": "1"
    },
    {
      "question": "10. For the geometric sequence $\\left\\{a_{n}\\right\\}$ with the first term $a_{1}=1536$, common ratio $q=-\\frac{1}{2}$, $\\Pi_{n}$ represents the product of its first $n$ terms $\\left(n \\in \\mathbf{N}^{*}\\right)$, then the value of $n$ that maximizes $\\mathrm{I}_{n}$ is $\\qquad$.",
      "answer": "12"
    },
    {
      "question": "Let's determine the minimum and maximum values of the expression $9-x^{2}-2 \\sqrt{9-x^{2}}$, where $x$ is a real number between -3 and 3.",
      "answer": "-13"
    },
    {
      "question": "## Task B-2.3.\n\nIn online shopping, a package of photocopy paper is 9 kuna cheaper than in a bookstore. In December, the secretary bought 8 more packages online than she bought in November for the same amount of 1440 kn in the bookstore. What is the price of photocopy paper in online shopping?",
      "answer": "36"
    },
    {
      "question": "2. Both rows of the following grid have the same sum. What is the value of *?\n\\begin{tabular}{|r|r|r|r|r|r|r|r|r|r|c|}\n\\hline 1 & 2 & 3 & 4 & 5 & 6 & 7 & 8 & 9 & 10 & 1050 \\\\\n\\hline 11 & 12 & 13 & 14 & 15 & 16 & 17 & 18 & 19 & 20 & $*$ \\\\\n\\hline\n\\end{tabular}",
      "answer": "950"
    },
    {
      "question": "10. (3 points) The greatest common divisor and the least common multiple of two numbers are 3 and 135, respectively. Find the minimum difference between these two numbers is $\\qquad$ .",
      "answer": "12"
    },
    {
      "question": "4. Find the smallest number $n>4$ for which there exists a set of $n$ people such that any two who know each other have no common acquaintances, and any two who do not know each other have exactly two common acquaintances. (Acquaintance is a symmetric relation: if $A$ knows $B, A \\neq B$, then $B$ also knows $A$.)\n\n(Bulgarian)",
      "answer": "16"
    },
    {
      "question": "1. For a finite set $A$, there exists a function $f: \\mathbf{N}_{+} \\rightarrow A$, with the following property: if $i, j \\in \\mathbf{N}_{+}$ and $|i-j|$ is a prime number, then $f(i) \\neq f(j)$. How many elements does the set $A$ have at minimum?",
      "answer": "4"
    },
    {
      "question": "7. Find the largest integer which cannot be expressed as sum of some of the numbers $135,136,137, \\ldots, 144$ (each number may occur many times in the sum or not at all).\n(1 mark)\n某整數不能寫成 135、136、‥144 當中的某些數之和（在和式中每個數出現的次數不限, 亦可以不出現）。求該整數的最大可能值。",
      "answer": "2024"
    },
    {
      "question": "10. To pack books when moving a school library, you can buy small boxes that hold 12 books or large ones that hold 25 books. If all the books are packed in small boxes, 7 books will be left, and if all the books are packed in large boxes, there will be room for 5 more books. The library's collection contains between 500 and 650 books. How many books are in the library?",
      "answer": "595"
    },
    {
      "question": "2. Let $f(x)=\\min (4 x+1, x+2,-2 x+4)$, then the maximum value of $f(x)$ is $\\qquad$",
      "answer": "\\frac{8}{3}"
    },
    {
      "question": "5. (5 points) If 6 is placed at the right end of a two-digit number, the resulting three-digit number is 474 greater than the original two-digit number. The original two-digit number is $\\qquad$ .",
      "answer": "52"
    },
    {
      "question": "4. Given are the numbers $a=-8$ and $b=36$.\n\na) Calculate the value of the expression $||a|-2 \\sqrt{b}|-\\frac{1}{2}|\\sqrt[3]{a}|$.\n\nb) Solve the equation $|x-\\sqrt[3]{a}|=\\sqrt{b}$.",
      "answer": "3"
    },
    {
      "question": "83. What is in the period? Determine what stands in the period of the decimal expansion of the fraction $\\frac{1}{49}$.[^3]",
      "answer": "0.020408163265306122448979591836734693877551"
    },
    {
      "question": "\\section*{Problem 6 - 320936}\n\na) Give three integers \\(x, y\\) and \\(z\\) such that:\n\n\\[\nx^{2}+y^{2}+z^{2}-4 x+12 y-14 z-57=0\n\\]\n\nb) Determine the number of all triples \\((x, y, z)\\) of integers \\(x, y, z\\) that satisfy equation (1)!",
      "answer": "192"
    },
    {
      "question": "ALB\n\n3) If $x^{3}-3 \\sqrt{3} x^{2}+9 x-3 \\sqrt{3}-64=0$, find the value of $x^{6}-8 x^{5}+13 x^{4}-5 x^{3}+49 x^{2}-137 x+2015$.",
      "answer": "1898"
    },
    {
      "question": "Consider the positive integer\n\n$$\nn=2^{r}-16^{s}\n$$\n\nwhere $r$ and $s$ are also positive integers. Find the conditions that $r$ and $s$ must satisfy for the remainder of the division of $n$ by 7 to be 5. Find the smallest number that satisfies this condition.\n\n## Solution.",
      "answer": "768"
    },
    {
      "question": "2. Find the smallest prime number $p$ for which the number $p^{3}+2 p^{2}+p$ has exactly 42 positive divisors.",
      "answer": "23"
    },
    {
      "question": "3. Let $v(X)$ be the sum of elements of a nonempty finite set $X$, where $X$ is a set of numbers. Calculate the sum of all numbers $v(X)$ where $X$ ranges over all nonempty subsets of the set $\\{1,2,3, \\ldots, 16\\}$.",
      "answer": "4456448"
    },
    {
      "question": "3. From points $A$ and $B$, two cars set off towards each other simultaneously with constant speeds. One hour before the first car arrived at $B$ and four hours before the second car arrived at $A$, they met. Find the ratio of the speeds of the cars.",
      "answer": "2"
    },
    {
      "question": "## Task 2 - 340812\n\nA three-digit natural number is called \"symmetric\" if and only if its hundreds digit is equal to its units digit.\n\na) Form all those three-digit symmetric numbers in which only the digits $0,1,2$ appear (each possibly multiple times or not at all)! No justification is required.\n\nb) Form all those three-digit symmetric numbers that are divisible by 6 and in which only the digits 2, 5, 8 appear! Prove that exactly the numbers you specify meet the given conditions!\n\nc) Determine the number of all three-digit symmetric numbers!\n\nd) Determine the sum of all three-digit symmetric numbers!",
      "answer": "49500"
    },
    {
      "question": "5. Solution. First, let's prove that building towers taller than 2 is meaningless.\n\nIndeed, suppose one of the towers has a height of \\( h > 2 \\). Let's remove the top floor from it, turning it into a new tower of height 1. How will this change the inspector's sum? The shortened tower no longer sees towers of height \\( h-1 \\). However, since \\( h > 2 \\), from all these towers (including the shortened one), the new tower is now visible. Therefore, the total sum has increased.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_4ca17689025243fab126g-18.jpg?height=319&width=616&top_left_y=1354&top_left_x=465)\n\nFig. 15\n\nIt remains to study the case where all towers have a height of 1 or 2. If there are \\( X \\) towers of height 1 and \\( Y \\) towers of height 2 (by the condition \\( X + 2Y = 30 \\)), then the inspector's sum is\n\n\\[\nX Y = (30 - 2Y) Y = 2Y(15 - Y) = 2\\left(7.5^2 - (Y - 7.5)^2\\right)\n\\]\n\nFor integer \\( Y \\), this expression takes its maximum value at \\( Y = 7 \\) and \\( Y = 8 \\), and equals 112.",
      "answer": "112"
    },
    {
      "question": "## Task B-4.4.\n\nHow many passwords consisting of 4 characters from the set $\\{a, b, c, 1,2,3,4,5,+,!\\}$ are there such that all characters in the password are different or the password contains exactly two, not necessarily different letters and two, not necessarily different digits?",
      "answer": "5670"
    },
    {
      "question": "4. (7 points) The numbers $a, b, c, d$ belong to the interval $[-10.5,10.5]$. Find the maximum value of the expression $a+2 b+c+2 d-a b-b c-c d-d a$.",
      "answer": "462"
    },
    {
      "question": "Circle inscribed in an angle  Area of a trapezoid\n\nA circle with a radius of 3 is inscribed in a right trapezoid, the smaller base of which is 4. Find the larger base of the trapezoid.",
      "answer": "12"
    },
    {
      "question": "1. (5 points) Find the degree measure of the angle\n\n$$\n\\delta=\\arccos \\left(\\left(\\sin 2195^{\\circ}+\\sin 2196^{\\circ}+\\cdots+\\sin 5795^{\\circ}\\right)^{\\cos } 2160^{\\circ}+\\cos 2161^{\\circ}+\\cdots+\\cos 5760^{\\circ}\\right)\n$$",
      "answer": "55"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n$\\lim _{x \\rightarrow 0}\\left(\\frac{2+x}{3-x}\\right)^{x}$",
      "answer": "1"
    },
    {
      "question": "3. Given is the quadratic equation in $x$\n\n$$\n4(p-2)^{2} x^{2}-4(p-2) q x+q(4 p-3 q+4)-(p-1)^{2}=0, p, q \\in \\mathbb{N}\n$$\n\nWhat condition must $p$ and $q$ satisfy for the equation to have a double root? Find at least one double root that is a natural number.",
      "answer": "1"
    },
    {
      "question": "Example 6.25. A clerk accepts on average two applications per hour. It is assumed that the flow of applications is a simple one. What is the probability of receiving four applications in four hours?",
      "answer": "0.9576"
    },
    {
      "question": "2. (10 points) Calculate:\n$$\n10 \\times 9 \\times 8 + 7 \\times 6 \\times 5 + 6 \\times 5 \\times 4 + 3 \\times 2 \\times 1 - 9 \\times 8 \\times 7 - 8 \\times 7 \\times 6 - 5 \\times 4 \\times 3 - 4 \\times 3 \\times 2\n$$",
      "answer": "132"
    },
    {
      "question": "1. Class A and Class B have a total of 83 people, Class B and Class C have a total of 86 people, Class C and Class D have a total of 88 people. How many people are there in Class A and Class D?",
      "answer": "85"
    },
    {
      "question": "8. In the rectangular prism $A B C D-A_{1} B_{1} C_{1} D_{1}$, $A B=$ $A D=5, A A_{1}=1, M$ is a point on edge $A B$ and satisfies $3 \\overrightarrow{A M}+2 \\overrightarrow{B M}=\\mathbf{0}$. A point $P$ is randomly selected within the square base $A B C D$, and the distance from point $P$ to line $A_{1} D_{1}$ is $d_{1}, P M=d_{2}$. Then the probability that the point $P$ satisfies $d_{1}^{2}-d_{2}^{2} \\geqslant 1$ is $\\qquad$",
      "answer": "\\frac{32}{75}"
    },
    {
      "question": "12. What is the value of $\\sqrt{20212020 \\times 20202021-20212021 \\times 20202020}$ ?",
      "answer": "100"
    },
    {
      "question": "50*. What is the greatest number of circles of radius 1 that can be placed on a plane so that they all intersect a certain fixed unit circle $S$ and no one of them contains the center of $S$ or the center of another circle inside it?",
      "answer": "18"
    },
    {
      "question": "1. Given two quadratic trinomials $f(x)$ and $g(x)$ with leading coefficients of 1, each having two real roots, and $f(1)=g(2)$, $g(1)=f(2)$. Find the sum of the four roots of these two quadratic trinomials.",
      "answer": "6"
    },
    {
      "question": "Example 19 (2002 National High School Competition Question) As shown in Figure 5-19, there is a sequence of curves $P_{0}, P_{1}, P_{2}, \\cdots$. It is known that the figure enclosed by $P_{0}$ is an equilateral triangle with an area of 1. $P_{k+1}$ is obtained by performing the following operation on $P_{k}$: trisect each side of $P_{k}$, construct an equilateral triangle outward using the middle segment of each side, and then remove the middle segment $(k=0,1,2, \\cdots)$. Let $S_{n}$ be the area of the figure enclosed by the curve $P_{n}$.\n(1) Find the general term formula for the sequence $\\left\\{S_{n}\\right\\}$;\n(2) Find $\\lim _{n \\rightarrow \\infty} S_{n}$.",
      "answer": "\\frac{8}{5}"
    },
    {
      "question": "## Task Condition\n\nFind the angle between the planes:\n\n$x+4 y-z+1=0$\n\n$2 x+y+4 z-3=0$",
      "answer": "845^{\\}4^{\\\\}"
    },
    {
      "question": "Water can be let into a basin through four different taps. If the first and second taps are opened simultaneously, the basin fills up in 2 hours. If the second and third taps are opened at the same time, it takes 3 hours, and if the third and fourth taps are opened simultaneously, it takes 4 hours for the basin to fill up. How long will it take to fill the basin if water flows in through the fourth and first taps at the same time?",
      "answer": "2"
    },
    {
      "question": "8. Let set $S$ consist of all positive integers not exceeding $2^{40}$, and whose binary representation contains exactly two 1s. If a number is randomly selected from $S$, the probability that this number is divisible by 9 is $\\qquad$ .",
      "answer": "\\frac{133}{780}"
    },
    {
      "question": "2. Calculate the value of the expression\n\n$$\n\\left(\\frac{-1+i \\sqrt{3}}{2}\\right)^{1986}+\\left(\\frac{-1-i \\sqrt{3}}{2}\\right)^{1986}\n$$\n\ni is the imaginary unit.",
      "answer": "2"
    },
    {
      "question": "A $180 \\mathrm{~cm}$ high, horizontally based, cylindrical fuel tank is equipped with a tap at the bottom and a filling nozzle at the top. The latter can fill the tank in 1 hour. If we open both the filling nozzle and the tap for 5 minutes, the fuel level in the tank rises by $10 \\mathrm{~cm}$. How long will it take to empty a full tank through the tap?",
      "answer": "3"
    },
    {
      "question": "251. Packing Cylinders. Forty cylinders with a diameter of 1 cm and the same height were tightly packed in a box in 5 rows of 8 cylinders each so that they would not \"rattle\" during transportation. How many cylinders need to be removed from the box so that, by moving the remaining cylinders and adding the removed cylinders and one more cylinder at the end, 41 cylinders of the same size can be packed in this box? Will the cylinders \"rattle\" in this case?\n\n![](https://cdn.mathpix.com/cropped/2024_05_21_2b40d984baf9da275c68g-054.jpg?height=325&width=512&top_left_y=1448&top_left_x=772)",
      "answer": "2"
    },
    {
      "question": "13th CanMO 1981 Problem 2 The circle C has radius 1 and touches the line L at P. The point X lies on C and Y is the foot of the perpendicular from X to L. Find the maximum possible value of area PXY (as X varies). Solution",
      "answer": "\\frac{3\\sqrt{3}}{8}"
    },
    {
      "question": "15. Find the sum:\n\n$$\n1+2-3-4+5+6-7-8+\\ldots+301+302\n$$",
      "answer": "303"
    },
    {
      "question": "(not treated in class) Determine the minimum of $x^{2}+y^{2}+z^{2}$ under the condition $x^{3}+y^{3}+z^{3}-3 x y z=1$.",
      "answer": "1"
    },
    {
      "question": "18. Prince Gvidon had three sons. Among his descendants, 93 each had two sons and no daughters, while all the others died childless. How many descendants did Prince Gvidon have in total?",
      "answer": "189"
    },
    {
      "question": "## Problem Statement\n\nCalculate the limit of the function:\n\n$\\lim _{x \\rightarrow \\frac{\\pi}{2}} \\frac{e^{\\tan 2 x}-e^{-\\sin 2 x}}{\\sin x-1}$",
      "answer": "0"
    },
    {
      "question": "Task B-4.2. For what values of the real number $x$ in the expansion of $\\left(\\sqrt{5^{x}}+\\frac{1}{\\sqrt[3]{25^{x}}}\\right)^{6}$ is the third term equal to 75?",
      "answer": "-\\frac{3}{2}"
    },
    {
      "question": "78. The smallest natural number whose digits sum to 218 is an $n$-digit number, then $n=$",
      "answer": "25"
    },
    {
      "question": "B4. Calculate the integer solution of the equation $\\log _{2}\\left(x^{2}+7\\right)=\\frac{\\log _{2}(x+1)}{\\log _{8} 2}-\\log _{2} x$. (7 points)\n\nSpace for solving problems in set $B$.\n\n## dIFA\n\n12th Mathematics Knowledge Competition for Students of Secondary Technical and Vocational Schools\n\nRegional Competition, March 28, 2012\n\n## PROBLEMS FOR THE FOURTH YEAR\n\nTime for solving: 90 minutes. In set A, we will evaluate a correct answer with two points, while we will deduct half a point for an incorrect answer. Write the answers for set A in the left table.\n![](https://cdn.mathpix.com/cropped/2024_06_07_d6b556bd14f8001ca6fag-08.jpg?height=172&width=1210&top_left_y=658&top_left_x=433)",
      "answer": "1"
    },
    {
      "question": "4. 42 Find the positive root of the equation $x=\\sqrt{2+\\sqrt{2+\\sqrt{2+\\sqrt{2+x}}}}$, and prove that there is only one positive root.",
      "answer": "2"
    },
    {
      "question": "5. In the number $2 * 0 * 1 * 6 * 0 *$, each of the 5 asterisks needs to be replaced with any of the digits $0,1,2,3,4,5,6,7,8$ (digits can repeat) so that the resulting 10-digit number is divisible by 18. In how many ways can this be done?",
      "answer": "3645"
    },
    {
      "question": "11. (10 points) Several people completed the task of planting 2013 trees, with each person planting the same number of trees. If 5 people did not participate in planting, the rest would need to plant 2 more trees each to fail to complete the task, while planting 3 more trees each would exceed the task. How many people participated in planting?",
      "answer": "61"
    },
    {
      "question": "3. A road 1 km long is fully illuminated by street lamps, each of which illuminates a section of the road 1 m long. What is the maximum number of street lamps that can be on the road, given that after turning off any street lamp, the road will no longer be fully illuminated?",
      "answer": "1998"
    },
    {
      "question": "6.164. $\\sqrt{x+8+2 \\sqrt{x+7}}+\\sqrt{x+1-\\sqrt{x+7}}=4$.",
      "answer": "2"
    },
    {
      "question": "10. (20 points) Given a line $l$ intersects the ellipse $C: \\frac{x^{2}}{2}+y^{2}=1$ at points $A, B$, and $O$ is the origin.\n(1) Find the maximum value of the area of $\\triangle A O B$, and the equation of the line $l$ at this time;\n(2) Let the set $L=\\{l \\mid$ line $l$ makes the area of $\\triangle A O B$ take the maximum value $\\}, l_{1}, l_{2}, l_{3}, l_{4} \\in L$ and satisfy $l_{1} / / l_{2}, l_{3} / / l_{4}$, $k_{1}, k_{2}, k_{3}, k_{4}$ are the slopes of $l_{1}, l_{2}, l_{3}, l_{4}$ respectively and satisfy $k_{1}+k_{2}+k_{3}+k_{4}=0$, find the minimum value of the area of the quadrilateral formed by these four lines.",
      "answer": "2\\sqrt{2}"
    },
    {
      "question": "1. Let $A B C$ be a triangle such that $A B=7$, and let the angle bisector of $\\angle B A C$ intersect line $B C$ at $D$. If there exist points $E$ and $F$ on sides $A C$ and $B C$, respectively, such that lines $A D$ and $E F$ are parallel and divide triangle $A B C$ into three parts of equal area, determine the number of possible integral values for $B C$.",
      "answer": "13"
    },
    {
      "question": "4. Consider the adjacent complementary angles $\\square A O B$ and $\\square B O C$, points $D, E \\in \\operatorname{Int}(\\square A O B)$, such that $\\square A O D \\equiv \\square D O E \\equiv \\square E O B$, and (OF is the bisector of angle $\\square B O C$. Given that $\\mathrm{m}(\\square E O F)=39^{\\circ}$, determine:\n\n(4p) a) $\\mathrm{m}(\\square A O B)$ and $\\mathrm{m}(\\square B O C)$.\n\n(3p) b) m( $\\square D O M$ ), where ( $O M$ is the bisector of angle $\\square E^{\\prime} O C$, and ( $O E$ and ( $O E^{\\prime}$ are opposite rays.\n\nAdina Oancea\n\nNote: All subjects are mandatory.\n\nEffective working time: 2 hours.",
      "answer": "135"
    },
    {
      "question": "Three. (50 points) Nine small balls numbered $1,2, \\cdots, 9$ are randomly placed on nine equally spaced points on a circle, with one ball on each point. Let $S$ be the sum of the absolute values of the differences between the numbers of all adjacent balls on the circle. Find the probability of the arrangement that minimizes $S$.",
      "answer": "\\frac{1}{315}"
    },
    {
      "question": "Problem 3. If $x y=6$ and $x^{2} y+x y^{2}+x+y=63$, then what is $x^{2}+y^{2}$?",
      "answer": "69"
    },
    {
      "question": "5. (40th IMO Problem) $n$ is a given integer, $n \\geqslant 2$, determine the smallest constant $c$, such that the inequality $\\sum_{1 \\leq i \\leq j \\leq n} x_{i} x_{j}\\left(x_{i}^{2}+x_{j}^{2}\\right) \\leqslant c\\left(\\sum_{i=1}^{n} x_{i}\\right)^{4}$ holds for all non-negative real numbers $x_{1}, x_{2}, \\cdots, x_{n}$.",
      "answer": "\\frac{1}{8}"
    },
    {
      "question": "2、The following is a calendar for May of a certain year. Using a $2 \\times 2$ box that can frame four numbers (excluding Chinese characters), the number of different ways to frame four numbers is $\\qquad$.\n\n---\n\nNote: The original text includes a placeholder for the answer (indicated by $\\qquad$). This has been preserved in the translation.",
      "answer": "20"
    },
    {
      "question": "27. If the team had scored 97 points in the last sport, the average number of points scored by it in one sport would have been 90; if it had scored 73 points in this last sport, the average number of points per sport would have been 87. In how many sports did the team compete?",
      "answer": "8"
    },
    {
      "question": "Problem 4.3. How many rectangles exist on this picture with sides running along the grid lines? (A square is also a rectangle.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_038330058549e9b1ba1cg-06.jpg?height=173&width=206&top_left_y=614&top_left_x=624)",
      "answer": "24"
    },
    {
      "question": "36. [20] In the game of Connect Four, there are seven vertical columns which have spaces for six tokens. These form a $7 \\times 6$ grid of spaces. Two players White and Black move alternately. A player takes a turn by picking a column which is not already full and dropping a token of their color into the lowest unoccupied space in that column. The game ends when there are four consecutive tokens of the same color in a line, either horizontally, vertically, or diagonally. The player who has four tokens in a row of their color wins.\nAssume two players play this game randomly. Each player, on their turn, picks a random column which is not full and drops a token of their color into that column. This happens until one player wins or all of the columns are filled. Let $P$ be the probability that all of the columns are filled without any player obtaining four tokens in a row of their color. Estimate $P$.\nAn estimate of $E>0$ earns $\\lfloor 20 \\min (P / E, E / P)\\rfloor$ points.",
      "answer": "0.0025632817"
    },
    {
      "question": "5.1. Let $S(n)$ be the sum of the digits in the decimal representation of the number $n$. Find $S\\left(S\\left(S\\left(S\\left(2017^{2017}\\right)\\right)\\right)\\right)$. Answer. 1.",
      "answer": "1"
    },
    {
      "question": "Anzhans A.\n\nThere are 100 silver coins, ordered by weight, and 101 gold coins, also ordered by weight. It is known that all coins have different weights. We have a two-pan balance that allows us to determine which of any two coins is heavier. How can we find the coin that ranks 101st in weight among all the coins with the fewest number of weighings?",
      "answer": "8"
    },
    {
      "question": "Let $\\left(a_{1}, a_{2}, \\ldots, a_{100}\\right)$ be a permutation of $(1,2, \\ldots, 100)$.\n\nWhat is the maximum number of perfect squares that can occur among the numbers $a_{1}, a_{1}+a_{2}, a_{1}+a_{2}+a_{3}, \\ldots, a_{1}+\\ldots+a_{100} ?$",
      "answer": "60"
    },
    {
      "question": "5.6. Among all the irreducible fractions, where the numerator and denominator are two-digit numbers, find the smallest fraction greater than $\\frac{4}{9}$. In your answer, specify its numerator.",
      "answer": "41"
    },
    {
      "question": "When Bořek was painting the garage door, he accidentally painted over the scale of the outdoor wall thermometer. However, the mercury tube remained undamaged, so Bořek covered the original scale with a homemade strip. On it, he carefully marked divisions, all of the same size and labeled with numbers. His division, however, had a different size than the original division, which represented one degree Celsius, and he also placed the zero differently than where $0{ }^{\\circ} \\mathrm{C}$ was. Thus, Bořek began measuring temperature in his own units: borcs. When the thermometer should have indicated a temperature of $11^{\\circ} \\mathrm{C}$, it showed 2 borcs. When it should have indicated $-4{ }^{\\circ} \\mathrm{C}$, it showed -8 borcs. What is the temperature in degrees Celsius if Bořek sees a temperature of -2 borcs on his thermometer?\n\n(L. Simünek)",
      "answer": "5"
    },
    {
      "question": "The Law of Cosines\n[Pythagorean Theorem (direct and inverse).]\n\nIn a right triangle \\(ABC\\), the leg \\(AC = 15\\) and the leg \\(BC = 20\\). On the hypotenuse \\(AB\\), a segment \\(AD\\) equal to 4 is laid out, and point \\(D\\) is connected to \\(C\\). Find \\(CD\\).",
      "answer": "13"
    },
    {
      "question": "6.24 In a geometric sequence with a common ratio greater than 1, what is the maximum number of terms that are integers between 100 and 1000.\n(4th Canadian Mathematics Competition, 1972)",
      "answer": "6"
    },
    {
      "question": "Consider 100 points in the plane, such that no three are collinear. We draw all the lines passing through two of these points, and we assume that if $A, B, C, D$ are four of these points, then $(A B)$ and $(C D)$ are not parallel, and their intersection point does not lie on another line. How many regions of the plane are thus formed by these lines?",
      "answer": "11778426"
    },
    {
      "question": "543. a) How can a right triangle be divided into two similar triangles? b) In how many ways can this be done?",
      "answer": "1"
    },
    {
      "question": "13. (2005 National High School Mathematics League Additional Question) Let positive numbers $a$, $b$, $c$, $x$, $y$, $z$ satisfy $c y + b z = a$; $a z + c x = b$; $b x + a y = c$. Find the minimum value of the function $f(x, y, z) = \\frac{x^2}{1 + x} + \\frac{y^2}{1 + y} + \\frac{z^2}{1 + z}$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "A dart is thrown at a square dartboard of side length 2 so that it hits completely randomly. What is the probability that it hits closer to the center than any corner, but within a distance 1 of a corner?",
      "answer": "\\frac{\\pi-2}{4}"
    },
    {
      "question": "14. As shown in the figure, four rectangles, Jia, Yi, Bing, and Ding, are pieced together to form square $E F G H$, with the middle shaded area being a square. It is known that the sum of the areas of rectangles Jia, Yi, Bing, and Ding is $32 \\mathrm{~cm}^{2}$, and the area of quadrilateral $A B C D$ is $20 \\mathrm{~cm}^{2}$. Find the total perimeter of rectangles Jia, Yi, Bing, and Ding in centimeters.",
      "answer": "48"
    },
    {
      "question": "7. If (1) $a, b, c, d$ all belong to $\\{1,2,3,4\\}$; (2) $a \\neq b, b \\neq c, c \\neq d, d \\neq a$; (3) $a$ is the smallest value among $a, b, c, d$, then the number of different four-digit numbers $\\overline{a b c d}$ that can be formed is $\\qquad$ .",
      "answer": "28"
    },
    {
      "question": "14. Gauss and his classmates were punished at school for being mischievous, and the teacher assigned a math problem:\nFrom the natural numbers 1 to 100, remove two consecutive odd numbers, the average of the remaining numbers is 51, what is the product of the two removed odd numbers?\nWhoever answers correctly can go home. Gauss once again quickly gave the correct answer. Gauss's answer is $\\qquad$.",
      "answer": "675"
    },
    {
      "question": "45. Given a 129-gon. Petya and Vasya are playing a game. They take turns marking the vertices of this polygon, with Petya making the first move. Petya can mark any unmarked vertex on each of his turns. Vasya can mark any unmarked vertex that is two positions away from the vertex Petya marked on his last turn. The game ends when Vasya cannot make his move. What is the maximum number of moves Vasya can make regardless of Petya's play?",
      "answer": "7"
    },
    {
      "question": "6. A circle of radius $t$ is tangent to the hypotenuse, the incircle, and one leg of an isosceles right triangle with inradius $r=1+\\sin \\frac{\\pi}{8}$. Find $r t$.",
      "answer": "\\frac{2+\\sqrt{2}}{4}"
    },
    {
      "question": "6. Solve the equation $\\sqrt{\\frac{x^{2}-16}{x-3}}+\\sqrt{x+3}=\\frac{7}{\\sqrt{x-3}}$.",
      "answer": "5"
    },
    {
      "question": "In the sequence 1, 3, 2, .. each term after the first two is equal to the preceding term, subtracted from the term that precedes it, that is, if $n>2$, then $a_{n}=a_{n-1}-a_{n-2}$. What is the sum of the first hundred terms of this sequence?",
      "answer": "5"
    },
    {
      "question": "Example 7 A positive integer $n$ cannot be divisible by 2 or 3, and there do not exist non-negative integers $a, b$ such that $\\left|2^{a}-3^{b}\\right|=n$. Find the minimum value of $n$.\n(2003 China Training Team Test)",
      "answer": "35"
    },
    {
      "question": "Question 4. Calculate\n$$\n\\log _{2}\\left(\\prod_{a=0}^{2018} \\prod_{b=0}^{2018}\\left(1+e^{\\frac{2 \\pi n k b}{2017}}\\right)\\right)\n$$",
      "answer": "6725"
    },
    {
      "question": "3. As shown in the figure, in trapezoid $A B C D$ with an area of 112, $A E: E F: F B=1: 3: 3, A D: B C=2$ : 3, then the area of triangle $C D H$ is $\\qquad$.",
      "answer": "36"
    },
    {
      "question": "5. In the country of Betia, there are 125 cities, some of which are connected by express trains that do not stop at intermediate stations. It is known that any four cities can be toured in a circle in some order. What is the minimum number of pairs of cities connected by express trains?",
      "answer": "7688"
    },
    {
      "question": "B1. Arthur is driving to David's house intending to arrive at a certain time. If he drives at 60 $\\mathrm{km} / \\mathrm{h}$, he will arrive 5 minutes late. If he drives at $90 \\mathrm{~km} / \\mathrm{h}$, he will arrive 5 minutes early. If he drives at $n \\mathrm{~km} / \\mathrm{h}$, he will arrive exactly on time. What is the value of $n$ ?",
      "answer": "72"
    },
    {
      "question": "1. Rectangle A has length $6 \\mathrm{~cm}$ and area $36 \\mathrm{~cm}^{2}$.\nRectangle B has length $12 \\mathrm{~cm}$ and area $36 \\mathrm{~cm}^{2}$.\nRectangle $\\mathrm{C}$ has length $9 \\mathrm{~cm}$ and area $36 \\mathrm{~cm}^{2}$.\nThe rectangle with the smallest width has a width of $x \\mathrm{~cm}$. What is the value of $x$ ?",
      "answer": "3"
    },
    {
      "question": "M . B.\n\nOn some cells of a $10 \\times 10$ board, there is a flea. Once a minute, the fleas jump simultaneously, and each one jumps to an adjacent cell (along a side). A flea jumps strictly in one of the four directions parallel to the sides of the board, maintains its direction as long as possible, and changes it to the opposite direction if it cannot continue. Dog Barboss observed the fleas for an hour and never saw two of them sitting on the same cell. What is the maximum number of fleas that could have been jumping on the board?",
      "answer": "40"
    },
    {
      "question": "9.3. The number 2019 is represented as the sum of different odd natural numbers. What is the maximum possible number of addends?",
      "answer": "43"
    },
    {
      "question": "A pedestrian set out from point $A$ to point $B$. At the same time, a cyclist set out from $B$ to $A$. After one hour, the pedestrian found himself exactly halfway between point $A$ and the cyclist. Another 15 minutes later, they met, and each continued on their way.\n\nHow much time did the pedestrian spend on the journey from $A$ to $B$? (The speeds of the pedestrian and the cyclist are constant.)\n\n#",
      "answer": "5"
    },
    {
      "question": "## - Problem 1\n\nA pucelana sequence is a strictly increasing sequence of sixteen consecutive positive odd numbers, whose sum is a perfect cube. How many pucelana sequences consist only of three-digit numbers?",
      "answer": "3"
    },
    {
      "question": "7.7. The distances from the midpoint of the height of a regular triangular pyramid to the lateral face and to the lateral edge are 2 and $\\sqrt{13}$, respectively. Find the volume of the pyramid. If necessary, round your answer to two decimal places.",
      "answer": "432.99"
    },
    {
      "question": "On a table are 10 coins, all with \"heads\" facing up. A move consists of flipping exactly 4 coins.\n\na) What is the minimum number of moves required for all to be \"tails\" facing up?\n\nb) If there were 11 coins, would it be possible to have all of them with tails facing up?",
      "answer": "3"
    },
    {
      "question": "6. Given $A=\\{1,-1, \\mathrm{i},-\\mathrm{i}\\}$ (i is the imaginary unit), $f(x)$ is a function with domain and range both being $A$, for any $x, y \\in A$, there is $f(x y)=f(x) f(y)$. Then the number of functions $f(x)$ that satisfy the condition is $\\qquad$.",
      "answer": "2"
    },
    {
      "question": "7.1. Does there exist a ten-digit number, divisible by 11, in which all digits from 0 to 9 appear?",
      "answer": "9576843210"
    },
    {
      "question": "1. Given $a>0, b>0$, and $\\log _{9} a=\\log _{12} b=\\log _{16}(a+b)$. Then the value of $\\frac{b}{a}$ is $\\qquad$ .",
      "answer": "\\frac{1+\\sqrt{5}}{2}"
    },
    {
      "question": "4. (7 points) The numbers $a, b, c, d$ belong to the interval $[-5.5,5.5]$. Find the maximum value of the expression $a+2 b+c+2 d-a b-b c-c d-d a$.",
      "answer": "132"
    },
    {
      "question": "Find the height of a triangular pyramid, the lateral edges of which are pairwise perpendicular and equal to 2, 3, and 4.\n\n#",
      "answer": "\\frac{12}{\\sqrt{61}}"
    },
    {
      "question": "10.366 Two circles touch each other externally at point $A$. Find the radii of the circles if the chords connecting point $A$ with the points of tangency of one of their common external tangents are 6 and 8 cm.",
      "answer": "\\frac{15}{4}"
    },
    {
      "question": "9. The equation $9 x^{3}-3 x^{2}-3 x-1=0$ has a real root of the form $\\frac{\\sqrt[3]{a}+\\sqrt[3]{b}+1}{c}$, where $a, b, c$ are positive integers. Find the value of $a+b+c$.\n(1 mark)\n方程 $9 x^{3}-3 x^{2}-3 x-1=0$ 有一個形如 $\\frac{\\sqrt[3]{a}+\\sqrt[3]{b}+1}{c}$ （其中 $a \\cdot b \\cdot c$ 為正整數）的實根。求 $a+b+c$ 的值。\n(1 分)",
      "answer": "119"
    },
    {
      "question": "4. In triangle $\\mathrm{ABC}$ with sides $a, b$ and $c$, it is known that $b=50 \\mathrm{~cm}, c=48 \\mathrm{~cm}$ and the internal angle at vertex A is twice the internal angle at vertex B. Determine the side $a$.",
      "answer": "70"
    },
    {
      "question": "9. In order to effectively improve the \"sandstorm\" phenomenon, a region attaches great importance to environmental protection, and the area of green vegetation is on the rise. According to the survey, from 1986 to 1995 over 10 years, it increased by $2 \\%$ every two years, and in 1994 and 1995, 8.15 million $\\mathrm{m}^{2}$ of vegetation was planted. The local government has decided to continue developing at this rate for the next 10 years. Therefore, from 1996 to 1999, the total area of green vegetation planted will be _____ million $\\mathrm{m}^{2}$ (accurate to the nearest million $\\mathrm{m}^{2}$).",
      "answer": "1679"
    },
    {
      "question": "1. Given is $\\triangle A B C$, over whose side $A C$ a circle $k$ is constructed as if over a diameter. The circle $k$ passes through the midpoint of side $B C$, and intersects side $A B$ at point $D$ such that $A D=\\frac{3}{2} D B$. If $A C=60$, calculate the area of $\\triangle A B C$.",
      "answer": "1440"
    },
    {
      "question": "Some people are playing paintball against each other. In a given situation, the distances between them are all different. In this case, everyone shoots at the person closest to them. What is the maximum number of people who can shoot at the same person?",
      "answer": "5"
    },
    {
      "question": "4. In the coordinate plane, there are 100 points. Let $N$ be the number of triples of points $(A, B, C)$ that satisfy the following conditions: the vertices are chosen from these 100 points, the y-coordinates of $A$ and $B$ are the same, and the x-coordinates of $B$ and $C$ are the same. Find the maximum value of $N$.",
      "answer": "8100"
    },
    {
      "question": "12. Following the order of King Pea, General Mushtralkin tried to arrange all the soldiers in rows first by 2, and then by $3,4,5,6,7,8,9,10$, but to his surprise, each time the last row was incomplete, as there were respectively\n\n$$\n1,2,3,4,5,6,7,8,9\n$$\n\nsoldiers left. What is the smallest number of soldiers there could have been?",
      "answer": "2519"
    },
    {
      "question": "B1. Find all real numbers $a \\geq 0$ for which the equation $2|x-a|+3|x+a|=1$ has at least one real solution.",
      "answer": "\\frac{1}{6}"
    },
    {
      "question": "191*. The sequence of natural numbers $a_{0}, a_{1}$, $a_{2}, a_{3}, \\ldots$ is formed according to the following rule:\n\n$$\na_{0}, a_{1}, a_{2}=\\left|a_{0}-a_{1}\\right|, a_{3}=\\left|a_{1}-a_{2}\\right|, \\ldots\n$$\n\n(and generally $a_{n}=\\left|a_{n-2}-a_{n-1}\\right|$ for all $n \\geqslant 2$); the sequence continues until the first zero. It is known that each number in the sequence does not exceed 1967. What is the maximum number of terms that such a sequence can contain?",
      "answer": "1967"
    },
    {
      "question": "11.005. The plane angle at the vertex of a regular triangular pyramid is $90^{\\circ}$. Find the ratio of the lateral surface area of the pyramid to the area of its base.",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "12.022. Find the angle of a triangle if it is known that the sides enclosing this angle are 1 and 3, and the bisector of the angle is \\(0.75 \\sqrt{3}\\).",
      "answer": "60"
    },
    {
      "question": "14. New Year's Problem (recommended for 8th grade, 4 points). On the New Year's table, there are 4 glasses in a row: the first and third are filled with orange juice, while the second and fourth are empty. While waiting for guests, Vanya absent-mindedly and randomly pours the juice from one glass to another. In one move, she can take a full glass and pour all its contents into one of the two empty glasses.\n\nFind the expected number of pourings after which, for the first time, everything will be the opposite: the first and third glasses will be empty, and the second and fourth will be full.",
      "answer": "6"
    },
    {
      "question": "10. Let $\\triangle A B C$ be equilateral, and let $D, E, F$ be points on sides $B C, C A, A B$ respectively, with $F A=9, A E=E C=6, C D=4$. Determine the measure (in degrees) of $\\angle D E F$.",
      "answer": "60"
    },
    {
      "question": "4. The three sides of a triangle can be expressed as $m^{2}+m+1, 2m+1, m^{2}-1$, then the largest angle of this triangle is . $\\qquad$",
      "answer": "\\frac{2\\pi}{3}"
    },
    {
      "question": "(11) Let $0<x<1,0<y<1$, and $(1-x y)^{2}=2(1-x)(1-y)$. Then the maximum value of the function $f(x, y)=\\frac{1}{2} x y(1-x y)$ is $\\qquad$ .",
      "answer": "5\\sqrt{2}-7"
    },
    {
      "question": "Exercise 6. Find the largest integer $n \\geqslant 3$ for which there exists a set $\\mathcal{S}$ of $n$ points in the plane with the following property: every triangle (even degenerate) whose vertices belong to $\\mathcal{S}$ is isosceles but not equilateral.",
      "answer": "6"
    },
    {
      "question": "4. A box contains two white socks, three blue socks and four grey socks. Rachel knows that three of the socks have holes in, but does not know what colour these socks are. She takes one sock at a time from the box without looking. How many socks must she take for her to be certain she has a pair of socks of the same colour without holes?",
      "answer": "7"
    },
    {
      "question": "Example 5 Find the maximum constant $k$, such that $\\frac{k a b c}{a+b+c} \\leqslant(a+b)^{2}+(a+b+4 c)^{2}$ holds for all positive real numbers $a, b, c$.",
      "answer": "100"
    },
    {
      "question": "Example 4. Solve the equation\n\n$$\n\\left|\\frac{x^{2}-6 \\sqrt{x}+7}{x^{2}+6 \\sqrt{x}+7}\\right|=1\n$$",
      "answer": "0"
    },
    {
      "question": "You are traveling in a foreign country whose currency consists of five different-looking kinds of coins. You have several of each coin in your pocket. You remember that the coins are worth 1, 2, 5, 10, and 20 florins, but you have no idea which coin is which and you don't speak the local language. You find a vending machine where a single candy can be bought for 1 florin: you insert any kind of coin, and receive\n\n1 candy plus any change owed. You can only buy one candy at a time, but you can buy as many as you want, one after the other.\n\nWhat is the least number of candies that you must buy to ensure that you can determine the values of all the coins? Prove that your answer is correct.\n\n#",
      "answer": "4"
    },
    {
      "question": "Determine the largest real number $a$ such that for all $n \\geqslant 1$ and for all real numbers $x_{0}, x_{1}, \\ldots, x_{n}$ satisfying $0=x_{0}<x_{1}<x_{2}<\\cdots<x_{n}$, we have  \n$$ \\frac{1}{x_{1}-x_{0}}+\\frac{1}{x_{2}-x_{1}}+\\cdots+\\frac{1}{x_{n}-x_{n-1}} \\geqslant a\\left(\\frac{2}{x_{1}}+\\frac{3}{x_{2}}+\\cdots+\\frac{n+1}{x_{n}}\\right) . $$",
      "answer": "\\frac{4}{9}"
    },
    {
      "question": "7.5. Determine the integer numbers $x$, for which the numbers $n=x^{2}+5 x+1$ and $m=x^{2}+3 x+$ 7 are simultaneously perfect squares.",
      "answer": "3"
    },
    {
      "question": "Find all strictly positive numbers $a, p, n$ such that $p^{a}-1=$ $2^{n}(p-1)$, where $p$ is a prime number.",
      "answer": "2"
    },
    {
      "question": "7. Suppose $A B C D$ is an isosceles trapezoid in which $\\overline{A B} \\| \\overline{C D}$. Two mutually externally tangent circles $\\omega_{1}$ and $\\omega_{2}$ are inscribed in $A B C D$ such that $\\omega_{1}$ is tangent to $\\overline{A B}, \\overline{B C}$, and $\\overline{C D}$ while $\\omega_{2}$ is tangent to $\\overline{A B}, \\overline{D A}$, and $\\overline{C D}$. Given that $A B=1, C D=6$, compute the radius of either circle.",
      "answer": "\\frac{3}{7}"
    },
    {
      "question": "If the price of a product increased from 5.00 to 5.55 reais, what was the percentage increase?",
      "answer": "11"
    },
    {
      "question": "11. (15 points) In the sequence $\\left\\{a_{n}\\right\\}$, $a_{1}, a_{2}$ are given non-zero integers, $a_{n+2}=\\left|a_{n+1}-a_{n}\\right|$.\n（1）If $a_{15}=2, a_{16}=-1$, find $a_{2008}$;\n（2）Prove: From $\\left\\{a_{n}\\right\\}$, it is always possible to select infinitely many terms to form two different constant sequences.",
      "answer": "1"
    },
    {
      "question": "## Task 7\n\nThe students of grades 1, 2, 3, and 4 of the Karl-Marx-Oberschule have committed to donating 300 marks for solidarity this school year.\n\nGrade 1 donated $48.70 \\mathrm{M}$, Grade 2 donated $40.50 \\mathrm{M}$, Grade 3 donated $53.20 \\mathrm{M}$, and Grade 4 donated $61.80 \\mathrm{M}$.\n\na) How many marks have all 4 classes donated in total?\n\nb) How many marks still need to be donated to fulfill the commitment?",
      "answer": "204.20"
    },
    {
      "question": "# 4. Variant 1\n\nThe train was moving from point A to point B at a constant speed. Halfway through the journey, a breakdown occurred, and the train stopped for 15 minutes. After that, the driver had to increase the train's speed by 4 times to arrive at point B on schedule. How many minutes does the train travel from point A to point B according to the schedule?",
      "answer": "40"
    },
    {
      "question": "21. Let's take any 1962-digit number divisible by 9. The sum of its digits will be denoted by $a$, the sum of the digits of $a$ by $b$, and the sum of the digits of $b$ by $c$. What is $c$?[^0]",
      "answer": "9"
    },
    {
      "question": "2. The numbers 1059, 1417, and 2312 leave the same remainder when divided by $d$, and $d>1$, find $d$.",
      "answer": "179"
    },
    {
      "question": "5. Arithmetic density. It is clear that if we take a random natural number from a sufficiently long segment of the natural number series, the probability that this number will be divisible by 10 will be closer to 0.1, the longer the segment.\n\nLet's ask a different question: how much can this probability differ from 0.1? It can be 0, for example, if we specified the segment $1,2,3$, where no number is divisible by 10. And what is:\n\na) (recommended for 6th grade, 1 point) the greatest possible probability;\n\nb) (recommended for 6th grade, 1 point) the smallest possible non-zero probability?",
      "answer": "\\frac{1}{19}"
    },
    {
      "question": "In the parking lot, there were cars and bicycles. If one more car arrived, there would be as many of them as there are bicycles. If five more bicycles arrived, all the bicycles would have as many wheels as all the cars.\n\nHow many cars and how many bicycles were in the parking lot?\n\n(M. Dillingerová)\n\nHint. Imagine a situation where the number of wheels of the cars and bicycles is the same.",
      "answer": "6"
    },
    {
      "question": "Problem 2. Determine the integers $x, y$, for which\n\n$$\n5^{x}-\\log _{2}(y+3)=3^{y} \\text { and } 5^{y}-\\log _{2}(x+3)=3^{x}\n$$",
      "answer": "1"
    },
    {
      "question": "6. Each boy is friends with 5 girls, while all girls are friends with a different number of boys. What is the minimum number of children that can be in this group?",
      "answer": "9"
    },
    {
      "question": "For what value of $k$ is $k \\%$ of 25 equal to $20 \\%$ of 30 ?",
      "answer": "24"
    },
    {
      "question": "Problem 5. Three square paths with a common center are 1 m apart from each other (see figure). Three ants start simultaneously from the lower left corners of the paths and run at the same speed: Mu and Ra counterclockwise, and Wei clockwise. When Mu reaches the lower right corner of the largest path, the other two, who have not yet completed a full circle, are on the right sides of their paths, and all three are on the same straight line. Find the sides of the squares.\n\n[6 points] (A. V. Shapovalov)",
      "answer": "4"
    },
    {
      "question": "It is known that the remainder of dividing a certain prime number by 60 is a composite number. Which one?",
      "answer": "49"
    },
    {
      "question": "25. For an exam answer, a student can receive one of the following grades: $5,4,3,2$. The probability that a student will receive a grade of 5 is 0.3; a grade of $4-0.4$; a grade of $3-0.2$ and a grade of $2-0.1$. Which of the named events form a complete group of mutually exclusive events? What event is opposite to the event: “the student will receive a grade of 5” and what is the probability of this event?",
      "answer": "0.7"
    },
    {
      "question": "Suppose we have a convex polygon in which all interior angles are integers when measured in degrees, and the interior angles at every two consecutive vertices differ by exactly $1^{\\circ}$. If the greatest and least interior angles in the polygon are $M^{\\circ}$ and $m^{\\circ}$, what is the maximum possible value of $M-m$ ?",
      "answer": "18"
    },
    {
      "question": "Folklore\n\nOn a chessboard, $n$ white and $n$ black rooks are placed such that rooks of different colors do not attack each other. Find the maximum possible value of $n$.\n\n#",
      "answer": "16"
    },
    {
      "question": "\\section*{Exercise 1 - 081221}\n\nGive all prime numbers \\(p\\) for which both \\(p+10\\) and \\(p+14\\) are also prime numbers!",
      "answer": "3"
    },
    {
      "question": "1-110 Given the line $a x+b y+c=0$ where the coefficients $a, b, c$ are 3 different elements chosen from the set $\\{-3,-2,-1,0,1,2,3\\}$, and the inclination angle of the line is acute. Find the number of all such different lines.",
      "answer": "43"
    },
    {
      "question": "8. (10 points) As shown in the figure, $A B C D$ is a parallelogram. $A E=2 E B$. If the area of triangle $C E F$ is $=1$. Then, the area of parallelogram $A B C D$ is $\\qquad$ .",
      "answer": "5"
    },
    {
      "question": "1572. Let the germination rate of rye seeds be $90 \\%$. What is the probability that out of 7 sown seeds, 5 will germinate?",
      "answer": "0.124"
    },
    {
      "question": "G2.2 In Figure 2(a), $A B C D$ is a rectangle. $D E: E C=1: 5$, and $D E=12^{\\frac{1}{4}} . \\triangle B C E$ is folded along the side BE. If $b$ is the area of the shaded part as shown in Figure 2(b), find the value of $b$.\n\nG2.3 Let the curve $y=x^{2}-7 x+12$ intersect the $x$-axis at points $A$ and $B$, and intersect the $y$-axis at $C$. If $c$ is the area of $\\triangle A B C$, find the value of $c$.",
      "answer": "6"
    },
    {
      "question": "The cost of withdrawing money from an ATM consists of two parts. There is a base fee, which is independent of the amount withdrawn. In addition, there is a part that is directly proportional to the amount withdrawn. What is the cost of withdrawing $85000 \\mathrm{Ft}$, if the cost of withdrawing $40000 \\mathrm{Ft}$ is $221 \\mathrm{Ft}$, and the cost of withdrawing $100000 \\mathrm{Ft}$ is $485 \\mathrm{Ft}$?",
      "answer": "419"
    },
    {
      "question": "Determine the value of $(666666666)^{2}-(333333333)^{2}$.",
      "answer": "333333332666666667"
    },
    {
      "question": "Let's determine the smallest positive number divisible by 999 that does not contain the digit 9.",
      "answer": "111888"
    },
    {
      "question": "4. Let $A$ be any set of 11 real numbers, and let the set $B=\\{u v \\mid u, v \\in A, u \\neq v\\}$. Find the minimum number of elements in $B$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "17"
    },
    {
      "question": "5. If the function $f(x)=\\frac{a+\\sin x}{2+\\cos x}+b \\tan x$ has a sum of its maximum and minimum values equal to 4, then $a+b=$ $\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "11. A food factory has made 4 different exquisite cards, and each bag of food produced by the factory randomly contains one card. It is stipulated that if one collects all 4 different cards, they can win a prize. Xiao Ming buys 6 bags of this food at once. What is the probability that Xiao Ming will win the prize? $\\qquad$ .",
      "answer": "\\frac{195}{512}"
    },
    {
      "question": "Problem 4.4. Along the road connecting Masha's and Sasha's houses, there are 17 apple trees and 20 poplars. When Masha was going to visit Sasha, she took photos of all the trees. Right after the tenth apple tree, Masha's phone memory ran out, and she couldn't photograph the remaining 13 trees. The next day, when Sasha was going to visit Masha, starting from the eighth apple tree, he picked one leaf from each tree. How many leaves did Sasha pick?",
      "answer": "24"
    },
    {
      "question": "Gombóc Artúr lives at 1 Édes Street, while the chocolate shop is at the other end of the street, at number $n$. Every day, Artúr follows this fitness routine: he starts in front of the house numbered 2. If he stands in front of the house numbered $k$ (where $1<k<n$), he flips an expired but fair chocolate coin. If it lands heads, he moves to the house numbered ( $k-1$ ), and if it lands tails, he moves to the house numbered ( $k+1$ ). If he reaches the chocolate shop, he goes in and drinks a chocolate milkshake, then moves to the house numbered ( $n-1$ ). When he gets home, his workout is over. On average, how many chocolate milkshakes does Artúr drink each day?",
      "answer": "1"
    },
    {
      "question": "Problem 8.7. The figure shows two equal triangles: $A B C$ and $E B D$. It turns out that $\\angle D A E = \\angle D E A = 37^{\\circ}$. Find the angle $B A C$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-27.jpg?height=432&width=711&top_left_y=91&top_left_x=369)",
      "answer": "7"
    },
    {
      "question": "3. If $n$ squares in a plane satisfy:\n(1) All squares are congruent;\n(2) If two squares have a common intersection point $P$, then $P$ is a vertex of each of the squares;\n(3) Each square intersects exactly three other squares, then this set of $n$ squares is called \"triconnected.\"\n\nQuestion: How many positive integers $n$ in $2018 \\leqslant n \\leqslant 3018$ allow for the existence of a triconnected set with $n$ squares?",
      "answer": "501"
    },
    {
      "question": "Example 1: $n$ teams are to hold a home-and-away double round-robin match (each pair of teams plays two matches, each having one home match), and each team can play multiple away matches within a week (from Sunday to Saturday). However, if a team has a home match in a certain week, no away matches for that team can be scheduled within the same week. If all matches can be completed within 4 weeks, find the maximum value of $n$.\n(Note: A match held at Team A's venue is called Team A's home match and Team B's away match.)",
      "answer": "6"
    },
    {
      "question": "Below is a part of a square grid consisting of 4 rows and 2023 columns.\n\nDetermine the number of squares, all of whose vertices are grid points of the square grid.\n\n(K. Pazourek)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_096aded327f4b802fe92g-7.jpg?height=337&width=1422&top_left_y=1214&top_left_x=317)",
      "answer": "40430"
    },
    {
      "question": "In Jen's baseball league, each team plays exactly 6 games against each of the other teams in the league. If a total of 396 games are played, how many teams are in the league?\n(A) 12\n(B) 16\n(C) 15\n(D) 13\n(E) 9",
      "answer": "12"
    },
    {
      "question": "24. What is the unit digit of $1^{3}+2^{3}+3^{3}+\\ldots+2017^{3}+2018^{3}$? (Note: $a^{3}=a \\times a \\times a$)",
      "answer": "1"
    },
    {
      "question": "12. In the figure, there are two squares of different sizes. The area of the shaded part is 1.25 times the area of the blank part. When the side length of the larger square is $a$, the side length of the smaller square is",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "30. There are 700 cards of 6 different colors: red, orange, yellow, green, blue, and white in a box. The ratio of the number of red, orange, and yellow cards is $1: 3: 4$, and the ratio of the number of green, blue, and white cards is 3:1:6. It is known that there are 50 more yellow cards than blue cards. How many cards must be drawn to ensure that at least 60 cards of the same color are drawn?",
      "answer": "312"
    },
    {
      "question": "Example 7 (2003 National High School Competition Question) Given that $a, b, c, d$ are all positive integers, and $\\log _{a} b=\\frac{3}{2}, \\log _{c} d=\\frac{5}{4}$, if $a-c=$ 9, then $b-d=$ $\\qquad$ .",
      "answer": "93"
    },
    {
      "question": "S1.4 Let $f(x)$ be a function such that $f(n)=(n-1) f(n-1)$ and $f(n) \\neq 0$ hold for all integers $n \\geq 6$.\nIf $U=\\frac{f(T)}{(T-1) f(T-3)}$, find the value of $U$.",
      "answer": "72"
    },
    {
      "question": "2. Positive integers $x, y$, for which $\\gcd(x, y)=3$, are the coordinates of a vertex of a square with its center at the origin and an area of $20 \\cdot \\text{lcm}(x, y)$. Find the perimeter of the square.",
      "answer": "24\\sqrt{5}"
    },
    {
      "question": "Problem 10.2. Points $A, B, C, D, E, F, G$ are located clockwise on a circle, as shown in the figure. It is known that $A E$ is the diameter of the circle. Also, it is known that $\\angle A B F=81^{\\circ}, \\angle E D G=76^{\\circ}$. How many degrees does the angle $F C G$ measure?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-36.jpg?height=561&width=616&top_left_y=89&top_left_x=419)",
      "answer": "67"
    },
    {
      "question": "Example 9 (2003 Bulgarian National Mathematical Olympiad) Find the number of real solutions of the system of equations $\\left\\{\\begin{array}{l}x+y+z=3 x y, \\\\ x^{2}+y^{2}+z^{2}=3 x z, \\\\ x^{3}+y^{3}+z^{3}=3 y z\\end{array}\\right.$.",
      "answer": "4"
    },
    {
      "question": "Is there a regular polygon in which the length of the shortest diagonal is equal to the radius of the circumscribed circle of the polygon?",
      "answer": "12"
    },
    {
      "question": "## Task $6 / 83$\n\nIn every rectangle, the angle bisectors intersect at four points that span a square (if the rectangle is a square, these four points coincide).\n\nThe area $A_{Q}$ of this square is to be represented as a function of the side ratio $x=a: b$ (where $a>b, b$ is constant). For which side ratio is the square area $A_{Q}$ equal to the rectangle area $A_{R}$?",
      "answer": "2+\\sqrt{3}"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nFour line segments intersect in points $A, B, C, D$, and $E$, as shown. The measure of $\\angle C E D$ is $x^{\\circ}$. What is the value of $x$ ?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_9c5b2a5bb2c0ebd3406fg-4.jpg?height=439&width=829&top_left_y=1447&top_left_x=651)\n\n##",
      "answer": "30"
    },
    {
      "question": "7. Given vectors $\\boldsymbol{a}, \\boldsymbol{b}, \\boldsymbol{c}$ satisfy\n$$\n\\begin{array}{l}\n|a|=|b|=3,|c|=4, \\boldsymbol{a} \\cdot \\boldsymbol{b}=-\\frac{7}{2}, \\\\\n\\boldsymbol{a} \\perp \\boldsymbol{c}, \\boldsymbol{b} \\perp \\boldsymbol{c} .\n\\end{array}\n$$\n\nThen for real numbers $x, y$, the expression\n$$\n|x \\boldsymbol{a}+y \\boldsymbol{b}+(1-x-y) \\boldsymbol{c}|\n$$\n\nhas the minimum value of $\\qquad$",
      "answer": "\\frac{4\\sqrt{33}}{15}"
    },
    {
      "question": "4. Find the area of the figure, the coordinates $(x ; y)$ of the points of which satisfy the system of inequalities\n\n$$\n\\left\\{\\begin{array}{l}\n|x|+|y| \\geq 1 \\\\\n(|x|-1)^{2}+(|y|-1)^{2} \\leq 1\n\\end{array}\\right.\n$$",
      "answer": "\\pi-2"
    },
    {
      "question": "7. In the addition problem on the right, the sum of the digits of the addend is three times the sum of the digits of the sum. What is the smallest possible addend?\n\n\n\nThe above text has been translated into English, preserving the original text's line breaks and format.",
      "answer": "18"
    },
    {
      "question": "3. If real numbers $x, y$ satisfy $4 x^{2}-4 x y+2 y^{2}=1$, then the sum of the maximum and minimum values of $3 x^{2}+x y+y^{2}$ is $\\qquad$",
      "answer": "3"
    },
    {
      "question": "$\\underline{\\text { Frankin B.R. }}$\n\nGiven a polynomial $P(x)$ with real coefficients. An infinite sequence of distinct natural numbers $a_{1}, a_{2}, a_{3}, \\ldots$ is such that\n\n$P\\left(a_{1}\\right)=0, P\\left(a_{2}\\right)=a_{1}, P\\left(a_{3}\\right)=a_{2}$, and so on. What degree can $P(x)$ have?",
      "answer": "1"
    },
    {
      "question": "Example 24 (IMO-18 Problem) Find the maximum number of elements in a set $S$ that satisfies the following conditions:\n(1) Each element of $S$ is a positive integer not exceeding 100;\n(2) For any two distinct elements $a, b$ in $S$, there exists an element $c$ in $S$ such that the greatest common divisor of $a$ and $c$ is 1, and the greatest common divisor of $b$ and $c$ is also 1;\n(3) For any two distinct elements $a, b$ in $S$, there exists an element $d$ in $S$ different from $a, b$ such that the greatest common divisor of $a$ and $d$ is greater than 1, and the greatest common divisor of $b$ and $d$ is also greater than 1.",
      "answer": "72"
    },
    {
      "question": "Bogdanov I.i.\n\nInitially, there are 111 pieces of plasticine of the same mass on the table. In one operation, you can choose several groups (possibly one) with the same number of pieces and in each group, combine all the plasticine into one piece. What is the minimum number of operations required to get exactly 11 pieces, each of which has a different mass?\n\n#",
      "answer": "2"
    },
    {
      "question": "$$\n2015 \\times 2016\n$$\n1. (8 points) The calculation result of the expression $\\frac{1}{1+\\frac{1}{2016}}+\\frac{2}{2+\\frac{2}{2016}}+\\frac{3}{3+\\frac{3}{2016}}+\\cdots+\\frac{2015}{2015+\\frac{2015}{2016}}$ is",
      "answer": "2017"
    },
    {
      "question": "There are 2017 unoccupied chairs in a row. Every minute, a person arrives and sits in one of them that is empty, and at the same instant, if it is occupied, a person in an adjacent chair stands up and leaves. What is the maximum number of people that can be sitting simultaneously in the row of chairs?",
      "answer": "2016"
    },
    {
      "question": "Let $A, B, C$ be points in a line on this order, with $A B=B C$. A car travels from $A$ to $B$ at 40 kilometers per hour and then from $B$ to $C$ at 60 kilometers per hour, without stopping in between. What is the overall speed of the trip from $A$ to $C$ ?",
      "answer": "48"
    },
    {
      "question": "Exercise 7. Consider a $10 \\times 10$ grid, composed of 100 cells. The rows of the grid are numbered from top to bottom in the order of 1 to 10: the topmost row has the number 1, the bottommost row has the number 10. The columns of the grid are numbered from left to right in the order of 1 to 10: the leftmost column has the number 1, the rightmost column has the number 10. Martin colors some cells of the grid such that for any colored cell $c$, there is at most one different colored cell whose column number is greater than or equal to that of the column of $c$ and whose row number is greater than or equal to that of the row of $c$.\n\nWhat is the maximum number of cells that Martin can color?",
      "answer": "15"
    },
    {
      "question": "10. (This question is worth 20 points) The sequence $\\left\\{a_{n}\\right\\}$ satisfies $a_{1}=\\frac{\\pi}{6}, a_{n+1}=\\arctan \\left(\\sec a_{n}\\right)\\left(n \\in \\mathbf{N}^{*}\\right)$. Find the positive integer $m$, such that\n$$\n\\sin a_{1} \\cdot \\sin a_{2} \\cdots \\cdot \\sin a_{m}=\\frac{1}{100} .\n$$",
      "answer": "3333"
    },
    {
      "question": "15. Given that the function $f(x)$ is a decreasing function defined on $(-\\infty, 1]$, and for all real numbers $x$, the inequality $f(k-\\sin x) \\geqslant$ $f\\left(k^{2}-\\sin ^{2} x\\right)$ always holds, then the value of $k$ is $\\qquad$.",
      "answer": "-1"
    },
    {
      "question": "G3.2 Suppose the equation $x^{2} y-x^{2}-3 y-14=0$ has only one positive integral solution $\\left(x_{0}, y_{0}\\right)$. If $x_{0}+y_{0}=b$, find the value of $b$.",
      "answer": "20"
    },
    {
      "question": "7.4. In triangle $A B C$, the angles $A$ and $C$ at the base are $20^{\\circ}$ and $40^{\\circ}$, respectively. It is known that $A C - A B = 5$ (cm). Find the length of the angle bisector of angle $B$.",
      "answer": "5"
    },
    {
      "question": "1. [3] Find the sum of all real solutions to $x^{2}+\\cos x=2019$.",
      "answer": "0"
    },
    {
      "question": "8. The maximum volume of a cone inscribed in a sphere compared to the volume of the sphere is $\\qquad$\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "\\frac{8}{27}"
    },
    {
      "question": "1. In a convex quadrilateral $ABCD$, $\\angle B=\\angle D=90^{\\circ}, AB=4, AD=5, \\angle A=60^{\\circ}$, $\\frac{BC}{CD}=$ $\\qquad$ .",
      "answer": "2"
    },
    {
      "question": "3. Can a certain number of circles, the sum of whose radii is 1999, be placed in a circle of radius 1 such that no two of them have common interior points?",
      "answer": "3998"
    },
    {
      "question": "Problem 1. The sum of the first four terms of an arithmetic progression, as well as the sum of the first seven of its terms, are natural numbers. In addition, its first term $a_{1}$ satisfies the inequality $a_{1} \\leqslant \\frac{2}{3}$. What is the greatest value that $a_{1}$ can take?",
      "answer": "\\frac{9}{14}"
    },
    {
      "question": "8. In an acute-angled triangle $A B C$, a point $Q$ is chosen on side $A C$ such that $A Q: Q C=1: 2$. From point $Q$, perpendiculars $Q M$ and $Q K$ are dropped to sides $A B$ and $B C$ respectively. It is given that $B M: M A=4: 1, B K=K C$. Find $M K: A C$.",
      "answer": "\\frac{2}{\\sqrt{10}}"
    },
    {
      "question": "Problem 7.1. In the picture, nine small squares are drawn, with arrows on eight of them. The numbers 1 and 9 are already placed. Replace the letters in the remaining squares with numbers from 2 to 8 so that the arrows from the square with the number 1 point in the direction of the square with the number 2 (the number 2 can be in square $A$ or $B$), the arrows from the square with the number 2 point in the direction of the square with the number 3, and so on, the arrows from the square with the number 8 point in the direction of the square with the number 9.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_564c13f715a760703913g-18.jpg?height=389&width=379&top_left_y=96&top_left_x=529)\n\nConstruct the correspondence.\n\n- In square $A$\n- In square $B$\n- In square $C$\n- In square $D$\n- In square $E$\n- In square $F$\n- In square $G$\n- stands the number 2.\n- stands the number 3.\n- stands the number 4.\n- stands the number 5.\n- stands the number 6.\n- stands the number 7.\n- stands the number 8.",
      "answer": "2"
    },
    {
      "question": "A square-based right pyramid has its lateral faces forming a $60^{\\circ}$ angle with the base. The plane that bisects the angle between the base and one of the lateral faces divides the pyramid into two parts. Determine the ratio of the volumes of the two resulting solids.",
      "answer": "\\frac{3}{5}"
    },
    {
      "question": "14. In $\\triangle A B C$, $B D=D C$. There is a piece of cheese on side $A C$, located at the point closest to point $C$ among the four equal segments. On $A D$, there are three mirrors $W_{1}, W_{2}, W_{3}$, which divide $A D$ into four equal segments. A very suspicious mouse is crawling on $A B$ (from $A$ to $B$), with $A B=400$ meters. The mouse can observe the cheese when the mouse, a certain mirror, and the cheese are aligned. Due to its suspicion, the mouse hopes to see the cheese multiple times to ensure that the cheese has not been eaten by other mice before it reaches it. Therefore, in the first minute, it crawls forward 80 meters, in the second minute, it retreats 20 meters, in the third minute, it crawls forward 80 meters, in the fourth minute, it retreats 20 meters, and so on. When the mouse reaches point $B$, it directly rushes along $B C$ to eat the cheese. Question: How many times can the mouse see the cheese on segment $A B$?",
      "answer": "5"
    },
    {
      "question": "Problem 1. At a round table, 60 people are sitting. Each of them is either a knight, who always tells the truth, or a liar, who always lies. Each person at the table said: \"Among the next 3 people sitting to my right, there is no more than one knight.\" How many knights could have been sitting at the table? List all possible options and prove that there are no others.",
      "answer": "30"
    },
    {
      "question": "2. (2 points) Point $M$ lies on the side of a regular hexagon with side length 12. Find the sum of the distances from point $M$ to the lines containing the other sides of the hexagon.",
      "answer": "36\\sqrt{3}"
    },
    {
      "question": "$11 \\cdot 42$ For an integer $x \\geqslant 1$, let $p(x)$ be the smallest prime that does not divide $x$, and $q(x)$ be the product of all primes less than $p(x)$. In particular, $p(1)=2$, and if there is some $x$ such that $p(x)=2$, then $q(x)=1$.\nThe sequence $x_{0}, x_{1}, x_{2}, \\cdots$ is defined by:\n$$\n\\begin{array}{c}\nx_{0}=1, \\\\\nx_{n+1}=\\frac{x_{n} p\\left(x_{n}\\right)}{q\\left(x_{n}\\right)}\n\\end{array}\n$$\n\nwhere $n \\geqslant 0$, find all integers $n$ such that $x_{n}=1995$.\n(36th International Mathematical Olympiad Preliminary Question, 1995)",
      "answer": "142"
    },
    {
      "question": "7. Given $P$ is the circumcenter of $\\triangle A B C$, and $\\overrightarrow{P A}+\\overrightarrow{P B}=\\lambda \\overrightarrow{P C}, \\tan C$ $=\\frac{12}{5}$, then the value of the real number $\\lambda$ is $\\qquad$.",
      "answer": "-\\frac{10}{13}"
    },
    {
      "question": "C3 a) In how many ways can we read the word SARAJEVO from the table below, if it is allowed to jump from cell to an adjacent cell (by vertex or a side) cell?\n\n![](https://cdn.mathpix.com/cropped/2024_06_05_17e61ec1e039d21af3a0g-11.jpg?height=497&width=596&top_left_y=631&top_left_x=730)\n\nb) After the letter in one cell was deleted, only 525 ways to read the word SARAJEVO remained. Find all possible positions of that cell.",
      "answer": "750"
    },
    {
      "question": "2. Let positive numbers $a, b, c, x, y, z$ satisfy $c y + b z = a, a z + c x = b, b x + a y = c$, find the minimum value of the function $f(x, y, z)=$\n$\\frac{x^{2}}{1+x}+\\frac{y^{2}}{1+y}+\\frac{z^{2}}{1+z}$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "3.10. In the square $A B C D$, an isosceles triangle $A E F$ is inscribed; point $E$ lies on side $B C$, point $F$ lies on side $C D$, and $A E=A F$. The tangent of angle $A E F$ is 3. Find the cosine of angle $F A D$.",
      "answer": "\\frac{2\\sqrt{5}}{5}"
    },
    {
      "question": "2. Radmila brought a basket of eggs to the market. To the first buyer, she sold half of all the eggs and one more egg. To the second, she sold half of the remaining eggs and one more egg. Similarly, to the third, she sold half of the remaining eggs and one more egg. Then, one egg was left in the basket. How many eggs did Radmila bring to the market?",
      "answer": "15"
    },
    {
      "question": "Problem 4. The maze for mice (see the figure) is a $5 \\times 5$ meter square, and the mice can only run along the paths. Two pieces of cheese (marked with crosses) are placed at two intersections. At another intersection, a mouse (marked with a circle) is sitting. She can smell the cheese, but she needs to run the same distance to both pieces, so she doesn't know which one to choose and sits thoughtfully in place.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_51d7eed368663d90d59bg-2.jpg?height=260&width=273&top_left_y=1152&top_left_x=1002)\n\na) Mark five more intersections where the mouse could sit thoughtfully (from where she would need to run the same distance to both pieces of cheese).\n\n[2 points]\n\nb) Come up with two intersections where you can place a piece of cheese so that there are as many suitable intersections for a thoughtful mouse as possible.\n\n[up to 5 points]",
      "answer": "26"
    },
    {
      "question": "## T-3\n\nFind the smallest integer $b$ with the following property: For each way of colouring exactly $b$ squares of an $8 \\times 8$ chessboard green, one can place 7 bishops on 7 green squares so that no two bishops attack each other.\n\nRemark. Two bishops attack each other if they are on the same diagonal.\n\nAnswer. 41",
      "answer": "41"
    },
    {
      "question": "6. Given a rectangle $A B C D$ with point $S$ being the intersection of the diagonals. On side $\\overline{A D}$, points $E$ and $F$ are chosen such that $|A E|=|E F|=|F D|$. Determine the ratio of the area of pentagon $E B S C F$ to the area of rectangle $A B C D$.",
      "answer": "\\frac{5}{12}"
    },
    {
      "question": "130. How many four-digit numbers can be formed from the digits $0,1,2,3,4,5$, if the numbers must be odd and digits may be repeated?",
      "answer": "540"
    },
    {
      "question": "4. The number $a_{n}$ is formed by writing down the first n consecutive natural numbers in sequence, for example, $a_{13}=12345678910111213$. Determine how many numbers divisible by 24 are among the numbers $a_{1}, a_{2}, \\ldots, a_{10000}$.",
      "answer": "834"
    },
    {
      "question": "Task 4. Masha wrote a three-digit number on the board, and Vera wrote the same number next to it, but she swapped the last two digits. After that, Polina added the obtained numbers and got a four-digit sum, the first three digits of which are 195. What is the last digit of this sum? (The answer needs to be justified.)\n\nAnswer: 4.",
      "answer": "4"
    },
    {
      "question": "## SUBJECT I\n\nDetermine the non-zero natural number n such that $\\frac{2}{3 \\cdot 5}+\\frac{2}{5 \\cdot 7}+\\ldots . .+\\frac{2}{(n-2) \\cdot n}=\\frac{332}{999}$.",
      "answer": "999"
    },
    {
      "question": "【Question 19】\nThere are 2014 apples in three baskets. If the same number of apples is taken from each basket, the second basket will have 49 apples left, and the number of apples left in the first basket is twice the number of apples left in the third basket. How many apples were originally in the third basket? $\\qquad$",
      "answer": "655"
    },
    {
      "question": "Zhendarov R.G.\n\nFind all such prime numbers $p$ that the number $p^{2}+11$ has exactly six distinct divisors (including one and the number itself).",
      "answer": "3"
    },
    {
      "question": "Problem 4.7. Vanya goes to the swimming pool every Wednesday and Friday. After one of his visits, he noticed that he had already gone 10 times this month. What will be the date of the third visit in the next month if he continues to go on Wednesdays and Fridays?",
      "answer": "12"
    },
    {
      "question": "18. As shown in the figure, the lengths of the base edges of the rectangular prism are $1 \\mathrm{~cm}$ and $3 \\mathrm{~cm}$, and the height is $6 \\mathrm{~cm}$. If a thin string is used to start from point $A$, pass through the four sides and wrap around once to reach point $B$, then the shortest length of the string used is $\\qquad$ $\\mathrm{cm}$.",
      "answer": "10"
    },
    {
      "question": "## Task $1 / 88$\n\nLet $x$ be the sum of the digits of $1988^{8891}$, $y$ the sum of the digits of $x$, and $z$ the sum of the digits of $y$. Determine $z$!",
      "answer": "8"
    },
    {
      "question": "1 In a certain municipality, they want to set up a hydroelectric power plant, and for this purpose, they wish to utilize the energy of the water flowing at the border. Preliminary calculations are to be made to determine how many light bulbs the water's energy can power in the case of the lowest water level. The river flows in a rectangular parallelogram-shaped channel at one point, which is $5 \\mathrm{~m}$ wide. Even at the lowest water level, the depth of the water is $0.9 \\mathrm{~m}$. The speed of the water was determined using a floating piece of wood and it was found that it covers a distance of $70 \\mathrm{~m}$ in 100 seconds. The terrain conditions are such that a drop of $3 \\mathrm{~m}$ can be easily established. The turbines and associated generators convert $80 \\%$ of the available energy into electrical energy, and $10 \\%$ of the energy is lost in the transmission lines. A 16 standard candlepower lamp in a carbon filament operates at 110 Volts and 0.5 Amperes.[^0]\n\n\n[^0]:    1* We also gladly accept solutions to starred problems from university students.",
      "answer": "1214"
    },
    {
      "question": "24. Mowgli asked his monkey friends to bring him some nuts. The monkeys gathered an equal number of nuts and carried them to Mowgli. But on the way, they quarreled, and each monkey threw a nut at every other monkey. As a result, Mowgli received only 35 nuts. How many nuts did the monkeys gather, if it is known that each of them brought more than one nut?",
      "answer": "11"
    },
    {
      "question": "7.2. Losharik is going to visit Sovunya along the river at a speed of 4 km/h. Every half hour, he launches paper boats that float to Sovunya at a speed of 10 km/h. With what time interval do the boats arrive at Sovunya? (Provide a complete solution, not just the answer.)",
      "answer": "18"
    },
    {
      "question": "98. There are 3 integers $\\mathrm{A}, \\mathrm{B}, \\mathrm{C}$, if 5 times $\\mathrm{B}$ is 4 times the difference between $\\mathrm{A}$ and 1, 5 times $\\mathrm{C}$ is 4 times the difference between $\\mathrm{B}$ and 1, then the smallest value of $\\mathrm{A}$ is $\\qquad$.",
      "answer": "21"
    },
    {
      "question": "For what values of $x$ is the expression $4^{\\sin ^{2} x}+8^{\\cos ^{2} x}$ minimal? What is the minimum value?",
      "answer": "5\\cdot\\sqrt[5]{\\frac{16}{27}}"
    },
    {
      "question": "Task A-3.5. (4 points)\n\nIf the lengths of the sides of a triangle satisfy the equation $\\frac{a+b}{b+c}=\\frac{c}{a-b}$, determine the measure of the largest angle of this triangle.",
      "answer": "120"
    },
    {
      "question": "In the diagram, points $B$ and $C$ lie on $A D$ and point $E$ lies on $C F$. The measures of four angles are shown. What is the value of $x+y+z$ ?\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_2efeb2001664c4e9869eg-1.jpg?height=429&width=656&top_left_y=1268&top_left_x=797)",
      "answer": "124"
    },
    {
      "question": "[ Systems of points and segments (other) ] [ Classical combinatorics (other) ]\n\nOn a circle, 20 points are marked. How many triples of chords with endpoints at these points exist such that each chord intersects the other two (possibly at the endpoints)?",
      "answer": "156180"
    },
    {
      "question": "# PROBLEM 1\n\na) Show that: $A=\\sqrt{1+3+5+\\ldots+2015} \\in Q$\n\nb) If the real numbers $a$ and $b$ satisfy the relation: $a^{2}+b^{2}-4 \\sqrt{3} a-6 \\sqrt{2} b+30=0$, then calculate:\n\n$$\n\\mathrm{E}=\\left(2 \\mathrm{a}^{-1}+3 \\mathrm{~b}^{-1}\\right)\\left(\\frac{1}{b^{-1}}-\\frac{1}{a^{-1}}\\right)\n$$",
      "answer": "1"
    },
    {
      "question": "10.1. The first term of the sequence is 934. Each subsequent term is equal to the sum of the digits of the previous term, multiplied by 13. Find the 2013-th term of the sequence.",
      "answer": "130"
    },
    {
      "question": "We consider all positive real numbers $a, b, c, d$ for which $(a+c)(b+d)=a c+b d$ holds.\nDetermine the smallest value that $S=\\frac{a}{b}+\\frac{b}{c}+\\frac{c}{d}+\\frac{d}{a}$ can take.",
      "answer": "8"
    },
    {
      "question": "1. Option 1. To a confectionery factory for cake packaging, 5 rolls of ribbon, each 60 m long, were delivered. How many cuts need to be made to get pieces of ribbon, each 1 m 50 cm long?",
      "answer": "195"
    },
    {
      "question": "97.3. Let $A, B, C$, and $D$ be four different points in the plane. Three of the line segments $A B, A C, A D, B C, B D$, and $C D$ have length a. The other three have length $b$, where $b>a$. Determine all possible values of the quotient $\\frac{b}{a}$.",
      "answer": "\\frac{\\sqrt{5}+1}{2}"
    },
    {
      "question": "[Mathematical logic (miscellaneous).] Case analysis $\\quad]$\n\nAfter a hockey game, Anton said that he scored 3 goals, and Ilya only one. Ilya said that he scored 4 goals, and Seryozha as many as 5. Seryozha said that he scored 6 goals, and Anton only two. Could it be that together they scored 10 goals, given that each of them told the truth once and lied once?",
      "answer": "8"
    },
    {
      "question": "13.4. 19 ** In $\\triangle A B C$ with a fixed perimeter, it is known that $|A B|=6$, and when vertex $C$ is at a fixed point $P$, $\\cos C$ has a minimum value of $\\frac{7}{25}$.\n（1）Establish an appropriate coordinate system and find the equation of the locus of vertex $C$;\n(2) Draw a line through point $A$ that intersects the curve from (1) at points $M$ and $N$, and find the minimum value of $|\\overrightarrow{B M}| \\cdot|\\overrightarrow{B N}|$.",
      "answer": "16"
    },
    {
      "question": "Problem 5.7. In a class, there are 31 students. Three of them have exactly three friends each, the next three have six each, the next three have nine each, ..., and the next three have thirty each. How many friends does the 31st student have? (Friendship between people is mutual.)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_32e5791ee8081231aabeg-6.jpg?height=494&width=525&top_left_y=80&top_left_x=464)\n\nFig. 2: to the solution of problem 5.6\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_32e5791ee8081231aabeg-6.jpg?height=260&width=521&top_left_y=661&top_left_x=466)\n\nFig. 3: to the solution of problem 5.6\n\n#",
      "answer": "15"
    },
    {
      "question": "4. Arrange the digits $1,2,3,4,5,6,7,8$ to form two 4-digit integers whose difference is as small as possible.\nExplain clearly why your arrangement achieves the smallest possible difference.",
      "answer": "247"
    },
    {
      "question": "In triangle $ABC$, side $BC$ is equal to 4, and the median drawn to this side is equal to 3. Find the length of the common chord of two circles, each of which passes through point $A$ and is tangent to $BC$, with one touching $BC$ at point $B$ and the other at point $C$.",
      "answer": "\\frac{5}{3}"
    },
    {
      "question": "12. (10 points) Xiaoming went to a store with 100 RMB to buy stationery. After returning home, he counted the RMB he got back, which included 4 different denominations of banknotes and 4 different denominations of coins. The banknotes were all worth more than 1 yuan, and the coins were all worth less than 1 yuan. Additionally, the sum of the values of all banknotes, in yuan, is divisible by 3, and the sum of the values of all coins, in fen, is divisible by 7. How much money did Xiaoming spend at most? (Note: The store has banknotes of 100 yuan, 50 yuan, 20 yuan, 10 yuan, 5 yuan, and 1 yuan, and coins of 5 jiao, 1 jiao, 5 fen, 2 fen, and 1 fen for change)",
      "answer": "63"
    },
    {
      "question": "## Task 2\n\nSubtract from 3182100 the product of the numbers 56823 and 56. By how much is the difference greater than 10?",
      "answer": "2"
    },
    {
      "question": "3. A three-digit number, where the hundreds and units digits are the same, and this three-digit number is a perfect square. There are $\\qquad$ such three-digit numbers.\n\nNote: A perfect square can be expressed in the form of $a \\times a$, such as $9=3 \\times 3, 25=5 \\times 5$, both 9 and 25 are perfect squares.",
      "answer": "3"
    },
    {
      "question": "3. Find the number of distinct reduced quadratic trinomials (i.e., with the leading coefficient equal to 1) with integer coefficients such that they have at least one root, all their roots are powers of the number 3 with integer non-negative exponents, and their coefficients in absolute value do not exceed \\(27^{47}\\).",
      "answer": "5111"
    },
    {
      "question": "3. [7 points] Let $M$ be a figure on the Cartesian plane consisting of all points $(x, y)$ such that there exists a pair of real numbers $a, b$ for which the system of inequalities is satisfied\n\n$$\n\\left\\{\\begin{array}{l}\n(x-a)^{2}+(y-b)^{2} \\leqslant 5 \\\\\na^{2}+b^{2} \\leqslant \\min (4 a-2 b ; 5)\n\\end{array}\\right.\n$$\n\nFind the area of the figure $M$.",
      "answer": "15\\pi-\\frac{5\\sqrt{3}}{2}"
    },
    {
      "question": "Paul has 6 boxes, each of which contains 12 trays. Paul also has 4 extra trays. If each tray can hold 8 apples, what is the largest possible number of apples that can be held by the 6 boxes and 4 extra trays?",
      "answer": "608"
    },
    {
      "question": "Example 3 Find all positive integers $k$ such that the indeterminate equation $x^{2}+y^{2}=k x y-1$ has positive integer solutions for $x$ and $y$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "3"
    },
    {
      "question": "3. Given three points $A, B, C$, forming a triangle with angles $30^{\\circ}, 45^{\\circ}, 105^{\\circ}$. Two of these points are chosen, and the perpendicular bisector of the segment connecting them is drawn, after which the third point is reflected across this perpendicular bisector. This results in a fourth point $D$. With the resulting set of 4 points, the same procedure is carried out - two points are chosen, the perpendicular bisector is drawn, and all points are reflected across it. What is the maximum number of distinct points that can be obtained by repeatedly performing this procedure?",
      "answer": "12"
    },
    {
      "question": "(2) Given the function $f(x)=\\sqrt{3} \\sin 2 x+2 \\cos ^{2} x+a$, if the minimum value of $f(x)$ on the interval $\\left[0, \\frac{\\pi}{2}\\right]$ is -1, then the value of $a$ is $\\qquad$.",
      "answer": "-1"
    },
    {
      "question": "How many positive five-digit whole numbers are there for which the product of their five digits is 900?\n\n(Karl Czakler)\n\nAnswer. 210",
      "answer": "210"
    },
    {
      "question": "Problem 3. In the morning, a dandelion blooms, it flowers yellow for two days, on the third day in the morning it turns white, and by evening it sheds its seeds. Yesterday afternoon, there were 20 yellow and 14 white dandelions on the meadow, and today there are 15 yellow and 11 white.\na) How many yellow dandelions were there on the meadow the day before yesterday\n\nb) How many white dandelions will there be on the meadow tomorrow? [6 points] (",
      "answer": "25"
    },
    {
      "question": "10. (5 points) According to the pattern shown in the figure, deduce that $M=$\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "1692"
    },
    {
      "question": "11.4. 2011 Warehouses are connected by roads in such a way that from any warehouse you can drive to any other, possibly by driving along several roads. On the warehouses, there are $x_{1}, \\ldots, x_{2011}$ kg of cement respectively. In one trip, you can transport an arbitrary amount of cement from any warehouse to another warehouse along the road connecting them. In the end, according to the plan, there should be $y_{1}, \\ldots, y_{2011}$ kg of cement on the warehouses respectively, with\n\n$$\nx_{1}+x_{2}+\\ldots+x_{2011}=y_{1}+y_{2}+\\ldots+y_{2011}\n$$\n\nWhat is the minimum number of trips required to fulfill the plan for any values of the numbers $x_{i}$ and $y_{i}$ and any road network?\n\n(P. Karasev)",
      "answer": "2010"
    },
    {
      "question": "[The ratio in which the bisector divides the side.] Law of Cosines\n\nIn triangle $ABC$, the bisector of angle $BAC$ intersects side $BC$ at point $M$. It is known that $AB = BC = 2AC$, $AM = 4$. Find the area of triangle $ABC$.",
      "answer": "\\frac{18\\sqrt{15}}{5}"
    },
    {
      "question": "【Question 21】\nFour football teams participate in a round-robin tournament, where each pair of teams plays one match. The winning team gets 3 points, the losing team gets 0 points, and in the case of a draw, both teams get 1 point. It is known that after all the matches, the points of the four teams are four consecutive positive integers. Find the product of these four numbers.",
      "answer": "120"
    },
    {
      "question": "8.6. A triangle is divided into triangular cells as shown in the figure. A natural number is written in each cell. For each side of the triangle, there are four layers parallel to this side, containing seven, five, three, and one cell, respectively. It turns out that the sum of the numbers in each of these twelve layers is a prime number. What is the smallest possible sum of all the written numbers?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6ab074edf53f8bb32518g-2.jpg?height=334&width=391&top_left_y=1843&top_left_x=1546)",
      "answer": "22"
    },
    {
      "question": "9.6. Find the largest $n$ such that the sum of the fourth powers of any $n$ prime numbers greater than 10 is divisible by $n$.",
      "answer": "240"
    },
    {
      "question": "Problem 5.3. A cuckoo clock is hanging on the wall. When a new hour begins, the cuckoo says \"cuckoo\" a number of times equal to the number the hour hand points to (for example, at 19:00, \"cuckoo\" sounds 7 times). One morning, Maxim approached the clock when it was 9:05. He started turning the minute hand until he advanced the clock by 7 hours. How many times did \"cuckoo\" sound during this time?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d43b1f0a7bd77fffa87ag-10.jpg?height=523&width=311&top_left_y=522&top_left_x=565)",
      "answer": "43"
    },
    {
      "question": "## Task B-1.1.\n\nFor real numbers $x$ and $y$, if $x-y=6$ and $x^{2}+y^{2}=22$, what is $x^{3}-y^{3}$?",
      "answer": "90"
    },
    {
      "question": "1. Calculate the value of the expression:\n\n$$\n\\frac{(\\sqrt{666}+\\sqrt{888})^{2}-\\sqrt{666^{2}+888^{2}}}{444}\n$$",
      "answer": "1+2\\sqrt{3}"
    },
    {
      "question": "3-ча 1. Determine the coefficients that will stand by $x^{17}$ and $x^{18}$ after expanding the brackets and combining like terms in the expression\n\n$$\n\\left(1+x^{5}+x^{7}\\right)^{20}\n$$",
      "answer": "3420"
    },
    {
      "question": "9. (10 points) In the figure, $AB$ is the diameter of circle $O$, 6 cm long, and square $BCDE$ has one vertex $E$ on the circumference of the circle, $\\angle ABE=45^{\\circ}$. The difference between the area of the non-shaded part of circle $O$ and the area of the non-shaded part of square $BCDE$ is $\\qquad$ square centimeters (take $\\pi=3.14$)",
      "answer": "10.26"
    },
    {
      "question": "4. It is known that $3 \\operatorname{tg}^{2} \\gamma+3 \\operatorname{ctg}^{2} \\gamma+\\frac{2}{\\sin ^{2} \\gamma}+\\frac{2}{\\cos ^{2} \\gamma}=19$. Find the value of the expression $\\sin ^{4} \\gamma-\\sin ^{2} \\gamma$.\n\nAnswer. $-\\frac{1}{5}$.",
      "answer": "-\\frac{1}{5}"
    },
    {
      "question": "The last digit of each number is 6. What is the condition for the product of the two numbers to end in 36?\n\n翻译完成，但根据您的要求，我保留了源文本的换行和格式。请注意，这里提供的翻译是将匈牙利语翻译成了英文。如果您需要进一步的帮助或有其他要求，请告诉我。",
      "answer": "35"
    },
    {
      "question": "## Task 1 - 291211\n\nDetermine the number of all natural numbers $z$ with the following properties:\n\n(1) The decimal representation of $z$ consists of five pairwise distinct digits.\n\n(2) The first and the last digit in it are different from 0.\n\n(3) If $z^{\\prime}$ is the number whose digit representation is obtained from that of $z$ by reversing the order, then the digit representation of the number $z+z^{\\prime}$ consists of all identical digits.",
      "answer": "56"
    },
    {
      "question": "Example 2.2. $I=\\int_{0}^{2}(x-1)^{3} d x$.",
      "answer": "0"
    },
    {
      "question": "Variant 11.3.3. The polynomial $G(x)$ with real coefficients takes the value 2022 at exactly five different points $x_{1}<x_{2}<x_{3}<x_{4}<x_{5}$. It is known that the graph of the function $y=G(x)$ is symmetric with respect to the line $x=-7$.\n\n(a) (2 points) Find $x_{1}+x_{3}+x_{5}$.\n\n(b) (2 points) What is the smallest degree that $G(x)$ can have?",
      "answer": "-21"
    },
    {
      "question": "3. A natural number greater than 0, if the sum of all its divisors equals twice itself, such a number is called a perfect number or a complete number. For example, the divisors of 6 are $1,2,3,6$, $1+2+3+6=12$, so 6 is the smallest perfect number. Whether there are infinitely many perfect numbers remains one of the challenges that have puzzled humans. Research on perfect numbers can start with calculating the sum of all divisors of a natural number, the sum of all divisors of 8128 is.",
      "answer": "16256"
    },
    {
      "question": "9. There is a heads up coin on every integer of the number line. Lucky is initially standing on the zero point of the number line facing in the positive direction. Lucky performs the following procedure: he looks at the coin (or lack thereof) underneath him, and then,\n- If the coin is heads up, Lucky flips it to tails up, turns around, and steps forward a distance of one unit.\n- If the coin is tails up, Lucky picks up the coin and steps forward a distance of one unit facing the same direction.\n- If there is no coin, Lucky places a coin heads up underneath him and steps forward a distance of one unit facing the same direction.\nHe repeats this procedure until there are 20 coins anywhere that are tails up. How many times has Lucky performed the procedure when the process stops?",
      "answer": "6098"
    },
    {
      "question": "2. Given two linear functions $f(x)$ and $g(x)$ such that the graphs $y=f(x)$ and $y=g(x)$ are parallel lines, not parallel to the coordinate axes. Find the minimum value of the function $(g(x))^{2}+$ $f(x)$, if the minimum value of the function $(f(x))^{2}+g(x)$ is -6.",
      "answer": "\\frac{11}{2}"
    },
    {
      "question": "1. How many four-digit natural numbers can be written using the digits $0,1,2,3,4,5,6$, if all digits must be different and the sum of the digits must be $15?$",
      "answer": "66"
    },
    {
      "question": "Problem 3. In a competition 8 judges marked the contestants by yes or no. It is known that for any two contestants, two judges gave both a yes; two judges gave the first one a yes and the second one a no; two judges gave the first one a no and the second one a yes, and finally, two judges gave both a no. What is the greatest possible number of contestants?\n\nEmil Kolev",
      "answer": "7"
    },
    {
      "question": "25.15. Calculate $\\lim _{n \\rightarrow \\infty}(\\sqrt{n+1}-\\sqrt{n})$.",
      "answer": "0"
    },
    {
      "question": "【Question 7】\nA car has a fuel tank capacity of 50 liters, and it departs from Shanghai to Harbin, which is 2560 kilometers away, with a full tank of fuel. It is known that the car consumes 8 liters of fuel for every 100 kilometers driven, and to ensure driving safety, at least 6 liters of fuel should be kept in the tank. Therefore, during the journey to Harbin, the car needs to refuel at least $\\qquad$ times.",
      "answer": "4"
    },
    {
      "question": "13. (15 points) As shown in the figure, in the isosceles right triangle $A B C$, $\\angle A=90^{\\circ}, A B=A C=1$, rectangle $E H G F$ is inside triangle $A B C$, and $G, H$ are on side $B C$. Find the maximum area of rectangle $E H G F$.",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Multiplied several natural numbers and got 224, and the smallest number was exactly half of the largest. How many numbers were multiplied\n\n#",
      "answer": "3"
    },
    {
      "question": "We roll a die 30 times. Determine the probability, to 8 decimal places, that among the numbers rolled there will be a six, and that before the first six, neither a five nor a four appears.",
      "answer": "0.33333333"
    },
    {
      "question": "Example 1. Using the method of tangents, find the real root of the equation $x^{3}+x-3=0$.",
      "answer": "1.213412"
    },
    {
      "question": "Example 4-15 The 6 faces of a die have $1,2,3,4,5,6$ dots respectively, how many different schemes are there?",
      "answer": "30"
    },
    {
      "question": "B1. Find all real numbers $x$ that satisfy the equation\n\n$$\n\\log _{2}(10 x)+\\log _{4}(100 x)+\\log _{8}(1000 x)-2 \\log _{64} x=9\n$$\n\nWrite the result in the form of a simplified fraction.\n\n$$\n\\text { (6 points) }\n$$",
      "answer": "\\frac{16}{25}"
    },
    {
      "question": "Task 1. Calculate the value of the numerical expression:\n\n$$\n\\frac{1}{17} \\cdot\\left(\\frac{2 \\frac{1}{2}+3 \\frac{1}{3}}{3 \\frac{1}{2}-2 \\frac{1}{3}}: \\frac{5 \\frac{3}{5}+1 \\frac{1}{3}}{6 \\frac{3}{5}-4 \\frac{1}{3}}\\right) \\cdot\\left(\\frac{\\frac{1}{4}-\\frac{1}{5}}{\\frac{1}{7}-\\frac{1}{8}}-\\frac{1}{5}\\right)\n$$",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "'1-188 Write the numbers $1,2,3, \\cdots, 1986,1987$ on the blackboard. At each step, determine some of the numbers written and erase them, replacing them with the remainder of their sum divided by 7. After several steps, two numbers remain on the blackboard, one of which is 987. What is the second remaining number?",
      "answer": "0"
    },
    {
      "question": "[ The ratio of the areas of triangles with a common base or common height ]\n\nIn one island tribe, there is a custom - during a ritual dance, the shaman tosses three thin straight rods of the same length, tied in a shape resembling the letter П, high into the air. Adjacent rods are connected by a short thread and can therefore rotate freely relative to each other. The rods fall onto the sand, forming a random figure. If a self-intersection occurs (the first and third rods cross each other), the tribe will face poor harvests and various troubles in the coming year. If there is no self-intersection, the year will be successful - bountiful and happy. Find the probability that the rods will predict good fortune for the year 2017.",
      "answer": "\\frac{11}{12}"
    },
    {
      "question": "# 7.2. (7 points)\n\nTwo pedestrians set out at dawn. Each walked at a constant speed. One walked from $A$ to $B$, the other from $B$ to $A$. They met at noon and, without stopping, arrived: one - in $B$ at 4 PM, and the other - in $A$ at 9 PM. At what hour was dawn that day?",
      "answer": "6"
    },
    {
      "question": "9.4. In a class, there are 18 children. The parents decided to gift the children from this class a cake. To do this, they first asked each child the area of the piece they wanted to receive. After that, they ordered a square cake, the area of which is exactly equal to the sum of the 18 named numbers. However, upon seeing the cake, the children wanted their pieces to be square as well. The parents can cut the cake with cuts parallel to the sides of the cake (cuts are not required to start or end on the side of the cake). For what largest $k$ can the parents guarantee to cut out $k$ square pieces from the ordered cake, which can be given to $k$ children, so that each of them gets what they want?\n\n(A. Ibragimov, I. Bogdanov)",
      "answer": "12"
    },
    {
      "question": "156. Given a cube $A B C D A_{1} B_{1} C_{1} D_{1}$. Points $M$ and $N$ are taken on segments $A A_{1}$ and $B C_{1}$ such that the line $M N$ intersects the line $B_{1} D$. Find\n\n$$\n\\frac{\\left|B C_{1}\\right|}{|B N|}-\\frac{|A M|}{\\left|A A_{1}\\right|}\n$$",
      "answer": "1"
    },
    {
      "question": "6. (3 points) Given the cryptarithm: ЖАЛО + ЛОЖА = ОСЕНЬ. Identical letters represent identical digits, different letters represent different digits. Find the value of the letter А.",
      "answer": "8"
    },
    {
      "question": "30.5. Let each of the characters $A, B, C, D, E$ denote a single digit, and $A B C D E 4$ and $4 A B C D E$ represent six-digit numbers. If\n$$\n4 \\times A B C D E 4=4 A B C D E,\n$$\nwhat is $C$ ?",
      "answer": "2"
    },
    {
      "question": "3. Given that quadrilateral $A B C D$ is an isosceles trapezoid, $A B / / C D, A B=6, C D=16 . \\triangle A C E$ is a right triangle, $\\angle A E C=90^{\\circ}, C E=B C=A D$. Then the length of $A E$ is $\\qquad$ .",
      "answer": "4\\sqrt{6}"
    },
    {
      "question": "11. A student missed the multiplication sign written between two three-digit numbers, mistaking them for a six-digit number, which turned out to be exactly 7 times the original product. What is the sum of these two three-digit numbers?",
      "answer": "286"
    },
    {
      "question": "Task B-2.7. Today is Valentine's Day, and Valentino wants to go out with his girlfriend Lorna. With his 120 kuna, he can buy several roses and two lemonades. The price of one rose is equal to the price of one lemonade. Since they have a math test tomorrow, they will postpone the outing until Saturday. On Saturday, the price of one rose will decrease by 7 kuna. Valentino wants to buy the same number of roses, and instead of lemonades, he wants to buy a jumbo pizza and a Coca-Cola, which together cost 78 kuna. However, he would be short of 6 kuna. How many roses will Valentino buy for Lorna? At what price will he buy the roses on Saturday?",
      "answer": "6"
    },
    {
      "question": "117. Given a regular tetrahedron with edge $a$. A sphere touches the three edges of the tetrahedron emanating from one vertex at their ends. Find the area of the part of the spherical surface located inside the tetrahedron.",
      "answer": "\\frac{\\pi^{2}}{6}(2\\sqrt{3}-3)"
    },
    {
      "question": "$\\underline{\\text { Folklore }}$\n\nIn a herd consisting of horses, two-humped and one-humped camels, there are a total of 200 humps.\n\nHow many animals are in the herd if the number of horses is equal to the number of two-humped camels? .",
      "answer": "200"
    },
    {
      "question": "Matěj and his friends went caroling. In addition to apples, walnuts, and gingerbread, each of the boys also received an orange. Jarda got one orange, Milan too. Radek, Patrik, Michal, and Dušan each got two oranges. Matěj got as many as four oranges, which was the most of all the boys. The other boys each got three oranges. How many boys went caroling if they received a total of 23 oranges?\n\n(M. Volfová)",
      "answer": "10"
    },
    {
      "question": "The equations of the sides of a quadrilateral are:\n\n$$\ny=-x+7, \\quad y=\\frac{x}{2}+1, \\quad y=-\\frac{3}{2} x+21, \\quad y=\\frac{7}{4} x+\\frac{3}{2}\n$$\n\nDetermine the coordinates of the vertices of the quadrilateral and its area.",
      "answer": "35"
    },
    {
      "question": "5. Vova has 19 math grades in his journal, all twos and threes, and the first four grades are twos. It turned out that among the quartets of consecutive grades, all 16 possible combinations of four twos and threes are present. What are Vova's last four grades?",
      "answer": "3222"
    },
    {
      "question": "46th Putnam 1985 Problem A2 ABC is an acute-angled triangle with area 1. A rectangle R has its vertices R i on the sides of the triangle, R 1 and R 2 on BC, R 3 on AC and R 4 on AB. Another rectangle S has it vertices on the sides of the triangle AR 3 R 4 , two on R 3 R 4 and one one each of the other two sides. What is the maximum total area of R and S over all possible choices of triangle and rectangles? Solution",
      "answer": "\\frac{2}{3}"
    },
    {
      "question": "24. Find the number of 7-digit integers formed by some or all of the five digits, namely, $0,1,2,3$, and 4 , such that these integers contain none of the three blocks 22,33 and 44 .",
      "answer": "29776"
    },
    {
      "question": "20. A palindrome number is the same either read from left to right or right to left, for example. 121 is a palindrome number. How many 5 -digit palindrome numbers are there together?",
      "answer": "900"
    },
    {
      "question": "20. Let $A$ be the sum of all non-negative integers $n$ satisfying\n$$\n\\left\\lfloor\\frac{n}{27}\\right\\rfloor=\\left\\lfloor\\frac{n}{28}\\right\\rfloor .\n$$\n\nDetermine $A$.",
      "answer": "95004"
    },
    {
      "question": "What are the last two digits of $7^{9^{9^{9}}}$?",
      "answer": "7"
    },
    {
      "question": "2. Determine all natural numbers that are 33 times larger than the sum of their digits.",
      "answer": "594"
    },
    {
      "question": "5.1. (13 points) On the picture, the entrance and exit of the maze are marked with arrows. You can move through it such that on this picture you can only move right, down, or up (you cannot turn around). How many different ways are there to navigate this maze?\n\n---\n\nThe translation maintains the original text's formatting and structure.",
      "answer": "16"
    },
    {
      "question": "Subject (3). Consider the triangle $ABC$ with $\\mathrm{m}(\\angle BAC) = 90^{\\circ}$. The bisector of $\\angle ACB$ intersects the segment $(AB)$ at point $E$. If there exists $D \\in (CE)$ such that $\\mathrm{m}(\\angle DAC) = \\mathrm{m}(\\angle BDE) = x^{\\circ}$, then calculate $x$.\n\nThe statement in the problem is unfortunate. Such a point $D$ always exists, namely $D \\equiv I$, the incenter of $\\triangle ABC$, for which $\\angle IAC = \\angle BIE = 45^{\\circ}$. ${ }^{2}$ Otherwise, by continuity considerations, as $D$ traverses $CE$, from $C$ to $E$, the angle $\\angle DAC$ increases from $0^{\\circ}$ to $90^{\\circ}$, and the angle $\\angle BDE$ increases from $\\frac{1}{2} \\angle C$ to $90^{\\circ} - \\frac{1}{2} \\angle C$. This means that the continuous function $f(D) = \\angle BDE - \\angle DAC$ takes the values $f(C) = \\frac{1}{2} \\angle C > 0$ and $f(E) = -\\frac{1}{2} \\angle C < 0$, so there must exist (at least) one point $D$ for which $f(D) = 0$. If more than one exists, and since the problem asks for a (unique) $x$, the value must be 45. Perhaps this observation should lead to the elegant synthetic solution that follows, where it is demonstrated that $D \\equiv I$ is the only such point. [^1]",
      "answer": "45"
    },
    {
      "question": "2. Calculate the value of the expression $\\sqrt{(\\sqrt{3}-3)^{2}}-(\\sqrt{3}-3)$.",
      "answer": "6-2\\sqrt{3}"
    },
    {
      "question": "7.101. $\\lg 8-\\lg \\sqrt{x+6}=\\lg 16-\\lg (x-2)$",
      "answer": "10"
    },
    {
      "question": "6. In the Cartesian coordinate system $x O y$, the area of the plane region corresponding to the point set $K=\\{(x, y) \\mid(|x|+|3 y|-6)(|3 x|+|y|-6) \\leqslant 0\\}$ is $\\qquad$ .",
      "answer": "24"
    },
    {
      "question": "11. Find the value of\n$$\n\\frac{2011^{2}+2111^{2}-2 \\times 2011 \\times 2111}{25}\n$$",
      "answer": "400"
    },
    {
      "question": "Write the digits $1,2,3,4,5,6,7,8,9$ in some order to form a nine-digit number $\\overline{\\operatorname{abcdefghi}}$. If $A=\\overline{a b c}+\\overline{b c d}+\\overline{c d e}+\\overline{d e f}+\\overline{e f g}+\\overline{f g h}+\\overline{g h i}$, find the maximum possible value of $A$.",
      "answer": "4648"
    },
    {
      "question": "1. Maurits cycles to school every day. He can choose between two different routes. Route B is $1.5 \\mathrm{~km}$ longer than route A, but because he encounters fewer traffic lights along the way, his average speed on route B is $2 \\mathrm{~km} / \\mathrm{u}$ higher than on route A. As a result, his cycling trip via route A takes exactly as long as via route B.\n\nHow long does his cycling trip to school take?",
      "answer": "45"
    },
    {
      "question": "From a barrel, we pour out 4 liters of wine and replace it with 4 liters of water. From the resulting mixture, we again pour out 4 liters and replace it with 4 liters of water. We repeat this process a total of three times, and in the final result, there is 2.5 liters more water than wine. How many liters of wine were originally in the barrel?",
      "answer": "16"
    },
    {
      "question": "3. Find the smallest distance from the point with coordinates $(5 ; 10 ; 13)$ to a point whose coordinates are positive and satisfy the inequality\n\n$$\n(x+y+z)\\left(\\frac{1}{x+y}+\\frac{1}{y+z}+\\frac{1}{z+x}\\right) \\geq 4.5 \\sqrt[4]{1-(2 y+x)^{2}}\n$$\n\nIn your answer, write the square of the found distance.",
      "answer": "115.2"
    },
    {
      "question": "Sonkin $M$.\n\nIn trapezoid $A B C D$ with area 1, the bases $B C$ and $A D$ are in the ratio $1: 2$. Let $K$ be the midpoint of diagonal $A C$. Line $D K$ intersects side $A B$ at point $L$. Find the area of quadrilateral $B C K L$.",
      "answer": "\\frac{2}{9}"
    },
    {
      "question": "Example 5 Color each vertex of a 2003-gon with one of three colors: red, blue, or green, such that adjacent vertices have different colors. How many such colorings are there? ${ }^{[3]}$\n(2002-2003, Hungarian Mathematical Olympiad)",
      "answer": "2^{2003}-2"
    },
    {
      "question": "5.19 It is known that 1982 people attended a gathering, and among any 4 people, at least 1 person knows the other 3. How many people at least know all the attendees at this gathering?",
      "answer": "1979"
    },
    {
      "question": "2. The cells of a $100 \\times 100$ table are painted white. In one move, it is allowed to select any 99 cells from one row or one column and repaint each of them to the opposite color - from white to black, and from black to white. What is the minimum number of moves required to obtain a table with a checkerboard pattern of cells?",
      "answer": "100"
    },
    {
      "question": "## [ Product of lengths of chord segments and lengths of secant segments\n\n Pythagorean Theorem (direct and inverse). [Cyclic quadrilateral with perpendicular diagonals] The diagonals $A C$ and $B D$ of the cyclic quadrilateral $A B C D$ are perpendicular to each other and intersect at point $M$. It is known that $A M=3, B M=4$ and $C M=6$. Find $C D$.\n\n#",
      "answer": "7.5"
    },
    {
      "question": "18. There is a sequence, the first number is 6, the second number is 3, starting from the second number, each number is 5 less than the sum of the number before it and the number after it. What is the sum of the first 200 numbers in this sequence, from the first number to the 200th number?",
      "answer": "999"
    },
    {
      "question": "In the case of Sándor Mátyás, the leader of the patriots sent an equation to his friends:\n\n$$\n2 x^{2}+4 x y+7 y^{2}-12 x-2 y+N=0\n$$\n\nThe secret message was that the uprising should take place on the $N$-th day of the month. The patriots often said that the uprising was the only solution. This also characterized $N$: the equation was satisfied by only one pair of real numbers $(x, y)$. What is the value of $N$?",
      "answer": "23"
    },
    {
      "question": "H4. The points $A, B$ and $C$ are the centres of three faces of a cuboid that meet at a vertex. The lengths of the sides of the triangle $A B C$ are 4,5 and 6 .\nWhat is the volume of the cuboid?",
      "answer": "90\\sqrt{6}"
    },
    {
      "question": "1. Variant 1.\n\nPetya has four cards with digits $1,2,3,4$. Each digit appears exactly once. How many natural numbers greater than 2222 can Petya make from these cards?",
      "answer": "16"
    },
    {
      "question": "4. Points $A, B, C, D$ lie on the circumference of a circle, and $B C=D C=4, A E=6$, the lengths of segments $B E$ and $D E$ are both integers, find the length of $B D$.",
      "answer": "7"
    },
    {
      "question": "1. A palindrome is a number that reads the same from left to right and from right to left. For example, the numbers 353 and $4884-$ are palindromes. It is known that a three-digit number $x$ is a palindrome. To it, 32 was added and a four-digit number was obtained, which is also a palindrome. Find $x$.",
      "answer": "969"
    },
    {
      "question": "[Example 3.6.6] Find all positive integers $n>1$, such that $\\frac{2^{n}+1}{n^{2}}$ is an integer.",
      "answer": "3"
    },
    {
      "question": "## Task B-2.3.\n\nThere are 8 steps to the top of the staircase. In how many ways can we reach the top if we can climb by one step or by two steps?",
      "answer": "34"
    },
    {
      "question": "Problem 3. There were several whole cheese heads on the warehouse. At night, rats came and ate 10 heads, and everyone ate equally. Several rats got stomachaches from overeating. The remaining 7 rats the next night finished off the remaining cheese, but each rat could eat only half as much cheese as the night before. How much cheese was originally on the warehouse? [5 points] (A.D. Blyunkov, I. V. Raskina)",
      "answer": "11"
    },
    {
      "question": "5. Calculate: $1+3 \\frac{1}{6}+5 \\frac{1}{12}+7 \\frac{1}{20}+9 \\frac{1}{30}+11 \\frac{1}{42}+13 \\frac{1}{56}+15 \\frac{1}{72}+17 \\frac{1}{90}$",
      "answer": "81\\frac{2}{5}"
    },
    {
      "question": "6. Convex quadrilateral $A B C D$ satisfies $\\angle C A B=\\angle A D B=30^{\\circ}, \\angle A B D=77^{\\circ}, B C=C D$, and $\\angle B C D=n^{\\circ}$ for some positive integer $n$. Compute $n$.",
      "answer": "68"
    },
    {
      "question": "Question 182, In a group of 9 people, arrange for each person to shake hands with exactly two other people in the group. Two different handshake arrangements are considered distinct if and only if there are at least two people who shake hands in one arrangement but do not shake hands in the other. How many different handshake arrangements are there? (2012 AIME Problem)",
      "answer": "30016"
    },
    {
      "question": "1. Let the set $A=\\{1,2,3, \\cdots, 99\\}, B=\\{2 x \\mid x \\in A\\}, C=\\{x \\mid 2 x \\in A\\}$, then the number of elements in $B \\cap C$ is $\\qquad$ ـ.",
      "answer": "24"
    },
    {
      "question": "$24, 2$ red balls and $11$ white balls are arranged in a row, satisfying the following conditions: the red balls are not adjacent, and in any consecutive $7$ balls, there is at least one red ball, then there are $\\qquad$ ways to arrange them.",
      "answer": "31"
    },
    {
      "question": "5. Two circles touch internally at point $A$. From the center of the larger circle $O$, a radius $O B$ of the larger circle is drawn, which is also tangent to the smaller circle at point $C$. Determine $\\measuredangle B A C$.",
      "answer": "45"
    },
    {
      "question": "139. Moving Digits. A certain number, which is written using fewer than 30 digits, starts (if we move from left to right) with the digits 15, that is, it has the form $15 \\ldots$. If we multiply it by 5, the result can be obtained simply by moving these two digits to the right end; in the end, we get a number of the form ... 15. Find the original number.",
      "answer": "157894736842105263"
    },
    {
      "question": "2. Place nine small balls numbered $1, 2, \\cdots, 9$ into a bag. These balls are identical except for their numbers. Person A draws a ball from the bag, with the number being $a$, and then returns it. Person B then draws a ball from the bag, with the number being $b$. The probability that the inequality $a - 2b + 10 > 0$ holds is $\\qquad$.",
      "answer": "\\frac{61}{81}"
    },
    {
      "question": "Problem 8.1. A square was cut into four equal rectangles, and from them, a large letter P was formed, as shown in the figure, with a perimeter of 56.\n![](https://cdn.mathpix.com/cropped/2024_05_06_6da73bfd3e09e8b55e3fg-27.jpg?height=356&width=720&top_left_y=274&top_left_x=366)\n\nWhat is the perimeter of the original square",
      "answer": "32"
    },
    {
      "question": "5. The natural numbers $1,2, \\ldots, 64$ are written in the cells of an $8 \\times 8$ table such that for all $k=1,2,3, \\ldots, 63$ the numbers $k$ and $k+1$ are in adjacent cells. What is the maximum possible value of the sum of the numbers on the main diagonal?",
      "answer": "432"
    },
    {
      "question": "2. A positive integer $n$ is said to be 'good' if $n^{2}-1$ can be written as the product of three distinct prime numbers. Find the sum of the five smallest 'good' integers.\n(1 mark)\n對於正整數 $n$, 若 $n^{2}-1$ 可寫成三個不同質數之積, 則 $n$ 稱為「好數」。求最小的五個「好數」之和。",
      "answer": "104"
    },
    {
      "question": "Problem 6.3. In a new math textbook, there are a total of 91 problems. Yura started solving them in the mornings, beginning on September 6.\n\nEvery morning, starting from September 7, he solves one fewer problem than the previous morning (until the problems run out).\n\nBy the evening of September 8, Yura realized that there were still 46 problems left to solve in the textbook. On which day of September will he finish solving the textbook?",
      "answer": "12"
    },
    {
      "question": "2. (13 points) A triathlon competitor swam 1 km in the first stage. In the second stage, he cycled 25 km, and in the third stage, he ran 4 km. He completed the entire distance in 1 hour 15 minutes. Before the competition, he tried out the course: he swam for $1 / 16$ hours, cycled and ran for $1 / 49$ hours each, covering a total of 5/4 km. During the competition, he completed each stage at the same speed as during the trial run. How long did he cycle and at what speed?",
      "answer": "\\frac{5}{7}"
    },
    {
      "question": "João's scientific calculator has a special key that transforms any number $x$ displayed on the screen and that is different from 1 into the number $\\frac{1}{1-x}$.\n\na) What happens if the number 2 is displayed on the screen and we press the special key three times?\n\nb) What happens if the number 2 is displayed on the screen and we press the special key ten times?\n\nc) Finally, what happens if the number 2 is displayed on the screen and we press the special key 2015 times?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "$\\begin{array}{ll}{\\left[\\begin{array}{l}\\text { Game Theory (other) }\\end{array}\\right]} \\\\ {\\left[\\begin{array}{l}\\text { Pairings and Groupings; Bijections }\\end{array}\\right]}\\end{array}$\n\nA token is placed on a chessboard. Two players take turns moving the token to an adjacent square. It is forbidden to place the token on a square it has already visited. The player who cannot make a move loses. Who wins with correct play?",
      "answer": "64"
    },
    {
      "question": "2. For a natural number ending not in zero, one of its digits was replaced by zero (if it is the leading digit, it was simply erased). As a result, the number decreased by 9 times. How many numbers exist for which this is possible?",
      "answer": "7"
    },
    {
      "question": "7. What? Where? When? The game wheel in the TV quiz is \n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0312347ad1209bb36d0eg-04.jpg?height=457&width=665&top_left_y=1693&top_left_x=296)\ndivided into 13 identical sectors. The sectors are numbered from 1 to 13. At the beginning of the game, each sector contains an envelope with a question. Players choose a random sector using a spinner with an arrow. If this sector has already been chosen before, there is no envelope in it, and then the next sector clockwise is played. If it is also empty, the next one, and so on, until a non-empty sector is found. Before the break, the players played 6 sectors. a) (from 6th grade, 1 point) What is more likely: that among the played sectors there is sector №1 or that among the played sectors there is sector №8?\n\nb) (from 7th grade, 5 points). Find the probability that as a result, the six sectors played in a row have numbers from №1 to №6.",
      "answer": "0.00348"
    },
    {
      "question": "58. From the 205 positive integers $1,2,3, \\cdots \\cdots, 205$, the maximum number of integers that can be selected such that for any three selected numbers $a, b, c(a<b<c)$, we have $a b \\neq c$ is $\\qquad$.",
      "answer": "193"
    },
    {
      "question": "Calculate\n\n$$\n\\sqrt{32 \\sqrt{16 \\sqrt{8 \\sqrt{4}}}}\n$$\n\nOnly a numerical answer is expected here.",
      "answer": "16"
    },
    {
      "question": "We draw the diagonals of the convex quadrilateral $A B C D$, and then the centroids of the 4 triangles thus formed. What fraction of the area of quadrilateral $A B C D$ is the area of the quadrilateral determined by the 4 centroids?",
      "answer": "\\frac{2}{9}"
    },
    {
      "question": "$3 \\cdot 26$ If real numbers $a, b, x, y$ satisfy $a x+b y=3, a x^{2}+b y^{2}=7, a x^{3}+$ $b y^{3}=16, a x^{4}+b y^{4}=42$, find the value of $a x^{5}+b y^{5}$.",
      "answer": "20"
    },
    {
      "question": "The positive integer $d$ has the property that each of 468,636, and 867 has the same remainder, $r$, when divided by $d$. What is the largest possible value of $d+r$ ?",
      "answer": "27"
    },
    {
      "question": "8 Let $x \\in \\mathbf{R}$, then the minimum value of the function $f(x)=\\sqrt{x^{2}+1}+\\sqrt{(x-12)^{2}+16}$ is $\\qquad$",
      "answer": "13"
    },
    {
      "question": "It is known that in the small lottery, 5 numbers are drawn from 90. If someone bets on 1, 2, 3, 4, or 5 numbers, what is the probability that these numbers will be drawn? In the past, when the small lottery was still in operation, the state paid out 14, 240, 4800, 60000, and 100000 times the stake for estratra (one number), ambora (two numbers), ternora, quaternora, and quinternora, respectively.\n\nWhat percentage of all stakes was the state's revenue?",
      "answer": "99.78"
    },
    {
      "question": "11. (12 points) Person A and Person B start from point A to point B at the same time: A is faster than B. After A reaches point B, A's speed doubles and A immediately returns to point A, meeting B 240 meters away from point B; after B meets A, B's speed also doubles, and B turns around to return; when A returns to point A, B is still 120 meters away from point A. What is the distance between points A and B in meters?",
      "answer": "420"
    },
    {
      "question": "B1. At least how many times must we roll two fair dice simultaneously so that the probability of rolling the same number of pips on both dice at least once is greater than $\\frac{1}{2}$?\n\n(20 points)",
      "answer": "4"
    },
    {
      "question": "Example 4. The mathematical expectations of two independent random variables $X$ and $Y$ are known: $M(X)=4, M(Y)=5$.\n\nFind the mathematical expectation of their product.",
      "answer": "20"
    },
    {
      "question": "Three friends Pankrác, Servác, and Bonifác went on a night walk through a natural labyrinth during their vacation. At the entrance, each received a candle and they set off in different directions. All of them successfully navigated the labyrinth, but each took a different path. In the following square grid, their paths are marked. We know that Pankrác never went south and that Servác never went west. How many meters did Bonifác walk in the labyrinth if Pankrác walked exactly $500 \\mathrm{~m}$?\n\n(M. Petrová)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_0d982c5cbb35b07eed65g-1.jpg?height=665&width=1114&top_left_y=787&top_left_x=471)\n\nIdea. Which path did Bonifác definitely not take?",
      "answer": "625"
    },
    {
      "question": "10.1. Find the sum $\\sin x + \\sin y + \\sin z$, given that $\\sin x = \\tan y$, $\\sin y = \\tan z$, $\\sin z = \\tan x$.",
      "answer": "0"
    },
    {
      "question": "1. Determine the geometric locus of points $M$ in the Cartesian coordinate plane such that the tangents drawn from point $M$ to the parabola $y=x^{2}$ are mutually perpendicular.",
      "answer": "-\\frac{1}{4}"
    },
    {
      "question": "43. As shown in the figure, two satellites $\\mathrm{A}, \\mathrm{B}$ are both orbiting the Earth's center in a counterclockwise direction at constant speeds. It is known that the time ratio for $\\mathrm{A}$ and $\\mathrm{B}$ to complete one orbit is 1:5. Starting from the positions shown in the figure, during the time it takes for $\\mathrm{B}$ to complete one orbit, satellites $\\mathrm{A}$ and $\\mathrm{B}$ and the Earth's center are collinear $\\qquad$ times.",
      "answer": "8"
    },
    {
      "question": "1.047. $\\left(\\frac{\\sqrt{561^{2}-459^{2}}}{4 \\frac{2}{7} \\cdot 0.15+4 \\frac{2}{7}: \\frac{20}{3}}+4 \\sqrt{10}\\right): \\frac{1}{3} \\sqrt{40}$.",
      "answer": "125"
    },
    {
      "question": "12. Find the remainder of the division of $67^{283}$ by 13.",
      "answer": "11"
    },
    {
      "question": "5. The maximum value of the orthogonal projection area of a regular tetrahedron with edge length 1 on a horizontal plane is $\\qquad$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "8. Suppose when repeatedly tossing a fair coin, the probability of getting five consecutive heads before two consecutive tails is $p$. Then the value of $p$ is $\\qquad$ .",
      "answer": "\\frac{3}{34}"
    },
    {
      "question": "7. There are two strips of paper, one 21 cm long and the other 13 cm long. After cutting off the same length from both strips, it is found that the remaining length of the shorter strip is thirteen-eighths of the remaining length of the longer strip. How long was the piece that was cut off?",
      "answer": "0.2"
    },
    {
      "question": "10. The recurring decimal $0 . \\dot{x} y \\dot{z}$, where $x, y, z$ denote digits between 0 and 9 inclusive, is converted to a fraction in lowest term. How many different possible values may the numerator take?\n(1 mark)\n若把循環小數 $0 \\dot{x} y \\dot{z}$ （其中 $x 、 y 、 z$ 代表 0 至 9 之間的數字，包括 0 和 9）化戌最簡分數，分子有多少個不同的可能値？",
      "answer": "660"
    },
    {
      "question": "A 100-row and 100-column table has all 1s in its first column, and in the $k$-th row, there is an arithmetic sequence with a difference of $k$. Among the numbers located along the diagonal stretching from the bottom left to the top right of the table, which is the largest?",
      "answer": "2501"
    },
    {
      "question": "## Problem B2\n\n$\\mathrm{P}_{1}, \\mathrm{P}_{2}, \\ldots, \\mathrm{P}_{\\mathrm{n}}$ are points in the plane and $\\mathrm{r}_{1}, \\mathrm{r}_{2}, \\ldots, \\mathrm{r}_{\\mathrm{n}}$ are real numbers such that the distance between $P_{i}$ and $P_{j}$ is $r_{i}+r_{j}$ (for $i$ not equal to $j$ ). Find the largest $n$ for which this is possible.",
      "answer": "4"
    },
    {
      "question": "Let's calculate as simply as possible the value of\n\n$$\na^{3}+b^{3}+3\\left(a^{3} b+a b^{3}\\right)+6\\left(a^{3} b^{2}+a^{2} b^{3}\\right)\n$$\n\nif $a+b=1$.",
      "answer": "1"
    },
    {
      "question": "3. Find all natural numbers $n$ for which $2^{n}+n^{2016}$ is a prime number.",
      "answer": "1"
    },
    {
      "question": "Task B-3.4. In the plane, 100 concentric circles are drawn with radii $1 \\mathrm{~cm}, 2 \\mathrm{~cm}, 3 \\mathrm{~cm}, \\ldots, 100 \\mathrm{~cm}$. The circle with a radius of $1 \\mathrm{~cm}$ is painted red, and all other areas bounded by consecutive circles are painted either red or green, but in such a way that adjacent areas are painted different colors. Determine the ratio of the area of all green regions to the area of all red regions.",
      "answer": "\\frac{101}{99}"
    },
    {
      "question": "For each positive integer $n$, the expression $1+2+3+\\cdots+(n-1)+n$ represents the sum of all of the integers from 1 to $n$ inclusive. What integer is equal to\n\n$$\n(1+2+3+\\cdots+2020+2021)-(1+2+3+\\cdots+2018+2019) ?\n$$",
      "answer": "4041"
    },
    {
      "question": "2. (1985 National High School League Question) A football invitational tournament involves sixteen cities, each sending two teams, Team A and Team B. According to the competition rules, each pair of teams plays at most one match, and teams from the same city do not play against each other. After several days of the tournament, a statistical review found that, except for Team A from City A, all other teams have played a different number of matches. How many matches has Team B from City A played? Please prove your conclusion.",
      "answer": "15"
    },
    {
      "question": "Four points lying on one circle Auxiliary circle\n\nIn a convex quadrilateral $A B C D$, it is known that $\\angle B C D=80^{\\circ}, \\angle A C B=50^{\\circ}$ and $\\angle A B D=30^{\\circ}$. Find the angle $A D B$.",
      "answer": "50"
    },
    {
      "question": "2. In the laboratory, there are flasks of two sizes (volume $V$ and volume $V / 2$) in a total of 100 pieces, with at least three flasks of each size. The lab assistant randomly selects three flasks in sequence, and fills the first one with an 80% salt solution, the second one with a 50% salt solution, and the third one with a 20% salt solution. Then he pours the contents of these three flasks into one dish and determines the percentage of salt in it. For what minimum number of large flasks $N$ will the event \"the percentage of salt in the dish is between $45\\%$ and $55\\%$ inclusive\" occur less frequently than the event \"when two fair coins are tossed, one head and one tail appear (in any order)\"? Justify your answer. (16 points)",
      "answer": "46"
    },
    {
      "question": "[ Limitedness, monotonicity]\n\nGiven a strictly increasing function \\$f \\colon \\mathbb{\\mathrm{N}}_0 \\to \\mathbb{\\mathrm{N}}_0\\$ (where $\\mathbb{\\mathrm{N}}_0$ is the set of non-negative integers), which satisfies the relation $\\mathrm{f}(\\mathrm{n}+\\mathrm{f}(\\mathrm{m}))=\\mathrm{f}(\\mathrm{n})+\\mathrm{m}+1$ for any $m,n \\in \\mathbb{\\mathrm{N}}_0$. Find all possible values that \\$f(2023)\\$ can take.",
      "answer": "2024"
    },
    {
      "question": "F5 (20-3, UK) Let $f, g: \\mathbf{N}^{*} \\rightarrow \\mathbf{N}^{*}$ be strictly increasing functions, and\n$$\nf\\left(\\mathbf{N}^{*}\\right) \\cup g\\left(\\mathbf{N}^{*}\\right)=\\mathbf{N}^{*}, f\\left(\\mathbf{N}^{*}\\right) \\cap g\\left(\\mathbf{N}^{*}\\right)=\\varnothing, g(n)=f(f(n))+1 .\n$$\n\nFind $f(240)$.",
      "answer": "388"
    },
    {
      "question": "10. (20 points) Find the smallest value of the parameter $c$ such that the system of equations has a unique solution\n\n$$\n\\left\\{\\begin{array}{l}\n8(x+7)^{4}+(y-4)^{4}=c \\\\\n(x+4)^{4}+8(y-7)^{4}=c\n\\end{array}\\right.\n$$",
      "answer": "24"
    },
    {
      "question": "5. The subset $X$ of the set $\\{00,01, \\cdots, 98,99\\}$ satisfies: in any infinite sequence of digits, there are two adjacent digits that form an element of $X$. What is the minimum number of elements that $X$ should contain?\n（52nd Moscow Olympiad problem）",
      "answer": "55"
    },
    {
      "question": "1.1. Find the maximum value of the function $\\sin ^{13} x+\\cos ^{14} x$.",
      "answer": "1"
    },
    {
      "question": "15. If $a=1.69, b=1.73$ and $c=0.48$, find the value of\n$$\n\\frac{1}{a^{2}-a c-a b+b c}+\\frac{2}{b^{2}-a b-b c+a c}+\\frac{1}{c^{2}-a c-b c+a b}\n$$",
      "answer": "20"
    },
    {
      "question": "3. Let $g(x)=\\frac{2}{x^{2}-8x+17}$. Find all possible values of the parameter $a$ such that the inequality holds for all real $x$. In your answer, write the difference between the largest and smallest possible values of the parameter $a$.\n\n(6 points)",
      "answer": "2.1"
    },
    {
      "question": "## Task 4 - 250514\n\nIn each of the figures a), b), c), the numbers 1, 2, 3, 4, 5, 6, 7, 8, 9 are to be entered into the circles.\n\nEach of these numbers should (in each such entry) occur exactly once. For some circles, the number to be entered is already specified. Furthermore, for each entry, the following should apply:\n\nIf you add the four numbers on each side of the triangle, the sum should be the same for each of the three sides.\n\na) Find an entry in Figure a) where the sum for each of the three sides is 17!\n\nb) Find as many entries as possible in Figure b) where the sum for each of the three sides is 21!\n\nc) Find as many entries as possible in Figure c) where the sum for each of the three sides is the same! For each of these entries, give this value!\n\na)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0108.jpg?height=314&width=369&top_left_y=865&top_left_x=455)\nb)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0108.jpg?height=320&width=357&top_left_y=862&top_left_x=878)\nc)\n\n![](https://cdn.mathpix.com/cropped/2024_06_06_b85a894aa8dbf722a7b3g-0108.jpg?height=323&width=351&top_left_y=861&top_left_x=1292)",
      "answer": "20"
    },
    {
      "question": "## SUBJECT III\n\nLet $O$ and $O^{\\prime}$ be the centers of the faces $A B C D$ and $B C C B^{\\prime}$ of the cube $A B C D A^{\\prime} B^{\\prime} C^{\\prime} D^{\\prime}, A B=a$.\n\na) Find the measure of the angle between the lines $D^{\\prime} O$ and $A^{\\prime} O^{\\prime}$.\n\nb) Calculate the distance from the point $O^{\\prime}$ to the plane ($D^{\\prime} A C$).\n\n(prof. Bunu Iulian, Liceul de Arte Baia Mare)",
      "answer": "\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "1. How many tiles with dimensions $15 \\times 15$ cm are needed to tile a wall that is 3 m 6 dm long and 27 dm wide $?$",
      "answer": "432"
    },
    {
      "question": "# Problem 5.\n\nNumbers $a_{1}, b_{1}, a_{2}, b_{2}, \\ldots, a_{n}, b_{n}$ satisfy the conditions\n\n$$\n\\begin{gathered}\na_{1}^{2}+a_{2}^{2}+\\cdots+a_{n}^{2}=2018^{2}, \\quad b_{1}^{2}+b_{2}^{2}+\\cdots+b_{n}^{2}=2017^{2} \\\\\na_{1} b_{1}+a_{2} b_{2}+\\cdots+a_{n} b_{n}=2017 \\cdot 2018\n\\end{gathered}\n$$\n\nFind the ratios $a_{1} / b_{1}, a_{2} / b_{2}, \\ldots, a_{n} / b_{n}$.",
      "answer": "\\frac{2018}{2017}"
    },
    {
      "question": "4. What is the greatest length that a closed, non-self-intersecting broken line can have, which runs along the grid lines of a $6 \\times 10$ cell field?",
      "answer": "76"
    },
    {
      "question": "3-5. There is a piece of chain consisting of 60 links, each weighing 1 g. What is the smallest number of links that need to be unbuckled so that from the resulting parts, all weights of 1 g, 2 g, 3 g, \\(\\ldots, 60\\) g can be formed (an unbuckled link also weighs 1 g)",
      "answer": "3"
    },
    {
      "question": "A Tim number is a five-digit positive integer with the property that it is a multiple of 15 , its hundreds digit is 3, and its tens digit is equal to the sum of its first (leftmost) three digits. How many Tim numbers are there?",
      "answer": "16"
    },
    {
      "question": "3. Nils has a goose farm. Nils calculated that if he sells 75 geese, the feed will last 20 days longer than if he doesn't sell any. If he buys an additional 100 geese, the feed will run out 15 days earlier than if he doesn't make such a purchase. How many geese does Nils have?",
      "answer": "300"
    },
    {
      "question": "14.24. How many different integer solutions does the inequality $|x|+|y|<100$ have?",
      "answer": "338350"
    },
    {
      "question": "7. From the set $\\{1,2, \\cdots, 100\\}$, randomly select elements $m$ and $n$ (which can be the same). Then the probability that the unit digit of $2^{m}+3^{n}$ is 3 is $\\qquad$ .",
      "answer": "\\frac{3}{16}"
    },
    {
      "question": "7. 14 A fair coin is tossed ten times, let $\\frac{i}{j}$ be the probability that heads do not appear consecutively, where $i$ and $j$ are coprime, find $i+j$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "73"
    },
    {
      "question": "Example 3 Find all pairs $(x, y)$ that satisfy the system of equations $\\left\\{\\begin{array}{l}\\frac{\\cos x}{\\cos y}=2 \\cos ^{2} y \\\\ \\frac{\\sin x}{\\sin y}=2 \\sin ^{2} y\\end{array}\\right.$, where $x$, $y \\in\\left(0, \\frac{\\pi}{2}\\right)$.",
      "answer": "\\frac{\\pi}{4}"
    },
    {
      "question": "3. For two numbers $a, b$, define a new operation: $a * b=a \\times b+a+b$, then $1 * 2+2 * 3=$",
      "answer": "16"
    },
    {
      "question": "If $3^{n}=2$ then what is the value of $27^{2 n}$?",
      "answer": "64"
    },
    {
      "question": "1. A game participant was offered 30 questions. For each correct answer, they were awarded 7 points, and for an incorrect answer (or no answer), 12 points were deducted. How many correct answers did the participant give if they scored 77 points?",
      "answer": "23"
    },
    {
      "question": "1. Given $a, b, c$ are the side lengths of $\\triangle ABC$, satisfying $a^{2}+b^{2}=2018 c^{2}$. Then the value of $(\\cot A+\\cot B) \\tan C$ is $\\qquad$",
      "answer": "\\frac{2}{2017}"
    },
    {
      "question": "2. What is the smallest area that a figure on the plane $Oxy$ can have, located between the lines $x=-26$ and $x=2$, bounded below by the $x$-axis, and above by the tangent to the graph of the function $y=5+\\sqrt{4-x}$ at the point of tangency with abscissa $x_{0}$, where $x_{0}$ lies in the interval $-26 \\leq x_{0} \\leq 2$?\n\n(12 points)",
      "answer": "504"
    },
    {
      "question": "A bus, a train, and a plane depart from city A to city B at the same time. If I take the bus, which has an average speed of 100 $\\mathrm{km} / \\mathrm{h}$, I will arrive in city B at 8 PM. If I take the train, which has an average speed of $300 \\mathrm{~km} / \\mathrm{h}$, I will arrive in city B at 2 PM. What will be the arrival time of the plane if its average speed is $900 \\mathrm{~km} / \\mathrm{h}$?",
      "answer": "12"
    },
    {
      "question": "4. (7 points) The numbers $a, b, c, d$ belong to the interval $[-12.5,12.5]$. Find the maximum value of the expression $a+2 b+c+2 d-a b-b c-c d-d a$.",
      "answer": "650"
    },
    {
      "question": "9.1. (12 points) Two circles touch each other externally at point $K$. On their common internal tangent, point $P$ is marked such that $K P=14$. Through point $P$, two secants are drawn to the circles, with one of them intercepting a chord $A B=45$ on the first circle, and the other intercepting a chord $C D=21$ on the second circle, with point $A$ lying between points $B$ and $P$, and point $C$ lying between points $D$ and $P$. Find the ratio $B C: A D$.",
      "answer": "1.75"
    },
    {
      "question": "16. (3 points) Wang Lei and her sister walk from home to the gym to play badminton. It is known that her sister walks 20 meters more per minute than Wang Lei. After 25 minutes, her sister arrives at the gym. At this point, her sister realizes she forgot the racket and immediately returns home along the same route to get it. She meets Wang Lei 300 meters away from the gym. What is the distance from Wang Lei's home to the gym in meters? $\\qquad$ meters.",
      "answer": "1500"
    },
    {
      "question": "1. Given non-zero real numbers $x, y$ satisfy\n$$\n\\begin{array}{l}\n(5 x+y)^{2019}+x^{2019}+30 x+5 y=0 . \\\\\n\\text { Then } \\frac{x}{y}=\n\\end{array}\n$$",
      "answer": "-\\frac{1}{6}"
    },
    {
      "question": "2. In quadrilateral $ABCD$, the diagonals intersect at point $P$, diagonal $AC$ is divided into four equal segments by points $P, N$, and $M$ $(|AP|=|PN|=|NM|=|MC|)$, and diagonal $BD$ is divided into four equal segments by points $L, K$, and $P$ $(|BL|=|LK|=|KP|=|PD|)$. Determine the ratio of the area of quadrilateral KLMN to the area of quadrilateral ABCD.",
      "answer": "\\frac{3}{16}"
    },
    {
      "question": "Example 22 (Mathematical Problem 1486 from \"Mathematical Bulletin\") Find the maximum value of the function $f(x)=\\frac{x(1-x)}{(x+1)(x+2)(2x+1)}$, $x \\in (0,1]$.",
      "answer": "\\frac{1}{3}(8\\sqrt{2}-5\\sqrt{5})"
    },
    {
      "question": "# Task 3. (8 points)\n\nIn the Sidorov family, there are 3 people: the father works as a programmer with an hourly rate of 1500 rubles. The mother works as a hairdresser at home and charges 1200 rubles per haircut, which takes her 1.5 hours. The son tutors in mathematics and earns 450 rubles per academic hour (45 minutes). Every day, the family needs to prepare food, walk the dog, and go to the store. The mother spends 2 hours preparing food, the father 1 hour, and the son 4 hours. Walking the dog takes 1 hour for any family member. A trip to the store takes 1 hour for the father, 2 hours for the mother, and 3 hours for the son. In addition, each family member sleeps for 8 hours and spends another 8 hours on rest and personal matters.\n\nWhat is the maximum amount of money the family can earn in one day?",
      "answer": "19600"
    },
    {
      "question": "If $n$ is a natural number, we denote by $n!$ the product of all integers from 1 to $n$. For example: $5! = 1 \\times 2 \\times 3 \\times 4 \\times 5$ and 13! $=$ $1 \\times 2 \\times 3 \\times 4 \\times 5 \\times \\ldots \\times 12 \\times 13$. By convention, $0! = 1$. Find three different integers $a, b$, and $c$, between 0 and 9 such that the three-digit number $abc$ is equal to $a! + b! + c!$.",
      "answer": "145"
    },
    {
      "question": "G1.3 Let $x, y$ be non-zero real numbers. If $x$ is $250 \\%$ of $y$ and $2 y$ is $c \\%$ of $x$, find the value of $c$.",
      "answer": "80"
    },
    {
      "question": "Let $P$ be an arbitrary interior point of the equilateral triangle $ABC$. Let the feet of the perpendiculars from $P$ to the sides $BC$, $CA$, and $AB$ be $D$, $E$, and $F$ respectively. Determine the value of the ratio\n\n$$\n\\frac{P D+P E+P F}{B D+C E+A F}\n$$",
      "answer": "\\frac{\\sqrt{3}}{3}"
    },
    {
      "question": "2.1. The numbers 7, 8, 9, 10, 11 are arranged in a row in some order. It turned out that the sum of the first three of them is 26, and the sum of the last three is 30. Determine the number standing in the middle.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_6202285ed0b3c14598f3g-1.jpg?height=254&width=765&top_left_y=1169&top_left_x=154)",
      "answer": "11"
    },
    {
      "question": "For example, let the function $f(x) = ax^2 + 8x + 3$ where $a < 0$. For a given negative number $a$, there is a maximum positive number $l(a)$ such that the inequality $|f(x)| \\leq 5$ holds for the entire interval $[0, l(a)]$. Question: For what value of $a$ is $l(a)$ maximized? Find this maximum $l(a)$ and prove your conclusion.\n(1998 National High School Mathematics Competition Problem)",
      "answer": "\\frac{\\sqrt{5}+1}{2}"
    },
    {
      "question": "6. If in the expression $9 \\times 8 \\times 7 \\times 6 \\times 5 \\times 4 \\times 3 \\times 2 \\times 1$ some of the “ $\\times$ ” are changed to “ $\\div$ ” so that the final result is still a natural number, denoted as $N$, then the smallest $N$ is $\\qquad$ .",
      "answer": "70"
    },
    {
      "question": "1. Given the set $S=\\{1,2,3, \\cdots, 2000,2001\\}$, where a subset $T$ has the property that for any three elements $x, y, z$ in $T$, $x+y \\neq z$, what is the maximum number of elements in $T$? Maximum value",
      "answer": "1001"
    },
    {
      "question": "10. Given a regular 2017-gon inscribed in a unit circle, let $S$ be the set of distances between any two distinct vertices of the 2017-gon, and $Q=\\prod_{s \\in S} s$. Then $Q^{2}=$ $\\qquad$",
      "answer": "2017"
    },
    {
      "question": "6. A circle with center $O_{1}$ and radius 2, inscribed in triangle $A B C$, touches side $B C$ at point $D$. A second circle with center $O_{2}$ and radius 4 touches the extensions of sides $A B$ and $A C$, as well as side $B C$ at point $E$. Find the area of quadrilateral $O_{1} D O_{2} E$, if the measure of angle $A C B$ is $120^{\\circ}$. (20 points)",
      "answer": "\\frac{30}{\\sqrt{3}}"
    },
    {
      "question": "64 The base of the triangular pyramid $S-ABC$ is an equilateral triangle $ABC$, with a side length of 4. It is also known that $AS=BS=\\sqrt{19}$, and $CS=3$. Find the surface area of the circumscribed sphere of this triangular pyramid.",
      "answer": "4\\pi\\sqrt{\\frac{61}{11}}"
    },
    {
      "question": "1. [20] For how many positive integers $n \\leq 1000$ does the equation in real numbers\n$$\nx^{\\lfloor x\\rfloor}=n\n$$\nhave a positive solution for $x$ ? (For a real number $x,\\lfloor x\\rfloor$ denotes the largest integer that is not greater than $x$.)",
      "answer": "412"
    },
    {
      "question": "19. On the number line, points $A, B, C, D$ were marked. It is known that the first three points correspond to the numbers 5, 8, and -10. What number corresponds to point $D$, if when the direction of the number line is reversed, the sum of the four numbers corresponding to these points does not change?",
      "answer": "-3"
    },
    {
      "question": "B4 In the following diagram, two lines that meet at a point $A$ are tangent to a circle at points $B$ and $C$. The line parallel to $A C$ passing through $B$ meets the circle again at $D$. Join the segments $C D$ and $A D$. Suppose $A B=49$ and $C D=28$. Determine the length of $A D$.",
      "answer": "63"
    },
    {
      "question": "11. The Clock Hands Problem. At what time after 12:00 will the minute hand first catch up with the hour hand?",
      "answer": "1"
    },
    {
      "question": "In the game \"What? Where? When?\", 13 envelopes with questions from the audience are played. The envelopes are chosen in a random order using a spinner. If the Experts answer correctly, they earn a point; if incorrectly, one point goes to the audience. The game ends as soon as one of the teams scores 6 points.\n\nAssume that the teams of Experts and Audience are equally matched.\n\na) Find the expected number of points scored by the Experts team in 100 games.\n\nb) Find the probability that in the next game, envelope No. 5 will be played.",
      "answer": "0.715"
    },
    {
      "question": "Problem 8.2. In the chat of students from one of the schools, a poll was being held: \"On which day to hold the disco: October 22 or October 29?\"\n\nThe graph shows how the votes were distributed an hour after the start of the poll.\n\nThen, 80 more people participated in the poll, voting only for October 22. After that, the poll ended. The final distribution of votes is also shown on the graph.\n\nHow many people participated in the poll?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f1bf0225c3b69484645g-24.jpg?height=494&width=747&top_left_y=1295&top_left_x=339)",
      "answer": "260"
    },
    {
      "question": "G9.3 If $S=a b+a-b-1$ and $a=101, b=49$, find $S$.",
      "answer": "5000"
    },
    {
      "question": "\\section*{Exercise 5 - 031025}\n\nBy which numbers is the product of three arbitrary, but consecutive positive integers divisible, whose sum is odd?",
      "answer": "24"
    },
    {
      "question": "5. (8 points) On the radius $A O$ of a circle with center $O$, a point $M$ is chosen. On one side of $A O$ on the circle, points $B$ and $C$ are chosen such that $\\angle A M B = \\angle O M C = \\alpha$. Find the length of $B C$ if the radius of the circle is $10$, and $\\cos \\alpha = \\frac{4}{5}$?",
      "answer": "16"
    },
    {
      "question": "[ Ratio of areas of triangles with a common angle ]\n\nOn the sides $A B, B C$ and $A C$ of triangle $A B C$, whose area is 75, points $M, N$ and $K$ are located respectively. It is known that $M$ is the midpoint of $A B$, the area of triangle $B M N$ is 15, and the area of triangle $A M K$ is 25. Find the area of triangle $C N K$.\n\n#",
      "answer": "15"
    },
    {
      "question": "6. If the centroid of the inscribed $\\triangle A B C$ of the curve $y^{2}=4 \\sqrt{2} x$ is its focus $F$. Then\n$$\n|F A|^{2}+|F B|^{2}+|F C|^{2}=\n$$\n$\\qquad$",
      "answer": "27"
    },
    {
      "question": "4. In an acute-angled triangle $A B C$, the altitudes $A A_{1}, B B_{1}$, and $C C_{1}$ are dropped. A point $T$ is chosen on the plane such that the lines $T A$ and $T B$ are tangents to the circumcircle of triangle $A B C$, and point $O$ is the center of this circle. The perpendicular dropped from point $T$ to the line $A_{1} B_{1}$ intersects the line $C C_{1}$ at point $K$, and the line passing through point $C_{1}$ parallel to $O K$ intersects the segment $C O$ at point $L$. Find the angle $\\angle C L A_{1}$.",
      "answer": "90"
    },
    {
      "question": "6. Thirteen identical metal rods are connected as follows (see fig.). It is known that the resistance of one rod \\( R_{0}=10 \\) Ohms. Determine the resistance of the entire structure if it is connected to a current source at points \\( A \\) and \\( B \\). (10 points)\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_84fc2054d2075e5d2c9fg-05.jpg?height=537&width=945&top_left_y=154&top_left_x=584)",
      "answer": "4"
    },
    {
      "question": "[ Monotonicity, boundedness $\\quad$ ]\n\nFind the maximum value of the expression $a+b+c+d-ab-bc-cd-da$, if each of the numbers $a, b, c$ and $d$ belongs to the interval $[0,1]$.\n\n#",
      "answer": "2"
    },
    {
      "question": "4. (4 points) Oleg usually arrives on a business trip by the 11 o'clock train. A car arrives at the station for this train. This time, the train arrived an hour earlier, and Oleg started walking towards the car. Meeting the car on the way, he got in, and as a result, arrived 10 minutes earlier than planned. Determine the time Oleg met the car.",
      "answer": "10"
    },
    {
      "question": "6. Given that $D$ is the intersection of the tangents to the circumcircle of $\\triangle A B C$ at $A$ and $B$, the circumcircle of $\\triangle A B D$ intersects line $A C$ and segment $B C$ at another point $E$ and $F$ respectively, and $C D$ intersects $B E$ at point $G$. If $\\frac{B C}{B F}=2$, find $\\frac{B G}{G E}$.",
      "answer": "2"
    },
    {
      "question": "1. (6 points) Calculate: $2017 \\times \\frac{2015}{2016}+\\frac{1}{2016}=$",
      "answer": "2016"
    },
    {
      "question": "577. a) From a sample of size $n$, extracted from a normal general population with a known standard deviation $\\sigma$, the sample mean $\\bar{x}$ has been found. At a significance level $\\alpha$, it is required to find the power function of the test criterion for testing the null hypothesis $H_{0}: a=a_{0}$ about the equality of the general mean $a$ to the hypothetical value $a_{0}$ under the alternative hypothesis $H_{1}: a=a_{1} \\neq a_{0}$.\n\n**Solution. The alternative hypothesis has the form** $a \\neq a_{0}$, therefore the critical region is two-sided. Using rule 1, we find the critical point $u_{\\text {cr }}$ from the equality $\\Phi\\left(u_{\\text {cr }}\\right)=(1-\\alpha) / 2$. Consequently, the two-sided critical region is defined by the inequality $|U|>u_{\\mathrm{cr}}$, or in more detail\n\n$$\n\\left|\\frac{\\left(\\bar{x}-a_{0}\\right)}{\\sigma / \\sqrt{n}}\\right|>u_{\\mathrm{cr}}\n$$\n\nWe will find the power of the considered criterion, i.e., the probability of the criterion falling into the critical region assuming that the alternative hypothesis $a=a_{1} \\neq a_{0}$ is true:\n\n$$\n1-\\beta=P\\left(\\left|\\frac{x-a_{0}}{\\sigma / \\sqrt{n}}\\right|>u_{\\mathrm{cr}} ; \\quad a=a_{1}\\right)\n$$\n\nTransform the expression under the modulus sign:\n\n$$\n\\frac{\\bar{x}-a_{0}}{\\sigma / \\sqrt{n}}=\\frac{\\bar{x}-a_{1}}{\\sigma / \\sqrt{n}}+\\frac{a_{1}-a_{0}}{\\sigma / \\sqrt{n}}=b+\\lambda\n$$\n\nwhere $b=\\frac{\\bar{x}-a_{1}}{\\sigma / \\sqrt{n}}, \\lambda=\\frac{a_{1}-a_{0}}{\\sigma / \\sqrt{n}}$. Using these relations, we get\n\n$$\n\\begin{gathered}\n1-\\beta=P\\left(|b+\\lambda|>u_{\\mathrm{cr}}\\right)=P\\left(b+\\lambda>u_{\\mathrm{cr}}\\right)+P\\left(b+\\lambda<-u_{\\mathrm{cr}}\\right)= \\\\\n=P\\left(b>u_{\\mathrm{cr}}-\\lambda\\right)+P\\left(b<-u_{\\mathrm{cr}}-\\lambda\\right)= \\\\\n=\\left[1-P\\left(b<u_{\\mathrm{cr}}-\\lambda\\right)\\right]+P\\left(b<-u_{\\mathrm{cr}}-\\lambda\\right)= \\\\\n=\\left[1-\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)\\right]+\\Phi\\left(-u_{\\mathrm{cr}}-\\lambda\\right)=1-\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)-\\Phi\\left(u_{\\mathrm{cr}}+\\lambda\\right) .\n\\end{gathered}\n$$\n\nThus, the power of the two-sided criterion at $a=a_{1}$ is\n\n$$\n1-\\beta=1-\\left[\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)+\\Phi\\left(u_{\\mathrm{cr}}+\\lambda\\right)\\right]\n$$\n\nwhere $\\lambda=\\left(a_{1}-a_{0}\\right) \\sqrt{n} / \\sigma$.\n\nEach value of $a_{1}$ corresponds to a certain power value, so the power of the criterion is a function of $a_{1}$; denote it by $\\pi_{2}\\left(a_{1}\\right)$.\n\nThus, the sought power of the two-sided criterion is\n\n$$\n\\pi_{2}\\left(a_{1}\\right)=1-\\left[\\Phi\\left(u_{\\mathrm{cr}}-\\lambda\\right)+\\Phi\\left(u_{\\mathrm{cr}}+\\lambda\\right)\\right]\n$$\n\nwhere $\\Phi(x)$ is the Laplace function, $\\lambda=\\left(a_{1}-a_{0}\\right) \\sqrt{n} / \\sigma, u_{\\text {cr }}$ is found from the equality $\\Phi\\left(u_{\\text {cr }}\\right)=(1-\\alpha) / 2$.\n\nb) From a sample of size $n=16$, extracted from a normal general population with a known standard deviation $\\sigma=5$, at a significance level of 0.05, the null hypothesis $H_{0}: a=a_{0}=20$ about the equality of the general mean $a$ to the hypothetical value $a_{0}=20$ is tested under the alternative hypothesis $H_{1}: a \\neq 20$. Find the power of the two-sided criterion for testing the considered hypothesis for the hypothetical value of the general mean $a_{1}=24$.",
      "answer": "0.8925"
    },
    {
      "question": "30. How many zeros does the product of all integers from 1 to 30 inclusive end with?\n\n## Problems for the sixth grade",
      "answer": "7"
    },
    {
      "question": "1. Foma and Yerema were walking in the same direction along a road with kilometer markers. In one hour, Foma passed by five markers, while Yerema passed by six. Could Foma's speed have been greater than Yerema's?",
      "answer": "5200"
    },
    {
      "question": "29th IMO 1988 shortlist Problem 20 Find the smallest n such that if {1, 2, ... , n} is divided into two disjoint subsets then we can always find three distinct numbers a, b, c in the same subset with ab = c.",
      "answer": "96"
    },
    {
      "question": "4. How many elements does the multiplicative group of 2x2 matrices with elements in residue classes modulo 3 have? What about the subgroup of matrices that have a determinant of 1 modulo 3?\n\nSupplement with exercises G.M.1/2014\n\nNOTE: All subjects are mandatory.\n\nEach subject is graded from 0 to 7 points.\n\nWorking time 3 hours.\n\nEach subject should be written on a separate sheet.",
      "answer": "24"
    },
    {
      "question": "7. (6 points) The number in the parentheses that satisfies the equation below is $\\qquad$\n$$\n\\frac{\\frac{(}{128}+1 \\frac{2}{7}}{5-4 \\frac{2}{21} \\times 0.75} \\div \\frac{\\frac{1}{3}+\\frac{5}{7} \\times 1.4}{\\left(4-2 \\frac{2}{3}\\right) \\times 3}=4.5\n$$",
      "answer": "\\frac{1440}{7}"
    },
    {
      "question": "[The product of the lengths of the segments of chords and the lengths of the segments of secants] [Properties and characteristics of an isosceles triangle.] [Thales' theorem and the theorem of proportional segments]\n\nOn the extension of side $A D$ of rhombus $A B C D$ beyond point $D$, point $K$ is taken. Lines $A C$ and $B K$ intersect at point $Q$. It is known that $A K=14$ and that points $A, B$ and $Q$ lie on a circle with radius 6, the center of which lies on segment $A A$. Find $B K$.",
      "answer": "7"
    },
    {
      "question": "Determine the number of convex polygons all of whose sides are the square roots of positive integers which can be inscribed in a unit circle. Polygons that are rotations or reflections of each other are considered the same.",
      "answer": "14"
    },
    {
      "question": "6. Let the side length of a regular $n$-sided polygon be $a$, and the longest and shortest diagonals be $b$ and $c$ respectively. If $a=b-c$, then $n=$ $\\qquad$",
      "answer": "9"
    },
    {
      "question": "## 2. Rectangles\n\nCalculate the sum of all possible different perimeters that rectangles with an area of 144 and whose side lengths are natural numbers can have.\n\nResult: $\\quad 830$",
      "answer": "830"
    },
    {
      "question": "1. We will divide the integers from 1 to 9 into three groups of three and then multiply the numbers within each group.\n\na) Determine these three products, given that two of them are equal and smaller than the third product.\n\nb) Assume that one of the three products, denoted as $S$, is smaller than the other two products (which may be the same). Find the largest possible value of $S$.",
      "answer": "70"
    },
    {
      "question": "Problem 5.3. Petya and Vasya were playing with toy soldiers. Petya arranged his knights in a \"rectangle\" - some number of columns and some number of rows. When all the knights from the first and second rows went on reconnaissance, 24 knights remained. Then Vasya's archers drove away all the knights who were left in the first and second columns. After this, 18 knights remained. How many knights did Petya have initially?",
      "answer": "40"
    },
    {
      "question": "2. If the three points\n$$\n\\begin{array}{l}\n(1, a, b) \\\\\n(a, 2, b) \\\\\n(a, b, 3)\n\\end{array}\n$$\nare collinear (in 3-space), what is the value of $a+b$ ?",
      "answer": "4"
    },
    {
      "question": "5. (8 points) Fill in the blanks with $1-6$, so that the numbers in each row and each column are not repeated. The two cells occupied by the same symbol in the figure have the same number combination, but the order is uncertain. Therefore, the five-digit number formed by the first five numbers from left to right in the last row is\n\n保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "46123"
    },
    {
      "question": "# Problem 2.\n\nIn a football tournament, each team is supposed to play one match against each of the others. But during the tournament, half of all the teams were disqualified and withdrew from further participation. As a result, 77 matches were played, and the teams that withdrew had managed to play all their matches among themselves, with each of the withdrawn teams playing the same number of matches. How many teams were there at the beginning of the tournament?",
      "answer": "14"
    },
    {
      "question": "6. If $f(x)=2 \\sin \\omega x(0<\\omega<1)$ has a maximum value of $\\sqrt{2}$ on the interval $\\left[0, \\frac{\\pi}{3}\\right]$, then $\\omega=$",
      "answer": "\\frac{3}{4}"
    },
    {
      "question": "## Task 1 - 250911\n\nFrom the digits 1, 9, 8, 5, all possible four-digit numbers are to be formed, where each of the digits appears exactly once in each of these numbers.\n\nDetermine the number of such numbers that are\n\na) divisible by 2,\n\nb) divisible by 3,\n\nc) divisible by 4,\n\nd) divisible by 5,\n\ne) divisible by 6,\n\nand list these numbers!",
      "answer": "0"
    },
    {
      "question": "7.5. Egor, Nikita, and Innokentiy took turns playing chess with each other (two play, one watches). After each game, the loser would give up their place at the board to the spectator (there were no draws). In the end, it turned out that Egor participated in 13 games, and Nikita in 27. How many games did Innokentiy play?",
      "answer": "14"
    },
    {
      "question": "97. For the first half of the journey, the motorcyclist traveled at a speed of 30 km/h, and for the second half - at a speed of 60 km/h. Find the average speed of the motorcyclist.",
      "answer": "40"
    },
    {
      "question": "[ [Decimal numeral system ]  [ Problems with constraints ]\n\nFind the number of five-digit numbers in decimal notation that contain at least one digit 8.",
      "answer": "37512"
    },
    {
      "question": "Example 14 (IMO-29 Problem) Let $\\mathbf{N}$ be the set of positive integers, and define the function $f$ on $\\mathbf{N}$ as follows:\n(i) $f(1)=1, f(3)=3$;\n(ii) For $n \\in \\mathbf{N}$, we have $f(2 n)=f(n), f(4 n+1)=2 f(2 n+1)-f(n), f(4 n+3)=$ $3 f(2 n+1)-2 f(n)$.\nFind all $n$ such that $n \\leqslant 1988$ and $f(n)=n$.",
      "answer": "92"
    },
    {
      "question": "7. Find the area of triangle $ABC$, the vertices of which have coordinates $A(0,0), B(1424233,2848467), C(1424234,2848469)$. Round your answer to the hundredths.",
      "answer": "0.50"
    },
    {
      "question": "Example 6 For the function $y=(n-1) t^{2}-10 t+10, t \\in(0,4]$, does there exist a positive integer $n$ such that $0<y \\leqslant 30$ always holds?",
      "answer": "4"
    },
    {
      "question": "9-3-1. The numbers from 1 to 217 are divided into two groups: one group has 10 numbers, and the other has 207. It turns out that the arithmetic means of the numbers in the two groups are equal. Find the sum of the numbers in the group of 10 numbers.",
      "answer": "1090"
    },
    {
      "question": "11. (10 points) Divide the natural numbers $2, 3, 4, \\cdots, n$ into two groups, satisfying (1) the product of any two numbers in the same group is not in this group; (2) any number and its square are not in the same group. Then the maximum value of $n$ is $\\qquad$",
      "answer": "31"
    },
    {
      "question": "Exercise 3. Let $A B C$ be an isosceles triangle such that $\\widehat{A B C}=60^{\\circ}$; it is not known at which vertex $A B C$ is isosceles. Find all possible values of $\\widehat{A C B}$.",
      "answer": "60"
    },
    {
      "question": "## Task B-4.4.\n\nThe set of all points that satisfy the equation $x^{2}+y^{2}=8|x|+6|y|$ is a curve in the plane. Determine the area of the part of the plane bounded by this curve. How many points $(x, y)$ with integer coordinates lie inside this curve (not on the curve) for which $x y>0$?",
      "answer": "104"
    },
    {
      "question": "3. (7 points) Rubik made a net of a $3 \\times 3 \\times 3$ cube and marked two points on it - see the image. What will be the distance between these points after Rubik glues the net into a cube?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_06b9f4bfa48aaf5de920g-2.jpg?height=388&width=511&top_left_y=523&top_left_x=1318)",
      "answer": "2\\sqrt{3}"
    },
    {
      "question": "10th Chinese 1995 Problem B1 Four solid balls radii 3, 3, 2, 2 touch each other. What is the radius of the solid ball which touches all four balls?",
      "answer": "\\frac{6}{11}"
    },
    {
      "question": "Example 9 Find the smallest positive integer $n(n>1)$, such that $\\sqrt{\\frac{1}{n} \\sum_{i=1}^{n} i^{2}}$ is an integer.\n(2017, Singapore Mathematical Olympiad)",
      "answer": "337"
    },
    {
      "question": "10. Find the sum of the roots of the equation param1 that lie in the interval param2. Write the answer in degrees.\n\n| param 1 | param2 |  |\n| :---: | :---: | :---: |\n| $\\sin x+\\sin ^{2} x+\\cos ^{3} x=0$ | $\\left[360^{\\circ} ; 720^{\\circ}\\right]$ |  |\n| $\\cos x-\\cos ^{2} x-\\sin ^{3} x=0$ | $\\left[180^{\\circ} ; 540^{\\circ}\\right]$ |  |\n| $\\sin x+\\sin ^{2} x+\\cos ^{3} x=0$ | $\\left[-360^{\\circ} ; 0^{\\circ}\\right]$ |  |\n| $\\cos x-\\cos ^{2} x-\\sin ^{3} x=0$ | $\\left[-540^{\\circ} ;-180^{\\circ}\\right]$ |  |",
      "answer": "-360"
    },
    {
      "question": "1. Let the sequence $\\left\\{a_{n}\\right\\}$ satisfy:\n$$\na_{1}=\\frac{1}{2}, a_{n+1}=\\frac{2 a_{n}}{3 a_{n}+2}\\left(n \\in \\mathbf{Z}_{+}\\right) \\text {. }\n$$\n\nThen the value of $a_{2017}$ is $\\qquad$ .",
      "answer": "\\frac{1}{3026}"
    },
    {
      "question": "$7.43 \\quad 5^{x} \\sqrt[x]{8^{x-1}}=500$.\n\n$7.44 \\quad 5^{\\frac{x}{\\sqrt{x}+2}} \\cdot 0.2^{\\frac{4}{\\sqrt{x}+2}}=125^{x-4} \\cdot 0.04^{x-2}$.",
      "answer": "3"
    },
    {
      "question": "Find the inverse of 42 modulo 43, then modulo 59\n\nTo verify if needed:",
      "answer": "42"
    },
    {
      "question": "Let $A B C D$ be a quadrilateral with $\\angle B A D=\\angle A B C=90^{\\circ}$, and suppose $A B=B C=1, A D=2$. The circumcircle of $A B C$ meets $\\overline{A D}$ and $\\overline{B D}$ at points $E$ and $F$, respectively. If lines $A F$ and $C D$ meet at $K$, compute $E K$.",
      "answer": "\\frac{\\sqrt{2}}{2}"
    },
    {
      "question": "## Task A-1.5.\n\nKarlo and Lovro are playing the following game. Karlo will cut a paper of size $9 \\times 9$ into rectangles of integer dimensions, where at least one dimension is 1. After that, Lovro will choose a natural number $k \\in \\{1, \\ldots, 9\\}$, and Karlo will give him as many coins as the total area of all rectangles of dimensions $1 \\times k$ and $k \\times 1$. Lovro will choose $k$ to maximize the number of coins he receives, while Karlo wants to save and give Lovro as few coins as possible.\n\nDetermine the smallest possible number of coins that Karlo will give to Lovro.",
      "answer": "12"
    },
    {
      "question": "In triangle $ABC$, the coordinates of two vertices are $A(-1 ; 1), \\quad B(1 ; 2)$. The third vertex $C$ is on the $x$-axis. The centroid of the triangle is equidistant from the two coordinate axes.\n\nWhat is the area of the triangle?",
      "answer": "3"
    },
    {
      "question": "$\\underline{\\text { by Banov I.S }}$\n\nThe height and the radius of the base of a cylinder are equal to 1. What is the smallest number of spheres with a radius of 1 that can completely cover this cylinder?",
      "answer": "3"
    },
    {
      "question": "1. 25 Find all natural numbers $x$ that satisfy the following conditions: the product of the digits of $x$ equals $44x - 86868$, and the sum of the digits is a perfect cube.",
      "answer": "1989"
    },
    {
      "question": "6. At the entrance of Sheep Village, there are three lights - yellow, red, and blue (the positions of the lights cannot be changed), used to indicate signals. For example, \"yellow light on, red light off, blue light on\" is one signal, and \"yellow light off, red light off, blue light off\" is another signal. There are $\\qquad$ such signals.",
      "answer": "8"
    },
    {
      "question": "3. The letters of the word MATHEMATICS are rearranged to form distinct strings of the same 11 letters. What proportion of these strings do not contain the string MATH?",
      "answer": "\\frac{491}{495}"
    },
    {
      "question": "8. Given $\\alpha, \\beta, \\gamma \\in(0, \\pi)$, and $\\alpha+\\beta+2 \\gamma=\\pi$, then the maximum value of $\\cos \\alpha+\\cos \\beta+\\sin 2 \\gamma$ is $\\qquad$",
      "answer": "\\frac{3\\sqrt{3}}{2}"
    },
    {
      "question": "7.187. $x^{2} \\cdot \\log _{x} 27 \\cdot \\log _{9} x=x+4$.",
      "answer": "2"
    },
    {
      "question": "6. (15 points) For a natural number $N$, if at least six numbers in the nine natural numbers from $1 \\sim 9$ can be factors of $N$, then $N$ is called a “Six-Union Number”. Among the natural numbers greater than 2000, the smallest “Six-Union Number” is $\\qquad$",
      "answer": "2016"
    },
    {
      "question": "Shapovalov A.V.\n\nThree square paths with a common center are spaced 1 m apart from each other (see figure). Three ants start simultaneously from the lower left corners of the paths and run at the same speed: Mu and Ra counterclockwise, and Wei clockwise. When Mu reaches the lower right corner of the largest path, the other two, not yet having completed a full circle, are on the right sides of their paths, and all three are on the same straight line. Find the sides of the squares.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1aa9346639cd54e59f2ag-06.jpg?height=683&width=640&top_left_y=1202&top_left_x=709)",
      "answer": "4"
    },
    {
      "question": "Example 7. Find $\\lim _{x \\rightarrow 0} \\frac{\\sin ^{2} \\frac{x}{3}}{x^{2}}$.",
      "answer": "\\frac{1}{9}"
    },
    {
      "question": "1. It is known that $\\sin x=\\frac{3}{2} \\sin y-\\frac{2}{3} \\cos y, \\cos x=\\frac{3}{2} \\cos y-\\frac{2}{3} \\sin y$. Find $\\sin 2 y$.",
      "answer": "\\frac{61}{72}"
    },
    {
      "question": "10. Among $1,2,3,4, \\cdots, 1000$, the numbers that can be written in the form $a^{2}-b^{2}+1(a, b \\in \\mathbf{N})$ and are not divisible by 3 are $\\qquad$ in number.",
      "answer": "501"
    },
    {
      "question": "3. We will call a number \"remarkable\" if it has exactly 4 distinct natural divisors, and among them, there are two such that neither is a multiple of the other. How many \"remarkable\" two-digit numbers exist?",
      "answer": "36"
    },
    {
      "question": "8. It is known that two engineering teams, Team A and Team B, have several people each. If 90 people are transferred from Team A to Team B, then the total number of people in Team B will be twice that of Team A; if some people are transferred from Team B to Team A, then the total number of people in Team A will be 6 times that of Team B. What is the minimum number of people originally in Team A?",
      "answer": "153"
    },
    {
      "question": "1. If $a b c d=1$, then\n$$\n\\frac{1}{1+a+a b+a b c}+\\frac{1}{1+b+b c+b c d}+\\frac{1}{1+c+c d+c d a}+\\frac{1}{1+d+d a+d a b}=\n$$\n$\\qquad$",
      "answer": "1"
    },
    {
      "question": "We have selected the vertices, the centroid, and the points that trisect the sides of an equilateral triangle. How many points can we retain from among them so that no three form an equilateral triangle?",
      "answer": "6"
    },
    {
      "question": "4. Find the sum\n\n$$\n\\log \\operatorname{tg} 1^{\\circ}+\\log \\operatorname{tg} 2^{\\circ}+\\ldots+\\log \\operatorname{tg} 89^{\\circ}\n$$",
      "answer": "0"
    },
    {
      "question": "VI OM - II - Task 3\n\nWhat should be the angle at the vertex of an isosceles triangle so that a triangle can be constructed with sides equal to the height, base, and one of the remaining sides of this isosceles triangle?",
      "answer": "106"
    },
    {
      "question": "11. Let $P$ be any point on the ellipse $\\frac{x^{2}}{16}+\\frac{y^{2}}{9}=1$ other than the endpoints of the major axis, $F_{1}$ and $F_{2}$ be its left and right foci respectively, and $O$ be the center. Then $\\left|P F_{1}\\right| \\cdot\\left|P F_{2}\\right|+|O P|^{2}=$ $\\qquad$ .",
      "answer": "25"
    },
    {
      "question": "9.5 The numbers $x$, $y$, and $z$ satisfy the equations\n\n$$\nx y + y z + z x = x y z, \\quad x + y + z = 1\n$$\n\nWhat values can the sum $x^{3} + y^{3} + z^{3}$ take?",
      "answer": "1"
    },
    {
      "question": "7.054. $\\log _{2} x+\\log _{4} x+\\log _{8} x=11$.",
      "answer": "64"
    },
    {
      "question": "Example 11 (25th IMO Shortlist) Find the last two digits of $\\left[(\\sqrt{29}+\\sqrt{21})^{1984}\\right]$.\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "71"
    },
    {
      "question": "The base of an isosceles triangle is $4 \\sqrt{2}$, and the median drawn to the lateral side is 5.\n\nFind the lateral sides.",
      "answer": "6"
    },
    {
      "question": "6. Given that the base of the tetrahedron $S-ABC$ is an equilateral triangle, the projection of point $A$ on the side face $SBC$ is $H$, which is the orthocenter of $\\triangle SBC$, the dihedral angle $H-AB-C$ is $30^{\\circ}$, and $SA=2\\sqrt{3}$. Then the volume of the tetrahedron $S-ABC$ is",
      "answer": "\\frac{9}{4}\\sqrt{3}"
    },
    {
      "question": "13. Ring Line (from 8th grade. 3 points). On weekdays, the Absent-Minded Scientist travels to work on the ring line of the Moscow metro from the station \"Taganskaya\" to the station \"Kievskaya\", and back in the evening (see the diagram).\n\nUpon entering the station, the Scientist boards the first train that arrives. It is known that trains run at approximately equal intervals in both directions, and that the train traveling on the northern route (through \"Belorusskaya\") takes 17 minutes to travel from \"Kievskaya\" to \"Taganskaya\" or vice versa, while the train on the southern route (through \"Paveletskaya\") takes 11 minutes.\n\nOut of habit, the Scientist always calculates everything. Once, he calculated that over many years of observation:\n\n- the train traveling counterclockwise arrives at \"Kievskaya\" on average 1 minute and 15 seconds after the train traveling clockwise arrives at the same station. The same is true for \"Taganskaya\";\n\n- the average travel time from home to work is 1 minute less than the travel time from work to home.\n\nFind the expected interval between trains traveling in the same direction.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b695f5b0d6056980ce62g-08.jpg?height=785&width=785&top_left_y=1606&top_left_x=1138)",
      "answer": "3"
    },
    {
      "question": "1703. Determine whether Chebyshev's theorem is applicable to the sequence of independent random variables $X_{1}, X_{2}, \\ldots$, $X_{n}, \\ldots$, if $X_{n}$ has the following distribution:\n\n| $x$ | $-5 n$ | 0 | $5 n$ |\n| :---: | :---: | :---: | :---: |\n| $p(x)$ | $\\frac{1}{3 n^{2}}$ | $1-\\frac{2}{3 n^{2}}$ | $\\frac{1}{3 n^{2}}$ |",
      "answer": "\\frac{50}{3}"
    },
    {
      "question": "One interior angle in a triangle measures $50^{\\circ}$. What is the angle between the bisectors of the remaining two interior angles?\n\n(L. Hozová)\n\nHint. You don't need to know the sizes of the remaining interior angles to solve the problem.",
      "answer": "65"
    },
    {
      "question": "Problem 5.4. The school principal, the caretaker, and the parent committee, failing to agree with each other, each bought a carpet for the school auditorium, which is $10 \\times 10$. After thinking about what to do, they decided to place all three carpets as shown in the picture: the first carpet $6 \\times 8$ - in one corner, the second carpet $6 \\times 6$ - in the opposite corner, and the third carpet $5 \\times 7$ - in one of the remaining corners (all dimensions are in meters).\n\nFind the area of the part of the hall covered by carpets in three layers (give the answer in square meters).\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_b46fbd582cc3a82460aeg-10.jpg?height=337&width=1003&top_left_y=304&top_left_x=225)",
      "answer": "6"
    },
    {
      "question": "4. In the maze diagram in Fig. 1, each segment (link) is a corridor, and each circle is a small room. In some rooms, there are beacons that hum - each with its own voice. When in any room, the robot hears the signal from each beacon and determines the distance to it by the attenuation, i.e., the number of links on the shortest path to the beacon. For example, if a beacon is located in room $A$ (in chess notation, this is room a4), and the robot is in room $B$ (this is room g4), the distance between them is 7 links. The robot has the maze diagram and the beacons are marked on it. For what minimum number of beacons can the robot, when in any room, uniquely determine its location by sound?\n\na) Indicate how such a number of beacons can be placed. Why will the robot be able to determine its room by them?\n\nb) Prove that it is impossible to manage with a smaller number of beacons.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_57b8ad17478f0226d916g-02.jpg?height=505&width=529&top_left_y=513&top_left_x=775)\n\nFig. 1: Maze",
      "answer": "3"
    },
    {
      "question": "Shveiov D.V.\n\nThe circle inscribed in the right triangle $ABC\\left(\\angle ABC=90^{\\circ}\\right)$ touches the sides $AB, BC, AC$ at points $C_{1}, A_{1}, B_{1}$ respectively. The excircle opposite to $A$ touches the side $BC$ at point $A_{2}$. $A_{0}$ is the center of the circumcircle of triangle $A_{1} A_{2} B_{1}$; the point $C_{0}$ is defined similarly. Find the angle $A_{0} B C_{0}$.",
      "answer": "45"
    },
    {
      "question": "5. Given that $f(x)$ and $g(x)$ are two quadratic functions with the leading coefficient of 1, if $g(6)=35, \\frac{f(-1)}{g(-1)}=$ $\\frac{f(1)}{g(1)}=\\frac{21}{20}$, then $f(6)=$ $\\qquad$ .",
      "answer": "35"
    },
    {
      "question": "8. There are some chess pieces in a box, with the quantity being less than 50. Xiao Ming and Xiao Liang take turns to take chess pieces from the box. If they take the pieces in the pattern of Xiao Ming taking 2, Xiao Liang taking 2, Xiao Ming taking 2, and Xiao Liang taking 2, in the end, Xiao Ming takes 2 more pieces than Xiao Liang; If they take the pieces in the pattern of Xiao Ming taking 3, Xiao Liang taking 3, Xiao Ming taking 3, and Xiao Liang taking 3, in the end, both of them take the same number of pieces. Therefore, the box contains at most $\\qquad$ pieces.",
      "answer": "42"
    },
    {
      "question": "Zeljko travelled at $30 \\mathrm{~km} / \\mathrm{h}$ for 20 minutes and then travelled at $20 \\mathrm{~km} / \\mathrm{h}$ for 30 minutes. How far did he travel, in kilometres?",
      "answer": "20"
    },
    {
      "question": "2. (5 points) The same remote control plane is sold online for 300 yuan, which is $20 \\%$ lower than the price at Star Toy Store. Therefore, the price of this remote control plane at Star Toy Store is $\\qquad$ yuan.",
      "answer": "375"
    },
    {
      "question": "6. As shown in Figure 1, in the tetrahedron\n$$\n\\begin{array}{l}\nA B C D \\text {, } D A=D B=D C \\\\\n=2, D A \\perp D B, D A \\perp\n\\end{array}\n$$\n$D C$, and the cosine of the angle between $D A$ and the plane $A B C$ is $\\frac{\\sqrt{6}}{3}$. Then the radius $R$ of the circumscribed sphere of the tetrahedron is $\\qquad$",
      "answer": "\\sqrt{3}"
    },
    {
      "question": "# 1. Clone 1\n\nThe figure is divided into 7 equal squares and several rectangles. The perimeter of rectangle A is 112 cm. What is the perimeter of rectangle B? Express your answer in centimeters.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_1f50971f08007f421b41g-01.jpg?height=485&width=483&top_left_y=1008&top_left_x=838)\n\n#",
      "answer": "168"
    },
    {
      "question": "SI. 4 In the coordinate plane, the distance from the point $(-C, 0)$ to the straight line $y=x$ is $\\sqrt{D}$, find the value of $D$.",
      "answer": "8"
    },
    {
      "question": "Problem 10.7. The graph of the function $f(x)=\\frac{1}{12} x^{2}+a x+b$ intersects the $O x$ axis at points $A$ and $C$, and the $O y$ axis at point $B$, as shown in the figure. It turned out that for the point $T$ with coordinates $(3 ; 3)$, the condition $T A=T B=T C$ is satisfied. Find $b$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_df4c0436ab68970fab15g-39.jpg?height=321&width=517&top_left_y=195&top_left_x=468)",
      "answer": "-6"
    },
    {
      "question": "5. Given that $a$, $b$, and $c$ are all positive numbers. Then\n$$\nM=\\min \\left\\{\\frac{1}{a}, \\frac{2}{b}, \\frac{4}{c}, \\sqrt[3]{a b c}\\right\\}\n$$\n\nthe maximum value of $M$ is . $\\qquad$",
      "answer": "\\sqrt{2}"
    },
    {
      "question": "7. The diagonals of a trapezoid are perpendicular to each other, and one of them is 17. Find the area of the trapezoid if its height is 15.\n\nAnswer. $\\frac{4335}{16}$.",
      "answer": "\\frac{4335}{16}"
    },
    {
      "question": "How many positive integers of at most $n$ digits are there whose square ends in the same $n$ digits (possibly with some leading zeros)?",
      "answer": "3"
    },
    {
      "question": "3. Given that the length of the major axis of an ellipse is 4, the focal distance is 2, and the sum of the lengths of two perpendicular chords passing through the left focus is $\\frac{48}{7}$. Try to find the product of the lengths of these two chords.\n\n untranslated text remains unchanged.",
      "answer": "\\frac{576}{49}"
    },
    {
      "question": "Variant 10.4.1. Given a parallelogram $A B C D$, point $M$ is the midpoint of side $B C$. On side $A D$, there is a point $K$ such that $B K=B M$ and quadrilateral $K B M D$ is cyclic.\n\n(a) (2 points) What is the length of segment $M D$, if $A D=17$?\n\n(b) (2 points) How many degrees does angle $K M D$ measure, if $\\angle B A D=46^{\\circ}$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_baaf26a33e2d9a36d9b4g-10.jpg?height=274&width=517&top_left_y=824&top_left_x=468)",
      "answer": "8.5"
    },
    {
      "question": "28. [14] Compute the nearest integer to\n$$\n100 \\sum_{n=1}^{\\infty} 3^{n} \\sin ^{3}\\left(\\frac{\\pi}{3^{n}}\\right)\n$$",
      "answer": "236"
    },
    {
      "question": "14. (3 points) The unit digit of $1^{8}+2^{8}+3^{8}+4^{8}+5^{8}+6^{8}+7^{8}+8^{8}+9^{8}+10^{8}+11^{8}$ is",
      "answer": "4"
    },
    {
      "question": "## 298. Math Puzzle $3 / 90$\n\nSebastian visits his grandmother every week, who lives in the neighboring town 10 km away. Most of the time, he rides his bicycle. On a nice day, it takes the grandmother's favorite grandson 1 hour and 20 minutes to get to the destination and back.\n\nOne day, however, a strong wind is blowing. The boy still sets off on his bike.\n\nThe speed increases by $4 \\mathrm{~km} / \\mathrm{h}$ on the way there, and decreases by the same amount on the way back. Sebastian thinks that this does not change his travel time? Is he right?",
      "answer": "86.1"
    },
    {
      "question": "8.5. Given a convex quadrilateral $A B C D$, where $A B=A D=1, \\angle A=80^{\\circ}$, $\\angle C=140^{\\circ}$. Find the length of the diagonal $A C$.",
      "answer": "1"
    },
    {
      "question": "The Law of Cosines\n\nThe bases of a trapezoid are 3 cm and 5 cm. One of the diagonals of the trapezoid is 8 cm, and the angle between the diagonals is $60^{\\circ}$. Find the perimeter of the trapezoid.",
      "answer": "22"
    },
    {
      "question": "Find all integers $n$ such that $n^{3}-3 n^{2}+n+2$ is a power of 5.",
      "answer": "3"
    },
    {
      "question": "The table below shows some of the results from the last Pirajuba Fishing Festival, displaying how many competitors $q$ caught $n$ fish for some values of $n$.\n\n| $n$ | 0 | 1 | 2 | 3 | $\\ldots$ | 13 | 14 | 15 |\n| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |\n| $q$ | 9 | 5 | 7 | 23 | $\\ldots$ | 5 | 2 | 1 |.\n\nThe news published in the city's newspaper reported that:\n\ni) the winner caught 15 fish;\n\nii) among those who caught 3 or more fish, the average was 6 fish caught; and\n\niii) among those who caught 12 or fewer fish, the average number of fish caught was 5.\n\na) What was the total number of fish caught during the festival?\n\nb) How many competitors caught between 4 and 12 fish?",
      "answer": "883"
    },
    {
      "question": "Task B-4.1. Determine the natural number $n \\geq 2$ such that the equality\n\n$$\n\\frac{(n-1)^{2} n(n+1)!}{(n+2)!}=\\binom{n}{2}\n$$\n\nholds.",
      "answer": "4"
    },
    {
      "question": "Let $t$ be TNYWR.\n\nWhat is the area of the triangle enclosed by the $x$-axis, the $y$-axis, and the line with equation $y=t x+6$ ?\n\n## The CENTRE for EDUCATION in MATHEMATICS and COMPUTING cemc.uwaterloo.ca 2022 Canadian Team Mathematics Contest",
      "answer": "\\frac{9}{2}"
    },
    {
      "question": "10. From the numbers $1,2,3, \\cdots, 14$, select $a_{1}, a_{2}$, $a_{3}$ in ascending order, and $a_{2}-a_{1} \\geqslant 3, a_{3}-a_{2} \\geqslant 3$, then the number of different ways to select that meet the conditions is $\\qquad$ kinds.",
      "answer": "120"
    },
    {
      "question": "## Task B-3.1.\n\nDetermine the maximum value of the function $f(x)=\\log _{2}^{4} x+12 \\log _{2}^{2} x \\cdot \\log _{2} \\frac{8}{x}$ on the interval $[1,64]$.",
      "answer": "81"
    },
    {
      "question": "$9 \\cdot 8$ For all positive integers from 1 to 1000000000, find the sum of the digits of each number, then for the resulting 1 billion numbers, find the sum of the digits of each number, $\\cdots$, until obtaining 1 billion single-digit numbers. Ask whether there are more 1s or 2s in the resulting numbers?\n(4th All-Russian Mathematical Olympiad, 1964)",
      "answer": "1"
    },
    {
      "question": "4. The sequence $\\left(x_{n}\\right)_{n=1}^{\\infty} \\mathrm{s}$ has the first term $x_{1}=1$ and for every $n>1$ satisfies the condition\n\n$$\nx_{n}=x_{n-1}^{ \\pm 1}+x_{n-2}^{ \\pm 1}+\\ldots+x_{1}^{ \\pm 1}\n$$\n\nwith an appropriate choice of the signs \"±\" in the exponents of the powers.\n\na) Determine whether any term of such a sequence must be greater than 1000.\n\nb) Find the smallest possible value of the term $x_{1000000}$.\n\nc) Prove that the inequality $x_{n}<4$ cannot hold for nine terms $x_{n}$ of such a sequence.\n\nII. round of category A takes place\n\n## on Tuesday, January 14, 2003\n\nso that it starts in the morning and the contestants have 4 hours of pure time to solve the problems. Each problem can earn the contestant 6 points, and a successful solver is a student who scores 10 points or more. These details will be communicated to the students before the start of the competition.",
      "answer": "3-2^{3-10^{6}}"
    },
    {
      "question": "[ Right triangle with an angle in ]\n\nThe hypotenuse $AB$ of the right triangle $ABC$ is 2 and is a chord of a certain circle. The leg $AC$ is 1 and lies inside the circle, and its extension intersects the circle at point $D$, with $CD=3$. Find the radius of the circle.",
      "answer": "2"
    },
    {
      "question": "2. On the first day, Masha collected $25 \\%$ fewer berries than Vanya, and on the second day, $20 \\%$ more than Vanya. Over the two days, Masha collected $10 \\%$ more berries than Vanya. What is the smallest number of berries they could have collected together?",
      "answer": "189"
    },
    {
      "question": "1.005. $\\frac{2 \\frac{3}{4}: 1.1+3 \\frac{1}{3}}{2.5-0.4 \\cdot 3 \\frac{1}{3}}: \\frac{5}{7}-\\frac{\\left(2 \\frac{1}{6}+4.5\\right) \\cdot 0.375}{2.75-1 \\frac{1}{2}}$.",
      "answer": "5"
    },
    {
      "question": "456. Every evening, Uncle Chernomor appoints 9 or 10 out of 33 bogatyrs for duty, at his discretion. What is the smallest number of days after which it can happen that each of the bogatyrs has been on duty the same number of times?",
      "answer": "7"
    },
    {
      "question": "14. A regular 201-sided polygon is inscribed inside a circle of center $C$. Triangles are drawn by connecting any three of the 201 vertices of the polygon. How many of these triangles have the point $C$ lying inside the triangle?\n14. 一個正 201 邊形內接於一個以 $C$ 為圓心的圓形內。把多邊形 201 個頂點中的任意三個頂點連起, 可以組成不同的三角形。有多少個這樣的三角形, 將 $C$ 點包含在其內部?\n(2 分)",
      "answer": "338350"
    },
    {
      "question": "# 5. Option 1\n\nUnit cubes were used to build a large cube. Two cubes will be called adjacent if they touch by faces. Thus, one cube can have up to 6 neighbors. It is known that the number of cubes that have exactly 4 neighbors is 132. Find the number of cubes that have exactly 5 neighbors.",
      "answer": "726"
    },
    {
      "question": "18. How many digits does the number $2^{100}$ have?",
      "answer": "31"
    },
    {
      "question": "5. There is a paper square $7 \\times 7$, all cells of which are white. What is the minimum number of cells that need to be painted black so that it is impossible to cut out a rectangle from it with at least 10 cells, all of which are white?",
      "answer": "4"
    },
    {
      "question": "2. Write down and read all seven-digit numbers, the sum of the digits in each of which is equal to 2. How many such numbers are there?",
      "answer": "7"
    },
    {
      "question": "Question 62: Given positive real numbers $a$ and $b$ satisfy $a(a+b)=27$, try to find the maximum value of $a^{2} b$.\n\n保留源文本的换行和格式，直接输出翻译结果。 \n\n(注意：最后一句是解释说明，不是原文的一部分，因此在最终输出时应去掉。)\n\nQuestion 62: Given positive real numbers $a$ and $b$ satisfy $a(a+b)=27$, try to find the maximum value of $a^{2} b$.",
      "answer": "54"
    },
    {
      "question": "5. Given that the diameter $A B$ of circle $O$ divides the circle into upper and lower semicircles, and points $C, D$ are on the upper and lower semicircles respectively (neither coinciding with points $A, B$), if $A C=2, A D=1$, then $\\overrightarrow{A B} \\cdot \\overrightarrow{D C}=$ $\\qquad$ .",
      "answer": "3"
    },
    {
      "question": "## Task A-2.6.\n\nIn a dining room measuring $6 \\mathrm{~m} \\times 6 \\mathrm{~m}$, one corner is occupied by a rectangular card table with dimensions $2 \\mathrm{~m} \\times$ $1.5 \\mathrm{~m}$. What is the radius of the largest circular rug that can be spread on the dining room floor?",
      "answer": "2.5"
    },
    {
      "question": "22. The Rainforest School, Desert School, and Tundra School in Zootopia are organizing a Spring Festival Gala, with a total of 15 performances. If each school must perform at least 3 programs, then the number of different ways the three schools can perform is $\\qquad$ kinds.",
      "answer": "28"
    },
    {
      "question": "# Problem 5. (based on materials from the Ural Tournaments)\n\nIn triangle $ABC$, angle $B$ is $80^{\\circ}$. On side $BC$, point $D$ is marked such that $AB = AD = CD$. On side $AB$, point $F$ is marked such that $AF = BD$. On segment $AC$, point $E$ is marked such that $AB = AE$. Find angle $AEF$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_19300bfa006950daee48g-3.jpg?height=1431&width=1697&top_left_y=344&top_left_x=192)",
      "answer": "20"
    },
    {
      "question": "5. Arrange 3 identical black balls and 3 identical white balls in a row from left to right. If, starting from any position (including this position) and counting to the left, the number of black balls is always not less than the number of white balls, such an arrangement is called a \"valid arrangement.\" The probability of a \"valid arrangement\" occurring is $\\qquad$ .",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Example 9 The function $f(x, y)$ satisfies for all non-negative integers: (1) $f(0, y)=y+1$; (2) $f(x+1,0)=$ $f(x, 1)$; (3) $f(x+1, y+1)=f(x, f(x+1, y))$. Determine $f(4,1981)$.",
      "answer": "2^{2^{1984}-3}"
    },
    {
      "question": "13.063. In the club's auditorium, there were 320 seats arranged in identical rows. After the number of seats in each row was increased by 4 and one more row was added, the auditorium had 420 seats. How many rows are there in the auditorium now?",
      "answer": "21"
    },
    {
      "question": "## Task A-1.2.\n\nGiven is a right triangle $A B C$ with a right angle at vertex $C$ and side lengths $|A B|=26$, $|B C|=24$. A semicircle is inscribed in it with the diameter on side $\\overline{B C}$ which contains point $C$. The semicircle touches side $\\overline{A B}$. What is the radius of the inscribed semicircle?",
      "answer": "12"
    },
    {
      "question": "3. From the 10 numbers $0,1,2,3,4,5,6,7,8,9$, take out 3 numbers so that their sum is an even number not less than 10. The number of different ways to do this is $\\qquad$.",
      "answer": "51"
    },
    {
      "question": "Um decorador distribuirá flores em oito pontos ao redor de um arranjo quadrado de flores, como indicado na figura abaixo. Ele quer fazer isso de modo tal que, em cada lado do arranjo, as pessoas vejam sempre a mesma quantidade de flores. No exemplo abaixo, temos o total de 11 flores e em cada um dos 4 lados do quadrado são vistas exatamente 4 delas.\n\na) Qual o número máximo de flores que podem ser usadas, considerando que em cada lado do quadrado devem ser vistas exatamente 9 flores?\n\nb) Qual o número mínimo de flores que podem ser usadas, considerando que em cada lado do quadrado devem ser vistas exatamente 12 flores?\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_360f8ce7ec440aed2c7ag-19.jpg?height=519&width=514&top_left_y=954&top_left_x=859)\n\n#",
      "answer": "24"
    },
    {
      "question": "Problem 11.3. (15 points) Natural numbers $a, b, c$ are such that $1 \\leqslant a<b<c \\leqslant 3000$. Find the greatest possible value of the quantity\n\n$$\n\\text { GCD }(a, b)+\\text { GCD }(b, c)+\\text { GCD }(c, a)\n$$",
      "answer": "3000"
    },
    {
      "question": "Problem 8.6. For quadrilateral $ABCD$, it is known that $AB=BD, \\angle ABD=\\angle DBC, \\angle BCD=90^{\\circ}$. A point $E$ is marked on segment $BC$ such that $AD=DE$. What is the length of segment $BD$, if it is known that $BE=7, EC=5$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_e7f9c87b6a37ffba3564g-30.jpg?height=476&width=517&top_left_y=91&top_left_x=468)",
      "answer": "17"
    },
    {
      "question": "7. In a chess tournament, 8 players participated. They scored $7, 6, 4, 4, 3, 2, 1, 5$ and 0.5 points respectively. How many points did the players who took the first four places lose in matches against the others?",
      "answer": "1"
    },
    {
      "question": "Problem 3. In $1000 \\mathrm{~kg}$ of fresh berries, there is 99% water. During transportation, a certain amount of water evaporated, so the weight of the berries after the water evaporation is $500 \\mathrm{~kg}$. What percentage of water do the berries now contain?",
      "answer": "98"
    },
    {
      "question": "## Task B-3.1.\n\nSolve the equation $5 \\cdot 0.16^{x}+8 \\cdot 0.4^{x}=4$.",
      "answer": "1"
    },
    {
      "question": "3. (CAN) Find the minimum value of  $$ \\max (a+b+c, b+c+d, c+d+e, d+e+f, e+f+g) $$  subject to the constraints (i) $a, b, c, d, e, f, g \\geq 0$, (ii) $a+b+c+d+e+f+g=1$.",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "Determine the integer equal to\n\n$$\n2020+1919+1818+1717+1616+1515+1414+1313+1212+1111+1010 .\n$$",
      "answer": "16665"
    },
    {
      "question": "20 Anna randomly picked five integers from the following list\n$$\n53,62,66,68,71,82,89\n$$\nand discover that the average value of the five integers she picked is still an integer. If two of the integers she picked were 62 and 89 , find the sum of the remaining three integers.",
      "answer": "219"
    },
    {
      "question": "4. Petya wrote down $n$ consecutive two-digit numbers $(n \\geqslant 2)$ in a row on the board, the first of which does not contain the digit 4, and the last one does not contain the digit 7. Vasya thought that this was the decimal representation of a natural number $x$, and he factored $x$ into prime factors. It turned out that there were only two of them and they differ by 4. What is written on the board?",
      "answer": "2021"
    },
    {
      "question": "9. In our country, individual income tax on wages and salaries is calculated in segments. Total income not exceeding 800 yuan is exempt from individual income tax on wages and salaries; the portion exceeding 800 yuan is subject to taxation. Let the taxable amount for the entire month be $x$, where $x=$ total monthly income - 800 yuan. The tax rates are as follows:\n\\begin{tabular}{|c|c|c|}\n\\hline Level & Monthly taxable amount $x$ & Tax \\\\\n\\hline 1 & Up to 500 yuan \\\\\n\\hline 2 & Over 500 yuan to 2000 yuan & $5 \\%$ \\\\\n\\hline 3 & Over 2000 yuan to 5000 yuan & $10 \\%$ \\\\\n\\hline$\\vdots$ & & $15 \\%$ \\\\\n\\hline 9 & Over 100000 yuan & $45 \\%$ \\\\\n\\hline\n\\end{tabular}\n(1) If the tax amount is $f(x)$, try to express the calculation formula for the tax amount $f(x)$ of levels 1-3 using a piecewise function.\n(2) A person's total monthly income in March 2000 was 3000 yuan. Calculate how much personal income tax this person should pay in March.",
      "answer": "205"
    },
    {
      "question": "246. $\\int 8 x^{3} d x$",
      "answer": "160"
    },
    {
      "question": "10.3. Point $F$ is the midpoint of side $B C$ of square $A B C D$. A perpendicular $A E$ is drawn to segment $D F$. Find the angle $C E F$.\n\n---\n\nTranslation provided as requested, maintaining the original format and line breaks.",
      "answer": "45"
    },
    {
      "question": "[Arithmetic. Mental arithmetic, etc.]\n\nThe tower clock strikes three times in 12 seconds. How long will it take for it to strike six times?\n\n#",
      "answer": "30"
    },
    {
      "question": "4. Given $(x+3)(x-1)^{4}=a_{0}+a_{1} x+a_{2} x^{2}+a_{3} x^{3}+a_{4} x^{4}+a_{5} x^{5}$, then $a_{3}+a_{4}+a_{5}=$",
      "answer": "-6"
    },
    {
      "question": "18. The number of real solutions to the equation $\\sin \\left(x-\\frac{\\pi}{4}\\right)=\\frac{1}{4} x$ is",
      "answer": "3"
    },
    {
      "question": "7. In the Magic Land, there is a magic stone that grows uniformly upwards. To prevent it from piercing the sky, the Elders of the Immortal World decided to send plant warriors to consume the magic stone and suppress its growth. Each plant warrior consumes the same amount every day. If 14 plant warriors are dispatched, the magic stone will pierce the sky after 16 days; if 15 plant warriors are dispatched, the magic stone will pierce the sky after 24 days. At least $\\qquad$ plant warriors need to be dispatched to ensure the sky is not pierced.",
      "answer": "17"
    },
    {
      "question": "Problem 3. Vasya wants to find all integers $a$ such that the expression $10 n^{3}+3 n^{5}+7 a n$ is divisible by 15 for all integers $n$. What remainders can the number $a$ give when divided by 15? Provide all possible answers or prove that there are no such integers $a$.",
      "answer": "11"
    },
    {
      "question": "## Task 5 - 311235\n\nDetermine whether it is possible to select fifty different natural numbers from 1 to 100 such that their sum is 2525 and no two of them add up to 101.",
      "answer": "2525"
    },
    {
      "question": "3. The product of three natural numbers is 13600. Calculate the product of the first and second number if the product of the first and third number is 544, and the product of the second and third is 425.",
      "answer": "800"
    },
    {
      "question": "Juarez used the digits 1, 2, 3, 4, and 5 to write the five-digit number $\\overline{a b c d e}$. Without revealing what this number is, he told Luciana that:\n\n$\\cdot$ the number $\\overline{a b c}$ is divisible by 4;\n\n- the number $\\overline{b c d}$ is divisible by 5;\n\n$\\cdot$ the number $\\overline{c d e}$ is divisible by 3.\n\nThen, Luciana told Juarez that it is possible to figure out what the number $\\overline{a b c d e}$ is. Show that Luciana is correct, that is, find the number $\\overline{a b c d e}$.",
      "answer": "12453"
    },
    {
      "question": "## 18. Expected Lifespan\n\nAn ethnographer found that in a certain primitive tribe he studied, the distribution of the lifespan of the tribe members could be described as follows: $25 \\%$ of them only lived up to 40 years, $50 \\%$ died at 50 years, and $25 \\%$ - at 60 years. He then randomly selected two individuals to study in more detail. What is the expected lifespan of the one of the two randomly selected individuals who will live longer?",
      "answer": "53"
    },
    {
      "question": "4. In triangle $\\triangle A B C$, the base $A B$ of which lies on the x-axis, altitudes $A M, B N$ and $C K$ are drawn. Find the length of the base $A B$, if the coordinates of points $M(2,2)$ and $N(4,4)$ are known.",
      "answer": "4\\sqrt{5}"
    },
    {
      "question": "13. (6 points) Two classes in Grade 4 have a total of 72 students, among whom there are 35 girls. Class 4(1) has 36 students, and Class 4(2) has 19 boys. Therefore, Class 4(1) has $\\qquad$ girls.",
      "answer": "18"
    },
    {
      "question": "Miscellaneous Problem 1 The crossed ladder problem. In an alley a ladder length 3 is propped up against one wall with its foot at the bottom of the other wall. It crosses another ladder length 2 leaning the other way a distance 1 from the ground. How wide is the alley? [In other words we are given a quadrilateral ABCD. Angles ADC and DCB are right angles. AC = 2, BD = 3 and AC, BD intersect at X a distance 1 from CD. Find CD.]",
      "answer": "1.2311857"
    },
    {
      "question": "5. Place eight balls with a radius of 1 in two layers inside a cylinder, such that each ball is tangent to its four adjacent balls, and is also tangent to one base and the side of the cylinder. Then the height of this cylinder is $\\qquad$ .",
      "answer": "\\sqrt[4]{8}+2"
    },
    {
      "question": "3 [Divisibility of numbers. General properties]\n\nIt is known that $x=2 a^{5}=5 b^{2}>0$, where numbers $a$ and $b$ are integers. What is the smallest possible value of $x$?",
      "answer": "200000"
    },
    {
      "question": "Problem 9.4. From left to right, intersecting squares with sides $12, 9, 7, 3$ are depicted respectively. By how much is the sum of the black areas greater than the sum of the gray areas?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_a7569897979ba1659de3g-34.jpg?height=686&width=872&top_left_y=927&top_left_x=289)",
      "answer": "103"
    },
    {
      "question": "Example: Sets $A, B, C$ (not necessarily distinct) satisfy $A \\cup B \\cup C=\\{1,2,3, \\cdots, 11\\}$. Find the number of solutions of the ordered triples $(A, B, C)$ that meet the condition.",
      "answer": "7^{11}"
    },
    {
      "question": "113. The base of the pyramid $A B C E H$ is a convex quadrilateral $A B C E$, which is divided into two equal-area triangles by the diagonal $B E$. The length of the edge $A B$ is 1, the lengths of the edges $B C$ and $C E$ are equal. The sum of the lengths of the edges $A H$ and $E H$ is $\\sqrt{2}$. The volume of the pyramid is $1 / 6$. Find the radius of the sphere with the largest volume among all spheres that can fit inside the pyramid.",
      "answer": "\\frac{3-\\sqrt{5}}{4}"
    },
    {
      "question": "# Problem 7. (4 points)\n\nIn a $7 \\times 7$ table, some cells are black, and the rest are white. In each white cell, the total number of black cells on the same row or column is written; nothing is written in the black cells. What is the maximum value that the sum of the numbers in the entire table can take?\n\n#",
      "answer": "168"
    },
    {
      "question": "1. Andrei, Boris, and Valentin participated in a 1 km race. (We assume that each of them ran at a constant speed). Andrei was 50 m ahead of Boris at the finish line. And Boris was 40 m ahead of Valentin at the finish line. What was the distance between Andrei and Valentin at the moment Andrei finished?",
      "answer": "88"
    },
    {
      "question": "3. The villages of Arkadino, Borisovo, and Vadimovo are connected by straight roads. A square field adjoins the road between Arkadino and Borisovo, one side of which completely coincides with this road. A rectangular field adjoins the road between Borisovo and Vadimovo, one side of which completely coincides with this road, and the second side is 4 times longer. $\\mathrm{K}$ road between Arkadino and Vadimovo adjoins a rectangular forest, one side of which completely coincides with this road, and the second side is 12 km. The area of the forest is 45 sq. km greater than the sum of the areas of the fields. Find the total area of the forest and fields in sq. km.\n\n$\\{135\\}$",
      "answer": "135"
    },
    {
      "question": "2+ [ Classical combinatorics (other) $]\n\nIn a bag, there are 70 balls that differ only in color: 20 red, 20 blue, 20 yellow, and the rest are black and white.\n\nWhat is the smallest number of balls that need to be drawn from the bag, without seeing them, to ensure that among them there are at least 10 balls of the same color?\n\n#",
      "answer": "38"
    },
    {
      "question": "13.262. It is known that a freely falling body travels 4.9 m in the first second, and in each subsequent second, it travels 9.8 m more than in the previous one. If two bodies start falling from the same height, one 5 seconds after the other, then after what time will they be 220.5 m apart?",
      "answer": "7"
    },
    {
      "question": "As shown in the figure, find the sum of the degrees of the 12 angles marked with numbers.\n\nTranslate the text above into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "720"
    },
    {
      "question": "Task 1. 600 kg of mushrooms with a moisture content of $98 \\%$ were collected. After drying, the moisture content was reduced to $96 \\%$. What is the mass of the mushrooms after drying?",
      "answer": "300"
    },
    {
      "question": "Let's determine all natural numbers $a, b, c, d$ for which $a b=c+d$, and $a+b=c d$ hold.",
      "answer": "9"
    },
    {
      "question": "Is there such an $n$, and if there is, how large is it, for which the sum of the first $n$ natural numbers is a three-digit number with equal digits?",
      "answer": "36"
    },
    {
      "question": "1. Immediately after harvesting, the water content in 1 ton of tomatoes was $99 \\%$. By the time of sale, the water share had decreased by $4 \\%$ (after drying). What did the total weight of the tomatoes become?",
      "answer": "200"
    },
    {
      "question": "SI. 2 In Figure $1, A B=A C$ and $A B / / E D$.\nIf $\\angle A B C=P^{\\circ}$ and $\\angle A D E=Q^{\\circ}$, find the value of $Q$.",
      "answer": "120"
    },
    {
      "question": "8. Let $x, y$ be nonnegative integers such that $x+2 y$ is a multiple of $5, x+y$ is a multiple of 3 and $2 x+y \\geq 99$. Find the minimum possible value of $7 x+5 y$.\n設 $x 、 y$ 少非負整數, 使得 $x+2 y$ 爲 5 的倍數、 $x+y$ 絧 3 的倍數, 且 $2 x+y \\geq 99$ 。求 $7 x+5 y$ 的最小可能値。",
      "answer": "366"
    },
    {
      "question": "A square is divided into 16 equal squares, obtaining the set of 25 different vertices. What is the least number of vertices one must remove from this set, so that no 4 points of the remaining set are the vertices of any square with sides parallel to the sides of the initial square?\n\nRemark. The proposed solution to this problem claimed that it is enough to remove 7 vertices but the example to demonstrate this appeared to be incorrect. Below we show that removing 6 vertices is not sufficient but removing 8 vertices is. It seems that removing 7 vertices is not sufficient but we currently know no potential way to prove this, apart from a tedious case study.",
      "answer": "8"
    },
    {
      "question": "9.4. In a square table of size $100 \\times 100$, some cells are shaded. Each shaded cell is the only shaded cell either in its column or in its row. What is the maximum number of cells that can be shaded?",
      "answer": "198"
    },
    {
      "question": "3. A rectangle $A D E C$ is described around a right triangle $A B C$ with legs $A B=5$ and $B C=6$, as shown in the figure. What is the area of $A D E C$?",
      "answer": "30"
    },
    {
      "question": "4. The Petrovs family decided to buy an apartment with a mortgage. The total cost of the apartment was 13,000,000 rubles, and the family did not have enough funds, so they decided to take a loan of 8,000,000 rubles. One of the bank's conditions for issuing the loan was the purchase of a comprehensive mortgage insurance policy for the first year of the loan, including insurance of the acquired apartment against risks of loss and damage (property insurance), title insurance (loss of ownership rights to the apartment), and life and health insurance of the borrowers. Maria Sergeyevna Petrova and Vasily Dmitrievich Petrov became co-borrowers in the ratio of 40% and 60% of the loan amount, respectively. Calculate the family's expenses on insurance for the year if the bank's requirement is formulated as follows: \"The insurance amount for the risks of loss, damage, and loss of ownership for the first year of insurance should be no less than the loan amount at the time of issuance, increased by the interest rate, and the insurance amount for the life and health insurance of each co-borrower, calculated in the above-mentioned manner, is divided proportionally according to each of their contributions to the loan.\" (20 points)\n\nAppendix: the interest rate on the loan is 9.5%, the tariff for property insurance is 0.09%, life and health insurance (women) is 0.17%, life and health insurance (men) is 0.19%, and title insurance is 0.27%.",
      "answer": "47481.20"
    },
    {
      "question": "13.222. The distance between stations $A$ and $B$ is 103 km. A train left $A$ for $B$ and, after traveling a certain distance, was delayed, and therefore the remaining distance to $B$ was traveled at a speed 4 km/h greater than the initial speed. Find the original speed of the train, given that the remaining distance to $B$ was 23 km longer than the distance traveled before the delay, and that the time taken to travel the remaining distance after the delay was 15 minutes more than the time taken to travel the distance before the delay.",
      "answer": "80"
    },
    {
      "question": "In the plane, let there be a parabola. Let its vertex be $O$ and an arbitrary point on it be $M$. We draw a perpendicular to the line $OM$ at point $O$, and draw a line parallel to the axis of the parabola through point $M$. Determine the geometric locus of the intersection point of the two lines as $M$ runs along the parabola.",
      "answer": "-4"
    },
    {
      "question": "12. The smallest positive period of the function $f(x)=\\cos ^{3} x-\\sin ^{2} x$ is",
      "answer": "2\\pi"
    },
    {
      "question": "1. In triangle $\\mathrm{ABC}$, angle $\\mathrm{C}$ is a right angle, and $\\mathrm{CD}$ is the altitude. Find the length of the radius of the circle inscribed in triangle $\\mathrm{ABC}$, if the lengths of the radii of the circles inscribed in triangles $\\mathrm{ACD}$ and $\\mathrm{BCD}$ are 6 and 8, respectively.",
      "answer": "10"
    },
    {
      "question": "Problem 6.5. In the park, paths are laid out as shown in the figure. Two workers started to asphalt them, starting simultaneously from point $A$. They lay asphalt at constant speeds: the first on the section $A-B-C$, the second on the section $A-D-E-F-C$. In the end, they finished the work simultaneously, spending 9 hours on it. It is known that the second works 1.2 times faster than the first. How many minutes did the second spend laying asphalt on the section $D E$?\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_899cf5197845501d962eg-16.jpg?height=259&width=299&top_left_y=1340&top_left_x=577)",
      "answer": "45"
    },
    {
      "question": "14. Place the numbers $1,2,3,4,5$ in a circle. We call a placement method a \"central ring\" placement if for some number $n$ in the range 1 to 15, it is impossible to select several consecutive numbers on the circle such that their sum is $n$. If two placement methods are the same after rotation or reflection, they are considered the same method. How many \"central ring\" placement methods are there in total?\n(Explanation: If placed as shown in the figure below, then $1-5$ can be taken as a single number, $6=5+1,7=3+4,8=5+1+2$, $9=2+3+4,10=1+2+3+4,11=5+1+2+3,12=4+5+1+2,13=3+4+5+1,14=2+3+4+5,15=1+2+3+4+5$, so all numbers from 1 to 15 are obtained, and this is not a \"central ring\" placement method.)",
      "answer": "2"
    },
    {
      "question": "5. In an $8 \\times 8$ table, natural numbers are arranged. The numbers in cells symmetric with respect to both diagonals of the table are equal. It is known that the sum of all numbers in the table is 1000, and the sum of the numbers on the diagonals is 200. For what smallest number $M$ can we assert that the sum of the numbers in each row does not exceed $M$?",
      "answer": "288"
    },
    {
      "question": "Three. (50 points) Given $n \\in \\mathbf{N}, n>4$, the set $A=$ $\\{1,2, \\cdots, n\\}$. Let the positive integer $m$ and the sets $A_{1}, A_{2}, \\cdots$, $A_{m}$ have the following properties:\n(1) $\\bigcup_{i=1}^{m} A_{i}=A$;\n(2) $\\left|A_{i}\\right|=4(i=1,2, \\cdots, m)$;\n(3) $X_{1}, X_{2}, \\cdots, X_{\\mathrm{C}_{n}^{2}}$ are all the two-element subsets of $A$, for any $X_{k}\\left(k=1,2, \\cdots, \\mathrm{C}_{n}^{2}\\right)$, there exists a unique $j_{k} \\in\\{1,2, \\cdots, m\\}$, such that $X_{k} \\subseteq A_{j_{k}}$.\nFind the minimum value of $n$.",
      "answer": "13"
    },
    {
      "question": "7.3. Egor borrowed 28 rubles from Nikita, and then returned them in four payments. It turned out that Egor always returned a whole number of rubles, and the amount of each payment increased and was divisible by the previous one. What was the last amount Egor paid?",
      "answer": "18"
    },
    {
      "question": "102. A certain number is written in the duodecimal system; for which divisor $m$ is the following divisibility rule valid: if the sum of the digits of the number is divisible by $m$, then the number is also divisible by $m$?",
      "answer": "11"
    },
    {
      "question": "G9.2 There are exactly $n$ values of $\\theta$ satisfying the equation $\\left(\\sin ^{2} \\theta-1\\right)\\left(2 \\sin ^{2} \\theta-1\\right)=0$, where $0^{\\circ} \\leq \\theta \\leq 360^{\\circ}$. Find $n$.",
      "answer": "6"
    },
    {
      "question": "111) Find the maximum value of the real number $\\lambda$. Such that, as long as point $P$ is inside the acute $\\triangle ABC$, $\\angle PAB = \\angle PBC = \\angle PCA$, the rays $AP$, $BP$, $CP$ intersect the circumcircles of $\\triangle PBC$, $\\triangle PCA$, $\\triangle PAB$ at points $A_1$, $B_1$, $C_1$ respectively, then $S_{\\triangle B C} + S_{\\triangle B_1 C A} + S_{\\triangle C_1 A B} \\geq \\lambda S_{\\triangle A B C}$.",
      "answer": "3"
    },
    {
      "question": "Example 1 (2004 National College Entrance Examination for Science) Given that there are 3 weak teams among 8 teams, the 8 teams are divided into groups $A$ and $B$ by drawing lots, with 4 teams in each group.\n(1) The probability that one of the groups $A$ or $B$ has exactly two weak teams;\n(2) The probability that group $A$ has at least two weak teams.",
      "answer": "\\frac{6}{7}"
    },
    {
      "question": "14. (15 points) As shown in the figure, the radii of circular gears $A$, $B$, and $C$ are 15 cm, 10 cm, and 5 cm, respectively, and they are part of a transmission mechanism. When $A$ rotates uniformly, it drives $B$ to rotate uniformly, which in turn drives $C$ to rotate uniformly. Please answer:\n(1) When $A$ rotates uniformly in a clockwise direction, does $C$ rotate in a clockwise or counterclockwise direction?\n(2) When $A$ completes one full rotation, how many full rotations does $C$ complete?",
      "answer": "3"
    },
    {
      "question": "5.1. (12 points) The equation $x^{2}+5 x+1=0$ has roots $x_{1}$ and $x_{2}$. Find the value of the expression\n\n$$\n\\left(\\frac{x_{1} \\sqrt{6}}{1+x_{2}}\\right)^{2}+\\left(\\frac{x_{2} \\sqrt{6}}{1+x_{1}}\\right)^{2}\n$$",
      "answer": "220"
    },
    {
      "question": "3. The height of an isosceles trapezoid is equal to $h$, and its area is equal to $h^{2}$. Determine the angle between the diagonals of this trapezoid.",
      "answer": "90"
    },
    {
      "question": "44. Given that the time it takes for a mouse to run 5 steps is the same as the time it takes for a cat to run 4 steps, and the distance a mouse runs in 9 steps is the same as the distance a cat runs in 7 steps. Now, the mouse and the cat are 2 meters apart. The cat starts to chase the mouse, the cat needs to run $\\qquad$ meters to catch the mouse.",
      "answer": "72"
    },
    {
      "question": "149. Scattered set. In a printing house, a multiplication line of the form $\\overline{a b c} \\cdot \\overline{b c a} \\cdot \\overline{c a b}^{*}$ was being set; but the set scattered, and the digits of the product got mixed up. As a result, the product was printed as 2342355286. It is known that $a>b>c$ and that the units digit of the printed product is the correct digit 6. Restore the true value of the product.",
      "answer": "328245326"
    },
    {
      "question": "Question 192: Nine small balls numbered $1, 2, \\ldots, 9$ are randomly placed on nine equally spaced points on a circle, with one ball on each point. Let $S$ be the sum of the absolute differences of the numbers on all adjacent balls on the circle. Find the probability of the arrangement that minimizes $S$ (Note: If one arrangement can be made to coincide with another by rotation or reflection, they are considered the same arrangement).",
      "answer": "\\frac{1}{315}"
    },
    {
      "question": "The surface of a 3 x 3 x 3 Rubik's cube consists of 54 cells. What is the maximum number of cells that can be marked so that the marked cells do not share any vertices?\n\n#",
      "answer": "14"
    },
    {
      "question": "Condition of the problem\n\nCalculate the limit of the numerical sequence:\n$\\lim _{n \\rightarrow \\infty} \\frac{e^{1 / n}+\\sin \\left(\\frac{n}{n^{2}+1}\\right) \\cdot \\cos n}{1+\\cos \\left(\\frac{1}{n}\\right)}$",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "Jirka drew a square grid with 25 squares, see the image. Then he wanted to color each square so that squares of the same color do not share any vertex.\n\nHow many colors did Jirka need at least?\n\n(M. Dillingerová)\n\n![](https://cdn.mathpix.com/cropped/2024_04_17_adee0188cf1f8dcaeca5g-4.jpg?height=383&width=380&top_left_y=485&top_left_x=838)\n\nHint. Start in one of the corner squares.",
      "answer": "4"
    },
    {
      "question": "Three distinct numbers are given. The average of the average of the two smaller numbers and the average of the two larger numbers is equal to the average of all three numbers. The average of the smallest and the largest number is 2022.\n\nDetermine the sum of the three given numbers.\n\n(K. Pazourek)\n\nHint. Express one of the numbers in terms of the remaining two.",
      "answer": "6066"
    },
    {
      "question": "114 The sides of a right triangle are all positive integers. One of the legs is 1575 less than the hypotenuse, and the other leg is less than 1991. Then the length of the hypotenuse of this right triangle is $\\qquad$ .",
      "answer": "1799"
    },
    {
      "question": "363. The velocity of a material point is given by the formula $v=\\left(4 t^{3}-2 t+1\\right)$ m/s. Find the distance traveled by the point in the first 4 s from the start of the motion.",
      "answer": "244"
    },
    {
      "question": "5. In a football tournament where each team played against each other once, teams $A$, B, V, G, D, and E participated. Teams received 3 points for a win, 1 point for a draw, and 0 points for a loss. In the end, it turned out that teams $A$, B, V, G, and D each scored 7 points. What is the maximum number of points that team E could have scored?",
      "answer": "7"
    },
    {
      "question": "B3 In the figure, three squares are inscribed in a right-angled triangle (see figure). Calculate the side $x$ of the third square exactly.\n\n(6 points)\n\n![](https://cdn.mathpix.com/cropped/2024_06_07_c987701c9bb62194911fg-05.jpg?height=371&width=739&top_left_y=1077&top_left_x=1178)",
      "answer": "\\frac{50}{3}"
    },
    {
      "question": "B2. The expression is $2-\\frac{x-2}{5}+\\frac{1+x}{4}$.\n\nA. Calculate the value of the expression for $x=5$.\n\nB. Determine the value of $x$ such that the value of the expression is 4.\n\nC. Determine the value of $x$ such that the value of the expression is at least 2.\n\nD. Find the smallest natural number $x$ such that the value of the expression is an integer.",
      "answer": "7"
    },
    {
      "question": "Raskinna I.V.\n\nWinnie-the-Poo, Piglet, Rabbit, and Donkey Eeyore emptied a barrel of honey. Piglet ate half of what Winnie-the-Poo ate, Rabbit ate half of what Winnie-the-Poo did not eat, and Donkey Eeyore got only a tenth of the barrel. What part of the barrel did Rabbit get?",
      "answer": "\\frac{3}{10}"
    },
    {
      "question": "$\\frac{\\sin \\alpha \\sin (\\alpha+\\beta)+\\cos \\alpha \\cos (\\alpha+\\beta)}{\\cos \\beta}=?$",
      "answer": "1"
    },
    {
      "question": "6. (10 points) A class plants trees on Tree Planting Day, divided into 3 groups. The first group plants 5 trees per person, the second group plants 4 trees per person, and the third group plants 3 trees per person. It is known that the number of people in the second group is one-third of the sum of the number of people in the first and third groups, and the number of trees planted by the second group is 72 less than the sum of the number of trees planted by the first and third groups. Then, the class has at least $\\qquad$ people.",
      "answer": "32"
    },
    {
      "question": "1. (5 points) Calculate: $3 \\frac{3}{4} \\times 1.3 + 3 \\div 2 \\frac{2}{3}=$",
      "answer": "6"
    },
    {
      "question": "Question 64: Given $a, b \\in [1, \\sqrt{3}]$, find the maximum and minimum values of $\\frac{a^{2}+b^{2}-1}{a b}$.\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "1"
    },
    {
      "question": "10. In a math competition, there are 5 questions, each with a different natural number score. The smaller the question number, the fewer points the question is worth (for example, the score of question 1 is less than the score of question 2). Xiao Ming answered all the questions correctly, and his total score for the first 2 questions is 10 points, and the total score for the last 2 questions is 18 points. Therefore, Xiao Ming's total score is ( ) points.",
      "answer": "35"
    },
    {
      "question": "11. Toll Road (from 8th grade. 3 points). The cost of traveling along a section of a toll road depends on the class of the vehicle: passenger cars belong to the first class, for which the travel cost is 200 rubles, and light trucks and minivans belong to the second class, for which the cost is 300 rubles.\n\nAt the entrance to the toll booth, an automatic system is installed that measures the height of the vehicle. If the height of the vehicle is less than a certain threshold value \\( h \\), the system automatically classifies it as class 1; if it is higher than \\( h \\), it is classified as class 2. Errors are possible. For example, a low minivan may be classified as class 1 by the system, and its driver will be pleased. A low SUV may be incorrectly classified as class 2, and its driver will not be happy. The driver can file a claim, and the operating company will have to refund 100 rubles.\n\nThe company's management tasked the engineers with the goal of configuring the system so that the number of errors is minimized.\n\nFor several weeks, the engineers collected data on the height of vehicles passing through the toll booth. Separately for class 1 vehicles and separately for class 2 vehicles (Fig. 5). On the x-axis, the height of the vehicle (in cm) is plotted, and on the y-axis, the average number of vehicles of such height passing through the toll booth per day. For clarity, the points are connected by a smooth line.\n![](https://cdn.mathpix.com/cropped/2024_05_06_a6149daf2f3a04ecb66bg-05.jpg?height=486&width=1628&top_left_y=274&top_left_x=266)\n\nFig. 5. Graphs - the number of class 1 and class 2 vehicles per day\n\nHaving gathered all this information and drawn the graphs, the engineers began to think about what to do next: how to determine the threshold value \\( h \\) so that the probability of error is the smallest possible? Solve this problem for them.",
      "answer": "190"
    },
    {
      "question": "Problem 9.8. On the side $CD$ of trapezoid $ABCD (AD \\| BC)$, a point $M$ is marked. A perpendicular $AH$ is dropped from vertex $A$ to segment $BM$. It turns out that $AD = HD$. Find the length of segment $AD$, given that $BC = 16$, $CM = 8$, and $MD = 9$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d55ea3e8288d58a3f771g-35.jpg?height=444&width=589&top_left_y=743&top_left_x=432)",
      "answer": "18"
    },
    {
      "question": "I am Ivan and I am three times younger than my father. I have brothers Vincent and Jakub, who are 11 and 9 years old. My age is equal to five times the third of the age of the younger of the brothers. A peculiarity of our family is that we were all born on April 12th, so today we are celebrating our birthdays.\n\nHow many years will it take for the age of our father to be equal to the sum of the ages of the three of us? (L. Hozová)",
      "answer": "5"
    },
    {
      "question": "4. The integer part of the result of the expression $0.81 \\times 0.6+0.83 \\times 0.6+\\cdots+0.97 \\times 0.6+0.99 \\times 0.6$ is\n$\\qquad$ .",
      "answer": "5"
    },
    {
      "question": "By how much should ${ }^{4} \\log 8$ be multiplied to obtain ${ }^{32} \\log 8$?",
      "answer": "\\frac{2}{5}"
    },
    {
      "question": "6. In the Cartesian coordinate system $x O y$, the point set $K=\\{(x, y) \\mid x, y=-1,0,1\\}$. If three points are randomly selected from $K$, then the probability that the distance between each pair of these three points is no more than 2 is $\\qquad$ .",
      "answer": "\\frac{5}{14}"
    },
    {
      "question": "In 8th grade class \"G\", there are enough underachievers, but Vovochka studies the worst of all. The pedagogical council decided that either Vovochka must correct his twos by the end of the quarter, or he will be expelled. If Vovochka corrects his twos, then 24% of the class will be underachievers, and if he is expelled, the underachievers will become 25%. What percentage of underachievers are there in 8 \"G\" now? #",
      "answer": "28"
    },
    {
      "question": "We consider those natural numbers to be somewhat unlucky whose sum of digits in decimal notation is divisible by 13. What is the maximum difference between consecutive somewhat unlucky numbers?",
      "answer": "79"
    },
    {
      "question": "10. $f(x)=\\frac{2}{4^{x}+2}+2$, then $f\\left(\\frac{1}{m}\\right)+f\\left(\\frac{2}{m}\\right)+\\cdots+f\\left(\\frac{m-1}{m}\\right)=$ $\\ldots(m \\geqslant 2, m \\in \\mathbf{N})$.",
      "answer": "\\frac{5}{2}(-1)"
    },
    {
      "question": "Problem 9. In a convex quadrilateral $A B C D$, side $A B$ is equal to diagonal $B D, \\angle A=65^{\\circ}$, $\\angle B=80^{\\circ}, \\angle C=75^{\\circ}$. What is $\\angle C A D$ (in degrees $) ?$",
      "answer": "15"
    },
    {
      "question": "3. Arrange the numbers in the set $\\left\\{2^{x}+2^{y} \\mid x 、 y \\in \\mathbf{N}, x<y\\right\\}$ in ascending order. Then the 60th number is $\\qquad$ (answer with a number).\n\nTranslate the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "2064"
    },
    {
      "question": "1. Find all prime numbers $p$ such that\n$$\n1+2^{p}+3^{p}+\\cdots+p^{p}\n$$\n\nis a prime number.",
      "answer": "5"
    },
    {
      "question": "Example 9 Let real numbers $s, t$ satisfy the equations $19 s^{2}+99 s+1=0, t^{2}+99 t+19=0$, and $s t \\neq 1$.\nThen $\\frac{s t+4 s+1}{t}=$ $\\qquad$ .",
      "answer": "-5"
    },
    {
      "question": "7. In tetrahedron $A B C D$, $\\angle A D B=\\angle B D C$ $=\\angle C D A=60^{\\circ}, A D=B D=3, C D=2$. Then the volume of the circumscribed sphere of tetrahedron $A B C D$ is $\\qquad$ .",
      "answer": "4\\sqrt{3}\\pi"
    },
    {
      "question": "5. In an $m \\times 10$ grid, each cell is filled with either 0 or 1, such that each column contains exactly three 1s. The sum of the numbers in the $i(i=1,2, \\cdots, m)$-th row is denoted as $x_{i}$, and for any two columns, there always exists a row where the cells at the intersection with these two columns are both filled with 1. Let $x=\\max _{1 \\leqslant i \\leqslant m}\\left\\{x_{i}\\right\\}$. Then $x_{\\text {min }}=$",
      "answer": "5"
    },
    {
      "question": "7. (10 points) If $2 \\times 3^{8}$ can be expressed as the sum of $k$ consecutive positive integers, then the maximum value of $k$ is\n\nExpress the above text in English, keeping the original text's line breaks and format, and output the translation result directly.",
      "answer": "108"
    },
    {
      "question": "13. (15 points) A speedboat departs from dock $A$, travels downstream along the river, passes through dock $B$, and continues downstream to dock $C$. Upon reaching dock $C$, it immediately turns around and heads back to dock $B$, taking a total of 10 hours. If the distance between $A$ and $B$ is 20 kilometers, the speed of the speedboat in still water is 40 kilometers/hour, and the speed of the river current is 10 kilometers/hour, find the distance between $B$ and $C$.",
      "answer": "180"
    },
    {
      "question": "Find the minimum of the function\n\n$$\nf(x, y)=\\sqrt{(x+1)^{2}+(2 y+1)^{2}}+\\sqrt{(2 x+1)^{2}+(3 y+1)^{2}}+\\sqrt{(3 x-4)^{2}+(5 y-6)^{2}} \\text {, }\n$$\n\ndefined for all real $x, y>0$.",
      "answer": "10"
    },
    {
      "question": "6. Let real numbers $a, b, c$ satisfy $a^{2}+b^{2}+c^{2}=1$, and let the maximum and minimum values of $ab+bc+ca$ be $M, m$ respectively. Then $M-m=$ $\\qquad$ .",
      "answer": "\\frac{3}{2}"
    },
    {
      "question": "Esquecinaldo has a terrible memory for remembering numbers, but excellent for remembering sequences of operations. Therefore, to remember his five-digit bank code, he can remember that the code does not have repeated digits, none of the digits is zero, the first two digits form a power of 5, the last two digits form a power of 2, the middle digit is a multiple of 3, and the sum of all the digits is an odd number. Now he no longer needs to memorize the number, because he knows that his code is the largest number that satisfies these conditions. What is this code?",
      "answer": "25916"
    },
    {
      "question": "21. Ten people exchanged photographs. How many photographs were needed?",
      "answer": "90"
    },
    {
      "question": "4. On a circle, 100 points are marked, painted either red or blue. Some of the points are connected by segments, with each segment having one blue end and one red end. It is known that no two red points belong to the same number of segments. What is the maximum possible number of red points?",
      "answer": "50"
    },
    {
      "question": "## Task 4.\n\nDetermine all natural numbers $n \\geqslant 3$ for which the product of the first $n$ natural numbers divides the product of all sums of distinct pairs of prime numbers not greater than $n$, i.e., for which\n\n$$\nn! \\mid \\prod_{\\substack{p<q \\leqslant n \\\\ p, q \\text{ prime }}}(p+q)\n$$",
      "answer": "7"
    },
    {
      "question": "19. Erin lists all three-digit primes that are 21 less than a square. What is the mean of the numbers in Erin's list?",
      "answer": "421"
    },
    {
      "question": "Every day, the dog Patrick gnaws one slipper from the available supply in the house. With a probability of 0.5, Patrick wants to gnaw a left slipper and with a probability of 0.5 - a right slipper. If the desired slipper is not available, Patrick gets upset. How many pairs of identical slippers need to be bought so that with a probability of at least 0.8, Patrick does not get upset for a whole week (7 days)?",
      "answer": "5"
    },
    {
      "question": "5.1. (12 points) The decreasing sequence $a, b, c$ is a geometric progression, and the sequence $19 a, \\frac{124 b}{13}, \\frac{c}{13}$ is an arithmetic progression. Find the common ratio of the geometric progression.",
      "answer": "247"
    },
    {
      "question": "3. Find the least positive number $x$ with the following property: if $a, b, c, d$ are arbitrary positive numbers whose product is 1 , then\n\n$$\na^{x}+b^{x}+c^{x}+d^{x} \\geqslant \\frac{1}{a}+\\frac{1}{b}+\\frac{1}{c}+\\frac{1}{d} .\n$$",
      "answer": "3"
    },
    {
      "question": "18. The Absent-Minded Scientist on the Bus (4 p. 6-11). In the bus, there are $n$ seats, and all tickets have been sold to $n$ passengers. The first to enter the bus is the Absent-Minded Scientist, who, without looking at the ticket, takes the first available seat. Subsequently, passengers enter one by one. If an entering passenger sees that their seat is free, they take their seat. If the seat is occupied, the entering passenger takes the first available free seat. Find the probability that the last passenger to enter will take the seat according to their ticket?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "(12) $O, A, B$ are three non-collinear points on a plane, vector $\\overrightarrow{O A}=\\vec{a}, \\overrightarrow{O B}=\\vec{b}$, let $P$ be any point on the perpendicular bisector of line segment $A B$, vector $\\overrightarrow{O P}=\\vec{p}$. If $|\\vec{a}|=5,|\\vec{b}|=3$, then the value of $\\vec{p} \\cdot(\\vec{a}-\\vec{b})$ is $\\qquad$.",
      "answer": "8"
    },
    {
      "question": "13. Move 1 matchstick to make the equation in the figure correct. Write the equation after moving on the line:\n$\\qquad$ (Note: The \" $\\mathrm{Z}$ \" in the equation represents 2 )\n\n\n\nPlease write the equation after moving the matchstick on the line:",
      "answer": "11+1-2-7"
    },
    {
      "question": "11. Define $a \\oplus b=(a+2)(b+2)-2$. The calculation result of $1 \\times 3 \\times 5 \\times 7 \\times 9 \\times 11 \\times 13-(1 \\bigcirc 30507 \\sim 9011)$ is $\\qquad$",
      "answer": "2"
    },
    {
      "question": "I4.1 若 $m$ 和 $n$ 為正整數及 $a=\\log _{2}\\left[\\left(\\frac{m^{4} n^{-4}}{m^{-1} n}\\right)^{-3} \\div\\left(\\frac{m^{-2} n^{2}}{m n^{-1}}\\right)^{5}\\right]$, 求 $a$ 的值。\nIf $m$ and $n$ are positive integers and $a=\\log _{2}\\left[\\left(\\frac{m^{4} n^{-4}}{m^{-1} n}\\right)^{-3} \\div\\left(\\frac{m^{-2} n^{2}}{m n^{-1}}\\right)^{5}\\right]$, determine the value of $a$.",
      "answer": "0"
    },
    {
      "question": "[ Height of a pyramid (tetrahedron).]\n\nThe base of the pyramid is a right triangle with a hypotenuse of 6 and an acute angle of \\(15^{\\circ}\\). All lateral edges are inclined to the plane of the base at an angle of \\(45^{\\circ}\\). Find the volume of the pyramid.",
      "answer": "4.5"
    },
    {
      "question": "117. Find the minimum value of the expression\n\n$$\n(x-5)(x-1)(x-6)(x-2)+9\n$$",
      "answer": "5"
    },
    {
      "question": "3. The equation $a x^{2}-b x+c=0$ has two distinct roots $x_{1}, x_{2}$. What is the maximum number of numbers in the set $\\left\\{a, b, c, x_{1}, x_{2}\\right\\}$ that can be prime (and, accordingly, natural) numbers? If a prime number appears in the set twice, it should be counted twice. The number 1 is not considered a prime number.",
      "answer": "4"
    },
    {
      "question": "[Text problems $]$\n\nLiza is 8 years older than Nastya. Two years ago, she was three times as old as Nastya. How old is Liza?\n\n#",
      "answer": "14"
    },
    {
      "question": "## Problem 4\n\nIn the cube $\\mathrm{ABCDA}^{\\prime} \\mathrm{B}^{\\prime} \\mathrm{C}^{\\prime} \\mathrm{D}^{\\prime}$, points $\\mathrm{M}, \\mathrm{N}$, and $\\mathrm{P}$ are the midpoints of the edges $\\mathrm{CC}$ ', $\\mathrm{A}^{\\prime} \\mathrm{D}^{\\prime}$, and $\\mathrm{C}^{\\prime} \\mathrm{D}^{\\prime}$, respectively. Find the sine of the angle formed by the lines BM and NP.\n\nProblems selected by Lung Ioan and Nicoara Florin\n\nNote: a) The actual working time is 3 hours.\n\nb) All problems are mandatory.\n\nc) Each problem is scored from 0 to 7.\n\n## NATIONAL MATHEMATICS OLYMPIAD\n\nLocal stage - 15.02.2014\n\n## Grade 8\n\n## Grading Rubric",
      "answer": "\\frac{\\sqrt{3}}{\\sqrt{5}}"
    },
    {
      "question": "[The ratio of the areas of triangles with a common base or common height] [The area of a figure is equal to the sum of the areas of the figures into which it is divided\n\nPoints $M$ and $N$ are located on side $A C$ of triangle $A B C$, and points $K$ and $L$ are on side $A B$, such that $A M: M N: N C=1: 3: 1$ and $A K=K L=L B$. It is known that the area of triangle $A B C$ is 1. Find the area of quadrilateral $K L N M$.",
      "answer": "\\frac{7}{15}"
    },
    {
      "question": "A round table has a diameter of $1.40 \\, \\text{m}$. For a party, the table is expanded by adding three planks, each $40 \\, \\text{cm}$ wide, as shown in the figure. If each person at the table should have a space of $60 \\, \\text{cm}$, how many guests can sit at the table?\n![](https://cdn.mathpix.com/cropped/2024_05_01_bd5219fde2a4a59a7005g-2.jpg?height=506&width=724&top_left_y=618&top_left_x=996)",
      "answer": "11"
    },
    {
      "question": "1. If the sequence $\\left\\{a_{n}\\right\\}$ satisfies $a_{1}=\\frac{1}{2}, a_{n+1}=\\frac{2 a_{n}}{3 a_{n}+2}, n \\in \\mathbf{N}_{+}$, then $a_{2017}=$ $\\qquad$ .",
      "answer": "\\frac{1}{3026}"
    },
    {
      "question": "4. In how many ways can seven different items (3 weighing 2 tons each, 4 weighing 1 ton each) be loaded into two trucks with capacities of 6 tons and 5 tons, if the arrangement of the items inside the trucks does not matter? (12 points)\n\n#",
      "answer": "46"
    },
    {
      "question": "$[$ Mathematical logic (miscellaneous).]\n\nAuthors: Folklore, Folklore\n\nIn a school of witchcraft, there are 13 students. Before the clairvoyance exam, the teacher seated them at a round table and asked them to guess who would receive the clairvoyant diploma. They modestly kept quiet about themselves and their two neighbors, but wrote about everyone else: \"None of these ten will get it!\" Of course, all those who passed the exam guessed correctly, while all the others were wrong. How many wizards received the diploma?\n\n#",
      "answer": "2"
    },
    {
      "question": "14. There is a ten-digit number. Counting from left to right, if its first digit is a certain number, then this ten-digit number contains that many 0s; if its second digit is a certain number, then this ten-digit number contains that many 1s; if its third digit is a certain number, then this ten-digit number contains that many 2s; and so on; if its tenth digit is a certain number, then this ten-digit number contains that many 9s. This ten-digit number is $\\qquad$",
      "answer": "6210001000"
    },
    {
      "question": "## 5. Black and White Balls\n\nAn urn contains 5 white and 5 black balls. Then, 3 balls are randomly drawn one by one. This is done in two ways: 1 - the drawn ball is returned to the urn, 2 - the balls are not returned. In which of these two cases is there a higher probability of drawing one white and two black balls?",
      "answer": "\\frac{10}{24}"
    },
    {
      "question": "1. Calculate in the most rational way\n\n$$\n12 \\times 171+29 \\times 9+171 \\times 13+29 \\times 16\n$$",
      "answer": "5000"
    },
    {
      "question": "[ Regular pyramid ] [ Sections, unfoldings, and other frameworks. ]\n\nOn the extension of edge $S T$ beyond point $T$ of the regular quadrilateral pyramid $S P Q R T$ with vertex $S$, a point $B$ is taken such that the distance from it to the plane $S P Q$ is $\\frac{9 \\sqrt{7}}{2}$. Find the segment $B T$, if $Q R=12$, and $S R=10$.",
      "answer": "5"
    },
    {
      "question": "10. Determine the units digit of $(2!+2)(3!+3)(4!+2)(5!+3) \\cdots(2018!+2)(2019!+3)$.",
      "answer": "8"
    },
    {
      "question": "## Task B-1.5.\n\nInside an equilateral triangle $ABC$ with side length $12 \\, \\text{cm}$, a point $T$ is chosen such that $|TP|:|TQ|:|TR|=1:2:3$, where $P, Q$, and $R$ are the feet of the perpendiculars from point $T$ to the sides of the triangle. What is the area of triangle $\\triangle PQR$?",
      "answer": "\\frac{33\\sqrt{3}}{4}^2"
    },
    {
      "question": "A square has side length $s$ and a diagonal of length $s+1$. Write the area of the square in the form $a+b \\sqrt{c}$ where $a, b$ and $c$ are positive integers.",
      "answer": "3+2\\sqrt{2}"
    },
    {
      "question": "13. (12 points) Cars A and B start from point A to point B at the same time. At the start, car A's speed is 2.5 kilometers per hour faster than car B's. After 10 minutes, car A slows down; 5 minutes later, car B also slows down, at which point car B is 0.5 kilometers per hour slower than car A. Another 25 minutes later, both cars arrive at point B at the same time. By how many kilometers per hour did car A's speed decrease at that time?",
      "answer": "10"
    },
    {
      "question": "1. For a set of real numbers $A$, let $A-A$ be the set of distinct pairwise differences of $A$; that is,\n$$\nA-A:=\\{a-b: a, b \\in A\\}\n$$\n\nIf $|A-A|=25$, find the sum of all possible values of $|A|$.",
      "answer": "76"
    },
    {
      "question": "7. Given the polynomial $P(x)=x^{5}+a_{4} x^{4}+a_{3} x^{3}+a_{2} x^{2}+a_{1} x+a_{0}$, it is known that $P(2014)=1, P(2015)=2, P(2016)=3, P(2017)=4$, $P(2018)=5$. Find $P(2013)$.",
      "answer": "-120"
    },
    {
      "question": "We denote by $\\mathbb{Z}[x, y, z]$ the set of polynomials with integer coefficients in the three variables $x, y$, and $z$. We then say that a polynomial $P \\in \\mathbb{Z}[x, y, z]$ is Olympic if $\\mathbb{Z}[x, y, z]$ contains polynomials $A, B$, and $C$ such that\n\n$$\nP(x, y, z)=(x+y+z) A(x, y, z)+(x y+y z+z x) B(x, y, z)+x y z C(x, y, z).\n$$\n\nFind the largest integer $n$ for which there exist natural numbers $i, j$, and $k$ with sum $i+j+k=n$ and such that the polynomial $x^{i} y^{j} z^{k}$ is not Olympic.\nNote: A polynomial with integer coefficients in the variables $x, y$, and $z$ is a function that can be written as a sum of terms of the form $\\lambda x^{i} y^{j} z^{k}$, where $\\lambda$ is an integer and $i, j$, and $k$ are natural numbers. For example, $x-y+1$ and $x y+y z+z x$ are such polynomials, but $\\pi x y z, \\exp (x), x /\\left(y^{2}+1\\right)$, and $\\sqrt{x y+z}$ are not.",
      "answer": "3"
    },
    {
      "question": "The points $P(2,0), Q(11,-3)$ and $R(x, 3)$ are the vertices of a triangle with $\\angle P Q R=90^{\\circ}$. What is the value of $x$ ?",
      "answer": "13"
    },
    {
      "question": "(10 points) Calculate: $2016 \\times 2016 - 2015 \\times 2016=$",
      "answer": "2016"
    },
    {
      "question": "1. Let $A B C$ be a triangle with $\\angle A=60^{\\circ}$. Line $\\ell$ intersects segments $A B$ and $A C$ and splits triangle $A B C$ into an equilateral triangle and a quadrilateral. Let $X$ and $Y$ be on $\\ell$ such that lines $B X$ and $C Y$ are perpendicular to $\\ell$. Given that $A B=20$ and $A C=22$, compute $X Y$.",
      "answer": "21"
    },
    {
      "question": "3. Compute the greatest common divisor of $4^{8}-1$ and $8^{12}-1$.",
      "answer": "15"
    },
    {
      "question": "1537. In an urn, there are 5 white and 4 black balls. Two balls are drawn in succession. Find the probability that both balls are white.",
      "answer": "\\frac{5}{18}"
    },
    {
      "question": "1. [2] What is the smallest non-square positive integer that is the product of four prime numbers (not necessarily distinct)?",
      "answer": "24"
    },
    {
      "question": "9. Let $F$ be the left focus of the ellipse $E: \\frac{x^{2}}{3}+y^{2}=1$. A line $l$ passing through point $F$ with a positive slope intersects $E$ at points $A$ and $B$. Lines $A M$ and $B N$ are drawn through points $A$ and $B$ respectively, such that $A M \\perp l, B N \\perp l$, and lines $A M, B N$ intersect the $x$-axis at points $M$ and $N$. Find the minimum value of $|M N|$.",
      "answer": "\\sqrt{6}"
    },
    {
      "question": "9. Team A and Team B each send out 7 players to participate in a Go chess tournament according to a pre-arranged order. Both sides start with the No. 1 player competing, the loser is eliminated, and the winner then competes with the No. 2 player of the losing side, ..., until all players of one side are eliminated, and the other side wins, forming a match process. Then the total number of all possible match processes is $\\qquad$ .",
      "answer": "3432"
    },
    {
      "question": "## Task B-4.3.\n\nHow many isosceles trapezoids, with bases of different lengths, have integer side lengths and a perimeter of 2010?",
      "answer": "251502"
    },
    {
      "question": "5. There were 5 times more strawberry bushes on the first bed than on the second. When 22 bushes were transplanted from the first bed to the second, the number of strawberry bushes on each bed became the same. How many bushes were there on each bed?",
      "answer": "11"
    },
    {
      "question": "Example 6 Given the function $f(x)=x^{2}-a x+\\frac{a}{2}(a>0), x \\in[0,1]$, find the minimum value $g(a)$ of $f(x)$, and then find the maximum value of $g(a)$.",
      "answer": "\\frac{1}{4}"
    },
    {
      "question": "Task 5. (20 points) In the center of a circular field stands a geologists' cabin. From it, 6 straight roads extend, dividing the field into 6 equal sectors. Two geologists set out on a journey from their cabin at a speed of 4 km/h along a road each arbitrarily chooses. Determine the probability that the distance between them after one hour will be at least 6 km.\n\n#",
      "answer": "0.5"
    },
    {
      "question": "1. Each of the 5 brothers owns a plot of land. One day, they pooled their money, bought a neighbor's garden, and divided the new land equally among themselves. As a result, Andrey's plot increased by $10 \\%$, Boris's plot - by $\\frac{1}{15}$, Vladimir's plot - by $5 \\%$, Grigory's plot - by $4 \\%$, and Dmitry's plot - by $\\frac{1}{30}$. By what percentage did the total area of their land increase as a result?",
      "answer": "5"
    },
    {
      "question": "Segment $A B$ is the diameter of a circle. A second circle with center at point $B$ has a radius of 2 and intersects the first circle at points $C$ and $D$. Chord $C E$ of the second circle is part of a tangent to the first circle and is equal to 3. Find the radius of the first circle.",
      "answer": "\\frac{4}{\\sqrt{7}}"
    },
    {
      "question": "30. If $\\sin \\theta-\\cos \\theta=\\frac{\\sqrt{6}-\\sqrt{2}}{2}$, find the value of $24\\left(\\sin ^{3} \\theta-\\cos ^{3} \\theta\\right)^{2}$,",
      "answer": "12"
    },
    {
      "question": "According to the conditions of the chess match, the winner is the one who outperforms the opponent by two wins. Draws do not count. The probabilities of winning for the opponents are equal. The number of decisive games in such a match is a random variable. Find its mathematical expectation.",
      "answer": "4"
    },
    {
      "question": "3. Petrov and Vasechkin were repairing a fence. Each had to nail a certain number of boards (the same amount). Petrov nailed two nails into some boards, and three nails into the rest. Vasechkin nailed three nails into some boards, and five nails into the rest. Find out how many boards each of them nailed, if it is known that Petrov nailed 87 nails, and Vasechkin nailed 94 nails.",
      "answer": "30"
    },
    {
      "question": "8. Xiao Yu's family consists of Dad, Mom, Brother, Sister, and Xiao Yu, a total of five people. From the 1st to the 5th of this month, there are two household chores every day: one person cooks, and another person washes the dishes, and everyone has to cook once and wash the dishes once in these 5 days. One day they had the following conversation:\nBrother: I just washed the dishes yesterday, and I have to cook tomorrow, so tiring!\nSister: I am even more tired, I have chores both tomorrow and the day after!\nXiao Yu: Dad and I always do chores on the same day.\nDad: Yes, fortunately, after cooking, I can rest for two days before it's my turn to wash the dishes again.\nSo the five-digit number formed by the dates when Dad, Mom, Brother, Sister, and Xiao Yu wash the dishes in sequence is $\\qquad$",
      "answer": "54132"
    },
    {
      "question": "1. Find all positive real numbers $x$ that satisfy the equation $x^{x \\sqrt[3]{x}}=(x \\sqrt[3]{x})^{x}$.",
      "answer": "1"
    },
    {
      "question": "12.11 $f(x)=\\frac{(x-2)^{2}}{x^{2}}$.",
      "answer": "2"
    },
    {
      "question": "7.2. Find the sum of all three-digit natural numbers that do not contain the digits 0 or 9 in their representation.",
      "answer": "255744"
    },
    {
      "question": "For example, among 211 theater troupes, some troupes perform each day while others watch (those who perform cannot watch). If each troupe has seen performances by 10 other troupes, how many days are needed for the performances at least?\n\n\n\nTranslating the above text into English, please keep the original text's line breaks and format, and output the translation result directly.",
      "answer": "6"
    },
    {
      "question": "5. Point $A_{1}$ lies on side $B C$ of triangle $A B C$, point $B_{1}$ lies on side $A C$. Let $K$ be the intersection point of segments $A A_{1}$ and $B B_{1}$. Find the area of quadrilateral $A_{1} C B_{1} K$, given that the area of triangle $A_{1} B K$ is 5, the area of triangle $A B_{1} K$ is 8, and the area of triangle $A B K$ is 10.",
      "answer": "22"
    },
    {
      "question": "A palindrome is a positive integer that reads the same forward and backward, like 2552 or 1991. Find a positive integer greater than 1 that divides all four-digit palindromes.",
      "answer": "11"
    },
    {
      "question": "18. Lonely Cars. (From 9th grade, 4 points.) On a very long narrow highway, where overtaking is impossible, $n$ cars are driving in a random order, each with its own preferred speed. If a fast car catches up to a slower one, the fast car has to slow down and drive at the same speed as the slower one. Thus, the cars form groups. Find the expected number of \"lonely\" cars, that is, groups consisting of a single car.",
      "answer": "1"
    },
    {
      "question": "4. In the computer center, there are 200 computers, some of which (in pairs) are connected by cables, a total of 345 cables are used. We will call a \"cluster\" a set of computers such that a signal from any computer in this set can reach all other computers via cables (possibly through intermediate computers). Initially, all computers formed one cluster. But one night, a malicious hacker cut several cables, resulting in 8 clusters. Find the maximum possible number of cables that were cut.",
      "answer": "153"
    },
    {
      "question": "146. Find $\\lim _{x \\rightarrow 0} \\frac{1-\\cos x}{x^{2}}$.",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "20. Let $a, b, c, d$ be positive integers such that the least common multiple (L.C.M.) of any three of them is $3^{3} \\times 7^{5}$. How many different sets of $(a, b, c, d)$ are possible if the order of the numbers is taken into consideration?\n(2 marks)\n已知 $a 、 b 、 c 、 d$ 爲正整數, 當中任意三個數的最小公倍數 (L.C.M. ) 圴爲 $3^{3} \\times 7^{5}$ 。若需考虑各數的先後次序, 問 $(a, b, c, d)$ 有多少組不同的可能値?",
      "answer": "11457"
    },
    {
      "question": "12. (NET) ${ }^{\\mathrm{IMO} 3}$ Determine the maximum value of $m^{2}+n^{2}$ where $m$ and $n$ are integers satisfying  \n$$ m, n \\in\\{1,2, \\ldots, 100\\} \\quad \\text { and } \\quad\\left(n^{2}-m n-m^{2}\\right)^{2}=1 $$",
      "answer": "987^2 + 1597^2"
    },
    {
      "question": "414. The line $L$ passes through the point $(2 ; 1 ;-1)$ and the point of intersection of the line\n\n$$\n\\frac{x-3}{2}=\\frac{y}{-1}=\\frac{z+1}{1}\n$$\n\nwith the plane\n\n$$\nx-y+z-1=0\n$$\n\nFind the angle formed by the line $L$ with the plane\n\n$$\nx+2 y-z+3=0\n$$",
      "answer": "197^{\\}"
    },
    {
      "question": "9.5. Three hundred non-zero integers are written in a circle such that each number is greater than the product of the next three numbers in the clockwise direction. What is the maximum number of positive numbers that can be among these 300 written numbers?",
      "answer": "200"
    },
    {
      "question": "(7) The function $f(x)=\\frac{\\sin \\left(x+45^{\\circ}\\right)}{\\sin \\left(x+60^{\\circ}\\right)}, x \\in\\left[0^{\\circ}, 90^{\\circ}\\right]$, then the product of the maximum and minimum values of $f(x)$ is . $\\qquad$",
      "answer": "\\frac{2\\sqrt{3}}{3}"
    },
    {
      "question": "11. Color the 17 squares in the \"T\" shape on the right using black and white. If each square can only be one color, and no two black squares can be adjacent (share an edge), then there are $\\qquad$ coloring schemes.\n\nPaint the 17 squares in the \"T\" shape on the right using black and white. If each square can only be one color, and no two black squares can be adjacent (share an edge), then there are $\\qquad$ coloring schemes.",
      "answer": "4370"
    },
    {
      "question": "# 11.2 Solve the equation\n\n$$\n\\frac{1}{\\sqrt{x+20}+\\sqrt{x+21}}+\\frac{1}{\\sqrt{x+21}+\\sqrt{x+22}}=1\n$$",
      "answer": "-\\frac{79}{4}"
    },
    {
      "question": "10. (10 points) Convert the 2015 fractions $\\frac{1}{2}, \\frac{1}{3}, \\frac{1}{4} \\cdots \\frac{1}{2014}, \\frac{1}{2015}, \\frac{1}{2016}$ into decimals. How many of them are finite decimals?",
      "answer": "33"
    },
    {
      "question": "3. (GDR 1) Knowing that the system \n\\[ \\begin{aligned} x+y+z & =3, \\\\ x^{3}+y^{3}+z^{3} & =15, \\\\ x^{4}+y^{4}+z^{4} & =35, \\end{aligned} \\]\nhas a real solution \\(x, y, z\\) for which \\(x^{2}+y^{2}+z^{2}<10\\), find the value of \\(x^{5}+y^{5}+z^{5}\\) for that solution.",
      "answer": "83"
    },
    {
      "question": "2. Square $A B C D$ and square $A B E F$ are in planes that form a $120^{\\circ}$ angle, $M, N$ are points on the diagonals $A C$, $B F$ respectively, and $A M=F N$. If $A B=1$, then the maximum value of $M N$ is $\\qquad$ .",
      "answer": "1"
    },
    {
      "question": "23. If $x$ is positive, find the minimum value of $\\frac{\\sqrt{x^{4}+x^{2}+2 x+1}+\\sqrt{x^{4}-2 x^{3}+5 x^{2}-4 x+1}}{x}$. If $x$ is positive, find the minimum value of $\\frac{\\sqrt{x^{4}+x^{2}+2 x+1}+\\sqrt{x^{4}-2 x^{3}+5 x^{2}-4 x+1}}{x}$.",
      "answer": "\\sqrt{10}"
    },
    {
      "question": "5-1. A square with a side of 100 was cut into two equal rectangles. They were placed next to each other as shown in the figure. Find the perimeter of the resulting figure.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_535cfcd84333341737d0g-04.jpg?height=277&width=594&top_left_y=684&top_left_x=731)",
      "answer": "500"
    },
    {
      "question": "Let * be a natural number $n$ a three-digit number, the sum of all three-digit numbers formed by the arbitrary permutation of its three non-zero digits minus $n$ equals 1990. Find $n$.",
      "answer": "452"
    },
    {
      "question": "Given a fixed triangle $\\triangle A B C$ and a point $P$, find the maximum value of\n\n$$\n\\frac{A B^{2}+B C^{2}+C A^{2}}{P A^{2}+P B^{2}+P C^{2}}\n$$",
      "answer": "3"
    },
    {
      "question": "2. Compute\n\n$$\n\\int_{0}^{\\pi / 2} \\cos ^{2}(\\cos x)+\\sin ^{2}(\\sin x) d x\n$$",
      "answer": "\\frac{\\pi}{2}"
    },
    {
      "question": "2. Plane $\\alpha // \\beta, AB=6 \\text{ cm}$ is a fixed line segment between $\\alpha$ and $\\beta$, $CD$ is a moving line segment between $\\alpha$ and $\\beta$, $CD \\perp AB$. Then when the angle between $AB$ and $\\beta$ is $60^{\\circ}$, the minimum value of $CD$ is $\\qquad$",
      "answer": "6\\sqrt{3}"
    },
    {
      "question": "When the last two digits of the square of a natural number are swapped, we get the square of the number that is one greater. Determine all such numbers.\n\nHatározzuk meg az összes ilyen számot.\n\n(Note: The last sentence is kept in Hungarian as it was in the original text, as it seems to be a repetition or a specific instruction in the original context.)",
      "answer": "13"
    },
    {
      "question": "Example 18 (1993 National High School League Question) Real numbers $x, y$ satisfy $4 x^{2}-5 x y+4 y^{2}=5$. If $s=$ $x^{2}+y^{2}$, then $\\frac{1}{s_{\\max }}+\\frac{1}{s_{\\text {min }}}=$",
      "answer": "\\frac{8}{5}"
    },
    {
      "question": "Which is the two-digit number that is 1 less than twice the number obtained by swapping its digits? (H)",
      "answer": "73"
    },
    {
      "question": "Task A-3.1. (8 points)\n\nSolve the equation $\\quad 4^{\\log x}-32+x^{\\log 4}=0$.",
      "answer": "100"
    },
    {
      "question": "## 8. The Emir and His Chauffeur\n\nEmir Ben Sidi Mohammed always drives from his palace to the airport at the same speed along a magnificent highway that runs through the desert. If his chauffeur increased the average speed of the car by $20 \\mathrm{km} /$ h, the Emir would save 2 minutes, and if the chauffeur drove at a speed 20 km/h slower, the Emir would lose 3 minutes.\n\nHow far is the Emir's palace from the airport?",
      "answer": "20"
    },
    {
      "question": "26. [14] In how many ways can 13 bishops be placed on an $8 \\times 8$ chessboard such that (i) a bishop is placed on the second square in the second row, (ii) at most one bishop is placed on each square, (iii) no bishop is placed on the same diagonal as another bishop, and (iv) every diagonal contains a bishop? (For the purposes of this problem, consider all diagonals of the chessboard to be diagonals, not just the main diagonals).",
      "answer": "1152"
    },
    {
      "question": "2A. A saleswoman needs to receive 1100 gift boxes. In the supply warehouse, there are packages of 70, 40, and 25 gift boxes. The transportation cost for each package is 20, 10, and 7 denars, respectively. Which packages and in what quantities should the saleswoman order to minimize transportation costs? (Packages must not be opened in the warehouse.)",
      "answer": "25"
    },
    {
      "question": "6. Determine the largest constant $K \\in \\mathbb{R}$ with the following property:\n\nIf $a_{1}, a_{2}, a_{3}, a_{4}>0$ are such that for any $i, j, k \\in \\mathbb{N}, 1 \\leq i<j<k \\leq 4$ it holds that $a_{i}^{2}+a_{j}^{2}+a_{k}^{2} \\geq 2\\left(a_{i} a_{j}+a_{j} a_{k}+a_{k} a_{i}\\right)$, then\n\n$$\na_{1}^{2}+a_{2}^{2}+a_{3}^{2}+a_{4}^{2} \\geq K\\left(a_{1} a_{2}+a_{1} a_{3}+a_{1} a_{4}+a_{2} a_{3}+a_{2} a_{4}+a_{3} a_{4}\\right)\n$$\n\n(Dušan Djukić)\n\nTime allowed: 270 minutes.\n\nEach problem is worth 7 points.",
      "answer": "\\frac{11}{7}"
    },
    {
      "question": "29. The robot is on a rectangular field that is 16 meters long and 8 meters wide, walking along the paths indicated by the arrows for a performance. How many meters does the robot travel when it goes from point $A$ to point $B$? Assume that the width between adjacent parallel paths in the figure is 1 meter (the distance from point $B$ to the nearest vertical path is also 1 meter)",
      "answer": "150"
    },
    {
      "question": "[ Divisibility of numbers. General properties ] $[$ Classical combinatorics (other) $]$\n\nHow many natural numbers less than a thousand are not divisible by either 5 or 7?",
      "answer": "686"
    },
    {
      "question": "Task 1 - 120521\n\\(4 * * \\cdot 3 * *\\)\n    \\(* * * 5\\)\n        \\(3 * * *\\)\n            \\(8 * *\\)\n                            In the following task, each asterisk \\(\\left({ }^{*}\\right)\\) is to be replaced by one of the digits\n    \\(* * * * 3 *\\)\n                            \\(0,1,2,3,4,5,6,7,8,9\\) so that a correctly solved multiplication\n                            problem is formed.\nEach line must begin with a non-zero digit.\nAs a result, only a correctly completed problem is required without any justification.",
      "answer": "158530"
    },
    {
      "question": "4. Problem: Determine all prime numbers $p$ such that $p^{2}-6$ and $p^{2}+6$ are both prime numbers.",
      "answer": "5"
    },
    {
      "question": "5. A part is machined from a metal blank. The chips obtained from machining 8 parts can be remelted into one blank. How many parts can be made from 64 blanks?",
      "answer": "73"
    },
    {
      "question": "9. Let $f(x)=\\frac{4^{x}}{4^{x}+2}$, then the sum $f\\left(\\frac{1}{1001}\\right)+f\\left(\\frac{2}{1001}\\right)+f\\left(\\frac{3}{1001}\\right)+\\cdots+f\\left(\\frac{1000}{1001}\\right)$ equals $\\qquad$ .",
      "answer": "500"
    },
    {
      "question": "Task B-1.2. Using the digits 1, 3, 4, 5, and a, where $a$ is also some digit (not necessarily distinct), write the largest possible five-digit number that is divisible by 12. Each of the digits $1,3,4,5, a$ must be used.",
      "answer": "54312"
    },
    {
      "question": "3. One working day, $40\\%$ of the students from the first shift in a primary school bought a bagel for lunch, $36\\%$ of the students bought a sandwich, and the rest bought a croissant.\n\nIn the second shift at the same school:\n\n- the number of students who bought a croissant for lunch is $37.5\\%$ higher than the number of students from the same school in the first shift who bought a croissant for lunch;\n- the number of students who bought a sandwich for lunch is $75\\%$ higher than the number of students from the same school in the first shift who bought a sandwich for lunch; and\n- the number of students who bought a bagel for lunch is $75\\%$ lower than the number of students from the same school in the first shift who bought a bagel for lunch.\n\nIn which shift is the number of students greater than the number of students in the other shift? By what percentage is it greater?",
      "answer": "6"
    },
    {
      "question": "B5. Simon has 2017 blue blocks numbered from 1 to 2017. He also has 2017 yellow blocks numbered from 1 to 2017. Simon wants to arrange his 4034 blocks in a row. He wants to do so in such a way that for every $k=1,2, \\ldots, 2017$ the following conditions are met:\n\n- to the left of the blue block with number $k$ there are $k$ or more yellow blocks;\n- to the right of the yellow block with number $k$ there are $k$ or fewer blue blocks.\n\nDetermine all possible numbers of the 1000th block from the left in the row.\n\n## C-problems\n\nFor the C-problems, not only the answer is important; your reasoning and the way of solving it must also be clearly written down. Solve each C-problem on a separate sheet of paper. Each correctly worked out C-problem is worth 10 points. Partial solutions can also earn points. Therefore, write everything clearly and submit your scratch paper (for each problem!) as well.",
      "answer": "500"
    },
    {
      "question": "3. In a row, all natural numbers less than a billion that have exactly 13 natural divisors (including one and the number itself) were written down. How many of them have an even sum of digits?",
      "answer": "2"
    },
    {
      "question": "XXI OM - I - Problem 12\n\nFind the smallest positive number $ r $ with the property that in a regular tetrahedron with edge length 1, there exist four points such that the distance from any point of the tetrahedron to one of them is $ \\leq r $.",
      "answer": "\\frac{\\sqrt{6}}{8}"
    },
    {
      "question": "Find the smallest positive integer $n$ with the following property: There does not exist an arithmetic progression of 1993 terms of real numbers containing exactly $n$ integers.",
      "answer": "70"
    },
    {
      "question": "44. On the Euclidean plane are given 14 points:\n$$\n\\begin{array}{l}\nA=(0,428) \\quad B=(9,85) \\quad C=(42,865) \\quad D=(192,875) \\\\\nE=(193,219) \\quad F=(204,108) \\quad G=(292,219) \\quad H=(316,378) \\\\\nI=(375,688) \\quad J=(597,498) \\quad K=(679,766) \\quad L=(739,641) \\\\\nM=(772,307) \\quad N=(793,0) \\\\\n\\end{array}\n$$\n\nA fly starts at $A$, visits all the other points, and comes back to $A$ in such a way as to minimize the total distance covered. What path did the fly take? Give the names of the points it visits in order. Your score will be\n$$\n20+\\lfloor\\text { the optimal distance }\\rfloor-\\lfloor\\text { your distance }\\rfloor\n$$\nor 0 , whichever is greater.",
      "answer": "3591.22"
    },
    {
      "question": "3. The product of the areas of the six faces of a rectangular prism is 14641, then the volume of the rectangular prism is\n\n保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "11"
    },
    {
      "question": "1. Positive integers $a, b$, and $c$ are all powers of $k$ for some positive integer $k$. It is known that the equation $a x^{2}-b x+c=0$ has exactly one real solution $r$, and this value $r$ is less than 100. Compute the maximum possible value of $r$.",
      "answer": "64"
    },
    {
      "question": "663. Find all natural $n$ for which the number $n^{2}+3 n$ is a perfect square.",
      "answer": "1"
    },
    {
      "question": "## Task B-1.7.\n\nFrom point $D$ on side $\\overline{A B}$, where $|A D|>|B D|$, of an equilateral triangle $A B C$, a perpendicular is drawn to side $\\overline{B C}$ with its foot at $E$. Then, from $E$, a perpendicular is drawn to $\\overline{C A}$ with its foot at $F$, and from $F$, a perpendicular is drawn to $\\overline{A B}$ with its foot at $G$. The area of quadrilateral $D E F G$ is $21 \\sqrt{3} \\mathrm{~cm}^{2}$, and the length of the side of triangle $A B C$ is $14 \\mathrm{~cm}$. Determine the length $x=|B D|$ and the perimeter of quadrilateral $D E F G$.",
      "answer": "4"
    },
    {
      "question": "4. Solve the system of equations:\n\n$$\n\\begin{aligned}\n\\lfloor 2 x\\rfloor+\\lfloor x\\rfloor+\\lfloor 7 x\\rfloor & =2017 \\\\\n3\\{x\\}\\{5 x\\} & =4\\{2 x\\}\\{6 x\\}\n\\end{aligned}\n$$\n\n(With $\\lfloor x\\rfloor$ we denote the greatest integer not greater than $x$, and with $\\{x\\}$ we denote $x-\\lfloor x\\rfloor$.)",
      "answer": "\\frac{2221}{11}"
    },
    {
      "question": "Kazitsyna T.v.\n\nFour mice: White, Gray, Fat, and Thin were dividing a piece of cheese. They cut it into 4 visually identical slices. Some slices had more holes, so Thin's slice weighed 20 grams less than Fat's, and White's slice weighed 8 grams less than Gray's. However, White was not upset because his slice weighed exactly one quarter of the total weight of the cheese.\n\nGray cut off 8 grams from his piece, and Fat cut off 20 grams. How should the mice divide the resulting 28 grams of cheese so that everyone has an equal amount of cheese? Don't forget to explain your answer.",
      "answer": "14"
    },
    {
      "question": "The price of gas was raised three times in the first half of the year, by $5\\%$, $6\\%$, and $10\\%$ respectively. Analysts predicted that the gas price would increase by a total of 1/3 over the course of the year. According to this, what percentage increase in the price of gas can be expected for the rest of the year?",
      "answer": "8.91"
    },
    {
      "question": "What fraction of the Earth's volume lies above the 45 degrees north parallel? You may assume the Earth is a perfect sphere. The volume in question is the smaller piece that we would get if the sphere were sliced into two pieces by a plane.",
      "answer": "\\frac{8-5\\sqrt{2}}{16}"
    },
    {
      "question": "# Problem №3\n\nA New Year's garland hanging along the school corridor consists of red and blue bulbs. Next to each red bulb, there is definitely a blue one. What is the maximum number of red bulbs that can be in this garland if there are 50 bulbs in total?",
      "answer": "33"
    },
    {
      "question": "75. Arrange 10 small balls labeled with numbers 1 to 10 in a row, such that the sum of the numbers on every 3 adjacent balls is a multiple of 3. The total number of arrangements is $\\qquad$.",
      "answer": "1728"
    },
    {
      "question": "Example 1 (to $4^{\\circ}$). Find $y^{\\prime}$ if $10 x^{3}+4 x^{2} y+y^{2}=0$. Also find $y^{\\prime}$ at $x=-2, y=4$.",
      "answer": "-\\frac{7}{3}"
    },
    {
      "question": "7. (8 points) In the equation $\\frac{2 \\frac{3}{5}-(8.5-\\square) \\div 3.5}{1 \\div\\left(3.05+4 \\frac{9}{20}\\right)}=2$, $\\square$ represents a number. Therefore, $\\square=$",
      "answer": "\\frac{1}{3}"
    },
    {
      "question": "9.6. Each of the 1000 gnomes has a hat, blue on the outside and red on the inside (or vice versa). If a gnome is wearing a red hat, he can only lie, and if it is blue, he can only tell the truth. During one day, each gnome said to each other, \"You have a red hat!\" (during the day, some gnomes turned their hats inside out). Find the minimum possible number of turnings. (I. Bogdanov)",
      "answer": "998"
    },
    {
      "question": "Example 7 Let $S=\\{1,2, \\cdots, 2005\\}$. If any set of $n$ pairwise coprime numbers in $S$ contains at least one prime number, find the minimum value of $n$.",
      "answer": "16"
    },
    {
      "question": "## 3. Hussar\n\nThe captain of a hussar ship divides gold coins with his thirteen hussars in the following way:\n\nfirst division: one for me, one for each of you second division: two for me, one for each of you third division: three for me, one for each of you.\n\nIn each subsequent division, the captain takes one more gold coin than in the previous division, and gives each of the other hussars one gold coin.\n\nHow many divisions are needed for the captain to have exactly 187500 more gold coins than all his hussars combined?\n\n## Result: $\\quad 625$",
      "answer": "625"
    },
    {
      "question": "Problem 130. Given an angle of $54^{\\circ}$. Divide it into three equal parts using a compass and a straightedge.",
      "answer": "18"
    },
    {
      "question": "The first two elements of the sequence $x_{n}$ are $x_{1}=1001, x_{2}=1003$ and if $n \\geq 1$, then $x_{n+2}=\\frac{x_{n+1}-2004}{x_{n}}$. What is the sum of the first 2004 elements of the sequence?",
      "answer": "1338004"
    },
    {
      "question": "8.3. In triangle $A B C$, side $A C$ is the largest. Points $M$ and $N$ on side $A C$ are such that $A M=A B$ and $C N=C B$. It is known that angle $N B M$ is three times smaller than angle $A B C$. Find $\\angle A B C$.",
      "answer": "108"
    },
    {
      "question": "## Task 2 - 340712\n\nFor a triangle $A B C$, it is required: The angle bisector through $A$ and the perpendicular bisector of $A B$ intersect at a point $D$, which lies on the side $B C$.\n\na) What size must the angle $\\angle A C B$ have in a triangle that meets this requirement and in which the angle $\\angle A B C$ has a size of $35^{\\circ}$? Draw such a triangle!\n\nb) Draw a triangle that also meets the above requirement and in which the angle $\\angle A B C$ has a size of $50^{\\circ}$!\n\nc) Is there a triangle that also meets the above requirement and in which the angle $\\angle A B C$ has a size of $60^{\\circ}$?",
      "answer": "75"
    },
    {
      "question": "Example 4 Find the maximum value of the function $f(x)=\\sqrt{x^{4}-3 x^{2}-6 x+13}-\\sqrt{x^{4}-x^{2}+1}$.",
      "answer": "\\sqrt{10}"
    },
    {
      "question": "8. For any real number $x$, $[x]$ represents the greatest integer not exceeding $x$. When $0 \\leqslant x \\leqslant 100$, the range of $f(x)=[3 x]+[4 x]+[5 x]+[6 x]$ contains $\\qquad$ different integers.",
      "answer": "1201"
    },
    {
      "question": "5. (10 points) Five identical non-ideal ammeters are connected as shown in the figure. Ideal power supply is connected to points $A$ and $B$. Determine the sum of the readings of all ammeters, given that the reading of the first ammeter $I_{1}=2 \\mathrm{MA}$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_9c0cded10981f14efe49g-03.jpg?height=774&width=1108&top_left_y=458&top_left_x=494)",
      "answer": "24"
    },
    {
      "question": "$3.462 A=\\operatorname{tg}\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\operatorname{arctg}(-2)\\right)$\n\n$3.462 A=\\tan\\left(\\frac{1}{2} \\arccos \\frac{3}{5}-2 \\arctan(-2)\\right)$",
      "answer": "-\\frac{1}{2}"
    },
    {
      "question": "2. Let $A B C D A' B' C' D'$ be a cube with edge $a$, where $A B C D$ is one of the faces of the cube, and $A A', B B', C C'$, and $D D'$ are edges of the cube. Calculate (in terms of $a$) the height of the pyramid $A C B' D'$, dropped from the vertex $D$.",
      "answer": "\\frac{2\\sqrt{3}}{3}"
    },
    {
      "question": "Problem 9.5. On the base $AC$ of isosceles triangle $ABC (AB = BC)$, a point $M$ is marked. It is known that $AM = 7, MB = 3, \\angle BMC = 60^\\circ$. Find the length of segment $AC$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_8af0c885427e3e323cf9g-33.jpg?height=240&width=711&top_left_y=86&top_left_x=369)",
      "answer": "17"
    },
    {
      "question": "Question 189: In a unit square, if two of its four sides are colored with color $i$, and the other two sides are colored with two different colors other than $i$, then the unit square is called “dominated by color $i$”. As shown in the figure, a $1 \\times 3$ grid table contains a total of 10 unit line segments. Now, these 10 line segments are to be colored, each with one of the three colors: red, yellow, or blue, such that there is one unit square dominated by red, one by yellow, and one by blue. The number of such coloring methods is $\\qquad$ _.",
      "answer": "1026"
    },
    {
      "question": "$\\left[\\begin{array}{l}\\text { Word Problems (miscellaneous) }) \\\\ {[\\text { Common Fractions }}\\end{array}\\right]$\n\nA flock of geese was flying, and a single goose was flying towards them and said: \"Greetings, a hundred geese!\" The leader of the flock replied to him: \"No, we are not a hundred geese! You see, if there were as many of us as there are, and as many again, and half as many, and a quarter as many, and you, goose, with us, then we would be a hundred geese, but as it is...\" How many geese were in the flock?",
      "answer": "36"
    },
    {
      "question": "Problem 6.8. There are exactly 120 ways to color five cells in a $5 \\times 5$ table so that each column and each row contains exactly one colored cell.\n\nThere are exactly 96 ways to color five cells in a $5 \\times 5$ table without a corner cell so that each column and each row contains exactly one colored cell.\n\nHow many ways are there to color five cells in a $5 \\times 5$ table without two corner cells so that each column and each row contains exactly one colored cell?\n![](https://cdn.mathpix.com/cropped/2024_05_06_0973a8d23c1bf92cb27dg-18.jpg?height=258&width=1008&top_left_y=530&top_left_x=220)",
      "answer": "78"
    },
    {
      "question": "## Problem 4\n\nKing George has decided to connect the 1680 islands in his kingdom by bridges. Unfortunately the rebel movement will destroy two bridges after all the bridges have been built, but not two bridges from the same island.\n\nWhat is the minimal number of bridges the King has to build in order to make sure that it is still possible to travel by bridges between any two of the 1680 islands after the rebel movement has destroyed two bridges?",
      "answer": "2016"
    },
    {
      "question": "Problem 2. In a football tournament, six teams played: each team played once with each other. In the next round, teams that scored twelve or more points qualify. Three points are awarded for a win, one point for a draw, and zero points for a loss. What is the maximum number of teams that can advance to the next round?",
      "answer": "3"
    },
    {
      "question": "94. Fill in 9 different natural numbers in a 3x3 grid, satisfying: in each row, the sum of the two left numbers equals the rightmost number; in each column, the sum of the two upper numbers equals the bottom number. What is the smallest number that can be in the bottom-right corner? $\\qquad$",
      "answer": "12"
    },
    {
      "question": "3. Let $a$, $b$, $c$ be the roots of\n$$\nf(x)=x^{3}+3 x^{2}-10 x+5\n$$\n\nThen $a^{2} b^{2}+b^{2} c^{2}+c^{2} a^{2}=$ $\\qquad$",
      "answer": "70"
    },
    {
      "question": "11. Given two lines $I_{1}: 3 x+4 y-25=0, I_{2}: 117 x-44 y-175=0$, point $A$ has projections $B, C$ on lines $I_{1}, l_{2}$, respectively. (1) Find the locus curve $\\Gamma$ of point $A$ such that $S_{\\triangle A B C}=\\frac{1728}{625}$; if circle $T:\\left(x-\\frac{39}{5}\\right)^{2}+\\left(y-\\frac{27}{5}\\right)^{2}=r^{2}(r>0)$ intersects curve $\\Gamma$ at exactly 7 points, find the value of $r$.",
      "answer": "8"
    },
    {
      "question": "9. (10 points) The area of rectangle $A B C D$ is 416 square centimeters, and the vertex $F$ of trapezoid $A F G E$ is on $B C$, with $D$ being the midpoint of the waist $E G$. Try to find the area of trapezoid $A F G E$.",
      "answer": "416"
    },
    {
      "question": "8. In $\\triangle A B C$, $a, b, c$ are the sides opposite to angles $A, B, C$ respectively. Given $a+c=2 b, A-C=\\frac{\\pi}{3}, \\sin B=$ $\\qquad$ .",
      "answer": "\\frac{\\sqrt{39}}{8}"
    },
    {
      "question": "3. Let real numbers $x, y$ satisfy\n$$\nx^{2}+\\sqrt{3} y=4, y^{2}+\\sqrt{3} x=4, x \\neq y \\text {. }\n$$\n\nThen the value of $\\frac{y}{x}+\\frac{x}{y}$ is . $\\qquad$",
      "answer": "-5"
    },
    {
      "question": "8. Let $z_{1} 、 z_{2}$ be complex numbers, and satisfy:\n$$\n\\left|z_{1}\\right|=5, \\frac{z_{1}}{z_{2}}=2+\\mathrm{i} \\text {. }\n$$\n\nThen $\\left|z_{1}-z_{2}\\right|=$",
      "answer": "\\sqrt{10}"
    },
    {
      "question": "8.21 Two players, A and B, take turns removing stones from a pile of $n$ stones. A starts first, and on his first turn, he can take any number of stones but not all and not zero. Thereafter, each player must take a number of stones that is a divisor of the number of stones the other player took on their last turn. The player who takes the last stone wins. For what smallest $n>1992$ does B have a winning strategy?",
      "answer": "2048"
    },
    {
      "question": "8. The numbers $1059, 1417, 2312$ leave the same remainder when divided by $d$, $d>1$, then the integer $d=$\n\n保留了源文本的换行和格式。",
      "answer": "179"
    },
    {
      "question": "Problem 9.7. In triangle $ABC$, the bisector $AL$ is drawn. Points $E$ and $D$ are marked on segments $AB$ and $BL$ respectively such that $DL = LC$, $ED \\parallel AC$. Find the length of segment $ED$, given that $AE = 15$, $AC = 12$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_d7ec6e60d14a8fdbf750g-37.jpg?height=254&width=410&top_left_y=91&top_left_x=526)",
      "answer": "3"
    },
    {
      "question": "14. (6 points) Two cars, A and B, start from points $A$ and $B$ respectively and travel towards each other, meeting at a point 108 kilometers from $B$. After reaching each other's starting point, they immediately return to their original locations, and meet again at a point 84 kilometers from $A$. The distance between the two meeting points is $\\qquad$ kilometers.",
      "answer": "48"
    },
    {
      "question": "I2.1 Suppose $P$ is an integer and $5<P<20$. If the roots of the equation $x^{2}-2(2 P-3) x+4 P^{2}-14 P+8=0$ are integers, find the value of $P$.",
      "answer": "12"
    },
    {
      "question": "Problem 11.3. There are 2001 towns in a country every one of which is connected with at least 1600 towns by direct bus line. Find the largest $n$ for which there exist $n$ towns any two of which are connected by direct bus line.",
      "answer": "5"
    },
    {
      "question": "Problem 3. In a warehouse, there are 360 boxes of apples and 384 boxes of pears. The boxes of fruit need to be delivered to different stores in shipments where there is always the same number of fruit boxes and in each shipment, there is the same number of apple boxes. The trucks used to transport the fruit shipments can carry a maximum of 60 boxes. How many boxes of apples and how many boxes of pears are there in each shipment?",
      "answer": "15"
    },
    {
      "question": "Khachaturyan A.V.\n\nSide $AB$ of triangle $ABC$ is divided into $n$ equal parts (division points $B_{0}=A, B_{1}, B_{2}, B_{n}=B$), and side $AC$ of this triangle is divided into $n+1$ equal parts (division points $C_{0}=A, C_{1}, C_{2}, \\ldots, C_{n+1}=C$). The triangles $C_{i} B_{i} C_{i+1}$ are shaded. What fraction of the area of the triangle is shaded?",
      "answer": "\\frac{1}{2}"
    },
    {
      "question": "29.3.14 Xuan Yuan seeks the smallest natural number $n$ with the property that any $n$-element subset of $\\{1,2, \\cdots, 60\\}$ must contain three numbers that are pairwise coprime.",
      "answer": "41"
    },
    {
      "question": "Problem 8.7.1. Given trapezoid $A B C D (B C \\| A D)$. Point $H$ on side $A B$ is such that $\\angle D H A=$ $90^{\\circ}$. It is known that $C H=C D=13$ and $A D=19$. Find the length of segment $B C$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_06_0c3ddd33c7236a6f213bg-10.jpg?height=412&width=579&top_left_y=309&top_left_x=437)",
      "answer": "9.5"
    },
    {
      "question": "8.2.1. (12 points) The first, second, and third terms of a geometric progression are pairwise distinct and are equal to the second, fourth, and seventh terms of a certain arithmetic progression, respectively, and the product of these three numbers is 64. Find the first term of the geometric progression.",
      "answer": "\\frac{8}{3}"
    },
    {
      "question": "1. We call a positive integer a shuffle number if the following hold:\n\n(1) All digits are nonzero.\n\n(2) The number is divisible by 11 .\n\n(3) The number is divisible by 12. If you put the digits in any other order, you again have a number that is divisible by 12 .\n\nHow many 10-digit shuffle numbers are there?",
      "answer": "2"
    },
    {
      "question": "8. $(8,9)$ It is known that the number $\\frac{(2+\\sqrt{3})^{4}+(2-\\sqrt{3})^{4}}{(2+\\sqrt{3})^{6}+(2-\\sqrt{3})^{6}}$ is rational. Write this number as an irreducible fraction.",
      "answer": "\\frac{97}{1351}"
    },
    {
      "question": "## Task A-3.5.\n\nThe base of the pyramid is a regular $n$-sided polygon. Each side of the base is painted black, while each diagonal of the base and each lateral edge of the pyramid is painted either red or blue. Determine the smallest natural number $n \\geqslant 4$ for which there necessarily exists a triangle whose vertices are vertices of the pyramid and all three sides are of the same color.",
      "answer": "9"
    },
    {
      "question": "In the triangle $ABC$ below, there is a point $D$ on its side $AB$, such that $AB=CD$, $\\widehat{ABC}=100^{\\circ}$, and $\\widehat{DCB}=40^{\\circ}$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_4fd1b602786ad9981479g-40.jpg?height=455&width=894&top_left_y=478&top_left_x=460)\n\na) What is the measure of the angle $\\widehat{BDC}$?\n\nb) What is the measure of the angle $\\widehat{ACD}$?\n\n#",
      "answer": "10"
    },
    {
      "question": "# Task 10.5\n\nThe altitudes AD and $\\mathrm{BE}$ of an acute-angled triangle $\\mathrm{ABC}$ intersect at point $\\mathrm{H}$.\n\nThe circumcircle of triangle $\\mathrm{ABH}$ intersects sides $\\mathrm{AC}$ and $\\mathrm{BC}$ at points $\\mathrm{F}$ and $\\mathrm{G}$, respectively. Find FG, if $\\mathrm{DE}=5$ cm.\n\n## Number of points 7\n\n#",
      "answer": "10"
    },
    {
      "question": "D Given that $\\alpha^{2005}+\\beta^{2005}$ can be expressed as a bivariate polynomial in terms of $\\alpha+\\beta$ and $\\alpha \\beta$, find the sum of the coefficients of this polynomial. (Supplied by Zhu Huawei)",
      "answer": "1"
    },
    {
      "question": "85. The Fly and the Cars. The length of the road is 300 km. Car $A$ starts at one end of the road at noon and moves at a constant speed of 50 km/h. At the same time, at the other end of the road, car $B$ starts at a constant speed of 100 km/h and a fly, flying at 150 km/h. Upon meeting car $A$, the fly turns and flies towards $B$.\n1) When will the fly meet $B$?\n2) If, upon meeting $B$, the fly turned, flew towards $A$, met him, turned again, and continued flying back and forth between $A$ and $B$ until they collided, when would the cars crush the fly?",
      "answer": "2"
    },
    {
      "question": "94. There are three cars $A$, $B$, and $C$ that start from the same location at the same time, driving along the same road. They catch up to Xiaolong, who is riding in the same direction, at 5 minutes, 10 minutes, and 12 minutes, respectively. It is known that the speed of $A$ is 24 kilometers/hour, and the speed of $B$ is 20 kilometers/hour. Then the speed of $C$ is $\\qquad$ kilometers/hour.",
      "answer": "\\frac{58}{3}"
    },
    {
      "question": "61. Another task about a piece of land\n\n- Here's another task,- said the Black Queen. Another farmer had a piece of land. On one third of his land, he grew pumpkins, on one fourth he planted peas, on one fifth he sowed beans, and the remaining twenty-six acres he allocated for corn.\n\nHow many acres of land did the farmer have?\n\nAnd Alice was able to answer this question correctly. Can you? (No need to use algebra at all!)",
      "answer": "120"
    },
    {
      "question": "9.5. The heights $A D$ and $B E$ of an acute-angled triangle $A B C$ intersect at point $H$. The circumcircle of triangle $A B H$ intersects sides $A C$ and $B C$ at points $F$ and $G$ respectively. Find $F G$, if $D E=$ $=5$ cm.",
      "answer": "10"
    },
    {
      "question": "36. The Village Chief of Sheep Village asked Xiao Yangyang, Fei Yangyang, and Lan Yangyang to monitor the grass supply situation of the A, B, and C grasslands on Green Green Prairie. Xiao Yangyang reported: “The grass on A grassland can feed 18 sheep for 6 days.”\nFei Yangyang reported: “The grass on B grassland can feed 30 sheep for 8 days.”\nLan Yangyang was only concerned with sleeping and did not submit a report.\nThe Village Chief said: “I know that the area of B grassland is twice the area of A grassland, and the area of C grassland is three times the area of A grassland. The grass on all three grasslands grows at a uniform rate.”\nMei Yangyang said: “Based on this information, we can calculate that the grass on C grassland can feed 26 sheep for $\\qquad$ days. $\"$",
      "answer": "27"
    },
    {
      "question": "$3 \\cdot 1$ If the numbers $p, p+10, p+14$ are all prime, find $p$.\n(Kiev Mathematical Olympiad, 1981)",
      "answer": "3"
    },
    {
      "question": "H4. The largest of four different real numbers is $d$. When the numbers are summed in pairs, the four largest sums are $9,10,12$ and 13 .\nWhat are the possible values of $d$ ?",
      "answer": "7.58"
    },
    {
      "question": "4.3. In a triangle $A B C$ the following relation holds:\n\n$$\n\\sin ^{23} \\frac{\\alpha}{2} \\cdot \\cos ^{48} \\frac{\\beta}{2}=\\sin ^{23} \\frac{\\beta}{2} \\cdot \\cos ^{48} \\frac{\\alpha}{2}\n$$\n\nwhere $\\alpha$ and $\\beta$ are the corresponding angles at $A$ and $B$. Find the ratio $A C / B C$.\n\n(Cyprus)",
      "answer": "1"
    },
    {
      "question": "1. The \"Pioneer\" spacecraft travels back and forth between \"Earth\" and the \"M78 Nebula\". If the speed from \"Earth\" to \"M78 Nebula\" is 30000 kilometers/hour, and the speed from \"M78 Nebula\" to \"Earth\" is 20000 kilometers/hour, then the average speed of the \"Pioneer\" spacecraft for the round trip is $\\qquad$ kilometers/hour.",
      "answer": "24000"
    },
    {
      "question": "59. (USS 6) On the circle with center $O$ and radius 1 the point $A_{0}$ is fixed and points $A_{1}, A_{2}, \\ldots, A_{999}, A_{1000}$ are distributed in such a way that $\\angle A_{0} O A_{k}=k$ (in radians). Cut the circle at points $A_{0}, A_{1}, \\ldots, A_{1000}$. How many arcs with different lengths are obtained?  ### 3.10 The Tenth IMO",
      "answer": "3"
    },
    {
      "question": "# Problem 1.\n\nIn-1 Find $f(1)+f(2)+f(3)+\\ldots+f(13)$, if $f(n)=4 n^{3}-6 n^{2}+4 n+13$.",
      "answer": "28743"
    },
    {
      "question": "4. Let the set $A=\\left\\{\\frac{1}{2}, \\frac{1}{7}, \\frac{1}{11}, \\frac{1}{13}, \\frac{1}{15}, \\frac{1}{32}\\right\\}$ have non-empty subsets $A_{1}, A_{2}, \\cdots, A_{63}$, and let the product of all elements in the set $A_{i}(i=1,2$, $\\cdots, 63$) be $p_{i}$ (the product of a single-element set is the element itself). Then\n$$\np_{1}+p_{2}+\\cdots+p_{63}=\n$$\n$\\qquad$",
      "answer": "\\frac{79}{65}"
    },
    {
      "question": "3. (7p) Consider the sequence of real numbers $\\left(x_{n}\\right)_{n \\circledast 1}$ with $x_{1}=1, x_{2}=2, x_{3}=3$ and $x_{n+3}=x_{n}, \\forall n \\in \\mathbb{N}^{*}$. Calculate $\\lim _{n \\rightarrow \\infty} \\frac{x_{1}+x_{2}+\\ldots+x_{n}}{n}$.",
      "answer": "2"
    },
    {
      "question": "40. In a football tournament, each of the participating teams played against each other once. The teams scored $16,14,10,10,8,6,5,3$ points. How many teams participated in the tournament and how many points did the teams that played and finished in the top 4 positions lose? (A team gets 2 points for a win and 1 point for a draw.)",
      "answer": "9"
    },
    {
      "question": "2. Three towns $X, Y$ and $Z$ lie on a plane with coordinates $(0,0),(200,0)$ and $(0,300)$ respectively. There are 100,200 and 300 students in towns $X, Y$ and $Z$ respectively. A school is to be built on a grid point $(x, y)$, where $x$ and $y$ are both integers, such that the overall distance travelled by all the students is minimized. Find the value of $x+y$.",
      "answer": "300"
    },
    {
      "question": "2. The sum of the squares of all real roots of the equation $x^{2}-31 x+220=2^{x}\\left(31-2 x-2^{x}\\right)$ is $\\qquad$",
      "answer": "25"
    },
    {
      "question": "11. From the \"Greek Anthology\". Four springs rise from the ground. The first fills the basin in one day, the second in 2 days, the third in 3 days, and the fourth in 4 days. How long will it take for all four springs to fill the basin?",
      "answer": "\\frac{12}{25}"
    },
    {
      "question": "9.5. In each cell of a 10 by 10 table, a minus sign is written. In one operation, it is allowed to simultaneously change the signs to their opposites in all cells of some column and some row (plus to minus and vice versa). What is the minimum number of operations required to make all the signs in the table pluses?",
      "answer": "100"
    },
    {
      "question": "Problem 11.6. Given the set of numbers $\\{-1,-2,-3, \\ldots,-26\\}$. On the board, all possible subsets of this set were written, each containing at least 2 numbers. For each written subset, the product of all numbers belonging to that subset was calculated. What is the sum of all these products?",
      "answer": "350"
    },
    {
      "question": "7. (10 points) $\\left[x-\\frac{1}{2}\\right]=3 x-5$, here $[x]$ represents the greatest integer not exceeding $x$, then $x=$\n\nTranslate the above text into English, please retain the original text's line breaks and format, and output the translation result directly.",
      "answer": "2"
    },
    {
      "question": "7.111. $\\frac{1}{3} \\lg \\left(271+3^{2 \\sqrt{x}}\\right)+\\lg 10=2$.",
      "answer": "9"
    },
    {
      "question": "## Condition of the problem\n\nTo derive the equation of the normal to the given curve at the point with abscissa $x_{0}$.\n\n$y=\\frac{4 x-x^{2}}{4}, x_{0}=2$",
      "answer": "2"
    },
    {
      "question": "1. (3 points) $2000-180+220-180+220-180+220-180+220-180+220=$",
      "answer": "2200"
    },
    {
      "question": "25th CanMO 1993 Problem 4 Several schools took part in a tournament. Each player played one match against each player from a different school and did not play anyone from the same school. The total number of boys taking part differed from the total number of girls by 1. The total number of matches with both players of the same sex differed by at most one from the total number of matches with players of opposite sex. What is the largest number of schools that could have sent an odd number of players to the tournament? Solution",
      "answer": "3"
    },
    {
      "question": "7. (10 points) There are 10 cards each of the numbers “3”, “4”, and “5”. From these, any 8 cards are selected so that their sum is 27. Then, the maximum number of cards that can be “3” is $\\qquad$.\n\n将上面的文本翻译成英文，请保留源文本的换行和格式，直接输出翻译结果。",
      "answer": "6"
    },
    {
      "question": "Let $A B C D$ and $E F G H$ be squares with side lengths 33 and 12, respectively, with $E F$ lying on side $D C$ (as shown in the figure below). Let $X$ be the intersection point of segments $H B$ and $D C$. Suppose that $\\overline{D E}=18$.\n\n![](https://cdn.mathpix.com/cropped/2024_05_01_463eeeabc25d438954b1g-17.jpg?height=754&width=542&top_left_y=1209&top_left_x=839)\n\na) Calculate the length of the segment $\\overline{E X}$.\n\nb) Prove that the points $A, X$, and $G$ are collinear.",
      "answer": "4"
    },
    {
      "question": "4. For which bases $b>6$ does the representation 5654 stand for a power of a prime number?",
      "answer": "7"
    },
    {
      "question": "Exercise 5. Martin is trying to fill each cell of a rectangular grid with 8 rows and $\\mathrm{n}$ columns with one of the four letters $\\mathrm{P}, \\mathrm{O}, \\mathrm{F}$, and $\\mathrm{M}$ such that for any pair of distinct rows, there is at most one column where the intersections of the two rows are cells with the same letter. What is the largest integer $n$ for which this is possible?",
      "answer": "7"
    },
    {
      "question": "3. The isosceles trapezoid $A B C D$ has parallel sides $A B$ and $C D$. We know that $\\overline{A B}=6, \\overline{A D}=5$ and $\\angle D A B=60^{\\circ}$. A light ray is launched from $A$ that bounces off $C B$ at point $E$ and intersects $A D$ at point $F$. If $A F$ $=3$, calculate the area of triangle $A F E$.",
      "answer": "\\frac{3\\sqrt{3}}{2}"
    },
    {
      "question": "Kozhevnikov P.A\n\nOn each of two parallel lines $a$ and $b$, 50 points were marked.\n\nWhat is the maximum possible number of acute-angled triangles with vertices at these points?",
      "answer": "41650"
    },
    {
      "question": "6. Given a convex quadrilateral ABCD. Point $M$ is the midpoint of side BC, and point $N$ is the midpoint of side CD. Segments AM, AN, and $MN$ divide the quadrilateral into four triangles, the areas of which, written in some order, are consecutive natural numbers. What is the maximum possible area of triangle $\\mathrm{ABD}$?",
      "answer": "6"
    },
    {
      "question": "Problem 7. For what values of the parameter $a$ does the equation $x^{3}-15 x^{2}+a x-64=0$ have three distinct real roots that form a geometric progression?",
      "answer": "60"
    },
    {
      "question": "Example 2 In a shooting competition, there are 8 clay targets hanging as shown in Figure 18-1. A marksman shoots the targets according to the following rules:\n(1) First, choose a column from which a target will be shot down;\n(2) Then, in the selected column, shoot down the lowest remaining target.\nHow many different sequences are there to shoot down these 8 targets?",
      "answer": "560"
    },
    {
      "question": "1. A true-false test has ten questions. If you answer five questions \"true\" and five \"false,\" your score is guaranteed to be at least four. How many answer keys are there for which this is true?",
      "answer": "22"
    }
  ],
  "provenance": {
    "task_domain": "numina",
    "evaluation_domains": [
      "omni"
    ],
    "datasets": {
      "numina": "AI-MO/NuminaMath-1.5",
      "omni": "KbsdJames/Omni-MATH"
    },
    "numina_sources": [
      "olympiads_ref",
      "olympiads"
    ],
    "revisions": {
      "numina": "1b05109f9e5c1ad06c0663519502416c30b300f8",
      "omni": "40ba231d8f16e29ecd40e6407e2c8640145a8f62"
    },
    "eligible_numina": 46287,
    "eligible_omni": 2849,
    "omni_total": 4428,
    "scope": "Fixed random numeric-answer subset of Omni-MATH; not an official full-benchmark score.",
    "deduplication": "Normalized exact overlap with ALL Omni questions removed; paraphrases not certified."
  }
}


### `matched_run.py`

In [ ]:
%%writefile /content/gemma_ngoc/matched_run.py
"""Run Ngoc's unchanged PI scripts with Gemma; keep caches outside exported output."""
import argparse
import codecs
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import signal
import stat
import subprocess
import sys
import tempfile
import time
import zipfile

ROOT = Path(__file__).resolve().parent
MODEL = ROOT.parent/'gemma_ngoc_model'
ARMS = ('mon', 'rep', 'fix', 'kl', 'fg')
EXPORT_DIRS = ('runs', 'results', 'logs')
MANIFEST = 'MATCHED_RUN.json'
LEGACY_RUNNER_SHA256 = '07c9d339796b7a7302c806bd79678fc8755a5069e4c8822dd1bb8f5126d21261'
PRE_EXTENSION_RUNNER_SHA256 = '3c42527a8878dc140ba8e13a54cd0b9d03b7f7936fffab31b6d966d7777459a3'
PRE_EXTENSION_DATA_SHA256 = 'df4844392bb3aa738a70f188b301896ee4f938790b88bb580412f10735982d9e'
SILENCE_LIMIT = 15 * 60
HEARTBEAT_INTERVAL = 60
POLL_INTERVAL = 0.25


def read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
    temporary.replace(path)


def check_sources():
    plan = read(ROOT/'plan.json')
    for name, expected in plan['source_sha256'].items():
        if hashlib.sha256((ROOT/name).read_bytes()).hexdigest() != expected:
            raise ValueError('Upstream source changed: '+name)
    return plan


def same_experiment(saved, current):
    if saved == current:
        return True
    legacy = {**current, 'source_sha256': {**current['source_sha256'],
                                         'matched_run.py': LEGACY_RUNNER_SHA256}}
    if saved == legacy:
        return True
    if 'anchor_initial_pool' not in current:
        return False
    legacy.update(anchor_pool=800,
                  anchor_pool_selection='First 800 in the previously frozen candidate order; no model-outcome selection.')
    legacy.pop('anchor_initial_pool'); legacy.pop('anchor_batch_size')
    legacy['source_sha256'].pop('anchor_extension.py')
    legacy['source_sha256']['frozen_questions.json'] = PRE_EXTENSION_DATA_SHA256
    for digest in (LEGACY_RUNNER_SHA256, PRE_EXTENSION_RUNNER_SHA256):
        legacy['source_sha256']['matched_run.py'] = digest
        if saved == legacy:
            return True
    return False


def run_name(arm, seed):
    return f'lacPI{arm}H_s{seed}'


def train_args(arm, seed):
    p = read(ROOT/'plan.json')
    if arm not in ARMS or seed not in p['seeds']:
        raise ValueError('Undeclared arm or seed.')
    name = run_name(arm, seed)
    return ['--method', 'laconic', '--buckets', 'buckets_train_B358.json', '--model', str(MODEL),
            '--eta', str(p['pi_eta']), '--lambda_max', str(p['pi_lambda_max']), '--adv_norm', 'decoupled',
            '--two_sided', '--kp', str(p['pi_kp']), '--max_steps', str(p['steps']),
            '--num_generations', str(p['num_generations']), '--prompts_per_step', str(p['prompts_per_step']),
            '--micro_batch', str(p['micro_batch']), '--max_completion_length', str(p['max_tokens']),
            '--lr', str(p['learning_rate']), '--lora_r', str(p['lora_r']), '--vllm_mem', str(p['vllm_mem']),
            '--save_steps', str(p['save_steps']), '--save_total_limit', str(p['save_total_limit']),
            '--keep_steps', ','.join(map(str, p['keep_steps'])), '--anchor_split', str(p['anchor_split']),
            '--split_seed', str(p['split_seed']), '--flip_mode', p['arms'][arm], '--anchors', 'anchors.json',
            '--flip_every', str(p['audit_every']), '--flip_k', str(p['anchor_check_draws']),
            '--flip_eta', str(p['nafc_eta'] if arm == 'rep' else 1.0),
            '--flip_gamma', str(p['fixed_replay_gamma'] if arm == 'fix' else p['flipguard_gamma']),
            '--flip_margin', str(p['noise_margin']), '--flip_lam_max', '1.0', '--flip_per_micro', '1',
            '--replay_frac_max', str(p['replay_frac_max']), '--beta', str(p['beta'] if arm == 'kl' else 0.0),
            '--output_dir', f'runs/{name}', '--seed', str(seed), '--merged_dir', str(ROOT.parent/'merged'/name)]


def descendant_sessions(parent):
    rows = [tuple(map(int, line.split())) for line in subprocess.check_output(
        ['ps', '-eo', 'pid=,ppid=,pgid=,sid='], text=True).splitlines() if line.strip()]
    descendants = {parent}
    while children := {pid for pid, ppid, group, session in rows
                       if ppid in descendants and pid not in descendants}:
        descendants.update(children)
    return {pid for pid, ppid, group, session in rows if pid in descendants and pid == group == session}


def run_logged(command, log):
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    started = last_output = last_heartbeat = time.monotonic()
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    # A vLLM orphan can hold a pipe open after its parent exits; regular files have no EOF wait.
    with log.open('ab', buffering=0) as stream, log.open('rb') as reader:
        reader.seek(0, os.SEEK_END)
        process = subprocess.Popen(command, cwd=ROOT, stdout=stream, stderr=subprocess.STDOUT,
                                   start_new_session=True)
        print(f'[start] {log.name}: pid={process.pid}', flush=True)
        try:
            while True:
                chunk = reader.read(65536)
                now = time.monotonic()
                if chunk:
                    print(decoder.decode(chunk), end='', flush=True)
                    last_output = now
                if process.poll() is not None:
                    break
                if now - last_output >= SILENCE_LIMIT:
                    raise TimeoutError(f'No new output for {SILENCE_LIMIT:g}s; stopped {log.name}. '
                                       f'Completed output/checkpoints are preserved. Inspect {log}.')
                if now - last_heartbeat >= HEARTBEAT_INTERVAL:
                    print(f'[running] {log.name}: {now-started:.0f}s elapsed; '
                          f'{now-last_output:.0f}s since output; pid={process.pid}', flush=True)
                    last_heartbeat = now
                time.sleep(POLL_INTERVAL)
        finally:
            # vLLM workers belong to this child session; never kill unrelated notebook processes.
            groups = {process.pid}
            try:
                if process.poll() is None and os.name == 'posix':
                    groups.update(descendant_sessions(process.pid))
            finally:
                for group in sorted(groups, reverse=True):
                    try:
                        os.killpg(group, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                process.wait(timeout=10)
            remaining = os.fstat(reader.fileno()).st_size - reader.tell()
            while remaining > 0 and (chunk := reader.read(min(65536, remaining))):
                print(decoder.decode(chunk), end='', flush=True)
                remaining -= len(chunk)
            print(decoder.decode(b'', final=True), end='', flush=True)
        if process.returncode:
            raise RuntimeError(f'Command failed ({process.returncode}); see {log}.')
        print(f'[done] {log.name}: {time.monotonic()-started:.1f}s', flush=True)


def execute(script, args, log):
    entry = ([str(ROOT/script)] if script in ('matched_run.py', 'anchor_extension.py')
             else [str(ROOT/'run_upstream.py'), script])
    run_logged([sys.executable, '-u', *entry, *map(str, args)], ROOT/'logs'/log)


def check_process_cleanup():
    with tempfile.TemporaryDirectory(prefix='nafc_process_check_') as folder:
        folder = Path(folder)
        code = ('import os,subprocess,sys; '
                'p=subprocess.Popen([sys.executable,"-c","import time; time.sleep(30)"]); '
                'print(p.pid,flush=True); os._exit(0)')
        started = time.monotonic()
        run_logged([sys.executable, '-u', '-c', code], folder/'orphan.log')
        if time.monotonic() - started >= 10:
            raise RuntimeError('Process cleanup self-check took too long; do not start GPU work.')
        worker = int((folder/'orphan.log').read_text().strip())
        status = Path(f'/proc/{worker}/stat')
        deadline = time.monotonic() + 2
        while status.exists():
            try:
                state = status.read_text().rsplit(')', 1)[1].split()[0]
            except FileNotFoundError:
                break
            if state in ('Z', 'X'):
                break
            if time.monotonic() >= deadline:
                raise RuntimeError('Worker survived process cleanup; do not start GPU work.')
            time.sleep(.05)
    print('[self-check] Orphan holding stdout was cleaned up without waiting for EOF.', flush=True)


def initialize():
    p = check_sources()
    import torch
    if not torch.cuda.is_available() or 'A100' not in torch.cuda.get_device_name(0):
        raise RuntimeError('Select an A100 GPU runtime.')
    for package, expected in p['versions'].items():
        if importlib.metadata.version(package) != expected:
            raise RuntimeError(f'{package} must be {expected}; use the notebook setup cell.')
    versions = {name: importlib.metadata.version(name) for name in (*p['versions'], 'math-verify', 'datasets')}
    current = {'plan': p, 'software': versions, 'gpu': torch.cuda.get_device_name(0)}
    target = ROOT/'experiment_identity.json'
    if target.exists():
        saved = read(target)
        if not same_experiment(saved.get('plan'), p) or {**saved, 'plan': p} != current:
            raise ValueError('Saved output belongs to a different model, configuration or environment.')
        if saved['plan']['anchor_pool'] != p['anchor_pool'] and any((ROOT/'runs').rglob('*')):
            raise ValueError('Cannot extend the anchor pool after training has started.')
    write(target, current)
    from huggingface_hub import snapshot_download
    snapshot_download(p['model'], revision=p['model_revision'], local_dir=MODEL,
                      allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model'])
    return p


def prepare():
    p = initialize()
    from run_upstream import load_split
    if any(len(load_split(p['dataset'], split, 2000)) != 2000 for split in ('train', 'test')):
        raise ValueError('The frozen Olympic partitions must contain 2,000 questions each.')
    common = ['--model', MODEL, '--dataset', p['dataset'], '--max_len', p['max_tokens'], '--n', 8, '--limit', 2000]
    if not (ROOT/'buckets_train_B440.json').exists():
        execute('prepare_buckets.py', common+['--split', 'train', '--budget', 440,
                '--out', 'buckets_train_B440.json'], 'prepare_train.log')
    if not (ROOT/'buckets_train_B358.json').exists():
        execute('prepare_buckets.py', ['--reuse_samples', 'buckets_train_B440.json', '--budget', 358,
                '--out', 'buckets_train_B358.json'], 'reuse_train_B358.log')
    if not (ROOT/'buckets_test_B440.json').exists():
        execute('prepare_buckets.py', common+['--split', 'test', '--budgets_from', 'buckets_train_B440.json',
                '--out', 'buckets_test_B440.json'], 'prepare_test.log')
    (ROOT/'results/texts').mkdir(parents=True, exist_ok=True)
    if not (ROOT/'results/base.json').exists():
        evaluate_model(MODEL, 'base')
    if not (ROOT/'anchors.json').exists() or len(read(ROOT/'anchors.json')['items']) < p['anchor_count']:
        execute('anchor_extension.py', [], 'anchors.log')
    for name in ('buckets_train_B440.json', 'buckets_train_B358.json', 'buckets_test_B440.json'):
        d = read(ROOT/name)
        if (len(d['items']) != 2000 or d['meta']['max_len'] != 1536
            or d['meta']['model'] != str(MODEL) or d['meta']['dataset'] != p['dataset']):
            raise ValueError('Wrong matched data partition: '+name)
    if len(read(ROOT/'anchors.json')['items']) != 128:
        raise ValueError('Fewer than 128 stable anchors; no arm may train. Preserve the diagnostic output.')
    print('Preparation complete: one 1536-token cap; 128 anchors; five arms x three seeds.', flush=True)


def evaluate_model(path, name):
    p = read(ROOT/'plan.json')
    execute('evaluate.py', ['--model', path, '--buckets', 'buckets_test_B440.json', '--n', p['test_draws'],
            '--seed', p['test_seed'], '--max_len', p['max_tokens'], '--gpu_mem', p['eval_vllm_mem'],
            '--out', f'results/{name}.json', '--save_texts', f'results/texts/{name}.jsonl.gz'], f'eval_{name}.log')


def latest_checkpoint(folder):
    checkpoints = sorted(folder.glob('checkpoint-*'), key=lambda path: int(path.name.split('-')[-1]), reverse=True)
    for path in checkpoints:
        needed = ('adapter_model.safetensors', 'adapter_config.json', 'trainer_state.json', 'optimizer.pt',
                  'scheduler.pt', 'rng_state.pth', 'lambda_state.json', 'flip_state.json')
        if all((path/name).exists() for name in needed):
            return path
    if checkpoints:
        raise ValueError('No complete resumable checkpoint; keep the output and inspect it before restarting.')
    return None


def merge_adapter(checkpoint, output):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map='cpu')
    model = PeftModel.from_pretrained(base, checkpoint).merge_and_unload()
    model.save_pretrained(output)
    AutoTokenizer.from_pretrained(MODEL).save_pretrained(output)


def train_one(arm, seed):
    initialize()
    if not (ROOT/'anchors.json').exists() or len(read(ROOT/'anchors.json')['items']) != 128:
        raise ValueError('Complete preparation before training.')
    name = run_name(arm, seed); folder = ROOT/'runs'/name
    args = train_args(arm, seed)
    if not (folder/'run_info.json').exists():
        checkpoint = latest_checkpoint(folder)
        if checkpoint:
            args += ['--resume', str(checkpoint)]
        elif folder.exists() and any(folder.iterdir()):
            raise ValueError('Partial run without a complete checkpoint; refusing to silently restart.')
        execute('train.py', args, f'train_{name}.log')
    p = read(ROOT/'plan.json')
    for step in [p['steps'], *p['keep_steps']]:
        key = name if step == p['steps'] else f'{name}_step{step}'
        if (ROOT/f'results/{key}.json').exists():
            continue
        checkpoint = folder/(f'checkpoint-{step}' if step == p['steps'] else f'keep/step-{step}')
        merged = ROOT.parent/'merged'/key
        # Merge in a separate process, releasing CPU tensors before the vLLM evaluator starts.
        if not (merged/'config.json').exists():
            execute('matched_run.py', ['merge', '--checkpoint', checkpoint, '--merged', merged], f'merge_{key}.log')
        evaluate_model(merged, key)
        if not merged.resolve().is_relative_to((ROOT.parent/'merged').resolve()):
            raise ValueError('Unexpected merged-model cleanup path.')
        shutil.rmtree(merged)


def report():
    from fixflip import Scores, fixflip, compare
    p = check_sources()
    base = Scores.load(str(ROOT/'buckets_test_B440.json'))
    noise = fixflip(base, Scores.load(str(ROOT/'results/base.json')))
    table = ['# Gemma / Olympic mathematics: Ngoc-matched PI configuration', '',
             '| Step | Seed | Arm | Accuracy (%) | Mean tokens | Robust regressions | Newly reliable | Boxed (%) | Capped (%) |',
             '|---|---|---|---:|---:|---:|---:|---:|---:|']
    payload = {'plan': p, 'noise': noise, 'milestones': {}}
    for step in p['test_steps']:
        paired = {}
        payload['milestones'][str(step)] = {}
        for seed in p['seeds']:
            paired[seed] = {}
            for arm in ARMS:
                name = run_name(arm, seed)+(f'_step{step}' if step != 300 else '')
                path = ROOT/f'results/{name}.json'
                if not path.exists():
                    raise ValueError('All prespecified seeds, arms and milestones are required: '+name)
                d = read(path); score = Scores.load(str(path)); paired[seed][arm] = score
                metrics = fixflip(base, score)
                payload['milestones'][str(step)][name] = metrics
                o = d['overall']
                broken = int(((base.p >= .75) & (score.p <= .25)).sum())
                fixed = int(((base.p <= .25) & (score.p >= .75)).sum())
                table.append(f"| {step} | {seed} | {arm} | {o['acc']:.2f} | {o['mean_len']:.1f} | {broken} | {fixed} | {o['boxed_rate']:.2f} | {o['truncated_rate']:.2f} |")
        comparisons = {}
        for control in ('mon', 'fix', 'kl', 'fg'):
            comparisons[control] = {str(seed): compare(base, paired[seed]['rep'], paired[seed][control])
                                    for seed in p['seeds']}
        payload['milestones'][str(step)]['paired_comparisons'] = comparisons
    table += ['', f"Base sampled accuracy: {100*base.acc:.2f}%. Unchanged-reference robust flip rate: {100*noise['robust_flip']:.2f}%.",
              '', 'All five arms and all three seeds are reported. Question-bootstrap intervals in comparison.json are conditional on each seed.',
              'Qwen-specific between-run variance estimates and numerical success thresholds are not transferred to Gemma.',
              'Historical 2048-token HF output is excluded. No longer-cap evaluation or best-checkpoint selection.']
    write(ROOT/'comparison.json', payload)
    (ROOT/'REPORT.md').write_text('\n'.join(table)+'\n', encoding='utf-8')
    print('\n'.join(table), flush=True)


def pack(archive):
    p = check_sources()
    allowed = {'plan.json', 'experiment_identity.json', 'anchors.json', 'buckets_train_B440.json',
               'buckets_train_B358.json', 'buckets_test_B440.json', 'comparison.json', 'REPORT.md'}
    archive = Path(archive).resolve()
    if archive.is_relative_to(ROOT):
        raise ValueError('Save the archive outside the experiment folder.')
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(archive)+'.tmp')
    files = [path for path in ROOT.rglob('*') if path.is_file() and not path.name.endswith('.tmp')
             and (path.relative_to(ROOT).parts[0] in EXPORT_DIRS or path.relative_to(ROOT).as_posix() in allowed)]
    try:
        with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as bundle:
            bundle.writestr(MANIFEST, json.dumps({'plan': p}))
            for path in files:
                if path.is_symlink():
                    raise ValueError('Symlinks are not exported.')
                bundle.write(path, path.relative_to(ROOT).as_posix())
        temporary.replace(archive)
    finally:
        temporary.unlink(missing_ok=True)
    return archive


def restore(archive):
    p = check_sources()
    with zipfile.ZipFile(archive) as bundle:
        if len(bundle.namelist()) != len(set(bundle.namelist())) or bundle.testzip() is not None:
            raise ValueError('Duplicate or damaged output archive.')
        saved_plan = read_zip_plan(bundle)
        if not same_experiment(saved_plan, p):
            raise ValueError('Different experiment: the historical 2048-token output cannot resume this run.')
        if saved_plan['anchor_pool'] != p['anchor_pool'] and any(n.startswith('runs/') for n in bundle.namelist()):
            raise ValueError('Cannot extend an anchor pool in an archive containing training progress.')
        if sum(info.file_size for info in bundle.infolist()) > shutil.disk_usage(ROOT).free:
            raise OSError('Not enough space to restore the output archive.')
        for info in bundle.infolist():
            if info.filename == MANIFEST:
                continue
            if info.filename == 'plan.json':
                if json.loads(bundle.read(info)) != saved_plan:
                    raise ValueError('Archive plan differs from its manifest.')
                continue
            name = PurePosixPath(info.filename)
            if (info.orig_filename != info.filename or name.is_absolute() or '..' in name.parts
                or chr(92) in info.filename or ':' in info.filename or not name.parts
                or stat.S_ISLNK(info.external_attr >> 16)
                or not (name.parts[0] in EXPORT_DIRS or info.filename in
                        {'experiment_identity.json', 'plan.json', 'anchors.json', 'buckets_train_B440.json',
                         'buckets_train_B358.json', 'buckets_test_B440.json', 'comparison.json', 'REPORT.md'})):
                raise ValueError('Unsafe archive member.')
            target = ROOT.joinpath(*name.parts).resolve()
            if not target.is_relative_to(ROOT):
                raise ValueError('Unsafe archive destination.')
            if target.exists() and target.read_bytes() != bundle.read(info):
                raise ValueError('Existing progress differs; restore into a fresh runtime.')
        for info in bundle.infolist():
            if info.filename in (MANIFEST, 'plan.json') or info.is_dir():
                continue
            target = ROOT/info.filename
            if target.exists():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            temporary = target.with_suffix(target.suffix+'.tmp')
            with bundle.open(info) as source, temporary.open('wb') as destination:
                shutil.copyfileobj(source, destination)
            temporary.replace(target)


def read_zip_plan(bundle):
    try:
        return json.loads(bundle.read(MANIFEST))['plan']
    except KeyError:
        raise ValueError('This archive is not from the Ngoc-matched Gemma notebook.') from None


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('command', choices=['prepare', 'train', 'report', 'merge'])
    parser.add_argument('--arm', choices=ARMS)
    parser.add_argument('--seed', type=int)
    parser.add_argument('--checkpoint')
    parser.add_argument('--merged')
    args = parser.parse_args()
    if args.command == 'prepare': prepare()
    elif args.command == 'train': train_one(args.arm, args.seed)
    elif args.command == 'report': report()
    elif args.command == 'merge': merge_adapter(args.checkpoint, args.merged)


### `prepare_buckets.py`

In [ ]:
%%writefile /content/gemma_ngoc/prepare_buckets.py
"""
prepare_buckets.py — Bước offline trước khi train (Proposal, mục 5, bước 1–2).

Cho model gốc giải mỗi câu N lần bằng vLLM, rồi:
  1. tính tỷ lệ giải đúng p(q) và gán nhóm độ khó (dễ / vừa / khó),
  2. phân bổ ngân sách B_k theo độ dài trung vị lời giải đúng, co giãn để trung bình có trọng số = B,
  3. ước lượng lambda ban đầu cho từng nhóm (và cho phương pháp toàn cục).

Ví dụ:
  python prepare_buckets.py --model Qwen/Qwen2.5-Math-1.5B-Instruct --dataset gsm8k --split train \
      --limit 2000 --budget 550 --out buckets_gsm8k_train.json

Chạy thêm một lần với --split test để có nhóm độ khó cho tập đánh giá
(dùng cùng --thresholds; ngân sách lấy từ file train, xem --budgets_from).
"""

from __future__ import annotations

import argparse
import json
import statistics

from common import chat_messages, is_correct, load_split
from constraints import calibrate_lambda


def bucket_of(p: float, thresholds: list[float]) -> int:
    """thresholds giảm dần, ví dụ [0.75, 0.25]: p>=0.75 -> 0 (dễ), p>=0.25 -> 1 (vừa), còn lại -> 2 (khó)."""
    for k, t in enumerate(thresholds):
        if p >= t:
            return k
    return len(thresholds)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=None)
    ap.add_argument("--dataset", default="gsm8k", choices=["gsm8k", "math500", "math", "olympic"])
    ap.add_argument("--split", default="train")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--n", type=int, default=8, help="số mẫu mỗi câu")
    ap.add_argument("--max_len", type=int, default=1024)
    ap.add_argument("--budget", type=float, default=550, help="ngân sách trung bình B")
    ap.add_argument("--thresholds", type=float, nargs="+", default=[0.75, 0.25])
    ap.add_argument("--beta", type=float, default=0.5, help="nhiệt độ KL cho hiệu chỉnh lambda")
    ap.add_argument("--budgets_from", default=None, help="file json train để tái dùng B_k (cho tập test)")
    ap.add_argument("--reuse_samples", default=None, help="file json cũ: dùng lại mẫu, chỉ tính lại B_k và lambda (không cần GPU)")
    ap.add_argument("--gpu_mem", type=float, default=0.85)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()

    if not args.reuse_samples and not args.model:
        ap.error("cần --model (hoặc --reuse_samples)")
    if args.budgets_from:   # tập test: dùng CÙNG ngưỡng chia nhóm và ngân sách với tập train
        _ref = json.load(open(args.budgets_from))
        args.budget, args.thresholds, args.max_len = _ref["budget"], _ref["thresholds"], _ref["max_len"]

    if args.reuse_samples:
        old = json.load(open(args.reuse_samples))
        raw = [(it["question"], it["answer"], [tuple(x) for x in it["samples"]]) for it in old["items"]]
        args.model, args.max_len = old["meta"]["model"], old["meta"]["max_len"]
        args.dataset, args.split = old["meta"].get("dataset", args.dataset), old["meta"].get("split", args.split)
    else:
        from transformers import AutoTokenizer
        from vllm import LLM, SamplingParams

        rows = load_split(args.dataset, args.split, args.limit)
        tok = AutoTokenizer.from_pretrained(args.model)
        prompts = [tok.apply_chat_template(chat_messages(r["question"]), tokenize=False, add_generation_prompt=True) for r in rows]
        llm = LLM(args.model, gpu_memory_utilization=args.gpu_mem, max_model_len=args.max_len + 512)
        sp = SamplingParams(n=args.n, temperature=1.0, top_p=1.0, max_tokens=args.max_len, seed=0)
        outs = llm.generate(prompts, sp)
        raw = [(r["question"], r["answer"], [(is_correct(c.text, r["answer"]), len(c.token_ids)) for c in o.outputs])
               for r, o in zip(rows, outs)]

    K = len(args.thresholds) + 1
    items = []
    for q, a, samples in raw:
        p = sum(ok for ok, _ in samples) / len(samples)
        items.append({"question": q, "answer": a, "p": p, "bucket": bucket_of(p, args.thresholds), "samples": samples})

    counts = [sum(it["bucket"] == k for it in items) for k in range(K)]
    weights = [c / len(items) for c in counts]

    # Độ dài tự nhiên của model gốc theo nhóm (trung vị lời giải đúng; nhóm không có lời giải đúng -> trung vị mọi mẫu)
    natural = []
    for k in range(K):
        grp = [s for it in items if it["bucket"] == k for s in it["samples"]]
        good = [L for ok, L in grp if ok] or [L for _, L in grp] or [args.budget]
        natural.append(statistics.median(good))
    base_stats = {
        "acc_by_bucket": [statistics.mean(it["p"] for it in items if it["bucket"] == k) if counts[k] else None for k in range(K)],
        "mean_len_by_bucket": [statistics.mean(L for it in items if it["bucket"] == k for _, L in it["samples"]) if counts[k] else None for k in range(K)],
        "acc": statistics.mean(it["p"] for it in items),
        "mean_len": statistics.mean(L for it in items for _, L in it["samples"]),
    }

    if args.budgets_from:
        ref = json.load(open(args.budgets_from))
        budgets, init_group, init_global = ref["budgets"], ref["init_lambdas"], ref["init_lambda_global"]
    else:
        scale = args.budget / sum(w * n for w, n in zip(weights, natural))
        budgets = [n * scale for n in natural]
        init_group = []
        for k in range(K):
            grp = [it["samples"] for it in items if it["bucket"] == k]
            lam_max = budgets[k] / max(args.max_len - budgets[k], 1.0)
            init_group.append(calibrate_lambda(grp, budgets[k], lam_max, beta=args.beta) if grp else 0.0)
        lam_max_g = args.budget / max(args.max_len - args.budget, 1.0)
        init_global = calibrate_lambda([it["samples"] for it in items], args.budget, lam_max_g, beta=args.beta)

    meta = {
        "model": args.model, "dataset": args.dataset, "split": args.split, "n": args.n,
        "max_len": args.max_len, "budget": args.budget, "thresholds": args.thresholds,
        "counts": counts, "weights": weights, "natural_len": natural,
        "budgets": budgets, "difficulty": [k / (K - 1) for k in range(K)],
        "init_lambdas": init_group, "init_lambda_global": init_global,
        "base": base_stats,
    }
    with open(args.out, "w") as f:
        json.dump({"meta": meta, **meta, "items": items}, f, ensure_ascii=False)

    print(json.dumps({k: v for k, v in meta.items()}, indent=2, ensure_ascii=False))
    print("\nRQ1 — độ chính xác model gốc theo nhóm:", base_stats["acc_by_bucket"])


if __name__ == "__main__":
    import os as _os, sys as _sys
    main()
    _sys.stdout.flush(); _sys.stderr.flush()
    _os._exit(0)   # thoát ngay: không chờ tiến trình nền của vLLM (từng làm ô treo hàng giờ)


### `run_upstream.py`

In [ ]:
%%writefile /content/gemma_ngoc/run_upstream.py
"""Supply the previously frozen Olympic data to Ngoc's training and evaluation scripts."""
import json
from pathlib import Path
import runpy
import sys
import common

ROOT = Path(__file__).resolve().parent
original_load_split = common.load_split


def load_split(dataset, split, limit=None):
    if dataset == 'olympic':
        data = json.loads((ROOT/'frozen_questions.json').read_text(encoding='utf-8'))
        if split == 'train':
            rows = data['train'] if limit is not None else data['train']+data['anchor_pool']
        elif split == 'test':
            rows = data['test']
        else:
            raise ValueError('Only the frozen Olympic train and test splits are available.')
        return rows if limit is None else rows[:limit]
    return original_load_split(dataset, split, limit)


if __name__ == '__main__':
    script, *arguments = sys.argv[1:]
    if script not in ('prepare_buckets.py', 'build_anchors.py', 'train.py', 'evaluate.py'):
        raise ValueError('Unexpected upstream script.')
    common.load_split = load_split
    sys.argv = [str(ROOT/script), *arguments]
    runpy.run_path(str(ROOT/script), run_name='__main__')


### `train.py`

In [ ]:
%%writefile /content/gemma_ngoc/train.py
"""
train.py — GRPO có ràng buộc độ dài, trên 1 GPU (Colab A100), bằng TRL + vLLM (colocate) + LoRA.

Mọi phương pháp dùng CÙNG script, chỉ khác --method, để so sánh công bằng:
  grpo | fixed | leash | laconic | dalg | dalg_param

Cơ chế: TRL gọi reward function một lần cho mỗi batch sinh (generation batch).
Trong reward function ta
  (1) tính reward Lagrange với lambda hiện tại  (bước primal dùng reward này),
  (2) cập nhật lambda từ độ dài vừa quan sát     (bước dual),
nên mỗi bước tối ưu có đúng một lần cập nhật đối ngẫu, như LACONIC/Leash.

Ví dụ:
  python train.py --method laconic --buckets buckets_gsm8k_train.json --output_dir runs/laconic_s0 --seed 0
  python train.py --method dalg --init calibrated --buckets buckets_gsm8k_train.json --output_dir runs/dalg_s0

v8 (nhánh RL của đồ án gộp) thêm, không đổi gì khi để mặc định:
  --beta 0.04                         GRPO + KL với model gốc (TRL tắt LoRA để lấy model tham chiếu)
  --flip_mode nafc|flipguard|robust_fixed|raw_adaptive|monitor --anchors anchors.json
                                      ràng buộc đúng → sai (flipcon.py); monitor = chỉ đo, không kéo
v9 thêm --flip_mode nafc_replay|replay_fixed (sửa on-policy: anchor vào batch GRPO thay cho NLL) và
  --replay_frac_max (tỷ lệ prompt tối đa thay bằng anchor khi λ = 1)
v10 thêm --anchor_split 0.5 --split_seed 0: nửa anchor SỬA (được kéo/trộn), nửa ĐO (chỉ đo, điều khiển λ)
v11 thêm --keep_steps 50,100,150,200 (giữ adapter giữa chừng trong output_dir/keep/) và --save_total_limit
"""

from __future__ import annotations

import argparse
import json
import os
import statistics

from common import chat_messages, completion_text, is_correct
from constraints import AccuracyController, LagrangianController, clipped_cost


def decoupled_advantages(group_keys, rewards, costs, lams, eps=1e-6):
    groups = {}
    for i, g in enumerate(group_keys):
        groups.setdefault(g, []).append(i)
    out = [0.0] * len(rewards)
    for idx in groups.values():
        def z(vals):
            m = sum(vals) / len(vals)
            s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
            return [0.0 if s < eps else (v - m) / s for v in vals]
        zr = z([rewards[i] for i in idx])
        zc = z([costs[i] for i in idx])
        for j, i in enumerate(idx):
            out[i] = zr[j] - lams[i] * zc[j]
    return out


def build_controller(args, meta) -> LagrangianController:
    grouped = args.method in ("dalg", "dalg_param")
    budgets = meta["budgets"] if grouped else [meta["budget"]]
    init = None
    if args.lambda_init is not None and args.method not in ("grpo", "fixed"):
        init = [args.lambda_init] * (len(budgets))
    elif args.init == "calibrated" and args.method not in ("grpo", "fixed"):
        init = meta["init_lambdas"] if grouped else [meta["init_lambda_global"]]
    return LagrangianController(
        method=args.method,
        budgets=budgets,
        max_len=args.max_completion_length,
        difficulty=meta["difficulty"] if args.method == "dalg_param" else None,
        eta=args.eta,
        eta_param=args.eta,
        fixed_lambda=args.fixed_lambda,
        optimistic=args.optimistic,
        init_lambdas=init,
        leash_lambda_max=args.lambda_max,
        two_sided=args.two_sided,
        kp=args.kp,
    )


def acccon_advantages(group_keys, correct, lengths, lams, eps=1e-6):
    """A = (lam * z(đúng) - z_đúng(L)) / (1 + lam), chuẩn hoá trong từng nhóm rollout của cùng một prompt.
    z_đúng(L): độ dài chỉ được chuẩn hoá và phạt GIỮA CÁC LỜI GIẢI ĐÚNG (cần ít nhất 2); lời giải sai không
    nhận tín hiệu độ dài. Nhóm toàn sai -> advantage 0 (như GRPO), không bị đẩy viết ngắn."""
    groups = {}
    for i, g in enumerate(group_keys):
        groups.setdefault(g, []).append(i)
    out = [0.0] * len(correct)
    for idx in groups.values():
        def z(vals):
            m = sum(vals) / len(vals)
            s = (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5
            return [0.0 if s < eps else (v - m) / s for v in vals]
        zc = z([1.0 if correct[i] else 0.0 for i in idx])
        ok = [i for i in idx if correct[i]]
        zl = dict(zip(ok, z([float(lengths[i]) for i in ok]))) if len(ok) >= 2 else {}
        for j, i in enumerate(idx):
            out[i] = (lams[i] * zc[j] - zl.get(i, 0.0)) / (1.0 + lams[i])
    return out


def make_acccon_reward_fn(ctrl: AccuracyController, num_buckets: int):
    def acccon_reward(completions, completion_ids, answer, bucket, log_metric=None, **kwargs):
        texts = [completion_text(c) for c in completions]
        correct = [is_correct(t, a) for t, a in zip(texts, answer)]
        lengths = [len(ids) for ids in completion_ids]
        buckets = [int(b) for b in bucket]
        lams = [ctrl.lambdas[b] for b in buckets]
        rewards = acccon_advantages([str(p) for p in kwargs["prompts"]], correct, lengths, lams)   # primal: lambda_t
        stats = ctrl.dual_update(correct, buckets)                                                # dual
        if log_metric is not None:
            log_metric("task/accuracy", sum(correct) / len(correct))
            log_metric("task/mean_len", statistics.mean(lengths))
            for k in range(num_buckets):
                idx = [i for i, b in enumerate(buckets) if b == k]
                if idx:
                    log_metric(f"task/accuracy_b{k}", sum(correct[i] for i in idx) / len(idx))
                    log_metric(f"task/mean_len_b{k}", statistics.mean(lengths[i] for i in idx))
            log_metric("constraint/mean_len_g0", statistics.mean(lengths))
            for name, v in stats.items():
                log_metric(name, v)
        return rewards

    return acccon_reward


def make_reward_fn(ctrl: LagrangianController, num_buckets: int, decoupled: bool = False, topq: float = 1.0):
    def lagrangian_reward(completions, completion_ids, answer, bucket, log_metric=None, **kwargs):
        texts = [completion_text(c) for c in completions]
        correct = [is_correct(t, a) for t, a in zip(texts, answer)]
        lengths = [len(ids) for ids in completion_ids]
        buckets = [int(b) for b in bucket]

        if decoupled:
            ks = [ctrl.group_of(b) for b in buckets]
            if ctrl.two_sided:   # chi phí có dấu: λ > 0 đẩy ngắn lại, λ < 0 kéo dài ra về phía B
                costs = [(L - ctrl.budget(k)) / ctrl.budget(k) for L, k in zip(lengths, ks)]
            else:
                costs = [clipped_cost(L, ctrl.budget(k)) for L, k in zip(lengths, ks)]
            lams = [ctrl.lambdas[k] for k in ks]
            keys = [str(p) for p in kwargs["prompts"]]
            if topq < 1.0:   # GRPO-λ: chỉ phạt độ dài ở các prompt có tỷ lệ đúng thuộc top q của batch
                acc_by = {}
                for i, kk in enumerate(keys):
                    acc_by.setdefault(kk, []).append(1.0 if correct[i] else 0.0)
                accs = {kk: sum(v) / len(v) for kk, v in acc_by.items()}
                ranked = sorted(accs.values(), reverse=True)
                n_keep = max(1, int(round(topq * len(ranked))))
                thr = ranked[n_keep - 1]
                lams = [l if accs[kk] >= thr and accs[kk] > 0 else 0.0 for l, kk in zip(lams, keys)]
            rewards = decoupled_advantages(keys, [1.0 if c else 0.0 for c in correct], costs, lams)
        else:
            rewards = ctrl.shaped_rewards(correct, lengths, buckets)   # primal: lambda_t
        stats = ctrl.dual_update(lengths, buckets)                 # dual: lambda_t -> lambda_{t+1}

        if log_metric is not None:
            log_metric("task/accuracy", sum(correct) / len(correct))
            log_metric("task/mean_len", statistics.mean(lengths))
            for k in range(num_buckets):
                idx = [i for i, b in enumerate(buckets) if b == k]
                if idx:
                    log_metric(f"task/accuracy_b{k}", sum(correct[i] for i in idx) / len(idx))
                    log_metric(f"task/mean_len_b{k}", statistics.mean(lengths[i] for i in idx))
            for name, v in stats.items():
                log_metric(name, v)
        return rewards

    return lagrangian_reward


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["grpo", "fixed", "leash", "laconic", "dalg", "dalg_param", "acccon"])
    ap.add_argument("--buckets", required=True, help="file json từ prepare_buckets.py (split train)")
    ap.add_argument("--model", default=None, help="mặc định lấy từ file buckets")
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--seed", type=int, default=0)
    # Đối ngẫu
    ap.add_argument("--eta", type=float, default=0.002)
    ap.add_argument("--fixed_lambda", type=float, default=0.01)
    ap.add_argument("--init", choices=["zero", "calibrated"], default="zero")
    ap.add_argument("--lambda_init", type=float, default=None, help="khởi tạo lambda bằng hằng số (Leash dùng 0.1)")
    ap.add_argument("--lambda_max", type=float, default=1.0, help="trần lambda (leash; và mọi phương pháp khi --adv_norm decoupled)")
    ap.add_argument("--optimistic", action="store_true")
    ap.add_argument("--two_sided", action="store_true", help="λ được âm, chi phí có dấu (cần --adv_norm decoupled)")
    ap.add_argument("--kp", type=float, default=0.0, help="hệ số tỉ lệ của bộ điều khiển PI cho λ")
    ap.add_argument("--topq", type=float, default=1.0, help="GRPO-λ: chỉ phạt độ dài ở tỷ lệ topq prompt đúng nhiều nhất (1.0 = tắt)")
    ap.add_argument("--acc_ref", default=None, help="acccon: file json {\"curves\": [[tau_0(t)...], ...]} (đã trừ eps)")
    ap.add_argument("--acc_eta", type=float, default=0.5)
    ap.add_argument("--acc_lam_init", type=float, default=1.0)
    ap.add_argument("--acc_lam_min", type=float, default=0.5)
    ap.add_argument("--acc_lam_max", type=float, default=4.0)
    # GRPO
    ap.add_argument("--max_steps", type=int, default=300)
    ap.add_argument("--num_generations", type=int, default=8)
    ap.add_argument("--prompts_per_step", type=int, default=8)
    ap.add_argument("--micro_batch", type=int, default=4, help="số completion mỗi lần forward/backward")
    ap.add_argument("--max_completion_length", type=int, default=None, help="mặc định = max_len trong file buckets")
    ap.add_argument("--lr", type=float, default=1e-5)
    ap.add_argument("--lora_r", type=int, default=32)
    ap.add_argument("--vllm_mem", type=float, default=0.3)
    ap.add_argument("--save_steps", type=int, default=25)
    ap.add_argument("--save_total_limit", type=int, default=3, help="số checkpoint đầy đủ giữ lại (để train tiếp)")
    ap.add_argument("--keep_steps", default="", help="vd '50,100,150,200': chép adapter LoRA ở các bước này vào output_dir/keep/step-N "
                    "(chỉ adapter + trạng thái λ, không có optimizer) để đánh giá giữa chừng")
    ap.add_argument("--resume", default=None, help="đường dẫn checkpoint-xxx để train tiếp")
    ap.add_argument("--scale_rewards", default="none", choices=["group", "batch", "none"])
    ap.add_argument("--adv_norm", default="sum", choices=["sum", "decoupled"])
    ap.add_argument("--report_to", default="none", help="wandb | none")
    ap.add_argument("--no_vllm", action="store_true", help="chỉ để chạy thử trên CPU")
    ap.add_argument("--merged_dir", default=None, help="nơi lưu model đã gộp LoRA (mặc định output_dir/merged); nên để ngoài Drive vì ~3 GB")
    # v8: KL và ràng buộc đúng → sai
    ap.add_argument("--beta", type=float, default=0.0, help="hệ số KL với model gốc (0 = tắt, như mọi lần chạy trước)")
    ap.add_argument("--flip_mode", default="none", choices=["none", "nafc", "flipguard", "robust_fixed", "raw_adaptive", "monitor",
                                                           "nafc_replay", "replay_fixed"])
    ap.add_argument("--anchors", default=None, help="file json từ build_anchors.py")
    ap.add_argument("--anchor_limit", type=int, default=None, help="chỉ dùng N anchor đầu (chạy thử)")
    ap.add_argument("--flip_every", type=int, default=25, help="kiểm tra anchor mỗi N bước tối ưu")
    ap.add_argument("--flip_k", type=int, default=4, help="số mẫu mỗi anchor mỗi lần kiểm tra")
    ap.add_argument("--flip_eps", type=float, default=None, help="ngân sách; mặc định = mức nhiễu của cách đếm + flip_margin")
    ap.add_argument("--flip_margin", type=float, default=0.02)
    ap.add_argument("--flip_eta", type=float, default=1.0, help="bước đối ngẫu của λ (chế độ tự điều chỉnh)")
    ap.add_argument("--flip_gamma", type=float, default=0.1, help="λ cố định (flipguard, robust_fixed)")
    ap.add_argument("--flip_lam_max", type=float, default=1.0)
    ap.add_argument("--flip_per_micro", type=int, default=1, help="số anchor tính NLL mỗi micro-batch khi đang kéo")
    ap.add_argument("--anchor_split", type=float, default=0.0,
                    help="tỷ lệ anchor SỬA (được kéo/trộn vào train); phần còn lại chỉ để đo và điều khiển λ. 0 = không tách")
    ap.add_argument("--split_seed", type=int, default=0, help="seed của phép tách anchor (cố định giữa các lần chạy)")
    ap.add_argument("--replay_frac_max", type=float, default=0.5,
                    help="nafc_replay/replay_fixed: tỷ lệ prompt mỗi batch thay bằng anchor khi λ = 1 (n = ceil(λ·tỷ lệ·G))")
    args = ap.parse_args()

    from datasets import Dataset
    from peft import LoraConfig
    from transformers import TrainerCallback
    from trl import GRPOConfig, GRPOTrainer

    data = json.load(open(args.buckets))
    meta = data["meta"]
    args.model = args.model or meta["model"]
    args.max_completion_length = args.max_completion_length or meta["max_len"]
    num_buckets = len(meta["budgets"])

    ds = Dataset.from_list([
        {"prompt": chat_messages(it["question"]), "answer": it["answer"], "bucket": it["bucket"]}
        for it in data["items"]
    ]).shuffle(seed=args.seed)

    if args.method == "acccon":
        assert args.acc_ref, "acccon cần --acc_ref"
        ctrl = AccuracyController(ref_curves=json.load(open(args.acc_ref))["curves"], eta=args.acc_eta,
                                  lam_init=args.acc_lam_init, lam_min=args.acc_lam_min, lam_max=args.acc_lam_max)
        args.scale_rewards, args.adv_norm = "none", "acccon"
    else:
        ctrl = build_controller(args, meta)
    if args.two_sided and args.adv_norm != "decoupled":
        ap.error("--two_sided cần --adv_norm decoupled")
    if args.method != "acccon" and args.adv_norm == "decoupled":
        args.scale_rewards = "none"
        ctrl.ceiling = lambda k: args.lambda_max
        ctrl.lambdas = [min(l, args.lambda_max) for l in ctrl.lambdas]
    if args.resume and os.path.exists(os.path.join(args.resume, "lambda_state.json")):
        ctrl.load_state_dict(json.load(open(os.path.join(args.resume, "lambda_state.json"))))
        print("Đã nạp lambda từ", args.resume, ctrl.lambdas)

    flip_ctrl, anchors = None, None
    if args.flip_mode != "none":
        from flipcon import AnchorBank, FlipController, default_eps, load_state, split_anchors
        assert args.anchors, "--flip_mode cần --anchors (file từ build_anchors.py)"
        anchors = AnchorBank.load(args.anchors, args.anchor_limit)
        eps = args.flip_eps if args.flip_eps is not None else default_eps(anchors.meta, args.flip_mode, args.flip_margin)
        repair_idx, held_idx = (split_anchors(len(anchors), args.anchor_split, args.split_seed)
                                if args.anchor_split > 0 else (None, None))
        flip_ctrl = FlipController(mode=args.flip_mode, eps=eps, eta=args.flip_eta, gamma=args.flip_gamma,
                                   lam_max=args.flip_lam_max, repair_idx=repair_idx, held_idx=held_idx)
        if held_idx is not None:
            print(f"Tách anchor: {len(repair_idx)} SỬA, {len(held_idx)} ĐO (split_seed={args.split_seed})", flush=True)
        if args.resume and load_state(flip_ctrl, args.resume):
            print("Đã nạp trạng thái ràng buộc đúng → sai từ", args.resume, "λ =", flip_ctrl.lam)
        print(f"Ràng buộc đúng → sai: mode={args.flip_mode}, {len(anchors)} anchor, ε={eps:.4f}, "
              f"mỗi {args.flip_every} bước × {args.flip_k} mẫu", flush=True)

    keep_steps = {int(x) for x in args.keep_steps.split(",") if x.strip()}
    bad = sorted(k for k in keep_steps if k % args.save_steps)
    if bad:
        ap.error(f"--keep_steps {bad} không chia hết cho --save_steps {args.save_steps}")

    class SaveLambda(TrainerCallback):
        def on_save(self, targs, state, control, **kw):
            path = os.path.join(targs.output_dir, f"checkpoint-{state.global_step}")
            os.makedirs(path, exist_ok=True)
            json.dump(ctrl.state_dict(), open(os.path.join(path, "lambda_state.json"), "w"))
            if flip_ctrl is not None:
                json.dump(flip_ctrl.state_dict(), open(os.path.join(path, "flip_state.json"), "w"))
            if state.global_step in keep_steps:
                import shutil
                dst = os.path.join(targs.output_dir, "keep", f"step-{state.global_step}")
                os.makedirs(dst, exist_ok=True)
                for f in ("adapter_model.safetensors", "adapter_config.json", "lambda_state.json", "flip_state.json"):
                    if os.path.exists(os.path.join(path, f)):
                        shutil.copy2(os.path.join(path, f), os.path.join(dst, f))
                print(f"Đã giữ adapter bước {state.global_step} → {dst}", flush=True)

    completions_per_step = args.prompts_per_step * args.num_generations
    assert completions_per_step % args.micro_batch == 0, "prompts_per_step*num_generations phải chia hết cho micro_batch"

    config = GRPOConfig(
        output_dir=args.output_dir,
        seed=args.seed,
        max_steps=args.max_steps,
        learning_rate=args.lr,
        lr_scheduler_type="constant_with_warmup",
        warmup_steps=10,
        per_device_train_batch_size=args.micro_batch,
        gradient_accumulation_steps=completions_per_step // args.micro_batch,
        num_generations=args.num_generations,
        max_completion_length=args.max_completion_length,
        temperature=1.0,
        beta=args.beta,
        loss_type="dr_grpo",        # bỏ chuẩn hoá theo độ dài câu trả lời, tránh nhiễu với ràng buộc độ dài
        scale_rewards=args.scale_rewards,
        use_vllm=not args.no_vllm,
        vllm_mode="colocate",
        vllm_gpu_memory_utilization=args.vllm_mem,
        bf16=not args.no_vllm,
        gradient_checkpointing=not args.no_vllm,
        logging_steps=1,
        save_steps=args.save_steps,
        save_total_limit=args.save_total_limit,
        report_to=args.report_to,
        log_completions=False,
        model_init_kwargs={"dtype": "bfloat16"} if not args.no_vllm else None,
    )

    trainer_cls, extra = GRPOTrainer, {}
    if flip_ctrl is not None:
        from flipcon import make_flip_trainer
        trainer_cls = make_flip_trainer(GRPOTrainer)
        extra = dict(flip_ctrl=flip_ctrl, anchors=anchors, flip_every=args.flip_every, flip_k=args.flip_k,
                     flip_per_micro=args.flip_per_micro, replay_frac_max=args.replay_frac_max)

    trainer = trainer_cls(
        model=args.model,
        reward_funcs=(make_acccon_reward_fn(ctrl, num_buckets) if args.method == "acccon"
                      else make_reward_fn(ctrl, num_buckets, args.adv_norm == "decoupled", args.topq)),
        args=config,
        train_dataset=ds,
        peft_config=LoraConfig(r=args.lora_r, lora_alpha=2 * args.lora_r, lora_dropout=0.0,
                               target_modules="all-linear", task_type="CAUSAL_LM"),
        callbacks=[SaveLambda()],
        **extra,
    )
    trainer.train(resume_from_checkpoint=args.resume)
    trainer.save_state()   # ghi trainer_state.json (log_history) để vẽ quỹ đạo lambda

    # Gộp LoRA để evaluate.py nạp thẳng bằng vLLM
    merged_dir = args.merged_dir or os.path.join(args.output_dir, "merged")
    merged = trainer.model.merge_and_unload()
    merged.save_pretrained(merged_dir)
    trainer.processing_class.save_pretrained(merged_dir)
    info = {"method": args.method, "final_lambda_state": ctrl.state_dict(), "args": vars(args)}
    if flip_ctrl is not None:
        from flipcon import read_flip_log, summarize_log
        info["flip"] = {"final_state": flip_ctrl.state_dict(),
                        "summary": summarize_log(read_flip_log(os.path.join(args.output_dir, "flip_log.jsonl")))}
    with open(os.path.join(args.output_dir, "run_info.json"), "w") as f:
        json.dump(info, f, indent=2)


if __name__ == "__main__":
    import os as _os, sys as _sys
    main()
    _sys.stdout.flush(); _sys.stderr.flush()
    _os._exit(0)   # thoát ngay: không chờ tiến trình nền của vLLM (từng làm ô treo hàng giờ)


### `anchor_extension.py`

In [ ]:
%%writefile /content/gemma_ngoc/anchor_extension.py
"""Resume anchor selection, extending only the pre-frozen unused candidate bank."""
import json
from pathlib import Path
import statistics

from build_anchors import noise_floors, norm_q, pick_anchors
from common import chat_messages, is_correct
from matched_run import MODEL, check_sources, read, write

ROOT = Path(__file__).resolve().parent


def validate_saved(saved, plan, bank):
    meta, items = saved['meta'], saved['items']
    expected = {'model': str(MODEL), 'dataset': plan['dataset'], 'train_limit': plan['train_count'],
                'k_select': 8, 'k_check': 4, 'hi': .75, 'lo': .25, 'max_len': plan['max_tokens'],
                'temperature': 1.0}
    if any(meta.get(k) != v for k, v in expected.items()):
        raise ValueError('Saved anchors use different model, data, decoding or selection settings.')
    allowed = {norm_q(r['question']): r['answer'] for r in bank}
    seen = set()
    for row in items:
        key = norm_q(row['question'])
        if (key in seen or key not in allowed or row['answer'] != allowed[key]
                or row['p_select'] < .75 or not row.get('base_response')
                or not row.get('base_response_ids')):
            raise ValueError('Saved anchor does not match the frozen eligible candidate bank.')
        seen.add(key)
    if meta.get('n_anchor') != len(items):
        raise ValueError('Saved anchor count differs from its metadata.')


def build(generate):
    plan = check_sources()
    frozen = read(ROOT/'frozen_questions.json')
    bank = frozen['anchor_pool']
    initial, target = plan['anchor_initial_pool'], plan['anchor_count']
    blocked = {norm_q(r['question']) for split in ('train', 'test') for r in frozen[split]}
    keys = [norm_q(r['question']) for r in bank]
    if len(bank) != plan['anchor_pool'] or len(keys) != len(set(keys)) or blocked.intersection(keys):
        raise ValueError('Anchor candidates must be unique and disjoint from frozen train/test data.')
    output = ROOT/'anchors.json'
    if output.exists():
        saved = read(output); validate_saved(saved, plan, bank)
        if len(saved['items']) == target:
            print(f'Anchors already complete: {target}; no generation repeated.', flush=True)
            return
    folder = ROOT/'results/anchor_extension'
    folder.mkdir(parents=True, exist_ok=True)
    cache_path = folder/'selection.json'
    identity = {'plan': plan, 'environment': read(ROOT/'experiment_identity.json')}
    if cache_path.exists():
        cache = read(cache_path)
        if cache['identity'] != identity:
            raise ValueError('Cached selection belongs to a different experiment or environment.')
    else:
        cache = {'identity': identity, 'retained': [], 'initial_count': 0,
                 'initial_correct_sum': 0.0, 'scored': []}
        if output.exists():
            saved = read(output); meta = saved['meta']
            validate_saved(saved, plan, bank[:initial])
            if meta['pool'] != initial or meta['candidates'] != len(saved['items']):
                raise ValueError('Cannot reconstruct the initial pool from a truncated anchor selection.')
            write(folder/'anchors_initial.json', saved)
            cache.update(retained=saved['items'], initial_count=initial,
                         initial_correct_sum=meta['pool_base_acc'] * initial)
            print(f'Reusing {len(saved["items"])} qualified anchors from {initial} tested candidates.', flush=True)
        write(cache_path, cache)

    def score_until(end):
        start = cache['initial_count'] + len(cache['scored'])
        while start < end:
            batch = bank[start:min(start + plan['anchor_batch_size'], end)]
            outputs = generate(batch, 8, 0)
            if len(outputs) != len(batch) or any(len(o.outputs) != 8 for o in outputs):
                raise ValueError('Incomplete anchor selection batch; prior cached batches are preserved.')
            scored = []
            for row, out in zip(batch, outputs):
                ok = [is_correct(c.text, row['answer']) for c in out.outputs]
                good = next((c for c, correct in zip(out.outputs, ok) if correct), None)
                scored.append({**row, 'p_select': sum(ok)/8, 'base_first_correct': bool(ok[0]),
                               'base_response': good.text if good else None,
                               'base_response_ids': list(good.token_ids) if good else None,
                               'base_response_len': len(good.token_ids) if good else None})
            cache['scored'].extend(scored)
            write(cache_path, cache)
            start += len(batch)
            print(f'[selection saved] {start}/{end} candidates; '
                  f'{sum(r["p_select"] >= .75 for r in cache["retained"] + cache["scored"])} qualified',
                  flush=True)

    score_until(initial)
    eligible = cache['retained'] + cache['scored']
    if cache.get('extended') or sum(r['p_select'] >= .75 for r in eligible) < target:
        cache['extended'] = True
        write(cache_path, cache)
        print(f'Initial pool insufficient; extending to the fixed maximum {len(bank)} candidates.', flush=True)
        score_until(len(bank))
        eligible = cache['retained'] + cache['scored']
    order = {key: i for i, key in enumerate(keys)}
    eligible.sort(key=lambda row: order[norm_q(row['question'])])
    anchors = pick_anchors(eligible, target, .75, 0)
    if len(anchors) != target:
        raise RuntimeError(f'Found {len(anchors)}/{target} eligible anchors after the fixed pool; '
                           'all selection results are cached. No threshold or training count was changed.')
    if 'calibrated' in cache:
        anchors = cache['calibrated']
    else:
        print(f'[calibration] {target} anchors x 4 independent responses', flush=True)
        outputs = generate(anchors, 4, 1)
        if len(outputs) != target or any(len(o.outputs) != 4 for o in outputs):
            raise ValueError('Incomplete calibration; selection cache is preserved.')
        for row, out in zip(anchors, outputs):
            ok = [is_correct(c.text, row['answer']) for c in out.outputs]
            row.update(p_check=sum(ok)/4, check_first_correct=bool(ok[0]))
        cache['calibrated'] = anchors
        write(cache_path, cache)
    tested = cache['initial_count'] + len(cache['scored'])
    meta = {'model': str(MODEL), 'dataset': plan['dataset'], 'train_limit': plan['train_count'],
            'train_total': len(frozen['train']) + len(bank), 'train_used': len(frozen['train']),
            'train_rest': len(bank), 'dup_with_test_removed': 0, 'pool': tested,
            'candidates': sum(r['p_select'] >= .75 for r in eligible), 'n_anchor': target,
            'k_select': 8, 'k_check': 4, 'hi': .75, 'lo': .25, 'max_len': plan['max_tokens'],
            'temperature': 1.0, 'initial_pool': initial, 'pool_max': len(bank),
            'reused_qualified': len(cache['retained']), 'selection_batch_size': plan['anchor_batch_size'],
            'pool_base_acc': (cache['initial_correct_sum'] + sum(r['p_select'] for r in cache['scored']))/tested,
            'anchor_resp_len_mean': statistics.mean(r['base_response_len'] for r in anchors),
            **noise_floors(anchors, .25)}
    result = {'meta': meta, 'items': anchors}
    validate_saved(result, plan, bank)
    write(output, result)
    print(json.dumps(meta, indent=2), flush=True)


def main():
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    llm = tokenizer = None

    def generate(rows, n, seed):
        nonlocal llm, tokenizer
        if llm is None:
            tokenizer = AutoTokenizer.from_pretrained(str(MODEL))
            llm = LLM(str(MODEL), gpu_memory_utilization=.85, max_model_len=1536 + 1024)
        prompts = [tokenizer.apply_chat_template(chat_messages(r['question']), tokenize=False,
                                                 add_generation_prompt=True) for r in rows]
        return llm.generate(prompts, SamplingParams(n=n, temperature=1.0, top_p=1.0,
                                                    max_tokens=1536, seed=seed))

    build(generate)


if __name__ == '__main__':
    import os, sys, traceback
    code = 0
    try:
        main()
    except Exception:
        traceback.print_exc()
        code = 1
    except KeyboardInterrupt:
        code = 130
    # The supervisor reaps vLLM workers; do not wait for their Python shutdown hooks.
    sys.stdout.flush(); sys.stderr.flush(); os._exit(code)


### `plan.json`

In [ ]:
%%writefile /content/gemma_ngoc/plan.json
{
  "upstream_repository": "https://github.com/HoangLeminh17/DATN-FPTU",
  "upstream_commit": "786291e",
  "model": "google/gemma-3-1b-it",
  "model_revision": "dcc83ea841ab6100d6b47a070329e1ba4cf78752",
  "dataset": "olympic",
  "train_count": 2000,
  "test_count": 2000,
  "max_tokens": 1536,
  "pi_budget": 358,
  "reference_budget": 440,
  "seeds": [
    1,
    2,
    3
  ],
  "steps": 300,
  "prompts_per_step": 8,
  "num_generations": 8,
  "micro_batch": 4,
  "lora_r": 32,
  "learning_rate": 1e-05,
  "warmup_steps": 10,
  "precision": "bfloat16",
  "pi_eta": 0.05,
  "pi_kp": 4.0,
  "pi_lambda_max": 3.0,
  "anchor_pool": 2048,
  "anchor_count": 128,
  "anchor_split": 0.5,
  "split_seed": 0,
  "anchor_select_draws": 8,
  "anchor_check_draws": 4,
  "anchor_noise_repeats": 1,
  "audit_every": 25,
  "noise_margin": 0.02,
  "nafc_eta": 10.0,
  "replay_frac_max": 0.5,
  "test_draws": 4,
  "test_seed": 1,
  "keep_steps": [
    50,
    100,
    150,
    200
  ],
  "test_steps": [
    50,
    100,
    150,
    200,
    300
  ],
  "save_steps": 25,
  "save_total_limit": 3,
  "vllm_mem": 0.3,
  "eval_vllm_mem": 0.85,
  "beta": 0.04,
  "flipguard_gamma": 0.1,
  "fixed_replay_gamma": 0.5,
  "arms": {
    "mon": "monitor",
    "rep": "nafc_replay",
    "fix": "replay_fixed",
    "kl": "monitor",
    "fg": "flipguard"
  },
  "versions": {
    "torch": "2.10.0+cu128",
    "vllm": "0.19.1",
    "trl": "1.13.0",
    "transformers": "5.18.0",
    "peft": "0.21.1"
  },
  "source_sha256": {
    "build_anchors.py": "53b5174fc257dcd0452ae57a2ad68131ba5ffc4b074a989f63da516f75c3b572",
    "common.py": "a4950e6ca79368f1f741ccc3c3c918272754c80751afd59452764a97badf68b3",
    "constraints.py": "b6faa0ad66e063e06603aa75e9780f67c4518edbba7a24f8723dc39fdc65bc43",
    "evaluate.py": "b0673f51f6ff6ffb82c3207e41c6389c3dff871c163e053cf850a01b82641b20",
    "fixflip.py": "51ba99e80c1316133804a5b076c6b1da6b3be06f18c2ce6accaae1b85500abb6",
    "flipcon.py": "6658769eef936200f557272948e3c7d3c7bf7171907be9bd4d04669195a54875",
    "frozen_questions.json": "6fe942bdfda17453163799bc4d4ec9d671d3b37adc4a5800bd41f05d8adbd263",
    "matched_run.py": "15c67b15732fd2eba5c03059951f50ef1c892fe91ab60a7d1f0db0b67776d40e",
    "prepare_buckets.py": "1b4c77d644b84598ddbaa8b41a6ea5ad5ba4ec3d37c1f7f414409aee5b601168",
    "run_upstream.py": "f291e99dadaa0ccea7d25889d4b3bde76538f239070fbd2e4b781f6905a56fa9",
    "train.py": "a41a6d510d8819144d0cd29d0faa573ecb58ae25a47a9a1760cd4ffb0dac696c",
    "anchor_extension.py": "3a756c23ed8d3bfe93c51e973d3cd9d44cde09184f1a37c3ab0e31c4b585a3ad"
  },
  "unpinned_dependencies_as_upstream": [
    "math-verify"
  ],
  "source_line_endings": "LF; seven upstream scripts unchanged; prepare_buckets only adds the Olympic CLI choice",
  "train_dataset": "AI-MO/NuminaMath-1.5",
  "test_dataset": "KbsdJames/Omni-MATH",
  "train_sources": [
    "olympiads_ref",
    "olympiads"
  ],
  "data_revisions": {
    "numina": "1b05109f9e5c1ad06c0663519502416c30b300f8",
    "omni": "40ba231d8f16e29ecd40e6407e2c8640145a8f62"
  },
  "data_split_seed": 42,
  "data_scope": "Fixed random numeric-answer subset of Omni-MATH; not an official full-benchmark score.",
  "anchor_pool_selection": "First 800 frozen candidates initially; if insufficient, evaluate all remaining candidates in the pre-frozen 2048-question bank. Selection results are cached; threshold remains 6/8.",
  "upstream_data_adaptation": {
    "file": "prepare_buckets.py",
    "change": "Add olympic to --dataset choices; data supplied by run_upstream.py. Training/controller/evaluation algorithms unchanged."
  },
  "anchor_initial_pool": 800,
  "anchor_batch_size": 128
}


## 4. Verify sources and restore only matching output

For a new run, leave `RESTORE_MATCHED_OUTPUT=False`. For a later interrupted run, enable it and select the ZIP
downloaded from this matched notebook. Historical 2,048-token ZIPs are intentionally rejected.


In [ ]:
import importlib.util, json, hashlib
from google.colab import files
# IPython adds a final newline to magic cells; restore bytes only when the pinned hash proves the match.
saved_plan = json.loads((ROOT/"plan.json").read_text(encoding="utf-8"))
for name, expected in saved_plan["source_sha256"].items():
    path = ROOT/name
    data = path.read_bytes()
    if hashlib.sha256(data).hexdigest() == expected:
        continue
    if data.endswith(b"\n") and hashlib.sha256(data[:-1]).hexdigest() == expected:
        path.write_bytes(data[:-1])
    else:
        raise ValueError("Embedded source differs from its pinned hash: " + name)
spec = importlib.util.spec_from_file_location("gemma_matched_export", ROOT/"matched_run.py")
exporter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exporter)
PLAN = exporter.check_sources()
exporter.check_process_cleanup()
if RESTORE_MATCHED_OUTPUT:
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError("Select one latest output ZIP from this Ngoc-matched Gemma notebook.")
    name, data = next(iter(uploads.items()))
    archive = ARCHIVES/"restore_gemma_matched.zip"
    archive.write_bytes(data)
    exporter.restore(archive)
    del uploads, data

def command(*args):
    print("Starting:", *args, flush=True)
    exporter.run_logged([sys.executable, "-u", str(ROOT/"matched_run.py"), *map(str, args)],
                        ROOT/"logs"/("command_"+"_".join(map(str,args)).replace("--", "")+".log"))

print("Matched plan: five arms x three seeds; 300 steps; 1,536-token cap; PI budget 358.")


## 5. Prepare once

Completed upstream preparation files are reused. An interrupted upstream generation command may need to restart
that command; there is no per-16-question HF resume in the unchanged vLLM scripts. Insufficient stable anchors stop
the experiment if the fixed maximum candidate bank is exhausted. The anchor count and success threshold are
never lowered. Selection results are saved after each 128-candidate batch; a partial batch may need to repeat.

The runner follows the parent process, not pipe EOF. Finished stages kill only their own process group. It prints
a heartbeat every minute and aborts a stage after 15 minutes without new output, preserving completed files and
checkpoints. This guard can also stop a legitimate long silent operation; inspect the log rather than bypassing it.
Output ZIPs from the previous matched Olympic notebook remain compatible; other experiment changes are rejected.


In [ ]:
command("prepare")

## 6. Run all prespecified arms and seeds

The scripts keep full checkpoints every 25 steps, retain the last three full checkpoints, and preserve milestone
adapters at steps 50-200. After each training run, evaluate its final model and every retained milestone.
All arms start from the unchanged Gemma checkpoint. No best-seed or best-step selection.


In [ ]:
for seed in PLAN["seeds"]:
    for arm in ("mon", "rep", "fix", "kl", "fg"):
        print("Starting", arm, "seed", seed, flush=True)
        command("train", "--arm", arm, "--seed", seed)


## 7. Report every arm, seed and milestone

In [ ]:
command("report")
from IPython.display import display, Markdown
display(Markdown((ROOT/"REPORT.md").read_text(encoding="utf-8")))


## 8. Download all output

After an error or planned interruption, stop the running cell, wait for its subprocess to finish, then run this cell
manually. Download before disconnecting. ZIP includes preparation, results, response texts, logs, all remaining
full checkpoints and retained adapters. Model cache, token and merged model copies are excluded.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
button = widgets.Button(description="Download all output (.zip)", layout=widgets.Layout(width="260px"))
status = widgets.Output()

def download_output(_=None):
    button.disabled = True
    try:
        with status:
            status.clear_output(wait=True)
            print("Creating output archive; wait for download to finish...")
            archive = exporter.pack(ARCHIVES/"nafc_gemma_ngoc_matched_output.zip")
            files.download(str(archive))
    finally:
        button.disabled = False

button.on_click(download_output)
display(button, status)
